In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()

candidates = [
    "GITHUB_TOKEN",
    "GH_TOKEN",
    "GITHUB_PAT",
    "GH_PAT",
    "github_token",
    "github_pat",
]

print("Checking existing GitHub secret labels...\n")

found = []

for name in candidates:
    try:
        value = client.get_secret(name)
        if value:
            print(f"[FOUND] {name}  ({len(value)} characters)")
            found.append(name)
    except Exception:
        print(f"[----]  {name}")

print()

if found:
    print("Usable GitHub secret:", found[0])
else:
    print("NO ACCESSIBLE GITHUB SECRET FOUND")
    print("Open Add-ons -> Secrets and enable notebook access for your existing secret.")

Checking existing GitHub secret labels...

[FOUND] GITHUB_TOKEN  (93 characters)
[----]  GH_TOKEN
[----]  GITHUB_PAT
[----]  GH_PAT
[FOUND] github_token  (93 characters)
[----]  github_pat

Usable GitHub secret: GITHUB_TOKEN


In [3]:
# =============================================================================
# STAGE23 — GITHUB SECRET PATCH
# Run this ONCE before the main bootstrap cell
# =============================================================================

import os
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()

# Exact secret confirmed by diagnostic
github_token = client.get_secret("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError("GITHUB_TOKEN exists but returned an empty value.")

github_token = github_token.strip()

# Git / GitHub CLI compatible environment variables
os.environ["GITHUB_TOKEN"] = github_token
os.environ["GH_TOKEN"] = github_token
os.environ["GIT_TERMINAL_PROMPT"] = "0"

print("GitHub authentication patched successfully.")
print("Secret source : GITHUB_TOKEN")
print("Token length  :", len(github_token))
print("Token value   : [REDACTED]")

GitHub authentication patched successfully.
Secret source : GITHUB_TOKEN
Token length  : 93
Token value   : [REDACTED]


In [1]:
# =============================================================================
# STAGE 23 — FRESH KAGGLE NOTEBOOK BOOTSTRAP
# =============================================================================
#
# PURPOSE ONLY:
#   1. Securely patch GitHub authentication from Kaggle Secrets
#   2. Clone / reconnect the repository
#   3. Verify the frozen Stage22R provenance base
#   4. Inventory TOP-LEVEL Kaggle input mounts only
#   5. Install a hard path guard against Mar1 / Mar2 raw access
#   6. Record environment/bootstrap metadata
#
# THIS CELL DOES NOT:
#   - create stage23_0
#   - define Stage23 feature subsets
#   - read any dataset file
#   - recursively scan /kaggle/input
#   - train any model
#   - calculate any Stage23 metric
#   - open Mar1 / Mar2
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import json
import os
import re
import sys
import random


# =============================================================================
# 0. FROZEN PROJECT CONSTANTS
# =============================================================================

REPO_FULL_NAME = "themubasshir/ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"

WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")

REPO_DIR = WORKING_ROOT / "ids2018-validation-safe-ablation"

# Stage23 results location is declared only.
# DO NOT create it in this bootstrap cell.
STAGE23_RESULTS_ROOT = (
    REPO_DIR / "results" / "stage23_shortcut_feature_audit"
)

# ---------------------------------------------------------------------------
# Stage22R scientific seal
# ---------------------------------------------------------------------------
STAGE22R_FINAL_COMMIT = (
    "b5e44615269198426cc8a9aa3b3e701c2ca9e48e"
)
STAGE22R_FINAL_TAG = (
    "stage22r-final-single-holdout-v1"
)

# ---------------------------------------------------------------------------
# Stage22R publication closeout — expected Stage23 starting point
# ---------------------------------------------------------------------------
STAGE22R_CLOSEOUT_COMMIT = (
    "fafb131981b6e15e47bdf35fd8c18ea228680fe3"
)
STAGE22R_CLOSEOUT_TAG = (
    "stage22r-publication-closeout-v1"
)


print("=" * 78)
print("STAGE 23 — FRESH NOTEBOOK BOOTSTRAP")
print("=" * 78)
print()
print("Repository :", REPO_FULL_NAME)
print("Branch     : main")
print("Stage22R scientific seal :", STAGE22R_FINAL_COMMIT)
print("Stage22R closeout base   :", STAGE22R_CLOSEOUT_COMMIT)
print()


# =============================================================================
# 1. COMMAND HELPER
# =============================================================================

def run(cmd, cwd=None, check=True, show=True):
    """
    Run a command without shell=True.

    This prevents accidental shell expansion and avoids embedding secrets
    into shell command strings.
    """

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


# =============================================================================
# 2. LOAD GITHUB TOKEN FROM KAGGLE SECRETS
# =============================================================================
#
# Preferred Kaggle secret name:
#
#       GITHUB_TOKEN
#
# Also accepts:
#       GH_TOKEN
#
# The token is NEVER inserted into the Git remote URL.
# =============================================================================

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError(
        "kaggle_secrets is unavailable. "
        "This cell is intended to run inside Kaggle."
    ) from exc


secret_client = UserSecretsClient()


def get_first_secret(names):
    for name in names:
        try:
            value = secret_client.get_secret(name)
            if value and value.strip():
                return name, value.strip()
        except Exception:
            pass

    return None, None


secret_name, github_token = get_first_secret(
    ["GITHUB_TOKEN", "GH_TOKEN"]
)

if not github_token:
    raise RuntimeError(
        "\nGitHub token not found.\n\n"
        "Add a Kaggle Secret named:\n\n"
        "    GITHUB_TOKEN\n\n"
        "Then rerun this cell.\n"
        "Do NOT paste the token directly into notebook source."
    )

# Export only into this ephemeral notebook process.
os.environ["GITHUB_TOKEN"] = github_token
os.environ["GH_TOKEN"] = github_token

# Never allow Git to fall back to an interactive terminal prompt.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

print(f"GitHub credential loaded from Kaggle Secret: {secret_name}")
print("Token value: [REDACTED]")
print()


# =============================================================================
# 3. SECURE GIT_ASKPASS
# =============================================================================
#
# This allows git clone/fetch/push WITHOUT:
#
#   https://TOKEN@github.com/...
#
# Therefore:
#   git remote -v
#
# remains clean and cannot reveal the PAT.
# =============================================================================

ASKPASS = WORKING_ROOT / ".stage23_git_askpass.sh"

ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
    *Username*|*username*)
        printf '%s\\n' "x-access-token"
        ;;
    *Password*|*password*)
        printf '%s\\n' "$GITHUB_TOKEN"
        ;;
    *)
        exit 1
        ;;
esac
""",
    encoding="utf-8",
)

ASKPASS.chmod(0o700)

os.environ["GIT_ASKPASS"] = str(ASKPASS)

# Remove ordinary Python variable containing token.
# Git still gets it through the process environment.
del github_token


# =============================================================================
# 4. GIT IDENTITY
# =============================================================================
#
# Optional Kaggle secrets:
#
#   GIT_USER_NAME
#   GIT_USER_EMAIL
#
# If absent, use the verified GitHub account identity.
# =============================================================================

def optional_secret(name, default):
    try:
        value = secret_client.get_secret(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass
    return default


GIT_USER_NAME = optional_secret(
    "GIT_USER_NAME",
    "themubasshir",
)

GIT_USER_EMAIL = optional_secret(
    "GIT_USER_EMAIL",
    "10107331+themubasshir@users.noreply.github.com",
)

run([
    "git", "config", "--global",
    "user.name", GIT_USER_NAME
], show=False)

run([
    "git", "config", "--global",
    "user.email", GIT_USER_EMAIL
], show=False)

run([
    "git", "config", "--global",
    "--add", "safe.directory", str(REPO_DIR)
], check=False, show=False)


# =============================================================================
# 5. CLONE / RECONNECT REPOSITORY
# =============================================================================

fresh_clone = False

if not REPO_DIR.exists():

    print("Fresh Kaggle notebook detected.")
    print("Cloning repository...")
    print()

    run([
        "git",
        "clone",
        "--branch", "main",
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ])

    fresh_clone = True

elif (REPO_DIR / ".git").exists():

    print("Existing Kaggle working-tree detected.")
    print("Reusing repository without destructive reset.")

else:
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. "
        "Refusing to overwrite it."
    )


# =============================================================================
# 6. FORCE CLEAN REMOTE URL
# =============================================================================

run([
    "git", "remote", "set-url",
    "origin",
    REPO_URL,
], cwd=REPO_DIR, show=False)

remote_url = run(
    ["git", "remote", "get-url", "origin"],
    cwd=REPO_DIR,
    show=False,
)

if remote_url != REPO_URL:
    raise RuntimeError(
        "Unexpected Git remote URL."
    )

# A token must never appear in the URL.
if "@" in remote_url.split("github.com")[0]:
    raise RuntimeError(
        "Potential credential embedded in Git remote URL."
    )

print()
print("Git remote:")
print(" ", remote_url)


# =============================================================================
# 7. FETCH CURRENT MAIN + TAGS
# =============================================================================

print()
print("Fetching GitHub refs/tags...")

run([
    "git",
    "fetch",
    "origin",
    "main",
    "--tags",
    "--prune",
], cwd=REPO_DIR)


# =============================================================================
# 8. VERIFY STAGE22R FROZEN TAGS
# =============================================================================

def verify_tag(tag, expected_commit):
    actual = run(
        ["git", "rev-list", "-n", "1", tag],
        cwd=REPO_DIR,
        show=False,
    ).strip()

    if actual != expected_commit:
        raise RuntimeError(
            f"\nFROZEN TAG VERIFICATION FAILED\n"
            f"tag      : {tag}\n"
            f"expected : {expected_commit}\n"
            f"actual   : {actual}\n"
        )

    print(f"[OK] {tag}")
    print(f"     {actual}")


print()
print("Verifying Stage22R seals...")

verify_tag(
    STAGE22R_FINAL_TAG,
    STAGE22R_FINAL_COMMIT,
)

verify_tag(
    STAGE22R_CLOSEOUT_TAG,
    STAGE22R_CLOSEOUT_COMMIT,
)


# =============================================================================
# 9. VERIFY REPOSITORY PROVENANCE
# =============================================================================

HEAD = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    show=False,
).strip()

BRANCH = run(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
    show=False,
).strip()

print()
print("Current repository state:")
print("  branch:", BRANCH)
print("  HEAD  :", HEAD)


if BRANCH != "main":
    raise RuntimeError(
        f"Expected branch 'main', found '{BRANCH}'."
    )


if fresh_clone:
    # On the first Stage23 notebook bootstrap, Stage23 must begin exactly
    # from the frozen Stage22R publication-closeout commit.
    if HEAD != STAGE22R_CLOSEOUT_COMMIT:
        raise RuntimeError(
            "\nSTAGE23 BASE MISMATCH\n"
            f"Expected main HEAD:\n  {STAGE22R_CLOSEOUT_COMMIT}\n"
            f"Actual:\n  {HEAD}\n\n"
            "Do not continue until the repository state is inspected."
        )

else:
    # On later reruns, allow Stage23 commits to exist, but Stage22R closeout
    # MUST remain an ancestor.
    ancestry = subprocess.run(
        [
            "git",
            "merge-base",
            "--is-ancestor",
            STAGE22R_CLOSEOUT_COMMIT,
            HEAD,
        ],
        cwd=str(REPO_DIR),
        env=os.environ.copy(),
    )

    if ancestry.returncode != 0:
        raise RuntimeError(
            "Stage22R publication closeout is not an ancestor of HEAD."
        )


print()
print("[OK] Stage22R provenance verified.")


# =============================================================================
# 10. VERIFY WORKTREE
# =============================================================================

git_status = run(
    ["git", "status", "--porcelain"],
    cwd=REPO_DIR,
    show=False,
)

if fresh_clone and git_status:
    raise RuntimeError(
        "Fresh repository clone is unexpectedly dirty:\n"
        + git_status
    )

print("[OK] Git working tree clean." if not git_status
      else "[INFO] Existing working tree contains changes.")


# =============================================================================
# 11. TEST GITHUB PUSH AUTH — DRY RUN ONLY
# =============================================================================
#
# This does NOT push anything.
#
# It merely verifies that the Kaggle token can authenticate for the repo.
# =============================================================================

print()
print("Checking GitHub write authentication (dry-run only)...")

push_test = subprocess.run(
    [
        "git",
        "push",
        "--dry-run",
        "origin",
        "HEAD:main",
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if push_test.returncode != 0:
    print(push_test.stdout.strip())
    raise RuntimeError(
        "\nGitHub authentication is configured, "
        "but dry-run push authentication failed.\n"
        "Check PAT repository permissions."
    )

print("[OK] GitHub token can authenticate for push.")


# =============================================================================
# 12. STAGE 23 RAW HOLDOUT SAFETY GUARD
# =============================================================================
#
# Stage23 MUST NOT access raw March 1 or March 2 data.
#
# This guard is intended to be called by ALL Stage23 dataset resolvers/loaders.
#
# IMPORTANT:
#   We deliberately do NOT recursively inspect /kaggle/input here.
# =============================================================================

FORBIDDEN_RAW_MARKERS = tuple(x.lower() for x in [

    # MM-DD-YYYY
    "03-01-2018",
    "03_01_2018",
    "03 01 2018",
    "03-02-2018",
    "03_02_2018",
    "03 02 2018",

    # DD-MM-YYYY
    "01-03-2018",
    "01_03_2018",
    "01 03 2018",
    "02-03-2018",
    "02_03_2018",
    "02 03 2018",

    # Compact
    "03012018",
    "03022018",
    "01032018",
    "02032018",

    # Textual
    "mar1",
    "mar_1",
    "mar-1",
    "march1",
    "march_1",
    "march-1",

    "mar2",
    "mar_2",
    "mar-2",
    "march2",
    "march_2",
    "march-2",
])


def assert_stage23_path_allowed(path):
    """
    Mandatory Stage23 path gate.

    Returns the resolved Path if permitted.
    Raises RuntimeError if the path appears to reference
    the permanently closed Mar1/Mar2 raw holdout.
    """

    path = Path(path).expanduser().resolve()

    normalized = str(path).lower()

    for marker in FORBIDDEN_RAW_MARKERS:
        if marker in normalized:
            raise RuntimeError(
                "\nSTAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}\n\n"
                "Raw Mar1 / Mar2 access is permanently forbidden."
            )

    return path


print()
print("[OK] Stage23 Mar1/Mar2 path guard installed.")


# =============================================================================
# 13. TOP-LEVEL KAGGLE INPUT INVENTORY ONLY
# =============================================================================
#
# NO recursive glob.
# NO CSV read.
# NO parquet read.
# NO file opening.
#
# Only the attached Kaggle dataset mount points are enumerated.
# =============================================================================

if not KAGGLE_INPUT.exists():
    raise RuntimeError(
        "/kaggle/input does not exist."
    )


input_mounts = sorted(
    p for p in KAGGLE_INPUT.iterdir()
    if p.is_dir()
)


print()
print("=" * 78)
print("ATTACHED KAGGLE DATASET MOUNTS — TOP LEVEL ONLY")
print("=" * 78)

if not input_mounts:
    print("No attached dataset directories found.")
else:
    for i, path in enumerate(input_mounts, 1):

        # Merely classify the mount name.
        # We do NOT open or recurse into it.
        normalized = path.name.lower()

        blocked_name = any(
            marker in normalized
            for marker in FORBIDDEN_RAW_MARKERS
        )

        status = (
            "BLOCKED-HOLDOUT-NAME"
            if blocked_name
            else "MOUNTED"
        )

        print(
            f"{i:02d}. [{status}] "
            f"/kaggle/input/{path.name}"
        )


# =============================================================================
# 14. ENVIRONMENT RECEIPT
# =============================================================================

PACKAGE_NAMES = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "lightgbm",
    "shap",
    "scipy",
    "matplotlib",
    "joblib",
    "pyarrow",
]


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


versions = {
    name: package_version(name)
    for name in PACKAGE_NAMES
}


print()
print("=" * 78)
print("PYTHON / ML ENVIRONMENT")
print("=" * 78)

print("Python:", sys.version.split()[0])

for name, version in versions.items():
    print(f"{name:16s}: {version or 'NOT INSTALLED'}")


# =============================================================================
# 15. DETERMINISTIC GENERAL SEED
# =============================================================================
#
# Stage23 model-specific seeds remain governed by Stage23-0 / inherited
# Stage22 definitions. This merely initializes generic notebook randomness.
# =============================================================================

BOOTSTRAP_SEED = 42

random.seed(BOOTSTRAP_SEED)

try:
    import numpy as np
    np.random.seed(BOOTSTRAP_SEED)
except Exception:
    pass


# =============================================================================
# 16. WRITE LOCAL BOOTSTRAP RECEIPT
# =============================================================================
#
# IMPORTANT:
#   Written to /kaggle/working only.
#
#   It is NOT committed to the repository.
#   It is NOT a Stage23 scientific result.
# =============================================================================

receipt = {
    "receipt_type": "stage23_fresh_notebook_bootstrap",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "repository": REPO_FULL_NAME,
    "branch": BRANCH,
    "head": HEAD,

    "stage22r_scientific_seal": {
        "commit": STAGE22R_FINAL_COMMIT,
        "tag": STAGE22R_FINAL_TAG,
    },

    "stage22r_publication_closeout": {
        "commit": STAGE22R_CLOSEOUT_COMMIT,
        "tag": STAGE22R_CLOSEOUT_TAG,
    },

    "fresh_clone": fresh_clone,

    "kaggle_input_mounts": [
        p.name for p in input_mounts
    ],

    "raw_mar1_mar2_access": "FORBIDDEN",

    "dataset_files_opened_by_bootstrap": 0,

    "stage23_experiments_run": 0,

    "stage23_0_created": False,

    "python": sys.version,

    "packages": versions,

    "git_remote": REPO_URL,
    "github_token_storage": "Kaggle Secret + ephemeral GIT_ASKPASS",
}


RECEIPT_PATH = (
    WORKING_ROOT /
    "stage23_fresh_bootstrap_receipt.json"
)

RECEIPT_PATH.write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# =============================================================================
# 17. FINAL BOOTSTRAP ASSERTIONS
# =============================================================================

assert REPO_DIR.exists()
assert (REPO_DIR / ".git").exists()

assert run(
    ["git", "rev-list", "-n", "1", STAGE22R_FINAL_TAG],
    cwd=REPO_DIR,
    show=False,
) == STAGE22R_FINAL_COMMIT

assert run(
    ["git", "rev-list", "-n", "1", STAGE22R_CLOSEOUT_TAG],
    cwd=REPO_DIR,
    show=False,
) == STAGE22R_CLOSEOUT_COMMIT


os.chdir(REPO_DIR)


print()
print("=" * 78)
print("STAGE 23 FRESH-BOOTSTRAP COMPLETE")
print("=" * 78)

print()
print("Repository:")
print(" ", REPO_DIR)

print()
print("HEAD:")
print(" ", HEAD)

print()
print("Scientific Stage22R seal:")
print(" ", STAGE22R_FINAL_TAG)
print(" ", STAGE22R_FINAL_COMMIT)

print()
print("Publication closeout:")
print(" ", STAGE22R_CLOSEOUT_TAG)
print(" ", STAGE22R_CLOSEOUT_COMMIT)

print()
print("Bootstrap receipt:")
print(" ", RECEIPT_PATH)

print()
print("Safety state:")
print("  Mar1 raw access : FORBIDDEN")
print("  Mar2 raw access : FORBIDDEN")
print("  dataset files read: 0")
print("  Stage23 models run: 0")
print("  stage23_0 created: NO")

print()
print("NEXT ACTION:")
print(
    "  Inspect the bootstrap output and attached development-data mounts."
)
print(
    "  Do NOT train anything. Next we construct Stage23-0 prospectively."
)

print()
print("=" * 78)

STAGE 23 — FRESH NOTEBOOK BOOTSTRAP

Repository : themubasshir/ids2018-validation-safe-ablation
Branch     : main
Stage22R scientific seal : b5e44615269198426cc8a9aa3b3e701c2ca9e48e
Stage22R closeout base   : fafb131981b6e15e47bdf35fd8c18ea228680fe3

GitHub credential loaded from Kaggle Secret: GITHUB_TOKEN
Token value: [REDACTED]

Fresh Kaggle notebook detected.
Cloning repository...

Cloning into '/kaggle/working/ids2018-validation-safe-ablation'...
Updating files:  37% (607/1624)
Updating files:  38% (618/1624)
Updating files:  39% (634/1624)
Updating files:  40% (650/1624)
Updating files:  41% (666/1624)
Updating files:  42% (683/1624)
Updating files:  43% (699/1624)
Updating files:  44% (715/1624)
Updating files:  45% (731/1624)
Updating files:  46% (748/1624)
Updating files:  47% (764/1624)
Updating files:  48% (780/1624)
Updating files:  49% (796/1624)
Updating files:  50% (812/1624)
Updating files:  51% (829/1624)
Updating files:  52% (845/1624)
Updating files:  53% (861/1624)


RuntimeError: 
STAGE23 BASE MISMATCH
Expected main HEAD:
  fafb131981b6e15e47bdf35fd8c18ea228680fe3
Actual:
  2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460

Do not continue until the repository state is inspected.

In [5]:
# =============================================================================
# STAGE23 — ATTACHED DATASET STRUCTURE AUDIT
# METADATA / PATHS ONLY — NO DATA FILE IS OPENED
# =============================================================================

from pathlib import Path
import os

DATASET_ROOT = Path("/kaggle/input/datasets")

if not DATASET_ROOT.exists():
    raise RuntimeError(
        f"Expected attached dataset root does not exist: {DATASET_ROOT}"
    )


# =============================================================================
# PERMANENTLY CLOSED HOLDOUT MARKERS
# =============================================================================

FORBIDDEN_MARKERS = tuple(x.lower() for x in [

    # MM-DD-YYYY
    "03-01-2018",
    "03_01_2018",
    "03 01 2018",
    "03-02-2018",
    "03_02_2018",
    "03 02 2018",

    # DD-MM-YYYY
    "01-03-2018",
    "01_03_2018",
    "01 03 2018",
    "02-03-2018",
    "02_03_2018",
    "02 03 2018",

    # Compact date forms
    "03012018",
    "03022018",
    "01032018",
    "02032018",

    # Text names
    "mar1",
    "mar_1",
    "mar-1",
    "march1",
    "march_1",
    "march-1",

    "mar2",
    "mar_2",
    "mar-2",
    "march2",
    "march_2",
    "march-2",
])


def is_forbidden(path):
    text = str(path).lower()
    return any(marker in text for marker in FORBIDDEN_MARKERS)


# =============================================================================
# DIRECTORY WALK
#
# IMPORTANT:
# - directory metadata only
# - filenames only
# - NO open()
# - NO pandas.read_*
# - NO pyarrow reads
# - NO hashing file contents
# =============================================================================

directories = []
allowed_files = []
blocked_files = []

for root, dirs, files in os.walk(DATASET_ROOT):

    root_path = Path(root)

    # Never descend into a directory whose path identifies Mar1/Mar2.
    safe_dirs = []

    for dirname in sorted(dirs):
        child = root_path / dirname

        if is_forbidden(child):
            print(
                "[BLOCKED DIRECTORY — NOT ENTERED]",
                child.relative_to(DATASET_ROOT)
            )
        else:
            safe_dirs.append(dirname)

    # Modify traversal in-place.
    dirs[:] = safe_dirs

    directories.append(root_path)

    for filename in sorted(files):

        path = root_path / filename

        if is_forbidden(path):
            blocked_files.append(path)
            continue

        allowed_files.append(path)


# =============================================================================
# REPORT
# =============================================================================

print()
print("=" * 88)
print("STAGE23 ATTACHED DATASET STRUCTURE AUDIT")
print("=" * 88)

print()
print("Root:")
print(" ", DATASET_ROOT)

print()
print("Safety:")
print("  raw Mar1 access : FORBIDDEN")
print("  raw Mar2 access : FORBIDDEN")
print("  data files read : 0")
print("  model runs      : 0")

print()
print("Directory count:", len(directories))
print("Allowed file-path count:", len(allowed_files))
print("Blocked Mar1/Mar2 file-path count:", len(blocked_files))


# =============================================================================
# DISPLAY DIRECTORY TREE
# =============================================================================

print()
print("=" * 88)
print("DIRECTORIES")
print("=" * 88)

for path in sorted(directories):

    rel = path.relative_to(DATASET_ROOT)

    depth = len(rel.parts)

    if str(rel) == ".":
        print("/datasets")
    else:
        indent = "  " * depth
        print(f"{indent}{rel.name}/")


# =============================================================================
# DISPLAY SAFE FILE PATHS
# =============================================================================

print()
print("=" * 88)
print("AUTHORIZED-NAME FILE INVENTORY")
print("NO FILE CONTENT HAS BEEN READ")
print("=" * 88)

for i, path in enumerate(sorted(allowed_files), 1):

    rel = path.relative_to(DATASET_ROOT)

    try:
        size = path.stat().st_size
    except Exception:
        size = None

    if size is None:
        size_text = "size unavailable"
    elif size >= 1024**3:
        size_text = f"{size / 1024**3:.3f} GiB"
    elif size >= 1024**2:
        size_text = f"{size / 1024**2:.3f} MiB"
    elif size >= 1024:
        size_text = f"{size / 1024:.3f} KiB"
    else:
        size_text = f"{size} B"

    print(
        f"{i:03d}. {rel}"
        f"    [{size_text}]"
    )


# =============================================================================
# BLOCKED PATH SUMMARY
#
# We intentionally do not print blocked filenames.
# =============================================================================

if blocked_files:
    print()
    print("=" * 88)
    print("PERMANENT HOLDOUT GUARD")
    print("=" * 88)

    print(
        f"{len(blocked_files)} file path(s) matched "
        "Mar1/Mar2 holdout markers."
    )

    print("Their contents were NOT opened.")
    print("Their names are intentionally not used by Stage23.")


# =============================================================================
# EXTENSION SUMMARY
# =============================================================================

extension_counts = {}

for path in allowed_files:
    ext = path.suffix.lower() or "<no extension>"
    extension_counts[ext] = extension_counts.get(ext, 0) + 1


print()
print("=" * 88)
print("SAFE FILE TYPE SUMMARY")
print("=" * 88)

for ext, count in sorted(
    extension_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"{ext:20s} {count:6d}")


print()
print("=" * 88)
print("STRUCTURE AUDIT COMPLETE")
print("=" * 88)

print()
print("No Stage23 protocol has been frozen.")
print("No Stage23 experiment has been run.")
print("No dataset content has been read.")
print()
print(
    "Next: map the attached development artifacts against the "
    "frozen Stage22R memberships and model-input receipts."
)
print("=" * 88)


STAGE23 ATTACHED DATASET STRUCTURE AUDIT

Root:
  /kaggle/input/datasets

Safety:
  raw Mar1 access : FORBIDDEN
  raw Mar2 access : FORBIDDEN
  data files read : 0
  model runs      : 0

Directory count: 10
Allowed file-path count: 35
Blocked Mar1/Mar2 file-path count: 2

DIRECTORIES
/datasets
  jmmubasshirrahman/
    ai-ids-research-kaggle/
      data/
        processed/
      models/
      results/
    stage22r-1c-70f-cache-3cd41c5f/
  solarmainframe/
    ids-intrusion-csv/

AUTHORIZED-NAME FILE INVENTORY
NO FILE CONTENT HAS BEEN READ
001. jmmubasshirrahman/ai-ids-research-kaggle/data/processed/merged_balanced_ids2018_safe.csv    [108.836 MiB]
002. jmmubasshirrahman/ai-ids-research-kaggle/models/final_xgboost_ids_model.pkl    [755.018 KiB]
003. jmmubasshirrahman/ai-ids-research-kaggle/results/additional_baseline_model_results.csv    [530 B]
004. jmmubasshirrahman/ai-ids-research-kaggle/results/all_csv_label_summary.csv    [1.142 KiB]
005. jmmubasshirrahman/ai-ids-research-kaggle/res

In [6]:
# =============================================================================
# STAGE23 PRE-FREEZE — STAGE22R DEVELOPMENT CACHE PROVENANCE VERIFICATION
# =============================================================================
#
# PURPOSE:
#   Verify that the 8 attached Stage22R development parquet files are
#   EXACTLY the frozen Stage22R-1C model-input cache.
#
# OPERATIONS:
#   - reads frozen receipt from cloned Git repository
#   - hashes ONLY the 8 explicitly authorized Feb14-Feb28 parquet files
#   - checks byte size
#   - checks SHA256
#   - checks parquet metadata row count
#   - checks schema against the frozen 70-feature ordering
#
# DOES NOT:
#   - open Mar1
#   - open Mar2
#   - read raw CICIDS CSV data
#   - train anything
#   - calculate model metrics
#   - create/freeze Stage23-0
#
# =============================================================================

from pathlib import Path
import csv
import hashlib
import json

import pyarrow.parquet as pq


# =============================================================================
# PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

STAGE22R_1C_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
)

CACHE_RECEIPT = (
    STAGE22R_1C_DIR
    / "stage22r_1c_cache_files.csv"
)

FEATURE_RECEIPT = (
    STAGE22R_1C_DIR
    / "stage22r_1c_70_feature_order.csv"
)


for required in [
    REPO_DIR,
    CACHE_DIR,
    STAGE22R_1C_DIR,
    CACHE_RECEIPT,
    FEATURE_RECEIPT,
]:
    if not required.exists():
        raise RuntimeError(
            f"Required Stage22R artifact missing: {required}"
        )


# =============================================================================
# FIXED AUTHORIZED DEVELOPMENT DAYS
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]

assert len(AUTHORIZED_CACHE_FILES) == 8


# =============================================================================
# HARD HOLDOUT GUARD
# =============================================================================

FORBIDDEN = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):
    path = Path(path).resolve()
    text = str(path).lower()

    for marker in FORBIDDEN:
        if marker in text:
            raise RuntimeError(
                f"PERMANENT HOLDOUT ACCESS BLOCKED: {path}"
            )

    return path


# =============================================================================
# READ FROZEN GIT RECEIPTS
# =============================================================================

with CACHE_RECEIPT.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:
    cache_rows = list(csv.DictReader(f))


with FEATURE_RECEIPT.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:
    feature_rows = list(csv.DictReader(f))


if len(cache_rows) != 8:
    raise RuntimeError(
        f"Frozen Stage22R cache receipt contains "
        f"{len(cache_rows)} rows, expected 8."
    )


feature_rows = sorted(
    feature_rows,
    key=lambda x: int(x["feature_position"]),
)

frozen_features = [
    row["feature_name"]
    for row in feature_rows
]


if len(frozen_features) != 70:
    raise RuntimeError(
        f"Frozen feature order contains "
        f"{len(frozen_features)} features, expected 70."
    )


# =============================================================================
# FEATURE-SPACE PRE-FREEZE FACTS
# =============================================================================

print("=" * 88)
print("FROZEN STAGE22R FEATURE SPACE")
print("=" * 88)

print()
print("Feature count :", len(frozen_features))
print("Dst Port      :", "Dst Port" in frozen_features)
print("Src Port      :", "Src Port" in frozen_features)
print("Protocol      :", "Protocol" in frozen_features)
print(
    "Init Fwd Win :",
    "Init Fwd Win Byts" in frozen_features
)
print(
    "Fwd Seg Min  :",
    "Fwd Seg Size Min" in frozen_features
)

print()

if "Src Port" not in frozen_features:
    print(
        "[IMPORTANT] Src Port is NOT part of the frozen "
        "70-feature Stage22R model input."
    )
    print(
        "[IMPORTANT] NO_PORTS therefore requires an explicit "
        "prospective definition before Stage23-0."
    )


# =============================================================================
# HASH HELPER
# =============================================================================

def sha256_file(path, chunk_size=16 * 1024 * 1024):

    path = guard_path(path)

    h = hashlib.sha256()
    total = 0

    with path.open("rb") as f:

        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)
            total += len(chunk)

    return h.hexdigest(), total


# =============================================================================
# VERIFY FROZEN RECEIPT FILE SET
# =============================================================================

receipt_names = [
    row["cache_file"]
    for row in cache_rows
]

if receipt_names != AUTHORIZED_CACHE_FILES:
    raise RuntimeError(
        "Frozen receipt cache-file order does not match "
        "the explicitly authorized Stage23 development list."
    )


# =============================================================================
# VERIFY EACH CACHE
# =============================================================================

audit_rows = []

print()
print("=" * 88)
print("STAGE22R DEVELOPMENT CACHE BYTE VERIFICATION")
print("=" * 88)
print()


for i, row in enumerate(cache_rows):

    name = row["cache_file"]

    if name not in AUTHORIZED_CACHE_FILES:
        raise RuntimeError(
            f"Unexpected cache in Stage22R receipt: {name}"
        )

    path = guard_path(CACHE_DIR / name)

    if not path.exists():
        raise RuntimeError(
            f"Attached Stage22R cache missing: {path}"
        )

    expected_bytes = int(row["bytes"])
    expected_sha = row["sha256"]
    expected_rows = int(row["rows"])

    actual_bytes_stat = path.stat().st_size

    # -------------------------------------------------------------------------
    # Byte count before hashing
    # -------------------------------------------------------------------------

    if actual_bytes_stat != expected_bytes:
        raise RuntimeError(
            f"\nBYTE SIZE MISMATCH\n"
            f"file     : {name}\n"
            f"expected : {expected_bytes}\n"
            f"actual   : {actual_bytes_stat}"
        )

    # -------------------------------------------------------------------------
    # Exact byte-level SHA256 verification
    # -------------------------------------------------------------------------

    actual_sha, hashed_bytes = sha256_file(path)

    if hashed_bytes != expected_bytes:
        raise RuntimeError(
            f"\nHASHED BYTE COUNT MISMATCH\n"
            f"file     : {name}\n"
            f"expected : {expected_bytes}\n"
            f"hashed   : {hashed_bytes}"
        )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"\nSHA256 MISMATCH\n"
            f"file     : {name}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    # -------------------------------------------------------------------------
    # Parquet metadata only
    # -------------------------------------------------------------------------

    pf = pq.ParquetFile(path)

    actual_rows = pf.metadata.num_rows

    if actual_rows != expected_rows:
        raise RuntimeError(
            f"\nPARQUET ROW COUNT MISMATCH\n"
            f"file     : {name}\n"
            f"expected : {expected_rows}\n"
            f"actual   : {actual_rows}"
        )

    schema_names = pf.schema_arrow.names

    audit_rows.append({
        "day_id": int(row["day_id"]),
        "source_file": row["source_file"],
        "cache_file": name,

        "expected_rows": expected_rows,
        "actual_rows": actual_rows,

        "expected_bytes": expected_bytes,
        "actual_bytes": actual_bytes_stat,

        "expected_sha256": expected_sha,
        "actual_sha256": actual_sha,

        "schema_column_count": len(schema_names),

        "sha256_match": actual_sha == expected_sha,
        "bytes_match": actual_bytes_stat == expected_bytes,
        "rows_match": actual_rows == expected_rows,
    })

    gib = expected_bytes / (1024 ** 3)

    print(
        f"[OK] day {i:02d}  "
        f"{name}"
    )

    print(
        f"     rows   : {actual_rows:,}"
    )

    print(
        f"     bytes  : {actual_bytes_stat:,} "
        f"({gib:.3f} GiB)"
    )

    print(
        f"     SHA256 : {actual_sha}"
    )

    print(
        f"     columns: {len(schema_names)}"
    )

    print()


# =============================================================================
# CROSS-DAY SCHEMA CONSISTENCY
# =============================================================================

schemas = []

for name in AUTHORIZED_CACHE_FILES:

    path = guard_path(CACHE_DIR / name)

    pf = pq.ParquetFile(path)

    schemas.append(
        tuple(pf.schema_arrow.names)
    )


reference_schema = schemas[0]

for i, schema in enumerate(schemas[1:], 1):

    if schema != reference_schema:
        raise RuntimeError(
            f"Parquet schema mismatch between day 00 and day {i:02d}."
        )


print("=" * 88)
print("SCHEMA CONSISTENCY")
print("=" * 88)

print()
print("All 8 cache schemas identical: YES")
print("Parquet columns:", len(reference_schema))


# =============================================================================
# VERIFY FROZEN 70 FEATURES ARE PRESENT IN THE CACHE
# =============================================================================

missing_features = [
    feature
    for feature in frozen_features
    if feature not in reference_schema
]

if missing_features:
    raise RuntimeError(
        "Frozen Stage22R features missing from cache schema:\n"
        + "\n".join(missing_features)
    )


print("Frozen 70 features present: YES")


# =============================================================================
# IDENTIFY NON-FEATURE CACHE COLUMNS
# =============================================================================

extra_columns = [
    column
    for column in reference_schema
    if column not in frozen_features
]


print()
print("Non-feature/cache-support columns:")

for column in extra_columns:
    print("  -", column)


# =============================================================================
# AGGREGATE COUNTS FROM THE FROZEN RECEIPT
# =============================================================================

total_rows = sum(
    int(row["rows"])
    for row in cache_rows
)

total_benign = sum(
    int(row["benign"])
    for row in cache_rows
)

total_attack = sum(
    int(row["attack"])
    for row in cache_rows
)


print()
print("=" * 88)
print("FROZEN DEVELOPMENT CORPUS TOTALS")
print("=" * 88)

print()
print(f"Rows   : {total_rows:,}")
print(f"Benign : {total_benign:,}")
print(f"Attack : {total_attack:,}")
print(
    f"Attack prevalence: "
    f"{total_attack / total_rows:.10f}"
)


# =============================================================================
# WRITE LOCAL PRE-FREEZE AUDIT RECEIPT
# =============================================================================
#
# This stays in /kaggle/working.
# It is NOT Stage23-0 and is NOT committed yet.
# =============================================================================

AUDIT_RECEIPT = Path(
    "/kaggle/working/"
    "stage23_prefreeze_stage22r_cache_audit.json"
)


payload = {
    "audit_type":
        "stage23_prefreeze_stage22r_cache_provenance",

    "stage23_experiment": False,

    "stage23_0_frozen": False,

    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,

    "authorized_cache_dir":
        str(CACHE_DIR),

    "authorized_cache_file_count":
        len(AUTHORIZED_CACHE_FILES),

    "feature_count":
        len(frozen_features),

    "src_port_present":
        "Src Port" in frozen_features,

    "dst_port_present":
        "Dst Port" in frozen_features,

    "protocol_present":
        "Protocol" in frozen_features,

    "all_sha256_match":
        all(x["sha256_match"] for x in audit_rows),

    "all_byte_sizes_match":
        all(x["bytes_match"] for x in audit_rows),

    "all_row_counts_match":
        all(x["rows_match"] for x in audit_rows),

    "all_schemas_identical":
        all(schema == reference_schema for schema in schemas),

    "development_rows":
        total_rows,

    "development_benign":
        total_benign,

    "development_attack":
        total_attack,

    "files":
        audit_rows,
}


AUDIT_RECEIPT.write_text(
    json.dumps(
        payload,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# =============================================================================
# FINAL ASSERTIONS
# =============================================================================

assert len(audit_rows) == 8

assert all(
    x["sha256_match"]
    for x in audit_rows
)

assert all(
    x["bytes_match"]
    for x in audit_rows
)

assert all(
    x["rows_match"]
    for x in audit_rows
)

assert all(
    schema == reference_schema
    for schema in schemas
)

assert len(frozen_features) == 70

assert "Dst Port" in frozen_features
assert "Init Fwd Win Byts" in frozen_features
assert "Fwd Seg Size Min" in frozen_features


print()
print("=" * 88)
print("STAGE22R DEVELOPMENT CACHE PROVENANCE: VERIFIED")
print("=" * 88)

print()
print("8 / 8 authorized development caches:")
print("  byte size : EXACT")
print("  SHA256    : EXACT")
print("  row count : EXACT")
print("  schema    : CONSISTENT")

print()
print("Raw Mar1 read : NO")
print("Raw Mar2 read : NO")
print("Stage23 model runs : 0")
print("Stage23-0 frozen   : NO")

print()
print("Local audit receipt:")
print(" ", AUDIT_RECEIPT)

print()
print("NEXT:")
print(
    "  Verify frozen RANDOM_NATURAL and CHRONOLOGICAL_NATURAL "
    "membership reconstruction against Stage22R receipts."
)

print("=" * 88)

FROZEN STAGE22R FEATURE SPACE

Feature count : 70
Dst Port      : True
Src Port      : False
Protocol      : True
Init Fwd Win : True
Fwd Seg Min  : True

[IMPORTANT] Src Port is NOT part of the frozen 70-feature Stage22R model input.
[IMPORTANT] NO_PORTS therefore requires an explicit prospective definition before Stage23-0.

STAGE22R DEVELOPMENT CACHE BYTE VERIFICATION

[OK] day 00  day_00_02-14-2018.parquet
     rows   : 822,947
     bytes  : 87,229,572 (0.081 GiB)
     SHA256 : a542ad551f5aab59c9ad27958d6bb065e3336b3ca4f040da21e49ad71936f990
     columns: 74

[OK] day 01  day_01_02-15-2018.parquet
     rows   : 1,046,154
     bytes  : 116,709,416 (0.109 GiB)
     SHA256 : 48b4173ac1d2711047be9284b73247537da9db3ade3e8bf63bc7ed878424ff72
     columns: 74

[OK] day 02  day_02_02-16-2018.parquet
     rows   : 900,988
     bytes  : 81,039,407 (0.075 GiB)
     SHA256 : 26c259c777fbb59faecbabc00071a601ff7d1546a1c4bbca87b62f60e285bb43
     columns: 74

[OK] day 03  day_03_02-20-2018.parque

In [7]:
# =============================================================================
# STAGE23 PRE-FREEZE
# VERIFY INHERITED STAGE22R RANDOM_NATURAL + CHRONOLOGICAL_NATURAL MEMBERSHIPS
# =============================================================================
#
# PURPOSE:
#   Prove that Stage23 can inherit the exact frozen Stage22R NATURAL splits.
#
# READS:
#   - ONLY the 8 authorized Feb14-Feb28 Stage22R cache files
#   - ONLY support columns:
#         clean_position
#         day_id
#         binary_label
#   - frozen Stage22R membership bitset / receipts from Git clone
#
# DOES NOT:
#   - touch Mar1 / Mar2
#   - read any 70 predictor features
#   - fit any model
#   - select threshold
#   - calculate Stage23 performance
#   - freeze Stage23-0
#
# =============================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pyarrow.parquet as pq


# =============================================================================
# PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

SUMMARY_PATH = (
    MEMBERSHIP_DIR
    / "stage22r_1b1_membership_summary.json"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)


AUTHORIZED_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


# =============================================================================
# HARD HOLDOUT GUARD
# =============================================================================

FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):
    path = Path(path).resolve()
    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:
        if marker in text:
            raise RuntimeError(
                "\nPERMANENT STAGE22R HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


# =============================================================================
# SHA256
# =============================================================================

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = guard_path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# LOAD FROZEN STAGE22R MEMBERSHIP RECEIPT
# =============================================================================

if not SUMMARY_PATH.exists():
    raise RuntimeError(
        f"Frozen membership receipt missing: {SUMMARY_PATH}"
    )

summary = json.loads(
    SUMMARY_PATH.read_text(encoding="utf-8")
)

assert summary["status"] == (
    "FOUR_DEVELOPMENT_MEMBERSHIPS_GENERATED_AND_FROZEN"
)

assert summary["clean_development"]["rows"] == 14_412_403


print("=" * 92)
print("STAGE22R FROZEN MEMBERSHIP RECEIPT")
print("=" * 92)

print()
print(
    "Canonical order:",
    summary["clean_development"]["canonical_order"]
)

print(
    "Development rows:",
    f'{summary["clean_development"]["rows"]:,}'
)

print(
    "Development benign:",
    f'{summary["clean_development"]["benign"]:,}'
)

print(
    "Development attack:",
    f'{summary["clean_development"]["attack"]:,}'
)


# =============================================================================
# VERIFY RANDOM VALIDATION BITSET ITSELF
# =============================================================================

bitset_receipt = summary["artifacts"]["random_validation.packbits"]

expected_bitset_sha = bitset_receipt["sha256"]
expected_bitset_bytes = bitset_receipt["bytes"]
expected_bitset_population = bitset_receipt["population"]
logical_length = bitset_receipt["logical_length"]
bit_order = bitset_receipt["bit_order"]


actual_bitset_bytes = RANDOM_VALIDATION_BITSET.stat().st_size
actual_bitset_sha = sha256_file(RANDOM_VALIDATION_BITSET)


if actual_bitset_bytes != expected_bitset_bytes:
    raise RuntimeError(
        "random_validation.packbits byte-size mismatch"
    )

if actual_bitset_sha != expected_bitset_sha:
    raise RuntimeError(
        "random_validation.packbits SHA256 mismatch"
    )


print()
print("=" * 92)
print("RANDOM VALIDATION BITSET")
print("=" * 92)

print()
print("Bytes   :", f"{actual_bitset_bytes:,}")
print("SHA256  :", actual_bitset_sha)
print("Bitorder:", bit_order)
print("Length  :", f"{logical_length:,}")
print("Expected population:", f"{expected_bitset_population:,}")
print()
print("[OK] Frozen random-validation bitset integrity verified.")


# =============================================================================
# UNPACK RANDOM VALIDATION MEMBERSHIP
# =============================================================================

packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)

random_validation_mask = np.unpackbits(
    packed,
    bitorder=bit_order,
)[:logical_length].astype(bool)


if len(random_validation_mask) != logical_length:
    raise RuntimeError(
        "Random membership logical length mismatch."
    )


random_population = int(random_validation_mask.sum())

if random_population != expected_bitset_population:
    raise RuntimeError(
        "\nRandom validation population mismatch\n"
        f"expected: {expected_bitset_population:,}\n"
        f"actual  : {random_population:,}"
    )


random_train_mask = ~random_validation_mask


print(
    "Actual validation population:",
    f"{random_population:,}"
)

print(
    "Actual training population:",
    f"{int(random_train_mask.sum()):,}"
)


# =============================================================================
# READ ONLY SUPPORT COLUMNS FROM 8 AUTHORIZED DEVELOPMENT PARQUETS
# =============================================================================
#
# No predictor feature is read.
# =============================================================================

SUPPORT_COLUMNS = [
    "clean_position",
    "day_id",
    "binary_label",
]

clean_positions = []
day_ids = []
labels = []


print()
print("=" * 92)
print("READING AUTHORIZED MEMBERSHIP SUPPORT COLUMNS")
print("=" * 92)
print()


running_start = 0


for expected_day_id, filename in enumerate(AUTHORIZED_FILES):

    path = guard_path(CACHE_DIR / filename)

    if not path.exists():
        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )

    table = pq.read_table(
        path,
        columns=SUPPORT_COLUMNS,
    )

    cp = table["clean_position"].to_numpy(
        zero_copy_only=False
    )

    did = table["day_id"].to_numpy(
        zero_copy_only=False
    )

    y = table["binary_label"].to_numpy(
        zero_copy_only=False
    )


    # -------------------------------------------------------------------------
    # Verify day_id is exactly what filename/receipt imply
    # -------------------------------------------------------------------------

    unique_days = np.unique(did)

    if not (
        len(unique_days) == 1
        and int(unique_days[0]) == expected_day_id
    ):
        raise RuntimeError(
            f"day_id mismatch in {filename}: {unique_days}"
        )


    # -------------------------------------------------------------------------
    # Verify clean_position forms the expected global canonical sequence
    # -------------------------------------------------------------------------

    expected_start = running_start
    expected_stop = running_start + len(cp)

    if int(cp[0]) != expected_start:
        raise RuntimeError(
            f"clean_position start mismatch in {filename}\n"
            f"expected: {expected_start}\n"
            f"actual  : {int(cp[0])}"
        )

    if int(cp[-1]) != expected_stop - 1:
        raise RuntimeError(
            f"clean_position stop mismatch in {filename}\n"
            f"expected last: {expected_stop - 1}\n"
            f"actual last  : {int(cp[-1])}"
        )

    # Stronger sequentiality check.
    expected_cp = np.arange(
        expected_start,
        expected_stop,
        dtype=cp.dtype,
    )

    if not np.array_equal(cp, expected_cp):
        raise RuntimeError(
            f"clean_position is not strictly canonical in {filename}"
        )


    # -------------------------------------------------------------------------
    # Verify binary labels
    # -------------------------------------------------------------------------

    unique_labels = set(
        int(v) for v in np.unique(y)
    )

    if not unique_labels.issubset({0, 1}):
        raise RuntimeError(
            f"Unexpected binary labels in {filename}: "
            f"{sorted(unique_labels)}"
        )


    benign = int((y == 0).sum())
    attack = int((y == 1).sum())


    print(
        f"[OK] day {expected_day_id:02d} "
        f"{filename}"
    )

    print(
        f"     positions : "
        f"{expected_start:,} .. {expected_stop - 1:,}"
    )

    print(
        f"     rows      : {len(y):,}"
    )

    print(
        f"     benign    : {benign:,}"
    )

    print(
        f"     attack    : {attack:,}"
    )

    print()


    clean_positions.append(cp)
    day_ids.append(did)
    labels.append(y)

    running_start = expected_stop


# =============================================================================
# CONCATENATE IN FROZEN CANONICAL ORDER
# =============================================================================

clean_positions = np.concatenate(clean_positions)
day_ids = np.concatenate(day_ids)
labels = np.concatenate(labels)


if len(labels) != logical_length:
    raise RuntimeError(
        "\nCanonical development length mismatch\n"
        f"bitset logical length : {logical_length:,}\n"
        f"loaded labels         : {len(labels):,}"
    )


# Final canonical-position identity.
expected_positions = np.arange(
    logical_length,
    dtype=clean_positions.dtype,
)

if not np.array_equal(
    clean_positions,
    expected_positions,
):
    raise RuntimeError(
        "Global clean_position sequence mismatch."
    )


print("=" * 92)
print("CANONICAL DEVELOPMENT ORDER")
print("=" * 92)

print()
print("Rows:", f"{len(labels):,}")
print("First clean_position:", int(clean_positions[0]))
print("Last clean_position :", int(clean_positions[-1]))
print("Canonical sequence  : EXACT")


# =============================================================================
# GENERAL COUNT HELPER
# =============================================================================

def membership_counts(mask, y):

    selected = y[mask]

    return {
        "rows": int(len(selected)),
        "benign": int((selected == 0).sum()),
        "attack": int((selected == 1).sum()),
    }


def assert_counts(name, actual, expected):

    for key in ["rows", "benign", "attack"]:

        if actual[key] != expected[key]:
            raise RuntimeError(
                f"\n{name} COUNT MISMATCH\n"
                f"field    : {key}\n"
                f"expected : {expected[key]:,}\n"
                f"actual   : {actual[key]:,}"
            )


# =============================================================================
# RANDOM_NATURAL
# =============================================================================

rn_expected = summary["cells"]["RANDOM_NATURAL"]

rn_train = membership_counts(
    random_train_mask,
    labels,
)

rn_validation = membership_counts(
    random_validation_mask,
    labels,
)


assert_counts(
    "RANDOM_NATURAL train",
    rn_train,
    rn_expected["train"],
)

assert_counts(
    "RANDOM_NATURAL validation",
    rn_validation,
    rn_expected["validation"],
)


# =============================================================================
# CHRONOLOGICAL_NATURAL
# =============================================================================
#
# Frozen rule:
#
#       train = day_id 0..6
#       validation = day_id 7
#
# =============================================================================

chronological_train_mask = day_ids <= 6
chronological_validation_mask = day_ids == 7


if np.any(
    chronological_train_mask
    & chronological_validation_mask
):
    raise RuntimeError(
        "Chronological train/validation overlap detected."
    )


if not np.all(
    chronological_train_mask
    | chronological_validation_mask
):
    raise RuntimeError(
        "Chronological membership leaves development rows unassigned."
    )


cn_expected = summary["cells"]["CHRONOLOGICAL_NATURAL"]

cn_train = membership_counts(
    chronological_train_mask,
    labels,
)

cn_validation = membership_counts(
    chronological_validation_mask,
    labels,
)


assert_counts(
    "CHRONOLOGICAL_NATURAL train",
    cn_train,
    cn_expected["train"],
)

assert_counts(
    "CHRONOLOGICAL_NATURAL validation",
    cn_validation,
    cn_expected["validation"],
)


# =============================================================================
# PRINT VERIFIED MEMBERSHIPS
# =============================================================================

print()
print("=" * 92)
print("RANDOM_NATURAL — VERIFIED")
print("=" * 92)

print()

print("TRAIN")
print(f"  rows   : {rn_train['rows']:,}")
print(f"  benign : {rn_train['benign']:,}")
print(f"  attack : {rn_train['attack']:,}")
print(
    f"  prevalence: "
    f"{rn_train['attack'] / rn_train['rows']:.10f}"
)

print()

print("VALIDATION")
print(f"  rows   : {rn_validation['rows']:,}")
print(f"  benign : {rn_validation['benign']:,}")
print(f"  attack : {rn_validation['attack']:,}")
print(
    f"  prevalence: "
    f"{rn_validation['attack'] / rn_validation['rows']:.10f}"
)


print()
print("=" * 92)
print("CHRONOLOGICAL_NATURAL — VERIFIED")
print("=" * 92)

print()

print("TRAIN")
print(f"  rows   : {cn_train['rows']:,}")
print(f"  benign : {cn_train['benign']:,}")
print(f"  attack : {cn_train['attack']:,}")
print(
    f"  prevalence: "
    f"{cn_train['attack'] / cn_train['rows']:.10f}"
)

print()

print("VALIDATION")
print(f"  rows   : {cn_validation['rows']:,}")
print(f"  benign : {cn_validation['benign']:,}")
print(f"  attack : {cn_validation['attack']:,}")
print(
    f"  prevalence: "
    f"{cn_validation['attack'] / cn_validation['rows']:.10f}"
)


# =============================================================================
# VERIFY RANDOM SPLIT PARTITION INVARIANTS
# =============================================================================

assert not np.any(
    random_train_mask
    & random_validation_mask
)

assert np.all(
    random_train_mask
    | random_validation_mask
)

assert (
    rn_train["rows"]
    + rn_validation["rows"]
    == logical_length
)


# =============================================================================
# VERIFY CHRONOLOGICAL PARTITION INVARIANTS
# =============================================================================

assert not np.any(
    chronological_train_mask
    & chronological_validation_mask
)

assert np.all(
    chronological_train_mask
    | chronological_validation_mask
)

assert (
    cn_train["rows"]
    + cn_validation["rows"]
    == logical_length
)


# =============================================================================
# VERIFY FROZEN RANDOM OPERATOR RECEIPT
# =============================================================================

random_operator = summary["random_operator"]

print()
print("=" * 92)
print("FROZEN RANDOM OPERATOR")
print("=" * 92)

print()

for key, value in random_operator.items():
    print(f"{key:20s}: {value}")


assert random_operator["random_state"] == 42
assert random_operator["shuffle"] is True
assert random_operator["stratify"] == "binary_label"
assert random_operator["test_size"] == 0.2
assert random_operator["sklearn_version"] == "1.6.1"


# =============================================================================
# WRITE LOCAL PRE-FREEZE RECEIPT
# =============================================================================

RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage23_prefreeze_natural_membership_verification.json"
)


receipt = {

    "audit_type":
        "stage23_prefreeze_inherited_natural_memberships",

    "stage23_0_frozen": False,

    "stage23_models_fit": 0,

    "stage23_metrics_calculated": 0,

    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,

    "development_rows":
        int(logical_length),

    "random_validation_bitset": {
        "file": str(RANDOM_VALIDATION_BITSET),
        "sha256": actual_bitset_sha,
        "bytes": actual_bitset_bytes,
        "logical_length": int(logical_length),
        "population": int(random_population),
        "bit_order": bit_order,
    },

    "RANDOM_NATURAL": {
        "train": rn_train,
        "validation": rn_validation,
        "derivation":
            summary["membership_derivation"][
                "RANDOM_NATURAL_train"
            ],
    },

    "CHRONOLOGICAL_NATURAL": {
        "train": cn_train,
        "validation": cn_validation,
        "train_derivation":
            summary["membership_derivation"][
                "CHRONOLOGICAL_NATURAL_train"
            ],
        "validation_derivation":
            summary["membership_derivation"][
                "CHRONOLOGICAL_NATURAL_validation"
            ],
    },

    "random_operator":
        random_operator,

    "canonical_order":
        summary["clean_development"]["canonical_order"],
}


RECEIPT_PATH.write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 92)
print("STAGE23 PRE-FREEZE NATURAL MEMBERSHIP VERIFICATION: COMPLETE")
print("=" * 92)

print()
print("RANDOM_NATURAL:")
print("  frozen membership reproduced : YES")
print("  train/validation overlap     : 0")
print("  unassigned rows              : 0")

print()
print("CHRONOLOGICAL_NATURAL:")
print("  frozen membership reproduced : YES")
print("  train/validation overlap     : 0")
print("  unassigned rows              : 0")

print()
print("Predictor features read : 0")
print("Stage23 models fit      : 0")
print("Stage23 metrics run     : 0")

print()
print("Raw Mar1 read : NO")
print("Raw Mar2 read : NO")

print()
print("Stage23-0 frozen : NO")

print()
print("Local receipt:")
print(" ", RECEIPT_PATH)

print()
print("NEXT:")
print(
    "  Resolve prospective Stage23 design decisions and "
    "construct the Stage23-0 protocol lock."
)

print("=" * 92)

STAGE22R FROZEN MEMBERSHIP RECEIPT

Canonical order: (day_id ASC, original_zero_based_row_index ASC) after frozen K79 exclusions
Development rows: 14,412,403
Development benign: 12,440,104
Development attack: 1,972,299

RANDOM VALIDATION BITSET

Bytes   : 1,801,551
SHA256  : 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
Bitorder: little
Length  : 14,412,403
Expected population: 2,882,481

[OK] Frozen random-validation bitset integrity verified.
Actual validation population: 2,882,481
Actual training population: 11,529,922

READING AUTHORIZED MEMBERSHIP SUPPORT COLUMNS

[OK] day 00 day_00_02-14-2018.parquet
     positions : 0 .. 822,946
     rows      : 822,947
     benign    : 666,273
     attack    : 156,674

[OK] day 01 day_01_02-15-2018.parquet
     positions : 822,947 .. 1,869,100
     rows      : 1,046,154
     benign    : 994,414
     attack    : 51,740

[OK] day 02 day_02_02-16-2018.parquet
     positions : 1,869,101 .. 2,770,088
     rows      : 900,988
     

In [8]:
# =============================================================================
# STAGE23 PRE-FREEZE — COMMIT + PUSH PROVENANCE CHECKPOINT
# =============================================================================
#
# PUSHES ONLY:
#   1. fresh bootstrap receipt
#   2. exact Stage22R cache provenance audit
#   3. exact NATURAL membership verification
#
# DOES NOT:
#   - create Stage23-0
#   - freeze feature subsets
#   - train anything
#   - create a Stage23 scientific tag
#
# =============================================================================

from pathlib import Path
import shutil
import subprocess
import os


REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "fafb131981b6e15e47bdf35fd8c18ea228680fe3"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_prefreeze_provenance"
)

SOURCE_FILES = [
    Path("/kaggle/working/stage23_fresh_bootstrap_receipt.json"),
    Path("/kaggle/working/stage23_prefreeze_stage22r_cache_audit.json"),
    Path("/kaggle/working/stage23_prefreeze_natural_membership_verification.json"),
]


def run(cmd, cwd=REPO_DIR, check=True):
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


print("=" * 88)
print("STAGE23 PRE-FREEZE PROVENANCE PUSH")
print("=" * 88)
print()


# =============================================================================
# 1. VERIFY REPOSITORY STATE BEFORE TOUCHING IT
# =============================================================================

branch = run(
    ["git", "branch", "--show-current"]
).strip()

head_before = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status_before = run(
    ["git", "status", "--porcelain"]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected branch 'main', found '{branch}'."
    )


if head_before != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected repository HEAD before Stage23 pre-freeze push.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual  : {head_before}\n"
        "\nStop and inspect before committing."
    )


if status_before:
    raise RuntimeError(
        "\nRepository was not clean before creating provenance files:\n"
        + status_before
    )


print("[OK] branch:", branch)
print("[OK] parent:", head_before)
print("[OK] worktree clean before checkpoint")


# =============================================================================
# 2. VERIFY ALL THREE LOCAL RECEIPTS EXIST
# =============================================================================

print()
print("Verifying local receipts...")

for src in SOURCE_FILES:

    if not src.exists():
        raise RuntimeError(
            f"Required local receipt missing: {src}"
        )

    print(
        f"[OK] {src.name} "
        f"({src.stat().st_size:,} bytes)"
    )


# =============================================================================
# 3. COPY RECEIPTS INTO REPOSITORY
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for src in SOURCE_FILES:

    dst = TARGET_DIR / src.name

    shutil.copy2(
        src,
        dst,
    )

    print(
        f"Copied: {src.name}"
    )


# =============================================================================
# 4. CREATE A SMALL GOVERNANCE README
# =============================================================================

README = TARGET_DIR / "README.md"

README.write_text(
"""# Stage23 Pre-Freeze Provenance

This directory records the prospective provenance checks completed before
creation of the Stage23-0 shortcut-feature-audit protocol lock.

Verified before any Stage23 experiment:

- Stage22R scientific seal verified.
- Stage22R publication closeout verified.
- Eight authorized development cache files match the frozen Stage22R-1C
  byte sizes, SHA-256 hashes, row counts, and schemas exactly.
- The frozen 70-feature model input is present in all eight cache files.
- `RANDOM_NATURAL` was reconstructed exactly from the frozen
  `random_validation.packbits` membership.
- `CHRONOLOGICAL_NATURAL` was reconstructed exactly using development
  days 0-6 for training and day 7 (Feb28) for validation.
- Random and chronological partitions have zero train/validation overlap
  and zero unassigned development rows.
- No predictor feature was read during membership verification.
- No Stage23 model was fit.
- No Stage23 metric was calculated.
- Raw Mar1 and Mar2 were not opened.

## Important unresolved prospective design point

The frozen Stage22R 70-feature input contains `Dst Port` and `Protocol`,
but does not contain `Src Port`.

Therefore the exact operational definition of the requested Stage23
`NO_PORTS` subset must be resolved and frozen prospectively in Stage23-0.
It has not been defined by this checkpoint.

## Status

This is a provenance checkpoint only.

`Stage23-0` is **not yet frozen**.
""",
    encoding="utf-8",
)


# =============================================================================
# 5. SHOW EXACT FILES ABOUT TO BE COMMITTED
# =============================================================================

relative_files = [
    p.relative_to(REPO_DIR)
    for p in [
        TARGET_DIR / SOURCE_FILES[0].name,
        TARGET_DIR / SOURCE_FILES[1].name,
        TARGET_DIR / SOURCE_FILES[2].name,
        README,
    ]
]


print()
print("=" * 88)
print("FILES TO COMMIT")
print("=" * 88)

for path in relative_files:
    print(" ", path)


# =============================================================================
# 6. STAGE ONLY THESE FILES
# =============================================================================

run(
    [
        "git",
        "add",
        "--",
        *[str(p) for p in relative_files],
    ]
)


staged_names = run(
    ["git", "diff", "--cached", "--name-only"]
).splitlines()


expected_names = sorted(
    str(p)
    for p in relative_files
)

actual_names = sorted(staged_names)


if actual_names != expected_names:
    raise RuntimeError(
        "\nUnexpected staged file set.\n\n"
        f"Expected:\n{expected_names}\n\n"
        f"Actual:\n{actual_names}"
    )


print()
print("[OK] Only intended Stage23 pre-freeze files are staged.")


# =============================================================================
# 7. SHOW STAGED DIFF STAT
# =============================================================================

print()
print("=" * 88)
print("STAGED DIFF")
print("=" * 88)

run(
    ["git", "diff", "--cached", "--stat"]
)


# =============================================================================
# 8. COMMIT
# =============================================================================

COMMIT_MESSAGE = (
    "Stage23 prefreeze: verify Stage22R provenance and natural memberships"
)

print()
print("Creating commit...")

run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


new_head = run(
    ["git", "rev-parse", "HEAD"]
).strip()


print()
print("New commit:")
print(" ", new_head)


# =============================================================================
# 9. PUSH DIRECTLY TO MAIN
# =============================================================================

print()
print("Pushing main...")

run(
    [
        "git",
        "push",
        "origin",
        "main",
    ]
)


# =============================================================================
# 10. FINAL VERIFY
# =============================================================================

remote_head = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).split()[0]


if remote_head != new_head:
    raise RuntimeError(
        "\nRemote main does not match local HEAD after push.\n"
        f"local : {new_head}\n"
        f"remote: {remote_head}"
    )


final_status = run(
    ["git", "status", "--porcelain"]
).strip()


if final_status:
    raise RuntimeError(
        "\nWorking tree is not clean after push:\n"
        + final_status
    )


print()
print("=" * 88)
print("STAGE23 PRE-FREEZE PROVENANCE CHECKPOINT PUSHED")
print("=" * 88)

print()
print("Parent:")
print(" ", EXPECTED_PARENT)

print()
print("Commit:")
print(" ", new_head)

print()
print("Remote main:")
print(" ", remote_head)

print()
print("Git status: CLEAN")

print()
print("Stage23-0 frozen : NO")
print("Stage23 models   : 0")
print("Stage23 metrics  : 0")
print("Mar1 raw read    : NO")
print("Mar2 raw read    : NO")

print()
print("Tag created      : NO")
print()
print("NEXT:")
print("  Construct and freeze the prospective Stage23-0 protocol lock.")

print("=" * 88)

STAGE23 PRE-FREEZE PROVENANCE PUSH

main
fafb131981b6e15e47bdf35fd8c18ea228680fe3
[OK] branch: main
[OK] parent: fafb131981b6e15e47bdf35fd8c18ea228680fe3
[OK] worktree clean before checkpoint

Verifying local receipts...
[OK] stage23_fresh_bootstrap_receipt.json (1,233 bytes)
[OK] stage23_prefreeze_stage22r_cache_audit.json (5,033 bytes)
[OK] stage23_prefreeze_natural_membership_verification.json (1,556 bytes)
Copied: stage23_fresh_bootstrap_receipt.json
Copied: stage23_prefreeze_stage22r_cache_audit.json
Copied: stage23_prefreeze_natural_membership_verification.json

FILES TO COMMIT
  results/stage23_shortcut_feature_audit/stage23_prefreeze_provenance/stage23_fresh_bootstrap_receipt.json
  results/stage23_shortcut_feature_audit/stage23_prefreeze_provenance/stage23_prefreeze_stage22r_cache_audit.json
  results/stage23_shortcut_feature_audit/stage23_prefreeze_provenance/stage23_prefreeze_natural_membership_verification.json
  results/stage23_shortcut_feature_audit/stage23_prefreeze_prov

In [2]:
# =============================================================================
# STAGE23-0A — PROSPECTIVE SHORTCUT-FEATURE AUDIT PROTOCOL CANDIDATE
# =============================================================================
#
# SCIENTIFIC STATUS:
#   PROSPECTIVE / PRE-RESULT
#
# THIS CELL:
#   - reads frozen Stage22R Git receipts
#   - reads ONLY support columns from the 8 authorized development caches
#   - defines every Stage23 subset prospectively
#   - freezes behavior-only whitelist
#   - freezes matched-size placebo ablations
#   - freezes model / metric / stump / SHAP / uncertainty specifications
#   - freezes exact SHAP + uncertainty cohorts
#   - hashes all candidate artifacts
#
# THIS CELL DOES NOT:
#   - fit any model
#   - calculate any Stage23 model metric
#   - read predictor values
#   - read Mar1
#   - read Mar2
#   - commit anything
#   - create a Git tag
#
# AFTER THIS CELL:
#   Inspect the complete printed whitelist/specification.
#   Only then will Stage23-0B copy these exact bytes into Git and seal them.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import csv
import os

import numpy as np
import pyarrow.parquet as pq


# =============================================================================
# 0. PATHS / GOVERNANCE
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

CANDIDATE_DIR = Path(
    "/kaggle/working/stage23_0_protocol_candidate"
)

STAGE22R_CLOSEOUT = (
    "fafb131981b6e15e47bdf35fd8c18ea228680fe3"
)

STAGE22R_SCIENTIFIC = (
    "b5e44615269198426cc8a9aa3b3e701c2ca9e48e"
)

INTENDED_TAG = "stage23-0-protocol-lock-v1"


def run(cmd, cwd=REPO_DIR, check=True):
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


# =============================================================================
# 1. REPOSITORY PRECONDITIONS
# =============================================================================

branch = run(
    ["git", "branch", "--show-current"]
).strip()

parent_head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Stage23-0 must be constructed from main; found {branch}"
    )

if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-0A:\n" + status
    )


ancestry = subprocess.run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        STAGE22R_CLOSEOUT,
        parent_head,
    ],
    cwd=str(REPO_DIR),
)

if ancestry.returncode != 0:
    raise RuntimeError(
        "Stage22R publication closeout is not an ancestor of HEAD."
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        INTENDED_TAG,
    ]
).strip()

if existing_tag:
    raise RuntimeError(
        f"Intended Stage23-0 tag already exists: {INTENDED_TAG}"
    )


if CANDIDATE_DIR.exists():
    raise RuntimeError(
        f"Candidate directory already exists:\n{CANDIDATE_DIR}\n"
        "Remove it only if intentionally rebuilding before Stage23-0 is sealed."
    )


CANDIDATE_DIR.mkdir(parents=True)


print("=" * 96)
print("STAGE23-0A — PROSPECTIVE PROTOCOL CONSTRUCTION")
print("=" * 96)
print()
print("Branch     :", branch)
print("Parent HEAD:", parent_head)
print("Git status : CLEAN")
print("Target tag :", INTENDED_TAG)
print()


# =============================================================================
# 2. FROZEN STAGE22R RECEIPTS
# =============================================================================

S22_INPUT_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
)

S22_MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RN_RESULT_PATH = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

CN_RESULT_PATH = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

FEATURE_ORDER_PATH = (
    S22_INPUT_DIR
    / "stage22r_1c_70_feature_order.csv"
)

INPUT_MANIFEST_PATH = (
    S22_INPUT_DIR
    / "stage22r_1c_development_model_input_manifest.json"
)

MEMBERSHIP_SUMMARY_PATH = (
    S22_MEMBERSHIP_DIR
    / "stage22r_1b1_membership_summary.json"
)

RANDOM_VALIDATION_BITSET = (
    S22_MEMBERSHIP_DIR
    / "random_validation.packbits"
)


required_receipts = [
    RN_RESULT_PATH,
    CN_RESULT_PATH,
    FEATURE_ORDER_PATH,
    INPUT_MANIFEST_PATH,
    MEMBERSHIP_SUMMARY_PATH,
    RANDOM_VALIDATION_BITSET,
]

for path in required_receipts:
    if not path.exists():
        raise RuntimeError(
            f"Frozen inherited artifact missing: {path}"
        )


rn_result = json.loads(
    RN_RESULT_PATH.read_text(encoding="utf-8")
)

cn_result = json.loads(
    CN_RESULT_PATH.read_text(encoding="utf-8")
)

input_manifest = json.loads(
    INPUT_MANIFEST_PATH.read_text(encoding="utf-8")
)

membership_summary = json.loads(
    MEMBERSHIP_SUMMARY_PATH.read_text(encoding="utf-8")
)


with FEATURE_ORDER_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:
    feature_rows = list(csv.DictReader(f))


feature_rows.sort(
    key=lambda x: int(x["feature_position"])
)

FULL_FEATURES = [
    row["feature_name"]
    for row in feature_rows
]


if len(FULL_FEATURES) != 70:
    raise RuntimeError(
        f"Expected 70 frozen features; found {len(FULL_FEATURES)}"
    )

if rn_result["data"]["feature_order"] != FULL_FEATURES:
    raise RuntimeError(
        "RANDOM_NATURAL feature order differs from Stage22R-1C."
    )

if cn_result["data"]["feature_order"] != FULL_FEATURES:
    raise RuntimeError(
        "CHRONOLOGICAL_NATURAL feature order differs from Stage22R-1C."
    )


# =============================================================================
# 3. VERIFY INHERITED MODEL RECIPE IS IDENTICAL ACROSS BOTH NATURAL CELLS
# =============================================================================

rn_lgbm = rn_result["models"]["lightgbm"]["executed_parameters"]
cn_lgbm = cn_result["models"]["lightgbm"]["executed_parameters"]

rn_xgb = rn_result["models"]["xgboost"]["parameters"]
cn_xgb = cn_result["models"]["xgboost"]["parameters"]


if rn_lgbm != cn_lgbm:
    raise RuntimeError(
        "Inherited LightGBM configurations differ across natural cells."
    )

if rn_xgb != cn_xgb:
    raise RuntimeError(
        "Inherited XGBoost configurations differ across natural cells."
    )


if rn_result["models"]["strategy"] != "ENS_LGBM_XGB_EQUAL":
    raise RuntimeError("Unexpected Stage22R ensemble strategy.")

if cn_result["models"]["strategy"] != "ENS_LGBM_XGB_EQUAL":
    raise RuntimeError("Unexpected Stage22R ensemble strategy.")


# =============================================================================
# 4. STAGE23 ENVIRONMENT LOCK
# =============================================================================

PACKAGE_NAMES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "shap": "0.51.0",
}


actual_packages = {}

for package, expected in PACKAGE_NAMES.items():

    actual = importlib_metadata.version(package)

    actual_packages[package] = actual

    if actual != expected:
        raise RuntimeError(
            f"Stage23 environment mismatch for {package}: "
            f"expected {expected}, actual {actual}"
        )


# =============================================================================
# 5. PRIMARY SUSPICIOUS GROUP — FROZEN PROSPECTIVELY
# =============================================================================

SUSPICIOUS_GROUP = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


for feature in SUSPICIOUS_GROUP:
    if feature not in FULL_FEATURES:
        raise RuntimeError(
            f"Suspicious feature absent: {feature}"
        )


# =============================================================================
# 6. BEHAVIOR-ONLY EXCLUSIONS / WHITELIST
# =============================================================================
#
# Exclusions are SEMANTIC and frozen before Stage23 model results:
#
#   Dst Port
#       destination/service context
#
#   Protocol
#       transport/network protocol identifier
#
#   Init Fwd Win Byts
#   Init Bwd Win Byts
#       TCP initial-window / stack-context candidates
#
#   Fwd Seg Size Min
#       TCP segment/configuration fingerprint candidate
#
#   Fwd Header Len
#   Bwd Header Len
#       header/configuration-related context candidates
#
# Everything else remains in original Stage22R order.
# =============================================================================

BEHAVIOR_EXCLUSIONS = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


for feature in BEHAVIOR_EXCLUSIONS:
    if feature not in FULL_FEATURES:
        raise RuntimeError(
            f"Behavior exclusion absent from FULL: {feature}"
        )


BEHAVIOR_ONLY = [
    feature
    for feature in FULL_FEATURES
    if feature not in set(BEHAVIOR_EXCLUSIONS)
]


if len(BEHAVIOR_ONLY) != 63:
    raise RuntimeError(
        f"Expected 63 behavior-only features; found {len(BEHAVIOR_ONLY)}"
    )


# =============================================================================
# 7. PRIMARY SEVEN SUBSETS
# =============================================================================
#
# IMPORTANT NO_PORTS RESOLUTION
#
# Src Port is absent from the frozen Stage22R 70-feature space.
#
# To preserve the REQUIRED subset name while avoiding duplication with
# NO_DST_PORT:
#
#   NO_PORTS = remove Dst Port + Protocol
#
# Operational semantic label:
#
#   TRANSPORT_IDENTIFIER_RESTRICTION
#
# It MUST NOT be described as removing Src Port.
# =============================================================================

PRIMARY_SUBSETS = {
    "FULL": {
        "mode": "all",
        "removed": [],
        "features": FULL_FEATURES,
        "semantic_label": "full_frozen_stage22r_feature_space",
    },

    "NO_DST_PORT": {
        "mode": "remove",
        "removed": [
            "Dst Port",
        ],
        "features": [
            f for f in FULL_FEATURES
            if f != "Dst Port"
        ],
        "semantic_label": "destination_port_ablation",
    },

    "NO_PORTS": {
        "mode": "remove",
        "removed": [
            "Dst Port",
            "Protocol",
        ],
        "features": [
            f for f in FULL_FEATURES
            if f not in {"Dst Port", "Protocol"}
        ],
        "semantic_label": "transport_identifier_restriction",
        "important_note":
            "Src Port does not exist in the frozen Stage22R 70-feature "
            "space. NO_PORTS prospectively removes Dst Port + Protocol "
            "to remain scientifically distinct from NO_DST_PORT.",
    },

    "NO_INIT_FWD_WIN_BYTS": {
        "mode": "remove",
        "removed": [
            "Init Fwd Win Byts",
        ],
        "features": [
            f for f in FULL_FEATURES
            if f != "Init Fwd Win Byts"
        ],
        "semantic_label": "initial_forward_window_ablation",
    },

    "NO_FWD_SEG_SIZE_MIN": {
        "mode": "remove",
        "removed": [
            "Fwd Seg Size Min",
        ],
        "features": [
            f for f in FULL_FEATURES
            if f != "Fwd Seg Size Min"
        ],
        "semantic_label": "minimum_forward_segment_ablation",
    },

    "NO_SUSPICIOUS_GROUP": {
        "mode": "remove",
        "removed": SUSPICIOUS_GROUP,
        "features": [
            f for f in FULL_FEATURES
            if f not in set(SUSPICIOUS_GROUP)
        ],
        "semantic_label": "joint_shortcut_prone_group_ablation",
    },

    "BEHAVIOR_ONLY": {
        "mode": "whitelist",
        "removed": BEHAVIOR_EXCLUSIONS,
        "features": BEHAVIOR_ONLY,
        "semantic_label": "behavior_restricted_feature_set",
    },
}


if list(PRIMARY_SUBSETS.keys()) != [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]:
    raise RuntimeError("Primary subset ordering changed.")


for name, spec in PRIMARY_SUBSETS.items():

    if len(spec["features"]) != len(set(spec["features"])):
        raise RuntimeError(
            f"Duplicate feature in subset {name}"
        )

    if any(f not in FULL_FEATURES for f in spec["features"]):
        raise RuntimeError(
            f"Unknown feature in subset {name}"
        )


# =============================================================================
# 8. MATCHED-SIZE PLACEBO ABLATIONS
# =============================================================================
#
# k = size of suspicious group = 3.
#
# Five preselected ordinary behavioral triplets.
# No model result was consulted.
#
# They span:
#   counts
#   volumes/direction
#   timing
#   packet-size dynamics
#   active/idle dynamics
# =============================================================================

PLACEBO_K = len(SUSPICIOUS_GROUP)

PLACEBO_SUBSETS = {
    "PLACEBO_COUNTS": [
        "Flow Duration",
        "Tot Fwd Pkts",
        "Tot Bwd Pkts",
    ],

    "PLACEBO_VOLUME_DIRECTION": [
        "TotLen Fwd Pkts",
        "TotLen Bwd Pkts",
        "Down/Up Ratio",
    ],

    "PLACEBO_IAT": [
        "Flow IAT Mean",
        "Flow IAT Std",
        "Flow IAT Max",
    ],

    "PLACEBO_PACKET_SIZE": [
        "Pkt Len Mean",
        "Pkt Len Std",
        "Pkt Len Var",
    ],

    "PLACEBO_ACTIVITY": [
        "Active Mean",
        "Idle Mean",
        "Fwd Act Data Pkts",
    ],
}


for name, removed in PLACEBO_SUBSETS.items():

    if len(removed) != PLACEBO_K:
        raise RuntimeError(
            f"{name} is not matched-size k={PLACEBO_K}"
        )

    for feature in removed:
        if feature not in FULL_FEATURES:
            raise RuntimeError(
                f"Unknown placebo feature: {feature}"
            )

        if feature in SUSPICIOUS_GROUP:
            raise RuntimeError(
                f"Placebo contains suspicious feature: {feature}"
            )


# =============================================================================
# 9. INHERITED SPLITS
# =============================================================================

SPLITS = {
    "RANDOM_NATURAL": {
        "train": membership_summary["cells"][
            "RANDOM_NATURAL"
        ]["train"],

        "validation": membership_summary["cells"][
            "RANDOM_NATURAL"
        ]["validation"],

        "derivation":
            "logical complement / membership of frozen "
            "random_validation.packbits",

        "random_operator":
            membership_summary["random_operator"],
    },

    "CHRONOLOGICAL_NATURAL": {
        "train": membership_summary["cells"][
            "CHRONOLOGICAL_NATURAL"
        ]["train"],

        "validation": membership_summary["cells"][
            "CHRONOLOGICAL_NATURAL"
        ]["validation"],

        "train_days": [
            "02-14-2018",
            "02-15-2018",
            "02-16-2018",
            "02-20-2018",
            "02-21-2018",
            "02-22-2018",
            "02-23-2018",
        ],

        "validation_days": [
            "02-28-2018",
        ],
    },
}


# =============================================================================
# 10. MODEL SPECIFICATION
# =============================================================================

MODEL_SPEC = {
    "strategy":
        "ENS_LGBM_XGB_EQUAL",

    "ensemble_probability":
        "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

    "probability_combination_dtype":
        "float64",

    "persisted_probability_dtype":
        "float32",

    "preprocessing": {
        "parse_dtype": "float64",
        "scaling": "NONE",
        "explicit_imputation": "NONE",
        "positive_infinity": "CONVERT_TO_NAN",
        "negative_infinity": "CONVERT_TO_NAN",
        "other_nonnumeric_token": "FAIL_CLOSED",
    },

    "lightgbm": {
        "configuration": "LGBM_11",
        "library_version": "4.6.0",
        "parameters": rn_lgbm,
        "backend_note":
            "Stage22R-frozen execution amendment uses CPU only; "
            "algorithmic hyperparameters unchanged.",
    },

    "xgboost": {
        "configuration": "XGB_11",
        "library_version": "3.2.0",
        "parameters": rn_xgb,
    },

    "hyperparameter_policy":
        "IDENTICAL_FOR_EVERY_FEATURE_SUBSET_AND_SPLIT; "
        "NO_SUBSET_SPECIFIC_TUNING",

    "full_reference_policy":
        "Reuse frozen Stage22R RANDOM_NATURAL and "
        "CHRONOLOGICAL_NATURAL FULL models/results where possible.",

    "rebalancing":
        "NOT_PART_OF_PRIMARY_STAGE23_AUDIT",
}


# =============================================================================
# 11. PRIMARY METRICS + INTERACTION
# =============================================================================

METRIC_SPEC = {
    "primary_ranking_metric":
        "PR_AUC",

    "secondary_ranking_metrics": [
        "ROC_AUC",
        "PR_AUC_MINUS_ATTACK_PREVALENCE",
    ],

    "secondary_fixed_operating_point": {
        "threshold": 0.50,
        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "metrics": [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "fpr",
            "fnr",
            "tn",
            "fp",
            "fn",
            "tp",
        ],
    },

    "threshold_optimization_policy":
        "NO_PER_SUBSET_THRESHOLD_OPTIMIZATION_FOR_PRIMARY_STAGE23_EVIDENCE",

    "primary_removal_penalty": {
        "random":
            "M_FULL_RANDOM - M_SUBSET_RANDOM",

        "chronological":
            "M_FULL_CHRONOLOGICAL - M_SUBSET_CHRONOLOGICAL",

        "shortcut_interaction":
            "(M_FULL_RANDOM - M_SUBSET_RANDOM) - "
            "(M_FULL_CHRONOLOGICAL - M_SUBSET_CHRONOLOGICAL)",
    },

    "headline_interaction_metrics": [
        "PR_AUC",
        "ROC_AUC",
    ],

    "supplementary_interaction_metrics": [
        "f1_at_0_50",
        "recall_at_0_50",
        "fpr_at_0_50",
    ],
}


# =============================================================================
# 12. STUMP SPECIFICATION
# =============================================================================

STUMP_SPEC = {
    "features": [
        "Dst Port",
        "Init Fwd Win Byts",
        "Fwd Seg Size Min",
    ],

    "splits": [
        "RANDOM_NATURAL",
        "CHRONOLOGICAL_NATURAL",
    ],

    "implementation":
        "sklearn.tree.DecisionTreeClassifier",

    "library_version":
        actual_packages["scikit-learn"],

    "parameters": {
        "criterion": "gini",
        "splitter": "best",
        "max_depth": 1,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "min_weight_fraction_leaf": 0.0,
        "max_features": None,
        "random_state": 42,
        "max_leaf_nodes": None,
        "min_impurity_decrease": 0.0,
        "class_weight": None,
        "ccp_alpha": 0.0,
    },

    "missing_value_policy": {
        "method":
            "TRAINING_MEDIAN_IMPUTATION_FOR_STUMP_ONLY",

        "implementation":
            "sklearn.impute.SimpleImputer(strategy='median')",

        "fit_scope":
            "fit on the stump training membership only; "
            "apply unchanged to validation",

        "reason":
            "Tree stump control requires a deterministic explicit "
            "missing-value rule while the boosted Stage23 models retain "
            "the inherited no-explicit-imputation policy.",
    },

    "metrics": [
        "PR_AUC",
        "PR_AUC_MINUS_ATTACK_PREVALENCE",
        "ROC_AUC",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "fpr",
        "fnr",
    ],

    "operating_threshold":
        0.50,

    "report_tree_structure": [
        "selected_feature",
        "split_threshold",
        "missing_imputation_median",
        "left_leaf_class_distribution",
        "right_leaf_class_distribution",
        "left_leaf_support",
        "right_leaf_support",
        "branch_predicting_attack",
    ],

    "expected_fits": 6,
}


# =============================================================================
# 13. LOAD ONLY SUPPORT COLUMNS TO FREEZE SHAP / UNCERTAINTY COHORTS
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    lower = str(path).lower()

    for marker in FORBIDDEN_MARKERS:
        if marker in lower:
            raise RuntimeError(
                f"PERMANENT HOLDOUT ACCESS BLOCKED: {path}"
            )

    return path


clean_position_parts = []
day_id_parts = []
label_parts = []


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR / filename
    )

    table = pq.read_table(
        path,
        columns=[
            "clean_position",
            "day_id",
            "binary_label",
        ],
    )

    clean_position_parts.append(
        table["clean_position"].to_numpy(
            zero_copy_only=False
        )
    )

    day_id_parts.append(
        table["day_id"].to_numpy(
            zero_copy_only=False
        )
    )

    label_parts.append(
        table["binary_label"].to_numpy(
            zero_copy_only=False
        )
    )


clean_positions = np.concatenate(
    clean_position_parts
)

day_ids = np.concatenate(
    day_id_parts
)

labels = np.concatenate(
    label_parts
)


N = len(labels)

if N != 14_412_403:
    raise RuntimeError(
        f"Development length mismatch: {N}"
    )

if not np.array_equal(
    clean_positions,
    np.arange(N, dtype=clean_positions.dtype),
):
    raise RuntimeError(
        "Canonical clean_position sequence changed."
    )


# RANDOM validation mask

packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)

random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N].astype(bool)


# CHRONOLOGICAL validation

chrono_validation_mask = (
    day_ids == 7
)


if int(random_validation_mask.sum()) != 2_882_481:
    raise RuntimeError(
        "RANDOM validation membership changed."
    )

if int(chrono_validation_mask.sum()) != 593_780:
    raise RuntimeError(
        "CHRONOLOGICAL validation membership changed."
    )


# =============================================================================
# 14. DETERMINISTIC COHORT BUILDERS
# =============================================================================

def sample_balanced_shap_cohort(
    validation_mask,
    seed,
    n_per_class=2500,
):

    rng = np.random.RandomState(seed)

    benign_positions = clean_positions[
        validation_mask & (labels == 0)
    ]

    attack_positions = clean_positions[
        validation_mask & (labels == 1)
    ]

    chosen_benign = rng.choice(
        benign_positions,
        size=n_per_class,
        replace=False,
    )

    chosen_attack = rng.choice(
        attack_positions,
        size=n_per_class,
        replace=False,
    )

    rows = [
        (int(pos), 0)
        for pos in chosen_benign
    ] + [
        (int(pos), 1)
        for pos in chosen_attack
    ]

    rows.sort(key=lambda x: x[0])

    return rows


def sample_natural_uncertainty_cohort(
    validation_mask,
    seed,
    total_size=50_000,
):

    rng = np.random.RandomState(seed)

    val_positions = clean_positions[
        validation_mask
    ]

    val_labels = labels[
        validation_mask
    ]

    prevalence = float(
        (val_labels == 1).mean()
    )

    attack_n = int(
        round(total_size * prevalence)
    )

    benign_n = total_size - attack_n

    benign_positions = val_positions[
        val_labels == 0
    ]

    attack_positions = val_positions[
        val_labels == 1
    ]

    chosen_benign = rng.choice(
        benign_positions,
        size=benign_n,
        replace=False,
    )

    chosen_attack = rng.choice(
        attack_positions,
        size=attack_n,
        replace=False,
    )

    rows = [
        (int(pos), 0)
        for pos in chosen_benign
    ] + [
        (int(pos), 1)
        for pos in chosen_attack
    ]

    rows.sort(key=lambda x: x[0])

    return rows, {
        "rows": total_size,
        "benign": benign_n,
        "attack": attack_n,
        "attack_prevalence":
            attack_n / total_size,
        "source_validation_prevalence":
            prevalence,
    }


# Use seed 42 restarted independently for each split.

SHAP_RANDOM = sample_balanced_shap_cohort(
    random_validation_mask,
    seed=42,
)

SHAP_CHRONO = sample_balanced_shap_cohort(
    chrono_validation_mask,
    seed=42,
)


UNC_RANDOM, UNC_RANDOM_COUNTS = (
    sample_natural_uncertainty_cohort(
        random_validation_mask,
        seed=42,
    )
)

UNC_CHRONO, UNC_CHRONO_COUNTS = (
    sample_natural_uncertainty_cohort(
        chrono_validation_mask,
        seed=42,
    )
)


# =============================================================================
# 15. WRITE COHORT CSV FILES
# =============================================================================

def write_cohort_csv(
    path,
    split_name,
    rows,
):

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "split",
            "clean_position",
            "binary_label",
        ])

        for clean_position, label in rows:
            writer.writerow([
                split_name,
                clean_position,
                label,
            ])


write_cohort_csv(
    CANDIDATE_DIR / "shap_cohort_random_natural.csv",
    "RANDOM_NATURAL",
    SHAP_RANDOM,
)

write_cohort_csv(
    CANDIDATE_DIR / "shap_cohort_chronological_natural.csv",
    "CHRONOLOGICAL_NATURAL",
    SHAP_CHRONO,
)

write_cohort_csv(
    CANDIDATE_DIR / "uncertainty_cohort_random_natural.csv",
    "RANDOM_NATURAL",
    UNC_RANDOM,
)

write_cohort_csv(
    CANDIDATE_DIR / "uncertainty_cohort_chronological_natural.csv",
    "CHRONOLOGICAL_NATURAL",
    UNC_CHRONO,
)


# =============================================================================
# 16. SHAP SPECIFICATION
# =============================================================================

SHAP_SPEC = {
    "purpose":
        "Quantify proxy absorption after removal of shortcut-prone features.",

    "cohort": {
        "rows_per_split": 5000,
        "benign_per_split": 2500,
        "attack_per_split": 2500,
        "sampling":
            "without replacement from frozen validation membership",
        "seed":
            42,
        "rng":
            "numpy.random.RandomState",
        "same_rows_reused_for_every_subset_within_split":
            True,
        "locator":
            "clean_position",
    },

    "components": [
        "LightGBM",
        "XGBoost",
    ],

    "method":
        "TreeSHAP computed separately for each component model.",

    "per_model_importance":
        "mean absolute SHAP over the frozen cohort",

    "aggregation": {
        "primary_component_reporting":
            "LightGBM and XGBoost reported separately",

        "descriptive_consensus":
            "For each component, divide mean_abs_SHAP(feature) by "
            "the sum of mean_abs_SHAP across retained features; "
            "then average the two normalized shares with weights 0.5/0.5.",

        "warning":
            "Consensus normalized importance is descriptive and must not "
            "be called exact SHAP for the probability-averaged ensemble.",
    },

    "proxy_absorption_measures": [
        "top_10_features",
        "top_20_features",
        "Jaccard_at_10_vs_FULL",
        "Jaccard_at_20_vs_FULL",
        "Spearman_rank_on_common_retained_features",
        "new_top_10_entrants",
        "new_top_20_entrants",
        "features_leaving_top_10",
        "features_leaving_top_20",
        "rank_displacement",
        "normalized_importance_share_change",
        "top_importance_gainers_after_removal",
    ],

    "retraining":
        "NONE; use frozen models from Stage23 primary/placebo runs",
}


# =============================================================================
# 17. UNCERTAINTY SPECIFICATION
# =============================================================================

UNCERTAINTY_SPEC = {
    "point_estimates":
        "Always computed on the complete frozen validation membership.",

    "ci_role":
        "Supplementary uncertainty for removal penalties and "
        "split-by-ablation interactions.",

    "method":
        "paired stratified bootstrap on a frozen natural-prevalence "
        "validation cohort",

    "cohort_size_per_split":
        50_000,

    "cohort_sampling":
        "without replacement; attack/benign counts selected to approximate "
        "the source validation prevalence as closely as possible",

    "cohort_seed":
        42,

    "bootstrap_replicates":
        1000,

    "bootstrap_rng":
        "numpy.random.RandomState(seed=42), restarted independently per split",

    "bootstrap_sampling":
        "Within each frozen cohort, benign and attack rows are resampled "
        "with replacement separately while preserving the cohort's fixed "
        "class counts.",

    "pairing":
        "Within a split and bootstrap replicate, exactly the same sampled "
        "rows are used for FULL and the compared ablated model.",

    "interaction_bootstrap":
        "For replicate b: "
        "(FULL_R_b - ABLATED_R_b) - "
        "(FULL_C_b - ABLATED_C_b).",

    "confidence_interval":
        "percentile 95% interval [2.5th, 97.5th]",

    "headline_ci_metrics": [
        "PR_AUC_removal_penalty",
        "ROC_AUC_removal_penalty",
        "PR_AUC_shortcut_interaction",
        "ROC_AUC_shortcut_interaction",
    ],

    "random_cohort":
        UNC_RANDOM_COUNTS,

    "chronological_cohort":
        UNC_CHRONO_COUNTS,

    "important_limitation":
        "Confidence intervals are based on the frozen uncertainty cohort; "
        "full-validation metrics remain the primary point estimates.",
}


# =============================================================================
# 18. ATTACK-FAMILY SPECIFICATION
# =============================================================================

ATTACK_FAMILY_SPEC = {
    "scope":
        "development-only; never Mar1/Mar2",

    "source_alignment":
        "Use day_id + original_row_index to recover development attack "
        "family labels from authorized Feb14-Feb28 source labels or "
        "existing repository attack-category infrastructure.",

    "no_retraining":
        True,

    "evaluation_definition":
        "For attack family A, form a validation evaluation set containing "
        "all benign rows from that split plus attack rows belonging to A.",

    "metrics": [
        "support_attack",
        "support_benign",
        "PR_AUC",
        "ROC_AUC",
        "precision_at_0_50",
        "recall_at_0_50",
        "f1_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
        "delta_f1_FULL_minus_ablated",
        "delta_recall_FULL_minus_ablated",
    ],

    "interpretation_minimum_attack_support":
        100,

    "low_support_policy":
        "Families with fewer than 100 validation attacks are listed "
        "descriptively but excluded from inferential interpretation.",

    "family_set_policy":
        "Do not add/drop attack families because of Stage23 results.",
}


# =============================================================================
# 19. INTERPRETATION MATRIX
# =============================================================================

INTERPRETATION_MATRIX = [
    {
        "pattern":
            "Removal penalties near zero under both split regimes",
        "permitted":
            "Little evidence that the tested feature/group materially "
            "drives ranking performance under these frozen validations.",
    },

    {
        "pattern":
            "Positive shortcut interaction with 95% CI excluding zero",
        "permitted":
            "Random validation benefits disproportionately from the "
            "tested feature/group relative to chronological validation.",
    },

    {
        "pattern":
            "Negative shortcut interaction with 95% CI excluding zero",
        "permitted":
            "Chronological validation depends more strongly on the "
            "tested feature/group than random validation.",
    },

    {
        "pattern":
            "Large positive removal penalties under both regimes",
        "permitted":
            "The tested information carries predictive information "
            "across both evaluation protocols.",
    },

    {
        "pattern":
            "Chronological removal penalty below zero",
        "permitted":
            "Removal is consistent with improved forward-temporal "
            "generalization; causal claims are not permitted.",
    },

    {
        "pattern":
            "SHAP importance moves toward semantically related retained features",
        "permitted":
            "Evidence consistent with proxy absorption.",
    },

    {
        "pattern":
            "BEHAVIOR_ONLY retains strong chronological discrimination",
        "permitted":
            "Substantial behavior-restricted capability survives under "
            "the frozen Stage23 development protocol.",
    },

    {
        "pattern":
            "BEHAVIOR_ONLY strongly degrades",
        "permitted":
            "Performance under this protocol depends substantially on "
            "information excluded by the behavior-restricted definition.",
    },
]


PROHIBITED_CLAIMS = [
    "Dst Port is leakage",
    "Init Fwd Win Byts is leakage",
    "Fwd Seg Size Min is leakage",
    "Random splitting is universally invalid",
    "Chronological splitting completely eliminates leakage",
    "K79 provides session independence",
    "K79 provides endpoint/IP/5-tuple disjointness",
    "Removing suspicious features proves causal reliance",
    "SHAP importance proves causation",
    "Behavior-only performance equals real-world deployment performance",
    "Stage23 has a new untouched final holdout",
]


# =============================================================================
# 20. FIGURE PLAN
# =============================================================================

FIGURE_PLAN = {
    "Figure_23_A": {
        "title":
            "Subset × split interaction",
        "content":
            "Seven primary subsets; RANDOM_NATURAL vs "
            "CHRONOLOGICAL_NATURAL PR-AUC and ROC-AUC.",
    },

    "Figure_23_B": {
        "title":
            "Removal penalties and shortcut interaction",
        "content":
            "FULL minus ablated penalties by split with frozen "
            "bootstrap uncertainty and random-minus-chronological interaction.",
    },

    "Figure_23_C": {
        "title":
            "Single-feature stump degradation",
        "content":
            "Dst Port, Init Fwd Win Byts, Fwd Seg Size Min; "
            "random versus chronological.",
    },

    "Figure_23_D": {
        "title":
            "SHAP proxy absorption",
        "content":
            "Top-rank changes and normalized importance redistribution "
            "after removal of shortcut-prone information.",
    },

    "supplementary": [
        "matched-size placebo ablations",
        "attack-family-conditioned degradation",
        "complete component-specific SHAP top-20 tables",
        "behavior-restricted operating metrics",
    ],
}


# =============================================================================
# 21. EXPECTED MODEL COUNT / STOPPING RULE
# =============================================================================

EXPECTED_MODEL_COUNT = {
    "FULL_new_ensemble_fits":
        0,

    "FULL_reference":
        "Reuse Stage22R RANDOM_NATURAL and CHRONOLOGICAL_NATURAL.",

    "primary_reduced_subset_split_cells":
        6 * 2,

    "new_primary_boosted_component_fits":
        6 * 2 * 2,

    "placebo_subset_split_cells":
        5 * 2,

    "new_placebo_boosted_component_fits":
        5 * 2 * 2,

    "stump_fits":
        3 * 2,

    "new_boosted_component_fits_total":
        44,

    "new_stump_fits_total":
        6,

    "total_new_model_fits":
        50,

    "shap_model_fits":
        0,

    "attack_family_model_fits":
        0,
}


STOPPING_RULE = {
    "complete_when": [
        "all 7 primary subsets have frozen results under both natural splits",
        "all 6 stump controls are complete",
        "all 5 matched-size placebo subsets have results under both splits",
        "all planned SHAP proxy-absorption analyses are complete",
        "attack-family analysis is complete under frozen support rules",
        "consolidated scientific audit is generated",
        "publication figures and manuscript integration are complete",
    ],

    "no_post_result_adaptation": [
        "no new primary subset",
        "no feature migration between subsets",
        "no change to suspicious group",
        "no behavior-only whitelist change",
        "no placebo membership change",
        "no model hyperparameter change per subset",
        "no split change",
        "no stump specification change",
        "no SHAP cohort change",
        "no uncertainty cohort/method change",
        "no primary metric change",
        "no favorable metric addition after results",
    ],

    "final_holdout":
        "Raw Mar1 and Mar2 permanently forbidden.",

    "rolling_forward_analysis":
        "NOT_INCLUDED_IN_FINAL_STAGE23_PROTOCOL",
}


# =============================================================================
# 22. INHERITED RECEIPTS
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


INHERITED_RECEIPTS = {
    "stage22r_scientific_commit":
        STAGE22R_SCIENTIFIC,

    "stage22r_publication_closeout_commit":
        STAGE22R_CLOSEOUT,

    "stage23_0_parent_commit":
        parent_head,

    "stage22r_1c_feature_order": {
        "path":
            str(FEATURE_ORDER_PATH.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(FEATURE_ORDER_PATH),
    },

    "stage22r_1c_model_input_manifest": {
        "path":
            str(INPUT_MANIFEST_PATH.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(INPUT_MANIFEST_PATH),
    },

    "stage22r_1b1_membership_summary": {
        "path":
            str(MEMBERSHIP_SUMMARY_PATH.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(MEMBERSHIP_SUMMARY_PATH),
    },

    "random_validation_packbits": {
        "path":
            str(RANDOM_VALIDATION_BITSET.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(RANDOM_VALIDATION_BITSET),
    },

    "random_natural_full_result": {
        "path":
            str(RN_RESULT_PATH.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(RN_RESULT_PATH),
    },

    "chronological_natural_full_result": {
        "path":
            str(CN_RESULT_PATH.relative_to(REPO_DIR)),
        "sha256":
            sha256_file(CN_RESULT_PATH),
    },

    "prefreeze_verifications": {
        "repository_provenance":
            "PASSED",

        "eight_cache_sha256":
            "8_OF_8_EXACT",

        "natural_memberships":
            "RANDOM_NATURAL_AND_CHRONOLOGICAL_NATURAL_EXACT",

        "mar1_raw_read":
            False,

        "mar2_raw_read":
            False,
    },
}


# =============================================================================
# 23. SERIALIZABLE PRIMARY SUBSET SPEC
# =============================================================================

FEATURE_SUBSET_SPEC = {
    "primary_subset_count":
        7,

    "full_feature_count":
        len(FULL_FEATURES),

    "full_feature_order":
        FULL_FEATURES,

    "subsets":
        {
            name: {
                **spec,
                "feature_count":
                    len(spec["features"]),
            }
            for name, spec
            in PRIMARY_SUBSETS.items()
        },

    "no_ports_resolution": {
        "src_port_present_in_frozen_70f":
            False,

        "dst_port_present_in_frozen_70f":
            True,

        "protocol_present_in_frozen_70f":
            True,

        "frozen_definition":
            [
                "Dst Port",
                "Protocol",
            ],

        "operational_semantics":
            "transport_identifier_restriction",

        "manuscript_rule":
            "Never state that Stage23 removed Src Port.",
    },
}


# =============================================================================
# 24. TOP-LEVEL MASTER PROTOCOL
# =============================================================================

MASTER_PROTOCOL = {
    "stage":
        "Stage23-0",

    "title":
        "Shortcut-Feature Audit Prospective Protocol Lock",

    "scientific_question":
        "To what extent does IDS performance depend on shortcut-prone "
        "contextual or TCP-stack features, and does that dependence "
        "change between random and chronologically separated evaluation?",

    "status":
        "PROSPECTIVE_CANDIDATE_READY_FOR_GIT_SEAL",

    "constructed_utc":
        datetime.now(timezone.utc).isoformat(),

    "parent_commit":
        parent_head,

    "intended_tag":
        INTENDED_TAG,

    "primary_splits": [
        "RANDOM_NATURAL",
        "CHRONOLOGICAL_NATURAL",
    ],

    "primary_subsets": list(
        PRIMARY_SUBSETS.keys()
    ),

    "primary_metric":
        "PR_AUC",

    "secondary_ranking_metric":
        "ROC_AUC",

    "fixed_operating_threshold":
        0.50,

    "matched_placebo_count":
        5,

    "stump_count":
        6,

    "shap_cohort_rows_per_split":
        5000,

    "uncertainty_cohort_rows_per_split":
        50_000,

    "bootstrap_replicates":
        1000,

    "expected_new_model_fits":
        50,

    "environment":
        actual_packages,

    "raw_mar1_access":
        "PERMANENTLY_FORBIDDEN",

    "raw_mar2_access":
        "PERMANENTLY_FORBIDDEN",

    "post_result_protocol_changes":
        "FORBIDDEN",

    "stage23_final_holdout":
        "NONE",
}


# =============================================================================
# 25. JSON WRITER
# =============================================================================

def write_json(filename, payload):

    path = CANDIDATE_DIR / filename

    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    return path


write_json(
    "stage23_0_protocol_lock.json",
    MASTER_PROTOCOL,
)

write_json(
    "feature_subset_spec.json",
    FEATURE_SUBSET_SPEC,
)

write_json(
    "suspicious_group.json",
    {
        "features": SUSPICIOUS_GROUP,
        "k": len(SUSPICIOUS_GROUP),
        "selection_basis":
            "Prospective semantic shortcut-prone candidates inherited "
            "from pre-Stage23 research design; not selected from Stage23 results.",
    },
)

write_json(
    "behavior_only_features.json",
    {
        "definition":
            "Explicit prospective whitelist",

        "excluded_features":
            BEHAVIOR_EXCLUSIONS,

        "feature_count":
            len(BEHAVIOR_ONLY),

        "features":
            BEHAVIOR_ONLY,

        "interpretation_label":
            "behavior-restricted performance",

        "prohibited_label":
            "true performance",
    },
)

write_json(
    "placebo_ablation_spec.json",
    {
        "matched_k":
            PLACEBO_K,

        "number_of_placebos":
            len(PLACEBO_SUBSETS),

        "selection_method":
            "Five prospectively chosen ordinary behavioral triplets "
            "spanning count, volume/direction, timing, packet-size, "
            "and active/idle families.",

        "seed":
            None,

        "subsets":
            {
                name: {
                    "removed":
                        features,

                    "retained":
                        [
                            f for f in FULL_FEATURES
                            if f not in set(features)
                        ],

                    "feature_count":
                        len(FULL_FEATURES) - len(features),
                }
                for name, features
                in PLACEBO_SUBSETS.items()
            },
    },
)

write_json(
    "inherited_splits.json",
    SPLITS,
)

write_json(
    "model_spec.json",
    MODEL_SPEC,
)

write_json(
    "metric_spec.json",
    METRIC_SPEC,
)

write_json(
    "stump_spec.json",
    STUMP_SPEC,
)

write_json(
    "shap_spec.json",
    SHAP_SPEC,
)

write_json(
    "uncertainty_spec.json",
    UNCERTAINTY_SPEC,
)

write_json(
    "attack_family_spec.json",
    ATTACK_FAMILY_SPEC,
)

write_json(
    "interpretation_matrix.json",
    {
        "rules":
            INTERPRETATION_MATRIX,

        "prohibited_claims":
            PROHIBITED_CLAIMS,
    },
)

write_json(
    "figure_plan.json",
    FIGURE_PLAN,
)

write_json(
    "expected_model_count.json",
    EXPECTED_MODEL_COUNT,
)

write_json(
    "stopping_rule.json",
    STOPPING_RULE,
)

write_json(
    "inherited_stage22_receipts.json",
    INHERITED_RECEIPTS,
)

write_json(
    "freeze_record.json",
    {
        "stage":
            "Stage23-0",

        "status":
            "LOCKED_ON_UNMODIFIED_GIT_COMMIT_AND_TAG",

        "parent_commit":
            parent_head,

        "intended_tag":
            INTENDED_TAG,

        "candidate_directory":
            str(CANDIDATE_DIR),

        "models_fit_during_protocol_construction":
            0,

        "stage23_metrics_calculated":
            0,

        "predictor_values_read":
            0,

        "support_columns_read": [
            "clean_position",
            "day_id",
            "binary_label",
        ],

        "raw_mar1_opened":
            False,

        "raw_mar2_opened":
            False,

        "manual_inspection_required_before_commit":
            True,

        "immutability_rule":
            "Stage23-0B must commit these candidate artifact bytes "
            "without modifying scientific definitions.",
    },
)


# =============================================================================
# 26. README
# =============================================================================

README = CANDIDATE_DIR / "README.md"

README.write_text(
"""# Stage23-0 — Prospective Shortcut-Feature Audit Protocol

## Scientific question

To what extent does IDS performance depend on shortcut-prone contextual or
TCP-stack features, and does that dependence change between random and
chronologically separated evaluation?

## Governance

This protocol was constructed before any Stage23 model was trained and before
any Stage23 performance metric was calculated.

The primary development evaluations are the exact frozen Stage22R
`RANDOM_NATURAL` and `CHRONOLOGICAL_NATURAL` memberships.

Raw March 1 and March 2 access is permanently forbidden.

## Primary subsets

1. `FULL`
2. `NO_DST_PORT`
3. `NO_PORTS`
4. `NO_INIT_FWD_WIN_BYTS`
5. `NO_FWD_SEG_SIZE_MIN`
6. `NO_SUSPICIOUS_GROUP`
7. `BEHAVIOR_ONLY`

### Important NO_PORTS definition

The frozen 70-feature Stage22R feature space contains `Dst Port` and `Protocol`
but does not contain `Src Port`.

Therefore Stage23 prospectively operationalizes `NO_PORTS` as removing:

- `Dst Port`
- `Protocol`

Its semantic interpretation is a transport-identifier restriction. No
Stage23 manuscript text may claim that `Src Port` was removed.

## Primary evidence

- PR-AUC
- ROC-AUC
- difference-of-removal-penalties across random and chronological validation

Fixed threshold 0.50 operating metrics are secondary.

No per-subset threshold optimization is part of the primary Stage23 evidence.

## Behavior-only

The behavior-only feature list is explicitly frozen in
`behavior_only_features.json` before training.

It must be called behavior-restricted performance, not true performance.

## Placebo controls

Five fixed 3-feature behavioral placebo ablations are frozen before training.

## SHAP

A fixed 5,000-row balanced SHAP cohort is frozen separately for each split.
The same row locators are reused for every subset within that split.

## Uncertainty

Full-validation metrics remain the point estimates. Paired stratified
bootstrap uncertainty uses a separately frozen 50,000-row natural-prevalence
cohort per split with 1,000 replicates.

## Anti-adaptation

No subset, suspicious-group membership, behavior-only feature, placebo
membership, primary metric, SHAP cohort, uncertainty method, model
hyperparameter, or split may change after this protocol is sealed.
""",
    encoding="utf-8",
)


# =============================================================================
# 27. HASH ALL CANDIDATE ARTIFACTS
# =============================================================================

candidate_files = sorted(
    p for p in CANDIDATE_DIR.iterdir()
    if p.is_file()
)


checksum_lines = []

for path in candidate_files:

    digest = sha256_file(path)

    checksum_lines.append(
        f"{digest}  {path.name}"
    )


CHECKSUM_PATH = (
    CANDIDATE_DIR
    / "checksums.sha256"
)

CHECKSUM_PATH.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 28. FINAL INVARIANTS
# =============================================================================

assert len(PRIMARY_SUBSETS) == 7

assert PRIMARY_SUBSETS[
    "NO_PORTS"
]["removed"] == [
    "Dst Port",
    "Protocol",
]

assert len(SUSPICIOUS_GROUP) == 3

assert len(BEHAVIOR_ONLY) == 63

assert len(PLACEBO_SUBSETS) == 5

assert all(
    len(features) == 3
    for features in PLACEBO_SUBSETS.values()
)

assert len(SHAP_RANDOM) == 5000
assert len(SHAP_CHRONO) == 5000

assert len(UNC_RANDOM) == 50_000
assert len(UNC_CHRONO) == 50_000

assert EXPECTED_MODEL_COUNT[
    "total_new_model_fits"
] == 50


# =============================================================================
# 29. PRINT COMPLETE PROSPECTIVE DEFINITIONS FOR HUMAN INSPECTION
# =============================================================================

print()
print("=" * 96)
print("PRIMARY SUBSET MATRIX — PROSPECTIVE")
print("=" * 96)

for name, spec in PRIMARY_SUBSETS.items():

    print()
    print(name)
    print("  feature count :", len(spec["features"]))

    if spec["removed"]:
        print(
            "  removed       :",
            ", ".join(spec["removed"])
        )
    else:
        print("  removed       : NONE")


print()
print("=" * 96)
print("BEHAVIOR_ONLY — COMPLETE 63-FEATURE WHITELIST")
print("=" * 96)

for i, feature in enumerate(
    BEHAVIOR_ONLY,
    start=1,
):
    print(
        f"{i:02d}. {feature}"
    )


print()
print("=" * 96)
print("BEHAVIOR_ONLY EXCLUSIONS")
print("=" * 96)

for feature in BEHAVIOR_EXCLUSIONS:
    print(" -", feature)


print()
print("=" * 96)
print("MATCHED-SIZE PLACEBO ABLATIONS")
print("=" * 96)

for name, features in PLACEBO_SUBSETS.items():

    print()
    print(name)

    for feature in features:
        print("  -", feature)


print()
print("=" * 96)
print("INHERITED MODEL RECIPE")
print("=" * 96)

print()
print("Ensemble:")
print(
    "  0.5 * LightGBM + 0.5 * XGBoost"
)

print()
print("LightGBM:")
for key, value in rn_lgbm.items():
    print(f"  {key}: {value}")

print()
print("XGBoost:")
for key, value in rn_xgb.items():
    print(f"  {key}: {value}")


print()
print("=" * 96)
print("FROZEN COHORTS")
print("=" * 96)

print()
print("SHAP RANDOM:")
print("  rows   :", len(SHAP_RANDOM))
print("  benign :", sum(y == 0 for _, y in SHAP_RANDOM))
print("  attack :", sum(y == 1 for _, y in SHAP_RANDOM))

print()
print("SHAP CHRONOLOGICAL:")
print("  rows   :", len(SHAP_CHRONO))
print("  benign :", sum(y == 0 for _, y in SHAP_CHRONO))
print("  attack :", sum(y == 1 for _, y in SHAP_CHRONO))

print()
print("UNCERTAINTY RANDOM:")
for key, value in UNC_RANDOM_COUNTS.items():
    print(f"  {key}: {value}")

print()
print("UNCERTAINTY CHRONOLOGICAL:")
for key, value in UNC_CHRONO_COUNTS.items():
    print(f"  {key}: {value}")


print()
print("=" * 96)
print("EXPECTED COMPUTE")
print("=" * 96)

for key, value in EXPECTED_MODEL_COUNT.items():
    print(f"{key}: {value}")


print()
print("=" * 96)
print("CANDIDATE ARTIFACT HASHES")
print("=" * 96)

print(
    CHECKSUM_PATH.read_text(
        encoding="utf-8"
    )
)


print("=" * 96)
print("STAGE23-0A PROSPECTIVE CANDIDATE CONSTRUCTED")
print("=" * 96)

print()
print("Candidate directory:")
print(" ", CANDIDATE_DIR)

print()
print("Stage23 models fit        : 0")
print("Stage23 metrics calculated: 0")
print("Predictor values read     : 0")
print("Raw Mar1 read             : NO")
print("Raw Mar2 read             : NO")

print()
print("Git commit created : NO")
print("Git tag created    : NO")
print("Stage23-0 sealed   : NO")

print()
print("IMPORTANT:")
print(
    "  Inspect the complete BEHAVIOR_ONLY whitelist, NO_PORTS definition,"
)
print(
    "  placebo groups, model recipe, cohort counts, and hashes above."
)

print()
print("NEXT ONLY AFTER INSPECTION:")
print(
    "  Stage23-0B will copy these exact candidate bytes into the repository,"
)
print(
    "  commit them, create tag stage23-0-protocol-lock-v1, and push."
)

print("=" * 96)

STAGE23-0A — PROSPECTIVE PROTOCOL CONSTRUCTION

Branch     : main
Parent HEAD: 324ee4fc0c0940912bda412d40fb28ea879a5977
Git status : CLEAN
Target tag : stage23-0-protocol-lock-v1


PRIMARY SUBSET MATRIX — PROSPECTIVE

FULL
  feature count : 70
  removed       : NONE

NO_DST_PORT
  feature count : 69
  removed       : Dst Port

NO_PORTS
  feature count : 68
  removed       : Dst Port, Protocol

NO_INIT_FWD_WIN_BYTS
  feature count : 69
  removed       : Init Fwd Win Byts

NO_FWD_SEG_SIZE_MIN
  feature count : 69
  removed       : Fwd Seg Size Min

NO_SUSPICIOUS_GROUP
  feature count : 67
  removed       : Dst Port, Init Fwd Win Byts, Fwd Seg Size Min

BEHAVIOR_ONLY
  feature count : 63
  removed       : Dst Port, Protocol, Fwd Header Len, Bwd Header Len, Init Fwd Win Byts, Init Bwd Win Byts, Fwd Seg Size Min

BEHAVIOR_ONLY — COMPLETE 63-FEATURE WHITELIST
01. Flow Duration
02. Tot Fwd Pkts
03. Tot Bwd Pkts
04. TotLen Fwd Pkts
05. TotLen Bwd Pkts
06. Fwd Pkt Len Max
07. Fwd Pkt Len Min
08

In [3]:
# =============================================================================
# STAGE23-0B — EXACT-BYTE PROTOCOL SEAL
# =============================================================================
#
# PURPOSE:
#   Seal the already-constructed Stage23-0A prospective protocol.
#
# THIS CELL:
#   1. Verifies candidate directory is unchanged
#   2. Verifies every SHA256 from checksums.sha256
#   3. Copies candidate bytes EXACTLY into Git repository
#   4. Verifies source == destination byte-for-byte
#   5. Stages ONLY the Stage23-0 protocol directory
#   6. Commits
#   7. Creates annotated tag:
#
#          stage23-0-protocol-lock-v1
#
#   8. Pushes main + tag
#   9. Verifies remote main and peeled tag both point to the new commit
#
# THIS CELL DOES NOT:
#   - read any dataset
#   - read any predictor
#   - read Mar1
#   - read Mar2
#   - train any model
#   - calculate any metric
#   - modify any Stage23 scientific definition
#
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import shutil
import os


# =============================================================================
# 0. PATHS / EXPECTED STATE
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CANDIDATE_DIR = Path(
    "/kaggle/working/stage23_0_protocol_candidate"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

EXPECTED_PARENT = (
    "324ee4fc0c0940912bda412d40fb28ea879a5977"
)

TAG_NAME = (
    "stage23-0-protocol-lock-v1"
)

COMMIT_MESSAGE = (
    "Stage23-0: freeze shortcut-feature audit protocol"
)

TAG_MESSAGE = (
    "Stage23-0 prospective shortcut-feature audit protocol lock"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=8 * 1024 * 1024):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 2. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-0B — PROTOCOL SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"],
    show=False,
).strip()

head_before = run(
    ["git", "rev-parse", "HEAD"],
    show=False,
).strip()

status_before = run(
    ["git", "status", "--porcelain"],
    show=False,
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head_before != EXPECTED_PARENT:
    raise RuntimeError(
        "\nSTAGE23-0 PARENT COMMIT CHANGED\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head_before}\n\n"
        "Do not seal until the repository state is inspected."
    )


if status_before:
    raise RuntimeError(
        "\nRepository must be clean before Stage23-0 seal:\n"
        + status_before
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        "\nStage23-0 target directory already exists:\n"
        f"{TARGET_DIR}\n\n"
        "Refusing to overwrite an existing protocol lock."
    )


if not CANDIDATE_DIR.exists():
    raise RuntimeError(
        f"Candidate directory missing:\n{CANDIDATE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        TAG_NAME,
    ],
    show=False,
).strip()


if existing_tag:
    raise RuntimeError(
        f"Protocol tag already exists: {TAG_NAME}"
    )


print("[OK] branch        :", branch)
print("[OK] parent commit :", head_before)
print("[OK] worktree      : CLEAN")
print("[OK] tag absent    :", TAG_NAME)


# =============================================================================
# 3. LOAD FROZEN CANDIDATE CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_PATH = (
    CANDIDATE_DIR
    / "checksums.sha256"
)


if not CHECKSUM_PATH.exists():
    raise RuntimeError(
        f"Candidate checksum manifest missing: {CHECKSUM_PATH}"
    )


checksum_entries = {}


for line in CHECKSUM_PATH.read_text(
    encoding="utf-8"
).splitlines():

    line = line.strip()

    if not line:
        continue

    try:
        digest, filename = line.split(
            "  ",
            1,
        )
    except ValueError:
        raise RuntimeError(
            f"Malformed checksum line: {line}"
        )

    if len(digest) != 64:
        raise RuntimeError(
            f"Invalid SHA256 digest for {filename}"
        )

    if filename in checksum_entries:
        raise RuntimeError(
            f"Duplicate checksum entry: {filename}"
        )

    checksum_entries[filename] = digest


if not checksum_entries:
    raise RuntimeError(
        "Candidate checksum manifest is empty."
    )


# =============================================================================
# 4. VERIFY CANDIDATE FILE SET HAS NOT CHANGED
# =============================================================================
#
# checksums.sha256 does not hash itself.
# Every OTHER file must have exactly one entry.
# =============================================================================

candidate_nonmanifest_files = sorted(
    p.name
    for p in CANDIDATE_DIR.iterdir()
    if p.is_file()
    and p.name != "checksums.sha256"
)


manifest_names = sorted(
    checksum_entries.keys()
)


if candidate_nonmanifest_files != manifest_names:
    raise RuntimeError(
        "\nCANDIDATE FILE SET CHANGED SINCE STAGE23-0A\n\n"
        f"Manifest files:\n{manifest_names}\n\n"
        f"Actual files:\n{candidate_nonmanifest_files}"
    )


print()
print("=" * 96)
print("VERIFYING STAGE23-0A CANDIDATE HASHES")
print("=" * 96)
print()


for filename in manifest_names:

    path = (
        CANDIDATE_DIR
        / filename
    )

    expected = checksum_entries[
        filename
    ]

    actual = sha256_file(
        path
    )

    if actual != expected:
        raise RuntimeError(
            "\nSTAGE23-0 CANDIDATE MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected}\n"
            f"actual   : {actual}"
        )

    print(
        f"[OK] {filename}"
    )


candidate_checksum_manifest_sha = (
    sha256_file(
        CHECKSUM_PATH
    )
)


print()
print(
    "[OK] All Stage23-0A candidate "
    "artifact hashes are unchanged."
)

print(
    "Checksum-manifest SHA256:",
    candidate_checksum_manifest_sha,
)


# =============================================================================
# 5. CRITICAL SCIENTIFIC CONTENT ASSERTIONS
# =============================================================================
#
# These assertions do not modify candidate bytes.
# They verify that the candidate being sealed is the one we inspected.
# =============================================================================

import json


feature_spec = json.loads(
    (
        CANDIDATE_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

behavior_spec = json.loads(
    (
        CANDIDATE_DIR
        / "behavior_only_features.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_count_spec = json.loads(
    (
        CANDIDATE_DIR
        / "expected_model_count.json"
    ).read_text(
        encoding="utf-8"
    )
)

master_protocol = json.loads(
    (
        CANDIDATE_DIR
        / "stage23_0_protocol_lock.json"
    ).read_text(
        encoding="utf-8"
    )
)


expected_primary_subsets = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]


if master_protocol["primary_subsets"] != expected_primary_subsets:
    raise RuntimeError(
        "Primary subset matrix changed."
    )


if (
    feature_spec["subsets"]["NO_PORTS"]["removed"]
    != ["Dst Port", "Protocol"]
):
    raise RuntimeError(
        "NO_PORTS definition changed."
    )


if feature_spec[
    "no_ports_resolution"
]["src_port_present_in_frozen_70f"] is not False:
    raise RuntimeError(
        "NO_PORTS Src Port governance changed."
    )


if behavior_spec["feature_count"] != 63:
    raise RuntimeError(
        "BEHAVIOR_ONLY feature count changed."
    )


if len(
    behavior_spec["excluded_features"]
) != 7:
    raise RuntimeError(
        "BEHAVIOR_ONLY exclusions changed."
    )


if model_count_spec[
    "total_new_model_fits"
] != 50:
    raise RuntimeError(
        "Expected Stage23 model count changed."
    )


if master_protocol[
    "raw_mar1_access"
] != "PERMANENTLY_FORBIDDEN":
    raise RuntimeError(
        "Mar1 governance changed."
    )


if master_protocol[
    "raw_mar2_access"
] != "PERMANENTLY_FORBIDDEN":
    raise RuntimeError(
        "Mar2 governance changed."
    )


print()
print("[OK] Scientific content assertions passed.")
print("     primary subsets : 7")
print("     behavior-only   : 63 features")
print("     NO_PORTS        : Dst Port + Protocol")
print("     total new fits  : 50")
print("     Mar1 / Mar2     : permanently forbidden")


# =============================================================================
# 6. COPY EXACT CANDIDATE BYTES INTO REPOSITORY
# =============================================================================

print()
print("=" * 96)
print("COPYING EXACT CANDIDATE BYTES INTO REPOSITORY")
print("=" * 96)
print()


TARGET_DIR.parent.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copytree(
    CANDIDATE_DIR,
    TARGET_DIR,
    copy_function=shutil.copy2,
)


print("Source:")
print(" ", CANDIDATE_DIR)

print()
print("Destination:")
print(" ", TARGET_DIR)


# =============================================================================
# 7. VERIFY SOURCE == DESTINATION EXACTLY
# =============================================================================

source_names = sorted(
    p.name
    for p in CANDIDATE_DIR.iterdir()
    if p.is_file()
)

destination_names = sorted(
    p.name
    for p in TARGET_DIR.iterdir()
    if p.is_file()
)


if source_names != destination_names:
    raise RuntimeError(
        "Destination file set differs from candidate."
    )


print()
print("Verifying destination bytes...")


for filename in source_names:

    src = (
        CANDIDATE_DIR
        / filename
    )

    dst = (
        TARGET_DIR
        / filename
    )

    src_sha = sha256_file(
        src
    )

    dst_sha = sha256_file(
        dst
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            "\nSOURCE / DESTINATION BYTE MISMATCH\n"
            f"file: {filename}\n"
            f"src : {src_sha}\n"
            f"dst : {dst_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


print()
print(
    "[OK] Repository protocol directory "
    "is byte-identical to Stage23-0A candidate."
)


# =============================================================================
# 8. REVERIFY DESTINATION CHECKSUM MANIFEST
# =============================================================================

destination_manifest = (
    TARGET_DIR
    / "checksums.sha256"
)


if (
    sha256_file(destination_manifest)
    != candidate_checksum_manifest_sha
):
    raise RuntimeError(
        "Destination checksum manifest itself changed."
    )


for filename, expected in checksum_entries.items():

    destination_file = (
        TARGET_DIR
        / filename
    )

    actual = sha256_file(
        destination_file
    )

    if actual != expected:
        raise RuntimeError(
            f"Destination checksum mismatch: {filename}"
        )


print()
print("[OK] Destination checksum manifest verified.")


# =============================================================================
# 9. STAGE ONLY STAGE23-0
# =============================================================================

relative_target = (
    TARGET_DIR
    .relative_to(REPO_DIR)
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ],
    show=False,
)


staged_files = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ],
    show=False,
).splitlines()


expected_prefix = (
    str(relative_target)
    + "/"
)


unexpected_staged = [
    path
    for path in staged_files
    if not path.startswith(
        expected_prefix
    )
]


if unexpected_staged:
    raise RuntimeError(
        "\nUNRELATED FILES ARE STAGED:\n"
        + "\n".join(
            unexpected_staged
        )
    )


if len(staged_files) != len(
    destination_names
):
    raise RuntimeError(
        "\nStaged file count mismatch.\n"
        f"expected: {len(destination_names)}\n"
        f"actual  : {len(staged_files)}"
    )


print()
print("=" * 96)
print("STAGED STAGE23-0 FILES")
print("=" * 96)

for path in staged_files:
    print(" ", path)


print()
print("Staged diff stat:")

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ]
)


# =============================================================================
# 10. COMMIT EXACT PROTOCOL LOCK
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-0 SCIENTIFIC PROTOCOL COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    show=False,
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Commit did not advance HEAD."
    )


print()
print("Stage23-0 commit:")
print(" ", sealed_commit)


# =============================================================================
# 11. VERIFY COMMITTED TREE STILL MATCHES CANDIDATE
# =============================================================================
#
# Verify files as present in working tree immediately after commit.
# =============================================================================

for filename in source_names:

    src_sha = sha256_file(
        CANDIDATE_DIR / filename
    )

    committed_worktree_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != committed_worktree_sha:
        raise RuntimeError(
            f"Post-commit byte mismatch: {filename}"
        )


post_commit_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ],
    show=False,
).strip()


if post_commit_status:
    raise RuntimeError(
        "\nRepository not clean immediately after Stage23-0 commit:\n"
        + post_commit_status
    )


print("[OK] Post-commit worktree clean.")


# =============================================================================
# 12. CREATE ANNOTATED SCIENTIFIC TAG
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-0 PROTOCOL TAG")
print("=" * 96)
print()


run(
    [
        "git",
        "tag",
        "-a",
        TAG_NAME,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ]
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        TAG_NAME,
    ],
    show=False,
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "\nLocal Stage23-0 tag does not resolve "
        "to protocol commit.\n"
        f"commit : {sealed_commit}\n"
        f"tag    : {local_tag_commit}"
    )


print("[OK] Local tag:")
print(" ", TAG_NAME)
print(" ->", local_tag_commit)


# =============================================================================
# 13. PUSH MAIN FIRST
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-0 COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ]
)


# =============================================================================
# 14. PUSH PROTOCOL TAG
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-0 TAG")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        TAG_NAME,
    ]
)


# =============================================================================
# 15. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ],
    show=False,
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Could not resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nREMOTE MAIN VERIFICATION FAILED\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 16. VERIFY REMOTE ANNOTATED TAG — PEELED COMMIT
# =============================================================================
#
# For annotated tags:
#
#   refs/tags/TAG       -> tag object
#   refs/tags/TAG^{}    -> underlying commit
#
# We verify the peeled commit.
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{TAG_NAME}",
        f"refs/tags/{TAG_NAME}^{{}}",
    ],
    show=False,
)


remote_tag_lines = (
    remote_tag_output.splitlines()
)


peeled_commit = None
tag_object = None


for line in remote_tag_lines:

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{TAG_NAME}":
        tag_object = sha

    elif ref == f"refs/tags/{TAG_NAME}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote annotated tag object was not found."
    )


if not peeled_commit:
    raise RuntimeError(
        "Remote annotated tag could not be peeled to commit."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nREMOTE TAG VERIFICATION FAILED\n"
        f"expected commit: {sealed_commit}\n"
        f"peeled tag     : {peeled_commit}"
    )


# =============================================================================
# 17. FINAL REPOSITORY CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ],
    show=False,
).strip()


if final_status:
    raise RuntimeError(
        "\nRepository dirty after Stage23-0 push:\n"
        + final_status
    )


# =============================================================================
# 18. FINAL SEAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-0 SCIENTIFIC PROTOCOL SEALED")
print("=" * 96)

print()
print("Parent provenance commit:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-0 protocol commit:")
print(" ", sealed_commit)

print()
print("Scientific protocol tag:")
print(" ", TAG_NAME)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote annotated tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Candidate checksum-manifest SHA256:")
print(" ", candidate_checksum_manifest_sha)

print()
print("Protocol:")
print("  primary subsets              : 7")
print("  behavior-only features       : 63")
print("  behavior exclusions          : 7")
print("  suspicious-group features    : 3")
print("  placebo ablations            : 5")
print("  stump fits                   : 6")
print("  new boosted component fits   : 44")
print("  total new model fits         : 50")
print("  SHAP rows / split            : 5,000")
print("  uncertainty rows / split     : 50,000")
print("  bootstrap replicates         : 1,000")

print()
print("Governance:")
print("  Stage23-0 immutable          : YES")
print("  post-result subset changes   : FORBIDDEN")
print("  post-result metric changes   : FORBIDDEN")
print("  post-result cohort changes   : FORBIDDEN")
print("  per-subset tuning            : FORBIDDEN")
print("  raw Mar1 access              : PERMANENTLY FORBIDDEN")
print("  raw Mar2 access              : PERMANENTLY FORBIDDEN")

print()
print("Scientific execution:")
print("  models fit in Stage23-0      : 0")
print("  Stage23 metrics calculated   : 0")
print("  predictor values read        : 0")

print()
print("Git status:")
print("  CLEAN")

print()
print("=" * 96)
print("STAGE23-0 COMPLETE — STAGE23-1 IS NOW AUTHORIZED")
print("=" * 96)

print()
print("NEXT:")
print(
    "  Stage23-1A — execute the first frozen reduced-feature "
    "RANDOM_NATURAL cell."
)

print("=" * 96)

STAGE23-0B — PROTOCOL SEAL

[OK] branch        : main
[OK] parent commit : 324ee4fc0c0940912bda412d40fb28ea879a5977
[OK] worktree      : CLEAN
[OK] tag absent    : stage23-0-protocol-lock-v1

VERIFYING STAGE23-0A CANDIDATE HASHES

[OK] README.md
[OK] attack_family_spec.json
[OK] behavior_only_features.json
[OK] expected_model_count.json
[OK] feature_subset_spec.json
[OK] figure_plan.json
[OK] freeze_record.json
[OK] inherited_splits.json
[OK] inherited_stage22_receipts.json
[OK] interpretation_matrix.json
[OK] metric_spec.json
[OK] model_spec.json
[OK] placebo_ablation_spec.json
[OK] shap_cohort_chronological_natural.csv
[OK] shap_cohort_random_natural.csv
[OK] shap_spec.json
[OK] stage23_0_protocol_lock.json
[OK] stopping_rule.json
[OK] stump_spec.json
[OK] suspicious_group.json
[OK] uncertainty_cohort_chronological_natural.csv
[OK] uncertainty_cohort_random_natural.csv
[OK] uncertainty_spec.json

[OK] All Stage23-0A candidate artifact hashes are unchanged.
Checksum-manifest SHA256: 3

In [2]:
# =============================================================================
# STAGE23-1A
# NO_DST_PORT × RANDOM_NATURAL
# =============================================================================
#
# AUTHORIZED BY:
#   tag    : stage23-0-protocol-lock-v1
#   commit : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# THIS CELL EXECUTES EXACTLY:
#
#   subset : NO_DST_PORT
#   split  : RANDOM_NATURAL
#
# New completed fits if successful:
#   1 × LightGBM
#   1 × XGBoost
#
# FULL is NOT retrained.
# Frozen Stage22R RANDOM_NATURAL FULL result is reused as reference.
#
# PRIMARY:
#   PR-AUC
#
# SECONDARY:
#   ROC-AUC
#   PR-AUC - attack prevalence
#   fixed threshold 0.50 operating metrics
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP yet
#   placebo yet
#   Mar1 / Mar2 access
#
# IMPORTANT:
#   This execution is fail-closed.
#   If a model fit completes and a later operation fails, DO NOT simply
#   rerun this cell. Send the traceback + execution_state.json state so
#   recovery can resume without duplicating a completed fit.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import csv
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1A"

TARGET_SUBSET = "NO_DST_PORT"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)

STAGE22_FULL_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1a_no_dst_port_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1a_no_dst_port_random_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:
            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1A OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"Directory:\n  {OUTPUT_DIR}\n\n"
        f"Existing state:\n{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly. "
        "A completed model fit may already exist."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {
    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total":
        0,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. GIT / PROTOCOL SEAL VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1A — NO_DST_PORT × RANDOM_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head != PROTOCOL_COMMIT:
    raise RuntimeError(
        "\nSTAGE23-1A MUST BEGIN DIRECTLY FROM THE SEALED "
        "STAGE23-0 COMMIT.\n"
        f"expected: {PROTOCOL_COMMIT}\n"
        f"actual  : {head}"
    )


if tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag no longer resolves to frozen commit."
    )


if status:
    raise RuntimeError(
        "Git worktree must be clean before Stage23-1A:\n"
        + status
    )


print("[OK] branch       :", branch)
print("[OK] HEAD         :", head)
print("[OK] protocol tag :", PROTOCOL_TAG)
print("[OK] worktree     : CLEAN")


# =============================================================================
# 5. VERIFY STAGE23-0 CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_MANIFEST = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if not CHECKSUM_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-0 checksum manifest is missing."
    )


actual_manifest_sha = sha256_file(
    CHECKSUM_MANIFEST
)


if actual_manifest_sha != PROTOCOL_MANIFEST_SHA256:
    raise RuntimeError(
        "\nSTAGE23-0 CHECKSUM MANIFEST CHANGED\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_manifest_sha}"
    )


entries = {}


for line in CHECKSUM_MANIFEST.read_text(
    encoding="utf-8"
).splitlines():

    line = line.strip()

    if not line:
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    entries[filename] = digest


for filename, expected_sha in entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nFROZEN STAGE23-0 ARTIFACT CHANGED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    "[OK] Stage23-0 protocol artifacts:"
    f" {len(entries)}/{len(entries)} exact"
)


# =============================================================================
# 6. LOAD FROZEN SCIENTIFIC SPECIFICATION
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)

metric_spec = json.loads(
    (
        PROTOCOL_DIR
        / "metric_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec["subsets"][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Dst Port"
]:
    raise RuntimeError(
        "Frozen NO_DST_PORT definition changed."
    )


if len(FEATURES) != 69:
    raise RuntimeError(
        f"Expected 69 NO_DST_PORT features; found {len(FEATURES)}"
    )


if "Dst Port" in FEATURES:
    raise RuntimeError(
        "Dst Port remains in NO_DST_PORT feature list."
    )


print()
print("Frozen target:")
print("  split         :", TARGET_SPLIT)
print("  subset        :", TARGET_SUBSET)
print("  feature count :", len(FEATURES))
print("  removed       :", REMOVED_FEATURES)


# =============================================================================
# 7. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[package] = actual

    if actual != expected:
        raise RuntimeError(
            f"Environment mismatch: {package} "
            f"expected={expected} actual={actual}"
        )


print()
print("[OK] Frozen Stage23 package versions verified.")


# =============================================================================
# 8. GPU PREFLIGHT
# =============================================================================

gpu_check = subprocess.run(
    ["nvidia-smi", "-L"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)


if gpu_check.returncode != 0:
    raise RuntimeError(
        "\nXGBoost is frozen to device='cuda', but no NVIDIA GPU "
        "was detected.\n"
        "Enable a Kaggle GPU accelerator before Stage23-1A."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(gpu_check.stdout.strip())


# =============================================================================
# 9. FROZEN RANDOM VALIDATION MEMBERSHIP
# =============================================================================

membership_receipt = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = membership_receipt[
    "train"
]

EXPECTED_VALIDATION = membership_receipt[
    "validation"
]


packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)


N_DEVELOPMENT = (
    EXPECTED_TRAIN["rows"]
    + EXPECTED_VALIDATION["rows"]
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(bool)


if int(
    random_validation_mask.sum()
) != EXPECTED_VALIDATION["rows"]:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL validation population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL frozen membership loaded.")
print(
    "     train rows      :",
    f"{EXPECTED_TRAIN['rows']:,}"
)
print(
    "     validation rows :",
    f"{EXPECTED_VALIDATION['rows']:,}"
)


# =============================================================================
# 10. AUTHORIZED CACHE FILES ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Authorized Stage22R cache missing: {path}"
        )


# =============================================================================
# 11. MEMMAP PLAN
# =============================================================================
#
# Frozen data representation remains float64.
#
# Disk-backed memmaps reduce notebook RAM pressure without changing:
#   - row order
#   - feature order
#   - dtype
#   - model parameters
#   - split membership
#
# =============================================================================

TRAIN_ROWS = int(
    EXPECTED_TRAIN["rows"]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION["rows"]
)

N_FEATURES = len(
    FEATURES
)


matrix_bytes = (
    (TRAIN_ROWS + VALIDATION_ROWS)
    * N_FEATURES
    * np.dtype(np.float64).itemsize
)

working_disk = shutil.disk_usage(
    "/kaggle/working"
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)

print()
print(
    "Feature matrix bytes:",
    f"{matrix_bytes:,}",
    f"({matrix_bytes / 1024**3:.3f} GiB)"
)

print(
    "Free /kaggle/working:",
    f"{working_disk.free / 1024**3:.3f} GiB"
)


# Require matrix + reasonable artifact/headroom.
minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


if working_disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk for frozen float64 "
        "Stage23-1A materialization.\n"
        f"required minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"available       : {working_disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 12. CREATE FROZEN TRAIN / VALIDATION MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)

y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)

X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)

y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)

validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 13. STREAM AUTHORIZED PARQUETS INTO EXACT FROZEN MEMBERSHIPS
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0

materialize_start = time.perf_counter()


print()
print("=" * 96)
print("MATERIALIZING NO_DST_PORT MATRICES")
print("=" * 96)
print()


for day_index, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows_seen = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = batch.num_rows

        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )

        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        # --------------------------------------------------------------
        # Canonical clean_position invariants
        # --------------------------------------------------------------

        expected_global_start = (
            file_rows_seen
            + int(cp[0])
            - file_rows_seen
        )

        if len(cp) == 0:
            continue

        if not np.array_equal(
            cp,
            np.arange(
                int(cp[0]),
                int(cp[0]) + batch_rows,
                dtype=np.int64,
            ),
        ):

            raise RuntimeError(
                f"Non-canonical clean_position sequence in {filename}"
            )


        if int(cp[-1]) >= N_DEVELOPMENT:

            raise RuntimeError(
                f"clean_position exceeds development universe in {filename}"
            )


        is_validation = (
            random_validation_mask[
                cp
            ]
        )

        is_train = (
            ~is_validation
        )


        # --------------------------------------------------------------
        # Predictor matrix in exact frozen subset order
        # --------------------------------------------------------------

        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        n_train = int(
            is_train.sum()
        )

        n_validation = int(
            is_validation.sum()
        )


        # --------------------------------------------------------------
        # Train
        # --------------------------------------------------------------

        if n_train:

            next_train = (
                train_cursor
                + n_train
            )

            X_train[
                train_cursor:next_train
            ] = X_batch[
                is_train
            ]

            y_train[
                train_cursor:next_train
            ] = y[
                is_train
            ]

            train_cursor = next_train


        # --------------------------------------------------------------
        # Validation
        # --------------------------------------------------------------

        if n_validation:

            next_validation = (
                validation_cursor
                + n_validation
            )

            X_validation[
                validation_cursor:next_validation
            ] = X_batch[
                is_validation
            ]

            y_validation[
                validation_cursor:next_validation
            ] = y[
                is_validation
            ]

            validation_positions[
                validation_cursor:next_validation
            ] = cp[
                is_validation
            ]

            validation_cursor = (
                next_validation
            )


        file_rows_seen += (
            batch_rows
        )

        del (
            X_batch,
            cp,
            y,
            is_train,
            is_validation,
        )

        gc.collect()


    print(
        f"[OK] day {day_index:02d} "
        f"{filename}"
    )


if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN MATERIALIZATION COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION MATERIALIZATION COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


# =============================================================================
# 14. VERIFY LABEL COUNTS
# =============================================================================

train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if (
    train_attack
    != EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "RANDOM_NATURAL training attack count mismatch."
    )


if (
    train_benign
    != EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "RANDOM_NATURAL training benign count mismatch."
    )


if (
    validation_attack
    != EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "RANDOM_NATURAL validation attack count mismatch."
    )


if (
    validation_benign
    != EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "RANDOM_NATURAL validation benign count mismatch."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean_position ordering is not strictly increasing."
    )


print()
print("[OK] Frozen matrices materialized.")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)
print(
    "     features   :",
    N_FEATURES
)
print(
    "     dtype      : float64"
)
print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({
    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 15. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


# Hard assertions against accidental mutation.

assert lgbm_params == {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


assert xgb_params == {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


# =============================================================================
# 16. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = time.perf_counter()


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_dst_port_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(LGBM_MODEL_PATH)
)


lgbm_probability = (
    lgbm_model.predict_proba(
        X_validation
    )[:, 1]
)


# Stage22R combination occurs in float64.
lgbm_probability = np.asarray(
    lgbm_probability,
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)

LGBM_PROB_SHA = sha256_file(
    LGBM_PROB_PATH
)


execution_state.update({
    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,

    "lightgbm_probability_sha256":
        LGBM_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


# Release LightGBM model memory before XGBoost.

del lgbm_model
gc.collect()


# =============================================================================
# 17. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = time.perf_counter()


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_dst_port_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = (
    xgb_model.predict_proba(
        X_validation
    )[:, 1]
)


xgb_probability = np.asarray(
    xgb_probability,
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)

XGB_PROB_SHA = sha256_file(
    XGB_PROB_PATH
)


execution_state.update({
    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,

    "xgboost_probability_sha256":
        XGB_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model
gc.collect()


# =============================================================================
# 18. ENSEMBLE PROBABILITY — EXACT FROZEN RULE
# =============================================================================

ensemble_float64 = (
    0.5 * lgbm_probability
    + 0.5 * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite validation ensemble probability produced."
    )


# =============================================================================
# 19. PRIMARY / SECONDARY RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 20. FIXED THRESHOLD 0.50
# =============================================================================

FIXED_THRESHOLD = np.float32(
    0.50
)


predicted = (
    ensemble_probability
    >= FIXED_THRESHOLD
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    )
    .ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 21. FROZEN FULL REFERENCE — NO RETRAIN
# =============================================================================

full_result = json.loads(
    STAGE22_FULL_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_result["cell"] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Unexpected frozen FULL reference cell."
    )


full_pr_auc = float(
    full_result[
        "validation_probability"
    ][
        "pr_auc"
    ]
)

full_roc_auc = float(
    full_result[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_result[
        "operating_points"
    ][
        "standard"
    ]
)


# Frozen removal penalties:
#       FULL - ABLATED

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty_at_050 = float(
    full_standard["f1"]
    - f1
)


recall_removal_penalty_at_050 = float(
    full_standard["recall"]
    - recall
)


fpr_change_full_minus_ablated_at_050 = float(
    full_standard["fpr"]
    - fpr
)


# =============================================================================
# 22. SAVE VALIDATION PROBABILITY ARTIFACT
# =============================================================================
#
# Needed later for:
#   - paired uncertainty
#   - comparison
#   - attack-family analysis
#
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_dst_port_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=y_validation_array,

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = sha256_file(
    VALIDATION_PROBABILITY_PATH
)


# =============================================================================
# 23. RESULT JSON
# =============================================================================

result = {
    "stage":
        STAGE,

    "status":
        "NO_DST_PORT_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "cell": {
        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,
    },

    "data": {
        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {
            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {
            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {
            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions["lightgbm"],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {
            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions["xgboost"],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {
        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {
        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {
        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard["f1"]
            ),

        "recall_at_0_50":
            float(
                full_standard["recall"]
            ),

        "fpr_at_0_50":
            float(
                full_standard["fpr"]
            ),
    },

    "removal_penalty_full_minus_ablated": {
        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty_at_050,

        "recall_at_0_50":
            recall_removal_penalty_at_050,

        "fpr_at_0_50":
            fpr_change_full_minus_ablated_at_050,
    },

    "shortcut_interaction": {
        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "NO_DST_PORT × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {
        "validation_probabilities": {
            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {
            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {
            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {
        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {
        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_total":
            2,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1A result before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 24. ARTIFACT CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        checksum_lines
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 25. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({
    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total":
        2,

    "stage23_metrics_calculated":
        True,

    "result_path":
        str(RESULT_PATH),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 26. REMOVE LARGE TRANSIENT TRAINING MATRICES
# =============================================================================
#
# Models + validation probabilities are retained.
# Training memmaps are purely runtime materialization and must not be committed.
#
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for transient_path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if transient_path.exists():
        transient_path.unlink()


# Remove runtime directory if now empty.

try:
    RUNTIME_DIR.rmdir()
except OSError:
    pass


# =============================================================================
# 27. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1A COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print("  subset :", TARGET_SUBSET)
print("  split  :", TARGET_SPLIT)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "  features   :",
    N_FEATURES
)
print(
    "  removed    :",
    REMOVED_FEATURES
)

print()
print("New model fits:")
print("  LightGBM : 1")
print("  XGBoost  : 1")
print("  TOTAL    : 2")

print()
print("=" * 96)
print("RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence          : {attack_prevalence:.12f}"
)
print(
    f"NO_DST_PORT PR-AUC         : {pr_auc:.12f}"
)
print(
    f"FULL PR-AUC                : {full_pr_auc:.12f}"
)
print(
    f"PR-AUC removal penalty     : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"NO_DST_PORT ROC-AUC        : {roc_auc:.12f}"
)
print(
    f"FULL ROC-AUC               : {full_roc_auc:.12f}"
)
print(
    f"ROC-AUC removal penalty    : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence        : {pr_auc_minus_prevalence:.12f}"
)

print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)

print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)

print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")

print()
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")

print()
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Shortcut interaction   : PENDING")
print(
    "  requires matched NO_DST_PORT × "
    "CHRONOLOGICAL_NATURAL result"
)

print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT run another Stage23 model cell yet."
)
print(
    "First seal and push this Stage23-1A result."
)

print("=" * 96)

STAGE23-1A — NO_DST_PORT × RANDOM_NATURAL

[OK] branch       : main
[OK] HEAD         : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] protocol tag : stage23-0-protocol-lock-v1
[OK] worktree     : CLEAN
[OK] Stage23-0 protocol artifacts: 23/23 exact

Frozen target:
  split         : RANDOM_NATURAL
  subset        : NO_DST_PORT
  feature count : 69
  removed       : ['Dst Port']

[OK] Frozen Stage23 package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] RANDOM_NATURAL frozen membership loaded.
     train rows      : 11,529,922
     validation rows : 2,882,481

MATERIALIZATION PLAN

Feature matrix bytes: 7,955,646,456 (7.409 GiB)
Free /kaggle/working: 18.418 GiB

MATERIALIZING NO_DST_PORT MATRICES

[OK] day 00 day_00_02-14-2018.parquet
[OK] day 01 day_01_02-15-2018.parquet
[OK] day 02 day_02_02-16-2018.parquet
[OK] day 03 day_03_02-20-2018.parquet
[OK] day

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 415.062 s
     model SHA256: b5ad1dd5e225bdbc6ea3cf7d61e6cf2556c5211a93cc354f25d9ddc775ebf870

FIT 2 / 2 — XGBOOST



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [07:51:53] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[OK] XGBoost completed in 105.072 s
     model SHA256: 413e931f0dfcdc77c6ef233ed677bc7d3941b3ac3b32b1927d087e18c897eb70

STAGE23-1A COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_DST_PORT
  split  : RANDOM_NATURAL

Data:
  train      : 11,529,922
  validation : 2,882,481
  features   : 69
  removed    : ['Dst Port']

New model fits:
  LightGBM : 1
  XGBoost  : 1
  TOTAL    : 2

RANKING METRICS

Attack prevalence          : 0.136847389454
NO_DST_PORT PR-AUC         : 0.995260198750
FULL PR-AUC                : 0.995590041899
PR-AUC removal penalty     : +0.000329843149

NO_DST_PORT ROC-AUC        : 0.998473210513
FULL ROC-AUC               : 0.998624564774
ROC-AUC removal penalty    : +0.000151354261

PR-AUC - prevalence        : 0.858412809297

FIXED OPERATING POINT — THRESHOLD 0.50

Accuracy  : 0.995601011767
Precision : 0.999458390242
Recall    : 0.968379556863
F1        : 0.983673553474
FPR       : 0.000083198655
FNR       : 0.031620443137

TN / FP / FN / TP: 2487814 207 12473 3819

In [4]:
# =============================================================================
# STAGE23-1A — SEAL + COMMIT + TAG + PUSH
# NO_DST_PORT × RANDOM_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO METRIC RECOMPUTATION.
# NO DATA ACCESS.
#
# This cell:
#   - verifies the exact Stage23-1A artifact hashes
#   - verifies the result governance
#   - copies permanent artifacts into Git
#   - creates a sealing receipt
#   - commits
#   - creates an annotated result tag
#   - pushes main + tag
#   - verifies remote refs
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1a_no_dst_port_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_dst_port"
)

EXPECTED_PARENT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

RESULT_TAG = (
    "stage23-1a-no-dst-port-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1A: freeze NO_DST_PORT random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1A frozen NO_DST_PORT RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXPECTED ARTIFACT HASHES FROM COMPLETED EXECUTION
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1a_no_dst_port_random_natural_result.json":
        "eabc9fd3ab70990f5cfbca49619ef5e22613345542c4ea16de202c640ec7a6ef",

    "no_dst_port_random_natural_lightgbm_model.txt":
        "b5ad1dd5e225bdbc6ea3cf7d61e6cf2556c5211a93cc354f25d9ddc775ebf870",

    "no_dst_port_random_natural_xgboost_model.json":
        "413e931f0dfcdc77c6ef233ed677bc7d3941b3ac3b32b1927d087e18c897eb70",

    "no_dst_port_random_natural_validation_probabilities.npz":
        "08cc16772c8aca26642a593a3ce15689993fa1f1c147b8ad21be6d2d015653f9",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "2a7b597fb7e7011543e8817468858a06eaa82f1f8c4d97288553cd6f151151a3"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1A — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1A seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-0 protocol tag no longer resolves "
        "to expected protocol commit."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before sealing:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1A target already exists:\n{TARGET_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1A result tag already exists: {RESULT_TAG}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1A source directory missing:\n{SOURCE_DIR}"
    )


print("[OK] branch       :", branch)
print("[OK] parent HEAD  :", head)
print("[OK] protocol tag :", PROTOCOL_TAG)
print("[OK] worktree     : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 4. VERIFY ORIGINAL SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1A checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1A CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 5. VERIFY EVERY CORE ARTIFACT EXACTLY
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1A CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1A artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1A ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 6. VERIFY RESULT SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1a_no_dst_port_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


assert result["stage"] == "Stage23-1A"

assert (
    result["cell"]["subset"]
    == "NO_DST_PORT"
)

assert (
    result["cell"]["split"]
    == "RANDOM_NATURAL"
)

assert (
    result["cell"]["removed_features"]
    == ["Dst Port"]
)

assert result["cell"]["feature_count"] == 69

assert (
    result["protocol"]["commit"]
    == EXPECTED_PARENT
)

assert (
    result["protocol"]["tag"]
    == PROTOCOL_TAG
)

assert (
    result["governance"]["new_model_fits_total"]
    == 2
)

assert (
    result["governance"]["new_lightgbm_fits"]
    == 1
)

assert (
    result["governance"]["new_xgboost_fits"]
    == 1
)

assert (
    result["governance"]["threshold_optimization"]
    is False
)

assert (
    result["governance"]["subset_specific_tuning"]
    is False
)

assert (
    result["governance"]["rebalancing"]
    is False
)

assert (
    result["governance"]["raw_mar1_accessed"]
    is False
)

assert (
    result["governance"]["raw_mar2_accessed"]
    is False
)

assert (
    result["shortcut_interaction"]["status"]
    == "PENDING_MATCHED_CHRONOLOGICAL_CELL"
)


# Exact observed ranking metrics.

# =============================================================================
# EXACT OBSERVED METRIC ASSERTIONS
# =============================================================================

EXPECTED_PR_AUC = 0.995260198750
EXPECTED_ROC_AUC = 0.998473210513

actual_pr_auc = float(
    result["ranking_metrics"]["pr_auc"]
)

actual_roc_auc = float(
    result["ranking_metrics"]["roc_auc"]
)


if abs(
    actual_pr_auc
    - EXPECTED_PR_AUC
) >= 1e-12:

    raise RuntimeError(
        "\nFrozen PR-AUC differs from observed Stage23-1A result.\n"
        f"expected: {EXPECTED_PR_AUC:.12f}\n"
        f"actual  : {actual_pr_auc:.12f}"
    )


if abs(
    actual_roc_auc
    - EXPECTED_ROC_AUC
) >= 1e-12:

    raise RuntimeError(
        "\nFrozen ROC-AUC differs from observed Stage23-1A result.\n"
        f"expected: {EXPECTED_ROC_AUC:.12f}\n"
        f"actual  : {actual_roc_auc:.12f}"
    )


print()
print("[OK] Scientific result assertions passed.")

print(
    "     PR-AUC :",
    f'{result["ranking_metrics"]["pr_auc"]:.12f}'
)

print(
    "     ROC-AUC:",
    f'{result["ranking_metrics"]["roc_auc"]:.12f}'
)

print(
    "     PR penalty:",
    f'{result["removal_penalty_full_minus_ablated"]["pr_auc"]:+.12f}'
)

print(
    "     ROC penalty:",
    f'{result["removal_penalty_full_minus_ablated"]["roc_auc"]:+.12f}'
)

print(
    "     interaction status:",
    result["shortcut_interaction"]["status"]
)


# =============================================================================
# 7. COPY PERMANENT EXECUTION ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 8. VERIFY SOURCE == DESTINATION
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            "\nSOURCE / DESTINATION BYTE MISMATCH\n"
            f"file: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 9. CREATE NON-SCIENTIFIC SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {
    "stage":
        "Stage23-1A",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {
        "subset":
            "NO_DST_PORT",

        "split":
            "RANDOM_NATURAL",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            PROTOCOL_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit":
        2,

    "scientific_interpretation_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "shortcut_interaction_status":
        "NOT_YET_AVAILABLE",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1B NO_DST_PORT × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 10. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1A — NO_DST_PORT × RANDOM_NATURAL

This directory freezes the first reduced-feature Stage23 primary-ablation cell.

## Frozen cell

- Subset: `NO_DST_PORT`
- Split: `RANDOM_NATURAL`
- Removed feature: `Dst Port`
- Retained features: 69
- New boosted-model fits: 2
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {result["ranking_metrics"]["pr_auc"]:.12f}
- ROC-AUC: {result["ranking_metrics"]["roc_auc"]:.12f}
- FULL − NO_DST_PORT PR-AUC penalty: {result["removal_penalty_full_minus_ablated"]["pr_auc"]:+.12f}
- FULL − NO_DST_PORT ROC-AUC penalty: {result["removal_penalty_full_minus_ablated"]["roc_auc"]:+.12f}

## Fixed threshold 0.50

- Accuracy: {result["fixed_threshold_0_50"]["accuracy"]:.12f}
- Precision: {result["fixed_threshold_0_50"]["precision"]:.12f}
- Recall: {result["fixed_threshold_0_50"]["recall"]:.12f}
- F1: {result["fixed_threshold_0_50"]["f1"]:.12f}
- FPR: {result["fixed_threshold_0_50"]["fpr"]:.12f}
- FNR: {result["fixed_threshold_0_50"]["fnr"]:.12f}

## Interpretation status

No split-by-ablation shortcut interaction is interpreted from this result alone.

The matched frozen `NO_DST_PORT × CHRONOLOGICAL_NATURAL` cell must be
completed before the primary interaction can be evaluated.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 11. REPOSITORY-LEVEL CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    p
    for p in TARGET_DIR.iterdir()
    if p.is_file()
    and p.name != "repository_checksums.sha256"
)


checksum_lines = [
    f"{sha256_file(path)}  {path.name}"
    for path in repo_artifacts
]


REPO_CHECKSUMS.write_text(
    "\n".join(
        checksum_lines
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 12. STAGE ONLY STAGE23-1A DIRECTORY
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(prefix)
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(unexpected)
    )


print()
print("=" * 96)
print("STAGED STAGE23-1A ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()
run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 13. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1A RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():

    raise RuntimeError(
        "Repository dirty after Stage23-1A commit."
    )


print()
print("Stage23-1A commit:")
print(" ", sealed_commit)


# =============================================================================
# 14. CREATE ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1A tag verification failed."
    )


# =============================================================================
# 15. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1A")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 16. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 17. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == (
        f"refs/tags/{RESULT_TAG}"
    ):
        tag_object = sha

    elif ref == (
        f"refs/tags/{RESULT_TAG}^{{}}"
    ):
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1A annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote Stage23-1A tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 18. FINAL
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1A push."
    )


print()
print("=" * 96)
print("STAGE23-1A SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Protocol parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1A commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)

print()
print("Frozen result:")
print(
    "  PR-AUC        :",
    f'{result["ranking_metrics"]["pr_auc"]:.12f}'
)
print(
    "  ROC-AUC       :",
    f'{result["ranking_metrics"]["roc_auc"]:.12f}'
)
print(
    "  PR penalty    :",
    f'{result["removal_penalty_full_minus_ablated"]["pr_auc"]:+.12f}'
)
print(
    "  ROC penalty   :",
    f'{result["removal_penalty_full_minus_ablated"]["roc_auc"]:+.12f}'
)

print()
print("Governance:")
print("  new fits sealed        : 2 / 50")
print("  threshold optimization : NO")
print("  subset-specific tuning : NO")
print("  Mar1 raw read          : NO")
print("  Mar2 raw read          : NO")
print("  Stage23-0 changed      : NO")
print("  Git status             : CLEAN")

print()
print("Scientific interpretation:")
print("  shortcut interaction   : PENDING")
print(
    "  matched chronological cell required."
)

print()
print("=" * 96)
print("STAGE23-1A COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1B — NO_DST_PORT × CHRONOLOGICAL_NATURAL"
)

print()
print(
    "Do not execute any other subset before Stage23-1B."
)

print("=" * 96)

STAGE23-1A — SCIENTIFIC RESULT SEAL

[OK] branch       : main
[OK] parent HEAD  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] protocol tag : stage23-0-protocol-lock-v1
[OK] worktree     : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  2a7b597fb7e7011543e8817468858a06eaa82f1f8c4d97288553cd6f151151a3

VERIFYING STAGE23-1A CORE ARTIFACTS

[EXACT] stage23_1a_no_dst_port_random_natural_result.json
[EXACT] no_dst_port_random_natural_lightgbm_model.txt
[EXACT] no_dst_port_random_natural_xgboost_model.json
[EXACT] no_dst_port_random_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC : 0.995260198750
     ROC-AUC: 0.998473210513
     PR penalty: +0.000329843149
     ROC penalty: +0.000151354261
     interaction status: PENDING_MATCHED_CHRONOLOGICAL_CELL

VERIFYING REPOSITORY COPIES

[EXACT] stage23_1a_no_dst_port_random_natural_result.json
[EXACT] no_dst_port_random_natural_lightgbm_model.txt
[EXACT] no_dst_port_random_natural_xg

In [5]:
# =============================================================================
# STAGE23-1B
# NO_DST_PORT × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# AUTHORIZED BY:
#   Stage23-0 protocol tag:
#       stage23-0-protocol-lock-v1
#       -> 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# REQUIRED PREDECESSOR:
#   Stage23-1A result tag:
#       stage23-1a-no-dst-port-random-natural-v1
#       -> b83ff3bd5014a87b256d82507f4dd2475df0b39f
#
# THIS CELL EXECUTES EXACTLY:
#
#   subset : NO_DST_PORT
#   split  : CHRONOLOGICAL_NATURAL
#
# NEW FITS:
#   1 × LightGBM
#   1 × XGBoost
#
# TOTAL STAGE23 FITS AFTER SUCCESS:
#   4 / 50
#
# FULL is NOT retrained.
#
# Frozen Stage22R CHRONOLOGICAL_NATURAL FULL result is reused.
# Frozen Stage23-1A RANDOM_NATURAL result is reused to calculate the
# first split × ablation interaction point estimate.
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   Mar1
#   Mar2
#
# IMPORTANT FAIL-CLOSED RULE:
#
# If either model completes and something later fails, DO NOT blindly rerun
# this cell. Send the traceback and execution_state.json contents.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1B"

TARGET_SUBSET = "NO_DST_PORT"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

STAGE23_1A_COMMIT = (
    "b83ff3bd5014a87b256d82507f4dd2475df0b39f"
)

STAGE23_1A_TAG = (
    "stage23-1a-no-dst-port-random-natural-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

STAGE23_1A_RESULT_SHA256 = (
    "eabc9fd3ab70990f5cfbca49619ef5e22613345542c4ea16de202c640ec7a6ef"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

STAGE23_1A_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_dst_port"
)

STAGE23_1A_RESULT = (
    STAGE23_1A_DIR
    / "stage23_1a_no_dst_port_random_natural_result.json"
)

STAGE22_FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1b_no_dst_port_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1b_no_dst_port_chronological_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1B OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly. "
        "A completed model fit may already exist."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {
    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "required_stage23_1a_commit":
        STAGE23_1A_COMMIT,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. GIT / PREDECESSOR / PROTOCOL VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1B — NO_DST_PORT × CHRONOLOGICAL_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

stage23_1a_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        STAGE23_1A_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != STAGE23_1A_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1B MUST BEGIN FROM THE SEALED STAGE23-1A COMMIT.\n"
        f"expected: {STAGE23_1A_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if stage23_1a_tag_commit != STAGE23_1A_COMMIT:

    raise RuntimeError(
        "Stage23-1A result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1B:\n"
        + status
    )


print("[OK] branch          :", branch)
print("[OK] HEAD            :", head)
print("[OK] Stage23-0 tag   :", protocol_tag_commit)
print("[OK] Stage23-1A tag  :", stage23_1a_tag_commit)
print("[OK] worktree        : CLEAN")


# =============================================================================
# 5. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Frozen Stage23-0 checksum-manifest SHA256 changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen Stage23-0 artifact missing: {filename}"
        )

    actual = sha256_file(
        path
    )

    if actual != expected:

        raise RuntimeError(
            "\nFROZEN STAGE23-0 ARTIFACT CHANGED\n"
            f"file     : {filename}\n"
            f"expected : {expected}\n"
            f"actual   : {actual}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 6. VERIFY SEALED STAGE23-1A RANDOM RESULT
# =============================================================================

if not STAGE23_1A_RESULT.exists():

    raise RuntimeError(
        "Sealed Stage23-1A random result is missing."
    )


actual_stage23_1a_sha = sha256_file(
    STAGE23_1A_RESULT
)


if (
    actual_stage23_1a_sha
    != STAGE23_1A_RESULT_SHA256
):

    raise RuntimeError(
        "\nSEALED STAGE23-1A RESULT CHANGED\n"
        f"expected: {STAGE23_1A_RESULT_SHA256}\n"
        f"actual  : {actual_stage23_1a_sha}"
    )


random_result = json.loads(
    STAGE23_1A_RESULT.read_text(
        encoding="utf-8"
    )
)


if (
    random_result["cell"]["subset"]
    != TARGET_SUBSET
):

    raise RuntimeError(
        "Stage23-1A subset mismatch."
    )


if (
    random_result["cell"]["split"]
    != "RANDOM_NATURAL"
):

    raise RuntimeError(
        "Stage23-1A split mismatch."
    )


print("[OK] Sealed Stage23-1A random result exact.")


# =============================================================================
# 7. LOAD FROZEN SCIENTIFIC SPECIFICATION
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)

metric_spec = json.loads(
    (
        PROTOCOL_DIR
        / "metric_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Dst Port"
]:

    raise RuntimeError(
        "Frozen NO_DST_PORT definition changed."
    )


if len(FEATURES) != 69:

    raise RuntimeError(
        f"Expected 69 features; found {len(FEATURES)}"
    )


if "Dst Port" in FEATURES:

    raise RuntimeError(
        "Dst Port remains in frozen NO_DST_PORT feature list."
    )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  retained      :", len(FEATURES))
print("  removed       :", REMOVED_FEATURES)


# =============================================================================
# 8. PACKAGE ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi not available. "
        "Stage23 XGBoost is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "\nXGBoost is frozen to device='cuda', "
        "but an NVIDIA GPU was not detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(gpu_check.stdout.strip())


# =============================================================================
# 10. FROZEN CHRONOLOGICAL MEMBERSHIP
# =============================================================================

chrono_spec = split_spec[
    TARGET_SPLIT
]


EXPECTED_TRAIN = chrono_spec[
    "train"
]

EXPECTED_VALIDATION = chrono_spec[
    "validation"
]


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)


if TRAIN_ROWS != 13_818_623:

    raise RuntimeError(
        "Frozen chronological train count changed."
    )


if VALIDATION_ROWS != 593_780:

    raise RuntimeError(
        "Frozen chronological validation count changed."
    )


print()
print("[OK] CHRONOLOGICAL_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "     train days : 0..6"
)
print(
    "     val day    : 7 / 02-28-2018"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE FILES
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. FLOAT64 MEMMAP MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


working_disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)
print()

print(
    "Train rows       :",
    f"{TRAIN_ROWS:,}"
)

print(
    "Validation rows  :",
    f"{VALIDATION_ROWS:,}"
)

print(
    "Features         :",
    N_FEATURES
)

print(
    "Matrix storage   :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)

print(
    "Working free     :",
    f"{working_disk.free / 1024**3:.3f} GiB"
)


if working_disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum : {minimum_free / 1024**3:.3f} GiB\n"
        f"free    : {working_disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. STREAM EXACT CHRONOLOGICAL MEMBERSHIPS
# =============================================================================
#
# days 0..6 -> TRAIN
# day 7     -> VALIDATION
#
# No row shuffling.
# No resampling.
#
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "day_id",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING CHRONOLOGICAL NO_DST_PORT MATRICES")
print("=" * 96)
print()


for expected_day_id, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        day_ids = batch.column(
            batch.schema.get_field_index(
                "day_id"
            )
        ).to_numpy(
            zero_copy_only=False
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        # ---------------------------------------------------------------------
        # Day identity
        # ---------------------------------------------------------------------

        unique_days = np.unique(
            day_ids
        )

        if not (
            len(unique_days) == 1
            and int(unique_days[0]) == expected_day_id
        ):

            raise RuntimeError(
                f"day_id mismatch in {filename}: "
                f"{unique_days}"
            )


        # ---------------------------------------------------------------------
        # Canonical clean_position sequence
        # ---------------------------------------------------------------------

        if expected_day_id <= 6:

            expected_cp_start = (
                train_cursor
            )

        else:

            expected_cp_start = (
                TRAIN_ROWS
                + validation_cursor
            )


        if int(cp[0]) != expected_cp_start:

            raise RuntimeError(
                "\nclean_position start mismatch\n"
                f"file     : {filename}\n"
                f"expected : {expected_cp_start:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        expected_cp = np.arange(
            expected_cp_start,
            expected_cp_start
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                f"Non-canonical clean_position sequence in {filename}"
            )


        # ---------------------------------------------------------------------
        # Feature matrix
        # ---------------------------------------------------------------------

        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        # ---------------------------------------------------------------------
        # Chronological assignment
        # ---------------------------------------------------------------------

        if expected_day_id <= 6:

            next_cursor = (
                train_cursor
                + batch_rows
            )

            if next_cursor > TRAIN_ROWS:

                raise RuntimeError(
                    "Chronological training memmap overflow."
                )


            X_train[
                train_cursor:next_cursor
            ] = X_batch

            y_train[
                train_cursor:next_cursor
            ] = y

            train_cursor = (
                next_cursor
            )


        else:

            next_cursor = (
                validation_cursor
                + batch_rows
            )

            if next_cursor > VALIDATION_ROWS:

                raise RuntimeError(
                    "Chronological validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_cursor
            ] = X_batch

            y_validation[
                validation_cursor:next_cursor
            ] = y

            validation_positions[
                validation_cursor:next_cursor
            ] = cp

            validation_cursor = (
                next_cursor
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            day_ids,
            y,
            expected_cp,
        )

        gc.collect()


    print(
        f"[OK] day {expected_day_id:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Chronological training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Chronological training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Chronological validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Chronological validation benign count mismatch."
    )


expected_validation_positions = np.arange(
    TRAIN_ROWS,
    TRAIN_ROWS
    + VALIDATION_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    validation_positions,
    expected_validation_positions,
):

    raise RuntimeError(
        "Chronological validation clean_position membership changed."
    )


print()
print("[OK] Frozen chronological matrices materialized.")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)
print(
    "     features   :",
    N_FEATURES
)
print(
    "     dtype      : float64"
)
print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({
    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


EXPECTED_XGB_PARAMS = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_dst_port_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)

LGBM_PROB_SHA = sha256_file(
    LGBM_PROB_PATH
)


execution_state.update({
    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,

    "lightgbm_probability_sha256":
        LGBM_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_dst_port_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)

XGB_PROB_SHA = sha256_file(
    XGB_PROB_PATH
)


execution_state.update({
    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,

    "xgboost_probability_sha256":
        XGB_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 19. EXACT FROZEN ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 20. PRIMARY RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 21. FIXED THRESHOLD 0.50
# =============================================================================

FIXED_THRESHOLD = np.float32(
    0.50
)


predicted = (
    ensemble_probability
    >= FIXED_THRESHOLD
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[
            0,
            1,
        ],
    )
    .ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)


precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp
    / (
        fp
        + tn
    )
    if (
        fp
        + tn
    )
    else 0.0
)


fnr = float(
    fn
    / (
        fn
        + tp
    )
    if (
        fn
        + tp
    )
    else 0.0
)


# =============================================================================
# 22. FROZEN FULL CHRONOLOGICAL REFERENCE
# =============================================================================

full_chrono = json.loads(
    STAGE22_FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


if (
    full_chrono["cell"]
    != "CHRONOLOGICAL_NATURAL"
):

    raise RuntimeError(
        "Unexpected Stage22R chronological FULL reference."
    )


full_chrono_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_chrono_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_chrono_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 23. CHRONOLOGICAL REMOVAL PENALTIES
# =============================================================================
#
# Frozen definition:
#
#   Δ_C(S) = FULL_C - SUBSET_C
#
# =============================================================================

chrono_pr_penalty = float(
    full_chrono_pr_auc
    - pr_auc
)


chrono_roc_penalty = float(
    full_chrono_roc_auc
    - roc_auc
)


chrono_f1_penalty = float(
    float(
        full_chrono_standard[
            "f1"
        ]
    )
    - f1
)


chrono_recall_penalty = float(
    float(
        full_chrono_standard[
            "recall"
        ]
    )
    - recall
)


chrono_fpr_change = float(
    float(
        full_chrono_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 24. SEALED RANDOM REMOVAL PENALTIES
# =============================================================================

random_pr_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)


random_roc_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


random_f1_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "f1_at_0_50"
    ]
)


random_recall_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "recall_at_0_50"
    ]
)


random_fpr_change = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "fpr_at_0_50"
    ]
)


# =============================================================================
# 25. FIRST STAGE23 SPLIT × ABLATION INTERACTION
# =============================================================================
#
# Frozen definition:
#
#   I(S) = Δ_R(S) - Δ_C(S)
#
# =============================================================================

interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)


interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)


interaction_f1_050 = float(
    random_f1_penalty
    - chrono_f1_penalty
)


interaction_recall_050 = float(
    random_recall_penalty
    - chrono_recall_penalty
)


interaction_fpr_050 = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 26. SAVE VALIDATION PROBABILITY ARTIFACT
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_dst_port_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=y_validation_array,

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = sha256_file(
    VALIDATION_PROBABILITY_PATH
)


# =============================================================================
# 27. RESULT JSON
# =============================================================================

result = {
    "stage":
        STAGE,

    "status":
        "NO_DST_PORT_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {
        "stage":
            "Stage23-1A",

        "commit":
            STAGE23_1A_COMMIT,

        "tag":
            STAGE23_1A_TAG,

        "result_sha256":
            STAGE23_1A_RESULT_SHA256,
    },

    "cell": {
        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,
    },

    "data": {
        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {
            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {
            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {
            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {
            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {
        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {
        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {
        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_chrono_pr_auc,

        "roc_auc":
            full_chrono_roc_auc,

        "f1_at_0_50":
            float(
                full_chrono_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_chrono_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_chrono_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {
        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {
        "source":
            "Sealed Stage23-1A",

        "result_sha256":
            STAGE23_1A_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {
        "definition":
            "I(S) = "
            "(FULL_RANDOM - SUBSET_RANDOM) - "
            "(FULL_CHRONOLOGICAL - SUBSET_CHRONOLOGICAL)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1_050,

        "recall_at_0_50":
            interaction_recall_050,

        "fpr_at_0_50":
            interaction_fpr_050,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "artifacts": {
        "validation_probabilities": {
            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {
            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {
            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {
        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {
        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_after_cell":
            4,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1B before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 28. ARTIFACT CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        checksum_lines
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 29. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({
    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        4,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 30. REMOVE TRANSIENT TRAINING MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


transient_paths = [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]


for transient_path in transient_paths:

    if transient_path.exists():

        transient_path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 31. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1B COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "  features   :",
    N_FEATURES
)
print(
    "  removed    :",
    REMOVED_FEATURES
)

print()
print("New model fits:")
print("  LightGBM : 1")
print("  XGBoost  : 1")
print("  THIS CELL: 2")
print("  STAGE23 TOTAL: 4 / 50")


print()
print("=" * 96)
print("CHRONOLOGICAL RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence          : {attack_prevalence:.12f}"
)

print(
    f"NO_DST_PORT PR-AUC         : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                : {full_chrono_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty   : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"NO_DST_PORT ROC-AUC        : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC               : {full_chrono_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty  : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence        : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("FIRST STAGE23 SPLIT × ABLATION INTERACTION")
print("=" * 96)

print()
print(
    "Definition:"
)
print(
    "  I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)"
)

print()
print("PR-AUC:")
print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")
print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary threshold-0.50 interactions:")
print(
    f"  F1 interaction     : {interaction_f1_050:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall_050:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr_050:+.12f}"
)

print()
print("IMPORTANT:")
print(
    "  Interaction confidence interval: PENDING"
)
print(
    "  Scientific interpretation      : POINT ESTIMATE ONLY"
)


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")

print()
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")

print()
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Stage23 fits completed : 4 / 50")

print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1B."
)

print("=" * 96)

STAGE23-1B — NO_DST_PORT × CHRONOLOGICAL_NATURAL

[OK] branch          : main
[OK] HEAD            : b83ff3bd5014a87b256d82507f4dd2475df0b39f
[OK] Stage23-0 tag   : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1A tag  : b83ff3bd5014a87b256d82507f4dd2475df0b39f
[OK] worktree        : CLEAN
[OK] Stage23-0 artifacts: 23/23 exact
[OK] Sealed Stage23-1A random result exact.

Frozen target:
  subset        : NO_DST_PORT
  split         : CHRONOLOGICAL_NATURAL
  retained      : 69
  removed       : ['Dst Port']

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] CHRONOLOGICAL_NATURAL membership:
     train      : 13,818,623
     validation : 593,780
     train days : 0..6
     val day    : 7 / 02-28-2018

MATERIALIZATION PLAN

Train rows       : 13,818,623
Validation rows  : 593,780
Features         : 69
Matrix storage   : 7.409 GiB
Wo

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 518.124 s
     model SHA256: 6b32caaac0e945f1341f011b8e346791dd3850bf045de7c654f96dbeb273c074

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 146.407 s
     model SHA256: a540c326dfd30e588bc6361030709eb7cc8c86e7933681158d75668a0554b564

STAGE23-1B COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_DST_PORT
  split  : CHRONOLOGICAL_NATURAL

Data:
  train      : 13,818,623
  validation : 593,780
  features   : 69
  removed    : ['Dst Port']

New model fits:
  LightGBM : 1
  XGBoost  : 1
  THIS CELL: 2
  STAGE23 TOTAL: 4 / 50

CHRONOLOGICAL RANKING METRICS

Attack prevalence          : 0.104846912998
NO_DST_PORT PR-AUC         : 0.102473686409
FULL PR-AUC                : 0.106215155134
Chronological PR penalty   : +0.003741468725

NO_DST_PORT ROC-AUC        : 0.488864390761
FULL ROC-AUC               : 0.514918426394
Chronological ROC penalty  : +0.026054035633

PR-AUC - prevalence        : -0.002373226589

FIXED OPERATING POINT — THRESHOLD 0.50

Accuracy  : 0.895

In [6]:
# =============================================================================
# STAGE23-1B — SEAL + COMMIT + TAG + PUSH
# NO_DST_PORT × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO METRIC RECOMPUTATION FROM DATA.
# NO DATASET ACCESS.
#
# Seals the already-completed Stage23-1B artifacts exactly.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1b_no_dst_port_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_dst_port"
)

EXPECTED_PARENT = (
    "b83ff3bd5014a87b256d82507f4dd2475df0b39f"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

STAGE23_1A_TAG = (
    "stage23-1a-no-dst-port-random-natural-v1"
)

RESULT_TAG = (
    "stage23-1b-no-dst-port-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1B: freeze NO_DST_PORT chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1B frozen NO_DST_PORT CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1b_no_dst_port_chronological_natural_result.json":
        "9101d521fa2791eacbb6c6626289aedf1f102e13d146c5da78a3b0a898b4c022",

    "no_dst_port_chronological_natural_lightgbm_model.txt":
        "6b32caaac0e945f1341f011b8e346791dd3850bf045de7c654f96dbeb273c074",

    "no_dst_port_chronological_natural_xgboost_model.json":
        "a540c326dfd30e588bc6361030709eb7cc8c86e7933681158d75668a0554b564",

    "no_dst_port_chronological_natural_validation_probabilities.npz":
        "47e681c10b97879c6f9ef171fb667f753258f717d13cb30e895024b232d6bcd5",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "05b1164a14e3f6248405c35977e1571df9235a34979020e9c60a218e988cd52e"
)


# =============================================================================
# 2. EXACT OBSERVED SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.102473686409
EXPECTED_ROC_AUC = 0.488864390761

EXPECTED_PR_PENALTY = 0.003741468725
EXPECTED_ROC_PENALTY = 0.026054035633

EXPECTED_PR_INTERACTION = -0.003411625576
EXPECTED_ROC_INTERACTION = -0.025902681372


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1B — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

stage23_1a_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        STAGE23_1A_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1B seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )

if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )

if stage23_1a_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1A result tag verification failed."
    )

if status:
    raise RuntimeError(
        "Repository must be clean before sealing:\n"
        + status
    )

if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1B target already exists:\n{TARGET_DIR}"
    )

if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1B source directory missing:\n{SOURCE_DIR}"
    )


existing_result_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_result_tag:
    raise RuntimeError(
        f"Stage23-1B result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1A tag:", stage23_1a_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1B source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1B CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS EXACTLY
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1B CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1B artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1B ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC RESULT CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1b_no_dst_port_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1B":
    raise RuntimeError(
        "Unexpected Stage23-1B stage identifier."
    )

if result["cell"]["subset"] != "NO_DST_PORT":
    raise RuntimeError(
        "Stage23-1B subset changed."
    )

if result["cell"]["split"] != "CHRONOLOGICAL_NATURAL":
    raise RuntimeError(
        "Stage23-1B split changed."
    )

if result["cell"]["removed_features"] != ["Dst Port"]:
    raise RuntimeError(
        "Stage23-1B removed-feature definition changed."
    )

if result["cell"]["feature_count"] != 69:
    raise RuntimeError(
        "Stage23-1B feature count changed."
    )

if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-1B protocol commit mismatch."
    )

if result["required_predecessor"]["commit"] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1B predecessor commit mismatch."
    )

if result["governance"]["new_model_fits_this_cell"] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1B model-fit count."
    )

if result["governance"]["stage23_total_model_fits_after_cell"] != 4:
    raise RuntimeError(
        "Unexpected total Stage23 fit count."
    )

if result["governance"]["threshold_optimization"] is not False:
    raise RuntimeError(
        "Threshold optimization governance changed."
    )

if result["governance"]["subset_specific_tuning"] is not False:
    raise RuntimeError(
        "Subset-specific tuning governance changed."
    )

if result["governance"]["rebalancing"] is not False:
    raise RuntimeError(
        "Rebalancing governance changed."
    )

if result["governance"]["raw_mar1_accessed"] is not False:
    raise RuntimeError(
        "Mar1 governance changed."
    )

if result["governance"]["raw_mar2_accessed"] is not False:
    raise RuntimeError(
        "Mar2 governance changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result["ranking_metrics"]["pr_auc"]
)

actual_roc_auc = float(
    result["ranking_metrics"]["roc_auc"]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["pr_auc"]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["roc_auc"]
)

actual_pr_interaction = float(
    result[
        "shortcut_interaction"
    ]["pr_auc"]
)

actual_roc_interaction = float(
    result[
        "shortcut_interaction"
    ]["roc_auc"]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
    (
        "PR-AUC interaction",
        actual_pr_interaction,
        EXPECTED_PR_INTERACTION,
    ),
    (
        "ROC-AUC interaction",
        actual_roc_interaction,
        EXPECTED_ROC_INTERACTION,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from observed Stage23-1B output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


if (
    result[
        "shortcut_interaction"
    ][
        "confidence_interval_status"
    ]
    != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS"
):
    raise RuntimeError(
        "Stage23 interaction uncertainty governance changed."
    )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC          : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC         : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty      : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty     : {actual_roc_penalty:+.12f}"
)

print(
    f"     PR interaction  : {actual_pr_interaction:+.12f}"
)

print(
    f"     ROC interaction : {actual_roc_interaction:+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 9. COPY PERMANENT EXECUTION ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY SOURCE == DESTINATION
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            "\nSOURCE / DESTINATION BYTE MISMATCH\n"
            f"file: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1B",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {
        "subset":
            "NO_DST_PORT",

        "split":
            "CHRONOLOGICAL_NATURAL",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            STAGE23_1A_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        4,

    "shortcut_interaction": {
        "pr_auc":
            actual_pr_interaction,

        "roc_auc":
            actual_roc_interaction,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1C NO_PORTS × RANDOM_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1B — NO_DST_PORT × CHRONOLOGICAL_NATURAL

This directory freezes the matched chronological partner of Stage23-1A.

## Frozen cell

- Subset: `NO_DST_PORT`
- Split: `CHRONOLOGICAL_NATURAL`
- Removed feature: `Dst Port`
- Retained features: 69
- New boosted-model fits: 2
- Total Stage23 fits after this cell: 4 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Chronological ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − NO_DST_PORT PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − NO_DST_PORT ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Split × ablation interaction

Frozen definition:

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

Point estimates:

- PR-AUC interaction: {actual_pr_interaction:+.12f}
- ROC-AUC interaction: {actual_roc_interaction:+.12f}

These are point estimates only.

The preregistered Stage23 uncertainty analysis must be completed before
inferential interpretation of the interaction.

## Governance

- Raw March 1 access: forbidden
- Raw March 2 access: forbidden
- Stage23-0 unchanged
- No threshold optimization
- No subset-specific tuning
- No rebalancing
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


repo_checksum_lines = [
    f"{sha256_file(path)}  {path.name}"
    for path in repo_artifacts
]


REPO_CHECKSUMS.write_text(
    "\n".join(
        repo_checksum_lines
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1B
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1B ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1B RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


post_commit_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if post_commit_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1B commit."
    )


print()
print("Stage23-1B commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1B tag verification failed."
    )


# =============================================================================
# 17. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1B")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Could not resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == (
        f"refs/tags/{RESULT_TAG}"
    ):
        tag_object = sha

    elif ref == (
        f"refs/tags/{RESULT_TAG}^{{}}"
    ):
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1B annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1B tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1B push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1B SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1A parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1B commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_DST_PORT PAIR")
print("=" * 96)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : "
    f'{result["matched_random_reference"]["pr_auc_removal_penalty"]:+.12f}'
)
print(
    f"  ROC penalty : "
    f'{result["matched_random_reference"]["roc_auc_removal_penalty"]:+.12f}'
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR penalty  : {actual_pr_penalty:+.12f}"
)
print(
    f"  ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {actual_pr_interaction:+.12f}"
)
print(
    f"  ROC-AUC : {actual_roc_interaction:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 4 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1B COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1C — NO_PORTS × RANDOM_NATURAL"
)

print()
print(
    "Do not execute another model cell before this seal succeeds."
)

print("=" * 96)

STAGE23-1B — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : b83ff3bd5014a87b256d82507f4dd2475df0b39f
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1A tag: b83ff3bd5014a87b256d82507f4dd2475df0b39f
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  05b1164a14e3f6248405c35977e1571df9235a34979020e9c60a218e988cd52e

VERIFYING STAGE23-1B CORE ARTIFACTS

[EXACT] stage23_1b_no_dst_port_chronological_natural_result.json
[EXACT] no_dst_port_chronological_natural_lightgbm_model.txt
[EXACT] no_dst_port_chronological_natural_xgboost_model.json
[EXACT] no_dst_port_chronological_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC          : 0.102473686409
     ROC-AUC         : 0.488864390761
     PR penalty      : +0.003741468725
     ROC penalty     : +0.026054035633
     PR interaction  : -0.003411625576
     ROC interaction : -0.025902681372
     CI status       : P

In [7]:
# =============================================================================
# STAGE23-1C
# NO_PORTS × RANDOM_NATURAL
# =============================================================================
#
# FROZEN OPERATIONAL DEFINITION:
#
#   NO_PORTS = remove:
#       - Dst Port
#       - Protocol
#
# IMPORTANT:
#   Src Port does NOT exist in the frozen Stage22R 70-feature model input.
#   Therefore this cell MUST NOT be described as removing Src Port.
#
# Semantic label:
#   transport_identifier_restriction
#
# AUTHORIZED BY:
#   Stage23-0:
#       stage23-0-protocol-lock-v1
#       -> 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# REQUIRED PREDECESSOR:
#   Stage23-1B:
#       stage23-1b-no-dst-port-chronological-natural-v1
#       -> f7aa59008b757e1fc074737a46cbcffbf7ed6f59
#
# THIS CELL EXECUTES EXACTLY:
#
#   subset : NO_PORTS
#   split  : RANDOM_NATURAL
#
# NEW FITS:
#   1 × LightGBM
#   1 × XGBoost
#
# STAGE23 TOTAL AFTER SUCCESS:
#   6 / 50
#
# NO:
#   - threshold optimization
#   - subset-specific tuning
#   - rebalancing
#   - SHAP
#   - placebo
#   - Mar1 access
#   - Mar2 access
#
# FAIL-CLOSED:
#   If either model completes and something later fails, do NOT blindly
#   rerun this cell. Preserve OUTPUT_DIR and send execution_state.json.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1C"

TARGET_SUBSET = "NO_PORTS"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_COMMIT = (
    "f7aa59008b757e1fc074737a46cbcffbf7ed6f59"
)

PREDECESSOR_TAG = (
    "stage23-1b-no-dst-port-chronological-natural-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_dst_port"
    / "seal_receipt.json"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)

STAGE22_FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1c_no_ports_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1c_no_ports_random_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1C OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "predecessor_commit":
        PREDECESSOR_COMMIT,

    "predecessor_tag":
        PREDECESSOR_TAG,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_total_model_fits_before_cell":
        4,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. REPOSITORY / PREDECESSOR VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1C — NO_PORTS × RANDOM_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1C MUST BEGIN FROM SEALED STAGE23-1B.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1B result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1C:\n"
        + status
    )


print("[OK] branch          :", branch)
print("[OK] HEAD            :", head)
print("[OK] Stage23-0 tag   :", protocol_tag_commit)
print("[OK] Stage23-1B tag  :", predecessor_tag_commit)
print("[OK] worktree        : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR GOVERNANCE — 4 FITS SEALED
# =============================================================================

if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1B seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 4:

    raise RuntimeError(
        "Expected exactly 4 sealed Stage23 fits before Stage23-1C."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1C NO_PORTS × RANDOM_NATURAL":

    raise RuntimeError(
        "Stage23-1B seal does not authorize Stage23-1C."
    )


print(
    "[OK] predecessor governance: 4 / 50 fits sealed"
)


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


actual_protocol_manifest_sha = (
    sha256_file(
        PROTOCOL_CHECKSUMS
    )
)


if (
    actual_protocol_manifest_sha
    != PROTOCOL_MANIFEST_SHA256
):

    raise RuntimeError(
        "\nStage23-0 checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_protocol_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SUBSET / MODEL / SPLIT SPEC
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


EXPECTED_REMOVED = [
    "Dst Port",
    "Protocol",
]


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nFrozen NO_PORTS definition changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if len(FEATURES) != 68:

    raise RuntimeError(
        f"Expected 68 NO_PORTS features; found {len(FEATURES)}"
    )


if "Dst Port" in FEATURES:

    raise RuntimeError(
        "Dst Port remains in NO_PORTS."
    )


if "Protocol" in FEATURES:

    raise RuntimeError(
        "Protocol remains in NO_PORTS."
    )


no_ports_resolution = feature_spec[
    "no_ports_resolution"
]


if (
    no_ports_resolution[
        "src_port_present_in_frozen_70f"
    ]
    is not False
):

    raise RuntimeError(
        "Frozen Src Port governance changed."
    )


if (
    no_ports_resolution[
        "frozen_definition"
    ]
    != EXPECTED_REMOVED
):

    raise RuntimeError(
        "Frozen NO_PORTS resolution changed."
    )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  feature count :", len(FEATURES))
print("  removed       :", REMOVED_FEATURES)
print(
    "  semantics     :",
    no_ports_resolution[
        "operational_semantics"
    ]
)
print("  Src Port      : NOT PRESENT IN FROZEN 70F")


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi not available; "
        "XGBoost is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected; "
        "XGBoost Stage23 backend is frozen to CUDA."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(gpu_check.stdout.strip())


# =============================================================================
# 10. FROZEN RANDOM MEMBERSHIP
# =============================================================================

random_spec = split_spec[
    TARGET_SPLIT
]


EXPECTED_TRAIN = random_spec[
    "train"
]

EXPECTED_VALIDATION = random_spec[
    "validation"
]


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)

N_DEVELOPMENT = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)


if TRAIN_ROWS != 11_529_922:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL training count changed."
    )


if VALIDATION_ROWS != 2_882_481:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL validation count changed."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(bool)


if len(random_validation_mask) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random validation mask length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation membership population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. FLOAT64 MEMMAP MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


working_disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)
print()

print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)

print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)

print(
    "Features        :",
    N_FEATURES
)

print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)

print(
    "Working free    :",
    f"{working_disk.free / 1024**3:.3f} GiB"
)


if working_disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"free   : {working_disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. STREAM EXACT RANDOM MEMBERSHIP
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0
expected_next_clean_position = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING RANDOM NO_PORTS MATRICES")
print("=" * 96)
print()


for day_index, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        # ---------------------------------------------------------------------
        # Canonical development order
        # ---------------------------------------------------------------------

        expected_cp = np.arange(
            expected_next_clean_position,
            expected_next_clean_position
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file       : {filename}\n"
                f"expected   : {expected_next_clean_position:,}\n"
                f"actual first: {int(cp[0]):,}"
            )


        expected_next_clean_position += (
            batch_rows
        )


        # ---------------------------------------------------------------------
        # Frozen RANDOM validation membership
        # ---------------------------------------------------------------------

        is_validation = (
            random_validation_mask[
                cp
            ]
        )

        is_train = (
            ~is_validation
        )


        # ---------------------------------------------------------------------
        # Feature matrix — exact frozen 68-feature order
        # ---------------------------------------------------------------------

        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        n_train = int(
            is_train.sum()
        )

        n_validation = int(
            is_validation.sum()
        )


        # ---------------------------------------------------------------------
        # TRAIN
        # ---------------------------------------------------------------------

        if n_train:

            next_train = (
                train_cursor
                + n_train
            )

            if next_train > TRAIN_ROWS:

                raise RuntimeError(
                    "Random training memmap overflow."
                )


            X_train[
                train_cursor:next_train
            ] = X_batch[
                is_train
            ]

            y_train[
                train_cursor:next_train
            ] = y[
                is_train
            ]

            train_cursor = (
                next_train
            )


        # ---------------------------------------------------------------------
        # VALIDATION
        # ---------------------------------------------------------------------

        if n_validation:

            next_validation = (
                validation_cursor
                + n_validation
            )

            if next_validation > VALIDATION_ROWS:

                raise RuntimeError(
                    "Random validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_validation
            ] = X_batch[
                is_validation
            ]

            y_validation[
                validation_cursor:next_validation
            ] = y[
                is_validation
            ]

            validation_positions[
                validation_cursor:next_validation
            ] = cp[
                is_validation
            ]

            validation_cursor = (
                next_validation
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            y,
            expected_cp,
            is_train,
            is_validation,
        )

        gc.collect()


    print(
        f"[OK] day {day_index:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if (
    expected_next_clean_position
    != N_DEVELOPMENT
):

    raise RuntimeError(
        "\nDevelopment canonical-length mismatch.\n"
        f"expected: {N_DEVELOPMENT:,}\n"
        f"actual  : {expected_next_clean_position:,}"
    )


if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Random training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Random training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Random validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Random validation benign count mismatch."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean_position order is not strictly increasing."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):

    raise RuntimeError(
        "Materialized validation positions disagree with frozen bitset."
    )


print()
print("[OK] Frozen RANDOM_NATURAL matrices materialized.")

print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)

print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)

print(
    "     features   :",
    N_FEATURES
)

print(
    "     dtype      : float64"
)

print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_ports_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)

LGBM_PROB_SHA = sha256_file(
    LGBM_PROB_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,

    "lightgbm_probability_sha256":
        LGBM_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_ports_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)

XGB_PROB_SHA = sha256_file(
    XGB_PROB_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,

    "xgboost_probability_sha256":
        XGB_PROB_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 19. EXACT FROZEN ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probability produced."
    )


# =============================================================================
# 20. RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 21. FIXED THRESHOLD 0.50
# =============================================================================

FIXED_THRESHOLD = np.float32(
    0.50
)


predicted = (
    ensemble_probability
    >= FIXED_THRESHOLD
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[
            0,
            1,
        ],
    )
    .ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)


precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 22. FROZEN FULL RANDOM REFERENCE
# =============================================================================

full_random = json.loads(
    STAGE22_FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if (
    full_random[
        "cell"
    ]
    != "RANDOM_NATURAL"
):

    raise RuntimeError(
        "Unexpected Stage22R FULL random reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 23. RANDOM REMOVAL PENALTIES
# =============================================================================
#
# Δ_R(S) = FULL_R - SUBSET_R
#
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty_at_050 = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


recall_removal_penalty_at_050 = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


fpr_change_full_minus_ablated_at_050 = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 24. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_ports_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 25. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_PORTS_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {

        "stage":
            "Stage23-1B",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            no_ports_resolution[
                "operational_semantics"
            ],

        "src_port_present_in_frozen_70f":
            False,

        "manuscript_rule":
            no_ports_resolution[
                "manuscript_rule"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty_at_050,

        "recall_at_0_50":
            recall_removal_penalty_at_050,

        "fpr_at_0_50":
            fpr_change_full_minus_ablated_at_050,
    },

    "shortcut_interaction": {

        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "NO_PORTS × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            4,

        "stage23_total_model_fits_after_cell":
            6,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1C before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 26. ARTIFACT CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        checksum_lines
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = (
    sha256_file(
        CHECKSUM_OUTPUT_PATH
    )
)


# =============================================================================
# 27. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        6,

    "stage23_metrics_calculated":
        True,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 28. REMOVE TRANSIENT MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for transient_path in [

    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if transient_path.exists():

        transient_path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 29. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1C COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen NO_PORTS definition:")
print("  removed : Dst Port, Protocol")
print("  Src Port: NOT PRESENT IN FROZEN 70F")
print(
    "  semantics:",
    no_ports_resolution[
        "operational_semantics"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "  features   :",
    N_FEATURES
)

print()
print("New model fits:")
print("  LightGBM : 1")
print("  XGBoost  : 1")
print("  THIS CELL: 2")
print("  STAGE23 TOTAL: 6 / 50")


print()
print("=" * 96)
print("RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence      : {attack_prevalence:.12f}"
)

print(
    f"NO_PORTS PR-AUC        : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC            : {full_pr_auc:.12f}"
)

print(
    f"PR-AUC removal penalty : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"NO_PORTS ROC-AUC       : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC           : {full_roc_auc:.12f}"
)

print(
    f"ROC removal penalty    : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence    : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")

print()
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")

print()
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Stage23 fits completed : 6 / 50")

print()
print("Shortcut interaction:")
print("  PENDING matched NO_PORTS chronological cell")


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1C."
)

print("=" * 96)

STAGE23-1C — NO_PORTS × RANDOM_NATURAL

[OK] branch          : main
[OK] HEAD            : f7aa59008b757e1fc074737a46cbcffbf7ed6f59
[OK] Stage23-0 tag   : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1B tag  : f7aa59008b757e1fc074737a46cbcffbf7ed6f59
[OK] worktree        : CLEAN
[OK] predecessor governance: 4 / 50 fits sealed
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_PORTS
  split         : RANDOM_NATURAL
  feature count : 68
  removed       : ['Dst Port', 'Protocol']
  semantics     : transport_identifier_restriction
  Src Port      : NOT PRESENT IN FROZEN 70F

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] RANDOM_NATURAL membership:
     train      : 11,529,922
     validation : 2,882,481

MATERIALIZATION PLAN

Train rows      : 11,529,922
Validation rows : 2,882,481
Features        : 68
Ma

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 389.965 s
     model SHA256: 934a80bcca2f5691742dd584875a57fd2a9f8e4237c220361301b060286aa1c6

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 108.828 s
     model SHA256: d6d4a45de5baa53a7ce94c067827c3f8df9483ed13daff502bc5e443e26b4f2f

STAGE23-1C COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_PORTS
  split  : RANDOM_NATURAL

Frozen NO_PORTS definition:
  removed : Dst Port, Protocol
  Src Port: NOT PRESENT IN FROZEN 70F
  semantics: transport_identifier_restriction

Data:
  train      : 11,529,922
  validation : 2,882,481
  features   : 68

New model fits:
  LightGBM : 1
  XGBoost  : 1
  THIS CELL: 2
  STAGE23 TOTAL: 6 / 50

RANKING METRICS

Attack prevalence      : 0.136847389454
NO_PORTS PR-AUC        : 0.995267437645
FULL PR-AUC            : 0.995590041899
PR-AUC removal penalty : +0.000322604254

NO_PORTS ROC-AUC       : 0.998475655106
FULL ROC-AUC           : 0.998624564774
ROC removal penalty    : +0.000148909668

PR-AUC - prevalence    : 0.8584200481

In [8]:
# =============================================================================
# STAGE23-1C — SEAL + COMMIT + TAG + PUSH
# NO_PORTS × RANDOM_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# Frozen NO_PORTS definition:
#   remove Dst Port + Protocol
#
# Src Port is NOT present in the frozen Stage22R 70-feature space.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/stage23_1c_no_ports_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_ports"
)

EXPECTED_PARENT = (
    "f7aa59008b757e1fc074737a46cbcffbf7ed6f59"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1b-no-dst-port-chronological-natural-v1"
)

RESULT_TAG = (
    "stage23-1c-no-ports-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1C: freeze NO_PORTS random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1C frozen NO_PORTS RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1c_no_ports_random_natural_result.json":
        "e36087d6387b936712ce302a56f6aa10fee6675d3b314f2e9ecc512b1b0e48d4",

    "no_ports_random_natural_lightgbm_model.txt":
        "934a80bcca2f5691742dd584875a57fd2a9f8e4237c220361301b060286aa1c6",

    "no_ports_random_natural_xgboost_model.json":
        "d6d4a45de5baa53a7ce94c067827c3f8df9483ed13daff502bc5e443e26b4f2f",

    "no_ports_random_natural_validation_probabilities.npz":
        "e518c928281e5c123c73467bca02b7c4317bcb3279d4fd8c21561a5f532e346e",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "88a2547b2cd223526c3a86db75847afb855572ffa6e9a9a7fd119aed3f2c6a76"
)


# =============================================================================
# 2. EXACT OBSERVED VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.995267437645
EXPECTED_ROC_AUC = 0.998475655106

EXPECTED_PR_PENALTY = 0.000322604254
EXPECTED_ROC_PENALTY = 0.000148909668


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1C — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1C seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1B predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1C seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1C target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1C source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1C result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1B tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1C source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1C CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1C CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1C artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1C ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1c_no_ports_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1C":
    raise RuntimeError(
        "Unexpected stage identifier."
    )


if result["cell"]["subset"] != "NO_PORTS":
    raise RuntimeError(
        "Stage23-1C subset changed."
    )


if result["cell"]["split"] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Stage23-1C split changed."
    )


if result["cell"]["removed_features"] != [
    "Dst Port",
    "Protocol",
]:
    raise RuntimeError(
        "Frozen NO_PORTS removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 68:
    raise RuntimeError(
        "Frozen NO_PORTS feature count changed."
    )


if result[
    "cell"
][
    "src_port_present_in_frozen_70f"
] is not False:

    raise RuntimeError(
        "Src Port governance changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "transport_identifier_restriction":

    raise RuntimeError(
        "NO_PORTS operational semantics changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result["required_predecessor"]["commit"] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1C predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance["new_model_fits_this_cell"] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1C model-fit count."
    )


if governance["stage23_total_model_fits_before_cell"] != 4:
    raise RuntimeError(
        "Unexpected pre-cell Stage23 fit count."
    )


if governance["stage23_total_model_fits_after_cell"] != 6:
    raise RuntimeError(
        "Unexpected post-cell Stage23 fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


if result[
    "shortcut_interaction"
][
    "status"
] != "PENDING_MATCHED_CHRONOLOGICAL_CELL":

    raise RuntimeError(
        "Stage23-1C interaction status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result[
        "ranking_metrics"
    ][
        "pr_auc"
    ]
)

actual_roc_auc = float(
    result[
        "ranking_metrics"
    ][
        "roc_auc"
    ]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from observed Stage23-1C output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty : {actual_roc_penalty:+.12f}"
)

print(
    "     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    source_sha = sha256_file(
        SOURCE_DIR / filename
    )

    target_sha = sha256_file(
        TARGET_DIR / filename
    )

    if source_sha != target_sha:
        raise RuntimeError(
            f"Source/destination mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1C",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_PORTS",

        "split":
            "RANDOM_NATURAL",

        "removed_features": [
            "Dst Port",
            "Protocol",
        ],

        "feature_count":
            68,

        "operational_semantics":
            "transport_identifier_restriction",

        "src_port_present_in_frozen_70f":
            False,
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        6,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1D NO_PORTS × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1C — NO_PORTS × RANDOM_NATURAL

This directory freezes the random-split `NO_PORTS` primary-ablation cell.

## Frozen operational definition

`NO_PORTS` removes:

- `Dst Port`
- `Protocol`

`Src Port` is not present in the frozen Stage22R 70-feature model input.

The operational semantic label is:

`transport_identifier_restriction`

## Frozen cell

- Split: `RANDOM_NATURAL`
- Retained features: 68
- New boosted-model fits: 2
- Total Stage23 fits after cell: 6 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − NO_PORTS PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − NO_PORTS ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Interpretation status

The split × ablation interaction is not yet available.

The matched frozen `NO_PORTS × CHRONOLOGICAL_NATURAL` cell must be
completed before the interaction point estimate is calculated.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1C
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1C ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1C RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():

    raise RuntimeError(
        "Repository dirty after Stage23-1C commit."
    )


print()
print("Stage23-1C commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1C tag verification failed."
    )


# =============================================================================
# 17. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1C")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == (
        f"refs/tags/{RESULT_TAG}"
    ):
        tag_object = sha

    elif ref == (
        f"refs/tags/{RESULT_TAG}^{{}}"
    ):
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1C tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1C tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1C push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1C SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1B parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1C commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_PORTS RANDOM RESULT")
print("=" * 96)

print()
print("Operational definition:")
print("  remove   : Dst Port + Protocol")
print("  Src Port : NOT PRESENT")
print("  retained : 68 features")

print()
print(
    f"PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("Shortcut interaction:")
print("  PENDING matched chronological cell")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 6 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1C COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1D — NO_PORTS × CHRONOLOGICAL_NATURAL"
)

print()
print(
    "Do not execute another model cell before this seal succeeds."
)

print("=" * 96)

STAGE23-1C — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : f7aa59008b757e1fc074737a46cbcffbf7ed6f59
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1B tag: f7aa59008b757e1fc074737a46cbcffbf7ed6f59
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  88a2547b2cd223526c3a86db75847afb855572ffa6e9a9a7fd119aed3f2c6a76

VERIFYING STAGE23-1C CORE ARTIFACTS

[EXACT] stage23_1c_no_ports_random_natural_result.json
[EXACT] no_ports_random_natural_lightgbm_model.txt
[EXACT] no_ports_random_natural_xgboost_model.json
[EXACT] no_ports_random_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC      : 0.995267437645
     ROC-AUC     : 0.998475655106
     PR penalty  : +0.000322604254
     ROC penalty : +0.000148909668
     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL

VERIFYING REPOSITORY COPIES

[EXACT] stage23_1c_no_ports_random_natural_result.json
[EXACT] no_ports_ran

In [9]:
# =============================================================================
# STAGE23-1D
# NO_PORTS × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# FROZEN NO_PORTS DEFINITION:
#   remove:
#       - Dst Port
#       - Protocol
#
# Src Port does NOT exist in the frozen Stage22R 70-feature input.
#
# AUTHORIZED BY:
#   Stage23-0:
#       stage23-0-protocol-lock-v1
#       -> 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# REQUIRED PREDECESSOR:
#   Stage23-1C:
#       stage23-1c-no-ports-random-natural-v1
#       -> ff853952366f731449579f64da44572b3a5a725e
#
# THIS CELL:
#   - fits 1 LightGBM
#   - fits 1 XGBoost
#   - evaluates CHRONOLOGICAL_NATURAL
#   - reuses frozen FULL chronological result
#   - reuses sealed Stage23-1C random result
#   - calculates NO_PORTS split × ablation point estimate
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   Mar1 / Mar2
#
# FAIL CLOSED:
#   If a model completes and a later step fails, do NOT blindly rerun.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1D"

TARGET_SUBSET = "NO_PORTS"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

STAGE23_1C_COMMIT = (
    "ff853952366f731449579f64da44572b3a5a725e"
)

STAGE23_1C_TAG = (
    "stage23-1c-no-ports-random-natural-v1"
)

STAGE23_1C_RESULT_SHA256 = (
    "e36087d6387b936712ce302a56f6aa10fee6675d3b314f2e9ecc512b1b0e48d4"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

STAGE23_1C_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_ports"
)

STAGE23_1C_RESULT = (
    STAGE23_1C_DIR
    / "stage23_1c_no_ports_random_natural_result.json"
)

STAGE23_1C_SEAL = (
    STAGE23_1C_DIR
    / "seal_receipt.json"
)

STAGE22_FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1d_no_ports_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1d_no_ports_chronological_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:
            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            existing_state = {
                "status": "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1D OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "required_stage23_1c_commit":
        STAGE23_1C_COMMIT,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_total_model_fits_before_cell":
        6,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. GIT / PREDECESSOR VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1D — NO_PORTS × CHRONOLOGICAL_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

stage23_1c_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        STAGE23_1C_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != STAGE23_1C_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1D MUST BEGIN FROM SEALED STAGE23-1C.\n"
        f"expected: {STAGE23_1C_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if stage23_1c_tag_commit != STAGE23_1C_COMMIT:

    raise RuntimeError(
        "Stage23-1C result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1D:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1C tag :", stage23_1c_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR SEAL
# =============================================================================

if not STAGE23_1C_SEAL.exists():

    raise RuntimeError(
        "Stage23-1C seal receipt missing."
    )


stage23_1c_seal = json.loads(
    STAGE23_1C_SEAL.read_text(
        encoding="utf-8"
    )
)


if stage23_1c_seal[
    "stage23_models_fit_total"
] != 6:

    raise RuntimeError(
        "Expected 6 sealed Stage23 fits before Stage23-1D."
    )


if stage23_1c_seal[
    "next_authorized_model_cell"
] != "Stage23-1D NO_PORTS × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1C seal does not authorize Stage23-1D."
    )


print(
    "[OK] predecessor governance: 6 / 50 fits sealed"
)


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum manifest changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )


    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. VERIFY SEALED RANDOM NO_PORTS RESULT
# =============================================================================

if not STAGE23_1C_RESULT.exists():

    raise RuntimeError(
        "Sealed Stage23-1C random result missing."
    )


actual_stage23_1c_sha = (
    sha256_file(
        STAGE23_1C_RESULT
    )
)


if (
    actual_stage23_1c_sha
    != STAGE23_1C_RESULT_SHA256
):

    raise RuntimeError(
        "\nSEALED STAGE23-1C RESULT CHANGED\n"
        f"expected: {STAGE23_1C_RESULT_SHA256}\n"
        f"actual  : {actual_stage23_1c_sha}"
    )


random_result = json.loads(
    STAGE23_1C_RESULT.read_text(
        encoding="utf-8"
    )
)


if random_result[
    "cell"
][
    "subset"
] != TARGET_SUBSET:

    raise RuntimeError(
        "Stage23-1C subset mismatch."
    )


if random_result[
    "cell"
][
    "split"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Stage23-1C split mismatch."
    )


print(
    "[OK] Sealed Stage23-1C random result exact."
)


# =============================================================================
# 8. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Dst Port",
    "Protocol",
]:

    raise RuntimeError(
        "Frozen NO_PORTS definition changed."
    )


if len(FEATURES) != 68:

    raise RuntimeError(
        f"Expected 68 features; found {len(FEATURES)}"
    )


if (
    "Dst Port" in FEATURES
    or "Protocol" in FEATURES
):

    raise RuntimeError(
        "Removed transport identifiers remain in NO_PORTS."
    )


no_ports_resolution = (
    feature_spec[
        "no_ports_resolution"
    ]
)


if no_ports_resolution[
    "src_port_present_in_frozen_70f"
] is not False:

    raise RuntimeError(
        "Src Port governance changed."
    )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  feature count :", len(FEATURES))
print("  removed       :", REMOVED_FEATURES)
print(
    "  semantics     :",
    no_ports_resolution[
        "operational_semantics"
    ]
)
print("  Src Port      : NOT PRESENT")


# =============================================================================
# 9. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 10. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        ["nvidia-smi", "-L"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; "
        "XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 11. FROZEN CHRONOLOGICAL MEMBERSHIP
# =============================================================================

chrono_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = chrono_spec[
    "train"
]

EXPECTED_VALIDATION = chrono_spec[
    "validation"
]


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)


if TRAIN_ROWS != 13_818_623:

    raise RuntimeError(
        "Chronological train count changed."
    )


if VALIDATION_ROWS != 593_780:

    raise RuntimeError(
        "Chronological validation count changed."
    )


print()
print("[OK] CHRONOLOGICAL_NATURAL:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "     train days : 0..6"
)
print(
    "     val day    : 7 / 02-28-2018"
)


# =============================================================================
# 12. AUTHORIZED CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 13. MEMMAP PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)

print()
print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)

print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)

print(
    "Features        :",
    N_FEATURES
)

print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)

print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"free   : {disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 14. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 15. MATERIALIZE CHRONOLOGICAL MATRICES
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "day_id",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING CHRONOLOGICAL NO_PORTS MATRICES")
print("=" * 96)
print()


for expected_day_id, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        day_ids = batch.column(
            batch.schema.get_field_index(
                "day_id"
            )
        ).to_numpy(
            zero_copy_only=False
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        unique_days = np.unique(
            day_ids
        )


        if not (
            len(unique_days) == 1
            and int(unique_days[0]) == expected_day_id
        ):

            raise RuntimeError(
                f"day_id mismatch in {filename}: {unique_days}"
            )


        if expected_day_id <= 6:

            expected_cp_start = (
                train_cursor
            )

        else:

            expected_cp_start = (
                TRAIN_ROWS
                + validation_cursor
            )


        expected_cp = np.arange(
            expected_cp_start,
            expected_cp_start
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file     : {filename}\n"
                f"expected : {expected_cp_start:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        if expected_day_id <= 6:

            next_cursor = (
                train_cursor
                + batch_rows
            )

            if next_cursor > TRAIN_ROWS:

                raise RuntimeError(
                    "Training memmap overflow."
                )


            X_train[
                train_cursor:next_cursor
            ] = X_batch

            y_train[
                train_cursor:next_cursor
            ] = y

            train_cursor = next_cursor


        else:

            next_cursor = (
                validation_cursor
                + batch_rows
            )

            if next_cursor > VALIDATION_ROWS:

                raise RuntimeError(
                    "Validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_cursor
            ] = X_batch

            y_validation[
                validation_cursor:next_cursor
            ] = y

            validation_positions[
                validation_cursor:next_cursor
            ] = cp

            validation_cursor = (
                next_cursor
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            day_ids,
            y,
            expected_cp,
        )

        gc.collect()


    print(
        f"[OK] day {expected_day_id:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 16. MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        f"Train row mismatch: {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        f"Validation row mismatch: {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Validation benign count mismatch."
    )


expected_validation_positions = np.arange(
    TRAIN_ROWS,
    TRAIN_ROWS
    + VALIDATION_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    validation_positions,
    expected_validation_positions,
):

    raise RuntimeError(
        "Chronological validation membership changed."
    )


print()
print("[OK] Frozen matrices materialized.")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)
print(
    "     features   :",
    N_FEATURES
)
print(
    "     dtype      : float64"
)
print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 17. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


EXPECTED_XGB = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if lgbm_params != EXPECTED_LGBM:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 18. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


start = time.perf_counter()


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_ports_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 19. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


start = time.perf_counter()


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_ports_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 20. ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5 * lgbm_probability
    + 0.5 * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 21. METRICS
# =============================================================================

y_val = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_val.mean()
)


pr_auc = float(
    average_precision_score(
        y_val,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_val,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_val,
        predicted,
    )
)

precision = float(
    precision_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

fpr = float(
    fp / (fp + tn)
)

fnr = float(
    fn / (fn + tp)
)


# =============================================================================
# 22. FULL CHRONOLOGICAL REFERENCE
# =============================================================================

full_chrono = json.loads(
    STAGE22_FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


full_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


chrono_pr_penalty = float(
    full_pr_auc
    - pr_auc
)


chrono_roc_penalty = float(
    full_roc_auc
    - roc_auc
)


chrono_f1_penalty = float(
    float(
        full_standard["f1"]
    )
    - f1
)


chrono_recall_penalty = float(
    float(
        full_standard["recall"]
    )
    - recall
)


chrono_fpr_change = float(
    float(
        full_standard["fpr"]
    )
    - fpr
)


# =============================================================================
# 23. SEALED RANDOM REFERENCE + INTERACTION
# =============================================================================

random_pr_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)


random_roc_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


random_f1_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "f1_at_0_50"
    ]
)


random_recall_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "recall_at_0_50"
    ]
)


random_fpr_change = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "fpr_at_0_50"
    ]
)


interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)


interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)


interaction_f1 = float(
    random_f1_penalty
    - chrono_f1_penalty
)


interaction_recall = float(
    random_recall_penalty
    - chrono_recall_penalty
)


interaction_fpr = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 24. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_ports_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=y_val,

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 25. RESULT
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_PORTS_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {
        "stage":
            "Stage23-1C",

        "commit":
            STAGE23_1C_COMMIT,

        "tag":
            STAGE23_1C_TAG,

        "result_sha256":
            STAGE23_1C_RESULT_SHA256,
    },

    "cell": {
        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            "transport_identifier_restriction",

        "src_port_present_in_frozen_70f":
            False,
    },

    "data": {
        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {
            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {
            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "lightgbm": {
            "parameters":
                lgbm_params,

            "fit_seconds":
                lgbm_seconds,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {
            "parameters":
                xgb_params,

            "fit_seconds":
                xgb_seconds,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {
        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {
        "threshold":
            0.50,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {
        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard["f1"]
            ),

        "recall_at_0_50":
            float(
                full_standard["recall"]
            ),

        "fpr_at_0_50":
            float(
                full_standard["fpr"]
            ),
    },

    "removal_penalty_full_minus_ablated": {
        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {
        "source":
            "Sealed Stage23-1C",

        "result_sha256":
            STAGE23_1C_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {
        "definition":
            "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1,

        "recall_at_0_50":
            interaction_recall,

        "fpr_at_0_50":
            interaction_fpr,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "artifacts": {
        "validation_probabilities": {
            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {
            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {
            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {
        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {
        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            6,

        "stage23_total_model_fits_after_cell":
            8,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1D before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 26. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifacts = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifacts
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = (
    sha256_file(
        CHECKSUM_OUTPUT_PATH
    )
)


# =============================================================================
# 27. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        8,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 28. REMOVE TRANSIENT MEMMAPS
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():
        path.unlink()


try:
    RUNTIME_DIR.rmdir()
except OSError:
    pass


# =============================================================================
# 29. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1D COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print("  subset : NO_PORTS")
print("  split  : CHRONOLOGICAL_NATURAL")

print()
print("Frozen NO_PORTS:")
print("  removed   : Dst Port, Protocol")
print("  retained  : 68")
print("  Src Port  : NOT PRESENT")

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 8 / 50")


print()
print("=" * 96)
print("CHRONOLOGICAL RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence          : {attack_prevalence:.12f}"
)

print(
    f"NO_PORTS PR-AUC            : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                : {full_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty   : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"NO_PORTS ROC-AUC           : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC               : {full_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty  : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence        : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("NO_PORTS SPLIT × ABLATION INTERACTION")
print("=" * 96)

print()
print("PR-AUC:")
print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")
print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary:")
print(
    f"  F1 interaction     : {interaction_f1:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 8 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1D."
)

print("=" * 96)

STAGE23-1D — NO_PORTS × CHRONOLOGICAL_NATURAL

[OK] branch         : main
[OK] HEAD           : ff853952366f731449579f64da44572b3a5a725e
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1C tag : ff853952366f731449579f64da44572b3a5a725e
[OK] worktree       : CLEAN
[OK] predecessor governance: 6 / 50 fits sealed
[OK] Stage23-0 artifacts: 23/23 exact
[OK] Sealed Stage23-1C random result exact.

Frozen target:
  subset        : NO_PORTS
  split         : CHRONOLOGICAL_NATURAL
  feature count : 68
  removed       : ['Dst Port', 'Protocol']
  semantics     : transport_identifier_restriction
  Src Port      : NOT PRESENT

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] CHRONOLOGICAL_NATURAL:
     train      : 13,818,623
     validation : 593,780
     train days : 0..6
     val day    : 7 / 02-28-2018

MATERIALIZATI

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 492.746 s
     model SHA256: 4a1dca073af9dbcbedf70429390f62282ac5f10db5eaf0533bf8b4b0ed239a0f

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 142.098 s
     model SHA256: 3bd7d4d51f69b111021f48ee319433bebd523fbefadec08b518cb078648f68af

STAGE23-1D COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_PORTS
  split  : CHRONOLOGICAL_NATURAL

Frozen NO_PORTS:
  removed   : Dst Port, Protocol
  retained  : 68
  Src Port  : NOT PRESENT

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 8 / 50

CHRONOLOGICAL RANKING METRICS

Attack prevalence          : 0.104846912998
NO_PORTS PR-AUC            : 0.102473487388
FULL PR-AUC                : 0.106215155134
Chronological PR penalty   : +0.003741667746

NO_PORTS ROC-AUC           : 0.488863937353
FULL ROC-AUC               : 0.514918426394
Chronological ROC penalty  : +0.026054489040

PR-AUC - prevalence        : -0.002373425610

FIXED OPERATING POINT — THRESHOLD 0.50

Accuracy  : 0.89

In [10]:
# =============================================================================
# STAGE23-1D — SEAL + COMMIT + TAG + PUSH
# NO_PORTS × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# Frozen NO_PORTS:
#   Dst Port + Protocol removed
#   Src Port absent from frozen 70-feature space
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/stage23_1d_no_ports_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_ports"
)

EXPECTED_PARENT = (
    "ff853952366f731449579f64da44572b3a5a725e"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1c-no-ports-random-natural-v1"
)

RESULT_TAG = (
    "stage23-1d-no-ports-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1D: freeze NO_PORTS chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1D frozen NO_PORTS CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1d_no_ports_chronological_natural_result.json":
        "47240e34b857d087eae699885284c35e0a530f98730edbe4b2662ca0fe15ac55",

    "no_ports_chronological_natural_lightgbm_model.txt":
        "4a1dca073af9dbcbedf70429390f62282ac5f10db5eaf0533bf8b4b0ed239a0f",

    "no_ports_chronological_natural_xgboost_model.json":
        "3bd7d4d51f69b111021f48ee319433bebd523fbefadec08b518cb078648f68af",

    "no_ports_chronological_natural_validation_probabilities.npz":
        "1cebe2d875ad342c8370a17689ff9301991c06c8cad94551290588465c98439e",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "79c3c0cf68d22dcffc0c25eabb4172fb9903091e513cd05143cabf2206fcbee6"
)


# =============================================================================
# 2. EXACT OBSERVED VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.102473487388
EXPECTED_ROC_AUC = 0.488863937353

EXPECTED_PR_PENALTY = 0.003741667746
EXPECTED_ROC_PENALTY = 0.026054489040

EXPECTED_PR_INTERACTION = -0.003419063492
EXPECTED_ROC_INTERACTION = -0.025905579373


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1D — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1D seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1C predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1D seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1D target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1D source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1D result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1C tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1D source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1D CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1D CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1D artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1D ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1d_no_ports_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1D":
    raise RuntimeError(
        "Unexpected Stage23-1D identifier."
    )


if result["cell"]["subset"] != "NO_PORTS":
    raise RuntimeError(
        "Stage23-1D subset changed."
    )


if result["cell"]["split"] != "CHRONOLOGICAL_NATURAL":
    raise RuntimeError(
        "Stage23-1D split changed."
    )


if result["cell"]["removed_features"] != [
    "Dst Port",
    "Protocol",
]:
    raise RuntimeError(
        "Frozen NO_PORTS removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 68:
    raise RuntimeError(
        "Frozen NO_PORTS feature count changed."
    )


if result[
    "cell"
][
    "src_port_present_in_frozen_70f"
] is not False:
    raise RuntimeError(
        "Src Port governance changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "transport_identifier_restriction":
    raise RuntimeError(
        "NO_PORTS semantic definition changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-1D protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1D predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance["new_model_fits_this_cell"] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1D fit count."
    )


if governance["stage23_total_model_fits_before_cell"] != 6:
    raise RuntimeError(
        "Unexpected pre-cell fit count."
    )


if governance["stage23_total_model_fits_after_cell"] != 8:
    raise RuntimeError(
        "Unexpected post-cell fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


if result[
    "shortcut_interaction"
][
    "confidence_interval_status"
] != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS":
    raise RuntimeError(
        "Interaction uncertainty status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result[
        "ranking_metrics"
    ][
        "pr_auc"
    ]
)

actual_roc_auc = float(
    result[
        "ranking_metrics"
    ][
        "roc_auc"
    ]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)

actual_pr_interaction = float(
    result[
        "shortcut_interaction"
    ][
        "pr_auc"
    ]
)

actual_roc_interaction = float(
    result[
        "shortcut_interaction"
    ][
        "roc_auc"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
    (
        "PR-AUC interaction",
        actual_pr_interaction,
        EXPECTED_PR_INTERACTION,
    ),
    (
        "ROC-AUC interaction",
        actual_roc_interaction,
        EXPECTED_ROC_INTERACTION,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1D output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC          : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC         : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty      : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty     : {actual_roc_penalty:+.12f}"
)

print(
    f"     PR interaction  : {actual_pr_interaction:+.12f}"
)

print(
    f"     ROC interaction : {actual_roc_interaction:+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1D",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {
        "subset":
            "NO_PORTS",

        "split":
            "CHRONOLOGICAL_NATURAL",

        "removed_features": [
            "Dst Port",
            "Protocol",
        ],

        "feature_count":
            68,

        "operational_semantics":
            "transport_identifier_restriction",

        "src_port_present_in_frozen_70f":
            False,
    },

    "sealed_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        8,

    "ranking": {
        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction": {
        "pr_auc":
            actual_pr_interaction,

        "roc_auc":
            actual_roc_interaction,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1E NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1D — NO_PORTS × CHRONOLOGICAL_NATURAL

This directory freezes the matched chronological `NO_PORTS` primary-ablation
cell.

## Frozen operational definition

`NO_PORTS` removes:

- `Dst Port`
- `Protocol`

`Src Port` is not present in the frozen Stage22R 70-feature model input.

## Frozen result

- Retained features: 68
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − NO_PORTS PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − NO_PORTS ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Split × ablation interaction

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

- PR-AUC interaction: {actual_pr_interaction:+.12f}
- ROC-AUC interaction: {actual_roc_interaction:+.12f}

These are point estimates only. The preregistered uncertainty analysis is
required before inferential interpretation.

## Governance

- Stage23 model fits sealed after this cell: 8 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none
- Raw March 1 and March 2: permanently closed
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1D
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1D ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1D RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():
    raise RuntimeError(
        "Repository dirty after Stage23-1D commit."
    )


print()
print("Stage23-1D commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1D tag verification failed."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1D")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":
        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1D annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1D tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1D push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1D SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1C parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1D commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_PORTS PAIR")
print("=" * 96)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : "
    f'{result["matched_random_reference"]["pr_auc_removal_penalty"]:+.12f}'
)
print(
    f"  ROC penalty : "
    f'{result["matched_random_reference"]["roc_auc_removal_penalty"]:+.12f}'
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR penalty  : {actual_pr_penalty:+.12f}"
)
print(
    f"  ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {actual_pr_interaction:+.12f}"
)
print(
    f"  ROC-AUC : {actual_roc_interaction:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 8 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1D COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1E — NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL"
)

print("=" * 96)

STAGE23-1D — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : ff853952366f731449579f64da44572b3a5a725e
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1C tag: ff853952366f731449579f64da44572b3a5a725e
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  79c3c0cf68d22dcffc0c25eabb4172fb9903091e513cd05143cabf2206fcbee6

VERIFYING STAGE23-1D CORE ARTIFACTS

[EXACT] stage23_1d_no_ports_chronological_natural_result.json
[EXACT] no_ports_chronological_natural_lightgbm_model.txt
[EXACT] no_ports_chronological_natural_xgboost_model.json
[EXACT] no_ports_chronological_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC          : 0.102473487388
     ROC-AUC         : 0.488863937353
     PR penalty      : +0.003741667746
     ROC penalty     : +0.026054489040
     PR interaction  : -0.003419063492
     ROC interaction : -0.025905579373
     CI status       : PENDING

VERI

In [11]:
# =============================================================================
# STAGE23-1E
# NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL
# =============================================================================
#
# FROZEN SUBSET:
#   remove exactly:
#       Init Fwd Win Byts
#
# AUTHORIZED BY:
#   Stage23-0:
#       stage23-0-protocol-lock-v1
#       -> 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# REQUIRED PREDECESSOR:
#   Stage23-1D:
#       stage23-1d-no-ports-chronological-natural-v1
#       -> 8b3ddc6cda51d5fba68db50f476829152d0b4ad4
#
# THIS CELL:
#   - fits 1 LightGBM
#   - fits 1 XGBoost
#   - evaluates RANDOM_NATURAL
#   - reuses frozen Stage22R FULL RANDOM_NATURAL reference
#
# STAGE23 FIT COUNT:
#   before : 8 / 50
#   after  : 10 / 50
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   Mar1
#   Mar2
#
# FAIL CLOSED:
#   If either model fit completes and a later step fails, DO NOT blindly
#   rerun this cell. Preserve OUTPUT_DIR and send execution_state.json.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1E"

TARGET_SUBSET = "NO_INIT_FWD_WIN_BYTS"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_COMMIT = (
    "8b3ddc6cda51d5fba68db50f476829152d0b4ad4"
)

PREDECESSOR_TAG = (
    "stage23-1d-no-ports-chronological-natural-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_ports"
    / "seal_receipt.json"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)

STAGE22_FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1e_no_init_fwd_win_byts_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1e_no_init_fwd_win_byts_random_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1E OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "predecessor_commit":
        PREDECESSOR_COMMIT,

    "predecessor_tag":
        PREDECESSOR_TAG,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_total_model_fits_before_cell":
        8,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. REPOSITORY / TAG / PREDECESSOR VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1E — NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1E MUST BEGIN FROM SEALED STAGE23-1D.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1D result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1E:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1D tag :", predecessor_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR GOVERNANCE
# =============================================================================

if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1D seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 8:

    raise RuntimeError(
        "Expected exactly 8 sealed Stage23 fits before Stage23-1E."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1E NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL":

    raise RuntimeError(
        "\nStage23-1D seal does not authorize Stage23-1E.\n"
        f"actual authorization: "
        f"{predecessor_seal.get('next_authorized_model_cell')}"
    )


print(
    "[OK] predecessor governance: 8 / 50 fits sealed"
)


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


actual_protocol_manifest_sha = (
    sha256_file(
        PROTOCOL_CHECKSUMS
    )
)


if actual_protocol_manifest_sha != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "\nStage23-0 checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_protocol_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )


    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SUBSET / MODEL / SPLIT SPEC
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


EXPECTED_REMOVED = [
    "Init Fwd Win Byts",
]


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nFrozen NO_INIT_FWD_WIN_BYTS definition changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if len(FEATURES) != 69:

    raise RuntimeError(
        f"Expected 69 retained features; found {len(FEATURES)}"
    )


if "Init Fwd Win Byts" in FEATURES:

    raise RuntimeError(
        "Init Fwd Win Byts remains in frozen ablation feature list."
    )


print()
print("Frozen target:")
print(
    "  subset        :",
    TARGET_SUBSET
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  feature count :",
    len(FEATURES)
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  semantics     :",
    subset_spec[
        "semantic_label"
    ]
)


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 10. FROZEN RANDOM_NATURAL MEMBERSHIP
# =============================================================================

random_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = random_spec[
    "train"
]

EXPECTED_VALIDATION = random_spec[
    "validation"
]


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)

N_DEVELOPMENT = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)


if TRAIN_ROWS != 11_529_922:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL training count changed."
    )


if VALIDATION_ROWS != 2_882_481:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL validation count changed."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random validation mask length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Frozen random validation population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 11. AUTHORIZED CACHE FILES ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. FLOAT64 MEMMAP MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)

print()
print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Features        :",
    N_FEATURES
)
print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)
print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"free   : {disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. STREAM EXACT RANDOM MEMBERSHIP
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0
expected_next_clean_position = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING RANDOM NO_INIT_FWD_WIN_BYTS MATRICES")
print("=" * 96)
print()


for day_index, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        expected_cp = np.arange(
            expected_next_clean_position,
            expected_next_clean_position
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file        : {filename}\n"
                f"expected    : {expected_next_clean_position:,}\n"
                f"actual first: {int(cp[0]):,}"
            )


        expected_next_clean_position += (
            batch_rows
        )


        is_validation = (
            random_validation_mask[
                cp
            ]
        )

        is_train = (
            ~is_validation
        )


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        n_train = int(
            is_train.sum()
        )

        n_validation = int(
            is_validation.sum()
        )


        if n_train:

            next_train = (
                train_cursor
                + n_train
            )

            if next_train > TRAIN_ROWS:

                raise RuntimeError(
                    "Random training memmap overflow."
                )


            X_train[
                train_cursor:next_train
            ] = X_batch[
                is_train
            ]

            y_train[
                train_cursor:next_train
            ] = y[
                is_train
            ]

            train_cursor = (
                next_train
            )


        if n_validation:

            next_validation = (
                validation_cursor
                + n_validation
            )

            if next_validation > VALIDATION_ROWS:

                raise RuntimeError(
                    "Random validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_validation
            ] = X_batch[
                is_validation
            ]

            y_validation[
                validation_cursor:next_validation
            ] = y[
                is_validation
            ]

            validation_positions[
                validation_cursor:next_validation
            ] = cp[
                is_validation
            ]

            validation_cursor = (
                next_validation
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            y,
            expected_cp,
            is_train,
            is_validation,
        )

        gc.collect()


    print(
        f"[OK] day {day_index:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if expected_next_clean_position != N_DEVELOPMENT:

    raise RuntimeError(
        "\nDevelopment canonical-length mismatch.\n"
        f"expected: {N_DEVELOPMENT:,}\n"
        f"actual  : {expected_next_clean_position:,}"
    )


if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Random training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Random training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Random validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Random validation benign count mismatch."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean_position order is not strictly increasing."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):

    raise RuntimeError(
        "Materialized validation positions disagree with frozen bitset."
    )


print()
print("[OK] Frozen RANDOM_NATURAL matrices materialized.")

print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)

print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)

print(
    "     features   :",
    N_FEATURES
)

print(
    "     dtype      : float64"
)

print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 19. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5 * lgbm_probability
    + 0.5 * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 20. RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 21. FIXED OPERATING POINT 0.50
# =============================================================================

FIXED_THRESHOLD = np.float32(
    0.50
)


predicted = (
    ensemble_probability
    >= FIXED_THRESHOLD
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 22. FROZEN FULL RANDOM REFERENCE
# =============================================================================

full_random = json.loads(
    STAGE22_FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_random["cell"] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Unexpected Stage22R FULL random reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 23. RANDOM REMOVAL PENALTIES
# =============================================================================
#
# Δ_R(S) = FULL_R - SUBSET_R
#
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty_at_050 = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


recall_removal_penalty_at_050 = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


fpr_change_full_minus_ablated_at_050 = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 24. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 25. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_INIT_FWD_WIN_BYTS_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {

        "stage":
            "Stage23-1D",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty_at_050,

        "recall_at_0_50":
            recall_removal_penalty_at_050,

        "fpr_at_0_50":
            fpr_change_full_minus_ablated_at_050,
    },

    "shortcut_interaction": {

        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            8,

        "stage23_total_model_fits_after_cell":
            10,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1E before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 26. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifacts = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifacts
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = (
    sha256_file(
        CHECKSUM_OUTPUT_PATH
    )
)


# =============================================================================
# 27. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        10,

    "stage23_metrics_calculated":
        True,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 28. REMOVE TRANSIENT MEMMAPS
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 29. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1E COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen ablation:")
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  retained      :",
    N_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 10 / 50")


print()
print("=" * 96)
print("RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence                    : {attack_prevalence:.12f}"
)

print(
    f"NO_INIT_FWD_WIN_BYTS PR-AUC          : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                          : {full_pr_auc:.12f}"
)

print(
    f"PR-AUC removal penalty               : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"NO_INIT_FWD_WIN_BYTS ROC-AUC         : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                         : {full_roc_auc:.12f}"
)

print(
    f"ROC-AUC removal penalty              : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence                  : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 10 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Shortcut interaction:")
print(
    "  PENDING matched "
    "NO_INIT_FWD_WIN_BYTS chronological cell"
)


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1E."
)

print("=" * 96)

STAGE23-1E — NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL

[OK] branch         : main
[OK] HEAD           : 8b3ddc6cda51d5fba68db50f476829152d0b4ad4
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1D tag : 8b3ddc6cda51d5fba68db50f476829152d0b4ad4
[OK] worktree       : CLEAN
[OK] predecessor governance: 8 / 50 fits sealed
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_INIT_FWD_WIN_BYTS
  split         : RANDOM_NATURAL
  feature count : 69
  removed       : ['Init Fwd Win Byts']
  semantics     : initial_forward_window_ablation

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] RANDOM_NATURAL membership:
     train      : 11,529,922
     validation : 2,882,481

MATERIALIZATION PLAN

Train rows      : 11,529,922
Validation rows : 2,882,481
Features        : 69
Matrix storage  : 7.409 GiB
Wor

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 389.178 s
     model SHA256: a2a5168a8b0bcb2b06946c77aa62d528a60a74d5ebe3547e9016da49f8ca744c

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 111.074 s
     model SHA256: 658bdda25fb2b08f2290ba73c81e98582c0114fc4e03a2f55ab3610e460e3500

STAGE23-1E COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_INIT_FWD_WIN_BYTS
  split  : RANDOM_NATURAL

Frozen ablation:
  removed       : ['Init Fwd Win Byts']
  retained      : 69
  semantic label: initial_forward_window_ablation

Data:
  train      : 11,529,922
  validation : 2,882,481

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 10 / 50

RANKING METRICS

Attack prevalence                    : 0.136847389454
NO_INIT_FWD_WIN_BYTS PR-AUC          : 0.993417448831
FULL PR-AUC                          : 0.995590041899
PR-AUC removal penalty               : +0.002172593068

NO_INIT_FWD_WIN_BYTS ROC-AUC         : 0.998192014039
FULL ROC-AUC                         : 0.998624564774
ROC

In [12]:
# =============================================================================
# STAGE23-1E — SEAL + COMMIT + TAG + PUSH
# NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1e_no_init_fwd_win_byts_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_init_fwd_win_byts"
)

EXPECTED_PARENT = (
    "8b3ddc6cda51d5fba68db50f476829152d0b4ad4"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1d-no-ports-chronological-natural-v1"
)

RESULT_TAG = (
    "stage23-1e-no-init-fwd-win-byts-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1E: freeze NO_INIT_FWD_WIN_BYTS random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1E frozen NO_INIT_FWD_WIN_BYTS RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1e_no_init_fwd_win_byts_random_natural_result.json":
        "bae6f10594df955857ba8653fca886c2e861cd6687e718da80d3dffadf65db7f",

    "no_init_fwd_win_byts_random_natural_lightgbm_model.txt":
        "a2a5168a8b0bcb2b06946c77aa62d528a60a74d5ebe3547e9016da49f8ca744c",

    "no_init_fwd_win_byts_random_natural_xgboost_model.json":
        "658bdda25fb2b08f2290ba73c81e98582c0114fc4e03a2f55ab3610e460e3500",

    "no_init_fwd_win_byts_random_natural_validation_probabilities.npz":
        "720fa25d788b65b880396daffff60ef2776548a7a48745bdd653dc6765f0c593",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "9eefe10eaaf5118403bbf0d00a790b0bebd3f932ee9be12b32711c7aabe43424"
)


# =============================================================================
# 2. EXACT OBSERVED VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.993417448831
EXPECTED_ROC_AUC = 0.998192014039

EXPECTED_PR_PENALTY = 0.002172593068
EXPECTED_ROC_PENALTY = 0.000432550735


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1E — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1E seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1D predecessor tag verification failed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage23-1E seal:\n"
        + status
    )


if TARGET_DIR.exists():

    raise RuntimeError(
        f"Stage23-1E target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():

    raise RuntimeError(
        f"Stage23-1E source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:

    raise RuntimeError(
        f"Stage23-1E result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1D tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():

    raise RuntimeError(
        "Stage23-1E source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:

    raise RuntimeError(
        "\nSTAGE23-1E CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1E CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-1E artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-1E ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1e_no_init_fwd_win_byts_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1E":

    raise RuntimeError(
        "Unexpected Stage23-1E stage identifier."
    )


if result["cell"]["subset"] != "NO_INIT_FWD_WIN_BYTS":

    raise RuntimeError(
        "Stage23-1E subset changed."
    )


if result["cell"]["split"] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Stage23-1E split changed."
    )


if result["cell"]["removed_features"] != [
    "Init Fwd Win Byts"
]:

    raise RuntimeError(
        "Frozen removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 69:

    raise RuntimeError(
        "Frozen Stage23-1E feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "initial_forward_window_ablation":

    raise RuntimeError(
        "Stage23-1E semantic label changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1E predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance["new_model_fits_this_cell"] != 2:

    raise RuntimeError(
        "Unexpected Stage23-1E fit count."
    )


if governance["stage23_total_model_fits_before_cell"] != 8:

    raise RuntimeError(
        "Unexpected pre-cell Stage23 fit count."
    )


if governance["stage23_total_model_fits_after_cell"] != 10:

    raise RuntimeError(
        "Unexpected post-cell Stage23 fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:

        raise RuntimeError(
            f"Governance violation: {key}"
        )


if result[
    "shortcut_interaction"
][
    "status"
] != "PENDING_MATCHED_CHRONOLOGICAL_CELL":

    raise RuntimeError(
        "Stage23-1E interaction status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result[
        "ranking_metrics"
    ][
        "pr_auc"
    ]
)

actual_roc_auc = float(
    result[
        "ranking_metrics"
    ][
        "roc_auc"
    ]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1E output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty : {actual_roc_penalty:+.12f}"
)

print(
    "     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:

        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1E",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_INIT_FWD_WIN_BYTS",

        "split":
            "RANDOM_NATURAL",

        "removed_features": [
            "Init Fwd Win Byts"
        ],

        "feature_count":
            69,

        "semantic_label":
            "initial_forward_window_ablation",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        10,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1F NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1E — NO_INIT_FWD_WIN_BYTS × RANDOM_NATURAL

This directory freezes the random-split single-feature ablation of
`Init Fwd Win Byts`.

## Frozen cell

- Subset: `NO_INIT_FWD_WIN_BYTS`
- Split: `RANDOM_NATURAL`
- Removed feature: `Init Fwd Win Byts`
- Retained features: 69
- New boosted-model fits: 2
- Total Stage23 fits after this cell: 10 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − ablated PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − ablated ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Fixed threshold 0.50

- Accuracy: {result["fixed_threshold_0_50"]["accuracy"]:.12f}
- Precision: {result["fixed_threshold_0_50"]["precision"]:.12f}
- Recall: {result["fixed_threshold_0_50"]["recall"]:.12f}
- F1: {result["fixed_threshold_0_50"]["f1"]:.12f}
- FPR: {result["fixed_threshold_0_50"]["fpr"]:.12f}
- FNR: {result["fixed_threshold_0_50"]["fnr"]:.12f}

## Interpretation status

The matched chronological cell has not yet been executed.

Therefore the preregistered split × ablation interaction remains pending.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1E
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1E ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1E RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():

    raise RuntimeError(
        "Repository dirty after Stage23-1E commit."
    )


print()
print("Stage23-1E commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:

    raise RuntimeError(
        "Local Stage23-1E tag verification failed."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1E")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":

        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":

        peeled_commit = sha


if not tag_object:

    raise RuntimeError(
        "Remote Stage23-1E annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote Stage23-1E tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-1E push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1E SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1D parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1E commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_INIT_FWD_WIN_BYTS RANDOM RESULT")
print("=" * 96)

print()
print(
    f"PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("Shortcut interaction:")
print("  PENDING matched chronological cell")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 10 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1E COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1F — "
    "NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL"
)

print("=" * 96)

STAGE23-1E — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : 8b3ddc6cda51d5fba68db50f476829152d0b4ad4
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1D tag: 8b3ddc6cda51d5fba68db50f476829152d0b4ad4
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  9eefe10eaaf5118403bbf0d00a790b0bebd3f932ee9be12b32711c7aabe43424

VERIFYING STAGE23-1E CORE ARTIFACTS

[EXACT] stage23_1e_no_init_fwd_win_byts_random_natural_result.json
[EXACT] no_init_fwd_win_byts_random_natural_lightgbm_model.txt
[EXACT] no_init_fwd_win_byts_random_natural_xgboost_model.json
[EXACT] no_init_fwd_win_byts_random_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC      : 0.993417448831
     ROC-AUC     : 0.998192014039
     PR penalty  : +0.002172593068
     ROC penalty : +0.000432550735
     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL

VERIFYING REPOSITORY COPIES

[EXACT] stage23_1e_no_init_

In [13]:
# =============================================================================
# STAGE23-1F
# NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# FROZEN SUBSET:
#   remove exactly:
#       Init Fwd Win Byts
#
# AUTHORIZED BY:
#   Stage23-0:
#       stage23-0-protocol-lock-v1
#       -> 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
#
# REQUIRED PREDECESSOR:
#   Stage23-1E:
#       stage23-1e-no-init-fwd-win-byts-random-natural-v1
#       -> 1e8f8bb3b204a445efd4f8ab629daa8a33d5c215
#
# THIS CELL:
#   - fits 1 LightGBM
#   - fits 1 XGBoost
#   - evaluates CHRONOLOGICAL_NATURAL
#   - reuses frozen Stage22R FULL chronological result
#   - reuses sealed Stage23-1E random result
#   - computes NO_INIT_FWD_WIN_BYTS split × ablation point estimate
#
# STAGE23 FIT COUNT:
#   before : 10 / 50
#   after  : 12 / 50
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   Mar1
#   Mar2
#
# FAIL CLOSED:
#   If a model completes and a later step fails, DO NOT blindly rerun.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1F"

TARGET_SUBSET = "NO_INIT_FWD_WIN_BYTS"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_COMMIT = (
    "1e8f8bb3b204a445efd4f8ab629daa8a33d5c215"
)

PREDECESSOR_TAG = (
    "stage23-1e-no-init-fwd-win-byts-random-natural-v1"
)

PREDECESSOR_RESULT_SHA256 = (
    "bae6f10594df955857ba8653fca886c2e861cd6687e718da80d3dffadf65db7f"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_init_fwd_win_byts"
)

PREDECESSOR_RESULT = (
    PREDECESSOR_DIR
    / "stage23_1e_no_init_fwd_win_byts_random_natural_result.json"
)

PREDECESSOR_SEAL = (
    PREDECESSOR_DIR
    / "seal_receipt.json"
)

STAGE22_FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1f_no_init_fwd_win_byts_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1f_no_init_fwd_win_byts_chronological_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:
            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1F OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "predecessor_commit":
        PREDECESSOR_COMMIT,

    "predecessor_tag":
        PREDECESSOR_TAG,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_total_model_fits_before_cell":
        10,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. REPOSITORY / TAG VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1F — NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:
    raise RuntimeError(
        "\nSTAGE23-1F MUST BEGIN FROM SEALED STAGE23-1E.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:
    raise RuntimeError(
        "Stage23-1E result tag verification failed."
    )


if status:
    raise RuntimeError(
        "Git worktree must be clean before Stage23-1F:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1E tag :", predecessor_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR SEAL + RESULT
# =============================================================================

if not PREDECESSOR_SEAL.exists():
    raise RuntimeError(
        "Stage23-1E seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 10:

    raise RuntimeError(
        "Expected 10 sealed Stage23 fits before Stage23-1F."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1F NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1E seal does not authorize Stage23-1F."
    )


if not PREDECESSOR_RESULT.exists():
    raise RuntimeError(
        "Stage23-1E sealed result missing."
    )


actual_predecessor_result_sha = (
    sha256_file(
        PREDECESSOR_RESULT
    )
)


if (
    actual_predecessor_result_sha
    != PREDECESSOR_RESULT_SHA256
):

    raise RuntimeError(
        "\nSEALED STAGE23-1E RESULT CHANGED\n"
        f"expected: {PREDECESSOR_RESULT_SHA256}\n"
        f"actual  : {actual_predecessor_result_sha}"
    )


random_result = json.loads(
    PREDECESSOR_RESULT.read_text(
        encoding="utf-8"
    )
)


if random_result["cell"]["subset"] != TARGET_SUBSET:
    raise RuntimeError(
        "Stage23-1E subset mismatch."
    )


if random_result["cell"]["split"] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Stage23-1E split mismatch."
    )


print(
    "[OK] predecessor governance: 10 / 50 fits sealed"
)
print(
    "[OK] Stage23-1E random result exact"
)


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum-manifest SHA256 changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Init Fwd Win Byts"
]:
    raise RuntimeError(
        "Frozen NO_INIT_FWD_WIN_BYTS definition changed."
    )


if len(FEATURES) != 69:
    raise RuntimeError(
        f"Expected 69 retained features; found {len(FEATURES)}"
    )


if "Init Fwd Win Byts" in FEATURES:
    raise RuntimeError(
        "Init Fwd Win Byts remains in ablated feature list."
    )


if subset_spec[
    "semantic_label"
] != "initial_forward_window_ablation":

    raise RuntimeError(
        "Frozen semantic label changed."
    )


print()
print("Frozen target:")
print(
    "  subset        :",
    TARGET_SUBSET
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  feature count :",
    len(FEATURES)
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  semantics     :",
    subset_spec[
        "semantic_label"
    ]
)


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; "
        "Stage23 XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 10. FROZEN CHRONOLOGICAL MEMBERSHIP
# =============================================================================

chrono_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = chrono_spec[
    "train"
]

EXPECTED_VALIDATION = chrono_spec[
    "validation"
]


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)


if TRAIN_ROWS != 13_818_623:

    raise RuntimeError(
        "Chronological train count changed."
    )


if VALIDATION_ROWS != 593_780:

    raise RuntimeError(
        "Chronological validation count changed."
    )


print()
print("[OK] CHRONOLOGICAL_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "     train days : 0..6"
)
print(
    "     val day    : 7 / 02-28-2018"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. MEMMAP MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)

print()
print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Features        :",
    N_FEATURES
)
print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)
print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"free   : {disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. MATERIALIZE CHRONOLOGICAL MATRICES
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "day_id",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING CHRONOLOGICAL NO_INIT_FWD_WIN_BYTS MATRICES")
print("=" * 96)
print()


for expected_day_id, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = batch.num_rows

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        day_ids = batch.column(
            batch.schema.get_field_index(
                "day_id"
            )
        ).to_numpy(
            zero_copy_only=False
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        unique_days = np.unique(
            day_ids
        )


        if not (
            len(unique_days) == 1
            and int(unique_days[0]) == expected_day_id
        ):

            raise RuntimeError(
                f"day_id mismatch in {filename}: {unique_days}"
            )


        if expected_day_id <= 6:

            expected_cp_start = (
                train_cursor
            )

        else:

            expected_cp_start = (
                TRAIN_ROWS
                + validation_cursor
            )


        expected_cp = np.arange(
            expected_cp_start,
            expected_cp_start
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file     : {filename}\n"
                f"expected : {expected_cp_start:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        if expected_day_id <= 6:

            next_cursor = (
                train_cursor
                + batch_rows
            )

            if next_cursor > TRAIN_ROWS:

                raise RuntimeError(
                    "Training memmap overflow."
                )


            X_train[
                train_cursor:next_cursor
            ] = X_batch

            y_train[
                train_cursor:next_cursor
            ] = y

            train_cursor = (
                next_cursor
            )


        else:

            next_cursor = (
                validation_cursor
                + batch_rows
            )

            if next_cursor > VALIDATION_ROWS:

                raise RuntimeError(
                    "Validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_cursor
            ] = X_batch

            y_validation[
                validation_cursor:next_cursor
            ] = y

            validation_positions[
                validation_cursor:next_cursor
            ] = cp

            validation_cursor = (
                next_cursor
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            day_ids,
            y,
            expected_cp,
        )

        gc.collect()


    print(
        f"[OK] day {expected_day_id:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Validation benign count mismatch."
    )


expected_validation_positions = np.arange(
    TRAIN_ROWS,
    TRAIN_ROWS
    + VALIDATION_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    validation_positions,
    expected_validation_positions,
):

    raise RuntimeError(
        "Chronological validation clean_position membership changed."
    )


print()
print("[OK] Frozen CHRONOLOGICAL_NATURAL matrices materialized.")

print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)

print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)

print(
    "     features   :",
    N_FEATURES
)

print(
    "     dtype      : float64"
)

print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


EXPECTED_XGB_PARAMS = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = time.perf_counter()


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = time.perf_counter()


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 19. FROZEN ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probability produced."
    )


# =============================================================================
# 20. METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[
            0,
            1,
        ],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 21. FROZEN FULL CHRONOLOGICAL REFERENCE
# =============================================================================

full_chrono = json.loads(
    STAGE22_FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


if (
    full_chrono[
        "cell"
    ]
    != "CHRONOLOGICAL_NATURAL"
):

    raise RuntimeError(
        "Unexpected Stage22R FULL chronological reference."
    )


full_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 22. CHRONOLOGICAL PENALTIES
# =============================================================================

chrono_pr_penalty = float(
    full_pr_auc
    - pr_auc
)


chrono_roc_penalty = float(
    full_roc_auc
    - roc_auc
)


chrono_f1_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


chrono_recall_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


chrono_fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 23. SEALED RANDOM PENALTIES
# =============================================================================

random_pr_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)


random_roc_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


random_f1_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "f1_at_0_50"
    ]
)


random_recall_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "recall_at_0_50"
    ]
)


random_fpr_change = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "fpr_at_0_50"
    ]
)


# =============================================================================
# 24. THIRD PRIMARY SPLIT × ABLATION INTERACTION
# =============================================================================
#
# I(S) = Δ_R(S) - Δ_C(S)
#
# =============================================================================

interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)


interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)


interaction_f1 = float(
    random_f1_penalty
    - chrono_f1_penalty
)


interaction_recall = float(
    random_recall_penalty
    - chrono_recall_penalty
)


interaction_fpr = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 25. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_init_fwd_win_byts_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 26. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_INIT_FWD_WIN_BYTS_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {

        "stage":
            "Stage23-1E",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,

        "result_sha256":
            PREDECESSOR_RESULT_SHA256,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {

        "source":
            "Sealed Stage23-1E",

        "result_sha256":
            PREDECESSOR_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {

        "definition":
            "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1,

        "recall_at_0_50":
            interaction_recall,

        "fpr_at_0_50":
            interaction_fpr,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            10,

        "stage23_total_model_fits_after_cell":
            12,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1F before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 27. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = (
    sha256_file(
        CHECKSUM_OUTPUT_PATH
    )
)


# =============================================================================
# 28. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        12,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 29. REMOVE TRANSIENT MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():
        path.unlink()


try:
    RUNTIME_DIR.rmdir()
except OSError:
    pass


# =============================================================================
# 30. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1F COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen ablation:")
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  retained      :",
    N_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 12 / 50")


print()
print("=" * 96)
print("CHRONOLOGICAL RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence                    : {attack_prevalence:.12f}"
)

print(
    f"NO_INIT_FWD_WIN_BYTS PR-AUC          : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                          : {full_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty             : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"NO_INIT_FWD_WIN_BYTS ROC-AUC         : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                         : {full_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty            : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence                  : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("NO_INIT_FWD_WIN_BYTS SPLIT × ABLATION INTERACTION")
print("=" * 96)

print()
print("PR-AUC:")
print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")
print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary:")
print(
    f"  F1 interaction     : {interaction_f1:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 12 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1F."
)

print("=" * 96)

STAGE23-1F — NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL

[OK] branch         : main
[OK] HEAD           : 1e8f8bb3b204a445efd4f8ab629daa8a33d5c215
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1E tag : 1e8f8bb3b204a445efd4f8ab629daa8a33d5c215
[OK] worktree       : CLEAN
[OK] predecessor governance: 10 / 50 fits sealed
[OK] Stage23-1E random result exact
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_INIT_FWD_WIN_BYTS
  split         : CHRONOLOGICAL_NATURAL
  feature count : 69
  removed       : ['Init Fwd Win Byts']
  semantics     : initial_forward_window_ablation

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] CHRONOLOGICAL_NATURAL membership:
     train      : 13,818,623
     validation : 593,780
     train days : 0..6
     val day    : 7 / 02-28-2018

MATERIALIZATION PLA

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 460.991 s
     model SHA256: 465af8a3a8a957a08583f8b5c9da5f8e8d3f3642998cda684326ad6bdc74848c

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 141.586 s
     model SHA256: d0fbfff87872e9fc37005ece5465e62056a9a121d9e43f738d88f1bf2b7ecf3a

STAGE23-1F COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_INIT_FWD_WIN_BYTS
  split  : CHRONOLOGICAL_NATURAL

Frozen ablation:
  removed       : ['Init Fwd Win Byts']
  retained      : 69
  semantic label: initial_forward_window_ablation

Data:
  train      : 13,818,623
  validation : 593,780

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 12 / 50

CHRONOLOGICAL RANKING METRICS

Attack prevalence                    : 0.104846912998
NO_INIT_FWD_WIN_BYTS PR-AUC          : 0.106031684604
FULL PR-AUC                          : 0.106215155134
Chronological PR penalty             : +0.000183470530

NO_INIT_FWD_WIN_BYTS ROC-AUC         : 0.509211682791
FULL ROC-AUC                         :

In [14]:
# =============================================================================
# STAGE23-1F — SEAL + COMMIT + TAG + PUSH
# NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1f_no_init_fwd_win_byts_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_init_fwd_win_byts"
)

EXPECTED_PARENT = (
    "1e8f8bb3b204a445efd4f8ab629daa8a33d5c215"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1e-no-init-fwd-win-byts-random-natural-v1"
)

RESULT_TAG = (
    "stage23-1f-no-init-fwd-win-byts-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1F: freeze NO_INIT_FWD_WIN_BYTS chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1F frozen NO_INIT_FWD_WIN_BYTS CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1f_no_init_fwd_win_byts_chronological_natural_result.json":
        "161989fae61fcfdba9c5c854e2876187905575d501b74eaf1ed2a857cbc936d7",

    "no_init_fwd_win_byts_chronological_natural_lightgbm_model.txt":
        "465af8a3a8a957a08583f8b5c9da5f8e8d3f3642998cda684326ad6bdc74848c",

    "no_init_fwd_win_byts_chronological_natural_xgboost_model.json":
        "d0fbfff87872e9fc37005ece5465e62056a9a121d9e43f738d88f1bf2b7ecf3a",

    "no_init_fwd_win_byts_chronological_natural_validation_probabilities.npz":
        "8cadb355ee2e99f83e3e0040c6082b895e2a4137b05a98e45cf2739340e9515c",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "da8c3d5671df1b7fb41814e9e62105d7a4ea17e3f09a68f461fc1e2463049160"
)


# =============================================================================
# 2. EXACT OBSERVED VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.106031684604
EXPECTED_ROC_AUC = 0.509211682791

EXPECTED_PR_PENALTY = 0.000183470530
EXPECTED_ROC_PENALTY = 0.005706743603

EXPECTED_PR_INTERACTION = 0.001989122538
EXPECTED_ROC_INTERACTION = -0.005274192868

EXPECTED_F1_INTERACTION = 0.012250129142
EXPECTED_RECALL_INTERACTION = 0.002946345630
EXPECTED_FPR_INTERACTION = -0.001946060588


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1F — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1F seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1E predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1F seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1F target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1F source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1F result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1E tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1F source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1F CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1F CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1F artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1F ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1f_no_init_fwd_win_byts_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1F":
    raise RuntimeError(
        "Unexpected Stage23-1F stage identifier."
    )


if result["cell"]["subset"] != "NO_INIT_FWD_WIN_BYTS":
    raise RuntimeError(
        "Stage23-1F subset changed."
    )


if result["cell"]["split"] != "CHRONOLOGICAL_NATURAL":
    raise RuntimeError(
        "Stage23-1F split changed."
    )


if result["cell"]["removed_features"] != [
    "Init Fwd Win Byts"
]:
    raise RuntimeError(
        "Frozen removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 69:
    raise RuntimeError(
        "Frozen Stage23-1F feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "initial_forward_window_ablation":
    raise RuntimeError(
        "Frozen semantic label changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1F predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance["new_model_fits_this_cell"] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1F fit count."
    )


if governance["stage23_total_model_fits_before_cell"] != 10:
    raise RuntimeError(
        "Unexpected pre-cell Stage23 fit count."
    )


if governance["stage23_total_model_fits_after_cell"] != 12:
    raise RuntimeError(
        "Unexpected post-cell Stage23 fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


interaction = result[
    "shortcut_interaction"
]


if interaction[
    "confidence_interval_status"
] != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS":
    raise RuntimeError(
        "Stage23-1F uncertainty status changed."
    )


if interaction[
    "interpretation_status"
] != "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE":
    raise RuntimeError(
        "Stage23-1F interpretation status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result["ranking_metrics"]["pr_auc"]
)

actual_roc_auc = float(
    result["ranking_metrics"]["roc_auc"]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["pr_auc"]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["roc_auc"]
)

actual_pr_interaction = float(
    interaction[
        "pr_auc"
    ]
)

actual_roc_interaction = float(
    interaction[
        "roc_auc"
    ]
)

actual_f1_interaction = float(
    interaction[
        "f1_at_0_50"
    ]
)

actual_recall_interaction = float(
    interaction[
        "recall_at_0_50"
    ]
)

actual_fpr_interaction = float(
    interaction[
        "fpr_at_0_50"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
    (
        "PR-AUC interaction",
        actual_pr_interaction,
        EXPECTED_PR_INTERACTION,
    ),
    (
        "ROC-AUC interaction",
        actual_roc_interaction,
        EXPECTED_ROC_INTERACTION,
    ),
    (
        "F1 interaction",
        actual_f1_interaction,
        EXPECTED_F1_INTERACTION,
    ),
    (
        "Recall interaction",
        actual_recall_interaction,
        EXPECTED_RECALL_INTERACTION,
    ),
    (
        "FPR interaction",
        actual_fpr_interaction,
        EXPECTED_FPR_INTERACTION,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1F output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC          : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC         : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty      : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty     : {actual_roc_penalty:+.12f}"
)

print(
    f"     PR interaction  : {actual_pr_interaction:+.12f}"
)

print(
    f"     ROC interaction : {actual_roc_interaction:+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1F",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_INIT_FWD_WIN_BYTS",

        "split":
            "CHRONOLOGICAL_NATURAL",

        "removed_features": [
            "Init Fwd Win Byts"
        ],

        "feature_count":
            69,

        "semantic_label":
            "initial_forward_window_ablation",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        12,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction": {

        "pr_auc":
            actual_pr_interaction,

        "roc_auc":
            actual_roc_interaction,

        "f1_at_0_50":
            actual_f1_interaction,

        "recall_at_0_50":
            actual_recall_interaction,

        "fpr_at_0_50":
            actual_fpr_interaction,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1G NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1F — NO_INIT_FWD_WIN_BYTS × CHRONOLOGICAL_NATURAL

This directory freezes the chronological partner of the
`NO_INIT_FWD_WIN_BYTS` primary ablation.

## Frozen cell

- Removed feature: `Init Fwd Win Byts`
- Retained features: 69
- Split: `CHRONOLOGICAL_NATURAL`
- New boosted-model fits: 2
- Total Stage23 fits after cell: 12 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Chronological ranking

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − ablated PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − ablated ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Split × ablation interaction

Frozen definition:

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

Point estimates:

- PR-AUC interaction: {actual_pr_interaction:+.12f}
- ROC-AUC interaction: {actual_roc_interaction:+.12f}
- F1@0.50 interaction: {actual_f1_interaction:+.12f}
- Recall@0.50 interaction: {actual_recall_interaction:+.12f}
- FPR@0.50 interaction: {actual_fpr_interaction:+.12f}

The PR-AUC and ROC-AUC point estimates have different signs.

No inferential interpretation is made here. The preregistered paired
bootstrap uncertainty analysis remains required.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1F
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1F ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1F RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():
    raise RuntimeError(
        "Repository dirty after Stage23-1F commit."
    )


print()
print("Stage23-1F commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1F tag verification failed."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1F")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":
        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1F annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1F tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1F push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1F SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1E parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1F commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_INIT_FWD_WIN_BYTS PAIR")
print("=" * 96)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : "
    f'{result["matched_random_reference"]["pr_auc_removal_penalty"]:+.12f}'
)
print(
    f"  ROC penalty : "
    f'{result["matched_random_reference"]["roc_auc_removal_penalty"]:+.12f}'
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR penalty  : {actual_pr_penalty:+.12f}"
)
print(
    f"  ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {actual_pr_interaction:+.12f}"
)
print(
    f"  ROC-AUC : {actual_roc_interaction:+.12f}"
)

print()
print("Supplementary interactions:")
print(
    f"  F1@0.50     : {actual_f1_interaction:+.12f}"
)
print(
    f"  Recall@0.50 : {actual_recall_interaction:+.12f}"
)
print(
    f"  FPR@0.50    : {actual_fpr_interaction:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 12 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1F COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1G — NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL"
)

print("=" * 96)

STAGE23-1F — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : 1e8f8bb3b204a445efd4f8ab629daa8a33d5c215
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1E tag: 1e8f8bb3b204a445efd4f8ab629daa8a33d5c215
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  da8c3d5671df1b7fb41814e9e62105d7a4ea17e3f09a68f461fc1e2463049160

VERIFYING STAGE23-1F CORE ARTIFACTS

[EXACT] stage23_1f_no_init_fwd_win_byts_chronological_natural_result.json
[EXACT] no_init_fwd_win_byts_chronological_natural_lightgbm_model.txt
[EXACT] no_init_fwd_win_byts_chronological_natural_xgboost_model.json
[EXACT] no_init_fwd_win_byts_chronological_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC          : 0.106031684604
     ROC-AUC         : 0.509211682791
     PR penalty      : +0.000183470530
     ROC penalty     : +0.005706743603
     PR interaction  : +0.001989122538
     ROC interaction : -0.0

In [15]:
# =============================================================================
# STAGE23-1G
# NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL
# =============================================================================
#
# Frozen ablation:
#   remove exactly:
#       Fwd Seg Size Min
#
# Parent:
#   Stage23-1F
#   f8cea7af7f239e254e427663a2d963483d0047c4
#
# Fits:
#   before = 12 / 50
#   this cell = 2
#   after = 14 / 50
#
# NO threshold optimization
# NO subset-specific tuning
# NO rebalancing
# NO SHAP
# NO placebo
# NO raw Mar1 / Mar2
#
# If a fit completes and a later step fails:
#   DO NOT blindly rerun.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1G"
TARGET_SUBSET = "NO_FWD_SEG_SIZE_MIN"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
PROTOCOL_TAG = "stage23-0-protocol-lock-v1"

PREDECESSOR_COMMIT = "f8cea7af7f239e254e427663a2d963483d0047c4"
PREDECESSOR_TAG = (
    "stage23-1f-no-init-fwd-win-byts-chronological-natural-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_init_fwd_win_byts"
    / "seal_receipt.json"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)

STAGE22_FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1g_no_fwd_seg_size_min_random_natural"
)

RUNTIME_DIR = OUTPUT_DIR / "runtime"

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1g_no_fwd_seg_size_min_random_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + out if out else "")
        )

    return out


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT STATE
# =============================================================================

if OUTPUT_DIR.exists():

    state = None

    if EXECUTION_STATE_PATH.exists():

        try:
            state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )
        except Exception:
            state = {
                "status": "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1G OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"{json.dumps(state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {
    "stage": STAGE,
    "subset": TARGET_SUBSET,
    "split": TARGET_SPLIT,
    "status": "INITIALIZED",
    "created_utc": now_utc(),

    "protocol_commit": PROTOCOL_COMMIT,
    "protocol_tag": PROTOCOL_TAG,

    "predecessor_commit": PREDECESSOR_COMMIT,
    "predecessor_tag": PREDECESSOR_TAG,

    "stage23_total_model_fits_before_cell": 12,

    "lightgbm_completed_fits": 0,
    "xgboost_completed_fits": 0,
    "models_completed_total_this_cell": 0,

    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. GIT / PROTOCOL / PREDECESSOR
# =============================================================================

print("=" * 96)
print("STAGE23-1G — NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:
    raise RuntimeError(
        "\nUnexpected Stage23-1G parent.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:
    raise RuntimeError(
        "Stage23-1F result tag mismatch."
    )


if status:
    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1F tag :", predecessor_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. PREDECESSOR GOVERNANCE
# =============================================================================

if not PREDECESSOR_SEAL.exists():
    raise RuntimeError(
        "Stage23-1F seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 12:
    raise RuntimeError(
        "Expected 12 sealed Stage23 fits before Stage23-1G."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1G NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL":

    raise RuntimeError(
        "\nStage23-1F does not authorize Stage23-1G.\n"
        f"actual: "
        f"{predecessor_seal.get('next_authorized_model_cell')}"
    )


print(
    "[OK] predecessor governance: 12 / 50 fits sealed"
)


# =============================================================================
# 6. VERIFY STAGE23-0 BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


actual_manifest_sha = sha256_file(
    PROTOCOL_CHECKSUMS
)


if actual_manifest_sha != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "\nStage23-0 checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Missing frozen protocol artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nFrozen Stage23-0 artifact changed.\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Fwd Seg Size Min"
]:
    raise RuntimeError(
        "\nFrozen subset definition changed.\n"
        f"actual: {REMOVED_FEATURES}"
    )


if len(FEATURES) != 69:
    raise RuntimeError(
        f"Expected 69 retained features; found {len(FEATURES)}"
    )


if "Fwd Seg Size Min" in FEATURES:
    raise RuntimeError(
        "Fwd Seg Size Min remains in ablated feature list."
    )


if subset_spec[
    "semantic_label"
] != "minimum_forward_segment_ablation":

    raise RuntimeError(
        "Frozen semantic label changed."
    )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  retained      :", len(FEATURES))
print("  removed       :", REMOVED_FEATURES)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "shap": "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; "
        "XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 10. FROZEN RANDOM MEMBERSHIP
# =============================================================================

random_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)

EXPECTED_TRAIN = (
    random_spec[
        "train"
    ]
)

EXPECTED_VALIDATION = (
    random_spec[
        "validation"
    ]
)


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)

N_DEVELOPMENT = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)


if TRAIN_ROWS != 11_529_922:
    raise RuntimeError(
        "Random training count changed."
    )


if VALIDATION_ROWS != 2_882_481:
    raise RuntimeError(
        "Random validation count changed."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:
    raise RuntimeError(
        "Random validation mask length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:
    raise RuntimeError(
        "Random validation population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()
    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:
            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():
        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)
print()
print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Features        :",
    N_FEATURES
)
print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)
print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:
    raise RuntimeError(
        "Insufficient /kaggle/working disk."
    )


# =============================================================================
# 13. MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)

y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)

X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)

y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)

validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. MATERIALIZE EXACT RANDOM SPLIT
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0
expected_next_clean_position = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING RANDOM NO_FWD_SEG_SIZE_MIN MATRICES")
print("=" * 96)
print()


for day_index, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = batch.num_rows

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        expected_cp = np.arange(
            expected_next_clean_position,
            expected_next_clean_position
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):
            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file     : {filename}\n"
                f"expected : {expected_next_clean_position:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        expected_next_clean_position += (
            batch_rows
        )


        is_validation = (
            random_validation_mask[
                cp
            ]
        )

        is_train = ~is_validation


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        n_train = int(
            is_train.sum()
        )

        n_validation = int(
            is_validation.sum()
        )


        if n_train:

            next_train = (
                train_cursor
                + n_train
            )

            if next_train > TRAIN_ROWS:
                raise RuntimeError(
                    "Training memmap overflow."
                )


            X_train[
                train_cursor:next_train
            ] = X_batch[
                is_train
            ]

            y_train[
                train_cursor:next_train
            ] = y[
                is_train
            ]

            train_cursor = (
                next_train
            )


        if n_validation:

            next_validation = (
                validation_cursor
                + n_validation
            )

            if next_validation > VALIDATION_ROWS:
                raise RuntimeError(
                    "Validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_validation
            ] = X_batch[
                is_validation
            ]

            y_validation[
                validation_cursor:next_validation
            ] = y[
                is_validation
            ]

            validation_positions[
                validation_cursor:next_validation
            ] = cp[
                is_validation
            ]

            validation_cursor = (
                next_validation
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            y,
            expected_cp,
            is_train,
            is_validation,
        )

        gc.collect()


    print(
        f"[OK] day {day_index:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if expected_next_clean_position != N_DEVELOPMENT:
    raise RuntimeError(
        "Development length mismatch."
    )


if train_cursor != TRAIN_ROWS:
    raise RuntimeError(
        "\nTraining count mismatch.\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:
    raise RuntimeError(
        "\nValidation count mismatch.\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN[
        "attack"
    ]
):
    raise RuntimeError(
        "Training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN[
        "benign"
    ]
):
    raise RuntimeError(
        "Training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION[
        "attack"
    ]
):
    raise RuntimeError(
        "Validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION[
        "benign"
    ]
):
    raise RuntimeError(
        "Validation benign count mismatch."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):
    raise RuntimeError(
        "Validation clean_position order changed."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):
    raise RuntimeError(
        "Validation positions disagree with frozen bitset."
    )


print()
print("[OK] Frozen RANDOM_NATURAL matrices materialized.")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)
print(
    "     features   :",
    N_FEATURES
)
print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({
    "status": "MATERIALIZED",
    "materialization_seconds": materialize_seconds,
    "train_rows": TRAIN_ROWS,
    "validation_rows": VALIDATION_ROWS,
    "feature_count": N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


EXPECTED_XGB_PARAMS = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:
    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:
    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


t0 = time.perf_counter()


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - t0
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({
    "status": "LIGHTGBM_COMPLETED",
    "lightgbm_completed_fits": 1,
    "models_completed_total_this_cell": 1,
    "lightgbm_fit_seconds": lgbm_seconds,
    "lightgbm_model_sha256": LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)
print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model
gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


t0 = time.perf_counter()


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - t0
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({
    "status": "XGBOOST_COMPLETED",
    "xgboost_completed_fits": 1,
    "models_completed_total_this_cell": 2,
    "xgboost_fit_seconds": xgb_seconds,
    "xgboost_model_sha256": XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)
print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model
gc.collect()


# =============================================================================
# 19. ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5 * lgbm_probability
    +
    0.5 * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):
    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 20. METRICS
# =============================================================================

y_val = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_val.mean()
)


pr_auc = float(
    average_precision_score(
        y_val,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_val,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_val,
        predicted,
    )
)

precision = float(
    precision_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

fpr = float(
    fp / (fp + tn)
)

fnr = float(
    fn / (fn + tp)
)


# =============================================================================
# 21. FROZEN FULL RANDOM REFERENCE
# =============================================================================

full_random = json.loads(
    STAGE22_FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_random[
    "cell"
] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Unexpected Stage22R FULL random reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 22. RANDOM PENALTIES
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)

roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)

f1_removal_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)

recall_removal_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)

fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 23. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=y_val,

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=ensemble_probability,
)


VALIDATION_PROBABILITY_SHA = sha256_file(
    VALIDATION_PROBABILITY_PATH
)


# =============================================================================
# 24. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_FWD_SEG_SIZE_MIN_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {
        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {
        "stage":
            "Stage23-1F",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "cell": {
        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {
        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {
            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {
            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {
            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {
            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {
        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {
        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {
        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {
        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty,

        "recall_at_0_50":
            recall_removal_penalty,

        "fpr_at_0_50":
            fpr_change,
    },

    "shortcut_interaction": {
        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {
        "validation_probabilities": {
            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {
            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {
            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {
        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {
        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            12,

        "stage23_total_model_fits_after_cell":
            14,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1G before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 25. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 26. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({
    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        14,

    "stage23_metrics_calculated":
        True,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 27. REMOVE TRANSIENT MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():
        path.unlink()


try:
    RUNTIME_DIR.rmdir()
except OSError:
    pass


# =============================================================================
# 28. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1G COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen ablation:")
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  retained      :",
    N_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 14 / 50")


print()
print("=" * 96)
print("RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence                 : {attack_prevalence:.12f}"
)
print(
    f"NO_FWD_SEG_SIZE_MIN PR-AUC        : {pr_auc:.12f}"
)
print(
    f"FULL PR-AUC                       : {full_pr_auc:.12f}"
)
print(
    f"PR-AUC removal penalty            : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"NO_FWD_SEG_SIZE_MIN ROC-AUC       : {roc_auc:.12f}"
)
print(
    f"FULL ROC-AUC                      : {full_roc_auc:.12f}"
)
print(
    f"ROC-AUC removal penalty           : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence               : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 14 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Shortcut interaction:")
print(
    "  PENDING matched NO_FWD_SEG_SIZE_MIN chronological cell"
)


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1G."
)

print("=" * 96)

STAGE23-1G — NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL

[OK] branch         : main
[OK] HEAD           : f8cea7af7f239e254e427663a2d963483d0047c4
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1F tag : f8cea7af7f239e254e427663a2d963483d0047c4
[OK] worktree       : CLEAN
[OK] predecessor governance: 12 / 50 fits sealed
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_FWD_SEG_SIZE_MIN
  split         : RANDOM_NATURAL
  retained      : 69
  removed       : ['Fwd Seg Size Min']
  semantic label: minimum_forward_segment_ablation

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] RANDOM_NATURAL membership:
     train      : 11,529,922
     validation : 2,882,481

MATERIALIZATION PLAN

Train rows      : 11,529,922
Validation rows : 2,882,481
Features        : 69
Matrix storage  : 7.409 GiB
Work

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 385.677 s
     model SHA256: 3c4270af0465133c233527e00880a12a5ab141f2106d597d0db7f8aa89f0de5f

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 109.031 s
     model SHA256: 02d64109bea0db48979a81b7b24b34164418306bddbaf39b9119ca4aa69fcd62

STAGE23-1G COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_FWD_SEG_SIZE_MIN
  split  : RANDOM_NATURAL

Frozen ablation:
  removed       : ['Fwd Seg Size Min']
  retained      : 69
  semantic label: minimum_forward_segment_ablation

Data:
  train      : 11,529,922
  validation : 2,882,481

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 14 / 50

RANKING METRICS

Attack prevalence                 : 0.136847389454
NO_FWD_SEG_SIZE_MIN PR-AUC        : 0.995576136093
FULL PR-AUC                       : 0.995590041899
PR-AUC removal penalty            : +0.000013905806

NO_FWD_SEG_SIZE_MIN ROC-AUC       : 0.998608787410
FULL ROC-AUC                      : 0.998624564774
ROC-AUC removal penalt

In [16]:
# =============================================================================
# STAGE23-1G — SEAL + COMMIT + TAG + PUSH
# NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1g_no_fwd_seg_size_min_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_fwd_seg_size_min"
)

EXPECTED_PARENT = (
    "f8cea7af7f239e254e427663a2d963483d0047c4"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1f-no-init-fwd-win-byts-chronological-natural-v1"
)

RESULT_TAG = (
    "stage23-1g-no-fwd-seg-size-min-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1G: freeze NO_FWD_SEG_SIZE_MIN random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1G frozen NO_FWD_SEG_SIZE_MIN RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1g_no_fwd_seg_size_min_random_natural_result.json":
        "c661853f87801897116cf62dd9623011cf4548e8be7e4fa81e4d89d84252599a",

    "no_fwd_seg_size_min_random_natural_lightgbm_model.txt":
        "3c4270af0465133c233527e00880a12a5ab141f2106d597d0db7f8aa89f0de5f",

    "no_fwd_seg_size_min_random_natural_xgboost_model.json":
        "02d64109bea0db48979a81b7b24b34164418306bddbaf39b9119ca4aa69fcd62",

    "no_fwd_seg_size_min_random_natural_validation_probabilities.npz":
        "aa2a52e49041e716d1670c19383b2331796a87b56ed6868708a3aed67a30281e",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "17760d2676c495b759ecdf80f917c0b00b6f6d70c40e2e91a4c3c64f3745b7ce"
)


# =============================================================================
# 2. EXACT OBSERVED SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.995576136093
EXPECTED_ROC_AUC = 0.998608787410

EXPECTED_PR_PENALTY = 0.000013905806
EXPECTED_ROC_PENALTY = 0.000015777364


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1G — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1G seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1F predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1G seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1G target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1G source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1G result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1F tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1G source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1G CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1G CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1G artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1G ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC RESULT CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1g_no_fwd_seg_size_min_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1G":
    raise RuntimeError(
        "Unexpected Stage23-1G stage identifier."
    )


if result["cell"]["subset"] != "NO_FWD_SEG_SIZE_MIN":
    raise RuntimeError(
        "Stage23-1G subset changed."
    )


if result["cell"]["split"] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Stage23-1G split changed."
    )


if result["cell"]["removed_features"] != [
    "Fwd Seg Size Min"
]:
    raise RuntimeError(
        "Frozen removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 69:
    raise RuntimeError(
        "Frozen Stage23-1G feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "minimum_forward_segment_ablation":
    raise RuntimeError(
        "Frozen semantic label changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1G predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1G fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 12:
    raise RuntimeError(
        "Unexpected pre-cell Stage23 fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 14:
    raise RuntimeError(
        "Unexpected post-cell Stage23 fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


if result[
    "shortcut_interaction"
][
    "status"
] != "PENDING_MATCHED_CHRONOLOGICAL_CELL":
    raise RuntimeError(
        "Stage23-1G interaction status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result[
        "ranking_metrics"
    ][
        "pr_auc"
    ]
)

actual_roc_auc = float(
    result[
        "ranking_metrics"
    ][
        "roc_auc"
    ]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1G output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty : {actual_roc_penalty:+.12f}"
)

print(
    "     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1G",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_FWD_SEG_SIZE_MIN",

        "split":
            "RANDOM_NATURAL",

        "removed_features": [
            "Fwd Seg Size Min"
        ],

        "feature_count":
            69,

        "semantic_label":
            "minimum_forward_segment_ablation",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        14,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1H NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1G — NO_FWD_SEG_SIZE_MIN × RANDOM_NATURAL

This directory freezes the random-split single-feature ablation of
`Fwd Seg Size Min`.

## Frozen cell

- Subset: `NO_FWD_SEG_SIZE_MIN`
- Split: `RANDOM_NATURAL`
- Removed feature: `Fwd Seg Size Min`
- Semantic label: `minimum_forward_segment_ablation`
- Retained features: 69
- New boosted-model fits: 2
- Total Stage23 fits after cell: 14 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − ablated PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − ablated ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Fixed threshold 0.50

- Accuracy: {result["fixed_threshold_0_50"]["accuracy"]:.12f}
- Precision: {result["fixed_threshold_0_50"]["precision"]:.12f}
- Recall: {result["fixed_threshold_0_50"]["recall"]:.12f}
- F1: {result["fixed_threshold_0_50"]["f1"]:.12f}
- FPR: {result["fixed_threshold_0_50"]["fpr"]:.12f}
- FNR: {result["fixed_threshold_0_50"]["fnr"]:.12f}

## Interpretation status

The matched chronological ablation has not yet been executed.

Therefore the preregistered split × ablation interaction remains pending.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1G
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1G ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1G RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():
    raise RuntimeError(
        "Repository dirty after Stage23-1G commit."
    )


print()
print("Stage23-1G commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1G tag verification failed."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1G")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":
        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1G annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1G tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1G push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1G SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1F parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1G commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_FWD_SEG_SIZE_MIN RANDOM RESULT")
print("=" * 96)

print()
print(
    f"PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("Shortcut interaction:")
print("  PENDING matched chronological cell")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 14 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1G COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1H — NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL"
)

print("=" * 96)

STAGE23-1G — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : f8cea7af7f239e254e427663a2d963483d0047c4
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1F tag: f8cea7af7f239e254e427663a2d963483d0047c4
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  17760d2676c495b759ecdf80f917c0b00b6f6d70c40e2e91a4c3c64f3745b7ce

VERIFYING STAGE23-1G CORE ARTIFACTS

[EXACT] stage23_1g_no_fwd_seg_size_min_random_natural_result.json
[EXACT] no_fwd_seg_size_min_random_natural_lightgbm_model.txt
[EXACT] no_fwd_seg_size_min_random_natural_xgboost_model.json
[EXACT] no_fwd_seg_size_min_random_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC      : 0.995576136093
     ROC-AUC     : 0.998608787410
     PR penalty  : +0.000013905806
     ROC penalty : +0.000015777364
     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL

VERIFYING REPOSITORY COPIES

[EXACT] stage23_1g_no_fwd_seg_s

In [17]:
# =============================================================================
# STAGE23-1H
# NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# Frozen ablation:
#   remove exactly:
#       Fwd Seg Size Min
#
# REQUIRED PREDECESSOR:
#   Stage23-1G
#   e24985f3b51cf436272b965d89d1a123a0cef239
#
# THIS CELL:
#   - 1 LightGBM fit
#   - 1 XGBoost fit
#   - CHRONOLOGICAL_NATURAL
#   - reuses frozen FULL chronological result
#   - reuses sealed Stage23-1G random result
#   - calculates fourth primary split × ablation interaction
#
# FIT COUNT:
#   before : 14 / 50
#   after  : 16 / 50
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   raw Mar1 / Mar2
#
# FAIL CLOSED:
#   If a fit completes and anything later fails, DO NOT blindly rerun.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1H"

TARGET_SUBSET = "NO_FWD_SEG_SIZE_MIN"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_COMMIT = (
    "e24985f3b51cf436272b965d89d1a123a0cef239"
)

PREDECESSOR_TAG = (
    "stage23-1g-no-fwd-seg-size-min-random-natural-v1"
)

PREDECESSOR_RESULT_SHA256 = (
    "c661853f87801897116cf62dd9623011cf4548e8be7e4fa81e4d89d84252599a"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_fwd_seg_size_min"
)

PREDECESSOR_RESULT = (
    PREDECESSOR_DIR
    / "stage23_1g_no_fwd_seg_size_min_random_natural_result.json"
)

PREDECESSOR_SEAL = (
    PREDECESSOR_DIR
    / "seal_receipt.json"
)

STAGE22_FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1h_no_fwd_seg_size_min_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1h_no_fwd_seg_size_min_chronological_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    state = None

    if EXECUTION_STATE_PATH.exists():

        try:
            state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1H OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"{json.dumps(state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "predecessor_commit":
        PREDECESSOR_COMMIT,

    "predecessor_tag":
        PREDECESSOR_TAG,

    "stage23_total_model_fits_before_cell":
        14,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. REPOSITORY / TAG VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1H — NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1H MUST BEGIN FROM SEALED STAGE23-1G.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1G result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1H:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1G tag :", predecessor_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR SEAL + RANDOM RESULT
# =============================================================================

if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1G seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 14:

    raise RuntimeError(
        "Expected 14 sealed Stage23 fits before Stage23-1H."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1H NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1G seal does not authorize Stage23-1H."
    )


if not PREDECESSOR_RESULT.exists():

    raise RuntimeError(
        "Stage23-1G sealed result missing."
    )


actual_predecessor_sha = (
    sha256_file(
        PREDECESSOR_RESULT
    )
)


if actual_predecessor_sha != PREDECESSOR_RESULT_SHA256:

    raise RuntimeError(
        "\nSEALED STAGE23-1G RESULT CHANGED\n"
        f"expected: {PREDECESSOR_RESULT_SHA256}\n"
        f"actual  : {actual_predecessor_sha}"
    )


random_result = json.loads(
    PREDECESSOR_RESULT.read_text(
        encoding="utf-8"
    )
)


if random_result[
    "cell"
][
    "subset"
] != TARGET_SUBSET:

    raise RuntimeError(
        "Stage23-1G subset mismatch."
    )


if random_result[
    "cell"
][
    "split"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Stage23-1G split mismatch."
    )


print(
    "[OK] predecessor governance: 14 / 50 fits sealed"
)
print(
    "[OK] sealed Stage23-1G random result exact"
)


# =============================================================================
# 6. VERIFY STAGE23-0 BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


actual_protocol_manifest_sha = sha256_file(
    PROTOCOL_CHECKSUMS
)


if actual_protocol_manifest_sha != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "\nStage23-0 checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_protocol_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing frozen protocol artifact: {filename}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


if REMOVED_FEATURES != [
    "Fwd Seg Size Min"
]:

    raise RuntimeError(
        "Frozen NO_FWD_SEG_SIZE_MIN definition changed."
    )


if len(FEATURES) != 69:

    raise RuntimeError(
        f"Expected 69 retained features; found {len(FEATURES)}"
    )


if "Fwd Seg Size Min" in FEATURES:

    raise RuntimeError(
        "Fwd Seg Size Min remains in ablated feature list."
    )


if subset_spec[
    "semantic_label"
] != "minimum_forward_segment_ablation":

    raise RuntimeError(
        "Frozen semantic label changed."
    )


print()
print("Frozen target:")
print(
    "  subset        :",
    TARGET_SUBSET
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  retained      :",
    len(FEATURES)
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "shap": "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; "
        "XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 10. FROZEN CHRONOLOGICAL MEMBERSHIP
# =============================================================================

chrono_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = (
    chrono_spec[
        "train"
    ]
)

EXPECTED_VALIDATION = (
    chrono_spec[
        "validation"
    ]
)


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)


if TRAIN_ROWS != 13_818_623:

    raise RuntimeError(
        "Chronological training count changed."
    )


if VALIDATION_ROWS != 593_780:

    raise RuntimeError(
        "Chronological validation count changed."
    )


print()
print("[OK] CHRONOLOGICAL_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "     train days : 0..6"
)
print(
    "     val day    : 7 / 02-28-2018"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)
print()

print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)

print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)

print(
    "Features        :",
    N_FEATURES
)

print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)

print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "Insufficient /kaggle/working disk."
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. MATERIALIZE CHRONOLOGICAL MATRICES
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "day_id",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING CHRONOLOGICAL NO_FWD_SEG_SIZE_MIN MATRICES")
print("=" * 96)
print()


for expected_day_id, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        day_ids = batch.column(
            batch.schema.get_field_index(
                "day_id"
            )
        ).to_numpy(
            zero_copy_only=False
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        unique_days = np.unique(
            day_ids
        )


        if not (
            len(unique_days) == 1
            and int(unique_days[0]) == expected_day_id
        ):

            raise RuntimeError(
                f"day_id mismatch in {filename}: {unique_days}"
            )


        if expected_day_id <= 6:

            expected_cp_start = (
                train_cursor
            )

        else:

            expected_cp_start = (
                TRAIN_ROWS
                + validation_cursor
            )


        expected_cp = np.arange(
            expected_cp_start,
            expected_cp_start
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file     : {filename}\n"
                f"expected : {expected_cp_start:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        if expected_day_id <= 6:

            next_cursor = (
                train_cursor
                + batch_rows
            )

            if next_cursor > TRAIN_ROWS:

                raise RuntimeError(
                    "Training memmap overflow."
                )


            X_train[
                train_cursor:next_cursor
            ] = X_batch

            y_train[
                train_cursor:next_cursor
            ] = y

            train_cursor = (
                next_cursor
            )


        else:

            next_cursor = (
                validation_cursor
                + batch_rows
            )

            if next_cursor > VALIDATION_ROWS:

                raise RuntimeError(
                    "Validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_cursor
            ] = X_batch

            y_validation[
                validation_cursor:next_cursor
            ] = y

            validation_positions[
                validation_cursor:next_cursor
            ] = cp

            validation_cursor = (
                next_cursor
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            day_ids,
            y,
            expected_cp,
        )

        gc.collect()


    print(
        f"[OK] day {expected_day_id:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN["attack"]
):

    raise RuntimeError(
        "Training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN["benign"]
):

    raise RuntimeError(
        "Training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION["attack"]
):

    raise RuntimeError(
        "Validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION["benign"]
):

    raise RuntimeError(
        "Validation benign count mismatch."
    )


expected_validation_positions = np.arange(
    TRAIN_ROWS,
    TRAIN_ROWS
    + VALIDATION_ROWS,
    dtype=np.int64,
)


if not np.array_equal(
    validation_positions,
    expected_validation_positions,
):

    raise RuntimeError(
        "Chronological validation membership changed."
    )


print()
print("[OK] Frozen CHRONOLOGICAL_NATURAL matrices materialized.")

print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)

print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)

print(
    "     features   :",
    N_FEATURES
)

print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


EXPECTED_XGB_PARAMS = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


t0 = time.perf_counter()


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - t0
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model
gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


t0 = time.perf_counter()


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - t0
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model
gc.collect()


# =============================================================================
# 19. FROZEN ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 20. METRICS
# =============================================================================

y_val = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_val.mean()
)


pr_auc = float(
    average_precision_score(
        y_val,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_val,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_val,
        predicted,
    )
)

precision = float(
    precision_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_val,
        predicted,
        zero_division=0,
    )
)

fpr = float(
    fp / (fp + tn)
)

fnr = float(
    fn / (fn + tp)
)


# =============================================================================
# 21. FROZEN FULL CHRONOLOGICAL REFERENCE
# =============================================================================

full_chrono = json.loads(
    STAGE22_FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_chrono[
    "cell"
] != "CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected Stage22R FULL chronological reference."
    )


full_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 22. CHRONOLOGICAL PENALTIES
# =============================================================================

chrono_pr_penalty = float(
    full_pr_auc
    - pr_auc
)

chrono_roc_penalty = float(
    full_roc_auc
    - roc_auc
)

chrono_f1_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)

chrono_recall_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)

chrono_fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 23. SEALED RANDOM PENALTIES
# =============================================================================

random_pr_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

random_roc_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)

random_f1_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "f1_at_0_50"
    ]
)

random_recall_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "recall_at_0_50"
    ]
)

random_fpr_change = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "fpr_at_0_50"
    ]
)


# =============================================================================
# 24. FOURTH PRIMARY SPLIT × ABLATION INTERACTION
# =============================================================================

interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)

interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)

interaction_f1 = float(
    random_f1_penalty
    - chrono_f1_penalty
)

interaction_recall = float(
    random_recall_penalty
    - chrono_recall_penalty
)

interaction_fpr = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 25. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_fwd_seg_size_min_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=y_val,

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = sha256_file(
    VALIDATION_PROBABILITY_PATH
)


# =============================================================================
# 26. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_FWD_SEG_SIZE_MIN_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {

        "stage":
            "Stage23-1G",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,

        "result_sha256":
            PREDECESSOR_RESULT_SHA256,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {

        "source":
            "Sealed Stage23-1G",

        "result_sha256":
            PREDECESSOR_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {

        "definition":
            "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1,

        "recall_at_0_50":
            interaction_recall,

        "fpr_at_0_50":
            interaction_fpr,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            14,

        "stage23_total_model_fits_after_cell":
            16,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1H before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 27. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 28. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        16,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 29. REMOVE TRANSIENT MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():
        path.unlink()


try:
    RUNTIME_DIR.rmdir()
except OSError:
    pass


# =============================================================================
# 30. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1H COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen ablation:")
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  retained      :",
    N_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 16 / 50")


print()
print("=" * 96)
print("CHRONOLOGICAL RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence                 : {attack_prevalence:.12f}"
)

print(
    f"NO_FWD_SEG_SIZE_MIN PR-AUC        : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                       : {full_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty          : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"NO_FWD_SEG_SIZE_MIN ROC-AUC       : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                      : {full_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty         : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence               : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("NO_FWD_SEG_SIZE_MIN SPLIT × ABLATION INTERACTION")
print("=" * 96)

print()
print("PR-AUC:")

print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")

print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary:")

print(
    f"  F1 interaction     : {interaction_f1:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 16 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1H."
)

print("=" * 96)

STAGE23-1H — NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL

[OK] branch         : main
[OK] HEAD           : e24985f3b51cf436272b965d89d1a123a0cef239
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1G tag : e24985f3b51cf436272b965d89d1a123a0cef239
[OK] worktree       : CLEAN
[OK] predecessor governance: 14 / 50 fits sealed
[OK] sealed Stage23-1G random result exact
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_FWD_SEG_SIZE_MIN
  split         : CHRONOLOGICAL_NATURAL
  retained      : 69
  removed       : ['Fwd Seg Size Min']
  semantic label: minimum_forward_segment_ablation

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] CHRONOLOGICAL_NATURAL membership:
     train      : 13,818,623
     validation : 593,780
     train days : 0..6
     val day    : 7 / 02-28-2018

MATERIALIZATIO

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 474.601 s
     model SHA256: fa801c99a29d71f0a8020916343731a200c78a9f7783824c33549fcaa7f8cadc

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 142.899 s
     model SHA256: 19b4bc394fb5f8c39f614a50decadb8567906a235a11afc8fdaf1d2887cba46f

STAGE23-1H COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_FWD_SEG_SIZE_MIN
  split  : CHRONOLOGICAL_NATURAL

Frozen ablation:
  removed       : ['Fwd Seg Size Min']
  retained      : 69
  semantic label: minimum_forward_segment_ablation

Data:
  train      : 13,818,623
  validation : 593,780

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 16 / 50

CHRONOLOGICAL RANKING METRICS

Attack prevalence                 : 0.104846912998
NO_FWD_SEG_SIZE_MIN PR-AUC        : 0.105454756063
FULL PR-AUC                       : 0.106215155134
Chronological PR penalty          : +0.000760399071

NO_FWD_SEG_SIZE_MIN ROC-AUC       : 0.511682110206
FULL ROC-AUC                      : 0.514918426394
Chr

In [18]:
# =============================================================================
# STAGE23-1H — SEAL + COMMIT + TAG + PUSH
# NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1h_no_fwd_seg_size_min_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_fwd_seg_size_min"
)

EXPECTED_PARENT = (
    "e24985f3b51cf436272b965d89d1a123a0cef239"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1g-no-fwd-seg-size-min-random-natural-v1"
)

RESULT_TAG = (
    "stage23-1h-no-fwd-seg-size-min-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1H: freeze NO_FWD_SEG_SIZE_MIN chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1H frozen NO_FWD_SEG_SIZE_MIN CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1h_no_fwd_seg_size_min_chronological_natural_result.json":
        "3e68cf65d666a7925068dc0b5918055a0a85df20f3f61bef1600d30266df2e06",

    "no_fwd_seg_size_min_chronological_natural_lightgbm_model.txt":
        "fa801c99a29d71f0a8020916343731a200c78a9f7783824c33549fcaa7f8cadc",

    "no_fwd_seg_size_min_chronological_natural_xgboost_model.json":
        "19b4bc394fb5f8c39f614a50decadb8567906a235a11afc8fdaf1d2887cba46f",

    "no_fwd_seg_size_min_chronological_natural_validation_probabilities.npz":
        "566c5b131d9024601616af2b5055acc48eb080018287bafb4c479bfeaab6a041",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "b69a842eb9a5c30c9e66e57d6e5c2aed860032037452c94db0e796c4de1f8370"
)


# =============================================================================
# 2. EXACT OBSERVED VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.105454756063
EXPECTED_ROC_AUC = 0.511682110206

EXPECTED_PR_PENALTY = 0.000760399071
EXPECTED_ROC_PENALTY = 0.003236316188

EXPECTED_PR_INTERACTION = -0.000746493265
EXPECTED_ROC_INTERACTION = -0.003220538824

EXPECTED_F1_INTERACTION = 0.000071623304
EXPECTED_RECALL_INTERACTION = 0.000021984972
EXPECTED_FPR_INTERACTION = 0.000018557332


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1H — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1H seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1G predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1H seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1H target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1H source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1H result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1G tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1H source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1H CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1H CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1H artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1H ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY SCIENTIFIC CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1h_no_fwd_seg_size_min_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1H":
    raise RuntimeError(
        "Unexpected Stage23-1H stage identifier."
    )


if result["cell"]["subset"] != "NO_FWD_SEG_SIZE_MIN":
    raise RuntimeError(
        "Stage23-1H subset changed."
    )


if result["cell"]["split"] != "CHRONOLOGICAL_NATURAL":
    raise RuntimeError(
        "Stage23-1H split changed."
    )


if result["cell"]["removed_features"] != [
    "Fwd Seg Size Min"
]:
    raise RuntimeError(
        "Frozen removed-feature definition changed."
    )


if result["cell"]["feature_count"] != 69:
    raise RuntimeError(
        "Frozen Stage23-1H feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "minimum_forward_segment_ablation":
    raise RuntimeError(
        "Frozen semantic label changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1H predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1H fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 14:
    raise RuntimeError(
        "Unexpected pre-cell Stage23 fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 16:
    raise RuntimeError(
        "Unexpected post-cell Stage23 fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


interaction = result[
    "shortcut_interaction"
]


if interaction[
    "confidence_interval_status"
] != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS":
    raise RuntimeError(
        "Stage23-1H uncertainty status changed."
    )


if interaction[
    "interpretation_status"
] != "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE":
    raise RuntimeError(
        "Stage23-1H interpretation status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result[
        "ranking_metrics"
    ][
        "pr_auc"
    ]
)

actual_roc_auc = float(
    result[
        "ranking_metrics"
    ][
        "roc_auc"
    ]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)

actual_pr_interaction = float(
    interaction[
        "pr_auc"
    ]
)

actual_roc_interaction = float(
    interaction[
        "roc_auc"
    ]
)

actual_f1_interaction = float(
    interaction[
        "f1_at_0_50"
    ]
)

actual_recall_interaction = float(
    interaction[
        "recall_at_0_50"
    ]
)

actual_fpr_interaction = float(
    interaction[
        "fpr_at_0_50"
    ]
)


checks = [
    (
        "PR-AUC",
        actual_pr_auc,
        EXPECTED_PR_AUC,
    ),
    (
        "ROC-AUC",
        actual_roc_auc,
        EXPECTED_ROC_AUC,
    ),
    (
        "PR-AUC penalty",
        actual_pr_penalty,
        EXPECTED_PR_PENALTY,
    ),
    (
        "ROC-AUC penalty",
        actual_roc_penalty,
        EXPECTED_ROC_PENALTY,
    ),
    (
        "PR-AUC interaction",
        actual_pr_interaction,
        EXPECTED_PR_INTERACTION,
    ),
    (
        "ROC-AUC interaction",
        actual_roc_interaction,
        EXPECTED_ROC_INTERACTION,
    ),
    (
        "F1 interaction",
        actual_f1_interaction,
        EXPECTED_F1_INTERACTION,
    ),
    (
        "Recall interaction",
        actual_recall_interaction,
        EXPECTED_RECALL_INTERACTION,
    ),
    (
        "FPR interaction",
        actual_fpr_interaction,
        EXPECTED_FPR_INTERACTION,
    ),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1H output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC          : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC         : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty      : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty     : {actual_roc_penalty:+.12f}"
)

print(
    f"     PR interaction  : {actual_pr_interaction:+.12f}"
)

print(
    f"     ROC interaction : {actual_roc_interaction:+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1H",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_FWD_SEG_SIZE_MIN",

        "split":
            "CHRONOLOGICAL_NATURAL",

        "removed_features": [
            "Fwd Seg Size Min"
        ],

        "feature_count":
            69,

        "semantic_label":
            "minimum_forward_segment_ablation",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        16,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction": {

        "pr_auc":
            actual_pr_interaction,

        "roc_auc":
            actual_roc_interaction,

        "f1_at_0_50":
            actual_f1_interaction,

        "recall_at_0_50":
            actual_recall_interaction,

        "fpr_at_0_50":
            actual_fpr_interaction,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1I NO_SUSPICIOUS_GROUP × RANDOM_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1H — NO_FWD_SEG_SIZE_MIN × CHRONOLOGICAL_NATURAL

This directory freezes the chronological partner of the
`NO_FWD_SEG_SIZE_MIN` primary ablation.

## Frozen cell

- Removed feature: `Fwd Seg Size Min`
- Semantic label: `minimum_forward_segment_ablation`
- Retained features: 69
- Split: `CHRONOLOGICAL_NATURAL`
- New boosted-model fits: 2
- Total Stage23 fits after cell: 16 / 50
- Threshold optimization: none
- Per-subset tuning: none
- Rebalancing: none

## Chronological ranking

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − ablated PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − ablated ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Split × ablation interaction

Frozen definition:

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

Point estimates:

- PR-AUC interaction: {actual_pr_interaction:+.12f}
- ROC-AUC interaction: {actual_roc_interaction:+.12f}
- F1@0.50 interaction: {actual_f1_interaction:+.12f}
- Recall@0.50 interaction: {actual_recall_interaction:+.12f}
- FPR@0.50 interaction: {actual_fpr_interaction:+.12f}

These remain point estimates only. The preregistered paired bootstrap
uncertainty analysis is required before inferential interpretation.

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1H
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1H ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1H RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():
    raise RuntimeError(
        "Repository dirty after Stage23-1H commit."
    )


print()
print("Stage23-1H commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1H tag verification failed."
    )


# =============================================================================
# 17. PUSH
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1H")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":
        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1H annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1H tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1H push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1H SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1G parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1H commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_FWD_SEG_SIZE_MIN PAIR")
print("=" * 96)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : "
    f'{result["matched_random_reference"]["pr_auc_removal_penalty"]:+.12f}'
)
print(
    f"  ROC penalty : "
    f'{result["matched_random_reference"]["roc_auc_removal_penalty"]:+.12f}'
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR penalty  : {actual_pr_penalty:+.12f}"
)
print(
    f"  ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {actual_pr_interaction:+.12f}"
)
print(
    f"  ROC-AUC : {actual_roc_interaction:+.12f}"
)

print()
print("Supplementary interactions:")
print(
    f"  F1@0.50     : {actual_f1_interaction:+.12f}"
)
print(
    f"  Recall@0.50 : {actual_recall_interaction:+.12f}"
)
print(
    f"  FPR@0.50    : {actual_fpr_interaction:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 16 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1H COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1I — NO_SUSPICIOUS_GROUP × RANDOM_NATURAL"
)

print("=" * 96)

STAGE23-1H — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : e24985f3b51cf436272b965d89d1a123a0cef239
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1G tag: e24985f3b51cf436272b965d89d1a123a0cef239
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  b69a842eb9a5c30c9e66e57d6e5c2aed860032037452c94db0e796c4de1f8370

VERIFYING STAGE23-1H CORE ARTIFACTS

[EXACT] stage23_1h_no_fwd_seg_size_min_chronological_natural_result.json
[EXACT] no_fwd_seg_size_min_chronological_natural_lightgbm_model.txt
[EXACT] no_fwd_seg_size_min_chronological_natural_xgboost_model.json
[EXACT] no_fwd_seg_size_min_chronological_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC          : 0.105454756063
     ROC-AUC         : 0.511682110206
     PR penalty      : +0.000760399071
     ROC penalty     : +0.003236316188
     PR interaction  : -0.000746493265
     ROC interaction : -0.00322

In [19]:
# =============================================================================
# STAGE23-1I
# NO_SUSPICIOUS_GROUP × RANDOM_NATURAL
# =============================================================================
#
# Frozen ablation removes EXACTLY:
#   - Dst Port
#   - Init Fwd Win Byts
#   - Fwd Seg Size Min
#
# Retained features: 67
#
# Semantic label:
#   joint_shortcut_prone_group_ablation
#
# REQUIRED PREDECESSOR:
#   Stage23-1H
#   3cc239339ad24b697d87ad9d9cc9c29cdfb55979
#
# FIT COUNT:
#   before : 16 / 50
#   this   :  2
#   after  : 18 / 50
#
# NO:
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#   raw Mar1
#   raw Mar2
#
# FAIL CLOSED:
# If either model completes and anything later fails, DO NOT blindly rerun.
# Preserve OUTPUT_DIR and inspect execution_state.json.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np
import pyarrow.parquet as pq

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1I"

TARGET_SUBSET = "NO_SUSPICIOUS_GROUP"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_COMMIT = (
    "3cc239339ad24b697d87ad9d9cc9c29cdfb55979"
)

PREDECESSOR_TAG = (
    "stage23-1h-no-fwd-seg-size-min-chronological-natural-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_fwd_seg_size_min"
    / "seal_receipt.json"
)

MEMBERSHIP_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_BITSET = (
    MEMBERSHIP_DIR
    / "random_validation.packbits"
)

STAGE22_FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1i_no_suspicious_group_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1i_no_suspicious_group_random_natural_result.json"
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 3. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1I OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(existing_state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "protocol_commit":
        PROTOCOL_COMMIT,

    "protocol_tag":
        PROTOCOL_TAG,

    "predecessor_commit":
        PREDECESSOR_COMMIT,

    "predecessor_tag":
        PREDECESSOR_TAG,

    "stage23_total_model_fits_before_cell":
        16,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 4. GIT / PROTOCOL / PREDECESSOR VERIFICATION
# =============================================================================

print("=" * 96)
print("STAGE23-1I — NO_SUSPICIOUS_GROUP × RANDOM_NATURAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1I MUST BEGIN FROM SEALED STAGE23-1H.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1H result tag verification failed."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1I:\n"
        + status
    )


print("[OK] branch         :", branch)
print("[OK] HEAD           :", head)
print("[OK] Stage23-0 tag  :", protocol_tag_commit)
print("[OK] Stage23-1H tag :", predecessor_tag_commit)
print("[OK] worktree       : CLEAN")


# =============================================================================
# 5. VERIFY PREDECESSOR GOVERNANCE
# =============================================================================

if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1H seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 16:

    raise RuntimeError(
        "Expected exactly 16 sealed Stage23 fits before Stage23-1I."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1I NO_SUSPICIOUS_GROUP × RANDOM_NATURAL":

    raise RuntimeError(
        "\nStage23-1H seal does not authorize Stage23-1I.\n"
        f"actual authorization: "
        f"{predecessor_seal.get('next_authorized_model_cell')}"
    )


print(
    "[OK] predecessor governance: 16 / 50 fits sealed"
)


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


actual_protocol_manifest_sha = (
    sha256_file(
        PROTOCOL_CHECKSUMS
    )
)


if actual_protocol_manifest_sha != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "\nStage23-0 checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {actual_protocol_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Frozen protocol artifact missing: {filename}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts: "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = subset_spec[
    "features"
]

REMOVED_FEATURES = subset_spec[
    "removed"
]


EXPECTED_REMOVED = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nFrozen NO_SUSPICIOUS_GROUP definition changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if len(FEATURES) != 67:

    raise RuntimeError(
        f"Expected 67 retained features; found {len(FEATURES)}"
    )


for removed_feature in EXPECTED_REMOVED:

    if removed_feature in FEATURES:

        raise RuntimeError(
            f"Removed feature remains in ablation: {removed_feature}"
        )


if subset_spec[
    "semantic_label"
] != "joint_shortcut_prone_group_ablation":

    raise RuntimeError(
        "Frozen NO_SUSPICIOUS_GROUP semantic label changed."
    )


if subset_spec[
    "mode"
] != "remove":

    raise RuntimeError(
        "Frozen NO_SUSPICIOUS_GROUP mode changed."
    )


print()
print("Frozen target:")
print(
    "  subset        :",
    TARGET_SUBSET
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  feature count :",
    len(FEATURES)
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)


# =============================================================================
# 8. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] Frozen package versions verified.")


# =============================================================================
# 9. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; "
        "Stage23 XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 10. FROZEN RANDOM_NATURAL MEMBERSHIP
# =============================================================================

random_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


EXPECTED_TRAIN = (
    random_spec[
        "train"
    ]
)

EXPECTED_VALIDATION = (
    random_spec[
        "validation"
    ]
)


TRAIN_ROWS = int(
    EXPECTED_TRAIN[
        "rows"
    ]
)

VALIDATION_ROWS = int(
    EXPECTED_VALIDATION[
        "rows"
    ]
)

N_FEATURES = len(
    FEATURES
)

N_DEVELOPMENT = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)


if TRAIN_ROWS != 11_529_922:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL training count changed."
    )


if VALIDATION_ROWS != 2_882_481:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL validation count changed."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_BITSET,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random validation mask length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Frozen random validation population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL membership:")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 11. AUTHORIZED DEVELOPMENT CACHE ONLY
# =============================================================================

AUTHORIZED_CACHE_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    path = Path(path).resolve()

    text = str(path).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {path}\n"
                f"marker : {marker}"
            )

    return path


for filename in AUTHORIZED_CACHE_FILES:

    path = guard_path(
        CACHE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Authorized cache missing: {path}"
        )


# =============================================================================
# 12. MATERIALIZATION PLAN
# =============================================================================

matrix_bytes = (
    (
        TRAIN_ROWS
        + VALIDATION_ROWS
    )
    * N_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    matrix_bytes
    + 2 * 1024**3
)


print()
print("=" * 96)
print("MATERIALIZATION PLAN")
print("=" * 96)
print()

print(
    "Train rows      :",
    f"{TRAIN_ROWS:,}"
)

print(
    "Validation rows :",
    f"{VALIDATION_ROWS:,}"
)

print(
    "Features        :",
    N_FEATURES
)

print(
    "Matrix storage  :",
    f"{matrix_bytes / 1024**3:.3f} GiB"
)

print(
    "Working free    :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nInsufficient /kaggle/working disk.\n"
        f"minimum: {minimum_free / 1024**3:.3f} GiB\n"
        f"free   : {disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 13. CREATE MEMMAPS
# =============================================================================

X_train_path = (
    RUNTIME_DIR
    / "X_train_float64.dat"
)

y_train_path = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

X_validation_path = (
    RUNTIME_DIR
    / "X_validation_float64.dat"
)

y_validation_path = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

validation_positions_path = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_train_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    y_train_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


X_validation = np.memmap(
    X_validation_path,
    dtype=np.float64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_validation = np.memmap(
    y_validation_path,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    validation_positions_path,
    dtype=np.int64,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 14. MATERIALIZE EXACT RANDOM SPLIT
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "binary_label",
    ]
    + FEATURES
)


train_cursor = 0
validation_cursor = 0
expected_next_clean_position = 0

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 96)
print("MATERIALIZING RANDOM NO_SUSPICIOUS_GROUP MATRICES")
print("=" * 96)
print()


for day_index, filename in enumerate(
    AUTHORIZED_CACHE_FILES
):

    path = guard_path(
        CACHE_DIR
        / filename
    )

    parquet = pq.ParquetFile(
        path
    )

    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        batch_rows = (
            batch.num_rows
        )

        if batch_rows == 0:
            continue


        cp = batch.column(
            batch.schema.get_field_index(
                "clean_position"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.int64,
            copy=False,
        )


        y = batch.column(
            batch.schema.get_field_index(
                "binary_label"
            )
        ).to_numpy(
            zero_copy_only=False
        ).astype(
            np.uint8,
            copy=False,
        )


        expected_cp = np.arange(
            expected_next_clean_position,
            expected_next_clean_position
            + batch_rows,
            dtype=np.int64,
        )


        if not np.array_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCanonical clean_position mismatch.\n"
                f"file        : {filename}\n"
                f"expected    : {expected_next_clean_position:,}\n"
                f"actual first: {int(cp[0]):,}"
            )


        expected_next_clean_position += (
            batch_rows
        )


        is_validation = (
            random_validation_mask[
                cp
            ]
        )

        is_train = (
            ~is_validation
        )


        X_batch = np.column_stack([
            batch.column(
                batch.schema.get_field_index(
                    feature
                )
            ).to_numpy(
                zero_copy_only=False
            )
            for feature in FEATURES
        ]).astype(
            np.float64,
            copy=False,
        )


        n_train = int(
            is_train.sum()
        )

        n_validation = int(
            is_validation.sum()
        )


        if n_train:

            next_train = (
                train_cursor
                + n_train
            )

            if next_train > TRAIN_ROWS:

                raise RuntimeError(
                    "Random training memmap overflow."
                )


            X_train[
                train_cursor:next_train
            ] = X_batch[
                is_train
            ]

            y_train[
                train_cursor:next_train
            ] = y[
                is_train
            ]

            train_cursor = (
                next_train
            )


        if n_validation:

            next_validation = (
                validation_cursor
                + n_validation
            )

            if next_validation > VALIDATION_ROWS:

                raise RuntimeError(
                    "Random validation memmap overflow."
                )


            X_validation[
                validation_cursor:next_validation
            ] = X_batch[
                is_validation
            ]

            y_validation[
                validation_cursor:next_validation
            ] = y[
                is_validation
            ]

            validation_positions[
                validation_cursor:next_validation
            ] = cp[
                is_validation
            ]

            validation_cursor = (
                next_validation
            )


        file_rows += (
            batch_rows
        )


        del (
            X_batch,
            cp,
            y,
            expected_cp,
            is_train,
            is_validation,
        )

        gc.collect()


    print(
        f"[OK] day {day_index:02d} "
        f"{filename} — {file_rows:,} rows"
    )


# =============================================================================
# 15. MATERIALIZATION ASSERTIONS
# =============================================================================

if expected_next_clean_position != N_DEVELOPMENT:

    raise RuntimeError(
        "\nDevelopment canonical-length mismatch.\n"
        f"expected: {N_DEVELOPMENT:,}\n"
        f"actual  : {expected_next_clean_position:,}"
    )


if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nTRAIN ROW COUNT MISMATCH\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nVALIDATION ROW COUNT MISMATCH\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


X_train.flush()
y_train.flush()
X_validation.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if train_attack != int(
    EXPECTED_TRAIN[
        "attack"
    ]
):

    raise RuntimeError(
        "Random training attack count mismatch."
    )


if train_benign != int(
    EXPECTED_TRAIN[
        "benign"
    ]
):

    raise RuntimeError(
        "Random training benign count mismatch."
    )


if validation_attack != int(
    EXPECTED_VALIDATION[
        "attack"
    ]
):

    raise RuntimeError(
        "Random validation attack count mismatch."
    )


if validation_benign != int(
    EXPECTED_VALIDATION[
        "benign"
    ]
):

    raise RuntimeError(
        "Random validation benign count mismatch."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean_position order is not strictly increasing."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):

    raise RuntimeError(
        "Materialized validation positions disagree with frozen bitset."
    )


print()
print("[OK] Frozen RANDOM_NATURAL matrices materialized.")

print(
    "     train      :",
    f"{TRAIN_ROWS:,}",
    f"(benign={train_benign:,}, attack={train_attack:,})"
)

print(
    "     validation :",
    f"{VALIDATION_ROWS:,}",
    f"(benign={validation_benign:,}, attack={validation_attack:,})"
)

print(
    "     features   :",
    N_FEATURES
)

print(
    "     dtype      : float64"
)

print(
    "     seconds    :",
    f"{materialize_seconds:.3f}"
)


execution_state.update({

    "status":
        "MATERIALIZED",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 16. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 17. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 96)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 18. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 96)
print("FIT 2 / 2 — XGBOOST")
print("=" * 96)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 19. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 20. RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 21. FIXED OPERATING POINT 0.50
# =============================================================================

predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 22. FROZEN FULL RANDOM REFERENCE
# =============================================================================

full_random = json.loads(
    STAGE22_FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_random[
    "cell"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Unexpected Stage22R FULL random reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 23. RANDOM REMOVAL PENALTIES
# =============================================================================
#
# Δ_R(S) = FULL_R - SUBSET_R
#
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty_at_050 = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


recall_removal_penalty_at_050 = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


fpr_change_full_minus_ablated_at_050 = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 24. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 25. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_SUSPICIOUS_GROUP_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent_commit":
        head,

    "required_predecessor": {

        "stage":
            "Stage23-1H",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty_at_050,

        "recall_at_0_50":
            recall_removal_penalty_at_050,

        "fpr_at_0_50":
            fpr_change_full_minus_ablated_at_050,
    },

    "shortcut_interaction": {

        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            16,

        "stage23_total_model_fits_after_cell":
            18,

        "stage23_total_authorized_model_fits":
            50,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1I before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = (
    sha256_file(
        RESULT_PATH
    )
)


# =============================================================================
# 26. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = (
    sha256_file(
        CHECKSUM_OUTPUT_PATH
    )
)


# =============================================================================
# 27. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        18,

    "stage23_metrics_calculated":
        True,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 28. REMOVE TRANSIENT MATRICES
# =============================================================================

del (
    X_train,
    y_train,
    X_validation,
    y_validation,
    validation_positions,
)

gc.collect()


for path in [
    X_train_path,
    y_train_path,
    X_validation_path,
    y_validation_path,
    validation_positions_path,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 29. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1I COMPLETE — RESULT UNSEALED")
print("=" * 96)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen suspicious group:")
print(
    "  removed :",
    REMOVED_FEATURES
)
print(
    "  retained:",
    N_FEATURES
)
print(
    "  semantics:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 18 / 50")


print()
print("=" * 96)
print("RANKING METRICS")
print("=" * 96)

print()
print(
    f"Attack prevalence                   : {attack_prevalence:.12f}"
)

print(
    f"NO_SUSPICIOUS_GROUP PR-AUC          : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                         : {full_pr_auc:.12f}"
)

print(
    f"PR-AUC removal penalty              : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"NO_SUSPICIOUS_GROUP ROC-AUC         : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                        : {full_roc_auc:.12f}"
)

print(
    f"ROC-AUC removal penalty             : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence                 : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 96)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 96)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)

print(
    f"Precision : {precision:.12f}"
)

print(
    f"Recall    : {recall:.12f}"
)

print(
    f"F1        : {f1:.12f}"
)

print(
    f"FPR       : {fpr:.12f}"
)

print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 96)
print("ARTIFACTS")
print("=" * 96)

print()
print("Result:")
print(" ", RESULT_PATH)
print(" SHA256:", RESULT_SHA)

print()
print("LightGBM:")
print(" ", LGBM_MODEL_PATH)
print(" SHA256:", LGBM_MODEL_SHA)

print()
print("XGBoost:")
print(" ", XGB_MODEL_PATH)
print(" SHA256:", XGB_MODEL_SHA)

print()
print("Validation probabilities:")
print(" ", VALIDATION_PROBABILITY_PATH)
print(" SHA256:", VALIDATION_PROBABILITY_SHA)

print()
print("Checksum manifest:")
print(" ", CHECKSUM_OUTPUT_PATH)
print(" SHA256:", CHECKSUM_OUTPUT_SHA)


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits completed : 18 / 50")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("Shortcut interaction:")
print(
    "  PENDING matched "
    "NO_SUSPICIOUS_GROUP chronological cell"
)


print()
print("=" * 96)
print("NEXT ACTION")
print("=" * 96)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1I."
)

print("=" * 96)

STAGE23-1I — NO_SUSPICIOUS_GROUP × RANDOM_NATURAL

[OK] branch         : main
[OK] HEAD           : 3cc239339ad24b697d87ad9d9cc9c29cdfb55979
[OK] Stage23-0 tag  : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1H tag : 3cc239339ad24b697d87ad9d9cc9c29cdfb55979
[OK] worktree       : CLEAN
[OK] predecessor governance: 16 / 50 fits sealed
[OK] Stage23-0 artifacts: 23/23 exact

Frozen target:
  subset        : NO_SUSPICIOUS_GROUP
  split         : RANDOM_NATURAL
  feature count : 67
  removed       : ['Dst Port', 'Init Fwd Win Byts', 'Fwd Seg Size Min']
  semantic label: joint_shortcut_prone_group_ablation

[OK] Frozen package versions verified.

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID: GPU-c2e6525e-6404-4205-c5c2-2499a82d5f9b)

[OK] RANDOM_NATURAL membership:
     train      : 11,529,922
     validation : 2,882,481

MATERIALIZATION PLAN

Train rows      : 11,529,922
Validation rows : 2,882,481
Features        :

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 390.147 s
     model SHA256: e787b53dfb5a34b8a6fdd50143dbd729ee11fae101fd5a1360cb26726003b1e8

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 111.896 s
     model SHA256: 0233b02f34d8afd8a248244e720b6419bfeccc932264ae2b1e5aa4825e618b16

STAGE23-1I COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_SUSPICIOUS_GROUP
  split  : RANDOM_NATURAL

Frozen suspicious group:
  removed : ['Dst Port', 'Init Fwd Win Byts', 'Fwd Seg Size Min']
  retained: 67
  semantics: joint_shortcut_prone_group_ablation

Data:
  train      : 11,529,922
  validation : 2,882,481

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 18 / 50

RANKING METRICS

Attack prevalence                   : 0.136847389454
NO_SUSPICIOUS_GROUP PR-AUC          : 0.989601245441
FULL PR-AUC                         : 0.995590041899
PR-AUC removal penalty              : +0.005988796458

NO_SUSPICIOUS_GROUP ROC-AUC         : 0.997408519130
FULL ROC-AUC                        

In [20]:
# =============================================================================
# STAGE23-1I — SEAL + COMMIT + TAG + PUSH
# NO_SUSPICIOUS_GROUP × RANDOM_NATURAL
# =============================================================================
#
# NO MODEL FITS.
# NO DATASET ACCESS.
# NO METRIC RECOMPUTATION FROM DATA.
#
# AFTER THIS SEAL:
#   - Stage23 remains at 18 / 50 sealed fits
#   - Stage23-1J remains the next authorized MODEL cell
#   - BUT first we will execute a ZERO-FIT deterministic cache optimization
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1i_no_suspicious_group_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_suspicious_group"
)

EXPECTED_PARENT = (
    "3cc239339ad24b697d87ad9d9cc9c29cdfb55979"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PREDECESSOR_TAG = (
    "stage23-1h-no-fwd-seg-size-min-chronological-natural-v1"
)

RESULT_TAG = (
    "stage23-1i-no-suspicious-group-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1I: freeze NO_SUSPICIOUS_GROUP random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1I frozen NO_SUSPICIOUS_GROUP RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1i_no_suspicious_group_random_natural_result.json":
        "20b2b6491d4e85738d4ab6a52579233ce3bc660164a3742ca9ccb3a84a005fde",

    "no_suspicious_group_random_natural_lightgbm_model.txt":
        "e787b53dfb5a34b8a6fdd50143dbd729ee11fae101fd5a1360cb26726003b1e8",

    "no_suspicious_group_random_natural_xgboost_model.json":
        "0233b02f34d8afd8a248244e720b6419bfeccc932264ae2b1e5aa4825e618b16",

    "no_suspicious_group_random_natural_validation_probabilities.npz":
        "f820304ca76f038d562d7b1de38ecfc6b34b6d43a764e68b601f83bfdfb21207",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "2a4a38884964f9ec851eada0b0a2f8e736baf095482ee645df68ea1ef596deab"
)


# =============================================================================
# 2. EXACT OBSERVED SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.989601245441
EXPECTED_ROC_AUC = 0.997408519130

EXPECTED_PR_PENALTY = 0.005988796458
EXPECTED_ROC_PENALTY = 0.001216045643

EXPECTED_ACCURACY = 0.989896550923
EXPECTED_PRECISION = 0.965033718849
EXPECTED_RECALL = 0.960989707448
EXPECTED_F1 = 0.963007467606
EXPECTED_FPR = 0.005520451797
EXPECTED_FNR = 0.039010292552

EXPECTED_TN = 2474286
EXPECTED_FP = 13735
EXPECTED_FN = 15388
EXPECTED_TP = 379072


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 96)
print("STAGE23-1I — SCIENTIFIC RESULT SEAL")
print("=" * 96)
print()


branch = run(
    ["git", "branch", "--show-current"]
).strip()

head = run(
    ["git", "rev-parse", "HEAD"]
).strip()

status = run(
    ["git", "status", "--porcelain"]
).strip()

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
).strip()

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
).strip()


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected Git parent before Stage23-1I seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1H predecessor tag verification failed."
    )


if status:
    raise RuntimeError(
        "Repository must be clean before Stage23-1I seal:\n"
        + status
    )


if TARGET_DIR.exists():
    raise RuntimeError(
        f"Stage23-1I target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():
    raise RuntimeError(
        f"Stage23-1I source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
).strip()


if existing_tag:
    raise RuntimeError(
        f"Stage23-1I result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch        :", branch)
print("[OK] parent HEAD   :", head)
print("[OK] Stage23-0 tag :", protocol_tag_commit)
print("[OK] Stage23-1H tag:", predecessor_tag_commit)
print("[OK] worktree      : CLEAN")
print("[OK] result tag absent")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():
    raise RuntimeError(
        "Stage23-1I source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:
    raise RuntimeError(
        "\nSTAGE23-1I CHECKSUM MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 96)
print("VERIFYING STAGE23-1I CORE ARTIFACTS")
print("=" * 96)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = SOURCE_DIR / filename

    if not path.exists():
        raise RuntimeError(
            f"Missing Stage23-1I artifact: {filename}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "\nSTAGE23-1I ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. VERIFY RESULT CONTENT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1i_no_suspicious_group_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result["stage"] != "Stage23-1I":
    raise RuntimeError(
        "Unexpected Stage23-1I stage identifier."
    )


if result["cell"]["subset"] != "NO_SUSPICIOUS_GROUP":
    raise RuntimeError(
        "Stage23-1I subset changed."
    )


if result["cell"]["split"] != "RANDOM_NATURAL":
    raise RuntimeError(
        "Stage23-1I split changed."
    )


EXPECTED_REMOVED = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


if result["cell"]["removed_features"] != EXPECTED_REMOVED:
    raise RuntimeError(
        "Frozen suspicious-group definition changed."
    )


if result["cell"]["feature_count"] != 67:
    raise RuntimeError(
        "Frozen Stage23-1I feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "joint_shortcut_prone_group_ablation":
    raise RuntimeError(
        "Frozen semantic label changed."
    )


if result["protocol"]["commit"] != PROTOCOL_COMMIT:
    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "required_predecessor"
][
    "commit"
] != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-1I predecessor mismatch."
    )


governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:
    raise RuntimeError(
        "Unexpected Stage23-1I fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 16:
    raise RuntimeError(
        "Unexpected Stage23-1I pre-fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 18:
    raise RuntimeError(
        "Unexpected Stage23-1I post-fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
]:

    if governance[key] is not False:
        raise RuntimeError(
            f"Governance violation: {key}"
        )


if result[
    "shortcut_interaction"
][
    "status"
] != "PENDING_MATCHED_CHRONOLOGICAL_CELL":
    raise RuntimeError(
        "Stage23-1I interaction status changed."
    )


# =============================================================================
# 8. EXACT METRIC ASSERTIONS
# =============================================================================

actual_pr_auc = float(
    result["ranking_metrics"]["pr_auc"]
)

actual_roc_auc = float(
    result["ranking_metrics"]["roc_auc"]
)

actual_pr_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["pr_auc"]
)

actual_roc_penalty = float(
    result[
        "removal_penalty_full_minus_ablated"
    ]["roc_auc"]
)


fixed = result[
    "fixed_threshold_0_50"
]


checks = [
    ("PR-AUC", actual_pr_auc, EXPECTED_PR_AUC),
    ("ROC-AUC", actual_roc_auc, EXPECTED_ROC_AUC),
    ("PR penalty", actual_pr_penalty, EXPECTED_PR_PENALTY),
    ("ROC penalty", actual_roc_penalty, EXPECTED_ROC_PENALTY),
    ("Accuracy", float(fixed["accuracy"]), EXPECTED_ACCURACY),
    ("Precision", float(fixed["precision"]), EXPECTED_PRECISION),
    ("Recall", float(fixed["recall"]), EXPECTED_RECALL),
    ("F1", float(fixed["f1"]), EXPECTED_F1),
    ("FPR", float(fixed["fpr"]), EXPECTED_FPR),
    ("FNR", float(fixed["fnr"]), EXPECTED_FNR),
]


for name, actual, expected in checks:

    if abs(
        actual
        - expected
    ) >= 1e-12:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1I output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


if (
    int(fixed["tn"]) != EXPECTED_TN
    or int(fixed["fp"]) != EXPECTED_FP
    or int(fixed["fn"]) != EXPECTED_FN
    or int(fixed["tp"]) != EXPECTED_TP
):

    raise RuntimeError(
        "Stage23-1I confusion matrix differs from execution output."
    )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"     ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"     PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"     ROC penalty : {actual_roc_penalty:+.12f}"
)

print(
    "     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL"
)


# =============================================================================
# 9. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR / filename,
        TARGET_DIR / filename,
    )


# =============================================================================
# 10. VERIFY COPIES
# =============================================================================

print()
print("=" * 96)
print("VERIFYING REPOSITORY COPIES")
print("=" * 96)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR / filename
    )

    if src_sha != dst_sha:
        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1I",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "cell": {

        "subset":
            "NO_SUSPICIOUS_GROUP",

        "split":
            "RANDOM_NATURAL",

        "removed_features":
            EXPECTED_REMOVED,

        "feature_count":
            67,

        "semantic_label":
            "joint_shortcut_prone_group_ablation",
    },

    "sealed_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        18,

    "ranking": {

        "pr_auc":
            actual_pr_auc,

        "roc_auc":
            actual_roc_auc,

        "pr_auc_removal_penalty":
            actual_pr_penalty,

        "roc_auc_removal_penalty":
            actual_roc_penalty,
    },

    "shortcut_interaction_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_nonfit_action":
        "Stage23 deterministic execution-cache optimization",

    "next_authorized_model_cell":
        "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. README
# =============================================================================

README = (
    TARGET_DIR
    / "README.md"
)


README.write_text(
f"""# Stage23-1I — NO_SUSPICIOUS_GROUP × RANDOM_NATURAL

This directory freezes the RANDOM_NATURAL result for the prospectively
defined three-feature suspicious-group ablation.

## Frozen subset

Removed exactly:

- `Dst Port`
- `Init Fwd Win Byts`
- `Fwd Seg Size Min`

Retained features: 67

Semantic label:

`joint_shortcut_prone_group_ablation`

## Execution

- Split: `RANDOM_NATURAL`
- Train rows: {result["data"]["train"]["rows"]:,}
- Validation rows: {result["data"]["validation"]["rows"]:,}
- New boosted-model fits: 2
- Stage23 fits after this cell: 18 / 50
- Threshold optimization: none
- Subset-specific tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {actual_pr_auc:.12f}
- ROC-AUC: {actual_roc_auc:.12f}
- FULL − ablated PR-AUC penalty: {actual_pr_penalty:+.12f}
- FULL − ablated ROC-AUC penalty: {actual_roc_penalty:+.12f}

## Fixed threshold 0.50

- Accuracy: {fixed["accuracy"]:.12f}
- Precision: {fixed["precision"]:.12f}
- Recall: {fixed["recall"]:.12f}
- F1: {fixed["f1"]:.12f}
- FPR: {fixed["fpr"]:.12f}
- FNR: {fixed["fnr"]:.12f}

Confusion matrix:

- TN: {int(fixed["tn"]):,}
- FP: {int(fixed["fp"]):,}
- FN: {int(fixed["fn"]):,}
- TP: {int(fixed["tp"]):,}

## Interaction status

The matched `CHRONOLOGICAL_NATURAL` result has not yet been executed.

Therefore the frozen split × ablation interaction remains pending.

## Execution optimization note

Before the next model cell, a zero-fit deterministic execution-cache
optimization may be constructed and verified. It must not change:

- row membership
- labels
- clean positions
- feature values
- feature order
- input dtype
- model specifications
- model seeds
- thresholds
- ensemble definition
- Stage23 fit accounting

The next authorized model cell remains:

`Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL`

Raw March 1 and March 2 remain permanently closed.
""",
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_CHECKSUMS = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_artifacts = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_artifacts
    ) + "\n",
    encoding="utf-8",
)


repo_checksum_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


# =============================================================================
# 14. STAGE ONLY STAGE23-1I
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(relative_target),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


prefix = (
    str(relative_target)
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        prefix
    )
]


if unexpected:
    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 96)
print("STAGED STAGE23-1I ARTIFACTS")
print("=" * 96)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 96)
print("CREATING STAGE23-1I RESULT COMMIT")
print("=" * 96)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
).strip()


if sealed_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip():
    raise RuntimeError(
        "Repository dirty after Stage23-1I commit."
    )


print()
print("Stage23-1I commit:")
print(" ", sealed_commit)


# =============================================================================
# 16. CREATE ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ],
    show=True,
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
).strip()


if local_tag_commit != sealed_commit:
    raise RuntimeError(
        "Local Stage23-1I tag verification failed."
    )


# =============================================================================
# 17. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 96)
print("PUSHING STAGE23-1I")
print("=" * 96)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 18. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
).strip()


if not remote_main_output:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 19. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":
        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":
        peeled_commit = sha


if not tag_object:
    raise RuntimeError(
        "Remote Stage23-1I annotated tag object missing."
    )


if peeled_commit != sealed_commit:
    raise RuntimeError(
        "\nRemote Stage23-1I tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 20. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).strip()


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage23-1I push."
    )


# =============================================================================
# 21. FINAL REPORT
# =============================================================================

print()
print("=" * 96)
print("STAGE23-1I SCIENTIFIC RESULT SEALED")
print("=" * 96)

print()
print("Stage23-1H parent:")
print(" ", EXPECTED_PARENT)

print()
print("Stage23-1I commit:")
print(" ", sealed_commit)

print()
print("Result tag:")
print(" ", RESULT_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag object:")
print(" ", tag_object)

print()
print("Remote tag peeled commit:")
print(" ", peeled_commit)

print()
print("Source execution checksum manifest:")
print(" ", EXPECTED_SOURCE_MANIFEST_SHA)

print()
print("Repository checksum manifest:")
print(" ", repo_checksum_manifest_sha)


print()
print("=" * 96)
print("FROZEN NO_SUSPICIOUS_GROUP RANDOM RESULT")
print("=" * 96)

print()
print(
    f"PR-AUC      : {actual_pr_auc:.12f}"
)

print(
    f"ROC-AUC     : {actual_roc_auc:.12f}"
)

print(
    f"PR penalty  : {actual_pr_penalty:+.12f}"
)

print(
    f"ROC penalty : {actual_roc_penalty:+.12f}"
)

print()
print("Shortcut interaction:")
print("  PENDING matched chronological cell")


print()
print("=" * 96)
print("GOVERNANCE")
print("=" * 96)

print()
print("Stage23 fits sealed      : 18 / 50")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Git status               : CLEAN")


print()
print("=" * 96)
print("STAGE23-1I COMPLETE AND SEALED")
print("=" * 96)

print()
print("NEXT AUTHORIZED NON-FIT ACTION:")
print(
    "  Stage23 deterministic execution-cache optimization"
)

print()
print("NEXT AUTHORIZED MODEL CELL AFTER CACHE VERIFICATION:")
print(
    "  Stage23-1J — NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL"
)

print("=" * 96)

STAGE23-1I — SCIENTIFIC RESULT SEAL

[OK] branch        : main
[OK] parent HEAD   : 3cc239339ad24b697d87ad9d9cc9c29cdfb55979
[OK] Stage23-0 tag : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1H tag: 3cc239339ad24b697d87ad9d9cc9c29cdfb55979
[OK] worktree      : CLEAN
[OK] result tag absent

[OK] source checksum-manifest SHA256:
  2a4a38884964f9ec851eada0b0a2f8e736baf095482ee645df68ea1ef596deab

VERIFYING STAGE23-1I CORE ARTIFACTS

[EXACT] stage23_1i_no_suspicious_group_random_natural_result.json
[EXACT] no_suspicious_group_random_natural_lightgbm_model.txt
[EXACT] no_suspicious_group_random_natural_xgboost_model.json
[EXACT] no_suspicious_group_random_natural_validation_probabilities.npz

[OK] Scientific result assertions passed.
     PR-AUC      : 0.989601245441
     ROC-AUC     : 0.997408519130
     PR penalty  : +0.005988796458
     ROC penalty : +0.001216045643
     interaction: PENDING_MATCHED_CHRONOLOGICAL_CELL

VERIFYING REPOSITORY COPIES

[EXACT] stage23_1i_no_suspiciou

In [21]:
# =============================================================================
# STAGE23 — DETERMINISTIC EXECUTION CACHE V1
# ZERO-FIT DATA-LAYER OPTIMIZATION
# =============================================================================
#
# PURPOSE
# -------
# Replace repeated Parquet decompression/conversion with an immutable,
# byte-verified raw-column cache.
#
# SCIENCE DOES NOT CHANGE.
#
# Preserved EXACTLY:
#   - 14,412,403 development rows
#   - canonical clean_position order
#   - binary labels
#   - day_id
#   - original_row_index
#   - all 70 frozen Stage22R feature values
#   - exact feature order
#   - float64 feature dtype
#   - RANDOM_NATURAL membership bitset
#   - CHRONOLOGICAL_NATURAL membership
#   - all frozen subset definitions
#   - model parameters
#   - seeds
#   - thresholds
#   - ensemble definition
#
# MODEL FITS:
#   before : 18 / 50
#   this   :  0
#   after  : 18 / 50
#
# RAW MAR1 / MAR2:
#   NEVER ACCESSED
#
# OUTPUT:
#   /kaggle/working/stage23_execution_cache_v1/
#
# IMPORTANT:
#   This cache is NOT committed to GitHub (it is ~7.8 GiB).
#   Only a small verification/seal receipt will later be committed.
#
# FAIL CLOSED:
#   If this cell fails, DO NOT delete the cache directory blindly.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import shutil
import gc
import time

import numpy as np
import pyarrow.parquet as pq


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-Execution-Cache-V1"

EXPECTED_PARENT = (
    "a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0"
)

STAGE23_1I_TAG = (
    "stage23-1i-no-suspicious-group-random-natural-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

EXPECTED_RANDOM_BITSET_SHA256 = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_CACHE_DIR = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

STAGE23_1I_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_suspicious_group"
    / "seal_receipt.json"
)

RANDOM_BITSET_SOURCE = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "random_validation.packbits"
)

OUTPUT_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

FEATURE_DIR = (
    OUTPUT_DIR
    / "features"
)

BUILD_STATE_PATH = (
    OUTPUT_DIR
    / "build_state.json"
)

METADATA_PATH = (
    OUTPUT_DIR
    / "metadata.json"
)

DATA_CHECKSUMS_PATH = (
    OUTPUT_DIR
    / "data_checksums.sha256"
)

CACHE_RECEIPT_PATH = (
    OUTPUT_DIR
    / "cache_build_receipt.json"
)


# =============================================================================
# 2. FROZEN DATASET SHAPE
# =============================================================================

N_DEVELOPMENT = 14_412_403
N_FEATURES = 70

EXPECTED_TOTAL_BENIGN = 12_440_104
EXPECTED_TOTAL_ATTACK = 1_972_299

RANDOM_TRAIN_ROWS = 11_529_922
RANDOM_VALIDATION_ROWS = 2_882_481

RANDOM_TRAIN_BENIGN = 9_952_083
RANDOM_TRAIN_ATTACK = 1_577_839

RANDOM_VALIDATION_BENIGN = 2_488_021
RANDOM_VALIDATION_ATTACK = 394_460

CHRONO_TRAIN_ROWS = 13_818_623
CHRONO_VALIDATION_ROWS = 593_780

CHRONO_TRAIN_BENIGN = 11_908_580
CHRONO_TRAIN_ATTACK = 1_910_043

CHRONO_VALIDATION_BENIGN = 531_524
CHRONO_VALIDATION_ATTACK = 62_256


AUTHORIZED_SOURCE_FILES = [
    (
        0,
        "day_00_02-14-2018.parquet",
        822_947,
    ),
    (
        1,
        "day_01_02-15-2018.parquet",
        1_046_154,
    ),
    (
        2,
        "day_02_02-16-2018.parquet",
        900_988,
    ),
    (
        3,
        "day_03_02-20-2018.parquet",
        7_926_258,
    ),
    (
        4,
        "day_04_02-21-2018.parquet",
        1_031_018,
    ),
    (
        5,
        "day_05_02-22-2018.parquet",
        1_045_297,
    ),
    (
        6,
        "day_06_02-23-2018.parquet",
        1_045_961,
    ),
    (
        7,
        "day_07_02-28-2018.parquet",
        593_780,
    ),
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + out if out else "")
        )

    return out


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    temp = path.with_suffix(
        path.suffix + ".tmp"
    )

    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    temp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    resolved = Path(path).resolve()

    lowered = str(
        resolved
    ).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in lowered:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {resolved}\n"
                f"marker : {marker}"
            )

    return resolved


def exact_bytes_equal(a, b):

    a = np.asarray(a)
    b = np.asarray(b)

    if a.shape != b.shape:
        return False

    if a.dtype != b.dtype:
        return False

    return (
        a.tobytes(order="C")
        ==
        b.tobytes(order="C")
    )


# =============================================================================
# 4. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if BUILD_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                BUILD_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_BUILD_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23 EXECUTION CACHE DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        "Do NOT delete or rebuild blindly.\n\n"
        "Existing state:\n"
        + json.dumps(
            existing_state,
            indent=2,
        )
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


build_state = {

    "stage":
        STAGE,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        EXPECTED_PARENT,

    "stage23_models_fit_before":
        18,

    "stage23_models_fit_this_action":
        0,

    "stage23_models_fit_after":
        18,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    BUILD_STATE_PATH,
    build_state,
)


# =============================================================================
# 5. REPOSITORY / GOVERNANCE PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23 — DETERMINISTIC EXECUTION CACHE V1")
print("ZERO-FIT DATA-LAYER OPTIMIZATION")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

stage23_1i_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        STAGE23_1I_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main branch; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nCACHE BUILD MUST START FROM SEALED STAGE23-1I.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if stage23_1i_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1I tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


if not STAGE23_1I_SEAL.exists():

    raise RuntimeError(
        "Stage23-1I seal receipt missing."
    )


stage23_1i_seal = json.loads(
    STAGE23_1I_SEAL.read_text(
        encoding="utf-8"
    )
)


if stage23_1i_seal[
    "stage23_models_fit_total"
] != 18:

    raise RuntimeError(
        "Expected 18 sealed Stage23 fits."
    )


if stage23_1i_seal[
    "next_authorized_nonfit_action"
] != "Stage23 deterministic execution-cache optimization":

    raise RuntimeError(
        "Stage23-1I seal does not authorize cache optimization."
    )


if stage23_1i_seal[
    "next_authorized_model_cell"
] != "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected next authorized model cell."
    )


print("[OK] branch              :", branch)
print("[OK] HEAD                :", head)
print("[OK] Stage23-0 tag       :", protocol_tag_commit)
print("[OK] Stage23-1I tag      :", stage23_1i_tag_commit)
print("[OK] worktree            : CLEAN")
print("[OK] sealed fits         : 18 / 50")
print("[OK] cache action        : AUTHORIZED")
print("[OK] next MODEL cell     : Stage23-1J")


# =============================================================================
# 6. VERIFY STAGE23-0 PROTOCOL BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


protocol_manifest_sha = sha256_file(
    PROTOCOL_CHECKSUMS
)


if protocol_manifest_sha != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "\nStage23-0 protocol checksum manifest changed.\n"
        f"expected: {PROTOCOL_MANIFEST_SHA256}\n"
        f"actual  : {protocol_manifest_sha}"
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    protocol_path = (
        PROTOCOL_DIR
        / filename
    )

    if not protocol_path.exists():

        raise RuntimeError(
            f"Missing Stage23-0 artifact: {filename}"
        )

    actual_sha = sha256_file(
        protocol_path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-0 MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


print(
    f"[OK] Stage23-0 artifacts : "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN FEATURE + SPLIT SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)


split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


FULL_FEATURES = (
    feature_spec[
        "full_feature_order"
    ]
)


if len(FULL_FEATURES) != N_FEATURES:

    raise RuntimeError(
        f"Expected 70 frozen features; found {len(FULL_FEATURES)}"
    )


if feature_spec[
    "full_feature_count"
] != 70:

    raise RuntimeError(
        "Frozen full feature count changed."
    )


if len(
    set(
        FULL_FEATURES
    )
) != 70:

    raise RuntimeError(
        "Frozen feature order contains duplicates."
    )


FEATURE_TO_INDEX = {
    feature: index
    for index, feature in enumerate(
        FULL_FEATURES
    )
}


# =============================================================================
# 8. VERIFY EVERY PRIMARY SUBSET MAPS EXACTLY INTO CACHE
# =============================================================================

primary_subset_indices = {}


for subset_name, subset in feature_spec[
    "subsets"
].items():

    subset_features = (
        subset[
            "features"
        ]
    )

    indices = [
        FEATURE_TO_INDEX[
            feature
        ]
        for feature in subset_features
    ]

    reconstructed = [
        FULL_FEATURES[
            index
        ]
        for index in indices
    ]


    if reconstructed != subset_features:

        raise RuntimeError(
            f"Feature projection mismatch for {subset_name}"
        )


    if len(indices) != subset[
        "feature_count"
    ]:

        raise RuntimeError(
            f"Feature count mismatch for {subset_name}"
        )


    primary_subset_indices[
        subset_name
    ] = indices


print()
print("[OK] Frozen primary subset projections:")

for subset_name in [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]:

    print(
        f"     {subset_name:<25}"
        f"{len(primary_subset_indices[subset_name]):>3} features"
    )


# =============================================================================
# 9. VERIFY RANDOM MEMBERSHIP BITSET
# =============================================================================

if not RANDOM_BITSET_SOURCE.exists():

    raise RuntimeError(
        "Frozen RANDOM_NATURAL membership bitset missing."
    )


actual_random_bitset_sha = sha256_file(
    RANDOM_BITSET_SOURCE
)


if actual_random_bitset_sha != EXPECTED_RANDOM_BITSET_SHA256:

    raise RuntimeError(
        "\nRandom validation membership bitset changed.\n"
        f"expected: {EXPECTED_RANDOM_BITSET_SHA256}\n"
        f"actual  : {actual_random_bitset_sha}"
    )


packed_random = np.fromfile(
    RANDOM_BITSET_SOURCE,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed_random,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random validation mask logical length mismatch."
    )


if int(
    random_validation_mask.sum()
) != RANDOM_VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation population mismatch."
    )


RANDOM_BITSET_CACHE = (
    OUTPUT_DIR
    / "random_validation.packbits"
)


shutil.copy2(
    RANDOM_BITSET_SOURCE,
    RANDOM_BITSET_CACHE,
)


if sha256_file(
    RANDOM_BITSET_CACHE
) != EXPECTED_RANDOM_BITSET_SHA256:

    raise RuntimeError(
        "Copied random membership bitset differs."
    )


print()
print("[OK] RANDOM_NATURAL bitset exact:")
print(
    "     SHA256:",
    EXPECTED_RANDOM_BITSET_SHA256
)
print(
    "     validation population:",
    f"{RANDOM_VALIDATION_ROWS:,}"
)


# =============================================================================
# 10. AUTHORIZE ONLY FEB14–FEB28 PARQUETS
# =============================================================================

for day_id, filename, expected_rows in AUTHORIZED_SOURCE_FILES:

    source_path = guard_path(
        SOURCE_CACHE_DIR
        / filename
    )

    if not source_path.exists():

        raise RuntimeError(
            f"Authorized Stage22R cache file missing:\n{source_path}"
        )

    parquet_rows = pq.ParquetFile(
        source_path
    ).metadata.num_rows

    if parquet_rows != expected_rows:

        raise RuntimeError(
            "\nAuthorized source row count changed.\n"
            f"file     : {filename}\n"
            f"expected : {expected_rows:,}\n"
            f"actual   : {parquet_rows:,}"
        )


if sum(
    rows
    for _, _, rows in AUTHORIZED_SOURCE_FILES
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Authorized source total does not equal 14,412,403."
    )


print()
print(
    "[OK] Authorized source files:"
    " 8 / 8, Feb14–Feb28 only"
)


# =============================================================================
# 11. STORAGE PREFLIGHT
# =============================================================================

FEATURE_BYTES = (
    N_DEVELOPMENT
    * N_FEATURES
    * 8
)

LABEL_BYTES = (
    N_DEVELOPMENT
    * 1
)

DAY_BYTES = (
    N_DEVELOPMENT
    * 1
)

CLEAN_POSITION_BYTES = (
    N_DEVELOPMENT
    * 8
)

ORIGINAL_ROW_INDEX_BYTES = (
    N_DEVELOPMENT
    * 8
)

EXPECTED_CORE_BYTES = (
    FEATURE_BYTES
    + LABEL_BYTES
    + DAY_BYTES
    + CLEAN_POSITION_BYTES
    + ORIGINAL_ROW_INDEX_BYTES
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


# require expected cache plus 2 GiB safety margin
minimum_free = (
    EXPECTED_CORE_BYTES
    + 2 * 1024**3
)


print()
print("=" * 100)
print("CACHE STORAGE PLAN")
print("=" * 100)
print()

print(
    "Development rows :",
    f"{N_DEVELOPMENT:,}"
)

print(
    "Frozen features  :",
    N_FEATURES
)

print(
    "70F float64 data :",
    f"{FEATURE_BYTES / 1024**3:.3f} GiB"
)

print(
    "Core cache total :",
    f"{EXPECTED_CORE_BYTES / 1024**3:.3f} GiB"
)

print(
    "Working free     :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nINSUFFICIENT DISK FOR CACHE BUILD\n"
        f"required with safety margin: "
        f"{minimum_free / 1024**3:.3f} GiB\n"
        f"available                  : "
        f"{disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 12. CREATE RAW COLUMN MEMMAPS
# =============================================================================
#
# One raw file per feature:
#
# features/feature_00.float64.dat
# ...
# features/feature_69.float64.dat
#
# Each file:
#   14,412,403 values
#   little-endian float64
#
# This makes later 3-feature placebos cheap:
#   read ONLY 3 feature columns.
#
# =============================================================================

feature_paths = []

feature_mmaps = []

source_feature_hashers = []
cache_feature_hashers = []


for feature_index in range(
    N_FEATURES
):

    feature_path = (
        FEATURE_DIR
        / f"feature_{feature_index:02d}.float64.dat"
    )

    feature_paths.append(
        feature_path
    )

    feature_mmaps.append(
        np.memmap(
            feature_path,
            dtype="<f8",
            mode="w+",
            shape=(
                N_DEVELOPMENT,
            ),
        )
    )

    source_feature_hashers.append(
        hashlib.sha256()
    )

    cache_feature_hashers.append(
        hashlib.sha256()
    )


LABEL_PATH = (
    OUTPUT_DIR
    / "binary_label.uint8.dat"
)

DAY_ID_PATH = (
    OUTPUT_DIR
    / "day_id.uint8.dat"
)

CLEAN_POSITION_PATH = (
    OUTPUT_DIR
    / "clean_position.int64.dat"
)

ORIGINAL_ROW_INDEX_PATH = (
    OUTPUT_DIR
    / "original_row_index.int64.dat"
)


label_mm = np.memmap(
    LABEL_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        N_DEVELOPMENT,
    ),
)


day_id_mm = np.memmap(
    DAY_ID_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        N_DEVELOPMENT,
    ),
)


clean_position_mm = np.memmap(
    CLEAN_POSITION_PATH,
    dtype="<i8",
    mode="w+",
    shape=(
        N_DEVELOPMENT,
    ),
)


original_row_index_mm = np.memmap(
    ORIGINAL_ROW_INDEX_PATH,
    dtype="<i8",
    mode="w+",
    shape=(
        N_DEVELOPMENT,
    ),
)


source_label_hasher = hashlib.sha256()
cache_label_hasher = hashlib.sha256()

source_day_hasher = hashlib.sha256()
cache_day_hasher = hashlib.sha256()

source_cp_hasher = hashlib.sha256()
cache_cp_hasher = hashlib.sha256()

source_ori_hasher = hashlib.sha256()
cache_ori_hasher = hashlib.sha256()

source_row_major_70f_hasher = hashlib.sha256()
cache_row_major_70f_hasher = hashlib.sha256()


# =============================================================================
# 13. BUILD CACHE — ONE AUTHORIZED PARQUET PASS
# =============================================================================

READ_COLUMNS = (
    [
        "clean_position",
        "day_id",
        "original_row_index",
        "binary_label",
    ]
    + FULL_FEATURES
)


global_cursor = 0

build_start = (
    time.perf_counter()
)


print()
print("=" * 100)
print("BUILDING RAW COLUMN CACHE")
print("=" * 100)
print()


for expected_day_id, filename, expected_file_rows in AUTHORIZED_SOURCE_FILES:

    source_path = guard_path(
        SOURCE_CACHE_DIR
        / filename
    )


    parquet = pq.ParquetFile(
        source_path
    )


    schema_names = set(
        parquet.schema_arrow.names
    )


    missing_columns = [
        column
        for column in READ_COLUMNS
        if column not in schema_names
    ]


    if missing_columns:

        raise RuntimeError(
            "\nAuthorized source schema missing columns.\n"
            f"file    : {filename}\n"
            f"missing : {missing_columns}"
        )


    file_rows = 0


    for batch in parquet.iter_batches(
        batch_size=100_000,
        columns=READ_COLUMNS,
        use_threads=True,
    ):

        n_rows = (
            batch.num_rows
        )

        if n_rows == 0:
            continue


        start = (
            global_cursor
        )

        end = (
            global_cursor
            + n_rows
        )


        if end > N_DEVELOPMENT:

            raise RuntimeError(
                "Execution cache overflow."
            )


        # ---------------------------------------------------------------------
        # canonical metadata
        # ---------------------------------------------------------------------

        cp = np.asarray(
            batch.column(
                batch.schema.get_field_index(
                    "clean_position"
                )
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype="<i8",
        )


        day = np.asarray(
            batch.column(
                batch.schema.get_field_index(
                    "day_id"
                )
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.uint8,
        )


        original_row_index = np.asarray(
            batch.column(
                batch.schema.get_field_index(
                    "original_row_index"
                )
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype="<i8",
        )


        labels = np.asarray(
            batch.column(
                batch.schema.get_field_index(
                    "binary_label"
                )
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.uint8,
        )


        expected_cp = np.arange(
            start,
            end,
            dtype="<i8",
        )


        if not exact_bytes_equal(
            cp,
            expected_cp,
        ):

            raise RuntimeError(
                "\nCANONICAL CLEAN_POSITION MISMATCH\n"
                f"file     : {filename}\n"
                f"expected : {start:,}\n"
                f"actual   : {int(cp[0]):,}"
            )


        unique_day = np.unique(
            day
        )


        if not (
            len(unique_day) == 1
            and int(unique_day[0]) == expected_day_id
        ):

            raise RuntimeError(
                "\nDAY_ID MISMATCH\n"
                f"file     : {filename}\n"
                f"expected : {expected_day_id}\n"
                f"actual   : {unique_day.tolist()}"
            )


        if not np.all(
            (
                labels == 0
            )
            |
            (
                labels == 1
            )
        ):

            raise RuntimeError(
                "Non-binary label encountered."
            )


        # ---------------------------------------------------------------------
        # EXACT old materialization semantics:
        #
        # np.column_stack([...]).astype(np.float64, copy=False)
        #
        # This reproduces the transformation used by the previous Stage23
        # model cells.
        # ---------------------------------------------------------------------

        feature_arrays = []


        for feature_name in FULL_FEATURES:

            feature_array = np.asarray(
                batch.column(
                    batch.schema.get_field_index(
                        feature_name
                    )
                ).to_numpy(
                    zero_copy_only=False
                ),
                dtype="<f8",
            )


            if feature_array.shape != (
                n_rows,
            ):

                raise RuntimeError(
                    f"Unexpected shape for feature: {feature_name}"
                )


            feature_arrays.append(
                feature_array
            )


        X_old_semantics = np.column_stack(
            feature_arrays
        ).astype(
            np.float64,
            copy=False,
        )


        if X_old_semantics.shape != (
            n_rows,
            N_FEATURES,
        ):

            raise RuntimeError(
                "70F batch materialization shape mismatch."
            )


        # row-major digest of exactly what the old cells would have built
        source_row_major_70f_hasher.update(
            np.ascontiguousarray(
                X_old_semantics,
                dtype="<f8",
            ).tobytes(
                order="C"
            )
        )


        # ---------------------------------------------------------------------
        # write all 70 raw feature columns
        # ---------------------------------------------------------------------

        for feature_index in range(
            N_FEATURES
        ):

            source_column = np.ascontiguousarray(
                X_old_semantics[
                    :,
                    feature_index
                ],
                dtype="<f8",
            )


            feature_mmaps[
                feature_index
            ][
                start:end
            ] = source_column


            source_feature_hashers[
                feature_index
            ].update(
                source_column.tobytes(
                    order="C"
                )
            )


        # ---------------------------------------------------------------------
        # write metadata arrays
        # ---------------------------------------------------------------------

        label_mm[
            start:end
        ] = labels


        day_id_mm[
            start:end
        ] = day


        clean_position_mm[
            start:end
        ] = cp


        original_row_index_mm[
            start:end
        ] = original_row_index


        # ---------------------------------------------------------------------
        # immediate read-back verification
        #
        # Reconstruct the row-major 70F matrix FROM THE NEW CACHE.
        #
        # This verifies:
        #   Parquet -> old float64 semantics
        #         ==
        #   raw cache -> reconstructed float64 semantics
        # ---------------------------------------------------------------------

        X_cache_readback = np.empty(
            (
                n_rows,
                N_FEATURES,
            ),
            dtype="<f8",
        )


        for feature_index in range(
            N_FEATURES
        ):

            cached_column = np.asarray(
                feature_mmaps[
                    feature_index
                ][
                    start:end
                ],
                dtype="<f8",
            )


            X_cache_readback[
                :,
                feature_index
            ] = cached_column


            cache_feature_hashers[
                feature_index
            ].update(
                np.ascontiguousarray(
                    cached_column,
                    dtype="<f8",
                ).tobytes(
                    order="C"
                )
            )


        if not exact_bytes_equal(
            np.ascontiguousarray(
                X_old_semantics,
                dtype="<f8",
            ),
            np.ascontiguousarray(
                X_cache_readback,
                dtype="<f8",
            ),
        ):

            raise RuntimeError(
                "\nFEATURE CACHE BYTE MISMATCH\n"
                f"file  : {filename}\n"
                f"rows  : {start:,}..{end - 1:,}"
            )


        cache_row_major_70f_hasher.update(
            np.ascontiguousarray(
                X_cache_readback,
                dtype="<f8",
            ).tobytes(
                order="C"
            )
        )


        # ---------------------------------------------------------------------
        # metadata byte verification
        # ---------------------------------------------------------------------

        cached_labels = np.asarray(
            label_mm[
                start:end
            ],
            dtype=np.uint8,
        )

        cached_day = np.asarray(
            day_id_mm[
                start:end
            ],
            dtype=np.uint8,
        )

        cached_cp = np.asarray(
            clean_position_mm[
                start:end
            ],
            dtype="<i8",
        )

        cached_ori = np.asarray(
            original_row_index_mm[
                start:end
            ],
            dtype="<i8",
        )


        if not exact_bytes_equal(
            labels,
            cached_labels,
        ):

            raise RuntimeError(
                "Label cache byte mismatch."
            )


        if not exact_bytes_equal(
            day,
            cached_day,
        ):

            raise RuntimeError(
                "day_id cache byte mismatch."
            )


        if not exact_bytes_equal(
            cp,
            cached_cp,
        ):

            raise RuntimeError(
                "clean_position cache byte mismatch."
            )


        if not exact_bytes_equal(
            original_row_index,
            cached_ori,
        ):

            raise RuntimeError(
                "original_row_index cache byte mismatch."
            )


        source_label_hasher.update(
            labels.tobytes(
                order="C"
            )
        )

        cache_label_hasher.update(
            cached_labels.tobytes(
                order="C"
            )
        )


        source_day_hasher.update(
            day.tobytes(
                order="C"
            )
        )

        cache_day_hasher.update(
            cached_day.tobytes(
                order="C"
            )
        )


        source_cp_hasher.update(
            cp.tobytes(
                order="C"
            )
        )

        cache_cp_hasher.update(
            cached_cp.tobytes(
                order="C"
            )
        )


        source_ori_hasher.update(
            original_row_index.tobytes(
                order="C"
            )
        )

        cache_ori_hasher.update(
            cached_ori.tobytes(
                order="C"
            )
        )


        global_cursor = (
            end
        )

        file_rows += (
            n_rows
        )


        del (
            cp,
            day,
            original_row_index,
            labels,
            expected_cp,
            feature_arrays,
            X_old_semantics,
            X_cache_readback,
            cached_labels,
            cached_day,
            cached_cp,
            cached_ori,
        )

        gc.collect()


    if file_rows != expected_file_rows:

        raise RuntimeError(
            "\nSOURCE FILE ROW COUNT MISMATCH DURING BUILD\n"
            f"file     : {filename}\n"
            f"expected : {expected_file_rows:,}\n"
            f"actual   : {file_rows:,}"
        )


    print(
        f"[EXACT] day {expected_day_id:02d} "
        f"{filename:<31} {file_rows:>10,} rows"
    )


# =============================================================================
# 14. FINAL BUILD SIZE ASSERTION
# =============================================================================

if global_cursor != N_DEVELOPMENT:

    raise RuntimeError(
        "\nCACHE LOGICAL LENGTH MISMATCH\n"
        f"expected: {N_DEVELOPMENT:,}\n"
        f"actual  : {global_cursor:,}"
    )


# =============================================================================
# 15. FLUSH ALL MEMMAPS
# =============================================================================

for mm in feature_mmaps:

    mm.flush()


label_mm.flush()
day_id_mm.flush()
clean_position_mm.flush()
original_row_index_mm.flush()


build_seconds = (
    time.perf_counter()
    - build_start
)


# =============================================================================
# 16. GLOBAL SEMANTIC DIGEST VERIFICATION
# =============================================================================

source_row_major_sha = (
    source_row_major_70f_hasher.hexdigest()
)

cache_row_major_sha = (
    cache_row_major_70f_hasher.hexdigest()
)


if source_row_major_sha != cache_row_major_sha:

    raise RuntimeError(
        "\nGLOBAL 70F SEMANTIC DIGEST MISMATCH\n"
        f"source semantics: {source_row_major_sha}\n"
        f"cache semantics : {cache_row_major_sha}"
    )


if (
    source_label_hasher.hexdigest()
    != cache_label_hasher.hexdigest()
):

    raise RuntimeError(
        "Global label digest mismatch."
    )


if (
    source_day_hasher.hexdigest()
    != cache_day_hasher.hexdigest()
):

    raise RuntimeError(
        "Global day_id digest mismatch."
    )


if (
    source_cp_hasher.hexdigest()
    != cache_cp_hasher.hexdigest()
):

    raise RuntimeError(
        "Global clean_position digest mismatch."
    )


if (
    source_ori_hasher.hexdigest()
    != cache_ori_hasher.hexdigest()
):

    raise RuntimeError(
        "Global original_row_index digest mismatch."
    )


# =============================================================================
# 17. VERIFY EACH FEATURE FILE DIGEST
# =============================================================================

feature_records = []


for feature_index, feature_name in enumerate(
    FULL_FEATURES
):

    source_digest = (
        source_feature_hashers[
            feature_index
        ].hexdigest()
    )

    cache_digest = (
        cache_feature_hashers[
            feature_index
        ].hexdigest()
    )


    if source_digest != cache_digest:

        raise RuntimeError(
            "\nFEATURE DIGEST MISMATCH\n"
            f"feature : {feature_name}\n"
            f"source  : {source_digest}\n"
            f"cache   : {cache_digest}"
        )


    expected_size = (
        N_DEVELOPMENT
        * 8
    )


    actual_size = (
        feature_paths[
            feature_index
        ].stat().st_size
    )


    if actual_size != expected_size:

        raise RuntimeError(
            "\nFEATURE FILE SIZE MISMATCH\n"
            f"feature : {feature_name}\n"
            f"expected: {expected_size}\n"
            f"actual  : {actual_size}"
        )


    feature_records.append({

        "index":
            feature_index,

        "feature":
            feature_name,

        "path":
            str(
                Path("features")
                / feature_paths[
                    feature_index
                ].name
            ),

        "dtype":
            "float64",

        "byte_order":
            "little-endian",

        "rows":
            N_DEVELOPMENT,

        "bytes":
            actual_size,

        "sha256":
            cache_digest,
    })


print()
print(
    "[OK] feature column digests:"
    " 70 / 70 source == cache"
)


# =============================================================================
# 18. VERIFY METADATA ARRAY SIZES
# =============================================================================

EXPECTED_METADATA_SIZES = {

    LABEL_PATH:
        N_DEVELOPMENT,

    DAY_ID_PATH:
        N_DEVELOPMENT,

    CLEAN_POSITION_PATH:
        N_DEVELOPMENT * 8,

    ORIGINAL_ROW_INDEX_PATH:
        N_DEVELOPMENT * 8,
}


for path, expected_size in EXPECTED_METADATA_SIZES.items():

    actual_size = (
        path.stat().st_size
    )

    if actual_size != expected_size:

        raise RuntimeError(
            "\nCACHE METADATA FILE SIZE MISMATCH\n"
            f"file     : {path.name}\n"
            f"expected : {expected_size:,}\n"
            f"actual   : {actual_size:,}"
        )


# =============================================================================
# 19. VALIDATE CANONICAL LABEL / SPLIT COUNTS FROM NEW CACHE
# =============================================================================

labels_cache = np.asarray(
    label_mm,
    dtype=np.uint8,
)


total_attack = int(
    np.count_nonzero(
        labels_cache == 1
    )
)

total_benign = int(
    np.count_nonzero(
        labels_cache == 0
    )
)


if total_attack != EXPECTED_TOTAL_ATTACK:

    raise RuntimeError(
        "Total attack count mismatch."
    )


if total_benign != EXPECTED_TOTAL_BENIGN:

    raise RuntimeError(
        "Total benign count mismatch."
    )


# -----------------------------------------------------------------------------
# RANDOM_NATURAL
# -----------------------------------------------------------------------------

random_val_labels = (
    labels_cache[
        random_validation_mask
    ]
)

random_train_labels = (
    labels_cache[
        ~random_validation_mask
    ]
)


random_val_attack = int(
    np.count_nonzero(
        random_val_labels == 1
    )
)

random_val_benign = int(
    np.count_nonzero(
        random_val_labels == 0
    )
)

random_train_attack = int(
    np.count_nonzero(
        random_train_labels == 1
    )
)

random_train_benign = int(
    np.count_nonzero(
        random_train_labels == 0
    )
)


if (
    len(random_train_labels)
    != RANDOM_TRAIN_ROWS
    or random_train_benign
    != RANDOM_TRAIN_BENIGN
    or random_train_attack
    != RANDOM_TRAIN_ATTACK
):

    raise RuntimeError(
        "RANDOM_NATURAL training membership mismatch."
    )


if (
    len(random_val_labels)
    != RANDOM_VALIDATION_ROWS
    or random_val_benign
    != RANDOM_VALIDATION_BENIGN
    or random_val_attack
    != RANDOM_VALIDATION_ATTACK
):

    raise RuntimeError(
        "RANDOM_NATURAL validation membership mismatch."
    )


# -----------------------------------------------------------------------------
# CHRONOLOGICAL_NATURAL
# -----------------------------------------------------------------------------

chrono_train_labels = (
    labels_cache[
        :CHRONO_TRAIN_ROWS
    ]
)

chrono_val_labels = (
    labels_cache[
        CHRONO_TRAIN_ROWS:
    ]
)


chrono_train_attack = int(
    np.count_nonzero(
        chrono_train_labels == 1
    )
)

chrono_train_benign = int(
    np.count_nonzero(
        chrono_train_labels == 0
    )
)

chrono_val_attack = int(
    np.count_nonzero(
        chrono_val_labels == 1
    )
)

chrono_val_benign = int(
    np.count_nonzero(
        chrono_val_labels == 0
    )
)


if (
    len(chrono_train_labels)
    != CHRONO_TRAIN_ROWS
    or chrono_train_benign
    != CHRONO_TRAIN_BENIGN
    or chrono_train_attack
    != CHRONO_TRAIN_ATTACK
):

    raise RuntimeError(
        "CHRONOLOGICAL_NATURAL training membership mismatch."
    )


if (
    len(chrono_val_labels)
    != CHRONO_VALIDATION_ROWS
    or chrono_val_benign
    != CHRONO_VALIDATION_BENIGN
    or chrono_val_attack
    != CHRONO_VALIDATION_ATTACK
):

    raise RuntimeError(
        "CHRONOLOGICAL_NATURAL validation membership mismatch."
    )


# =============================================================================
# 20. VERIFY DAY BOUNDARY FOR CHRONO SPLIT
# =============================================================================

if not np.all(
    np.asarray(
        day_id_mm[
            :CHRONO_TRAIN_ROWS
        ]
    ) <= 6
):

    raise RuntimeError(
        "Chronological train contains unexpected day_id."
    )


if not np.all(
    np.asarray(
        day_id_mm[
            CHRONO_TRAIN_ROWS:
        ]
    ) == 7
):

    raise RuntimeError(
        "Chronological validation is not exactly day_id 7."
    )


# =============================================================================
# 21. VERIFY CLEAN_POSITION IS CANONICAL
# =============================================================================

# Avoid allocating another 115 MiB arange at once.
POSITION_CHUNK = 1_000_000


for start in range(
    0,
    N_DEVELOPMENT,
    POSITION_CHUNK,
):

    end = min(
        start + POSITION_CHUNK,
        N_DEVELOPMENT,
    )

    expected = np.arange(
        start,
        end,
        dtype="<i8",
    )

    actual = np.asarray(
        clean_position_mm[
            start:end
        ],
        dtype="<i8",
    )

    if not exact_bytes_equal(
        expected,
        actual,
    ):

        raise RuntimeError(
            "\nCACHE CLEAN_POSITION NOT CANONICAL\n"
            f"range: {start:,}..{end - 1:,}"
        )


print()
print("[OK] Canonical cache validation:")
print(
    "     total rows       :",
    f"{N_DEVELOPMENT:,}"
)
print(
    "     benign           :",
    f"{total_benign:,}"
)
print(
    "     attack           :",
    f"{total_attack:,}"
)

print()
print("[OK] RANDOM_NATURAL from cache:")
print(
    "     train            :",
    f"{len(random_train_labels):,}",
    f"(B={random_train_benign:,}, A={random_train_attack:,})"
)
print(
    "     validation       :",
    f"{len(random_val_labels):,}",
    f"(B={random_val_benign:,}, A={random_val_attack:,})"
)

print()
print("[OK] CHRONOLOGICAL_NATURAL from cache:")
print(
    "     train            :",
    f"{len(chrono_train_labels):,}",
    f"(B={chrono_train_benign:,}, A={chrono_train_attack:,})"
)
print(
    "     validation       :",
    f"{len(chrono_val_labels):,}",
    f"(B={chrono_val_benign:,}, A={chrono_val_attack:,})"
)


# =============================================================================
# 22. CREATE DATA CHECKSUM MANIFEST
# =============================================================================

data_checksums = []


for record in feature_records:

    data_checksums.append(
        (
            record[
                "sha256"
            ],
            record[
                "path"
            ],
        )
    )


metadata_array_records = {

    "binary_label.uint8.dat": {
        "sha256":
            cache_label_hasher.hexdigest(),
        "dtype":
            "uint8",
        "rows":
            N_DEVELOPMENT,
    },

    "day_id.uint8.dat": {
        "sha256":
            cache_day_hasher.hexdigest(),
        "dtype":
            "uint8",
        "rows":
            N_DEVELOPMENT,
    },

    "clean_position.int64.dat": {
        "sha256":
            cache_cp_hasher.hexdigest(),
        "dtype":
            "int64",
        "rows":
            N_DEVELOPMENT,
    },

    "original_row_index.int64.dat": {
        "sha256":
            cache_ori_hasher.hexdigest(),
        "dtype":
            "int64",
        "rows":
            N_DEVELOPMENT,
    },

    "random_validation.packbits": {
        "sha256":
            EXPECTED_RANDOM_BITSET_SHA256,
        "dtype":
            "packed-bitset",
        "logical_rows":
            N_DEVELOPMENT,
        "validation_population":
            RANDOM_VALIDATION_ROWS,
    },
}


for filename, record in metadata_array_records.items():

    data_checksums.append(
        (
            record[
                "sha256"
            ],
            filename,
        )
    )


DATA_CHECKSUMS_PATH.write_text(
    "\n".join(
        f"{digest}  {filename}"
        for digest, filename in data_checksums
    ) + "\n",
    encoding="utf-8",
)


DATA_CHECKSUMS_SHA = sha256_file(
    DATA_CHECKSUMS_PATH
)


# =============================================================================
# 23. METADATA
# =============================================================================

core_cache_bytes = sum(
    path.stat().st_size
    for path in feature_paths
) + sum(
    path.stat().st_size
    for path in [
        LABEL_PATH,
        DAY_ID_PATH,
        CLEAN_POSITION_PATH,
        ORIGINAL_ROW_INDEX_PATH,
        RANDOM_BITSET_CACHE,
    ]
)


metadata = {

    "cache":
        "stage23_execution_cache_v1",

    "status":
        "BUILT_AND_BYTE_VERIFIED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        EXPECTED_PARENT,

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "source": {

        "type":
            "Stage22R authorized 70-feature development Parquet cache",

        "root":
            str(
                SOURCE_CACHE_DIR
            ),

        "authorized_files": [
            {
                "day_id":
                    day_id,

                "filename":
                    filename,

                "rows":
                    rows,
            }
            for day_id, filename, rows
            in AUTHORIZED_SOURCE_FILES
        ],

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,
    },

    "shape": {

        "rows":
            N_DEVELOPMENT,

        "features":
            N_FEATURES,
    },

    "feature_dtype":
        "float64",

    "feature_byte_order":
        "little-endian",

    "feature_storage":
        "ONE_RAW_CONTIGUOUS_COLUMN_PER_FEATURE",

    "full_feature_order":
        FULL_FEATURES,

    "feature_columns":
        feature_records,

    "metadata_arrays":
        metadata_array_records,

    "primary_subset_feature_indices":
        primary_subset_indices,

    "semantic_equivalence": {

        "old_stage23_materialization":
            "np.column_stack(feature_arrays).astype(np.float64, copy=False)",

        "source_row_major_70f_sha256":
            source_row_major_sha,

        "cache_reconstructed_row_major_70f_sha256":
            cache_row_major_sha,

        "exact_byte_equality":
            True,

        "per_feature_exact_byte_equality":
            True,

        "label_exact_byte_equality":
            True,

        "day_id_exact_byte_equality":
            True,

        "clean_position_exact_byte_equality":
            True,

        "original_row_index_exact_byte_equality":
            True,
    },

    "split_validation": {

        "RANDOM_NATURAL": {

            "train": {
                "rows":
                    RANDOM_TRAIN_ROWS,

                "benign":
                    RANDOM_TRAIN_BENIGN,

                "attack":
                    RANDOM_TRAIN_ATTACK,
            },

            "validation": {
                "rows":
                    RANDOM_VALIDATION_ROWS,

                "benign":
                    RANDOM_VALIDATION_BENIGN,

                "attack":
                    RANDOM_VALIDATION_ATTACK,
            },

            "membership_bitset_sha256":
                EXPECTED_RANDOM_BITSET_SHA256,
        },

        "CHRONOLOGICAL_NATURAL": {

            "train": {
                "rows":
                    CHRONO_TRAIN_ROWS,

                "benign":
                    CHRONO_TRAIN_BENIGN,

                "attack":
                    CHRONO_TRAIN_ATTACK,

                "day_ids":
                    [0, 1, 2, 3, 4, 5, 6],
            },

            "validation": {
                "rows":
                    CHRONO_VALIDATION_ROWS,

                "benign":
                    CHRONO_VALIDATION_BENIGN,

                "attack":
                    CHRONO_VALIDATION_ATTACK,

                "day_ids":
                    [7],
            },
        },
    },

    "storage": {

        "core_cache_bytes":
            core_cache_bytes,

        "core_cache_gib":
            core_cache_bytes
            / 1024**3,

        "data_checksums_manifest":
            DATA_CHECKSUMS_PATH.name,

        "data_checksums_manifest_sha256":
            DATA_CHECKSUMS_SHA,
    },

    "governance": {

        "new_model_fits":
            0,

        "stage23_models_fit_before":
            18,

        "stage23_models_fit_after":
            18,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "model_specification_changed":
            False,

        "feature_definition_changed":
            False,

        "split_definition_changed":
            False,

        "input_dtype_changed":
            False,

        "stage23_0_modified":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,
    },

    "next_authorized_model_cell":
        "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL",
}


write_json_atomic(
    METADATA_PATH,
    metadata,
)


METADATA_SHA = sha256_file(
    METADATA_PATH
)


# =============================================================================
# 24. BUILD RECEIPT
# =============================================================================

cache_receipt = {

    "stage":
        STAGE,

    "status":
        "CACHE_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        EXPECTED_PARENT,

    "cache_path":
        str(
            OUTPUT_DIR
        ),

    "rows":
        N_DEVELOPMENT,

    "feature_count":
        N_FEATURES,

    "feature_dtype":
        "float64",

    "semantic_row_major_70f_sha256":
        cache_row_major_sha,

    "source_semantic_row_major_70f_sha256":
        source_row_major_sha,

    "semantic_byte_equivalence":
        True,

    "data_checksums_manifest_sha256":
        DATA_CHECKSUMS_SHA,

    "metadata_sha256":
        METADATA_SHA,

    "random_validation_bitset_sha256":
        EXPECTED_RANDOM_BITSET_SHA256,

    "build_seconds":
        build_seconds,

    "core_cache_bytes":
        core_cache_bytes,

    "core_cache_gib":
        core_cache_bytes
        / 1024**3,

    "model_fits_this_action":
        0,

    "stage23_models_fit_total":
        18,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "next_action":
        "Seal deterministic execution-cache verification receipt.",

    "next_authorized_model_cell_after_cache_seal":
        "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL",
}


write_json_atomic(
    CACHE_RECEIPT_PATH,
    cache_receipt,
)


CACHE_RECEIPT_SHA = sha256_file(
    CACHE_RECEIPT_PATH
)


# =============================================================================
# 25. FINAL BUILD STATE
# =============================================================================

build_state.update({

    "status":
        "CACHE_COMPLETE_UNSEALED",

    "completed_utc":
        now_utc(),

    "rows":
        N_DEVELOPMENT,

    "features":
        N_FEATURES,

    "feature_dtype":
        "float64",

    "semantic_byte_equivalence":
        True,

    "semantic_row_major_70f_sha256":
        cache_row_major_sha,

    "data_checksums_manifest_sha256":
        DATA_CHECKSUMS_SHA,

    "metadata_sha256":
        METADATA_SHA,

    "cache_receipt_sha256":
        CACHE_RECEIPT_SHA,

    "core_cache_bytes":
        core_cache_bytes,

    "stage23_models_fit_before":
        18,

    "stage23_models_fit_this_action":
        0,

    "stage23_models_fit_after":
        18,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
})


write_json_atomic(
    BUILD_STATE_PATH,
    build_state,
)


# =============================================================================
# 26. CLOSE LARGE MEMMAP OBJECTS
# =============================================================================

del random_train_labels
del random_val_labels
del chrono_train_labels
del chrono_val_labels
del labels_cache

del label_mm
del day_id_mm
del clean_position_mm
del original_row_index_mm

for index in range(
    len(feature_mmaps)
):

    feature_mmaps[
        index
    ] = None


del feature_mmaps

gc.collect()


# =============================================================================
# 27. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23 EXECUTION CACHE V1 COMPLETE — UNSEALED")
print("=" * 100)

print()
print("Parent:")
print(
    " ",
    EXPECTED_PARENT
)

print()
print("Cache:")
print(
    " ",
    OUTPUT_DIR
)

print()
print("Canonical development corpus:")
print(
    "  rows          :",
    f"{N_DEVELOPMENT:,}"
)
print(
    "  features      :",
    N_FEATURES
)
print(
    "  feature dtype : float64"
)
print(
    "  core size     :",
    f"{core_cache_bytes / 1024**3:.3f} GiB"
)

print()
print("=" * 100)
print("BYTE-EQUIVALENCE VERIFICATION")
print("=" * 100)

print()
print("Old Stage23 row-major 70F semantic SHA256:")
print(
    " ",
    source_row_major_sha
)

print()
print("New cache reconstructed 70F semantic SHA256:")
print(
    " ",
    cache_row_major_sha
)

print()
print(
    "Exact global 70F byte equivalence : YES"
)
print(
    "Exact per-feature byte equivalence: 70 / 70"
)
print(
    "Exact binary_label equivalence    : YES"
)
print(
    "Exact day_id equivalence          : YES"
)
print(
    "Exact clean_position equivalence  : YES"
)
print(
    "Exact original_row_index equiv.   : YES"
)


print()
print("=" * 100)
print("SPLIT VERIFICATION")
print("=" * 100)

print()
print("RANDOM_NATURAL:")
print(
    "  train      :",
    f"{RANDOM_TRAIN_ROWS:,}",
    f"(B={RANDOM_TRAIN_BENIGN:,}, A={RANDOM_TRAIN_ATTACK:,})"
)
print(
    "  validation :",
    f"{RANDOM_VALIDATION_ROWS:,}",
    f"(B={RANDOM_VALIDATION_BENIGN:,}, A={RANDOM_VALIDATION_ATTACK:,})"
)
print(
    "  bitset SHA :",
    EXPECTED_RANDOM_BITSET_SHA256
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    "  train      :",
    f"{CHRONO_TRAIN_ROWS:,}",
    f"(B={CHRONO_TRAIN_BENIGN:,}, A={CHRONO_TRAIN_ATTACK:,})"
)
print(
    "  validation :",
    f"{CHRONO_VALIDATION_ROWS:,}",
    f"(B={CHRONO_VALIDATION_BENIGN:,}, A={CHRONO_VALIDATION_ATTACK:,})"
)
print(
    "  train days : 0..6"
)
print(
    "  val day    : 7"
)


print()
print("=" * 100)
print("CACHE ARTIFACTS")
print("=" * 100)

print()
print("Data checksum manifest:")
print(
    " ",
    DATA_CHECKSUMS_PATH
)
print(
    " SHA256:",
    DATA_CHECKSUMS_SHA
)

print()
print("Metadata:")
print(
    " ",
    METADATA_PATH
)
print(
    " SHA256:",
    METADATA_SHA
)

print()
print("Build receipt:")
print(
    " ",
    CACHE_RECEIPT_PATH
)
print(
    " SHA256:",
    CACHE_RECEIPT_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits before      : 18 / 50")
print("Model fits this action   : 0")
print("Stage23 fits after       : 18 / 50")
print("Feature definitions      : UNCHANGED")
print("Split definitions        : UNCHANGED")
print("Input dtype              : UNCHANGED — float64")
print("Model parameters         : UNCHANGED")
print("Thresholds               : UNCHANGED")
print("Ensemble definition      : UNCHANGED")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 modified       : NO")
print("Git commit created       : NO")
print("Git tag created          : NO")


print()
print("=" * 100)
print("NEXT ACTION")
print("=" * 100)

print()
print(
    "Do NOT execute Stage23-1J yet."
)

print(
    "First seal the deterministic execution-cache verification receipt."
)

print()
print(
    "After cache seal, next authorized MODEL cell:"
)
print(
    "  Stage23-1J — "
    "NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL"
)

print("=" * 100)

STAGE23 — DETERMINISTIC EXECUTION CACHE V1
ZERO-FIT DATA-LAYER OPTIMIZATION

[OK] branch              : main
[OK] HEAD                : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] Stage23-0 tag       : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1I tag      : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] worktree            : CLEAN
[OK] sealed fits         : 18 / 50
[OK] cache action        : AUTHORIZED
[OK] next MODEL cell     : Stage23-1J
[OK] Stage23-0 artifacts : 23/23 exact

[OK] Frozen primary subset projections:
     FULL                      70 features
     NO_DST_PORT               69 features
     NO_PORTS                  68 features
     NO_INIT_FWD_WIN_BYTS      69 features
     NO_FWD_SEG_SIZE_MIN       69 features
     NO_SUSPICIOUS_GROUP       67 features
     BEHAVIOR_ONLY             63 features

[OK] RANDOM_NATURAL bitset exact:
     SHA256: 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
     validation population: 2,882,481

[OK] Autho

In [22]:
# =============================================================================
# STAGE23 — DETERMINISTIC EXECUTION CACHE V1
# VERIFICATION SEAL + COMMIT + TAG + PUSH
# =============================================================================
#
# ZERO MODEL FITS.
# ZERO PARQUET ACCESS.
# ZERO MAR1 / MAR2 ACCESS.
#
# IMPORTANT:
#   The ~7.76 GiB binary execution cache is NOT committed to GitHub.
#
#   We commit only:
#       data_checksums.sha256
#       metadata.json
#       cache_build_receipt.json
#       README.md
#       seal_receipt.json
#       repository_checksums.sha256
#
# The physical cache is re-hashed against its frozen manifest before sealing.
#
# FIT ACCOUNTING:
#   before : 18 / 50
#   this   :  0
#   after  : 18 / 50
#
# NEXT MODEL CELL AFTER SUCCESSFUL SEAL:
#   Stage23-1J
#   NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import os


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

EXPECTED_PARENT = (
    "a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0"
)

PREDECESSOR_TAG = (
    "stage23-1i-no-suspicious-group-random-natural-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

CACHE_TAG = (
    "stage23-execution-cache-v1"
)

COMMIT_MESSAGE = (
    "Stage23: seal deterministic execution cache v1"
)

TAG_MESSAGE = (
    "Stage23 deterministic execution cache v1 verification seal"
)


# =============================================================================
# 1. EXACT CACHE BUILD HASHES
# =============================================================================

EXPECTED_DATA_CHECKSUMS_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

EXPECTED_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

EXPECTED_CACHE_BUILD_RECEIPT_SHA256 = (
    "024e8724cd0dd672f44015af909bf4f9b620f44b03e4ed09c48ac19ecc35a8f2"
)

EXPECTED_SEMANTIC_70F_SHA256 = (
    "8c43ab0e36a65de1c095acb1ebf9e7d029a9e2b1cc353455a66617db8a899bb2"
)

EXPECTED_RANDOM_BITSET_SHA256 = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =============================================================================
# 2. FROZEN COUNTS
# =============================================================================

N_DEVELOPMENT = 14_412_403
N_FEATURES = 70

EXPECTED_TOTAL_BENIGN = 12_440_104
EXPECTED_TOTAL_ATTACK = 1_972_299

EXPECTED_RANDOM_TRAIN_ROWS = 11_529_922
EXPECTED_RANDOM_VAL_ROWS = 2_882_481

EXPECTED_CHRONO_TRAIN_ROWS = 13_818_623
EXPECTED_CHRONO_VAL_ROWS = 593_780


# =============================================================================
# 3. SOURCE ARTIFACTS
# =============================================================================

DATA_CHECKSUMS_SOURCE = (
    CACHE_DIR
    / "data_checksums.sha256"
)

METADATA_SOURCE = (
    CACHE_DIR
    / "metadata.json"
)

BUILD_RECEIPT_SOURCE = (
    CACHE_DIR
    / "cache_build_receipt.json"
)


# =============================================================================
# 4. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if show and out:
        print(out)

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + out if out else "")
        )

    return out


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


# =============================================================================
# 5. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23 — EXECUTION CACHE V1 VERIFICATION SEAL")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nCACHE SEAL MUST FOLLOW SEALED STAGE23-1I.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if predecessor_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1I predecessor tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


if TARGET_DIR.exists():

    raise RuntimeError(
        f"Cache seal target already exists:\n{TARGET_DIR}"
    )


if not CACHE_DIR.exists():

    raise RuntimeError(
        f"Execution cache missing:\n{CACHE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        CACHE_TAG,
    ]
)


if existing_tag:

    raise RuntimeError(
        f"Cache tag already exists: {CACHE_TAG}"
    )


print("[OK] branch            :", branch)
print("[OK] parent HEAD       :", head)
print("[OK] Stage23-0 tag     :", protocol_tag_commit)
print("[OK] Stage23-1I tag    :", predecessor_tag_commit)
print("[OK] worktree          : CLEAN")
print("[OK] cache tag absent")


# =============================================================================
# 6. VERIFY SMALL BUILD ARTIFACT HASHES
# =============================================================================

EXPECTED_SMALL_HASHES = {

    DATA_CHECKSUMS_SOURCE:
        EXPECTED_DATA_CHECKSUMS_SHA256,

    METADATA_SOURCE:
        EXPECTED_METADATA_SHA256,

    BUILD_RECEIPT_SOURCE:
        EXPECTED_CACHE_BUILD_RECEIPT_SHA256,
}


print()
print("=" * 100)
print("VERIFYING CACHE BUILD ARTIFACTS")
print("=" * 100)
print()


for path, expected_sha in EXPECTED_SMALL_HASHES.items():

    if not path.exists():

        raise RuntimeError(
            f"Missing cache artifact: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nCACHE BUILD ARTIFACT CHANGED\n"
            f"file     : {path.name}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )

    print(
        f"[EXACT] {path.name}"
    )


# =============================================================================
# 7. VERIFY METADATA CONTENT
# =============================================================================

metadata = json.loads(
    METADATA_SOURCE.read_text(
        encoding="utf-8"
    )
)


if metadata[
    "cache"
] != "stage23_execution_cache_v1":

    raise RuntimeError(
        "Unexpected cache identifier."
    )


if metadata[
    "status"
] != "BUILT_AND_BYTE_VERIFIED":

    raise RuntimeError(
        "Cache metadata is not in verified state."
    )


if metadata[
    "execution_parent_commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Cache parent commit mismatch."
    )


if metadata[
    "protocol"
][
    "commit"
] != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Cache protocol commit mismatch."
    )


if metadata[
    "shape"
][
    "rows"
] != N_DEVELOPMENT:

    raise RuntimeError(
        "Cache row count changed."
    )


if metadata[
    "shape"
][
    "features"
] != N_FEATURES:

    raise RuntimeError(
        "Cache feature count changed."
    )


if metadata[
    "feature_dtype"
] != "float64":

    raise RuntimeError(
        "Cache feature dtype changed."
    )


semantic = metadata[
    "semantic_equivalence"
]


if semantic[
    "source_row_major_70f_sha256"
] != EXPECTED_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Source semantic 70F digest mismatch."
    )


if semantic[
    "cache_reconstructed_row_major_70f_sha256"
] != EXPECTED_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Cache semantic 70F digest mismatch."
    )


for key in [
    "exact_byte_equality",
    "per_feature_exact_byte_equality",
    "label_exact_byte_equality",
    "day_id_exact_byte_equality",
    "clean_position_exact_byte_equality",
    "original_row_index_exact_byte_equality",
]:

    if semantic[key] is not True:

        raise RuntimeError(
            f"Cache equivalence assertion failed: {key}"
        )


if len(
    metadata[
        "full_feature_order"
    ]
) != 70:

    raise RuntimeError(
        "Frozen cache feature order length changed."
    )


if len(
    metadata[
        "feature_columns"
    ]
) != 70:

    raise RuntimeError(
        "Expected exactly 70 cache feature records."
    )


# =============================================================================
# 8. VERIFY PRIMARY SUBSET PROJECTION COUNTS
# =============================================================================

EXPECTED_SUBSET_COUNTS = {

    "FULL":
        70,

    "NO_DST_PORT":
        69,

    "NO_PORTS":
        68,

    "NO_INIT_FWD_WIN_BYTS":
        69,

    "NO_FWD_SEG_SIZE_MIN":
        69,

    "NO_SUSPICIOUS_GROUP":
        67,

    "BEHAVIOR_ONLY":
        63,
}


subset_indices = metadata[
    "primary_subset_feature_indices"
]


for subset, expected_count in EXPECTED_SUBSET_COUNTS.items():

    if subset not in subset_indices:

        raise RuntimeError(
            f"Missing cache projection: {subset}"
        )

    if len(
        subset_indices[
            subset
        ]
    ) != expected_count:

        raise RuntimeError(
            f"Projection count mismatch: {subset}"
        )


print()
print(
    "[OK] primary subset projections:"
    " 7 / 7 exact"
)


# =============================================================================
# 9. VERIFY FROZEN SPLITS IN METADATA
# =============================================================================

split_validation = metadata[
    "split_validation"
]


random_split = split_validation[
    "RANDOM_NATURAL"
]

chrono_split = split_validation[
    "CHRONOLOGICAL_NATURAL"
]


if random_split[
    "train"
][
    "rows"
] != EXPECTED_RANDOM_TRAIN_ROWS:

    raise RuntimeError(
        "Random train count mismatch."
    )


if random_split[
    "validation"
][
    "rows"
] != EXPECTED_RANDOM_VAL_ROWS:

    raise RuntimeError(
        "Random validation count mismatch."
    )


if random_split[
    "membership_bitset_sha256"
] != EXPECTED_RANDOM_BITSET_SHA256:

    raise RuntimeError(
        "Random split bitset digest mismatch."
    )


if chrono_split[
    "train"
][
    "rows"
] != EXPECTED_CHRONO_TRAIN_ROWS:

    raise RuntimeError(
        "Chronological train count mismatch."
    )


if chrono_split[
    "validation"
][
    "rows"
] != EXPECTED_CHRONO_VAL_ROWS:

    raise RuntimeError(
        "Chronological validation count mismatch."
    )


if chrono_split[
    "train"
][
    "day_ids"
] != [
    0, 1, 2, 3, 4, 5, 6
]:

    raise RuntimeError(
        "Chronological training days changed."
    )


if chrono_split[
    "validation"
][
    "day_ids"
] != [7]:

    raise RuntimeError(
        "Chronological validation day changed."
    )


print(
    "[OK] RANDOM_NATURAL split exact"
)

print(
    "[OK] CHRONOLOGICAL_NATURAL split exact"
)


# =============================================================================
# 10. VERIFY GOVERNANCE
# =============================================================================

governance = metadata[
    "governance"
]


if governance[
    "new_model_fits"
] != 0:

    raise RuntimeError(
        "Cache build unexpectedly consumed model fits."
    )


if governance[
    "stage23_models_fit_before"
] != 18:

    raise RuntimeError(
        "Unexpected pre-cache fit count."
    )


if governance[
    "stage23_models_fit_after"
] != 18:

    raise RuntimeError(
        "Unexpected post-cache fit count."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "model_specification_changed",
    "feature_definition_changed",
    "split_definition_changed",
    "input_dtype_changed",
    "stage23_0_modified",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
]:

    if governance[
        key
    ] is not False:

        raise RuntimeError(
            f"Cache governance violation: {key}"
        )


if metadata[
    "next_authorized_model_cell"
] != "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected next authorized model cell."
    )


print(
    "[OK] governance exact: 18 / 50 fits, zero new fits"
)


# =============================================================================
# 11. VERIFY BUILD RECEIPT
# =============================================================================

build_receipt = json.loads(
    BUILD_RECEIPT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if build_receipt[
    "status"
] != "CACHE_COMPLETE_UNSEALED":

    raise RuntimeError(
        "Unexpected cache build receipt status."
    )


if build_receipt[
    "execution_parent_commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Build receipt parent mismatch."
    )


if build_receipt[
    "rows"
] != N_DEVELOPMENT:

    raise RuntimeError(
        "Build receipt row count mismatch."
    )


if build_receipt[
    "feature_count"
] != 70:

    raise RuntimeError(
        "Build receipt feature count mismatch."
    )


if build_receipt[
    "feature_dtype"
] != "float64":

    raise RuntimeError(
        "Build receipt feature dtype mismatch."
    )


if build_receipt[
    "semantic_row_major_70f_sha256"
] != EXPECTED_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Build receipt cache semantic digest mismatch."
    )


if build_receipt[
    "source_semantic_row_major_70f_sha256"
] != EXPECTED_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Build receipt source semantic digest mismatch."
    )


if build_receipt[
    "semantic_byte_equivalence"
] is not True:

    raise RuntimeError(
        "Build receipt does not assert byte equivalence."
    )


if build_receipt[
    "data_checksums_manifest_sha256"
] != EXPECTED_DATA_CHECKSUMS_SHA256:

    raise RuntimeError(
        "Build receipt checksum-manifest digest mismatch."
    )


if build_receipt[
    "metadata_sha256"
] != EXPECTED_METADATA_SHA256:

    raise RuntimeError(
        "Build receipt metadata digest mismatch."
    )


if build_receipt[
    "model_fits_this_action"
] != 0:

    raise RuntimeError(
        "Build receipt reports model fits."
    )


if build_receipt[
    "stage23_models_fit_total"
] != 18:

    raise RuntimeError(
        "Build receipt fit count mismatch."
    )


if build_receipt[
    "raw_mar1_accessed"
] is not False:

    raise RuntimeError(
        "Unexpected Mar1 access flag."
    )


if build_receipt[
    "raw_mar2_accessed"
] is not False:

    raise RuntimeError(
        "Unexpected Mar2 access flag."
    )


# =============================================================================
# 12. PHYSICAL CACHE RE-HASH
# =============================================================================
#
# Re-read every physical cache file listed in the frozen manifest.
#
# This verifies that the ~7.76 GiB cache on disk still exactly matches
# the manifest produced during construction.
#
# =============================================================================

print()
print("=" * 100)
print("RE-HASHING PHYSICAL CACHE AGAINST FROZEN MANIFEST")
print("=" * 100)
print()


manifest_entries = []


for line in DATA_CHECKSUMS_SOURCE.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, relative_path = line.split(
        "  ",
        1,
    )

    manifest_entries.append(
        (
            digest,
            relative_path,
        )
    )


if len(
    manifest_entries
) != 75:

    raise RuntimeError(
        "\nUnexpected cache manifest entry count.\n"
        f"expected: 75\n"
        f"actual  : {len(manifest_entries)}"
    )


verified_entries = 0
verified_bytes = 0


for expected_sha, relative_path in manifest_entries:

    path = (
        CACHE_DIR
        / relative_path
    )


    if not path.exists():

        raise RuntimeError(
            f"Physical cache file missing:\n{path}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nPHYSICAL CACHE MUTATION DETECTED\n"
            f"file     : {relative_path}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


    verified_entries += 1

    verified_bytes += (
        path.stat().st_size
    )


    if (
        verified_entries <= 5
        or verified_entries % 10 == 0
        or verified_entries == len(manifest_entries)
    ):

        print(
            f"  verified {verified_entries:>2}/"
            f"{len(manifest_entries)}"
            f"  ({verified_bytes / 1024**3:.3f} GiB read)"
        )


print()
print(
    f"[OK] physical cache: "
    f"{verified_entries}/{len(manifest_entries)} files exact"
)

print(
    f"[OK] bytes verified : "
    f"{verified_bytes / 1024**3:.3f} GiB"
)


# =============================================================================
# 13. CREATE SMALL REPOSITORY SEAL DIRECTORY
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


for source_path in [
    DATA_CHECKSUMS_SOURCE,
    METADATA_SOURCE,
    BUILD_RECEIPT_SOURCE,
]:

    shutil.copy2(
        source_path,
        TARGET_DIR
        / source_path.name,
    )


# =============================================================================
# 14. VERIFY COPIES
# =============================================================================

for source_path in [
    DATA_CHECKSUMS_SOURCE,
    METADATA_SOURCE,
    BUILD_RECEIPT_SOURCE,
]:

    destination = (
        TARGET_DIR
        / source_path.name
    )

    if (
        sha256_file(
            source_path
        )
        !=
        sha256_file(
            destination
        )
    ):

        raise RuntimeError(
            f"Repository copy mismatch: {source_path.name}"
        )


print()
print(
    "[OK] verification metadata copied byte-exactly"
)


# =============================================================================
# 15. CREATE CACHE SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT_PATH = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-Execution-Cache-V1",

    "status":
        "EXECUTION_CACHE_VERIFICATION_FROZEN",

    "sealed_utc":
        now_utc(),

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "cache_tag":
        CACHE_TAG,

    "cache_storage": {

        "runtime_path":
            str(
                CACHE_DIR
            ),

        "large_binary_cache_committed_to_git":
            False,

        "rows":
            N_DEVELOPMENT,

        "feature_count":
            N_FEATURES,

        "feature_dtype":
            "float64",

        "physical_manifest_entries":
            len(
                manifest_entries
            ),

        "physical_bytes_verified":
            verified_bytes,
    },

    "frozen_build_artifacts": {

        "data_checksums.sha256":
            EXPECTED_DATA_CHECKSUMS_SHA256,

        "metadata.json":
            EXPECTED_METADATA_SHA256,

        "cache_build_receipt.json":
            EXPECTED_CACHE_BUILD_RECEIPT_SHA256,
    },

    "semantic_equivalence": {

        "old_stage23_row_major_70f_sha256":
            EXPECTED_SEMANTIC_70F_SHA256,

        "cache_reconstructed_row_major_70f_sha256":
            EXPECTED_SEMANTIC_70F_SHA256,

        "exact_byte_equivalence":
            True,

        "per_feature_exact_byte_equivalence":
            True,

        "labels_exact":
            True,

        "clean_positions_exact":
            True,

        "day_ids_exact":
            True,

        "original_row_indices_exact":
            True,

        "random_membership_exact":
            True,

        "chronological_membership_exact":
            True,
    },

    "governance": {

        "stage23_models_fit_before":
            18,

        "new_model_fits":
            0,

        "stage23_models_fit_after":
            18,

        "feature_definitions_changed":
            False,

        "split_definitions_changed":
            False,

        "input_dtype_changed":
            False,

        "model_specification_changed":
            False,

        "thresholds_changed":
            False,

        "ensemble_definition_changed":
            False,

        "stage23_0_modified":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,
    },

    "next_authorized_model_cell":
        "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT_PATH.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 16. README
# =============================================================================

README_PATH = (
    TARGET_DIR
    / "README.md"
)


README_PATH.write_text(
f"""# Stage23 Deterministic Execution Cache V1

This directory freezes the verification metadata for the Stage23
execution-cache optimization introduced after Stage23-1I.

## Purpose

The cache removes repeated Parquet decompression and Python/PyArrow column
conversion from subsequent Stage23 execution.

The large binary cache itself is intentionally **not stored in Git**.

Runtime location:

`/kaggle/working/stage23_execution_cache_v1`

## Canonical corpus

- Development rows: {N_DEVELOPMENT:,}
- Frozen features: 70
- Feature dtype: float64
- Source days: February 14 through February 28 only
- Raw March 1 access: no
- Raw March 2 access: no

## Exact semantic equivalence

Old Stage23 row-major 70-feature semantic SHA256:

`{EXPECTED_SEMANTIC_70F_SHA256}`

New cache reconstructed row-major 70-feature semantic SHA256:

`{EXPECTED_SEMANTIC_70F_SHA256}`

Therefore the cache is byte-equivalent to the previously used Stage23
float64 materialization semantics.

Verified:

- all 70 feature columns
- binary labels
- day IDs
- clean positions
- original row indices
- RANDOM_NATURAL membership
- CHRONOLOGICAL_NATURAL membership

## Frozen split counts

### RANDOM_NATURAL

- Train: 11,529,922
- Validation: 2,882,481

### CHRONOLOGICAL_NATURAL

- Train: 13,818,623
- Validation: 593,780
- Train day IDs: 0–6
- Validation day ID: 7

## Scientific governance

This optimization consumed **zero model fits**.

Stage23 fit accounting remains:

`18 / 50`

No change was made to:

- feature definitions
- feature order
- row membership
- input dtype
- model parameters
- model seeds
- thresholds
- ensemble definition
- Stage23-0 protocol

## Next authorized model cell

`Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL`
""",
    encoding="utf-8",
)


# =============================================================================
# 17. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPOSITORY_CHECKSUMS_PATH = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_files = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPOSITORY_CHECKSUMS_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_files
    ) + "\n",
    encoding="utf-8",
)


REPOSITORY_MANIFEST_SHA = (
    sha256_file(
        REPOSITORY_CHECKSUMS_PATH
    )
)


# =============================================================================
# 18. STAGE ONLY CACHE VERIFICATION METADATA
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(
            relative_target
        ),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


expected_prefix = (
    str(
        relative_target
    )
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        expected_prefix
    )
]


if unexpected:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 100)
print("STAGED CACHE VERIFICATION ARTIFACTS")
print("=" * 100)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 19. COMMIT
# =============================================================================

print()
print("=" * 100)
print("CREATING EXECUTION CACHE SEAL COMMIT")
print("=" * 100)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)


if sealed_commit == EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
):

    raise RuntimeError(
        "Repository dirty after cache seal commit."
    )


print()
print("Cache seal commit:")
print(" ", sealed_commit)


# =============================================================================
# 20. ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        CACHE_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ]
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_TAG,
    ]
)


if local_tag_commit != sealed_commit:

    raise RuntimeError(
        "Local cache tag verification failed."
    )


# =============================================================================
# 21. PUSH MAIN + CACHE TAG
# =============================================================================

print()
print("=" * 100)
print("PUSHING EXECUTION CACHE SEAL")
print("=" * 100)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        CACHE_TAG,
    ],
    show=True,
)


# =============================================================================
# 22. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
)


if not remote_main_output:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 23. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{CACHE_TAG}",
        f"refs/tags/{CACHE_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{CACHE_TAG}":

        tag_object = sha

    elif ref == f"refs/tags/{CACHE_TAG}^{{}}":

        peeled_commit = sha


if not tag_object:

    raise RuntimeError(
        "Remote cache annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote cache tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 24. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)


if final_status:

    raise RuntimeError(
        "Repository dirty after cache seal push."
    )


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23 EXECUTION CACHE V1 — SEALED")
print("=" * 100)

print()
print("Stage23-1I scientific parent:")
print(
    " ",
    EXPECTED_PARENT
)

print()
print("Cache verification commit:")
print(
    " ",
    sealed_commit
)

print()
print("Cache tag:")
print(
    " ",
    CACHE_TAG
)

print()
print("Remote main:")
print(
    " ",
    remote_main
)

print()
print("Remote tag object:")
print(
    " ",
    tag_object
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    peeled_commit
)


print()
print("=" * 100)
print("FROZEN CACHE IDENTITY")
print("=" * 100)

print()
print("Development rows:")
print(
    " ",
    f"{N_DEVELOPMENT:,}"
)

print()
print("Feature count:")
print(
    " ",
    N_FEATURES
)

print()
print("Feature dtype:")
print(
    "  float64"
)

print()
print("Semantic 70F SHA256:")
print(
    " ",
    EXPECTED_SEMANTIC_70F_SHA256
)

print()
print("Data checksum manifest SHA256:")
print(
    " ",
    EXPECTED_DATA_CHECKSUMS_SHA256
)

print()
print("Metadata SHA256:")
print(
    " ",
    EXPECTED_METADATA_SHA256
)

print()
print("Cache build receipt SHA256:")
print(
    " ",
    EXPECTED_CACHE_BUILD_RECEIPT_SHA256
)

print()
print("Repository checksum manifest SHA256:")
print(
    " ",
    REPOSITORY_MANIFEST_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits sealed        : 18 / 50")
print("Model fits cache build     : 0")
print("Model fits cache seal      : 0")
print("Feature definitions changed: NO")
print("Split definitions changed  : NO")
print("Input dtype changed        : NO")
print("Model specification changed: NO")
print("Thresholds changed         : NO")
print("Ensemble changed           : NO")
print("Raw Mar1 read              : NO")
print("Raw Mar2 read              : NO")
print("Stage23-0 changed          : NO")
print("Git status                 : CLEAN")


print()
print("=" * 100)
print("NEXT AUTHORIZED MODEL CELL")
print("=" * 100)

print()
print(
    "Stage23-1J — "
    "NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL"
)

print()
print(
    "Stage23-1J may now use the sealed deterministic execution cache."
)

print("=" * 100)

STAGE23 — EXECUTION CACHE V1 VERIFICATION SEAL

[OK] branch            : main
[OK] parent HEAD       : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] Stage23-0 tag     : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1I tag    : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] worktree          : CLEAN
[OK] cache tag absent

VERIFYING CACHE BUILD ARTIFACTS

[EXACT] data_checksums.sha256
[EXACT] metadata.json
[EXACT] cache_build_receipt.json

[OK] primary subset projections: 7 / 7 exact
[OK] RANDOM_NATURAL split exact
[OK] CHRONOLOGICAL_NATURAL split exact
[OK] governance exact: 18 / 50 fits, zero new fits

RE-HASHING PHYSICAL CACHE AGAINST FROZEN MANIFEST

  verified  1/75  (0.107 GiB read)
  verified  2/75  (0.215 GiB read)
  verified  3/75  (0.322 GiB read)
  verified  4/75  (0.430 GiB read)
  verified  5/75  (0.537 GiB read)
  verified 10/75  (1.074 GiB read)
  verified 20/75  (2.148 GiB read)
  verified 30/75  (3.221 GiB read)
  verified 40/75  (4.295 GiB read)
  verified 50/75 

In [23]:
# =============================================================================
# STAGE23-1J
# NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL
# OPTIMIZED — SEALED EXECUTION CACHE V1
# =============================================================================
#
# Frozen ablation removes EXACTLY:
#   - Dst Port
#   - Init Fwd Win Byts
#   - Fwd Seg Size Min
#
# Retained features: 67
# Semantic label: joint_shortcut_prone_group_ablation
#
# REQUIRED EXECUTION PARENT:
#   Stage23 deterministic execution-cache seal
#   66e1c6382caa63699766e605df63df02beff894f
#
# SCIENTIFIC RANDOM PARTNER:
#   Stage23-1I
#   a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
#
# FIT COUNT:
#   before : 18 / 50
#   this   :  2
#   after  : 20 / 50
#
# INPUT PATH:
#   SEALED EXECUTION CACHE ONLY
#
# NO:
#   Parquet decoding
#   raw Mar1
#   raw Mar2
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#
# FAIL CLOSED:
#   If a model fit completes and anything later fails, DO NOT blindly rerun.
#   Inspect execution_state.json and preserve OUTPUT_DIR.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1J"

TARGET_SUBSET = "NO_SUSPICIOUS_GROUP"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

SCIENTIFIC_PREDECESSOR_COMMIT = (
    "a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0"
)

SCIENTIFIC_PREDECESSOR_TAG = (
    "stage23-1i-no-suspicious-group-random-natural-v1"
)

SCIENTIFIC_PREDECESSOR_RESULT_SHA256 = (
    "20b2b6491d4e85738d4ab6a52579233ce3bc660164a3742ca9ccb3a84a005fde"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_DATA_MANIFEST_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

CACHE_BUILD_RECEIPT_SHA256 = (
    "024e8724cd0dd672f44015af909bf4f9b620f44b03e4ed09c48ac19ecc35a8f2"
)

CACHE_SEMANTIC_70F_SHA256 = (
    "8c43ab0e36a65de1c095acb1ebf9e7d029a9e2b1cc353455a66617db8a899bb2"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

CACHE_SEAL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

RANDOM_PARTNER_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "no_suspicious_group"
)

RANDOM_PARTNER_RESULT = (
    RANDOM_PARTNER_DIR
    / "stage23_1i_no_suspicious_group_random_natural_result.json"
)

RANDOM_PARTNER_SEAL = (
    RANDOM_PARTNER_DIR
    / "seal_receipt.json"
)

FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1j_no_suspicious_group_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1j_no_suspicious_group_chronological_natural_result.json"
)


# =============================================================================
# 2. FROZEN SHAPE
# =============================================================================

N_DEVELOPMENT = 14_412_403

TRAIN_ROWS = 13_818_623
VALIDATION_ROWS = 593_780

TRAIN_BENIGN = 11_908_580
TRAIN_ATTACK = 1_910_043

VALIDATION_BENIGN = 531_524
VALIDATION_ATTACK = 62_256

N_FEATURES = 67


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    temp = path.with_suffix(
        path.suffix + ".tmp"
    )

    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    temp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    resolved = Path(path).resolve()

    lowered = str(
        resolved
    ).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in lowered:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {resolved}\n"
                f"marker : {marker}"
            )

    return resolved


# =============================================================================
# 4. INITIAL FAIL-CLOSED CHECK
# =============================================================================

if OUTPUT_DIR.exists():

    state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1J OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"{json.dumps(state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


# =============================================================================
# 5. GIT / TAG PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1J — NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL")
print("OPTIMIZED EXECUTION CACHE V1")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

random_partner_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        SCIENTIFIC_PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_SEAL_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != CACHE_SEAL_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1J MUST START FROM SEALED EXECUTION CACHE COMMIT.\n"
        f"expected: {CACHE_SEAL_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if random_partner_tag_commit != SCIENTIFIC_PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1I scientific predecessor tag mismatch."
    )


if cache_tag_commit != CACHE_SEAL_COMMIT:

    raise RuntimeError(
        "Execution-cache seal tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean before Stage23-1J:\n"
        + status
    )


print("[OK] branch             :", branch)
print("[OK] HEAD               :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1I tag     :", random_partner_tag_commit)
print("[OK] cache tag          :", cache_tag_commit)
print("[OK] worktree           : CLEAN")


# =============================================================================
# 6. VERIFY CACHE SEAL
# =============================================================================

CACHE_SEAL_RECEIPT = (
    CACHE_SEAL_DIR
    / "seal_receipt.json"
)

CACHE_METADATA_REPO = (
    CACHE_SEAL_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_REPO = (
    CACHE_SEAL_DIR
    / "data_checksums.sha256"
)

CACHE_BUILD_RECEIPT_REPO = (
    CACHE_SEAL_DIR
    / "cache_build_receipt.json"
)


for path in [
    CACHE_SEAL_RECEIPT,
    CACHE_METADATA_REPO,
    CACHE_DATA_MANIFEST_REPO,
    CACHE_BUILD_RECEIPT_REPO,
]:

    if not path.exists():

        raise RuntimeError(
            f"Missing sealed cache artifact: {path}"
        )


if sha256_file(
    CACHE_METADATA_REPO
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Sealed cache metadata SHA256 mismatch."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_REPO
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Sealed cache data-manifest SHA256 mismatch."
    )


if sha256_file(
    CACHE_BUILD_RECEIPT_REPO
) != CACHE_BUILD_RECEIPT_SHA256:

    raise RuntimeError(
        "Sealed cache build-receipt SHA256 mismatch."
    )


cache_seal = json.loads(
    CACHE_SEAL_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if cache_seal[
    "status"
] != "EXECUTION_CACHE_VERIFICATION_FROZEN":

    raise RuntimeError(
        "Cache is not scientifically sealed."
    )


if cache_seal[
    "governance"
][
    "stage23_models_fit_after"
] != 18:

    raise RuntimeError(
        "Cache seal Stage23 fit accounting mismatch."
    )


if cache_seal[
    "governance"
][
    "new_model_fits"
] != 0:

    raise RuntimeError(
        "Cache optimization unexpectedly consumed model fits."
    )


if cache_seal[
    "next_authorized_model_cell"
] != "Stage23-1J NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Cache seal does not authorize Stage23-1J."
    )


print("[OK] cache seal          : FROZEN")
print("[OK] sealed fits before  : 18 / 50")
print("[OK] Stage23-1J          : AUTHORIZED")


# =============================================================================
# 7. VERIFY RUNTIME CACHE SMALL ARTIFACTS AGAINST SEALED REPO
# =============================================================================

CACHE_METADATA_RUNTIME = guard_path(
    CACHE_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_RUNTIME = guard_path(
    CACHE_DIR
    / "data_checksums.sha256"
)

CACHE_BUILD_RECEIPT_RUNTIME = guard_path(
    CACHE_DIR
    / "cache_build_receipt.json"
)


for path in [
    CACHE_METADATA_RUNTIME,
    CACHE_DATA_MANIFEST_RUNTIME,
    CACHE_BUILD_RECEIPT_RUNTIME,
]:

    if not path.exists():

        raise RuntimeError(
            f"Runtime execution-cache artifact missing: {path}"
        )


if sha256_file(
    CACHE_METADATA_RUNTIME
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Runtime cache metadata differs from sealed metadata."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_RUNTIME
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Runtime cache manifest differs from sealed manifest."
    )


if sha256_file(
    CACHE_BUILD_RECEIPT_RUNTIME
) != CACHE_BUILD_RECEIPT_SHA256:

    raise RuntimeError(
        "Runtime cache build receipt differs from sealed receipt."
    )


cache_metadata = json.loads(
    CACHE_METADATA_RUNTIME.read_text(
        encoding="utf-8"
    )
)


if cache_metadata[
    "semantic_equivalence"
][
    "cache_reconstructed_row_major_70f_sha256"
] != CACHE_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Runtime cache semantic identity mismatch."
    )


if cache_metadata[
    "semantic_equivalence"
][
    "exact_byte_equality"
] is not True:

    raise RuntimeError(
        "Runtime cache equivalence flag is false."
    )


print("[OK] runtime cache metadata exactly matches sealed cache")


# =============================================================================
# 8. VERIFY STAGE23-0 BYTES
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum manifest changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-0 artifact: {filename}"
        )


    if sha256_file(
        path
    ) != expected_sha:

        raise RuntimeError(
            f"Stage23-0 artifact changed: {filename}"
        )


print(
    f"[OK] Stage23-0 artifacts : "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 9. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = (
    subset_spec[
        "features"
    ]
)

REMOVED_FEATURES = (
    subset_spec[
        "removed"
    ]
)


EXPECTED_REMOVED = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "Frozen suspicious-group definition changed."
    )


if len(
    FEATURES
) != N_FEATURES:

    raise RuntimeError(
        f"Expected 67 retained features; found {len(FEATURES)}"
    )


if subset_spec[
    "semantic_label"
] != "joint_shortcut_prone_group_ablation":

    raise RuntimeError(
        "Frozen suspicious-group semantic label changed."
    )


for feature in EXPECTED_REMOVED:

    if feature in FEATURES:

        raise RuntimeError(
            f"Removed feature still present: {feature}"
        )


# =============================================================================
# 10. VERIFY CACHE FEATURE PROJECTION
# =============================================================================

FULL_FEATURES = (
    cache_metadata[
        "full_feature_order"
    ]
)


if FULL_FEATURES != feature_spec[
    "full_feature_order"
]:

    raise RuntimeError(
        "Cache full feature order differs from frozen protocol."
    )


CACHE_INDICES = (
    cache_metadata[
        "primary_subset_feature_indices"
    ][
        TARGET_SUBSET
    ]
)


if len(
    CACHE_INDICES
) != 67:

    raise RuntimeError(
        "Cache suspicious-group projection count mismatch."
    )


CACHE_PROJECTED_FEATURES = [
    FULL_FEATURES[
        index
    ]
    for index in CACHE_INDICES
]


if CACHE_PROJECTED_FEATURES != FEATURES:

    raise RuntimeError(
        "Cache suspicious-group feature order differs from protocol."
    )


print()
print("Frozen target:")
print(
    "  subset        :",
    TARGET_SUBSET
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  retained      :",
    len(FEATURES)
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)
print(
    "  data source   : SEALED EXECUTION CACHE V1"
)


# =============================================================================
# 11. VERIFY FROZEN CHRONO SPLIT
# =============================================================================

chrono_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


if int(
    chrono_spec[
        "train"
    ][
        "rows"
    ]
) != TRAIN_ROWS:

    raise RuntimeError(
        "Chronological train count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "rows"
    ]
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Chronological validation count changed."
    )


if int(
    chrono_spec[
        "train"
    ][
        "benign"
    ]
) != TRAIN_BENIGN:

    raise RuntimeError(
        "Chronological train benign count changed."
    )


if int(
    chrono_spec[
        "train"
    ][
        "attack"
    ]
) != TRAIN_ATTACK:

    raise RuntimeError(
        "Chronological train attack count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "benign"
    ]
) != VALIDATION_BENIGN:

    raise RuntimeError(
        "Chronological validation benign count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "attack"
    ]
) != VALIDATION_ATTACK:

    raise RuntimeError(
        "Chronological validation attack count changed."
    )


print()
print("[OK] frozen CHRONOLOGICAL_NATURAL membership")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 12. VERIFY SEALED RANDOM PARTNER
# =============================================================================

if not RANDOM_PARTNER_RESULT.exists():

    raise RuntimeError(
        "Sealed Stage23-1I random result missing."
    )


if sha256_file(
    RANDOM_PARTNER_RESULT
) != SCIENTIFIC_PREDECESSOR_RESULT_SHA256:

    raise RuntimeError(
        "Stage23-1I random result SHA256 changed."
    )


if not RANDOM_PARTNER_SEAL.exists():

    raise RuntimeError(
        "Stage23-1I seal receipt missing."
    )


random_seal = json.loads(
    RANDOM_PARTNER_SEAL.read_text(
        encoding="utf-8"
    )
)


if random_seal[
    "stage23_models_fit_total"
] != 18:

    raise RuntimeError(
        "Stage23-1I seal fit count mismatch."
    )


random_result = json.loads(
    RANDOM_PARTNER_RESULT.read_text(
        encoding="utf-8"
    )
)


if random_result[
    "cell"
][
    "subset"
] != TARGET_SUBSET:

    raise RuntimeError(
        "Matched random subset mismatch."
    )


if random_result[
    "cell"
][
    "split"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Matched random split mismatch."
    )


print("[OK] matched Stage23-1I RANDOM_NATURAL result exact")


# =============================================================================
# 13. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] frozen package versions verified")


# =============================================================================
# 14. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 15. PARSE SEALED CACHE MANIFEST
# =============================================================================

manifest_hashes = {}


for line in CACHE_DATA_MANIFEST_RUNTIME.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, relative_path = line.split(
        "  ",
        1,
    )

    manifest_hashes[
        relative_path
    ] = digest


if len(
    manifest_hashes
) != 75:

    raise RuntimeError(
        f"Expected 75 cache-manifest entries; found {len(manifest_hashes)}"
    )


feature_records = {
    int(record["index"]):
        record
    for record in cache_metadata[
        "feature_columns"
    ]
}


SELECTED_FEATURE_RECORDS = []


for index in CACHE_INDICES:

    if index not in feature_records:

        raise RuntimeError(
            f"Missing cache feature record: {index}"
        )

    record = feature_records[
        index
    ]

    if record[
        "feature"
    ] != FULL_FEATURES[
        index
    ]:

        raise RuntimeError(
            f"Cache feature record/name mismatch at index {index}"
        )

    relative_path = record[
        "path"
    ]

    if relative_path not in manifest_hashes:

        raise RuntimeError(
            f"Feature absent from sealed manifest: {relative_path}"
        )

    if record[
        "sha256"
    ] != manifest_hashes[
        relative_path
    ]:

        raise RuntimeError(
            f"Feature metadata/manifest SHA disagreement: {relative_path}"
        )

    physical_path = guard_path(
        CACHE_DIR
        / relative_path
    )

    if not physical_path.exists():

        raise RuntimeError(
            f"Cache feature file missing: {physical_path}"
        )

    expected_bytes = (
        N_DEVELOPMENT
        * 8
    )

    if physical_path.stat().st_size != expected_bytes:

        raise RuntimeError(
            "\nCache feature size mismatch.\n"
            f"feature : {record['feature']}\n"
            f"expected: {expected_bytes:,}\n"
            f"actual  : {physical_path.stat().st_size:,}"
        )

    SELECTED_FEATURE_RECORDS.append(
        {
            **record,
            "physical_path":
                physical_path,
        }
    )


# =============================================================================
# 16. VERIFY CACHE METADATA ARRAYS
# =============================================================================

LABEL_PATH = guard_path(
    CACHE_DIR
    / "binary_label.uint8.dat"
)

DAY_ID_PATH = guard_path(
    CACHE_DIR
    / "day_id.uint8.dat"
)

CLEAN_POSITION_PATH = guard_path(
    CACHE_DIR
    / "clean_position.int64.dat"
)


for path in [
    LABEL_PATH,
    DAY_ID_PATH,
    CLEAN_POSITION_PATH,
]:

    if not path.exists():

        raise RuntimeError(
            f"Required cache metadata array missing: {path}"
        )


if sha256_file(
    LABEL_PATH
) != manifest_hashes[
    "binary_label.uint8.dat"
]:

    raise RuntimeError(
        "Physical binary-label cache differs from sealed manifest."
    )


if sha256_file(
    DAY_ID_PATH
) != manifest_hashes[
    "day_id.uint8.dat"
]:

    raise RuntimeError(
        "Physical day_id cache differs from sealed manifest."
    )


if sha256_file(
    CLEAN_POSITION_PATH
) != manifest_hashes[
    "clean_position.int64.dat"
]:

    raise RuntimeError(
        "Physical clean_position cache differs from sealed manifest."
    )


# =============================================================================
# 17. OPEN CACHE METADATA ARRAYS
# =============================================================================

labels_all = np.memmap(
    LABEL_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


day_ids_all = np.memmap(
    DAY_ID_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


clean_positions_all = np.memmap(
    CLEAN_POSITION_PATH,
    dtype="<i8",
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


if not np.all(
    day_ids_all[
        :TRAIN_ROWS
    ] <= 6
):

    raise RuntimeError(
        "Chronological training cache contains unexpected day_id."
    )


if not np.all(
    day_ids_all[
        TRAIN_ROWS:
    ] == 7
):

    raise RuntimeError(
        "Chronological validation cache is not exactly day_id 7."
    )


y_train = labels_all[
    :TRAIN_ROWS
]

y_validation = labels_all[
    TRAIN_ROWS:
]


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if (
    train_attack != TRAIN_ATTACK
    or train_benign != TRAIN_BENIGN
):

    raise RuntimeError(
        "Chronological training labels differ from frozen split."
    )


if (
    validation_attack != VALIDATION_ATTACK
    or validation_benign != VALIDATION_BENIGN
):

    raise RuntimeError(
        "Chronological validation labels differ from frozen split."
    )


validation_clean_positions = np.asarray(
    clean_positions_all[
        TRAIN_ROWS:
    ],
    dtype=np.int64,
)


expected_validation_positions = np.arange(
    TRAIN_ROWS,
    N_DEVELOPMENT,
    dtype=np.int64,
)


if not np.array_equal(
    validation_clean_positions,
    expected_validation_positions,
):

    raise RuntimeError(
        "Chronological validation clean_position membership changed."
    )


del expected_validation_positions


# =============================================================================
# 18. STORAGE PREFLIGHT FOR TEMPORARY 67F ROW-MAJOR MATRIX
# =============================================================================

TEMP_MATRIX_BYTES = (
    N_DEVELOPMENT
    * N_FEATURES
    * 8
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


# Keep 1 GiB safety margin beyond temporary row-major matrix.
minimum_free = (
    TEMP_MATRIX_BYTES
    + 1 * 1024**3
)


print()
print("=" * 100)
print("OPTIMIZED MATERIALIZATION PLAN")
print("=" * 100)
print()

print(
    "Source               : sealed raw-column cache"
)
print(
    "Parquet files read   : 0"
)
print(
    "Development rows     :",
    f"{N_DEVELOPMENT:,}"
)
print(
    "Selected features    :",
    N_FEATURES
)
print(
    "Temporary matrix     :",
    f"{TEMP_MATRIX_BYTES / 1024**3:.3f} GiB"
)
print(
    "Working free         :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nINSUFFICIENT DISK FOR OPTIMIZED 67F MATERIALIZATION\n"
        f"required with safety margin: "
        f"{minimum_free / 1024**3:.3f} GiB\n"
        f"available                  : "
        f"{disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 19. CREATE FAIL-CLOSED OUTPUT STATE
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        CACHE_SEAL_COMMIT,

    "cache_seal_commit":
        CACHE_SEAL_COMMIT,

    "cache_seal_tag":
        CACHE_SEAL_TAG,

    "scientific_random_partner_commit":
        SCIENTIFIC_PREDECESSOR_COMMIT,

    "stage23_total_model_fits_before_cell":
        18,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 20. OPEN 67 SEALED RAW FEATURE COLUMNS
# =============================================================================

source_feature_mmaps = []


for record in SELECTED_FEATURE_RECORDS:

    source_feature_mmaps.append(
        np.memmap(
            record[
                "physical_path"
            ],
            dtype="<f8",
            mode="r",
            shape=(
                N_DEVELOPMENT,
            ),
        )
    )


# =============================================================================
# 21. OPTIMIZED ROW-MAJOR MATERIALIZATION
# =============================================================================
#
# Unlike previous Stage23 cells:
#
#   NO Parquet decode
#   NO PyArrow conversion
#   NO day-file traversal
#
# We only perform sequential local-cache reads + one temporary row-major write.
#
# =============================================================================

X_ALL_PATH = (
    RUNTIME_DIR
    / "X_no_suspicious_group_float64.dat"
)


X_all = np.memmap(
    X_ALL_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        N_DEVELOPMENT,
        N_FEATURES,
    ),
)


MATERIALIZE_CHUNK_ROWS = 250_000

materialize_start = (
    time.perf_counter()
)


print()
print("=" * 100)
print("MATERIALIZING 67F MATRIX FROM SEALED EXECUTION CACHE")
print("=" * 100)
print()


total_chunks = (
    N_DEVELOPMENT
    + MATERIALIZE_CHUNK_ROWS
    - 1
) // MATERIALIZE_CHUNK_ROWS


for chunk_index, start in enumerate(
    range(
        0,
        N_DEVELOPMENT,
        MATERIALIZE_CHUNK_ROWS,
    ),
    start=1,
):

    end = min(
        start
        + MATERIALIZE_CHUNK_ROWS,
        N_DEVELOPMENT,
    )

    n_rows = (
        end
        - start
    )


    chunk = np.empty(
        (
            n_rows,
            N_FEATURES,
        ),
        dtype=np.float64,
    )


    for output_column, source_mm in enumerate(
        source_feature_mmaps
    ):

        chunk[
            :,
            output_column
        ] = source_mm[
            start:end
        ]


    X_all[
        start:end,
        :
    ] = chunk


    if (
        chunk_index == 1
        or chunk_index % 10 == 0
        or chunk_index == total_chunks
    ):

        print(
            f"  chunk {chunk_index:>3}/{total_chunks}"
            f" — rows {start:,}..{end - 1:,}"
        )


    del chunk

    gc.collect()


X_all.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


if X_ALL_PATH.stat().st_size != TEMP_MATRIX_BYTES:

    raise RuntimeError(
        "\nTemporary optimized matrix size mismatch.\n"
        f"expected: {TEMP_MATRIX_BYTES:,}\n"
        f"actual  : {X_ALL_PATH.stat().st_size:,}"
    )


X_train = X_all[
    :TRAIN_ROWS,
    :
]

X_validation = X_all[
    TRAIN_ROWS:,
    :
]


if X_train.shape != (
    TRAIN_ROWS,
    N_FEATURES,
):

    raise RuntimeError(
        "Optimized training matrix shape mismatch."
    )


if X_validation.shape != (
    VALIDATION_ROWS,
    N_FEATURES,
):

    raise RuntimeError(
        "Optimized validation matrix shape mismatch."
    )


print()
print("[OK] optimized cache materialization complete")
print(
    "     train shape       :",
    X_train.shape
)
print(
    "     validation shape  :",
    X_validation.shape
)
print(
    "     dtype             :",
    X_all.dtype
)
print(
    "     seconds           :",
    f"{materialize_seconds:.3f}"
)
print(
    "     Parquet reads     : 0"
)


execution_state.update({

    "status":
        "MATERIALIZED_FROM_SEALED_CACHE",

    "materialization_seconds":
        materialize_seconds,

    "temporary_matrix_bytes":
        TEMP_MATRIX_BYTES,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,

    "input_dtype":
        "float64",
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 22. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 23. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 100)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 100)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 24. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 100)
print("FIT 2 / 2 — XGBOOST")
print("=" * 100)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    str(
        XGB_MODEL_PATH
    )
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 25. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 26. METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 27. FROZEN FULL CHRONO REFERENCE
# =============================================================================

full_chrono = json.loads(
    FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_chrono[
    "cell"
] != "CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected Stage22R FULL chronological reference."
    )


full_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 28. CHRONOLOGICAL PENALTIES
# =============================================================================

chrono_pr_penalty = float(
    full_pr_auc
    - pr_auc
)

chrono_roc_penalty = float(
    full_roc_auc
    - roc_auc
)

chrono_f1_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)

chrono_recall_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)

chrono_fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 29. SEALED RANDOM PENALTIES
# =============================================================================

random_pr_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "pr_auc"
    ]
)

random_roc_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "roc_auc"
    ]
)

random_f1_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "f1_at_0_50"
    ]
)

random_recall_penalty = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "recall_at_0_50"
    ]
)

random_fpr_change = float(
    random_result[
        "removal_penalty_full_minus_ablated"
    ][
        "fpr_at_0_50"
    ]
)


# =============================================================================
# 30. PRIMARY SPLIT × ABLATION INTERACTION
# =============================================================================

interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)

interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)

interaction_f1 = float(
    random_f1_penalty
    - chrono_f1_penalty
)

interaction_recall = float(
    random_recall_penalty
    - chrono_recall_penalty
)

interaction_fpr = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 31. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "no_suspicious_group_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=(
        validation_clean_positions
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 32. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "NO_SUSPICIOUS_GROUP_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent": {

        "commit":
            CACHE_SEAL_COMMIT,

        "tag":
            CACHE_SEAL_TAG,

        "type":
            "SEALED_DETERMINISTIC_EXECUTION_CACHE",
    },

    "scientific_random_partner": {

        "stage":
            "Stage23-1I",

        "commit":
            SCIENTIFIC_PREDECESSOR_COMMIT,

        "tag":
            SCIENTIFIC_PREDECESSOR_TAG,

        "result_sha256":
            SCIENTIFIC_PREDECESSOR_RESULT_SHA256,
    },

    "cache": {

        "identifier":
            "stage23_execution_cache_v1",

        "runtime_path":
            str(
                CACHE_DIR
            ),

        "sealed":
            True,

        "seal_commit":
            CACHE_SEAL_COMMIT,

        "seal_tag":
            CACHE_SEAL_TAG,

        "metadata_sha256":
            CACHE_METADATA_SHA256,

        "data_manifest_sha256":
            CACHE_DATA_MANIFEST_SHA256,

        "semantic_70f_sha256":
            CACHE_SEMANTIC_70F_SHA256,

        "parquet_files_read":
            0,

        "materialization_seconds":
            materialize_seconds,

        "temporary_row_major_matrix_bytes":
            TEMP_MATRIX_BYTES,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {

        "source":
            "Sealed Stage23-1I",

        "result_sha256":
            SCIENTIFIC_PREDECESSOR_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {

        "definition":
            "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1,

        "recall_at_0_50":
            interaction_recall,

        "fpr_at_0_50":
            interaction_fpr,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "optimized_cache_materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,

        "parquet_files_read":
            0,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            18,

        "stage23_total_model_fits_after_cell":
            20,

        "stage23_total_authorized_model_fits":
            50,

        "execution_cache_optimization":
            True,

        "execution_cache_scientifically_sealed":
            True,

        "parquet_files_read":
            0,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,

        "feature_definition_changed":
            False,

        "split_definition_changed":
            False,

        "input_dtype_changed":
            False,

        "model_specification_changed":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1J before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 33. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 34. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        20,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "optimized_cache_used":
        True,

    "parquet_files_read":
        0,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 35. REMOVE ONLY TEMPORARY ROW-MAJOR MATRIX
# =============================================================================
#
# DO NOT DELETE:
#   /kaggle/working/stage23_execution_cache_v1
#
# That sealed cache is reused by later Stage23 cells.
#
# =============================================================================

del X_train
del X_validation
del X_all

del y_train
del y_validation

del labels_all
del day_ids_all
del clean_positions_all

for index in range(
    len(source_feature_mmaps)
):

    source_feature_mmaps[
        index
    ] = None


del source_feature_mmaps

gc.collect()


if X_ALL_PATH.exists():

    X_ALL_PATH.unlink()


for path in [
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 36. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1J COMPLETE — RESULT UNSEALED")
print("=" * 100)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Frozen suspicious group:")
print(
    "  removed :",
    REMOVED_FEATURES
)
print(
    "  retained:",
    N_FEATURES
)
print(
    "  semantics:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Optimized execution:")
print(
    "  cache             : stage23_execution_cache_v1"
)
print(
    "  cache seal commit :",
    CACHE_SEAL_COMMIT
)
print(
    "  Parquet reads     : 0"
)
print(
    "  materialization   :",
    f"{materialize_seconds:.3f} s"
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 20 / 50")


print()
print("=" * 100)
print("CHRONOLOGICAL RANKING METRICS")
print("=" * 100)

print()
print(
    f"Attack prevalence                : {attack_prevalence:.12f}"
)

print(
    f"NO_SUSPICIOUS_GROUP PR-AUC       : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                      : {full_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty         : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"NO_SUSPICIOUS_GROUP ROC-AUC      : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                     : {full_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty        : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence              : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 100)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 100)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 100)
print("NO_SUSPICIOUS_GROUP SPLIT × ABLATION INTERACTION")
print("=" * 100)

print()
print("PR-AUC:")
print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")
print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary:")
print(
    f"  F1 interaction     : {interaction_f1:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 100)
print("ARTIFACTS")
print("=" * 100)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits completed : 20 / 50")
print("Optimized cache used   : YES")
print("Parquet files read     : 0")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")


print()
print("=" * 100)
print("NEXT ACTION")
print("=" * 100)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1J."
)

print("=" * 100)

STAGE23-1J — NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL
OPTIMIZED EXECUTION CACHE V1

[OK] branch             : main
[OK] HEAD               : 66e1c6382caa63699766e605df63df02beff894f
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1I tag     : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] cache tag          : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] cache seal          : FROZEN
[OK] sealed fits before  : 18 / 50
[OK] Stage23-1J          : AUTHORIZED
[OK] runtime cache metadata exactly matches sealed cache
[OK] Stage23-0 artifacts : 23/23 exact

Frozen target:
  subset        : NO_SUSPICIOUS_GROUP
  split         : CHRONOLOGICAL_NATURAL
  retained      : 67
  removed       : ['Dst Port', 'Init Fwd Win Byts', 'Fwd Seg Size Min']
  semantic label: joint_shortcut_prone_group_ablation
  data source   : SEALED EXECUTION CACHE V1

[OK] frozen CHRONOLOGICAL_NATURAL membership
     train      : 13,818,623
     validation : 5

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 488.759 s
     model SHA256: 243cb59d498973e4c4313b8d03aafa14ac2149b0e5aa70ce1dabc2d4155298d3

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 144.959 s
     model SHA256: c70f60058ad6bf3bf01cb335d7ceb0a9994c2c0dae4dc3cfb624b003c589eae3

STAGE23-1J COMPLETE — RESULT UNSEALED

Cell:
  subset : NO_SUSPICIOUS_GROUP
  split  : CHRONOLOGICAL_NATURAL

Frozen suspicious group:
  removed : ['Dst Port', 'Init Fwd Win Byts', 'Fwd Seg Size Min']
  retained: 67
  semantics: joint_shortcut_prone_group_ablation

Optimized execution:
  cache             : stage23_execution_cache_v1
  cache seal commit : 66e1c6382caa63699766e605df63df02beff894f
  Parquet reads     : 0
  materialization   : 41.670 s

Data:
  train      : 13,818,623
  validation : 593,780

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 20 / 50

CHRONOLOGICAL RANKING METRICS

Attack prevalence                : 0.104846912998
NO_SUSPICIOUS_GROUP PR-AUC       : 0.10466692

In [24]:
# =============================================================================
# STAGE23-1J — SCIENTIFIC RESULT SEAL
# NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL
# =============================================================================
#
# ZERO MODEL FITS.
# ZERO DATASET ACCESS.
# ZERO CACHE MATERIALIZATION.
# ZERO METRIC RECOMPUTATION FROM DATA.
#
# FIT ACCOUNTING:
#   before seal : 20 / 50
#   this seal   :  0
#   after seal  : 20 / 50
#
# NEXT AUTHORIZED MODEL CELL AFTER SUCCESSFUL SEAL:
#   Stage23-1K — BEHAVIOR_ONLY × RANDOM_NATURAL
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1j_no_suspicious_group_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_suspicious_group"
)

EXPECTED_PARENT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_TAG = (
    "stage23-execution-cache-v1"
)

SCIENTIFIC_RANDOM_COMMIT = (
    "a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0"
)

SCIENTIFIC_RANDOM_TAG = (
    "stage23-1i-no-suspicious-group-random-natural-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

RESULT_TAG = (
    "stage23-1j-no-suspicious-group-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1J: freeze NO_SUSPICIOUS_GROUP chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1J frozen NO_SUSPICIOUS_GROUP CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1j_no_suspicious_group_chronological_natural_result.json":
        "49913686033cdd2ab946803d3ac3def3bed09eb7fca9592a079a13c55f36a621",

    "no_suspicious_group_chronological_natural_lightgbm_model.txt":
        "243cb59d498973e4c4313b8d03aafa14ac2149b0e5aa70ce1dabc2d4155298d3",

    "no_suspicious_group_chronological_natural_xgboost_model.json":
        "c70f60058ad6bf3bf01cb335d7ceb0a9994c2c0dae4dc3cfb624b003c589eae3",

    "no_suspicious_group_chronological_natural_validation_probabilities.npz":
        "8bd661093859cb0e6c2b6248cc99eddc278e55e95d2b68f4ffee9939a8cba22a",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "974cb32c20360e82c1481544ed58f4592d5c8fd5527bb776911c676b95243843"
)


# =============================================================================
# 2. EXACT SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.104666926088
EXPECTED_ROC_AUC = 0.489392507661

EXPECTED_PR_PENALTY = 0.001548229046
EXPECTED_ROC_PENALTY = 0.025525918733

EXPECTED_RANDOM_PR_PENALTY = 0.005988796458
EXPECTED_RANDOM_ROC_PENALTY = 0.001216045643

EXPECTED_PR_INTERACTION = 0.004440567412
EXPECTED_ROC_INTERACTION = -0.024309873089

EXPECTED_F1_INTERACTION = 0.039938361373
EXPECTED_RECALL_INTERACTION = 0.017636132544
EXPECTED_FPR_INTERACTION = 0.000952205052

EXPECTED_ACCURACY = 0.890407221530
EXPECTED_PRECISION = 0.155332681018
EXPECTED_RECALL = 0.010199820098
EXPECTED_F1 = 0.019142650428
EXPECTED_FPR = 0.006496414085
EXPECTED_FNR = 0.989800179902

EXPECTED_TN = 528071
EXPECTED_FP = 3453
EXPECTED_FN = 61621
EXPECTED_TP = 635

EXPECTED_MATERIALIZATION_SECONDS = 41.670


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if show and output:
        print(output)

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


def assert_float(name, actual, expected, tolerance=1e-12):

    actual = float(actual)
    expected = float(expected)

    if abs(actual - expected) >= tolerance:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1J output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1J — SCIENTIFIC RESULT SEAL")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_TAG,
    ]
)

random_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        SCIENTIFIC_RANDOM_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected parent before Stage23-1J seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if cache_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Execution-cache tag verification failed."
    )


if random_tag_commit != SCIENTIFIC_RANDOM_COMMIT:

    raise RuntimeError(
        "Stage23-1I scientific partner tag verification failed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage23-1J seal:\n"
        + status
    )


if TARGET_DIR.exists():

    raise RuntimeError(
        f"Stage23-1J repository target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():

    raise RuntimeError(
        f"Stage23-1J source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
)


if existing_tag:

    raise RuntimeError(
        f"Stage23-1J result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch             :", branch)
print("[OK] parent HEAD        :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] cache seal tag     :", cache_tag_commit)
print("[OK] Stage23-1I tag     :", random_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] Stage23-1J tag     : ABSENT")


# =============================================================================
# 5. VERIFY SOURCE MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():

    raise RuntimeError(
        "Stage23-1J source checksum manifest missing."
    )


actual_source_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_source_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:

    raise RuntimeError(
        "\nSTAGE23-1J SOURCE MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_source_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_source_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 100)
print("VERIFYING STAGE23-1J CORE ARTIFACTS")
print("=" * 100)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-1J artifact: {filename}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-1J ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. LOAD / VERIFY RESULT JSON
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1j_no_suspicious_group_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result[
    "stage"
] != "Stage23-1J":

    raise RuntimeError(
        "Unexpected Stage23-1J stage identifier."
    )


if result[
    "cell"
][
    "subset"
] != "NO_SUSPICIOUS_GROUP":

    raise RuntimeError(
        "Stage23-1J subset mismatch."
    )


if result[
    "cell"
][
    "split"
] != "CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1J split mismatch."
    )


EXPECTED_REMOVED = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


if result[
    "cell"
][
    "removed_features"
] != EXPECTED_REMOVED:

    raise RuntimeError(
        "Frozen suspicious-group definition changed."
    )


if result[
    "cell"
][
    "feature_count"
] != 67:

    raise RuntimeError(
        "Frozen Stage23-1J feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "joint_shortcut_prone_group_ablation":

    raise RuntimeError(
        "Frozen Stage23-1J semantic label changed."
    )


# =============================================================================
# 8. VERIFY PROTOCOL / CACHE / RANDOM PARTNER PROVENANCE
# =============================================================================

if result[
    "protocol"
][
    "commit"
] != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "execution_parent"
][
    "commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1J execution parent mismatch."
    )


if result[
    "execution_parent"
][
    "tag"
] != CACHE_TAG:

    raise RuntimeError(
        "Stage23-1J cache parent tag mismatch."
    )


if result[
    "execution_parent"
][
    "type"
] != "SEALED_DETERMINISTIC_EXECUTION_CACHE":

    raise RuntimeError(
        "Unexpected Stage23-1J execution-parent type."
    )


if result[
    "scientific_random_partner"
][
    "commit"
] != SCIENTIFIC_RANDOM_COMMIT:

    raise RuntimeError(
        "Stage23-1J scientific random partner changed."
    )


if result[
    "scientific_random_partner"
][
    "tag"
] != SCIENTIFIC_RANDOM_TAG:

    raise RuntimeError(
        "Stage23-1J scientific random partner tag changed."
    )


if result[
    "cache"
][
    "identifier"
] != "stage23_execution_cache_v1":

    raise RuntimeError(
        "Unexpected Stage23 execution-cache identifier."
    )


if result[
    "cache"
][
    "seal_commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1J cache seal commit mismatch."
    )


if result[
    "cache"
][
    "seal_tag"
] != CACHE_TAG:

    raise RuntimeError(
        "Stage23-1J cache seal tag mismatch."
    )


if result[
    "cache"
][
    "sealed"
] is not True:

    raise RuntimeError(
        "Stage23-1J did not use a sealed cache."
    )


if result[
    "cache"
][
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Stage23-1J unexpectedly read Parquet files."
    )


# =============================================================================
# 9. GOVERNANCE ASSERTIONS
# =============================================================================

governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:

    raise RuntimeError(
        "Unexpected Stage23-1J fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 18:

    raise RuntimeError(
        "Unexpected pre-Stage23-1J fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 20:

    raise RuntimeError(
        "Unexpected post-Stage23-1J fit count."
    )


if governance[
    "execution_cache_optimization"
] is not True:

    raise RuntimeError(
        "Stage23-1J optimized-cache flag missing."
    )


if governance[
    "execution_cache_scientifically_sealed"
] is not True:

    raise RuntimeError(
        "Stage23-1J cache-seal flag missing."
    )


if governance[
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Stage23-1J governance reports Parquet access."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
    "feature_definition_changed",
    "split_definition_changed",
    "input_dtype_changed",
    "model_specification_changed",
]:

    if governance[
        key
    ] is not False:

        raise RuntimeError(
            f"Governance violation: {key}"
        )


print()
print("[OK] governance assertions passed")
print("     Stage23 fits : 20 / 50")
print("     Parquet reads: 0")
print("     cache        : SEALED")


# =============================================================================
# 10. EXACT RANKING METRICS
# =============================================================================

ranking = result[
    "ranking_metrics"
]

penalty = result[
    "removal_penalty_full_minus_ablated"
]

interaction = result[
    "shortcut_interaction"
]

random_reference = result[
    "matched_random_reference"
]

fixed = result[
    "fixed_threshold_0_50"
]


assert_float(
    "PR-AUC",
    ranking[
        "pr_auc"
    ],
    EXPECTED_PR_AUC,
)

assert_float(
    "ROC-AUC",
    ranking[
        "roc_auc"
    ],
    EXPECTED_ROC_AUC,
)

assert_float(
    "chronological PR penalty",
    penalty[
        "pr_auc"
    ],
    EXPECTED_PR_PENALTY,
)

assert_float(
    "chronological ROC penalty",
    penalty[
        "roc_auc"
    ],
    EXPECTED_ROC_PENALTY,
)

assert_float(
    "random PR penalty",
    random_reference[
        "pr_auc_removal_penalty"
    ],
    EXPECTED_RANDOM_PR_PENALTY,
)

assert_float(
    "random ROC penalty",
    random_reference[
        "roc_auc_removal_penalty"
    ],
    EXPECTED_RANDOM_ROC_PENALTY,
)


# =============================================================================
# 11. EXACT INTERACTION ASSERTIONS
# =============================================================================

if interaction[
    "point_estimate_available"
] is not True:

    raise RuntimeError(
        "Stage23-1J interaction point estimate missing."
    )


if interaction[
    "confidence_interval_status"
] != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS":

    raise RuntimeError(
        "Stage23-1J uncertainty status changed."
    )


if interaction[
    "interpretation_status"
] != "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE":

    raise RuntimeError(
        "Stage23-1J interaction interpretation status changed."
    )


assert_float(
    "PR interaction",
    interaction[
        "pr_auc"
    ],
    EXPECTED_PR_INTERACTION,
)

assert_float(
    "ROC interaction",
    interaction[
        "roc_auc"
    ],
    EXPECTED_ROC_INTERACTION,
)

assert_float(
    "F1 interaction",
    interaction[
        "f1_at_0_50"
    ],
    EXPECTED_F1_INTERACTION,
)

assert_float(
    "Recall interaction",
    interaction[
        "recall_at_0_50"
    ],
    EXPECTED_RECALL_INTERACTION,
)

assert_float(
    "FPR interaction",
    interaction[
        "fpr_at_0_50"
    ],
    EXPECTED_FPR_INTERACTION,
)


# =============================================================================
# 12. EXACT FIXED-THRESHOLD ASSERTIONS
# =============================================================================

assert_float(
    "Accuracy",
    fixed["accuracy"],
    EXPECTED_ACCURACY,
)

assert_float(
    "Precision",
    fixed["precision"],
    EXPECTED_PRECISION,
)

assert_float(
    "Recall",
    fixed["recall"],
    EXPECTED_RECALL,
)

assert_float(
    "F1",
    fixed["f1"],
    EXPECTED_F1,
)

assert_float(
    "FPR",
    fixed["fpr"],
    EXPECTED_FPR,
)

assert_float(
    "FNR",
    fixed["fnr"],
    EXPECTED_FNR,
)


if (
    int(fixed["tn"]) != EXPECTED_TN
    or int(fixed["fp"]) != EXPECTED_FP
    or int(fixed["fn"]) != EXPECTED_FN
    or int(fixed["tp"]) != EXPECTED_TP
):

    raise RuntimeError(
        "Stage23-1J confusion matrix differs from execution output."
    )


print()
print("[OK] Scientific result assertions passed.")

print(
    f"     PR-AUC          : {float(ranking['pr_auc']):.12f}"
)

print(
    f"     ROC-AUC         : {float(ranking['roc_auc']):.12f}"
)

print(
    f"     PR penalty      : {float(penalty['pr_auc']):+.12f}"
)

print(
    f"     ROC penalty     : {float(penalty['roc_auc']):+.12f}"
)

print(
    f"     PR interaction  : {float(interaction['pr_auc']):+.12f}"
)

print(
    f"     ROC interaction : {float(interaction['roc_auc']):+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 13. COPY PERMANENT CORE ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR
        / filename,
        TARGET_DIR
        / filename,
    )


# =============================================================================
# 14. VERIFY REPOSITORY COPIES
# =============================================================================

print()
print("=" * 100)
print("VERIFYING REPOSITORY COPIES")
print("=" * 100)
print()


for filename in FILES_TO_COPY:

    source_sha = sha256_file(
        SOURCE_DIR
        / filename
    )

    destination_sha = sha256_file(
        TARGET_DIR
        / filename
    )

    if source_sha != destination_sha:

        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )


    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 15. CREATE SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT_PATH = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1J",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "sealed_utc":
        now_utc(),

    "cell": {

        "subset":
            "NO_SUSPICIOUS_GROUP",

        "split":
            "CHRONOLOGICAL_NATURAL",

        "removed_features":
            EXPECTED_REMOVED,

        "feature_count":
            67,

        "semantic_label":
            "joint_shortcut_prone_group_ablation",
    },

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "execution_parent": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            CACHE_TAG,

        "type":
            "SEALED_DETERMINISTIC_EXECUTION_CACHE",
    },

    "scientific_random_partner": {

        "commit":
            SCIENTIFIC_RANDOM_COMMIT,

        "tag":
            SCIENTIFIC_RANDOM_TAG,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        20,

    "optimized_cache_used":
        True,

    "parquet_files_read":
        0,

    "ranking": {

        "pr_auc":
            float(
                ranking[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                ranking[
                    "roc_auc"
                ]
            ),

        "pr_auc_removal_penalty":
            float(
                penalty[
                    "pr_auc"
                ]
            ),

        "roc_auc_removal_penalty":
            float(
                penalty[
                    "roc_auc"
                ]
            ),
    },

    "shortcut_interaction": {

        "pr_auc":
            float(
                interaction[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                interaction[
                    "roc_auc"
                ]
            ),

        "f1_at_0_50":
            float(
                interaction[
                    "f1_at_0_50"
                ]
            ),

        "recall_at_0_50":
            float(
                interaction[
                    "recall_at_0_50"
                ]
            ),

        "fpr_at_0_50":
            float(
                interaction[
                    "fpr_at_0_50"
                ]
            ),

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1K BEHAVIOR_ONLY × RANDOM_NATURAL",
}


SEAL_RECEIPT_PATH.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 16. README
# =============================================================================

README_PATH = (
    TARGET_DIR
    / "README.md"
)


README_PATH.write_text(
f"""# Stage23-1J — NO_SUSPICIOUS_GROUP × CHRONOLOGICAL_NATURAL

This directory freezes the chronological partner of the prospectively
defined three-feature suspicious-group ablation.

## Frozen subset

Removed exactly:

- `Dst Port`
- `Init Fwd Win Byts`
- `Fwd Seg Size Min`

Retained features: 67

Semantic label:

`joint_shortcut_prone_group_ablation`

## Execution

- Split: `CHRONOLOGICAL_NATURAL`
- Train rows: 13,818,623
- Validation rows: 593,780
- New boosted-model fits: 2
- Stage23 fits after this cell: 20 / 50
- Execution source: sealed deterministic execution cache v1
- Parquet reads: 0
- Cache materialization: {float(result["cache"]["materialization_seconds"]):.3f} s
- Threshold optimization: none
- Subset-specific tuning: none
- Rebalancing: none

## Chronological ranking

- Attack prevalence: {result["data"]["validation"]["attack_prevalence"]:.12f}
- PR-AUC: {float(ranking["pr_auc"]):.12f}
- ROC-AUC: {float(ranking["roc_auc"]):.12f}
- FULL − ablated PR-AUC penalty: {float(penalty["pr_auc"]):+.12f}
- FULL − ablated ROC-AUC penalty: {float(penalty["roc_auc"]):+.12f}

## Fixed threshold 0.50

- Accuracy: {float(fixed["accuracy"]):.12f}
- Precision: {float(fixed["precision"]):.12f}
- Recall: {float(fixed["recall"]):.12f}
- F1: {float(fixed["f1"]):.12f}
- FPR: {float(fixed["fpr"]):.12f}
- FNR: {float(fixed["fnr"]):.12f}

Confusion matrix:

- TN: {int(fixed["tn"]):,}
- FP: {int(fixed["fp"]):,}
- FN: {int(fixed["fn"]):,}
- TP: {int(fixed["tp"]):,}

## Frozen split × ablation point interaction

Definition:

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

Point estimates:

- PR-AUC interaction: {float(interaction["pr_auc"]):+.12f}
- ROC-AUC interaction: {float(interaction["roc_auc"]):+.12f}
- F1@0.50 interaction: {float(interaction["f1_at_0_50"]):+.12f}
- Recall@0.50 interaction: {float(interaction["recall_at_0_50"]):+.12f}
- FPR@0.50 interaction: {float(interaction["fpr_at_0_50"]):+.12f}

The PR-AUC and ROC-AUC interactions currently have opposite signs.
Accordingly, interpretation remains metric-specific.

These remain **point estimates only**. The preregistered paired bootstrap
uncertainty analysis is required before inferential interpretation.

## Governance

- Stage23 fits sealed after this result: 20 / 50
- Execution-cache optimization: yes
- Cache scientifically sealed: yes
- Parquet files read: 0
- Raw March 1 access: no
- Raw March 2 access: no
- Stage23-0 modified: no
- Threshold optimization: no
- Per-subset tuning: no

## Next authorized model cell

`Stage23-1K BEHAVIOR_ONLY × RANDOM_NATURAL`
""",
    encoding="utf-8",
)


# =============================================================================
# 17. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPOSITORY_CHECKSUMS_PATH = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_files = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPOSITORY_CHECKSUMS_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_files
    ) + "\n",
    encoding="utf-8",
)


REPOSITORY_MANIFEST_SHA = sha256_file(
    REPOSITORY_CHECKSUMS_PATH
)


# =============================================================================
# 18. STAGE ONLY STAGE23-1J
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(
            relative_target
        ),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


expected_prefix = (
    str(
        relative_target
    )
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        expected_prefix
    )
]


if unexpected:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 100)
print("STAGED STAGE23-1J ARTIFACTS")
print("=" * 100)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 19. CREATE RESULT COMMIT
# =============================================================================

print()
print("=" * 100)
print("CREATING STAGE23-1J RESULT COMMIT")
print("=" * 100)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)


if sealed_commit == EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
):

    raise RuntimeError(
        "Repository dirty after Stage23-1J commit."
    )


print()
print("Stage23-1J commit:")
print(
    " ",
    sealed_commit
)


# =============================================================================
# 20. CREATE ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ]
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
)


if local_tag_commit != sealed_commit:

    raise RuntimeError(
        "Local Stage23-1J tag verification failed."
    )


# =============================================================================
# 21. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 100)
print("PUSHING STAGE23-1J")
print("=" * 100)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 22. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
)


if not remote_main_output:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 23. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":

        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":

        peeled_commit = sha


if not tag_object:

    raise RuntimeError(
        "Remote Stage23-1J annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote Stage23-1J tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 24. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-1J push."
    )


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1J SCIENTIFIC RESULT SEALED")
print("=" * 100)

print()
print("Execution-cache parent:")
print(
    " ",
    EXPECTED_PARENT
)

print()
print("Scientific random partner:")
print(
    " ",
    SCIENTIFIC_RANDOM_COMMIT
)

print()
print("Stage23-1J commit:")
print(
    " ",
    sealed_commit
)

print()
print("Result tag:")
print(
    " ",
    RESULT_TAG
)

print()
print("Remote main:")
print(
    " ",
    remote_main
)

print()
print("Remote tag object:")
print(
    " ",
    tag_object
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    peeled_commit
)

print()
print("Source execution checksum manifest:")
print(
    " ",
    EXPECTED_SOURCE_MANIFEST_SHA
)

print()
print("Repository checksum manifest:")
print(
    " ",
    REPOSITORY_MANIFEST_SHA
)


print()
print("=" * 100)
print("FROZEN NO_SUSPICIOUS_GROUP PAIR")
print("=" * 100)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : {EXPECTED_RANDOM_PR_PENALTY:+.12f}"
)
print(
    f"  ROC penalty : {EXPECTED_RANDOM_ROC_PENALTY:+.12f}"
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR penalty  : {EXPECTED_PR_PENALTY:+.12f}"
)
print(
    f"  ROC penalty : {EXPECTED_ROC_PENALTY:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {EXPECTED_PR_INTERACTION:+.12f}"
)
print(
    f"  ROC-AUC : {EXPECTED_ROC_INTERACTION:+.12f}"
)

print()
print("Supplementary interactions:")
print(
    f"  F1@0.50     : {EXPECTED_F1_INTERACTION:+.12f}"
)
print(
    f"  Recall@0.50 : {EXPECTED_RECALL_INTERACTION:+.12f}"
)
print(
    f"  FPR@0.50    : {EXPECTED_FPR_INTERACTION:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 100)
print("OPTIMIZED EXECUTION")
print("=" * 100)

print()
print("Cache used       : stage23_execution_cache_v1")
print("Parquet reads    : 0")
print(
    "Materialization :",
    f"{float(result['cache']['materialization_seconds']):.3f} s"
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits sealed      : 20 / 50")
print("Model fits seal          : 0")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Execution cache changed  : NO")
print("Git status               : CLEAN")


print()
print("=" * 100)
print("STAGE23-1J COMPLETE AND SEALED")
print("=" * 100)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1K — BEHAVIOR_ONLY × RANDOM_NATURAL"
)

print("=" * 100)

STAGE23-1J — SCIENTIFIC RESULT SEAL

[OK] branch             : main
[OK] parent HEAD        : 66e1c6382caa63699766e605df63df02beff894f
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] cache seal tag     : 66e1c6382caa63699766e605df63df02beff894f
[OK] Stage23-1I tag     : a68ed8da15972c9fa9ee7efb80c8f86bf263cdd0
[OK] worktree           : CLEAN
[OK] Stage23-1J tag     : ABSENT

[OK] source checksum-manifest SHA256:
  974cb32c20360e82c1481544ed58f4592d5c8fd5527bb776911c676b95243843

VERIFYING STAGE23-1J CORE ARTIFACTS

[EXACT] stage23_1j_no_suspicious_group_chronological_natural_result.json
[EXACT] no_suspicious_group_chronological_natural_lightgbm_model.txt
[EXACT] no_suspicious_group_chronological_natural_xgboost_model.json
[EXACT] no_suspicious_group_chronological_natural_validation_probabilities.npz

[OK] governance assertions passed
     Stage23 fits : 20 / 50
     Parquet reads: 0
     cache        : SEALED

[OK] Scientific result assertions passed.
     PR-AU

In [25]:
# =============================================================================
# STAGE23-1K
# BEHAVIOR_ONLY × RANDOM_NATURAL
# OPTIMIZED — SEALED EXECUTION CACHE V1
# =============================================================================
#
# FROZEN SUBSET
# -------------
# BEHAVIOR_ONLY
#
# Retained features : 63
#
# Excluded EXACTLY:
#   - Dst Port
#   - Protocol
#   - Fwd Header Len
#   - Bwd Header Len
#   - Init Fwd Win Byts
#   - Init Bwd Win Byts
#   - Fwd Seg Size Min
#
# Semantic label:
#   behavior_restricted_feature_set
#
# IMPORTANT MANUSCRIPT TERMINOLOGY:
#   "behavior-restricted performance"
#   NOT "true performance"
#
# REQUIRED PARENT:
#   Stage23-1J
#   12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74
#
# SEALED CACHE:
#   stage23-execution-cache-v1
#   66e1c6382caa63699766e605df63df02beff894f
#
# FIT ACCOUNTING:
#   before : 20 / 50
#   this   :  2
#   after  : 22 / 50
#
# NO:
#   Parquet reads
#   raw Mar1
#   raw Mar2
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#
# INTERACTION:
#   PENDING matched Stage23-1L CHRONOLOGICAL_NATURAL
#
# FAIL CLOSED:
# If either model completes and anything later fails, DO NOT blindly rerun.
# Inspect execution_state.json first.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1K"

TARGET_SUBSET = "BEHAVIOR_ONLY"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

PREDECESSOR_COMMIT = (
    "12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74"
)

PREDECESSOR_TAG = (
    "stage23-1j-no-suspicious-group-chronological-natural-v1"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_DATA_MANIFEST_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

CACHE_SEMANTIC_70F_SHA256 = (
    "8c43ab0e36a65de1c095acb1ebf9e7d029a9e2b1cc353455a66617db8a899bb2"
)

EXPECTED_RANDOM_BITSET_SHA256 = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

CACHE_SEAL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "no_suspicious_group"
    / "seal_receipt.json"
)

FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1k_behavior_only_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1k_behavior_only_random_natural_result.json"
)


# =============================================================================
# 2. FROZEN COUNTS
# =============================================================================

N_DEVELOPMENT = 14_412_403

TRAIN_ROWS = 11_529_922
VALIDATION_ROWS = 2_882_481

TRAIN_BENIGN = 9_952_083
TRAIN_ATTACK = 1_577_839

VALIDATION_BENIGN = 2_488_021
VALIDATION_ATTACK = 394_460

N_FEATURES = 63


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (result.stdout or "").strip()

    if check and result.returncode != 0:

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    resolved = Path(path).resolve()

    text = str(
        resolved
    ).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in text:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {resolved}\n"
                f"marker : {marker}"
            )

    return resolved


# =============================================================================
# 4. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1K OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        f"Existing state:\n"
        f"{json.dumps(state, indent=2)}\n\n"
        "Do NOT delete or rerun blindly."
    )


# =============================================================================
# 5. GIT / GOVERNANCE PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1K — BEHAVIOR_ONLY × RANDOM_NATURAL")
print("OPTIMIZED EXECUTION CACHE V1")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_SEAL_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1K MUST START FROM SEALED STAGE23-1J.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1J result tag mismatch."
    )


if cache_tag_commit != CACHE_SEAL_COMMIT:

    raise RuntimeError(
        "Stage23 execution-cache tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1J seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 20:

    raise RuntimeError(
        "Expected exactly 20 sealed Stage23 fits."
    )


if predecessor_seal[
    "next_authorized_model_cell"
] != "Stage23-1K BEHAVIOR_ONLY × RANDOM_NATURAL":

    raise RuntimeError(
        "\nStage23-1J does not authorize Stage23-1K.\n"
        f"actual: "
        f"{predecessor_seal.get('next_authorized_model_cell')}"
    )


print("[OK] branch             :", branch)
print("[OK] HEAD               :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1J tag     :", predecessor_tag_commit)
print("[OK] cache seal tag     :", cache_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] fits before cell   : 20 / 50")
print("[OK] Stage23-1K         : AUTHORIZED")


# =============================================================================
# 6. VERIFY STAGE23-0
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum manifest changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Stage23-0 artifact missing: {filename}"
        )


    if sha256_file(
        path
    ) != expected_sha:

        raise RuntimeError(
            f"Stage23-0 artifact mutated: {filename}"
        )


print(
    f"[OK] Stage23-0 artifacts : "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. VERIFY SEALED CACHE METADATA
# =============================================================================

CACHE_METADATA_REPO = (
    CACHE_SEAL_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_REPO = (
    CACHE_SEAL_DIR
    / "data_checksums.sha256"
)

CACHE_SEAL_RECEIPT = (
    CACHE_SEAL_DIR
    / "seal_receipt.json"
)


for path in [
    CACHE_METADATA_REPO,
    CACHE_DATA_MANIFEST_REPO,
    CACHE_SEAL_RECEIPT,
]:

    if not path.exists():

        raise RuntimeError(
            f"Missing sealed cache artifact: {path}"
        )


if sha256_file(
    CACHE_METADATA_REPO
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Sealed cache metadata SHA mismatch."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_REPO
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Sealed cache data-manifest SHA mismatch."
    )


cache_seal = json.loads(
    CACHE_SEAL_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if cache_seal[
    "status"
] != "EXECUTION_CACHE_VERIFICATION_FROZEN":

    raise RuntimeError(
        "Execution cache is not sealed."
    )


if cache_seal[
    "semantic_equivalence"
][
    "cache_reconstructed_row_major_70f_sha256"
] != CACHE_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Execution cache semantic SHA mismatch."
    )


# =============================================================================
# 8. VERIFY RUNTIME CACHE SMALL ARTIFACTS
# =============================================================================

CACHE_METADATA_RUNTIME = guard_path(
    CACHE_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_RUNTIME = guard_path(
    CACHE_DIR
    / "data_checksums.sha256"
)


if not CACHE_METADATA_RUNTIME.exists():

    raise RuntimeError(
        "Runtime cache metadata missing."
    )


if not CACHE_DATA_MANIFEST_RUNTIME.exists():

    raise RuntimeError(
        "Runtime cache manifest missing."
    )


if sha256_file(
    CACHE_METADATA_RUNTIME
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Runtime cache metadata differs from sealed metadata."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_RUNTIME
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Runtime cache manifest differs from sealed manifest."
    )


cache_metadata = json.loads(
    CACHE_METADATA_RUNTIME.read_text(
        encoding="utf-8"
    )
)


if cache_metadata[
    "semantic_equivalence"
][
    "exact_byte_equality"
] is not True:

    raise RuntimeError(
        "Cache semantic equivalence flag is false."
    )


print("[OK] sealed runtime cache metadata exact")


# =============================================================================
# 9. LOAD FROZEN FEATURE / MODEL / SPLIT SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = list(
    subset_spec[
        "features"
    ]
)

REMOVED_FEATURES = list(
    subset_spec[
        "removed"
    ]
)


EXPECTED_REMOVED = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nFrozen BEHAVIOR_ONLY exclusions changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if len(
    FEATURES
) != 63:

    raise RuntimeError(
        f"Expected 63 behavior-restricted features; found {len(FEATURES)}"
    )


if subset_spec[
    "feature_count"
] != 63:

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY feature_count changed."
    )


if subset_spec[
    "mode"
] != "whitelist":

    raise RuntimeError(
        "BEHAVIOR_ONLY must remain a frozen whitelist."
    )


if subset_spec[
    "semantic_label"
] != "behavior_restricted_feature_set":

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY semantic label changed."
    )


for feature in EXPECTED_REMOVED:

    if feature in FEATURES:

        raise RuntimeError(
            f"Excluded feature present in BEHAVIOR_ONLY: {feature}"
        )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  retained      :", len(FEATURES))
print("  excluded      :", REMOVED_FEATURES)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)
print(
    "  interpretation:",
    "behavior-restricted performance"
)


# =============================================================================
# 10. VERIFY CACHE FEATURE PROJECTION
# =============================================================================

FULL_FEATURES = (
    cache_metadata[
        "full_feature_order"
    ]
)


if FULL_FEATURES != feature_spec[
    "full_feature_order"
]:

    raise RuntimeError(
        "Cache full feature order differs from protocol."
    )


CACHE_INDICES = (
    cache_metadata[
        "primary_subset_feature_indices"
    ][
        TARGET_SUBSET
    ]
)


if len(
    CACHE_INDICES
) != 63:

    raise RuntimeError(
        "Cache BEHAVIOR_ONLY projection count mismatch."
    )


projected_features = [
    FULL_FEATURES[
        index
    ]
    for index in CACHE_INDICES
]


if projected_features != FEATURES:

    raise RuntimeError(
        "Cache BEHAVIOR_ONLY feature order differs from protocol."
    )


# =============================================================================
# 11. VERIFY FROZEN RANDOM SPLIT COUNTS
# =============================================================================

random_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


if int(
    random_spec[
        "train"
    ][
        "rows"
    ]
) != TRAIN_ROWS:

    raise RuntimeError(
        "Random train row count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "rows"
    ]
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation row count changed."
    )


if int(
    random_spec[
        "train"
    ][
        "benign"
    ]
) != TRAIN_BENIGN:

    raise RuntimeError(
        "Random train benign count changed."
    )


if int(
    random_spec[
        "train"
    ][
        "attack"
    ]
) != TRAIN_ATTACK:

    raise RuntimeError(
        "Random train attack count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "benign"
    ]
) != VALIDATION_BENIGN:

    raise RuntimeError(
        "Random validation benign count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "attack"
    ]
) != VALIDATION_ATTACK:

    raise RuntimeError(
        "Random validation attack count changed."
    )


# =============================================================================
# 12. PARSE SEALED CACHE MANIFEST
# =============================================================================

manifest_hashes = {}


for line in CACHE_DATA_MANIFEST_RUNTIME.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, relative_path = line.split(
        "  ",
        1,
    )

    manifest_hashes[
        relative_path
    ] = digest


if len(
    manifest_hashes
) != 75:

    raise RuntimeError(
        f"Expected 75 cache manifest entries; found {len(manifest_hashes)}"
    )


# =============================================================================
# 13. RANDOM MEMBERSHIP BITSET
# =============================================================================

RANDOM_BITSET_PATH = guard_path(
    CACHE_DIR
    / "random_validation.packbits"
)


if not RANDOM_BITSET_PATH.exists():

    raise RuntimeError(
        "Frozen random membership bitset missing from cache."
    )


if sha256_file(
    RANDOM_BITSET_PATH
) != EXPECTED_RANDOM_BITSET_SHA256:

    raise RuntimeError(
        "Random validation bitset changed."
    )


packed = np.fromfile(
    RANDOM_BITSET_PATH,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random membership logical length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation population mismatch."
    )


print()
print("[OK] RANDOM_NATURAL bitset exact")
print(
    "     SHA256:",
    EXPECTED_RANDOM_BITSET_SHA256
)


# =============================================================================
# 14. CACHE METADATA ARRAYS
# =============================================================================

LABEL_PATH = guard_path(
    CACHE_DIR
    / "binary_label.uint8.dat"
)

CLEAN_POSITION_PATH = guard_path(
    CACHE_DIR
    / "clean_position.int64.dat"
)


for path, manifest_name in [
    (
        LABEL_PATH,
        "binary_label.uint8.dat",
    ),
    (
        CLEAN_POSITION_PATH,
        "clean_position.int64.dat",
    ),
]:

    if not path.exists():

        raise RuntimeError(
            f"Required cache array missing: {path}"
        )


    if sha256_file(
        path
    ) != manifest_hashes[
        manifest_name
    ]:

        raise RuntimeError(
            f"Physical cache array differs from seal: {manifest_name}"
        )


labels_all = np.memmap(
    LABEL_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


clean_positions_all = np.memmap(
    CLEAN_POSITION_PATH,
    dtype="<i8",
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


# =============================================================================
# 15. VERIFY CACHE FEATURE RECORDS
# =============================================================================

feature_records = {
    int(record["index"]):
        record
    for record in cache_metadata[
        "feature_columns"
    ]
}


SELECTED_FEATURE_RECORDS = []


for index in CACHE_INDICES:

    record = feature_records[
        index
    ]

    if record[
        "feature"
    ] != FULL_FEATURES[
        index
    ]:

        raise RuntimeError(
            f"Cache feature record mismatch at index {index}"
        )


    relative_path = record[
        "path"
    ]


    if relative_path not in manifest_hashes:

        raise RuntimeError(
            f"Feature absent from sealed manifest: {relative_path}"
        )


    if record[
        "sha256"
    ] != manifest_hashes[
        relative_path
    ]:

        raise RuntimeError(
            f"Metadata/manifest digest disagreement: {relative_path}"
        )


    physical_path = guard_path(
        CACHE_DIR
        / relative_path
    )


    if not physical_path.exists():

        raise RuntimeError(
            f"Cache feature file missing: {physical_path}"
        )


    expected_size = (
        N_DEVELOPMENT
        * 8
    )


    if physical_path.stat().st_size != expected_size:

        raise RuntimeError(
            f"Cache feature size mismatch: {record['feature']}"
        )


    SELECTED_FEATURE_RECORDS.append(
        {
            **record,
            "physical_path":
                physical_path,
        }
    )


# =============================================================================
# 16. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] frozen package versions verified")


# =============================================================================
# 17. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 18. STORAGE PREFLIGHT
# =============================================================================

MATRIX_BYTES = (
    N_DEVELOPMENT
    * N_FEATURES
    * 8
)

LABEL_OUTPUT_BYTES = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)

POSITION_OUTPUT_BYTES = (
    VALIDATION_ROWS
    * 8
)


required_bytes = (
    MATRIX_BYTES
    + LABEL_OUTPUT_BYTES
    + POSITION_OUTPUT_BYTES
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    required_bytes
    + 1 * 1024**3
)


print()
print("=" * 100)
print("OPTIMIZED RANDOM MATERIALIZATION PLAN")
print("=" * 100)
print()

print(
    "Source             : sealed execution cache v1"
)
print(
    "Parquet reads      : 0"
)
print(
    "Selected features  :",
    N_FEATURES
)
print(
    "Train rows         :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows    :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Matrix storage     :",
    f"{MATRIX_BYTES / 1024**3:.3f} GiB"
)
print(
    "Working free       :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nINSUFFICIENT DISK FOR STAGE23-1K\n"
        f"required with safety margin: "
        f"{minimum_free / 1024**3:.3f} GiB\n"
        f"available                  : "
        f"{disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 19. CREATE OUTPUT + EXECUTION STATE
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        PREDECESSOR_COMMIT,

    "cache_seal_commit":
        CACHE_SEAL_COMMIT,

    "cache_seal_tag":
        CACHE_SEAL_TAG,

    "stage23_total_model_fits_before_cell":
        20,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 20. CREATE RANDOM TRAIN / VALIDATION MEMMAPS
# =============================================================================

X_TRAIN_PATH = (
    RUNTIME_DIR
    / "X_train_behavior_only_float64.dat"
)

X_VALIDATION_PATH = (
    RUNTIME_DIR
    / "X_validation_behavior_only_float64.dat"
)

Y_TRAIN_PATH = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

Y_VALIDATION_PATH = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

VAL_POSITION_PATH = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_TRAIN_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


X_validation = np.memmap(
    X_VALIDATION_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    Y_TRAIN_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


y_validation = np.memmap(
    Y_VALIDATION_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    VAL_POSITION_PATH,
    dtype="<i8",
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 21. OPEN SELECTED RAW FEATURE COLUMNS
# =============================================================================

source_feature_mmaps = []


for record in SELECTED_FEATURE_RECORDS:

    source_feature_mmaps.append(
        np.memmap(
            record[
                "physical_path"
            ],
            dtype="<f8",
            mode="r",
            shape=(
                N_DEVELOPMENT,
            ),
        )
    )


# =============================================================================
# 22. MATERIALIZE RANDOM SPLIT FROM CACHE
# =============================================================================

CHUNK_ROWS = 250_000

train_cursor = 0
validation_cursor = 0

feature_hashers = [
    hashlib.sha256()
    for _ in range(
        N_FEATURES
    )
]


materialize_start = (
    time.perf_counter()
)


print()
print("=" * 100)
print("MATERIALIZING BEHAVIOR_ONLY RANDOM SPLIT FROM SEALED CACHE")
print("=" * 100)
print()


total_chunks = (
    N_DEVELOPMENT
    + CHUNK_ROWS
    - 1
) // CHUNK_ROWS


for chunk_index, start in enumerate(
    range(
        0,
        N_DEVELOPMENT,
        CHUNK_ROWS,
    ),
    start=1,
):

    end = min(
        start
        + CHUNK_ROWS,
        N_DEVELOPMENT,
    )

    n_rows = (
        end
        - start
    )


    is_validation = (
        random_validation_mask[
            start:end
        ]
    )

    is_train = (
        ~is_validation
    )


    n_train = int(
        is_train.sum()
    )

    n_validation = int(
        is_validation.sum()
    )


    chunk = np.empty(
        (
            n_rows,
            N_FEATURES,
        ),
        dtype=np.float64,
    )


    for output_col, source_mm in enumerate(
        source_feature_mmaps
    ):

        source_slice = np.asarray(
            source_mm[
                start:end
            ],
            dtype="<f8",
        )


        # Integrity digest is accumulated during the SAME read pass.
        feature_hashers[
            output_col
        ].update(
            np.ascontiguousarray(
                source_slice,
                dtype="<f8",
            ).tobytes(
                order="C"
            )
        )


        chunk[
            :,
            output_col
        ] = source_slice


    if n_train:

        train_end = (
            train_cursor
            + n_train
        )

        X_train[
            train_cursor:train_end,
            :
        ] = chunk[
            is_train,
            :
        ]

        y_train[
            train_cursor:train_end
        ] = labels_all[
            start:end
        ][
            is_train
        ]

        train_cursor = (
            train_end
        )


    if n_validation:

        validation_end = (
            validation_cursor
            + n_validation
        )

        X_validation[
            validation_cursor:validation_end,
            :
        ] = chunk[
            is_validation,
            :
        ]

        y_validation[
            validation_cursor:validation_end
        ] = labels_all[
            start:end
        ][
            is_validation
        ]

        validation_positions[
            validation_cursor:validation_end
        ] = clean_positions_all[
            start:end
        ][
            is_validation
        ]

        validation_cursor = (
            validation_end
        )


    if (
        chunk_index == 1
        or chunk_index % 10 == 0
        or chunk_index == total_chunks
    ):

        print(
            f"  chunk {chunk_index:>3}/{total_chunks}"
            f" — train={train_cursor:,}"
            f" val={validation_cursor:,}"
        )


    del (
        chunk,
        is_train,
        is_validation,
    )

    gc.collect()


X_train.flush()
X_validation.flush()
y_train.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


# =============================================================================
# 23. VERIFY ALL SELECTED FEATURE DIGESTS
# =============================================================================

for output_col, record in enumerate(
    SELECTED_FEATURE_RECORDS
):

    actual_digest = (
        feature_hashers[
            output_col
        ].hexdigest()
    )

    expected_digest = (
        record[
            "sha256"
        ]
    )


    if actual_digest != expected_digest:

        raise RuntimeError(
            "\nPHYSICAL CACHE FEATURE MUTATION DETECTED\n"
            f"feature  : {record['feature']}\n"
            f"expected : {expected_digest}\n"
            f"actual   : {actual_digest}"
        )


print()
print(
    "[OK] selected physical cache columns:"
    " 63 / 63 exact during materialization"
)


# =============================================================================
# 24. MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nRandom training row count mismatch.\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nRandom validation row count mismatch.\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if (
    train_attack != TRAIN_ATTACK
    or train_benign != TRAIN_BENIGN
):

    raise RuntimeError(
        "Random training class counts changed."
    )


if (
    validation_attack != VALIDATION_ATTACK
    or validation_benign != VALIDATION_BENIGN
):

    raise RuntimeError(
        "Random validation class counts changed."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean positions are not strictly increasing."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):

    raise RuntimeError(
        "Validation clean positions disagree with frozen bitset."
    )


print()
print("[OK] optimized RANDOM_NATURAL materialization")
print(
    "     train          :",
    f"{TRAIN_ROWS:,}",
    f"(B={train_benign:,}, A={train_attack:,})"
)
print(
    "     validation     :",
    f"{VALIDATION_ROWS:,}",
    f"(B={validation_benign:,}, A={validation_attack:,})"
)
print(
    "     features       :",
    N_FEATURES
)
print(
    "     dtype          : float64"
)
print(
    "     Parquet reads  : 0"
)
print(
    "     materialization:",
    f"{materialize_seconds:.3f} s"
)


execution_state.update({

    "status":
        "MATERIALIZED_FROM_SEALED_CACHE",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,

    "input_dtype":
        "float64",

    "physical_feature_digests_verified":
        63,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 25. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 26. LIGHTGBM — FIT 1 / 2
# =============================================================================

print()
print("=" * 100)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 100)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "behavior_only_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 27. XGBOOST — FIT 2 / 2
# =============================================================================

print()
print("=" * 100)
print("FIT 2 / 2 — XGBOOST")
print("=" * 100)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "behavior_only_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    str(
        XGB_MODEL_PATH
    )
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 28. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 29. RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 30. FIXED THRESHOLD 0.50
# =============================================================================

predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 31. FROZEN FULL RANDOM REFERENCE
# =============================================================================

if not FULL_RANDOM_RESULT.exists():

    raise RuntimeError(
        f"Frozen FULL random reference missing:\n{FULL_RANDOM_RESULT}"
    )


full_random = json.loads(
    FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_random[
    "cell"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Unexpected frozen FULL random reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 32. REMOVAL PENALTIES
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


recall_removal_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 33. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "behavior_only_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 34. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "BEHAVIOR_ONLY_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent": {

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "cache": {

        "identifier":
            "stage23_execution_cache_v1",

        "seal_commit":
            CACHE_SEAL_COMMIT,

        "seal_tag":
            CACHE_SEAL_TAG,

        "sealed":
            True,

        "metadata_sha256":
            CACHE_METADATA_SHA256,

        "data_manifest_sha256":
            CACHE_DATA_MANIFEST_SHA256,

        "semantic_70f_sha256":
            CACHE_SEMANTIC_70F_SHA256,

        "selected_physical_feature_digests_verified":
            63,

        "parquet_files_read":
            0,

        "materialization_seconds":
            materialize_seconds,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],

        "interpretation_term":
            "behavior-restricted performance",
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty,

        "recall_at_0_50":
            recall_removal_penalty,

        "fpr_at_0_50":
            fpr_change,
    },

    "shortcut_interaction": {

        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "optimized_cache_materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,

        "parquet_files_read":
            0,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            20,

        "stage23_total_model_fits_after_cell":
            22,

        "stage23_total_authorized_model_fits":
            50,

        "execution_cache_optimization":
            True,

        "execution_cache_scientifically_sealed":
            True,

        "parquet_files_read":
            0,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,

        "feature_definition_changed":
            False,

        "split_definition_changed":
            False,

        "input_dtype_changed":
            False,

        "model_specification_changed":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1K before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 35. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 36. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        22,

    "stage23_metrics_calculated":
        True,

    "optimized_cache_used":
        True,

    "parquet_files_read":
        0,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 37. REMOVE ONLY TRANSIENT TRAINING MATRICES
# =============================================================================
#
# DO NOT DELETE:
#   /kaggle/working/stage23_execution_cache_v1
#
# =============================================================================

del X_train
del X_validation
del y_train
del y_validation
del validation_positions

del labels_all
del clean_positions_all

for i in range(
    len(source_feature_mmaps)
):

    source_feature_mmaps[
        i
    ] = None


del source_feature_mmaps

gc.collect()


for path in [
    X_TRAIN_PATH,
    X_VALIDATION_PATH,
    Y_TRAIN_PATH,
    Y_VALIDATION_PATH,
    VAL_POSITION_PATH,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 38. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1K COMPLETE — RESULT UNSEALED")
print("=" * 100)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Behavior-restricted feature set:")
print(
    "  retained:",
    N_FEATURES
)
print(
    "  excluded:",
    REMOVED_FEATURES
)
print(
    "  semantics:",
    subset_spec[
        "semantic_label"
    ]
)

print()
print("Optimized execution:")
print(
    "  cache            : stage23_execution_cache_v1"
)
print(
    "  Parquet reads    : 0"
)
print(
    "  materialization  :",
    f"{materialize_seconds:.3f} s"
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 22 / 50")


print()
print("=" * 100)
print("BEHAVIOR-RESTRICTED RANDOM RANKING")
print("=" * 100)

print()
print(
    f"Attack prevalence            : {attack_prevalence:.12f}"
)

print(
    f"BEHAVIOR_ONLY PR-AUC         : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                  : {full_pr_auc:.12f}"
)

print(
    f"PR-AUC removal penalty       : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"BEHAVIOR_ONLY ROC-AUC        : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                 : {full_roc_auc:.12f}"
)

print(
    f"ROC-AUC removal penalty      : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence          : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 100)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 100)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 100)
print("INTERACTION STATUS")
print("=" * 100)

print()
print(
    "PENDING matched Stage23-1L "
    "BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL"
)


print()
print("=" * 100)
print("ARTIFACTS")
print("=" * 100)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits completed : 22 / 50")
print("Optimized cache used   : YES")
print("Parquet files read     : 0")
print("Threshold optimization : NO")
print("Subset-specific tuning : NO")
print("Rebalancing            : NO")
print("SHAP executed          : NO")
print("Placebo executed       : NO")
print("Raw Mar1 read          : NO")
print("Raw Mar2 read          : NO")
print("Stage23-0 modified     : NO")
print("Git commit created     : NO")
print("Git tag created        : NO")

print()
print("=" * 100)
print("NEXT ACTION")
print("=" * 100)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1K."
)

print()
print(
    "After sealing, only ONE primary boosted execution remains:"
)

print(
    "  Stage23-1L — "
    "BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL"
)

print("=" * 100)

STAGE23-1K — BEHAVIOR_ONLY × RANDOM_NATURAL
OPTIMIZED EXECUTION CACHE V1

[OK] branch             : main
[OK] HEAD               : 12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1J tag     : 12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74
[OK] cache seal tag     : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] fits before cell   : 20 / 50
[OK] Stage23-1K         : AUTHORIZED
[OK] Stage23-0 artifacts : 23/23 exact
[OK] sealed runtime cache metadata exact

Frozen target:
  subset        : BEHAVIOR_ONLY
  split         : RANDOM_NATURAL
  retained      : 63
  excluded      : ['Dst Port', 'Protocol', 'Fwd Header Len', 'Bwd Header Len', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Min']
  semantic label: behavior_restricted_feature_set
  interpretation: behavior-restricted performance

[OK] RANDOM_NATURAL bitset exact
     SHA256: 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 397.871 s
     model SHA256: bb5c3b433582d7129fa765b0c211ed02b4801681b81560a5a5f8f08ae3360eb3

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 107.481 s
     model SHA256: 54c7c7064f7efcd54a6eaaeae375c58df93d2a852a8d7c43d4f9844efa82d913

STAGE23-1K COMPLETE — RESULT UNSEALED

Cell:
  subset : BEHAVIOR_ONLY
  split  : RANDOM_NATURAL

Behavior-restricted feature set:
  retained: 63
  excluded: ['Dst Port', 'Protocol', 'Fwd Header Len', 'Bwd Header Len', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Min']
  semantics: behavior_restricted_feature_set

Optimized execution:
  cache            : stage23_execution_cache_v1
  Parquet reads    : 0
  materialization  : 65.198 s

Data:
  train      : 11,529,922
  validation : 2,882,481

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 22 / 50

BEHAVIOR-RESTRICTED RANDOM RANKING

Attack prevalence            : 0.136847389454
BEHAVIOR_ONLY PR-AUC         : 0.981588003857
FU

In [26]:
# =============================================================================
# STAGE23-1K — SCIENTIFIC RESULT SEAL
# BEHAVIOR_ONLY × RANDOM_NATURAL
# =============================================================================
#
# ZERO MODEL FITS.
# ZERO DATASET ACCESS.
# ZERO CACHE MATERIALIZATION.
# ZERO METRIC RECOMPUTATION FROM DATA.
#
# FIT ACCOUNTING:
#   before seal : 22 / 50
#   seal        :  0
#   after seal  : 22 / 50
#
# NEXT AUTHORIZED MODEL CELL:
#   Stage23-1L — BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/stage23_1k_behavior_only_random_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "behavior_only"
)

EXPECTED_PARENT = (
    "12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74"
)

PREDECESSOR_TAG = (
    "stage23-1j-no-suspicious-group-chronological-natural-v1"
)

CACHE_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_TAG = (
    "stage23-execution-cache-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

RESULT_TAG = (
    "stage23-1k-behavior-only-random-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1K: freeze BEHAVIOR_ONLY random-natural result"
)

TAG_MESSAGE = (
    "Stage23-1K frozen BEHAVIOR_ONLY RANDOM_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1k_behavior_only_random_natural_result.json":
        "de40919dfced8471a48cbef06b1816b05a8e3b797eec10e35d2fade2effa0d61",

    "behavior_only_random_natural_lightgbm_model.txt":
        "bb5c3b433582d7129fa765b0c211ed02b4801681b81560a5a5f8f08ae3360eb3",

    "behavior_only_random_natural_xgboost_model.json":
        "54c7c7064f7efcd54a6eaaeae375c58df93d2a852a8d7c43d4f9844efa82d913",

    "behavior_only_random_natural_validation_probabilities.npz":
        "0b0118a89bc63ada1e63c1f913bceb30c8bf234d25e3c060a03336e5a3e58b0a",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "c3ccb4eda01778f72facdadaef2d2c1a2b766715df87530f6cc9a142a2f55d48"
)


# =============================================================================
# 2. EXACT OBSERVED SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.981588003857
EXPECTED_ROC_AUC = 0.996141240052

EXPECTED_PR_PENALTY = 0.014002038042
EXPECTED_ROC_PENALTY = 0.002483324722

EXPECTED_ACCURACY = 0.985431994174
EXPECTED_PRECISION = 0.950588054817
EXPECTED_RECALL = 0.942539167469
EXPECTED_F1 = 0.946546500675
EXPECTED_FPR = 0.007767619325
EXPECTED_FNR = 0.057460832531

EXPECTED_TN = 2468695
EXPECTED_FP = 19326
EXPECTED_FN = 22666
EXPECTED_TP = 371794


EXPECTED_REMOVED = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (p.stdout or "").strip()

    if show and output:
        print(output)

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


def assert_float(name, actual, expected, tolerance=1e-12):

    actual = float(actual)
    expected = float(expected)

    if abs(actual - expected) >= tolerance:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1K output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1K — SCIENTIFIC RESULT SEAL")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected parent before Stage23-1K seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1J predecessor tag verification failed."
    )


if cache_tag_commit != CACHE_COMMIT:

    raise RuntimeError(
        "Stage23 execution-cache tag verification failed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage23-1K seal:\n"
        + status
    )


if TARGET_DIR.exists():

    raise RuntimeError(
        f"Stage23-1K repository target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():

    raise RuntimeError(
        f"Stage23-1K source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
)


if existing_tag:

    raise RuntimeError(
        f"Stage23-1K result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch             :", branch)
print("[OK] parent HEAD        :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1J tag     :", predecessor_tag_commit)
print("[OK] execution-cache tag:", cache_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] Stage23-1K tag     : ABSENT")


# =============================================================================
# 5. VERIFY SOURCE CHECKSUM MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():

    raise RuntimeError(
        "Stage23-1K source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:

    raise RuntimeError(
        "\nSTAGE23-1K SOURCE MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 100)
print("VERIFYING STAGE23-1K CORE ARTIFACTS")
print("=" * 100)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-1K artifact: {filename}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-1K ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. LOAD RESULT JSON
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1k_behavior_only_random_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result[
    "stage"
] != "Stage23-1K":

    raise RuntimeError(
        "Unexpected Stage23-1K stage identifier."
    )


if result[
    "cell"
][
    "subset"
] != "BEHAVIOR_ONLY":

    raise RuntimeError(
        "Stage23-1K subset mismatch."
    )


if result[
    "cell"
][
    "split"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Stage23-1K split mismatch."
    )


if result[
    "cell"
][
    "removed_features"
] != EXPECTED_REMOVED:

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY exclusion list changed."
    )


if result[
    "cell"
][
    "feature_count"
] != 63:

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "behavior_restricted_feature_set":

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY semantic label changed."
    )


if result[
    "cell"
][
    "interpretation_term"
] != "behavior-restricted performance":

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY interpretation terminology changed."
    )


# =============================================================================
# 8. VERIFY PROVENANCE
# =============================================================================

if result[
    "protocol"
][
    "commit"
] != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "execution_parent"
][
    "commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1K execution parent mismatch."
    )


if result[
    "execution_parent"
][
    "tag"
] != PREDECESSOR_TAG:

    raise RuntimeError(
        "Stage23-1K predecessor tag mismatch."
    )


if result[
    "cache"
][
    "identifier"
] != "stage23_execution_cache_v1":

    raise RuntimeError(
        "Unexpected execution-cache identifier."
    )


if result[
    "cache"
][
    "seal_commit"
] != CACHE_COMMIT:

    raise RuntimeError(
        "Stage23-1K cache seal commit mismatch."
    )


if result[
    "cache"
][
    "seal_tag"
] != CACHE_TAG:

    raise RuntimeError(
        "Stage23-1K cache tag mismatch."
    )


if result[
    "cache"
][
    "sealed"
] is not True:

    raise RuntimeError(
        "Stage23-1K cache was not sealed."
    )


if result[
    "cache"
][
    "selected_physical_feature_digests_verified"
] != 63:

    raise RuntimeError(
        "Stage23-1K did not verify all 63 physical cache columns."
    )


if result[
    "cache"
][
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Stage23-1K unexpectedly read Parquet files."
    )


# =============================================================================
# 9. GOVERNANCE
# =============================================================================

governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:

    raise RuntimeError(
        "Unexpected Stage23-1K fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 20:

    raise RuntimeError(
        "Unexpected Stage23-1K pre-fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 22:

    raise RuntimeError(
        "Unexpected Stage23-1K post-fit count."
    )


if governance[
    "execution_cache_optimization"
] is not True:

    raise RuntimeError(
        "Optimized-cache flag missing."
    )


if governance[
    "execution_cache_scientifically_sealed"
] is not True:

    raise RuntimeError(
        "Cache-seal governance flag missing."
    )


if governance[
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Governance reports unexpected Parquet access."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
    "feature_definition_changed",
    "split_definition_changed",
    "input_dtype_changed",
    "model_specification_changed",
]:

    if governance[
        key
    ] is not False:

        raise RuntimeError(
            f"Governance violation: {key}"
        )


print()
print("[OK] governance assertions passed")
print("     Stage23 fits : 22 / 50")
print("     Parquet reads: 0")
print("     cache        : SEALED")


# =============================================================================
# 10. SCIENTIFIC ASSERTIONS
# =============================================================================

ranking = result[
    "ranking_metrics"
]

penalty = result[
    "removal_penalty_full_minus_ablated"
]

fixed = result[
    "fixed_threshold_0_50"
]

interaction = result[
    "shortcut_interaction"
]


assert_float(
    "PR-AUC",
    ranking[
        "pr_auc"
    ],
    EXPECTED_PR_AUC,
)

assert_float(
    "ROC-AUC",
    ranking[
        "roc_auc"
    ],
    EXPECTED_ROC_AUC,
)

assert_float(
    "PR-AUC removal penalty",
    penalty[
        "pr_auc"
    ],
    EXPECTED_PR_PENALTY,
)

assert_float(
    "ROC-AUC removal penalty",
    penalty[
        "roc_auc"
    ],
    EXPECTED_ROC_PENALTY,
)


assert_float(
    "Accuracy",
    fixed[
        "accuracy"
    ],
    EXPECTED_ACCURACY,
)

assert_float(
    "Precision",
    fixed[
        "precision"
    ],
    EXPECTED_PRECISION,
)

assert_float(
    "Recall",
    fixed[
        "recall"
    ],
    EXPECTED_RECALL,
)

assert_float(
    "F1",
    fixed[
        "f1"
    ],
    EXPECTED_F1,
)

assert_float(
    "FPR",
    fixed[
        "fpr"
    ],
    EXPECTED_FPR,
)

assert_float(
    "FNR",
    fixed[
        "fnr"
    ],
    EXPECTED_FNR,
)


if (
    int(fixed["tn"]) != EXPECTED_TN
    or int(fixed["fp"]) != EXPECTED_FP
    or int(fixed["fn"]) != EXPECTED_FN
    or int(fixed["tp"]) != EXPECTED_TP
):

    raise RuntimeError(
        "Stage23-1K confusion matrix differs from execution output."
    )


if interaction[
    "status"
] != "PENDING_MATCHED_CHRONOLOGICAL_CELL":

    raise RuntimeError(
        "Stage23-1K interaction status changed."
    )


if interaction[
    "required_future_cell"
] != "Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected matched chronological cell."
    )


print()
print("[OK] scientific result assertions passed")

print(
    f"     PR-AUC      : {float(ranking['pr_auc']):.12f}"
)

print(
    f"     ROC-AUC     : {float(ranking['roc_auc']):.12f}"
)

print(
    f"     PR penalty  : {float(penalty['pr_auc']):+.12f}"
)

print(
    f"     ROC penalty : {float(penalty['roc_auc']):+.12f}"
)

print(
    "     interaction : PENDING Stage23-1L"
)


# =============================================================================
# 11. COPY PERMANENT ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR
        / filename,
        TARGET_DIR
        / filename,
    )


# =============================================================================
# 12. VERIFY COPIES
# =============================================================================

print()
print("=" * 100)
print("VERIFYING REPOSITORY COPIES")
print("=" * 100)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR
        / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR
        / filename
    )


    if src_sha != dst_sha:

        raise RuntimeError(
            f"Source/destination byte mismatch: {filename}"
        )


    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 13. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT_PATH = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1K",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "sealed_utc":
        now_utc(),

    "cell": {

        "subset":
            "BEHAVIOR_ONLY",

        "split":
            "RANDOM_NATURAL",

        "removed_features":
            EXPECTED_REMOVED,

        "feature_count":
            63,

        "semantic_label":
            "behavior_restricted_feature_set",

        "interpretation_term":
            "behavior-restricted performance",
    },

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "execution_cache": {

        "commit":
            CACHE_COMMIT,

        "tag":
            CACHE_TAG,

        "scientifically_sealed":
            True,

        "parquet_files_read":
            0,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        22,

    "ranking": {

        "pr_auc":
            float(
                ranking[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                ranking[
                    "roc_auc"
                ]
            ),

        "pr_auc_removal_penalty":
            float(
                penalty[
                    "pr_auc"
                ]
            ),

        "roc_auc_removal_penalty":
            float(
                penalty[
                    "roc_auc"
                ]
            ),
    },

    "shortcut_interaction_status":
        "PENDING_MATCHED_CHRONOLOGICAL_CELL",

    "required_matched_cell":
        "Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL",

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_authorized_model_cell":
        "Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL",
}


SEAL_RECEIPT_PATH.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 14. README
# =============================================================================

README_PATH = (
    TARGET_DIR
    / "README.md"
)


README_PATH.write_text(
f"""# Stage23-1K — BEHAVIOR_ONLY × RANDOM_NATURAL

This directory freezes the RANDOM_NATURAL result for the prospectively
defined behavior-restricted feature set.

## Frozen feature restriction

Excluded exactly:

- `Dst Port`
- `Protocol`
- `Fwd Header Len`
- `Bwd Header Len`
- `Init Fwd Win Byts`
- `Init Bwd Win Byts`
- `Fwd Seg Size Min`

Retained features: 63

Semantic label:

`behavior_restricted_feature_set`

The correct interpretation term is **behavior-restricted performance**.

This result must not be described as “true performance” or as a direct
deployment-performance estimate.

## Execution

- Split: `RANDOM_NATURAL`
- Train rows: 11,529,922
- Validation rows: 2,882,481
- New boosted-model fits: 2
- Stage23 fits after this result: 22 / 50
- Execution source: sealed deterministic execution cache v1
- Parquet reads: 0
- Cache materialization: {float(result["cache"]["materialization_seconds"]):.3f} s
- Threshold optimization: none
- Subset-specific tuning: none
- Rebalancing: none

## Ranking results

- Attack prevalence: {float(result["data"]["validation"]["attack_prevalence"]):.12f}
- BEHAVIOR_ONLY PR-AUC: {float(ranking["pr_auc"]):.12f}
- FULL PR-AUC: {float(result["full_reference"]["pr_auc"]):.12f}
- FULL − BEHAVIOR_ONLY PR-AUC penalty: {float(penalty["pr_auc"]):+.12f}

- BEHAVIOR_ONLY ROC-AUC: {float(ranking["roc_auc"]):.12f}
- FULL ROC-AUC: {float(result["full_reference"]["roc_auc"]):.12f}
- FULL − BEHAVIOR_ONLY ROC-AUC penalty: {float(penalty["roc_auc"]):+.12f}

## Fixed threshold 0.50

- Accuracy: {float(fixed["accuracy"]):.12f}
- Precision: {float(fixed["precision"]):.12f}
- Recall: {float(fixed["recall"]):.12f}
- F1: {float(fixed["f1"]):.12f}
- FPR: {float(fixed["fpr"]):.12f}
- FNR: {float(fixed["fnr"]):.12f}

Confusion matrix:

- TN: {int(fixed["tn"]):,}
- FP: {int(fixed["fp"]):,}
- FN: {int(fixed["fn"]):,}
- TP: {int(fixed["tp"]):,}

## Current interpretation

Despite excluding the frozen contextual/TCP-stack feature set,
RANDOM_NATURAL ranking performance remains high:

- PR-AUC: {float(ranking["pr_auc"]):.12f}
- ROC-AUC: {float(ranking["roc_auc"]):.12f}

However, the restriction produces a larger RANDOM_NATURAL penalty than any
earlier frozen primary subset evaluated so far.

This is descriptive only at this stage.

## Interaction status

The matched chronological partner has not yet been executed.

Therefore the split × ablation interaction remains pending until:

`Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL`

## Governance

- Stage23 fits sealed after this result: 22 / 50
- Optimized execution cache used: yes
- Parquet files read: 0
- Threshold optimization: no
- Subset-specific tuning: no
- Rebalancing: no
- SHAP: no
- Placebo: no
- Raw March 1 access: no
- Raw March 2 access: no
- Stage23-0 modified: no

## Next authorized model cell

`Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL`
""",
    encoding="utf-8",
)


# =============================================================================
# 15. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPOSITORY_CHECKSUMS_PATH = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_files = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPOSITORY_CHECKSUMS_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_files
    ) + "\n",
    encoding="utf-8",
)


REPOSITORY_MANIFEST_SHA = sha256_file(
    REPOSITORY_CHECKSUMS_PATH
)


# =============================================================================
# 16. STAGE ONLY STAGE23-1K
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(
            relative_target
        ),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


expected_prefix = (
    str(
        relative_target
    )
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        expected_prefix
    )
]


if unexpected:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 100)
print("STAGED STAGE23-1K ARTIFACTS")
print("=" * 100)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 17. COMMIT
# =============================================================================

print()
print("=" * 100)
print("CREATING STAGE23-1K RESULT COMMIT")
print("=" * 100)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)


if sealed_commit == EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
):

    raise RuntimeError(
        "Repository dirty after Stage23-1K commit."
    )


print()
print("Stage23-1K commit:")
print(" ", sealed_commit)


# =============================================================================
# 18. ANNOTATED TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ]
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
)


if local_tag_commit != sealed_commit:

    raise RuntimeError(
        "Local Stage23-1K tag verification failed."
    )


# =============================================================================
# 19. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 100)
print("PUSHING STAGE23-1K")
print("=" * 100)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 20. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
)


if not remote_main_output:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 21. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":

        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":

        peeled_commit = sha


if not tag_object:

    raise RuntimeError(
        "Remote Stage23-1K annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote Stage23-1K tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 22. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-1K push."
    )


# =============================================================================
# 23. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1K SCIENTIFIC RESULT SEALED")
print("=" * 100)

print()
print("Stage23-1J parent:")
print(
    " ",
    EXPECTED_PARENT
)

print()
print("Stage23-1K commit:")
print(
    " ",
    sealed_commit
)

print()
print("Result tag:")
print(
    " ",
    RESULT_TAG
)

print()
print("Remote main:")
print(
    " ",
    remote_main
)

print()
print("Remote tag object:")
print(
    " ",
    tag_object
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    peeled_commit
)

print()
print("Source execution checksum manifest:")
print(
    " ",
    EXPECTED_SOURCE_MANIFEST_SHA
)

print()
print("Repository checksum manifest:")
print(
    " ",
    REPOSITORY_MANIFEST_SHA
)


print()
print("=" * 100)
print("FROZEN BEHAVIOR-RESTRICTED RANDOM RESULT")
print("=" * 100)

print()
print(
    f"PR-AUC      : {float(ranking['pr_auc']):.12f}"
)

print(
    f"ROC-AUC     : {float(ranking['roc_auc']):.12f}"
)

print(
    f"PR penalty  : {float(penalty['pr_auc']):+.12f}"
)

print(
    f"ROC penalty : {float(penalty['roc_auc']):+.12f}"
)

print()
print(
    "Interpretation term: behavior-restricted performance"
)

print()
print(
    "Interaction: PENDING matched chronological cell"
)


print()
print("=" * 100)
print("OPTIMIZED EXECUTION")
print("=" * 100)

print()
print("Cache used       : stage23_execution_cache_v1")
print("Parquet reads    : 0")
print(
    "Materialization :",
    f"{float(result['cache']['materialization_seconds']):.3f} s"
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits sealed      : 22 / 50")
print("Model fits seal          : 0")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Execution cache changed  : NO")
print("Git status               : CLEAN")


print()
print("=" * 100)
print("STAGE23-1K COMPLETE AND SEALED")
print("=" * 100)

print()
print("NEXT AUTHORIZED MODEL CELL:")
print(
    "  Stage23-1L — "
    "BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL"
)

print()
print(
    "After Stage23-1L executes and is sealed,"
)
print(
    "the PRIMARY boosted-model ablation block will be complete at 24 / 50 fits."
)

print("=" * 100)

STAGE23-1K — SCIENTIFIC RESULT SEAL

[OK] branch             : main
[OK] parent HEAD        : 12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1J tag     : 12e1d1c45a6060473d1d5aa3d2a53d1a459fcc74
[OK] execution-cache tag: 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] Stage23-1K tag     : ABSENT

[OK] source checksum-manifest SHA256:
  c3ccb4eda01778f72facdadaef2d2c1a2b766715df87530f6cc9a142a2f55d48

VERIFYING STAGE23-1K CORE ARTIFACTS

[EXACT] stage23_1k_behavior_only_random_natural_result.json
[EXACT] behavior_only_random_natural_lightgbm_model.txt
[EXACT] behavior_only_random_natural_xgboost_model.json
[EXACT] behavior_only_random_natural_validation_probabilities.npz

[OK] governance assertions passed
     Stage23 fits : 22 / 50
     Parquet reads: 0
     cache        : SEALED

[OK] scientific result assertions passed
     PR-AUC      : 0.981588003857
     ROC-AUC     : 0.99614124

In [27]:
# =============================================================================
# STAGE23-1L
# BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL
# FINAL PRIMARY BOOSTED-MODEL EXECUTION
# OPTIMIZED — SEALED EXECUTION CACHE V1
# =============================================================================
#
# FROZEN SUBSET
# -------------
# BEHAVIOR_ONLY
#
# Retained features: 63
#
# Excluded EXACTLY:
#   - Dst Port
#   - Protocol
#   - Fwd Header Len
#   - Bwd Header Len
#   - Init Fwd Win Byts
#   - Init Bwd Win Byts
#   - Fwd Seg Size Min
#
# Semantic label:
#   behavior_restricted_feature_set
#
# Correct interpretation:
#   "behavior-restricted performance"
#
# DO NOT call this:
#   "true performance"
#
# REQUIRED SCIENTIFIC PREDECESSOR:
#   Stage23-1K
#   eae3a7c409687f086924cce84be65bca5e7b10ba
#
# SEALED EXECUTION CACHE:
#   stage23-execution-cache-v1
#   66e1c6382caa63699766e605df63df02beff894f
#
# FIT ACCOUNTING:
#   before : 22 / 50
#   this   :  2
#   after  : 24 / 50
#
# AFTER SUCCESS:
#   ALL PRIMARY BOOSTED ABLATION FITS COMPLETE
#
# NO:
#   Parquet reads
#   raw Mar1
#   raw Mar2
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   placebo
#
# FAIL CLOSED:
# If either fit completes and anything later fails, DO NOT blindly rerun.
# Inspect execution_state.json first.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-1L"

TARGET_SUBSET = "BEHAVIOR_ONLY"
TARGET_SPLIT = "CHRONOLOGICAL_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

PREDECESSOR_COMMIT = (
    "eae3a7c409687f086924cce84be65bca5e7b10ba"
)

PREDECESSOR_TAG = (
    "stage23-1k-behavior-only-random-natural-v1"
)

RANDOM_RESULT_SHA256 = (
    "de40919dfced8471a48cbef06b1816b05a8e3b797eec10e35d2fade2effa0d61"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_DATA_MANIFEST_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

CACHE_SEMANTIC_70F_SHA256 = (
    "8c43ab0e36a65de1c095acb1ebf9e7d029a9e2b1cc353455a66617db8a899bb2"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

CACHE_SEAL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

RANDOM_PARTNER_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "random"
    / "behavior_only"
)

RANDOM_PARTNER_RESULT = (
    RANDOM_PARTNER_DIR
    / "stage23_1k_behavior_only_random_natural_result.json"
)

RANDOM_PARTNER_SEAL = (
    RANDOM_PARTNER_DIR
    / "seal_receipt.json"
)

FULL_CHRONO_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_1l_behavior_only_chronological_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_1l_behavior_only_chronological_natural_result.json"
)


# =============================================================================
# 2. FROZEN DATA COUNTS
# =============================================================================

N_DEVELOPMENT = 14_412_403

TRAIN_ROWS = 13_818_623
VALIDATION_ROWS = 593_780

TRAIN_BENIGN = 11_908_580
TRAIN_ATTACK = 1_910_043

VALIDATION_BENIGN = 531_524
VALIDATION_ATTACK = 62_256

N_FEATURES = 63


# =============================================================================
# 3. FROZEN EXCLUSION LIST
# =============================================================================

EXPECTED_REMOVED = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


# =============================================================================
# 4. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (p.stdout or "").strip()

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    temp = path.with_suffix(
        path.suffix + ".tmp"
    )

    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    temp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    resolved = Path(path).resolve()

    lowered = str(resolved).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in lowered:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {resolved}\n"
                f"marker : {marker}"
            )

    return resolved


# =============================================================================
# 5. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-1L OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        "Do NOT delete or rerun blindly.\n\n"
        "Existing execution state:\n"
        + json.dumps(
            existing_state,
            indent=2,
        )
    )


# =============================================================================
# 6. GIT / GOVERNANCE PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1L — BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL")
print("FINAL PRIMARY BOOSTED-MODEL EXECUTION")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_SEAL_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-1L MUST START FROM SEALED STAGE23-1K.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1K result tag mismatch."
    )


if cache_tag_commit != CACHE_SEAL_COMMIT:

    raise RuntimeError(
        "Stage23 execution-cache tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


if not RANDOM_PARTNER_SEAL.exists():

    raise RuntimeError(
        "Stage23-1K seal receipt missing."
    )


random_seal = json.loads(
    RANDOM_PARTNER_SEAL.read_text(
        encoding="utf-8"
    )
)


if random_seal[
    "stage23_models_fit_total"
] != 22:

    raise RuntimeError(
        "Expected Stage23-1K to seal 22 / 50 fits."
    )


if random_seal[
    "next_authorized_model_cell"
] != "Stage23-1L BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1K does not authorize Stage23-1L."
    )


print("[OK] branch             :", branch)
print("[OK] HEAD               :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1K tag     :", predecessor_tag_commit)
print("[OK] cache seal tag     :", cache_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] fits before cell   : 22 / 50")
print("[OK] Stage23-1L         : AUTHORIZED")


# =============================================================================
# 7. VERIFY STAGE23-0
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum manifest changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-0 artifact: {filename}"
        )


    if sha256_file(
        path
    ) != expected_sha:

        raise RuntimeError(
            f"Stage23-0 artifact mutated: {filename}"
        )


print(
    f"[OK] Stage23-0 artifacts : "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 8. VERIFY SEALED CACHE
# =============================================================================

CACHE_METADATA_REPO = (
    CACHE_SEAL_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_REPO = (
    CACHE_SEAL_DIR
    / "data_checksums.sha256"
)

CACHE_SEAL_RECEIPT = (
    CACHE_SEAL_DIR
    / "seal_receipt.json"
)


for path in [
    CACHE_METADATA_REPO,
    CACHE_DATA_MANIFEST_REPO,
    CACHE_SEAL_RECEIPT,
]:

    if not path.exists():

        raise RuntimeError(
            f"Missing sealed cache artifact: {path}"
        )


if sha256_file(
    CACHE_METADATA_REPO
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Sealed cache metadata SHA256 mismatch."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_REPO
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Sealed cache manifest SHA256 mismatch."
    )


cache_seal = json.loads(
    CACHE_SEAL_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if cache_seal[
    "status"
] != "EXECUTION_CACHE_VERIFICATION_FROZEN":

    raise RuntimeError(
        "Execution cache is not scientifically sealed."
    )


if cache_seal[
    "semantic_equivalence"
][
    "cache_reconstructed_row_major_70f_sha256"
] != CACHE_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Sealed execution-cache semantic digest mismatch."
    )


# =============================================================================
# 9. VERIFY RUNTIME CACHE
# =============================================================================

CACHE_METADATA_RUNTIME = guard_path(
    CACHE_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_RUNTIME = guard_path(
    CACHE_DIR
    / "data_checksums.sha256"
)


if not CACHE_METADATA_RUNTIME.exists():

    raise RuntimeError(
        "Runtime cache metadata missing."
    )


if not CACHE_DATA_MANIFEST_RUNTIME.exists():

    raise RuntimeError(
        "Runtime cache manifest missing."
    )


if sha256_file(
    CACHE_METADATA_RUNTIME
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Runtime cache metadata differs from sealed metadata."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_RUNTIME
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Runtime cache manifest differs from sealed manifest."
    )


cache_metadata = json.loads(
    CACHE_METADATA_RUNTIME.read_text(
        encoding="utf-8"
    )
)


if cache_metadata[
    "semantic_equivalence"
][
    "exact_byte_equality"
] is not True:

    raise RuntimeError(
        "Execution-cache semantic equivalence flag is false."
    )


print("[OK] sealed runtime cache metadata exact")


# =============================================================================
# 10. LOAD FROZEN SPECS
# =============================================================================

feature_spec = json.loads(
    (
        PROTOCOL_DIR
        / "feature_subset_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    (
        PROTOCOL_DIR
        / "model_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    (
        PROTOCOL_DIR
        / "inherited_splits.json"
    ).read_text(
        encoding="utf-8"
    )
)


subset_spec = (
    feature_spec[
        "subsets"
    ][
        TARGET_SUBSET
    ]
)


FEATURES = list(
    subset_spec[
        "features"
    ]
)

REMOVED_FEATURES = list(
    subset_spec[
        "removed"
    ]
)


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nFrozen BEHAVIOR_ONLY exclusion list changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if subset_spec[
    "mode"
] != "whitelist":

    raise RuntimeError(
        "BEHAVIOR_ONLY must remain the frozen whitelist."
    )


if subset_spec[
    "feature_count"
] != 63:

    raise RuntimeError(
        "BEHAVIOR_ONLY feature count changed."
    )


if len(
    FEATURES
) != 63:

    raise RuntimeError(
        "Expected exactly 63 retained BEHAVIOR_ONLY features."
    )


if subset_spec[
    "semantic_label"
] != "behavior_restricted_feature_set":

    raise RuntimeError(
        "BEHAVIOR_ONLY semantic label changed."
    )


for feature in EXPECTED_REMOVED:

    if feature in FEATURES:

        raise RuntimeError(
            f"Excluded feature present: {feature}"
        )


print()
print("Frozen target:")
print("  subset        :", TARGET_SUBSET)
print("  split         :", TARGET_SPLIT)
print("  retained      :", len(FEATURES))
print("  excluded      :", REMOVED_FEATURES)
print(
    "  semantic label:",
    subset_spec[
        "semantic_label"
    ]
)
print(
    "  interpretation:",
    "behavior-restricted performance"
)


# =============================================================================
# 11. VERIFY CACHE FEATURE PROJECTION
# =============================================================================

FULL_FEATURES = (
    cache_metadata[
        "full_feature_order"
    ]
)


if FULL_FEATURES != feature_spec[
    "full_feature_order"
]:

    raise RuntimeError(
        "Cache full feature order differs from Stage23 protocol."
    )


CACHE_INDICES = (
    cache_metadata[
        "primary_subset_feature_indices"
    ][
        TARGET_SUBSET
    ]
)


if len(
    CACHE_INDICES
) != 63:

    raise RuntimeError(
        "Cache BEHAVIOR_ONLY projection count mismatch."
    )


projected_features = [
    FULL_FEATURES[
        index
    ]
    for index in CACHE_INDICES
]


if projected_features != FEATURES:

    raise RuntimeError(
        "Cache BEHAVIOR_ONLY feature order differs from protocol."
    )


# =============================================================================
# 12. VERIFY CHRONO SPLIT
# =============================================================================

chrono_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


if int(
    chrono_spec[
        "train"
    ][
        "rows"
    ]
) != TRAIN_ROWS:

    raise RuntimeError(
        "Chronological train row count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "rows"
    ]
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Chronological validation row count changed."
    )


if int(
    chrono_spec[
        "train"
    ][
        "benign"
    ]
) != TRAIN_BENIGN:

    raise RuntimeError(
        "Chronological train benign count changed."
    )


if int(
    chrono_spec[
        "train"
    ][
        "attack"
    ]
) != TRAIN_ATTACK:

    raise RuntimeError(
        "Chronological train attack count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "benign"
    ]
) != VALIDATION_BENIGN:

    raise RuntimeError(
        "Chronological validation benign count changed."
    )


if int(
    chrono_spec[
        "validation"
    ][
        "attack"
    ]
) != VALIDATION_ATTACK:

    raise RuntimeError(
        "Chronological validation attack count changed."
    )


print()
print("[OK] frozen CHRONOLOGICAL_NATURAL membership")
print(
    "     train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "     validation :",
    f"{VALIDATION_ROWS:,}"
)


# =============================================================================
# 13. VERIFY SEALED RANDOM PARTNER
# =============================================================================

if not RANDOM_PARTNER_RESULT.exists():

    raise RuntimeError(
        "Stage23-1K random partner result missing."
    )


if sha256_file(
    RANDOM_PARTNER_RESULT
) != RANDOM_RESULT_SHA256:

    raise RuntimeError(
        "Stage23-1K random result SHA256 changed."
    )


random_result = json.loads(
    RANDOM_PARTNER_RESULT.read_text(
        encoding="utf-8"
    )
)


if random_result[
    "cell"
][
    "subset"
] != "BEHAVIOR_ONLY":

    raise RuntimeError(
        "Matched random subset mismatch."
    )


if random_result[
    "cell"
][
    "split"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Matched random split mismatch."
    )


if random_result[
    "cell"
][
    "feature_count"
] != 63:

    raise RuntimeError(
        "Matched random feature count mismatch."
    )


print("[OK] sealed Stage23-1K RANDOM_NATURAL partner exact")


# =============================================================================
# 14. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] frozen package versions verified")


# =============================================================================
# 15. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 16. PARSE SEALED CACHE MANIFEST
# =============================================================================

manifest_hashes = {}


for line in CACHE_DATA_MANIFEST_RUNTIME.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, relative_path = line.split(
        "  ",
        1,
    )

    manifest_hashes[
        relative_path
    ] = digest


if len(
    manifest_hashes
) != 75:

    raise RuntimeError(
        f"Expected 75 cache manifest entries; "
        f"found {len(manifest_hashes)}"
    )


# =============================================================================
# 17. VERIFY METADATA ARRAYS
# =============================================================================

LABEL_PATH = guard_path(
    CACHE_DIR
    / "binary_label.uint8.dat"
)

DAY_ID_PATH = guard_path(
    CACHE_DIR
    / "day_id.uint8.dat"
)

CLEAN_POSITION_PATH = guard_path(
    CACHE_DIR
    / "clean_position.int64.dat"
)


for path, manifest_name in [
    (
        LABEL_PATH,
        "binary_label.uint8.dat",
    ),
    (
        DAY_ID_PATH,
        "day_id.uint8.dat",
    ),
    (
        CLEAN_POSITION_PATH,
        "clean_position.int64.dat",
    ),
]:

    if not path.exists():

        raise RuntimeError(
            f"Required cache array missing: {path}"
        )


    if sha256_file(
        path
    ) != manifest_hashes[
        manifest_name
    ]:

        raise RuntimeError(
            f"Cache metadata array differs from seal: {manifest_name}"
        )


labels_all = np.memmap(
    LABEL_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


day_ids_all = np.memmap(
    DAY_ID_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


clean_positions_all = np.memmap(
    CLEAN_POSITION_PATH,
    dtype="<i8",
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


# =============================================================================
# 18. VERIFY CHRONO DAY BOUNDARY
# =============================================================================

if not np.all(
    day_ids_all[
        :TRAIN_ROWS
    ] <= 6
):

    raise RuntimeError(
        "Chronological training cache contains invalid day_id."
    )


if not np.all(
    day_ids_all[
        TRAIN_ROWS:
    ] == 7
):

    raise RuntimeError(
        "Chronological validation is not exactly day_id 7."
    )


y_train = labels_all[
    :TRAIN_ROWS
]

y_validation = labels_all[
    TRAIN_ROWS:
]


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if (
    train_attack != TRAIN_ATTACK
    or train_benign != TRAIN_BENIGN
):

    raise RuntimeError(
        "Chronological training class counts changed."
    )


if (
    validation_attack != VALIDATION_ATTACK
    or validation_benign != VALIDATION_BENIGN
):

    raise RuntimeError(
        "Chronological validation class counts changed."
    )


validation_clean_positions = np.asarray(
    clean_positions_all[
        TRAIN_ROWS:
    ],
    dtype=np.int64,
)


expected_positions = np.arange(
    TRAIN_ROWS,
    N_DEVELOPMENT,
    dtype=np.int64,
)


if not np.array_equal(
    validation_clean_positions,
    expected_positions,
):

    raise RuntimeError(
        "Chronological validation clean_position membership changed."
    )


del expected_positions


# =============================================================================
# 19. VERIFY SELECTED FEATURE RECORDS
# =============================================================================

feature_records = {
    int(record["index"]):
        record
    for record in cache_metadata[
        "feature_columns"
    ]
}


SELECTED_FEATURE_RECORDS = []


for index in CACHE_INDICES:

    if index not in feature_records:

        raise RuntimeError(
            f"Missing cache feature record: {index}"
        )


    record = feature_records[
        index
    ]


    if record[
        "feature"
    ] != FULL_FEATURES[
        index
    ]:

        raise RuntimeError(
            f"Cache feature-name mismatch at index {index}"
        )


    relative_path = record[
        "path"
    ]


    if relative_path not in manifest_hashes:

        raise RuntimeError(
            f"Feature missing from sealed manifest: {relative_path}"
        )


    if record[
        "sha256"
    ] != manifest_hashes[
        relative_path
    ]:

        raise RuntimeError(
            f"Cache metadata/manifest SHA mismatch: {relative_path}"
        )


    physical_path = guard_path(
        CACHE_DIR
        / relative_path
    )


    if not physical_path.exists():

        raise RuntimeError(
            f"Physical cache feature missing: {physical_path}"
        )


    expected_bytes = (
        N_DEVELOPMENT
        * 8
    )


    if physical_path.stat().st_size != expected_bytes:

        raise RuntimeError(
            f"Physical cache feature size mismatch: {record['feature']}"
        )


    SELECTED_FEATURE_RECORDS.append(
        {
            **record,
            "physical_path":
                physical_path,
        }
    )


# =============================================================================
# 20. STORAGE PREFLIGHT
# =============================================================================

MATRIX_BYTES = (
    N_DEVELOPMENT
    * N_FEATURES
    * 8
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    MATRIX_BYTES
    + 1 * 1024**3
)


print()
print("=" * 100)
print("OPTIMIZED CHRONOLOGICAL MATERIALIZATION PLAN")
print("=" * 100)
print()

print(
    "Source             : sealed execution cache v1"
)
print(
    "Parquet reads      : 0"
)
print(
    "Selected features  :",
    N_FEATURES
)
print(
    "Train rows         :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows    :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Temporary matrix   :",
    f"{MATRIX_BYTES / 1024**3:.3f} GiB"
)
print(
    "Working free       :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nINSUFFICIENT DISK FOR STAGE23-1L\n"
        f"required with safety margin: "
        f"{minimum_free / 1024**3:.3f} GiB\n"
        f"available                  : "
        f"{disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 21. CREATE OUTPUT + FAIL-CLOSED STATE
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "subset":
        TARGET_SUBSET,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        PREDECESSOR_COMMIT,

    "cache_seal_commit":
        CACHE_SEAL_COMMIT,

    "cache_seal_tag":
        CACHE_SEAL_TAG,

    "stage23_total_model_fits_before_cell":
        22,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 22. OPEN SELECTED RAW FEATURE COLUMNS
# =============================================================================

source_feature_mmaps = []


for record in SELECTED_FEATURE_RECORDS:

    source_feature_mmaps.append(
        np.memmap(
            record[
                "physical_path"
            ],
            dtype="<f8",
            mode="r",
            shape=(
                N_DEVELOPMENT,
            ),
        )
    )


# =============================================================================
# 23. MATERIALIZE 63F ROW-MAJOR MATRIX
# =============================================================================

X_ALL_PATH = (
    RUNTIME_DIR
    / "X_behavior_only_float64.dat"
)


X_all = np.memmap(
    X_ALL_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        N_DEVELOPMENT,
        N_FEATURES,
    ),
)


CHUNK_ROWS = 250_000

feature_hashers = [
    hashlib.sha256()
    for _ in range(
        N_FEATURES
    )
]


materialize_start = (
    time.perf_counter()
)


print()
print("=" * 100)
print("MATERIALIZING BEHAVIOR_ONLY FROM SEALED EXECUTION CACHE")
print("=" * 100)
print()


total_chunks = (
    N_DEVELOPMENT
    + CHUNK_ROWS
    - 1
) // CHUNK_ROWS


for chunk_index, start in enumerate(
    range(
        0,
        N_DEVELOPMENT,
        CHUNK_ROWS,
    ),
    start=1,
):

    end = min(
        start
        + CHUNK_ROWS,
        N_DEVELOPMENT,
    )

    n_rows = (
        end
        - start
    )


    chunk = np.empty(
        (
            n_rows,
            N_FEATURES,
        ),
        dtype=np.float64,
    )


    for output_col, source_mm in enumerate(
        source_feature_mmaps
    ):

        source_slice = np.asarray(
            source_mm[
                start:end
            ],
            dtype="<f8",
        )


        feature_hashers[
            output_col
        ].update(
            np.ascontiguousarray(
                source_slice,
                dtype="<f8",
            ).tobytes(
                order="C"
            )
        )


        chunk[
            :,
            output_col
        ] = source_slice


    X_all[
        start:end,
        :
    ] = chunk


    if (
        chunk_index == 1
        or chunk_index % 10 == 0
        or chunk_index == total_chunks
    ):

        print(
            f"  chunk {chunk_index:>3}/{total_chunks}"
            f" — rows {start:,}..{end - 1:,}"
        )


    del chunk

    gc.collect()


X_all.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


# =============================================================================
# 24. VERIFY ALL 63 PHYSICAL FEATURE DIGESTS
# =============================================================================

for output_col, record in enumerate(
    SELECTED_FEATURE_RECORDS
):

    actual_digest = (
        feature_hashers[
            output_col
        ].hexdigest()
    )

    expected_digest = (
        record[
            "sha256"
        ]
    )


    if actual_digest != expected_digest:

        raise RuntimeError(
            "\nPHYSICAL CACHE FEATURE MUTATION DETECTED\n"
            f"feature  : {record['feature']}\n"
            f"expected : {expected_digest}\n"
            f"actual   : {actual_digest}"
        )


print()
print(
    "[OK] selected physical cache columns:"
    " 63 / 63 exact"
)


if X_ALL_PATH.stat().st_size != MATRIX_BYTES:

    raise RuntimeError(
        "\nTemporary behavior-only matrix size mismatch.\n"
        f"expected: {MATRIX_BYTES:,}\n"
        f"actual  : {X_ALL_PATH.stat().st_size:,}"
    )


X_train = X_all[
    :TRAIN_ROWS,
    :
]

X_validation = X_all[
    TRAIN_ROWS:,
    :
]


if X_train.shape != (
    TRAIN_ROWS,
    N_FEATURES,
):

    raise RuntimeError(
        "Training matrix shape mismatch."
    )


if X_validation.shape != (
    VALIDATION_ROWS,
    N_FEATURES,
):

    raise RuntimeError(
        "Validation matrix shape mismatch."
    )


print()
print("[OK] optimized chronological materialization")
print(
    "     train          :",
    X_train.shape
)
print(
    "     validation     :",
    X_validation.shape
)
print(
    "     dtype          :",
    X_all.dtype
)
print(
    "     Parquet reads  : 0"
)
print(
    "     materialization:",
    f"{materialize_seconds:.3f} s"
)


execution_state.update({

    "status":
        "MATERIALIZED_FROM_SEALED_CACHE",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,

    "input_dtype":
        "float64",

    "physical_feature_digests_verified":
        63,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 25. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 26. FIT 1 / 2 — LIGHTGBM
# =============================================================================

print()
print("=" * 100)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 100)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "behavior_only_chronological_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 27. FIT 2 / 2 — XGBOOST
# =============================================================================

print()
print("=" * 100)
print("FIT 2 / 2 — XGBOOST")
print("=" * 100)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "behavior_only_chronological_natural_xgboost_model.json"
)


xgb_model.save_model(
    str(
        XGB_MODEL_PATH
    )
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 28. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 29. METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 30. FIXED THRESHOLD 0.50
# =============================================================================

predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 31. FROZEN FULL CHRONO REFERENCE
# =============================================================================

if not FULL_CHRONO_RESULT.exists():

    raise RuntimeError(
        f"Frozen FULL chronological result missing:\n{FULL_CHRONO_RESULT}"
    )


full_chrono = json.loads(
    FULL_CHRONO_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_chrono[
    "cell"
] != "CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Unexpected FULL chronological reference."
    )


full_pr_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_chrono[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_chrono[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 32. CHRONOLOGICAL REMOVAL PENALTIES
# =============================================================================

chrono_pr_penalty = float(
    full_pr_auc
    - pr_auc
)


chrono_roc_penalty = float(
    full_roc_auc
    - roc_auc
)


chrono_f1_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


chrono_recall_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


chrono_fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 33. FROZEN RANDOM PENALTIES FROM STAGE23-1K
# =============================================================================

random_penalty = (
    random_result[
        "removal_penalty_full_minus_ablated"
    ]
)


random_pr_penalty = float(
    random_penalty[
        "pr_auc"
    ]
)


random_roc_penalty = float(
    random_penalty[
        "roc_auc"
    ]
)


random_f1_penalty = float(
    random_penalty[
        "f1_at_0_50"
    ]
)


random_recall_penalty = float(
    random_penalty[
        "recall_at_0_50"
    ]
)


random_fpr_change = float(
    random_penalty[
        "fpr_at_0_50"
    ]
)


# =============================================================================
# 34. FINAL PRIMARY BEHAVIOR_ONLY INTERACTION
# =============================================================================
#
# Frozen definition:
#
#   Δ_R(S) = FULL_R - ABL_R
#   Δ_C(S) = FULL_C - ABL_C
#
#   I(S) = Δ_R(S) - Δ_C(S)
#
# =============================================================================

interaction_pr_auc = float(
    random_pr_penalty
    - chrono_pr_penalty
)


interaction_roc_auc = float(
    random_roc_penalty
    - chrono_roc_penalty
)


interaction_f1 = float(
    random_f1_penalty
    - chrono_f1_penalty
)


interaction_recall = float(
    random_recall_penalty
    - chrono_recall_penalty
)


interaction_fpr = float(
    random_fpr_change
    - chrono_fpr_change
)


# =============================================================================
# 35. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "behavior_only_chronological_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=(
        validation_clean_positions
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 36. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "BEHAVIOR_ONLY_CHRONOLOGICAL_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent": {

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "scientific_random_partner": {

        "stage":
            "Stage23-1K",

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,

        "result_sha256":
            RANDOM_RESULT_SHA256,
    },

    "cache": {

        "identifier":
            "stage23_execution_cache_v1",

        "seal_commit":
            CACHE_SEAL_COMMIT,

        "seal_tag":
            CACHE_SEAL_TAG,

        "sealed":
            True,

        "metadata_sha256":
            CACHE_METADATA_SHA256,

        "data_manifest_sha256":
            CACHE_DATA_MANIFEST_SHA256,

        "semantic_70f_sha256":
            CACHE_SEMANTIC_70F_SHA256,

        "selected_physical_feature_digests_verified":
            63,

        "parquet_files_read":
            0,

        "materialization_seconds":
            materialize_seconds,

        "temporary_row_major_matrix_bytes":
            MATRIX_BYTES,
    },

    "cell": {

        "subset":
            TARGET_SUBSET,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,

        "semantic_label":
            subset_spec[
                "semantic_label"
            ],

        "interpretation_term":
            "behavior-restricted performance",
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "row_randomization":
            False,

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,

            "days":
                chrono_spec[
                    "train_days"
                ],
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,

            "days":
                chrono_spec[
                    "validation_days"
                ],
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R CHRONOLOGICAL_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_ablated": {

        "pr_auc":
            chrono_pr_penalty,

        "roc_auc":
            chrono_roc_penalty,

        "f1_at_0_50":
            chrono_f1_penalty,

        "recall_at_0_50":
            chrono_recall_penalty,

        "fpr_at_0_50":
            chrono_fpr_change,
    },

    "matched_random_reference": {

        "source":
            "Sealed Stage23-1K",

        "result_sha256":
            RANDOM_RESULT_SHA256,

        "pr_auc_removal_penalty":
            random_pr_penalty,

        "roc_auc_removal_penalty":
            random_roc_penalty,

        "f1_at_0_50_removal_penalty":
            random_f1_penalty,

        "recall_at_0_50_removal_penalty":
            random_recall_penalty,

        "fpr_at_0_50_change":
            random_fpr_change,
    },

    "shortcut_interaction": {

        "definition":
            "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

        "point_estimate_available":
            True,

        "pr_auc":
            interaction_pr_auc,

        "roc_auc":
            interaction_roc_auc,

        "f1_at_0_50":
            interaction_f1,

        "recall_at_0_50":
            interaction_recall,

        "fpr_at_0_50":
            interaction_fpr,

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE",
    },

    "primary_boosted_ablation_status": {

        "fits_complete_after_this_cell":
            True,

        "boosted_primary_fits_completed":
            24,

        "boosted_primary_fits_expected":
            24,

        "all_primary_subsets_have_both_splits":
            True,

        "seal_pending":
            True,
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "optimized_cache_materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,

        "parquet_files_read":
            0,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            22,

        "stage23_total_model_fits_after_cell":
            24,

        "stage23_total_authorized_model_fits":
            50,

        "primary_boosted_model_fits_complete":
            True,

        "primary_boosted_model_fits_total":
            24,

        "execution_cache_optimization":
            True,

        "execution_cache_scientifically_sealed":
            True,

        "parquet_files_read":
            0,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "placebo_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,

        "feature_definition_changed":
            False,

        "split_definition_changed":
            False,

        "input_dtype_changed":
            False,

        "model_specification_changed":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-1L before any further Stage23 execution.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 37. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 38. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        24,

    "primary_boosted_model_fits_complete":
        True,

    "stage23_metrics_calculated":
        True,

    "interaction_point_estimate_calculated":
        True,

    "interaction_ci_calculated":
        False,

    "optimized_cache_used":
        True,

    "parquet_files_read":
        0,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 39. DELETE ONLY TRANSIENT TRAINING MATRIX
# =============================================================================
#
# KEEP:
#   /kaggle/working/stage23_execution_cache_v1
#
# =============================================================================

del X_train
del X_validation
del X_all

del y_train
del y_validation

del labels_all
del day_ids_all
del clean_positions_all

for i in range(
    len(source_feature_mmaps)
):

    source_feature_mmaps[
        i
    ] = None


del source_feature_mmaps

gc.collect()


if X_ALL_PATH.exists():

    X_ALL_PATH.unlink()


for path in [
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 40. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1L COMPLETE — RESULT UNSEALED")
print("=" * 100)

print()
print("Cell:")
print(
    "  subset :",
    TARGET_SUBSET
)
print(
    "  split  :",
    TARGET_SPLIT
)

print()
print("Behavior-restricted feature set:")
print(
    "  retained:",
    N_FEATURES
)
print(
    "  excluded:",
    REMOVED_FEATURES
)
print(
    "  semantics:",
    subset_spec[
        "semantic_label"
    ]
)
print(
    "  interpretation:",
    "behavior-restricted performance"
)

print()
print("Optimized execution:")
print(
    "  cache           : stage23_execution_cache_v1"
)
print(
    "  Parquet reads   : 0"
)
print(
    "  materialization :",
    f"{materialize_seconds:.3f} s"
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM     : 1")
print("  XGBoost      : 1")
print("  THIS CELL    : 2")
print("  STAGE23 TOTAL: 24 / 50")


print()
print("=" * 100)
print("BEHAVIOR-RESTRICTED CHRONOLOGICAL RANKING")
print("=" * 100)

print()
print(
    f"Attack prevalence            : {attack_prevalence:.12f}"
)

print(
    f"BEHAVIOR_ONLY PR-AUC         : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                  : {full_pr_auc:.12f}"
)

print(
    f"Chronological PR penalty     : {chrono_pr_penalty:+.12f}"
)

print()
print(
    f"BEHAVIOR_ONLY ROC-AUC        : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                 : {full_roc_auc:.12f}"
)

print(
    f"Chronological ROC penalty    : {chrono_roc_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence          : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 100)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 100)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 100)
print("BEHAVIOR_ONLY SPLIT × ABLATION INTERACTION")
print("=" * 100)

print()
print("PR-AUC:")
print(
    f"  random penalty        : {random_pr_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_pr_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_pr_auc:+.12f}"
)

print()
print("ROC-AUC:")
print(
    f"  random penalty        : {random_roc_penalty:+.12f}"
)
print(
    f"  chronological penalty : {chrono_roc_penalty:+.12f}"
)
print(
    f"  interaction           : {interaction_roc_auc:+.12f}"
)

print()
print("Supplementary:")
print(
    f"  F1 interaction     : {interaction_f1:+.12f}"
)
print(
    f"  Recall interaction : {interaction_recall:+.12f}"
)
print(
    f"  FPR interaction    : {interaction_fpr:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 100)
print("PRIMARY BOOSTED-MODEL STATUS")
print("=" * 100)

print()
print("Frozen primary subsets : 7")
print("Splits per subset      : 2")
print("Models per reduced cell: 2")
print()
print("PRIMARY BOOSTED FITS COMPLETE : 24 / 24")
print("STAGE23 TOTAL FITS            : 24 / 50")
print()
print(
    "All seven frozen primary subsets now have"
)
print(
    "RANDOM_NATURAL and CHRONOLOGICAL_NATURAL results."
)


print()
print("=" * 100)
print("ARTIFACTS")
print("=" * 100)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits completed      : 24 / 50")
print("Primary boosted fits        : 24 / 24 COMPLETE")
print("Optimized cache used        : YES")
print("Parquet files read          : 0")
print("Threshold optimization      : NO")
print("Subset-specific tuning      : NO")
print("Rebalancing                 : NO")
print("SHAP executed               : NO")
print("Placebo executed            : NO")
print("Raw Mar1 read               : NO")
print("Raw Mar2 read               : NO")
print("Stage23-0 modified          : NO")
print("Git commit created          : NO")
print("Git tag created             : NO")


print()
print("=" * 100)
print("NEXT ACTION")
print("=" * 100)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-1L."
)

print()
print(
    "After the Stage23-1L seal:"
)
print(
    "  PRIMARY BOOSTED ABLATION BLOCK = COMPLETE"
)
print(
    "  Stage23 fit accounting          = 24 / 50"
)
print(
    "  Remaining model fits            = 26"
)
print(
    "    - placebo boosted fits         = 20"
)
print(
    "    - depth-1 stump fits           = 6"
)

print("=" * 100)

STAGE23-1L — BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL
FINAL PRIMARY BOOSTED-MODEL EXECUTION

[OK] branch             : main
[OK] HEAD               : eae3a7c409687f086924cce84be65bca5e7b10ba
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1K tag     : eae3a7c409687f086924cce84be65bca5e7b10ba
[OK] cache seal tag     : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] fits before cell   : 22 / 50
[OK] Stage23-1L         : AUTHORIZED
[OK] Stage23-0 artifacts : 23/23 exact
[OK] sealed runtime cache metadata exact

Frozen target:
  subset        : BEHAVIOR_ONLY
  split         : CHRONOLOGICAL_NATURAL
  retained      : 63
  excluded      : ['Dst Port', 'Protocol', 'Fwd Header Len', 'Bwd Header Len', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Min']
  semantic label: behavior_restricted_feature_set
  interpretation: behavior-restricted performance

[OK] frozen CHRONOLOGICAL_NATURAL membership
     train      : 13,818,623
   

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 487.911 s
     model SHA256: 65af6dcf849230508f4ffa738a93ad4c1cc6eed80c67083793085d6319d78d7c

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 136.064 s
     model SHA256: 2816b36504a717b87bf3bd2c761ef7d7df0f23988f8f377f6a63a0b45cfb5310

STAGE23-1L COMPLETE — RESULT UNSEALED

Cell:
  subset : BEHAVIOR_ONLY
  split  : CHRONOLOGICAL_NATURAL

Behavior-restricted feature set:
  retained: 63
  excluded: ['Dst Port', 'Protocol', 'Fwd Header Len', 'Bwd Header Len', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Min']
  semantics: behavior_restricted_feature_set
  interpretation: behavior-restricted performance

Optimized execution:
  cache           : stage23_execution_cache_v1
  Parquet reads   : 0
  materialization : 58.791 s

Data:
  train      : 13,818,623
  validation : 593,780

New model fits:
  LightGBM     : 1
  XGBoost      : 1
  THIS CELL    : 2
  STAGE23 TOTAL: 24 / 50

BEHAVIOR-RESTRICTED CHRONOLOGICAL RANKING

Attack prevalence            : 0.10

In [28]:
# =============================================================================
# STAGE23-1L — SCIENTIFIC RESULT SEAL
# BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL
# FINAL PRIMARY BOOSTED-ABLATION SEAL
# =============================================================================
#
# ZERO MODEL FITS.
# ZERO DATASET ACCESS.
# ZERO CACHE MATERIALIZATION.
# ZERO METRIC RECOMPUTATION FROM DATA.
#
# AFTER SUCCESS:
#   PRIMARY BOOSTED ABLATION BLOCK = PERMANENTLY SEALED
#
# FIT ACCOUNTING:
#   before seal : 24 / 50
#   seal        :  0
#   after seal  : 24 / 50
#
# PRIMARY BOOSTED FITS:
#   24 / 24 COMPLETE
#
# REMAINING MODEL FITS:
#   placebo boosted fits : 20
#   depth-1 stump fits   :  6
#   total remaining      : 26
#
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import shutil
import json
import os


# =============================================================================
# 0. IDENTIFIERS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_DIR = Path(
    "/kaggle/working/"
    "stage23_1l_behavior_only_chronological_natural"
)

TARGET_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "behavior_only"
)

EXPECTED_PARENT = (
    "eae3a7c409687f086924cce84be65bca5e7b10ba"
)

PREDECESSOR_TAG = (
    "stage23-1k-behavior-only-random-natural-v1"
)

CACHE_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_TAG = (
    "stage23-execution-cache-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

RESULT_TAG = (
    "stage23-1l-behavior-only-chronological-natural-v1"
)

COMMIT_MESSAGE = (
    "Stage23-1L: freeze BEHAVIOR_ONLY chronological-natural result"
)

TAG_MESSAGE = (
    "Stage23-1L frozen BEHAVIOR_ONLY CHRONOLOGICAL_NATURAL result"
)


# =============================================================================
# 1. EXACT EXECUTION HASHES
# =============================================================================

EXPECTED_HASHES = {

    "stage23_1l_behavior_only_chronological_natural_result.json":
        "185b5b247d51cc776a4506e64765c88ef23f0e69deb3e3f9d479014427097459",

    "behavior_only_chronological_natural_lightgbm_model.txt":
        "65af6dcf849230508f4ffa738a93ad4c1cc6eed80c67083793085d6319d78d7c",

    "behavior_only_chronological_natural_xgboost_model.json":
        "2816b36504a717b87bf3bd2c761ef7d7df0f23988f8f377f6a63a0b45cfb5310",

    "behavior_only_chronological_natural_validation_probabilities.npz":
        "6b084ca96d8474dbaf6c20c52957ecea38ff2729b7facf158981e78840eb94b5",
}


EXPECTED_SOURCE_MANIFEST_SHA = (
    "371096c436476258f241c8605601372c3528d58591adf046386e0ab95a643e10"
)


# =============================================================================
# 2. EXACT SCIENTIFIC VALUES
# =============================================================================

EXPECTED_PR_AUC = 0.095539807865
EXPECTED_ROC_AUC = 0.476481778469

EXPECTED_CHRONO_PR_PENALTY = 0.010675347269
EXPECTED_CHRONO_ROC_PENALTY = 0.038436647924

EXPECTED_RANDOM_PR_PENALTY = 0.014002038042
EXPECTED_RANDOM_ROC_PENALTY = 0.002483324722

EXPECTED_PR_INTERACTION = 0.003326690773
EXPECTED_ROC_INTERACTION = -0.035953323203

EXPECTED_F1_INTERACTION = 0.066711960023
EXPECTED_RECALL_INTERACTION = 0.046671997632
EXPECTED_FPR_INTERACTION = 0.037948796978

EXPECTED_ACCURACY = 0.856387887770
EXPECTED_PRECISION = 0.050535030852
EXPECTED_RECALL = 0.020785145207
EXPECTED_F1 = 0.029455282147
EXPECTED_FPR = 0.045740173539
EXPECTED_FNR = 0.979214854793

EXPECTED_TN = 507212
EXPECTED_FP = 24312
EXPECTED_FN = 60962
EXPECTED_TP = 1294

EXPECTED_REMOVED = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True, show=False):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (p.stdout or "").strip()

    if show and output:
        print(output)

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


def assert_float(name, actual, expected, tolerance=1e-12):

    actual = float(actual)
    expected = float(expected)

    if abs(actual - expected) >= tolerance:

        raise RuntimeError(
            f"\nFrozen {name} differs from Stage23-1L output.\n"
            f"expected: {expected:+.12f}\n"
            f"actual  : {actual:+.12f}"
        )


# =============================================================================
# 4. REPOSITORY PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-1L — SCIENTIFIC RESULT SEAL")
print("FINAL PRIMARY BOOSTED-ABLATION SEAL")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected parent before Stage23-1L seal.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag verification failed."
    )


if predecessor_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1K predecessor tag verification failed."
    )


if cache_tag_commit != CACHE_COMMIT:

    raise RuntimeError(
        "Stage23 execution-cache tag verification failed."
    )


if status:

    raise RuntimeError(
        "Repository must be clean before Stage23-1L seal:\n"
        + status
    )


if TARGET_DIR.exists():

    raise RuntimeError(
        f"Stage23-1L repository target already exists:\n{TARGET_DIR}"
    )


if not SOURCE_DIR.exists():

    raise RuntimeError(
        f"Stage23-1L source directory missing:\n{SOURCE_DIR}"
    )


existing_tag = run(
    [
        "git",
        "tag",
        "--list",
        RESULT_TAG,
    ]
)


if existing_tag:

    raise RuntimeError(
        f"Stage23-1L result tag already exists: {RESULT_TAG}"
    )


print("[OK] branch             :", branch)
print("[OK] parent HEAD        :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1K tag     :", predecessor_tag_commit)
print("[OK] execution-cache tag:", cache_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] Stage23-1L tag     : ABSENT")


# =============================================================================
# 5. VERIFY SOURCE MANIFEST
# =============================================================================

SOURCE_MANIFEST = (
    SOURCE_DIR
    / "checksums.sha256"
)


if not SOURCE_MANIFEST.exists():

    raise RuntimeError(
        "Stage23-1L source checksum manifest missing."
    )


actual_manifest_sha = sha256_file(
    SOURCE_MANIFEST
)


if actual_manifest_sha != EXPECTED_SOURCE_MANIFEST_SHA:

    raise RuntimeError(
        "\nSTAGE23-1L SOURCE MANIFEST CHANGED\n"
        f"expected: {EXPECTED_SOURCE_MANIFEST_SHA}\n"
        f"actual  : {actual_manifest_sha}"
    )


print()
print("[OK] source checksum-manifest SHA256:")
print(" ", actual_manifest_sha)


# =============================================================================
# 6. VERIFY CORE ARTIFACTS
# =============================================================================

print()
print("=" * 100)
print("VERIFYING STAGE23-1L CORE ARTIFACTS")
print("=" * 100)
print()


for filename, expected_sha in EXPECTED_HASHES.items():

    path = (
        SOURCE_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-1L artifact: {filename}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            "\nSTAGE23-1L ARTIFACT MUTATION DETECTED\n"
            f"file     : {filename}\n"
            f"expected : {expected_sha}\n"
            f"actual   : {actual_sha}"
        )


    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 7. LOAD RESULT
# =============================================================================

RESULT_SOURCE = (
    SOURCE_DIR
    / "stage23_1l_behavior_only_chronological_natural_result.json"
)


result = json.loads(
    RESULT_SOURCE.read_text(
        encoding="utf-8"
    )
)


if result[
    "stage"
] != "Stage23-1L":

    raise RuntimeError(
        "Unexpected Stage23-1L identifier."
    )


if result[
    "cell"
][
    "subset"
] != "BEHAVIOR_ONLY":

    raise RuntimeError(
        "Stage23-1L subset mismatch."
    )


if result[
    "cell"
][
    "split"
] != "CHRONOLOGICAL_NATURAL":

    raise RuntimeError(
        "Stage23-1L split mismatch."
    )


if result[
    "cell"
][
    "removed_features"
] != EXPECTED_REMOVED:

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY exclusions changed."
    )


if result[
    "cell"
][
    "feature_count"
] != 63:

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY feature count changed."
    )


if result[
    "cell"
][
    "semantic_label"
] != "behavior_restricted_feature_set":

    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY semantic label changed."
    )


if result[
    "cell"
][
    "interpretation_term"
] != "behavior-restricted performance":

    raise RuntimeError(
        "Frozen interpretation terminology changed."
    )


# =============================================================================
# 8. PROVENANCE ASSERTIONS
# =============================================================================

if result[
    "protocol"
][
    "commit"
] != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Protocol commit mismatch."
    )


if result[
    "execution_parent"
][
    "commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1L execution parent mismatch."
    )


if result[
    "execution_parent"
][
    "tag"
] != PREDECESSOR_TAG:

    raise RuntimeError(
        "Stage23-1L predecessor tag mismatch."
    )


if result[
    "scientific_random_partner"
][
    "commit"
] != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-1L random partner commit mismatch."
    )


if result[
    "scientific_random_partner"
][
    "tag"
] != PREDECESSOR_TAG:

    raise RuntimeError(
        "Stage23-1L random partner tag mismatch."
    )


if result[
    "cache"
][
    "identifier"
] != "stage23_execution_cache_v1":

    raise RuntimeError(
        "Unexpected execution-cache identifier."
    )


if result[
    "cache"
][
    "seal_commit"
] != CACHE_COMMIT:

    raise RuntimeError(
        "Stage23-1L cache seal commit mismatch."
    )


if result[
    "cache"
][
    "seal_tag"
] != CACHE_TAG:

    raise RuntimeError(
        "Stage23-1L cache tag mismatch."
    )


if result[
    "cache"
][
    "sealed"
] is not True:

    raise RuntimeError(
        "Stage23-1L did not use a sealed cache."
    )


if result[
    "cache"
][
    "selected_physical_feature_digests_verified"
] != 63:

    raise RuntimeError(
        "Stage23-1L did not verify all 63 cache columns."
    )


if result[
    "cache"
][
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Stage23-1L unexpectedly read Parquet files."
    )


# =============================================================================
# 9. GOVERNANCE ASSERTIONS
# =============================================================================

governance = result[
    "governance"
]


if governance[
    "new_model_fits_this_cell"
] != 2:

    raise RuntimeError(
        "Unexpected Stage23-1L fit count."
    )


if governance[
    "stage23_total_model_fits_before_cell"
] != 22:

    raise RuntimeError(
        "Unexpected Stage23-1L pre-fit count."
    )


if governance[
    "stage23_total_model_fits_after_cell"
] != 24:

    raise RuntimeError(
        "Unexpected Stage23-1L post-fit count."
    )


if governance[
    "primary_boosted_model_fits_complete"
] is not True:

    raise RuntimeError(
        "Primary boosted completion flag missing."
    )


if governance[
    "primary_boosted_model_fits_total"
] != 24:

    raise RuntimeError(
        "Primary boosted fit total mismatch."
    )


if governance[
    "execution_cache_optimization"
] is not True:

    raise RuntimeError(
        "Execution-cache optimization flag missing."
    )


if governance[
    "execution_cache_scientifically_sealed"
] is not True:

    raise RuntimeError(
        "Execution-cache seal flag missing."
    )


if governance[
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Governance reports unexpected Parquet reads."
    )


for key in [
    "threshold_optimization",
    "subset_specific_tuning",
    "rebalancing",
    "shap_executed",
    "placebo_executed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
    "stage23_0_modified",
    "feature_definition_changed",
    "split_definition_changed",
    "input_dtype_changed",
    "model_specification_changed",
]:

    if governance[
        key
    ] is not False:

        raise RuntimeError(
            f"Governance violation: {key}"
        )


primary_status = result[
    "primary_boosted_ablation_status"
]


if primary_status[
    "fits_complete_after_this_cell"
] is not True:

    raise RuntimeError(
        "Primary boosted block not marked complete."
    )


if primary_status[
    "boosted_primary_fits_completed"
] != 24:

    raise RuntimeError(
        "Primary boosted completed-fit count mismatch."
    )


if primary_status[
    "boosted_primary_fits_expected"
] != 24:

    raise RuntimeError(
        "Primary boosted expected-fit count mismatch."
    )


if primary_status[
    "all_primary_subsets_have_both_splits"
] is not True:

    raise RuntimeError(
        "Primary subset/split completeness assertion failed."
    )


print()
print("[OK] governance assertions passed")
print("     Stage23 fits       : 24 / 50")
print("     Primary boosted    : 24 / 24 COMPLETE")
print("     Parquet reads      : 0")
print("     execution cache    : SEALED")


# =============================================================================
# 10. SCIENTIFIC ASSERTIONS
# =============================================================================

ranking = result[
    "ranking_metrics"
]

penalty = result[
    "removal_penalty_full_minus_ablated"
]

random_reference = result[
    "matched_random_reference"
]

interaction = result[
    "shortcut_interaction"
]

fixed = result[
    "fixed_threshold_0_50"
]


assert_float(
    "PR-AUC",
    ranking[
        "pr_auc"
    ],
    EXPECTED_PR_AUC,
)

assert_float(
    "ROC-AUC",
    ranking[
        "roc_auc"
    ],
    EXPECTED_ROC_AUC,
)

assert_float(
    "chronological PR penalty",
    penalty[
        "pr_auc"
    ],
    EXPECTED_CHRONO_PR_PENALTY,
)

assert_float(
    "chronological ROC penalty",
    penalty[
        "roc_auc"
    ],
    EXPECTED_CHRONO_ROC_PENALTY,
)

assert_float(
    "random PR penalty",
    random_reference[
        "pr_auc_removal_penalty"
    ],
    EXPECTED_RANDOM_PR_PENALTY,
)

assert_float(
    "random ROC penalty",
    random_reference[
        "roc_auc_removal_penalty"
    ],
    EXPECTED_RANDOM_ROC_PENALTY,
)


# =============================================================================
# 11. INTERACTION ASSERTIONS
# =============================================================================

if interaction[
    "point_estimate_available"
] is not True:

    raise RuntimeError(
        "BEHAVIOR_ONLY interaction point estimate missing."
    )


if interaction[
    "confidence_interval_status"
] != "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS":

    raise RuntimeError(
        "Unexpected interaction CI status."
    )


if interaction[
    "interpretation_status"
] != "POINT_ESTIMATE_ONLY_UNTIL_UNCERTAINTY_STAGE":

    raise RuntimeError(
        "Unexpected interaction interpretation status."
    )


assert_float(
    "PR interaction",
    interaction[
        "pr_auc"
    ],
    EXPECTED_PR_INTERACTION,
)

assert_float(
    "ROC interaction",
    interaction[
        "roc_auc"
    ],
    EXPECTED_ROC_INTERACTION,
)

assert_float(
    "F1 interaction",
    interaction[
        "f1_at_0_50"
    ],
    EXPECTED_F1_INTERACTION,
)

assert_float(
    "Recall interaction",
    interaction[
        "recall_at_0_50"
    ],
    EXPECTED_RECALL_INTERACTION,
)

assert_float(
    "FPR interaction",
    interaction[
        "fpr_at_0_50"
    ],
    EXPECTED_FPR_INTERACTION,
)


# =============================================================================
# 12. FIXED THRESHOLD ASSERTIONS
# =============================================================================

assert_float(
    "Accuracy",
    fixed[
        "accuracy"
    ],
    EXPECTED_ACCURACY,
)

assert_float(
    "Precision",
    fixed[
        "precision"
    ],
    EXPECTED_PRECISION,
)

assert_float(
    "Recall",
    fixed[
        "recall"
    ],
    EXPECTED_RECALL,
)

assert_float(
    "F1",
    fixed[
        "f1"
    ],
    EXPECTED_F1,
)

assert_float(
    "FPR",
    fixed[
        "fpr"
    ],
    EXPECTED_FPR,
)

assert_float(
    "FNR",
    fixed[
        "fnr"
    ],
    EXPECTED_FNR,
)


if (
    int(fixed["tn"]) != EXPECTED_TN
    or int(fixed["fp"]) != EXPECTED_FP
    or int(fixed["fn"]) != EXPECTED_FN
    or int(fixed["tp"]) != EXPECTED_TP
):

    raise RuntimeError(
        "Stage23-1L confusion matrix differs from execution output."
    )


print()
print("[OK] scientific result assertions passed")

print(
    f"     PR-AUC          : {float(ranking['pr_auc']):.12f}"
)

print(
    f"     ROC-AUC         : {float(ranking['roc_auc']):.12f}"
)

print(
    f"     Chrono PR pen.  : {float(penalty['pr_auc']):+.12f}"
)

print(
    f"     Chrono ROC pen. : {float(penalty['roc_auc']):+.12f}"
)

print(
    f"     PR interaction  : {float(interaction['pr_auc']):+.12f}"
)

print(
    f"     ROC interaction : {float(interaction['roc_auc']):+.12f}"
)

print(
    "     CI status       : PENDING"
)


# =============================================================================
# 13. COPY CORE ARTIFACTS
# =============================================================================

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


FILES_TO_COPY = list(
    EXPECTED_HASHES.keys()
) + [
    "checksums.sha256",
]


for filename in FILES_TO_COPY:

    shutil.copy2(
        SOURCE_DIR
        / filename,
        TARGET_DIR
        / filename,
    )


# =============================================================================
# 14. VERIFY COPIES
# =============================================================================

print()
print("=" * 100)
print("VERIFYING REPOSITORY COPIES")
print("=" * 100)
print()


for filename in FILES_TO_COPY:

    src_sha = sha256_file(
        SOURCE_DIR
        / filename
    )

    dst_sha = sha256_file(
        TARGET_DIR
        / filename
    )

    if src_sha != dst_sha:

        raise RuntimeError(
            f"Source/destination mismatch: {filename}"
        )

    print(
        f"[EXACT] {filename}"
    )


# =============================================================================
# 15. CREATE SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT_PATH = (
    TARGET_DIR
    / "seal_receipt.json"
)


seal_receipt = {

    "stage":
        "Stage23-1L",

    "status":
        "SCIENTIFIC_RESULT_FROZEN",

    "sealed_utc":
        now_utc(),

    "cell": {

        "subset":
            "BEHAVIOR_ONLY",

        "split":
            "CHRONOLOGICAL_NATURAL",

        "removed_features":
            EXPECTED_REMOVED,

        "feature_count":
            63,

        "semantic_label":
            "behavior_restricted_feature_set",

        "interpretation_term":
            "behavior-restricted performance",
    },

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,
    },

    "required_predecessor": {

        "commit":
            EXPECTED_PARENT,

        "tag":
            PREDECESSOR_TAG,
    },

    "execution_cache": {

        "commit":
            CACHE_COMMIT,

        "tag":
            CACHE_TAG,

        "scientifically_sealed":
            True,

        "parquet_files_read":
            0,
    },

    "result_tag":
        RESULT_TAG,

    "source_checksum_manifest_sha256":
        EXPECTED_SOURCE_MANIFEST_SHA,

    "core_artifact_sha256":
        EXPECTED_HASHES,

    "models_fit_this_cell":
        2,

    "stage23_models_fit_total":
        24,

    "primary_boosted_ablation": {

        "status":
            "COMPLETE_AND_FROZEN_AFTER_THIS_SEAL",

        "primary_boosted_fits":
            24,

        "primary_boosted_fits_expected":
            24,

        "frozen_primary_subsets":
            7,

        "splits_per_subset":
            2,

        "all_primary_subsets_have_both_splits":
            True,
    },

    "ranking": {

        "pr_auc":
            float(
                ranking[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                ranking[
                    "roc_auc"
                ]
            ),

        "pr_auc_removal_penalty":
            float(
                penalty[
                    "pr_auc"
                ]
            ),

        "roc_auc_removal_penalty":
            float(
                penalty[
                    "roc_auc"
                ]
            ),
    },

    "shortcut_interaction": {

        "pr_auc":
            float(
                interaction[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                interaction[
                    "roc_auc"
                ]
            ),

        "f1_at_0_50":
            float(
                interaction[
                    "f1_at_0_50"
                ]
            ),

        "recall_at_0_50":
            float(
                interaction[
                    "recall_at_0_50"
                ]
            ),

        "fpr_at_0_50":
            float(
                interaction[
                    "fpr_at_0_50"
                ]
            ),

        "confidence_interval_status":
            "PENDING_FROZEN_STAGE23_UNCERTAINTY_ANALYSIS",

        "interpretation_status":
            "POINT_ESTIMATE_ONLY",
    },

    "remaining_stage23_model_fits": {

        "placebo_boosted":
            20,

        "depth_1_stumps":
            6,

        "total":
            26,
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "post_result_adaptation":
        False,

    "next_stage":
        "Stage23 placebo ablation block",
}


SEAL_RECEIPT_PATH.write_text(
    json.dumps(
        seal_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 16. README
# =============================================================================

README_PATH = (
    TARGET_DIR
    / "README.md"
)


README_PATH.write_text(
f"""# Stage23-1L — BEHAVIOR_ONLY × CHRONOLOGICAL_NATURAL

This directory freezes the chronological partner of the prospectively
defined behavior-restricted feature set and closes the Stage23 primary
boosted-ablation block.

## Frozen feature restriction

Excluded exactly:

- `Dst Port`
- `Protocol`
- `Fwd Header Len`
- `Bwd Header Len`
- `Init Fwd Win Byts`
- `Init Bwd Win Byts`
- `Fwd Seg Size Min`

Retained features: 63

Semantic label:

`behavior_restricted_feature_set`

Correct interpretation term:

**behavior-restricted performance**

This must not be described as “true performance” or as a deployment
performance estimate.

## Chronological execution

- Train rows: 13,818,623
- Validation rows: 593,780
- New boosted-model fits: 2
- Stage23 fits after execution: 24 / 50
- Execution source: sealed deterministic execution cache v1
- Parquet reads: 0
- Cache materialization: {float(result["cache"]["materialization_seconds"]):.3f} s

## Behavior-restricted chronological ranking

- Attack prevalence: {float(result["data"]["validation"]["attack_prevalence"]):.12f}
- PR-AUC: {float(ranking["pr_auc"]):.12f}
- FULL PR-AUC: {float(result["full_reference"]["pr_auc"]):.12f}
- FULL − BEHAVIOR_ONLY PR penalty: {float(penalty["pr_auc"]):+.12f}

- ROC-AUC: {float(ranking["roc_auc"]):.12f}
- FULL ROC-AUC: {float(result["full_reference"]["roc_auc"]):.12f}
- FULL − BEHAVIOR_ONLY ROC penalty: {float(penalty["roc_auc"]):+.12f}

The chronological BEHAVIOR_ONLY PR-AUC is below attack prevalence and
ROC-AUC is below 0.5 on this development split. This is a descriptive
observation only and does not by itself establish causal shortcut
dependence.

## Fixed threshold 0.50

- Accuracy: {float(fixed["accuracy"]):.12f}
- Precision: {float(fixed["precision"]):.12f}
- Recall: {float(fixed["recall"]):.12f}
- F1: {float(fixed["f1"]):.12f}
- FPR: {float(fixed["fpr"]):.12f}
- FNR: {float(fixed["fnr"]):.12f}

Confusion matrix:

- TN: {int(fixed["tn"]):,}
- FP: {int(fixed["fp"]):,}
- FN: {int(fixed["fn"]):,}
- TP: {int(fixed["tp"]):,}

## Frozen BEHAVIOR_ONLY interaction point estimates

Definition:

`I(S) = Δ_RANDOM(S) - Δ_CHRONOLOGICAL(S)`

- Random PR penalty: {EXPECTED_RANDOM_PR_PENALTY:+.12f}
- Chronological PR penalty: {EXPECTED_CHRONO_PR_PENALTY:+.12f}
- PR-AUC interaction: {EXPECTED_PR_INTERACTION:+.12f}

- Random ROC penalty: {EXPECTED_RANDOM_ROC_PENALTY:+.12f}
- Chronological ROC penalty: {EXPECTED_CHRONO_ROC_PENALTY:+.12f}
- ROC-AUC interaction: {EXPECTED_ROC_INTERACTION:+.12f}

Supplementary:

- F1@0.50 interaction: {EXPECTED_F1_INTERACTION:+.12f}
- Recall@0.50 interaction: {EXPECTED_RECALL_INTERACTION:+.12f}
- FPR@0.50 interaction: {EXPECTED_FPR_INTERACTION:+.12f}

The PR-AUC and ROC-AUC interaction point estimates have opposite signs.
Interpretation therefore remains metric-specific.

All interaction values remain **point estimates only** until the frozen
paired-bootstrap uncertainty analysis is executed.

## Primary boosted-ablation block

After this seal:

- Frozen primary subsets: 7
- Evaluation splits per subset: 2
- Primary boosted fits complete: 24 / 24
- Stage23 total model fits complete: 24 / 50

The Stage23 primary boosted-ablation block is now complete.

## Remaining Stage23 model fits

- Placebo boosted fits: 20
- Depth-1 stump fits: 6
- Remaining total: 26

## Governance

- Threshold optimization: no
- Subset-specific tuning: no
- Rebalancing: no
- SHAP: no
- Placebo execution in Stage23-1L: no
- Raw March 1 access: no
- Raw March 2 access: no
- Stage23-0 modified: no
- Execution-cache definition modified: no

## Next block

`Stage23 placebo ablation block`
""",
    encoding="utf-8",
)


# =============================================================================
# 17. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPOSITORY_CHECKSUMS_PATH = (
    TARGET_DIR
    / "repository_checksums.sha256"
)


repo_files = sorted(
    path
    for path in TARGET_DIR.iterdir()
    if path.is_file()
    and path.name != "repository_checksums.sha256"
)


REPOSITORY_CHECKSUMS_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in repo_files
    ) + "\n",
    encoding="utf-8",
)


REPOSITORY_MANIFEST_SHA = sha256_file(
    REPOSITORY_CHECKSUMS_PATH
)


# =============================================================================
# 18. STAGE ONLY STAGE23-1L
# =============================================================================

relative_target = (
    TARGET_DIR.relative_to(
        REPO_DIR
    )
)


run(
    [
        "git",
        "add",
        "--",
        str(
            relative_target
        ),
    ]
)


staged = run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ]
).splitlines()


expected_prefix = (
    str(
        relative_target
    )
    + "/"
)


unexpected = [
    path
    for path in staged
    if not path.startswith(
        expected_prefix
    )
]


if unexpected:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            unexpected
        )
    )


print()
print("=" * 100)
print("STAGED STAGE23-1L ARTIFACTS")
print("=" * 100)

for path in staged:
    print(" ", path)


print()

run(
    [
        "git",
        "diff",
        "--cached",
        "--stat",
    ],
    show=True,
)


# =============================================================================
# 19. COMMIT
# =============================================================================

print()
print("=" * 100)
print("CREATING STAGE23-1L RESULT COMMIT")
print("=" * 100)
print()


run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ],
    show=True,
)


sealed_commit = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)


if sealed_commit == EXPECTED_PARENT:

    raise RuntimeError(
        "Git HEAD did not advance."
    )


if run(
    [
        "git",
        "status",
        "--porcelain",
    ]
):

    raise RuntimeError(
        "Repository dirty after Stage23-1L commit."
    )


print()
print("Stage23-1L commit:")
print(
    " ",
    sealed_commit
)


# =============================================================================
# 20. ANNOTATED RESULT TAG
# =============================================================================

run(
    [
        "git",
        "tag",
        "-a",
        RESULT_TAG,
        sealed_commit,
        "-m",
        TAG_MESSAGE,
    ]
)


local_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        RESULT_TAG,
    ]
)


if local_tag_commit != sealed_commit:

    raise RuntimeError(
        "Local Stage23-1L tag verification failed."
    )


# =============================================================================
# 21. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 100)
print("PUSHING STAGE23-1L")
print("=" * 100)
print()


run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    show=True,
)


run(
    [
        "git",
        "push",
        "origin",
        RESULT_TAG,
    ],
    show=True,
)


# =============================================================================
# 22. VERIFY REMOTE MAIN
# =============================================================================

remote_main_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ]
)


if not remote_main_output:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_main = (
    remote_main_output.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "\nRemote main mismatch.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {remote_main}"
    )


# =============================================================================
# 23. VERIFY REMOTE ANNOTATED TAG
# =============================================================================

remote_tag_output = run(
    [
        "git",
        "ls-remote",
        "origin",
        f"refs/tags/{RESULT_TAG}",
        f"refs/tags/{RESULT_TAG}^{{}}",
    ]
)


tag_object = None
peeled_commit = None


for line in remote_tag_output.splitlines():

    parts = line.split()

    if len(parts) != 2:
        continue

    sha, ref = parts

    if ref == f"refs/tags/{RESULT_TAG}":

        tag_object = sha

    elif ref == f"refs/tags/{RESULT_TAG}^{{}}":

        peeled_commit = sha


if not tag_object:

    raise RuntimeError(
        "Remote Stage23-1L annotated tag object missing."
    )


if peeled_commit != sealed_commit:

    raise RuntimeError(
        "\nRemote Stage23-1L tag verification failed.\n"
        f"expected: {sealed_commit}\n"
        f"actual  : {peeled_commit}"
    )


# =============================================================================
# 24. FINAL CLEANLINESS
# =============================================================================

final_status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-1L push."
    )


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-1L SCIENTIFIC RESULT SEALED")
print("=" * 100)

print()
print("Stage23-1K parent:")
print(
    " ",
    EXPECTED_PARENT
)

print()
print("Stage23-1L commit:")
print(
    " ",
    sealed_commit
)

print()
print("Result tag:")
print(
    " ",
    RESULT_TAG
)

print()
print("Remote main:")
print(
    " ",
    remote_main
)

print()
print("Remote tag object:")
print(
    " ",
    tag_object
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    peeled_commit
)

print()
print("Source execution checksum manifest:")
print(
    " ",
    EXPECTED_SOURCE_MANIFEST_SHA
)

print()
print("Repository checksum manifest:")
print(
    " ",
    REPOSITORY_MANIFEST_SHA
)


print()
print("=" * 100)
print("FROZEN BEHAVIOR-RESTRICTED PAIR")
print("=" * 100)

print()
print("RANDOM_NATURAL:")
print(
    f"  PR penalty  : {EXPECTED_RANDOM_PR_PENALTY:+.12f}"
)
print(
    f"  ROC penalty : {EXPECTED_RANDOM_ROC_PENALTY:+.12f}"
)

print()
print("CHRONOLOGICAL_NATURAL:")
print(
    f"  PR-AUC      : {EXPECTED_PR_AUC:.12f}"
)
print(
    f"  ROC-AUC     : {EXPECTED_ROC_AUC:.12f}"
)
print(
    f"  PR penalty  : {EXPECTED_CHRONO_PR_PENALTY:+.12f}"
)
print(
    f"  ROC penalty : {EXPECTED_CHRONO_ROC_PENALTY:+.12f}"
)

print()
print("SPLIT × ABLATION INTERACTION:")
print(
    f"  PR-AUC  : {EXPECTED_PR_INTERACTION:+.12f}"
)
print(
    f"  ROC-AUC : {EXPECTED_ROC_INTERACTION:+.12f}"
)

print()
print("Supplementary interactions:")
print(
    f"  F1@0.50     : {EXPECTED_F1_INTERACTION:+.12f}"
)
print(
    f"  Recall@0.50 : {EXPECTED_RECALL_INTERACTION:+.12f}"
)
print(
    f"  FPR@0.50    : {EXPECTED_FPR_INTERACTION:+.12f}"
)

print()
print("Uncertainty:")
print("  95% CI : PENDING")
print("  status : POINT ESTIMATE ONLY")


print()
print("=" * 100)
print("PRIMARY BOOSTED-ABLATION BLOCK")
print("=" * 100)

print()
print("Frozen primary subsets       : 7")
print("Evaluation splits per subset : 2")
print("Primary boosted fits sealed  : 24 / 24")
print("Stage23 total fits sealed    : 24 / 50")
print()
print("PRIMARY BOOSTED ABLATION BLOCK: COMPLETE")


print()
print("=" * 100)
print("REMAINING STAGE23 FITS")
print("=" * 100)

print()
print("Placebo boosted fits : 20")
print("Depth-1 stump fits   :  6")
print("---------------------------")
print("Total remaining      : 26")


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Model fits seal          : 0")
print("Threshold optimization   : NO")
print("Subset-specific tuning   : NO")
print("Rebalancing              : NO")
print("Raw Mar1 read            : NO")
print("Raw Mar2 read            : NO")
print("Stage23-0 changed        : NO")
print("Execution cache changed  : NO")
print("Git status               : CLEAN")


print()
print("=" * 100)
print("STAGE23-1L COMPLETE AND SEALED")
print("=" * 100)

print()
print("PRIMARY BOOSTED ABLATION BLOCK:")
print(
    "  COMPLETE — 24 / 24 primary boosted fits sealed"
)

print()
print("STAGE23 TOTAL:")
print(
    "  24 / 50 fits sealed"
)

print()
print("NEXT BLOCK:")
print(
    "  Stage23 placebo ablations"
)

print("=" * 100)

STAGE23-1L — SCIENTIFIC RESULT SEAL
FINAL PRIMARY BOOSTED-ABLATION SEAL

[OK] branch             : main
[OK] parent HEAD        : eae3a7c409687f086924cce84be65bca5e7b10ba
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1K tag     : eae3a7c409687f086924cce84be65bca5e7b10ba
[OK] execution-cache tag: 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] Stage23-1L tag     : ABSENT

[OK] source checksum-manifest SHA256:
  371096c436476258f241c8605601372c3528d58591adf046386e0ab95a643e10

VERIFYING STAGE23-1L CORE ARTIFACTS

[EXACT] stage23_1l_behavior_only_chronological_natural_result.json
[EXACT] behavior_only_chronological_natural_lightgbm_model.txt
[EXACT] behavior_only_chronological_natural_xgboost_model.json
[EXACT] behavior_only_chronological_natural_validation_probabilities.npz

[OK] governance assertions passed
     Stage23 fits       : 24 / 50
     Primary boosted    : 24 / 24 COMPLETE
     Parquet reads      : 0
     executio

In [29]:
# =============================================================================
# STAGE23-2A
# PLACEBO_COUNTS × RANDOM_NATURAL
# FIRST FROZEN PLACEBO BOOSTED EXECUTION
# OPTIMIZED — SEALED EXECUTION CACHE V1
# =============================================================================
#
# FROZEN PLACEBO
# --------------
# PLACEBO_COUNTS
#
# Remove EXACTLY:
#   - Flow Duration
#   - Tot Fwd Pkts
#   - Tot Bwd Pkts
#
# Retained features: 67
#
# This is one of five prospectively frozen ordinary-behavioral
# three-feature placebo ablations.
#
# REQUIRED PARENT:
#   Stage23-1L
#   359b6a2c1b548e6b9fc84d2712f0fb13714b1c24
#
# SEALED CACHE:
#   stage23-execution-cache-v1
#   66e1c6382caa63699766e605df63df02beff894f
#
# FIT ACCOUNTING:
#   before : 24 / 50
#   this   :  2
#   after  : 26 / 50
#
# PLACEBO BOOSTED FITS:
#   before : 0 / 20
#   this   : 2
#   after  : 2 / 20
#
# NO:
#   Parquet reads
#   raw Mar1
#   raw Mar2
#   threshold optimization
#   subset-specific tuning
#   rebalancing
#   SHAP
#   stumps
#
# INTERACTION:
#   PENDING matched PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL
#
# FAIL CLOSED:
# If either model completes and anything later fails, DO NOT blindly rerun.
# Inspect execution_state.json first.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. FROZEN IDENTIFIERS
# =============================================================================

STAGE = "Stage23-2A"

TARGET_PLACEBO = "PLACEBO_COUNTS"
TARGET_SPLIT = "RANDOM_NATURAL"

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

PREDECESSOR_COMMIT = (
    "359b6a2c1b548e6b9fc84d2712f0fb13714b1c24"
)

PREDECESSOR_TAG = (
    "stage23-1l-behavior-only-chronological-natural-v1"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_DATA_MANIFEST_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

CACHE_SEMANTIC_70F_SHA256 = (
    "8c43ab0e36a65de1c095acb1ebf9e7d029a9e2b1cc353455a66617db8a899bb2"
)

EXPECTED_RANDOM_BITSET_SHA256 = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =============================================================================
# 1. PATHS
# =============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE_DIR = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

CACHE_SEAL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

PROTOCOL_DIR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

PREDECESSOR_SEAL = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
    / "chronological"
    / "behavior_only"
    / "seal_receipt.json"
)

FULL_RANDOM_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "stage23_2a_placebo_counts_random_natural"
)

RUNTIME_DIR = (
    OUTPUT_DIR
    / "runtime"
)

EXECUTION_STATE_PATH = (
    OUTPUT_DIR
    / "execution_state.json"
)

RESULT_PATH = (
    OUTPUT_DIR
    / "stage23_2a_placebo_counts_random_natural_result.json"
)


# =============================================================================
# 2. FROZEN COUNTS
# =============================================================================

N_DEVELOPMENT = 14_412_403

TRAIN_ROWS = 11_529_922
VALIDATION_ROWS = 2_882_481

TRAIN_BENIGN = 9_952_083
TRAIN_ATTACK = 1_577_839

VALIDATION_BENIGN = 2_488_021
VALIDATION_ATTACK = 394_460

N_FEATURES = 67

STAGE23_FITS_BEFORE = 24
STAGE23_FITS_AFTER = 26

PLACEBO_FITS_BEFORE = 0
PLACEBO_FITS_AFTER = 2

EXPECTED_REMOVED = [
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def run(cmd, cwd=REPO_DIR, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    output = (p.stdout or "").strip()

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + ("\n" + output if output else "")
        )

    return output


def sha256_file(path, chunk_size=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def write_json_atomic(path, payload):

    path = Path(path)

    temp = path.with_suffix(
        path.suffix + ".tmp"
    )

    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    temp.replace(path)


def now_utc():

    return datetime.now(
        timezone.utc
    ).isoformat()


FORBIDDEN_MARKERS = (
    "03-01-2018",
    "03-02-2018",
    "01-03-2018",
    "02-03-2018",
    "03012018",
    "03022018",
    "01032018",
    "02032018",
    "mar1",
    "mar2",
    "march1",
    "march2",
)


def guard_path(path):

    resolved = Path(path).resolve()

    lowered = str(
        resolved
    ).lower()

    for marker in FORBIDDEN_MARKERS:

        if marker in lowered:

            raise RuntimeError(
                "\nPERMANENT STAGE23 HOLDOUT ACCESS BLOCKED\n"
                f"path   : {resolved}\n"
                f"marker : {marker}"
            )

    return resolved


# =============================================================================
# 4. FAIL-CLOSED OUTPUT POLICY
# =============================================================================

if OUTPUT_DIR.exists():

    existing_state = None

    if EXECUTION_STATE_PATH.exists():

        try:

            existing_state = json.loads(
                EXECUTION_STATE_PATH.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:

            existing_state = {
                "status":
                    "UNREADABLE_EXECUTION_STATE"
            }

    raise RuntimeError(
        "\nSTAGE23-2A OUTPUT DIRECTORY ALREADY EXISTS.\n\n"
        f"{OUTPUT_DIR}\n\n"
        "Do NOT delete or rerun blindly.\n\n"
        "Existing execution state:\n"
        + json.dumps(
            existing_state,
            indent=2,
        )
    )


# =============================================================================
# 5. REPOSITORY / GOVERNANCE PRECONDITIONS
# =============================================================================

print("=" * 100)
print("STAGE23-2A — PLACEBO_COUNTS × RANDOM_NATURAL")
print("FIRST FROZEN PLACEBO BOOSTED EXECUTION")
print("=" * 100)
print()


branch = run(
    ["git", "branch", "--show-current"]
)

head = run(
    ["git", "rev-parse", "HEAD"]
)

status = run(
    ["git", "status", "--porcelain"]
)

protocol_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    ]
)

predecessor_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        PREDECESSOR_TAG,
    ]
)

cache_tag_commit = run(
    [
        "git",
        "rev-list",
        "-n",
        "1",
        CACHE_SEAL_TAG,
    ]
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "\nSTAGE23-2A MUST START FROM SEALED STAGE23-1L.\n"
        f"expected: {PREDECESSOR_COMMIT}\n"
        f"actual  : {head}"
    )


if protocol_tag_commit != PROTOCOL_COMMIT:

    raise RuntimeError(
        "Stage23-0 protocol tag mismatch."
    )


if predecessor_tag_commit != PREDECESSOR_COMMIT:

    raise RuntimeError(
        "Stage23-1L result tag mismatch."
    )


if cache_tag_commit != CACHE_SEAL_COMMIT:

    raise RuntimeError(
        "Stage23 execution-cache tag mismatch."
    )


if status:

    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )


if not PREDECESSOR_SEAL.exists():

    raise RuntimeError(
        "Stage23-1L seal receipt missing."
    )


predecessor_seal = json.loads(
    PREDECESSOR_SEAL.read_text(
        encoding="utf-8"
    )
)


if predecessor_seal[
    "stage23_models_fit_total"
] != 24:

    raise RuntimeError(
        "Expected exactly 24 sealed Stage23 fits."
    )


if predecessor_seal[
    "primary_boosted_ablation"
][
    "status"
] != "COMPLETE_AND_FROZEN_AFTER_THIS_SEAL":

    raise RuntimeError(
        "Primary boosted block is not frozen."
    )


if predecessor_seal[
    "next_stage"
] != "Stage23 placebo ablation block":

    raise RuntimeError(
        "Stage23-1L does not authorize placebo block."
    )


print("[OK] branch             :", branch)
print("[OK] HEAD               :", head)
print("[OK] Stage23-0 tag      :", protocol_tag_commit)
print("[OK] Stage23-1L tag     :", predecessor_tag_commit)
print("[OK] cache seal tag     :", cache_tag_commit)
print("[OK] worktree           : CLEAN")
print("[OK] Stage23 fits       : 24 / 50")
print("[OK] Primary boosted    : 24 / 24 FROZEN")
print("[OK] placebo block      : AUTHORIZED")


# =============================================================================
# 6. VERIFY STAGE23-0 ARTIFACTS
# =============================================================================

PROTOCOL_CHECKSUMS = (
    PROTOCOL_DIR
    / "checksums.sha256"
)


if sha256_file(
    PROTOCOL_CHECKSUMS
) != PROTOCOL_MANIFEST_SHA256:

    raise RuntimeError(
        "Stage23-0 checksum manifest changed."
    )


protocol_entries = {}


for line in PROTOCOL_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        "  ",
        1,
    )

    protocol_entries[
        filename
    ] = digest


for filename, expected_sha in protocol_entries.items():

    path = (
        PROTOCOL_DIR
        / filename
    )

    if not path.exists():

        raise RuntimeError(
            f"Missing Stage23-0 artifact: {filename}"
        )


    if sha256_file(
        path
    ) != expected_sha:

        raise RuntimeError(
            f"Stage23-0 artifact mutated: {filename}"
        )


print(
    f"[OK] Stage23-0 artifacts : "
    f"{len(protocol_entries)}/{len(protocol_entries)} exact"
)


# =============================================================================
# 7. LOAD FROZEN PLACEBO / MODEL / SPLIT / FEATURE SPECS
# =============================================================================

PLACEBO_SPEC_PATH = (
    PROTOCOL_DIR
    / "placebo_ablation_spec.json"
)

FEATURE_SPEC_PATH = (
    PROTOCOL_DIR
    / "feature_subset_spec.json"
)

MODEL_SPEC_PATH = (
    PROTOCOL_DIR
    / "model_spec.json"
)

SPLIT_SPEC_PATH = (
    PROTOCOL_DIR
    / "inherited_splits.json"
)


placebo_spec = json.loads(
    PLACEBO_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

feature_spec = json.loads(
    FEATURE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

model_spec = json.loads(
    MODEL_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

split_spec = json.loads(
    SPLIT_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


if placebo_spec[
    "matched_k"
] != 3:

    raise RuntimeError(
        "Frozen placebo matched_k changed."
    )


if placebo_spec[
    "number_of_placebos"
] != 5:

    raise RuntimeError(
        "Frozen placebo count changed."
    )


EXPECTED_SELECTION_METHOD = (
    "Five prospectively chosen ordinary behavioral triplets spanning "
    "count, volume/direction, timing, packet-size, and active/idle families."
)


if placebo_spec[
    "selection_method"
] != EXPECTED_SELECTION_METHOD:

    raise RuntimeError(
        "Frozen placebo selection method changed."
    )


if set(
    placebo_spec[
        "subsets"
    ].keys()
) != {
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
}:

    raise RuntimeError(
        "Frozen placebo subset set changed."
    )


target_placebo_spec = (
    placebo_spec[
        "subsets"
    ][
        TARGET_PLACEBO
    ]
)


REMOVED_FEATURES = list(
    target_placebo_spec[
        "removed"
    ]
)

FEATURES = list(
    target_placebo_spec[
        "retained"
    ]
)


if REMOVED_FEATURES != EXPECTED_REMOVED:

    raise RuntimeError(
        "\nPLACEBO_COUNTS removal definition changed.\n"
        f"expected: {EXPECTED_REMOVED}\n"
        f"actual  : {REMOVED_FEATURES}"
    )


if target_placebo_spec[
    "feature_count"
] != 67:

    raise RuntimeError(
        "Frozen PLACEBO_COUNTS feature_count changed."
    )


if len(
    FEATURES
) != 67:

    raise RuntimeError(
        f"Expected 67 retained placebo features; found {len(FEATURES)}"
    )


if len(
    set(
        FEATURES
    )
) != 67:

    raise RuntimeError(
        "PLACEBO_COUNTS retained features contain duplicates."
    )


for feature in EXPECTED_REMOVED:

    if feature in FEATURES:

        raise RuntimeError(
            f"Removed placebo feature remains present: {feature}"
        )


FULL_FEATURES = list(
    feature_spec[
        "full_feature_order"
    ]
)


expected_retained = [
    feature
    for feature in FULL_FEATURES
    if feature not in EXPECTED_REMOVED
]


if FEATURES != expected_retained:

    raise RuntimeError(
        "PLACEBO_COUNTS retained feature order is not "
        "FULL minus frozen three-feature removal."
    )


print()
print("Frozen placebo:")
print(
    "  placebo       :",
    TARGET_PLACEBO
)
print(
    "  split         :",
    TARGET_SPLIT
)
print(
    "  removed       :",
    REMOVED_FEATURES
)
print(
    "  retained      :",
    len(FEATURES)
)
print(
    "  matched k     :",
    placebo_spec[
        "matched_k"
    ]
)


# =============================================================================
# 8. VERIFY SEALED EXECUTION CACHE
# =============================================================================

CACHE_METADATA_REPO = (
    CACHE_SEAL_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_REPO = (
    CACHE_SEAL_DIR
    / "data_checksums.sha256"
)

CACHE_SEAL_RECEIPT = (
    CACHE_SEAL_DIR
    / "seal_receipt.json"
)


for path in [
    CACHE_METADATA_REPO,
    CACHE_DATA_MANIFEST_REPO,
    CACHE_SEAL_RECEIPT,
]:

    if not path.exists():

        raise RuntimeError(
            f"Missing sealed cache artifact: {path}"
        )


if sha256_file(
    CACHE_METADATA_REPO
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Sealed cache metadata SHA mismatch."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_REPO
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Sealed cache data-manifest SHA mismatch."
    )


cache_seal = json.loads(
    CACHE_SEAL_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if cache_seal[
    "status"
] != "EXECUTION_CACHE_VERIFICATION_FROZEN":

    raise RuntimeError(
        "Execution cache is not scientifically sealed."
    )


if cache_seal[
    "semantic_equivalence"
][
    "cache_reconstructed_row_major_70f_sha256"
] != CACHE_SEMANTIC_70F_SHA256:

    raise RuntimeError(
        "Execution-cache semantic SHA mismatch."
    )


# =============================================================================
# 9. VERIFY RUNTIME CACHE SMALL ARTIFACTS
# =============================================================================

CACHE_METADATA_RUNTIME = guard_path(
    CACHE_DIR
    / "metadata.json"
)

CACHE_DATA_MANIFEST_RUNTIME = guard_path(
    CACHE_DIR
    / "data_checksums.sha256"
)


if not CACHE_METADATA_RUNTIME.exists():

    raise RuntimeError(
        "Runtime execution-cache metadata missing."
    )


if not CACHE_DATA_MANIFEST_RUNTIME.exists():

    raise RuntimeError(
        "Runtime execution-cache manifest missing."
    )


if sha256_file(
    CACHE_METADATA_RUNTIME
) != CACHE_METADATA_SHA256:

    raise RuntimeError(
        "Runtime cache metadata differs from sealed metadata."
    )


if sha256_file(
    CACHE_DATA_MANIFEST_RUNTIME
) != CACHE_DATA_MANIFEST_SHA256:

    raise RuntimeError(
        "Runtime cache manifest differs from sealed manifest."
    )


cache_metadata = json.loads(
    CACHE_METADATA_RUNTIME.read_text(
        encoding="utf-8"
    )
)


if cache_metadata[
    "semantic_equivalence"
][
    "exact_byte_equality"
] is not True:

    raise RuntimeError(
        "Execution-cache exact equivalence flag is false."
    )


CACHE_FULL_FEATURES = list(
    cache_metadata[
        "full_feature_order"
    ]
)


if CACHE_FULL_FEATURES != FULL_FEATURES:

    raise RuntimeError(
        "Execution-cache feature order differs from Stage23 protocol."
    )


print("[OK] sealed execution cache metadata exact")


# =============================================================================
# 10. DERIVE EXACT PLACEBO CACHE PROJECTION
# =============================================================================

FEATURE_TO_INDEX = {
    feature: index
    for index, feature in enumerate(
        CACHE_FULL_FEATURES
    )
}


CACHE_INDICES = [
    FEATURE_TO_INDEX[
        feature
    ]
    for feature in FEATURES
]


if len(
    CACHE_INDICES
) != 67:

    raise RuntimeError(
        "PLACEBO_COUNTS cache projection count mismatch."
    )


if [
    CACHE_FULL_FEATURES[
        index
    ]
    for index in CACHE_INDICES
] != FEATURES:

    raise RuntimeError(
        "PLACEBO_COUNTS cache projection order mismatch."
    )


# =============================================================================
# 11. VERIFY FROZEN RANDOM SPLIT
# =============================================================================

random_spec = (
    split_spec[
        TARGET_SPLIT
    ]
)


if int(
    random_spec[
        "train"
    ][
        "rows"
    ]
) != TRAIN_ROWS:

    raise RuntimeError(
        "Random train row count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "rows"
    ]
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation row count changed."
    )


if int(
    random_spec[
        "train"
    ][
        "benign"
    ]
) != TRAIN_BENIGN:

    raise RuntimeError(
        "Random train benign count changed."
    )


if int(
    random_spec[
        "train"
    ][
        "attack"
    ]
) != TRAIN_ATTACK:

    raise RuntimeError(
        "Random train attack count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "benign"
    ]
) != VALIDATION_BENIGN:

    raise RuntimeError(
        "Random validation benign count changed."
    )


if int(
    random_spec[
        "validation"
    ][
        "attack"
    ]
) != VALIDATION_ATTACK:

    raise RuntimeError(
        "Random validation attack count changed."
    )


# =============================================================================
# 12. PARSE SEALED CACHE MANIFEST
# =============================================================================

manifest_hashes = {}


for line in CACHE_DATA_MANIFEST_RUNTIME.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, relative_path = line.split(
        "  ",
        1,
    )

    manifest_hashes[
        relative_path
    ] = digest


if len(
    manifest_hashes
) != 75:

    raise RuntimeError(
        f"Expected 75 execution-cache manifest entries; "
        f"found {len(manifest_hashes)}"
    )


# =============================================================================
# 13. RANDOM MEMBERSHIP BITSET
# =============================================================================

RANDOM_BITSET_PATH = guard_path(
    CACHE_DIR
    / "random_validation.packbits"
)


if not RANDOM_BITSET_PATH.exists():

    raise RuntimeError(
        "Frozen RANDOM_NATURAL bitset missing."
    )


if sha256_file(
    RANDOM_BITSET_PATH
) != EXPECTED_RANDOM_BITSET_SHA256:

    raise RuntimeError(
        "Frozen RANDOM_NATURAL bitset changed."
    )


packed_random = np.fromfile(
    RANDOM_BITSET_PATH,
    dtype=np.uint8,
)


random_validation_mask = np.unpackbits(
    packed_random,
    bitorder="little",
)[:N_DEVELOPMENT].astype(
    bool
)


if len(
    random_validation_mask
) != N_DEVELOPMENT:

    raise RuntimeError(
        "Random validation mask logical length mismatch."
    )


if int(
    random_validation_mask.sum()
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Random validation mask population changed."
    )


print()
print("[OK] RANDOM_NATURAL membership bitset exact")


# =============================================================================
# 14. VERIFY CACHE LABEL / POSITION ARRAYS
# =============================================================================

LABEL_PATH = guard_path(
    CACHE_DIR
    / "binary_label.uint8.dat"
)

CLEAN_POSITION_PATH = guard_path(
    CACHE_DIR
    / "clean_position.int64.dat"
)


for path, manifest_name in [
    (
        LABEL_PATH,
        "binary_label.uint8.dat",
    ),
    (
        CLEAN_POSITION_PATH,
        "clean_position.int64.dat",
    ),
]:

    if not path.exists():

        raise RuntimeError(
            f"Required execution-cache array missing: {path}"
        )


    if sha256_file(
        path
    ) != manifest_hashes[
        manifest_name
    ]:

        raise RuntimeError(
            f"Execution-cache array changed: {manifest_name}"
        )


labels_all = np.memmap(
    LABEL_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


clean_positions_all = np.memmap(
    CLEAN_POSITION_PATH,
    dtype="<i8",
    mode="r",
    shape=(
        N_DEVELOPMENT,
    ),
)


# =============================================================================
# 15. VERIFY SELECTED FEATURE RECORDS
# =============================================================================

feature_records = {
    int(record["index"]):
        record
    for record in cache_metadata[
        "feature_columns"
    ]
}


SELECTED_FEATURE_RECORDS = []


for index in CACHE_INDICES:

    if index not in feature_records:

        raise RuntimeError(
            f"Missing execution-cache feature record: {index}"
        )


    record = feature_records[
        index
    ]


    if record[
        "feature"
    ] != CACHE_FULL_FEATURES[
        index
    ]:

        raise RuntimeError(
            f"Execution-cache feature record mismatch at index {index}"
        )


    relative_path = record[
        "path"
    ]


    if relative_path not in manifest_hashes:

        raise RuntimeError(
            f"Feature missing from sealed manifest: {relative_path}"
        )


    if record[
        "sha256"
    ] != manifest_hashes[
        relative_path
    ]:

        raise RuntimeError(
            f"Feature metadata/manifest SHA mismatch: {relative_path}"
        )


    physical_path = guard_path(
        CACHE_DIR
        / relative_path
    )


    if not physical_path.exists():

        raise RuntimeError(
            f"Physical execution-cache feature missing: {physical_path}"
        )


    expected_bytes = (
        N_DEVELOPMENT
        * 8
    )


    if physical_path.stat().st_size != expected_bytes:

        raise RuntimeError(
            f"Execution-cache feature size mismatch: {record['feature']}"
        )


    SELECTED_FEATURE_RECORDS.append(
        {
            **record,
            "physical_path":
                physical_path,
        }
    )


# =============================================================================
# 16. ENVIRONMENT LOCK
# =============================================================================

EXPECTED_VERSIONS = {

    "numpy":
        "2.0.2",

    "pandas":
        "2.3.3",

    "scikit-learn":
        "1.6.1",

    "xgboost":
        "3.2.0",

    "lightgbm":
        "4.6.0",

    "shap":
        "0.51.0",
}


actual_versions = {}


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    actual_versions[
        package
    ] = actual

    if actual != expected:

        raise RuntimeError(
            f"Environment mismatch: {package}; "
            f"expected={expected}, actual={actual}"
        )


print()
print("[OK] frozen package versions verified")


# =============================================================================
# 17. GPU PREFLIGHT
# =============================================================================

try:

    gpu_check = subprocess.run(
        [
            "nvidia-smi",
            "-L",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

except FileNotFoundError as exc:

    raise RuntimeError(
        "nvidia-smi unavailable; XGBoost backend is frozen to CUDA."
    ) from exc


if gpu_check.returncode != 0:

    raise RuntimeError(
        "No NVIDIA GPU detected."
    )


print()
print("[OK] NVIDIA GPU detected:")
print(
    gpu_check.stdout.strip()
)


# =============================================================================
# 18. STORAGE PREFLIGHT
# =============================================================================

MATRIX_BYTES = (
    N_DEVELOPMENT
    * N_FEATURES
    * 8
)

LABEL_OUTPUT_BYTES = (
    TRAIN_ROWS
    + VALIDATION_ROWS
)

POSITION_OUTPUT_BYTES = (
    VALIDATION_ROWS
    * 8
)


required_bytes = (
    MATRIX_BYTES
    + LABEL_OUTPUT_BYTES
    + POSITION_OUTPUT_BYTES
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


minimum_free = (
    required_bytes
    + 1 * 1024**3
)


print()
print("=" * 100)
print("OPTIMIZED PLACEBO RANDOM MATERIALIZATION PLAN")
print("=" * 100)
print()

print(
    "Placebo            :",
    TARGET_PLACEBO
)
print(
    "Source             : sealed execution cache v1"
)
print(
    "Parquet reads      : 0"
)
print(
    "Selected features  :",
    N_FEATURES
)
print(
    "Train rows         :",
    f"{TRAIN_ROWS:,}"
)
print(
    "Validation rows    :",
    f"{VALIDATION_ROWS:,}"
)
print(
    "Matrix storage     :",
    f"{MATRIX_BYTES / 1024**3:.3f} GiB"
)
print(
    "Working free       :",
    f"{disk.free / 1024**3:.3f} GiB"
)


if disk.free < minimum_free:

    raise RuntimeError(
        "\nINSUFFICIENT DISK FOR STAGE23-2A\n"
        f"required with safety margin: "
        f"{minimum_free / 1024**3:.3f} GiB\n"
        f"available                  : "
        f"{disk.free / 1024**3:.3f} GiB"
    )


# =============================================================================
# 19. CREATE OUTPUT + EXECUTION STATE
# =============================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


execution_state = {

    "stage":
        STAGE,

    "placebo":
        TARGET_PLACEBO,

    "split":
        TARGET_SPLIT,

    "status":
        "INITIALIZED",

    "created_utc":
        now_utc(),

    "execution_parent_commit":
        PREDECESSOR_COMMIT,

    "cache_seal_commit":
        CACHE_SEAL_COMMIT,

    "stage23_total_model_fits_before_cell":
        STAGE23_FITS_BEFORE,

    "placebo_boosted_fits_before_cell":
        PLACEBO_FITS_BEFORE,

    "lightgbm_completed_fits":
        0,

    "xgboost_completed_fits":
        0,

    "models_completed_total_this_cell":
        0,

    "stage23_metrics_calculated":
        False,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 20. CREATE RANDOM TRAIN / VALIDATION MEMMAPS
# =============================================================================

X_TRAIN_PATH = (
    RUNTIME_DIR
    / "X_train_placebo_counts_float64.dat"
)

X_VALIDATION_PATH = (
    RUNTIME_DIR
    / "X_validation_placebo_counts_float64.dat"
)

Y_TRAIN_PATH = (
    RUNTIME_DIR
    / "y_train_uint8.dat"
)

Y_VALIDATION_PATH = (
    RUNTIME_DIR
    / "y_validation_uint8.dat"
)

VAL_POSITION_PATH = (
    RUNTIME_DIR
    / "validation_clean_position_int64.dat"
)


X_train = np.memmap(
    X_TRAIN_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        TRAIN_ROWS,
        N_FEATURES,
    ),
)


X_validation = np.memmap(
    X_VALIDATION_PATH,
    dtype="<f8",
    mode="w+",
    shape=(
        VALIDATION_ROWS,
        N_FEATURES,
    ),
)


y_train = np.memmap(
    Y_TRAIN_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        TRAIN_ROWS,
    ),
)


y_validation = np.memmap(
    Y_VALIDATION_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


validation_positions = np.memmap(
    VAL_POSITION_PATH,
    dtype="<i8",
    mode="w+",
    shape=(
        VALIDATION_ROWS,
    ),
)


# =============================================================================
# 21. OPEN 67 SELECTED RAW FEATURE COLUMNS
# =============================================================================

source_feature_mmaps = []


for record in SELECTED_FEATURE_RECORDS:

    source_feature_mmaps.append(
        np.memmap(
            record[
                "physical_path"
            ],
            dtype="<f8",
            mode="r",
            shape=(
                N_DEVELOPMENT,
            ),
        )
    )


# =============================================================================
# 22. MATERIALIZE RANDOM SPLIT FROM SEALED CACHE
# =============================================================================

CHUNK_ROWS = 250_000

train_cursor = 0
validation_cursor = 0

feature_hashers = [
    hashlib.sha256()
    for _ in range(
        N_FEATURES
    )
]


materialize_start = (
    time.perf_counter()
)


print()
print("=" * 100)
print("MATERIALIZING PLACEBO_COUNTS RANDOM SPLIT")
print("=" * 100)
print()


total_chunks = (
    N_DEVELOPMENT
    + CHUNK_ROWS
    - 1
) // CHUNK_ROWS


for chunk_index, start in enumerate(
    range(
        0,
        N_DEVELOPMENT,
        CHUNK_ROWS,
    ),
    start=1,
):

    end = min(
        start
        + CHUNK_ROWS,
        N_DEVELOPMENT,
    )

    n_rows = (
        end
        - start
    )


    is_validation = (
        random_validation_mask[
            start:end
        ]
    )

    is_train = (
        ~is_validation
    )


    n_train = int(
        is_train.sum()
    )

    n_validation = int(
        is_validation.sum()
    )


    chunk = np.empty(
        (
            n_rows,
            N_FEATURES,
        ),
        dtype=np.float64,
    )


    for output_col, source_mm in enumerate(
        source_feature_mmaps
    ):

        source_slice = np.asarray(
            source_mm[
                start:end
            ],
            dtype="<f8",
        )


        feature_hashers[
            output_col
        ].update(
            np.ascontiguousarray(
                source_slice,
                dtype="<f8",
            ).tobytes(
                order="C"
            )
        )


        chunk[
            :,
            output_col
        ] = source_slice


    if n_train:

        train_end = (
            train_cursor
            + n_train
        )

        X_train[
            train_cursor:train_end,
            :
        ] = chunk[
            is_train,
            :
        ]

        y_train[
            train_cursor:train_end
        ] = labels_all[
            start:end
        ][
            is_train
        ]

        train_cursor = (
            train_end
        )


    if n_validation:

        validation_end = (
            validation_cursor
            + n_validation
        )

        X_validation[
            validation_cursor:validation_end,
            :
        ] = chunk[
            is_validation,
            :
        ]

        y_validation[
            validation_cursor:validation_end
        ] = labels_all[
            start:end
        ][
            is_validation
        ]

        validation_positions[
            validation_cursor:validation_end
        ] = clean_positions_all[
            start:end
        ][
            is_validation
        ]

        validation_cursor = (
            validation_end
        )


    if (
        chunk_index == 1
        or chunk_index % 10 == 0
        or chunk_index == total_chunks
    ):

        print(
            f"  chunk {chunk_index:>3}/{total_chunks}"
            f" — train={train_cursor:,}"
            f" val={validation_cursor:,}"
        )


    del (
        chunk,
        is_train,
        is_validation,
    )

    gc.collect()


X_train.flush()
X_validation.flush()
y_train.flush()
y_validation.flush()
validation_positions.flush()


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


# =============================================================================
# 23. VERIFY ALL 67 SELECTED PHYSICAL FEATURE DIGESTS
# =============================================================================

for output_col, record in enumerate(
    SELECTED_FEATURE_RECORDS
):

    actual_digest = (
        feature_hashers[
            output_col
        ].hexdigest()
    )

    expected_digest = (
        record[
            "sha256"
        ]
    )


    if actual_digest != expected_digest:

        raise RuntimeError(
            "\nPHYSICAL CACHE FEATURE MUTATION DETECTED\n"
            f"feature  : {record['feature']}\n"
            f"expected : {expected_digest}\n"
            f"actual   : {actual_digest}"
        )


print()
print(
    "[OK] selected physical cache columns:"
    " 67 / 67 exact during materialization"
)


# =============================================================================
# 24. RANDOM MATERIALIZATION ASSERTIONS
# =============================================================================

if train_cursor != TRAIN_ROWS:

    raise RuntimeError(
        "\nRandom training row count mismatch.\n"
        f"expected: {TRAIN_ROWS:,}\n"
        f"actual  : {train_cursor:,}"
    )


if validation_cursor != VALIDATION_ROWS:

    raise RuntimeError(
        "\nRandom validation row count mismatch.\n"
        f"expected: {VALIDATION_ROWS:,}\n"
        f"actual  : {validation_cursor:,}"
    )


train_attack = int(
    np.count_nonzero(
        y_train == 1
    )
)

train_benign = int(
    np.count_nonzero(
        y_train == 0
    )
)

validation_attack = int(
    np.count_nonzero(
        y_validation == 1
    )
)

validation_benign = int(
    np.count_nonzero(
        y_validation == 0
    )
)


if (
    train_attack != TRAIN_ATTACK
    or train_benign != TRAIN_BENIGN
):

    raise RuntimeError(
        "Random training class counts changed."
    )


if (
    validation_attack != VALIDATION_ATTACK
    or validation_benign != VALIDATION_BENIGN
):

    raise RuntimeError(
        "Random validation class counts changed."
    )


if not np.all(
    validation_positions[:-1]
    < validation_positions[1:]
):

    raise RuntimeError(
        "Validation clean positions are not strictly increasing."
    )


if not np.all(
    random_validation_mask[
        np.asarray(
            validation_positions,
            dtype=np.int64,
        )
    ]
):

    raise RuntimeError(
        "Validation clean positions disagree with frozen bitset."
    )


print()
print("[OK] PLACEBO_COUNTS RANDOM_NATURAL materialization")
print(
    "     train          :",
    f"{TRAIN_ROWS:,}",
    f"(B={train_benign:,}, A={train_attack:,})"
)
print(
    "     validation     :",
    f"{VALIDATION_ROWS:,}",
    f"(B={validation_benign:,}, A={validation_attack:,})"
)
print(
    "     retained       :",
    N_FEATURES
)
print(
    "     dtype          : float64"
)
print(
    "     Parquet reads  : 0"
)
print(
    "     materialization:",
    f"{materialize_seconds:.3f} s"
)


execution_state.update({

    "status":
        "MATERIALIZED_FROM_SEALED_CACHE",

    "materialization_seconds":
        materialize_seconds,

    "train_rows":
        TRAIN_ROWS,

    "validation_rows":
        VALIDATION_ROWS,

    "feature_count":
        N_FEATURES,

    "input_dtype":
        "float64",

    "physical_feature_digests_verified":
        N_FEATURES,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 25. FROZEN MODEL PARAMETERS
# =============================================================================

lgbm_params = dict(
    model_spec[
        "lightgbm"
    ][
        "parameters"
    ]
)

xgb_params = dict(
    model_spec[
        "xgboost"
    ][
        "parameters"
    ]
)


EXPECTED_LGBM_PARAMS = {

    "boosting_type":
        "gbdt",

    "colsample_bytree":
        1.0,

    "device_type":
        "cpu",

    "learning_rate":
        0.06,

    "max_depth":
        12,

    "min_child_samples":
        20,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "num_leaves":
        127,

    "objective":
        "binary",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "subsample_freq":
        1,

    "verbosity":
        -1,
}


EXPECTED_XGB_PARAMS = {

    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "n_estimators":
        400,

    "n_jobs":
        -1,

    "objective":
        "binary:logistic",

    "random_state":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


if lgbm_params != EXPECTED_LGBM_PARAMS:

    raise RuntimeError(
        "Frozen LightGBM parameters changed."
    )


if xgb_params != EXPECTED_XGB_PARAMS:

    raise RuntimeError(
        "Frozen XGBoost parameters changed."
    )


# =============================================================================
# 26. FIT 1 / 2 — LIGHTGBM
# =============================================================================

print()
print("=" * 100)
print("FIT 1 / 2 — LIGHTGBM")
print("=" * 100)
print()


lgbm_model = LGBMClassifier(
    **lgbm_params
)


lgbm_start = (
    time.perf_counter()
)


lgbm_model.fit(
    X_train,
    y_train,
)


lgbm_seconds = (
    time.perf_counter()
    - lgbm_start
)


LGBM_MODEL_PATH = (
    OUTPUT_DIR
    / "placebo_counts_random_natural_lightgbm_model.txt"
)


lgbm_model.booster_.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_probability = np.asarray(
    lgbm_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


LGBM_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_lightgbm_float64.npy"
)


np.save(
    LGBM_PROB_PATH,
    lgbm_probability,
    allow_pickle=False,
)


LGBM_MODEL_SHA = sha256_file(
    LGBM_MODEL_PATH
)


execution_state.update({

    "status":
        "LIGHTGBM_COMPLETED",

    "lightgbm_completed_fits":
        1,

    "models_completed_total_this_cell":
        1,

    "lightgbm_fit_seconds":
        lgbm_seconds,

    "lightgbm_model_sha256":
        LGBM_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] LightGBM completed in",
    f"{lgbm_seconds:.3f} s"
)

print(
    "     model SHA256:",
    LGBM_MODEL_SHA
)


del lgbm_model

gc.collect()


# =============================================================================
# 27. FIT 2 / 2 — XGBOOST
# =============================================================================

print()
print("=" * 100)
print("FIT 2 / 2 — XGBOOST")
print("=" * 100)
print()


xgb_model = XGBClassifier(
    **xgb_params
)


xgb_start = (
    time.perf_counter()
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_seconds = (
    time.perf_counter()
    - xgb_start
)


XGB_MODEL_PATH = (
    OUTPUT_DIR
    / "placebo_counts_random_natural_xgboost_model.json"
)


xgb_model.save_model(
    str(
        XGB_MODEL_PATH
    )
)


xgb_probability = np.asarray(
    xgb_model.predict_proba(
        X_validation
    )[:, 1],
    dtype=np.float64,
)


XGB_PROB_PATH = (
    RUNTIME_DIR
    / "validation_probability_xgboost_float64.npy"
)


np.save(
    XGB_PROB_PATH,
    xgb_probability,
    allow_pickle=False,
)


XGB_MODEL_SHA = sha256_file(
    XGB_MODEL_PATH
)


execution_state.update({

    "status":
        "XGBOOST_COMPLETED",

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "xgboost_fit_seconds":
        xgb_seconds,

    "xgboost_model_sha256":
        XGB_MODEL_SHA,
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


print(
    "[OK] XGBoost completed in",
    f"{xgb_seconds:.3f} s"
)

print(
    "     model SHA256:",
    XGB_MODEL_SHA
)


del xgb_model

gc.collect()


# =============================================================================
# 28. FROZEN 50/50 ENSEMBLE
# =============================================================================

ensemble_float64 = (
    0.5
    * lgbm_probability
    +
    0.5
    * xgb_probability
)


ensemble_probability = np.asarray(
    ensemble_float64,
    dtype=np.float32,
)


if len(
    ensemble_probability
) != VALIDATION_ROWS:

    raise RuntimeError(
        "Ensemble probability length mismatch."
    )


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probabilities produced."
    )


# =============================================================================
# 29. RANKING METRICS
# =============================================================================

y_validation_array = np.asarray(
    y_validation,
    dtype=np.uint8,
)


attack_prevalence = float(
    y_validation_array.mean()
)


pr_auc = float(
    average_precision_score(
        y_validation_array,
        ensemble_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_validation_array,
        ensemble_probability,
    )
)


pr_auc_minus_prevalence = float(
    pr_auc
    - attack_prevalence
)


# =============================================================================
# 30. FIXED THRESHOLD 0.50
# =============================================================================

predicted = (
    ensemble_probability
    >= np.float32(0.50)
).astype(
    np.uint8
)


tn, fp, fn, tp = (
    confusion_matrix(
        y_validation_array,
        predicted,
        labels=[0, 1],
    ).ravel()
)


accuracy = float(
    accuracy_score(
        y_validation_array,
        predicted,
    )
)

precision = float(
    precision_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

recall = float(
    recall_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)

f1 = float(
    f1_score(
        y_validation_array,
        predicted,
        zero_division=0,
    )
)


fpr = float(
    fp / (fp + tn)
    if (fp + tn)
    else 0.0
)


fnr = float(
    fn / (fn + tp)
    if (fn + tp)
    else 0.0
)


# =============================================================================
# 31. FROZEN FULL RANDOM REFERENCE
# =============================================================================

if not FULL_RANDOM_RESULT.exists():

    raise RuntimeError(
        f"Frozen FULL random reference missing:\n{FULL_RANDOM_RESULT}"
    )


full_random = json.loads(
    FULL_RANDOM_RESULT.read_text(
        encoding="utf-8"
    )
)


if full_random[
    "cell"
] != "RANDOM_NATURAL":

    raise RuntimeError(
        "Unexpected frozen FULL RANDOM_NATURAL reference."
    )


full_pr_auc = float(
    full_random[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    full_random[
        "validation_probability"
    ][
        "roc_auc"
    ]
)


full_standard = (
    full_random[
        "operating_points"
    ][
        "standard"
    ]
)


# =============================================================================
# 32. PLACEBO REMOVAL PENALTIES
# =============================================================================

pr_auc_removal_penalty = float(
    full_pr_auc
    - pr_auc
)


roc_auc_removal_penalty = float(
    full_roc_auc
    - roc_auc
)


f1_removal_penalty = float(
    float(
        full_standard[
            "f1"
        ]
    )
    - f1
)


recall_removal_penalty = float(
    float(
        full_standard[
            "recall"
        ]
    )
    - recall
)


fpr_change = float(
    float(
        full_standard[
            "fpr"
        ]
    )
    - fpr
)


# =============================================================================
# 33. SAVE VALIDATION PROBABILITIES
# =============================================================================

VALIDATION_PROBABILITY_PATH = (
    OUTPUT_DIR
    / "placebo_counts_random_natural_validation_probabilities.npz"
)


np.savez_compressed(
    VALIDATION_PROBABILITY_PATH,

    clean_position=np.asarray(
        validation_positions,
        dtype=np.int64,
    ),

    binary_label=(
        y_validation_array
    ),

    lightgbm_probability=np.asarray(
        lgbm_probability,
        dtype=np.float32,
    ),

    xgboost_probability=np.asarray(
        xgb_probability,
        dtype=np.float32,
    ),

    ensemble_probability=(
        ensemble_probability
    ),
)


VALIDATION_PROBABILITY_SHA = (
    sha256_file(
        VALIDATION_PROBABILITY_PATH
    )
)


# =============================================================================
# 34. RESULT JSON
# =============================================================================

result = {

    "stage":
        STAGE,

    "status":
        "PLACEBO_COUNTS_RANDOM_NATURAL_RESULT_COMPLETE_UNSEALED",

    "created_utc":
        now_utc(),

    "protocol": {

        "commit":
            PROTOCOL_COMMIT,

        "tag":
            PROTOCOL_TAG,

        "checksum_manifest_sha256":
            PROTOCOL_MANIFEST_SHA256,
    },

    "execution_parent": {

        "commit":
            PREDECESSOR_COMMIT,

        "tag":
            PREDECESSOR_TAG,
    },

    "placebo_protocol": {

        "placebo":
            TARGET_PLACEBO,

        "matched_k":
            placebo_spec[
                "matched_k"
            ],

        "number_of_placebos":
            placebo_spec[
                "number_of_placebos"
            ],

        "seed":
            placebo_spec[
                "seed"
            ],

        "selection_method":
            placebo_spec[
                "selection_method"
            ],
    },

    "cache": {

        "identifier":
            "stage23_execution_cache_v1",

        "seal_commit":
            CACHE_SEAL_COMMIT,

        "seal_tag":
            CACHE_SEAL_TAG,

        "sealed":
            True,

        "metadata_sha256":
            CACHE_METADATA_SHA256,

        "data_manifest_sha256":
            CACHE_DATA_MANIFEST_SHA256,

        "semantic_70f_sha256":
            CACHE_SEMANTIC_70F_SHA256,

        "selected_physical_feature_digests_verified":
            N_FEATURES,

        "parquet_files_read":
            0,

        "materialization_seconds":
            materialize_seconds,
    },

    "cell": {

        "placebo":
            TARGET_PLACEBO,

        "split":
            TARGET_SPLIT,

        "removed_features":
            REMOVED_FEATURES,

        "feature_count":
            N_FEATURES,

        "feature_order":
            FEATURES,
    },

    "data": {

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "train": {

            "rows":
                TRAIN_ROWS,

            "benign":
                train_benign,

            "attack":
                train_attack,
        },

        "validation": {

            "rows":
                VALIDATION_ROWS,

            "benign":
                validation_benign,

            "attack":
                validation_attack,

            "attack_prevalence":
                attack_prevalence,
        },
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "parameters":
                lgbm_params,

            "library_version":
                actual_versions[
                    "lightgbm"
                ],

            "fit_seconds":
                lgbm_seconds,

            "completed_fits":
                1,

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "library_version":
                actual_versions[
                    "xgboost"
                ],

            "fit_seconds":
                xgb_seconds,

            "completed_fits":
                1,

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                XGB_MODEL_SHA,
        },
    },

    "ranking_metrics": {

        "primary_metric":
            "PR_AUC",

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc_minus_prevalence,
    },

    "fixed_threshold_0_50": {

        "threshold":
            0.50,

        "prediction_rule":
            "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_PROBABILITY_GTE_0_50",

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),
    },

    "full_reference": {

        "source":
            "Frozen Stage22R RANDOM_NATURAL FULL",

        "retrained":
            False,

        "pr_auc":
            full_pr_auc,

        "roc_auc":
            full_roc_auc,

        "f1_at_0_50":
            float(
                full_standard[
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                full_standard[
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                full_standard[
                    "fpr"
                ]
            ),
    },

    "removal_penalty_full_minus_placebo": {

        "pr_auc":
            pr_auc_removal_penalty,

        "roc_auc":
            roc_auc_removal_penalty,

        "f1_at_0_50":
            f1_removal_penalty,

        "recall_at_0_50":
            recall_removal_penalty,

        "fpr_at_0_50":
            fpr_change,
    },

    "placebo_interaction": {

        "status":
            "PENDING_MATCHED_CHRONOLOGICAL_CELL",

        "required_future_cell":
            "Stage23-2B PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL",
    },

    "artifacts": {

        "validation_probabilities": {

            "path":
                VALIDATION_PROBABILITY_PATH.name,

            "sha256":
                VALIDATION_PROBABILITY_SHA,
        },

        "lightgbm_model": {

            "path":
                LGBM_MODEL_PATH.name,

            "sha256":
                LGBM_MODEL_SHA,
        },

        "xgboost_model": {

            "path":
                XGB_MODEL_PATH.name,

            "sha256":
                XGB_MODEL_SHA,
        },
    },

    "runtime": {

        "optimized_cache_materialization_seconds":
            materialize_seconds,

        "lightgbm_fit_seconds":
            lgbm_seconds,

        "xgboost_fit_seconds":
            xgb_seconds,

        "parquet_files_read":
            0,
    },

    "environment":
        actual_versions,

    "governance": {

        "new_lightgbm_fits":
            1,

        "new_xgboost_fits":
            1,

        "new_model_fits_this_cell":
            2,

        "stage23_total_model_fits_before_cell":
            STAGE23_FITS_BEFORE,

        "stage23_total_model_fits_after_cell":
            STAGE23_FITS_AFTER,

        "stage23_total_authorized_model_fits":
            50,

        "placebo_boosted_fits_before_cell":
            PLACEBO_FITS_BEFORE,

        "placebo_boosted_fits_after_cell":
            PLACEBO_FITS_AFTER,

        "placebo_boosted_fits_expected":
            20,

        "primary_boosted_model_fits_complete":
            True,

        "primary_boosted_model_fits_total":
            24,

        "execution_cache_optimization":
            True,

        "execution_cache_scientifically_sealed":
            True,

        "parquet_files_read":
            0,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "rebalancing":
            False,

        "shap_executed":
            False,

        "stump_executed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "stage23_0_modified":
            False,

        "placebo_definition_changed":
            False,

        "feature_definition_changed":
            False,

        "split_definition_changed":
            False,

        "input_dtype_changed":
            False,

        "model_specification_changed":
            False,
    },

    "next_authorized_action":
        "Seal Stage23-2A before executing another Stage23 model cell.",
}


write_json_atomic(
    RESULT_PATH,
    result,
)


RESULT_SHA = sha256_file(
    RESULT_PATH
)


# =============================================================================
# 35. CHECKSUM MANIFEST
# =============================================================================

CHECKSUM_OUTPUT_PATH = (
    OUTPUT_DIR
    / "checksums.sha256"
)


artifact_paths = [
    LGBM_MODEL_PATH,
    XGB_MODEL_PATH,
    VALIDATION_PROBABILITY_PATH,
    RESULT_PATH,
]


CHECKSUM_OUTPUT_PATH.write_text(
    "\n".join(
        f"{sha256_file(path)}  {path.name}"
        for path in artifact_paths
    ) + "\n",
    encoding="utf-8",
)


CHECKSUM_OUTPUT_SHA = sha256_file(
    CHECKSUM_OUTPUT_PATH
)


# =============================================================================
# 36. FINAL EXECUTION STATE
# =============================================================================

execution_state.update({

    "status":
        "RESULT_COMPLETE_UNSEALED",

    "lightgbm_completed_fits":
        1,

    "xgboost_completed_fits":
        1,

    "models_completed_total_this_cell":
        2,

    "stage23_total_model_fits_after_cell":
        STAGE23_FITS_AFTER,

    "placebo_boosted_fits_after_cell":
        PLACEBO_FITS_AFTER,

    "stage23_metrics_calculated":
        True,

    "optimized_cache_used":
        True,

    "parquet_files_read":
        0,

    "result_path":
        str(
            RESULT_PATH
        ),

    "result_sha256":
        RESULT_SHA,

    "checksum_manifest_sha256":
        CHECKSUM_OUTPUT_SHA,

    "completed_utc":
        now_utc(),
})


write_json_atomic(
    EXECUTION_STATE_PATH,
    execution_state,
)


# =============================================================================
# 37. REMOVE ONLY TRANSIENT MATRICES
# =============================================================================
#
# KEEP:
#   /kaggle/working/stage23_execution_cache_v1
#
# =============================================================================

del X_train
del X_validation
del y_train
del y_validation
del validation_positions

del labels_all
del clean_positions_all

for index in range(
    len(source_feature_mmaps)
):

    source_feature_mmaps[
        index
    ] = None


del source_feature_mmaps

gc.collect()


for path in [
    X_TRAIN_PATH,
    X_VALIDATION_PATH,
    Y_TRAIN_PATH,
    Y_VALIDATION_PATH,
    VAL_POSITION_PATH,
    LGBM_PROB_PATH,
    XGB_PROB_PATH,
]:

    if path.exists():

        path.unlink()


try:

    RUNTIME_DIR.rmdir()

except OSError:

    pass


# =============================================================================
# 38. FINAL REPORT
# =============================================================================

print()
print("=" * 100)
print("STAGE23-2A COMPLETE — RESULT UNSEALED")
print("=" * 100)

print()
print("Cell:")
print(
    "  placebo :",
    TARGET_PLACEBO
)
print(
    "  split   :",
    TARGET_SPLIT
)

print()
print("Frozen placebo removal:")
print(
    "  removed :",
    REMOVED_FEATURES
)
print(
    "  retained:",
    N_FEATURES
)

print()
print("Optimized execution:")
print(
    "  cache           : stage23_execution_cache_v1"
)
print(
    "  Parquet reads   : 0"
)
print(
    "  materialization :",
    f"{materialize_seconds:.3f} s"
)

print()
print("Data:")
print(
    "  train      :",
    f"{TRAIN_ROWS:,}"
)
print(
    "  validation :",
    f"{VALIDATION_ROWS:,}"
)

print()
print("New model fits:")
print("  LightGBM       : 1")
print("  XGBoost        : 1")
print("  THIS CELL      : 2")
print("  PLACEBO BLOCK  : 2 / 20")
print("  STAGE23 TOTAL  : 26 / 50")


print()
print("=" * 100)
print("PLACEBO_COUNTS RANDOM RANKING")
print("=" * 100)

print()
print(
    f"Attack prevalence             : {attack_prevalence:.12f}"
)

print(
    f"PLACEBO_COUNTS PR-AUC         : {pr_auc:.12f}"
)

print(
    f"FULL PR-AUC                   : {full_pr_auc:.12f}"
)

print(
    f"PR-AUC placebo penalty        : {pr_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PLACEBO_COUNTS ROC-AUC        : {roc_auc:.12f}"
)

print(
    f"FULL ROC-AUC                  : {full_roc_auc:.12f}"
)

print(
    f"ROC-AUC placebo penalty       : {roc_auc_removal_penalty:+.12f}"
)

print()
print(
    f"PR-AUC - prevalence           : {pr_auc_minus_prevalence:.12f}"
)


print()
print("=" * 100)
print("FIXED OPERATING POINT — THRESHOLD 0.50")
print("=" * 100)

print()
print(
    f"Accuracy  : {accuracy:.12f}"
)
print(
    f"Precision : {precision:.12f}"
)
print(
    f"Recall    : {recall:.12f}"
)
print(
    f"F1        : {f1:.12f}"
)
print(
    f"FPR       : {fpr:.12f}"
)
print(
    f"FNR       : {fnr:.12f}"
)

print()
print(
    "TN / FP / FN / TP:",
    int(tn),
    int(fp),
    int(fn),
    int(tp),
)


print()
print("=" * 100)
print("PLACEBO INTERACTION STATUS")
print("=" * 100)

print()
print(
    "PENDING matched Stage23-2B "
    "PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL"
)


print()
print("=" * 100)
print("ARTIFACTS")
print("=" * 100)

print()
print("Result:")
print(
    " ",
    RESULT_PATH
)
print(
    " SHA256:",
    RESULT_SHA
)

print()
print("LightGBM:")
print(
    " ",
    LGBM_MODEL_PATH
)
print(
    " SHA256:",
    LGBM_MODEL_SHA
)

print()
print("XGBoost:")
print(
    " ",
    XGB_MODEL_PATH
)
print(
    " SHA256:",
    XGB_MODEL_SHA
)

print()
print("Validation probabilities:")
print(
    " ",
    VALIDATION_PROBABILITY_PATH
)
print(
    " SHA256:",
    VALIDATION_PROBABILITY_SHA
)

print()
print("Checksum manifest:")
print(
    " ",
    CHECKSUM_OUTPUT_PATH
)
print(
    " SHA256:",
    CHECKSUM_OUTPUT_SHA
)


print()
print("=" * 100)
print("GOVERNANCE")
print("=" * 100)

print()
print("Stage23 fits completed     : 26 / 50")
print("Primary boosted fits       : 24 / 24 COMPLETE")
print("Placebo boosted fits       :  2 / 20")
print("Optimized cache used       : YES")
print("Parquet files read         : 0")
print("Threshold optimization     : NO")
print("Subset-specific tuning     : NO")
print("Rebalancing                : NO")
print("SHAP executed              : NO")
print("Stump executed             : NO")
print("Raw Mar1 read              : NO")
print("Raw Mar2 read              : NO")
print("Stage23-0 modified         : NO")
print("Placebo definition changed : NO")
print("Git commit created         : NO")
print("Git tag created            : NO")


print()
print("=" * 100)
print("NEXT ACTION")
print("=" * 100)

print()
print(
    "Do NOT execute another Stage23 model cell."
)

print(
    "First seal and push Stage23-2A."
)

print()
print(
    "After sealing, next matched placebo cell:"
)

print(
    "  Stage23-2B — "
    "PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL"
)

print("=" * 100)

STAGE23-2A — PLACEBO_COUNTS × RANDOM_NATURAL
FIRST FROZEN PLACEBO BOOSTED EXECUTION

[OK] branch             : main
[OK] HEAD               : 359b6a2c1b548e6b9fc84d2712f0fb13714b1c24
[OK] Stage23-0 tag      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] Stage23-1L tag     : 359b6a2c1b548e6b9fc84d2712f0fb13714b1c24
[OK] cache seal tag     : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree           : CLEAN
[OK] Stage23 fits       : 24 / 50
[OK] Primary boosted    : 24 / 24 FROZEN
[OK] placebo block      : AUTHORIZED
[OK] Stage23-0 artifacts : 23/23 exact

Frozen placebo:
  placebo       : PLACEBO_COUNTS
  split         : RANDOM_NATURAL
  removed       : ['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
  retained      : 67
  matched k     : 3
[OK] sealed execution cache metadata exact

[OK] RANDOM_NATURAL membership bitset exact

[OK] frozen package versions verified

[OK] NVIDIA GPU detected:
GPU 0: Tesla T4 (UUID: GPU-ae5cf237-2aaf-2843-df92-7772a07b6647)
GPU 1: Tesla T4 (UUID:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM completed in 378.475 s
     model SHA256: e9aa94b7d71d49454422c41e46c7e1999936c1b92dc6686afbea8d2394e271a7

FIT 2 / 2 — XGBOOST

[OK] XGBoost completed in 104.815 s
     model SHA256: a5363e87c833b75d7a6e5f8b7a2f6371b4c9867cb7ac9f2ab30654949b707de0

STAGE23-2A COMPLETE — RESULT UNSEALED

Cell:
  placebo : PLACEBO_COUNTS
  split   : RANDOM_NATURAL

Frozen placebo removal:
  removed : ['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
  retained: 67

Optimized execution:
  cache           : stage23_execution_cache_v1
  Parquet reads   : 0
  materialization : 62.595 s

Data:
  train      : 11,529,922
  validation : 2,882,481

New model fits:
  LightGBM       : 1
  XGBoost        : 1
  THIS CELL      : 2
  PLACEBO BLOCK  : 2 / 20
  STAGE23 TOTAL  : 26 / 50

PLACEBO_COUNTS RANDOM RANKING

Attack prevalence             : 0.136847389454
PLACEBO_COUNTS PR-AUC         : 0.995586516336
FULL PR-AUC                   : 0.995590041899
PR-AUC placebo penalty        : +0.000003525563

PL

In [30]:
# =============================================================================
# STAGE23-2A — SEAL
# PLACEBO_COUNTS × RANDOM_NATURAL
#
# ZERO FITS
# ZERO DATASET READS
#
# BEFORE: 26 / 50 executed, 24 / 50 sealed
# AFTER : 26 / 50 sealed
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess, hashlib, shutil, json, os

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
SRC = Path("/kaggle/working/stage23_2a_placebo_counts_random_natural")

DST = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
    / "random"
    / "placebo_counts"
)

EXPECTED_PARENT = "359b6a2c1b548e6b9fc84d2712f0fb13714b1c24"
PARENT_TAG = "stage23-1l-behavior-only-chronological-natural-v1"

TAG = "stage23-2a-placebo-counts-random-natural-v1"
COMMIT_MESSAGE = "Stage23-2A: freeze PLACEBO_COUNTS random-natural result"

EXPECTED = {
    "stage23_2a_placebo_counts_random_natural_result.json":
        "8eab55f5d983f8510e0225cbea7574d52b00d5017675511419f79b4f9b4ba4aa",
    "placebo_counts_random_natural_lightgbm_model.txt":
        "e9aa94b7d71d49454422c41e46c7e1999936c1b92dc6686afbea8d2394e271a7",
    "placebo_counts_random_natural_xgboost_model.json":
        "a5363e87c833b75d7a6e5f8b7a2f6371b4c9867cb7ac9f2ab30654949b707de0",
    "placebo_counts_random_natural_validation_probabilities.npz":
        "3c91f1b1d5848a920dda970d4c07b2fd094c8c7f1770f88785c9896c24e0911a",
    "checksums.sha256":
        "914d734a243a7fe9ee231cfef0ca9143982e4d9f26e73998b1517faab20b125f",
}


def run(cmd, check=True, show=False):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    out = (p.stdout or "").strip()
    if show and out:
        print(out)
    if check and p.returncode != 0:
        raise RuntimeError(out)
    return out


def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(32 * 1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


print("=" * 100)
print("STAGE23-2A — PLACEBO_COUNTS RANDOM SEAL")
print("=" * 100)

assert run(["git", "branch", "--show-current"]) == "main"
assert run(["git", "rev-parse", "HEAD"]) == EXPECTED_PARENT
assert run(["git", "rev-list", "-n", "1", PARENT_TAG]) == EXPECTED_PARENT
assert run(["git", "status", "--porcelain"]) == ""

if run(["git", "tag", "--list", TAG]):
    raise RuntimeError(f"Tag already exists: {TAG}")

if DST.exists():
    raise RuntimeError(f"Destination already exists: {DST}")

for name, digest in EXPECTED.items():
    p = SRC / name
    if not p.exists():
        raise RuntimeError(f"Missing: {p}")
    actual = sha(p)
    if actual != digest:
        raise RuntimeError(
            f"SHA mismatch: {name}\nexpected={digest}\nactual={actual}"
        )
    print("[EXACT]", name)

result = json.loads(
    (SRC / "stage23_2a_placebo_counts_random_natural_result.json")
    .read_text(encoding="utf-8")
)

assert result["stage"] == "Stage23-2A"
assert result["cell"]["placebo"] == "PLACEBO_COUNTS"
assert result["cell"]["split"] == "RANDOM_NATURAL"
assert result["cell"]["removed_features"] == [
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
]
assert result["cell"]["feature_count"] == 67

g = result["governance"]
assert g["new_model_fits_this_cell"] == 2
assert g["stage23_total_model_fits_before_cell"] == 24
assert g["stage23_total_model_fits_after_cell"] == 26
assert g["placebo_boosted_fits_after_cell"] == 2
assert g["parquet_files_read"] == 0
assert g["raw_mar1_accessed"] is False
assert g["raw_mar2_accessed"] is False
assert g["threshold_optimization"] is False
assert g["subset_specific_tuning"] is False

DST.mkdir(parents=True)

for name in EXPECTED:
    shutil.copy2(SRC / name, DST / name)

receipt = {
    "stage": "Stage23-2A",
    "status": "SCIENTIFIC_RESULT_FROZEN",
    "sealed_utc": datetime.now(timezone.utc).isoformat(),
    "placebo": "PLACEBO_COUNTS",
    "split": "RANDOM_NATURAL",
    "removed_features": [
        "Flow Duration",
        "Tot Fwd Pkts",
        "Tot Bwd Pkts",
    ],
    "feature_count": 67,
    "parent_commit": EXPECTED_PARENT,
    "parent_tag": PARENT_TAG,
    "result_tag": TAG,
    "stage23_models_fit_total": 26,
    "primary_boosted_fits": 24,
    "placebo_boosted_fits": 2,
    "remaining_placebo_boosted_fits": 18,
    "remaining_stump_fits": 6,
    "core_sha256": EXPECTED,
    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,
    "next_authorized_execution": (
        "Stage23 remaining frozen placebo batch: "
        "Stage23-2B through Stage23-2J, beginning with "
        "PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL"
    ),
}

(DST / "seal_receipt.json").write_text(
    json.dumps(receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

(DST / "README.md").write_text(
    f"""# Stage23-2A — PLACEBO_COUNTS × RANDOM_NATURAL

Frozen placebo removal:

- Flow Duration
- Tot Fwd Pkts
- Tot Bwd Pkts

Retained features: 67

PR-AUC: {result["ranking_metrics"]["pr_auc"]:.12f}
ROC-AUC: {result["ranking_metrics"]["roc_auc"]:.12f}

FULL − PLACEBO PR-AUC penalty:
{result["removal_penalty_full_minus_placebo"]["pr_auc"]:+.12f}

FULL − PLACEBO ROC-AUC penalty:
{result["removal_penalty_full_minus_placebo"]["roc_auc"]:+.12f}

Stage23 fits after execution: 26 / 50.
Placebo boosted fits: 2 / 20.

Matched chronological interaction remains pending.

No raw March 1 or March 2 access occurred.
""",
    encoding="utf-8",
)

repo_manifest = DST / "repository_checksums.sha256"
files = sorted(
    p for p in DST.iterdir()
    if p.is_file() and p.name != repo_manifest.name
)
repo_manifest.write_text(
    "\n".join(f"{sha(p)}  {p.name}" for p in files) + "\n",
    encoding="utf-8",
)

rel = DST.relative_to(REPO)
run(["git", "add", "--", str(rel)])

staged = run(["git", "diff", "--cached", "--name-only"]).splitlines()
if not staged or any(not x.startswith(str(rel) + "/") for x in staged):
    raise RuntimeError("Unexpected staged files.")

run(["git", "commit", "-m", COMMIT_MESSAGE], show=True)

sealed_commit = run(["git", "rev-parse", "HEAD"])

run([
    "git", "tag", "-a", TAG, sealed_commit,
    "-m", "Stage23-2A frozen PLACEBO_COUNTS RANDOM_NATURAL result"
])

run(["git", "push", "origin", "main"], show=True)
run(["git", "push", "origin", TAG], show=True)

remote_main = run(
    ["git", "ls-remote", "origin", "refs/heads/main"]
).split()[0]

if remote_main != sealed_commit:
    raise RuntimeError("Remote main mismatch.")

remote_tag = run([
    "git", "ls-remote", "origin",
    f"refs/tags/{TAG}^{{}}"
])

if not remote_tag or remote_tag.split()[0] != sealed_commit:
    raise RuntimeError("Remote tag mismatch.")

if run(["git", "status", "--porcelain"]):
    raise RuntimeError("Repository dirty after seal.")

print()
print("=" * 100)
print("STAGE23-2A SEALED")
print("=" * 100)
print("commit :", sealed_commit)
print("tag    :", TAG)
print("fits   : 26 / 50")
print("placebo:  2 / 20")
print()
print("NEXT: run the 18-fit remaining-placebo batch cell.")
print("=" * 100)

STAGE23-2A — PLACEBO_COUNTS RANDOM SEAL
[EXACT] stage23_2a_placebo_counts_random_natural_result.json
[EXACT] placebo_counts_random_natural_lightgbm_model.txt
[EXACT] placebo_counts_random_natural_xgboost_model.json
[EXACT] placebo_counts_random_natural_validation_probabilities.npz
[EXACT] checksums.sha256
[main 660c39b] Stage23-2A: freeze PLACEBO_COUNTS random-natural result
 8 files changed, 8154 insertions(+)
 create mode 100644 results/stage23_shortcut_feature_audit/stage23_2_placebo_ablation/random/placebo_counts/README.md
 create mode 100644 results/stage23_shortcut_feature_audit/stage23_2_placebo_ablation/random/placebo_counts/checksums.sha256
 create mode 100644 results/stage23_shortcut_feature_audit/stage23_2_placebo_ablation/random/placebo_counts/placebo_counts_random_natural_lightgbm_model.txt
 create mode 100644 results/stage23_shortcut_feature_audit/stage23_2_placebo_ablation/random/placebo_counts/placebo_counts_random_natural_validation_probabilities.npz
 create mode 10064

In [31]:
# =============================================================================
# STAGE23-2B..2J — REMAINING PLACEBO BOOSTED BATCH
#
# EXECUTES EXACTLY THE REMAINING 18 PLACEBO MODEL FITS:
#
# 2B  PLACEBO_COUNTS           × CHRONOLOGICAL_NATURAL   = 2 fits
# 2C  PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL          = 2 fits
# 2D  PLACEBO_VOLUME_DIRECTION × CHRONOLOGICAL_NATURAL   = 2 fits
# 2E  PLACEBO_IAT              × RANDOM_NATURAL          = 2 fits
# 2F  PLACEBO_IAT              × CHRONOLOGICAL_NATURAL   = 2 fits
# 2G  PLACEBO_PACKET_SIZE      × RANDOM_NATURAL          = 2 fits
# 2H  PLACEBO_PACKET_SIZE      × CHRONOLOGICAL_NATURAL   = 2 fits
# 2I  PLACEBO_ACTIVITY         × RANDOM_NATURAL          = 2 fits
# 2J  PLACEBO_ACTIVITY         × CHRONOLOGICAL_NATURAL   = 2 fits
#
# BEFORE: Stage23 26 / 50
# AFTER : Stage23 44 / 50
#
# PLACEBO BEFORE:  2 / 20
# PLACEBO AFTER : 20 / 20
#
# LightGBM remains CPU — FROZEN.
# XGBoost remains CUDA — FROZEN.
#
# RESUME SAFE:
# - every component fit is recorded immediately
# - completed component fits are never repeated
# - rerunning this same cell resumes from saved task state
#
# ZERO PARQUET READS.
# RAW MAR1/MAR2 FORBIDDEN.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import shutil
import time
import gc

import numpy as np

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# =============================================================================
# 0. PATHS / FROZEN IDENTIFIERS
# =============================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
CACHE = Path("/kaggle/working/stage23_execution_cache_v1")

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

BATCH = Path(
    "/kaggle/working/stage23_remaining_placebo_batch"
)

BATCH.mkdir(parents=True, exist_ok=True)

BATCH_STATE = BATCH / "batch_state.json"

PREDECESSOR_TAG = "stage23-2a-placebo-counts-random-natural-v1"

PROTOCOL_COMMIT = "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
PROTOCOL_TAG = "stage23-0-protocol-lock-v1"

CACHE_TAG = "stage23-execution-cache-v1"
CACHE_COMMIT = "66e1c6382caa63699766e605df63df02beff894f"

CACHE_METADATA_SHA = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)
CACHE_MANIFEST_SHA = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)
RANDOM_BITSET_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)

N = 14_412_403
RANDOM_TRAIN = 11_529_922
RANDOM_VAL = 2_882_481

CHRONO_TRAIN = 13_818_623
CHRONO_VAL = 593_780

CHUNK = 250_000


# =============================================================================
# 1. TASK LIST — FROZEN
# =============================================================================

TASKS = [
    ("Stage23-2B", "PLACEBO_COUNTS", "CHRONOLOGICAL_NATURAL"),

    ("Stage23-2C", "PLACEBO_VOLUME_DIRECTION", "RANDOM_NATURAL"),
    ("Stage23-2D", "PLACEBO_VOLUME_DIRECTION", "CHRONOLOGICAL_NATURAL"),

    ("Stage23-2E", "PLACEBO_IAT", "RANDOM_NATURAL"),
    ("Stage23-2F", "PLACEBO_IAT", "CHRONOLOGICAL_NATURAL"),

    ("Stage23-2G", "PLACEBO_PACKET_SIZE", "RANDOM_NATURAL"),
    ("Stage23-2H", "PLACEBO_PACKET_SIZE", "CHRONOLOGICAL_NATURAL"),

    ("Stage23-2I", "PLACEBO_ACTIVITY", "RANDOM_NATURAL"),
    ("Stage23-2J", "PLACEBO_ACTIVITY", "CHRONOLOGICAL_NATURAL"),
]

EXPECTED_REMOVALS = {
    "PLACEBO_COUNTS":
        ["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"],

    "PLACEBO_VOLUME_DIRECTION":
        ["TotLen Fwd Pkts", "TotLen Bwd Pkts", "Down/Up Ratio"],

    "PLACEBO_IAT":
        ["Flow IAT Mean", "Flow IAT Std", "Flow IAT Max"],

    "PLACEBO_PACKET_SIZE":
        ["Pkt Len Mean", "Pkt Len Std", "Pkt Len Var"],

    "PLACEBO_ACTIVITY":
        ["Active Mean", "Idle Mean", "Fwd Act Data Pkts"],
}


# =============================================================================
# 2. HELPERS
# =============================================================================

def now():
    return datetime.now(timezone.utc).isoformat()


def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    if p.returncode != 0:
        raise RuntimeError(p.stdout)
    return (p.stdout or "").strip()


def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(32 * 1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    tmp.replace(path)


def slug(s):
    return s.lower().replace("placebo_", "").replace("_", "-")


def metric_block(y, p):
    p = np.asarray(p, dtype=np.float32)
    y = np.asarray(y, dtype=np.uint8)

    prevalence = float(y.mean())
    pr = float(average_precision_score(y, p))
    roc = float(roc_auc_score(y, p))

    pred = (p >= np.float32(0.50)).astype(np.uint8)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    return {
        "attack_prevalence": prevalence,
        "pr_auc": pr,
        "roc_auc": roc,
        "pr_auc_minus_attack_prevalence": pr - prevalence,

        "threshold_0_50": {
            "accuracy": float(accuracy_score(y, pred)),
            "precision": float(precision_score(y, pred, zero_division=0)),
            "recall": float(recall_score(y, pred, zero_division=0)),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "fpr": float(fp / (fp + tn)),
            "fnr": float(fn / (fn + tp)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
    }


# =============================================================================
# 3. GOVERNANCE PREFLIGHT
# =============================================================================

print("=" * 100)
print("STAGE23 — REMAINING 18 PLACEBO FITS")
print("=" * 100)

assert run(["git", "branch", "--show-current"]) == "main"
assert run(["git", "status", "--porcelain"]) == ""

head = run(["git", "rev-parse", "HEAD"])
tag_commit = run(["git", "rev-list", "-n", "1", PREDECESSOR_TAG])

if head != tag_commit:
    raise RuntimeError(
        "HEAD must be the sealed Stage23-2A commit.\n"
        f"HEAD={head}\ntag={tag_commit}"
    )

assert run(["git", "rev-list", "-n", "1", PROTOCOL_TAG]) == PROTOCOL_COMMIT
assert run(["git", "rev-list", "-n", "1", CACHE_TAG]) == CACHE_COMMIT

print("[OK] Stage23-2A sealed parent:", head)
print("[OK] Stage23-0 protocol       :", PROTOCOL_COMMIT)
print("[OK] execution cache          :", CACHE_COMMIT)
print("[OK] worktree                 : CLEAN")


# =============================================================================
# 4. LOAD FROZEN SPECS
# =============================================================================

placebo_spec = json.loads(
    (PROTOCOL / "placebo_ablation_spec.json").read_text()
)

feature_spec = json.loads(
    (PROTOCOL / "feature_subset_spec.json").read_text()
)

model_spec = json.loads(
    (PROTOCOL / "model_spec.json").read_text()
)

split_spec = json.loads(
    (PROTOCOL / "inherited_splits.json").read_text()
)

assert placebo_spec["matched_k"] == 3
assert placebo_spec["number_of_placebos"] == 5

full_features = feature_spec["full_feature_order"]

for name, removed in EXPECTED_REMOVALS.items():
    spec = placebo_spec["subsets"][name]

    assert spec["removed"] == removed
    assert spec["feature_count"] == 67
    assert len(spec["retained"]) == 67
    assert spec["retained"] == [
        f for f in full_features if f not in removed
    ]


# =============================================================================
# 5. ENVIRONMENT / BACKENDS
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "shap": "0.51.0",
}

versions = {
    p: importlib_metadata.version(p)
    for p in EXPECTED_VERSIONS
}

for p, v in EXPECTED_VERSIONS.items():
    assert versions[p] == v, (p, versions[p], v)

lgbm_params = dict(model_spec["lightgbm"]["parameters"])
xgb_params = dict(model_spec["xgboost"]["parameters"])

if lgbm_params["device_type"] != "cpu":
    raise RuntimeError("LightGBM backend changed from frozen CPU.")

if xgb_params["device"] != "cuda":
    raise RuntimeError("XGBoost backend changed from frozen CUDA.")

gpu = subprocess.run(
    ["nvidia-smi", "-L"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if gpu.returncode != 0:
    raise RuntimeError("CUDA GPU unavailable.")

print("[OK] LightGBM backend : CPU — frozen")
print("[OK] XGBoost backend  : CUDA — frozen")


# =============================================================================
# 6. CACHE VERIFICATION
# =============================================================================

metadata_path = CACHE / "metadata.json"
manifest_path = CACHE / "data_checksums.sha256"
bitset_path = CACHE / "random_validation.packbits"

assert sha(metadata_path) == CACHE_METADATA_SHA
assert sha(manifest_path) == CACHE_MANIFEST_SHA
assert sha(bitset_path) == RANDOM_BITSET_SHA

cache_meta = json.loads(metadata_path.read_text())

assert cache_meta["semantic_equivalence"]["exact_byte_equality"] is True
assert cache_meta["full_feature_order"] == full_features

manifest = {}
for line in manifest_path.read_text().splitlines():
    if line.strip():
        digest, rel = line.split("  ", 1)
        manifest[rel] = digest

feature_records = {
    r["feature"]: r
    for r in cache_meta["feature_columns"]
}

labels_path = CACHE / "binary_label.uint8.dat"
positions_path = CACHE / "clean_position.int64.dat"

if sha(labels_path) != manifest["binary_label.uint8.dat"]:
    raise RuntimeError("Cache labels mutated.")

if sha(positions_path) != manifest["clean_position.int64.dat"]:
    raise RuntimeError("Cache positions mutated.")

labels = np.memmap(
    labels_path,
    dtype=np.uint8,
    mode="r",
    shape=(N,),
)

positions = np.memmap(
    positions_path,
    dtype="<i8",
    mode="r",
    shape=(N,),
)

packed = np.fromfile(bitset_path, dtype=np.uint8)
random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N].astype(bool)

assert int(random_val_mask.sum()) == RANDOM_VAL

print("[OK] sealed cache metadata / split membership verified")


# =============================================================================
# 7. FULL REFERENCES
# =============================================================================

FULL_RANDOM_PATH = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

FULL_CHRONO_PATH = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

full_random = json.loads(FULL_RANDOM_PATH.read_text())
full_chrono = json.loads(FULL_CHRONO_PATH.read_text())

assert full_random["cell"] == "RANDOM_NATURAL"
assert full_chrono["cell"] == "CHRONOLOGICAL_NATURAL"


def full_reference(split):
    src = full_random if split == "RANDOM_NATURAL" else full_chrono
    return {
        "pr_auc": float(src["validation_probability"]["pr_auc"]),
        "roc_auc": float(src["validation_probability"]["roc_auc"]),
        "f1": float(src["operating_points"]["standard"]["f1"]),
        "recall": float(src["operating_points"]["standard"]["recall"]),
        "fpr": float(src["operating_points"]["standard"]["fpr"]),
    }


# =============================================================================
# 8. SEALED 2A RANDOM RESULT FOR MATCHED COUNTS INTERACTION
# =============================================================================

COUNTS_RANDOM_RESULT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
    / "random"
    / "placebo_counts"
    / "stage23_2a_placebo_counts_random_natural_result.json"
)

if not COUNTS_RANDOM_RESULT.exists():
    raise RuntimeError("Sealed Stage23-2A result missing.")

counts_random = json.loads(COUNTS_RANDOM_RESULT.read_text())


# =============================================================================
# 9. BATCH STATE
# =============================================================================

if BATCH_STATE.exists():
    batch_state = json.loads(BATCH_STATE.read_text())
else:
    batch_state = {
        "status": "RUNNING",
        "created_utc": now(),
        "parent_commit": head,
        "fits_before": 26,
        "fits_target_after": 44,
        "placebo_fits_before": 2,
        "placebo_fits_target_after": 20,
        "tasks": {},
        "raw_mar1_accessed": False,
        "raw_mar2_accessed": False,
    }
    write_json(BATCH_STATE, batch_state)

if batch_state["parent_commit"] != head:
    raise RuntimeError("Batch parent changed.")

completed_fits = 0

for task in batch_state["tasks"].values():
    completed_fits += int(task.get("lightgbm_fit_complete", False))
    completed_fits += int(task.get("xgboost_fit_complete", False))

print()
print("Already completed inside this batch:", completed_fits, "/ 18")


# =============================================================================
# 10. TASK EXECUTOR
# =============================================================================

def execute_task(stage, placebo, split):

    global batch_state

    task_key = f"{stage}_{placebo}_{split}"
    pslug = slug(placebo)
    sslug = "random-natural" if split == "RANDOM_NATURAL" else "chronological-natural"

    out = BATCH / f"{stage.lower().replace('stage23-', 'stage23_')}_{pslug}_{sslug}"
    out.mkdir(parents=True, exist_ok=True)

    state_path = out / "task_state.json"
    result_path = out / f"{stage.lower().replace('stage23-', 'stage23_')}_{pslug}_{sslug}_result.json"

    lgbm_model_path = out / f"{pslug}_{sslug}_lightgbm_model.txt"
    xgb_model_path = out / f"{pslug}_{sslug}_xgboost_model.json"

    lgbm_prob_path = out / "lightgbm_validation_probability_float64.npy"
    xgb_prob_path = out / "xgboost_validation_probability_float64.npy"

    probs_final = out / f"{pslug}_{sslug}_validation_probabilities.npz"
    checksums_path = out / "checksums.sha256"

    runtime = out / "runtime"
    runtime.mkdir(exist_ok=True)

    if state_path.exists():
        st = json.loads(state_path.read_text())
    else:
        st = {
            "stage": stage,
            "placebo": placebo,
            "split": split,
            "status": "INITIALIZED",
            "lightgbm_fit_complete": False,
            "xgboost_fit_complete": False,
            "result_complete": False,
            "created_utc": now(),
        }
        write_json(state_path, st)

    if st.get("result_complete"):
        print(f"[SKIP COMPLETE] {stage} {placebo} × {split}")
        batch_state["tasks"][task_key] = st
        write_json(BATCH_STATE, batch_state)
        return

    spec = placebo_spec["subsets"][placebo]
    features = spec["retained"]

    assert spec["removed"] == EXPECTED_REMOVALS[placebo]
    assert len(features) == 67

    if split == "RANDOM_NATURAL":
        train_rows, val_rows = RANDOM_TRAIN, RANDOM_VAL
    else:
        train_rows, val_rows = CHRONO_TRAIN, CHRONO_VAL

    Xtr_path = runtime / "X_train.float64.dat"
    Xva_path = runtime / "X_val.float64.dat"
    ytr_path = runtime / "y_train.uint8.dat"
    yva_path = runtime / "y_val.uint8.dat"
    pos_path = runtime / "val_position.int64.dat"

    expected_xtr_bytes = train_rows * 67 * 8
    expected_xva_bytes = val_rows * 67 * 8

    matrices_ready = (
        Xtr_path.exists()
        and Xtr_path.stat().st_size == expected_xtr_bytes
        and Xva_path.exists()
        and Xva_path.stat().st_size == expected_xva_bytes
        and ytr_path.exists()
        and ytr_path.stat().st_size == train_rows
        and yva_path.exists()
        and yva_path.stat().st_size == val_rows
        and pos_path.exists()
        and pos_path.stat().st_size == val_rows * 8
        and st.get("materialization_complete", False)
    )

    if not matrices_ready:

        for p in [Xtr_path, Xva_path, ytr_path, yva_path, pos_path]:
            if p.exists():
                p.unlink()

        print()
        print("=" * 100)
        print(f"{stage} — MATERIALIZE {placebo} × {split}")
        print("=" * 100)

        Xtr = np.memmap(
            Xtr_path, dtype="<f8", mode="w+",
            shape=(train_rows, 67)
        )
        Xva = np.memmap(
            Xva_path, dtype="<f8", mode="w+",
            shape=(val_rows, 67)
        )
        ytr = np.memmap(
            ytr_path, dtype=np.uint8, mode="w+",
            shape=(train_rows,)
        )
        yva = np.memmap(
            yva_path, dtype=np.uint8, mode="w+",
            shape=(val_rows,)
        )
        vpos = np.memmap(
            pos_path, dtype="<i8", mode="w+",
            shape=(val_rows,)
        )

        srcs = []
        for feature in features:
            rec = feature_records[feature]
            path = CACHE / rec["path"]

            if not path.exists():
                raise RuntimeError(f"Missing cache feature: {feature}")

            if path.stat().st_size != N * 8:
                raise RuntimeError(f"Cache size mismatch: {feature}")

            srcs.append(
                np.memmap(path, dtype="<f8", mode="r", shape=(N,))
            )

        tr_cursor = 0
        va_cursor = 0
        t0 = time.perf_counter()

        for start in range(0, N, CHUNK):

            end = min(start + CHUNK, N)
            nr = end - start

            chunk = np.empty((nr, 67), dtype=np.float64)

            for j, mm in enumerate(srcs):
                chunk[:, j] = mm[start:end]

            if split == "RANDOM_NATURAL":

                vm = random_val_mask[start:end]
                tm = ~vm

                nt = int(tm.sum())
                nv = int(vm.sum())

                if nt:
                    Xtr[tr_cursor:tr_cursor+nt] = chunk[tm]
                    ytr[tr_cursor:tr_cursor+nt] = labels[start:end][tm]
                    tr_cursor += nt

                if nv:
                    Xva[va_cursor:va_cursor+nv] = chunk[vm]
                    yva[va_cursor:va_cursor+nv] = labels[start:end][vm]
                    vpos[va_cursor:va_cursor+nv] = positions[start:end][vm]
                    va_cursor += nv

            else:

                # Chronological split is contiguous:
                # [0:13,818,623) train, remainder validation.

                if start < CHRONO_TRAIN:
                    te = min(end, CHRONO_TRAIN)
                    ntrain_chunk = te - start

                    if ntrain_chunk:
                        Xtr[tr_cursor:tr_cursor+ntrain_chunk] = chunk[:ntrain_chunk]
                        ytr[tr_cursor:tr_cursor+ntrain_chunk] = labels[start:te]
                        tr_cursor += ntrain_chunk

                if end > CHRONO_TRAIN:
                    vs = max(start, CHRONO_TRAIN)
                    offset = vs - start
                    nv = end - vs

                    if nv:
                        Xva[va_cursor:va_cursor+nv] = chunk[offset:]
                        yva[va_cursor:va_cursor+nv] = labels[vs:end]
                        vpos[va_cursor:va_cursor+nv] = positions[vs:end]
                        va_cursor += nv

            del chunk
            gc.collect()

        Xtr.flush(); Xva.flush()
        ytr.flush(); yva.flush(); vpos.flush()

        if tr_cursor != train_rows or va_cursor != val_rows:
            raise RuntimeError(
                f"Materialization count mismatch: "
                f"train {tr_cursor}/{train_rows}, val {va_cursor}/{val_rows}"
            )

        seconds = time.perf_counter() - t0

        st["materialization_complete"] = True
        st["materialization_seconds"] = seconds
        st["status"] = "MATERIALIZED"
        write_json(state_path, st)

        del Xtr, Xva, ytr, yva, vpos, srcs
        gc.collect()

    # reopen existing or newly created matrices
    Xtr = np.memmap(
        Xtr_path, dtype="<f8", mode="r",
        shape=(train_rows, 67)
    )
    Xva = np.memmap(
        Xva_path, dtype="<f8", mode="r",
        shape=(val_rows, 67)
    )
    ytr = np.memmap(
        ytr_path, dtype=np.uint8, mode="r",
        shape=(train_rows,)
    )
    yva = np.memmap(
        yva_path, dtype=np.uint8, mode="r",
        shape=(val_rows,)
    )
    vpos = np.memmap(
        pos_path, dtype="<i8", mode="r",
        shape=(val_rows,)
    )

    # -------------------------------------------------------------------------
    # LightGBM — fit only if not already consumed
    # -------------------------------------------------------------------------

    if not st.get("lightgbm_fit_complete", False):

        print()
        print("=" * 100)
        print(f"{stage} — LIGHTGBM — {placebo} × {split}")
        print("=" * 100)

        model = LGBMClassifier(**lgbm_params)

        t0 = time.perf_counter()
        model.fit(Xtr, ytr)
        seconds = time.perf_counter() - t0

        model.booster_.save_model(str(lgbm_model_path))

        lp = np.asarray(
            model.predict_proba(Xva)[:, 1],
            dtype=np.float64,
        )
        np.save(lgbm_prob_path, lp, allow_pickle=False)

        st["lightgbm_fit_complete"] = True
        st["lightgbm_fit_seconds"] = seconds
        st["lightgbm_model_sha256"] = sha(lgbm_model_path)
        st["status"] = "LIGHTGBM_COMPLETE"
        write_json(state_path, st)

        batch_state["tasks"][task_key] = st
        write_json(BATCH_STATE, batch_state)

        print(f"[OK] LightGBM {seconds:.3f}s")
        print("     SHA:", st["lightgbm_model_sha256"])

        del model, lp
        gc.collect()

    elif not lgbm_prob_path.exists():

        raise RuntimeError(
            f"{stage}: LightGBM fit consumed but probability file missing. "
            "DO NOT rerun automatically."
        )

    # -------------------------------------------------------------------------
    # XGBoost — fit only if not already consumed
    # -------------------------------------------------------------------------

    if not st.get("xgboost_fit_complete", False):

        print()
        print("=" * 100)
        print(f"{stage} — XGBOOST — {placebo} × {split}")
        print("=" * 100)

        model = XGBClassifier(**xgb_params)

        t0 = time.perf_counter()
        model.fit(Xtr, ytr)
        seconds = time.perf_counter() - t0

        model.save_model(str(xgb_model_path))

        xp = np.asarray(
            model.predict_proba(Xva)[:, 1],
            dtype=np.float64,
        )
        np.save(xgb_prob_path, xp, allow_pickle=False)

        st["xgboost_fit_complete"] = True
        st["xgboost_fit_seconds"] = seconds
        st["xgboost_model_sha256"] = sha(xgb_model_path)
        st["status"] = "XGBOOST_COMPLETE"
        write_json(state_path, st)

        batch_state["tasks"][task_key] = st
        write_json(BATCH_STATE, batch_state)

        print(f"[OK] XGBoost {seconds:.3f}s")
        print("     SHA:", st["xgboost_model_sha256"])

        del model, xp
        gc.collect()

    elif not xgb_prob_path.exists():

        raise RuntimeError(
            f"{stage}: XGBoost fit consumed but probability file missing. "
            "DO NOT rerun automatically."
        )

    # -------------------------------------------------------------------------
    # Result
    # -------------------------------------------------------------------------

    lp = np.load(lgbm_prob_path, allow_pickle=False)
    xp = np.load(xgb_prob_path, allow_pickle=False)

    ensemble = np.asarray(
        0.5 * np.asarray(lp, dtype=np.float64)
        + 0.5 * np.asarray(xp, dtype=np.float64),
        dtype=np.float32,
    )

    metrics = metric_block(yva, ensemble)

    full = full_reference(split)

    penalty = {
        "pr_auc": full["pr_auc"] - metrics["pr_auc"],
        "roc_auc": full["roc_auc"] - metrics["roc_auc"],
        "f1_at_0_50":
            full["f1"] - metrics["threshold_0_50"]["f1"],
        "recall_at_0_50":
            full["recall"] - metrics["threshold_0_50"]["recall"],
        "fpr_at_0_50":
            full["fpr"] - metrics["threshold_0_50"]["fpr"],
    }

    np.savez_compressed(
        probs_final,
        clean_position=np.asarray(vpos, dtype=np.int64),
        binary_label=np.asarray(yva, dtype=np.uint8),
        lightgbm_probability=np.asarray(lp, dtype=np.float32),
        xgboost_probability=np.asarray(xp, dtype=np.float32),
        ensemble_probability=ensemble,
    )

    result = {
        "stage": stage,
        "status": "RESULT_COMPLETE_UNSEALED",
        "created_utc": now(),

        "cell": {
            "placebo": placebo,
            "split": split,
            "removed_features": EXPECTED_REMOVALS[placebo],
            "feature_count": 67,
            "feature_order": features,
        },

        "execution_parent": {
            "commit": head,
            "tag": PREDECESSOR_TAG,
        },

        "models": {
            "lightgbm": {
                "backend": "cpu",
                "fit_seconds": st["lightgbm_fit_seconds"],
                "model_sha256": st["lightgbm_model_sha256"],
            },
            "xgboost": {
                "backend": "cuda",
                "fit_seconds": st["xgboost_fit_seconds"],
                "model_sha256": st["xgboost_model_sha256"],
            },
            "ensemble": "0.5 * LIGHTGBM + 0.5 * XGBOOST",
        },

        "ranking_metrics": {
            "attack_prevalence": metrics["attack_prevalence"],
            "pr_auc": metrics["pr_auc"],
            "roc_auc": metrics["roc_auc"],
            "pr_auc_minus_attack_prevalence":
                metrics["pr_auc_minus_attack_prevalence"],
        },

        "fixed_threshold_0_50": metrics["threshold_0_50"],

        "full_reference": full,

        "removal_penalty_full_minus_placebo": penalty,

        "cache": {
            "identifier": "stage23_execution_cache_v1",
            "parquet_files_read": 0,
            "materialization_seconds": st["materialization_seconds"],
        },

        "governance": {
            "new_model_fits_this_task": 2,
            "threshold_optimization": False,
            "subset_specific_tuning": False,
            "rebalancing": False,
            "raw_mar1_accessed": False,
            "raw_mar2_accessed": False,
            "stage23_0_modified": False,
            "placebo_definition_changed": False,
        },
    }

    # matched interaction once chronological member exists
    if split == "CHRONOLOGICAL_NATURAL":

        if placebo == "PLACEBO_COUNTS":
            random_result = counts_random
            rp = random_result["removal_penalty_full_minus_placebo"]

        else:
            random_stage = {
                "PLACEBO_VOLUME_DIRECTION": "Stage23-2C",
                "PLACEBO_IAT": "Stage23-2E",
                "PLACEBO_PACKET_SIZE": "Stage23-2G",
                "PLACEBO_ACTIVITY": "Stage23-2I",
            }[placebo]

            random_key_dir = (
                BATCH
                / f"{random_stage.lower().replace('stage23-', 'stage23_')}_"
                  f"{pslug}_random-natural"
            )

            random_result_files = list(
                random_key_dir.glob("*_result.json")
            )

            if len(random_result_files) != 1:
                raise RuntimeError(
                    f"Matched random result missing for {placebo}"
                )

            random_result = json.loads(
                random_result_files[0].read_text()
            )
            rp = random_result["removal_penalty_full_minus_placebo"]

        result["placebo_interaction"] = {
            "definition":
                "I(S) = DELTA_RANDOM(S) - DELTA_CHRONOLOGICAL(S)",

            "pr_auc":
                float(rp["pr_auc"] - penalty["pr_auc"]),

            "roc_auc":
                float(rp["roc_auc"] - penalty["roc_auc"]),

            "f1_at_0_50":
                float(rp["f1_at_0_50"] - penalty["f1_at_0_50"]),

            "recall_at_0_50":
                float(rp["recall_at_0_50"] - penalty["recall_at_0_50"]),

            "fpr_at_0_50":
                float(rp["fpr_at_0_50"] - penalty["fpr_at_0_50"]),

            "status":
                "POINT_ESTIMATE_ONLY",

            "confidence_interval_status":
                "PENDING_FROZEN_UNCERTAINTY_ANALYSIS",
        }

    else:
        result["placebo_interaction"] = {
            "status": "PENDING_MATCHED_CHRONOLOGICAL_TASK"
        }

    write_json(result_path, result)

    manifest_files = [
        result_path,
        lgbm_model_path,
        xgb_model_path,
        probs_final,
    ]

    checksums_path.write_text(
        "\n".join(
            f"{sha(p)}  {p.name}"
            for p in manifest_files
        ) + "\n",
        encoding="utf-8",
    )

    st["result_complete"] = True
    st["status"] = "RESULT_COMPLETE_UNSEALED"
    st["result_sha256"] = sha(result_path)
    st["checksums_sha256"] = sha(checksums_path)
    st["completed_utc"] = now()
    write_json(state_path, st)

    batch_state["tasks"][task_key] = st
    write_json(BATCH_STATE, batch_state)

    print()
    print("-" * 100)
    print(f"[COMPLETE] {stage} — {placebo} × {split}")
    print(f"PR-AUC      : {metrics['pr_auc']:.12f}")
    print(f"ROC-AUC     : {metrics['roc_auc']:.12f}")
    print(f"PR penalty  : {penalty['pr_auc']:+.12f}")
    print(f"ROC penalty : {penalty['roc_auc']:+.12f}")
    print("-" * 100)

    # cleanup transient matrices + component float64 probability files
    del Xtr, Xva, ytr, yva, vpos, lp, xp, ensemble
    gc.collect()

    for p in [
        Xtr_path, Xva_path,
        ytr_path, yva_path,
        pos_path,
        lgbm_prob_path,
        xgb_prob_path,
    ]:
        if p.exists():
            p.unlink()

    try:
        runtime.rmdir()
    except OSError:
        pass


# =============================================================================
# 11. EXECUTE / RESUME ALL 9 TASKS
# =============================================================================

for i, (stage, placebo, split) in enumerate(TASKS, start=1):

    print()
    print("#" * 100)
    print(f"TASK {i}/9 — {stage}: {placebo} × {split}")
    print("#" * 100)

    execute_task(stage, placebo, split)


# =============================================================================
# 12. FINAL FIT ACCOUNTING
# =============================================================================

task_states = []

for stage, placebo, split in TASKS:

    task_key = f"{stage}_{placebo}_{split}"

    if task_key not in batch_state["tasks"]:
        raise RuntimeError(f"Missing batch task state: {task_key}")

    st = batch_state["tasks"][task_key]

    if not (
        st.get("lightgbm_fit_complete")
        and st.get("xgboost_fit_complete")
        and st.get("result_complete")
    ):
        raise RuntimeError(f"Incomplete task: {task_key}")

    task_states.append(st)


new_fits = sum(
    int(st["lightgbm_fit_complete"])
    + int(st["xgboost_fit_complete"])
    for st in task_states
)

if new_fits != 18:
    raise RuntimeError(
        f"Expected exactly 18 completed batch fits; found {new_fits}"
    )


# =============================================================================
# 13. BATCH SUMMARY
# =============================================================================

summary_rows = []

for stage, placebo, split in TASKS:

    pslug = slug(placebo)
    sslug = "random-natural" if split == "RANDOM_NATURAL" else "chronological-natural"

    d = (
        BATCH
        / f"{stage.lower().replace('stage23-', 'stage23_')}_{pslug}_{sslug}"
    )

    results = list(d.glob("*_result.json"))

    if len(results) != 1:
        raise RuntimeError(f"Result count error: {d}")

    r = json.loads(results[0].read_text())

    row = {
        "stage": stage,
        "placebo": placebo,
        "split": split,
        "pr_auc": r["ranking_metrics"]["pr_auc"],
        "roc_auc": r["ranking_metrics"]["roc_auc"],
        "pr_penalty":
            r["removal_penalty_full_minus_placebo"]["pr_auc"],
        "roc_penalty":
            r["removal_penalty_full_minus_placebo"]["roc_auc"],
    }

    if split == "CHRONOLOGICAL_NATURAL":
        row["pr_interaction"] = r["placebo_interaction"]["pr_auc"]
        row["roc_interaction"] = r["placebo_interaction"]["roc_auc"]

    summary_rows.append(row)


summary = {
    "stage": "Stage23 remaining placebo batch",
    "status": "ALL_REMAINING_PLACEBO_RESULTS_COMPLETE_UNSEALED",
    "completed_utc": now(),
    "parent_commit": head,
    "parent_tag": PREDECESSOR_TAG,
    "new_model_fits": 18,
    "stage23_fits_before_batch": 26,
    "stage23_fits_after_batch": 44,
    "placebo_fits_before_batch": 2,
    "placebo_fits_after_batch": 20,
    "primary_boosted_fits": 24,
    "remaining_stump_fits": 6,
    "parquet_files_read": 0,
    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,
    "results": summary_rows,
}

summary_path = BATCH / "remaining_placebo_batch_summary.json"
write_json(summary_path, summary)

batch_state["status"] = "COMPLETE_UNSEALED"
batch_state["fits_completed_in_batch"] = 18
batch_state["stage23_fits_after"] = 44
batch_state["placebo_fits_after"] = 20
batch_state["completed_utc"] = now()
batch_state["summary_sha256"] = sha(summary_path)
write_json(BATCH_STATE, batch_state)


# =============================================================================
# 14. FINAL OUTPUT
# =============================================================================

print()
print("=" * 100)
print("ALL REMAINING PLACEBO BOOSTED FITS COMPLETE — UNSEALED")
print("=" * 100)
print()

for row in summary_rows:

    print(
        f"{row['stage']:>11}  "
        f"{row['placebo']:<26}  "
        f"{row['split']:<23}  "
        f"PR={row['pr_auc']:.9f}  "
        f"ROC={row['roc_auc']:.9f}"
    )

    if "pr_interaction" in row:
        print(
            f"{'':>66}"
            f"I_PR={row['pr_interaction']:+.9f}  "
            f"I_ROC={row['roc_interaction']:+.9f}"
        )

print()
print("=" * 100)
print("FIT ACCOUNTING")
print("=" * 100)
print()
print("Primary boosted fits : 24 / 24 COMPLETE")
print("Placebo boosted fits : 20 / 20 COMPLETE")
print("Batch fits completed : 18 / 18")
print("Stage23 total        : 44 / 50")
print("Remaining fits       : 6 / 50 — DEPTH-1 STUMPS ONLY")
print()
print("Parquet reads        : 0")
print("Raw Mar1 read        : NO")
print("Raw Mar2 read        : NO")
print("LightGBM backend     : CPU — frozen")
print("XGBoost backend      : CUDA — frozen")
print()
print("DO NOT RUN MORE BOOSTED MODELS.")
print("NEXT ACTION: seal this placebo batch, then run all 6 stumps in one cell.")
print("=" * 100)


STAGE23 — REMAINING 18 PLACEBO FITS
[OK] Stage23-2A sealed parent: 660c39bfb8207db326c73f4e5637239ca2140139
[OK] Stage23-0 protocol       : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] execution cache          : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree                 : CLEAN
[OK] LightGBM backend : CPU — frozen
[OK] XGBoost backend  : CUDA — frozen
[OK] sealed cache metadata / split membership verified

Already completed inside this batch: 0 / 18

####################################################################################################
TASK 1/9 — Stage23-2B: PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL
####################################################################################################

Stage23-2B — MATERIALIZE PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL

Stage23-2B — LIGHTGBM — PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM 464.396s
     SHA: f6e2747e30f376289bb43f1bd03780f36be095f4827135517f25e2c493b20da5

Stage23-2B — XGBOOST — PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL
[OK] XGBoost 140.105s
     SHA: 5152e767cd99c8a41361001fc603db946d38f1bc69f321db7df366d9c593f08e

----------------------------------------------------------------------------------------------------
[COMPLETE] Stage23-2B — PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL
PR-AUC      : 0.106566505817
ROC-AUC     : 0.516785367433
PR penalty  : -0.000351350683
ROC penalty : -0.001866941039
----------------------------------------------------------------------------------------------------

####################################################################################################
TASK 2/9 — Stage23-2C: PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL
####################################################################################################

Stage23-2C — MATERIALIZE PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL

Stage23-2C — LIGHTGBM

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM 379.150s
     SHA: b8ea7e3d449efecfc9d7b6bd9035b3f7774f43f81b531e163233bda9300b8c19

Stage23-2C — XGBOOST — PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL
[OK] XGBoost 106.051s
     SHA: 12381f2146e421a7a905897f54c197ac39fc935642f09a47bd36d0ae2abb3b6d

----------------------------------------------------------------------------------------------------
[COMPLETE] Stage23-2C — PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL
PR-AUC      : 0.995581152110
ROC-AUC     : 0.998616948486
PR penalty  : +0.000008889789
ROC penalty : +0.000007616287
----------------------------------------------------------------------------------------------------

####################################################################################################
TASK 3/9 — Stage23-2D: PLACEBO_VOLUME_DIRECTION × CHRONOLOGICAL_NATURAL
####################################################################################################

Stage23-2D — MATERIALIZE PLACEBO_VOLUME_DIRECTION × CHRONOLOGICAL_NATURAL

S

...

[OK] LightGBM 461.454s
     SHA: 025936171c5b0a6602730bfb8d9761027b65a1d338eb9079174140cec8b62275

Stage23-2H — XGBOOST — PLACEBO_PACKET_SIZE × CHRONOLOGICAL_NATURAL
[OK] XGBoost 141.475s
     SHA: 6daf16f58c8cb99abcda492a2f9224447e289621a7526b5a424e87b70d7a460c

----------------------------------------------------------------------------------------------------
[COMPLETE] Stage23-2H — PLACEBO_PACKET_SIZE × CHRONOLOGICAL_NATURAL
PR-AUC      : 0.106017252331
ROC-AUC     : 0.515719675375
PR penalty  : +0.000197902803
ROC penalty : -0.000801248981
----------------------------------------------------------------------------------------------------

####################################################################################################
TASK 8/9 — Stage23-2I: PLACEBO_ACTIVITY × RANDOM_NATURAL
####################################################################################################

Stage23-2I — MATERIALIZE PLACEBO_ACTIVITY × RANDOM_NATURAL

Stage23-2I — LIGHTGBM — PLA

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM 384.114s
     SHA: 3810cc9c083db44a8677e273a2fd55cebf555729315b6defc94f6d9e9ca0b608

Stage23-2I — XGBOOST — PLACEBO_ACTIVITY × RANDOM_NATURAL
[OK] XGBoost 105.209s
     SHA: d69c060e1c5fef4022f939c86f965a0781fe779952c30b2d0685cdfb51224d40

----------------------------------------------------------------------------------------------------
[COMPLETE] Stage23-2I — PLACEBO_ACTIVITY × RANDOM_NATURAL
PR-AUC      : 0.995584022042
ROC-AUC     : 0.998616983938
PR penalty  : +0.000006019857
ROC penalty : +0.000007580836
----------------------------------------------------------------------------------------------------

####################################################################################################
TASK 9/9 — Stage23-2J: PLACEBO_ACTIVITY × CHRONOLOGICAL_NATURAL
####################################################################################################

Stage23-2J — MATERIALIZE PLACEBO_ACTIVITY × CHRONOLOGICAL_NATURAL

Stage23-2J — LIGHTGBM — PLACEBO_A

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[OK] LightGBM 469.759s
     SHA: 9102444dd42048c94a605dd77134cf8cf695e92bb8b6e4dfddcc1e826fa86a93

Stage23-2J — XGBOOST — PLACEBO_ACTIVITY × CHRONOLOGICAL_NATURAL
[OK] XGBoost 137.844s
     SHA: 1ab84f44b70d302f8ce3f981338518f6debf74e6cce603a3d6228604bef314b1

----------------------------------------------------------------------------------------------------
[COMPLETE] Stage23-2J — PLACEBO_ACTIVITY × CHRONOLOGICAL_NATURAL
PR-AUC      : 0.106554516270
ROC-AUC     : 0.516519011938
PR penalty  : -0.000339361136
ROC penalty : -0.001600585544
----------------------------------------------------------------------------------------------------

ALL REMAINING PLACEBO BOOSTED FITS COMPLETE — UNSEALED

 Stage23-2B  PLACEBO_COUNTS              CHRONOLOGICAL_NATURAL    PR=0.106566506  ROC=0.516785367
                                                                  I_PR=+0.000354876  I_ROC=+0.001874086
 Stage23-2C  PLACEBO_VOLUME_DIRECTION    RANDOM_NATURAL           PR=0.995581152  ROC=0.9986169

In [32]:
# =============================================================================
# STAGE23 — AUTO-SEAL + PUSH REMAINING PLACEBO BATCH
#
# QUEUE THIS CELL WHILE THE CURRENT TRAINING CELL IS STILL RUNNING.
#
# ZERO MODEL FITS
# ZERO DATASET READS
# ZERO CACHE READS
#
# It will execute only after the current Stage23 placebo batch finishes.
#
# REQUIRED INPUT STATE:
#   Stage23-2A already sealed at:
#   660c39bfb8207db326c73f4e5637239ca2140139
#
# EXPECTED BATCH RESULT:
#   Stage23 total:        44 / 50
#   Primary boosted:      24 / 24
#   Placebo boosted:      20 / 20
#   Remaining stumps:      6 / 6
#
# FAIL CLOSED:
#   If the batch is incomplete, NOTHING is committed or pushed.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import shutil
import os


# =============================================================================
# CONFIG
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

BATCH = Path(
    "/kaggle/working/stage23_remaining_placebo_batch"
)

DEST_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

EXPECTED_PARENT = (
    "660c39bfb8207db326c73f4e5637239ca2140139"
)

PARENT_TAG = (
    "stage23-2a-placebo-counts-random-natural-v1"
)

SEAL_TAG = (
    "stage23-placebo-boosted-block-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: freeze complete placebo boosted block"
)


TASKS = [
    {
        "stage": "Stage23-2B",
        "placebo": "PLACEBO_COUNTS",
        "split": "CHRONOLOGICAL_NATURAL",
        "src": "stage23_2b_counts_chronological-natural",
        "dst": "chronological/placebo_counts",
    },
    {
        "stage": "Stage23-2C",
        "placebo": "PLACEBO_VOLUME_DIRECTION",
        "split": "RANDOM_NATURAL",
        "src": "stage23_2c_volume-direction_random-natural",
        "dst": "random/placebo_volume_direction",
    },
    {
        "stage": "Stage23-2D",
        "placebo": "PLACEBO_VOLUME_DIRECTION",
        "split": "CHRONOLOGICAL_NATURAL",
        "src": "stage23_2d_volume-direction_chronological-natural",
        "dst": "chronological/placebo_volume_direction",
    },
    {
        "stage": "Stage23-2E",
        "placebo": "PLACEBO_IAT",
        "split": "RANDOM_NATURAL",
        "src": "stage23_2e_iat_random-natural",
        "dst": "random/placebo_iat",
    },
    {
        "stage": "Stage23-2F",
        "placebo": "PLACEBO_IAT",
        "split": "CHRONOLOGICAL_NATURAL",
        "src": "stage23_2f_iat_chronological-natural",
        "dst": "chronological/placebo_iat",
    },
    {
        "stage": "Stage23-2G",
        "placebo": "PLACEBO_PACKET_SIZE",
        "split": "RANDOM_NATURAL",
        "src": "stage23_2g_packet-size_random-natural",
        "dst": "random/placebo_packet_size",
    },
    {
        "stage": "Stage23-2H",
        "placebo": "PLACEBO_PACKET_SIZE",
        "split": "CHRONOLOGICAL_NATURAL",
        "src": "stage23_2h_packet-size_chronological-natural",
        "dst": "chronological/placebo_packet_size",
    },
    {
        "stage": "Stage23-2I",
        "placebo": "PLACEBO_ACTIVITY",
        "split": "RANDOM_NATURAL",
        "src": "stage23_2i_activity_random-natural",
        "dst": "random/placebo_activity",
    },
    {
        "stage": "Stage23-2J",
        "placebo": "PLACEBO_ACTIVITY",
        "split": "CHRONOLOGICAL_NATURAL",
        "src": "stage23_2j_activity_chronological-natural",
        "dst": "chronological/placebo_activity",
    },
]


# =============================================================================
# HELPERS
# =============================================================================

def now():
    return datetime.now(timezone.utc).isoformat()


def git(*args, show=False):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if show and out:
        print(out)

    if p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n{out}"
        )

    return out


def sha256_file(path, chunk=32 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )


# =============================================================================
# START
# =============================================================================

print("=" * 100)
print("STAGE23 — AUTO-SEAL COMPLETE PLACEBO BOOSTED BLOCK")
print("=" * 100)
print()


# =============================================================================
# 1. VERIFY REPOSITORY
# =============================================================================

if not REPO.exists():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

status = git(
    "status",
    "--porcelain",
)


if branch != "main":
    raise RuntimeError(
        f"Expected main branch; found {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected repository HEAD.\n"
        f"expected: {EXPECTED_PARENT}\n"
        f"actual:   {head}"
    )

if status:
    raise RuntimeError(
        "Repository is dirty BEFORE sealing:\n"
        + status
    )

parent_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    PARENT_TAG,
)

if parent_tag_commit != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-2A parent tag does not resolve "
        "to expected parent."
    )

if git("tag", "--list", SEAL_TAG):
    raise RuntimeError(
        f"Seal tag already exists: {SEAL_TAG}"
    )


print("[OK] repository HEAD :", head)
print("[OK] parent tag      :", PARENT_TAG)
print("[OK] worktree        : CLEAN")


# =============================================================================
# 2. VERIFY BATCH STATE
# =============================================================================

BATCH_STATE = (
    BATCH
    / "batch_state.json"
)

SUMMARY_PATH = (
    BATCH
    / "remaining_placebo_batch_summary.json"
)


if not BATCH_STATE.exists():
    raise RuntimeError(
        "Batch state missing. "
        "Current training cell did not finish correctly."
    )

if not SUMMARY_PATH.exists():
    raise RuntimeError(
        "Batch summary missing. "
        "Current training cell did not finish correctly."
    )


batch_state = json.loads(
    BATCH_STATE.read_text(
        encoding="utf-8"
    )
)

summary = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


print()
print("=" * 100)
print("VERIFY COMPLETED BATCH")
print("=" * 100)
print()


if batch_state.get("status") != "COMPLETE_UNSEALED":
    raise RuntimeError(
        "Batch did not reach COMPLETE_UNSEALED.\n"
        f"status={batch_state.get('status')}"
    )

if summary.get("status") != (
    "ALL_REMAINING_PLACEBO_RESULTS_COMPLETE_UNSEALED"
):
    raise RuntimeError(
        "Unexpected batch summary status:\n"
        f"{summary.get('status')}"
    )

assert summary["new_model_fits"] == 18
assert summary["stage23_fits_before_batch"] == 26
assert summary["stage23_fits_after_batch"] == 44

assert summary["placebo_fits_before_batch"] == 2
assert summary["placebo_fits_after_batch"] == 20

assert summary["primary_boosted_fits"] == 24
assert summary["remaining_stump_fits"] == 6

assert summary["parquet_files_read"] == 0
assert summary["raw_mar1_accessed"] is False
assert summary["raw_mar2_accessed"] is False


print("[EXACT] batch fits          : 18 / 18")
print("[EXACT] placebo boosted     : 20 / 20")
print("[EXACT] primary boosted     : 24 / 24")
print("[EXACT] Stage23 total       : 44 / 50")
print("[EXACT] remaining stumps    : 6")
print("[EXACT] raw Mar1 access     : NO")
print("[EXACT] raw Mar2 access     : NO")
print("[EXACT] parquet reads       : 0")


# =============================================================================
# 3. VERIFY ALL NINE TASKS INDIVIDUALLY
# =============================================================================

print()
print("=" * 100)
print("VERIFY 9 COMPLETED TASKS")
print("=" * 100)
print()


verified_tasks = []


for task in TASKS:

    src = (
        BATCH
        / task["src"]
    )

    state_path = (
        src
        / "task_state.json"
    )

    if not src.exists():
        raise RuntimeError(
            f"Missing task directory:\n{src}"
        )

    if not state_path.exists():
        raise RuntimeError(
            f"Missing task state:\n{state_path}"
        )

    state = json.loads(
        state_path.read_text(
            encoding="utf-8"
        )
    )

    if state.get("stage") != task["stage"]:
        raise RuntimeError(
            f"{task['stage']}: stage mismatch"
        )

    if state.get("placebo") != task["placebo"]:
        raise RuntimeError(
            f"{task['stage']}: placebo mismatch"
        )

    if state.get("split") != task["split"]:
        raise RuntimeError(
            f"{task['stage']}: split mismatch"
        )

    if state.get("status") != "RESULT_COMPLETE_UNSEALED":
        raise RuntimeError(
            f"{task['stage']}: not RESULT_COMPLETE_UNSEALED"
        )

    if not state.get(
        "lightgbm_fit_complete",
        False,
    ):
        raise RuntimeError(
            f"{task['stage']}: LightGBM incomplete"
        )

    if not state.get(
        "xgboost_fit_complete",
        False,
    ):
        raise RuntimeError(
            f"{task['stage']}: XGBoost incomplete"
        )

    if not state.get(
        "result_complete",
        False,
    ):
        raise RuntimeError(
            f"{task['stage']}: result incomplete"
        )

    result_files = list(
        src.glob("*_result.json")
    )

    if len(result_files) != 1:
        raise RuntimeError(
            f"{task['stage']}: expected exactly 1 result JSON, "
            f"found {len(result_files)}"
        )

    result = json.loads(
        result_files[0].read_text(
            encoding="utf-8"
        )
    )

    assert result["stage"] == task["stage"]
    assert (
        result["status"]
        == "RESULT_COMPLETE_UNSEALED"
    )
    assert (
        result["cell"]["placebo"]
        == task["placebo"]
    )
    assert (
        result["cell"]["split"]
        == task["split"]
    )
    assert (
        result["cell"]["feature_count"]
        == 67
    )

    assert (
        result["models"]["lightgbm"]["backend"]
        == "cpu"
    )
    assert (
        result["models"]["xgboost"]["backend"]
        == "cuda"
    )

    assert (
        result["governance"]["new_model_fits_this_task"]
        == 2
    )
    assert (
        result["governance"]["raw_mar1_accessed"]
        is False
    )
    assert (
        result["governance"]["raw_mar2_accessed"]
        is False
    )
    assert (
        result["governance"]["threshold_optimization"]
        is False
    )
    assert (
        result["governance"]["subset_specific_tuning"]
        is False
    )

    checksums = (
        src
        / "checksums.sha256"
    )

    if not checksums.exists():
        raise RuntimeError(
            f"{task['stage']}: checksums.sha256 missing"
        )

    verified_tasks.append(
        {
            "stage": task["stage"],
            "placebo": task["placebo"],
            "split": task["split"],
            "source": str(src),
            "result_file": result_files[0].name,
            "result_sha256":
                sha256_file(result_files[0]),
            "checksums_sha256":
                sha256_file(checksums),
            "pr_auc":
                result["ranking_metrics"]["pr_auc"],
            "roc_auc":
                result["ranking_metrics"]["roc_auc"],
        }
    )

    print(
        f"[EXACT] {task['stage']}  "
        f"{task['placebo']} × {task['split']}"
    )


# =============================================================================
# 4. COPY TASK ARTIFACTS INTO REPOSITORY
# =============================================================================

print()
print("=" * 100)
print("COPYING SCIENTIFIC ARTIFACTS")
print("=" * 100)
print()


for task in TASKS:

    src = (
        BATCH
        / task["src"]
    )

    dst = (
        DEST_ROOT
        / task["dst"]
    )

    if dst.exists():
        raise RuntimeError(
            "Destination already exists; refusing overwrite:\n"
            f"{dst}"
        )

    dst.mkdir(
        parents=True,
        exist_ok=False,
    )

    allowed_files = []

    for p in src.iterdir():

        if not p.is_file():
            continue

        # Persist only scientific/recovery artifacts.
        if (
            p.name.endswith("_result.json")
            or p.name.endswith("_lightgbm_model.txt")
            or p.name.endswith("_xgboost_model.json")
            or p.name.endswith(
                "_validation_probabilities.npz"
            )
            or p.name == "checksums.sha256"
            or p.name == "task_state.json"
        ):
            allowed_files.append(p)

    if len(allowed_files) < 6:
        raise RuntimeError(
            f"{task['stage']}: unexpectedly few artifacts "
            f"({len(allowed_files)})"
        )

    for p in allowed_files:
        shutil.copy2(
            p,
            dst / p.name,
        )

    print(
        f"[COPIED] {task['stage']} -> "
        f"{dst.relative_to(REPO)}"
    )


# =============================================================================
# 5. COPY BATCH SUMMARY / STATE
# =============================================================================

BATCH_SEAL_DIR = (
    DEST_ROOT
    / "placebo_boosted_block_seal"
)

if BATCH_SEAL_DIR.exists():
    raise RuntimeError(
        f"Batch seal directory already exists:\n{BATCH_SEAL_DIR}"
    )

BATCH_SEAL_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

shutil.copy2(
    SUMMARY_PATH,
    BATCH_SEAL_DIR
    / SUMMARY_PATH.name,
)

shutil.copy2(
    BATCH_STATE,
    BATCH_SEAL_DIR
    / BATCH_STATE.name,
)


# =============================================================================
# 6. CREATE SEAL RECEIPT
# =============================================================================

seal_receipt = {
    "stage": "Stage23 placebo boosted block",
    "status": "SCIENTIFIC_RESULTS_FROZEN",
    "sealed_utc": now(),

    "parent_commit": EXPECTED_PARENT,
    "parent_tag": PARENT_TAG,

    "seal_tag": SEAL_TAG,

    "fit_accounting": {
        "primary_boosted": {
            "completed": 24,
            "expected": 24,
        },
        "placebo_boosted": {
            "completed": 20,
            "expected": 20,
        },
        "stumps": {
            "completed": 0,
            "expected": 6,
        },
        "stage23_total": {
            "completed": 44,
            "expected": 50,
        },
    },

    "batch": {
        "new_fits": 18,
        "tasks": verified_tasks,
        "summary_sha256":
            sha256_file(SUMMARY_PATH),
        "batch_state_sha256":
            sha256_file(BATCH_STATE),
    },

    "governance": {
        "lightgbm_backend": "cpu",
        "xgboost_backend": "cuda",
        "parquet_files_read": 0,
        "raw_mar1_accessed": False,
        "raw_mar2_accessed": False,
        "threshold_optimization": False,
        "subset_specific_tuning": False,
        "rebalancing": False,
        "placebo_definition_changed": False,
        "stage23_0_modified": False,
    },

    "next_authorized_model_work": {
        "block": "Stage23 depth-1 stump controls",
        "new_fits": 6,
        "stage23_after_completion": "50 / 50",
    },
}

write_json(
    BATCH_SEAL_DIR
    / "seal_receipt.json",
    seal_receipt,
)


# =============================================================================
# 7. README
# =============================================================================

README = """# Stage23 — Complete Placebo Boosted Block

This seal freezes the complete Stage23 matched-size placebo boosted-model
block.

## Fit accounting

- Primary boosted fits: 24 / 24
- Placebo boosted fits: 20 / 20
- Stage23 total after this seal: 44 / 50
- Remaining model fits: 6 depth-1 stump controls

The sealed placebo block contains:

1. PLACEBO_COUNTS
2. PLACEBO_VOLUME_DIRECTION
3. PLACEBO_IAT
4. PLACEBO_PACKET_SIZE
5. PLACEBO_ACTIVITY

Each placebo was evaluated under:

- RANDOM_NATURAL
- CHRONOLOGICAL_NATURAL

using the prospectively frozen Stage23 model specification.

No raw March 1 or March 2 data were accessed.
No threshold optimization, subset-specific tuning, rebalancing, or
placebo-definition changes were performed.

The next authorized model work is the six prospectively frozen
depth-1 stump controls.
"""

(
    BATCH_SEAL_DIR
    / "README.md"
).write_text(
    README,
    encoding="utf-8",
)


# =============================================================================
# 8. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

manifest_path = (
    BATCH_SEAL_DIR
    / "repository_checksums.sha256"
)

all_files = sorted(
    p
    for p in DEST_ROOT.rglob("*")
    if p.is_file()
    and p.name != "repository_checksums.sha256"
)

manifest_lines = []

for p in all_files:

    rel = p.relative_to(DEST_ROOT)

    manifest_lines.append(
        f"{sha256_file(p)}  {rel}"
    )

manifest_path.write_text(
    "\n".join(manifest_lines) + "\n",
    encoding="utf-8",
)

print()
print(
    "[OK] repository checksum manifest:",
    sha256_file(manifest_path),
)


# =============================================================================
# 9. STAGE ONLY THE PLACEBO BLOCK
# =============================================================================

rel_root = (
    DEST_ROOT.relative_to(REPO)
)

git(
    "add",
    "--",
    str(rel_root),
)

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()


if not staged:
    raise RuntimeError(
        "Nothing staged."
    )

for path in staged:

    if not path.startswith(
        str(rel_root) + "/"
    ):
        raise RuntimeError(
            "Unexpected staged file:\n"
            f"{path}"
        )


print()
print("[OK] staged files:", len(staged))


# =============================================================================
# 10. COMMIT
# =============================================================================

print()
print("=" * 100)
print("COMMIT")
print("=" * 100)
print()

git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)

sealed_commit = git(
    "rev-parse",
    "HEAD",
)

print()
print("[OK] sealed commit:", sealed_commit)


# =============================================================================
# 11. ANNOTATED TAG
# =============================================================================

git(
    "tag",
    "-a",
    SEAL_TAG,
    sealed_commit,
    "-m",
    (
        "Stage23 complete placebo boosted block: "
        "20/20 placebo fits, Stage23 44/50"
    ),
)


# =============================================================================
# 12. PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 100)
print("PUSH")
print("=" * 100)
print()

git(
    "push",
    "origin",
    "main",
    show=True,
)

git(
    "push",
    "origin",
    SEAL_TAG,
    show=True,
)


# =============================================================================
# 13. REMOTE VERIFICATION
# =============================================================================

remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_main_line:
    raise RuntimeError(
        "Could not resolve remote main."
    )

remote_main = (
    remote_main_line
    .split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_main}"
    )


remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEAL_TAG}^{{}}",
)

if not remote_tag_line:
    raise RuntimeError(
        "Could not resolve peeled remote tag."
    )

remote_tag_commit = (
    remote_tag_line
    .split()[0]
)


if remote_tag_commit != sealed_commit:
    raise RuntimeError(
        "Remote tag mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_tag_commit}"
    )


# =============================================================================
# 14. FINAL CLEAN STATE
# =============================================================================

final_status = git(
    "status",
    "--porcelain",
)

if final_status:
    raise RuntimeError(
        "Repository dirty AFTER seal:\n"
        + final_status
    )


print()
print("=" * 100)
print("STAGE23 PLACEBO BLOCK SEALED + PUSHED")
print("=" * 100)
print()
print("Commit:")
print(" ", sealed_commit)
print()
print("Tag:")
print(" ", SEAL_TAG)
print()
print("Remote main:")
print(" ", remote_main)
print()
print("Remote tag peeled commit:")
print(" ", remote_tag_commit)
print()
print("FIT ACCOUNTING")
print("  Primary boosted : 24 / 24")
print("  Placebo boosted : 20 / 20")
print("  Stage23 total   : 44 / 50")
print("  Remaining       : 6 stump fits")
print()
print("RESULTS ARE NOW SAFE ON GITHUB.")
print("=" * 100)

STAGE23 — AUTO-SEAL COMPLETE PLACEBO BOOSTED BLOCK

[OK] repository HEAD : 660c39bfb8207db326c73f4e5637239ca2140139
[OK] parent tag      : stage23-2a-placebo-counts-random-natural-v1
[OK] worktree        : CLEAN

VERIFY COMPLETED BATCH

[EXACT] batch fits          : 18 / 18
[EXACT] placebo boosted     : 20 / 20
[EXACT] primary boosted     : 24 / 24
[EXACT] Stage23 total       : 44 / 50
[EXACT] remaining stumps    : 6
[EXACT] raw Mar1 access     : NO
[EXACT] raw Mar2 access     : NO
[EXACT] parquet reads       : 0

VERIFY 9 COMPLETED TASKS

[EXACT] Stage23-2B  PLACEBO_COUNTS × CHRONOLOGICAL_NATURAL
[EXACT] Stage23-2C  PLACEBO_VOLUME_DIRECTION × RANDOM_NATURAL
[EXACT] Stage23-2D  PLACEBO_VOLUME_DIRECTION × CHRONOLOGICAL_NATURAL
[EXACT] Stage23-2E  PLACEBO_IAT × RANDOM_NATURAL
[EXACT] Stage23-2F  PLACEBO_IAT × CHRONOLOGICAL_NATURAL
[EXACT] Stage23-2G  PLACEBO_PACKET_SIZE × RANDOM_NATURAL
[EXACT] Stage23-2H  PLACEBO_PACKET_SIZE × CHRONOLOGICAL_NATURAL
[EXACT] Stage23-2I  PLACEBO_ACTIVITY ×

In [3]:
# =============================================================================
# STAGE23-3 — COMPLETE FROZEN DEPTH-1 STUMP CONTROL BLOCK
#
# SIX AND ONLY SIX NEW FITS:
#
# 3A  Dst Port           × RANDOM_NATURAL
# 3B  Dst Port           × CHRONOLOGICAL_NATURAL
# 3C  Init Fwd Win Byts  × RANDOM_NATURAL
# 3D  Init Fwd Win Byts  × CHRONOLOGICAL_NATURAL
# 3E  Fwd Seg Size Min   × RANDOM_NATURAL
# 3F  Fwd Seg Size Min   × CHRONOLOGICAL_NATURAL
#
# BEFORE: 44 / 50
# AFTER : 50 / 50
#
# Uses:
#   sklearn.tree.DecisionTreeClassifier(max_depth=1)
#   SimpleImputer(strategy="median")
#
# CPU ONLY.
# NO BOOSTED FITS.
# NO PARQUET.
# NO RAW MAR1.
# NO RAW MAR2.
#
# THREE independent feature workers run concurrently.
# Each worker runs RANDOM then CHRONO for its frozen feature.
#
# RESUME-SAFE:
#   completed result tasks are skipped.
#   ambiguous interrupted model fits FAIL CLOSED.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib.metadata as importlib_metadata
import hashlib
import json
import os
import time
import gc

import numpy as np
import joblib

from joblib import Parallel, delayed

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


# =============================================================================
# 0. FROZEN PATHS / IDENTIFIERS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

OUT = Path(
    "/kaggle/working/stage23_3_stump_controls"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_PARENT = (
    "eca21e3f6a918d59c64af92eb701361dde91ada7"
)

EXPECTED_PARENT_TAG = (
    "stage23-placebo-boosted-block-complete-v1"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

CACHE_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)


# =============================================================================
# 1. SEALED CACHE CONSTANTS
# =============================================================================

N = 14_412_403

RANDOM_TRAIN_ROWS = 11_529_922
RANDOM_VAL_ROWS = 2_882_481

RANDOM_TRAIN_ATTACK = 1_577_839
RANDOM_TRAIN_BENIGN = 9_952_083

RANDOM_VAL_ATTACK = 394_460
RANDOM_VAL_BENIGN = 2_488_021


CHRONO_TRAIN_ROWS = 13_818_623
CHRONO_VAL_ROWS = 593_780

CHRONO_TRAIN_ATTACK = 1_910_043
CHRONO_TRAIN_BENIGN = 11_908_580

CHRONO_VAL_ATTACK = 62_256
CHRONO_VAL_BENIGN = 531_524


CACHE_METADATA_SHA = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_MANIFEST_SHA = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

RANDOM_BITSET_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =============================================================================
# 2. FROZEN STUMP FEATURES
# =============================================================================

FEATURES = [
    {
        "feature": "Dst Port",
        "index": 0,
        "slug": "dst_port",
        "random_stage": "Stage23-3A",
        "chrono_stage": "Stage23-3B",
    },
    {
        "feature": "Init Fwd Win Byts",
        "index": 58,
        "slug": "init_fwd_win_byts",
        "random_stage": "Stage23-3C",
        "chrono_stage": "Stage23-3D",
    },
    {
        "feature": "Fwd Seg Size Min",
        "index": 61,
        "slug": "fwd_seg_size_min",
        "random_stage": "Stage23-3E",
        "chrono_stage": "Stage23-3F",
    },
]


# =============================================================================
# 3. HELPERS
# =============================================================================

def now():
    return datetime.now(timezone.utc).isoformat()


def run_git(*args):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n{out}"
        )

    return out


def sha256_file(path, chunk=32 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(path, obj):

    path = Path(path)

    tmp = Path(
        str(path) + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    tmp.replace(path)


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def metric_block(y, probability):

    y = np.asarray(
        y,
        dtype=np.uint8,
    )

    p = np.asarray(
        probability,
        dtype=np.float64,
    )

    prevalence = float(
        y.mean()
    )

    pr_auc = float(
        average_precision_score(
            y,
            p,
        )
    )

    roc_auc = float(
        roc_auc_score(
            y,
            p,
        )
    )

    pred = (
        p >= 0.50
    ).astype(
        np.uint8
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    ).ravel()

    return {
        "attack_prevalence":
            prevalence,

        "pr_auc":
            pr_auc,

        "pr_auc_minus_attack_prevalence":
            pr_auc - prevalence,

        "roc_auc":
            roc_auc,

        "accuracy":
            float(
                accuracy_score(
                    y,
                    pred,
                )
            ),

        "precision":
            float(
                precision_score(
                    y,
                    pred,
                    zero_division=0,
                )
            ),

        "recall":
            float(
                recall_score(
                    y,
                    pred,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    y,
                    pred,
                    zero_division=0,
                )
            ),

        "fpr":
            float(
                fp / (fp + tn)
            ),

        "fnr":
            float(
                fn / (fn + tp)
            ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


# =============================================================================
# 4. REPOSITORY PREFLIGHT
# =============================================================================

print("=" * 100)
print("STAGE23-3 — SIX FROZEN DEPTH-1 STUMPS")
print("=" * 100)
print()


branch = run_git(
    "branch",
    "--show-current",
)

head = run_git(
    "rev-parse",
    "HEAD",
)

status = run_git(
    "status",
    "--porcelain",
)


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if status:
    raise RuntimeError(
        "Repository dirty before stump execution:\n"
        + status
    )


if (
    run_git(
        "rev-list",
        "-n",
        "1",
        EXPECTED_PARENT_TAG,
    )
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Placebo-block seal tag mismatch."
    )


if (
    run_git(
        "rev-list",
        "-n",
        "1",
        PROTOCOL_TAG,
    )
    != PROTOCOL_COMMIT
):
    raise RuntimeError(
        "Stage23 protocol tag mismatch."
    )


if (
    run_git(
        "rev-list",
        "-n",
        "1",
        CACHE_TAG,
    )
    != CACHE_COMMIT
):
    raise RuntimeError(
        "Stage23 cache tag mismatch."
    )


print("[OK] HEAD          :", head)
print("[OK] placebo seal  :", EXPECTED_PARENT_TAG)
print("[OK] protocol      :", PROTOCOL_COMMIT)
print("[OK] cache seal    :", CACHE_COMMIT)
print("[OK] worktree      : CLEAN")


# =============================================================================
# 5. ENVIRONMENT
# =============================================================================

EXPECTED_VERSIONS = {
    "numpy": "2.0.2",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
}

print()
print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)
print()


for package, expected in EXPECTED_VERSIONS.items():

    actual = importlib_metadata.version(
        package
    )

    print(
        f"{package:<16}"
        f" expected={expected:<10}"
        f" actual={actual}"
    )

    if actual != expected:

        raise RuntimeError(
            f"{package} mismatch."
        )


# =============================================================================
# 6. VERIFY FROZEN STUMP SPEC
# =============================================================================

STUMP_SPEC_PATH = (
    PROTOCOL
    / "stump_spec.json"
)

SPLIT_SPEC_PATH = (
    PROTOCOL
    / "inherited_splits.json"
)


stump_spec = load_json(
    STUMP_SPEC_PATH
)

split_spec = load_json(
    SPLIT_SPEC_PATH
)


EXPECTED_PARAMS = {
    "ccp_alpha": 0.0,
    "class_weight": None,
    "criterion": "gini",
    "max_depth": 1,
    "max_features": None,
    "max_leaf_nodes": None,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "random_state": 42,
    "splitter": "best",
}


assert stump_spec["expected_fits"] == 6

assert stump_spec["features"] == [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]

assert stump_spec["splits"] == [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

assert (
    stump_spec["implementation"]
    == "sklearn.tree.DecisionTreeClassifier"
)

assert (
    stump_spec["library_version"]
    == "1.6.1"
)

assert (
    stump_spec["missing_value_policy"]["implementation"]
    == "sklearn.impute.SimpleImputer(strategy='median')"
)

assert (
    stump_spec["missing_value_policy"]["method"]
    == "TRAINING_MEDIAN_IMPUTATION_FOR_STUMP_ONLY"
)

assert (
    stump_spec["missing_value_policy"]["fit_scope"]
    == "fit on the stump training membership only; apply unchanged to validation"
)

assert (
    stump_spec["operating_threshold"]
    == 0.5
)

assert (
    stump_spec["parameters"]
    == EXPECTED_PARAMS
)


# Split-spec counts.

assert (
    split_spec["RANDOM_NATURAL"]["train"]["rows"]
    == RANDOM_TRAIN_ROWS
)

assert (
    split_spec["RANDOM_NATURAL"]["validation"]["rows"]
    == RANDOM_VAL_ROWS
)

assert (
    split_spec["CHRONOLOGICAL_NATURAL"]["train"]["rows"]
    == CHRONO_TRAIN_ROWS
)

assert (
    split_spec["CHRONOLOGICAL_NATURAL"]["validation"]["rows"]
    == CHRONO_VAL_ROWS
)


print()
print("[EXACT] six stump fits")
print("[EXACT] max_depth=1")
print("[EXACT] training-only median imputation")
print("[EXACT] threshold=0.50")
print("[EXACT] random + chronological natural splits")


# =============================================================================
# 7. VERIFY SEALED CACHE
# =============================================================================

metadata_path = (
    CACHE
    / "metadata.json"
)

manifest_path = (
    CACHE
    / "data_checksums.sha256"
)

labels_path = (
    CACHE
    / "binary_label.uint8.dat"
)

positions_path = (
    CACHE
    / "clean_position.int64.dat"
)

bitset_path = (
    CACHE
    / "random_validation.packbits"
)


if sha256_file(metadata_path) != CACHE_METADATA_SHA:
    raise RuntimeError(
        "Cache metadata SHA mismatch."
    )

if sha256_file(manifest_path) != CACHE_MANIFEST_SHA:
    raise RuntimeError(
        "Cache checksum-manifest SHA mismatch."
    )

if sha256_file(bitset_path) != RANDOM_BITSET_SHA:
    raise RuntimeError(
        "Random-validation bitset SHA mismatch."
    )


manifest = {}

for line in manifest_path.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, rel = line.split(
        "  ",
        1,
    )

    manifest[rel] = digest


required_files = [
    "binary_label.uint8.dat",
    "clean_position.int64.dat",
    "features/feature_00.float64.dat",
    "features/feature_58.float64.dat",
    "features/feature_61.float64.dat",
]


print()
print("=" * 100)
print("CACHE PHYSICAL SHA VERIFICATION")
print("=" * 100)
print()


for rel in required_files:

    p = (
        CACHE
        / rel
    )

    if not p.exists():

        raise RuntimeError(
            f"Missing cache file: {rel}"
        )

    actual = sha256_file(
        p
    )

    expected = manifest.get(
        rel
    )

    if expected is None:

        raise RuntimeError(
            f"No frozen SHA for {rel}"
        )

    if actual != expected:

        raise RuntimeError(
            f"SHA mismatch: {rel}"
        )

    print(
        "[EXACT]",
        rel,
    )


# =============================================================================
# 8. VERIFY SPLIT MEMBERSHIP / CLASS COUNTS
# =============================================================================

labels = np.memmap(
    labels_path,
    dtype=np.uint8,
    mode="r",
    shape=(N,),
)


packed = np.fromfile(
    bitset_path,
    dtype=np.uint8,
)

random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N].astype(
    bool
)


if int(
    random_val_mask.sum()
) != RANDOM_VAL_ROWS:

    raise RuntimeError(
        "Random validation population mismatch."
    )


random_train_mask = (
    ~random_val_mask
)


# RANDOM counts.

random_train_y = labels[
    random_train_mask
]

random_val_y = labels[
    random_val_mask
]


assert len(
    random_train_y
) == RANDOM_TRAIN_ROWS

assert len(
    random_val_y
) == RANDOM_VAL_ROWS


assert int(
    random_train_y.sum()
) == RANDOM_TRAIN_ATTACK

assert (
    RANDOM_TRAIN_ROWS
    - int(random_train_y.sum())
    == RANDOM_TRAIN_BENIGN
)

assert int(
    random_val_y.sum()
) == RANDOM_VAL_ATTACK

assert (
    RANDOM_VAL_ROWS
    - int(random_val_y.sum())
    == RANDOM_VAL_BENIGN
)


# CHRONO:
# Canonical development-cache ordering places frozen chronological
# training membership first and Feb28 validation membership last.
#
# We fail closed by checking the exact frozen class populations.

chrono_train_y = labels[
    :CHRONO_TRAIN_ROWS
]

chrono_val_y = labels[
    CHRONO_TRAIN_ROWS:
]


assert len(
    chrono_train_y
) == CHRONO_TRAIN_ROWS

assert len(
    chrono_val_y
) == CHRONO_VAL_ROWS


assert int(
    chrono_train_y.sum()
) == CHRONO_TRAIN_ATTACK

assert (
    CHRONO_TRAIN_ROWS
    - int(chrono_train_y.sum())
    == CHRONO_TRAIN_BENIGN
)

assert int(
    chrono_val_y.sum()
) == CHRONO_VAL_ATTACK

assert (
    CHRONO_VAL_ROWS
    - int(chrono_val_y.sum())
    == CHRONO_VAL_BENIGN
)


print()
print("[EXACT] RANDOM_NATURAL populations")
print(
    "        train:",
    RANDOM_TRAIN_ROWS,
    "validation:",
    RANDOM_VAL_ROWS,
)

print("[EXACT] CHRONOLOGICAL_NATURAL populations")
print(
    "        train:",
    CHRONO_TRAIN_ROWS,
    "validation:",
    CHRONO_VAL_ROWS,
)


# Release temporary copied labels before workers.
del random_train_y
del random_val_y
del chrono_train_y
del chrono_val_y
del labels

gc.collect()


# =============================================================================
# 9. TASK DEFINITIONS
# =============================================================================

TASKS = []

for feature_cfg in FEATURES:

    TASKS.append(
        {
            **feature_cfg,
            "stage":
                feature_cfg["random_stage"],
            "split":
                "RANDOM_NATURAL",
        }
    )

    TASKS.append(
        {
            **feature_cfg,
            "stage":
                feature_cfg["chrono_stage"],
            "split":
                "CHRONOLOGICAL_NATURAL",
        }
    )


# =============================================================================
# 10. SINGLE STUMP TASK
# =============================================================================

def execute_stump(task):

    feature = task["feature"]
    feature_index = task["index"]
    feature_slug = task["slug"]
    split = task["split"]
    stage = task["stage"]

    split_slug = (
        "random_natural"
        if split == "RANDOM_NATURAL"
        else "chronological_natural"
    )

    task_dir = (
        OUT
        / feature_slug
        / split_slug
    )

    task_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    state_path = (
        task_dir
        / "execution_state.json"
    )

    model_path = (
        task_dir
        / f"{feature_slug}_{split_slug}_stump.joblib"
    )

    prob_path = (
        task_dir
        / f"{feature_slug}_{split_slug}_validation_probabilities.npz"
    )

    result_path = (
        task_dir
        / f"{stage.lower().replace('-', '_')}_{feature_slug}_{split_slug}_result.json"
    )

    checksum_path = (
        task_dir
        / "checksums.sha256"
    )


    # -------------------------------------------------------------------------
    # Existing state / resume
    # -------------------------------------------------------------------------

    if state_path.exists():

        state = load_json(
            state_path
        )

    else:

        state = {
            "stage": stage,
            "feature": feature,
            "feature_index": feature_index,
            "split": split,
            "status": "INITIALIZED",

            "imputer_fit_complete": False,
            "stump_fit_started": False,
            "stump_fit_complete": False,
            "result_complete": False,

            "new_stump_fits": 0,

            "created_utc": now(),
        }

        write_json(
            state_path,
            state,
        )


    # Fully completed tasks are safe to skip.

    if state.get(
        "result_complete",
        False,
    ):

        if not result_path.exists():

            raise RuntimeError(
                f"{stage}: result state complete but result file missing."
            )

        result = load_json(
            result_path
        )

        print(
            f"[SKIP COMPLETE] {stage} — "
            f"{feature} × {split}",
            flush=True,
        )

        return result


    # Ambiguous interrupted model fit:
    # do NOT blindly rerun.

    if (
        state.get(
            "stump_fit_started",
            False,
        )
        and not state.get(
            "stump_fit_complete",
            False,
        )
    ):

        raise RuntimeError(
            f"{stage}: stump fit was started but completion is ambiguous. "
            "DO NOT RERUN BLINDLY."
        )


    # -------------------------------------------------------------------------
    # Open exact feature + labels
    # -------------------------------------------------------------------------

    feature_path = (
        CACHE
        / "features"
        / f"feature_{feature_index:02d}.float64.dat"
    )


    x_all = np.memmap(
        feature_path,
        dtype="<f8",
        mode="r",
        shape=(N,),
    )

    y_all = np.memmap(
        labels_path,
        dtype=np.uint8,
        mode="r",
        shape=(N,),
    )


    if split == "RANDOM_NATURAL":

        packed_local = np.fromfile(
            bitset_path,
            dtype=np.uint8,
        )

        val_mask = np.unpackbits(
            packed_local,
            bitorder="little",
        )[:N].astype(
            bool
        )

        train_mask = ~val_mask

        X_train_raw = np.asarray(
            x_all[train_mask],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        X_val_raw = np.asarray(
            x_all[val_mask],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        y_train = np.asarray(
            y_all[train_mask],
            dtype=np.uint8,
        )

        y_val = np.asarray(
            y_all[val_mask],
            dtype=np.uint8,
        )

        pos_all = np.memmap(
            positions_path,
            dtype="<i8",
            mode="r",
            shape=(N,),
        )

        val_positions = np.asarray(
            pos_all[val_mask],
            dtype=np.int64,
        )

        del val_mask
        del train_mask
        del packed_local
        del pos_all


    else:

        X_train_raw = np.asarray(
            x_all[:CHRONO_TRAIN_ROWS],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        X_val_raw = np.asarray(
            x_all[CHRONO_TRAIN_ROWS:],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        y_train = np.asarray(
            y_all[:CHRONO_TRAIN_ROWS],
            dtype=np.uint8,
        )

        y_val = np.asarray(
            y_all[CHRONO_TRAIN_ROWS:],
            dtype=np.uint8,
        )

        pos_all = np.memmap(
            positions_path,
            dtype="<i8",
            mode="r",
            shape=(N,),
        )

        val_positions = np.asarray(
            pos_all[CHRONO_TRAIN_ROWS:],
            dtype=np.int64,
        )

        del pos_all


    del x_all
    del y_all

    gc.collect()


    # -------------------------------------------------------------------------
    # Fit new stump OR recover already completed stump artifact
    # -------------------------------------------------------------------------

    if not state.get(
        "stump_fit_complete",
        False,
    ):

        print(
            f"[START] {stage} — "
            f"{feature} × {split}",
            flush=True,
        )


        # Frozen training-only median imputation.

        imputer = SimpleImputer(
            strategy="median"
        )

        t0_imputer = time.perf_counter()

        X_train = imputer.fit_transform(
            X_train_raw
        )

        X_val = imputer.transform(
            X_val_raw
        )

        imputer_seconds = (
            time.perf_counter()
            - t0_imputer
        )

        median = float(
            imputer.statistics_[0]
        )


        state["imputer_fit_complete"] = True
        state["imputer_seconds"] = imputer_seconds
        state["missing_imputation_median"] = median
        state["status"] = "IMPUTER_COMPLETE"

        write_json(
            state_path,
            state,
        )


        # Frozen stump.

        stump = DecisionTreeClassifier(
            **EXPECTED_PARAMS
        )


        state["stump_fit_started"] = True
        state["stump_fit_started_utc"] = now()
        state["status"] = "STUMP_FIT_STARTED"

        write_json(
            state_path,
            state,
        )


        t0 = time.perf_counter()

        stump.fit(
            X_train,
            y_train,
        )

        stump_seconds = (
            time.perf_counter()
            - t0
        )


        # Mark completion IMMEDIATELY after successful fit.

        state["stump_fit_complete"] = True
        state["stump_fit_completed_utc"] = now()
        state["stump_fit_seconds"] = stump_seconds
        state["new_stump_fits"] = 1
        state["status"] = "STUMP_FIT_COMPLETE_ARTIFACT_PENDING"

        write_json(
            state_path,
            state,
        )


        # Persist exact fitted imputer + stump.

        joblib.dump(
            {
                "stage": stage,
                "feature": feature,
                "feature_index": feature_index,
                "split": split,
                "imputer": imputer,
                "stump": stump,
            },
            model_path,
            compress=0,
        )


        state["model_sha256"] = sha256_file(
            model_path
        )

        state["status"] = "STUMP_ARTIFACT_SAVED"

        write_json(
            state_path,
            state,
        )


    else:

        # Fit already consumed; recover artifact only.

        if not model_path.exists():

            raise RuntimeError(
                f"{stage}: stump fit already consumed but "
                "model artifact missing. Refusing rerun."
            )


        artifact = joblib.load(
            model_path
        )

        imputer = artifact["imputer"]
        stump = artifact["stump"]

        X_train = imputer.transform(
            X_train_raw
        )

        X_val = imputer.transform(
            X_val_raw
        )

        median = float(
            imputer.statistics_[0]
        )


    # Raw arrays no longer needed.

    del X_train_raw
    del X_val_raw

    gc.collect()


    # -------------------------------------------------------------------------
    # Tree structure
    # -------------------------------------------------------------------------

    tree = stump.tree_

    left_node = int(
        tree.children_left[0]
    )

    right_node = int(
        tree.children_right[0]
    )


    if (
        left_node < 0
        or right_node < 0
    ):

        raise RuntimeError(
            f"{stage}: frozen depth-1 control produced no root split."
        )


    split_threshold = float(
        tree.threshold[0]
    )


    left_mask_train = (
        X_train[:, 0]
        <= split_threshold
    )

    right_mask_train = (
        ~left_mask_train
    )


    left_support = int(
        left_mask_train.sum()
    )

    right_support = int(
        right_mask_train.sum()
    )


    if (
        left_support
        + right_support
        != len(y_train)
    ):

        raise RuntimeError(
            f"{stage}: leaf support accounting mismatch."
        )


    left_benign = int(
        np.sum(
            y_train[
                left_mask_train
            ] == 0
        )
    )

    left_attack = int(
        np.sum(
            y_train[
                left_mask_train
            ] == 1
        )
    )


    right_benign = int(
        np.sum(
            y_train[
                right_mask_train
            ] == 0
        )
    )

    right_attack = int(
        np.sum(
            y_train[
                right_mask_train
            ] == 1
        )
    )


    left_attack_probability = float(
        left_attack
        / left_support
    )

    right_attack_probability = float(
        right_attack
        / right_support
    )


    left_predicts_attack = (
        left_attack_probability
        >= 0.50
    )

    right_predicts_attack = (
        right_attack_probability
        >= 0.50
    )


    if (
        left_predicts_attack
        and right_predicts_attack
    ):

        branch_predicting_attack = "BOTH"

    elif left_predicts_attack:

        branch_predicting_attack = "LEFT"

    elif right_predicts_attack:

        branch_predicting_attack = "RIGHT"

    else:

        branch_predicting_attack = "NEITHER"


    # -------------------------------------------------------------------------
    # Validation probability / metrics
    # -------------------------------------------------------------------------

    class_index = np.where(
        stump.classes_ == 1
    )[0]

    if len(class_index) != 1:

        raise RuntimeError(
            f"{stage}: attack class not uniquely represented."
        )

    attack_index = int(
        class_index[0]
    )


    probability = np.asarray(
        stump.predict_proba(
            X_val
        )[:, attack_index],
        dtype=np.float64,
    )


    metrics = metric_block(
        y_val,
        probability,
    )


    # -------------------------------------------------------------------------
    # Persist validation predictions
    # -------------------------------------------------------------------------

    # Uncompressed NPZ for speed.
    np.savez(
        prob_path,
        clean_position=np.asarray(
            val_positions,
            dtype=np.int64,
        ),
        binary_label=np.asarray(
            y_val,
            dtype=np.uint8,
        ),
        probability=probability,
    )


    # -------------------------------------------------------------------------
    # Result
    # -------------------------------------------------------------------------

    result = {
        "stage": stage,
        "status": "RESULT_COMPLETE_UNSEALED",
        "created_utc": now(),

        "fit_accounting": {
            "new_stump_fits_this_task": 1,
            "stage23_total_before_stump_block": 44,
            "stump_block_expected_fits": 6,
        },

        "feature": {
            "name": feature,
            "index": feature_index,
        },

        "split": split,

        "training": {
            "rows": int(
                len(y_train)
            ),

            "attack": int(
                y_train.sum()
            ),

            "benign": int(
                len(y_train)
                - y_train.sum()
            ),
        },

        "validation": {
            "rows": int(
                len(y_val)
            ),

            "attack": int(
                y_val.sum()
            ),

            "benign": int(
                len(y_val)
                - y_val.sum()
            ),
        },

        "imputation": {
            "implementation":
                "sklearn.impute.SimpleImputer(strategy='median')",

            "fit_scope":
                "TRAINING_MEMBERSHIP_ONLY",

            "missing_imputation_median":
                median,

            "seconds":
                float(
                    state.get(
                        "imputer_seconds",
                        0.0,
                    )
                ),
        },

        "stump": {
            "implementation":
                "sklearn.tree.DecisionTreeClassifier",

            "parameters":
                EXPECTED_PARAMS,

            "fit_seconds":
                float(
                    state[
                        "stump_fit_seconds"
                    ]
                ),

            "selected_feature":
                feature,

            "split_threshold":
                split_threshold,

            "left_leaf_support":
                left_support,

            "right_leaf_support":
                right_support,

            "left_leaf_class_distribution": {
                "benign_count":
                    left_benign,

                "attack_count":
                    left_attack,

                "benign_proportion":
                    float(
                        left_benign
                        / left_support
                    ),

                "attack_proportion":
                    left_attack_probability,
            },

            "right_leaf_class_distribution": {
                "benign_count":
                    right_benign,

                "attack_count":
                    right_attack,

                "benign_proportion":
                    float(
                        right_benign
                        / right_support
                    ),

                "attack_proportion":
                    right_attack_probability,
            },

            "branch_predicting_attack":
                branch_predicting_attack,
        },

        "metrics": {
            "attack_prevalence":
                metrics[
                    "attack_prevalence"
                ],

            "pr_auc":
                metrics[
                    "pr_auc"
                ],

            "pr_auc_minus_attack_prevalence":
                metrics[
                    "pr_auc_minus_attack_prevalence"
                ],

            "roc_auc":
                metrics[
                    "roc_auc"
                ],

            "threshold_0_50": {
                "accuracy":
                    metrics[
                        "accuracy"
                    ],

                "precision":
                    metrics[
                        "precision"
                    ],

                "recall":
                    metrics[
                        "recall"
                    ],

                "f1":
                    metrics[
                        "f1"
                    ],

                "fpr":
                    metrics[
                        "fpr"
                    ],

                "fnr":
                    metrics[
                        "fnr"
                    ],

                "tn":
                    metrics[
                        "tn"
                    ],

                "fp":
                    metrics[
                        "fp"
                    ],

                "fn":
                    metrics[
                        "fn"
                    ],

                "tp":
                    metrics[
                        "tp"
                    ],
            },
        },

        "artifacts": {
            "model":
                model_path.name,

            "model_sha256":
                sha256_file(
                    model_path
                ),

            "validation_probabilities":
                prob_path.name,

            "validation_probabilities_sha256":
                sha256_file(
                    prob_path
                ),

            "probability_dtype":
                "float64",
        },

        "governance": {
            "parquet_files_read": 0,
            "raw_mar1_accessed": False,
            "raw_mar2_accessed": False,
            "threshold_optimization": False,
            "boosted_model_fit": False,
            "stage23_0_modified": False,
            "stump_spec_changed": False,
        },
    }


    write_json(
        result_path,
        result,
    )


    checksum_path.write_text(
        "\n".join(
            [
                f"{sha256_file(result_path)}  {result_path.name}",
                f"{sha256_file(model_path)}  {model_path.name}",
                f"{sha256_file(prob_path)}  {prob_path.name}",
            ]
        ) + "\n",
        encoding="utf-8",
    )


    state["result_complete"] = True
    state["result_sha256"] = sha256_file(
        result_path
    )

    state["validation_probabilities_sha256"] = sha256_file(
        prob_path
    )

    state["checksums_sha256"] = sha256_file(
        checksum_path
    )

    state["status"] = "RESULT_COMPLETE_UNSEALED"
    state["completed_utc"] = now()

    write_json(
        state_path,
        state,
    )


    print(
        f"[COMPLETE] {stage} — "
        f"{feature} × {split} — "
        f"PR={metrics['pr_auc']:.9f} "
        f"ROC={metrics['roc_auc']:.9f}",
        flush=True,
    )


    del X_train
    del X_val
    del y_train
    del y_val
    del val_positions
    del probability
    del left_mask_train
    del right_mask_train

    gc.collect()


    return result


# =============================================================================
# 11. RUN THREE FEATURES IN PARALLEL
#
# Each feature worker executes its RANDOM + CHRONO task sequentially.
# This keeps feature-level work independent and limits memory pressure.
# =============================================================================

def execute_feature_pair(feature_cfg):

    random_task = {
        **feature_cfg,
        "stage":
            feature_cfg[
                "random_stage"
            ],
        "split":
            "RANDOM_NATURAL",
    }

    chrono_task = {
        **feature_cfg,
        "stage":
            feature_cfg[
                "chrono_stage"
            ],
        "split":
            "CHRONOLOGICAL_NATURAL",
    }


    random_result = execute_stump(
        random_task
    )

    chrono_result = execute_stump(
        chrono_task
    )


    return {
        "feature":
            feature_cfg[
                "feature"
            ],

        "random":
            random_result,

        "chronological":
            chrono_result,
    }


print()
print("=" * 100)
print("RUNNING 3 FEATURE WORKERS IN PARALLEL")
print("=" * 100)
print()
print("Worker 1: Dst Port")
print("Worker 2: Init Fwd Win Byts")
print("Worker 3: Fwd Seg Size Min")
print()
print("Each worker: RANDOM -> CHRONO")
print()


pairs = Parallel(
    n_jobs=3,
    backend="loky",
    verbose=10,
)(
    delayed(
        execute_feature_pair
    )(
        cfg
    )
    for cfg in FEATURES
)


# =============================================================================
# 12. CONSOLIDATE / VERIFY SIX FITS
# =============================================================================

all_results = []

for pair in pairs:

    all_results.append(
        pair["random"]
    )

    all_results.append(
        pair["chronological"]
    )


if len(
    all_results
) != 6:

    raise RuntimeError(
        "Expected six stump results."
    )


for result in all_results:

    if (
        result[
            "status"
        ]
        != "RESULT_COMPLETE_UNSEALED"
    ):

        raise RuntimeError(
            "Incomplete stump result."
        )


# =============================================================================
# 13. RANDOM -> CHRONO DEGRADATION
# =============================================================================

degradation = {}


for pair in pairs:

    feature = pair[
        "feature"
    ]

    r = pair[
        "random"
    ][
        "metrics"
    ]

    c = pair[
        "chronological"
    ][
        "metrics"
    ]


    degradation[
        feature
    ] = {
        "definition":
            "RANDOM_NATURAL metric minus CHRONOLOGICAL_NATURAL metric",

        "pr_auc":
            float(
                r["pr_auc"]
                - c["pr_auc"]
            ),

        "roc_auc":
            float(
                r["roc_auc"]
                - c["roc_auc"]
            ),

        "f1_at_0_50":
            float(
                r[
                    "threshold_0_50"
                ][
                    "f1"
                ]
                - c[
                    "threshold_0_50"
                ][
                    "f1"
                ]
            ),

        "recall_at_0_50":
            float(
                r[
                    "threshold_0_50"
                ][
                    "recall"
                ]
                - c[
                    "threshold_0_50"
                ][
                    "recall"
                ]
            ),

        "fpr_at_0_50":
            float(
                r[
                    "threshold_0_50"
                ][
                    "fpr"
                ]
                - c[
                    "threshold_0_50"
                ][
                    "fpr"
                ]
            ),
    }


# =============================================================================
# 14. BLOCK SUMMARY
# =============================================================================

summary = {
    "stage":
        "Stage23-3 stump control block",

    "status":
        "ALL_SIX_STUMP_RESULTS_COMPLETE_UNSEALED",

    "completed_utc":
        now(),

    "parent_commit":
        EXPECTED_PARENT,

    "parent_tag":
        EXPECTED_PARENT_TAG,

    "fit_accounting": {
        "primary_boosted":
            "24 / 24",

        "placebo_boosted":
            "20 / 20",

        "stump_controls":
            "6 / 6",

        "stage23_total":
            "50 / 50",

        "new_fits_this_cell":
            6,

        "new_boosted_fits_this_cell":
            0,
    },

    "random_to_chronological_degradation":
        degradation,

    "results":
        all_results,

    "governance": {
        "cache":
            "stage23_execution_cache_v1",

        "parquet_files_read":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "threshold_optimization":
            False,

        "new_boosted_models":
            0,

        "stage23_0_modified":
            False,

        "stump_spec_changed":
            False,
    },
}


summary_path = (
    OUT
    / "stage23_3_stump_block_summary.json"
)

write_json(
    summary_path,
    summary,
)


# =============================================================================
# 15. BLOCK CHECKSUM MANIFEST
# =============================================================================

manifest_out = (
    OUT
    / "checksums.sha256"
)


files_to_hash = sorted(
    p
    for p in OUT.rglob("*")
    if p.is_file()
    and p.name != "checksums.sha256"
)


manifest_out.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.relative_to(OUT)}"
        for p in files_to_hash
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 16. FINAL OUTPUT
# =============================================================================

print()
print("=" * 100)
print("STAGE23 MODEL-FIT BUDGET COMPLETE — RESULT UNSEALED")
print("=" * 100)
print()


stage_order = {
    "Stage23-3A": 0,
    "Stage23-3B": 1,
    "Stage23-3C": 2,
    "Stage23-3D": 3,
    "Stage23-3E": 4,
    "Stage23-3F": 5,
}


for result in sorted(
    all_results,
    key=lambda x: stage_order[
        x["stage"]
    ],
):

    m = result[
        "metrics"
    ]

    s = result[
        "stump"
    ]

    print(
        f"{result['stage']:<11} "
        f"{result['feature']['name']:<20} "
        f"{result['split']:<23}"
    )

    print(
        f"  median    : "
        f"{result['imputation']['missing_imputation_median']}"
    )

    print(
        f"  split     : "
        f"{s['split_threshold']}"
    )

    print(
        f"  attack arm: "
        f"{s['branch_predicting_attack']}"
    )

    print(
        f"  PR-AUC    : "
        f"{m['pr_auc']:.12f}"
    )

    print(
        f"  ROC-AUC   : "
        f"{m['roc_auc']:.12f}"
    )

    print(
        f"  F1@.50    : "
        f"{m['threshold_0_50']['f1']:.12f}"
    )

    print()


print("=" * 100)
print("RANDOM -> CHRONOLOGICAL DEGRADATION")
print("=" * 100)
print()


for feature, d in degradation.items():

    print(feature)

    print(
        f"  PR-AUC : "
        f"{d['pr_auc']:+.12f}"
    )

    print(
        f"  ROC-AUC: "
        f"{d['roc_auc']:+.12f}"
    )

    print()


print("=" * 100)
print("FIT ACCOUNTING")
print("=" * 100)
print()
print("Primary boosted : 24 / 24 COMPLETE")
print("Placebo boosted : 20 / 20 COMPLETE")
print("Stump controls  :  6 /  6 COMPLETE")
print("------------------------------------")
print("STAGE23 FITS    : 50 / 50 COMPLETE")
print()
print("New fits this cell        : 6")
print("New boosted fits this cell: 0")
print()
print("Parquet reads : 0")
print("Raw Mar1 read : NO")
print("Raw Mar2 read : NO")
print()
print("Summary:")
print(" ", summary_path)
print(" SHA256:")
print(" ", sha256_file(summary_path))
print()
print("Checksum manifest:")
print(" ", manifest_out)
print(" SHA256:")
print(" ", sha256_file(manifest_out))
print()
print("NO GIT COMMIT CREATED.")
print("NO GIT TAG CREATED.")
print()
print("NEXT ACTION:")
print("  Seal and push the complete Stage23 stump block / 50-of-50 model-fit state.")
print("=" * 100)

STAGE23-3 — SIX FROZEN DEPTH-1 STUMPS

[OK] HEAD          : eca21e3f6a918d59c64af92eb701361dde91ada7
[OK] placebo seal  : stage23-placebo-boosted-block-complete-v1
[OK] protocol      : 2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] cache seal    : 66e1c6382caa63699766e605df63df02beff894f
[OK] worktree      : CLEAN

ENVIRONMENT

numpy            expected=2.0.2      actual=2.0.2
scipy            expected=1.16.3     actual=1.16.3
scikit-learn     expected=1.6.1      actual=1.6.1
joblib           expected=1.5.3      actual=1.5.3

[EXACT] six stump fits
[EXACT] max_depth=1
[EXACT] training-only median imputation
[EXACT] threshold=0.50
[EXACT] random + chronological natural splits


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/stage23_execution_cache_v1/metadata.json'

In [2]:
# =============================================================================
# STAGE23-3 — SESSION RESET RECOVERY
#
# ZERO FITS
# Reclone exact 44/50 sealed state.
# Then locate/check Stage23 execution cache.
# =============================================================================

from pathlib import Path
import subprocess
import shutil
import os

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

EXPECTED_HEAD = "eca21e3f6a918d59c64af92eb701361dde91ada7"
EXPECTED_TAG = "stage23-placebo-boosted-block-complete-v1"

print("=" * 100)
print("STAGE23-3 — KAGGLE SESSION RESET RECOVERY")
print("=" * 100)
print()

# -----------------------------------------------------------------------------
# 1. Reclone repository
# -----------------------------------------------------------------------------

if REPO.exists():
    shutil.rmtree(REPO)

print("[INFO] Re-cloning repository with live progress...")
print()

p = subprocess.run(
    [
        "git",
        "clone",
        "--progress",
        "--no-single-branch",
        REPO_URL,
        str(REPO),
    ],
    text=True,
)

if p.returncode != 0:
    raise RuntimeError("git clone failed")

subprocess.run(
    ["git", "fetch", "--tags", "--force"],
    cwd=REPO,
    check=True,
)

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

tag_commit = subprocess.check_output(
    ["git", "rev-list", "-n", "1", EXPECTED_TAG],
    cwd=REPO,
    text=True,
).strip()

status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
).strip()

print()
print("=" * 100)
print("REPOSITORY VERIFICATION")
print("=" * 100)
print()

print("HEAD:")
print(" ", head)

print("44/50 seal tag:")
print(" ", tag_commit)

print("Worktree:")
print(" ", "CLEAN" if not status else status)

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"HEAD mismatch\nexpected={EXPECTED_HEAD}\nactual={head}"
    )

if tag_commit != EXPECTED_HEAD:
    raise RuntimeError(
        f"Tag mismatch\nexpected={EXPECTED_HEAD}\nactual={tag_commit}"
    )

if status:
    raise RuntimeError("Repository is unexpectedly dirty.")

print()
print("[OK] Exact Stage23 44/50 state restored.")


# -----------------------------------------------------------------------------
# 2. Search for execution cache
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("SEARCHING FOR STAGE23 EXECUTION CACHE")
print("=" * 100)
print()

candidate_paths = [
    Path("/kaggle/working/stage23_execution_cache_v1"),
    Path("/kaggle/input/stage23-execution-cache-v1"),
    Path("/kaggle/input/stage23_execution_cache_v1"),
]

# Also inspect Kaggle inputs for anything cache-like.
input_root = Path("/kaggle/input")

if input_root.exists():
    for p in input_root.iterdir():
        name = p.name.lower()
        if "stage23" in name and "cache" in name:
            candidate_paths.append(p)

seen = set()
found = []

for p in candidate_paths:
    key = str(p.resolve()) if p.exists() else str(p)

    if key in seen:
        continue

    seen.add(key)

    if p.exists():
        found.append(p)
        print("[FOUND]", p)
    else:
        print("[MISS ]", p)


# -----------------------------------------------------------------------------
# 3. Check exact required stump cache files
# -----------------------------------------------------------------------------

REQUIRED = [
    "features/feature_00.float64.dat",
    "features/feature_58.float64.dat",
    "features/feature_61.float64.dat",
    "binary_label.uint8.dat",
    "clean_position.int64.dat",
    "random_validation.packbits",
    "metadata.json",
    "data_checksums.sha256",
]

usable_cache = None

print()
print("=" * 100)
print("CACHE CONTENT CHECK")
print("=" * 100)
print()

for root in found:

    print()
    print("Candidate:", root)

    ok = True

    for rel in REQUIRED:

        path = root / rel

        if path.exists():
            print(
                f"  [FOUND] {rel:<42}"
                f"{path.stat().st_size / (1024**2):>10.2f} MiB"
            )
        else:
            print(f"  [MISS ] {rel}")
            ok = False

    if ok:
        usable_cache = root
        break


# -----------------------------------------------------------------------------
# 4. Final decision
# -----------------------------------------------------------------------------

print()
print("=" * 100)
print("RECOVERY STATE")
print("=" * 100)
print()

print("Stage23 sealed fits : 44 / 50")
print("Stump fits consumed : 0 / 6")

if usable_cache is not None:

    print()
    print("[READY] Exact stump cache found:")
    print(" ", usable_cache)

    print()
    print("NEXT:")
    print("  Run all six frozen stump fits -> Stage23 50/50.")

else:

    print()
    print("[CACHE MISSING]")
    print("The Kaggle reset removed the working execution cache.")
    print()
    print("IMPORTANT:")
    print("  Repository/results are safe on GitHub.")
    print("  No stump fit has been consumed.")
    print("  Do NOT rerun the old stump cell yet.")
    print()
    print("NEXT:")
    print("  Recover only the three stump feature columns + labels/split")
    print("  from the sealed development sources, then execute 6 stumps.")

print("=" * 100)

STAGE23-3 — KAGGLE SESSION RESET RECOVERY

[INFO] Re-cloning repository with live progress...



Cloning into '/kaggle/working/ids2018-validation-safe-ablation'...
remote: Enumerating objects: 2890, done.        
remote: Counting objects: 100% (1884/1884), done.        
remote: Compressing objects: 100% (1621/1621), done.        
remote: Total 2890 (delta 768), reused 1369 (delta 258), pack-reused 1006 (from 3)        
Receiving objects: 100% (2890/2890), 882.25 MiB | 29.44 MiB/s, done.
Resolving deltas: 100% (1020/1020), done.
Updating files: 100% (1793/1793), done.



REPOSITORY VERIFICATION

HEAD:
  eca21e3f6a918d59c64af92eb701361dde91ada7
44/50 seal tag:
  eca21e3f6a918d59c64af92eb701361dde91ada7
Worktree:
  CLEAN

[OK] Exact Stage23 44/50 state restored.

SEARCHING FOR STAGE23 EXECUTION CACHE

[MISS ] /kaggle/working/stage23_execution_cache_v1
[MISS ] /kaggle/input/stage23-execution-cache-v1
[MISS ] /kaggle/input/stage23_execution_cache_v1

CACHE CONTENT CHECK


RECOVERY STATE

Stage23 sealed fits : 44 / 50
Stump fits consumed : 0 / 6

[CACHE MISSING]
The Kaggle reset removed the working execution cache.

IMPORTANT:
  Repository/results are safe on GitHub.
  No stump fit has been consumed.
  Do NOT rerun the old stump cell yet.

NEXT:
  Recover only the three stump feature columns + labels/split
  from the sealed development sources, then execute 6 stumps.


In [4]:
# =============================================================================
# STAGE23 — MINIMAL STUMP CACHE RECOVERY
#
# ZERO MODEL FITS.
#
# Reconstructs ONLY:
#   feature_00 = Dst Port
#   feature_58 = Init Fwd Win Byts
#   feature_61 = Fwd Seg Size Min
#   binary_label
#   clean_position
#   random_validation.packbits
#
# from the exact frozen Stage22R development Parquets.
#
# BEFORE: 44 / 50 SEALED
# AFTER : 44 / 50 SEALED
#
# Any byte mismatch => FAIL CLOSED.
# RAW MAR1/MAR2 FORBIDDEN.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import shutil
import os
import gc
import time

import numpy as np
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "eca21e3f6a918d59c64af92eb701361dde91ada7"
)

EXPECTED_HEAD_TAG = (
    "stage23-placebo-boosted-block-complete-v1"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_RECEIPT_DIR = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_execution_cache_v1"
)

FROZEN_METADATA = (
    CACHE_RECEIPT_DIR
    / "metadata.json"
)

FROZEN_MANIFEST = (
    CACHE_RECEIPT_DIR
    / "data_checksums.sha256"
)

CACHE = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

FEATURE_DIR = CACHE / "features"


N = 14_412_403

EXPECTED_METADATA_SHA = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

EXPECTED_MANIFEST_SHA = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)


EXPECTED_SHA = {
    "features/feature_00.float64.dat":
        "9865224fb0aa281adb042e5af22f7a972fa32283080c602b380f056b50f12a85",

    "features/feature_58.float64.dat":
        "95e36c6fdf6040044061eeb5edb8cf6ec34eb3ecf2dc252ebc9bb6d1440dd42a",

    "features/feature_61.float64.dat":
        "20a3b83c19131a1a307f74d9ad15a7b4332288f18dd0a4cce6accc1c45c48ea5",

    "binary_label.uint8.dat":
        "571b2492929810425fee9c5d28f6b7a16df50f76a681a9df644626cb606f8f34",

    "clean_position.int64.dat":
        "1b839f004115c11bf2e6b21c12512ecef264051f989403e6a00ba5ae85a3b25a",

    "random_validation.packbits":
        "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad",
}


AUTHORIZED_DAYS = [
    ("day_00_02-14-2018.parquet",   822_947),
    ("day_01_02-15-2018.parquet", 1_046_154),
    ("day_02_02-16-2018.parquet",   900_988),
    ("day_03_02-20-2018.parquet", 7_926_258),
    ("day_04_02-21-2018.parquet", 1_031_018),
    ("day_05_02-22-2018.parquet", 1_045_297),
    ("day_06_02-23-2018.parquet", 1_045_961),
    ("day_07_02-28-2018.parquet",   593_780),
]

assert sum(rows for _, rows in AUTHORIZED_DAYS) == N


REQUIRED_COLUMNS = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
    "binary_label",
    "clean_position",
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def sha256_file(path, chunk=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (p.stdout or "").strip()


# =============================================================================
# 2. REPOSITORY / SEAL VERIFICATION
# =============================================================================

print("=" * 100)
print("STAGE23 — MINIMAL STUMP CACHE RECOVERY")
print("=" * 100)
print()


head = git(
    "rev-parse",
    "HEAD",
)

status = git(
    "status",
    "--porcelain",
)

parent_tag = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_HEAD_TAG,
)

cache_tag = git(
    "rev-list",
    "-n",
    "1",
    CACHE_SEAL_TAG,
)


assert head == EXPECTED_HEAD
assert parent_tag == EXPECTED_HEAD
assert cache_tag == CACHE_SEAL_COMMIT
assert status == ""


print("[OK] Stage23 HEAD       :", head)
print("[OK] 44/50 seal tag     :", EXPECTED_HEAD_TAG)
print("[OK] cache seal commit  :", CACHE_SEAL_COMMIT)
print("[OK] repository         : CLEAN")


# =============================================================================
# 3. VERIFY FROZEN CACHE RECEIPT FILES
# =============================================================================

if not FROZEN_METADATA.exists():
    raise RuntimeError(
        f"Frozen metadata missing: {FROZEN_METADATA}"
    )

if not FROZEN_MANIFEST.exists():
    raise RuntimeError(
        f"Frozen checksum manifest missing: {FROZEN_MANIFEST}"
    )


metadata_sha = sha256_file(
    FROZEN_METADATA
)

manifest_sha = sha256_file(
    FROZEN_MANIFEST
)


if metadata_sha != EXPECTED_METADATA_SHA:
    raise RuntimeError(
        "Frozen cache metadata SHA mismatch."
    )

if manifest_sha != EXPECTED_MANIFEST_SHA:
    raise RuntimeError(
        "Frozen cache manifest SHA mismatch."
    )


metadata = json.loads(
    FROZEN_METADATA.read_text(
        encoding="utf-8"
    )
)


assert metadata["shape"]["rows"] == N
assert metadata["shape"]["features"] == 70

assert metadata["source"]["raw_mar1_accessed"] is False
assert metadata["source"]["raw_mar2_accessed"] is False

assert [
    (x["filename"], x["rows"])
    for x in metadata["source"]["authorized_files"]
] == AUTHORIZED_DAYS


print()
print("[EXACT] frozen metadata :", metadata_sha)
print("[EXACT] frozen manifest :", manifest_sha)


# =============================================================================
# 4. LOCATE EXACT AUTHORIZED PARQUET SOURCE
#
# First try the root recorded in the seal.
# If Kaggle mounted the same dataset under a shortened /kaggle/input path,
# identify exactly ONE directory containing ALL eight exact filenames.
# =============================================================================

print()
print("=" * 100)
print("LOCATING FROZEN DEVELOPMENT SOURCE")
print("=" * 100)
print()


recorded_root = Path(
    metadata["source"]["root"]
)


def contains_all_authorized(root):

    return (
        root.exists()
        and root.is_dir()
        and all(
            (root / filename).is_file()
            for filename, _ in AUTHORIZED_DAYS
        )
    )


candidates = []


if contains_all_authorized(
    recorded_root
):
    candidates.append(
        recorded_root
    )


input_root = Path(
    "/kaggle/input"
)


if input_root.exists():

    # Find day_00, then require the same parent to contain
    # all eight exact authorized files.

    for p in input_root.rglob(
        "day_00_02-14-2018.parquet"
    ):

        parent = p.parent

        if contains_all_authorized(
            parent
        ):
            candidates.append(
                parent
            )


# Unique resolved candidates.

unique = []

seen = set()

for p in candidates:

    rp = str(
        p.resolve()
    )

    if rp not in seen:

        seen.add(
            rp
        )

        unique.append(
            p
        )


if len(unique) != 1:

    print(
        "Authorized-source candidates:",
        len(unique),
    )

    for p in unique:
        print(" ", p)

    raise RuntimeError(
        "Could not uniquely identify the frozen "
        "Stage22R development Parquet source."
    )


SOURCE = unique[0]


print("[OK] source root:")
print(" ", SOURCE)


# =============================================================================
# 5. VERIFY ONLY AUTHORIZED FILES + ROW COUNTS + SCHEMA
# =============================================================================

print()
print("=" * 100)
print("VERIFYING 8 AUTHORIZED FEBRUARY PARQUETS")
print("=" * 100)
print()


source_paths = []


for filename, expected_rows in AUTHORIZED_DAYS:

    path = SOURCE / filename

    if not path.is_file():

        raise RuntimeError(
            f"Authorized source missing: {filename}"
        )


    pf = pq.ParquetFile(
        path
    )

    actual_rows = (
        pf.metadata.num_rows
    )

    schema_names = (
        pf.schema_arrow.names
    )


    if actual_rows != expected_rows:

        raise RuntimeError(
            f"{filename}: row mismatch "
            f"{actual_rows} != {expected_rows}"
        )


    missing = [
        c
        for c in REQUIRED_COLUMNS
        if c not in schema_names
    ]


    if missing:

        print()
        print(
            f"{filename} columns:"
        )

        print(
            schema_names
        )

        raise RuntimeError(
            f"{filename}: missing required columns {missing}"
        )


    source_paths.append(
        path
    )


    print(
        f"[EXACT] {filename:<30}"
        f"{actual_rows:>12,} rows"
    )


print()
print(
    "[OK] No March 1 or March 2 file is authorized or read."
)


# =============================================================================
# 6. CREATE MINIMAL CACHE
# =============================================================================

if CACHE.exists():

    # This is a recovery cache from a previous failed attempt.
    # No stump fits have been consumed, so remove it and rebuild
    # only after repository state above is verified.

    shutil.rmtree(
        CACHE
    )


FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


paths = {
    "Dst Port":
        FEATURE_DIR
        / "feature_00.float64.dat",

    "Init Fwd Win Byts":
        FEATURE_DIR
        / "feature_58.float64.dat",

    "Fwd Seg Size Min":
        FEATURE_DIR
        / "feature_61.float64.dat",

    "binary_label":
        CACHE
        / "binary_label.uint8.dat",

    "clean_position":
        CACHE
        / "clean_position.int64.dat",
}


dst_port = np.memmap(
    paths["Dst Port"],
    dtype="<f8",
    mode="w+",
    shape=(N,),
)

init_win = np.memmap(
    paths["Init Fwd Win Byts"],
    dtype="<f8",
    mode="w+",
    shape=(N,),
)

seg_min = np.memmap(
    paths["Fwd Seg Size Min"],
    dtype="<f8",
    mode="w+",
    shape=(N,),
)

labels = np.memmap(
    paths["binary_label"],
    dtype=np.uint8,
    mode="w+",
    shape=(N,),
)

clean_position = np.memmap(
    paths["clean_position"],
    dtype="<i8",
    mode="w+",
    shape=(N,),
)


# =============================================================================
# 7. STREAM EXACT COLUMNS FROM AUTHORIZED PARQUETS
# =============================================================================

print()
print("=" * 100)
print("RECONSTRUCTING MINIMAL CACHE")
print("=" * 100)
print()


cursor = 0

t0 = time.perf_counter()


for day_no, (
    filename,
    expected_rows,
) in enumerate(
    AUTHORIZED_DAYS,
    start=1,
):

    path = SOURCE / filename

    day_start = cursor

    print()
    print(
        f"[{day_no}/8] {filename}"
    )


    pf = pq.ParquetFile(
        path
    )


    for batch in pf.iter_batches(
        batch_size=500_000,
        columns=REQUIRED_COLUMNS,
        use_threads=True,
    ):

        n = batch.num_rows

        end = cursor + n


        names = batch.schema.names

        idx = {
            name: names.index(name)
            for name in REQUIRED_COLUMNS
        }


        # Feature conversion semantics:
        # exact float64, little-endian storage.

        x0 = np.asarray(
            batch.column(
                idx["Dst Port"]
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )

        x58 = np.asarray(
            batch.column(
                idx["Init Fwd Win Byts"]
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )

        x61 = np.asarray(
            batch.column(
                idx["Fwd Seg Size Min"]
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float64,
        )


        y = np.asarray(
            batch.column(
                idx["binary_label"]
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.uint8,
        )


        cp = np.asarray(
            batch.column(
                idx["clean_position"]
            ).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.int64,
        )


        dst_port[
            cursor:end
        ] = x0

        init_win[
            cursor:end
        ] = x58

        seg_min[
            cursor:end
        ] = x61

        labels[
            cursor:end
        ] = y

        clean_position[
            cursor:end
        ] = cp


        cursor = end


        del (
            batch,
            x0,
            x58,
            x61,
            y,
            cp,
        )

        gc.collect()


    day_rows = (
        cursor
        - day_start
    )


    if day_rows != expected_rows:

        raise RuntimeError(
            f"{filename}: streamed {day_rows:,}, "
            f"expected {expected_rows:,}"
        )


    print(
        f"      completed "
        f"{day_rows:,} rows"
    )


if cursor != N:

    raise RuntimeError(
        f"Global row mismatch: {cursor:,} != {N:,}"
    )


dst_port.flush()
init_win.flush()
seg_min.flush()
labels.flush()
clean_position.flush()


build_seconds = (
    time.perf_counter()
    - t0
)


print()
print(
    f"[OK] reconstruction completed in "
    f"{build_seconds:.2f} s"
)


# =============================================================================
# 8. VERIFY FIVE RECONSTRUCTED PHYSICAL ARRAYS
# =============================================================================

print()
print("=" * 100)
print("BYTE-FOR-BYTE SHA256 VERIFICATION")
print("=" * 100)
print()


for rel in [
    "features/feature_00.float64.dat",
    "features/feature_58.float64.dat",
    "features/feature_61.float64.dat",
    "binary_label.uint8.dat",
    "clean_position.int64.dat",
]:

    path = CACHE / rel

    actual = sha256_file(
        path
    )

    expected = EXPECTED_SHA[
        rel
    ]


    print(
        rel
    )

    print(
        "  expected:",
        expected,
    )

    print(
        "  actual:  ",
        actual,
    )


    if actual != expected:

        raise RuntimeError(
            f"BYTE MISMATCH: {rel}"
        )


    print(
        "  [EXACT]"
    )


# =============================================================================
# 9. RECONSTRUCT EXACT RANDOM_NATURAL MEMBERSHIP
# =============================================================================

print()
print("=" * 100)
print("RECONSTRUCT RANDOM_NATURAL MEMBERSHIP")
print("=" * 100)
print()


# Re-open read-only after all writes are flushed.

del (
    dst_port,
    init_win,
    seg_min,
    labels,
    clean_position,
)

gc.collect()


labels_ro = np.memmap(
    CACHE / "binary_label.uint8.dat",
    dtype=np.uint8,
    mode="r",
    shape=(N,),
)


all_positions = np.arange(
    N,
    dtype=np.int64,
)


train_idx, val_idx = train_test_split(
    all_positions,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=np.asarray(
        labels_ro,
        dtype=np.uint8,
    ),
)


if len(train_idx) != 11_529_922:
    raise RuntimeError(
        "Random train count mismatch."
    )

if len(val_idx) != 2_882_481:
    raise RuntimeError(
        "Random validation count mismatch."
    )


if int(
    labels_ro[
        train_idx
    ].sum()
) != 1_577_839:

    raise RuntimeError(
        "Random train attack count mismatch."
    )


if int(
    labels_ro[
        val_idx
    ].sum()
) != 394_460:

    raise RuntimeError(
        "Random validation attack count mismatch."
    )


val_mask = np.zeros(
    N,
    dtype=np.bool_,
)

val_mask[
    val_idx
] = True


packed = np.packbits(
    val_mask,
    bitorder="little",
)


bitset_path = (
    CACHE
    / "random_validation.packbits"
)


packed.tofile(
    bitset_path
)


actual_bitset_sha = sha256_file(
    bitset_path
)


print(
    "expected:",
    EXPECTED_SHA[
        "random_validation.packbits"
    ]
)

print(
    "actual:  ",
    actual_bitset_sha,
)


if (
    actual_bitset_sha
    != EXPECTED_SHA[
        "random_validation.packbits"
    ]
):

    raise RuntimeError(
        "Random membership bitset does not match "
        "the frozen Stage23 split byte-for-byte."
    )


print(
    "[EXACT] RANDOM_NATURAL membership"
)


del (
    all_positions,
    train_idx,
    val_idx,
    val_mask,
    packed,
    labels_ro,
)

gc.collect()


# =============================================================================
# 10. COPY ORIGINAL SEALED METADATA + MANIFEST
# =============================================================================

shutil.copy2(
    FROZEN_METADATA,
    CACHE / "metadata.json",
)

shutil.copy2(
    FROZEN_MANIFEST,
    CACHE / "data_checksums.sha256",
)


assert (
    sha256_file(
        CACHE / "metadata.json"
    )
    == EXPECTED_METADATA_SHA
)

assert (
    sha256_file(
        CACHE / "data_checksums.sha256"
    )
    == EXPECTED_MANIFEST_SHA
)


# =============================================================================
# 11. RECOVERY RECEIPT
# =============================================================================

receipt = {
    "stage":
        "Stage23-Minimal-Stump-Cache-Recovery",

    "status":
        "RECOVERED_AND_BYTE_VERIFIED",

    "stage23_fit_count":
        "44 / 50",

    "model_fits_this_action":
        0,

    "rows":
        N,

    "source_root":
        str(SOURCE),

    "authorized_files":
        [
            {
                "filename": filename,
                "rows": rows,
            }
            for filename, rows
            in AUTHORIZED_DAYS
        ],

    "recovered_arrays": {
        rel: {
            "sha256":
                sha256_file(
                    CACHE / rel
                ),

            "matches_original_sealed_cache":
                True,
        }
        for rel
        in EXPECTED_SHA
    },

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "new_model_fits":
        0,

    "next_authorized_action":
        "Execute six frozen Stage23 depth-1 stump controls.",
}


(
    CACHE
    / "minimal_stump_cache_recovery_receipt.json"
).write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 12. FINAL
# =============================================================================

print()
print("=" * 100)
print("MINIMAL STUMP CACHE RECOVERED — EXACT")
print("=" * 100)
print()
print("Stage23 sealed state : 44 / 50")
print("New model fits       : 0")
print()
print("Recovered:")
print("  Dst Port")
print("  Init Fwd Win Byts")
print("  Fwd Seg Size Min")
print("  binary labels")
print("  clean positions")
print("  RANDOM_NATURAL membership")
print()
print("All recovered arrays match the ORIGINAL")
print("Stage23 deterministic cache SHA256 values.")
print()
print("Raw Mar1 read : NO")
print("Raw Mar2 read : NO")
print()
print("NEXT:")
print("  Run the six frozen depth-1 stumps.")
print("  Stage23 will move 44 / 50 -> 50 / 50.")
print("=" * 100)

STAGE23 — MINIMAL STUMP CACHE RECOVERY

[OK] Stage23 HEAD       : eca21e3f6a918d59c64af92eb701361dde91ada7
[OK] 44/50 seal tag     : stage23-placebo-boosted-block-complete-v1
[OK] cache seal commit  : 66e1c6382caa63699766e605df63df02beff894f
[OK] repository         : CLEAN

[EXACT] frozen metadata : e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439
[EXACT] frozen manifest : 063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b

LOCATING FROZEN DEVELOPMENT SOURCE

[OK] source root:
  /kaggle/input/datasets/jmmubasshirrahman/stage22r-1c-70f-cache-3cd41c5f

VERIFYING 8 AUTHORIZED FEBRUARY PARQUETS

[EXACT] day_00_02-14-2018.parquet          822,947 rows
[EXACT] day_01_02-15-2018.parquet        1,046,154 rows
[EXACT] day_02_02-16-2018.parquet          900,988 rows
[EXACT] day_03_02-20-2018.parquet        7,926,258 rows
[EXACT] day_04_02-21-2018.parquet        1,031,018 rows
[EXACT] day_05_02-22-2018.parquet        1,045,297 rows
[EXACT] day_06_02-23-2018.parquet  

In [9]:
# =============================================================================
# STAGE23-3 — FROZEN DEPTH-1 STUMP CONTROL BLOCK
#
# Executes the FINAL SIX Stage23 model fits:
#
#   3A  Dst Port             × RANDOM_NATURAL
#   3B  Dst Port             × CHRONOLOGICAL_NATURAL
#   3C  Init Fwd Win Byts    × RANDOM_NATURAL
#   3D  Init Fwd Win Byts    × CHRONOLOGICAL_NATURAL
#   3E  Fwd Seg Size Min     × RANDOM_NATURAL
#   3F  Fwd Seg Size Min     × CHRONOLOGICAL_NATURAL
#
# BEFORE : 44 / 50 SEALED
# AFTER  : 50 / 50 COMPLETE, UNSEALED
#
# IMPORTANT:
# - Exactly 6 DecisionTreeClassifier fits.
# - SimpleImputer fits are preprocessing, NOT model fits.
# - No boosted models.
# - No Parquet.
# - No raw Mar1.
# - No raw Mar2.
# - No threshold optimization.
# - No tuning.
# - Fail closed.
# - If this cell fails after any stump fit completes:
#       DO NOT BLINDLY RERUN.
#       Inspect execution_state.json first.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import os
import gc
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import sklearn
import joblib

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CACHE = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

OUT = Path(
    "/kaggle/working/stage23_3_stump_controls"
)


EXPECTED_HEAD = (
    "eca21e3f6a918d59c64af92eb701361dde91ada7"
)

EXPECTED_HEAD_TAG = (
    "stage23-placebo-boosted-block-complete-v1"
)

PROTOCOL_COMMIT = (
    "2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460"
)

PROTOCOL_TAG = (
    "stage23-0-protocol-lock-v1"
)

PROTOCOL_MANIFEST_SHA256 = (
    "36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b"
)

CACHE_SEAL_COMMIT = (
    "66e1c6382caa63699766e605df63df02beff894f"
)

CACHE_SEAL_TAG = (
    "stage23-execution-cache-v1"
)

CACHE_METADATA_SHA256 = (
    "e810d19f04a60233c85894205d29de718e91ac28cad42065fd258c0e0f966439"
)

CACHE_MANIFEST_SHA256 = (
    "063c7263017936b18dd8cb5d508b0a9ed4e68326f8b31e97ac602651d789517b"
)

STUMP_SPEC_BLOB_SHA = (
    "c2c8803e7250b1ff781e45dbf4665e172865e555"
)

N = 14_412_403

RANDOM_TRAIN_N = 11_529_922
RANDOM_VAL_N = 2_882_481

RANDOM_TRAIN_ATTACK = 1_577_839
RANDOM_VAL_ATTACK = 394_460

CHRONO_TRAIN_N = 13_818_623
CHRONO_VAL_N = 593_780

CHRONO_TRAIN_ATTACK = 1_910_043
CHRONO_VAL_ATTACK = 62_256

TOTAL_ATTACK = 1_972_299


PROTOCOL_DIR = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

STUMP_SPEC_PATH = (
    PROTOCOL_DIR
    / "stump_spec.json"
)

PROTOCOL_CHECKSUMS_PATH = (
    PROTOCOL_DIR
    / "checksums.sha256"
)

CACHE_METADATA_PATH = (
    CACHE
    / "metadata.json"
)

CACHE_MANIFEST_PATH = (
    CACHE
    / "data_checksums.sha256"
)

RECOVERY_RECEIPT_PATH = (
    CACHE
    / "minimal_stump_cache_recovery_receipt.json"
)


FEATURES = [
    {
        "feature": "Dst Port",
        "index": 0,
        "path": CACHE / "features" / "feature_00.float64.dat",
        "sha256":
            "9865224fb0aa281adb042e5af22f7a972fa32283080c602b380f056b50f12a85",
        "random_stage": "Stage23-3A",
        "chrono_stage": "Stage23-3B",
        "slug": "dst_port",
    },
    {
        "feature": "Init Fwd Win Byts",
        "index": 58,
        "path": CACHE / "features" / "feature_58.float64.dat",
        "sha256":
            "95e36c6fdf6040044061eeb5edb8cf6ec34eb3ecf2dc252ebc9bb6d1440dd42a",
        "random_stage": "Stage23-3C",
        "chrono_stage": "Stage23-3D",
        "slug": "init_fwd_win_byts",
    },
    {
        "feature": "Fwd Seg Size Min",
        "index": 61,
        "path": CACHE / "features" / "feature_61.float64.dat",
        "sha256":
            "20a3b83c19131a1a307f74d9ad15a7b4332288f18dd0a4cce6accc1c45c48ea5",
        "random_stage": "Stage23-3E",
        "chrono_stage": "Stage23-3F",
        "slug": "fwd_seg_size_min",
    },
]


EXPECTED_CACHE_SHA = {
    "features/feature_00.float64.dat":
        "9865224fb0aa281adb042e5af22f7a972fa32283080c602b380f056b50f12a85",

    "features/feature_58.float64.dat":
        "95e36c6fdf6040044061eeb5edb8cf6ec34eb3ecf2dc252ebc9bb6d1440dd42a",

    "features/feature_61.float64.dat":
        "20a3b83c19131a1a307f74d9ad15a7b4332288f18dd0a4cce6accc1c45c48ea5",

    "binary_label.uint8.dat":
        "571b2492929810425fee9c5d28f6b7a16df50f76a681a9df644626cb606f8f34",

    "clean_position.int64.dat":
        "1b839f004115c11bf2e6b21c12512ecef264051f989403e6a00ba5ae85a3b25a",

    "random_validation.packbits":
        "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad",
}


EXPECTED_TREE_PARAMETERS = {
    "ccp_alpha": 0.0,
    "class_weight": None,
    "criterion": "gini",
    "max_depth": 1,
    "max_features": None,
    "max_leaf_nodes": None,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "random_state": 42,
    "splitter": "best",
}


EXPECTED_METRICS = [
    "PR_AUC",
    "PR_AUC_MINUS_ATTACK_PREVALENCE",
    "ROC_AUC",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "fpr",
    "fnr",
]


EXPECTED_REPORT_STRUCTURE = [
    "selected_feature",
    "split_threshold",
    "missing_imputation_median",
    "left_leaf_class_distribution",
    "right_leaf_class_distribution",
    "left_leaf_support",
    "right_leaf_support",
    "branch_predicting_attack",
]


TASKS = [
    {
        "stage": "Stage23-3A",
        "feature": FEATURES[0],
        "split": "RANDOM_NATURAL",
    },
    {
        "stage": "Stage23-3B",
        "feature": FEATURES[0],
        "split": "CHRONOLOGICAL_NATURAL",
    },
    {
        "stage": "Stage23-3C",
        "feature": FEATURES[1],
        "split": "RANDOM_NATURAL",
    },
    {
        "stage": "Stage23-3D",
        "feature": FEATURES[1],
        "split": "CHRONOLOGICAL_NATURAL",
    },
    {
        "stage": "Stage23-3E",
        "feature": FEATURES[2],
        "split": "RANDOM_NATURAL",
    },
    {
        "stage": "Stage23-3F",
        "feature": FEATURES[2],
        "split": "CHRONOLOGICAL_NATURAL",
    },
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def git(*args):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (p.stdout or "").strip()


def write_json(
    path,
    payload,
):
    Path(path).write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def safe_float(x):
    return float(x)


def safe_int(x):
    return int(x)


def parse_sha_manifest(path):
    result = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        line = line.strip()

        if not line:
            continue

        sha, rel = line.split(
            maxsplit=1
        )

        result[rel.strip()] = (
            sha.strip()
        )

    return result


# =============================================================================
# 2. FAIL-CLOSED ONE-SHOT OUTPUT POLICY
# =============================================================================

if OUT.exists():

    state_path = (
        OUT / "execution_state.json"
    )

    print("=" * 100)
    print("REFUSING TO REUSE EXISTING STUMP EXECUTION DIRECTORY")
    print("=" * 100)

    if state_path.exists():
        print(
            state_path.read_text(
                encoding="utf-8"
            )
        )

    raise RuntimeError(
        "Stage23 stump output directory already exists. "
        "Do NOT blindly rerun. Inspect execution_state.json first."
    )


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


STATE_PATH = (
    OUT / "execution_state.json"
)


state = {
    "stage":
        "Stage23-3-Stump-Control-Block",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "execution_parent_commit":
        EXPECTED_HEAD,

    "execution_parent_tag":
        EXPECTED_HEAD_TAG,

    "stage23_fits_before":
        44,

    "stage23_fits_after_completed":
        44,

    "expected_stump_fits":
        6,

    "completed_stump_fits":
        0,

    "completed_tasks":
        [],

    "current_task":
        None,

    "fit_in_progress":
        False,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "parquet_files_read":
        0,

    "boosted_model_fits":
        0,

    "warning":
        (
            "If execution fails after a stump fit completes, "
            "do not blindly rerun this cell."
        ),
}


write_json(
    STATE_PATH,
    state,
)


# =============================================================================
# 3. REPOSITORY + PROTOCOL VERIFICATION
# =============================================================================

print("=" * 100)
print("STAGE23-3 — FROZEN DEPTH-1 STUMP CONTROL BLOCK")
print("=" * 100)
print()

head = git(
    "rev-parse",
    "HEAD",
)

worktree = git(
    "status",
    "--porcelain",
)

head_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_HEAD_TAG,
)

protocol_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    PROTOCOL_TAG,
)

cache_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    CACHE_SEAL_TAG,
)


assert head == EXPECTED_HEAD
assert worktree == ""
assert head_tag_commit == EXPECTED_HEAD
assert protocol_tag_commit == PROTOCOL_COMMIT
assert cache_tag_commit == CACHE_SEAL_COMMIT


print("[OK] HEAD:")
print(" ", head)

print("[OK] 44/50 parent tag:")
print(" ", EXPECTED_HEAD_TAG)

print("[OK] Stage23-0 protocol tag:")
print(" ", protocol_tag_commit)

print("[OK] execution-cache seal:")
print(" ", cache_tag_commit)

print("[OK] worktree: CLEAN")


# =============================================================================
# 4. VERIFY FROZEN STUMP SPEC EXACTLY
# =============================================================================

print()
print("=" * 100)
print("VERIFY FROZEN STUMP SPEC")
print("=" * 100)
print()


if not STUMP_SPEC_PATH.is_file():
    raise RuntimeError(
        f"Missing stump spec: {STUMP_SPEC_PATH}"
    )


spec_blob = git(
    "hash-object",
    STUMP_SPEC_PATH,
)

if spec_blob != STUMP_SPEC_BLOB_SHA:
    raise RuntimeError(
        "Frozen stump_spec.json Git blob mismatch."
    )


protocol_manifest_sha = sha256_file(
    PROTOCOL_CHECKSUMS_PATH
)

if (
    protocol_manifest_sha
    != PROTOCOL_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage23-0 protocol checksum-manifest mismatch."
    )


spec = json.loads(
    STUMP_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


assert spec["expected_fits"] == 6

assert spec["features"] == [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]

assert spec["splits"] == [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

assert (
    spec["implementation"]
    == "sklearn.tree.DecisionTreeClassifier"
)

assert (
    spec["library_version"]
    == "1.6.1"
)

assert sklearn.__version__ == "1.6.1"

assert (
    spec["missing_value_policy"]["implementation"]
    == "sklearn.impute.SimpleImputer(strategy='median')"
)

assert (
    spec["missing_value_policy"]["method"]
    == "TRAINING_MEDIAN_IMPUTATION_FOR_STUMP_ONLY"
)

assert (
    spec["missing_value_policy"]["fit_scope"]
    == (
        "fit on the stump training membership only; "
        "apply unchanged to validation"
    )
)

assert (
    spec["operating_threshold"]
    == 0.5
)

assert (
    spec["parameters"]
    == EXPECTED_TREE_PARAMETERS
)

assert (
    spec["metrics"]
    == EXPECTED_METRICS
)

assert (
    spec["report_tree_structure"]
    == EXPECTED_REPORT_STRUCTURE
)


print("[EXACT] stump_spec Git blob:")
print(" ", spec_blob)

print("[EXACT] Stage23-0 checksums SHA256:")
print(" ", protocol_manifest_sha)

print("[EXACT] sklearn:")
print(" ", sklearn.__version__)

print("[OK] expected stump fits: 6")
print("[OK] features: 3")
print("[OK] splits: 2")
print("[OK] threshold: 0.5")
print("[OK] training-only median imputation")
print("[OK] exact depth-1 tree parameters")


# =============================================================================
# 5. VERIFY RECOVERED MINIMAL CACHE
# =============================================================================

print()
print("=" * 100)
print("VERIFY RECOVERED MINIMAL STUMP CACHE")
print("=" * 100)
print()


if sha256_file(
    CACHE_METADATA_PATH
) != CACHE_METADATA_SHA256:
    raise RuntimeError(
        "Recovered cache metadata mismatch."
    )


if sha256_file(
    CACHE_MANIFEST_PATH
) != CACHE_MANIFEST_SHA256:
    raise RuntimeError(
        "Recovered cache checksum manifest mismatch."
    )


if not RECOVERY_RECEIPT_PATH.is_file():
    raise RuntimeError(
        "Minimal stump recovery receipt missing."
    )


recovery_receipt = json.loads(
    RECOVERY_RECEIPT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    recovery_receipt["status"]
    == "RECOVERED_AND_BYTE_VERIFIED"
)

assert (
    recovery_receipt["model_fits_this_action"]
    == 0
)

assert (
    recovery_receipt["raw_mar1_accessed"]
    is False
)

assert (
    recovery_receipt["raw_mar2_accessed"]
    is False
)


sealed_manifest = parse_sha_manifest(
    CACHE_MANIFEST_PATH
)


for rel, expected_sha in EXPECTED_CACHE_SHA.items():

    p = CACHE / rel

    if not p.is_file():
        raise RuntimeError(
            f"Required stump cache artifact missing: {rel}"
        )

    if sealed_manifest.get(rel) != expected_sha:
        raise RuntimeError(
            f"Frozen manifest disagreement for {rel}"
        )

    actual_sha = sha256_file(
        p
    )

    print(rel)
    print("  expected:", expected_sha)
    print("  actual:  ", actual_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Recovered cache byte mismatch: {rel}"
        )

    print("  [EXACT]")


# =============================================================================
# 6. OPEN ONLY REQUIRED CACHE ARRAYS
# =============================================================================

labels = np.memmap(
    CACHE / "binary_label.uint8.dat",
    dtype=np.uint8,
    mode="r",
    shape=(N,),
)

clean_position = np.memmap(
    CACHE / "clean_position.int64.dat",
    dtype="<i8",
    mode="r",
    shape=(N,),
)


if int(
    np.asarray(labels).sum()
) != TOTAL_ATTACK:

    raise RuntimeError(
        "Global attack count mismatch."
    )


# =============================================================================
# 7. RANDOM_NATURAL MEMBERSHIP — EXACT SEALED BITSET
# =============================================================================

packed = np.fromfile(
    CACHE / "random_validation.packbits",
    dtype=np.uint8,
)

unpacked = np.unpackbits(
    packed,
    bitorder="little",
)


if unpacked.size < N:
    raise RuntimeError(
        "Random validation bitset is too short."
    )


# Padding bits after N must not contain membership.

if int(
    unpacked[N:].sum()
) != 0:

    raise RuntimeError(
        "Unexpected set bits in random membership padding."
    )


random_val_mask = (
    unpacked[:N]
    .astype(
        np.bool_,
        copy=False,
    )
)

random_train_mask = (
    ~random_val_mask
)


assert int(
    random_val_mask.sum()
) == RANDOM_VAL_N

assert int(
    random_train_mask.sum()
) == RANDOM_TRAIN_N


random_train_attack_actual = int(
    labels[
        random_train_mask
    ].sum()
)

random_val_attack_actual = int(
    labels[
        random_val_mask
    ].sum()
)


assert (
    random_train_attack_actual
    == RANDOM_TRAIN_ATTACK
)

assert (
    random_val_attack_actual
    == RANDOM_VAL_ATTACK
)


print()
print("[EXACT] RANDOM_NATURAL")
print(
    "  train:",
    f"{RANDOM_TRAIN_N:,}",
    "attack:",
    f"{RANDOM_TRAIN_ATTACK:,}",
)

print(
    "  val:  ",
    f"{RANDOM_VAL_N:,}",
    "attack:",
    f"{RANDOM_VAL_ATTACK:,}",
)


# =============================================================================
# 8. CHRONOLOGICAL_NATURAL — EXACT CANONICAL DAY BOUNDARY
#
# Canonical cache order:
#   day IDs 0–6 -> first 13,818,623 rows
#   day ID 7   -> final   593,780 rows
#
# No day_id cache recovery is required because the byte-exact canonical
# row order and frozen boundary are already known and verified.
# =============================================================================

if (
    CHRONO_TRAIN_N
    + CHRONO_VAL_N
    != N
):
    raise RuntimeError(
        "Chronological boundary arithmetic mismatch."
    )


chrono_train_attack_actual = int(
    labels[
        :CHRONO_TRAIN_N
    ].sum()
)

chrono_val_attack_actual = int(
    labels[
        CHRONO_TRAIN_N:
    ].sum()
)


assert (
    chrono_train_attack_actual
    == CHRONO_TRAIN_ATTACK
)

assert (
    chrono_val_attack_actual
    == CHRONO_VAL_ATTACK
)


print()
print("[EXACT] CHRONOLOGICAL_NATURAL")
print(
    "  train:",
    f"{CHRONO_TRAIN_N:,}",
    "attack:",
    f"{CHRONO_TRAIN_ATTACK:,}",
)

print(
    "  val:  ",
    f"{CHRONO_VAL_N:,}",
    "attack:",
    f"{CHRONO_VAL_ATTACK:,}",
)


# Release unpacked byte array; masks remain.

del packed
del unpacked
gc.collect()


# =============================================================================
# 9. MATERIALIZE LABEL / POSITION MEMBERSHIPS ONCE
# =============================================================================

y_random_train = np.asarray(
    labels[
        random_train_mask
    ],
    dtype=np.uint8,
)

y_random_val = np.asarray(
    labels[
        random_val_mask
    ],
    dtype=np.uint8,
)

cp_random_val = np.asarray(
    clean_position[
        random_val_mask
    ],
    dtype=np.int64,
)


# Chronological selections remain canonical contiguous slices.

y_chrono_train = np.asarray(
    labels[
        :CHRONO_TRAIN_N
    ],
    dtype=np.uint8,
)

y_chrono_val = np.asarray(
    labels[
        CHRONO_TRAIN_N:
    ],
    dtype=np.uint8,
)

cp_chrono_val = np.asarray(
    clean_position[
        CHRONO_TRAIN_N:
    ],
    dtype=np.int64,
)


# =============================================================================
# 10. METRIC / TREE HELPERS
# =============================================================================

def calculate_metrics(
    y_true,
    probability,
    threshold=0.5,
):

    y_true = np.asarray(
        y_true,
        dtype=np.uint8,
    )

    probability = np.asarray(
        probability,
        dtype=np.float64,
    )

    prediction = (
        probability
        >= threshold
    ).astype(
        np.uint8
    )


    prevalence = safe_float(
        y_true.mean()
    )

    pr_auc = safe_float(
        average_precision_score(
            y_true,
            probability,
        )
    )

    roc_auc = safe_float(
        roc_auc_score(
            y_true,
            probability,
        )
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            prediction,
            labels=[0, 1],
        )
        .ravel()
        .tolist()
    )


    tn = safe_int(tn)
    fp = safe_int(fp)
    fn = safe_int(fn)
    tp = safe_int(tp)


    fpr = (
        fp / (fp + tn)
        if (fp + tn)
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp)
        else 0.0
    )


    metrics = {
        "PR_AUC":
            pr_auc,

        "PR_AUC_MINUS_ATTACK_PREVALENCE":
            pr_auc
            - prevalence,

        "ROC_AUC":
            roc_auc,

        "accuracy":
            safe_float(
                accuracy_score(
                    y_true,
                    prediction,
                )
            ),

        "precision":
            safe_float(
                precision_score(
                    y_true,
                    prediction,
                    zero_division=0,
                )
            ),

        "recall":
            safe_float(
                recall_score(
                    y_true,
                    prediction,
                    zero_division=0,
                )
            ),

        "f1":
            safe_float(
                f1_score(
                    y_true,
                    prediction,
                    zero_division=0,
                )
            ),

        "fpr":
            safe_float(fpr),

        "fnr":
            safe_float(fnr),
    }


    return {
        "attack_prevalence":
            prevalence,

        "operating_threshold":
            safe_float(threshold),

        "metrics":
            metrics,

        "confusion_matrix": {
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        },
    }


def class_distribution(
    y,
):
    counts = np.bincount(
        np.asarray(
            y,
            dtype=np.uint8,
        ),
        minlength=2,
    )

    benign = int(
        counts[0]
    )

    attack = int(
        counts[1]
    )

    support = (
        benign + attack
    )

    attack_rate = (
        attack / support
        if support
        else None
    )

    return {
        "benign":
            benign,

        "attack":
            attack,

        "attack_rate":
            (
                safe_float(
                    attack_rate
                )
                if attack_rate
                is not None
                else None
            ),
    }


def extract_tree_structure(
    clf,
    X_train_imputed,
    y_train,
    feature_name,
    median,
):
    tree = clf.tree_

    root_feature = int(
        tree.feature[0]
    )

    # sklearn uses TREE_UNDEFINED = -2 when the root never split.
    split_performed = (
        root_feature >= 0
    )


    if not split_performed:

        root_distribution = (
            class_distribution(
                y_train
            )
        )

        predicts_attack = (
            root_distribution[
                "attack_rate"
            ]
            >= 0.5
        )

        return {
            "selected_feature":
                feature_name,

            "tree_split_performed":
                False,

            "split_threshold":
                None,

            "missing_imputation_median":
                safe_float(
                    median
                ),

            "left_leaf_class_distribution":
                None,

            "right_leaf_class_distribution":
                None,

            "left_leaf_support":
                0,

            "right_leaf_support":
                0,

            "branch_predicting_attack":
                (
                    "NO_SPLIT_ROOT"
                    if predicts_attack
                    else "NEITHER"
                ),

            "root_class_distribution":
                root_distribution,

            "attack_branch_rule":
                (
                    "Attack iff leaf P(y=1) >= 0.5."
                ),
        }


    if root_feature != 0:

        raise RuntimeError(
            "Single-feature stump split on unexpected feature index."
        )


    threshold = safe_float(
        tree.threshold[0]
    )


    x = np.asarray(
        X_train_imputed[
            :,
            0
        ],
        dtype=np.float64,
    )


    left_mask = (
        x <= threshold
    )

    right_mask = (
        ~left_mask
    )


    left_y = (
        y_train[
            left_mask
        ]
    )

    right_y = (
        y_train[
            right_mask
        ]
    )


    left_distribution = (
        class_distribution(
            left_y
        )
    )

    right_distribution = (
        class_distribution(
            right_y
        )
    )


    left_support = int(
        left_mask.sum()
    )

    right_support = int(
        right_mask.sum()
    )


    if (
        left_support
        + right_support
        != len(y_train)
    ):
        raise RuntimeError(
            "Stump leaf support does not sum to training membership."
        )


    # Same decision rule as frozen threshold 0.5.

    left_attack = (
        left_distribution[
            "attack_rate"
        ]
        >= 0.5
    )

    right_attack = (
        right_distribution[
            "attack_rate"
        ]
        >= 0.5
    )


    if (
        left_attack
        and right_attack
    ):
        branch = "BOTH"

    elif left_attack:
        branch = "LEFT"

    elif right_attack:
        branch = "RIGHT"

    else:
        branch = "NEITHER"


    return {
        "selected_feature":
            feature_name,

        "tree_split_performed":
            True,

        "split_threshold":
            threshold,

        "missing_imputation_median":
            safe_float(
                median
            ),

        "left_leaf_class_distribution":
            left_distribution,

        "right_leaf_class_distribution":
            right_distribution,

        "left_leaf_support":
            left_support,

        "right_leaf_support":
            right_support,

        "branch_predicting_attack":
            branch,

        "attack_branch_rule":
            (
                "Attack iff leaf P(y=1) >= 0.5."
            ),

        "sklearn_tree_n_node_samples": {
            "root":
                int(
                    tree.n_node_samples[
                        0
                    ]
                ),

            "left":
                int(
                    tree.n_node_samples[
                        tree.children_left[
                            0
                        ]
                    ]
                ),

            "right":
                int(
                    tree.n_node_samples[
                        tree.children_right[
                            0
                        ]
                    ]
                ),
        },
    }


# =============================================================================
# 11. EXECUTE EXACTLY SIX FROZEN STUMP FITS
# =============================================================================

results = {}

state["status"] = (
    "VERIFICATION_COMPLETE_READY_FOR_FITS"
)

write_json(
    STATE_PATH,
    state,
)


try:

    for ordinal, task in enumerate(
        TASKS,
        start=1,
    ):

        stage = (
            task["stage"]
        )

        feature_meta = (
            task["feature"]
        )

        feature_name = (
            feature_meta[
                "feature"
            ]
        )

        feature_slug = (
            feature_meta[
                "slug"
            ]
        )

        split = (
            task["split"]
        )


        print()
        print("=" * 100)
        print(
            f"{stage} — {feature_name} × {split}"
        )
        print("=" * 100)
        print()


        # ---------------------------------------------------------------------
        # Re-verify physical feature bytes immediately before use.
        # ---------------------------------------------------------------------

        feature_path = (
            feature_meta[
                "path"
            ]
        )

        feature_sha = sha256_file(
            feature_path
        )

        if (
            feature_sha
            != feature_meta[
                "sha256"
            ]
        ):
            raise RuntimeError(
                f"{stage}: feature SHA mismatch."
            )


        feature_array = np.memmap(
            feature_path,
            dtype="<f8",
            mode="r",
            shape=(N,),
        )


        # ---------------------------------------------------------------------
        # Materialize exact split membership.
        # ---------------------------------------------------------------------

        materialize_t0 = (
            time.perf_counter()
        )


        if split == "RANDOM_NATURAL":

            X_train = np.asarray(
                feature_array[
                    random_train_mask
                ],
                dtype=np.float64,
            ).reshape(
                -1,
                1,
            )

            X_val = np.asarray(
                feature_array[
                    random_val_mask
                ],
                dtype=np.float64,
            ).reshape(
                -1,
                1,
            )

            y_train = (
                y_random_train
            )

            y_val = (
                y_random_val
            )

            val_clean_position = (
                cp_random_val
            )

            expected_train_n = (
                RANDOM_TRAIN_N
            )

            expected_val_n = (
                RANDOM_VAL_N
            )

            expected_train_attack = (
                RANDOM_TRAIN_ATTACK
            )

            expected_val_attack = (
                RANDOM_VAL_ATTACK
            )


        elif split == "CHRONOLOGICAL_NATURAL":

            X_train = np.asarray(
                feature_array[
                    :CHRONO_TRAIN_N
                ],
                dtype=np.float64,
            ).reshape(
                -1,
                1,
            )

            X_val = np.asarray(
                feature_array[
                    CHRONO_TRAIN_N:
                ],
                dtype=np.float64,
            ).reshape(
                -1,
                1,
            )

            y_train = (
                y_chrono_train
            )

            y_val = (
                y_chrono_val
            )

            val_clean_position = (
                cp_chrono_val
            )

            expected_train_n = (
                CHRONO_TRAIN_N
            )

            expected_val_n = (
                CHRONO_VAL_N
            )

            expected_train_attack = (
                CHRONO_TRAIN_ATTACK
            )

            expected_val_attack = (
                CHRONO_VAL_ATTACK
            )


        else:
            raise RuntimeError(
                f"Unexpected split: {split}"
            )


        materialization_seconds = (
            time.perf_counter()
            - materialize_t0
        )


        assert (
            X_train.shape
            == (
                expected_train_n,
                1,
            )
        )

        assert (
            X_val.shape
            == (
                expected_val_n,
                1,
            )
        )

        assert (
            len(y_train)
            == expected_train_n
        )

        assert (
            len(y_val)
            == expected_val_n
        )

        assert (
            int(
                y_train.sum()
            )
            == expected_train_attack
        )

        assert (
            int(
                y_val.sum()
            )
            == expected_val_attack
        )


        # ---------------------------------------------------------------------
        # Frozen stump-only missing-value treatment:
        #
        # fit median on TRAINING MEMBERSHIP ONLY,
        # then apply unchanged to validation.
        # ---------------------------------------------------------------------

        train_missing = int(
            np.isnan(
                X_train[
                    :,
                    0
                ]
            ).sum()
        )

        val_missing = int(
            np.isnan(
                X_val[
                    :,
                    0
                ]
            ).sum()
        )


        imputer = SimpleImputer(
            strategy="median"
        )


        impute_t0 = (
            time.perf_counter()
        )

        X_train_imp = (
            imputer.fit_transform(
                X_train
            )
        )

        median = safe_float(
            imputer.statistics_[
                0
            ]
        )

        X_val_imp = (
            imputer.transform(
                X_val
            )
        )

        imputation_seconds = (
            time.perf_counter()
            - impute_t0
        )


        if not np.isfinite(
            median
        ):
            raise RuntimeError(
                f"{stage}: non-finite training median."
            )


        # Original raw matrices no longer needed.
        del X_train
        del X_val
        gc.collect()


        # ---------------------------------------------------------------------
        # EXACT FROZEN DecisionTreeClassifier.
        #
        # STATE IS WRITTEN BEFORE FIT.
        # If process interruption occurs here, user must inspect state.
        # ---------------------------------------------------------------------

        state["status"] = (
            "STUMP_FIT_IN_PROGRESS"
        )

        state["current_task"] = {
            "ordinal":
                ordinal,

            "stage":
                stage,

            "feature":
                feature_name,

            "split":
                split,
        }

        state["fit_in_progress"] = (
            True
        )

        state[
            "stage23_fits_after_completed"
        ] = (
            44
            + state[
                "completed_stump_fits"
            ]
        )

        write_json(
            STATE_PATH,
            state,
        )


        stump = DecisionTreeClassifier(
            **spec["parameters"]
        )


        fit_t0 = (
            time.perf_counter()
        )

        stump.fit(
            X_train_imp,
            y_train,
        )

        fit_seconds = (
            time.perf_counter()
            - fit_t0
        )


        # ---------------------------------------------------------------------
        # FIT COMPLETED.
        # Count it IMMEDIATELY before any later artifact/metric work.
        # ---------------------------------------------------------------------

        state[
            "completed_stump_fits"
        ] += 1

        state[
            "stage23_fits_after_completed"
        ] = (
            44
            + state[
                "completed_stump_fits"
            ]
        )

        state[
            "completed_tasks"
        ].append(
            {
                "ordinal":
                    ordinal,

                "stage":
                    stage,

                "feature":
                    feature_name,

                "split":
                    split,

                "fit_seconds":
                    safe_float(
                        fit_seconds
                    ),

                "fit_completed_utc":
                    utc_now(),
            }
        )

        state["fit_in_progress"] = (
            False
        )

        state["status"] = (
            "STUMP_FIT_COMPLETE_POSTPROCESSING"
        )

        write_json(
            STATE_PATH,
            state,
        )


        print(
            f"[FIT {ordinal}/6 COMPLETE] "
            f"{fit_seconds:.3f} s"
        )

        print(
            "Stage23 consumed:",
            f"{state['stage23_fits_after_completed']}/50"
        )


        # ---------------------------------------------------------------------
        # Validation probabilities.
        # ---------------------------------------------------------------------

        predict_t0 = (
            time.perf_counter()
        )


        classes = (
            stump.classes_
            .tolist()
        )


        if (
            0 not in classes
            or 1 not in classes
        ):
            raise RuntimeError(
                f"{stage}: stump does not contain both binary classes."
            )


        attack_col = (
            classes.index(
                1
            )
        )


        probability = (
            stump.predict_proba(
                X_val_imp
            )[
                :,
                attack_col
            ]
            .astype(
                np.float64,
                copy=False,
            )
        )


        prediction_seconds = (
            time.perf_counter()
            - predict_t0
        )


        # ---------------------------------------------------------------------
        # Frozen metrics.
        # ---------------------------------------------------------------------

        metric_block = (
            calculate_metrics(
                y_val,
                probability,
                threshold=0.5,
            )
        )


        # ---------------------------------------------------------------------
        # Frozen tree-structure reporting.
        # ---------------------------------------------------------------------

        tree_structure = (
            extract_tree_structure(
                clf=stump,
                X_train_imputed=X_train_imp,
                y_train=y_train,
                feature_name=feature_name,
                median=median,
            )
        )


        # ---------------------------------------------------------------------
        # Persist exact fitted preprocessor + stump.
        # ---------------------------------------------------------------------

        stage_slug = (
            stage
            .lower()
            .replace(
                "-",
                "_",
            )
        )

        split_slug = (
            split.lower()
        )


        model_name = (
            f"{stage_slug}_"
            f"{feature_slug}_"
            f"{split_slug}_model.joblib"
        )

        probability_name = (
            f"{stage_slug}_"
            f"{feature_slug}_"
            f"{split_slug}_"
            f"validation_probabilities.npz"
        )

        result_name = (
            f"{stage_slug}_"
            f"{feature_slug}_"
            f"{split_slug}_result.json"
        )


        model_path = (
            OUT / model_name
        )

        probability_path = (
            OUT / probability_name
        )

        result_path = (
            OUT / result_name
        )


        joblib.dump(
            {
                "stage":
                    stage,

                "feature":
                    feature_name,

                "feature_index":
                    feature_meta[
                        "index"
                    ],

                "split":
                    split,

                "operating_threshold":
                    0.5,

                "missing_value_method":
                    (
                        "TRAINING_MEDIAN_IMPUTATION_FOR_STUMP_ONLY"
                    ),

                "imputer":
                    imputer,

                "stump":
                    stump,
            },
            model_path,
        )


        np.savez_compressed(
            probability_path,
            clean_position=np.asarray(
                val_clean_position,
                dtype=np.int64,
            ),
            binary_label=np.asarray(
                y_val,
                dtype=np.uint8,
            ),
            probability=np.asarray(
                probability,
                dtype=np.float64,
            ),
        )


        model_sha = sha256_file(
            model_path
        )

        probability_sha = sha256_file(
            probability_path
        )


        # ---------------------------------------------------------------------
        # Point result.
        # ---------------------------------------------------------------------

        result = {
            "stage":
                stage,

            "status":
                "RESULT_COMPLETE_UNSEALED",

            "created_utc":
                utc_now(),

            "execution_parent": {
                "commit":
                    EXPECTED_HEAD,

                "tag":
                    EXPECTED_HEAD_TAG,
            },

            "protocol": {
                "commit":
                    PROTOCOL_COMMIT,

                "tag":
                    PROTOCOL_TAG,

                "stump_spec_git_blob":
                    STUMP_SPEC_BLOB_SHA,

                "protocol_checksums_sha256":
                    PROTOCOL_MANIFEST_SHA256,
            },

            "cache": {
                "identifier":
                    "stage23_execution_cache_v1",

                "recovery":
                    "MINIMAL_STUMP_CACHE_RECOVERED_BYTE_EXACT",

                "feature_file":
                    str(
                        feature_path
                    ),

                "feature_file_sha256":
                    feature_sha,

                "parquet_files_read":
                    0,
            },

            "cell": {
                "feature":
                    feature_name,

                "feature_index":
                    feature_meta[
                        "index"
                    ],

                "split":
                    split,

                "train_rows":
                    safe_int(
                        expected_train_n
                    ),

                "validation_rows":
                    safe_int(
                        expected_val_n
                    ),

                "train_attack":
                    safe_int(
                        expected_train_attack
                    ),

                "validation_attack":
                    safe_int(
                        expected_val_attack
                    ),

                "input_dtype":
                    "float64",
            },

            "missing_value_handling": {
                "implementation":
                    (
                        "sklearn.impute.SimpleImputer(strategy='median')"
                    ),

                "method":
                    (
                        "TRAINING_MEDIAN_IMPUTATION_FOR_STUMP_ONLY"
                    ),

                "fit_scope":
                    (
                        "training membership only"
                    ),

                "training_missing_values":
                    train_missing,

                "validation_missing_values":
                    val_missing,

                "training_median":
                    median,

                "imputation_seconds":
                    safe_float(
                        imputation_seconds
                    ),
            },

            "model": {
                "implementation":
                    (
                        "sklearn.tree.DecisionTreeClassifier"
                    ),

                "sklearn_version":
                    sklearn.__version__,

                "parameters":
                    spec[
                        "parameters"
                    ],

                "fit_seconds":
                    safe_float(
                        fit_seconds
                    ),

                "prediction_seconds":
                    safe_float(
                        prediction_seconds
                    ),

                "materialization_seconds":
                    safe_float(
                        materialization_seconds
                    ),

                "model_artifact":
                    model_name,

                "model_sha256":
                    model_sha,
            },

            "tree_structure":
                tree_structure,

            "evaluation":
                metric_block,

            "validation_probabilities": {
                "artifact":
                    probability_name,

                "probability_dtype":
                    "float64",

                "sha256":
                    probability_sha,
            },

            "governance": {
                "new_model_fits_this_result":
                    1,

                "model_type":
                    "DEPTH_1_STUMP_CONTROL",

                "boosted_model_fits":
                    0,

                "subset_specific_tuning":
                    False,

                "threshold_optimization":
                    False,

                "rebalancing":
                    False,

                "raw_mar1_accessed":
                    False,

                "raw_mar2_accessed":
                    False,

                "stage23_0_modified":
                    False,

                "stump_spec_modified":
                    False,
            },
        }


        # Random member cannot yet report degradation until matched chrono
        # result exists. Chronological result gets the paired degradation later
        # in the consolidated summary.

        if split == "RANDOM_NATURAL":

            result[
                "random_to_chronological_degradation"
            ] = {
                "status":
                    "PENDING_MATCHED_CHRONO_STUMP"
            }

        else:

            result[
                "random_to_chronological_degradation"
            ] = {
                "status":
                    "REPORTED_IN_CONSOLIDATED_STUMP_SUMMARY"
            }


        write_json(
            result_path,
            result,
        )


        result_sha = sha256_file(
            result_path
        )


        results[
            (
                feature_name,
                split,
            )
        ] = {
            "result":
                result,

            "result_file":
                result_name,

            "result_sha256":
                result_sha,

            "model_file":
                model_name,

            "model_sha256":
                model_sha,

            "probability_file":
                probability_name,

            "probability_sha256":
                probability_sha,
        }


        # State postprocessing complete.

        state["status"] = (
            "TASK_COMPLETE"
        )

        state["current_task"] = (
            None
        )

        state["fit_in_progress"] = (
            False
        )

        write_json(
            STATE_PATH,
            state,
        )


        m = metric_block[
            "metrics"
        ]

        print()
        print(
            f"{feature_name} × {split}"
        )

        print(
            f"  median       : {median:.12g}"
        )

        print(
            f"  threshold    : "
            f"{tree_structure['split_threshold']}"
        )

        print(
            f"  attack branch: "
            f"{tree_structure['branch_predicting_attack']}"
        )

        print(
            f"  prevalence   : "
            f"{metric_block['attack_prevalence']:.12f}"
        )

        print(
            f"  PR-AUC       : "
            f"{m['PR_AUC']:.12f}"
        )

        print(
            f"  PR-prev      : "
            f"{m['PR_AUC_MINUS_ATTACK_PREVALENCE']:.12f}"
        )

        print(
            f"  ROC-AUC      : "
            f"{m['ROC_AUC']:.12f}"
        )

        print(
            f"  F1 @ .50     : "
            f"{m['f1']:.12f}"
        )

        print(
            f"  Recall @ .50 : "
            f"{m['recall']:.12f}"
        )

        print(
            f"  FPR @ .50    : "
            f"{m['fpr']:.12f}"
        )


        # Release large per-task allocations before next fit.

        del feature_array
        del X_train_imp
        del X_val_imp
        del stump
        del imputer
        del probability

        gc.collect()


    # =========================================================================
    # 12. CONSOLIDATED RANDOM -> CHRONO DEGRADATION
    # =========================================================================

    print()
    print("=" * 100)
    print("RANDOM → CHRONOLOGICAL STUMP DEGRADATION")
    print("=" * 100)
    print()


    paired = {}


    for feature_meta in FEATURES:

        feature_name = (
            feature_meta[
                "feature"
            ]
        )


        random_result = results[
            (
                feature_name,
                "RANDOM_NATURAL",
            )
        ][
            "result"
        ]


        chrono_result = results[
            (
                feature_name,
                "CHRONOLOGICAL_NATURAL",
            )
        ][
            "result"
        ]


        rm = random_result[
            "evaluation"
        ][
            "metrics"
        ]

        cm = chrono_result[
            "evaluation"
        ][
            "metrics"
        ]


        degradation = {
            "definition":
                (
                    "M_RANDOM_NATURAL - "
                    "M_CHRONOLOGICAL_NATURAL"
                ),

            "PR_AUC":
                safe_float(
                    rm["PR_AUC"]
                    - cm["PR_AUC"]
                ),

            "PR_AUC_MINUS_ATTACK_PREVALENCE":
                safe_float(
                    rm[
                        "PR_AUC_MINUS_ATTACK_PREVALENCE"
                    ]
                    - cm[
                        "PR_AUC_MINUS_ATTACK_PREVALENCE"
                    ]
                ),

            "ROC_AUC":
                safe_float(
                    rm["ROC_AUC"]
                    - cm["ROC_AUC"]
                ),

            "accuracy":
                safe_float(
                    rm["accuracy"]
                    - cm["accuracy"]
                ),

            "precision":
                safe_float(
                    rm["precision"]
                    - cm["precision"]
                ),

            "recall":
                safe_float(
                    rm["recall"]
                    - cm["recall"]
                ),

            "f1":
                safe_float(
                    rm["f1"]
                    - cm["f1"]
                ),

            "fpr":
                safe_float(
                    rm["fpr"]
                    - cm["fpr"]
                ),

            "fnr":
                safe_float(
                    rm["fnr"]
                    - cm["fnr"]
                ),
        }


        paired[
            feature_name
        ] = {
            "random_stage":
                random_result[
                    "stage"
                ],

            "chronological_stage":
                chrono_result[
                    "stage"
                ],

            "random_metrics":
                rm,

            "chronological_metrics":
                cm,

            "random_to_chronological_degradation":
                degradation,

            "random_tree_structure":
                random_result[
                    "tree_structure"
                ],

            "chronological_tree_structure":
                chrono_result[
                    "tree_structure"
                ],
        }


        print(
            feature_name
        )

        print(
            f"  Δ PR-AUC  : "
            f"{degradation['PR_AUC']:+.12f}"
        )

        print(
            f"  Δ ROC-AUC : "
            f"{degradation['ROC_AUC']:+.12f}"
        )

        print(
            f"  Δ F1      : "
            f"{degradation['f1']:+.12f}"
        )

        print()


    # =========================================================================
    # 13. CONSOLIDATED BLOCK SUMMARY
    # =========================================================================

    if (
        state[
            "completed_stump_fits"
        ]
        != 6
    ):
        raise RuntimeError(
            "Expected exactly six completed stump fits."
        )


    if (
        state[
            "stage23_fits_after_completed"
        ]
        != 50
    ):
        raise RuntimeError(
            "Stage23 fit accounting did not reach exactly 50."
        )


    summary = {
        "stage":
            "Stage23-3-Stump-Control-Block",

        "status":
            "ALL_SIX_STUMPS_COMPLETE_UNSEALED",

        "completed_utc":
            utc_now(),

        "execution_parent": {
            "commit":
                EXPECTED_HEAD,

            "tag":
                EXPECTED_HEAD_TAG,
        },

        "fit_accounting": {
            "stage23_before":
                44,

            "stump_fits_expected":
                6,

            "stump_fits_completed":
                6,

            "stage23_after":
                50,

            "boosted_model_fits_this_block":
                0,

            "total_model_fits_this_block":
                6,
        },

        "frozen_design": {
            "features":
                spec["features"],

            "splits":
                spec["splits"],

            "implementation":
                spec["implementation"],

            "library_version":
                spec["library_version"],

            "parameters":
                spec["parameters"],

            "operating_threshold":
                spec[
                    "operating_threshold"
                ],

            "missing_value_policy":
                spec[
                    "missing_value_policy"
                ],
        },

        "paired_random_to_chronological":
            paired,

        "artifacts": {
            (
                f"{feature}::{split}"
            ): {
                "result_file":
                    entry[
                        "result_file"
                    ],

                "result_sha256":
                    entry[
                        "result_sha256"
                    ],

                "model_file":
                    entry[
                        "model_file"
                    ],

                "model_sha256":
                    entry[
                        "model_sha256"
                    ],

                "probability_file":
                    entry[
                        "probability_file"
                    ],

                "probability_sha256":
                    entry[
                        "probability_sha256"
                    ],
            }
            for (
                feature,
                split,
            ), entry
            in results.items()
        },

        "governance": {
            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "parquet_files_read":
                0,

            "boosted_models_fit":
                0,

            "new_stump_fits":
                6,

            "threshold_optimization":
                False,

            "subset_specific_tuning":
                False,

            "stump_spec_modified":
                False,

            "stage23_0_modified":
                False,

            "final_status":
                (
                    "MODEL FIT BUDGET EXHAUSTED; "
                    "RESULTS REQUIRE SEAL"
                ),
        },

        "next_action":
            (
                "Zero-fit seal of Stage23-3 stump-control block. "
                "Do not execute any additional Stage23 model fit."
            ),
    }


    SUMMARY_PATH = (
        OUT
        / "stage23_3_stump_controls_summary.json"
    )


    write_json(
        SUMMARY_PATH,
        summary,
    )


    # =========================================================================
    # 14. FINAL EXECUTION STATE
    # =========================================================================

    state["status"] = (
        "ALL_SIX_STUMPS_COMPLETE_UNSEALED"
    )

    state["completed_utc"] = (
        utc_now()
    )

    state["current_task"] = (
        None
    )

    state["fit_in_progress"] = (
        False
    )

    state[
        "stage23_fits_after_completed"
    ] = 50

    state["next_action"] = (
        "ZERO_FIT_SEAL_ONLY"
    )

    state["additional_model_fits_authorized"] = (
        0
    )


    write_json(
        STATE_PATH,
        state,
    )


    # =========================================================================
    # 15. FINAL ARTIFACT CHECKSUM MANIFEST
    # =========================================================================

    artifact_paths = sorted(
        p
        for p in OUT.iterdir()
        if (
            p.is_file()
            and p.name
            != "checksums.sha256"
        )
    )


    manifest_lines = []

    for p in artifact_paths:

        manifest_lines.append(
            f"{sha256_file(p)}  {p.name}"
        )


    CHECKSUMS_PATH = (
        OUT
        / "checksums.sha256"
    )


    CHECKSUMS_PATH.write_text(
        "\n".join(
            manifest_lines
        ) + "\n",
        encoding="utf-8",
    )


    checksums_sha = sha256_file(
        CHECKSUMS_PATH
    )


    # =========================================================================
    # 16. FINAL REPORT
    # =========================================================================

    print()
    print("=" * 100)
    print("STAGE23-3 STUMP CONTROL BLOCK COMPLETE — UNSEALED")
    print("=" * 100)
    print()

    print(
        "Completed stump fits :",
        state[
            "completed_stump_fits"
        ],
        "/ 6",
    )

    print(
        "Stage23 fit count    :",
        "50 / 50",
    )

    print(
        "Boosted fits here    :",
        0,
    )

    print(
        "Parquet files read   :",
        0,
    )

    print(
        "Raw Mar1 accessed    :",
        "NO",
    )

    print(
        "Raw Mar2 accessed    :",
        "NO",
    )

    print()

    print(
        "Summary SHA256       :",
        sha256_file(
            SUMMARY_PATH
        ),
    )

    print(
        "Checksums SHA256     :",
        checksums_sha,
    )

    print()

    print(
        "STATUS:"
    )

    print(
        "  ALL 50 / 50 FROZEN STAGE23 MODEL FITS HAVE NOW BEEN CONSUMED."
    )

    print(
        "  THE SIX STUMP RESULTS ARE COMPLETE BUT NOT YET SEALED."
    )

    print(
        "  NO ADDITIONAL STAGE23 MODEL FIT IS AUTHORIZED."
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "  ZERO-FIT GIT/GITHUB SEAL OF THE COMPLETE STUMP BLOCK."
    )

    print("=" * 100)


except Exception as exc:

    # =========================================================================
    # FAIL-CLOSED RECOVERY STATE
    # =========================================================================

    state["status"] = (
        "FAILED_INSPECT_BEFORE_ANY_RERUN"
    )

    state["failed_utc"] = (
        utc_now()
    )

    state["fit_in_progress_at_failure"] = (
        state.get(
            "fit_in_progress",
            False,
        )
    )

    state["error_type"] = (
        type(exc).__name__
    )

    state["error"] = (
        str(exc)
    )

    state["traceback"] = (
        traceback.format_exc()
    )

    state[
        "stage23_fits_after_completed"
    ] = (
        44
        + state[
            "completed_stump_fits"
        ]
    )

    state["warning"] = (
        "DO NOT BLINDLY RERUN. "
        "At least execution_state.json must be inspected first, "
        "because one or more stump fits may already have completed."
    )


    write_json(
        STATE_PATH,
        state,
    )


    print()
    print("=" * 100)
    print("FAIL-CLOSED")
    print("=" * 100)

    print(
        "Completed stump fits recorded:",
        state[
            "completed_stump_fits"
        ],
        "/ 6",
    )

    print(
        "Stage23 consumed according to state:",
        state[
            "stage23_fits_after_completed"
        ],
        "/ 50",
    )

    print()
    print(
        "DO NOT BLINDLY RERUN THIS CELL."
    )

    print(
        "Inspect:"
    )

    print(
        STATE_PATH
    )

    print("=" * 100)

    raise

STAGE23-3 — FROZEN DEPTH-1 STUMP CONTROL BLOCK

[OK] HEAD:
  eca21e3f6a918d59c64af92eb701361dde91ada7
[OK] 44/50 parent tag:
  stage23-placebo-boosted-block-complete-v1
[OK] Stage23-0 protocol tag:
  2c36cb669aa1efcd6ca15dfaddecdd0d6fbbb460
[OK] execution-cache seal:
  66e1c6382caa63699766e605df63df02beff894f
[OK] worktree: CLEAN

VERIFY FROZEN STUMP SPEC

[EXACT] stump_spec Git blob:
  c2c8803e7250b1ff781e45dbf4665e172865e555
[EXACT] Stage23-0 checksums SHA256:
  36ae0ded6f9b768943d3781610a30426a4b250d928d4207a6ea36ba40768100b
[EXACT] sklearn:
  1.6.1
[OK] expected stump fits: 6
[OK] features: 3
[OK] splits: 2
[OK] threshold: 0.5
[OK] training-only median imputation
[OK] exact depth-1 tree parameters

VERIFY RECOVERED MINIMAL STUMP CACHE

features/feature_00.float64.dat
  expected: 9865224fb0aa281adb042e5af22f7a972fa32283080c602b380f056b50f12a85
  actual:   9865224fb0aa281adb042e5af22f7a972fa32283080c602b380f056b50f12a85
  [EXACT]
features/feature_58.float64.dat
  expected: 95e36c6fdf

In [7]:
# =============================================================================
# STAGE23-3 — INSPECT EXISTING STUMP EXECUTION STATE
#
# ZERO FITS
# ZERO WRITES
# DO NOT DELETE ANYTHING
# =============================================================================

from pathlib import Path
import json
import os

OUT = Path("/kaggle/working/stage23_3_stump_controls")
STATE = OUT / "execution_state.json"

print("=" * 100)
print("STAGE23-3 — EXISTING OUTPUT INSPECTION")
print("=" * 100)
print()

print("Output directory exists :", OUT.exists())
print("State file exists       :", STATE.exists())

print()
print("=" * 100)
print("DIRECTORY CONTENTS")
print("=" * 100)
print()

if OUT.exists():
    files = sorted(OUT.rglob("*"))

    if not files:
        print("[EMPTY DIRECTORY]")
    else:
        for p in files:
            if p.is_file():
                print(
                    f"[FILE] {p.relative_to(OUT)} "
                    f"({p.stat().st_size:,} bytes)"
                )
            elif p.is_dir():
                print(f"[DIR ] {p.relative_to(OUT)}")

print()
print("=" * 100)
print("EXECUTION STATE")
print("=" * 100)
print()

if STATE.exists():

    state = json.loads(
        STATE.read_text(
            encoding="utf-8"
        )
    )

    print(json.dumps(state, indent=2, sort_keys=True))

    print()
    print("=" * 100)
    print("FIT ACCOUNTING INTERPRETATION")
    print("=" * 100)
    print()

    completed = int(
        state.get("completed_stump_fits", 0)
    )

    in_progress = bool(
        state.get("fit_in_progress", False)
    )

    status = state.get("status")

    print("status                  :", status)
    print("completed stump fits    :", completed, "/ 6")
    print("fit_in_progress         :", in_progress)
    print("Stage23 definite total  :", 44 + completed, "/ 50")

    if completed == 0 and not in_progress:
        print()
        print("[SAFE RESIDUE]")
        print("No stump fit is recorded as consumed.")

    elif completed > 0 and not in_progress:
        print()
        print("[PARTIAL COMPLETION]")
        print(
            f"{completed} stump fit(s) definitely completed."
        )
        print("DO NOT DELETE OR RERUN THEM.")

    elif in_progress:
        print()
        print("[AMBIGUOUS FIT STATE]")
        print(
            "A fit may have been interrupted after starting."
        )
        print("DO NOT RERUN until artifacts/state are reconciled.")

else:

    print("[NO execution_state.json]")

    print()
    print("This usually means the directory was created by an")
    print("earlier pre-fit attempt before any stump execution.")
    print("We will confirm from the directory contents before deleting it.")

print()
print("=" * 100)
print("NO FILES MODIFIED. NO MODEL FITS EXECUTED.")
print("=" * 100)

STAGE23-3 — EXISTING OUTPUT INSPECTION

Output directory exists : True
State file exists       : False

DIRECTORY CONTENTS

[EMPTY DIRECTORY]

EXECUTION STATE

[NO execution_state.json]

This usually means the directory was created by an
earlier pre-fit attempt before any stump execution.
We will confirm from the directory contents before deleting it.

NO FILES MODIFIED. NO MODEL FITS EXECUTED.


In [8]:
from pathlib import Path

OUT = Path("/kaggle/working/stage23_3_stump_controls")

print("=" * 90)
print("STAGE23-3 — REMOVE VERIFIED EMPTY RESIDUE")
print("=" * 90)

if not OUT.exists():
    print("[OK] Directory already absent.")

else:
    contents = list(OUT.iterdir())

    if contents:
        raise RuntimeError(
            "REFUSING deletion: stump directory is no longer empty."
        )

    OUT.rmdir()

    print("[OK] Removed verified-empty directory:")
    print(" ", OUT)

print()
print("Stump fits consumed : 0 / 6")
print("Stage23 state       : 44 / 50")
print("Model fits this cell: 0")
print()
print("[READY] Six-stump execution cell may now run.")
print("=" * 90)

STAGE23-3 — REMOVE VERIFIED EMPTY RESIDUE
[OK] Removed verified-empty directory:
  /kaggle/working/stage23_3_stump_controls

Stump fits consumed : 0 / 6
Stage23 state       : 44 / 50
Model fits this cell: 0

[READY] Six-stump execution cell may now run.


In [12]:
# =============================================================================
# STAGE23-3 — ZERO-FIT FINAL STUMP SEAL + PUSH
#
# INPUT:
#   Stage23 parent: 44 / 50 sealed
#   Stump block   :  6 /  6 complete, unsealed
#
# OUTPUT:
#   Stage23       : 50 / 50 SEALED + PUSHED
#
# ZERO MODEL FITS.
# NO RAW DATA.
# NO PARQUET.
# Reads only:
#   - existing stump artifacts
#   - byte-verified minimal Stage23 cache
#
# Also derives validation leaf distributions/support WITHOUT fitting anything.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import shutil
import os
import gc

import numpy as np
import joblib


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE = Path(
    "/kaggle/working/stage23_3_stump_controls"
)

CACHE = Path(
    "/kaggle/working/stage23_execution_cache_v1"
)

DEST = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_3_stump_controls"
)

EXPECTED_PARENT = (
    "eca21e3f6a918d59c64af92eb701361dde91ada7"
)

EXPECTED_PARENT_TAG = (
    "stage23-placebo-boosted-block-complete-v1"
)

EXPECTED_SUMMARY_SHA = (
    "b78009175e2817b4d2d5f51d9712d5232d279b79e417d19f523bc74821ae89a3"
)

EXPECTED_SOURCE_CHECKSUMS_SHA = (
    "fe0c2879929ea83dad9a11e025575b2262bdae418c77ab23c4e7367de217720a"
)

SEAL_TAG = (
    "stage23-3-stump-controls-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: seal stump controls and complete 50-fit budget"
)

N = 14_412_403
CHRONO_TRAIN_N = 13_818_623

FEATURE_INDEX = {
    "Dst Port": 0,
    "Init Fwd Win Byts": 58,
    "Fwd Seg Size Min": 61,
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def git(
    *args,
    check=True,
    show=False,
):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (
        p.stdout or ""
    ).strip()

    if show and out:
        print(out)

    if check and p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(map(str,args))} failed:\n{out}"
        )

    return p.returncode, out


def distribution(y):
    y = np.asarray(
        y,
        dtype=np.uint8,
    )

    benign = int(
        np.sum(y == 0)
    )

    attack = int(
        np.sum(y == 1)
    )

    support = benign + attack

    return {
        "support": support,
        "benign": benign,
        "attack": attack,
        "attack_rate":
            float(
                attack / support
            )
            if support
            else None,
    }


# =============================================================================
# 2. START
# =============================================================================

print("=" * 100)
print("STAGE23-3 — ZERO-FIT FINAL STUMP SEAL")
print("=" * 100)
print()


# =============================================================================
# 3. REPOSITORY PREFLIGHT
# =============================================================================

if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing: {REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, parent_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)


if branch != "main":
    raise RuntimeError(
        f"Expected main; found {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if status:
    raise RuntimeError(
        "Repository dirty before seal:\n"
        + status
    )

if (
    parent_tag_commit
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "44/50 parent tag mismatch."
    )


# Seal tag must not already exist locally.

_, existing_local_tag = git(
    "tag",
    "--list",
    SEAL_TAG,
)

if existing_local_tag:
    raise RuntimeError(
        f"Seal tag already exists locally: {SEAL_TAG}"
    )


# Nor remotely.

_, remote_tag = git(
    "ls-remote",
    "--tags",
    "origin",
    f"refs/tags/{SEAL_TAG}",
)

if remote_tag:
    raise RuntimeError(
        f"Seal tag already exists remotely: {SEAL_TAG}"
    )


print("[OK] HEAD       :", head)
print("[OK] parent tag :", EXPECTED_PARENT_TAG)
print("[OK] worktree   : CLEAN")


# =============================================================================
# 4. PUSH-AUTH PREFLIGHT BEFORE TOUCHING REPOSITORY
# =============================================================================

print()
print("=" * 100)
print("PUSH AUTHENTICATION PREFLIGHT")
print("=" * 100)
print()


rc, push_test = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
)


if rc != 0:

    print(push_test)

    raise RuntimeError(
        "GitHub push authentication is unavailable in this fresh Kaggle "
        "session. No repository files have been modified."
    )


print("[OK] GitHub push authentication available.")


# =============================================================================
# 5. VERIFY SOURCE STUMP BLOCK
# =============================================================================

print()
print("=" * 100)
print("VERIFY UNSEALED SIX-STUMP BLOCK")
print("=" * 100)
print()


if not SOURCE.is_dir():
    raise RuntimeError(
        f"Stump source missing: {SOURCE}"
    )


STATE = (
    SOURCE
    / "execution_state.json"
)

SUMMARY = (
    SOURCE
    / "stage23_3_stump_controls_summary.json"
)

SOURCE_CHECKSUMS = (
    SOURCE
    / "checksums.sha256"
)


for p in [
    STATE,
    SUMMARY,
    SOURCE_CHECKSUMS,
]:
    if not p.is_file():
        raise RuntimeError(
            f"Required stump artifact missing: {p}"
        )


summary_sha = sha256_file(
    SUMMARY
)

checksums_sha = sha256_file(
    SOURCE_CHECKSUMS
)


print(
    "Summary expected:",
    EXPECTED_SUMMARY_SHA,
)

print(
    "Summary actual:  ",
    summary_sha,
)


if summary_sha != EXPECTED_SUMMARY_SHA:
    raise RuntimeError(
        "Stump summary SHA mismatch."
    )


print()
print(
    "Checksums expected:",
    EXPECTED_SOURCE_CHECKSUMS_SHA,
)

print(
    "Checksums actual:  ",
    checksums_sha,
)


if (
    checksums_sha
    != EXPECTED_SOURCE_CHECKSUMS_SHA
):
    raise RuntimeError(
        "Stump checksum-manifest SHA mismatch."
    )


state = json.loads(
    STATE.read_text(
        encoding="utf-8"
    )
)

summary = json.loads(
    SUMMARY.read_text(
        encoding="utf-8"
    )
)


assert (
    state["status"]
    == "ALL_SIX_STUMPS_COMPLETE_UNSEALED"
)

assert (
    state["completed_stump_fits"]
    == 6
)

assert (
    state["stage23_fits_after_completed"]
    == 50
)

assert (
    state["fit_in_progress"]
    is False
)

assert (
    state["additional_model_fits_authorized"]
    == 0
)


assert (
    summary["status"]
    == "ALL_SIX_STUMPS_COMPLETE_UNSEALED"
)

assert (
    summary["fit_accounting"]["stage23_before"]
    == 44
)

assert (
    summary["fit_accounting"]["stump_fits_completed"]
    == 6
)

assert (
    summary["fit_accounting"]["stage23_after"]
    == 50
)

assert (
    summary["fit_accounting"]["boosted_model_fits_this_block"]
    == 0
)


print()
print("[EXACT] stump fits       : 6 / 6")
print("[EXACT] Stage23 total    : 50 / 50")
print("[EXACT] fit in progress  : NO")
print("[EXACT] further fits     : 0 authorized")


# =============================================================================
# 6. VERIFY EVERY FILE AGAINST SOURCE CHECKSUM MANIFEST
# =============================================================================

manifest_entries = {}


for line in SOURCE_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        maxsplit=1
    )

    manifest_entries[
        filename.strip()
    ] = digest.strip()


for filename, expected in manifest_entries.items():

    p = SOURCE / filename

    if not p.is_file():
        raise RuntimeError(
            f"Manifest artifact missing: {filename}"
        )

    actual = sha256_file(
        p
    )

    if actual != expected:
        raise RuntimeError(
            f"Artifact SHA mismatch: {filename}"
        )


print(
    f"[EXACT] source manifest verified "
    f"for {len(manifest_entries)} files"
)


# =============================================================================
# 7. GITHUB FILE-SIZE SAFETY CHECK
# =============================================================================

print()
print("=" * 100)
print("GITHUB ARTIFACT SIZE CHECK")
print("=" * 100)
print()


largest = (
    None,
    -1,
)


for p in SOURCE.iterdir():

    if not p.is_file():
        continue

    size = p.stat().st_size

    if size > largest[1]:
        largest = (
            p,
            size,
        )

    print(
        f"{p.name:<85} "
        f"{size / (1024**2):>8.2f} MiB"
    )


if (
    largest[1]
    >= 95 * 1024 * 1024
):
    raise RuntimeError(
        "At least one stump artifact is >=95 MiB. "
        "Refusing to create a commit that may exceed GitHub's file limit."
    )


print()
print(
    "[OK] largest artifact:",
    largest[0].name,
    f"{largest[1] / (1024**2):.2f} MiB",
)


# =============================================================================
# 8. DERIVE VALIDATION LEAF DISTRIBUTIONS — ZERO FITS
#
# This is supplementary reporting from ALREADY-FITTED models.
# There are NO .fit() calls below.
# =============================================================================

print()
print("=" * 100)
print("DERIVE VALIDATION LEAF DISTRIBUTIONS — ZERO FITS")
print("=" * 100)
print()


labels = np.memmap(
    CACHE
    / "binary_label.uint8.dat",
    dtype=np.uint8,
    mode="r",
    shape=(N,),
)


packed = np.fromfile(
    CACHE
    / "random_validation.packbits",
    dtype=np.uint8,
)

random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:N].astype(
    np.bool_,
    copy=False,
)


result_files = sorted(
    SOURCE.glob(
        "*_result.json"
    )
)


if len(result_files) != 6:
    raise RuntimeError(
        f"Expected exactly six stump result JSONs; found {len(result_files)}"
    )


validation_leaf_report = {
    "stage":
        "Stage23-3 validation leaf distributions",

    "status":
        "ZERO_FIT_DERIVED_SUPPLEMENT",

    "created_utc":
        now(),

    "model_fits":
        0,

    "source_summary_sha256":
        EXPECTED_SUMMARY_SHA,

    "definition":
        (
            "Validation rows are routed through the already-fitted "
            "training-only median imputer and depth-1 stump. "
            "No estimator or imputer is refit."
        ),

    "results":
        [],
}


for result_file in result_files:

    result = json.loads(
        result_file.read_text(
            encoding="utf-8"
        )
    )

    stage = result["stage"]
    feature = result["cell"]["feature"]
    split = result["cell"]["split"]

    index = FEATURE_INDEX[
        feature
    ]


    model_name = (
        result[
            "model"
        ][
            "model_artifact"
        ]
    )

    model_path = (
        SOURCE
        / model_name
    )


    artifact = joblib.load(
        model_path
    )

    imputer = (
        artifact[
            "imputer"
        ]
    )

    stump = (
        artifact[
            "stump"
        ]
    )


    x = np.memmap(
        CACHE
        / "features"
        / f"feature_{index:02d}.float64.dat",
        dtype="<f8",
        mode="r",
        shape=(N,),
    )


    if split == "RANDOM_NATURAL":

        X_val_raw = np.asarray(
            x[
                random_val_mask
            ],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        y_val = np.asarray(
            labels[
                random_val_mask
            ],
            dtype=np.uint8,
        )


    elif split == "CHRONOLOGICAL_NATURAL":

        X_val_raw = np.asarray(
            x[
                CHRONO_TRAIN_N:
            ],
            dtype=np.float64,
        ).reshape(
            -1,
            1,
        )

        y_val = np.asarray(
            labels[
                CHRONO_TRAIN_N:
            ],
            dtype=np.uint8,
        )


    else:
        raise RuntimeError(
            f"Unexpected split: {split}"
        )


    # TRANSFORM ONLY — no fit.
    X_val = imputer.transform(
        X_val_raw
    )


    tree = stump.tree_

    if int(
        tree.feature[0]
    ) < 0:

        entry = {
            "stage":
                stage,

            "feature":
                feature,

            "split":
                split,

            "split_performed":
                False,

            "validation_root_distribution":
                distribution(
                    y_val
                ),
        }


    else:

        threshold = float(
            tree.threshold[0]
        )

        left = (
            X_val[
                :,
                0
            ]
            <= threshold
        )

        right = ~left


        entry = {
            "stage":
                stage,

            "feature":
                feature,

            "split":
                split,

            "split_performed":
                True,

            "split_threshold":
                threshold,

            "training_imputation_median":
                float(
                    imputer.statistics_[0]
                ),

            "left_validation_leaf":
                distribution(
                    y_val[
                        left
                    ]
                ),

            "right_validation_leaf":
                distribution(
                    y_val[
                        right
                    ]
                ),

            "branch_predicting_attack":
                result[
                    "tree_structure"
                ][
                    "branch_predicting_attack"
                ],
        }


    validation_leaf_report[
        "results"
    ].append(
        entry
    )


    print(
        f"[DERIVED] {stage} — "
        f"{feature} × {split}"
    )


    del (
        artifact,
        imputer,
        stump,
        x,
        X_val_raw,
        X_val,
        y_val,
    )

    gc.collect()


del (
    labels,
    packed,
    random_val_mask,
)

gc.collect()


# =============================================================================
# 9. COPY UNSEALED BLOCK INTO REPOSITORY
# =============================================================================

print()
print("=" * 100)
print("COPY STUMP BLOCK INTO REPOSITORY")
print("=" * 100)
print()


if DEST.exists():
    raise RuntimeError(
        f"Destination already exists: {DEST}"
    )


DEST.mkdir(
    parents=True,
    exist_ok=False,
)


for p in SOURCE.iterdir():

    if p.is_file():

        shutil.copy2(
            p,
            DEST / p.name,
        )


print(
    "[OK] copied source stump artifacts:",
    len(
        [
            p
            for p in SOURCE.iterdir()
            if p.is_file()
        ]
    ),
)


# =============================================================================
# 10. WRITE ZERO-FIT VALIDATION-LEAF SUPPLEMENT
# =============================================================================

VALIDATION_REPORT = (
    DEST
    / "validation_leaf_distributions.json"
)


write_json(
    VALIDATION_REPORT,
    validation_leaf_report,
)


print(
    "[OK] validation leaf supplement SHA256:",
    sha256_file(
        VALIDATION_REPORT
    ),
)


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

seal_receipt = {
    "stage":
        "Stage23-3 stump controls",

    "status":
        "SEALED_COMPLETE_MODEL_FIT_BUDGET",

    "sealed_utc":
        now(),

    "execution_parent": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            EXPECTED_PARENT_TAG,
    },

    "fit_accounting": {
        "primary_boosted":
            "24 / 24",

        "placebo_boosted":
            "20 / 20",

        "depth_1_stumps":
            "6 / 6",

        "stage23_total":
            "50 / 50",

        "additional_stage23_model_fits_authorized":
            0,
    },

    "stump_block": {
        "summary_sha256":
            EXPECTED_SUMMARY_SHA,

        "source_checksums_sha256":
            EXPECTED_SOURCE_CHECKSUMS_SHA,

        "validation_leaf_distributions_sha256":
            sha256_file(
                VALIDATION_REPORT
            ),
    },

    "governance": {
        "new_model_fits_this_seal":
            0,

        "new_boosted_fits_this_seal":
            0,

        "new_stump_fits_this_seal":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "parquet_files_read":
            0,

        "threshold_optimization":
            False,

        "subset_specific_tuning":
            False,

        "stage23_model_fit_budget_exhausted":
            True,
    },

    "scientific_status":
        (
            "All prospectively frozen Stage23 model fits are complete. "
            "Any subsequent Stage23 work must be analysis-only and must "
            "not introduce additional model fits."
        ),

    "tag":
        SEAL_TAG,
}


SEAL_RECEIPT = (
    DEST
    / "seal_receipt.json"
)


write_json(
    SEAL_RECEIPT,
    seal_receipt,
)


# =============================================================================
# 12. README
# =============================================================================

README = """# Stage23-3 — Depth-1 Stump Controls

This directory seals the six prospectively frozen Stage23 depth-1
single-feature stump controls.

## Frozen design

Features:

1. `Dst Port`
2. `Init Fwd Win Byts`
3. `Fwd Seg Size Min`

Each feature was evaluated under:

- `RANDOM_NATURAL`
- `CHRONOLOGICAL_NATURAL`

Implementation:

- `sklearn.tree.DecisionTreeClassifier`
- `max_depth = 1`
- `random_state = 42`
- training-membership-only median imputation
- fixed operating threshold = 0.50

## Fit accounting

- Primary boosted fits: 24 / 24
- Placebo boosted fits: 20 / 20
- Depth-1 stump fits: 6 / 6
- **Stage23 total: 50 / 50**

The Stage23 frozen model-fit budget is exhausted.
No additional Stage23 model fit is authorized.

## Interpretation boundary

The stump controls test whether individual pre-specified features provide
discriminative signal under the frozen random and chronological validation
regimes.

Large random-to-chronological degradation is consistent with split-specific
or shortcut-like signal that transfers poorly. It does not, by itself, prove
that a feature is leakage or establish causality.

Raw March 1 and March 2 data were not accessed.
"""


(
    DEST
    / "README.md"
).write_text(
    README,
    encoding="utf-8",
)


# =============================================================================
# 13. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_MANIFEST = (
    DEST
    / "repository_checksums.sha256"
)


manifest_files = sorted(
    p
    for p in DEST.iterdir()
    if (
        p.is_file()
        and p.name
        != "repository_checksums.sha256"
    )
)


REPO_MANIFEST.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.name}"
        for p in manifest_files
    ) + "\n",
    encoding="utf-8",
)


repo_manifest_sha = (
    sha256_file(
        REPO_MANIFEST
    )
)


print(
    "[OK] repository checksum manifest:",
    repo_manifest_sha,
)


# =============================================================================
# 14. ENSURE COMMIT IDENTITY EXISTS
# =============================================================================

_, user_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

_, user_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not user_name:

    _, parent_author = git(
        "log",
        "-1",
        "--format=%an",
    )

    git(
        "config",
        "user.name",
        parent_author,
    )


if not user_email:

    _, parent_email = git(
        "log",
        "-1",
        "--format=%ae",
    )

    git(
        "config",
        "user.email",
        parent_email,
    )


# =============================================================================
# 15. STAGE ONLY STAGE23-3
# =============================================================================

relative_dest = (
    DEST.relative_to(
        REPO
    )
)


git(
    "add",
    "--",
    str(
        relative_dest
    ),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    x
    for x in staged.splitlines()
    if x.strip()
]


if not staged_files:
    raise RuntimeError(
        "No files staged."
    )


for path in staged_files:

    if not path.startswith(
        str(relative_dest)
        + "/"
    ):
        raise RuntimeError(
            f"Unexpected staged file: {path}"
        )


print(
    "[OK] staged files:",
    len(staged_files),
)


# =============================================================================
# 16. COMMIT
# =============================================================================

print()
print("=" * 100)
print("COMMIT")
print("=" * 100)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, sealed_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] sealed commit:",
    sealed_commit,
)


# =============================================================================
# 17. ANNOTATED TAG
# =============================================================================

git(
    "tag",
    "-a",
    SEAL_TAG,
    sealed_commit,
    "-m",
    (
        "Stage23 complete model-fit budget: "
        "24 primary boosted + 20 placebo boosted + "
        "6 depth-1 stumps = 50/50"
    ),
)


# =============================================================================
# 18. ATOMIC PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 100)
print("ATOMIC PUSH MAIN + TAG")
print("=" * 100)
print()


git(
    "push",
    "--atomic",
    "origin",
    "main",
    f"refs/tags/{SEAL_TAG}",
    show=True,
)


# =============================================================================
# 19. REMOTE VERIFICATION
# =============================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_main_line:
    raise RuntimeError(
        "Remote main could not be resolved."
    )


remote_main = (
    remote_main_line
    .split()[0]
)


_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEAL_TAG}^{{}}",
)


if not remote_tag_line:
    raise RuntimeError(
        "Remote annotated tag could not be peeled."
    )


remote_tag_commit = (
    remote_tag_line
    .split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_main}"
    )


if remote_tag_commit != sealed_commit:

    raise RuntimeError(
        "Remote tag mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_tag_commit}"
    )


# =============================================================================
# 20. FINAL CLEAN WORKTREE
# =============================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:
    raise RuntimeError(
        "Repository dirty after final seal:\n"
        + final_status
    )


# =============================================================================
# 21. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23 MODEL-FIT BUDGET — SEALED AND PUSHED")
print("=" * 100)
print()

print("Commit:")
print(" ", sealed_commit)

print()
print("Tag:")
print(" ", SEAL_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag peeled commit:")
print(" ", remote_tag_commit)

print()
print("Repository manifest SHA256:")
print(" ", repo_manifest_sha)

print()
print("FIT ACCOUNTING")
print("  Primary boosted : 24 / 24")
print("  Placebo boosted : 20 / 20")
print("  Depth-1 stumps  :  6 /  6")
print("  -------------------------")
print("  STAGE23 TOTAL   : 50 / 50")

print()
print("New model fits during seal : 0")
print("Raw Mar1 accessed          : NO")
print("Raw Mar2 accessed          : NO")
print("Parquet files read         : 0")

print()
print("FINAL MODEL-FIT STATUS:")
print("  STAGE23 MODEL-FIT BUDGET EXHAUSTED.")
print("  NO ADDITIONAL STAGE23 MODEL FIT IS AUTHORIZED.")

print()
print("NEXT:")
print("  Continue with frozen analysis-only Stage23 work")
print("  (uncertainty / SHAP proxy-shift / attack-family / figures as applicable).")

print("=" * 100)

STAGE23-3 — ZERO-FIT FINAL STUMP SEAL

[OK] HEAD       : eca21e3f6a918d59c64af92eb701361dde91ada7
[OK] parent tag : stage23-placebo-boosted-block-complete-v1
[OK] worktree   : CLEAN

PUSH AUTHENTICATION PREFLIGHT

[OK] GitHub push authentication available.

VERIFY UNSEALED SIX-STUMP BLOCK

Summary expected: b78009175e2817b4d2d5f51d9712d5232d279b79e417d19f523bc74821ae89a3
Summary actual:   b78009175e2817b4d2d5f51d9712d5232d279b79e417d19f523bc74821ae89a3

Checksums expected: fe0c2879929ea83dad9a11e025575b2262bdae418c77ab23c4e7367de217720a
Checksums actual:   fe0c2879929ea83dad9a11e025575b2262bdae418c77ab23c4e7367de217720a

[EXACT] stump fits       : 6 / 6
[EXACT] Stage23 total    : 50 / 50
[EXACT] fit in progress  : NO
[EXACT] further fits     : 0 authorized
[EXACT] source manifest verified for 20 files

GITHUB ARTIFACT SIZE CHECK

stage23_3a_dst_port_random_natural_model.joblib                                           0.00 MiB
stage23_3f_fwd_seg_size_min_chronological_natural_model.job

In [11]:
# =============================================================================
# RESTORE GITHUB PUSH AUTH — ZERO FITS / ZERO SCIENTIFIC WRITES
# =============================================================================

from pathlib import Path
import subprocess
import os

from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
EXPECTED_HEAD = "eca21e3f6a918d59c64af92eb701361dde91ada7"

print("=" * 90)
print("RESTORE GITHUB PUSH AUTH")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. Scientific/repository state must still be untouched
# ---------------------------------------------------------------------------

def git(*args, check=True):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if check and p.returncode != 0:
        raise RuntimeError(out)

    return p.returncode, out


_, head = git("rev-parse", "HEAD")
_, status = git("status", "--porcelain")

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"Unexpected HEAD:\nexpected={EXPECTED_HEAD}\nactual={head}"
    )

if status:
    raise RuntimeError(
        "Repository is dirty before authentication:\n" + status
    )

print("[OK] HEAD     :", head)
print("[OK] worktree : CLEAN")


# ---------------------------------------------------------------------------
# 2. Retrieve GitHub token from Kaggle Secrets
#
# Nothing is printed except the secret LABEL that succeeded.
# ---------------------------------------------------------------------------

client = UserSecretsClient()

candidate_labels = [
    "GITHUB_TOKEN",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_token",
    "github_pat",
    "github",
]

token = None
used_label = None

for label in candidate_labels:
    try:
        value = client.get_secret(label)

        if value and value.strip():
            token = value.strip()
            used_label = label
            break

    except Exception:
        pass


if token is None:
    raise RuntimeError(
        "No usable GitHub token was found in Kaggle Secrets.\n\n"
        "Checked labels:\n  "
        + "\n  ".join(candidate_labels)
        + "\n\nAdd/enable your existing GitHub token secret for this notebook, "
          "then run this cell again."
    )


print("[OK] Kaggle secret loaded:", used_label)
print("[OK] token value          : HIDDEN")


# ---------------------------------------------------------------------------
# 3. Keep credential in CURRENT KERNEL ENVIRONMENT ONLY
# ---------------------------------------------------------------------------

os.environ["GITHUB_TOKEN"] = token
os.environ["GH_TOKEN"] = token

# Do not embed the actual token in the remote URL or git config.
#
# Git invokes this helper and reads GITHUB_TOKEN from the inherited process
# environment.

helper = (
    '!f() { '
    'if [ "$1" = "get" ]; then '
    'echo "username=x-access-token"; '
    'echo "password=$GITHUB_TOKEN"; '
    'fi; '
    '}; f'
)

git(
    "config",
    "--local",
    "--unset-all",
    "credential.helper",
    check=False,
)

git(
    "config",
    "--local",
    "credential.helper",
    helper,
)


# Ensure normal HTTPS origin.

_, remote = git(
    "remote",
    "get-url",
    "origin",
)

print("[OK] origin:", remote)


# ---------------------------------------------------------------------------
# 4. Authentication verification — DRY RUN ONLY
# ---------------------------------------------------------------------------

rc, output = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
)

if rc != 0:
    print()
    print(output)

    raise RuntimeError(
        "GitHub credential was found, but push authentication still failed."
    )


print()
print("[OK] GitHub push authentication restored.")
print("[OK] dry-run push succeeded.")
print()
print("Repository modifications : 0")
print("Model fits               : 0")
print("Stage23 scientific state : 50 / 50 COMPLETE, UNSEALED")
print()
print("NEXT: rerun the ZERO-FIT final stump seal + push cell.")
print("=" * 90)

RESTORE GITHUB PUSH AUTH
[OK] HEAD     : eca21e3f6a918d59c64af92eb701361dde91ada7
[OK] worktree : CLEAN
[OK] Kaggle secret loaded: GITHUB_TOKEN
[OK] token value          : HIDDEN
[OK] origin: https://github.com/themubasshir/ids2018-validation-safe-ablation.git

[OK] GitHub push authentication restored.
[OK] dry-run push succeeded.

Repository modifications : 0
Model fits               : 0
Stage23 scientific state : 50 / 50 COMPLETE, UNSEALED

NEXT: rerun the ZERO-FIT final stump seal + push cell.


In [13]:
# =============================================================================
# STAGE23-4 — FROZEN UNCERTAINTY ANALYSIS PREFLIGHT
#
# ZERO MODEL FITS
# ZERO SCIENTIFIC WRITES
#
# Purpose:
#   Verify the 50/50 Stage23 seal, frozen uncertainty specification/cohorts,
#   and inventory all FULL / primary / placebo validation-probability artifacts
#   before executing the prospectively frozen bootstrap.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import os

import numpy as np
import pandas as pd


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "f2cd2b59172179f9f0884d88e579276517de7a01"
)

EXPECTED_TAG = (
    "stage23-3-stump-controls-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

UNCERTAINTY_SPEC = (
    PROTOCOL
    / "uncertainty_spec.json"
)

RANDOM_COHORT = (
    PROTOCOL
    / "uncertainty_cohort_random_natural.csv"
)

CHRONO_COHORT = (
    PROTOCOL
    / "uncertainty_cohort_chronological_natural.csv"
)

PROTOCOL_CHECKSUMS = (
    PROTOCOL
    / "checksums.sha256"
)

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)

PLACEBO_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

FULL_RANDOM = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "random_natural_validation_ensemble_probabilities.npz"
)

FULL_CHRONO_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)


def git(*args):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return (p.stdout or "").strip()


def sha256_file(path, chunk=32 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)

    return h.hexdigest()


def parse_manifest(path):
    out = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        out[rel.strip()] = digest.strip()

    return out


print("=" * 100)
print("STAGE23-4 — UNCERTAINTY PREFLIGHT")
print("=" * 100)
print()


# =============================================================================
# 1. SEALED PARENT
# =============================================================================

head = git(
    "rev-parse",
    "HEAD",
)

tag = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)

assert head == EXPECTED_HEAD
assert tag == EXPECTED_HEAD
assert status == ""

print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 — no additional fits authorized")


# =============================================================================
# 2. VERIFY FROZEN UNCERTAINTY FILES USING STAGE23-0 MANIFEST
# =============================================================================

manifest = parse_manifest(
    PROTOCOL_CHECKSUMS
)

for p in [
    UNCERTAINTY_SPEC,
    RANDOM_COHORT,
    CHRONO_COHORT,
]:

    expected = manifest.get(
        p.name
    )

    if expected is None:
        raise RuntimeError(
            f"{p.name} absent from frozen protocol checksum manifest"
        )

    actual = sha256_file(
        p
    )

    print()
    print(p.name)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:
        raise RuntimeError(
            f"Frozen protocol SHA mismatch: {p.name}"
        )

    print("  [EXACT]")


# =============================================================================
# 3. VERIFY SPEC
# =============================================================================

spec = json.loads(
    UNCERTAINTY_SPEC.read_text(
        encoding="utf-8"
    )
)

assert spec["bootstrap_replicates"] == 1000
assert spec["cohort_size_per_split"] == 50000
assert spec["cohort_seed"] == 42

assert spec["random_cohort"]["rows"] == 50000
assert spec["random_cohort"]["benign"] == 43158
assert spec["random_cohort"]["attack"] == 6842

assert spec["chronological_cohort"]["rows"] == 50000
assert spec["chronological_cohort"]["benign"] == 44758
assert spec["chronological_cohort"]["attack"] == 5242

print()
print("=" * 100)
print("FROZEN UNCERTAINTY SPEC")
print("=" * 100)
print()

print("Bootstrap replicates :", spec["bootstrap_replicates"])
print("Cohort rows/split    :", spec["cohort_size_per_split"])
print("Cohort seed          :", spec["cohort_seed"])
print("Method               :", spec["method"])
print("CI                    :", spec["confidence_interval"])
print("Pairing               :", spec["pairing"])


# =============================================================================
# 4. INSPECT FROZEN COHORTS
# =============================================================================

random_cohort = pd.read_csv(
    RANDOM_COHORT
)

chrono_cohort = pd.read_csv(
    CHRONO_COHORT
)


if len(random_cohort) != 50000:
    raise RuntimeError(
        "Random uncertainty cohort row count mismatch."
    )

if len(chrono_cohort) != 50000:
    raise RuntimeError(
        "Chronological uncertainty cohort row count mismatch."
    )


print()
print("=" * 100)
print("UNCERTAINTY COHORT SCHEMA")
print("=" * 100)
print()

print("RANDOM columns:")
print(" ", list(random_cohort.columns))

print()
print("CHRONO columns:")
print(" ", list(chrono_cohort.columns))

print()
print("RANDOM rows :", len(random_cohort))
print("CHRONO rows :", len(chrono_cohort))


# =============================================================================
# 5. LOCATE FULL PROBABILITY VECTORS
# =============================================================================

if not FULL_RANDOM.is_file():
    raise RuntimeError(
        f"FULL random probability artifact missing:\n{FULL_RANDOM}"
    )


chrono_candidates = sorted(
    FULL_CHRONO_DIR.glob(
        "*validation*probabilit*.npz"
    )
)

if len(chrono_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one FULL chronological probability artifact; "
        f"found {len(chrono_candidates)}"
    )

FULL_CHRONO = chrono_candidates[0]


print()
print("=" * 100)
print("FULL PROBABILITY ARTIFACTS")
print("=" * 100)
print()


for split, p in [
    ("RANDOM_NATURAL", FULL_RANDOM),
    ("CHRONOLOGICAL_NATURAL", FULL_CHRONO),
]:

    with np.load(
        p,
        allow_pickle=False,
    ) as z:

        print(split)
        print("  path :", p.relative_to(REPO))
        print("  SHA  :", sha256_file(p))
        print("  keys :", list(z.files))

        for key in z.files:
            a = z[key]
            print(
                f"    {key:<30} "
                f"shape={a.shape} dtype={a.dtype}"
            )


# =============================================================================
# 6. INVENTORY PRIMARY + PLACEBO RESULT LEAVES
#
# Each Stage23 leaf must contain:
#   exactly one *_result.json
#   exactly one *validation_probabilities.npz
# =============================================================================

def inventory_block(
    root,
    expected_leaf_count,
    label,
):

    leaves = []

    for result_json in sorted(
        root.rglob("*_result.json")
    ):

        d = result_json.parent

        probs = sorted(
            d.glob(
                "*validation_probabilities.npz"
            )
        )

        if len(probs) != 1:
            raise RuntimeError(
                f"{label}: {d} has {len(probs)} "
                "validation-probability artifacts"
            )

        result = json.loads(
            result_json.read_text(
                encoding="utf-8"
            )
        )

        with np.load(
            probs[0],
            allow_pickle=False,
        ) as z:

            leaves.append(
                {
                    "stage":
                        result.get(
                            "stage",
                            "UNKNOWN",
                        ),

                    "directory":
                        str(
                            d.relative_to(
                                REPO
                            )
                        ),

                    "result_json":
                        result_json.name,

                    "probability_file":
                        probs[0].name,

                    "probability_sha256":
                        sha256_file(
                            probs[0]
                        ),

                    "npz_keys":
                        list(
                            z.files
                        ),

                    "npz_shapes":
                        {
                            key:
                                list(
                                    z[key].shape
                                )
                            for key
                            in z.files
                        },

                    "npz_dtypes":
                        {
                            key:
                                str(
                                    z[key].dtype
                                )
                            for key
                            in z.files
                        },
                }
            )


    if len(leaves) != expected_leaf_count:
        raise RuntimeError(
            f"{label}: expected {expected_leaf_count} results, "
            f"found {len(leaves)}"
        )


    print()
    print("=" * 100)
    print(label)
    print("=" * 100)
    print()

    for item in leaves:

        print(
            f"{item['stage']:<12} "
            f"{item['directory']}"
        )

        print(
            "  probability:",
            item[
                "probability_file"
            ],
        )

        print(
            "  keys:",
            item[
                "npz_keys"
            ],
        )


    return leaves


primary = inventory_block(
    PRIMARY_ROOT,
    expected_leaf_count=12,
    label="PRIMARY REDUCED ABLATIONS — 12 SPLIT CELLS",
)

placebo = inventory_block(
    PLACEBO_ROOT,
    expected_leaf_count=10,
    label="PLACEBO ABLATIONS — 10 SPLIT CELLS",
)


# =============================================================================
# 7. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23 UNCERTAINTY PREFLIGHT COMPLETE")
print("=" * 100)
print()

print("Stage23 model fits       : 50 / 50 SEALED")
print("New model fits           : 0")
print("Scientific writes        : 0")

print()
print("Frozen cohorts verified  : 2 / 2")
print("FULL probability vectors : 2 / 2")
print("Primary split cells      :", len(primary), "/ 12")
print("Placebo split cells      :", len(placebo), "/ 10")

print()
print("NEXT:")
print("  Execute frozen 1,000-replicate paired stratified bootstrap")
print("  using these exact probability artifacts and frozen cohorts.")
print("=" * 100)

STAGE23-4 — UNCERTAINTY PREFLIGHT

[OK] HEAD      : f2cd2b59172179f9f0884d88e579276517de7a01
[OK] seal tag  : stage23-3-stump-controls-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 — no additional fits authorized

uncertainty_spec.json
  expected: 36cd396892fcf8f01e598ff885c010dde5725fd1bfb1816d07183eacda6e5b02
  actual:   36cd396892fcf8f01e598ff885c010dde5725fd1bfb1816d07183eacda6e5b02
  [EXACT]

uncertainty_cohort_random_natural.csv
  expected: 38d8c08f61ebf9806d28bcefe5bd562a3aa2a9fd5577c083234730a3254d260f
  actual:   38d8c08f61ebf9806d28bcefe5bd562a3aa2a9fd5577c083234730a3254d260f
  [EXACT]

uncertainty_cohort_chronological_natural.csv
  expected: 6361a0bfbc070bd4610690cb443bf4f9f6a322d789c5363d8e5d047526aab8a7
  actual:   6361a0bfbc070bd4610690cb443bf4f9f6a322d789c5363d8e5d047526aab8a7
  [EXACT]

FROZEN UNCERTAINTY SPEC

Bootstrap replicates : 1000
Cohort rows/split    : 50000
Cohort seed          : 42
Method               : paired stratified bootstrap on a frozen n

In [14]:
# =============================================================================
# STAGE23-4 — FROZEN PAIRED STRATIFIED BOOTSTRAP UNCERTAINTY ANALYSIS
#
# ANALYSIS ONLY — ZERO MODEL FITS
#
# 11 comparisons:
#   PRIMARY (6)
#     NO_DST_PORT
#     NO_PORTS
#     NO_INIT_FWD_WIN_BYTS
#     NO_FWD_SEG_SIZE_MIN
#     NO_SUSPICIOUS_GROUP
#     BEHAVIOR_ONLY
#
#   PLACEBO (5)
#     PLACEBO_COUNTS
#     PLACEBO_VOLUME_DIRECTION
#     PLACEBO_IAT
#     PLACEBO_PACKET_SIZE
#     PLACEBO_ACTIVITY
#
# Frozen uncertainty design:
#   - 50,000 rows / split
#   - RANDOM: 43,158 benign + 6,842 attack
#   - CHRONO: 44,758 benign + 5,242 attack
#   - 1,000 bootstrap replicates
#   - RandomState(42), restarted independently per split
#   - benign and attack resampled separately WITH replacement
#   - FULL and every ablated model use SAME sampled rows within a split/replicate
#   - percentile 95% CI [2.5, 97.5]
#
# Headline:
#   PR_AUC_removal_penalty
#   ROC_AUC_removal_penalty
#   PR_AUC_shortcut_interaction
#   ROC_AUC_shortcut_interaction
#
# Removal penalty:
#   DELTA_R(S) = FULL_R - ABLATED_R
#   DELTA_C(S) = FULL_C - ABLATED_C
#
# Interaction:
#   I(S) = DELTA_R(S) - DELTA_C(S)
#
# NO RAW MAR1
# NO RAW MAR2
# NO PARQUET
# NO MODEL FIT
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# =============================================================================
# 0. FROZEN PATHS / CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "f2cd2b59172179f9f0884d88e579276517de7a01"
)

EXPECTED_TAG = (
    "stage23-3-stump-controls-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

UNCERTAINTY_SPEC = (
    PROTOCOL
    / "uncertainty_spec.json"
)

RANDOM_COHORT_PATH = (
    PROTOCOL
    / "uncertainty_cohort_random_natural.csv"
)

CHRONO_COHORT_PATH = (
    PROTOCOL
    / "uncertainty_cohort_chronological_natural.csv"
)

PROTOCOL_CHECKSUMS = (
    PROTOCOL
    / "checksums.sha256"
)

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)

PLACEBO_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

FULL_RANDOM_PROB = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "random_natural_validation_ensemble_probabilities.npz"
)

FULL_RANDOM_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

FULL_CHRONO_PROB = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "chronological_natural_validation_ensemble_probabilities.npz"
)

FULL_CHRONO_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

EXPECTED_FULL_PROB_SHA = {
    "RANDOM_NATURAL":
        "9dc64ccdb6580dad727668ff3e511e0a8572cbb1de49b9ffe8999b0c3d190394",

    "CHRONOLOGICAL_NATURAL":
        "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f",
}

OUT = Path(
    "/kaggle/working/stage23_4_uncertainty_analysis"
)

BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 42


COMPARISONS = [
    {
        "name": "NO_DST_PORT",
        "family": "PRIMARY",
        "slug": "no_dst_port",
    },
    {
        "name": "NO_PORTS",
        "family": "PRIMARY",
        "slug": "no_ports",
    },
    {
        "name": "NO_INIT_FWD_WIN_BYTS",
        "family": "PRIMARY",
        "slug": "no_init_fwd_win_byts",
    },
    {
        "name": "NO_FWD_SEG_SIZE_MIN",
        "family": "PRIMARY",
        "slug": "no_fwd_seg_size_min",
    },
    {
        "name": "NO_SUSPICIOUS_GROUP",
        "family": "PRIMARY",
        "slug": "no_suspicious_group",
    },
    {
        "name": "BEHAVIOR_ONLY",
        "family": "PRIMARY",
        "slug": "behavior_only",
    },

    {
        "name": "PLACEBO_COUNTS",
        "family": "PLACEBO",
        "slug": "placebo_counts",
    },
    {
        "name": "PLACEBO_VOLUME_DIRECTION",
        "family": "PLACEBO",
        "slug": "placebo_volume_direction",
    },
    {
        "name": "PLACEBO_IAT",
        "family": "PLACEBO",
        "slug": "placebo_iat",
    },
    {
        "name": "PLACEBO_PACKET_SIZE",
        "family": "PLACEBO",
        "slug": "placebo_packet_size",
    },
    {
        "name": "PLACEBO_ACTIVITY",
        "family": "PLACEBO",
        "slug": "placebo_activity",
    },
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def parse_manifest(path):

    result = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        result[
            rel.strip()
        ] = digest.strip()

    return result


def metric_pair(
    y,
    probability,
):
    return (
        float(
            average_precision_score(
                y,
                probability,
            )
        ),
        float(
            roc_auc_score(
                y,
                probability,
            )
        ),
    )


def ci95(values):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    q = np.percentile(
        values,
        [2.5, 97.5],
    )

    return {
        "lower_2_5":
            float(q[0]),

        "upper_97_5":
            float(q[1]),
    }


def locate_cohort_rows(
    probability_clean_position,
    cohort_clean_position,
):

    cp = np.asarray(
        probability_clean_position,
        dtype=np.int64,
    )

    target = np.asarray(
        cohort_clean_position,
        dtype=np.int64,
    )

    # Fast path if the full validation artifact already contains
    # cohort positions in a searchable canonical order.
    order = np.argsort(
        cp,
        kind="mergesort",
    )

    cp_sorted = cp[
        order
    ]

    loc = np.searchsorted(
        cp_sorted,
        target,
    )

    if np.any(
        loc >= len(cp_sorted)
    ):
        raise RuntimeError(
            "Cohort clean_position not found in validation artifact."
        )

    if not np.array_equal(
        cp_sorted[loc],
        target,
    ):
        raise RuntimeError(
            "Frozen uncertainty cohort does not map exactly "
            "onto validation clean_position."
        )

    original_indices = (
        order[
            loc
        ]
    )

    if not np.array_equal(
        cp[
            original_indices
        ],
        target,
    ):
        raise RuntimeError(
            "clean_position mapping verification failed."
        )

    return original_indices


def get_leaf(
    comparison,
    split,
):

    split_dir = (
        "random"
        if split == "RANDOM_NATURAL"
        else "chronological"
    )

    root = (
        PRIMARY_ROOT
        if comparison["family"] == "PRIMARY"
        else PLACEBO_ROOT
    )

    d = (
        root
        / split_dir
        / comparison["slug"]
    )

    if not d.is_dir():
        raise RuntimeError(
            f"Missing comparison directory: {d}"
        )

    result_files = sorted(
        d.glob(
            "*_result.json"
        )
    )

    probability_files = sorted(
        d.glob(
            "*validation_probabilities.npz"
        )
    )

    if len(result_files) != 1:
        raise RuntimeError(
            f"{d}: expected 1 result JSON; "
            f"found {len(result_files)}"
        )

    if len(probability_files) != 1:
        raise RuntimeError(
            f"{d}: expected 1 probability NPZ; "
            f"found {len(probability_files)}"
        )

    return (
        result_files[0],
        probability_files[0],
    )


def result_point_metrics(
    result_path,
):

    r = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    if "ranking_metrics" not in r:
        raise RuntimeError(
            f"ranking_metrics missing: {result_path}"
        )

    return {
        "pr_auc":
            float(
                r[
                    "ranking_metrics"
                ][
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                r[
                    "ranking_metrics"
                ][
                    "roc_auc"
                ]
            ),

        "stage":
            r.get(
                "stage",
                "UNKNOWN",
            ),
    }


def full_result_point_metrics(
    result_path,
):

    r = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    vp = (
        r[
            "validation_probability"
        ]
    )

    return {
        "pr_auc":
            float(
                vp[
                    "pr_auc"
                ]
            ),

        "roc_auc":
            float(
                vp[
                    "roc_auc"
                ]
            ),
    }


# =============================================================================
# 2. START / FAIL-CLOSED OUTPUT
# =============================================================================

print("=" * 100)
print("STAGE23-4 — FROZEN UNCERTAINTY ANALYSIS")
print("=" * 100)
print()


if OUT.exists():
    raise RuntimeError(
        "Uncertainty output directory already exists.\n"
        "Do not overwrite it blindly:\n"
        f"{OUT}"
    )


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


STATE_PATH = (
    OUT
    / "execution_state.json"
)


state = {
    "stage":
        "Stage23-4 uncertainty analysis",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "model_fits":
        0,

    "bootstrap_replicates_expected":
        1000,

    "bootstrap_random_completed":
        0,

    "bootstrap_chronological_completed":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "parquet_files_read":
        0,
}


write_json(
    STATE_PATH,
    state,
)


# =============================================================================
# 3. SEALED HEAD / WORKTREE
# =============================================================================

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

worktree = git(
    "status",
    "--porcelain",
)


assert head == EXPECTED_HEAD
assert tag_commit == EXPECTED_HEAD
assert worktree == ""


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50")
print("[OK] new fits  : 0 authorized")


# =============================================================================
# 4. FROZEN SPEC + COHORT VERIFICATION
# =============================================================================

manifest = parse_manifest(
    PROTOCOL_CHECKSUMS
)


for p in [
    UNCERTAINTY_SPEC,
    RANDOM_COHORT_PATH,
    CHRONO_COHORT_PATH,
]:

    expected = manifest.get(
        p.name
    )

    actual = sha256_file(
        p
    )

    if expected is None:
        raise RuntimeError(
            f"Frozen manifest missing {p.name}"
        )

    if actual != expected:
        raise RuntimeError(
            f"SHA mismatch: {p.name}"
        )


spec = json.loads(
    UNCERTAINTY_SPEC.read_text(
        encoding="utf-8"
    )
)


assert (
    spec[
        "bootstrap_replicates"
    ]
    == BOOTSTRAP_REPLICATES
)

assert (
    spec[
        "bootstrap_rng"
    ]
    == (
        "numpy.random.RandomState(seed=42), "
        "restarted independently per split"
    )
)

assert (
    spec[
        "cohort_size_per_split"
    ]
    == 50_000
)

assert (
    spec[
        "cohort_seed"
    ]
    == 42
)

assert (
    spec[
        "random_cohort"
    ][
        "benign"
    ]
    == 43_158
)

assert (
    spec[
        "random_cohort"
    ][
        "attack"
    ]
    == 6_842
)

assert (
    spec[
        "chronological_cohort"
    ][
        "benign"
    ]
    == 44_758
)

assert (
    spec[
        "chronological_cohort"
    ][
        "attack"
    ]
    == 5_242
)


print()
print("[EXACT] frozen uncertainty specification")


# =============================================================================
# 5. LOAD FROZEN COHORTS
# =============================================================================

random_cohort = pd.read_csv(
    RANDOM_COHORT_PATH
)

chrono_cohort = pd.read_csv(
    CHRONO_COHORT_PATH
)


for df, split, benign, attack in [
    (
        random_cohort,
        "RANDOM_NATURAL",
        43_158,
        6_842,
    ),
    (
        chrono_cohort,
        "CHRONOLOGICAL_NATURAL",
        44_758,
        5_242,
    ),
]:

    required = [
        "split",
        "clean_position",
        "binary_label",
    ]

    if list(
        df.columns
    ) != required:
        raise RuntimeError(
            f"{split}: unexpected cohort schema."
        )

    if len(df) != 50_000:
        raise RuntimeError(
            f"{split}: cohort row count mismatch."
        )

    if int(
        (
            df[
                "binary_label"
            ]
            == 0
        ).sum()
    ) != benign:
        raise RuntimeError(
            f"{split}: benign cohort count mismatch."
        )

    if int(
        (
            df[
                "binary_label"
            ]
            == 1
        ).sum()
    ) != attack:
        raise RuntimeError(
            f"{split}: attack cohort count mismatch."
        )


print(
    "[EXACT] RANDOM cohort : "
    "43,158 benign + 6,842 attack"
)

print(
    "[EXACT] CHRONO cohort : "
    "44,758 benign + 5,242 attack"
)


# =============================================================================
# 6. FULL SEALED POINT ESTIMATES
# =============================================================================

full_point = {
    "RANDOM_NATURAL":
        full_result_point_metrics(
            FULL_RANDOM_RESULT
        ),

    "CHRONOLOGICAL_NATURAL":
        full_result_point_metrics(
            FULL_CHRONO_RESULT
        ),
}


print()
print("=" * 100)
print("SEALED FULL POINT ESTIMATES")
print("=" * 100)
print()


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print(split)

    print(
        "  PR-AUC :",
        f"{full_point[split]['pr_auc']:.12f}",
    )

    print(
        "  ROC-AUC:",
        f"{full_point[split]['roc_auc']:.12f}",
    )


# =============================================================================
# 7. LOAD FULL COHORT PROBABILITIES
# =============================================================================

split_config = {
    "RANDOM_NATURAL": {
        "cohort":
            random_cohort,

        "full_probability_path":
            FULL_RANDOM_PROB,
    },

    "CHRONOLOGICAL_NATURAL": {
        "cohort":
            chrono_cohort,

        "full_probability_path":
            FULL_CHRONO_PROB,
    },
}


cohort_data = {}


for split, cfg in split_config.items():

    p = (
        cfg[
            "full_probability_path"
        ]
    )

    actual_sha = sha256_file(
        p
    )

    if (
        actual_sha
        != EXPECTED_FULL_PROB_SHA[
            split
        ]
    ):
        raise RuntimeError(
            f"{split}: FULL probability SHA mismatch."
        )


    with np.load(
        p,
        allow_pickle=False,
    ) as z:

        required_keys = {
            "clean_position",
            "binary_label",
            "ensemble_probability",
        }

        if set(
            z.files
        ) != required_keys:
            raise RuntimeError(
                f"{split}: unexpected FULL NPZ keys."
            )

        full_cp = np.asarray(
            z[
                "clean_position"
            ],
            dtype=np.int64,
        )

        full_y = np.asarray(
            z[
                "binary_label"
            ],
            dtype=np.uint8,
        )

        full_probability = np.asarray(
            z[
                "ensemble_probability"
            ],
            dtype=np.float64,
        )


    cohort_cp = np.asarray(
        cfg[
            "cohort"
        ][
            "clean_position"
        ],
        dtype=np.int64,
    )

    cohort_y = np.asarray(
        cfg[
            "cohort"
        ][
            "binary_label"
        ],
        dtype=np.uint8,
    )


    cohort_indices = locate_cohort_rows(
        full_cp,
        cohort_cp,
    )


    if not np.array_equal(
        full_y[
            cohort_indices
        ],
        cohort_y,
    ):
        raise RuntimeError(
            f"{split}: FULL probability labels do not "
            "match frozen uncertainty cohort."
        )


    cohort_probability = (
        full_probability[
            cohort_indices
        ]
    )


    full_cohort_pr, full_cohort_roc = (
        metric_pair(
            cohort_y,
            cohort_probability,
        )
    )


    cohort_data[
        split
    ] = {
        "clean_position":
            cohort_cp,

        "y":
            cohort_y,

        "probabilities": {
            "FULL":
                cohort_probability,
        },

        "cohort_point_metrics": {
            "FULL": {
                "pr_auc":
                    full_cohort_pr,

                "roc_auc":
                    full_cohort_roc,
            }
        },

        "sealed_full_validation_point":
            full_point[
                split
            ],
    }


    print()
    print(
        f"[EXACT] {split} FULL cohort mapped"
    )

    print(
        f"        cohort PR  = "
        f"{full_cohort_pr:.12f}"
    )

    print(
        f"        cohort ROC = "
        f"{full_cohort_roc:.12f}"
    )


    del (
        full_cp,
        full_y,
        full_probability,
        cohort_indices,
    )

    gc.collect()


# =============================================================================
# 8. LOAD ALL 22 ABLATED SPLIT-CELL PROBABILITIES
# =============================================================================

sealed_ablated_points = {}


print()
print("=" * 100)
print("LOAD ABLATED COHORT PROBABILITIES")
print("=" * 100)
print()


for comparison in COMPARISONS:

    name = (
        comparison[
            "name"
        ]
    )

    sealed_ablated_points[
        name
    ] = {}


    for split in [
        "RANDOM_NATURAL",
        "CHRONOLOGICAL_NATURAL",
    ]:

        result_path, probability_path = (
            get_leaf(
                comparison,
                split,
            )
        )


        point = result_point_metrics(
            result_path
        )


        with np.load(
            probability_path,
            allow_pickle=False,
        ) as z:

            required = {
                "clean_position",
                "binary_label",
                "lightgbm_probability",
                "xgboost_probability",
                "ensemble_probability",
            }

            if set(
                z.files
            ) != required:
                raise RuntimeError(
                    f"{name} × {split}: "
                    "unexpected probability NPZ schema."
                )


            cp = np.asarray(
                z[
                    "clean_position"
                ],
                dtype=np.int64,
            )

            y = np.asarray(
                z[
                    "binary_label"
                ],
                dtype=np.uint8,
            )

            probability = np.asarray(
                z[
                    "ensemble_probability"
                ],
                dtype=np.float64,
            )


        target_cp = (
            cohort_data[
                split
            ][
                "clean_position"
            ]
        )

        target_y = (
            cohort_data[
                split
            ][
                "y"
            ]
        )


        indices = locate_cohort_rows(
            cp,
            target_cp,
        )


        if not np.array_equal(
            y[
                indices
            ],
            target_y,
        ):
            raise RuntimeError(
                f"{name} × {split}: "
                "cohort labels do not match."
            )


        cohort_probability = (
            probability[
                indices
            ]
        )


        cohort_pr, cohort_roc = (
            metric_pair(
                target_y,
                cohort_probability,
            )
        )


        cohort_data[
            split
        ][
            "probabilities"
        ][
            name
        ] = (
            cohort_probability
        )


        cohort_data[
            split
        ][
            "cohort_point_metrics"
        ][
            name
        ] = {
            "pr_auc":
                cohort_pr,

            "roc_auc":
                cohort_roc,
        }


        sealed_ablated_points[
            name
        ][
            split
        ] = {
            "pr_auc":
                point[
                    "pr_auc"
                ],

            "roc_auc":
                point[
                    "roc_auc"
                ],

            "stage":
                point[
                    "stage"
                ],

            "result_file":
                str(
                    result_path.relative_to(
                        REPO
                    )
                ),

            "probability_file":
                str(
                    probability_path.relative_to(
                        REPO
                    )
                ),

            "probability_sha256":
                sha256_file(
                    probability_path
                ),
        }


        print(
            f"[LOADED] "
            f"{name:<29} "
            f"{split:<23} "
            f"cohort PR={cohort_pr:.9f} "
            f"ROC={cohort_roc:.9f}"
        )


        del (
            cp,
            y,
            probability,
            indices,
        )

        gc.collect()


# =============================================================================
# 9. POINT ESTIMATES ON COMPLETE FROZEN VALIDATION MEMBERSHIPS
#
# These remain the PRIMARY point estimates.
# Bootstrap below is supplementary uncertainty on frozen 50k cohorts.
# =============================================================================

point_summary = {}


for comparison in COMPARISONS:

    name = (
        comparison[
            "name"
        ]
    )


    r_abl = (
        sealed_ablated_points[
            name
        ][
            "RANDOM_NATURAL"
        ]
    )

    c_abl = (
        sealed_ablated_points[
            name
        ][
            "CHRONOLOGICAL_NATURAL"
        ]
    )


    delta_r_pr = (
        full_point[
            "RANDOM_NATURAL"
        ][
            "pr_auc"
        ]
        - r_abl[
            "pr_auc"
        ]
    )

    delta_c_pr = (
        full_point[
            "CHRONOLOGICAL_NATURAL"
        ][
            "pr_auc"
        ]
        - c_abl[
            "pr_auc"
        ]
    )

    delta_r_roc = (
        full_point[
            "RANDOM_NATURAL"
        ][
            "roc_auc"
        ]
        - r_abl[
            "roc_auc"
        ]
    )

    delta_c_roc = (
        full_point[
            "CHRONOLOGICAL_NATURAL"
        ][
            "roc_auc"
        ]
        - c_abl[
            "roc_auc"
        ]
    )


    point_summary[
        name
    ] = {
        "family":
            comparison[
                "family"
            ],

        "random": {
            "pr_auc_removal_penalty":
                float(
                    delta_r_pr
                ),

            "roc_auc_removal_penalty":
                float(
                    delta_r_roc
                ),
        },

        "chronological": {
            "pr_auc_removal_penalty":
                float(
                    delta_c_pr
                ),

            "roc_auc_removal_penalty":
                float(
                    delta_c_roc
                ),
        },

        "interaction": {
            "pr_auc":
                float(
                    delta_r_pr
                    - delta_c_pr
                ),

            "roc_auc":
                float(
                    delta_r_roc
                    - delta_c_roc
                ),
        },
    }


# =============================================================================
# 10. FROZEN BOOTSTRAP
#
# SAME sampled rows are applied to FULL and ALL compared ablations
# within each split / replicate.
#
# RNG is restarted independently for RANDOM and CHRONO,
# exactly as frozen.
# =============================================================================

def run_split_bootstrap(
    split,
):

    print()
    print("=" * 100)
    print(
        f"BOOTSTRAP — {split}"
    )
    print("=" * 100)
    print()


    y = (
        cohort_data[
            split
        ][
            "y"
        ]
    )

    probabilities = (
        cohort_data[
            split
        ][
            "probabilities"
        ]
    )


    benign_index = np.flatnonzero(
        y == 0
    )

    attack_index = np.flatnonzero(
        y == 1
    )


    expected_benign = (
        43_158
        if split == "RANDOM_NATURAL"
        else 44_758
    )

    expected_attack = (
        6_842
        if split == "RANDOM_NATURAL"
        else 5_242
    )


    assert (
        len(
            benign_index
        )
        == expected_benign
    )

    assert (
        len(
            attack_index
        )
        == expected_attack
    )


    rng = np.random.RandomState(
        BOOTSTRAP_SEED
    )


    full_pr = np.empty(
        BOOTSTRAP_REPLICATES,
        dtype=np.float64,
    )

    full_roc = np.empty(
        BOOTSTRAP_REPLICATES,
        dtype=np.float64,
    )


    pr_penalty = {
        comparison[
            "name"
        ]:
            np.empty(
                BOOTSTRAP_REPLICATES,
                dtype=np.float64,
            )
        for comparison
        in COMPARISONS
    }


    roc_penalty = {
        comparison[
            "name"
        ]:
            np.empty(
                BOOTSTRAP_REPLICATES,
                dtype=np.float64,
            )
        for comparison
        in COMPARISONS
    }


    t0 = time.perf_counter()


    for b in range(
        BOOTSTRAP_REPLICATES
    ):

        # Frozen class-preserving bootstrap.
        #
        # Benign and attack are sampled separately WITH replacement,
        # preserving exact cohort class counts.

        sampled_benign = rng.choice(
            benign_index,
            size=len(
                benign_index
            ),
            replace=True,
        )

        sampled_attack = rng.choice(
            attack_index,
            size=len(
                attack_index
            ),
            replace=True,
        )

        sampled = np.concatenate(
            [
                sampled_benign,
                sampled_attack,
            ]
        )


        y_b = y[
            sampled
        ]


        # FULL once for this split / replicate.

        full_pr_b, full_roc_b = (
            metric_pair(
                y_b,
                probabilities[
                    "FULL"
                ][
                    sampled
                ],
            )
        )


        full_pr[
            b
        ] = full_pr_b

        full_roc[
            b
        ] = full_roc_b


        # SAME sampled membership for every compared ablation.

        for comparison in COMPARISONS:

            name = (
                comparison[
                    "name"
                ]
            )

            ablated_pr_b, ablated_roc_b = (
                metric_pair(
                    y_b,
                    probabilities[
                        name
                    ][
                        sampled
                    ],
                )
            )


            pr_penalty[
                name
            ][
                b
            ] = (
                full_pr_b
                - ablated_pr_b
            )


            roc_penalty[
                name
            ][
                b
            ] = (
                full_roc_b
                - ablated_roc_b
            )


        completed = (
            b + 1
        )


        if (
            completed % 100 == 0
            or completed
            == BOOTSTRAP_REPLICATES
        ):

            elapsed = (
                time.perf_counter()
                - t0
            )

            print(
                f"[{split}] "
                f"{completed:4d}/"
                f"{BOOTSTRAP_REPLICATES} "
                f"replicates complete "
                f"({elapsed:.1f} s)",
                flush=True,
            )


            if split == "RANDOM_NATURAL":

                state[
                    "bootstrap_random_completed"
                ] = completed

            else:

                state[
                    "bootstrap_chronological_completed"
                ] = completed


            state[
                "status"
            ] = (
                f"{split}_BOOTSTRAP_IN_PROGRESS"
                if completed
                < BOOTSTRAP_REPLICATES
                else f"{split}_BOOTSTRAP_COMPLETE"
            )


            write_json(
                STATE_PATH,
                state,
            )


    return {
        "full_pr":
            full_pr,

        "full_roc":
            full_roc,

        "pr_penalty":
            pr_penalty,

        "roc_penalty":
            roc_penalty,
    }


random_boot = run_split_bootstrap(
    "RANDOM_NATURAL"
)

chrono_boot = run_split_bootstrap(
    "CHRONOLOGICAL_NATURAL"
)


# =============================================================================
# 11. INTERACTIONS + CI SUMMARY
# =============================================================================

print()
print("=" * 100)
print("FROZEN 95% UNCERTAINTY INTERVALS")
print("=" * 100)
print()


summary_rows = []

replicate_rows = []

uncertainty_summary = {}


for comparison in COMPARISONS:

    name = (
        comparison[
            "name"
        ]
    )

    family = (
        comparison[
            "family"
        ]
    )


    r_pr = (
        random_boot[
            "pr_penalty"
        ][
            name
        ]
    )

    c_pr = (
        chrono_boot[
            "pr_penalty"
        ][
            name
        ]
    )

    r_roc = (
        random_boot[
            "roc_penalty"
        ][
            name
        ]
    )

    c_roc = (
        chrono_boot[
            "roc_penalty"
        ][
            name
        ]
    )


    interaction_pr = (
        r_pr
        - c_pr
    )

    interaction_roc = (
        r_roc
        - c_roc
    )


    uncertainty_summary[
        name
    ] = {
        "family":
            family,

        "complete_validation_point_estimates":
            point_summary[
                name
            ],

        "bootstrap_95_percentile_ci": {
            "random_pr_auc_removal_penalty":
                ci95(
                    r_pr
                ),

            "chronological_pr_auc_removal_penalty":
                ci95(
                    c_pr
                ),

            "random_roc_auc_removal_penalty":
                ci95(
                    r_roc
                ),

            "chronological_roc_auc_removal_penalty":
                ci95(
                    c_roc
                ),

            "pr_auc_shortcut_interaction":
                ci95(
                    interaction_pr
                ),

            "roc_auc_shortcut_interaction":
                ci95(
                    interaction_roc
                ),
        },
    }


    p = point_summary[
        name
    ]

    pr_ci = ci95(
        interaction_pr
    )

    roc_ci = ci95(
        interaction_roc
    )


    summary_rows.append(
        {
            "comparison":
                name,

            "family":
                family,

            "point_delta_random_pr_auc":
                p[
                    "random"
                ][
                    "pr_auc_removal_penalty"
                ],

            "point_delta_chrono_pr_auc":
                p[
                    "chronological"
                ][
                    "pr_auc_removal_penalty"
                ],

            "point_pr_auc_interaction":
                p[
                    "interaction"
                ][
                    "pr_auc"
                ],

            "pr_auc_interaction_ci_low":
                pr_ci[
                    "lower_2_5"
                ],

            "pr_auc_interaction_ci_high":
                pr_ci[
                    "upper_97_5"
                ],

            "point_delta_random_roc_auc":
                p[
                    "random"
                ][
                    "roc_auc_removal_penalty"
                ],

            "point_delta_chrono_roc_auc":
                p[
                    "chronological"
                ][
                    "roc_auc_removal_penalty"
                ],

            "point_roc_auc_interaction":
                p[
                    "interaction"
                ][
                    "roc_auc"
                ],

            "roc_auc_interaction_ci_low":
                roc_ci[
                    "lower_2_5"
                ],

            "roc_auc_interaction_ci_high":
                roc_ci[
                    "upper_97_5"
                ],
        }
    )


    for b in range(
        BOOTSTRAP_REPLICATES
    ):

        replicate_rows.append(
            {
                "comparison":
                    name,

                "family":
                    family,

                "replicate":
                    b,

                "random_pr_auc_removal_penalty":
                    float(
                        r_pr[
                            b
                        ]
                    ),

                "chronological_pr_auc_removal_penalty":
                    float(
                        c_pr[
                            b
                        ]
                    ),

                "pr_auc_shortcut_interaction":
                    float(
                        interaction_pr[
                            b
                        ]
                    ),

                "random_roc_auc_removal_penalty":
                    float(
                        r_roc[
                            b
                        ]
                    ),

                "chronological_roc_auc_removal_penalty":
                    float(
                        c_roc[
                            b
                        ]
                    ),

                "roc_auc_shortcut_interaction":
                    float(
                        interaction_roc[
                            b
                        ]
                    ),
            }
        )


    print(
        name
    )

    print(
        "  PR interaction point : "
        f"{p['interaction']['pr_auc']:+.12f}"
    )

    print(
        "  PR interaction 95% CI: "
        f"[{pr_ci['lower_2_5']:+.12f}, "
        f"{pr_ci['upper_97_5']:+.12f}]"
    )

    print(
        "  ROC interaction point: "
        f"{p['interaction']['roc_auc']:+.12f}"
    )

    print(
        "  ROC interaction 95% CI: "
        f"[{roc_ci['lower_2_5']:+.12f}, "
        f"{roc_ci['upper_97_5']:+.12f}]"
    )

    print()


# =============================================================================
# 12. PERSIST ANALYSIS ARTIFACTS
# =============================================================================

SUMMARY_JSON = (
    OUT
    / "stage23_4_uncertainty_summary.json"
)

SUMMARY_CSV = (
    OUT
    / "stage23_4_uncertainty_summary.csv"
)

REPLICATES_CSV = (
    OUT
    / "stage23_4_bootstrap_replicates.csv"
)


summary_payload = {
    "stage":
        "Stage23-4 uncertainty analysis",

    "status":
        "UNCERTAINTY_ANALYSIS_COMPLETE_UNSEALED",

    "completed_utc":
        utc_now(),

    "execution_parent": {
        "commit":
            EXPECTED_HEAD,

        "tag":
            EXPECTED_TAG,
    },

    "frozen_method": {
        "bootstrap_replicates":
            BOOTSTRAP_REPLICATES,

        "bootstrap_rng":
            (
                "numpy.random.RandomState(seed=42), "
                "restarted independently per split"
            ),

        "sampling":
            (
                "benign and attack rows resampled with replacement "
                "separately, preserving fixed cohort class counts"
            ),

        "pairing":
            (
                "same sampled rows within split/replicate "
                "for FULL and every compared ablated model"
            ),

        "confidence_interval":
            (
                "percentile 95% interval [2.5th, 97.5th]"
            ),

        "numpy_percentile_method":
            "default linear",

        "interaction":
            (
                "(FULL_R - ABLATED_R) - "
                "(FULL_C - ABLATED_C)"
            ),

        "important_limitation":
            spec[
                "important_limitation"
            ],
    },

    "cohorts": {
        "RANDOM_NATURAL": {
            "rows":
                50_000,

            "benign":
                43_158,

            "attack":
                6_842,
        },

        "CHRONOLOGICAL_NATURAL": {
            "rows":
                50_000,

            "benign":
                44_758,

            "attack":
                5_242,
        },
    },

    "comparisons":
        uncertainty_summary,

    "governance": {
        "model_fits":
            0,

        "additional_stage23_model_fits_authorized":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "parquet_files_read":
            0,

        "new_subset":
            False,

        "split_changed":
            False,

        "uncertainty_cohort_changed":
            False,

        "uncertainty_method_changed":
            False,

        "primary_metric_changed":
            False,
    },

    "status_note":
        (
            "Full-validation metrics remain primary point estimates. "
            "Bootstrap intervals are supplementary uncertainty derived "
            "from the prospectively frozen 50,000-row cohorts."
        ),
}


write_json(
    SUMMARY_JSON,
    summary_payload,
)


pd.DataFrame(
    summary_rows
).to_csv(
    SUMMARY_CSV,
    index=False,
)


pd.DataFrame(
    replicate_rows
).to_csv(
    REPLICATES_CSV,
    index=False,
)


# =============================================================================
# 13. CHECKSUMS
# =============================================================================

state[
    "status"
] = (
    "UNCERTAINTY_ANALYSIS_COMPLETE_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "bootstrap_random_completed"
] = BOOTSTRAP_REPLICATES

state[
    "bootstrap_chronological_completed"
] = BOOTSTRAP_REPLICATES

state[
    "model_fits"
] = 0

state[
    "next_action"
] = (
    "ZERO_FIT_SEAL_UNCERTAINTY_BLOCK"
)


write_json(
    STATE_PATH,
    state,
)


CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


artifact_files = sorted(
    p
    for p in OUT.iterdir()
    if (
        p.is_file()
        and p.name
        != "checksums.sha256"
    )
)


CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.name}"
        for p in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-4 UNCERTAINTY ANALYSIS COMPLETE — UNSEALED")
print("=" * 100)
print()

print("Bootstrap replicates / split : 1000")
print("Comparisons                  : 11")
print("Primary comparisons          : 6")
print("Placebo comparisons          : 5")
print()
print("New model fits               : 0")
print("Stage23 model fit state      : 50 / 50 SEALED")
print("Raw Mar1 accessed            : NO")
print("Raw Mar2 accessed            : NO")
print("Parquet files read           : 0")
print()
print("Summary:")
print(" ", SUMMARY_JSON)
print(" SHA256:")
print(" ", sha256_file(SUMMARY_JSON))
print()
print("Summary CSV:")
print(" ", SUMMARY_CSV)
print(" SHA256:")
print(" ", sha256_file(SUMMARY_CSV))
print()
print("Replicates CSV:")
print(" ", REPLICATES_CSV)
print(" SHA256:")
print(" ", sha256_file(REPLICATES_CSV))
print()
print("Checksum manifest SHA256:")
print(" ", sha256_file(CHECKSUMS))
print()
print("NEXT:")
print("  ZERO-FIT seal + push of Stage23-4 uncertainty analysis.")
print("=" * 100)

STAGE23-4 — FROZEN UNCERTAINTY ANALYSIS

[OK] HEAD      : f2cd2b59172179f9f0884d88e579276517de7a01
[OK] seal tag  : stage23-3-stump-controls-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50
[OK] new fits  : 0 authorized

[EXACT] frozen uncertainty specification
[EXACT] RANDOM cohort : 43,158 benign + 6,842 attack
[EXACT] CHRONO cohort : 44,758 benign + 5,242 attack

SEALED FULL POINT ESTIMATES

RANDOM_NATURAL
  PR-AUC : 0.995590041899
  ROC-AUC: 0.998624564774
CHRONOLOGICAL_NATURAL
  PR-AUC : 0.106215155134
  ROC-AUC: 0.514918426394

[EXACT] RANDOM_NATURAL FULL cohort mapped
        cohort PR  = 0.996315693563
        cohort ROC = 0.998798445388

[EXACT] CHRONOLOGICAL_NATURAL FULL cohort mapped
        cohort PR  = 0.107085529425
        cohort ROC = 0.518687757499

LOAD ABLATED COHORT PROBABILITIES

[LOADED] NO_DST_PORT                   RANDOM_NATURAL          cohort PR=0.995852440 ROC=0.998484337
[LOADED] NO_DST_PORT                   CHRONOLOGICAL_NATURAL   cohort PR=0.1

In [15]:
# =============================================================================
# STAGE23-4 — ZERO-FIT UNCERTAINTY ANALYSIS SEAL + PUSH
#
# INPUT:
#   Stage23 model-fit state: 50 / 50 SEALED
#   Stage23-4 uncertainty : COMPLETE, UNSEALED
#
# OUTPUT:
#   Stage23-4 uncertainty : SEALED + PUSHED
#
# ZERO MODEL FITS
# ZERO BOOTSTRAP RECOMPUTATION
# ZERO RAW DATA
# ZERO PARQUET
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import os


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE = Path(
    "/kaggle/working/stage23_4_uncertainty_analysis"
)

DEST = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_4_uncertainty_analysis"
)

EXPECTED_PARENT = (
    "f2cd2b59172179f9f0884d88e579276517de7a01"
)

EXPECTED_PARENT_TAG = (
    "stage23-3-stump-controls-complete-v1"
)

EXPECTED_SUMMARY_SHA = (
    "218c9c8df4b39d74d683862132a87c48c539212cfec2c12fd495ddb227cf550b"
)

EXPECTED_SUMMARY_CSV_SHA = (
    "a2ae4c6ff92182ab05d569ef166ac42fda6ca345b7aa9c96a743e611ae0924e5"
)

EXPECTED_REPLICATES_SHA = (
    "736bb0e8f2b4d883f49145892ba89b9b4ca77559be51e02a43290b0d40589b34"
)

EXPECTED_SOURCE_CHECKSUMS_SHA = (
    "9f847e8de363b59c0d9797d7774ce61721e2be5a88e10968976cfc90fd12a654"
)

SEAL_TAG = (
    "stage23-4-uncertainty-analysis-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: seal frozen uncertainty analysis"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def git(
    *args,
    check=True,
    show=False,
):
    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (
        p.stdout or ""
    ).strip()

    if show and out:
        print(out)

    if check and p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(map(str,args))} failed:\n{out}"
        )

    return p.returncode, out


# =============================================================================
# 2. START
# =============================================================================

print("=" * 100)
print("STAGE23-4 — ZERO-FIT UNCERTAINTY SEAL")
print("=" * 100)
print()


# =============================================================================
# 3. REPOSITORY PREFLIGHT
# =============================================================================

_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, parent_tag = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)


if branch != "main":
    raise RuntimeError(
        f"Expected main branch; found {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected repository HEAD.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if status:
    raise RuntimeError(
        "Repository dirty before Stage23-4 seal:\n"
        + status
    )

if parent_tag != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage23-3 parent tag mismatch."
    )


print("[OK] HEAD       :", head)
print("[OK] parent tag :", EXPECTED_PARENT_TAG)
print("[OK] worktree   : CLEAN")


# =============================================================================
# 4. ENSURE SEAL TAG DOES NOT ALREADY EXIST
# =============================================================================

_, local_tag = git(
    "tag",
    "--list",
    SEAL_TAG,
)

if local_tag:
    raise RuntimeError(
        f"Seal tag already exists locally: {SEAL_TAG}"
    )


_, remote_tag = git(
    "ls-remote",
    "--tags",
    "origin",
    f"refs/tags/{SEAL_TAG}",
)

if remote_tag:
    raise RuntimeError(
        f"Seal tag already exists remotely: {SEAL_TAG}"
    )


# =============================================================================
# 5. PUSH AUTH PREFLIGHT
# =============================================================================

print()
print("=" * 100)
print("PUSH AUTHENTICATION PREFLIGHT")
print("=" * 100)
print()


rc, push_test = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
)


if rc != 0:
    print(push_test)

    raise RuntimeError(
        "GitHub push authentication unavailable. "
        "No repository files modified."
    )


print("[OK] GitHub push authentication available.")


# =============================================================================
# 6. VERIFY UNSEALED UNCERTAINTY BLOCK
# =============================================================================

print()
print("=" * 100)
print("VERIFY UNCERTAINTY BLOCK")
print("=" * 100)
print()


if not SOURCE.is_dir():
    raise RuntimeError(
        f"Missing uncertainty directory: {SOURCE}"
    )


STATE = (
    SOURCE
    / "execution_state.json"
)

SUMMARY_JSON = (
    SOURCE
    / "stage23_4_uncertainty_summary.json"
)

SUMMARY_CSV = (
    SOURCE
    / "stage23_4_uncertainty_summary.csv"
)

REPLICATES_CSV = (
    SOURCE
    / "stage23_4_bootstrap_replicates.csv"
)

SOURCE_CHECKSUMS = (
    SOURCE
    / "checksums.sha256"
)


for p in [
    STATE,
    SUMMARY_JSON,
    SUMMARY_CSV,
    REPLICATES_CSV,
    SOURCE_CHECKSUMS,
]:
    if not p.is_file():
        raise RuntimeError(
            f"Required Stage23-4 artifact missing: {p}"
        )


actual_summary_sha = sha256_file(
    SUMMARY_JSON
)

actual_summary_csv_sha = sha256_file(
    SUMMARY_CSV
)

actual_replicates_sha = sha256_file(
    REPLICATES_CSV
)

actual_source_checksums_sha = sha256_file(
    SOURCE_CHECKSUMS
)


checks = [
    (
        "summary JSON",
        EXPECTED_SUMMARY_SHA,
        actual_summary_sha,
    ),
    (
        "summary CSV",
        EXPECTED_SUMMARY_CSV_SHA,
        actual_summary_csv_sha,
    ),
    (
        "replicates CSV",
        EXPECTED_REPLICATES_SHA,
        actual_replicates_sha,
    ),
    (
        "source checksums",
        EXPECTED_SOURCE_CHECKSUMS_SHA,
        actual_source_checksums_sha,
    ),
]


for label, expected, actual in checks:

    print(label)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:
        raise RuntimeError(
            f"{label} SHA mismatch."
        )

    print("  [EXACT]")


# =============================================================================
# 7. VERIFY EXECUTION STATE
# =============================================================================

state = json.loads(
    STATE.read_text(
        encoding="utf-8"
    )
)

summary = json.loads(
    SUMMARY_JSON.read_text(
        encoding="utf-8"
    )
)


assert (
    state["status"]
    == "UNCERTAINTY_ANALYSIS_COMPLETE_UNSEALED"
)

assert (
    state["bootstrap_random_completed"]
    == 1000
)

assert (
    state["bootstrap_chronological_completed"]
    == 1000
)

assert (
    state["model_fits"]
    == 0
)


assert (
    summary["status"]
    == "UNCERTAINTY_ANALYSIS_COMPLETE_UNSEALED"
)

assert (
    summary["execution_parent"]["commit"]
    == EXPECTED_PARENT
)

assert (
    summary["execution_parent"]["tag"]
    == EXPECTED_PARENT_TAG
)

assert (
    summary["frozen_method"]["bootstrap_replicates"]
    == 1000
)

assert (
    len(
        summary["comparisons"]
    )
    == 11
)

assert (
    summary["governance"]["model_fits"]
    == 0
)

assert (
    summary["governance"][
        "additional_stage23_model_fits_authorized"
    ]
    == 0
)

assert (
    summary["governance"]["raw_mar1_accessed"]
    is False
)

assert (
    summary["governance"]["raw_mar2_accessed"]
    is False
)

assert (
    summary["governance"]["parquet_files_read"]
    == 0
)

assert (
    summary["governance"]["uncertainty_cohort_changed"]
    is False
)

assert (
    summary["governance"]["uncertainty_method_changed"]
    is False
)


print()
print("[EXACT] bootstrap RANDOM       : 1000 / 1000")
print("[EXACT] bootstrap CHRONO       : 1000 / 1000")
print("[EXACT] comparisons            : 11")
print("[EXACT] model fits             : 0")
print("[EXACT] raw Mar1/Mar2          : NO / NO")
print("[EXACT] parquet reads          : 0")
print("[EXACT] uncertainty method     : unchanged")


# =============================================================================
# 8. VERIFY SOURCE MANIFEST
# =============================================================================

source_manifest = {}


for line in SOURCE_CHECKSUMS.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    digest, filename = line.split(
        maxsplit=1
    )

    source_manifest[
        filename.strip()
    ] = digest.strip()


for filename, expected_sha in source_manifest.items():

    p = SOURCE / filename

    if not p.is_file():
        raise RuntimeError(
            f"Manifest file missing: {filename}"
        )

    actual_sha = sha256_file(
        p
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Manifest SHA mismatch: {filename}"
        )


print()
print(
    "[EXACT] source manifest verified:",
    len(source_manifest),
    "files",
)


# =============================================================================
# 9. SIZE SAFETY
# =============================================================================

print()
print("=" * 100)
print("GITHUB ARTIFACT SIZE CHECK")
print("=" * 100)
print()


largest_path = None
largest_size = -1


for p in sorted(
    SOURCE.iterdir()
):

    if not p.is_file():
        continue

    size = p.stat().st_size

    print(
        f"{p.name:<55} "
        f"{size / (1024**2):>9.2f} MiB"
    )

    if size > largest_size:
        largest_size = size
        largest_path = p


if largest_size >= 95 * 1024 * 1024:
    raise RuntimeError(
        "Stage23-4 artifact exceeds safe GitHub file size."
    )


print()
print(
    "[OK] largest artifact:",
    largest_path.name,
    f"{largest_size / (1024**2):.2f} MiB",
)


# =============================================================================
# 10. COPY ANALYSIS ARTIFACTS INTO REPOSITORY
# =============================================================================

print()
print("=" * 100)
print("COPY STAGE23-4 ARTIFACTS")
print("=" * 100)
print()


if DEST.exists():
    raise RuntimeError(
        f"Destination already exists: {DEST}"
    )


DEST.mkdir(
    parents=True,
    exist_ok=False,
)


for p in SOURCE.iterdir():

    if p.is_file():
        shutil.copy2(
            p,
            DEST / p.name,
        )


print(
    "[OK] copied source files:",
    len(
        [
            p
            for p in SOURCE.iterdir()
            if p.is_file()
        ]
    ),
)


# =============================================================================
# 11. SEALED INTERPRETATION CLASSIFICATION
#
# This DOES NOT create a new statistical rule.
#
# It simply records whether the frozen percentile interval:
#   - lies entirely above zero
#   - lies entirely below zero
#   - includes zero
# =============================================================================

classification = {}


for name, result in summary[
    "comparisons"
].items():

    ci_block = (
        result[
            "bootstrap_95_percentile_ci"
        ]
    )


    def classify(ci):

        lo = float(
            ci[
                "lower_2_5"
            ]
        )

        hi = float(
            ci[
                "upper_97_5"
            ]
        )

        if lo > 0:
            return "ABOVE_ZERO"

        if hi < 0:
            return "BELOW_ZERO"

        return "INCLUDES_ZERO"


    classification[
        name
    ] = {
        "family":
            result[
                "family"
            ],

        "pr_auc_shortcut_interaction_ci_status":
            classify(
                ci_block[
                    "pr_auc_shortcut_interaction"
                ]
            ),

        "roc_auc_shortcut_interaction_ci_status":
            classify(
                ci_block[
                    "roc_auc_shortcut_interaction"
                ]
            ),
    }


CLASSIFICATION_PATH = (
    DEST
    / "interaction_ci_zero_classification.json"
)


write_json(
    CLASSIFICATION_PATH,
    {
        "stage":
            "Stage23-4",

        "status":
            "DERIVED_FROM_FROZEN_95_PERCENTILE_INTERVALS",

        "created_utc":
            utc_now(),

        "decision_rule":
            (
                "Descriptive zero-location classification only: "
                "ABOVE_ZERO if CI lower bound > 0; "
                "BELOW_ZERO if CI upper bound < 0; "
                "otherwise INCLUDES_ZERO."
            ),

        "important_note":
            (
                "This classification is not evidence of feature leakage "
                "or causality. Placebo interactions are retained as the "
                "matched reference context."
            ),

        "comparisons":
            classification,

        "model_fits":
            0,
    },
)


# =============================================================================
# 12. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    DEST
    / "seal_receipt.json"
)


write_json(
    SEAL_RECEIPT,
    {
        "stage":
            "Stage23-4 uncertainty analysis",

        "status":
            "SEALED",

        "sealed_utc":
            utc_now(),

        "execution_parent": {
            "commit":
                EXPECTED_PARENT,

            "tag":
                EXPECTED_PARENT_TAG,
        },

        "analysis": {
            "comparisons":
                11,

            "primary_comparisons":
                6,

            "placebo_comparisons":
                5,

            "bootstrap_replicates_per_split":
                1000,

            "random_cohort_rows":
                50000,

            "chronological_cohort_rows":
                50000,

            "confidence_interval":
                "percentile 95% [2.5th, 97.5th]",
        },

        "artifact_sha256": {
            "summary_json":
                EXPECTED_SUMMARY_SHA,

            "summary_csv":
                EXPECTED_SUMMARY_CSV_SHA,

            "bootstrap_replicates_csv":
                EXPECTED_REPLICATES_SHA,

            "source_checksums":
                EXPECTED_SOURCE_CHECKSUMS_SHA,

            "interaction_ci_zero_classification":
                sha256_file(
                    CLASSIFICATION_PATH
                ),
        },

        "governance": {
            "new_model_fits":
                0,

            "stage23_model_fit_count":
                "50 / 50",

            "additional_model_fits_authorized":
                0,

            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "parquet_files_read":
                0,

            "bootstrap_recomputed_during_seal":
                False,

            "uncertainty_method_changed":
                False,

            "uncertainty_cohort_changed":
                False,
        },

        "next_authorized_work":
            (
                "Continue Stage23 analysis-only work under frozen protocol."
            ),

        "tag":
            SEAL_TAG,
    },
)


# =============================================================================
# 13. README
# =============================================================================

README = """# Stage23-4 — Frozen Uncertainty Analysis

This directory seals the prospectively frozen uncertainty analysis for
Stage23 shortcut-feature auditing.

## Method

- 50,000-row frozen cohort per natural split
- RANDOM_NATURAL:
  - 43,158 benign
  - 6,842 attack
- CHRONOLOGICAL_NATURAL:
  - 44,758 benign
  - 5,242 attack
- 1,000 paired stratified bootstrap replicates
- `numpy.random.RandomState(seed=42)` restarted independently per split
- benign and attack rows resampled separately with replacement
- identical sampled rows within each split/replicate for FULL and ablated models
- percentile 95% interval `[2.5th, 97.5th]`

## Headline uncertainty quantities

- PR-AUC removal penalty
- ROC-AUC removal penalty
- PR-AUC split-by-ablation interaction
- ROC-AUC split-by-ablation interaction

Interaction:

`I(S) = (FULL_R - ABLATED_R) - (FULL_C - ABLATED_C)`

Full-validation metrics remain the primary point estimates.
Bootstrap intervals are supplementary uncertainty from the frozen cohorts.

## Interpretation boundary

An interaction CI lying above or below zero indicates that the frozen
cohort bootstrap does not span zero for that interaction estimate.

It does **not** establish that a feature is leakage or prove causality.
Placebo interactions must remain part of the interpretation.

## Governance

- Stage23 model-fit budget: **50 / 50**
- New fits in Stage23-4: **0**
- Raw Mar1 accessed: **NO**
- Raw Mar2 accessed: **NO**
- Parquet files read: **0**
"""


(
    DEST
    / "README.md"
).write_text(
    README,
    encoding="utf-8",
)


# =============================================================================
# 14. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

REPO_MANIFEST = (
    DEST
    / "repository_checksums.sha256"
)


manifest_files = sorted(
    p
    for p in DEST.iterdir()
    if (
        p.is_file()
        and p.name
        != "repository_checksums.sha256"
    )
)


REPO_MANIFEST.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.name}"
        for p in manifest_files
    ) + "\n",
    encoding="utf-8",
)


repo_manifest_sha = sha256_file(
    REPO_MANIFEST
)


print()
print(
    "[OK] repository manifest SHA256:",
    repo_manifest_sha,
)


# =============================================================================
# 15. COMMIT IDENTITY
# =============================================================================

_, user_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

_, user_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not user_name:

    _, parent_author = git(
        "log",
        "-1",
        "--format=%an",
    )

    git(
        "config",
        "user.name",
        parent_author,
    )


if not user_email:

    _, parent_email = git(
        "log",
        "-1",
        "--format=%ae",
    )

    git(
        "config",
        "user.email",
        parent_email,
    )


# =============================================================================
# 16. STAGE ONLY STAGE23-4
# =============================================================================

relative_dest = (
    DEST.relative_to(
        REPO
    )
)


git(
    "add",
    "--",
    str(
        relative_dest
    ),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    x
    for x in staged.splitlines()
    if x.strip()
]


if not staged_files:
    raise RuntimeError(
        "No Stage23-4 files staged."
    )


for path in staged_files:

    if not path.startswith(
        str(relative_dest)
        + "/"
    ):
        raise RuntimeError(
            f"Unexpected staged file: {path}"
        )


print(
    "[OK] staged files:",
    len(staged_files),
)


# =============================================================================
# 17. COMMIT
# =============================================================================

print()
print("=" * 100)
print("COMMIT")
print("=" * 100)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, sealed_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] sealed commit:",
    sealed_commit,
)


# =============================================================================
# 18. ANNOTATED TAG
# =============================================================================

git(
    "tag",
    "-a",
    SEAL_TAG,
    sealed_commit,
    "-m",
    (
        "Stage23-4 frozen uncertainty analysis complete: "
        "11 comparisons, 1000 paired stratified bootstrap "
        "replicates per natural split, zero model fits"
    ),
)


# =============================================================================
# 19. ATOMIC PUSH
# =============================================================================

print()
print("=" * 100)
print("ATOMIC PUSH MAIN + TAG")
print("=" * 100)
print()


git(
    "push",
    "--atomic",
    "origin",
    "main",
    f"refs/tags/{SEAL_TAG}",
    show=True,
)


# =============================================================================
# 20. REMOTE VERIFY
# =============================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEAL_TAG}^{{}}",
)


if not remote_main_line:
    raise RuntimeError(
        "Remote main could not be resolved."
    )

if not remote_tag_line:
    raise RuntimeError(
        "Remote annotated tag could not be peeled."
    )


remote_main = (
    remote_main_line.split()[0]
)

remote_tag_commit = (
    remote_tag_line.split()[0]
)


if remote_main != sealed_commit:
    raise RuntimeError(
        "Remote main mismatch."
    )

if remote_tag_commit != sealed_commit:
    raise RuntimeError(
        "Remote tag mismatch."
    )


# =============================================================================
# 21. FINAL CLEAN WORKTREE
# =============================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:
    raise RuntimeError(
        "Repository dirty after uncertainty seal:\n"
        + final_status
    )


# =============================================================================
# 22. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-4 UNCERTAINTY ANALYSIS — SEALED AND PUSHED")
print("=" * 100)
print()

print("Commit:")
print(" ", sealed_commit)

print()
print("Tag:")
print(" ", SEAL_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag peeled commit:")
print(" ", remote_tag_commit)

print()
print("Repository manifest SHA256:")
print(" ", repo_manifest_sha)

print()
print("UNCERTAINTY ACCOUNTING")
print("  Comparisons       : 11")
print("  Primary           : 6")
print("  Placebo           : 5")
print("  Bootstrap / split : 1000")
print()
print("MODEL ACCOUNTING")
print("  Stage23 fits      : 50 / 50 SEALED")
print("  New fits in 23-4 : 0")
print()
print("Raw Mar1 accessed   : NO")
print("Raw Mar2 accessed   : NO")
print("Parquet files read  : 0")

print()
print("NEXT:")
print("  Continue Stage23 frozen analysis-only work.")
print("  No additional Stage23 model fit is authorized.")

print("=" * 100)

STAGE23-4 — ZERO-FIT UNCERTAINTY SEAL

[OK] HEAD       : f2cd2b59172179f9f0884d88e579276517de7a01
[OK] parent tag : stage23-3-stump-controls-complete-v1
[OK] worktree   : CLEAN

PUSH AUTHENTICATION PREFLIGHT

[OK] GitHub push authentication available.

VERIFY UNCERTAINTY BLOCK

summary JSON
  expected: 218c9c8df4b39d74d683862132a87c48c539212cfec2c12fd495ddb227cf550b
  actual:   218c9c8df4b39d74d683862132a87c48c539212cfec2c12fd495ddb227cf550b
  [EXACT]
summary CSV
  expected: a2ae4c6ff92182ab05d569ef166ac42fda6ca345b7aa9c96a743e611ae0924e5
  actual:   a2ae4c6ff92182ab05d569ef166ac42fda6ca345b7aa9c96a743e611ae0924e5
  [EXACT]
replicates CSV
  expected: 736bb0e8f2b4d883f49145892ba89b9b4ca77559be51e02a43290b0d40589b34
  actual:   736bb0e8f2b4d883f49145892ba89b9b4ca77559be51e02a43290b0d40589b34
  [EXACT]
source checksums
  expected: 9f847e8de363b59c0d9797d7774ce61721e2be5a88e10968976cfc90fd12a654
  actual:   9f847e8de363b59c0d9797d7774ce61721e2be5a88e10968976cfc90fd12a654
  [EXACT]

[EXACT]

In [16]:
# =============================================================================
# STAGE23-5 — SHAP PROXY-ABSORPTION PREFLIGHT
#
# ZERO MODEL FITS
# ZERO SHAP COMPUTATION
# ZERO SCIENTIFIC WRITES
# ZERO RAW MAR1 / MAR2
# ZERO PARQUET READS
#
# Verifies:
#   - Stage23-4 sealed parent
#   - frozen SHAP specification
#   - 2 frozen 5,000-row balanced cohorts
#   - exact 70-feature order
#   - all frozen FULL / primary / placebo component models
#   - 24 model-pairs = 48 component models
#   - LightGBM / XGBoost feature dimensions
#   - TreeExplainer runtime availability
#
# Nothing is modified.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import os
import gc

import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
import sklearn

try:
    import shap
except Exception as exc:
    raise RuntimeError(
        "SHAP import failed. Do NOT install or upgrade anything yet.\n"
        f"{type(exc).__name__}: {exc}"
    )


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "1e687d964aa5247a3e5d0678223dc8de92cc5ade"
)

EXPECTED_TAG = (
    "stage23-4-uncertainty-analysis-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

SHAP_SPEC = (
    PROTOCOL
    / "shap_spec.json"
)

SHAP_RANDOM = (
    PROTOCOL
    / "shap_cohort_random_natural.csv"
)

SHAP_CHRONO = (
    PROTOCOL
    / "shap_cohort_chronological_natural.csv"
)

PROTOCOL_CHECKSUMS = (
    PROTOCOL
    / "checksums.sha256"
)

FEATURE_ORDER_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_70_feature_order.csv"
)

FULL_ROOTS = {
    "RANDOM_NATURAL":
        REPO
        / "results"
        / "stage22r_training"
        / "stage22r_2a_random_natural",

    "CHRONOLOGICAL_NATURAL":
        REPO
        / "results"
        / "stage22r_training"
        / "stage22r_2c_chronological_natural",
}

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)

PLACEBO_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

EXPECTED_SUBSETS = [
    "FULL",

    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",

    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def parse_manifest(path):

    out = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        out[
            rel.strip()
        ] = digest.strip()

    return out


def find_exact_one(
    directory,
    pattern,
    label,
):

    matches = sorted(
        directory.glob(
            pattern
        )
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"{label}: expected exactly one {pattern} under\n"
            f"{directory}\n"
            f"found={len(matches)}"
        )

    return matches[0]


def normalize_subset_from_dir(
    name,
):

    mapping = {
        "no_dst_port":
            "NO_DST_PORT",

        "no_ports":
            "NO_PORTS",

        "no_init_fwd_win_byts":
            "NO_INIT_FWD_WIN_BYTS",

        "no_fwd_seg_size_min":
            "NO_FWD_SEG_SIZE_MIN",

        "no_suspicious_group":
            "NO_SUSPICIOUS_GROUP",

        "behavior_only":
            "BEHAVIOR_ONLY",

        "placebo_counts":
            "PLACEBO_COUNTS",

        "placebo_volume_direction":
            "PLACEBO_VOLUME_DIRECTION",

        "placebo_iat":
            "PLACEBO_IAT",

        "placebo_packet_size":
            "PLACEBO_PACKET_SIZE",

        "placebo_activity":
            "PLACEBO_ACTIVITY",
    }

    if name not in mapping:
        raise RuntimeError(
            f"Unknown frozen subset directory: {name}"
        )

    return mapping[name]


# =============================================================================
# 2. SEALED REPOSITORY STATE
# =============================================================================

print("=" * 100)
print("STAGE23-5 — SHAP PROXY-ABSORPTION PREFLIGHT")
print("=" * 100)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "HEAD mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )

if tag_commit != EXPECTED_HEAD:
    raise RuntimeError(
        "Stage23-4 tag mismatch."
    )

if status:
    raise RuntimeError(
        "Repository is dirty:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] model fits: 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")


# =============================================================================
# 3. VERIFY FROZEN SHAP SPEC + COHORT HASHES
# =============================================================================

manifest = parse_manifest(
    PROTOCOL_CHECKSUMS
)


for p in [
    SHAP_SPEC,
    SHAP_RANDOM,
    SHAP_CHRONO,
]:

    if p.name not in manifest:
        raise RuntimeError(
            f"{p.name} missing from Stage23-0 checksum manifest."
        )

    expected = manifest[
        p.name
    ]

    actual = sha256_file(
        p
    )

    print()
    print(p.name)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:
        raise RuntimeError(
            f"Frozen SHA mismatch: {p.name}"
        )

    print("  [EXACT]")


spec = json.loads(
    SHAP_SPEC.read_text(
        encoding="utf-8"
    )
)


assert (
    spec["cohort"]["rows_per_split"]
    == 5000
)

assert (
    spec["cohort"]["benign_per_split"]
    == 2500
)

assert (
    spec["cohort"]["attack_per_split"]
    == 2500
)

assert (
    spec["cohort"]["seed"]
    == 42
)

assert (
    spec["cohort"]["locator"]
    == "clean_position"
)

assert (
    spec["cohort"][
        "same_rows_reused_for_every_subset_within_split"
    ]
    is True
)

assert (
    spec["retraining"]
    == "NONE; use frozen models from Stage23 primary/placebo runs"
)


print()
print("[EXACT] frozen SHAP specification")


# =============================================================================
# 4. VERIFY FROZEN SHAP COHORTS
# =============================================================================

print()
print("=" * 100)
print("FROZEN SHAP COHORTS")
print("=" * 100)
print()


cohorts = {}


for split, path in [
    (
        "RANDOM_NATURAL",
        SHAP_RANDOM,
    ),
    (
        "CHRONOLOGICAL_NATURAL",
        SHAP_CHRONO,
    ),
]:

    df = pd.read_csv(
        path
    )


    expected_columns = [
        "split",
        "clean_position",
        "binary_label",
    ]


    if list(
        df.columns
    ) != expected_columns:

        raise RuntimeError(
            f"{split}: unexpected SHAP cohort schema:\n"
            f"{list(df.columns)}"
        )


    if len(df) != 5000:
        raise RuntimeError(
            f"{split}: expected 5000 rows; got {len(df)}"
        )


    benign = int(
        (
            df[
                "binary_label"
            ]
            == 0
        ).sum()
    )

    attack = int(
        (
            df[
                "binary_label"
            ]
            == 1
        ).sum()
    )


    if benign != 2500:
        raise RuntimeError(
            f"{split}: benign count {benign} != 2500"
        )

    if attack != 2500:
        raise RuntimeError(
            f"{split}: attack count {attack} != 2500"
        )


    if (
        df[
            "clean_position"
        ].duplicated().any()
    ):
        raise RuntimeError(
            f"{split}: duplicate clean_position in SHAP cohort."
        )


    cohorts[
        split
    ] = df


    print(
        f"[EXACT] {split:<23} "
        f"rows=5,000 benign=2,500 attack=2,500"
    )


# =============================================================================
# 5. FEATURE ORDER
# =============================================================================

print()
print("=" * 100)
print("FROZEN 70-FEATURE ORDER")
print("=" * 100)
print()


if not FEATURE_ORDER_PATH.is_file():
    raise RuntimeError(
        f"Feature-order file missing:\n{FEATURE_ORDER_PATH}"
    )


feature_order_df = pd.read_csv(
    FEATURE_ORDER_PATH
)


print(
    "columns:",
    list(
        feature_order_df.columns
    ),
)

print(
    "rows   :",
    len(
        feature_order_df
    ),
)


if len(
    feature_order_df
) != 70:
    raise RuntimeError(
        "Frozen feature-order file does not contain 70 rows."
    )


# Do not assume its feature-name column; identify it safely.

candidate_columns = [
    c
    for c in feature_order_df.columns
    if (
        "feature" in c.lower()
        or "name" in c.lower()
    )
]


if len(candidate_columns) != 1:

    print()
    print(feature_order_df.head())

    raise RuntimeError(
        "Could not uniquely identify feature-name column."
    )


FEATURE_COLUMN = (
    candidate_columns[0]
)

FULL_FEATURE_ORDER = (
    feature_order_df[
        FEATURE_COLUMN
    ]
    .astype(str)
    .tolist()
)


if len(
    set(
        FULL_FEATURE_ORDER
    )
) != 70:
    raise RuntimeError(
        "Frozen feature order contains duplicates."
    )


print(
    "[OK] feature-name column:",
    FEATURE_COLUMN,
)

print(
    "[OK] first feature       :",
    FULL_FEATURE_ORDER[0],
)

print(
    "[OK] final feature       :",
    FULL_FEATURE_ORDER[-1],
)


# =============================================================================
# 6. RUNTIME VERSIONS
# =============================================================================

print()
print("=" * 100)
print("SHAP RUNTIME")
print("=" * 100)
print()

print("numpy    :", np.__version__)
print("pandas   :", pd.__version__)
print("sklearn  :", sklearn.__version__)
print("lightgbm :", lgb.__version__)
print("xgboost  :", xgb.__version__)
print("shap     :", shap.__version__)


# =============================================================================
# 7. INVENTORY ALL FROZEN MODEL PAIRS
# =============================================================================

print()
print("=" * 100)
print("FROZEN MODEL INVENTORY")
print("=" * 100)
print()


inventory = []


# -----------------------------------------------------------------------------
# FULL
# -----------------------------------------------------------------------------

for split, d in FULL_ROOTS.items():

    lgb_path = find_exact_one(
        d,
        "*lightgbm_model.txt",
        f"FULL {split}",
    )

    xgb_path = find_exact_one(
        d,
        "*xgboost_model.json",
        f"FULL {split}",
    )

    inventory.append(
        {
            "subset":
                "FULL",

            "family":
                "FULL",

            "split":
                split,

            "directory":
                d,

            "lightgbm":
                lgb_path,

            "xgboost":
                xgb_path,
        }
    )


# -----------------------------------------------------------------------------
# PRIMARY — 12 split cells
# -----------------------------------------------------------------------------

for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PRIMARY_ROOT
        / split_dir
    )


    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )


    if len(leaves) != 6:
        raise RuntimeError(
            f"{root}: expected 6 primary subset directories; "
            f"found {len(leaves)}"
        )


    for d in leaves:

        subset = normalize_subset_from_dir(
            d.name
        )

        lgb_path = find_exact_one(
            d,
            "*lightgbm_model.txt",
            f"{subset} {split}",
        )

        xgb_path = find_exact_one(
            d,
            "*xgboost_model.json",
            f"{subset} {split}",
        )

        inventory.append(
            {
                "subset":
                    subset,

                "family":
                    "PRIMARY",

                "split":
                    split,

                "directory":
                    d,

                "lightgbm":
                    lgb_path,

                "xgboost":
                    xgb_path,
            }
        )


# -----------------------------------------------------------------------------
# PLACEBO — 10 split cells
# -----------------------------------------------------------------------------

for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PLACEBO_ROOT
        / split_dir
    )


    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )


    if len(leaves) != 5:
        raise RuntimeError(
            f"{root}: expected 5 placebo subset directories; "
            f"found {len(leaves)}"
        )


    for d in leaves:

        subset = normalize_subset_from_dir(
            d.name
        )

        lgb_path = find_exact_one(
            d,
            "*lightgbm_model.txt",
            f"{subset} {split}",
        )

        xgb_path = find_exact_one(
            d,
            "*xgboost_model.json",
            f"{subset} {split}",
        )

        inventory.append(
            {
                "subset":
                    subset,

                "family":
                    "PLACEBO",

                "split":
                    split,

                "directory":
                    d,

                "lightgbm":
                    lgb_path,

                "xgboost":
                    xgb_path,
            }
        )


# Exactly:
#   12 subsets × 2 splits = 24 model pairs
#   each pair has LGBM + XGB = 48 component models.

if len(inventory) != 24:
    raise RuntimeError(
        f"Expected 24 frozen model pairs; found {len(inventory)}"
    )


observed_by_split = {
    "RANDOM_NATURAL":
        set(),

    "CHRONOLOGICAL_NATURAL":
        set(),
}


for item in inventory:

    observed_by_split[
        item["split"]
    ].add(
        item["subset"]
    )


for split in observed_by_split:

    if (
        observed_by_split[
            split
        ]
        != set(
            EXPECTED_SUBSETS
        )
    ):

        raise RuntimeError(
            f"{split}: frozen subset inventory mismatch.\n"
            f"expected={sorted(EXPECTED_SUBSETS)}\n"
            f"actual={sorted(observed_by_split[split])}"
        )


print("[EXACT] frozen model pairs : 24 / 24")
print("[EXACT] component models   : 48 / 48")
print("[EXACT] subsets per split  : 12 / 12")


# =============================================================================
# 8. LOAD EACH FROZEN MODEL — NO PREDICTION, NO SHAP
#
# Verify feature dimensions and capture LightGBM feature-name order.
# =============================================================================

print()
print("=" * 100)
print("MODEL LOAD + FEATURE-DIMENSION VERIFICATION")
print("=" * 100)
print()


model_report = []


for i, item in enumerate(
    sorted(
        inventory,
        key=lambda x: (
            x["split"],
            x["family"],
            x["subset"],
        ),
    ),
    start=1,
):

    subset = item[
        "subset"
    ]

    split = item[
        "split"
    ]


    # -------------------------------------------------------------------------
    # LightGBM
    # -------------------------------------------------------------------------

    booster_lgb = lgb.Booster(
        model_file=str(
            item[
                "lightgbm"
            ]
        )
    )


    lgb_n = int(
        booster_lgb.num_feature()
    )

    lgb_names = list(
        booster_lgb.feature_name()
    )


    if len(
        lgb_names
    ) != lgb_n:
        raise RuntimeError(
            f"{subset} {split}: LightGBM feature-name count mismatch."
        )


    # -------------------------------------------------------------------------
    # XGBoost
    # -------------------------------------------------------------------------

    booster_xgb = xgb.Booster()

    booster_xgb.load_model(
        str(
            item[
                "xgboost"
            ]
        )
    )


    xgb_n = int(
        booster_xgb.num_features()
    )

    xgb_names = (
        list(
            booster_xgb.feature_names
        )
        if booster_xgb.feature_names
        is not None
        else None
    )


    if xgb_n != lgb_n:
        raise RuntimeError(
            f"{subset} {split}: "
            f"LGBM features={lgb_n}, XGB features={xgb_n}"
        )


    if (
        xgb_names is not None
        and len(
            xgb_names
        ) != xgb_n
    ):
        raise RuntimeError(
            f"{subset} {split}: XGB feature-name count mismatch."
        )


    # If XGBoost stored actual names, require exact agreement
    # with LightGBM retained order.

    if (
        xgb_names is not None
        and xgb_names
        != lgb_names
    ):
        raise RuntimeError(
            f"{subset} {split}: component feature-order disagreement."
        )


    model_report.append(
        {
            "subset":
                subset,

            "family":
                item[
                    "family"
                ],

            "split":
                split,

            "retained_features":
                lgb_n,

            "lightgbm_feature_names":
                lgb_names,

            "xgboost_feature_names_present":
                (
                    xgb_names
                    is not None
                ),

            "lightgbm_path":
                str(
                    item[
                        "lightgbm"
                    ].relative_to(
                        REPO
                    )
                ),

            "xgboost_path":
                str(
                    item[
                        "xgboost"
                    ].relative_to(
                        REPO
                    )
                ),

            "lightgbm_sha256":
                sha256_file(
                    item[
                        "lightgbm"
                    ]
                ),

            "xgboost_sha256":
                sha256_file(
                    item[
                        "xgboost"
                    ]
                ),
        }
    )


    print(
        f"[{i:02d}/24] "
        f"{split:<23} "
        f"{subset:<27} "
        f"features={lgb_n:2d} "
        f"XGB_names={'YES' if xgb_names is not None else 'NO'}"
    )


    del (
        booster_lgb,
        booster_xgb,
    )

    gc.collect()


# =============================================================================
# 9. VERIFY EXPECTED RETAINED DIMENSIONS
# =============================================================================

expected_dimensions = {
    "FULL":
        70,

    "NO_DST_PORT":
        69,

    "NO_PORTS":
        68,

    "NO_INIT_FWD_WIN_BYTS":
        69,

    "NO_FWD_SEG_SIZE_MIN":
        69,

    "NO_SUSPICIOUS_GROUP":
        67,

    "BEHAVIOR_ONLY":
        63,

    "PLACEBO_COUNTS":
        67,

    "PLACEBO_VOLUME_DIRECTION":
        67,

    "PLACEBO_IAT":
        67,

    "PLACEBO_PACKET_SIZE":
        67,

    "PLACEBO_ACTIVITY":
        67,
}


for row in model_report:

    expected = expected_dimensions[
        row[
            "subset"
        ]
    ]

    actual = row[
        "retained_features"
    ]

    if actual != expected:

        raise RuntimeError(
            f"{row['subset']} × {row['split']}: "
            f"expected {expected} features, got {actual}"
        )


print()
print("[EXACT] all retained feature dimensions match frozen subsets")


# =============================================================================
# 10. VERIFY SAME RETAINED FEATURE ORDER BETWEEN SPLITS
# =============================================================================

for subset in EXPECTED_SUBSETS:

    rows = [
        r
        for r in model_report
        if r[
            "subset"
        ] == subset
    ]

    if len(rows) != 2:
        raise RuntimeError(
            f"{subset}: expected two split models."
        )

    if (
        rows[0][
            "lightgbm_feature_names"
        ]
        != rows[1][
            "lightgbm_feature_names"
        ]
    ):
        raise RuntimeError(
            f"{subset}: retained feature order differs between splits."
        )


print("[EXACT] retained feature order identical across splits")


# =============================================================================
# 11. VERIFY SUBSET FEATURES ARE DRAWN FROM FROZEN 70-FEATURE SPACE
# =============================================================================

full_feature_set = set(
    FULL_FEATURE_ORDER
)


for row in model_report:

    names = row[
        "lightgbm_feature_names"
    ]

    unknown = [
        x
        for x in names
        if x not in full_feature_set
    ]

    if unknown:

        raise RuntimeError(
            f"{row['subset']}: model contains features outside "
            f"frozen 70-feature space: {unknown}"
        )


print("[EXACT] every retained feature belongs to frozen 70-feature space")


# =============================================================================
# 12. TREEEXPLAINER CONSTRUCTOR SMOKE TEST
#
# No SHAP values are computed.
# No data are passed.
# No model is modified.
# =============================================================================

print()
print("=" * 100)
print("TREEEXPLAINER CONSTRUCTOR SMOKE TEST")
print("=" * 100)
print()


# Use FULL RANDOM models only for constructor compatibility test.

full_random = [
    x
    for x in inventory
    if (
        x[
            "subset"
        ] == "FULL"
        and x[
            "split"
        ] == "RANDOM_NATURAL"
    )
]


if len(full_random) != 1:
    raise RuntimeError(
        "Could not uniquely identify FULL RANDOM model pair."
    )


full_random = full_random[0]


test_lgb = lgb.Booster(
    model_file=str(
        full_random[
            "lightgbm"
        ]
    )
)

test_xgb = xgb.Booster()

test_xgb.load_model(
    str(
        full_random[
            "xgboost"
        ]
    )
)


try:

    explainer_lgb = shap.TreeExplainer(
        test_lgb
    )

    print("[OK] shap.TreeExplainer(LightGBM Booster)")


except Exception as exc:

    raise RuntimeError(
        "TreeExplainer failed for frozen LightGBM model.\n"
        "Do NOT install/upgrade packages yet.\n"
        f"{type(exc).__name__}: {exc}"
    )


try:

    explainer_xgb = shap.TreeExplainer(
        test_xgb
    )

    print("[OK] shap.TreeExplainer(XGBoost Booster)")


except Exception as exc:

    raise RuntimeError(
        "TreeExplainer failed for frozen XGBoost model.\n"
        "Do NOT install/upgrade packages yet.\n"
        f"{type(exc).__name__}: {exc}"
    )


del (
    test_lgb,
    test_xgb,
    explainer_lgb,
    explainer_xgb,
)

gc.collect()


# =============================================================================
# 13. LOCATE AUTHORIZED FEBRUARY DEVELOPMENT SOURCE — NO READ
#
# We only verify the source path exists here.
# Actual selective SHAP-cohort materialization will be the next cell.
# =============================================================================

recorded_source = Path(
    "/kaggle/input/datasets/jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)


if not recorded_source.is_dir():

    candidates = []

    input_root = Path(
        "/kaggle/input"
    )

    if input_root.exists():

        for p in input_root.rglob(
            "day_00_02-14-2018.parquet"
        ):

            parent = p.parent

            if all(
                (
                    parent
                    / f"day_{i:02d}_{date}.parquet"
                ).is_file()

                for i, date in [
                    (0, "02-14-2018"),
                    (1, "02-15-2018"),
                    (2, "02-16-2018"),
                    (3, "02-20-2018"),
                    (4, "02-21-2018"),
                    (5, "02-22-2018"),
                    (6, "02-23-2018"),
                    (7, "02-28-2018"),
                ]
            ):
                candidates.append(
                    parent
                )


    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    if len(candidates) != 1:

        raise RuntimeError(
            "Could not uniquely locate authorized February source."
        )

    recorded_source = (
        candidates[0]
    )


print()
print("[OK] authorized February source located:")
print(" ", recorded_source)
print("[OK] source files read in this cell: 0")


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-5 SHAP PREFLIGHT COMPLETE")
print("=" * 100)
print()

print("Frozen SHAP cohorts         : 2 / 2 EXACT")
print("Rows per cohort             : 5,000")
print("Class composition           : 2,500 benign + 2,500 attack")
print()
print("Frozen subsets / split      : 12")
print("Frozen model pairs          : 24 / 24")
print("Frozen component models     : 48 / 48")
print()
print("LightGBM TreeExplainer      : AVAILABLE")
print("XGBoost TreeExplainer       : AVAILABLE")
print()
print("New model fits              : 0")
print("SHAP values computed        : 0")
print("Scientific writes           : 0")
print("Parquet files read          : 0")
print("Raw Mar1 accessed           : NO")
print("Raw Mar2 accessed           : NO")
print()
print("Stage23 model state         : 50 / 50 SEALED")
print()
print("NEXT:")
print("  Materialize ONLY the two frozen 5,000-row SHAP cohorts")
print("  from the authorized February source, then compute frozen")
print("  TreeSHAP proxy-absorption measures using the 48 sealed")
print("  component models. NO RETRAINING.")
print("=" * 100)

STAGE23-5 — SHAP PROXY-ABSORPTION PREFLIGHT

[OK] HEAD      : 1e687d964aa5247a3e5d0678223dc8de92cc5ade
[OK] seal tag  : stage23-4-uncertainty-analysis-complete-v1
[OK] worktree  : CLEAN
[OK] model fits: 50 / 50 SEALED
[OK] new fits  : 0 authorized

shap_spec.json
  expected: 4e664f95b6fabc5bdfed05d1195b0fe932e297d1a0690f074c411582438ad135
  actual:   4e664f95b6fabc5bdfed05d1195b0fe932e297d1a0690f074c411582438ad135
  [EXACT]

shap_cohort_random_natural.csv
  expected: fc8df6fad72566a6649ec04db5c63630de8eb3e2f67a92be3f3466a8b5574e55
  actual:   fc8df6fad72566a6649ec04db5c63630de8eb3e2f67a92be3f3466a8b5574e55
  [EXACT]

shap_cohort_chronological_natural.csv
  expected: 510d2657ffebd9e1c89ef738767aadf72208161dfd575a4a95a1966e84316773
  actual:   510d2657ffebd9e1c89ef738767aadf72208161dfd575a4a95a1966e84316773
  [EXACT]

[EXACT] frozen SHAP specification

FROZEN SHAP COHORTS

[EXACT] RANDOM_NATURAL          rows=5,000 benign=2,500 attack=2,500
[EXACT] CHRONOLOGICAL_NATURAL   rows=5,000 beni

RuntimeError: Could not uniquely identify feature-name column.

In [17]:
# =============================================================================
# STAGE23-5 — SHAP PREFLIGHT CONTINUATION AFTER VERIFIED COLUMN-NAME STOP
#
# ZERO FITS
# ZERO SHAP VALUES
# ZERO SCIENTIFIC WRITES
# ZERO PARQUET READS
#
# Prior cell already verified:
#   - HEAD/tag/worktree
#   - frozen SHAP spec SHA
#   - both frozen cohort SHAs
#   - 5,000 rows/split, 2,500 benign + 2,500 attack
#   - feature-order CSV schema:
#         feature_position
#         feature_name
#
# This cell fixes ONLY the deterministic column selection and continues.
# =============================================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import sklearn
import shap


# =============================================================================
# 1. EXACT FEATURE-NAME COLUMN
# =============================================================================

print("=" * 100)
print("STAGE23-5 — SHAP PREFLIGHT CONTINUATION")
print("=" * 100)
print()

if "feature_order_df" not in globals():
    raise RuntimeError(
        "Prior SHAP preflight state is not present in this kernel. "
        "Do not guess; rerun a corrected full preflight instead."
    )

if list(feature_order_df.columns) != [
    "feature_position",
    "feature_name",
]:
    raise RuntimeError(
        "Frozen feature-order schema changed unexpectedly:\n"
        f"{list(feature_order_df.columns)}"
    )

if len(feature_order_df) != 70:
    raise RuntimeError(
        f"Expected 70 feature rows; found {len(feature_order_df)}"
    )

expected_positions = np.arange(
    70,
    dtype=np.int64,
)

actual_positions = np.asarray(
    feature_order_df["feature_position"],
    dtype=np.int64,
)

if not np.array_equal(
    actual_positions,
    expected_positions,
):
    raise RuntimeError(
        "feature_position is not the exact canonical 0..69 sequence."
    )

FEATURE_COLUMN = "feature_name"

FULL_FEATURE_ORDER = (
    feature_order_df[
        FEATURE_COLUMN
    ]
    .astype(str)
    .tolist()
)

if len(FULL_FEATURE_ORDER) != 70:
    raise RuntimeError(
        "Frozen feature-name count is not 70."
    )

if len(set(FULL_FEATURE_ORDER)) != 70:
    raise RuntimeError(
        "Frozen feature names are not unique."
    )

print("[EXACT] feature-name column :", FEATURE_COLUMN)
print("[EXACT] feature positions    : 0..69")
print("[EXACT] feature count        : 70")
print("[OK] first feature          :", FULL_FEATURE_ORDER[0])
print("[OK] final feature          :", FULL_FEATURE_ORDER[-1])


# =============================================================================
# 2. RUNTIME
# =============================================================================

print()
print("=" * 100)
print("SHAP RUNTIME")
print("=" * 100)
print()

print("numpy    :", np.__version__)
print("pandas   :", pd.__version__)
print("sklearn  :", sklearn.__version__)
print("lightgbm :", lgb.__version__)
print("xgboost  :", xgb.__version__)
print("shap     :", shap.__version__)


# =============================================================================
# 3. VERIFY REQUIRED PRIOR DEFINITIONS
# =============================================================================

required_globals = [
    "REPO",
    "FULL_ROOTS",
    "PRIMARY_ROOT",
    "PLACEBO_ROOT",
    "EXPECTED_SUBSETS",
    "find_exact_one",
    "normalize_subset_from_dir",
    "sha256_file",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Prior preflight definitions missing:\n"
        + "\n".join(missing_globals)
    )


# =============================================================================
# 4. INVENTORY ALL FROZEN MODEL PAIRS
# =============================================================================

print()
print("=" * 100)
print("FROZEN MODEL INVENTORY")
print("=" * 100)
print()

inventory = []


# FULL: 2 split cells
for split, d in FULL_ROOTS.items():

    lgb_path = find_exact_one(
        d,
        "*lightgbm_model.txt",
        f"FULL {split}",
    )

    xgb_path = find_exact_one(
        d,
        "*xgboost_model.json",
        f"FULL {split}",
    )

    inventory.append(
        {
            "subset": "FULL",
            "family": "FULL",
            "split": split,
            "directory": d,
            "lightgbm": lgb_path,
            "xgboost": xgb_path,
        }
    )


# PRIMARY: 6 subsets × 2 splits
for split_dir, split in [
    ("random", "RANDOM_NATURAL"),
    ("chronological", "CHRONOLOGICAL_NATURAL"),
]:

    root = (
        PRIMARY_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 6:
        raise RuntimeError(
            f"{root}: expected 6 primary directories; "
            f"found {len(leaves)}"
        )

    for d in leaves:

        subset = normalize_subset_from_dir(
            d.name
        )

        lgb_path = find_exact_one(
            d,
            "*lightgbm_model.txt",
            f"{subset} {split}",
        )

        xgb_path = find_exact_one(
            d,
            "*xgboost_model.json",
            f"{subset} {split}",
        )

        inventory.append(
            {
                "subset": subset,
                "family": "PRIMARY",
                "split": split,
                "directory": d,
                "lightgbm": lgb_path,
                "xgboost": xgb_path,
            }
        )


# PLACEBO: 5 subsets × 2 splits
for split_dir, split in [
    ("random", "RANDOM_NATURAL"),
    ("chronological", "CHRONOLOGICAL_NATURAL"),
]:

    root = (
        PLACEBO_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 5:
        raise RuntimeError(
            f"{root}: expected 5 placebo directories; "
            f"found {len(leaves)}"
        )

    for d in leaves:

        subset = normalize_subset_from_dir(
            d.name
        )

        lgb_path = find_exact_one(
            d,
            "*lightgbm_model.txt",
            f"{subset} {split}",
        )

        xgb_path = find_exact_one(
            d,
            "*xgboost_model.json",
            f"{subset} {split}",
        )

        inventory.append(
            {
                "subset": subset,
                "family": "PLACEBO",
                "split": split,
                "directory": d,
                "lightgbm": lgb_path,
                "xgboost": xgb_path,
            }
        )


if len(inventory) != 24:
    raise RuntimeError(
        f"Expected 24 frozen model pairs; found {len(inventory)}"
    )


observed_by_split = {
    "RANDOM_NATURAL": set(),
    "CHRONOLOGICAL_NATURAL": set(),
}

for item in inventory:
    observed_by_split[
        item["split"]
    ].add(
        item["subset"]
    )

for split, observed in observed_by_split.items():

    if observed != set(EXPECTED_SUBSETS):
        raise RuntimeError(
            f"{split}: subset inventory mismatch.\n"
            f"expected={sorted(EXPECTED_SUBSETS)}\n"
            f"actual={sorted(observed)}"
        )


print("[EXACT] frozen model pairs : 24 / 24")
print("[EXACT] component models   : 48 / 48")
print("[EXACT] subsets per split  : 12 / 12")


# =============================================================================
# 5. LOAD MODELS AND VERIFY RETAINED FEATURE DIMENSIONS
#
# Loading only. NO prediction. NO SHAP values. NO fitting.
# =============================================================================

print()
print("=" * 100)
print("MODEL LOAD + FEATURE-DIMENSION VERIFICATION")
print("=" * 100)
print()

expected_dimensions = {
    "FULL": 70,
    "NO_DST_PORT": 69,
    "NO_PORTS": 68,
    "NO_INIT_FWD_WIN_BYTS": 69,
    "NO_FWD_SEG_SIZE_MIN": 69,
    "NO_SUSPICIOUS_GROUP": 67,
    "BEHAVIOR_ONLY": 63,
    "PLACEBO_COUNTS": 67,
    "PLACEBO_VOLUME_DIRECTION": 67,
    "PLACEBO_IAT": 67,
    "PLACEBO_PACKET_SIZE": 67,
    "PLACEBO_ACTIVITY": 67,
}

model_report = []


for i, item in enumerate(
    sorted(
        inventory,
        key=lambda x: (
            x["split"],
            x["family"],
            x["subset"],
        ),
    ),
    start=1,
):

    subset = item["subset"]
    split = item["split"]

    # LightGBM
    booster_lgb = lgb.Booster(
        model_file=str(
            item["lightgbm"]
        )
    )

    lgb_n = int(
        booster_lgb.num_feature()
    )

    lgb_names = list(
        booster_lgb.feature_name()
    )

    if len(lgb_names) != lgb_n:
        raise RuntimeError(
            f"{subset} {split}: "
            "LightGBM feature-name count mismatch."
        )

    # XGBoost
    booster_xgb = xgb.Booster()

    booster_xgb.load_model(
        str(
            item["xgboost"]
        )
    )

    xgb_n = int(
        booster_xgb.num_features()
    )

    xgb_names = (
        list(
            booster_xgb.feature_names
        )
        if booster_xgb.feature_names is not None
        else None
    )

    if xgb_n != lgb_n:
        raise RuntimeError(
            f"{subset} {split}: "
            f"LGBM={lgb_n}, XGB={xgb_n}"
        )

    expected_n = expected_dimensions[
        subset
    ]

    if lgb_n != expected_n:
        raise RuntimeError(
            f"{subset} {split}: "
            f"expected {expected_n} features, "
            f"found {lgb_n}"
        )

    if (
        xgb_names is not None
        and len(xgb_names) != xgb_n
    ):
        raise RuntimeError(
            f"{subset} {split}: "
            "XGBoost feature-name count mismatch."
        )

    if (
        xgb_names is not None
        and xgb_names != lgb_names
    ):
        raise RuntimeError(
            f"{subset} {split}: "
            "LightGBM/XGBoost feature order disagreement."
        )

    unknown = [
        name
        for name in lgb_names
        if name not in set(
            FULL_FEATURE_ORDER
        )
    ]

    if unknown:
        raise RuntimeError(
            f"{subset} {split}: "
            f"unknown retained features: {unknown}"
        )

    model_report.append(
        {
            "subset": subset,
            "family": item["family"],
            "split": split,
            "retained_features": lgb_n,
            "lightgbm_feature_names": lgb_names,
            "xgboost_feature_names_present":
                xgb_names is not None,
            "lightgbm_path":
                str(
                    item["lightgbm"].relative_to(
                        REPO
                    )
                ),
            "xgboost_path":
                str(
                    item["xgboost"].relative_to(
                        REPO
                    )
                ),
            "lightgbm_sha256":
                sha256_file(
                    item["lightgbm"]
                ),
            "xgboost_sha256":
                sha256_file(
                    item["xgboost"]
                ),
        }
    )

    print(
        f"[{i:02d}/24] "
        f"{split:<23} "
        f"{subset:<27} "
        f"features={lgb_n:2d} "
        f"XGB_names="
        f"{'YES' if xgb_names is not None else 'NO'}"
    )

    del (
        booster_lgb,
        booster_xgb,
    )

    gc.collect()


print()
print("[EXACT] all retained feature dimensions match frozen subsets")


# =============================================================================
# 6. RETAINED FEATURE ORDER MUST MATCH ACROSS SPLITS
# =============================================================================

for subset in EXPECTED_SUBSETS:

    rows = [
        row
        for row in model_report
        if row["subset"] == subset
    ]

    if len(rows) != 2:
        raise RuntimeError(
            f"{subset}: expected exactly two split models."
        )

    if (
        rows[0]["lightgbm_feature_names"]
        != rows[1]["lightgbm_feature_names"]
    ):
        raise RuntimeError(
            f"{subset}: retained feature order differs by split."
        )


print("[EXACT] retained feature order identical across splits")


# =============================================================================
# 7. FULL MODEL ORDER MUST MATCH FROZEN 70-FEATURE ORDER
# =============================================================================

full_rows = [
    row
    for row in model_report
    if row["subset"] == "FULL"
]

if len(full_rows) != 2:
    raise RuntimeError(
        "Expected exactly two FULL models."
    )

for row in full_rows:

    if (
        row["lightgbm_feature_names"]
        != FULL_FEATURE_ORDER
    ):
        raise RuntimeError(
            f"FULL {row['split']}: "
            "model feature order does not match frozen 70-feature order."
        )


print("[EXACT] FULL model order matches frozen canonical 70-feature order")


# =============================================================================
# 8. TREEEXPLAINER CONSTRUCTOR SMOKE TEST
#
# NO SHAP VALUES.
# =============================================================================

print()
print("=" * 100)
print("TREEEXPLAINER CONSTRUCTOR SMOKE TEST")
print("=" * 100)
print()

full_random_matches = [
    item
    for item in inventory
    if (
        item["subset"] == "FULL"
        and item["split"] == "RANDOM_NATURAL"
    )
]

if len(full_random_matches) != 1:
    raise RuntimeError(
        "Could not uniquely identify FULL RANDOM model pair."
    )

full_random = full_random_matches[0]


test_lgb = lgb.Booster(
    model_file=str(
        full_random["lightgbm"]
    )
)

test_xgb = xgb.Booster()

test_xgb.load_model(
    str(
        full_random["xgboost"]
    )
)


try:
    explainer_lgb = shap.TreeExplainer(
        test_lgb
    )

    print(
        "[OK] shap.TreeExplainer(LightGBM Booster)"
    )

except Exception as exc:
    raise RuntimeError(
        "TreeExplainer constructor failed for LightGBM.\n"
        "Do NOT install/upgrade packages.\n"
        f"{type(exc).__name__}: {exc}"
    )


try:
    explainer_xgb = shap.TreeExplainer(
        test_xgb
    )

    print(
        "[OK] shap.TreeExplainer(XGBoost Booster)"
    )

except Exception as exc:
    raise RuntimeError(
        "TreeExplainer constructor failed for XGBoost.\n"
        "Do NOT install/upgrade packages.\n"
        f"{type(exc).__name__}: {exc}"
    )


del (
    test_lgb,
    test_xgb,
    explainer_lgb,
    explainer_xgb,
)

gc.collect()


# =============================================================================
# 9. LOCATE AUTHORIZED FEBRUARY SOURCE — EXISTENCE ONLY
#
# NO PARQUET READ.
# =============================================================================

recorded_source = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

if not recorded_source.is_dir():

    candidates = []

    input_root = Path(
        "/kaggle/input"
    )

    if input_root.exists():

        for p in input_root.rglob(
            "day_00_02-14-2018.parquet"
        ):

            parent = p.parent

            required = [
                "day_00_02-14-2018.parquet",
                "day_01_02-15-2018.parquet",
                "day_02_02-16-2018.parquet",
                "day_03_02-20-2018.parquet",
                "day_04_02-21-2018.parquet",
                "day_05_02-22-2018.parquet",
                "day_06_02-23-2018.parquet",
                "day_07_02-28-2018.parquet",
            ]

            if all(
                (
                    parent / name
                ).is_file()
                for name in required
            ):
                candidates.append(
                    parent
                )

    candidates = list(
        dict.fromkeys(
            candidates
        )
    )

    if len(candidates) != 1:
        raise RuntimeError(
            "Could not uniquely locate authorized February source."
        )

    recorded_source = candidates[0]


print()
print("[OK] authorized February source:")
print(" ", recorded_source)
print("[OK] Parquet files read in this cell: 0")


# =============================================================================
# 10. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-5 SHAP PREFLIGHT COMPLETE")
print("=" * 100)
print()

print("Frozen SHAP cohorts         : 2 / 2 EXACT")
print("Rows per cohort             : 5,000")
print("Class composition           : 2,500 benign + 2,500 attack")
print()
print("Frozen feature order        : 70 / 70 EXACT")
print("Frozen subsets / split      : 12")
print("Frozen model pairs          : 24 / 24")
print("Frozen component models     : 48 / 48")
print()
print("LightGBM TreeExplainer      : AVAILABLE")
print("XGBoost TreeExplainer       : AVAILABLE")
print()
print("New model fits              : 0")
print("SHAP values computed        : 0")
print("Scientific writes           : 0")
print("Parquet files read          : 0")
print("Raw Mar1 accessed           : NO")
print("Raw Mar2 accessed           : NO")
print()
print("Stage23 model state         : 50 / 50 SEALED")
print()
print("NEXT:")
print("  Materialize ONLY the two frozen 5,000-row SHAP cohorts")
print("  and execute the frozen TreeSHAP proxy-absorption analysis.")
print("  NO RETRAINING.")
print("=" * 100)

STAGE23-5 — SHAP PREFLIGHT CONTINUATION

[EXACT] feature-name column : feature_name
[EXACT] feature positions    : 0..69
[EXACT] feature count        : 70
[OK] first feature          : Dst Port
[OK] final feature          : Idle Min

SHAP RUNTIME

numpy    : 2.0.2
pandas   : 2.3.3
sklearn  : 1.6.1
lightgbm : 4.6.0
xgboost  : 3.2.0
shap     : 0.51.0

FROZEN MODEL INVENTORY

[EXACT] frozen model pairs : 24 / 24
[EXACT] component models   : 48 / 48
[EXACT] subsets per split  : 12 / 12

MODEL LOAD + FEATURE-DIMENSION VERIFICATION



RuntimeError: FULL CHRONOLOGICAL_NATURAL: unknown retained features: ['Column_0', 'Column_1', 'Column_2', 'Column_3', 'Column_4', 'Column_5', 'Column_6', 'Column_7', 'Column_8', 'Column_9', 'Column_10', 'Column_11', 'Column_12', 'Column_13', 'Column_14', 'Column_15', 'Column_16', 'Column_17', 'Column_18', 'Column_19', 'Column_20', 'Column_21', 'Column_22', 'Column_23', 'Column_24', 'Column_25', 'Column_26', 'Column_27', 'Column_28', 'Column_29', 'Column_30', 'Column_31', 'Column_32', 'Column_33', 'Column_34', 'Column_35', 'Column_36', 'Column_37', 'Column_38', 'Column_39', 'Column_40', 'Column_41', 'Column_42', 'Column_43', 'Column_44', 'Column_45', 'Column_46', 'Column_47', 'Column_48', 'Column_49', 'Column_50', 'Column_51', 'Column_52', 'Column_53', 'Column_54', 'Column_55', 'Column_56', 'Column_57', 'Column_58', 'Column_59', 'Column_60', 'Column_61', 'Column_62', 'Column_63', 'Column_64', 'Column_65', 'Column_66', 'Column_67', 'Column_68', 'Column_69']

In [18]:
# =============================================================================
# STAGE23-5 — SHAP PREFLIGHT CONTINUATION V2
#
# FIX:
#   Booster-internal names such as Column_0 ... Column_N are positional
#   placeholders and are NOT used as semantic feature identities.
#
# Semantic feature order must come from:
#   1. frozen feature_subset_spec.json / placebo_ablation_spec.json
#   2. persisted result JSON feature_order
#
# ZERO MODEL FITS
# ZERO SHAP VALUES
# ZERO SCIENTIFIC WRITES
# ZERO PARQUET READS
# =============================================================================

from pathlib import Path
import json
import gc
import subprocess
import os

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import shap


# =============================================================================
# 0. REQUIRE PRIOR PREFLIGHT STATE
# =============================================================================

print("=" * 100)
print("STAGE23-5 — SHAP PREFLIGHT CONTINUATION V2")
print("=" * 100)
print()


required_globals = [
    "REPO",
    "EXPECTED_HEAD",
    "EXPECTED_TAG",
    "PROTOCOL",
    "FULL_ROOTS",
    "PRIMARY_ROOT",
    "PLACEBO_ROOT",
    "EXPECTED_SUBSETS",
    "FULL_FEATURE_ORDER",
    "find_exact_one",
    "normalize_subset_from_dir",
    "sha256_file",
]

missing = [
    x
    for x in required_globals
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Required prior preflight state missing:\n"
        + "\n".join(missing)
    )


# =============================================================================
# 1. REVERIFY SEALED STATE
# =============================================================================

def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (p.stdout or "").strip()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


assert head == EXPECTED_HEAD
assert tag_commit == EXPECTED_HEAD
assert status == ""


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")


# =============================================================================
# 2. LOAD FROZEN FEATURE-ORDER AUTHORITIES
# =============================================================================

PRIMARY_SPEC_PATH = (
    PROTOCOL
    / "feature_subset_spec.json"
)

PLACEBO_SPEC_PATH = (
    PROTOCOL
    / "placebo_ablation_spec.json"
)


primary_spec = json.loads(
    PRIMARY_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

placebo_spec = json.loads(
    PLACEBO_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    primary_spec[
        "full_feature_order"
    ]
    != FULL_FEATURE_ORDER
):
    raise RuntimeError(
        "Frozen Stage23 feature subset spec does not "
        "match canonical Stage22R feature order."
    )


SEMANTIC_ORDER = {
    "FULL":
        list(
            primary_spec[
                "subsets"
            ][
                "FULL"
            ][
                "features"
            ]
        ),

    "NO_DST_PORT":
        list(
            primary_spec[
                "subsets"
            ][
                "NO_DST_PORT"
            ][
                "features"
            ]
        ),

    "NO_PORTS":
        list(
            primary_spec[
                "subsets"
            ][
                "NO_PORTS"
            ][
                "features"
            ]
        ),

    "NO_INIT_FWD_WIN_BYTS":
        list(
            primary_spec[
                "subsets"
            ][
                "NO_INIT_FWD_WIN_BYTS"
            ][
                "features"
            ]
        ),

    "NO_FWD_SEG_SIZE_MIN":
        list(
            primary_spec[
                "subsets"
            ][
                "NO_FWD_SEG_SIZE_MIN"
            ][
                "features"
            ]
        ),

    "NO_SUSPICIOUS_GROUP":
        list(
            primary_spec[
                "subsets"
            ][
                "NO_SUSPICIOUS_GROUP"
            ][
                "features"
            ]
        ),

    "BEHAVIOR_ONLY":
        list(
            primary_spec[
                "subsets"
            ][
                "BEHAVIOR_ONLY"
            ][
                "features"
            ]
        ),
}


for placebo_name in [
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]:

    SEMANTIC_ORDER[
        placebo_name
    ] = list(
        placebo_spec[
            "subsets"
        ][
            placebo_name
        ][
            "retained"
        ]
    )


if set(
    SEMANTIC_ORDER
) != set(
    EXPECTED_SUBSETS
):
    raise RuntimeError(
        "Frozen semantic-order subset inventory mismatch."
    )


print()
print("[EXACT] semantic feature order loaded for all 12 subsets")


# =============================================================================
# 3. EXPECTED DIMENSIONS
# =============================================================================

EXPECTED_DIMENSIONS = {
    name:
        len(
            features
        )
    for name, features
    in SEMANTIC_ORDER.items()
}


expected_fixed = {
    "FULL": 70,
    "NO_DST_PORT": 69,
    "NO_PORTS": 68,
    "NO_INIT_FWD_WIN_BYTS": 69,
    "NO_FWD_SEG_SIZE_MIN": 69,
    "NO_SUSPICIOUS_GROUP": 67,
    "BEHAVIOR_ONLY": 63,
    "PLACEBO_COUNTS": 67,
    "PLACEBO_VOLUME_DIRECTION": 67,
    "PLACEBO_IAT": 67,
    "PLACEBO_PACKET_SIZE": 67,
    "PLACEBO_ACTIVITY": 67,
}


if EXPECTED_DIMENSIONS != expected_fixed:
    raise RuntimeError(
        "Frozen subset dimensions do not match expected Stage23 design."
    )


print("[EXACT] frozen subset dimensions verified")


# =============================================================================
# 4. REBUILD 24-PAIR MODEL INVENTORY
# =============================================================================

inventory = []


# FULL
for split, directory in FULL_ROOTS.items():

    inventory.append(
        {
            "subset":
                "FULL",

            "family":
                "FULL",

            "split":
                split,

            "directory":
                directory,

            "lightgbm":
                find_exact_one(
                    directory,
                    "*lightgbm_model.txt",
                    f"FULL {split}",
                ),

            "xgboost":
                find_exact_one(
                    directory,
                    "*xgboost_model.json",
                    f"FULL {split}",
                ),
        }
    )


# PRIMARY
for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PRIMARY_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 6:
        raise RuntimeError(
            f"{root}: expected 6 primary leaves."
        )

    for directory in leaves:

        subset = normalize_subset_from_dir(
            directory.name
        )

        inventory.append(
            {
                "subset":
                    subset,

                "family":
                    "PRIMARY",

                "split":
                    split,

                "directory":
                    directory,

                "lightgbm":
                    find_exact_one(
                        directory,
                        "*lightgbm_model.txt",
                        f"{subset} {split}",
                    ),

                "xgboost":
                    find_exact_one(
                        directory,
                        "*xgboost_model.json",
                        f"{subset} {split}",
                    ),
            }
        )


# PLACEBO
for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PLACEBO_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 5:
        raise RuntimeError(
            f"{root}: expected 5 placebo leaves."
        )

    for directory in leaves:

        subset = normalize_subset_from_dir(
            directory.name
        )

        inventory.append(
            {
                "subset":
                    subset,

                "family":
                    "PLACEBO",

                "split":
                    split,

                "directory":
                    directory,

                "lightgbm":
                    find_exact_one(
                        directory,
                        "*lightgbm_model.txt",
                        f"{subset} {split}",
                    ),

                "xgboost":
                    find_exact_one(
                        directory,
                        "*xgboost_model.json",
                        f"{subset} {split}",
                    ),
            }
        )


if len(inventory) != 24:
    raise RuntimeError(
        f"Expected 24 model pairs; found {len(inventory)}"
    )


print()
print("[EXACT] frozen model pairs : 24 / 24")
print("[EXACT] component models   : 48 / 48")


# =============================================================================
# 5. RESULT JSON FEATURE-ORDER EXTRACTION
# =============================================================================

def result_json_for_item(
    item,
):

    subset = item[
        "subset"
    ]

    split = item[
        "split"
    ]

    directory = item[
        "directory"
    ]


    if subset == "FULL":

        if split == "RANDOM_NATURAL":

            path = (
                directory
                / "stage22r_2a_random_natural_result.json"
            )

        else:

            path = (
                directory
                / "stage22r_2c_chronological_natural_result.json"
            )


        if not path.is_file():
            raise RuntimeError(
                f"FULL result missing: {path}"
            )

        return path


    matches = sorted(
        directory.glob(
            "*_result.json"
        )
    )


    if len(matches) != 1:

        raise RuntimeError(
            f"{subset} × {split}: "
            f"expected exactly one result JSON; "
            f"found {len(matches)}"
        )


    return matches[0]


def persisted_feature_order(
    result,
    subset,
):

    # Stage23 primary/placebo result schema.
    if (
        isinstance(
            result.get(
                "cell"
            ),
            dict,
        )
        and "feature_order"
        in result[
            "cell"
        ]
    ):

        return list(
            result[
                "cell"
            ][
                "feature_order"
            ]
        )


    # Stage22R FULL result schema.
    if (
        isinstance(
            result.get(
                "data"
            ),
            dict,
        )
        and "feature_order"
        in result[
            "data"
        ]
    ):

        return list(
            result[
                "data"
            ][
                "feature_order"
            ]
        )


    raise RuntimeError(
        f"{subset}: persisted feature_order not found in result JSON."
    )


# =============================================================================
# 6. INTERNAL NAME CLASSIFICATION
#
# Semantic names are NOT inferred from model metadata.
# Model metadata may legitimately be:
#   - exact semantic names
#   - Column_0 ... Column_N
#   - f0 ... fN
#   - None (XGBoost)
# =============================================================================

def classify_internal_names(
    names,
    expected_semantic,
    component,
):

    n = len(
        expected_semantic
    )


    if names is None:

        if component != "XGBoost":
            raise RuntimeError(
                "Only XGBoost may have absent feature_names."
            )

        return "ABSENT_POSITIONAL"


    names = list(
        names
    )


    if names == expected_semantic:

        return "SEMANTIC_EXACT"


    lightgbm_columns = [
        f"Column_{i}"
        for i in range(n)
    ]


    xgboost_columns = [
        f"f{i}"
        for i in range(n)
    ]


    if names == lightgbm_columns:

        return "POSITIONAL_COLUMN_N"


    if names == xgboost_columns:

        return "POSITIONAL_F_N"


    raise RuntimeError(
        f"{component}: internal feature names are neither "
        "the exact frozen semantic order nor an accepted "
        "pure positional sequence.\n"
        f"first names={names[:5]}"
    )


# =============================================================================
# 7. VERIFY EVERY RESULT + MODEL POSITIONALLY
# =============================================================================

print()
print("=" * 100)
print("MODEL + RESULT FEATURE-ORDER VERIFICATION")
print("=" * 100)
print()


model_report = []


for i, item in enumerate(
    sorted(
        inventory,
        key=lambda x: (
            x[
                "split"
            ],
            x[
                "family"
            ],
            x[
                "subset"
            ],
        ),
    ),
    start=1,
):

    subset = item[
        "subset"
    ]

    split = item[
        "split"
    ]

    expected_order = (
        SEMANTIC_ORDER[
            subset
        ]
    )

    expected_n = len(
        expected_order
    )


    # -------------------------------------------------------------------------
    # Persisted scientific result feature order.
    # -------------------------------------------------------------------------

    result_path = (
        result_json_for_item(
            item
        )
    )

    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    result_order = (
        persisted_feature_order(
            result,
            subset,
        )
    )


    if result_order != expected_order:

        raise RuntimeError(
            f"{subset} × {split}: persisted result feature order "
            "does not match frozen subset specification."
        )


    # -------------------------------------------------------------------------
    # LightGBM model
    # -------------------------------------------------------------------------

    booster_lgb = lgb.Booster(
        model_file=str(
            item[
                "lightgbm"
            ]
        )
    )

    lgb_n = int(
        booster_lgb.num_feature()
    )

    lgb_internal = list(
        booster_lgb.feature_name()
    )


    if lgb_n != expected_n:

        raise RuntimeError(
            f"{subset} × {split}: LightGBM feature count "
            f"{lgb_n} != {expected_n}"
        )


    lgb_name_mode = (
        classify_internal_names(
            lgb_internal,
            expected_order,
            "LightGBM",
        )
    )


    # -------------------------------------------------------------------------
    # XGBoost model
    # -------------------------------------------------------------------------

    booster_xgb = (
        xgb.Booster()
    )

    booster_xgb.load_model(
        str(
            item[
                "xgboost"
            ]
        )
    )

    xgb_n = int(
        booster_xgb.num_features()
    )


    if xgb_n != expected_n:

        raise RuntimeError(
            f"{subset} × {split}: XGBoost feature count "
            f"{xgb_n} != {expected_n}"
        )


    xgb_internal = (
        list(
            booster_xgb.feature_names
        )
        if booster_xgb.feature_names
        is not None
        else None
    )


    xgb_name_mode = (
        classify_internal_names(
            xgb_internal,
            expected_order,
            "XGBoost",
        )
    )


    model_report.append(
        {
            "subset":
                subset,

            "family":
                item[
                    "family"
                ],

            "split":
                split,

            "feature_count":
                expected_n,

            "semantic_feature_order":
                expected_order,

            "result_path":
                str(
                    result_path.relative_to(
                        REPO
                    )
                ),

            "result_feature_order_verified":
                True,

            "lightgbm_internal_name_mode":
                lgb_name_mode,

            "xgboost_internal_name_mode":
                xgb_name_mode,

            "lightgbm_sha256":
                sha256_file(
                    item[
                        "lightgbm"
                    ]
                ),

            "xgboost_sha256":
                sha256_file(
                    item[
                        "xgboost"
                    ]
                ),
        }
    )


    print(
        f"[{i:02d}/24] "
        f"{split:<23} "
        f"{subset:<27} "
        f"features={expected_n:2d} "
        f"LGB={lgb_name_mode:<20} "
        f"XGB={xgb_name_mode}"
    )


    del (
        booster_lgb,
        booster_xgb,
        result,
    )

    gc.collect()


print()
print(
    "[EXACT] all 24 result feature orders match frozen specifications"
)

print(
    "[EXACT] all 48 component model dimensions match those orders"
)


# =============================================================================
# 8. SAME SEMANTIC ORDER ACROSS SPLITS
# =============================================================================

for subset in EXPECTED_SUBSETS:

    rows = [
        row
        for row in model_report
        if row[
            "subset"
        ] == subset
    ]


    if len(rows) != 2:

        raise RuntimeError(
            f"{subset}: expected exactly two split records."
        )


    if (
        rows[0][
            "semantic_feature_order"
        ]
        != rows[1][
            "semantic_feature_order"
        ]
    ):

        raise RuntimeError(
            f"{subset}: semantic feature order differs between splits."
        )


print(
    "[EXACT] semantic retained feature order identical across splits"
)


# =============================================================================
# 9. TREEEXPLAINER CONSTRUCTOR SMOKE TEST
#
# Still NO SHAP values.
# =============================================================================

print()
print("=" * 100)
print("TREEEXPLAINER CONSTRUCTOR SMOKE TEST")
print("=" * 100)
print()


full_random = [
    item
    for item in inventory
    if (
        item[
            "subset"
        ] == "FULL"
        and item[
            "split"
        ] == "RANDOM_NATURAL"
    )
]


if len(full_random) != 1:

    raise RuntimeError(
        "Could not uniquely identify FULL RANDOM model pair."
    )


full_random = (
    full_random[0]
)


test_lgb = lgb.Booster(
    model_file=str(
        full_random[
            "lightgbm"
        ]
    )
)


test_xgb = xgb.Booster()

test_xgb.load_model(
    str(
        full_random[
            "xgboost"
        ]
    )
)


explainer_lgb = shap.TreeExplainer(
    test_lgb
)

print(
    "[OK] shap.TreeExplainer(LightGBM Booster)"
)


explainer_xgb = shap.TreeExplainer(
    test_xgb
)

print(
    "[OK] shap.TreeExplainer(XGBoost Booster)"
)


del (
    test_lgb,
    test_xgb,
    explainer_lgb,
    explainer_xgb,
)

gc.collect()


# =============================================================================
# 10. AUTHORIZED FEBRUARY SOURCE — EXISTENCE ONLY
# =============================================================================

recorded_source = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)


if not recorded_source.is_dir():

    candidates = []

    input_root = Path(
        "/kaggle/input"
    )


    if input_root.exists():

        required_files = [
            "day_00_02-14-2018.parquet",
            "day_01_02-15-2018.parquet",
            "day_02_02-16-2018.parquet",
            "day_03_02-20-2018.parquet",
            "day_04_02-21-2018.parquet",
            "day_05_02-22-2018.parquet",
            "day_06_02-23-2018.parquet",
            "day_07_02-28-2018.parquet",
        ]


        for first in input_root.rglob(
            required_files[0]
        ):

            parent = (
                first.parent
            )


            if all(
                (
                    parent
                    / filename
                ).is_file()

                for filename
                in required_files
            ):

                candidates.append(
                    parent
                )


    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    if len(candidates) != 1:

        raise RuntimeError(
            "Could not uniquely locate authorized February source."
        )


    recorded_source = (
        candidates[0]
    )


print()
print(
    "[OK] authorized February source:"
)

print(
    " ",
    recorded_source,
)

print(
    "[OK] Parquet files read in this cell: 0"
)


# =============================================================================
# 11. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-5 SHAP PREFLIGHT COMPLETE")
print("=" * 100)
print()

print("Frozen semantic subsets     : 12 / 12")
print("Frozen model pairs          : 24 / 24")
print("Frozen component models     : 48 / 48")
print("Persisted result orders     : 24 / 24 EXACT")
print("Component dimensions        : 48 / 48 EXACT")
print()
print("Semantic source:")
print("  Frozen Stage23 subset specifications")
print("  + persisted result feature_order")
print()
print("Model internal names:")
print("  Semantic names OR positional placeholders accepted")
print("  only after exact result/spec positional verification.")
print()
print("LightGBM TreeExplainer      : AVAILABLE")
print("XGBoost TreeExplainer       : AVAILABLE")
print()
print("New model fits              : 0")
print("SHAP values computed        : 0")
print("Scientific writes           : 0")
print("Parquet files read          : 0")
print("Raw Mar1 accessed           : NO")
print("Raw Mar2 accessed           : NO")
print()
print("Stage23 model state         : 50 / 50 SEALED")
print()
print("NEXT:")
print("  Materialize ONLY the two frozen 5,000-row SHAP cohorts")
print("  and execute frozen component-wise TreeSHAP.")
print("  NO RETRAINING.")
print("=" * 100)

STAGE23-5 — SHAP PREFLIGHT CONTINUATION V2

[OK] HEAD      : 1e687d964aa5247a3e5d0678223dc8de92cc5ade
[OK] seal tag  : stage23-4-uncertainty-analysis-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] new fits  : 0 authorized

[EXACT] semantic feature order loaded for all 12 subsets
[EXACT] frozen subset dimensions verified

[EXACT] frozen model pairs : 24 / 24
[EXACT] component models   : 48 / 48

MODEL + RESULT FEATURE-ORDER VERIFICATION

[01/24] CHRONOLOGICAL_NATURAL   FULL                        features=70 LGB=POSITIONAL_COLUMN_N  XGB=ABSENT_POSITIONAL
[02/24] CHRONOLOGICAL_NATURAL   PLACEBO_ACTIVITY            features=67 LGB=POSITIONAL_COLUMN_N  XGB=ABSENT_POSITIONAL
[03/24] CHRONOLOGICAL_NATURAL   PLACEBO_COUNTS              features=67 LGB=POSITIONAL_COLUMN_N  XGB=ABSENT_POSITIONAL
[04/24] CHRONOLOGICAL_NATURAL   PLACEBO_IAT                 features=67 LGB=POSITIONAL_COLUMN_N  XGB=ABSENT_POSITIONAL
[05/24] CHRONOLOGICAL_NATURAL   PLACEBO_PACKET_SIZE       

In [19]:
# =============================================================================
# STAGE23-5A — MATERIALIZE FROZEN SHAP COHORTS
#
# MATERIALIZATION ONLY
#
# ZERO MODEL FITS
# ZERO SHAP VALUES
# NO RAW MAR1
# NO RAW MAR2
#
# Reads ONLY the authorized February development Parquets and ONLY the
# row groups needed to recover the prospectively frozen 5,000-row SHAP
# cohort for each natural split.
#
# Output:
#   /kaggle/working/stage23_5a_shap_materialization/
#
#   random_natural_shap_cohort_full70.npz
#   chronological_natural_shap_cohort_full70.npz
#   stage23_5a_materialization_manifest.json
#   execution_state.json
#   checksums.sha256
#
# No TreeSHAP computation occurs in this cell.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "1e687d964aa5247a3e5d0678223dc8de92cc5ade"
)

EXPECTED_TAG = (
    "stage23-4-uncertainty-analysis-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

RANDOM_COHORT_PATH = (
    PROTOCOL
    / "shap_cohort_random_natural.csv"
)

CHRONO_COHORT_PATH = (
    PROTOCOL
    / "shap_cohort_chronological_natural.csv"
)

PROTOCOL_CHECKSUMS = (
    PROTOCOL
    / "checksums.sha256"
)

FEATURE_ORDER_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_70_feature_order.csv"
)

SOURCE = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

OUT = Path(
    "/kaggle/working/stage23_5a_shap_materialization"
)


DAY_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


EXPECTED_ROWS = [
    822_947,
    1_046_154,
    900_988,
    7_926_258,
    1_031_018,
    1_045_297,
    1_045_961,
    593_780,
]


EXPECTED_TOTAL_ROWS = (
    14_412_403
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def parse_manifest(path):

    result = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        result[
            rel.strip()
        ] = digest.strip()

    return result


# =============================================================================
# 2. FAIL-CLOSED OUTPUT GUARD
# =============================================================================

print("=" * 100)
print("STAGE23-5A — FROZEN SHAP COHORT MATERIALIZATION")
print("=" * 100)
print()


if OUT.exists():

    raise RuntimeError(
        "Stage23-5A output directory already exists.\n"
        "Do NOT overwrite blindly:\n"
        f"{OUT}"
    )


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


STATE_PATH = (
    OUT
    / "execution_state.json"
)


state = {
    "stage":
        "Stage23-5A SHAP cohort materialization",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "model_fits":
        0,

    "shap_values_computed":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,

    "parquet_files_read":
        0,

    "parquet_row_groups_read":
        0,

    "completed_splits":
        [],
}


write_json(
    STATE_PATH,
    state,
)


# =============================================================================
# 3. SEALED REPOSITORY STATE
# =============================================================================

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-4 seal tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository is dirty:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")
print("[OK] SHAP done : 0")


# =============================================================================
# 4. VERIFY FROZEN SHAP COHORTS AGAIN
# =============================================================================

protocol_manifest = parse_manifest(
    PROTOCOL_CHECKSUMS
)


for p in [
    RANDOM_COHORT_PATH,
    CHRONO_COHORT_PATH,
]:

    expected = (
        protocol_manifest.get(
            p.name
        )
    )

    actual = sha256_file(
        p
    )


    if expected is None:

        raise RuntimeError(
            f"{p.name} absent from frozen Stage23-0 manifest."
        )


    if actual != expected:

        raise RuntimeError(
            f"Frozen SHA mismatch: {p.name}"
        )


    print()
    print(p.name)
    print("  expected:", expected)
    print("  actual:  ", actual)
    print("  [EXACT]")


# =============================================================================
# 5. FROZEN 70-FEATURE ORDER
# =============================================================================

feature_order_df = pd.read_csv(
    FEATURE_ORDER_PATH
)


if list(
    feature_order_df.columns
) != [
    "feature_position",
    "feature_name",
]:

    raise RuntimeError(
        "Unexpected feature-order schema."
    )


if not np.array_equal(
    np.asarray(
        feature_order_df[
            "feature_position"
        ],
        dtype=np.int64,
    ),
    np.arange(
        70,
        dtype=np.int64,
    ),
):

    raise RuntimeError(
        "Frozen feature positions are not canonical 0..69."
    )


FULL_FEATURE_ORDER = (
    feature_order_df[
        "feature_name"
    ]
    .astype(str)
    .tolist()
)


if (
    len(
        FULL_FEATURE_ORDER
    )
    != 70
):

    raise RuntimeError(
        "Frozen feature count != 70."
    )


if (
    len(
        set(
            FULL_FEATURE_ORDER
        )
    )
    != 70
):

    raise RuntimeError(
        "Frozen feature names are not unique."
    )


print()
print("[EXACT] frozen feature order: 70 / 70")


# =============================================================================
# 6. VERIFY AUTHORIZED FEBRUARY PARQUET METADATA
#
# Metadata only at this stage.
# =============================================================================

print()
print("=" * 100)
print("AUTHORIZED FEBRUARY SOURCE VERIFICATION")
print("=" * 100)
print()


if not SOURCE.is_dir():

    raise RuntimeError(
        f"Authorized source missing:\n{SOURCE}"
    )


parquet_meta = []

running_start = 0


for filename, expected_rows in zip(
    DAY_FILES,
    EXPECTED_ROWS,
):

    path = (
        SOURCE
        / filename
    )


    if not path.is_file():

        raise RuntimeError(
            f"Authorized Parquet missing:\n{path}"
        )


    pf = pq.ParquetFile(
        path
    )


    rows = int(
        pf.metadata.num_rows
    )


    if rows != expected_rows:

        raise RuntimeError(
            f"{filename}: row-count mismatch.\n"
            f"expected={expected_rows}\n"
            f"actual={rows}"
        )


    schema_names = list(
        pf.schema_arrow.names
    )


    missing_features = [
        feature
        for feature
        in FULL_FEATURE_ORDER
        if feature not in schema_names
    ]


    if missing_features:

        raise RuntimeError(
            f"{filename}: frozen model features missing from Parquet:\n"
            + "\n".join(
                missing_features
            )
        )


    parquet_meta.append(
        {
            "filename":
                filename,

            "path":
                path,

            "rows":
                rows,

            "start":
                running_start,

            "end_exclusive":
                running_start
                + rows,

            "num_row_groups":
                int(
                    pf.num_row_groups
                ),

            "schema_names":
                schema_names,
        }
    )


    print(
        f"[EXACT] {filename:<33} "
        f"rows={rows:>10,} "
        f"row_groups={pf.num_row_groups}"
    )


    running_start += rows


if running_start != EXPECTED_TOTAL_ROWS:

    raise RuntimeError(
        "Authorized February total row count mismatch."
    )


print()
print(
    "[EXACT] total canonical development rows:",
    f"{running_start:,}",
)


# =============================================================================
# 7. LOAD FROZEN SHAP LOCATORS
# =============================================================================

cohort_frames = {
    "RANDOM_NATURAL":
        pd.read_csv(
            RANDOM_COHORT_PATH
        ),

    "CHRONOLOGICAL_NATURAL":
        pd.read_csv(
            CHRONO_COHORT_PATH
        ),
}


for split, df in cohort_frames.items():

    if list(
        df.columns
    ) != [
        "split",
        "clean_position",
        "binary_label",
    ]:

        raise RuntimeError(
            f"{split}: unexpected cohort schema."
        )


    if len(df) != 5000:

        raise RuntimeError(
            f"{split}: expected 5000 frozen rows."
        )


    if (
        df[
            "clean_position"
        ].duplicated().any()
    ):

        raise RuntimeError(
            f"{split}: duplicate clean_position."
        )


    clean_position = np.asarray(
        df[
            "clean_position"
        ],
        dtype=np.int64,
    )


    if np.any(
        clean_position < 0
    ):

        raise RuntimeError(
            f"{split}: negative clean_position."
        )


    if np.any(
        clean_position
        >= EXPECTED_TOTAL_ROWS
    ):

        raise RuntimeError(
            f"{split}: clean_position outside canonical "
            "February development range."
        )


    y = np.asarray(
        df[
            "binary_label"
        ],
        dtype=np.uint8,
    )


    benign = int(
        np.sum(
            y == 0
        )
    )

    attack = int(
        np.sum(
            y == 1
        )
    )


    if (
        benign != 2500
        or attack != 2500
    ):

        raise RuntimeError(
            f"{split}: frozen class counts incorrect."
        )


    expected_split_literal = split


    if not (
        df[
            "split"
        ]
        == expected_split_literal
    ).all():

        raise RuntimeError(
            f"{split}: split column contains unexpected values."
        )


    print()
    print(
        f"[EXACT] {split:<23} "
        "rows=5,000 benign=2,500 attack=2,500"
    )

    print(
        "        clean_position range:",
        int(
            clean_position.min()
        ),
        "→",
        int(
            clean_position.max()
        ),
    )


# =============================================================================
# 8. MAP CANONICAL clean_position -> DAY + LOCAL ROW
# =============================================================================

day_ends = np.asarray(
    [
        item[
            "end_exclusive"
        ]
        for item in parquet_meta
    ],
    dtype=np.int64,
)


def map_clean_positions(
    clean_positions,
):

    clean_positions = np.asarray(
        clean_positions,
        dtype=np.int64,
    )


    day_index = np.searchsorted(
        day_ends,
        clean_positions,
        side="right",
    )


    if np.any(
        day_index < 0
    ) or np.any(
        day_index
        >= len(
            parquet_meta
        )
    ):

        raise RuntimeError(
            "clean_position -> day mapping failed."
        )


    local_row = np.empty(
        len(
            clean_positions
        ),
        dtype=np.int64,
    )


    for i in range(
        len(
            parquet_meta
        )
    ):

        mask = (
            day_index == i
        )

        if not np.any(
            mask
        ):
            continue


        local_row[
            mask
        ] = (
            clean_positions[
                mask
            ]
            - parquet_meta[
                i
            ][
                "start"
            ]
        )


        if np.any(
            local_row[
                mask
            ] < 0
        ) or np.any(
            local_row[
                mask
            ]
            >= parquet_meta[
                i
            ][
                "rows"
            ]
        ):

            raise RuntimeError(
                "Local-row mapping outside day bounds."
            )


    return (
        day_index,
        local_row,
    )


# =============================================================================
# 9. SELECTIVE ROW-GROUP MATERIALIZATION
# =============================================================================

def materialize_split(
    split,
    cohort_df,
):

    print()
    print("=" * 100)
    print(
        f"MATERIALIZE — {split}"
    )
    print("=" * 100)
    print()


    clean_position = np.asarray(
        cohort_df[
            "clean_position"
        ],
        dtype=np.int64,
    )

    binary_label = np.asarray(
        cohort_df[
            "binary_label"
        ],
        dtype=np.uint8,
    )


    day_index, local_rows = (
        map_clean_positions(
            clean_position
        )
    )


    X = np.empty(
        (
            len(
                clean_position
            ),
            70,
        ),
        dtype=np.float64,
    )


    populated = np.zeros(
        len(
            clean_position
        ),
        dtype=np.bool_,
    )


    files_touched = 0
    row_groups_touched = 0


    for day_i, meta in enumerate(
        parquet_meta
    ):

        target_mask = (
            day_index
            == day_i
        )


        if not np.any(
            target_mask
        ):
            continue


        files_touched += 1


        target_output_indices = np.flatnonzero(
            target_mask
        )

        target_local_rows = local_rows[
            target_mask
        ]


        path = meta[
            "path"
        ]

        pf = pq.ParquetFile(
            path
        )


        # -------------------------------------------------------------
        # Build row-group boundaries.
        # -------------------------------------------------------------

        rg_starts = []
        rg_ends = []

        cursor = 0


        for rg in range(
            pf.num_row_groups
        ):

            rg_rows = int(
                pf.metadata
                .row_group(
                    rg
                )
                .num_rows
            )

            rg_starts.append(
                cursor
            )

            cursor += rg_rows

            rg_ends.append(
                cursor
            )


        if cursor != meta[
            "rows"
        ]:

            raise RuntimeError(
                f"{meta['filename']}: row-group total mismatch."
            )


        rg_starts = np.asarray(
            rg_starts,
            dtype=np.int64,
        )

        rg_ends = np.asarray(
            rg_ends,
            dtype=np.int64,
        )


        target_rg = np.searchsorted(
            rg_ends,
            target_local_rows,
            side="right",
        )


        if np.any(
            target_rg
            >= pf.num_row_groups
        ):

            raise RuntimeError(
                f"{meta['filename']}: target row-group mapping failed."
            )


        unique_rgs = np.unique(
            target_rg
        )


        print(
            f"{meta['filename']}"
        )

        print(
            "  frozen rows needed:",
            len(
                target_local_rows
            ),
        )

        print(
            "  row groups needed :",
            len(
                unique_rgs
            ),
            "/",
            pf.num_row_groups,
        )


        for rg in unique_rgs:

            rg = int(
                rg
            )

            rg_mask = (
                target_rg
                == rg
            )

            output_indices = (
                target_output_indices[
                    rg_mask
                ]
            )

            local_targets = (
                target_local_rows[
                    rg_mask
                ]
            )

            within_rg = (
                local_targets
                - rg_starts[
                    rg
                ]
            )


            table = pf.read_row_group(
                rg,
                columns=FULL_FEATURE_ORDER,
                use_threads=True,
            )


            if table.num_rows != (
                rg_ends[
                    rg
                ]
                - rg_starts[
                    rg
                ]
            ):

                raise RuntimeError(
                    f"{meta['filename']} row-group {rg}: "
                    "read row count mismatch."
                )


            if list(
                table.column_names
            ) != FULL_FEATURE_ORDER:

                raise RuntimeError(
                    f"{meta['filename']} row-group {rg}: "
                    "feature order changed during read."
                )


            # ---------------------------------------------------------
            # Convert only selected frozen rows to float64.
            # ---------------------------------------------------------

            for feature_position, feature_name in enumerate(
                FULL_FEATURE_ORDER
            ):

                column = np.asarray(
                    table[
                        feature_name
                    ].to_numpy(
                        zero_copy_only=False
                    ),
                    dtype=np.float64,
                )


                X[
                    output_indices,
                    feature_position,
                ] = column[
                    within_rg
                ]


            populated[
                output_indices
            ] = True


            row_groups_touched += 1


            del (
                table,
                within_rg,
                output_indices,
                local_targets,
            )

            gc.collect()


        del pf

        gc.collect()


    if not np.all(
        populated
    ):

        missing = np.flatnonzero(
            ~populated
        )

        raise RuntimeError(
            f"{split}: not every frozen cohort row was materialized. "
            f"missing={len(missing)}"
        )


    if X.shape != (
        5000,
        70,
    ):

        raise RuntimeError(
            f"{split}: unexpected materialized matrix shape {X.shape}"
        )


    # No feature transformation, no imputation.
    # Preserve raw frozen float64 model-input values.

    output_name = (
        "random_natural_shap_cohort_full70.npz"
        if split
        == "RANDOM_NATURAL"
        else
        "chronological_natural_shap_cohort_full70.npz"
    )


    output_path = (
        OUT
        / output_name
    )


    np.savez_compressed(
        output_path,
        clean_position=
            clean_position,

        binary_label=
            binary_label,

        feature_names=
            np.asarray(
                FULL_FEATURE_ORDER,
                dtype="U",
            ),

        X_full70=
            X,
    )


    print()
    print("[COMPLETE]")
    print("  rows             :", X.shape[0])
    print("  features         :", X.shape[1])
    print("  dtype            :", X.dtype)
    print("  parquet files    :", files_touched)
    print("  row groups read  :", row_groups_touched)
    print("  output           :", output_path)
    print("  SHA256           :", sha256_file(output_path))


    return {
        "split":
            split,

        "rows":
            int(
                X.shape[
                    0
                ]
            ),

        "features":
            int(
                X.shape[
                    1
                ]
            ),

        "dtype":
            str(
                X.dtype
            ),

        "benign":
            int(
                np.sum(
                    binary_label
                    == 0
                )
            ),

        "attack":
            int(
                np.sum(
                    binary_label
                    == 1
                )
            ),

        "clean_position_min":
            int(
                clean_position.min()
            ),

        "clean_position_max":
            int(
                clean_position.max()
            ),

        "parquet_files_read":
            files_touched,

        "parquet_row_groups_read":
            row_groups_touched,

        "output_file":
            output_name,

        "output_sha256":
            sha256_file(
                output_path
            ),
    }


# =============================================================================
# 10. MATERIALIZE BOTH FROZEN SPLITS
# =============================================================================

results = []


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    try:

        result = materialize_split(
            split,
            cohort_frames[
                split
            ],
        )


        results.append(
            result
        )


        state[
            "completed_splits"
        ].append(
            split
        )

        state[
            "parquet_files_read"
        ] += result[
            "parquet_files_read"
        ]

        state[
            "parquet_row_groups_read"
        ] += result[
            "parquet_row_groups_read"
        ]

        state[
            "status"
        ] = (
            f"{split}_MATERIALIZATION_COMPLETE"
        )


        write_json(
            STATE_PATH,
            state,
        )


    except Exception:

        state[
            "status"
        ] = (
            "MATERIALIZATION_FAILED"
        )

        write_json(
            STATE_PATH,
            state,
        )

        raise


# =============================================================================
# 11. VERIFY OUTPUT CONTENTS
# =============================================================================

print()
print("=" * 100)
print("VERIFY MATERIALIZED COHORT ARTIFACTS")
print("=" * 100)
print()


for result in results:

    p = (
        OUT
        / result[
            "output_file"
        ]
    )


    with np.load(
        p,
        allow_pickle=False,
    ) as z:

        required_keys = {
            "clean_position",
            "binary_label",
            "feature_names",
            "X_full70",
        }


        if set(
            z.files
        ) != required_keys:

            raise RuntimeError(
                f"{result['split']}: unexpected NPZ schema."
            )


        cp = np.asarray(
            z[
                "clean_position"
            ],
            dtype=np.int64,
        )

        y = np.asarray(
            z[
                "binary_label"
            ],
            dtype=np.uint8,
        )

        names = (
            z[
                "feature_names"
            ].astype(str).tolist()
        )

        X = np.asarray(
            z[
                "X_full70"
            ]
        )


    source_df = cohort_frames[
        result[
            "split"
        ]
    ]


    if not np.array_equal(
        cp,
        np.asarray(
            source_df[
                "clean_position"
            ],
            dtype=np.int64,
        ),
    ):

        raise RuntimeError(
            f"{result['split']}: clean_position order changed."
        )


    if not np.array_equal(
        y,
        np.asarray(
            source_df[
                "binary_label"
            ],
            dtype=np.uint8,
        ),
    ):

        raise RuntimeError(
            f"{result['split']}: binary_label order changed."
        )


    if names != FULL_FEATURE_ORDER:

        raise RuntimeError(
            f"{result['split']}: feature order changed."
        )


    if X.shape != (
        5000,
        70,
    ):

        raise RuntimeError(
            f"{result['split']}: matrix shape changed."
        )


    if X.dtype != np.float64:

        raise RuntimeError(
            f"{result['split']}: dtype changed from float64."
        )


    print(
        f"[EXACT] {result['split']:<23} "
        "rows=5000 features=70 dtype=float64"
    )


# =============================================================================
# 12. MANIFEST
# =============================================================================

MANIFEST_PATH = (
    OUT
    / "stage23_5a_materialization_manifest.json"
)


manifest_payload = {
    "stage":
        "Stage23-5A frozen SHAP cohort materialization",

    "status":
        "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED_UNSEALED",

    "completed_utc":
        utc_now(),

    "execution_parent": {
        "commit":
            EXPECTED_HEAD,

        "tag":
            EXPECTED_TAG,
    },

    "authorized_source": {
        "root":
            str(
                SOURCE
            ),

        "files":
            [
                {
                    "filename":
                        item[
                            "filename"
                        ],

                    "rows":
                        item[
                            "rows"
                        ],

                    "canonical_start":
                        item[
                            "start"
                        ],

                    "canonical_end_exclusive":
                        item[
                            "end_exclusive"
                        ],

                    "row_groups":
                        item[
                            "num_row_groups"
                        ],
                }

                for item
                in parquet_meta
            ],

        "total_rows":
            EXPECTED_TOTAL_ROWS,
    },

    "feature_space": {
        "count":
            70,

        "order":
            FULL_FEATURE_ORDER,

        "dtype":
            "float64",

        "transformations":
            "NONE",

        "imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "cohorts":
        results,

    "governance": {
        "model_fits":
            0,

        "shap_values_computed":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "only_authorized_february_source_read":
            True,

        "stage23_model_fit_state":
            "50 / 50 SEALED",

        "additional_model_fits_authorized":
            0,
    },

    "next_action":
        (
            "Execute frozen component-wise TreeSHAP on these "
            "two materialized 5,000-row cohorts using only "
            "the 48 sealed component models."
        ),
}


write_json(
    MANIFEST_PATH,
    manifest_payload,
)


# =============================================================================
# 13. FINAL STATE + CHECKSUMS
# =============================================================================

state[
    "status"
] = (
    "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "model_fits"
] = 0

state[
    "shap_values_computed"
] = 0

state[
    "next_action"
] = (
    "FROZEN_TREESHAP_EXECUTION"
)


write_json(
    STATE_PATH,
    state,
)


CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


artifact_files = sorted(
    p
    for p in OUT.iterdir()
    if (
        p.is_file()
        and p.name
        != "checksums.sha256"
    )
)


CHECKSUMS.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.name}"
        for p in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23-5A SHAP COHORT MATERIALIZATION COMPLETE — UNSEALED")
print("=" * 100)
print()

for result in results:

    print(result["split"])
    print("  rows              :", result["rows"])
    print("  benign            :", result["benign"])
    print("  attack            :", result["attack"])
    print("  features          :", result["features"])
    print("  dtype             :", result["dtype"])
    print("  parquet files read:", result["parquet_files_read"])
    print("  row groups read   :", result["parquet_row_groups_read"])
    print("  artifact SHA256   :", result["output_sha256"])
    print()

print("Model fits                 : 0")
print("SHAP values computed       : 0")
print("Stage23 model state        : 50 / 50 SEALED")
print("Raw Mar1 accessed          : NO")
print("Raw Mar2 accessed          : NO")
print()
print("Materialization manifest:")
print(" ", MANIFEST_PATH)
print(" SHA256:")
print(" ", sha256_file(MANIFEST_PATH))
print()
print("Checksum manifest SHA256:")
print(" ", sha256_file(CHECKSUMS))
print()
print("NEXT:")
print("  Execute frozen TreeSHAP proxy-absorption analysis")
print("  using ONLY these materialized cohorts and sealed models.")
print("  NO RETRAINING.")
print("=" * 100)

STAGE23-5A — FROZEN SHAP COHORT MATERIALIZATION

[OK] HEAD      : 1e687d964aa5247a3e5d0678223dc8de92cc5ade
[OK] seal tag  : stage23-4-uncertainty-analysis-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] new fits  : 0 authorized
[OK] SHAP done : 0

shap_cohort_random_natural.csv
  expected: fc8df6fad72566a6649ec04db5c63630de8eb3e2f67a92be3f3466a8b5574e55
  actual:   fc8df6fad72566a6649ec04db5c63630de8eb3e2f67a92be3f3466a8b5574e55
  [EXACT]

shap_cohort_chronological_natural.csv
  expected: 510d2657ffebd9e1c89ef738767aadf72208161dfd575a4a95a1966e84316773
  actual:   510d2657ffebd9e1c89ef738767aadf72208161dfd575a4a95a1966e84316773
  [EXACT]

[EXACT] frozen feature order: 70 / 70

AUTHORIZED FEBRUARY SOURCE VERIFICATION

[EXACT] day_00_02-14-2018.parquet         rows=   822,947 row_groups=12
[EXACT] day_01_02-15-2018.parquet         rows= 1,046,154 row_groups=14
[EXACT] day_02_02-16-2018.parquet         rows=   900,988 row_groups=12
[EXACT] day_03_02-20-2018.parquet

In [20]:
# =============================================================================
# STAGE23-5B — FROZEN TREESHAP PROXY-ABSORPTION ANALYSIS
#
# ZERO MODEL FITS
# NO RAW DATA / NO PARQUET
#
# Frozen design:
#   - 2 splits
#   - 12 subsets / split
#   - 2 component models / subset
#   - 48 component TreeSHAP explanations total
#   - exact frozen 5,000-row balanced cohort reused across subsets within split
#
# Component reporting:
#   LightGBM and XGBoost separately.
#
# Descriptive consensus:
#   normalize mean-|SHAP| within each component, then
#   consensus_share = 0.5 * LGB_share + 0.5 * XGB_share
#
# IMPORTANT:
#   Consensus is descriptive normalized importance.
#   It is NOT exact SHAP for the probability-averaged ensemble.
#
# Proxy-absorption measures:
#   - top 10 / top 20
#   - Jaccard@10 / Jaccard@20 vs FULL
#   - Spearman rank on common retained features vs FULL
#   - new top-10 / top-20 entrants
#   - features leaving top-10 / top-20
#   - rank displacement
#   - normalized importance-share change
#   - positive importance gainers after removal, ranked
#
# Tie-breaking for top-k only:
#   frozen canonical 70-feature order.
#
# Output is CHECKPOINTED / RESUMABLE.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
import shap

from scipy.stats import spearmanr


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "1e687d964aa5247a3e5d0678223dc8de92cc5ade"
)

EXPECTED_PARENT_TAG = (
    "stage23-4-uncertainty-analysis-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

SHAP_SPEC_PATH = (
    PROTOCOL
    / "shap_spec.json"
)

PRIMARY_SPEC_PATH = (
    PROTOCOL
    / "feature_subset_spec.json"
)

PLACEBO_SPEC_PATH = (
    PROTOCOL
    / "placebo_ablation_spec.json"
)

MATERIALIZED = Path(
    "/kaggle/working/stage23_5a_shap_materialization"
)

RANDOM_MATERIALIZED = (
    MATERIALIZED
    / "random_natural_shap_cohort_full70.npz"
)

CHRONO_MATERIALIZED = (
    MATERIALIZED
    / "chronological_natural_shap_cohort_full70.npz"
)

MATERIALIZATION_MANIFEST = (
    MATERIALIZED
    / "stage23_5a_materialization_manifest.json"
)

MATERIALIZATION_CHECKSUMS = (
    MATERIALIZED
    / "checksums.sha256"
)

EXPECTED_RANDOM_MATERIALIZED_SHA = (
    "5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85"
)

EXPECTED_CHRONO_MATERIALIZED_SHA = (
    "2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4"
)

EXPECTED_MATERIALIZATION_MANIFEST_SHA = (
    "ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4"
)

EXPECTED_MATERIALIZATION_CHECKSUMS_SHA = (
    "03e8013746e650183082c2504976b7f407c13a8c1f2dbf6576e6c33f4ae39f3d"
)

FULL_ROOTS = {
    "RANDOM_NATURAL":
        REPO
        / "results"
        / "stage22r_training"
        / "stage22r_2a_random_natural",

    "CHRONOLOGICAL_NATURAL":
        REPO
        / "results"
        / "stage22r_training"
        / "stage22r_2c_chronological_natural",
}

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)

PLACEBO_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

OUT = Path(
    "/kaggle/working/stage23_5b_treeshap_proxy_absorption"
)

COMPONENT_ROOT = (
    OUT
    / "component_checkpoints"
)

STATE_PATH = (
    OUT
    / "execution_state.json"
)


EXPECTED_SUBSETS = [
    "FULL",

    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",

    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]


DIRECTORY_TO_SUBSET = {
    "no_dst_port":
        "NO_DST_PORT",

    "no_ports":
        "NO_PORTS",

    "no_init_fwd_win_byts":
        "NO_INIT_FWD_WIN_BYTS",

    "no_fwd_seg_size_min":
        "NO_FWD_SEG_SIZE_MIN",

    "no_suspicious_group":
        "NO_SUSPICIOUS_GROUP",

    "behavior_only":
        "BEHAVIOR_ONLY",

    "placebo_counts":
        "PLACEBO_COUNTS",

    "placebo_volume_direction":
        "PLACEBO_VOLUME_DIRECTION",

    "placebo_iat":
        "PLACEBO_IAT",

    "placebo_packet_size":
        "PLACEBO_PACKET_SIZE",

    "placebo_activity":
        "PLACEBO_ACTIVITY",
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def one(
    directory,
    pattern,
):

    matches = sorted(
        directory.glob(
            pattern
        )
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one {pattern} under\n"
            f"{directory}\n"
            f"found={len(matches)}"
        )

    return matches[0]


def safe_slug(
    value,
):

    return (
        value
        .lower()
        .replace(
            " ",
            "_",
        )
        .replace(
            "/",
            "_",
        )
    )


# =============================================================================
# 2. REPOSITORY GOVERNANCE
# =============================================================================

print("=" * 110)
print("STAGE23-5B — FROZEN TREESHAP PROXY-ABSORPTION ANALYSIS")
print("=" * 110)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-4 parent tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before SHAP analysis:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] parent tag:", EXPECTED_PARENT_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23 model fits : 50 / 50 SEALED")
print("[OK] additional fits    : 0 authorized")


# =============================================================================
# 3. VERIFY MATERIALIZATION CHECKPOINT
# =============================================================================

print()
print("=" * 110)
print("VERIFY STAGE23-5A MATERIALIZED COHORTS")
print("=" * 110)
print()


materialization_checks = [
    (
        RANDOM_MATERIALIZED,
        EXPECTED_RANDOM_MATERIALIZED_SHA,
    ),

    (
        CHRONO_MATERIALIZED,
        EXPECTED_CHRONO_MATERIALIZED_SHA,
    ),

    (
        MATERIALIZATION_MANIFEST,
        EXPECTED_MATERIALIZATION_MANIFEST_SHA,
    ),

    (
        MATERIALIZATION_CHECKSUMS,
        EXPECTED_MATERIALIZATION_CHECKSUMS_SHA,
    ),
]


for path, expected in materialization_checks:

    if not path.is_file():

        raise RuntimeError(
            f"Required Stage23-5A artifact missing:\n{path}"
        )

    actual = sha256_file(
        path
    )

    print(path.name)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:

        raise RuntimeError(
            f"Stage23-5A SHA mismatch: {path.name}"
        )

    print("  [EXACT]")


materialization_manifest = json.loads(
    MATERIALIZATION_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if (
    materialization_manifest[
        "status"
    ]
    != "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5A materialization status is not complete."
    )


if (
    materialization_manifest[
        "governance"
    ][
        "model_fits"
    ]
    != 0
):

    raise RuntimeError(
        "Unexpected model fitting recorded in Stage23-5A."
    )


if (
    materialization_manifest[
        "governance"
    ][
        "shap_values_computed"
    ]
    != 0
):

    raise RuntimeError(
        "Stage23-5A unexpectedly reports SHAP computation."
    )


print()
print("[EXACT] Stage23-5A materialization verified")


# =============================================================================
# 4. VERIFY FROZEN SHAP SPEC
# =============================================================================

shap_spec = json.loads(
    SHAP_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    shap_spec[
        "cohort"
    ][
        "rows_per_split"
    ]
    == 5000
)

assert (
    shap_spec[
        "cohort"
    ][
        "benign_per_split"
    ]
    == 2500
)

assert (
    shap_spec[
        "cohort"
    ][
        "attack_per_split"
    ]
    == 2500
)

assert (
    shap_spec[
        "cohort"
    ][
        "same_rows_reused_for_every_subset_within_split"
    ]
    is True
)

assert (
    shap_spec[
        "retraining"
    ]
    == "NONE; use frozen models from Stage23 primary/placebo runs"
)


print("[EXACT] frozen SHAP protocol")


# =============================================================================
# 5. SEMANTIC FEATURE ORDERS
# =============================================================================

primary_spec = json.loads(
    PRIMARY_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

placebo_spec = json.loads(
    PLACEBO_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


FULL_FEATURE_ORDER = list(
    primary_spec[
        "full_feature_order"
    ]
)


if len(
    FULL_FEATURE_ORDER
) != 70:

    raise RuntimeError(
        "Frozen full feature count is not 70."
    )


CANONICAL_POSITION = {
    feature:
        i
    for i, feature
    in enumerate(
        FULL_FEATURE_ORDER
    )
}


SEMANTIC_ORDER = {}


for name in [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]:

    SEMANTIC_ORDER[
        name
    ] = list(
        primary_spec[
            "subsets"
        ][
            name
        ][
            "features"
        ]
    )


for name in [
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]:

    SEMANTIC_ORDER[
        name
    ] = list(
        placebo_spec[
            "subsets"
        ][
            name
        ][
            "retained"
        ]
    )


if set(
    SEMANTIC_ORDER
) != set(
    EXPECTED_SUBSETS
):

    raise RuntimeError(
        "Semantic subset inventory mismatch."
    )


# Map semantic subset columns back into the frozen full70 matrix.

SUBSET_COLUMN_INDEX = {}


for subset, features in SEMANTIC_ORDER.items():

    indices = []

    for feature in features:

        if feature not in CANONICAL_POSITION:

            raise RuntimeError(
                f"{subset}: feature outside frozen full space: {feature}"
            )

        indices.append(
            CANONICAL_POSITION[
                feature
            ]
        )


    # Retained order must remain canonical-order preserving.
    if indices != sorted(
        indices
    ):

        raise RuntimeError(
            f"{subset}: retained order is not canonical-order preserving."
        )


    SUBSET_COLUMN_INDEX[
        subset
    ] = np.asarray(
        indices,
        dtype=np.int64,
    )


print(
    "[EXACT] semantic orders + full70 column mappings verified for 12 subsets"
)


# =============================================================================
# 6. LOAD BOTH MATERIALIZED COHORTS
# =============================================================================

COHORTS = {}


for split, path in [
    (
        "RANDOM_NATURAL",
        RANDOM_MATERIALIZED,
    ),
    (
        "CHRONOLOGICAL_NATURAL",
        CHRONO_MATERIALIZED,
    ),
]:

    with np.load(
        path,
        allow_pickle=False,
    ) as z:

        keys = set(
            z.files
        )

        required = {
            "clean_position",
            "binary_label",
            "feature_names",
            "X_full70",
        }

        if keys != required:

            raise RuntimeError(
                f"{split}: unexpected Stage23-5A NPZ keys."
            )


        clean_position = np.asarray(
            z[
                "clean_position"
            ],
            dtype=np.int64,
        )

        binary_label = np.asarray(
            z[
                "binary_label"
            ],
            dtype=np.uint8,
        )

        feature_names = (
            z[
                "feature_names"
            ]
            .astype(str)
            .tolist()
        )

        X_full70 = np.asarray(
            z[
                "X_full70"
            ],
            dtype=np.float64,
        )


    if feature_names != FULL_FEATURE_ORDER:

        raise RuntimeError(
            f"{split}: materialized semantic feature order mismatch."
        )


    if X_full70.shape != (
        5000,
        70,
    ):

        raise RuntimeError(
            f"{split}: materialized shape mismatch."
        )


    if (
        int(
            np.sum(
                binary_label
                == 0
            )
        )
        != 2500
        or int(
            np.sum(
                binary_label
                == 1
            )
        )
        != 2500
    ):

        raise RuntimeError(
            f"{split}: frozen balanced class count mismatch."
        )


    COHORTS[
        split
    ] = {
        "clean_position":
            clean_position,

        "binary_label":
            binary_label,

        "X_full70":
            X_full70,

        "artifact":
            str(
                path
            ),

        "artifact_sha256":
            sha256_file(
                path
            ),
    }


    print(
        f"[EXACT] {split:<23} "
        "rows=5000 features=70 benign=2500 attack=2500"
    )


# =============================================================================
# 7. BUILD EXACT 24 MODEL-PAIR INVENTORY
# =============================================================================

inventory = []


for split, directory in FULL_ROOTS.items():

    inventory.append(
        {
            "split":
                split,

            "family":
                "FULL",

            "subset":
                "FULL",

            "lightgbm":
                one(
                    directory,
                    "*lightgbm_model.txt",
                ),

            "xgboost":
                one(
                    directory,
                    "*xgboost_model.json",
                ),
        }
    )


for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PRIMARY_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 6:

        raise RuntimeError(
            f"{root}: expected six primary directories."
        )


    for directory in leaves:

        subset = DIRECTORY_TO_SUBSET[
            directory.name
        ]

        inventory.append(
            {
                "split":
                    split,

                "family":
                    "PRIMARY",

                "subset":
                    subset,

                "lightgbm":
                    one(
                        directory,
                        "*lightgbm_model.txt",
                    ),

                "xgboost":
                    one(
                        directory,
                        "*xgboost_model.json",
                    ),
            }
        )


for split_dir, split in [
    (
        "random",
        "RANDOM_NATURAL",
    ),
    (
        "chronological",
        "CHRONOLOGICAL_NATURAL",
    ),
]:

    root = (
        PLACEBO_ROOT
        / split_dir
    )

    leaves = sorted(
        p
        for p in root.iterdir()
        if p.is_dir()
    )

    if len(leaves) != 5:

        raise RuntimeError(
            f"{root}: expected five placebo directories."
        )


    for directory in leaves:

        subset = DIRECTORY_TO_SUBSET[
            directory.name
        ]

        inventory.append(
            {
                "split":
                    split,

                "family":
                    "PLACEBO",

                "subset":
                    subset,

                "lightgbm":
                    one(
                        directory,
                        "*lightgbm_model.txt",
                    ),

                "xgboost":
                    one(
                        directory,
                        "*xgboost_model.json",
                    ),
            }
        )


if len(
    inventory
) != 24:

    raise RuntimeError(
        f"Expected 24 model pairs; found {len(inventory)}"
    )


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    names = {
        item[
            "subset"
        ]
        for item in inventory
        if item[
            "split"
        ] == split
    }

    if names != set(
        EXPECTED_SUBSETS
    ):

        raise RuntimeError(
            f"{split}: subset inventory mismatch."
        )


print()
print("[EXACT] frozen model pairs : 24 / 24")
print("[EXACT] component models   : 48 / 48")


# =============================================================================
# 8. INITIALIZE / VERIFY RESUMABLE OUTPUT
# =============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

COMPONENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


if STATE_PATH.exists():

    state = json.loads(
        STATE_PATH.read_text(
            encoding="utf-8"
        )
    )


    if state.get(
        "execution_parent_commit"
    ) != EXPECTED_HEAD:

        raise RuntimeError(
            "Existing Stage23-5B state belongs to a different parent commit."
        )


    if state.get(
        "random_materialized_sha256"
    ) != EXPECTED_RANDOM_MATERIALIZED_SHA:

        raise RuntimeError(
            "Existing Stage23-5B state has different RANDOM cohort SHA."
        )


    if state.get(
        "chronological_materialized_sha256"
    ) != EXPECTED_CHRONO_MATERIALIZED_SHA:

        raise RuntimeError(
            "Existing Stage23-5B state has different CHRONO cohort SHA."
        )


    print()
    print("[RESUME] Existing Stage23-5B state verified.")


else:

    state = {
        "stage":
            "Stage23-5B frozen TreeSHAP proxy-absorption",

        "status":
            "INITIALIZED",

        "created_utc":
            utc_now(),

        "execution_parent_commit":
            EXPECTED_HEAD,

        "execution_parent_tag":
            EXPECTED_PARENT_TAG,

        "random_materialized_sha256":
            EXPECTED_RANDOM_MATERIALIZED_SHA,

        "chronological_materialized_sha256":
            EXPECTED_CHRONO_MATERIALIZED_SHA,

        "component_explanations_expected":
            48,

        "component_explanations_completed":
            0,

        "model_fits":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "parquet_files_read":
            0,

        "retraining":
            False,
    }


    write_json(
        STATE_PATH,
        state,
    )


# =============================================================================
# 9. SHAP OUTPUT NORMALIZATION
# =============================================================================

def normalize_binary_shap(
    values,
    n_rows,
    n_features,
):

    extraction_mode = None


    if isinstance(
        values,
        list,
    ):

        # Binary classifier positive-class output if library returns
        # class-specific lists.
        if len(
            values
        ) == 2:

            arr = np.asarray(
                values[
                    1
                ],
                dtype=np.float64,
            )

            extraction_mode = (
                "BINARY_LIST_POSITIVE_CLASS_INDEX_1"
            )

        elif len(
            values
        ) == 1:

            arr = np.asarray(
                values[
                    0
                ],
                dtype=np.float64,
            )

            extraction_mode = (
                "SINGLE_LIST_OUTPUT"
            )

        else:

            raise RuntimeError(
                f"Unexpected SHAP list output count: {len(values)}"
            )


    else:

        arr = np.asarray(
            values,
            dtype=np.float64,
        )


        if arr.ndim == 2:

            extraction_mode = (
                "SINGLE_BINARY_OUTPUT_MATRIX"
            )


        elif (
            arr.ndim == 3
            and arr.shape[
                0
            ] == n_rows
            and arr.shape[
                1
            ] == n_features
            and arr.shape[
                2
            ] == 2
        ):

            arr = arr[
                :,
                :,
                1,
            ]

            extraction_mode = (
                "BINARY_OUTPUT_AXIS_POSITIVE_CLASS_INDEX_1"
            )


        else:

            raise RuntimeError(
                f"Unexpected SHAP ndarray shape: {arr.shape}"
            )


    if arr.shape != (
        n_rows,
        n_features,
    ):

        raise RuntimeError(
            "Normalized SHAP matrix shape mismatch.\n"
            f"expected={(n_rows, n_features)}\n"
            f"actual={arr.shape}"
        )


    if not np.all(
        np.isfinite(
            arr
        )
    ):

        raise RuntimeError(
            "Non-finite SHAP values produced."
        )


    return (
        arr,
        extraction_mode,
    )


# =============================================================================
# 10. DETERMINISTIC IMPORTANCE / RANK HELPERS
# =============================================================================

def ordered_features(
    feature_names,
    importance,
):

    pairs = list(
        zip(
            feature_names,
            importance,
        )
    )


    # Importance descending.
    # Exact ties resolved only by prospectively frozen canonical feature order.

    pairs.sort(
        key=lambda pair: (
            -float(
                pair[
                    1
                ]
            ),
            CANONICAL_POSITION[
                pair[
                    0
                ]
            ],
        )
    )


    return [
        feature
        for feature, _
        in pairs
    ]


def rank_map(
    ordered,
):

    return {
        feature:
            rank
        for rank, feature
        in enumerate(
            ordered,
            start=1,
        )
    }


def jaccard(
    a,
    b,
):

    a = set(
        a
    )

    b = set(
        b
    )

    union = (
        a
        | b
    )


    if not union:

        return 1.0


    return float(
        len(
            a
            & b
        )
        / len(
            union
        )
    )


# =============================================================================
# 11. CHECKPOINT PATH
# =============================================================================

def component_checkpoint_path(
    split,
    subset,
    component,
):

    d = (
        COMPONENT_ROOT
        / safe_slug(
            split
        )
        / safe_slug(
            subset
        )
    )

    d.mkdir(
        parents=True,
        exist_ok=True,
    )


    return (
        d
        / (
            safe_slug(
                component
            )
            + "_importance.json"
        )
    )


# =============================================================================
# 12. COMPUTE / RESUME ALL 48 COMPONENT EXPLANATIONS
# =============================================================================

print()
print("=" * 110)
print("COMPONENT-WISE TREESHAP")
print("=" * 110)
print()


component_records = []


tasks = []


for item in inventory:

    for component in [
        "LightGBM",
        "XGBoost",
    ]:

        tasks.append(
            (
                item,
                component,
            )
        )


tasks.sort(
    key=lambda x: (
        x[
            0
        ][
            "split"
        ],
        x[
            0
        ][
            "family"
        ],
        x[
            0
        ][
            "subset"
        ],
        x[
            1
        ],
    )
)


if len(
    tasks
) != 48:

    raise RuntimeError(
        "Internal task count != 48."
    )


for task_number, (
    item,
    component,
) in enumerate(
    tasks,
    start=1,
):

    split = item[
        "split"
    ]

    subset = item[
        "subset"
    ]

    family = item[
        "family"
    ]

    feature_names = (
        SEMANTIC_ORDER[
            subset
        ]
    )

    column_indices = (
        SUBSET_COLUMN_INDEX[
            subset
        ]
    )

    X = (
        COHORTS[
            split
        ][
            "X_full70"
        ][
            :,
            column_indices,
        ]
    )


    if X.shape != (
        5000,
        len(
            feature_names
        ),
    ):

        raise RuntimeError(
            f"{subset} × {split}: subset matrix shape mismatch."
        )


    model_path = (
        item[
            "lightgbm"
        ]
        if component
        == "LightGBM"
        else item[
            "xgboost"
        ]
    )

    model_sha = sha256_file(
        model_path
    )


    checkpoint = (
        component_checkpoint_path(
            split,
            subset,
            component,
        )
    )


    # -------------------------------------------------------------------------
    # RESUME PATH
    # -------------------------------------------------------------------------

    if checkpoint.exists():

        record = json.loads(
            checkpoint.read_text(
                encoding="utf-8"
            )
        )


        if record[
            "status"
        ] != "COMPONENT_TREESHAP_COMPLETE":

            raise RuntimeError(
                f"Incomplete existing checkpoint:\n{checkpoint}"
            )


        if record[
            "model_sha256"
        ] != model_sha:

            raise RuntimeError(
                f"Existing checkpoint model SHA mismatch:\n{checkpoint}"
            )


        if record[
            "cohort_sha256"
        ] != COHORTS[
            split
        ][
            "artifact_sha256"
        ]:

            raise RuntimeError(
                f"Existing checkpoint cohort SHA mismatch:\n{checkpoint}"
            )


        if record[
            "feature_order"
        ] != feature_names:

            raise RuntimeError(
                f"Existing checkpoint feature order mismatch:\n{checkpoint}"
            )


        if len(
            record[
                "mean_abs_shap"
            ]
        ) != len(
            feature_names
        ):

            raise RuntimeError(
                f"Existing checkpoint importance length mismatch:\n{checkpoint}"
            )


        component_records.append(
            record
        )


        print(
            f"[{task_number:02d}/48] "
            f"[RESUME] "
            f"{split:<23} "
            f"{subset:<27} "
            f"{component:<8}"
        )


        continue


    # -------------------------------------------------------------------------
    # FRESH COMPONENT TREESHAP
    # -------------------------------------------------------------------------

    t0 = time.perf_counter()


    if component == "LightGBM":

        model = lgb.Booster(
            model_file=str(
                model_path
            )
        )


        if int(
            model.num_feature()
        ) != len(
            feature_names
        ):

            raise RuntimeError(
                f"{subset} × {split}: LightGBM feature-count mismatch."
            )


    else:

        model = xgb.Booster()

        model.load_model(
            str(
                model_path
            )
        )


        if int(
            model.num_features()
        ) != len(
            feature_names
        ):

            raise RuntimeError(
                f"{subset} × {split}: XGBoost feature-count mismatch."
            )


    explainer = shap.TreeExplainer(
        model
    )


    raw_values = explainer.shap_values(
        X,
        check_additivity=False,
    )


    shap_matrix, extraction_mode = (
        normalize_binary_shap(
            raw_values,
            n_rows=5000,
            n_features=len(
                feature_names
            ),
        )
    )


    mean_abs = np.mean(
        np.abs(
            shap_matrix
        ),
        axis=0,
        dtype=np.float64,
    )


    if not np.all(
        np.isfinite(
            mean_abs
        )
    ):

        raise RuntimeError(
            f"{subset} × {split} × {component}: "
            "non-finite mean absolute SHAP."
        )


    importance_sum = float(
        np.sum(
            mean_abs,
            dtype=np.float64,
        )
    )


    if not (
        importance_sum > 0
    ):

        raise RuntimeError(
            f"{subset} × {split} × {component}: "
            "total mean-|SHAP| is not positive."
        )


    normalized_share = (
        mean_abs
        / importance_sum
    )


    ordered = ordered_features(
        feature_names,
        mean_abs,
    )

    ranks = rank_map(
        ordered
    )


    elapsed = float(
        time.perf_counter()
        - t0
    )


    record = {
        "stage":
            "Stage23-5B",

        "status":
            "COMPONENT_TREESHAP_COMPLETE",

        "completed_utc":
            utc_now(),

        "split":
            split,

        "family":
            family,

        "subset":
            subset,

        "component":
            component,

        "cohort_rows":
            5000,

        "cohort_sha256":
            COHORTS[
                split
            ][
                "artifact_sha256"
            ],

        "model_path":
            str(
                model_path.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,

        "feature_count":
            len(
                feature_names
            ),

        "feature_order":
            feature_names,

        "shap_method":
            "shap.TreeExplainer",

        "shap_output_extraction":
            extraction_mode,

        "mean_abs_shap":
            {
                feature:
                    float(
                        value
                    )
                for feature, value
                in zip(
                    feature_names,
                    mean_abs,
                )
            },

        "normalized_importance_share":
            {
                feature:
                    float(
                        value
                    )
                for feature, value
                in zip(
                    feature_names,
                    normalized_share,
                )
            },

        "ordered_features":
            ordered,

        "rank":
            ranks,

        "top_10_features":
            ordered[
                :10
            ],

        "top_20_features":
            ordered[
                :20
            ],

        "total_mean_abs_shap":
            importance_sum,

        "seconds":
            elapsed,

        "governance": {
            "model_fit":
                False,

            "retraining":
                False,

            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "parquet_files_read":
                0,
        },
    }


    write_json(
        checkpoint,
        record,
    )


    component_records.append(
        record
    )


    state[
        "component_explanations_completed"
    ] = len(
        list(
            COMPONENT_ROOT.rglob(
                "*_importance.json"
            )
        )
    )

    state[
        "status"
    ] = (
        "COMPONENT_TREESHAP_IN_PROGRESS"
    )

    state[
        "last_completed"
    ] = {
        "split":
            split,

        "subset":
            subset,

        "component":
            component,
    }


    write_json(
        STATE_PATH,
        state,
    )


    print(
        f"[{task_number:02d}/48] "
        f"[COMPLETE] "
        f"{split:<23} "
        f"{subset:<27} "
        f"{component:<8} "
        f"{elapsed:.2f}s"
    )


    # Row-level SHAP values are intentionally not retained.
    del (
        raw_values,
        shap_matrix,
        mean_abs,
        normalized_share,
        explainer,
        model,
        X,
    )

    gc.collect()


# =============================================================================
# 13. VERIFY COMPLETE COMPONENT SET
# =============================================================================

if len(
    component_records
) != 48:

    raise RuntimeError(
        f"Expected 48 component records; got {len(component_records)}"
    )


record_key = {
    (
        r[
            "split"
        ],
        r[
            "subset"
        ],
        r[
            "component"
        ],
    ):
        r

    for r
    in component_records
}


if len(
    record_key
) != 48:

    raise RuntimeError(
        "Duplicate component record key detected."
    )


print()
print("[EXACT] component TreeSHAP : 48 / 48 complete")


# =============================================================================
# 14. BUILD COMPONENT IMPORTANCE TABLE
# =============================================================================

component_rows = []


for record in component_records:

    for feature in record[
        "feature_order"
    ]:

        component_rows.append(
            {
                "split":
                    record[
                        "split"
                    ],

                "family":
                    record[
                        "family"
                    ],

                "subset":
                    record[
                        "subset"
                    ],

                "component":
                    record[
                        "component"
                    ],

                "feature":
                    feature,

                "mean_abs_shap":
                    record[
                        "mean_abs_shap"
                    ][
                        feature
                    ],

                "normalized_importance_share":
                    record[
                        "normalized_importance_share"
                    ][
                        feature
                    ],

                "rank":
                    record[
                        "rank"
                    ][
                        feature
                    ],
            }
        )


COMPONENT_IMPORTANCE_CSV = (
    OUT
    / "stage23_5b_component_importance.csv"
)


pd.DataFrame(
    component_rows
).to_csv(
    COMPONENT_IMPORTANCE_CSV,
    index=False,
)


# =============================================================================
# 15. DESCRIPTIVE CONSENSUS
# =============================================================================

consensus_records = {}


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    for subset in EXPECTED_SUBSETS:

        lgb_record = record_key[
            (
                split,
                subset,
                "LightGBM",
            )
        ]

        xgb_record = record_key[
            (
                split,
                subset,
                "XGBoost",
            )
        ]


        features = SEMANTIC_ORDER[
            subset
        ]


        consensus_share = {}


        for feature in features:

            consensus_share[
                feature
            ] = float(
                0.5
                * lgb_record[
                    "normalized_importance_share"
                ][
                    feature
                ]
                + 0.5
                * xgb_record[
                    "normalized_importance_share"
                ][
                    feature
                ]
            )


        share_sum = float(
            sum(
                consensus_share.values()
            )
        )


        if not np.isclose(
            share_sum,
            1.0,
            atol=1e-12,
            rtol=1e-10,
        ):

            raise RuntimeError(
                f"{split} × {subset}: consensus shares do not sum to one."
            )


        ordered = sorted(
            features,
            key=lambda feature: (
                -consensus_share[
                    feature
                ],
                CANONICAL_POSITION[
                    feature
                ],
            ),
        )


        consensus_records[
            (
                split,
                subset,
            )
        ] = {
            "split":
                split,

            "subset":
                subset,

            "feature_order":
                features,

            "normalized_importance_share":
                consensus_share,

            "ordered_features":
                ordered,

            "rank":
                rank_map(
                    ordered
                ),

            "top_10_features":
                ordered[
                    :10
                ],

            "top_20_features":
                ordered[
                    :20
                ],

            "description":
                (
                    "0.5/0.5 average of component-wise normalized "
                    "mean absolute SHAP shares; descriptive only, "
                    "not exact SHAP for the probability-averaged ensemble."
                ),
        }


CONSENSUS_IMPORTANCE_CSV = (
    OUT
    / "stage23_5b_descriptive_consensus_importance.csv"
)


consensus_rows = []


for (
    split,
    subset,
), record in consensus_records.items():

    family = (
        "FULL"
        if subset
        == "FULL"
        else (
            "PLACEBO"
            if subset.startswith(
                "PLACEBO_"
            )
            else "PRIMARY"
        )
    )


    for feature in record[
        "feature_order"
    ]:

        consensus_rows.append(
            {
                "split":
                    split,

                "family":
                    family,

                "subset":
                    subset,

                "feature":
                    feature,

                "consensus_normalized_importance_share":
                    record[
                        "normalized_importance_share"
                    ][
                        feature
                    ],

                "consensus_rank":
                    record[
                        "rank"
                    ][
                        feature
                    ],
            }
        )


pd.DataFrame(
    consensus_rows
).to_csv(
    CONSENSUS_IMPORTANCE_CSV,
    index=False,
)


# =============================================================================
# 16. PROXY-ABSORPTION COMPARISON FUNCTION
# =============================================================================

def compare_to_full(
    full_record,
    subset_record,
):

    full_features = set(
        full_record[
            "feature_order"
        ]
    )

    subset_features = set(
        subset_record[
            "feature_order"
        ]
    )


    common = [
        feature
        for feature
        in FULL_FEATURE_ORDER
        if (
            feature
            in full_features
            and feature
            in subset_features
        )
    ]


    if not common:

        raise RuntimeError(
            "No common retained features for proxy-absorption comparison."
        )


    full_rank = full_record[
        "rank"
    ]

    subset_rank = subset_record[
        "rank"
    ]


    full_rank_vector = np.asarray(
        [
            full_rank[
                feature
            ]
            for feature
            in common
        ],
        dtype=np.float64,
    )

    subset_rank_vector = np.asarray(
        [
            subset_rank[
                feature
            ]
            for feature
            in common
        ],
        dtype=np.float64,
    )


    spearman_result = spearmanr(
        full_rank_vector,
        subset_rank_vector,
    )


    spearman_rho = float(
        spearman_result.statistic
    )


    if not np.isfinite(
        spearman_rho
    ):

        raise RuntimeError(
            "Non-finite Spearman rank correlation."
        )


    full_top10 = full_record[
        "top_10_features"
    ]

    subset_top10 = subset_record[
        "top_10_features"
    ]

    full_top20 = full_record[
        "top_20_features"
    ]

    subset_top20 = subset_record[
        "top_20_features"
    ]


    rank_displacement = {
        feature:
            int(
                subset_rank[
                    feature
                ]
                - full_rank[
                    feature
                ]
            )

        for feature
        in common
    }


    normalized_share_change = {
        feature:
            float(
                subset_record[
                    "normalized_importance_share"
                ][
                    feature
                ]
                - full_record[
                    "normalized_importance_share"
                ][
                    feature
                ]
            )

        for feature
        in common
    }


    # No arbitrary K was frozen for "top importance gainers".
    # Therefore retain ALL positive gainers, deterministically sorted.

    positive_gainers = [
        feature
        for feature
        in common
        if normalized_share_change[
            feature
        ] > 0
    ]


    positive_gainers.sort(
        key=lambda feature: (
            -normalized_share_change[
                feature
            ],
            CANONICAL_POSITION[
                feature
            ],
        )
    )


    return {
        "common_retained_feature_count":
            len(
                common
            ),

        "top_10_features":
            subset_top10,

        "top_20_features":
            subset_top20,

        "full_top_10_features":
            full_top10,

        "full_top_20_features":
            full_top20,

        "Jaccard_at_10_vs_FULL":
            jaccard(
                subset_top10,
                full_top10,
            ),

        "Jaccard_at_20_vs_FULL":
            jaccard(
                subset_top20,
                full_top20,
            ),

        "Spearman_rank_on_common_retained_features":
            spearman_rho,

        "new_top_10_entrants":
            [
                feature
                for feature
                in subset_top10
                if feature
                not in set(
                    full_top10
                )
            ],

        "new_top_20_entrants":
            [
                feature
                for feature
                in subset_top20
                if feature
                not in set(
                    full_top20
                )
            ],

        "features_leaving_top_10":
            [
                feature
                for feature
                in full_top10
                if feature
                not in set(
                    subset_top10
                )
            ],

        "features_leaving_top_20":
            [
                feature
                for feature
                in full_top20
                if feature
                not in set(
                    subset_top20
                )
            ],

        "rank_displacement":
            rank_displacement,

        "normalized_importance_share_change":
            normalized_share_change,

        "top_importance_gainers_after_removal":
            [
                {
                    "feature":
                        feature,

                    "normalized_importance_share_change":
                        normalized_share_change[
                            feature
                        ],

                    "full_rank":
                        int(
                            full_rank[
                                feature
                            ]
                        ),

                    "subset_rank":
                        int(
                            subset_rank[
                                feature
                            ]
                        ),

                    "rank_displacement":
                        int(
                            rank_displacement[
                                feature
                            ]
                        ),
                }

                for feature
                in positive_gainers
            ],
    }


# =============================================================================
# 17. COMPONENT-WISE + CONSENSUS PROXY-ABSORPTION
# =============================================================================

comparisons = []

comparison_flat_rows = []


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    for subset in EXPECTED_SUBSETS:

        if subset == "FULL":
            continue


        family = (
            "PLACEBO"
            if subset.startswith(
                "PLACEBO_"
            )
            else "PRIMARY"
        )


        comparison = {
            "split":
                split,

            "family":
                family,

            "subset":
                subset,

            "components":
                {},

            "descriptive_consensus":
                None,
        }


        for component in [
            "LightGBM",
            "XGBoost",
        ]:

            full = record_key[
                (
                    split,
                    "FULL",
                    component,
                )
            ]

            reduced = record_key[
                (
                    split,
                    subset,
                    component,
                )
            ]


            result = compare_to_full(
                full,
                reduced,
            )


            comparison[
                "components"
            ][
                component
            ] = result


            comparison_flat_rows.append(
                {
                    "split":
                        split,

                    "family":
                        family,

                    "subset":
                        subset,

                    "reporting":
                        component,

                    "jaccard_at_10_vs_full":
                        result[
                            "Jaccard_at_10_vs_FULL"
                        ],

                    "jaccard_at_20_vs_full":
                        result[
                            "Jaccard_at_20_vs_FULL"
                        ],

                    "spearman_common_retained":
                        result[
                            "Spearman_rank_on_common_retained_features"
                        ],

                    "new_top10_entrants":
                        "|".join(
                            result[
                                "new_top_10_entrants"
                            ]
                        ),

                    "new_top20_entrants":
                        "|".join(
                            result[
                                "new_top_20_entrants"
                            ]
                        ),

                    "features_leaving_top10":
                        "|".join(
                            result[
                                "features_leaving_top_10"
                            ]
                        ),

                    "features_leaving_top20":
                        "|".join(
                            result[
                                "features_leaving_top_20"
                            ]
                        ),

                    "number_positive_share_gainers":
                        len(
                            result[
                                "top_importance_gainers_after_removal"
                            ]
                        ),
                }
            )


        full_consensus = (
            consensus_records[
                (
                    split,
                    "FULL",
                )
            ]
        )

        reduced_consensus = (
            consensus_records[
                (
                    split,
                    subset,
                )
            ]
        )


        consensus_result = compare_to_full(
            full_consensus,
            reduced_consensus,
        )


        comparison[
            "descriptive_consensus"
        ] = consensus_result


        comparison_flat_rows.append(
            {
                "split":
                    split,

                "family":
                    family,

                "subset":
                    subset,

                "reporting":
                    "DESCRIPTIVE_CONSENSUS_0.5_0.5",

                "jaccard_at_10_vs_full":
                    consensus_result[
                        "Jaccard_at_10_vs_FULL"
                    ],

                "jaccard_at_20_vs_full":
                    consensus_result[
                        "Jaccard_at_20_vs_FULL"
                    ],

                "spearman_common_retained":
                    consensus_result[
                        "Spearman_rank_on_common_retained_features"
                    ],

                "new_top10_entrants":
                    "|".join(
                        consensus_result[
                            "new_top_10_entrants"
                        ]
                    ),

                "new_top20_entrants":
                    "|".join(
                        consensus_result[
                            "new_top_20_entrants"
                        ]
                    ),

                "features_leaving_top10":
                    "|".join(
                        consensus_result[
                            "features_leaving_top_10"
                        ]
                    ),

                "features_leaving_top20":
                    "|".join(
                        consensus_result[
                            "features_leaving_top_20"
                        ]
                    ),

                "number_positive_share_gainers":
                    len(
                        consensus_result[
                            "top_importance_gainers_after_removal"
                        ]
                    ),
            }
        )


        comparisons.append(
            comparison
        )


if len(
    comparisons
) != 22:

    raise RuntimeError(
        f"Expected 22 subset-vs-FULL split comparisons; got {len(comparisons)}"
    )


# =============================================================================
# 18. WRITE FINAL SCIENTIFIC ARTIFACTS
# =============================================================================

PROXY_JSON = (
    OUT
    / "stage23_5b_proxy_absorption_results.json"
)

PROXY_CSV = (
    OUT
    / "stage23_5b_proxy_absorption_summary.csv"
)

CONSENSUS_JSON = (
    OUT
    / "stage23_5b_descriptive_consensus.json"
)

SUMMARY_JSON = (
    OUT
    / "stage23_5b_treeshap_summary.json"
)


write_json(
    PROXY_JSON,
    {
        "stage":
            "Stage23-5B",

        "status":
            "FROZEN_PROXY_ABSORPTION_ANALYSIS_COMPLETE_UNSEALED",

        "method":
            (
                "TreeSHAP separately for LightGBM and XGBoost; "
                "mean absolute SHAP over the frozen 5,000-row cohort."
            ),

        "comparisons":
            comparisons,
    },
)


pd.DataFrame(
    comparison_flat_rows
).to_csv(
    PROXY_CSV,
    index=False,
)


write_json(
    CONSENSUS_JSON,
    {
        "stage":
            "Stage23-5B",

        "status":
            "DESCRIPTIVE_CONSENSUS_COMPLETE",

        "definition":
            (
                "For each component, mean_abs_SHAP(feature) divided by "
                "sum(mean_abs_SHAP across retained features); "
                "then 0.5/0.5 average of LightGBM and XGBoost normalized shares."
            ),

        "warning":
            (
                "Descriptive normalized importance only; "
                "must not be called exact SHAP for the probability-averaged ensemble."
            ),

        "records":
            {
                f"{split}::{subset}":
                    record

                for (
                    split,
                    subset,
                ), record
                in consensus_records.items()
            },
    },
)


summary_payload = {
    "stage":
        "Stage23-5B frozen TreeSHAP proxy-absorption",

    "status":
        "TREESHAP_PROXY_ABSORPTION_COMPLETE_UNSEALED",

    "completed_utc":
        utc_now(),

    "execution_parent": {
        "commit":
            EXPECTED_HEAD,

        "tag":
            EXPECTED_PARENT_TAG,
    },

    "frozen_design": {
        "splits":
            2,

        "subsets_per_split":
            12,

        "model_pairs":
            24,

        "component_models":
            48,

        "cohort_rows_per_split":
            5000,

        "benign_per_split":
            2500,

        "attack_per_split":
            2500,

        "same_rows_reused_within_split":
            True,

        "components": [
            "LightGBM",
            "XGBoost",
        ],

        "component_importance":
            "mean absolute TreeSHAP",

        "descriptive_consensus_weights":
            {
                "LightGBM":
                    0.5,

                "XGBoost":
                    0.5,
            },

        "tie_break_for_top_k":
            "frozen canonical 70-feature order",

        "top_importance_gainers_handling":
            (
                "All positive normalized-share gainers retained and "
                "sorted descending because the frozen protocol did not "
                "specify a truncation K for this measure."
            ),
    },

    "materialized_cohorts": {
        "RANDOM_NATURAL":
            EXPECTED_RANDOM_MATERIALIZED_SHA,

        "CHRONOLOGICAL_NATURAL":
            EXPECTED_CHRONO_MATERIALIZED_SHA,
    },

    "accounting": {
        "component_treeshap_expected":
            48,

        "component_treeshap_completed":
            48,

        "proxy_absorption_split_comparisons":
            22,

        "model_fits":
            0,

        "stage23_model_fit_state":
            "50 / 50 SEALED",
    },

    "governance": {
        "retraining":
            False,

        "new_model_fits":
            0,

        "new_subset":
            False,

        "shap_cohort_changed":
            False,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "parquet_files_read":
            0,
    },

    "interpretation_boundary": (
        "SHAP proxy-shift patterns are descriptive evidence consistent "
        "with proxy absorption after feature removal. "
        "SHAP does not establish causation."
    ),
}


write_json(
    SUMMARY_JSON,
    summary_payload,
)


# =============================================================================
# 19. FINAL STATE
# =============================================================================

state[
    "status"
] = (
    "TREESHAP_PROXY_ABSORPTION_COMPLETE_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "component_explanations_completed"
] = 48

state[
    "proxy_absorption_split_comparisons"
] = 22

state[
    "model_fits"
] = 0

state[
    "next_action"
] = (
    "ZERO_FIT_SEAL_STAGE23_5A_AND_5B_SHAP_BLOCK"
)


write_json(
    STATE_PATH,
    state,
)


# =============================================================================
# 20. CHECKSUMS
# =============================================================================

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


artifact_files = sorted(
    p
    for p in OUT.rglob(
        "*"
    )
    if (
        p.is_file()
        and p != CHECKSUMS
    )
)


CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(p)}  "
            f"{p.relative_to(OUT)}"
        )

        for p
        in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 21. CONCISE RESULTS DISPLAY
# =============================================================================

print()
print("=" * 110)
print("DESCRIPTIVE CONSENSUS — PROXY-ABSORPTION SUMMARY")
print("=" * 110)
print()


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print(split)
    print("-" * 110)


    for comparison in comparisons:

        if comparison[
            "split"
        ] != split:

            continue


        c = comparison[
            "descriptive_consensus"
        ]


        gainers = c[
            "top_importance_gainers_after_removal"
        ]


        first_gainer = (
            gainers[
                0
            ][
                "feature"
            ]
            if gainers
            else "NONE"
        )


        print(
            f"{comparison['subset']:<29} "
            f"J10={c['Jaccard_at_10_vs_FULL']:.4f}  "
            f"J20={c['Jaccard_at_20_vs_FULL']:.4f}  "
            f"rho={c['Spearman_rank_on_common_retained_features']:+.4f}  "
            f"top-gainer={first_gainer}"
        )


# =============================================================================
# 22. FINAL
# =============================================================================

print()
print("=" * 110)
print("STAGE23-5B TREESHAP PROXY-ABSORPTION COMPLETE — UNSEALED")
print("=" * 110)
print()

print("Component TreeSHAP        : 48 / 48")
print("Model pairs               : 24 / 24")
print("Splits                    : 2")
print("Subsets / split           : 12")
print("Subset-vs-FULL comparisons: 22")
print()
print("Cohort rows / split       : 5,000")
print("Benign / split            : 2,500")
print("Attack / split            : 2,500")
print()
print("Model fits                : 0")
print("Stage23 model state       : 50 / 50 SEALED")
print("Parquet files read        : 0")
print("Raw Mar1 accessed         : NO")
print("Raw Mar2 accessed         : NO")
print()
print("Component importance CSV:")
print(" ", COMPONENT_IMPORTANCE_CSV)
print(" SHA256:")
print(" ", sha256_file(COMPONENT_IMPORTANCE_CSV))
print()
print("Consensus importance CSV:")
print(" ", CONSENSUS_IMPORTANCE_CSV)
print(" SHA256:")
print(" ", sha256_file(CONSENSUS_IMPORTANCE_CSV))
print()
print("Proxy-absorption JSON:")
print(" ", PROXY_JSON)
print(" SHA256:")
print(" ", sha256_file(PROXY_JSON))
print()
print("Proxy-absorption CSV:")
print(" ", PROXY_CSV)
print(" SHA256:")
print(" ", sha256_file(PROXY_CSV))
print()
print("TreeSHAP summary:")
print(" ", SUMMARY_JSON)
print(" SHA256:")
print(" ", sha256_file(SUMMARY_JSON))
print()
print("Checksums SHA256:")
print(" ", sha256_file(CHECKSUMS))
print()
print("NEXT:")
print("  ZERO-FIT seal + push Stage23-5A materialization")
print("  and Stage23-5B frozen SHAP proxy-absorption results.")
print("=" * 110)

STAGE23-5B — FROZEN TREESHAP PROXY-ABSORPTION ANALYSIS

[OK] HEAD      : 1e687d964aa5247a3e5d0678223dc8de92cc5ade
[OK] parent tag: stage23-4-uncertainty-analysis-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23 model fits : 50 / 50 SEALED
[OK] additional fits    : 0 authorized

VERIFY STAGE23-5A MATERIALIZED COHORTS

random_natural_shap_cohort_full70.npz
  expected: 5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  actual:   5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  [EXACT]
chronological_natural_shap_cohort_full70.npz
  expected: 2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  actual:   2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  [EXACT]
stage23_5a_materialization_manifest.json
  expected: ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4
  actual:   ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4
  [EXACT]
checksums.sha256
  expected: 03e8013746e650183082c2504976b7f407c13

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[01/48] [COMPLETE] CHRONOLOGICAL_NATURAL   FULL                        LightGBM 34.98s
[02/48] [COMPLETE] CHRONOLOGICAL_NATURAL   FULL                        XGBoost  6.27s
[03/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_ACTIVITY            LightGBM 34.35s
[04/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_ACTIVITY            XGBoost  6.14s
[05/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_COUNTS              LightGBM 33.36s
[06/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_COUNTS              XGBoost  6.20s
[07/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_IAT                 LightGBM 33.68s
[08/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_IAT                 XGBoost  6.31s
[09/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_PACKET_SIZE         LightGBM 34.30s
[10/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_PACKET_SIZE         XGBoost  6.19s
[11/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACEBO_VOLUME_DIRECTION    LightGBM 33.88s
[12/48] [COMPLETE] CHRONOLOGICAL_NATURAL   PLACE

In [21]:
# =============================================================================
# QUEUED NEXT CELL
# STAGE23-5 — ZERO-FIT SHAP BLOCK SEAL + GITHUB PUSH
#
# Run/queue this directly AFTER the currently-running Stage23-5B cell.
#
# It will:
#   1. Verify Stage23-5A materialization
#   2. Verify Stage23-5B completed all 48 component TreeSHAP explanations
#   3. Verify all Stage23-5B checksums
#   4. Copy 5A + 5B into repository
#   5. Add seal metadata
#   6. Commit
#   7. Annotated tag
#   8. Atomic push main + tag
#   9. Remote verification
#
# FAIL CLOSED:
#   If Stage23-5B is incomplete or failed, this cell stops BEFORE repository
#   modification.
#
# ZERO MODEL FITS
# ZERO SHAP RECOMPUTATION
# ZERO PARQUET READS
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import os


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE_5A = Path(
    "/kaggle/working/stage23_5a_shap_materialization"
)

SOURCE_5B = Path(
    "/kaggle/working/stage23_5b_treeshap_proxy_absorption"
)

DEST_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
)

DEST_5A = (
    DEST_ROOT
    / "stage23_5a_shap_materialization"
)

DEST_5B = (
    DEST_ROOT
    / "stage23_5b_treeshap_proxy_absorption"
)

EXPECTED_PARENT = (
    "1e687d964aa5247a3e5d0678223dc8de92cc5ade"
)

EXPECTED_PARENT_TAG = (
    "stage23-4-uncertainty-analysis-complete-v1"
)

# Known exact Stage23-5A hashes from successful materialization.
EXPECTED_5A_RANDOM_SHA = (
    "5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85"
)

EXPECTED_5A_CHRONO_SHA = (
    "2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4"
)

EXPECTED_5A_MANIFEST_SHA = (
    "ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4"
)

EXPECTED_5A_CHECKSUMS_SHA = (
    "03e8013746e650183082c2504976b7f407c13a8c1f2dbf6576e6c33f4ae39f3d"
)

SEAL_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: seal frozen SHAP proxy-absorption analysis"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def git(
    *args,
    check=True,
    show=False,
):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (
        p.stdout or ""
    ).strip()

    if show and out:
        print(out)

    if check and p.returncode != 0:

        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n{out}"
        )

    return (
        p.returncode,
        out,
    )


def verify_checksum_manifest(
    root,
    manifest_path,
):

    count = 0

    for line in Path(
        manifest_path
    ).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, relative = line.split(
            maxsplit=1
        )

        relative = relative.strip()

        path = (
            root
            / relative
        )

        if not path.is_file():

            raise RuntimeError(
                "Checksum manifest references missing file:\n"
                f"{path}"
            )

        actual = sha256_file(
            path
        )

        if actual != digest:

            raise RuntimeError(
                "Checksum mismatch:\n"
                f"{relative}\n"
                f"expected={digest}\n"
                f"actual={actual}"
            )

        count += 1

    return count


def copy_tree_exact(
    source,
    dest,
):

    if dest.exists():

        raise RuntimeError(
            f"Repository destination already exists:\n{dest}"
        )

    shutil.copytree(
        source,
        dest,
        copy_function=shutil.copy2,
    )


# =============================================================================
# 2. START
# =============================================================================

print("=" * 110)
print("QUEUED STAGE23-5 — ZERO-FIT SHAP SEAL + PUSH")
print("=" * 110)
print()


# =============================================================================
# 3. REPOSITORY PARENT MUST STILL BE STAGE23-4
# =============================================================================

if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, parent_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main branch; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected HEAD before Stage23-5 seal.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )


if parent_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-4 parent tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-5 seal:\n"
        + status
    )


print("[OK] HEAD       :", head)
print("[OK] parent tag :", EXPECTED_PARENT_TAG)
print("[OK] worktree   : CLEAN")


# =============================================================================
# 4. SEAL TAG MUST NOT ALREADY EXIST
# =============================================================================

_, local_tag = git(
    "tag",
    "--list",
    SEAL_TAG,
)


if local_tag:

    raise RuntimeError(
        f"Stage23-5 seal tag already exists locally:\n{SEAL_TAG}"
    )


_, remote_existing_tag = git(
    "ls-remote",
    "--tags",
    "origin",
    f"refs/tags/{SEAL_TAG}",
)


if remote_existing_tag:

    raise RuntimeError(
        f"Stage23-5 seal tag already exists remotely:\n{SEAL_TAG}"
    )


# =============================================================================
# 5. GITHUB AUTH BEFORE REPOSITORY MODIFICATION
# =============================================================================

print()
print("=" * 110)
print("GITHUB PUSH PREFLIGHT")
print("=" * 110)
print()


rc, dryrun = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
)


if rc != 0:

    print(dryrun)

    raise RuntimeError(
        "GitHub push authentication unavailable.\n"
        "No repository artifacts have been modified."
    )


print("[OK] GitHub push authentication available.")


# =============================================================================
# 6. VERIFY STAGE23-5A
# =============================================================================

print()
print("=" * 110)
print("VERIFY STAGE23-5A MATERIALIZATION")
print("=" * 110)
print()


if not SOURCE_5A.is_dir():

    raise RuntimeError(
        f"Stage23-5A source missing:\n{SOURCE_5A}"
    )


A_RANDOM = (
    SOURCE_5A
    / "random_natural_shap_cohort_full70.npz"
)

A_CHRONO = (
    SOURCE_5A
    / "chronological_natural_shap_cohort_full70.npz"
)

A_MANIFEST = (
    SOURCE_5A
    / "stage23_5a_materialization_manifest.json"
)

A_STATE = (
    SOURCE_5A
    / "execution_state.json"
)

A_CHECKSUMS = (
    SOURCE_5A
    / "checksums.sha256"
)


for p in [
    A_RANDOM,
    A_CHRONO,
    A_MANIFEST,
    A_STATE,
    A_CHECKSUMS,
]:

    if not p.is_file():

        raise RuntimeError(
            f"Stage23-5A required artifact missing:\n{p}"
        )


checks_5a = [
    (
        A_RANDOM,
        EXPECTED_5A_RANDOM_SHA,
    ),
    (
        A_CHRONO,
        EXPECTED_5A_CHRONO_SHA,
    ),
    (
        A_MANIFEST,
        EXPECTED_5A_MANIFEST_SHA,
    ),
    (
        A_CHECKSUMS,
        EXPECTED_5A_CHECKSUMS_SHA,
    ),
]


for p, expected in checks_5a:

    actual = sha256_file(
        p
    )

    print(p.name)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:

        raise RuntimeError(
            f"Stage23-5A SHA mismatch: {p.name}"
        )

    print("  [EXACT]")


manifest_5a = json.loads(
    A_MANIFEST.read_text(
        encoding="utf-8"
    )
)

state_5a = json.loads(
    A_STATE.read_text(
        encoding="utf-8"
    )
)


if (
    manifest_5a["status"]
    != "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5A manifest does not report successful completion."
    )


if (
    state_5a["status"]
    != "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5A execution state does not report successful completion."
    )


if manifest_5a[
    "governance"
][
    "model_fits"
] != 0:

    raise RuntimeError(
        "Stage23-5A unexpectedly reports model fitting."
    )


if manifest_5a[
    "governance"
][
    "shap_values_computed"
] != 0:

    raise RuntimeError(
        "Stage23-5A unexpectedly reports SHAP evaluation."
    )


a_manifest_count = (
    verify_checksum_manifest(
        SOURCE_5A,
        A_CHECKSUMS,
    )
)


print()
print("[EXACT] Stage23-5A manifest entries:", a_manifest_count)
print("[EXACT] Stage23-5A model fits      : 0")
print("[EXACT] Stage23-5A SHAP evaluations: 0")


# =============================================================================
# 7. VERIFY STAGE23-5B COMPLETION
#
# IF SHAP CELL FAILED, QUEUED CELL STOPS HERE.
# Repository still untouched.
# =============================================================================

print()
print("=" * 110)
print("VERIFY STAGE23-5B TREESHAP")
print("=" * 110)
print()


if not SOURCE_5B.is_dir():

    raise RuntimeError(
        "Stage23-5B output directory does not exist.\n"
        "The SHAP execution likely did not complete."
    )


B_STATE = (
    SOURCE_5B
    / "execution_state.json"
)

B_COMPONENT = (
    SOURCE_5B
    / "stage23_5b_component_importance.csv"
)

B_CONSENSUS = (
    SOURCE_5B
    / "stage23_5b_descriptive_consensus_importance.csv"
)

B_CONSENSUS_JSON = (
    SOURCE_5B
    / "stage23_5b_descriptive_consensus.json"
)

B_PROXY_JSON = (
    SOURCE_5B
    / "stage23_5b_proxy_absorption_results.json"
)

B_PROXY_CSV = (
    SOURCE_5B
    / "stage23_5b_proxy_absorption_summary.csv"
)

B_SUMMARY = (
    SOURCE_5B
    / "stage23_5b_treeshap_summary.json"
)

B_CHECKSUMS = (
    SOURCE_5B
    / "checksums.sha256"
)


for p in [
    B_STATE,
    B_COMPONENT,
    B_CONSENSUS,
    B_CONSENSUS_JSON,
    B_PROXY_JSON,
    B_PROXY_CSV,
    B_SUMMARY,
    B_CHECKSUMS,
]:

    if not p.is_file():

        raise RuntimeError(
            "Stage23-5B is NOT seal-ready.\n"
            f"Missing required artifact:\n{p}\n\n"
            "No repository changes have been made."
        )


state_5b = json.loads(
    B_STATE.read_text(
        encoding="utf-8"
    )
)

summary_5b = json.loads(
    B_SUMMARY.read_text(
        encoding="utf-8"
    )
)

proxy_5b = json.loads(
    B_PROXY_JSON.read_text(
        encoding="utf-8"
    )
)


if (
    state_5b.get("status")
    != "TREESHAP_PROXY_ABSORPTION_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5B has not reached successful completion.\n"
        f"status={state_5b.get('status')}\n"
        f"completed="
        f"{state_5b.get('component_explanations_completed')}/48\n\n"
        "REFUSING seal/push."
    )


if (
    state_5b.get(
        "component_explanations_completed"
    )
    != 48
):

    raise RuntimeError(
        "Stage23-5B does not contain 48 completed component explanations."
    )


if (
    state_5b.get(
        "proxy_absorption_split_comparisons"
    )
    != 22
):

    raise RuntimeError(
        "Stage23-5B does not contain 22 proxy-absorption comparisons."
    )


if state_5b.get(
    "model_fits"
) != 0:

    raise RuntimeError(
        "Stage23-5B unexpectedly records model fitting."
    )


if (
    summary_5b.get("status")
    != "TREESHAP_PROXY_ABSORPTION_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5B summary status mismatch."
    )


accounting = summary_5b[
    "accounting"
]


if accounting[
    "component_treeshap_expected"
] != 48:

    raise RuntimeError(
        "Stage23-5B expected component count changed."
    )


if accounting[
    "component_treeshap_completed"
] != 48:

    raise RuntimeError(
        "Stage23-5B component execution incomplete."
    )


if accounting[
    "proxy_absorption_split_comparisons"
] != 22:

    raise RuntimeError(
        "Stage23-5B comparison count mismatch."
    )


if accounting[
    "model_fits"
] != 0:

    raise RuntimeError(
        "Stage23-5B summary unexpectedly records fitting."
    )


governance = summary_5b[
    "governance"
]


required_false = [
    "retraining",
    "new_subset",
    "shap_cohort_changed",
    "raw_mar1_accessed",
    "raw_mar2_accessed",
]


for key in required_false:

    if governance[key] is not False:

        raise RuntimeError(
            f"Stage23-5B governance violation: {key}"
        )


if governance[
    "new_model_fits"
] != 0:

    raise RuntimeError(
        "Stage23-5B new_model_fits != 0."
    )


if governance[
    "parquet_files_read"
] != 0:

    raise RuntimeError(
        "Stage23-5B unexpectedly read Parquet files."
    )


if (
    proxy_5b.get("status")
    != "FROZEN_PROXY_ABSORPTION_ANALYSIS_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Stage23-5B proxy result status mismatch."
    )


if len(
    proxy_5b[
        "comparisons"
    ]
) != 22:

    raise RuntimeError(
        "Stage23-5B proxy result comparison count != 22."
    )


# Exact checkpoint inventory.

component_checkpoints = sorted(
    (
        SOURCE_5B
        / "component_checkpoints"
    ).rglob(
        "*_importance.json"
    )
)


if len(
    component_checkpoints
) != 48:

    raise RuntimeError(
        "Stage23-5B component checkpoint count mismatch.\n"
        f"expected=48\n"
        f"actual={len(component_checkpoints)}"
    )


seen_component_keys = set()


for checkpoint in component_checkpoints:

    r = json.loads(
        checkpoint.read_text(
            encoding="utf-8"
        )
    )


    if (
        r.get("status")
        != "COMPONENT_TREESHAP_COMPLETE"
    ):

        raise RuntimeError(
            f"Incomplete TreeSHAP checkpoint:\n{checkpoint}"
        )


    if r[
        "governance"
    ][
        "model_fit"
    ] is not False:

        raise RuntimeError(
            f"Checkpoint unexpectedly records fit:\n{checkpoint}"
        )


    key = (
        r[
            "split"
        ],
        r[
            "subset"
        ],
        r[
            "component"
        ],
    )


    if key in seen_component_keys:

        raise RuntimeError(
            f"Duplicate component checkpoint key: {key}"
        )


    seen_component_keys.add(
        key
    )


if len(
    seen_component_keys
) != 48:

    raise RuntimeError(
        "Unique component checkpoint count != 48."
    )


b_manifest_count = (
    verify_checksum_manifest(
        SOURCE_5B,
        B_CHECKSUMS,
    )
)


print("[EXACT] component TreeSHAP        : 48 / 48")
print("[EXACT] proxy comparisons         : 22 / 22")
print("[EXACT] component checkpoints     : 48 / 48")
print("[EXACT] checksum manifest entries :", b_manifest_count)
print("[EXACT] new model fits            : 0")
print("[EXACT] Parquet reads in 5B       : 0")
print("[EXACT] raw Mar1 / Mar2           : NO / NO")


# =============================================================================
# 8. RECORD FINAL SOURCE HASHES BEFORE COPY
# =============================================================================

source_hashes = {
    "stage23_5a": {
        "random_cohort":
            sha256_file(
                A_RANDOM
            ),

        "chronological_cohort":
            sha256_file(
                A_CHRONO
            ),

        "materialization_manifest":
            sha256_file(
                A_MANIFEST
            ),

        "checksums":
            sha256_file(
                A_CHECKSUMS
            ),
    },

    "stage23_5b": {
        "component_importance_csv":
            sha256_file(
                B_COMPONENT
            ),

        "descriptive_consensus_importance_csv":
            sha256_file(
                B_CONSENSUS
            ),

        "descriptive_consensus_json":
            sha256_file(
                B_CONSENSUS_JSON
            ),

        "proxy_absorption_json":
            sha256_file(
                B_PROXY_JSON
            ),

        "proxy_absorption_csv":
            sha256_file(
                B_PROXY_CSV
            ),

        "treeshap_summary":
            sha256_file(
                B_SUMMARY
            ),

        "checksums":
            sha256_file(
                B_CHECKSUMS
            ),
    },
}


print()
print("=" * 110)
print("FINAL STAGE23-5 SOURCE HASHES")
print("=" * 110)
print()

print("Stage23-5B summary SHA256:")
print(
    " ",
    source_hashes[
        "stage23_5b"
    ][
        "treeshap_summary"
    ],
)

print("Stage23-5B proxy JSON SHA256:")
print(
    " ",
    source_hashes[
        "stage23_5b"
    ][
        "proxy_absorption_json"
    ],
)

print("Stage23-5B checksums SHA256:")
print(
    " ",
    source_hashes[
        "stage23_5b"
    ][
        "checksums"
    ],
)


# =============================================================================
# 9. GITHUB SIZE SAFETY — BEFORE REPOSITORY WRITE
# =============================================================================

print()
print("=" * 110)
print("GITHUB FILE-SIZE SAFETY")
print("=" * 110)
print()


all_source_files = (
    [
        p
        for p in SOURCE_5A.rglob("*")
        if p.is_file()
    ]
    +
    [
        p
        for p in SOURCE_5B.rglob("*")
        if p.is_file()
    ]
)


largest = max(
    all_source_files,
    key=lambda p:
        p.stat().st_size,
)


largest_mb = (
    largest.stat().st_size
    / (1024 ** 2)
)


print(
    "Largest file:",
    largest,
)

print(
    "Size:",
    f"{largest_mb:.2f} MiB",
)


if (
    largest.stat().st_size
    >= 95 * 1024 * 1024
):

    raise RuntimeError(
        "A Stage23-5 artifact is >=95 MiB.\n"
        "REFUSING GitHub commit."
    )


print("[OK] all Stage23-5 files below safety limit.")


# =============================================================================
# 10. ONLY NOW MODIFY REPOSITORY
# =============================================================================

print()
print("=" * 110)
print("COPY STAGE23-5A + STAGE23-5B INTO REPOSITORY")
print("=" * 110)
print()


copy_tree_exact(
    SOURCE_5A,
    DEST_5A,
)

copy_tree_exact(
    SOURCE_5B,
    DEST_5B,
)


print("[OK] Stage23-5A copied.")
print("[OK] Stage23-5B copied.")


# =============================================================================
# 11. SEAL RECEIPT
# =============================================================================

SEAL_RECEIPT = (
    DEST_ROOT
    / "stage23_5_shap_block_seal_receipt.json"
)


write_json(
    SEAL_RECEIPT,
    {
        "stage":
            "Stage23-5 frozen SHAP proxy-absorption",

        "status":
            "SEALED",

        "sealed_utc":
            utc_now(),

        "execution_parent": {
            "commit":
                EXPECTED_PARENT,

            "tag":
                EXPECTED_PARENT_TAG,
        },

        "stage23_5a": {
            "status":
                "BOTH_FROZEN_SHAP_COHORTS_MATERIALIZED",

            "rows_per_split":
                5000,

            "features":
                70,

            "model_fits":
                0,

            "shap_values_computed":
                0,

            "source_sha256":
                source_hashes[
                    "stage23_5a"
                ],
        },

        "stage23_5b": {
            "status":
                "TREESHAP_PROXY_ABSORPTION_COMPLETE",

            "component_treeshap":
                "48 / 48",

            "model_pairs":
                "24 / 24",

            "proxy_absorption_split_comparisons":
                "22 / 22",

            "model_fits":
                0,

            "source_sha256":
                source_hashes[
                    "stage23_5b"
                ],
        },

        "governance": {
            "stage23_model_fit_budget":
                "50 / 50 SEALED",

            "additional_stage23_model_fits_authorized":
                0,

            "model_fits_during_seal":
                0,

            "shap_recomputed_during_seal":
                False,

            "parquet_reads_during_seal":
                0,

            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "retraining":
                False,
        },

        "interpretation_boundary":
            (
                "Component TreeSHAP is reported separately for "
                "LightGBM and XGBoost. The 0.5/0.5 normalized "
                "importance consensus is descriptive only and is "
                "not exact SHAP for the probability-averaged ensemble. "
                "Observed proxy shifts do not establish causality."
            ),

        "tag":
            SEAL_TAG,
    },
)


# =============================================================================
# 12. REPOSITORY CHECKSUM MANIFEST FOR ENTIRE STAGE23-5 BLOCK
# =============================================================================

REPO_CHECKSUMS = (
    DEST_ROOT
    / "stage23_5_shap_block_repository_checksums.sha256"
)


seal_files = []


for root in [
    DEST_5A,
    DEST_5B,
]:

    seal_files.extend(
        p
        for p in root.rglob("*")
        if p.is_file()
    )


seal_files.append(
    SEAL_RECEIPT
)


seal_files = sorted(
    seal_files,
    key=lambda p:
        str(
            p.relative_to(
                REPO
            )
        )
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(p)}  "
            f"{p.relative_to(REPO)}"
        )

        for p
        in seal_files
    ) + "\n",
    encoding="utf-8",
)


repo_checksums_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


print()
print(
    "[OK] Stage23-5 repository manifest SHA256:",
    repo_checksums_sha,
)


# =============================================================================
# 13. COMMIT IDENTITY
# =============================================================================

_, user_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

_, user_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not user_name:

    _, value = git(
        "log",
        "-1",
        "--format=%an",
    )

    git(
        "config",
        "user.name",
        value,
    )


if not user_email:

    _, value = git(
        "log",
        "-1",
        "--format=%ae",
    )

    git(
        "config",
        "user.email",
        value,
    )


# =============================================================================
# 14. STAGE ONLY AUTHORIZED STAGE23-5 PATHS
# =============================================================================

paths_to_stage = [
    str(
        DEST_5A.relative_to(
            REPO
        )
    ),

    str(
        DEST_5B.relative_to(
            REPO
        )
    ),

    str(
        SEAL_RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        REPO_CHECKSUMS.relative_to(
            REPO
        )
    ),
]


git(
    "add",
    "--",
    *paths_to_stage,
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    x
    for x in staged.splitlines()
    if x.strip()
]


if not staged_files:

    raise RuntimeError(
        "No Stage23-5 files staged."
    )


allowed_prefixes = [
    str(
        DEST_5A.relative_to(
            REPO
        )
    ) + "/",

    str(
        DEST_5B.relative_to(
            REPO
        )
    ) + "/",
]


allowed_exact = {
    str(
        SEAL_RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        REPO_CHECKSUMS.relative_to(
            REPO
        )
    ),
}


for path in staged_files:

    if (
        path not in allowed_exact
        and not any(
            path.startswith(
                prefix
            )
            for prefix
            in allowed_prefixes
        )
    ):

        raise RuntimeError(
            f"Unexpected staged file:\n{path}"
        )


print()
print(
    "[OK] staged Stage23-5 files:",
    len(
        staged_files
    ),
)


# =============================================================================
# 15. COMMIT
# =============================================================================

print()
print("=" * 110)
print("COMMIT")
print("=" * 110)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, sealed_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] sealed commit:",
    sealed_commit,
)


# =============================================================================
# 16. ANNOTATED TAG
# =============================================================================

git(
    "tag",
    "-a",
    SEAL_TAG,
    sealed_commit,
    "-m",
    (
        "Stage23-5 frozen SHAP proxy-absorption complete: "
        "two 5,000-row frozen cohorts, 48 component TreeSHAP "
        "explanations, 22 subset-vs-FULL comparisons, zero model fits"
    ),
)


# =============================================================================
# 17. ATOMIC PUSH
# =============================================================================

print()
print("=" * 110)
print("ATOMIC PUSH MAIN + TAG")
print("=" * 110)
print()


git(
    "push",
    "--atomic",
    "origin",
    "main",
    f"refs/tags/{SEAL_TAG}",
    show=True,
)


# =============================================================================
# 18. REMOTE VERIFY
# =============================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEAL_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Remote main could not be resolved."
    )


if not remote_tag_line:

    raise RuntimeError(
        "Remote Stage23-5 annotated tag could not be peeled."
    )


remote_main = (
    remote_main_line.split()[0]
)

remote_tag_commit = (
    remote_tag_line.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_main}"
    )


if remote_tag_commit != sealed_commit:

    raise RuntimeError(
        "Remote tag mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_tag_commit}"
    )


# =============================================================================
# 19. FINAL CLEAN WORKTREE
# =============================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-5 seal:\n"
        + final_status
    )


# =============================================================================
# 20. FINAL
# =============================================================================

print()
print("=" * 110)
print("STAGE23-5 SHAP BLOCK — SEALED AND PUSHED")
print("=" * 110)
print()

print("Commit:")
print(" ", sealed_commit)

print()
print("Tag:")
print(" ", SEAL_TAG)

print()
print("Remote main:")
print(" ", remote_main)

print()
print("Remote tag peeled commit:")
print(" ", remote_tag_commit)

print()
print("Repository manifest SHA256:")
print(" ", repo_checksums_sha)

print()
print("STAGE23-5A")
print("  frozen cohorts       : 2 / 2")
print("  rows / split         : 5,000")
print("  features             : 70")
print("  model fits           : 0")
print("  SHAP evaluations     : 0")

print()
print("STAGE23-5B")
print("  model pairs          : 24 / 24")
print("  component TreeSHAP   : 48 / 48")
print("  proxy comparisons    : 22 / 22")
print("  new model fits       : 0")

print()
print("STAGE23")
print("  model-fit budget     : 50 / 50 SEALED")
print("  additional fits      : 0 authorized")
print("  Raw Mar1 accessed    : NO")
print("  Raw Mar2 accessed    : NO")
print("  Parquet reads seal   : 0")

print()
print("NEXT:")
print("  Continue frozen Stage23 analysis-only work.")
print("  Attack-family analysis remains; NO model fitting.")

print("=" * 110)

QUEUED STAGE23-5 — ZERO-FIT SHAP SEAL + PUSH

[OK] HEAD       : 1e687d964aa5247a3e5d0678223dc8de92cc5ade
[OK] parent tag : stage23-4-uncertainty-analysis-complete-v1
[OK] worktree   : CLEAN

GITHUB PUSH PREFLIGHT

[OK] GitHub push authentication available.

VERIFY STAGE23-5A MATERIALIZATION

random_natural_shap_cohort_full70.npz
  expected: 5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  actual:   5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  [EXACT]
chronological_natural_shap_cohort_full70.npz
  expected: 2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  actual:   2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  [EXACT]
stage23_5a_materialization_manifest.json
  expected: ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4
  actual:   ae7cc8c5bbffa8e8ff54e04fbb2efe698023999400df703d080683d94be685d4
  [EXACT]
checksums.sha256
  expected: 03e8013746e650183082c2504976b7f407c13a8c1f2dbf6576e6c33f4ae39f

In [1]:
# =============================================================================
# STAGE23 — FRESH SESSION RECOVERY FROM SEALED STAGE23-5
#
# ZERO MODEL FITS
# ZERO SHAP COMPUTATION
# ZERO PARQUET READS
# ZERO RAW DATA READS
#
# Restores the repository from the exact Stage23-5 GitHub seal.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import shutil


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

EXPECTED_REPO_MANIFEST_SHA = (
    "3ea1437e65691567f64a201415c8aa4372415fc0273c9ce2b6465475135a04c6"
)

EXPECTED_5A_RANDOM_SHA = (
    "5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85"
)

EXPECTED_5A_CHRONO_SHA = (
    "2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4"
)

EXPECTED_5B_SUMMARY_SHA = (
    "85ea2da694f837c8d303dd2caa73fb778b8ee715840624dc620000d8af87a2d3"
)

EXPECTED_5B_PROXY_SHA = (
    "5d7c3077cce7fb105a3d2e45779b561361f93322df42f77a82faeda1a385e322"
)

EXPECTED_5B_CHECKSUMS_SHA = (
    "fa37b1f7551821ac0b06e729bd606fff5b739df4e236981d602d550dc9130a58"
)


def run(
    *args,
    cwd=None,
):

    p = subprocess.run(
        list(
            map(
                str,
                args,
            )
        ),
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def git(
    *args,
):

    return run(
        "git",
        *args,
        cwd=REPO,
    )


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


print("=" * 100)
print("STAGE23 — FRESH SESSION RECOVERY FROM STAGE23-5 SEAL")
print("=" * 100)
print()


# =============================================================================
# 1. FAIL CLOSED IF AN UNEXPECTED REPOSITORY ALREADY EXISTS
# =============================================================================

if REPO.exists():

    raise RuntimeError(
        "Repository path already exists in this fresh session.\n"
        "Do NOT delete it blindly:\n"
        f"{REPO}"
    )


# =============================================================================
# 2. CLONE
# =============================================================================

print("Cloning sealed repository...")

run(
    "git",
    "clone",
    REPO_URL,
    str(
        REPO
    ),
)

git(
    "fetch",
    "--tags",
    "--force",
)


# =============================================================================
# 3. VERIFY EXACT REMOTE-SEAL COMMIT
# =============================================================================

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Fresh clone HEAD mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 tag does not peel to expected commit.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={tag_commit}"
    )


if status:

    raise RuntimeError(
        "Fresh clone is unexpectedly dirty:\n"
        + status
    )


print("[EXACT] HEAD:")
print(" ", head)

print("[EXACT] tag:")
print(" ", EXPECTED_TAG)

print("[EXACT] worktree: CLEAN")


# =============================================================================
# 4. VERIFY SEALED STAGE23-5 ARTIFACTS
# =============================================================================

ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
)

REPO_MANIFEST = (
    ROOT
    / "stage23_5_shap_block_repository_checksums.sha256"
)

SEAL_RECEIPT = (
    ROOT
    / "stage23_5_shap_block_seal_receipt.json"
)

RANDOM_5A = (
    ROOT
    / "stage23_5a_shap_materialization"
    / "random_natural_shap_cohort_full70.npz"
)

CHRONO_5A = (
    ROOT
    / "stage23_5a_shap_materialization"
    / "chronological_natural_shap_cohort_full70.npz"
)

SUMMARY_5B = (
    ROOT
    / "stage23_5b_treeshap_proxy_absorption"
    / "stage23_5b_treeshap_summary.json"
)

PROXY_5B = (
    ROOT
    / "stage23_5b_treeshap_proxy_absorption"
    / "stage23_5b_proxy_absorption_results.json"
)

CHECKSUMS_5B = (
    ROOT
    / "stage23_5b_treeshap_proxy_absorption"
    / "checksums.sha256"
)


required = [
    REPO_MANIFEST,
    SEAL_RECEIPT,
    RANDOM_5A,
    CHRONO_5A,
    SUMMARY_5B,
    PROXY_5B,
    CHECKSUMS_5B,
]


for p in required:

    if not p.is_file():

        raise RuntimeError(
            f"Required sealed artifact missing:\n{p}"
        )


checks = [
    (
        "Stage23-5 repository manifest",
        REPO_MANIFEST,
        EXPECTED_REPO_MANIFEST_SHA,
    ),
    (
        "Stage23-5A RANDOM cohort",
        RANDOM_5A,
        EXPECTED_5A_RANDOM_SHA,
    ),
    (
        "Stage23-5A CHRONO cohort",
        CHRONO_5A,
        EXPECTED_5A_CHRONO_SHA,
    ),
    (
        "Stage23-5B summary",
        SUMMARY_5B,
        EXPECTED_5B_SUMMARY_SHA,
    ),
    (
        "Stage23-5B proxy results",
        PROXY_5B,
        EXPECTED_5B_PROXY_SHA,
    ),
    (
        "Stage23-5B checksums",
        CHECKSUMS_5B,
        EXPECTED_5B_CHECKSUMS_SHA,
    ),
]


print()
print("=" * 100)
print("VERIFY SEALED STAGE23-5 HASHES")
print("=" * 100)
print()


for label, path, expected in checks:

    actual = sha256_file(
        path
    )

    print(label)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:

        raise RuntimeError(
            f"SHA mismatch: {label}"
        )

    print("  [EXACT]")


# =============================================================================
# 5. VERIFY COMPLETE REPOSITORY MANIFEST
# =============================================================================

verified = 0


for line in REPO_MANIFEST.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    expected_sha, relative_path = line.split(
        maxsplit=1
    )

    path = (
        REPO
        / relative_path.strip()
    )

    if not path.is_file():

        raise RuntimeError(
            "Stage23-5 repository manifest references missing artifact:\n"
            f"{relative_path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "Stage23-5 repository manifest verification failed:\n"
            f"{relative_path}"
        )

    verified += 1


print()
print(
    "[EXACT] Stage23-5 repository manifest entries:",
    verified,
)


# =============================================================================
# 6. VERIFY SEAL RECEIPT / SCIENTIFIC STATE
# =============================================================================

receipt = json.loads(
    SEAL_RECEIPT.read_text(
        encoding="utf-8"
    )
)


if receipt[
    "status"
] != "SEALED":

    raise RuntimeError(
        "Stage23-5 receipt is not SEALED."
    )


gov = receipt[
    "governance"
]


if gov[
    "stage23_model_fit_budget"
] != "50 / 50 SEALED":

    raise RuntimeError(
        "Stage23 fit-budget receipt mismatch."
    )


if gov[
    "additional_stage23_model_fits_authorized"
] != 0:

    raise RuntimeError(
        "Unexpected additional Stage23 fits authorized."
    )


if gov[
    "raw_mar1_accessed"
] is not False:

    raise RuntimeError(
        "Unexpected Mar1 access recorded."
    )


if gov[
    "raw_mar2_accessed"
] is not False:

    raise RuntimeError(
        "Unexpected Mar2 access recorded."
    )


if receipt[
    "stage23_5b"
][
    "component_treeshap"
] != "48 / 48":

    raise RuntimeError(
        "Stage23-5B component count mismatch."
    )


if receipt[
    "stage23_5b"
][
    "proxy_absorption_split_comparisons"
] != "22 / 22":

    raise RuntimeError(
        "Stage23-5B comparison count mismatch."
    )


# =============================================================================
# 7. FINAL
# =============================================================================

print()
print("=" * 100)
print("STAGE23 FRESH RECOVERY COMPLETE")
print("=" * 100)
print()

print("GitHub seal commit:")
print(" ", EXPECTED_HEAD)

print()
print("GitHub seal tag:")
print(" ", EXPECTED_TAG)

print()
print("Stage23 model fits      : 50 / 50 SEALED")
print("Additional fits         : 0 authorized")
print()
print("Stage23-5A cohorts      : 2 / 2 RECOVERED")
print("Stage23-5B TreeSHAP     : 48 / 48 RECOVERED")
print("Proxy comparisons       : 22 / 22 RECOVERED")
print()
print("Model fits this cell    : 0")
print("SHAP computations       : 0")
print("Parquet reads           : 0")
print("Raw Mar1 accessed       : NO")
print("Raw Mar2 accessed       : NO")
print()
print("[READY]")
print("Next frozen Stage23 block: ATTACK-FAMILY ANALYSIS.")
print("NO RETRAINING.")
print("=" * 100)

STAGE23 — FRESH SESSION RECOVERY FROM STAGE23-5 SEAL

Cloning sealed repository...
[EXACT] HEAD:
  b927499efdf7d4dd5013054cc796f1115f879f8b
[EXACT] tag:
  stage23-5-shap-proxy-absorption-complete-v1
[EXACT] worktree: CLEAN

VERIFY SEALED STAGE23-5 HASHES

Stage23-5 repository manifest
  expected: 3ea1437e65691567f64a201415c8aa4372415fc0273c9ce2b6465475135a04c6
  actual:   3ea1437e65691567f64a201415c8aa4372415fc0273c9ce2b6465475135a04c6
  [EXACT]
Stage23-5A RANDOM cohort
  expected: 5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  actual:   5cd9ab402e4d1773c8f5e92fe927834dc1601661060857c82283c8d4bb223b85
  [EXACT]
Stage23-5A CHRONO cohort
  expected: 2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  actual:   2e6ebccfcc6b5321de41f96326a0645e56fedcc4cea792df746c04edc1e389a4
  [EXACT]
Stage23-5B summary
  expected: 85ea2da694f837c8d303dd2caa73fb778b8ee715840624dc620000d8af87a2d3
  actual:   85ea2da694f837c8d303dd2caa73fb778b8ee715840624dc620000d8af87a2d3

In [2]:
# =============================================================================
# STAGE23-6 — ATTACK-FAMILY ANALYSIS PREFLIGHT
#
# ZERO MODEL FITS
# ZERO PREDICTIONS
# ZERO SCIENTIFIC WRITES
# ZERO PARQUET ROW READS
# ZERO RAW MAR1 / MAR2 ACCESS
#
# Purpose:
#   1. Verify Stage23-5 sealed parent.
#   2. Verify frozen attack_family_spec.json.
#   3. Inspect ONLY metadata/schema of authorized Feb14-Feb28 Parquets.
#   4. Determine whether day_id / original_row_index / categorical attack label
#      already exist in the sealed development cache.
#   5. Search TRACKED repository text for existing attack-category alignment
#      infrastructure.
#
# No attack-family metric is calculated here.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import json
import os

import pyarrow.parquet as pq


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
)

ATTACK_SPEC = (
    PROTOCOL
    / "attack_family_spec.json"
)

PROTOCOL_CHECKSUMS = (
    PROTOCOL
    / "checksums.sha256"
)

SOURCE = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

DAY_FILES = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]

EXPECTED_ROWS = [
    822_947,
    1_046_154,
    900_988,
    7_926_258,
    1_031_018,
    1_045_297,
    1_045_961,
    593_780,
]

SEARCH_TERMS = [
    "original_row_index",
    "day_id",
    "attack_family",
    "attack_category",
    "attack family",
    "attack category",
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def git(*args, check=True):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    out = (p.stdout or "").strip()

    if check and p.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(map(str,args))} failed:\n{out}"
        )

    return p.returncode, out


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def parse_manifest(path):

    out = {}

    for line in Path(path).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        out[rel.strip()] = digest.strip()

    return out


def candidate_columns(columns):

    hits = []

    keywords = [
        "label",
        "class",
        "attack",
        "family",
        "category",
        "day",
        "row",
        "index",
        "position",
        "binary",
    ]

    for col in columns:

        lower = col.lower()

        if any(
            keyword in lower
            for keyword in keywords
        ):
            hits.append(col)

    return hits


# =============================================================================
# 2. SEALED REPOSITORY STATE
# =============================================================================

print("=" * 110)
print("STAGE23-6 — ATTACK-FAMILY ANALYSIS PREFLIGHT")
print("=" * 110)
print()


_, head = git(
    "rev-parse",
    "HEAD",
)

_, tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

_, status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 seal tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-6 preflight:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23 model fits : 50 / 50 SEALED")
print("[OK] additional fits    : 0 authorized")


# =============================================================================
# 3. VERIFY FROZEN ATTACK-FAMILY SPEC
# =============================================================================

manifest = parse_manifest(
    PROTOCOL_CHECKSUMS
)


if ATTACK_SPEC.name not in manifest:

    raise RuntimeError(
        "attack_family_spec.json absent from frozen protocol manifest."
    )


expected_attack_spec_sha = (
    manifest[
        ATTACK_SPEC.name
    ]
)

actual_attack_spec_sha = (
    sha256_file(
        ATTACK_SPEC
    )
)


print()
print("attack_family_spec.json")
print("  expected:", expected_attack_spec_sha)
print("  actual:  ", actual_attack_spec_sha)


if (
    actual_attack_spec_sha
    != expected_attack_spec_sha
):

    raise RuntimeError(
        "Frozen attack-family spec SHA mismatch."
    )


print("  [EXACT]")


spec = json.loads(
    ATTACK_SPEC.read_text(
        encoding="utf-8"
    )
)


required_metrics = [
    "support_attack",
    "support_benign",
    "PR_AUC",
    "ROC_AUC",
    "precision_at_0_50",
    "recall_at_0_50",
    "f1_at_0_50",
    "fpr_at_0_50",
    "fnr_at_0_50",
    "delta_f1_FULL_minus_ablated",
    "delta_recall_FULL_minus_ablated",
]


if spec["metrics"] != required_metrics:

    raise RuntimeError(
        "Frozen attack-family metric list differs from expected."
    )


if (
    spec[
        "interpretation_minimum_attack_support"
    ]
    != 100
):

    raise RuntimeError(
        "Frozen minimum attack support is not 100."
    )


if spec[
    "no_retraining"
] is not True:

    raise RuntimeError(
        "Frozen attack-family spec does not forbid retraining."
    )


print()
print("=" * 110)
print("FROZEN ATTACK-FAMILY SPEC")
print("=" * 110)
print()

print("Evaluation:")
print(" ", spec["evaluation_definition"])

print()
print("Alignment:")
print(" ", spec["source_alignment"])

print()
print(
    "Minimum attack support for interpretation:",
    spec["interpretation_minimum_attack_support"],
)

print(
    "Low-support policy:",
    spec["low_support_policy"],
)

print()
print("Metrics:")
for metric in spec["metrics"]:
    print(" ", metric)


# =============================================================================
# 4. AUTHORIZED FEBRUARY SOURCE — DIRECTORY LISTING ONLY
# =============================================================================

print()
print("=" * 110)
print("AUTHORIZED FEBRUARY SOURCE")
print("=" * 110)
print()


if not SOURCE.is_dir():

    raise RuntimeError(
        f"Authorized Stage22R source missing:\n{SOURCE}"
    )


print("Root:")
print(" ", SOURCE)

print()
print("Top-level files/directories:")


for p in sorted(
    SOURCE.iterdir(),
    key=lambda x:
        x.name.lower(),
):

    kind = (
        "DIR"
        if p.is_dir()
        else "FILE"
    )

    print(
        f"  [{kind}] {p.name}"
    )


# =============================================================================
# 5. PARQUET METADATA / SCHEMA ONLY
#
# NO row groups or row values are read.
# =============================================================================

print()
print("=" * 110)
print("AUTHORIZED FEBRUARY PARQUET SCHEMAS — METADATA ONLY")
print("=" * 110)
print()


schemas = {}


for filename, expected_rows in zip(
    DAY_FILES,
    EXPECTED_ROWS,
):

    path = (
        SOURCE
        / filename
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing authorized February Parquet:\n{path}"
        )


    pf = pq.ParquetFile(
        path
    )


    rows = int(
        pf.metadata.num_rows
    )


    if rows != expected_rows:

        raise RuntimeError(
            f"{filename}: row-count mismatch.\n"
            f"expected={expected_rows}\n"
            f"actual={rows}"
        )


    columns = list(
        pf.schema_arrow.names
    )


    schemas[
        filename
    ] = columns


    hits = candidate_columns(
        columns
    )


    print(filename)

    print(
        "  rows       :",
        f"{rows:,}",
    )

    print(
        "  columns    :",
        len(columns),
    )

    print(
        "  candidates :",
        hits,
    )

    print()


# Require all eight development Parquets to have the same schema.

first_schema = schemas[
    DAY_FILES[0]
]


for filename in DAY_FILES[1:]:

    if (
        schemas[
            filename
        ]
        != first_schema
    ):

        raise RuntimeError(
            f"Authorized development schema differs for {filename}."
        )


print("[EXACT] all eight authorized February Parquets share one schema")


# =============================================================================
# 6. EXACT CANDIDATE COLUMN REPORT
# =============================================================================

print()
print("=" * 110)
print("DEVELOPMENT CACHE LOCATOR / LABEL COLUMNS")
print("=" * 110)
print()


interesting_names = [
    "clean_position",
    "day_id",
    "original_row_index",
    "binary_label",
    "Label",
    "label",
    "attack_family",
    "attack_category",
    "category",
]


for name in interesting_names:

    present = (
        name
        in first_schema
    )

    print(
        f"{name:<24}: "
        f"{'PRESENT' if present else 'ABSENT'}"
    )


print()
print("All candidate-like columns:")

for col in candidate_columns(
    first_schema
):
    print(" ", col)


# =============================================================================
# 7. SEARCH TRACKED REPOSITORY TEXT
#
# git grep only searches tracked repository content.
# No Kaggle data is read.
# =============================================================================

print()
print("=" * 110)
print("TRACKED REPOSITORY ALIGNMENT INFRASTRUCTURE")
print("=" * 110)
print()


for term in SEARCH_TERMS:

    print()
    print("-" * 110)
    print("SEARCH:", term)
    print("-" * 110)


    rc, output = git(
        "grep",
        "-n",
        "-I",
        "-i",
        "-F",
        term,
        "--",
        "results",
        "scripts",
        "src",
        check=False,
    )


    # git grep returns 1 when no matches exist.
    if rc not in (
        0,
        1,
    ):

        raise RuntimeError(
            f"git grep failed for {term}:\n{output}"
        )


    if not output:

        print("[NONE]")

        continue


    lines = output.splitlines()


    # Preflight only; prevent enormous output.
    max_lines = 40


    for line in lines[
        :max_lines
    ]:

        print(line)


    if len(lines) > max_lines:

        print(
            f"... {len(lines)-max_lines} additional matches omitted"
        )


# =============================================================================
# 8. SEARCH LIKELY TRACKED FILENAMES
# =============================================================================

print()
print("=" * 110)
print("LIKELY TRACKED ATTACK/LABEL FILES")
print("=" * 110)
print()


_, tracked = git(
    "ls-files",
)


tracked_files = tracked.splitlines()


keywords = [
    "attack",
    "family",
    "category",
    "label",
    "row_index",
    "day_id",
]


likely_files = [
    path
    for path in tracked_files
    if any(
        keyword in path.lower()
        for keyword in keywords
    )
]


if likely_files:

    for path in likely_files[
        :100
    ]:

        print(path)


    if len(
        likely_files
    ) > 100:

        print(
            f"... {len(likely_files)-100} additional paths omitted"
        )

else:

    print("[NONE]")


# =============================================================================
# 9. FINAL
# =============================================================================

print()
print("=" * 110)
print("STAGE23-6 ATTACK-FAMILY PREFLIGHT COMPLETE")
print("=" * 110)
print()

print("Stage23 model fits        : 50 / 50 SEALED")
print("New model fits            : 0")
print("Predictions computed      : 0")
print("Attack-family metrics     : 0")
print("Scientific writes         : 0")
print()
print("Authorized Parquet rows   : NOT READ")
print("Parquet row groups read   : 0")
print("Only schema/metadata used : YES")
print()
print("Raw Mar1 accessed         : NO")
print("Raw Mar2 accessed         : NO")
print()
print("NEXT:")
print("  Resolve the exact frozen day_id + original_row_index")
print("  attack-family label source from the preflight output.")
print("  Then perform analysis-only family evaluation using")
print("  already-sealed FULL and ablated probabilities.")
print("  NO RETRAINING.")
print("=" * 110)

STAGE23-6 — ATTACK-FAMILY ANALYSIS PREFLIGHT

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23 model fits : 50 / 50 SEALED
[OK] additional fits    : 0 authorized

attack_family_spec.json
  expected: 9382b75a4baf3bae1a3babbd6384b82d3b5f3a2698fbf5c85a006a8f22081842
  actual:   9382b75a4baf3bae1a3babbd6384b82d3b5f3a2698fbf5c85a006a8f22081842
  [EXACT]

FROZEN ATTACK-FAMILY SPEC

Evaluation:
  For attack family A, form a validation evaluation set containing all benign rows from that split plus attack rows belonging to A.

Alignment:
  Use day_id + original_row_index to recover development attack family labels from authorized Feb14-Feb28 source labels or existing repository attack-category infrastructure.

Minimum attack support for interpretation: 100
Low-support policy: Families with fewer than 100 validation attacks are listed descriptively but excluded from inferential interpretatio

In [3]:
# =============================================================================
# STAGE23-6A — AUTHORITATIVE DEVELOPMENT LABEL-SOURCE DISCOVERY
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO METRICS
# ZERO SCIENTIFIC WRITES
# NO MAR1 / MAR2
#
# Purpose:
#   Find the authoritative Feb14-Feb28 source CSVs containing the original
#   categorical Label column required by the frozen Stage23 attack-family
#   alignment:
#
#       day_id + original_row_index
#
# Header discovery only. No CSV data rows are loaded.
# =============================================================================

from pathlib import Path
import subprocess
import hashlib
import csv
import os


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

INPUT_ROOT = Path(
    "/kaggle/input"
)


DAY_SPEC = [
    {
        "day_id": 0,
        "source_file": "02-14-2018.csv",
        "date_tokens": [
            "02-14-2018",
            "14-02-2018",
        ],
    },
    {
        "day_id": 1,
        "source_file": "02-15-2018.csv",
        "date_tokens": [
            "02-15-2018",
            "15-02-2018",
        ],
    },
    {
        "day_id": 2,
        "source_file": "02-16-2018.csv",
        "date_tokens": [
            "02-16-2018",
            "16-02-2018",
        ],
    },
    {
        "day_id": 3,
        "source_file": "02-20-2018.csv",
        "date_tokens": [
            "02-20-2018",
            "20-02-2018",
        ],
    },
    {
        "day_id": 4,
        "source_file": "02-21-2018.csv",
        "date_tokens": [
            "02-21-2018",
            "21-02-2018",
        ],
    },
    {
        "day_id": 5,
        "source_file": "02-22-2018.csv",
        "date_tokens": [
            "02-22-2018",
            "22-02-2018",
        ],
    },
    {
        "day_id": 6,
        "source_file": "02-23-2018.csv",
        "date_tokens": [
            "02-23-2018",
            "23-02-2018",
        ],
    },
    {
        "day_id": 7,
        "source_file": "02-28-2018.csv",
        "date_tokens": [
            "02-28-2018",
            "28-02-2018",
        ],
    },
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def read_csv_header(path):

    # Header only.
    # Try common CICIDS-compatible encodings.

    last_error = None

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "latin-1",
    ]:

        try:

            with Path(path).open(
                "r",
                encoding=encoding,
                newline="",
                errors="strict",
            ) as f:

                first_line = f.readline()

            if not first_line:
                raise RuntimeError(
                    "File is empty."
                )

            row = next(
                csv.reader(
                    [first_line]
                )
            )

            normalized = [
                str(x)
                .replace("\ufeff", "")
                .strip()

                for x in row
            ]

            return {
                "encoding":
                    encoding,

                "columns":
                    normalized,
            }


        except Exception as exc:

            last_error = exc


    raise RuntimeError(
        f"Could not read CSV header: {path}\n"
        f"{type(last_error).__name__}: {last_error}"
    )


# =============================================================================
# 2. SEALED STATE
# =============================================================================

print("=" * 110)
print("STAGE23-6A — DEVELOPMENT ATTACK-LABEL SOURCE DISCOVERY")
print("=" * 110)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 seal-tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")


# =============================================================================
# 3. INVENTORY CSV FILES UNDER /kaggle/input
#
# Directory enumeration only.
# =============================================================================

if not INPUT_ROOT.is_dir():

    raise RuntimeError(
        "/kaggle/input is unavailable."
    )


print()
print("Scanning /kaggle/input for February source CSV filenames...")


all_csvs = sorted(
    p
    for p in INPUT_ROOT.rglob(
        "*.csv"
    )
    if p.is_file()
)


print(
    "[OK] CSV files visible:",
    len(
        all_csvs
    ),
)


# Explicit safety: final holdout source names may NEVER be opened here.

for forbidden in [
    "03-01-2018.csv",
    "03-02-2018.csv",
]:

    forbidden_visible = [
        p
        for p in all_csvs
        if p.name
        == forbidden
    ]

    if forbidden_visible:

        print(
            f"[INFO] {forbidden} exists in Kaggle input, "
            "but WILL NOT be opened."
        )


# =============================================================================
# 4. FIND EXACT + FUZZY CANDIDATES
# =============================================================================

resolved = {}
unresolved = []


print()
print("=" * 110)
print("FEBRUARY LABEL-SOURCE CANDIDATES")
print("=" * 110)
print()


for day in DAY_SPEC:

    day_id = day[
        "day_id"
    ]

    exact_name = day[
        "source_file"
    ]

    tokens = [
        token.lower()
        for token
        in day[
            "date_tokens"
        ]
    ]


    exact_candidates = [
        p
        for p in all_csvs
        if p.name
        == exact_name
    ]


    fuzzy_candidates = [
        p
        for p in all_csvs

        if (
            p not in exact_candidates
            and any(
                token
                in p.name.lower()

                for token
                in tokens
            )
        )
    ]


    print(
        f"day_id={day_id}  expected={exact_name}"
    )


    if exact_candidates:

        print(
            "  exact candidates:",
            len(
                exact_candidates
            ),
        )

        for p in exact_candidates:

            print(
                "   ",
                p,
                f"({p.stat().st_size / (1024**2):.2f} MiB)",
            )


    else:

        print(
            "  exact candidates: 0"
        )


    if fuzzy_candidates:

        print(
            "  fuzzy candidates:",
            len(
                fuzzy_candidates
            ),
        )

        for p in fuzzy_candidates[
            :20
        ]:

            print(
                "   ",
                p,
                f"({p.stat().st_size / (1024**2):.2f} MiB)",
            )


        if len(
            fuzzy_candidates
        ) > 20:

            print(
                f"    ... "
                f"{len(fuzzy_candidates)-20} more"
            )


    else:

        print(
            "  fuzzy candidates: 0"
        )


    # -------------------------------------------------------------------------
    # We auto-resolve ONLY an exact filename candidate containing Label.
    #
    # Never silently choose among fuzzy candidates.
    # -------------------------------------------------------------------------

    valid_exact = []


    for p in exact_candidates:

        header = read_csv_header(
            p
        )


        columns = header[
            "columns"
        ]


        label_matches = [
            col
            for col in columns
            if col.lower()
            == "label"
        ]


        print(
            "  header:",
            p,
        )

        print(
            "    encoding :",
            header[
                "encoding"
            ],
        )

        print(
            "    columns  :",
            len(
                columns
            ),
        )

        print(
            "    Label    :",
            (
                label_matches[
                    0
                ]
                if len(
                    label_matches
                ) == 1
                else "NOT_UNIQUE_OR_ABSENT"
            ),
        )


        if len(
            label_matches
        ) == 1:

            valid_exact.append(
                {
                    "path":
                        p,

                    "label_column":
                        label_matches[
                            0
                        ],

                    "column_count":
                        len(
                            columns
                        ),

                    "encoding":
                        header[
                            "encoding"
                        ],
                }
            )


    if len(
        valid_exact
    ) == 1:

        resolved[
            day_id
        ] = (
            valid_exact[
                0
            ]
        )

        print(
            "  [RESOLVED EXACT]"
        )


    else:

        unresolved.append(
            {
                "day_id":
                    day_id,

                "expected":
                    exact_name,

                "valid_exact_count":
                    len(
                        valid_exact
                    ),

                "fuzzy_count":
                    len(
                        fuzzy_candidates
                    ),
            }
        )

        print(
            "  [NOT RESOLVED]"
        )


    print()


# =============================================================================
# 5. FINAL REPORT
# =============================================================================

print("=" * 110)
print("AUTHORITATIVE LABEL-SOURCE DISCOVERY RESULT")
print("=" * 110)
print()


print(
    "Exact Label-bearing sources resolved:",
    len(
        resolved
    ),
    "/ 8",
)


for day_id in sorted(
    resolved
):

    r = resolved[
        day_id
    ]

    print()
    print(
        f"day_id={day_id}"
    )

    print(
        "  path        :",
        r[
            "path"
        ],
    )

    print(
        "  Label column:",
        r[
            "label_column"
        ],
    )

    print(
        "  columns     :",
        r[
            "column_count"
        ],
    )

    print(
        "  encoding    :",
        r[
            "encoding"
        ],
    )


print()
print("-" * 110)


if unresolved:

    print()
    print("[STOP] Not all eight authoritative source CSVs were resolved.")
    print()

    for row in unresolved:

        print(
            f"day_id={row['day_id']} "
            f"expected={row['expected']} "
            f"valid_exact={row['valid_exact_count']} "
            f"fuzzy={row['fuzzy_count']}"
        )


    print()
    print(
        "Do NOT proceed to attack-family label recovery yet."
    )


else:

    print()
    print("[READY] All eight authoritative February source-label CSVs resolved.")
    print()
    print("NEXT:")
    print(
        "  Recover categorical Label ONLY through the frozen"
    )
    print(
        "  day_id + original_row_index alignment and verify"
    )
    print(
        "  Label→binary_label agreement before computing any"
    )
    print(
        "  attack-family metrics."
    )


print()
print("MODEL / DATA GOVERNANCE")
print("  New model fits         : 0")
print("  Model inference        : 0")
print("  LightGBM execution     : 0")
print("  XGBoost execution      : 0")
print("  CSV data rows read     : 0")
print("  CSV headers read       : February candidates only")
print("  Raw Mar1 rows accessed : NO")
print("  Raw Mar2 rows accessed : NO")
print("=" * 110)

STAGE23-6A — DEVELOPMENT ATTACK-LABEL SOURCE DISCOVERY

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] new fits  : 0 authorized

Scanning /kaggle/input for February source CSV filenames...
[OK] CSV files visible: 28
[INFO] 03-01-2018.csv exists in Kaggle input, but WILL NOT be opened.
[INFO] 03-02-2018.csv exists in Kaggle input, but WILL NOT be opened.

FEBRUARY LABEL-SOURCE CANDIDATES

day_id=0  expected=02-14-2018.csv
  exact candidates: 1
    /kaggle/input/datasets/solarmainframe/ids-intrusion-csv/02-14-2018.csv (341.63 MiB)
  fuzzy candidates: 0
  header: /kaggle/input/datasets/solarmainframe/ids-intrusion-csv/02-14-2018.csv
    encoding : utf-8-sig
    columns  : 80
    Label    : Label
  [RESOLVED EXACT]

day_id=1  expected=02-15-2018.csv
  exact candidates: 1
    /kaggle/input/datasets/solarmainframe/ids-intrusion-csv/02-15-2018.csv (358.53 MiB)
  

In [4]:
# =============================================================================
# STAGE23-6B — original_row_index SEMANTICS PREFLIGHT
#
# PURPOSE
#   Determine EXACTLY how Stage22R original_row_index maps back to the
#   authoritative CIC-IDS2018 CSV source when embedded header rows exist.
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO ATTACK-FAMILY METRICS
# ZERO SCIENTIFIC WRITES
# NO LIGHTGBM / XGBOOST
# NO MAR1 / MAR2
#
# We intentionally inspect only:
#   - Stage22R metadata columns from the eight February Parquets
#   - Label column from authoritative February source CSVs
#
# This cell tests alignment and FAILS CLOSED if binary labels disagree.
# =============================================================================

from pathlib import Path
import subprocess
import os
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

CACHE = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

RAW = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/"
    "ids-intrusion-csv"
)

DAYS = [
    (0, "02-14-2018.csv", "day_00_02-14-2018.parquet"),
    (1, "02-15-2018.csv", "day_01_02-15-2018.parquet"),
    (2, "02-16-2018.csv", "day_02_02-16-2018.parquet"),
    (3, "02-20-2018.csv", "day_03_02-20-2018.parquet"),
    (4, "02-21-2018.csv", "day_04_02-21-2018.parquet"),
    (5, "02-22-2018.csv", "day_05_02-22-2018.parquet"),
    (6, "02-23-2018.csv", "day_06_02-23-2018.parquet"),
    (7, "02-28-2018.csv", "day_07_02-28-2018.parquet"),
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(p.stdout)

    return (p.stdout or "").strip()


def normalize_label(x):

    return (
        str(x)
        .replace("\ufeff", "")
        .strip()
    )


def to_binary(label):

    label = normalize_label(label)

    if label.upper() == "BENIGN":
        return 0

    # Embedded header is NOT an attack.
    if label.upper() == "LABEL":
        return -1

    return 1


# =============================================================================
# 2. SEALED STATE
# =============================================================================

print("=" * 110)
print("STAGE23-6B — original_row_index SEMANTICS PREFLIGHT")
print("=" * 110)
print()

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"Unexpected HEAD:\n{head}"
    )

if tag_commit != EXPECTED_HEAD:
    raise RuntimeError(
        "Stage23-5 tag mismatch."
    )

if status:
    raise RuntimeError(
        "Repository dirty:\n"
        + status
    )

print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] model use : NONE")


# =============================================================================
# 3. INSPECT TRACKED TERMINOLOGY
# =============================================================================

print()
print("=" * 110)
print("TRACKED ROW-INDEX TERMINOLOGY")
print("=" * 110)
print()

for term in [
    "original_zero_based_row_index",
    "original_row_index",
    "embedded_header",
]:

    p = subprocess.run(
        [
            "git",
            "grep",
            "-n",
            "-I",
            "-F",
            term,
            "--",
            "results/stage22r_protocol_recovery",
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(f"SEARCH: {term}")

    if p.returncode == 0:

        lines = (
            p.stdout or ""
        ).splitlines()

        for line in lines[:25]:
            print(" ", line)

        if len(lines) > 25:
            print(
                f"  ... {len(lines)-25} additional matches"
            )

    elif p.returncode == 1:

        print("  [NONE]")

    else:

        raise RuntimeError(
            p.stdout
        )

    print()


# =============================================================================
# 4. TEST INDEX SEMANTICS DAY BY DAY
#
# We use deterministic locator samples:
#   first 16
#   last 16
#   32 evenly spaced
#
# For days with embedded headers we also test around every raw row whose
# Label equals "Label".
#
# =============================================================================

print("=" * 110)
print("SOURCE-ROW ALIGNMENT TEST")
print("=" * 110)
print()

results = []


for day_id, raw_name, parquet_name in DAYS:

    raw_path = (
        RAW
        / raw_name
    )

    parquet_path = (
        CACHE
        / parquet_name
    )

    if not raw_path.is_file():
        raise RuntimeError(
            f"Missing raw February source:\n{raw_path}"
        )

    if not parquet_path.is_file():
        raise RuntimeError(
            f"Missing Stage22R cache:\n{parquet_path}"
        )


    # -------------------------------------------------------------------------
    # Read ONLY Stage22R metadata required for alignment.
    # -------------------------------------------------------------------------

    table = pq.read_table(
        parquet_path,
        columns=[
            "original_row_index",
            "binary_label",
        ],
    )

    original_index = np.asarray(
        table[
            "original_row_index"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )

    binary_label = np.asarray(
        table[
            "binary_label"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int8,
    )

    n_clean = len(
        original_index
    )


    if n_clean == 0:
        raise RuntimeError(
            f"{raw_name}: empty Stage22R cache."
        )


    # -------------------------------------------------------------------------
    # Read ONLY Label from authoritative source.
    #
    # Keep every physical CSV data row. Embedded repeated headers therefore
    # appear naturally as Label == "Label".
    # -------------------------------------------------------------------------

    labels_df = pd.read_csv(
        raw_path,
        usecols=["Label"],
        dtype=str,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_filter=False,
        low_memory=False,
    )

    raw_labels = (
        labels_df[
            "Label"
        ]
        .astype(str)
        .map(normalize_label)
        .to_numpy()
    )

    raw_rows = len(
        raw_labels
    )

    embedded = np.flatnonzero(
        np.char.upper(
            raw_labels.astype(str)
        )
        == "LABEL"
    ).astype(
        np.int64
    )


    # -------------------------------------------------------------------------
    # Basic index-bound test.
    # -------------------------------------------------------------------------

    if original_index.min() < 0:
        raise RuntimeError(
            f"{raw_name}: negative original_row_index."
        )

    physical_index_possible = (
        original_index.max()
        < raw_rows
    )


    # -------------------------------------------------------------------------
    # Deterministic sample of CLEAN CACHE rows.
    # -------------------------------------------------------------------------

    sample_cache_positions = np.unique(
        np.concatenate(
            [
                np.arange(
                    min(
                        16,
                        n_clean
                    ),
                    dtype=np.int64,
                ),

                np.arange(
                    max(
                        0,
                        n_clean - 16
                    ),
                    n_clean,
                    dtype=np.int64,
                ),

                np.linspace(
                    0,
                    n_clean - 1,
                    num=min(
                        32,
                        n_clean
                    ),
                    dtype=np.int64,
                ),
            ]
        )
    )


    # Add retained rows near embedded-header source positions.
    if len(
        embedded
    ):

        for h in embedded:

            nearby = np.flatnonzero(
                (
                    original_index
                    >= max(
                        0,
                        int(h) - 3
                    )
                )
                &
                (
                    original_index
                    <= int(h) + 3
                )
            )

            if len(nearby):
                sample_cache_positions = np.unique(
                    np.concatenate(
                        [
                            sample_cache_positions,
                            nearby,
                        ]
                    )
                )


    sample_original = original_index[
        sample_cache_positions
    ]

    sample_binary = binary_label[
        sample_cache_positions
    ]


    # -------------------------------------------------------------------------
    # MODE A:
    # original_row_index is physical zero-based CSV data-row index.
    # -------------------------------------------------------------------------

    physical_matches = None
    physical_mismatches = None


    if physical_index_possible:

        labels_a = raw_labels[
            sample_original
        ]

        binary_a = np.asarray(
            [
                to_binary(x)
                for x in labels_a
            ],
            dtype=np.int8,
        )

        valid_a = (
            binary_a
            >= 0
        )

        physical_matches = int(
            np.sum(
                binary_a[
                    valid_a
                ]
                ==
                sample_binary[
                    valid_a
                ]
            )
        )

        physical_mismatches = int(
            np.sum(
                binary_a[
                    valid_a
                ]
                !=
                sample_binary[
                    valid_a
                ]
            )
        )

        embedded_hits_a = int(
            np.sum(
                ~valid_a
            )
        )

    else:

        embedded_hits_a = None


    # -------------------------------------------------------------------------
    # MODE B:
    # original_row_index is zero-based AFTER embedded-header rows are removed.
    #
    # Construct effective Label vector only for this comparison.
    # -------------------------------------------------------------------------

    effective_mask = (
        np.char.upper(
            raw_labels.astype(str)
        )
        != "LABEL"
    )

    effective_labels = raw_labels[
        effective_mask
    ]


    effective_index_possible = (
        original_index.max()
        < len(
            effective_labels
        )
    )


    effective_matches = None
    effective_mismatches = None


    if effective_index_possible:

        labels_b = effective_labels[
            sample_original
        ]

        binary_b = np.asarray(
            [
                to_binary(x)
                for x in labels_b
            ],
            dtype=np.int8,
        )

        effective_matches = int(
            np.sum(
                binary_b
                ==
                sample_binary
            )
        )

        effective_mismatches = int(
            np.sum(
                binary_b
                !=
                sample_binary
            )
        )


    # -------------------------------------------------------------------------
    # Decide only when one mode is unambiguously supported.
    # -------------------------------------------------------------------------

    sample_n = len(
        sample_cache_positions
    )


    physical_exact = (
        physical_index_possible
        and physical_mismatches == 0
        and embedded_hits_a == 0
    )

    effective_exact = (
        effective_index_possible
        and effective_mismatches == 0
    )


    if physical_exact and not effective_exact:

        mode = (
            "PHYSICAL_ZERO_BASED_CSV_DATA_ROW_INDEX"
        )

    elif effective_exact and not physical_exact:

        mode = (
            "POST_EMBEDDED_HEADER_EFFECTIVE_ROW_INDEX"
        )

    elif physical_exact and effective_exact:

        # This is expected on days with zero embedded headers:
        # both mappings are mathematically identical.
        if len(
            embedded
        ) == 0:

            mode = (
                "IDENTICAL_BECAUSE_NO_EMBEDDED_HEADERS"
            )

        else:

            raise RuntimeError(
                f"{raw_name}: both mappings appear exact despite "
                "embedded headers; cannot resolve semantics safely."
            )

    else:

        raise RuntimeError(
            f"{raw_name}: neither candidate original_row_index mapping "
            "passed exact binary-label agreement.\n"
            f"physical mismatches={physical_mismatches}, "
            f"embedded hits={embedded_hits_a}\n"
            f"effective mismatches={effective_mismatches}"
        )


    print(raw_name)
    print(
        "  raw CSV data rows      :",
        f"{raw_rows:,}",
    )
    print(
        "  Stage22R retained rows :",
        f"{n_clean:,}",
    )
    print(
        "  embedded header rows   :",
        len(
            embedded
        ),
    )

    if len(
        embedded
    ):

        print(
            "  embedded positions     :",
            embedded[
                :20
            ].tolist(),
            (
                ""
                if len(
                    embedded
                ) <= 20
                else " ..."
            ),
        )

    print(
        "  original index min/max :",
        int(
            original_index.min()
        ),
        "/",
        int(
            original_index.max()
        ),
    )

    print(
        "  deterministic test rows:",
        sample_n,
    )

    print(
        "  physical mapping       :",
        (
            f"matches={physical_matches}, "
            f"mismatches={physical_mismatches}, "
            f"embedded_hits={embedded_hits_a}"
            if physical_index_possible
            else "OUT_OF_RANGE"
        ),
    )

    print(
        "  effective mapping      :",
        (
            f"matches={effective_matches}, "
            f"mismatches={effective_mismatches}"
            if effective_index_possible
            else "OUT_OF_RANGE"
        ),
    )

    print(
        "  [RESOLVED]             :",
        mode,
    )

    print()


    results.append(
        {
            "day_id":
                day_id,

            "file":
                raw_name,

            "raw_rows":
                raw_rows,

            "retained_rows":
                n_clean,

            "embedded_headers":
                int(
                    len(
                        embedded
                    )
                ),

            "mapping_mode":
                mode,

            "sample_rows":
                int(
                    sample_n
                ),

            "physical_mismatches":
                physical_mismatches,

            "effective_mismatches":
                effective_mismatches,
        }
    )


    del (
        table,
        original_index,
        binary_label,
        labels_df,
        raw_labels,
        effective_labels,
        effective_mask,
    )

    gc.collect()


# =============================================================================
# 5. GLOBAL RESOLUTION
# =============================================================================

embedded_days = [
    r
    for r in results
    if r[
        "embedded_headers"
    ] > 0
]


if not embedded_days:

    raise RuntimeError(
        "Expected at least one February source with embedded headers."
    )


embedded_modes = {
    r[
        "mapping_mode"
    ]
    for r in embedded_days
}


if len(
    embedded_modes
) != 1:

    raise RuntimeError(
        "Embedded-header days disagree on original_row_index semantics:\n"
        + repr(
            embedded_modes
        )
    )


resolved_embedded_mode = next(
    iter(
        embedded_modes
    )
)


# =============================================================================
# 6. FINAL
# =============================================================================

print("=" * 110)
print("STAGE23-6B original_row_index SEMANTICS RESOLVED")
print("=" * 110)
print()

print(
    "Embedded-header mapping:",
    resolved_embedded_mode,
)

print()
print("Per-day status:")

for r in results:

    print(
        f"  day_id={r['day_id']} "
        f"{r['file']:<14} "
        f"embedded={r['embedded_headers']:<3} "
        f"mode={r['mapping_mode']}"
    )

print()
print("Model fits              : 0")
print("Model inference         : 0")
print("LightGBM execution      : 0")
print("XGBoost execution       : 0")
print("Attack-family metrics   : 0")
print("Scientific writes       : 0")
print()
print("Raw Mar1 accessed       : NO")
print("Raw Mar2 accessed       : NO")
print()
print("NEXT:")
print("  Recover categorical attack labels for the EXACT")
print("  RANDOM_NATURAL and CHRONOLOGICAL_NATURAL validation")
print("  memberships, verify 100% Label→binary_label agreement,")
print("  then compute frozen attack-family metrics from the")
print("  SEALED probability artifacts only.")
print("  NO MODEL EXECUTION.")
print("=" * 110)

STAGE23-6B — original_row_index SEMANTICS PREFLIGHT

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] model use : NONE

TRACKED ROW-INDEX TERMINOLOGY

SEARCH: original_zero_based_row_index
  results/stage22r_protocol_recovery/stage22r_1b0_membership_execution_lock/stage22r_1b0_membership_execution_lock.json:14:    "definition": "All K79-retained development rows sorted by (day_id ASC, original_zero_based_row_index ASC)."
  results/stage22r_protocol_recovery/stage22r_1b1_development_memberships/stage22r_1b1_membership_summary.json:88:    "canonical_order": "(day_id ASC, original_zero_based_row_index ASC) after frozen K79 exclusions",
  results/stage22r_protocol_recovery/stage22r_1c_development_model_inputs/stage22r_1c_development_model_input_manifest.json:177:    "canonical_row_order": "clean_position ASC, where clean_position follows (day_id ASC, original_ze

RuntimeError: 02-16-2018.csv: both mappings appear exact despite embedded headers; cannot resolve semantics safely.

In [5]:
# =============================================================================
# STAGE23-6B2 — DECISIVE original_row_index SEMANTICS RESOLUTION
#
# Previous Stage23-6B stopped safely because a SMALL binary-label sample
# could not distinguish physical vs post-header indexing on 02-16.
#
# This cell resolves the ambiguity using EVERY K79-retained row from ONLY
# the two development days containing embedded headers:
#
#   day_id=2 -> 02-16-2018.csv : 1 embedded header
#   day_id=7 -> 02-28-2018.csv : 33 embedded headers
#
# READS:
#   Parquet : original_row_index + binary_label ONLY
#   Raw CSV : Label ONLY
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM
# ZERO XGBOOST
# ZERO ATTACK-FAMILY METRICS
# ZERO SCIENTIFIC WRITES
# NO MAR1 / MAR2
# =============================================================================

from pathlib import Path
import subprocess
import os
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

CACHE = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

RAW = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/"
    "ids-intrusion-csv"
)


TEST_DAYS = [
    {
        "day_id": 2,
        "raw_file": "02-16-2018.csv",
        "cache_file": "day_02_02-16-2018.parquet",
        "expected_embedded_headers": 1,
        "expected_retained_rows": 900_988,
    },
    {
        "day_id": 7,
        "raw_file": "02-28-2018.csv",
        "cache_file": "day_07_02-28-2018.parquet",
        "expected_embedded_headers": 33,
        "expected_retained_rows": 593_780,
    },
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (p.stdout or "").strip()


def normalize_label_series(series):

    return (
        series
        .astype(str)
        .str.replace(
            "\ufeff",
            "",
            regex=False,
        )
        .str.strip()
    )


def labels_to_binary(labels):

    labels = np.asarray(
        labels,
        dtype=object,
    )

    out = np.ones(
        len(labels),
        dtype=np.int8,
    )

    for i, value in enumerate(labels):

        text = str(
            value
        ).strip()

        upper = text.upper()

        if upper == "BENIGN":

            out[i] = 0

        elif upper == "LABEL":

            # Embedded header.
            out[i] = -1

        else:

            # Every non-BENIGN traffic category is attack.
            out[i] = 1

    return out


# =============================================================================
# 2. SEALED STATE
# =============================================================================

print("=" * 110)
print("STAGE23-6B2 — DECISIVE original_row_index SEMANTICS RESOLUTION")
print("=" * 110)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] model use : NONE")


# =============================================================================
# 3. FULL-POPULATION ALIGNMENT TEST
# =============================================================================

print()
print("=" * 110)
print("FULL RETAINED-ROW ALIGNMENT TEST — EMBEDDED-HEADER DAYS ONLY")
print("=" * 110)
print()


results = []


for spec in TEST_DAYS:

    day_id = spec[
        "day_id"
    ]

    raw_path = (
        RAW
        / spec[
            "raw_file"
        ]
    )

    cache_path = (
        CACHE
        / spec[
            "cache_file"
        ]
    )


    if not raw_path.is_file():

        raise RuntimeError(
            f"Missing authoritative source:\n{raw_path}"
        )


    if not cache_path.is_file():

        raise RuntimeError(
            f"Missing authorized Stage22R cache:\n{cache_path}"
        )


    print("-" * 110)

    print(
        f"day_id={day_id}  "
        f"{spec['raw_file']}"
    )

    print("-" * 110)


    # =========================================================================
    # A. LOAD EXACT STAGE22R LOCATORS + BINARY LABELS
    # =========================================================================

    table = pq.read_table(
        cache_path,
        columns=[
            "original_row_index",
            "binary_label",
        ],
    )


    original_index = np.asarray(
        table[
            "original_row_index"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )


    cache_binary = np.asarray(
        table[
            "binary_label"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int8,
    )


    if len(
        original_index
    ) != spec[
        "expected_retained_rows"
    ]:

        raise RuntimeError(
            f"{spec['raw_file']}: retained-row count mismatch."
        )


    if len(
        original_index
    ) != len(
        cache_binary
    ):

        raise RuntimeError(
            f"{spec['raw_file']}: metadata length mismatch."
        )


    if np.any(
        original_index < 0
    ):

        raise RuntimeError(
            f"{spec['raw_file']}: negative original_row_index."
        )


    # Stage22R canonical order requires original row locators increasing
    # within each day after K79 exclusion.

    if not np.all(
        np.diff(
            original_index
        ) > 0
    ):

        raise RuntimeError(
            f"{spec['raw_file']}: original_row_index is not "
            "strictly increasing."
        )


    # =========================================================================
    # B. LOAD ONLY AUTHORITATIVE CATEGORICAL LABEL
    # =========================================================================

    raw_df = pd.read_csv(
        raw_path,
        usecols=[
            "Label"
        ],
        dtype=str,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_filter=False,
        low_memory=False,
    )


    raw_label_series = (
        normalize_label_series(
            raw_df[
                "Label"
            ]
        )
    )


    raw_labels = (
        raw_label_series
        .to_numpy(
            dtype=object
        )
    )


    raw_binary = (
        labels_to_binary(
            raw_labels
        )
    )


    embedded_positions = np.flatnonzero(
        raw_binary
        == -1
    ).astype(
        np.int64
    )


    if len(
        embedded_positions
    ) != spec[
        "expected_embedded_headers"
    ]:

        raise RuntimeError(
            f"{spec['raw_file']}: embedded-header count mismatch.\n"
            f"expected={spec['expected_embedded_headers']}\n"
            f"actual={len(embedded_positions)}"
        )


    print(
        "Raw CSV data rows          :",
        f"{len(raw_labels):,}",
    )

    print(
        "Stage22R retained rows      :",
        f"{len(original_index):,}",
    )

    print(
        "Embedded headers            :",
        len(
            embedded_positions
        ),
    )

    print(
        "Embedded physical positions :",
        embedded_positions.tolist(),
    )

    print(
        "original_row_index min/max  :",
        int(
            original_index.min()
        ),
        "/",
        int(
            original_index.max()
        ),
    )


    # =========================================================================
    # C. CANDIDATE 1 — PHYSICAL ZERO-BASED CSV DATA-ROW INDEX
    #
    # Stage22R locator points directly into raw_df before embedded-header
    # removal.
    # =========================================================================

    physical_in_range = bool(
        original_index.max()
        < len(
            raw_labels
        )
    )


    physical_header_hits = None
    physical_mismatches = None
    physical_matches = None


    if physical_in_range:

        physical_source_binary = (
            raw_binary[
                original_index
            ]
        )


        physical_header_hits = int(
            np.sum(
                physical_source_binary
                == -1
            )
        )


        physical_valid = (
            physical_source_binary
            >= 0
        )


        physical_mismatch_mask = (
            physical_valid
            &
            (
                physical_source_binary
                != cache_binary
            )
        )


        physical_mismatches = int(
            np.sum(
                physical_mismatch_mask
            )
        )


        physical_matches = int(
            np.sum(
                physical_valid
                &
                (
                    physical_source_binary
                    == cache_binary
                )
            )
        )


        physical_exact = (
            physical_header_hits == 0
            and physical_mismatches == 0
            and physical_matches
            == len(
                original_index
            )
        )


    else:

        physical_exact = False


    # =========================================================================
    # D. CANDIDATE 2 — POST-HEADER EFFECTIVE ROW INDEX
    #
    # Remove all embedded headers first, then interpret original_row_index
    # against the compressed traffic-row vector.
    # =========================================================================

    traffic_mask = (
        raw_binary
        >= 0
    )


    effective_binary = (
        raw_binary[
            traffic_mask
        ]
    )


    effective_in_range = bool(
        original_index.max()
        < len(
            effective_binary
        )
    )


    effective_mismatches = None
    effective_matches = None


    if effective_in_range:

        effective_source_binary = (
            effective_binary[
                original_index
            ]
        )


        effective_mismatch_mask = (
            effective_source_binary
            != cache_binary
        )


        effective_mismatches = int(
            np.sum(
                effective_mismatch_mask
            )
        )


        effective_matches = int(
            np.sum(
                ~effective_mismatch_mask
            )
        )


        effective_exact = (
            effective_mismatches == 0
            and effective_matches
            == len(
                original_index
            )
        )


    else:

        effective_exact = False


    # =========================================================================
    # E. ADDITIONAL STRUCTURAL EVIDENCE
    #
    # Under physical indexing, no retained locator may point to a detected
    # embedded-header row.
    # =========================================================================

    header_locator_intersection = np.intersect1d(
        original_index,
        embedded_positions,
        assume_unique=False,
    )


    print()
    print("PHYSICAL candidate")
    print(
        "  in range       :",
        physical_in_range,
    )

    print(
        "  exact matches  :",
        physical_matches,
    )

    print(
        "  mismatches     :",
        physical_mismatches,
    )

    print(
        "  header hits    :",
        physical_header_hits,
    )

    print(
        "  exact          :",
        physical_exact,
    )


    print()
    print("POST-HEADER EFFECTIVE candidate")
    print(
        "  in range       :",
        effective_in_range,
    )

    print(
        "  exact matches  :",
        effective_matches,
    )

    print(
        "  mismatches     :",
        effective_mismatches,
    )

    print(
        "  exact          :",
        effective_exact,
    )


    print()
    print(
        "Retained locator ∩ embedded-header positions:",
        header_locator_intersection.tolist(),
    )


    # =========================================================================
    # F. RESOLVE DAY
    # =========================================================================

    if (
        physical_exact
        and not effective_exact
    ):

        resolved = (
            "PHYSICAL_ZERO_BASED_CSV_DATA_ROW_INDEX"
        )


    elif (
        effective_exact
        and not physical_exact
    ):

        resolved = (
            "POST_EMBEDDED_HEADER_EFFECTIVE_ROW_INDEX"
        )


    elif (
        physical_exact
        and effective_exact
    ):

        resolved = (
            "AMBIGUOUS_BOTH_EXACT"
        )


    else:

        resolved = (
            "NEITHER_MAPPING_EXACT"
        )


    print()
    print(
        "[DAY RESOLUTION]:",
        resolved,
    )

    print()


    results.append(
        {
            "day_id":
                day_id,

            "file":
                spec[
                    "raw_file"
                ],

            "embedded_headers":
                int(
                    len(
                        embedded_positions
                    )
                ),

            "physical_exact":
                bool(
                    physical_exact
                ),

            "physical_mismatches":
                physical_mismatches,

            "physical_header_hits":
                physical_header_hits,

            "effective_exact":
                bool(
                    effective_exact
                ),

            "effective_mismatches":
                effective_mismatches,

            "resolution":
                resolved,
        }
    )


    del (
        table,
        original_index,
        cache_binary,
        raw_df,
        raw_label_series,
        raw_labels,
        raw_binary,
        embedded_positions,
        traffic_mask,
        effective_binary,
    )

    if physical_in_range:

        del (
            physical_source_binary,
            physical_valid,
            physical_mismatch_mask,
        )


    if effective_in_range:

        del (
            effective_source_binary,
            effective_mismatch_mask,
        )


    gc.collect()


# =============================================================================
# 4. GLOBAL SEMANTICS
#
# At least one embedded-header day must be decisive.
# Ambiguous days are allowed only if another day resolves the common
# ingestion convention and no day contradicts it.
# =============================================================================

decisive = [
    row
    for row in results
    if row[
        "resolution"
    ]
    in {
        "PHYSICAL_ZERO_BASED_CSV_DATA_ROW_INDEX",
        "POST_EMBEDDED_HEADER_EFFECTIVE_ROW_INDEX",
    }
]


if not decisive:

    raise RuntimeError(
        "Neither embedded-header day decisively resolves "
        "original_row_index semantics.\n"
        "Do NOT proceed to attack-family recovery."
    )


decisive_modes = {
    row[
        "resolution"
    ]
    for row in decisive
}


if len(
    decisive_modes
) != 1:

    raise RuntimeError(
        "Embedded-header days contradict one another:\n"
        + repr(
            decisive_modes
        )
    )


global_mode = next(
    iter(
        decisive_modes
    )
)


for row in results:

    if (
        row[
            "resolution"
        ]
        == "NEITHER_MAPPING_EXACT"
    ):

        raise RuntimeError(
            f"{row['file']}: neither mapping is exact."
        )


    if (
        row[
            "resolution"
        ]
        not in {
            global_mode,
            "AMBIGUOUS_BOTH_EXACT",
        }
    ):

        raise RuntimeError(
            f"{row['file']}: mapping contradicts global resolution."
        )


# =============================================================================
# 5. FINAL
# =============================================================================

print("=" * 110)
print("STAGE23-6B2 original_row_index SEMANTICS RESOLVED")
print("=" * 110)
print()

print(
    "GLOBAL MAPPING:",
    global_mode,
)

print()
print("Embedded-header evidence:")


for row in results:

    print(
        f"  day_id={row['day_id']} "
        f"{row['file']:<14} "
        f"headers={row['embedded_headers']:<3} "
        f"physical_mismatch={row['physical_mismatches']} "
        f"effective_mismatch={row['effective_mismatches']} "
        f"resolution={row['resolution']}"
    )


print()
print("Governance:")
print("  Model fits            : 0")
print("  Model inference       : 0")
print("  LightGBM execution    : 0")
print("  XGBoost execution     : 0")
print("  Attack-family metrics : 0")
print("  Scientific writes     : 0")
print("  Raw Mar1 accessed     : NO")
print("  Raw Mar2 accessed     : NO")

print()
print("NEXT:")
print("  Use this exact mapping to recover categorical Labels")
print("  for the two frozen validation memberships.")
print("  Then require 100% categorical-source Label ->")
print("  Stage22R binary_label agreement BEFORE calculating")
print("  any family-specific model metric.")
print("  SEALED probability artifacts only; NO MODEL EXECUTION.")

print("=" * 110)

STAGE23-6B2 — DECISIVE original_row_index SEMANTICS RESOLUTION

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] model use : NONE

FULL RETAINED-ROW ALIGNMENT TEST — EMBEDDED-HEADER DAYS ONLY

--------------------------------------------------------------------------------------------------------------
day_id=2  02-16-2018.csv
--------------------------------------------------------------------------------------------------------------
Raw CSV data rows          : 1,048,575
Stage22R retained rows      : 900,988
Embedded headers            : 1
Embedded physical positions : [999999]
original_row_index min/max  : 0 / 999998

PHYSICAL candidate
  in range       : True
  exact matches  : 900988
  mismatches     : 0
  header hits    : 0
  exact          : True

POST-HEADER EFFECTIVE candidate
  in range       : True
  exact matches  : 900988
  mismatches     : 0
 

In [6]:
# =============================================================================
# STAGE23-6C — EXACT DEVELOPMENT ATTACK-FAMILY ALIGNMENT MATERIALIZATION
#
# PURPOSE
#   Recover original categorical CIC-IDS2018 Label values using the now-proven:
#
#       day_id + PHYSICAL zero-based original_row_index
#
#   Establish the attack-family universe from ALL K79-retained development
#   ATTACK rows, independent of Stage23 model results.
#
#   Verify 100% source Label -> binary_label agreement for EVERY row that
#   can participate in frozen attack-family evaluation:
#
#       RANDOM_NATURAL validation        2,882,481 rows
#       CHRONOLOGICAL_NATURAL validation   593,780 rows
#
#   Also verify all 1,972,299 retained development attack locators map to
#   a non-BENIGN, non-header categorical Label.
#
# OUTPUT
#   - retained attack locator -> categorical family Parquet
#   - frozen family/support table
#   - alignment manifest/state/checksums
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM EXECUTION
# ZERO XGBOOST EXECUTION
# ZERO ATTACK-FAMILY MODEL METRICS
# NO MAR1 / MAR2
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)

CACHE = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

RAW = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/"
    "ids-intrusion-csv"
)

MEMBERSHIP_ROOT = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

RANDOM_VALIDATION_PACKBITS = (
    MEMBERSHIP_ROOT
    / "random_validation.packbits"
)

MEMBERSHIP_SUMMARY_PATH = (
    MEMBERSHIP_ROOT
    / "stage22r_1b1_membership_summary.json"
)

CLEANING_BY_DAY = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
    / "stage22r_k79_cleaning_by_day.csv"
)

ATTACK_FAMILY_SPEC = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "attack_family_spec.json"
)

EXPECTED_RANDOM_VALIDATION_SHA = (
    "8a308aa5c28008895559a87ba2335a82"
    "eac69a4a3b99303d42d598f7afbe2fad"
)

EXPECTED_RANDOM_VALIDATION_POPULATION = 2_882_481

EXPECTED_CLEAN_ROWS = 14_412_403
EXPECTED_CLEAN_ATTACK = 1_972_299
EXPECTED_CLEAN_BENIGN = 12_440_104

EXPECTED_RANDOM_ROWS = 2_882_481
EXPECTED_RANDOM_ATTACK = 394_460
EXPECTED_RANDOM_BENIGN = 2_488_021

EXPECTED_CHRONO_ROWS = 593_780
EXPECTED_CHRONO_ATTACK = 62_256
EXPECTED_CHRONO_BENIGN = 531_524

CHUNK_ROWS = 250_000

OUT = Path(
    "/kaggle/working/stage23_6c_attack_family_alignment"
)

ATTACK_LABELS_PARQUET = (
    OUT
    / "retained_development_attack_family_labels.parquet"
)

SUPPORT_CSV = (
    OUT
    / "attack_family_support_by_split.csv"
)

MANIFEST_JSON = (
    OUT
    / "stage23_6c_alignment_manifest.json"
)

STATE_JSON = (
    OUT
    / "execution_state.json"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


DAY_FILES = [
    (0, "02-14-2018.csv", "day_00_02-14-2018.parquet"),
    (1, "02-15-2018.csv", "day_01_02-15-2018.parquet"),
    (2, "02-16-2018.csv", "day_02_02-16-2018.parquet"),
    (3, "02-20-2018.csv", "day_03_02-20-2018.parquet"),
    (4, "02-21-2018.csv", "day_04_02-21-2018.parquet"),
    (5, "02-22-2018.csv", "day_05_02-22-2018.parquet"),
    (6, "02-23-2018.csv", "day_06_02-23-2018.parquet"),
    (7, "02-28-2018.csv", "day_07_02-28-2018.parquet"),
]


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:
        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def normalize_label_array(values):

    return np.asarray(
        [
            str(x)
            .replace("\ufeff", "")
            .strip()

            for x in values
        ],
        dtype=object,
    )


def labels_to_binary(
    labels,
):

    out = np.empty(
        len(labels),
        dtype=np.int8,
    )

    for i, value in enumerate(
        labels
    ):

        text = (
            str(value)
            .replace("\ufeff", "")
            .strip()
        )

        upper = text.upper()

        if upper == "BENIGN":
            out[i] = 0

        elif upper == "LABEL":
            out[i] = -1

        else:
            out[i] = 1

    return out


def extract_target_labels(
    csv_path,
    target_original_indices,
    expected_physical_rows,
):

    """
    Stream ONLY Label through the authoritative source CSV.

    target_original_indices are proven PHYSICAL zero-based CSV data-row
    indices, excluding the top-level CSV header line.
    """

    targets = np.asarray(
        target_original_indices,
        dtype=np.int64,
    )

    if len(targets) == 0:

        return np.empty(
            0,
            dtype=object,
        )


    if not np.all(
        np.diff(
            targets
        ) > 0
    ):

        raise RuntimeError(
            f"{csv_path.name}: target indices are not strictly increasing."
        )


    if targets[0] < 0:

        raise RuntimeError(
            f"{csv_path.name}: negative target locator."
        )


    recovered = np.empty(
        len(targets),
        dtype=object,
    )

    filled = np.zeros(
        len(targets),
        dtype=bool,
    )

    physical_row_base = 0


    reader = pd.read_csv(
        csv_path,
        usecols=[
            "Label"
        ],
        dtype=str,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_filter=False,
        chunksize=CHUNK_ROWS,
        low_memory=False,
    )


    for chunk in reader:

        n = len(
            chunk
        )

        start = physical_row_base
        stop = start + n


        left = int(
            np.searchsorted(
                targets,
                start,
                side="left",
            )
        )

        right = int(
            np.searchsorted(
                targets,
                stop,
                side="left",
            )
        )


        if right > left:

            wanted = targets[
                left:right
            ]

            local = (
                wanted
                - start
            ).astype(
                np.int64
            )


            chunk_labels = normalize_label_array(
                chunk[
                    "Label"
                ].to_numpy(
                    dtype=object
                )
            )


            recovered[
                left:right
            ] = chunk_labels[
                local
            ]

            filled[
                left:right
            ] = True


        physical_row_base = stop


    if physical_row_base != int(
        expected_physical_rows
    ):

        raise RuntimeError(
            f"{csv_path.name}: physical row-count mismatch.\n"
            f"expected={expected_physical_rows}\n"
            f"actual={physical_row_base}"
        )


    if not np.all(
        filled
    ):

        missing = targets[
            ~filled
        ]

        raise RuntimeError(
            f"{csv_path.name}: failed to recover "
            f"{len(missing)} requested source rows.\n"
            f"first missing={missing[:10].tolist()}"
        )


    return recovered


# =============================================================================
# 2. SEALED REPOSITORY STATE
# =============================================================================

print("=" * 110)
print("STAGE23-6C — EXACT DEVELOPMENT ATTACK-FAMILY ALIGNMENT")
print("=" * 110)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 seal-tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-6C:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] model use : NONE")


# =============================================================================
# 3. FROZEN SPEC / MEMBERSHIP VERIFICATION
# =============================================================================

spec = json.loads(
    ATTACK_FAMILY_SPEC.read_text(
        encoding="utf-8"
    )
)

if spec[
    "no_retraining"
] is not True:

    raise RuntimeError(
        "Frozen attack-family spec unexpectedly permits retraining."
    )

if spec[
    "scope"
] != "development-only; never Mar1/Mar2":

    raise RuntimeError(
        "Frozen attack-family scope mismatch."
    )

if spec[
    "interpretation_minimum_attack_support"
] != 100:

    raise RuntimeError(
        "Frozen support threshold mismatch."
    )


membership_summary = json.loads(
    MEMBERSHIP_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)


if membership_summary[
    "clean_development"
][
    "rows"
] != EXPECTED_CLEAN_ROWS:

    raise RuntimeError(
        "Clean-development row count mismatch."
    )


if membership_summary[
    "clean_development"
][
    "attack"
] != EXPECTED_CLEAN_ATTACK:

    raise RuntimeError(
        "Clean-development attack count mismatch."
    )


random_validation_sha = sha256_file(
    RANDOM_VALIDATION_PACKBITS
)

print()
print("random_validation.packbits")
print(
    "  expected:",
    EXPECTED_RANDOM_VALIDATION_SHA,
)
print(
    "  actual:  ",
    random_validation_sha,
)


if (
    random_validation_sha
    != EXPECTED_RANDOM_VALIDATION_SHA
):

    raise RuntimeError(
        "Frozen random-validation membership SHA mismatch."
    )


print("  [EXACT]")


packed = np.fromfile(
    RANDOM_VALIDATION_PACKBITS,
    dtype=np.uint8,
)


random_validation = np.unpackbits(
    packed,
    bitorder="little",
)[:EXPECTED_CLEAN_ROWS].astype(
    bool,
    copy=False,
)


if len(
    random_validation
) != EXPECTED_CLEAN_ROWS:

    raise RuntimeError(
        "Random-validation logical length mismatch."
    )


if int(
    np.sum(
        random_validation
    )
) != EXPECTED_RANDOM_VALIDATION_POPULATION:

    raise RuntimeError(
        "Random-validation population mismatch."
    )


print(
    "[EXACT] RANDOM validation population:",
    f"{int(random_validation.sum()):,}",
)


# =============================================================================
# 4. FROZEN DAY SOURCE TABLE
# =============================================================================

cleaning = pd.read_csv(
    CLEANING_BY_DAY
)


if cleaning.shape[
    0
] != 8:

    raise RuntimeError(
        "Expected exactly eight development days."
    )


cleaning = cleaning.sort_values(
    "day_id"
).reset_index(
    drop=True
)


for expected_day, (
    day_id,
    raw_name,
    _
) in enumerate(
    DAY_FILES
):

    row = cleaning.iloc[
        expected_day
    ]

    if int(
        row[
            "day_id"
        ]
    ) != day_id:

        raise RuntimeError(
            "Frozen day-id order mismatch."
        )

    if str(
        row[
            "file"
        ]
    ) != raw_name:

        raise RuntimeError(
            f"Frozen source filename mismatch for day {day_id}."
        )


print("[EXACT] eight frozen development sources")


# =============================================================================
# 5. OUTPUT INITIALIZATION
# =============================================================================

if OUT.exists():

    raise RuntimeError(
        "Stage23-6C output directory already exists.\n"
        "Do NOT delete or rerun blindly:\n"
        f"{OUT}"
    )


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


state = {
    "stage":
        "Stage23-6C exact attack-family alignment",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "original_row_index_mapping":
        "PHYSICAL_ZERO_BASED_CSV_DATA_ROW_INDEX",

    "random_validation_membership_sha256":
        EXPECTED_RANDOM_VALIDATION_SHA,

    "days_completed":
        0,

    "model_fits":
        0,

    "model_inference":
        0,

    "lightgbm_execution":
        0,

    "xgboost_execution":
        0,

    "attack_family_metrics":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 6. PROCESS DAY BY DAY
#
# We write one temporary attack-family Parquet per day, then consolidate.
#
# Required source-label extraction universe per day:
#
#   ALL retained attacks
#   UNION RANDOM validation
#   UNION CHRONO validation (all day7 rows)
#
# Thus:
#   - family universe is established from ALL retained development attacks
#   - EVERY RANDOM validation row gets exact source-binary verification
#   - EVERY CHRONO validation row gets exact source-binary verification
# =============================================================================

per_day_attack_files = []

random_rows_total = 0
random_attack_total = 0
random_benign_total = 0

chrono_rows_total = 0
chrono_attack_total = 0
chrono_benign_total = 0

retained_attack_total = 0

random_family_counts = {}
chrono_family_counts = {}
development_family_counts = {}

source_audit = []


for day_id, raw_name, cache_name in DAY_FILES:

    print()
    print("=" * 110)
    print(
        f"DAY {day_id} — {raw_name}"
    )
    print("=" * 110)
    print()


    raw_path = (
        RAW
        / raw_name
    )

    cache_path = (
        CACHE
        / cache_name
    )


    if not raw_path.is_file():

        raise RuntimeError(
            f"Missing authoritative source:\n{raw_path}"
        )


    if not cache_path.is_file():

        raise RuntimeError(
            f"Missing authorized Stage22R cache:\n{cache_path}"
        )


    frozen_day = cleaning.loc[
        cleaning[
            "day_id"
        ] == day_id
    ].iloc[
        0
    ]


    expected_physical_rows = int(
        frozen_day[
            "physical_rows"
        ]
    )

    expected_retained_rows = int(
        frozen_day[
            "retained_rows"
        ]
    )

    expected_retained_attack = int(
        frozen_day[
            "retained_attack"
        ]
    )

    expected_retained_benign = int(
        frozen_day[
            "retained_benign"
        ]
    )

    expected_embedded = int(
        frozen_day[
            "embedded_header_rows"
        ]
    )


    # -------------------------------------------------------------------------
    # Stage22R metadata only.
    # -------------------------------------------------------------------------

    table = pq.read_table(
        cache_path,
        columns=[
            "clean_position",
            "day_id",
            "original_row_index",
            "binary_label",
        ],
    )


    clean_position = np.asarray(
        table[
            "clean_position"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )

    parquet_day_id = np.asarray(
        table[
            "day_id"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int16,
    )

    original_row_index = np.asarray(
        table[
            "original_row_index"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int64,
    )

    binary_label = np.asarray(
        table[
            "binary_label"
        ].to_numpy(
            zero_copy_only=False
        ),
        dtype=np.int8,
    )


    n = len(
        clean_position
    )


    if n != expected_retained_rows:

        raise RuntimeError(
            f"{raw_name}: retained-row count mismatch."
        )


    if not np.all(
        parquet_day_id
        == day_id
    ):

        raise RuntimeError(
            f"{raw_name}: day_id metadata mismatch."
        )


    if not np.all(
        np.diff(
            clean_position
        ) == 1
    ):

        raise RuntimeError(
            f"{raw_name}: clean_position not contiguous."
        )


    if not np.all(
        np.diff(
            original_row_index
        ) > 0
    ):

        raise RuntimeError(
            f"{raw_name}: original_row_index not strictly increasing."
        )


    actual_attack = int(
        np.sum(
            binary_label
            == 1
        )
    )

    actual_benign = int(
        np.sum(
            binary_label
            == 0
        )
    )


    if (
        actual_attack
        != expected_retained_attack
        or actual_benign
        != expected_retained_benign
    ):

        raise RuntimeError(
            f"{raw_name}: frozen class-count mismatch."
        )


    # -------------------------------------------------------------------------
    # Exact frozen validation memberships.
    # -------------------------------------------------------------------------

    random_mask = random_validation[
        clean_position
    ]


    chrono_mask = np.full(
        n,
        day_id == 7,
        dtype=bool,
    )


    attack_mask = (
        binary_label
        == 1
    )


    needed_mask = (
        attack_mask
        | random_mask
        | chrono_mask
    )


    target_original = np.unique(
        original_row_index[
            needed_mask
        ]
    )


    # -------------------------------------------------------------------------
    # Stream Label only and recover exact requested PHYSICAL positions.
    # -------------------------------------------------------------------------

    recovered_labels = extract_target_labels(
        raw_path,
        target_original,
        expected_physical_rows,
    )


    recovered_binary = labels_to_binary(
        recovered_labels
    )


    # Embedded-header positions may NEVER correspond to a retained locator.
    if np.any(
        recovered_binary
        == -1
    ):

        bad = target_original[
            recovered_binary
            == -1
        ]

        raise RuntimeError(
            f"{raw_name}: retained/validation locator points to "
            f"embedded header rows:\n{bad[:20].tolist()}"
        )


    # -------------------------------------------------------------------------
    # Helper to map arbitrary selected retained rows into recovered labels.
    # -------------------------------------------------------------------------

    def labels_for_mask(mask):

        selected_original = original_row_index[
            mask
        ]

        pos = np.searchsorted(
            target_original,
            selected_original,
        )


        if np.any(
            pos
            >= len(
                target_original
            )
        ):

            raise RuntimeError(
                f"{raw_name}: source-label search overflow."
            )


        if not np.array_equal(
            target_original[
                pos
            ],
            selected_original,
        ):

            raise RuntimeError(
                f"{raw_name}: exact locator lookup failed."
            )


        return recovered_labels[
            pos
        ]


    # -------------------------------------------------------------------------
    # Verify ALL RANDOM validation rows.
    # -------------------------------------------------------------------------

    random_labels = labels_for_mask(
        random_mask
    )

    random_source_binary = labels_to_binary(
        random_labels
    )

    random_cache_binary = binary_label[
        random_mask
    ]


    random_mismatches = int(
        np.sum(
            random_source_binary
            != random_cache_binary
        )
    )


    if random_mismatches != 0:

        raise RuntimeError(
            f"{raw_name}: RANDOM validation Label->binary mismatch "
            f"count={random_mismatches}"
        )


    day_random_rows = int(
        np.sum(
            random_mask
        )
    )

    day_random_attack = int(
        np.sum(
            random_cache_binary
            == 1
        )
    )

    day_random_benign = int(
        np.sum(
            random_cache_binary
            == 0
        )
    )


    random_rows_total += day_random_rows
    random_attack_total += day_random_attack
    random_benign_total += day_random_benign


    # -------------------------------------------------------------------------
    # Verify ALL CHRONO validation rows (day 7 only).
    # -------------------------------------------------------------------------

    if day_id == 7:

        chrono_labels = labels_for_mask(
            chrono_mask
        )

        chrono_source_binary = labels_to_binary(
            chrono_labels
        )

        chrono_cache_binary = binary_label[
            chrono_mask
        ]


        chrono_mismatches = int(
            np.sum(
                chrono_source_binary
                != chrono_cache_binary
            )
        )


        if chrono_mismatches != 0:

            raise RuntimeError(
                f"{raw_name}: CHRONO validation Label->binary mismatch "
                f"count={chrono_mismatches}"
            )


        day_chrono_rows = int(
            np.sum(
                chrono_mask
            )
        )

        day_chrono_attack = int(
            np.sum(
                chrono_cache_binary
                == 1
            )
        )

        day_chrono_benign = int(
            np.sum(
                chrono_cache_binary
                == 0
            )
        )


        chrono_rows_total += day_chrono_rows
        chrono_attack_total += day_chrono_attack
        chrono_benign_total += day_chrono_benign


    else:

        chrono_mismatches = 0
        day_chrono_rows = 0
        day_chrono_attack = 0
        day_chrono_benign = 0


    # -------------------------------------------------------------------------
    # Recover ALL retained development attack families.
    # -------------------------------------------------------------------------

    attack_labels = labels_for_mask(
        attack_mask
    )


    attack_source_binary = labels_to_binary(
        attack_labels
    )


    if not np.all(
        attack_source_binary
        == 1
    ):

        raise RuntimeError(
            f"{raw_name}: retained attack locator recovered as "
            "BENIGN/header in source Label."
        )


    day_attack_count = len(
        attack_labels
    )


    if day_attack_count != expected_retained_attack:

        raise RuntimeError(
            f"{raw_name}: retained attack-family count mismatch."
        )


    retained_attack_total += day_attack_count


    # -------------------------------------------------------------------------
    # Family counts — full clean development.
    # -------------------------------------------------------------------------

    unique_family, unique_count = np.unique(
        attack_labels,
        return_counts=True,
    )


    for family, count in zip(
        unique_family,
        unique_count,
    ):

        family = str(
            family
        )

        development_family_counts[
            family
        ] = (
            development_family_counts.get(
                family,
                0,
            )
            + int(
                count
            )
        )


    # -------------------------------------------------------------------------
    # Family counts — RANDOM validation attacks.
    # -------------------------------------------------------------------------

    random_attack_labels = random_labels[
        random_cache_binary
        == 1
    ]


    unique_family, unique_count = np.unique(
        random_attack_labels,
        return_counts=True,
    )


    for family, count in zip(
        unique_family,
        unique_count,
    ):

        family = str(
            family
        )

        random_family_counts[
            family
        ] = (
            random_family_counts.get(
                family,
                0,
            )
            + int(
                count
            )
        )


    # -------------------------------------------------------------------------
    # Family counts — CHRONO validation attacks.
    # -------------------------------------------------------------------------

    if day_id == 7:

        chrono_attack_labels = chrono_labels[
            chrono_cache_binary
            == 1
        ]


        unique_family, unique_count = np.unique(
            chrono_attack_labels,
            return_counts=True,
        )


        for family, count in zip(
            unique_family,
            unique_count,
        ):

            family = str(
                family
            )

            chrono_family_counts[
                family
            ] = (
                chrono_family_counts.get(
                    family,
                    0,
                )
                + int(
                    count
                )
            )


    # -------------------------------------------------------------------------
    # Persist ALL retained attack locators + categorical family.
    # -------------------------------------------------------------------------

    attack_df = pd.DataFrame(
        {
            "clean_position":
                clean_position[
                    attack_mask
                ].astype(
                    np.int64
                ),

            "day_id":
                np.full(
                    day_attack_count,
                    day_id,
                    dtype=np.uint8,
                ),

            "original_row_index":
                original_row_index[
                    attack_mask
                ].astype(
                    np.int64
                ),

            "attack_family":
                attack_labels,
        }
    )


    day_output = (
        OUT
        / (
            f"day_{day_id:02d}_"
            "retained_attack_family_labels.parquet"
        )
    )


    attack_df.to_parquet(
        day_output,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )


    per_day_attack_files.append(
        day_output
    )


    print(
        "retained rows           :",
        f"{n:,}",
    )

    print(
        "retained attacks        :",
        f"{day_attack_count:,}",
    )

    print(
        "RANDOM validation       :",
        f"{day_random_rows:,}",
        f"(attack={day_random_attack:,}, benign={day_random_benign:,})",
    )

    print(
        "RANDOM binary mismatches:",
        random_mismatches,
    )

    if day_id == 7:

        print(
            "CHRONO validation       :",
            f"{day_chrono_rows:,}",
            f"(attack={day_chrono_attack:,}, benign={day_chrono_benign:,})",
        )

        print(
            "CHRONO binary mismatches:",
            chrono_mismatches,
        )


    print(
        "families on retained attacks:",
        len(
            np.unique(
                attack_labels
            )
        ),
    )

    print(
        "[EXACT] source alignment passed"
    )


    source_audit.append(
        {
            "day_id":
                day_id,

            "source_file":
                raw_name,

            "cache_file":
                cache_name,

            "physical_rows":
                expected_physical_rows,

            "embedded_headers":
                expected_embedded,

            "retained_rows":
                n,

            "retained_attack":
                day_attack_count,

            "retained_benign":
                actual_benign,

            "random_validation_rows":
                day_random_rows,

            "random_validation_attack":
                day_random_attack,

            "random_validation_benign":
                day_random_benign,

            "random_label_binary_mismatches":
                random_mismatches,

            "chronological_validation_rows":
                day_chrono_rows,

            "chronological_validation_attack":
                day_chrono_attack,

            "chronological_validation_benign":
                day_chrono_benign,

            "chronological_label_binary_mismatches":
                chrono_mismatches,

            "attack_family_output":
                day_output.name,

            "attack_family_output_sha256":
                sha256_file(
                    day_output
                ),
        }
    )


    state[
        "days_completed"
    ] = day_id + 1

    state[
        "status"
    ] = "ALIGNMENT_IN_PROGRESS"

    state[
        "last_completed_day"
    ] = day_id

    write_json(
        STATE_JSON,
        state,
    )


    del (
        table,
        clean_position,
        parquet_day_id,
        original_row_index,
        binary_label,
        random_mask,
        chrono_mask,
        attack_mask,
        needed_mask,
        target_original,
        recovered_labels,
        recovered_binary,
        random_labels,
        random_source_binary,
        random_cache_binary,
        random_attack_labels,
        attack_labels,
        attack_source_binary,
        attack_df,
    )


    if day_id == 7:

        del (
            chrono_labels,
            chrono_source_binary,
            chrono_cache_binary,
            chrono_attack_labels,
        )


    gc.collect()


# =============================================================================
# 7. GLOBAL COUNT VERIFICATION
# =============================================================================

print()
print("=" * 110)
print("GLOBAL ALIGNMENT COUNT VERIFICATION")
print("=" * 110)
print()


if retained_attack_total != EXPECTED_CLEAN_ATTACK:

    raise RuntimeError(
        "Total retained development attack count mismatch.\n"
        f"expected={EXPECTED_CLEAN_ATTACK}\n"
        f"actual={retained_attack_total}"
    )


if random_rows_total != EXPECTED_RANDOM_ROWS:

    raise RuntimeError(
        "RANDOM validation row count mismatch."
    )


if random_attack_total != EXPECTED_RANDOM_ATTACK:

    raise RuntimeError(
        "RANDOM validation attack count mismatch."
    )


if random_benign_total != EXPECTED_RANDOM_BENIGN:

    raise RuntimeError(
        "RANDOM validation benign count mismatch."
    )


if chrono_rows_total != EXPECTED_CHRONO_ROWS:

    raise RuntimeError(
        "CHRONO validation row count mismatch."
    )


if chrono_attack_total != EXPECTED_CHRONO_ATTACK:

    raise RuntimeError(
        "CHRONO validation attack count mismatch."
    )


if chrono_benign_total != EXPECTED_CHRONO_BENIGN:

    raise RuntimeError(
        "CHRONO validation benign count mismatch."
    )


print(
    "[EXACT] clean development attacks:",
    f"{retained_attack_total:,}",
)

print(
    "[EXACT] RANDOM validation:",
    f"{random_rows_total:,}",
    f"attack={random_attack_total:,}",
    f"benign={random_benign_total:,}",
)

print(
    "[EXACT] CHRONO validation:",
    f"{chrono_rows_total:,}",
    f"attack={chrono_attack_total:,}",
    f"benign={chrono_benign_total:,}",
)


# =============================================================================
# 8. CONSOLIDATE RETAINED ATTACK-FAMILY ARTIFACT
# =============================================================================

print()
print("=" * 110)
print("CONSOLIDATE RETAINED ATTACK-FAMILY LOCATORS")
print("=" * 110)
print()


tables = [
    pq.read_table(
        p
    )
    for p in per_day_attack_files
]


combined = pa.concat_tables(
    tables,
    promote_options="none",
)


if combined.num_rows != EXPECTED_CLEAN_ATTACK:

    raise RuntimeError(
        "Consolidated retained-attack row count mismatch."
    )


pq.write_table(
    combined,
    ATTACK_LABELS_PARQUET,
    compression="zstd",
)


consolidated_df = combined.to_pandas()


if not np.all(
    np.diff(
        consolidated_df[
            "clean_position"
        ].to_numpy(
            dtype=np.int64
        )
    ) > 0
):

    raise RuntimeError(
        "Consolidated clean_position is not strictly increasing."
    )


if consolidated_df[
    "clean_position"
].duplicated().any():

    raise RuntimeError(
        "Duplicate clean_position in attack-family alignment."
    )


if consolidated_df[
    [
        "day_id",
        "original_row_index",
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate day_id + original_row_index locator."
    )


print(
    "[EXACT] consolidated attack rows:",
    f"{len(consolidated_df):,}",
)

print(
    "Artifact:",
    ATTACK_LABELS_PARQUET,
)

print(
    "SHA256 :",
    sha256_file(
        ATTACK_LABELS_PARQUET
    ),
)


# =============================================================================
# 9. FAMILY SET + SUPPORT TABLE
#
# IMPORTANT:
#   Family universe = ALL categorical families observed among K79-retained
#   development ATTACK rows.
#
#   It is NOT selected from Stage23 performance.
# =============================================================================

families = sorted(
    development_family_counts.keys()
)


if not families:

    raise RuntimeError(
        "No retained development attack families recovered."
    )


support_rows = []


for family in families:

    development_support = int(
        development_family_counts.get(
            family,
            0,
        )
    )

    random_support = int(
        random_family_counts.get(
            family,
            0,
        )
    )

    chrono_support = int(
        chrono_family_counts.get(
            family,
            0,
        )
    )


    support_rows.append(
        {
            "attack_family":
                family,

            "development_retained_attack_support":
                development_support,

            "random_natural_validation_attack_support":
                random_support,

            "random_interpretable_ge_100":
                bool(
                    random_support
                    >= 100
                ),

            "chronological_natural_validation_attack_support":
                chrono_support,

            "chronological_interpretable_ge_100":
                bool(
                    chrono_support
                    >= 100
                ),
        }
    )


support_df = pd.DataFrame(
    support_rows
)


if int(
    support_df[
        "development_retained_attack_support"
    ].sum()
) != EXPECTED_CLEAN_ATTACK:

    raise RuntimeError(
        "Development family-support sum mismatch."
    )


if int(
    support_df[
        "random_natural_validation_attack_support"
    ].sum()
) != EXPECTED_RANDOM_ATTACK:

    raise RuntimeError(
        "RANDOM family-support sum mismatch."
    )


if int(
    support_df[
        "chronological_natural_validation_attack_support"
    ].sum()
) != EXPECTED_CHRONO_ATTACK:

    raise RuntimeError(
        "CHRONO family-support sum mismatch."
    )


support_df.to_csv(
    SUPPORT_CSV,
    index=False,
)


print()
print("=" * 110)
print("FROZEN DEVELOPMENT ATTACK-FAMILY SET")
print("=" * 110)
print()


print(
    "Attack families:",
    len(
        support_df
    ),
)

print()


for row in support_df.itertuples(
    index=False
):

    print(
        f"{row.attack_family:<32} "
        f"DEV={row.development_retained_attack_support:>8,}  "
        f"RANDOM={row.random_natural_validation_attack_support:>7,} "
        f"{'INTERPRETABLE' if row.random_interpretable_ge_100 else 'DESCRIPTIVE':<13}  "
        f"CHRONO={row.chronological_natural_validation_attack_support:>7,} "
        f"{'INTERPRETABLE' if row.chronological_interpretable_ge_100 else 'DESCRIPTIVE'}"
    )


# =============================================================================
# 10. REMOVE PER-DAY TEMPORARY FILES ONLY AFTER CONSOLIDATION VERIFIED
# =============================================================================

del (
    combined,
    consolidated_df,
)

for table in tables:
    del table

gc.collect()


for p in per_day_attack_files:

    p.unlink()


# =============================================================================
# 11. MANIFEST
# =============================================================================

manifest = {
    "stage":
        "Stage23-6C",

    "status":
        "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED_UNSEALED",

    "completed_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "frozen_attack_family_spec":
        str(
            ATTACK_FAMILY_SPEC.relative_to(
                REPO
            )
        ),

    "mapping": {
        "key":
            [
                "day_id",
                "original_row_index",
            ],

        "original_row_index_semantics":
            "PHYSICAL_ZERO_BASED_CSV_DATA_ROW_INDEX",

        "source_label_column":
            "Label",

        "source_files":
            [
                raw_name
                for _, raw_name, _
                in DAY_FILES
            ],

        "family_universe":
            (
                "All distinct non-BENIGN categorical Label values "
                "among all 1,972,299 K79-retained development attack rows."
            ),

        "family_selection_uses_model_results":
            False,
    },

    "verification": {
        "retained_development_attack_rows":
            retained_attack_total,

        "random_validation_rows_verified":
            random_rows_total,

        "random_validation_attack":
            random_attack_total,

        "random_validation_benign":
            random_benign_total,

        "random_label_to_binary_mismatches":
            0,

        "chronological_validation_rows_verified":
            chrono_rows_total,

        "chronological_validation_attack":
            chrono_attack_total,

        "chronological_validation_benign":
            chrono_benign_total,

        "chronological_label_to_binary_mismatches":
            0,

        "attack_family_count":
            len(
                families
            ),

        "minimum_support_for_interpretation":
            100,
    },

    "source_audit":
        source_audit,

    "artifacts": {
        ATTACK_LABELS_PARQUET.name: {
            "rows":
                EXPECTED_CLEAN_ATTACK,

            "sha256":
                sha256_file(
                    ATTACK_LABELS_PARQUET
                ),
        },

        SUPPORT_CSV.name: {
            "rows":
                len(
                    support_df
                ),

            "sha256":
                sha256_file(
                    SUPPORT_CSV
                ),
        },
    },

    "governance": {
        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_execution":
            0,

        "xgboost_execution":
            0,

        "attack_family_model_metrics":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "retraining":
            False,
    },

    "next":
        (
            "Stage23-6D: join this frozen categorical family alignment "
            "to sealed RANDOM_NATURAL and CHRONOLOGICAL_NATURAL "
            "validation probability artifacts and compute the "
            "pre-registered attack-family metrics. "
            "NO MODEL EXECUTION."
        ),
}


write_json(
    MANIFEST_JSON,
    manifest,
)


state[
    "status"
] = (
    "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "days_completed"
] = 8

state[
    "retained_development_attack_rows"
] = EXPECTED_CLEAN_ATTACK

state[
    "random_validation_rows_verified"
] = EXPECTED_RANDOM_ROWS

state[
    "chronological_validation_rows_verified"
] = EXPECTED_CHRONO_ROWS

state[
    "attack_family_count"
] = len(
    families
)

state[
    "next_action"
] = (
    "STAGE23_6D_SEALED_PROBABILITY_ATTACK_FAMILY_METRICS"
)


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 12. CHECKSUMS
# =============================================================================

artifact_files = sorted(
    p
    for p in OUT.iterdir()
    if (
        p.is_file()
        and p != CHECKSUMS
    )
)


CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(p)}  {p.name}"
        )

        for p
        in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 13. FINAL
# =============================================================================

print()
print("=" * 110)
print("STAGE23-6C ATTACK-FAMILY ALIGNMENT COMPLETE — UNSEALED")
print("=" * 110)
print()

print(
    "Development attack families :",
    len(
        families
    ),
)

print(
    "Retained development attacks:",
    f"{retained_attack_total:,}",
)

print()
print("RANDOM_NATURAL")
print(
    "  validation rows          :",
    f"{random_rows_total:,}",
)
print(
    "  attack                   :",
    f"{random_attack_total:,}",
)
print(
    "  benign                   :",
    f"{random_benign_total:,}",
)
print(
    "  Label→binary mismatches  : 0"
)

print()
print("CHRONOLOGICAL_NATURAL")
print(
    "  validation rows          :",
    f"{chrono_rows_total:,}",
)
print(
    "  attack                   :",
    f"{chrono_attack_total:,}",
)
print(
    "  benign                   :",
    f"{chrono_benign_total:,}",
)
print(
    "  Label→binary mismatches  : 0"
)

print()
print("Attack-family locator artifact:")
print(
    " ",
    ATTACK_LABELS_PARQUET,
)
print(" SHA256:")
print(
    " ",
    sha256_file(
        ATTACK_LABELS_PARQUET
    ),
)

print()
print("Support table:")
print(
    " ",
    SUPPORT_CSV,
)
print(" SHA256:")
print(
    " ",
    sha256_file(
        SUPPORT_CSV
    ),
)

print()
print("Alignment manifest:")
print(
    " ",
    MANIFEST_JSON,
)
print(" SHA256:")
print(
    " ",
    sha256_file(
        MANIFEST_JSON
    ),
)

print()
print("Checksum manifest SHA256:")
print(
    " ",
    sha256_file(
        CHECKSUMS
    ),
)

print()
print("GOVERNANCE")
print("  Model fits             : 0")
print("  Model inference        : 0")
print("  LightGBM execution     : 0")
print("  XGBoost execution      : 0")
print("  Attack-family metrics  : 0")
print("  Stage23 fit state      : 50 / 50 SEALED")
print("  Raw Mar1 accessed      : NO")
print("  Raw Mar2 accessed      : NO")

print()
print("NEXT:")
print("  Stage23-6D — compute frozen attack-family metrics")
print("  from SEALED validation probabilities only.")
print("  NO LightGBM/XGBoost execution.")

print("=" * 110)

STAGE23-6C — EXACT DEVELOPMENT ATTACK-FAMILY ALIGNMENT

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] model use : NONE

random_validation.packbits
  expected: 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
  actual:   8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
  [EXACT]
[EXACT] RANDOM validation population: 2,882,481
[EXACT] eight frozen development sources

DAY 0 — 02-14-2018.csv

retained rows           : 822,947
retained attacks        : 156,674
RANDOM validation       : 164,375 (attack=31,231, benign=133,144)
RANDOM binary mismatches: 0
families on retained attacks: 2
[EXACT] source alignment passed

DAY 1 — 02-15-2018.csv

retained rows           : 1,046,154
retained attacks        : 51,740
RANDOM validation       : 209,432 (attack=10,345, benign=199,087)
RANDOM binary mismatches: 0
families on retained attacks

In [7]:
# =============================================================================
# STAGE23-6D — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS
#
# INPUTS
#   Stage23-6C exact categorical family alignment
#   Frozen Stage22R FULL validation probability vectors
#   Frozen Stage23 PRIMARY ablation validation probability vectors
#
# PRIMARY SUBSETS
#   FULL
#   NO_DST_PORT
#   NO_PORTS
#   NO_INIT_FWD_WIN_BYTS
#   NO_FWD_SEG_SIZE_MIN
#   NO_SUSPICIOUS_GROUP
#   BEHAVIOR_ONLY
#
# SPLITS
#   RANDOM_NATURAL
#   CHRONOLOGICAL_NATURAL
#
# EVALUATION FOR FAMILY A
#   ALL benign validation rows
#   + attack validation rows belonging to family A
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM IMPORT / EXECUTION
# ZERO XGBOOST IMPORT / EXECUTION
# ZERO PARQUET DEVELOPMENT-SOURCE READS
# ZERO RAW CSV READS
# NO MAR1 / MAR2
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import math
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)


ALIGN_ROOT = Path(
    "/kaggle/working/stage23_6c_attack_family_alignment"
)

ALIGN_PARQUET = (
    ALIGN_ROOT
    / "retained_development_attack_family_labels.parquet"
)

ALIGN_SUPPORT = (
    ALIGN_ROOT
    / "attack_family_support_by_split.csv"
)

ALIGN_MANIFEST = (
    ALIGN_ROOT
    / "stage23_6c_alignment_manifest.json"
)

ALIGN_CHECKSUMS = (
    ALIGN_ROOT
    / "checksums.sha256"
)


EXPECTED_ALIGN_PARQUET_SHA = (
    "3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26"
)

EXPECTED_ALIGN_SUPPORT_SHA = (
    "d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377"
)

EXPECTED_ALIGN_MANIFEST_SHA = (
    "bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8"
)

EXPECTED_ALIGN_CHECKSUMS_SHA = (
    "a54e01b733e093602a03b9b8bff5430f7cdfa4e987e0a36577ee686e736113b4"
)


EXPECTED_FULL_RANDOM_SHA = (
    "9dc64ccdb6580dad727668ff885c010dde5725fd1bfb1816d07183eacda6e5b02"
)

EXPECTED_FULL_CHRONO_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)


EXPECTED_FULL_METRICS = {
    "RANDOM_NATURAL": {
        "PR_AUC":
            0.9955900418992819,

        "ROC_AUC":
            0.9986245647735994,
    },

    "CHRONOLOGICAL_NATURAL": {
        "PR_AUC":
            0.10621515513397227,

        "ROC_AUC":
            0.5149184263937692,
    },
}


EXPECTED_SPLIT_COUNTS = {
    "RANDOM_NATURAL": {
        "rows": 2_882_481,
        "attack": 394_460,
        "benign": 2_488_021,
    },

    "CHRONOLOGICAL_NATURAL": {
        "rows": 593_780,
        "attack": 62_256,
        "benign": 531_524,
    },
}


MIN_INTERPRETATION_SUPPORT = 100

THRESHOLD = np.float32(
    0.50
)


SUBSET_ORDER = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]


SUBSET_SLUG = {
    "NO_DST_PORT":
        "no_dst_port",

    "NO_PORTS":
        "no_ports",

    "NO_INIT_FWD_WIN_BYTS":
        "no_init_fwd_win_byts",

    "NO_FWD_SEG_SIZE_MIN":
        "no_fwd_seg_size_min",

    "NO_SUSPICIOUS_GROUP":
        "no_suspicious_group",

    "BEHAVIOR_ONLY":
        "behavior_only",
}


OUT = Path(
    "/kaggle/working/stage23_6d_attack_family_metrics"
)

METRICS_CSV = (
    OUT
    / "stage23_6d_attack_family_metrics.csv"
)

METRICS_JSON = (
    OUT
    / "stage23_6d_attack_family_metrics.json"
)

SUMMARY_CSV = (
    OUT
    / "stage23_6d_primary_delta_summary.csv"
)

MANIFEST_JSON = (
    OUT
    / "stage23_6d_manifest.json"
)

STATE_JSON = (
    OUT
    / "execution_state.json"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def json_safe_value(v):

    if v is None:
        return None

    if isinstance(
        v,
        (
            np.bool_,
            bool,
        ),
    ):
        return bool(v)

    if isinstance(
        v,
        (
            np.integer,
            int,
        ),
    ):
        return int(v)

    if isinstance(
        v,
        (
            np.floating,
            float,
        ),
    ):

        f = float(v)

        if not math.isfinite(
            f
        ):
            return None

        return f

    return str(v)


def dataframe_records_json_safe(
    df,
):

    return [
        {
            k:
                json_safe_value(v)

            for k, v
            in row.items()
        }

        for row
        in df.to_dict(
            orient="records"
        )
    ]


def expected_sha_from_manifest(
    artifact_path,
):

    manifest_path = (
        artifact_path.parent
        / "checksums.sha256"
    )

    if not manifest_path.is_file():

        raise RuntimeError(
            "Sibling checksum manifest missing:\n"
            f"{manifest_path}"
        )


    matches = []


    for line in manifest_path.read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        rel = rel.strip()


        if Path(rel).name == artifact_path.name:

            matches.append(
                digest.strip()
            )


    if len(
        matches
    ) != 1:

        raise RuntimeError(
            "Could not uniquely resolve checksum for:\n"
            f"{artifact_path}\n"
            f"matches={len(matches)}"
        )


    return matches[
        0
    ]


def verify_probability_artifact(
    path,
    expected_sha=None,
):

    if not path.is_file():

        raise RuntimeError(
            f"Probability artifact missing:\n{path}"
        )


    if expected_sha is None:

        expected_sha = (
            expected_sha_from_manifest(
                path
            )
        )


    actual = sha256_file(
        path
    )


    if actual != expected_sha:

        raise RuntimeError(
            "Probability artifact SHA mismatch:\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual}"
        )


    return actual


def load_probability_npz(
    path,
):

    z = np.load(
        path,
        allow_pickle=False,
    )


    required = {
        "clean_position",
        "binary_label",
        "ensemble_probability",
    }


    if not required.issubset(
        set(
            z.files
        )
    ):

        raise RuntimeError(
            f"{path.name}: required NPZ keys absent.\n"
            f"keys={sorted(z.files)}"
        )


    clean = np.asarray(
        z[
            "clean_position"
        ],
        dtype=np.int64,
    )


    y = np.asarray(
        z[
            "binary_label"
        ],
        dtype=np.int8,
    )


    p_raw = z[
        "ensemble_probability"
    ]


    if p_raw.dtype != np.float32:

        raise RuntimeError(
            f"{path.name}: ensemble_probability is not float32.\n"
            f"dtype={p_raw.dtype}"
        )


    probability = np.asarray(
        p_raw,
        dtype=np.float32,
    )


    z.close()


    if not (
        len(clean)
        == len(y)
        == len(probability)
    ):

        raise RuntimeError(
            f"{path.name}: NPZ vector-length mismatch."
        )


    if not np.all(
        np.isfinite(
            probability
        )
    ):

        raise RuntimeError(
            f"{path.name}: non-finite probabilities."
        )


    if (
        probability.min()
        < 0.0
        or probability.max()
        > 1.0
    ):

        raise RuntimeError(
            f"{path.name}: probability outside [0,1]."
        )


    if not np.all(
        np.isin(
            y,
            [
                0,
                1,
            ],
        )
    ):

        raise RuntimeError(
            f"{path.name}: non-binary labels."
        )


    return (
        clean,
        y,
        probability,
    )


# =============================================================================
# 2. FAST EXACT BINARY METRICS
#
# Family A evaluation contains:
#   every benign score
#   + positive scores for family A
#
# We sort benign scores ONCE per subset.
#
# PR_AUC definition matches sklearn average_precision_score:
#
#   sum over unique positive-score thresholds:
#       delta_recall * precision
#
# ROC_AUC is exact pairwise ranking:
#   P(score_positive > score_negative)
# + 0.5 * P(tie)
# =============================================================================

def fast_family_ranking_metrics(
    benign_sorted,
    positive_scores,
):

    B = int(
        len(
            benign_sorted
        )
    )

    P = int(
        len(
            positive_scores
        )
    )


    if B <= 0:

        raise RuntimeError(
            "Family evaluation has no benign rows."
        )


    if P == 0:

        return (
            np.nan,
            np.nan,
        )


    positive_scores = np.asarray(
        positive_scores,
        dtype=np.float32,
    )


    # -------------------------------------------------------------------------
    # ROC-AUC
    # -------------------------------------------------------------------------

    left = np.searchsorted(
        benign_sorted,
        positive_scores,
        side="left",
    )

    right = np.searchsorted(
        benign_sorted,
        positive_scores,
        side="right",
    )


    wins = (
        left.astype(
            np.float64
        )
        +
        0.5
        * (
            right
            - left
        ).astype(
            np.float64
        )
    )


    roc_auc = float(
        np.sum(
            wins,
            dtype=np.float64,
        )
        / (
            float(P)
            * float(B)
        )
    )


    # -------------------------------------------------------------------------
    # Average precision / PR-AUC
    # -------------------------------------------------------------------------

    unique_pos, counts = np.unique(
        positive_scores,
        return_counts=True,
    )


    score_desc = unique_pos[
        ::-1
    ]

    count_desc = counts[
        ::-1
    ].astype(
        np.int64
    )


    tp_cumulative = np.cumsum(
        count_desc,
        dtype=np.int64,
    )


    # At threshold score >= s:
    # false positives = number of benign scores >= s.
    fp_cumulative = (
        B
        -
        np.searchsorted(
            benign_sorted,
            score_desc,
            side="left",
        )
    ).astype(
        np.int64
    )


    precision = (
        tp_cumulative.astype(
            np.float64
        )
        /
        (
            tp_cumulative
            +
            fp_cumulative
        ).astype(
            np.float64
        )
    )


    recall_increment = (
        count_desc.astype(
            np.float64
        )
        / float(P)
    )


    pr_auc = float(
        np.sum(
            recall_increment
            * precision,
            dtype=np.float64,
        )
    )


    return (
        pr_auc,
        roc_auc,
    )


def operating_metrics(
    benign_scores,
    positive_scores,
):

    B = int(
        len(
            benign_scores
        )
    )

    P = int(
        len(
            positive_scores
        )
    )


    fp = int(
        np.sum(
            benign_scores
            >= THRESHOLD
        )
    )

    tn = (
        B
        - fp
    )


    fpr = (
        fp
        / B
        if B > 0
        else np.nan
    )


    # No attacks from this family in this validation split.
    if P == 0:

        return {
            "precision_at_0_50":
                np.nan,

            "recall_at_0_50":
                np.nan,

            "f1_at_0_50":
                np.nan,

            "fpr_at_0_50":
                float(
                    fpr
                ),

            "fnr_at_0_50":
                np.nan,

            "tp":
                0,

            "fn":
                0,

            "fp":
                fp,

            "tn":
                tn,
        }


    tp = int(
        np.sum(
            positive_scores
            >= THRESHOLD
        )
    )

    fn = (
        P
        - tp
    )


    precision = (
        tp
        / (
            tp
            + fp
        )
        if (
            tp
            + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp
        / P
    )


    f1 = (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
        if (
            precision
            + recall
        ) > 0
        else 0.0
    )


    fnr = (
        fn
        / P
    )


    return {
        "precision_at_0_50":
            float(
                precision
            ),

        "recall_at_0_50":
            float(
                recall
            ),

        "f1_at_0_50":
            float(
                f1
            ),

        "fpr_at_0_50":
            float(
                fpr
            ),

        "fnr_at_0_50":
            float(
                fnr
            ),

        "tp":
            tp,

        "fn":
            fn,

        "fp":
            fp,

        "tn":
            tn,
    }


# =============================================================================
# 3. SELF-TEST FAST RANKING METRICS AGAINST SKLEARN
#
# Includes score ties deliberately.
# =============================================================================

synthetic_y = np.asarray(
    [
        0, 1, 0, 1,
        0, 1, 0, 1,
        0, 1,
    ],
    dtype=np.int8,
)

synthetic_p = np.asarray(
    [
        0.1,
        0.8,
        0.8,
        0.8,
        0.2,
        0.5,
        0.5,
        0.3,
        0.9,
        0.9,
    ],
    dtype=np.float32,
)


syn_b = np.sort(
    synthetic_p[
        synthetic_y
        == 0
    ]
)

syn_a = synthetic_p[
    synthetic_y
    == 1
]


fast_ap, fast_roc = (
    fast_family_ranking_metrics(
        syn_b,
        syn_a,
    )
)


sk_ap = float(
    average_precision_score(
        synthetic_y,
        synthetic_p,
    )
)

sk_roc = float(
    roc_auc_score(
        synthetic_y,
        synthetic_p,
    )
)


if not np.isclose(
    fast_ap,
    sk_ap,
    rtol=0.0,
    atol=1e-15,
):

    raise RuntimeError(
        "Fast PR-AUC implementation failed sklearn self-test.\n"
        f"fast={fast_ap}\n"
        f"sklearn={sk_ap}"
    )


if not np.isclose(
    fast_roc,
    sk_roc,
    rtol=0.0,
    atol=1e-15,
):

    raise RuntimeError(
        "Fast ROC-AUC implementation failed sklearn self-test.\n"
        f"fast={fast_roc}\n"
        f"sklearn={sk_roc}"
    )


# =============================================================================
# 4. SEALED REPOSITORY STATE
# =============================================================================

print("=" * 112)
print("STAGE23-6D — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS")
print("=" * 112)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-6D:\n"
        + status
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")
print("[EXACT] custom PR/ROC implementation matches sklearn on tied-score self-test")


# =============================================================================
# 5. VERIFY STAGE23-6C
# =============================================================================

print()
print("=" * 112)
print("VERIFY STAGE23-6C ALIGNMENT")
print("=" * 112)
print()


alignment_checks = [
    (
        ALIGN_PARQUET,
        EXPECTED_ALIGN_PARQUET_SHA,
    ),

    (
        ALIGN_SUPPORT,
        EXPECTED_ALIGN_SUPPORT_SHA,
    ),

    (
        ALIGN_MANIFEST,
        EXPECTED_ALIGN_MANIFEST_SHA,
    ),

    (
        ALIGN_CHECKSUMS,
        EXPECTED_ALIGN_CHECKSUMS_SHA,
    ),
]


for path, expected in alignment_checks:

    if not path.is_file():

        raise RuntimeError(
            f"Stage23-6C artifact missing:\n{path}"
        )


    actual = sha256_file(
        path
    )


    print(path.name)
    print(
        "  expected:",
        expected,
    )
    print(
        "  actual:  ",
        actual,
    )


    if actual != expected:

        raise RuntimeError(
            f"Stage23-6C SHA mismatch: {path.name}"
        )


    print("  [EXACT]")


align_manifest = json.loads(
    ALIGN_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if align_manifest[
    "status"
] != "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED_UNSEALED":

    raise RuntimeError(
        "Stage23-6C manifest status mismatch."
    )


if align_manifest[
    "verification"
][
    "random_label_to_binary_mismatches"
] != 0:

    raise RuntimeError(
        "Stage23-6C RANDOM alignment mismatch recorded."
    )


if align_manifest[
    "verification"
][
    "chronological_label_to_binary_mismatches"
] != 0:

    raise RuntimeError(
        "Stage23-6C CHRONO alignment mismatch recorded."
    )


if align_manifest[
    "verification"
][
    "attack_family_count"
] != 13:

    raise RuntimeError(
        "Unexpected Stage23-6C family count."
    )


# =============================================================================
# 6. LOAD FROZEN FAMILY ALIGNMENT
# =============================================================================

attack_alignment = pd.read_parquet(
    ALIGN_PARQUET,
    columns=[
        "clean_position",
        "attack_family",
    ],
)


attack_clean = attack_alignment[
    "clean_position"
].to_numpy(
    dtype=np.int64
)


attack_family = attack_alignment[
    "attack_family"
].astype(
    str
).to_numpy(
    dtype=object
)


if len(
    attack_clean
) != 1_972_299:

    raise RuntimeError(
        "Retained attack-alignment row count mismatch."
    )


if not np.all(
    np.diff(
        attack_clean
    ) > 0
):

    raise RuntimeError(
        "Attack alignment clean_position not strictly increasing."
    )


support_df = pd.read_csv(
    ALIGN_SUPPORT
)


families = support_df[
    "attack_family"
].astype(
    str
).tolist()


if len(
    families
) != 13:

    raise RuntimeError(
        "Expected exactly 13 frozen development attack families."
    )


if len(
    set(
        families
    )
) != 13:

    raise RuntimeError(
        "Duplicate family in Stage23-6C support table."
    )


print()
print(
    "[EXACT] frozen family universe:",
    len(
        families
    ),
)

for family in families:
    print(
        " ",
        family,
    )


# =============================================================================
# 7. BUILD EXACT PROBABILITY ARTIFACT INVENTORY
# =============================================================================

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)


FULL_RANDOM = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "random_natural_validation_ensemble_probabilities.npz"
)


FULL_CHRONO = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "chronological_natural_validation_ensemble_probabilities.npz"
)


PROBABILITIES = {
    "RANDOM_NATURAL": {
        "FULL":
            FULL_RANDOM,
    },

    "CHRONOLOGICAL_NATURAL": {
        "FULL":
            FULL_CHRONO,
    },
}


for subset in SUBSET_ORDER[
    1:
]:

    slug = SUBSET_SLUG[
        subset
    ]


    PROBABILITIES[
        "RANDOM_NATURAL"
    ][
        subset
    ] = (
        PRIMARY_ROOT
        / "random"
        / slug
        / (
            f"{slug}_random_natural_"
            "validation_probabilities.npz"
        )
    )


    PROBABILITIES[
        "CHRONOLOGICAL_NATURAL"
    ][
        subset
    ] = (
        PRIMARY_ROOT
        / "chronological"
        / slug
        / (
            f"{slug}_chronological_natural_"
            "validation_probabilities.npz"
        )
    )


# =============================================================================
# 8. VERIFY ALL 14 PROBABILITY ARTIFACTS
# =============================================================================

print()
print("=" * 112)
print("SEALED PROBABILITY ARTIFACT VERIFICATION")
print("=" * 112)
print()


probability_sha = {}


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    probability_sha[
        split
    ] = {}


    for subset in SUBSET_ORDER:

        path = PROBABILITIES[
            split
        ][
            subset
        ]


        if (
            split
            == "RANDOM_NATURAL"
            and subset
            == "FULL"
        ):

            expected = (
                EXPECTED_FULL_RANDOM_SHA
            )


        elif (
            split
            == "CHRONOLOGICAL_NATURAL"
            and subset
            == "FULL"
        ):

            expected = (
                EXPECTED_FULL_CHRONO_SHA
            )


        else:

            expected = None


        actual = verify_probability_artifact(
            path,
            expected_sha=expected,
        )


        probability_sha[
            split
        ][
            subset
        ] = actual


        print(
            f"[EXACT] "
            f"{split:<25} "
            f"{subset:<24} "
            f"{actual}"
        )


print()
print(
    "[EXACT] sealed validation probability artifacts: 14 / 14"
)


# =============================================================================
# 9. OUTPUT INITIALIZATION
# =============================================================================

if OUT.exists():

    raise RuntimeError(
        "Stage23-6D output directory already exists.\n"
        "Do NOT delete/rerun blindly:\n"
        f"{OUT}"
    )


OUT.mkdir(
    parents=True,
    exist_ok=False,
)


state = {
    "stage":
        "Stage23-6D frozen attack-family primary-ablation metrics",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "primary_subset_count":
        7,

    "split_count":
        2,

    "family_count":
        13,

    "expected_metric_rows":
        182,

    "subsets_completed":
        0,

    "model_fits":
        0,

    "model_inference":
        0,

    "lightgbm_imported":
        False,

    "lightgbm_execution":
        0,

    "xgboost_imported":
        False,

    "xgboost_execution":
        0,

    "raw_february_csv_reads":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 10. SPLIT-BY-SPLIT FAMILY EVALUATION
# =============================================================================

metric_rows = []

split_family_membership = {}


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print("=" * 112)
    print(
        f"ATTACK-FAMILY METRICS — {split}"
    )
    print("=" * 112)
    print()


    expected_counts = EXPECTED_SPLIT_COUNTS[
        split
    ]


    # -------------------------------------------------------------------------
    # FULL establishes exact validation row order and family membership.
    # -------------------------------------------------------------------------

    full_path = PROBABILITIES[
        split
    ][
        "FULL"
    ]


    (
        reference_clean,
        reference_y,
        full_probability,
    ) = load_probability_npz(
        full_path
    )


    if len(
        reference_clean
    ) != expected_counts[
        "rows"
    ]:

        raise RuntimeError(
            f"{split}: FULL validation row count mismatch."
        )


    actual_attack = int(
        np.sum(
            reference_y
            == 1
        )
    )

    actual_benign = int(
        np.sum(
            reference_y
            == 0
        )
    )


    if actual_attack != expected_counts[
        "attack"
    ]:

        raise RuntimeError(
            f"{split}: attack count mismatch."
        )


    if actual_benign != expected_counts[
        "benign"
    ]:

        raise RuntimeError(
            f"{split}: benign count mismatch."
        )


    attack_row_idx = np.flatnonzero(
        reference_y
        == 1
    )


    benign_mask = (
        reference_y
        == 0
    )


    validation_attack_clean = reference_clean[
        attack_row_idx
    ]


    pos = np.searchsorted(
        attack_clean,
        validation_attack_clean,
    )


    if np.any(
        pos
        >= len(
            attack_clean
        )
    ):

        raise RuntimeError(
            f"{split}: attack-family locator overflow."
        )


    if not np.array_equal(
        attack_clean[
            pos
        ],
        validation_attack_clean,
    ):

        raise RuntimeError(
            f"{split}: attack-family clean_position join failed."
        )


    validation_attack_family = (
        attack_family[
            pos
        ]
    )


    # Precompute validation row indices for every frozen family.
    family_row_indices = {}


    for family in families:

        family_row_indices[
            family
        ] = attack_row_idx[
            validation_attack_family
            == family
        ]


    # Verify supports exactly match Stage23-6C.
    support_column = (
        "random_natural_validation_attack_support"
        if split
        == "RANDOM_NATURAL"
        else
        "chronological_natural_validation_attack_support"
    )


    for family in families:

        expected_support = int(
            support_df.loc[
                support_df[
                    "attack_family"
                ].astype(
                    str
                )
                == family,
                support_column,
            ].iloc[
                0
            ]
        )


        actual_support = len(
            family_row_indices[
                family
            ]
        )


        if actual_support != expected_support:

            raise RuntimeError(
                f"{split} / {family}: support mismatch.\n"
                f"expected={expected_support}\n"
                f"actual={actual_support}"
            )


    split_family_membership[
        split
    ] = {
        family:
            int(
                len(
                    family_row_indices[
                        family
                    ]
                )
            )

        for family
        in families
    }


    print(
        "[EXACT] validation rows:",
        f"{len(reference_clean):,}",
    )

    print(
        "[EXACT] attack / benign:",
        f"{actual_attack:,}",
        "/",
        f"{actual_benign:,}",
    )

    print(
        "[EXACT] all family supports match Stage23-6C"
    )


    # -------------------------------------------------------------------------
    # Process each primary subset.
    # -------------------------------------------------------------------------

    for subset_index, subset in enumerate(
        SUBSET_ORDER
    ):

        path = PROBABILITIES[
            split
        ][
            subset
        ]


        if subset == "FULL":

            clean = reference_clean
            y = reference_y
            probability = full_probability


        else:

            (
                clean,
                y,
                probability,
            ) = load_probability_npz(
                path
            )


            if not np.array_equal(
                clean,
                reference_clean,
            ):

                raise RuntimeError(
                    f"{split}/{subset}: clean_position differs from FULL."
                )


            if not np.array_equal(
                y,
                reference_y,
            ):

                raise RuntimeError(
                    f"{split}/{subset}: binary_label differs from FULL."
                )


        # ---------------------------------------------------------------------
        # Verify the FULL ranking metric against the frozen Stage22R result.
        # ---------------------------------------------------------------------

        benign_scores = probability[
            benign_mask
        ]


        benign_sorted = np.sort(
            benign_scores
        )


        if subset == "FULL":

            overall_attack_scores = probability[
                attack_row_idx
            ]


            overall_pr, overall_roc = (
                fast_family_ranking_metrics(
                    benign_sorted,
                    overall_attack_scores,
                )
            )


            expected_metric = EXPECTED_FULL_METRICS[
                split
            ]


            if not np.isclose(
                overall_pr,
                expected_metric[
                    "PR_AUC"
                ],
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    f"{split}: FULL PR-AUC verification failed.\n"
                    f"expected={expected_metric['PR_AUC']}\n"
                    f"actual={overall_pr}"
                )


            if not np.isclose(
                overall_roc,
                expected_metric[
                    "ROC_AUC"
                ],
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    f"{split}: FULL ROC-AUC verification failed.\n"
                    f"expected={expected_metric['ROC_AUC']}\n"
                    f"actual={overall_roc}"
                )


            print()
            print(
                f"[EXACT] {split} FULL "
                f"PR={overall_pr:.12f} "
                f"ROC={overall_roc:.12f}"
            )


        # ---------------------------------------------------------------------
        # Family metrics.
        # ---------------------------------------------------------------------

        benign_support = len(
            benign_scores
        )


        for family in families:

            positive_idx = family_row_indices[
                family
            ]


            positive_scores = probability[
                positive_idx
            ]


            support_attack = int(
                len(
                    positive_scores
                )
            )


            pr_auc, roc_auc = (
                fast_family_ranking_metrics(
                    benign_sorted,
                    positive_scores,
                )
            )


            op = operating_metrics(
                benign_scores,
                positive_scores,
            )


            interpretation_status = (
                "INTERPRETABLE"
                if support_attack
                >= MIN_INTERPRETATION_SUPPORT
                else
                "DESCRIPTIVE_ONLY"
            )


            metric_rows.append(
                {
                    "split":
                        split,

                    "subset":
                        subset,

                    "attack_family":
                        family,

                    "support_attack":
                        support_attack,

                    "support_benign":
                        int(
                            benign_support
                        ),

                    "interpretation_status":
                        interpretation_status,

                    "PR_AUC":
                        pr_auc,

                    "ROC_AUC":
                        roc_auc,

                    "precision_at_0_50":
                        op[
                            "precision_at_0_50"
                        ],

                    "recall_at_0_50":
                        op[
                            "recall_at_0_50"
                        ],

                    "f1_at_0_50":
                        op[
                            "f1_at_0_50"
                        ],

                    "fpr_at_0_50":
                        op[
                            "fpr_at_0_50"
                        ],

                    "fnr_at_0_50":
                        op[
                            "fnr_at_0_50"
                        ],

                    "tp_at_0_50":
                        op[
                            "tp"
                        ],

                    "fn_at_0_50":
                        op[
                            "fn"
                        ],

                    "fp_at_0_50":
                        op[
                            "fp"
                        ],

                    "tn_at_0_50":
                        op[
                            "tn"
                        ],
                }
            )


        print(
            f"[{subset_index + 1}/7] "
            f"{split:<25} "
            f"{subset:<24} COMPLETE"
        )


        state[
            "subsets_completed"
        ] += 1

        state[
            "status"
        ] = "METRICS_IN_PROGRESS"

        state[
            "last_completed_split"
        ] = split

        state[
            "last_completed_subset"
        ] = subset


        write_json(
            STATE_JSON,
            state,
        )


        del (
            benign_scores,
            benign_sorted,
        )


        if subset != "FULL":

            del (
                clean,
                y,
                probability,
            )


        gc.collect()


    del (
        reference_clean,
        reference_y,
        full_probability,
        attack_row_idx,
        benign_mask,
        validation_attack_clean,
        pos,
        validation_attack_family,
        family_row_indices,
    )

    gc.collect()


# =============================================================================
# 11. BUILD DATAFRAME + FULL-MINUS-ABLATED DELTAS
# =============================================================================

metrics_df = pd.DataFrame(
    metric_rows
)


EXPECTED_METRIC_ROWS = (
    2
    * 7
    * 13
)


if len(
    metrics_df
) != EXPECTED_METRIC_ROWS:

    raise RuntimeError(
        "Attack-family metric row-count mismatch.\n"
        f"expected={EXPECTED_METRIC_ROWS}\n"
        f"actual={len(metrics_df)}"
    )


metrics_df[
    "delta_f1_FULL_minus_ablated"
] = np.nan

metrics_df[
    "delta_recall_FULL_minus_ablated"
] = np.nan


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    for family in families:

        full_row = metrics_df[
            (
                metrics_df[
                    "split"
                ]
                == split
            )
            &
            (
                metrics_df[
                    "subset"
                ]
                == "FULL"
            )
            &
            (
                metrics_df[
                    "attack_family"
                ]
                == family
            )
        ]


        if len(
            full_row
        ) != 1:

            raise RuntimeError(
                f"FULL family row not unique: {split}/{family}"
            )


        full_f1 = float(
            full_row[
                "f1_at_0_50"
            ].iloc[
                0
            ]
        )


        full_recall = float(
            full_row[
                "recall_at_0_50"
            ].iloc[
                0
            ]
        )


        for subset in SUBSET_ORDER:

            row_mask = (
                (
                    metrics_df[
                        "split"
                    ]
                    == split
                )
                &
                (
                    metrics_df[
                        "subset"
                    ]
                    == subset
                )
                &
                (
                    metrics_df[
                        "attack_family"
                    ]
                    == family
                )
            )


            if int(
                row_mask.sum()
            ) != 1:

                raise RuntimeError(
                    f"Family metric row not unique: "
                    f"{split}/{subset}/{family}"
                )


            ablated_f1 = float(
                metrics_df.loc[
                    row_mask,
                    "f1_at_0_50",
                ].iloc[
                    0
                ]
            )


            ablated_recall = float(
                metrics_df.loc[
                    row_mask,
                    "recall_at_0_50",
                ].iloc[
                    0
                ]
            )


            if (
                math.isfinite(
                    full_f1
                )
                and
                math.isfinite(
                    ablated_f1
                )
            ):

                metrics_df.loc[
                    row_mask,
                    "delta_f1_FULL_minus_ablated",
                ] = (
                    full_f1
                    - ablated_f1
                )


            if (
                math.isfinite(
                    full_recall
                )
                and
                math.isfinite(
                    ablated_recall
                )
            ):

                metrics_df.loc[
                    row_mask,
                    "delta_recall_FULL_minus_ablated",
                ] = (
                    full_recall
                    - ablated_recall
                )


# =============================================================================
# 12. INTERNAL CONSISTENCY
# =============================================================================

# Every family within a split/subset must use the same benign support.
for (
    split,
    subset,
), group in metrics_df.groupby(
    [
        "split",
        "subset",
    ]
):

    if group[
        "support_benign"
    ].nunique() != 1:

        raise RuntimeError(
            f"{split}/{subset}: inconsistent benign support."
        )


    expected_benign = EXPECTED_SPLIT_COUNTS[
        split
    ][
        "benign"
    ]


    actual_benign = int(
        group[
            "support_benign"
        ].iloc[
            0
        ]
    )


    if actual_benign != expected_benign:

        raise RuntimeError(
            f"{split}/{subset}: benign support mismatch."
        )


# FULL delta must be exactly zero whenever defined.
full_defined_f1 = metrics_df[
    (
        metrics_df[
            "subset"
        ]
        == "FULL"
    )
    &
    (
        metrics_df[
            "delta_f1_FULL_minus_ablated"
        ].notna()
    )
]


if not np.allclose(
    full_defined_f1[
        "delta_f1_FULL_minus_ablated"
    ].to_numpy(
        dtype=float
    ),
    0.0,
    rtol=0.0,
    atol=0.0,
):

    raise RuntimeError(
        "FULL F1 deltas are not exactly zero."
    )


full_defined_recall = metrics_df[
    (
        metrics_df[
            "subset"
        ]
        == "FULL"
    )
    &
    (
        metrics_df[
            "delta_recall_FULL_minus_ablated"
        ].notna()
    )
]


if not np.allclose(
    full_defined_recall[
        "delta_recall_FULL_minus_ablated"
    ].to_numpy(
        dtype=float
    ),
    0.0,
    rtol=0.0,
    atol=0.0,
):

    raise RuntimeError(
        "FULL recall deltas are not exactly zero."
    )


# =============================================================================
# 13. AGGREGATE PRIMARY DELTA SUMMARY
#
# Only >=100-support families contribute to inferential summary.
# FULL excluded because delta is trivially zero.
# =============================================================================

summary_rows = []


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    for subset in SUBSET_ORDER[
        1:
    ]:

        g = metrics_df[
            (
                metrics_df[
                    "split"
                ]
                == split
            )
            &
            (
                metrics_df[
                    "subset"
                ]
                == subset
            )
            &
            (
                metrics_df[
                    "interpretation_status"
                ]
                == "INTERPRETABLE"
            )
        ].copy()


        if len(
            g
        ) == 0:

            raise RuntimeError(
                f"No interpretable families for {split}/{subset}."
            )


        if g[
            "delta_f1_FULL_minus_ablated"
        ].isna().any():

            raise RuntimeError(
                f"Undefined interpretable F1 delta: {split}/{subset}"
            )


        if g[
            "delta_recall_FULL_minus_ablated"
        ].isna().any():

            raise RuntimeError(
                f"Undefined interpretable recall delta: {split}/{subset}"
            )


        f1_abs_idx = g[
            "delta_f1_FULL_minus_ablated"
        ].abs().idxmax()


        recall_abs_idx = g[
            "delta_recall_FULL_minus_ablated"
        ].abs().idxmax()


        summary_rows.append(
            {
                "split":
                    split,

                "subset":
                    subset,

                "interpretable_family_count":
                    int(
                        len(
                            g
                        )
                    ),

                "mean_delta_f1_FULL_minus_ablated":
                    float(
                        g[
                            "delta_f1_FULL_minus_ablated"
                        ].mean()
                    ),

                "median_delta_f1_FULL_minus_ablated":
                    float(
                        g[
                            "delta_f1_FULL_minus_ablated"
                        ].median()
                    ),

                "max_abs_delta_f1_family":
                    str(
                        g.loc[
                            f1_abs_idx,
                            "attack_family",
                        ]
                    ),

                "max_abs_delta_f1":
                    float(
                        g.loc[
                            f1_abs_idx,
                            "delta_f1_FULL_minus_ablated",
                        ]
                    ),

                "mean_delta_recall_FULL_minus_ablated":
                    float(
                        g[
                            "delta_recall_FULL_minus_ablated"
                        ].mean()
                    ),

                "median_delta_recall_FULL_minus_ablated":
                    float(
                        g[
                            "delta_recall_FULL_minus_ablated"
                        ].median()
                    ),

                "max_abs_delta_recall_family":
                    str(
                        g.loc[
                            recall_abs_idx,
                            "attack_family",
                        ]
                    ),

                "max_abs_delta_recall":
                    float(
                        g.loc[
                            recall_abs_idx,
                            "delta_recall_FULL_minus_ablated",
                        ]
                    ),
            }
        )


summary_df = pd.DataFrame(
    summary_rows
)


if len(
    summary_df
) != 12:

    raise RuntimeError(
        "Primary delta summary must contain 12 rows."
    )


# =============================================================================
# 14. WRITE SCIENTIFIC OUTPUTS
# =============================================================================

metrics_df.to_csv(
    METRICS_CSV,
    index=False,
)


summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)


metrics_json_payload = {
    "stage":
        "Stage23-6D",

    "status":
        "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED",

    "evaluation_definition":
        (
            "For attack family A, evaluate all benign validation "
            "rows plus attack validation rows belonging to A."
        ),

    "family_set":
        families,

    "minimum_support_for_interpretation":
        MIN_INTERPRETATION_SUPPORT,

    "subsets":
        SUBSET_ORDER,

    "splits":
        [
            "RANDOM_NATURAL",
            "CHRONOLOGICAL_NATURAL",
        ],

    "metrics":
        dataframe_records_json_safe(
            metrics_df
        ),
}


write_json(
    METRICS_JSON,
    metrics_json_payload,
)


# =============================================================================
# 15. PRINT FULL FAMILY BASELINES
# =============================================================================

print()
print("=" * 112)
print("FULL ENSEMBLE — ATTACK-FAMILY BASELINES")
print("=" * 112)
print()


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print(split)
    print("-" * 112)


    g = metrics_df[
        (
            metrics_df[
                "split"
            ]
            == split
        )
        &
        (
            metrics_df[
                "subset"
            ]
            == "FULL"
        )
    ]


    for row in g.itertuples(
        index=False
    ):

        support = int(
            row.support_attack
        )


        if support == 0:

            print(
                f"{row.attack_family:<32} "
                f"support={support:>7,}  "
                f"{row.interpretation_status:<16} "
                f"PR=NA ROC=NA F1=NA Recall=NA "
                f"FPR={row.fpr_at_0_50:.9f}"
            )

        else:

            print(
                f"{row.attack_family:<32} "
                f"support={support:>7,}  "
                f"{row.interpretation_status:<16} "
                f"PR={row.PR_AUC:.9f} "
                f"ROC={row.ROC_AUC:.9f} "
                f"F1={row.f1_at_0_50:.9f} "
                f"Recall={row.recall_at_0_50:.9f} "
                f"FPR={row.fpr_at_0_50:.9f}"
            )


# =============================================================================
# 16. PRINT PRIMARY DELTA SUMMARY
# =============================================================================

print()
print("=" * 112)
print("PRIMARY ABLATION DELTA SUMMARY — INTERPRETABLE FAMILIES ONLY")
print("=" * 112)
print()

print(
    "Sign convention: positive FULL-minus-ablated delta = "
    "ablation performs worse than FULL."
)

print()


for row in summary_df.itertuples(
    index=False
):

    print(
        f"{row.split:<25} "
        f"{row.subset:<24} "
        f"n={row.interpretable_family_count:<2} "
        f"meanΔF1={row.mean_delta_f1_FULL_minus_ablated:+.9f} "
        f"max|ΔF1|={row.max_abs_delta_f1:+.9f} "
        f"[{row.max_abs_delta_f1_family}] "
        f"meanΔRecall={row.mean_delta_recall_FULL_minus_ablated:+.9f} "
        f"max|ΔRecall|={row.max_abs_delta_recall:+.9f} "
        f"[{row.max_abs_delta_recall_family}]"
    )


# =============================================================================
# 17. PRINT CHRONOLOGICAL INFILTERATION EXACT DELTAS
# =============================================================================

print()
print("=" * 112)
print("CHRONOLOGICAL_NATURAL — INFILTERATION PRIMARY ABLATION DELTAS")
print("=" * 112)
print()


chrono_inf = metrics_df[
    (
        metrics_df[
            "split"
        ]
        == "CHRONOLOGICAL_NATURAL"
    )
    &
    (
        metrics_df[
            "attack_family"
        ]
        == "Infilteration"
    )
].copy()


for subset in SUBSET_ORDER:

    row = chrono_inf[
        chrono_inf[
            "subset"
        ]
        == subset
    ].iloc[
        0
    ]


    print(
        f"{subset:<24} "
        f"support={int(row['support_attack']):>7,} "
        f"PR={row['PR_AUC']:.9f} "
        f"ROC={row['ROC_AUC']:.9f} "
        f"F1={row['f1_at_0_50']:.9f} "
        f"Recall={row['recall_at_0_50']:.9f} "
        f"ΔF1={row['delta_f1_FULL_minus_ablated']:+.9f} "
        f"ΔRecall={row['delta_recall_FULL_minus_ablated']:+.9f}"
    )


# =============================================================================
# 18. MANIFEST / STATE
# =============================================================================

manifest = {
    "stage":
        "Stage23-6D",

    "status":
        "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED",

    "completed_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "stage23_6c": {
        "attack_alignment_sha256":
            EXPECTED_ALIGN_PARQUET_SHA,

        "support_sha256":
            EXPECTED_ALIGN_SUPPORT_SHA,

        "manifest_sha256":
            EXPECTED_ALIGN_MANIFEST_SHA,

        "checksum_manifest_sha256":
            EXPECTED_ALIGN_CHECKSUMS_SHA,
    },

    "evaluation": {
        "splits":
            [
                "RANDOM_NATURAL",
                "CHRONOLOGICAL_NATURAL",
            ],

        "subsets":
            SUBSET_ORDER,

        "family_count":
            13,

        "metric_rows":
            int(
                len(
                    metrics_df
                )
            ),

        "minimum_attack_support_for_interpretation":
            MIN_INTERPRETATION_SUPPORT,

        "fixed_threshold":
            0.5,

        "probability_source":
            "persisted float32 ensemble probability",

        "family_definition":
            (
                "All frozen development attack families. "
                "For family A: all benign validation rows "
                "+ attack rows in A."
            ),

        "zero_support_policy":
            (
                "Ranking, precision, recall, F1, and FNR are "
                "undefined/null when support_attack=0; "
                "FPR remains defined from benign rows."
            ),
    },

    "probability_artifact_sha256":
        probability_sha,

    "family_support":
        split_family_membership,

    "artifacts": {
        METRICS_CSV.name: {
            "rows":
                int(
                    len(
                        metrics_df
                    )
                ),

            "sha256":
                sha256_file(
                    METRICS_CSV
                ),
        },

        METRICS_JSON.name: {
            "sha256":
                sha256_file(
                    METRICS_JSON
                ),
        },

        SUMMARY_CSV.name: {
            "rows":
                int(
                    len(
                        summary_df
                    )
                ),

            "sha256":
                sha256_file(
                    SUMMARY_CSV
                ),
        },
    },

    "governance": {
        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_imported":
            False,

        "lightgbm_execution":
            0,

        "xgboost_imported":
            False,

        "xgboost_execution":
            0,

        "raw_february_csv_reads":
            0,

        "development_source_parquet_reads":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "threshold_optimization":
            False,

        "new_subset":
            False,

        "new_metric":
            False,
    },

    "interpretation_boundary": {
        "chronological_attack_family_coverage":
            (
                "Only attack families with support in the frozen "
                "chronological validation day can be evaluated. "
                "Zero-support development families are retained "
                "descriptively rather than dropped."
            ),

        "causality":
            (
                "Family-specific ablation differences are descriptive "
                "associations under frozen validation memberships and "
                "do not establish causal feature dependence."
            ),
    },
}


write_json(
    MANIFEST_JSON,
    manifest,
)


state[
    "status"
] = (
    "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "subsets_completed"
] = 14

state[
    "metric_rows"
] = int(
    len(
        metrics_df
    )
)

state[
    "next_action"
] = (
    "REVIEW_THEN_ZERO_FIT_SEAL_STAGE23_6C_AND_6D"
)


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 19. CHECKSUM MANIFEST
# =============================================================================

artifact_files = sorted(
    p
    for p in OUT.iterdir()
    if (
        p.is_file()
        and p != CHECKSUMS
    )
)


CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(p)}  {p.name}"
        )

        for p
        in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 20. FINAL
# =============================================================================

print()
print("=" * 112)
print("STAGE23-6D ATTACK-FAMILY METRICS COMPLETE — UNSEALED")
print("=" * 112)
print()

print("Frozen development families : 13")
print("Primary subsets              : 7 / 7")
print("Splits                       : 2 / 2")
print("Probability artifacts        : 14 / 14")
print("Family metric rows           :", len(metrics_df))
print()
print(
    "RANDOM interpretable families:",
    int(
        (
            support_df[
                "random_natural_validation_attack_support"
            ]
            >= 100
        ).sum()
    ),
)
print(
    "CHRONO interpretable families:",
    int(
        (
            support_df[
                "chronological_natural_validation_attack_support"
            ]
            >= 100
        ).sum()
    ),
)

print()
print("Metrics CSV:")
print(" ", METRICS_CSV)
print(" SHA256:")
print(" ", sha256_file(METRICS_CSV))

print()
print("Metrics JSON:")
print(" ", METRICS_JSON)
print(" SHA256:")
print(" ", sha256_file(METRICS_JSON))

print()
print("Primary delta summary:")
print(" ", SUMMARY_CSV)
print(" SHA256:")
print(" ", sha256_file(SUMMARY_CSV))

print()
print("Manifest:")
print(" ", MANIFEST_JSON)
print(" SHA256:")
print(" ", sha256_file(MANIFEST_JSON))

print()
print("Checksum manifest SHA256:")
print(" ", sha256_file(CHECKSUMS))

print()
print("GOVERNANCE")
print("  Stage23 fit state        : 50 / 50 SEALED")
print("  New model fits           : 0")
print("  Model inference          : 0")
print("  LightGBM imported        : NO")
print("  LightGBM execution       : 0")
print("  XGBoost imported         : NO")
print("  XGBoost execution        : 0")
print("  Threshold optimization   : NO")
print("  Raw February CSV reads   : 0")
print("  Development Parquet read : 0")
print("  Raw Mar1 accessed        : NO")
print("  Raw Mar2 accessed        : NO")

print()
print("NEXT:")
print("  Review scientific results.")
print("  Then ZERO-FIT seal + push Stage23-6C and Stage23-6D.")

print("=" * 112)

STAGE23-6D — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] new fits  : 0 authorized
[EXACT] custom PR/ROC implementation matches sklearn on tied-score self-test

VERIFY STAGE23-6C ALIGNMENT

retained_development_attack_family_labels.parquet
  expected: 3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  actual:   3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  [EXACT]
attack_family_support_by_split.csv
  expected: d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  actual:   d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  [EXACT]
stage23_6c_alignment_manifest.json
  expected: bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  actual:   bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  [EXACT]
checksums.sha256


RuntimeError: Probability artifact SHA mismatch:
/kaggle/working/ids2018-validation-safe-ablation/results/stage22r_training/stage22r_2a_random_natural/random_natural_validation_ensemble_probabilities.npz
expected=9dc64ccdb6580dad727668ff885c010dde5725fd1bfb1816d07183eacda6e5b02
actual=9dc64ccdb6580dad727668ff3e511e0a8572cbb1de49b9ffe8999b0c3d190394

In [8]:
# =============================================================================
# STAGE23-6D — HASH TYPO CORRECTION PREFLIGHT
#
# ZERO FITS
# ZERO INFERENCE
# ZERO MODEL EXECUTION
# ZERO SCIENTIFIC WRITES
# =============================================================================

from pathlib import Path
import hashlib


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

FULL_RANDOM = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "random_natural_validation_ensemble_probabilities.npz"
)

OUT_6D = Path(
    "/kaggle/working/stage23_6d_attack_family_metrics"
)

CORRECT_RANDOM_FULL_SHA = (
    "9dc64ccdb6580dad727668ff3e511e0a"
    "8572cbb1de49b9ffe8999b0c3d190394"
)


def sha256_file(path, chunk=32 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


print("=" * 100)
print("STAGE23-6D — HASH TYPO CORRECTION PREFLIGHT")
print("=" * 100)
print()


if not FULL_RANDOM.is_file():

    raise RuntimeError(
        f"FULL RANDOM probability artifact missing:\n{FULL_RANDOM}"
    )


actual = sha256_file(
    FULL_RANDOM
)


print("RANDOM_NATURAL FULL probabilities")
print("  corrected expected:", CORRECT_RANDOM_FULL_SHA)
print("  actual:            ", actual)


if actual != CORRECT_RANDOM_FULL_SHA:

    raise RuntimeError(
        "Correct sealed RANDOM_NATURAL FULL SHA still does not match."
    )


print("  [EXACT]")


if OUT_6D.exists():

    raise RuntimeError(
        "Stage23-6D output directory unexpectedly exists.\n"
        "Do NOT rerun blindly:\n"
        f"{OUT_6D}"
    )


print()
print("[EXACT] Stage23-6D output directory does not exist.")
print("[EXACT] previous failure occurred before scientific writes.")

print()
print("Model fits           : 0")
print("Model inference      : 0")
print("LightGBM execution   : 0")
print("XGBoost execution    : 0")
print("Attack-family metrics: 0")

print()
print("[READY]")
print("Stage23-6D can be rerun with the corrected RANDOM FULL SHA.")
print("=" * 100)

STAGE23-6D — HASH TYPO CORRECTION PREFLIGHT

RANDOM_NATURAL FULL probabilities
  corrected expected: 9dc64ccdb6580dad727668ff3e511e0a8572cbb1de49b9ffe8999b0c3d190394
  actual:             9dc64ccdb6580dad727668ff3e511e0a8572cbb1de49b9ffe8999b0c3d190394
  [EXACT]

[EXACT] Stage23-6D output directory does not exist.
[EXACT] previous failure occurred before scientific writes.

Model fits           : 0
Model inference      : 0
LightGBM execution   : 0
XGBoost execution    : 0
Attack-family metrics: 0

[READY]
Stage23-6D can be rerun with the corrected RANDOM FULL SHA.


In [9]:
# =============================================================================
# STAGE23-6D V2 — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS
#
# IMPORTANT IMPLEMENTATION CORRECTIONS
#   1. Correct sealed RANDOM_NATURAL FULL probability SHA256.
#   2. Output ONLY metrics frozen in attack_family_spec.json.
#   3. No post-hoc aggregate mean/max family metrics.
#
# FROZEN METRICS
#   support_attack
#   support_benign
#   PR_AUC
#   ROC_AUC
#   precision_at_0_50
#   recall_at_0_50
#   f1_at_0_50
#   fpr_at_0_50
#   fnr_at_0_50
#   delta_f1_FULL_minus_ablated
#   delta_recall_FULL_minus_ablated
#
# PRIMARY SUBSETS
#   FULL
#   NO_DST_PORT
#   NO_PORTS
#   NO_INIT_FWD_WIN_BYTS
#   NO_FWD_SEG_SIZE_MIN
#   NO_SUSPICIOUS_GROUP
#   BEHAVIOR_ONLY
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM IMPORT / EXECUTION
# ZERO XGBOOST IMPORT / EXECUTION
# ZERO RAW FEBRUARY CSV READS
# ZERO DEVELOPMENT SOURCE PARQUET READS
# NO MAR1 / MAR2
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import math
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)


ALIGN_ROOT = Path(
    "/kaggle/working/stage23_6c_attack_family_alignment"
)

ALIGN_PARQUET = (
    ALIGN_ROOT
    / "retained_development_attack_family_labels.parquet"
)

ALIGN_SUPPORT = (
    ALIGN_ROOT
    / "attack_family_support_by_split.csv"
)

ALIGN_MANIFEST = (
    ALIGN_ROOT
    / "stage23_6c_alignment_manifest.json"
)

ALIGN_CHECKSUMS = (
    ALIGN_ROOT
    / "checksums.sha256"
)


EXPECTED_ALIGN_PARQUET_SHA = (
    "3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26"
)

EXPECTED_ALIGN_SUPPORT_SHA = (
    "d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377"
)

EXPECTED_ALIGN_MANIFEST_SHA = (
    "bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8"
)

EXPECTED_ALIGN_CHECKSUMS_SHA = (
    "a54e01b733e093602a03b9b8bff5430f7cdfa4e987e0a36577ee686e736113b4"
)


# CORRECT sealed hashes.
EXPECTED_FULL_RANDOM_SHA = (
    "9dc64ccdb6580dad727668ff3e511e0a"
    "8572cbb1de49b9ffe8999b0c3d190394"
)

EXPECTED_FULL_CHRONO_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd"
    "86628e9586eb01f6f5d962e35fff65e3f"
)


EXPECTED_FULL_METRICS = {
    "RANDOM_NATURAL": {
        "PR_AUC":
            0.9955900418992819,

        "ROC_AUC":
            0.9986245647735994,
    },

    "CHRONOLOGICAL_NATURAL": {
        "PR_AUC":
            0.10621515513397227,

        "ROC_AUC":
            0.5149184263937692,
    },
}


EXPECTED_SPLIT_COUNTS = {
    "RANDOM_NATURAL": {
        "rows": 2_882_481,
        "attack": 394_460,
        "benign": 2_488_021,
    },

    "CHRONOLOGICAL_NATURAL": {
        "rows": 593_780,
        "attack": 62_256,
        "benign": 531_524,
    },
}


MIN_INTERPRETATION_SUPPORT = 100
THRESHOLD = np.float32(0.50)


SUBSET_ORDER = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]


SUBSET_SLUG = {
    "NO_DST_PORT":
        "no_dst_port",

    "NO_PORTS":
        "no_ports",

    "NO_INIT_FWD_WIN_BYTS":
        "no_init_fwd_win_byts",

    "NO_FWD_SEG_SIZE_MIN":
        "no_fwd_seg_size_min",

    "NO_SUSPICIOUS_GROUP":
        "no_suspicious_group",

    "BEHAVIOR_ONLY":
        "behavior_only",
}


FROZEN_METRIC_COLUMNS = [
    "support_attack",
    "support_benign",
    "PR_AUC",
    "ROC_AUC",
    "precision_at_0_50",
    "recall_at_0_50",
    "f1_at_0_50",
    "fpr_at_0_50",
    "fnr_at_0_50",
    "delta_f1_FULL_minus_ablated",
    "delta_recall_FULL_minus_ablated",
]


OUT = Path(
    "/kaggle/working/stage23_6d_attack_family_metrics"
)

METRICS_CSV = (
    OUT
    / "stage23_6d_attack_family_metrics.csv"
)

METRICS_JSON = (
    OUT
    / "stage23_6d_attack_family_metrics.json"
)

MANIFEST_JSON = (
    OUT
    / "stage23_6d_manifest.json"
)

STATE_JSON = (
    OUT
    / "execution_state.json"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def git(*args):

    p = subprocess.run(
        ["git", *map(str, args)],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

    if p.returncode != 0:

        raise RuntimeError(
            p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def safe_json(v):

    if v is None:
        return None

    if isinstance(
        v,
        (bool, np.bool_),
    ):
        return bool(v)

    if isinstance(
        v,
        (int, np.integer),
    ):
        return int(v)

    if isinstance(
        v,
        (float, np.floating),
    ):

        x = float(v)

        if not math.isfinite(x):
            return None

        return x

    return str(v)


def records_json_safe(df):

    return [
        {
            k: safe_json(v)
            for k, v in row.items()
        }
        for row in df.to_dict(
            orient="records"
        )
    ]


def expected_sha_from_manifest(
    artifact_path,
):

    manifest_path = (
        artifact_path.parent
        / "checksums.sha256"
    )

    if not manifest_path.is_file():

        raise RuntimeError(
            "Sibling checksum manifest missing:\n"
            f"{manifest_path}"
        )


    matches = []


    for line in manifest_path.read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        digest, rel = line.split(
            maxsplit=1
        )

        if Path(
            rel.strip()
        ).name == artifact_path.name:

            matches.append(
                digest.strip()
            )


    if len(matches) != 1:

        raise RuntimeError(
            "Could not uniquely resolve frozen SHA:\n"
            f"{artifact_path}\n"
            f"matches={len(matches)}"
        )


    return matches[0]


def verify_probability_artifact(
    path,
    expected_sha=None,
):

    if not path.is_file():

        raise RuntimeError(
            f"Probability artifact missing:\n{path}"
        )


    if expected_sha is None:

        expected_sha = (
            expected_sha_from_manifest(
                path
            )
        )


    actual = sha256_file(
        path
    )


    if actual != expected_sha:

        raise RuntimeError(
            "Probability artifact SHA mismatch:\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual}"
        )


    return actual


def load_probability_npz(
    path,
):

    with np.load(
        path,
        allow_pickle=False,
    ) as z:

        required = {
            "clean_position",
            "binary_label",
            "ensemble_probability",
        }


        if not required.issubset(
            set(z.files)
        ):

            raise RuntimeError(
                f"{path.name}: required NPZ keys missing.\n"
                f"keys={sorted(z.files)}"
            )


        clean = np.asarray(
            z["clean_position"],
            dtype=np.int64,
        ).copy()


        y = np.asarray(
            z["binary_label"],
            dtype=np.int8,
        ).copy()


        raw_probability = z[
            "ensemble_probability"
        ]


        if raw_probability.dtype != np.float32:

            raise RuntimeError(
                f"{path.name}: probability storage dtype "
                f"is {raw_probability.dtype}, expected float32."
            )


        probability = np.asarray(
            raw_probability,
            dtype=np.float32,
        ).copy()


    if not (
        len(clean)
        == len(y)
        == len(probability)
    ):

        raise RuntimeError(
            f"{path.name}: vector length mismatch."
        )


    if not np.all(
        np.isin(
            y,
            [0, 1],
        )
    ):

        raise RuntimeError(
            f"{path.name}: non-binary labels."
        )


    if not np.all(
        np.isfinite(
            probability
        )
    ):

        raise RuntimeError(
            f"{path.name}: non-finite probabilities."
        )


    if (
        probability.min() < 0.0
        or probability.max() > 1.0
    ):

        raise RuntimeError(
            f"{path.name}: probability outside [0,1]."
        )


    return (
        clean,
        y,
        probability,
    )


# =============================================================================
# 2. EXACT RANKING METRICS
# =============================================================================

def fast_family_ranking_metrics(
    benign_sorted,
    attack_scores,
):

    B = int(
        len(benign_sorted)
    )

    P = int(
        len(attack_scores)
    )


    if B <= 0:

        raise RuntimeError(
            "No benign validation rows."
        )


    if P == 0:

        return (
            np.nan,
            np.nan,
        )


    attack_scores = np.asarray(
        attack_scores,
        dtype=np.float32,
    )


    # ROC-AUC.
    left = np.searchsorted(
        benign_sorted,
        attack_scores,
        side="left",
    )

    right = np.searchsorted(
        benign_sorted,
        attack_scores,
        side="right",
    )


    wins = (
        left.astype(np.float64)
        +
        0.5
        * (
            right
            - left
        ).astype(np.float64)
    )


    roc_auc = float(
        np.sum(
            wins,
            dtype=np.float64,
        )
        /
        (
            float(P)
            * float(B)
        )
    )


    # Average precision / PR-AUC.
    unique_scores, counts = np.unique(
        attack_scores,
        return_counts=True,
    )


    score_desc = unique_scores[::-1]

    count_desc = counts[
        ::-1
    ].astype(
        np.int64
    )


    tp_cumulative = np.cumsum(
        count_desc,
        dtype=np.int64,
    )


    fp_cumulative = (
        B
        -
        np.searchsorted(
            benign_sorted,
            score_desc,
            side="left",
        )
    ).astype(
        np.int64
    )


    precision = (
        tp_cumulative.astype(
            np.float64
        )
        /
        (
            tp_cumulative
            + fp_cumulative
        ).astype(
            np.float64
        )
    )


    recall_increment = (
        count_desc.astype(
            np.float64
        )
        / float(P)
    )


    pr_auc = float(
        np.sum(
            recall_increment
            * precision,
            dtype=np.float64,
        )
    )


    return (
        pr_auc,
        roc_auc,
    )


def frozen_operating_metrics(
    benign_scores,
    attack_scores,
):

    B = int(
        len(benign_scores)
    )

    P = int(
        len(attack_scores)
    )


    fp = int(
        np.sum(
            benign_scores
            >= THRESHOLD
        )
    )


    fpr = (
        float(fp / B)
        if B > 0
        else np.nan
    )


    # No family-A attacks in this validation split.
    if P == 0:

        return {
            "precision_at_0_50":
                np.nan,

            "recall_at_0_50":
                np.nan,

            "f1_at_0_50":
                np.nan,

            "fpr_at_0_50":
                fpr,

            "fnr_at_0_50":
                np.nan,
        }


    tp = int(
        np.sum(
            attack_scores
            >= THRESHOLD
        )
    )


    fn = (
        P
        - tp
    )


    precision = (
        float(
            tp
            / (
                tp
                + fp
            )
        )
        if (
            tp
            + fp
        ) > 0
        else 0.0
    )


    recall = float(
        tp
        / P
    )


    f1 = (
        float(
            2.0
            * precision
            * recall
            / (
                precision
                + recall
            )
        )
        if (
            precision
            + recall
        ) > 0
        else 0.0
    )


    fnr = float(
        fn
        / P
    )


    return {
        "precision_at_0_50":
            precision,

        "recall_at_0_50":
            recall,

        "f1_at_0_50":
            f1,

        "fpr_at_0_50":
            fpr,

        "fnr_at_0_50":
            fnr,
    }


# =============================================================================
# 3. FAST-METRIC SELF-TEST AGAINST SKLEARN
# =============================================================================

synthetic_y = np.asarray(
    [
        0, 1, 0, 1, 0,
        1, 0, 1, 0, 1,
    ],
    dtype=np.int8,
)

synthetic_p = np.asarray(
    [
        0.1, 0.8, 0.8, 0.8, 0.2,
        0.5, 0.5, 0.3, 0.9, 0.9,
    ],
    dtype=np.float32,
)


syn_b = np.sort(
    synthetic_p[
        synthetic_y == 0
    ]
)

syn_a = synthetic_p[
    synthetic_y == 1
]


fast_ap, fast_roc = (
    fast_family_ranking_metrics(
        syn_b,
        syn_a,
    )
)


sk_ap = float(
    average_precision_score(
        synthetic_y,
        synthetic_p,
    )
)

sk_roc = float(
    roc_auc_score(
        synthetic_y,
        synthetic_p,
    )
)


if not np.isclose(
    fast_ap,
    sk_ap,
    rtol=0.0,
    atol=1e-15,
):

    raise RuntimeError(
        "PR-AUC self-test failed."
    )


if not np.isclose(
    fast_roc,
    sk_roc,
    rtol=0.0,
    atol=1e-15,
):

    raise RuntimeError(
        "ROC-AUC self-test failed."
    )


# =============================================================================
# 4. SEALED REPOSITORY STATE
# =============================================================================

print("=" * 116)
print("STAGE23-6D V2 — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS")
print("=" * 116)
print()


head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-5 tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-6D:\n"
        + status
    )


if OUT.exists():

    raise RuntimeError(
        "Stage23-6D output directory already exists.\n"
        "Do NOT delete/rerun blindly:\n"
        f"{OUT}"
    )


print("[OK] HEAD      :", head)
print("[OK] seal tag  :", EXPECTED_TAG)
print("[OK] worktree  : CLEAN")
print("[OK] Stage23   : 50 / 50 SEALED")
print("[OK] new fits  : 0 authorized")
print(
    "[EXACT] custom PR/ROC implementation "
    "matches sklearn tied-score self-test"
)


# =============================================================================
# 5. VERIFY STAGE23-6C
# =============================================================================

print()
print("=" * 116)
print("VERIFY STAGE23-6C")
print("=" * 116)
print()


checks_6c = [
    (
        ALIGN_PARQUET,
        EXPECTED_ALIGN_PARQUET_SHA,
    ),
    (
        ALIGN_SUPPORT,
        EXPECTED_ALIGN_SUPPORT_SHA,
    ),
    (
        ALIGN_MANIFEST,
        EXPECTED_ALIGN_MANIFEST_SHA,
    ),
    (
        ALIGN_CHECKSUMS,
        EXPECTED_ALIGN_CHECKSUMS_SHA,
    ),
]


for path, expected in checks_6c:

    if not path.is_file():

        raise RuntimeError(
            f"Stage23-6C artifact missing:\n{path}"
        )


    actual = sha256_file(
        path
    )


    print(path.name)
    print("  expected:", expected)
    print("  actual:  ", actual)


    if actual != expected:

        raise RuntimeError(
            f"Stage23-6C SHA mismatch: {path.name}"
        )


    print("  [EXACT]")


align_manifest = json.loads(
    ALIGN_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if (
    align_manifest["status"]
    !=
    "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED_UNSEALED"
):

    raise RuntimeError(
        "Stage23-6C status mismatch."
    )


if (
    align_manifest[
        "verification"
    ][
        "random_label_to_binary_mismatches"
    ]
    != 0
):

    raise RuntimeError(
        "Stage23-6C RANDOM alignment mismatch."
    )


if (
    align_manifest[
        "verification"
    ][
        "chronological_label_to_binary_mismatches"
    ]
    != 0
):

    raise RuntimeError(
        "Stage23-6C CHRONO alignment mismatch."
    )


# =============================================================================
# 6. LOAD FROZEN FAMILY ALIGNMENT
# =============================================================================

attack_alignment = pd.read_parquet(
    ALIGN_PARQUET,
    columns=[
        "clean_position",
        "attack_family",
    ],
)


attack_clean = attack_alignment[
    "clean_position"
].to_numpy(
    dtype=np.int64
)


attack_family = attack_alignment[
    "attack_family"
].astype(
    str
).to_numpy(
    dtype=object
)


if len(
    attack_clean
) != 1_972_299:

    raise RuntimeError(
        "Attack-family alignment row count mismatch."
    )


if not np.all(
    np.diff(
        attack_clean
    ) > 0
):

    raise RuntimeError(
        "Attack-family clean_position not strictly increasing."
    )


support_df = pd.read_csv(
    ALIGN_SUPPORT
)


families = support_df[
    "attack_family"
].astype(
    str
).tolist()


if len(
    families
) != 13:

    raise RuntimeError(
        "Frozen family count != 13."
    )


if len(
    set(families)
) != 13:

    raise RuntimeError(
        "Duplicate frozen attack family."
    )


print()
print(
    "[EXACT] frozen attack families:",
    len(families),
)


# =============================================================================
# 7. PROBABILITY INVENTORY
# =============================================================================

PRIMARY_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)


PROBABILITIES = {
    "RANDOM_NATURAL": {
        "FULL":
            REPO
            / "results"
            / "stage22r_training"
            / "stage22r_2a_random_natural"
            / "random_natural_validation_ensemble_probabilities.npz",
    },

    "CHRONOLOGICAL_NATURAL": {
        "FULL":
            REPO
            / "results"
            / "stage22r_training"
            / "stage22r_2c_chronological_natural"
            / "chronological_natural_validation_ensemble_probabilities.npz",
    },
}


for subset in SUBSET_ORDER[1:]:

    slug = SUBSET_SLUG[
        subset
    ]


    PROBABILITIES[
        "RANDOM_NATURAL"
    ][
        subset
    ] = (
        PRIMARY_ROOT
        / "random"
        / slug
        / (
            f"{slug}_random_natural_"
            "validation_probabilities.npz"
        )
    )


    PROBABILITIES[
        "CHRONOLOGICAL_NATURAL"
    ][
        subset
    ] = (
        PRIMARY_ROOT
        / "chronological"
        / slug
        / (
            f"{slug}_chronological_natural_"
            "validation_probabilities.npz"
        )
    )


# =============================================================================
# 8. VERIFY ALL 14 SEALED PROBABILITY FILES BEFORE WRITING ANYTHING
# =============================================================================

print()
print("=" * 116)
print("SEALED PROBABILITY ARTIFACT VERIFICATION")
print("=" * 116)
print()


probability_sha = {}


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    probability_sha[
        split
    ] = {}


    for subset in SUBSET_ORDER:

        path = PROBABILITIES[
            split
        ][
            subset
        ]


        if (
            split == "RANDOM_NATURAL"
            and subset == "FULL"
        ):

            expected = (
                EXPECTED_FULL_RANDOM_SHA
            )


        elif (
            split == "CHRONOLOGICAL_NATURAL"
            and subset == "FULL"
        ):

            expected = (
                EXPECTED_FULL_CHRONO_SHA
            )


        else:

            expected = None


        actual = verify_probability_artifact(
            path,
            expected_sha=expected,
        )


        probability_sha[
            split
        ][
            subset
        ] = actual


        print(
            f"[EXACT] "
            f"{split:<25} "
            f"{subset:<24} "
            f"{actual}"
        )


print()
print(
    "[EXACT] sealed probability artifacts: 14 / 14"
)


# =============================================================================
# 9. OUTPUT INITIALIZATION
# =============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


state = {
    "stage":
        "Stage23-6D frozen attack-family primary-ablation metrics",

    "status":
        "INITIALIZED",

    "created_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "family_count":
        13,

    "primary_subset_count":
        7,

    "split_count":
        2,

    "expected_metric_rows":
        182,

    "subsets_completed":
        0,

    "model_fits":
        0,

    "model_inference":
        0,

    "lightgbm_imported":
        False,

    "lightgbm_execution":
        0,

    "xgboost_imported":
        False,

    "xgboost_execution":
        0,

    "raw_february_csv_reads":
        0,

    "development_source_parquet_reads":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 10. ATTACK-FAMILY METRICS
# =============================================================================

metric_rows = []


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print("=" * 116)
    print(
        f"ATTACK-FAMILY METRICS — {split}"
    )
    print("=" * 116)
    print()


    expected_counts = (
        EXPECTED_SPLIT_COUNTS[
            split
        ]
    )


    (
        reference_clean,
        reference_y,
        full_probability,
    ) = load_probability_npz(
        PROBABILITIES[
            split
        ][
            "FULL"
        ]
    )


    if (
        len(reference_clean)
        != expected_counts["rows"]
    ):

        raise RuntimeError(
            f"{split}: validation row mismatch."
        )


    if int(
        np.sum(
            reference_y == 1
        )
    ) != expected_counts["attack"]:

        raise RuntimeError(
            f"{split}: attack count mismatch."
        )


    if int(
        np.sum(
            reference_y == 0
        )
    ) != expected_counts["benign"]:

        raise RuntimeError(
            f"{split}: benign count mismatch."
        )


    benign_mask = (
        reference_y == 0
    )


    attack_row_idx = np.flatnonzero(
        reference_y == 1
    )


    validation_attack_clean = (
        reference_clean[
            attack_row_idx
        ]
    )


    join_pos = np.searchsorted(
        attack_clean,
        validation_attack_clean,
    )


    if np.any(
        join_pos
        >= len(attack_clean)
    ):

        raise RuntimeError(
            f"{split}: attack-family join overflow."
        )


    if not np.array_equal(
        attack_clean[
            join_pos
        ],
        validation_attack_clean,
    ):

        raise RuntimeError(
            f"{split}: attack-family join failed."
        )


    validation_attack_family = (
        attack_family[
            join_pos
        ]
    )


    family_row_indices = {
        family:
            attack_row_idx[
                validation_attack_family
                == family
            ]

        for family in families
    }


    support_column = (
        "random_natural_validation_attack_support"
        if split == "RANDOM_NATURAL"
        else
        "chronological_natural_validation_attack_support"
    )


    for family in families:

        expected_support = int(
            support_df.loc[
                support_df[
                    "attack_family"
                ].astype(str)
                == family,
                support_column,
            ].iloc[0]
        )


        actual_support = int(
            len(
                family_row_indices[
                    family
                ]
            )
        )


        if actual_support != expected_support:

            raise RuntimeError(
                f"{split}/{family}: support mismatch.\n"
                f"expected={expected_support}\n"
                f"actual={actual_support}"
            )


    print(
        "[EXACT] validation rows:",
        f"{len(reference_clean):,}",
    )

    print(
        "[EXACT] all 13 family supports match Stage23-6C"
    )


    # -------------------------------------------------------------------------
    # Process seven frozen primary subsets.
    # -------------------------------------------------------------------------

    for subset_number, subset in enumerate(
        SUBSET_ORDER,
        start=1,
    ):

        if subset == "FULL":

            clean = reference_clean
            y = reference_y
            probability = full_probability


        else:

            (
                clean,
                y,
                probability,
            ) = load_probability_npz(
                PROBABILITIES[
                    split
                ][
                    subset
                ]
            )


            if not np.array_equal(
                clean,
                reference_clean,
            ):

                raise RuntimeError(
                    f"{split}/{subset}: "
                    "clean_position differs from FULL."
                )


            if not np.array_equal(
                y,
                reference_y,
            ):

                raise RuntimeError(
                    f"{split}/{subset}: "
                    "binary_label differs from FULL."
                )


        benign_scores = probability[
            benign_mask
        ]


        benign_sorted = np.sort(
            benign_scores
        )


        # ---------------------------------------------------------------------
        # FULL aggregate metric verification against frozen result.
        # Verification only; not a new reported family metric.
        # ---------------------------------------------------------------------

        if subset == "FULL":

            overall_attack_scores = (
                probability[
                    attack_row_idx
                ]
            )


            check_pr, check_roc = (
                fast_family_ranking_metrics(
                    benign_sorted,
                    overall_attack_scores,
                )
            )


            if not np.isclose(
                check_pr,
                EXPECTED_FULL_METRICS[
                    split
                ][
                    "PR_AUC"
                ],
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    f"{split}: frozen FULL PR-AUC verification failed.\n"
                    f"expected="
                    f"{EXPECTED_FULL_METRICS[split]['PR_AUC']}\n"
                    f"actual={check_pr}"
                )


            if not np.isclose(
                check_roc,
                EXPECTED_FULL_METRICS[
                    split
                ][
                    "ROC_AUC"
                ],
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    f"{split}: frozen FULL ROC-AUC verification failed.\n"
                    f"expected="
                    f"{EXPECTED_FULL_METRICS[split]['ROC_AUC']}\n"
                    f"actual={check_roc}"
                )


            print(
                f"[EXACT] FULL verification "
                f"PR={check_pr:.12f} "
                f"ROC={check_roc:.12f}"
            )


        # ---------------------------------------------------------------------
        # Frozen family metrics only.
        # ---------------------------------------------------------------------

        for family in families:

            family_idx = (
                family_row_indices[
                    family
                ]
            )


            attack_scores = (
                probability[
                    family_idx
                ]
            )


            support_attack = int(
                len(
                    attack_scores
                )
            )


            support_benign = int(
                len(
                    benign_scores
                )
            )


            pr_auc, roc_auc = (
                fast_family_ranking_metrics(
                    benign_sorted,
                    attack_scores,
                )
            )


            operating = (
                frozen_operating_metrics(
                    benign_scores,
                    attack_scores,
                )
            )


            metric_rows.append(
                {
                    "split":
                        split,

                    "subset":
                        subset,

                    "attack_family":
                        family,

                    "interpretation_status":
                        (
                            "INTERPRETABLE"
                            if support_attack
                            >= MIN_INTERPRETATION_SUPPORT
                            else
                            "DESCRIPTIVE_ONLY"
                        ),

                    "support_attack":
                        support_attack,

                    "support_benign":
                        support_benign,

                    "PR_AUC":
                        pr_auc,

                    "ROC_AUC":
                        roc_auc,

                    "precision_at_0_50":
                        operating[
                            "precision_at_0_50"
                        ],

                    "recall_at_0_50":
                        operating[
                            "recall_at_0_50"
                        ],

                    "f1_at_0_50":
                        operating[
                            "f1_at_0_50"
                        ],

                    "fpr_at_0_50":
                        operating[
                            "fpr_at_0_50"
                        ],

                    "fnr_at_0_50":
                        operating[
                            "fnr_at_0_50"
                        ],

                    "delta_f1_FULL_minus_ablated":
                        np.nan,

                    "delta_recall_FULL_minus_ablated":
                        np.nan,
                }
            )


        print(
            f"[{subset_number}/7] "
            f"{split:<25} "
            f"{subset:<24} COMPLETE"
        )


        state[
            "subsets_completed"
        ] += 1

        state[
            "status"
        ] = "METRICS_IN_PROGRESS"

        state[
            "last_completed_split"
        ] = split

        state[
            "last_completed_subset"
        ] = subset


        write_json(
            STATE_JSON,
            state,
        )


        del (
            benign_scores,
            benign_sorted,
        )


        if subset != "FULL":

            del (
                clean,
                y,
                probability,
            )


        gc.collect()


    del (
        reference_clean,
        reference_y,
        full_probability,
        benign_mask,
        attack_row_idx,
        validation_attack_clean,
        join_pos,
        validation_attack_family,
        family_row_indices,
    )

    gc.collect()


# =============================================================================
# 11. EXACT ROW COUNT
# =============================================================================

metrics_df = pd.DataFrame(
    metric_rows
)


EXPECTED_METRIC_ROWS = (
    2
    * 7
    * 13
)


if len(
    metrics_df
) != EXPECTED_METRIC_ROWS:

    raise RuntimeError(
        "Metric row-count mismatch.\n"
        f"expected={EXPECTED_METRIC_ROWS}\n"
        f"actual={len(metrics_df)}"
    )


# =============================================================================
# 12. FROZEN FULL-MINUS-ABLATED DELTAS
# =============================================================================

for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    for family in families:

        full_mask = (
            (
                metrics_df["split"]
                == split
            )
            &
            (
                metrics_df["subset"]
                == "FULL"
            )
            &
            (
                metrics_df["attack_family"]
                == family
            )
        )


        if int(
            full_mask.sum()
        ) != 1:

            raise RuntimeError(
                f"FULL row not unique: "
                f"{split}/{family}"
            )


        full_f1 = metrics_df.loc[
            full_mask,
            "f1_at_0_50",
        ].iloc[0]


        full_recall = metrics_df.loc[
            full_mask,
            "recall_at_0_50",
        ].iloc[0]


        for subset in SUBSET_ORDER:

            row_mask = (
                (
                    metrics_df["split"]
                    == split
                )
                &
                (
                    metrics_df["subset"]
                    == subset
                )
                &
                (
                    metrics_df["attack_family"]
                    == family
                )
            )


            if int(
                row_mask.sum()
            ) != 1:

                raise RuntimeError(
                    f"Metric row not unique: "
                    f"{split}/{subset}/{family}"
                )


            ablated_f1 = metrics_df.loc[
                row_mask,
                "f1_at_0_50",
            ].iloc[0]


            ablated_recall = metrics_df.loc[
                row_mask,
                "recall_at_0_50",
            ].iloc[0]


            if (
                pd.notna(full_f1)
                and pd.notna(ablated_f1)
            ):

                metrics_df.loc[
                    row_mask,
                    "delta_f1_FULL_minus_ablated",
                ] = (
                    float(full_f1)
                    - float(ablated_f1)
                )


            if (
                pd.notna(full_recall)
                and pd.notna(
                    ablated_recall
                )
            ):

                metrics_df.loc[
                    row_mask,
                    "delta_recall_FULL_minus_ablated",
                ] = (
                    float(full_recall)
                    - float(
                        ablated_recall
                    )
                )


# =============================================================================
# 13. FROZEN METRIC-SCHEMA ASSERTION
# =============================================================================

actual_metric_columns = [
    column
    for column in metrics_df.columns
    if column in FROZEN_METRIC_COLUMNS
]


if actual_metric_columns != FROZEN_METRIC_COLUMNS:

    raise RuntimeError(
        "Frozen metric column order mismatch.\n"
        f"expected={FROZEN_METRIC_COLUMNS}\n"
        f"actual={actual_metric_columns}"
    )


# No unauthorized numerical scientific metric columns.
authorized_non_metric_columns = {
    "split",
    "subset",
    "attack_family",
    "interpretation_status",
}


all_allowed_columns = (
    list(
        authorized_non_metric_columns
    )
    +
    FROZEN_METRIC_COLUMNS
)


unexpected_columns = (
    set(
        metrics_df.columns
    )
    -
    set(
        all_allowed_columns
    )
)


if unexpected_columns:

    raise RuntimeError(
        "Unauthorized metric/output columns detected:\n"
        + repr(
            sorted(
                unexpected_columns
            )
        )
    )


# FULL deltas must be zero wherever family support > 0.
full_defined = metrics_df[
    (
        metrics_df["subset"]
        == "FULL"
    )
    &
    (
        metrics_df["support_attack"]
        > 0
    )
]


if not np.allclose(
    full_defined[
        "delta_f1_FULL_minus_ablated"
    ].to_numpy(
        dtype=float
    ),
    0.0,
    rtol=0.0,
    atol=0.0,
):

    raise RuntimeError(
        "FULL F1 deltas are not exactly zero."
    )


if not np.allclose(
    full_defined[
        "delta_recall_FULL_minus_ablated"
    ].to_numpy(
        dtype=float
    ),
    0.0,
    rtol=0.0,
    atol=0.0,
):

    raise RuntimeError(
        "FULL recall deltas are not exactly zero."
    )


# =============================================================================
# 14. WRITE ONLY FROZEN SCIENTIFIC METRICS
# =============================================================================

metrics_df.to_csv(
    METRICS_CSV,
    index=False,
)


write_json(
    METRICS_JSON,
    {
        "stage":
            "Stage23-6D",

        "status":
            "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED",

        "evaluation_definition":
            (
                "For attack family A, form a validation "
                "evaluation set containing all benign rows "
                "from that split plus attack rows belonging to A."
            ),

        "minimum_attack_support_for_interpretation":
            MIN_INTERPRETATION_SUPPORT,

        "family_set":
            families,

        "subsets":
            SUBSET_ORDER,

        "splits":
            [
                "RANDOM_NATURAL",
                "CHRONOLOGICAL_NATURAL",
            ],

        "frozen_metrics":
            FROZEN_METRIC_COLUMNS,

        "rows":
            records_json_safe(
                metrics_df
            ),
    },
)


# =============================================================================
# 15. FULL ENSEMBLE FAMILY BASELINES
# =============================================================================

print()
print("=" * 116)
print("FULL ENSEMBLE — ATTACK-FAMILY BASELINES")
print("=" * 116)


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print(split)
    print("-" * 116)


    g = metrics_df[
        (
            metrics_df["split"]
            == split
        )
        &
        (
            metrics_df["subset"]
            == "FULL"
        )
    ]


    for row in g.itertuples(
        index=False
    ):

        if row.support_attack == 0:

            print(
                f"{row.attack_family:<32} "
                f"support={row.support_attack:>7,} "
                f"{row.interpretation_status:<16} "
                f"PR=NA ROC=NA "
                f"Precision=NA Recall=NA "
                f"F1=NA FNR=NA "
                f"FPR={row.fpr_at_0_50:.9f}"
            )

        else:

            print(
                f"{row.attack_family:<32} "
                f"support={row.support_attack:>7,} "
                f"{row.interpretation_status:<16} "
                f"PR={row.PR_AUC:.9f} "
                f"ROC={row.ROC_AUC:.9f} "
                f"Precision={row.precision_at_0_50:.9f} "
                f"Recall={row.recall_at_0_50:.9f} "
                f"F1={row.f1_at_0_50:.9f} "
                f"FPR={row.fpr_at_0_50:.9f} "
                f"FNR={row.fnr_at_0_50:.9f}"
            )


# =============================================================================
# 16. ALL INTERPRETABLE PRIMARY ABLATION DELTAS
#
# No ranking / no top-K / no aggregate mean / no post-hoc selection.
# Every interpretable frozen family is printed.
# =============================================================================

print()
print("=" * 116)
print("PRIMARY ABLATION DELTAS — ALL INTERPRETABLE FAMILIES")
print("=" * 116)
print()

print(
    "Sign: positive FULL-minus-ablated = "
    "ablation performs worse than FULL."
)


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    print()
    print(split)
    print("-" * 116)


    g = metrics_df[
        (
            metrics_df["split"]
            == split
        )
        &
        (
            metrics_df[
                "interpretation_status"
            ]
            == "INTERPRETABLE"
        )
        &
        (
            metrics_df["subset"]
            != "FULL"
        )
    ]


    for subset in SUBSET_ORDER[1:]:

        print()
        print(subset)


        s = g[
            g["subset"]
            == subset
        ]


        for row in s.itertuples(
            index=False
        ):

            print(
                f"  {row.attack_family:<30} "
                f"support={row.support_attack:>7,} "
                f"ΔF1="
                f"{row.delta_f1_FULL_minus_ablated:+.9f} "
                f"ΔRecall="
                f"{row.delta_recall_FULL_minus_ablated:+.9f}"
            )


# =============================================================================
# 17. CHRONOLOGICAL INFILTERATION EXACT METRICS
# =============================================================================

print()
print("=" * 116)
print("CHRONOLOGICAL_NATURAL — INFILTERATION")
print("=" * 116)
print()


chrono_inf = metrics_df[
    (
        metrics_df["split"]
        == "CHRONOLOGICAL_NATURAL"
    )
    &
    (
        metrics_df["attack_family"]
        == "Infilteration"
    )
]


for subset in SUBSET_ORDER:

    row = chrono_inf[
        chrono_inf["subset"]
        == subset
    ].iloc[0]


    print(
        f"{subset:<24} "
        f"support={int(row['support_attack']):>7,} "
        f"PR={row['PR_AUC']:.9f} "
        f"ROC={row['ROC_AUC']:.9f} "
        f"Precision={row['precision_at_0_50']:.9f} "
        f"Recall={row['recall_at_0_50']:.9f} "
        f"F1={row['f1_at_0_50']:.9f} "
        f"FPR={row['fpr_at_0_50']:.9f} "
        f"FNR={row['fnr_at_0_50']:.9f} "
        f"ΔF1="
        f"{row['delta_f1_FULL_minus_ablated']:+.9f} "
        f"ΔRecall="
        f"{row['delta_recall_FULL_minus_ablated']:+.9f}"
    )


# =============================================================================
# 18. MANIFEST
# =============================================================================

manifest = {
    "stage":
        "Stage23-6D",

    "status":
        "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED",

    "completed_utc":
        utc_now(),

    "parent_commit":
        EXPECTED_HEAD,

    "parent_tag":
        EXPECTED_TAG,

    "stage23_6c": {
        "attack_alignment_sha256":
            EXPECTED_ALIGN_PARQUET_SHA,

        "support_sha256":
            EXPECTED_ALIGN_SUPPORT_SHA,

        "manifest_sha256":
            EXPECTED_ALIGN_MANIFEST_SHA,

        "checksum_manifest_sha256":
            EXPECTED_ALIGN_CHECKSUMS_SHA,
    },

    "evaluation": {
        "family_count":
            13,

        "subsets":
            SUBSET_ORDER,

        "splits":
            [
                "RANDOM_NATURAL",
                "CHRONOLOGICAL_NATURAL",
            ],

        "metric_rows":
            int(
                len(metrics_df)
            ),

        "frozen_metrics":
            FROZEN_METRIC_COLUMNS,

        "minimum_attack_support_for_interpretation":
            100,

        "fixed_threshold":
            0.5,

        "probability_source":
            "persisted float32 ensemble_probability",

        "zero_support_policy":
            (
                "For support_attack=0, attack-dependent "
                "metrics are undefined/null; FPR remains "
                "defined from all benign validation rows."
            ),
    },

    "probability_artifact_sha256":
        probability_sha,

    "artifacts": {
        METRICS_CSV.name: {
            "rows":
                int(
                    len(metrics_df)
                ),

            "sha256":
                sha256_file(
                    METRICS_CSV
                ),
        },

        METRICS_JSON.name: {
            "sha256":
                sha256_file(
                    METRICS_JSON
                ),
        },
    },

    "governance": {
        "stage23_model_fit_budget":
            "50 / 50 SEALED",

        "additional_model_fits_authorized":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_imported":
            False,

        "lightgbm_execution":
            0,

        "xgboost_imported":
            False,

        "xgboost_execution":
            0,

        "raw_february_csv_reads":
            0,

        "development_source_parquet_reads":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "threshold_optimization":
            False,

        "new_subset":
            False,

        "new_scientific_metric":
            False,

        "post_result_family_selection":
            False,
    },

    "interpretation_boundary": {
        "chronological_family_coverage":
            (
                "The frozen chronological validation "
                "contains Infilteration attacks only. "
                "The other frozen development families have "
                "zero chronological attack support and are "
                "retained descriptively."
            ),

        "causality":
            (
                "Family-specific ablation differences are "
                "descriptive associations under frozen "
                "memberships and do not establish causality."
            ),
    },
}


write_json(
    MANIFEST_JSON,
    manifest,
)


state[
    "status"
] = (
    "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED"
)

state[
    "completed_utc"
] = utc_now()

state[
    "subsets_completed"
] = 14

state[
    "metric_rows"
] = int(
    len(metrics_df)
)

state[
    "next_action"
] = (
    "REVIEW_THEN_ZERO_FIT_SEAL_STAGE23_6C_AND_6D"
)


write_json(
    STATE_JSON,
    state,
)


# =============================================================================
# 19. CHECKSUMS
# =============================================================================

artifact_files = sorted(
    p
    for p in OUT.iterdir()
    if (
        p.is_file()
        and p != CHECKSUMS
    )
)


CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(p)}  {p.name}"
        )
        for p in artifact_files
    ) + "\n",
    encoding="utf-8",
)


# =============================================================================
# 20. FINAL
# =============================================================================

print()
print("=" * 116)
print("STAGE23-6D ATTACK-FAMILY METRICS COMPLETE — UNSEALED")
print("=" * 116)
print()

print("Frozen development families : 13")
print("Primary subsets              : 7 / 7")
print("Splits                       : 2 / 2")
print("Probability artifacts        : 14 / 14")
print("Family metric rows           :", len(metrics_df))

print()
print(
    "RANDOM interpretable families:",
    int(
        (
            support_df[
                "random_natural_validation_attack_support"
            ]
            >= 100
        ).sum()
    ),
)

print(
    "CHRONO interpretable families:",
    int(
        (
            support_df[
                "chronological_natural_validation_attack_support"
            ]
            >= 100
        ).sum()
    ),
)

print()
print("Metrics CSV:")
print(" ", METRICS_CSV)
print(" SHA256:")
print(" ", sha256_file(METRICS_CSV))

print()
print("Metrics JSON:")
print(" ", METRICS_JSON)
print(" SHA256:")
print(" ", sha256_file(METRICS_JSON))

print()
print("Manifest:")
print(" ", MANIFEST_JSON)
print(" SHA256:")
print(" ", sha256_file(MANIFEST_JSON))

print()
print("Checksum manifest SHA256:")
print(" ", sha256_file(CHECKSUMS))

print()
print("GOVERNANCE")
print("  Stage23 fit state        : 50 / 50 SEALED")
print("  New model fits           : 0")
print("  Model inference          : 0")
print("  LightGBM imported        : NO")
print("  LightGBM execution       : 0")
print("  XGBoost imported         : NO")
print("  XGBoost execution        : 0")
print("  Threshold optimization   : NO")
print("  New scientific metrics   : NO")
print("  Raw February CSV reads   : 0")
print("  Development source reads : 0")
print("  Raw Mar1 accessed        : NO")
print("  Raw Mar2 accessed        : NO")

print()
print("NEXT:")
print("  Review Stage23-6C + Stage23-6D.")
print("  Then ZERO-FIT seal and push the attack-family block.")

print("=" * 116)

STAGE23-6D V2 — FROZEN ATTACK-FAMILY PRIMARY-ABLATION METRICS

[OK] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[OK] seal tag  : stage23-5-shap-proxy-absorption-complete-v1
[OK] worktree  : CLEAN
[OK] Stage23   : 50 / 50 SEALED
[OK] new fits  : 0 authorized
[EXACT] custom PR/ROC implementation matches sklearn tied-score self-test

VERIFY STAGE23-6C

retained_development_attack_family_labels.parquet
  expected: 3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  actual:   3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  [EXACT]
attack_family_support_by_split.csv
  expected: d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  actual:   d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  [EXACT]
stage23_6c_alignment_manifest.json
  expected: bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  actual:   bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  [EXACT]
checksums.sha256
  expected

In [10]:
# =============================================================================
# STAGE23-6 — ZERO-FIT ATTACK-FAMILY BLOCK SEAL + GITHUB PUSH
#
# Seals:
#   Stage23-6C exact attack-family alignment
#   Stage23-6D frozen primary-ablation family metrics
#
# FAIL CLOSED
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM
# ZERO XGBOOST
# ZERO SHAP
# ZERO SCIENTIFIC METRIC COMPUTATION
# ZERO RAW DATA READS
# NO MAR1 / MAR2
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import os
import stat

import pandas as pd


# =============================================================================
# 0. CONSTANTS
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "b927499efdf7d4dd5013054cc796f1115f879f8b"
)

EXPECTED_PARENT_TAG = (
    "stage23-5-shap-proxy-absorption-complete-v1"
)


SOURCE_6C = Path(
    "/kaggle/working/stage23_6c_attack_family_alignment"
)

SOURCE_6D = Path(
    "/kaggle/working/stage23_6d_attack_family_metrics"
)


DEST_ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_6_attack_family_analysis"
)

DEST_6C = (
    DEST_ROOT
    / "stage23_6c_attack_family_alignment"
)

DEST_6D = (
    DEST_ROOT
    / "stage23_6d_attack_family_metrics"
)


SEAL_RECEIPT = (
    DEST_ROOT
    / "stage23_6_attack_family_block_seal_receipt.json"
)

REPO_CHECKSUMS = (
    DEST_ROOT
    / "stage23_6_attack_family_block_repository_checksums.sha256"
)


SEAL_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: seal frozen attack-family analysis"
)


# Exact successful Stage23-6C hashes.

EXPECTED_6C = {
    "retained_development_attack_family_labels.parquet":
        "3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26",

    "attack_family_support_by_split.csv":
        "d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377",

    "stage23_6c_alignment_manifest.json":
        "bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8",

    "checksums.sha256":
        "a54e01b733e093602a03b9b8bff5430f7cdfa4e987e0a36577ee686e736113b4",
}


# Exact successful Stage23-6D hashes.

EXPECTED_6D = {
    "stage23_6d_attack_family_metrics.csv":
        "43ceb003356dcf4b2272629b685331161221ac067f4c58164c50af1f83fa98e4",

    "stage23_6d_attack_family_metrics.json":
        "e0a3656dbc577e917f4b0644cc636b1aeda2c39b24e544b2cee340ce2e4f6247",

    "stage23_6d_manifest.json":
        "2ad711416e0c540165fb3b5912ac4def28028763e912e72c28536891637dfc1d",

    "checksums.sha256":
        "72c11c90e2db332d05ac40e414df01ff895e822e4b40cb09c31bd251e51e110e",
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        ) + "\n",
        encoding="utf-8",
    )


def git(
    *args,
    check=True,
    env=None,
    show=False,
):

    merged_env = os.environ.copy()

    if env:
        merged_env.update(
            env
        )


    p = subprocess.run(
        [
            "git",
            *map(
                str,
                args,
            ),
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )


    output = (
        p.stdout
        or ""
    ).strip()


    if show and output:
        print(output)


    if (
        check
        and p.returncode != 0
    ):

        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n"
            f"{output}"
        )


    return (
        p.returncode,
        output,
    )


def verify_manifest(
    root,
    manifest_path,
):

    count = 0


    for line in Path(
        manifest_path
    ).read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue


        digest, relative = line.split(
            maxsplit=1
        )


        relative = relative.strip()

        path = (
            root
            / relative
        )


        if not path.is_file():

            raise RuntimeError(
                "Checksum manifest references missing file:\n"
                f"{path}"
            )


        actual = sha256_file(
            path
        )


        if actual != digest:

            raise RuntimeError(
                "Checksum mismatch:\n"
                f"{relative}\n"
                f"expected={digest}\n"
                f"actual={actual}"
            )


        count += 1


    return count


def verify_named_hashes(
    root,
    expected,
):

    for name, digest in expected.items():

        path = (
            root
            / name
        )


        if not path.is_file():

            raise RuntimeError(
                f"Required artifact missing:\n{path}"
            )


        actual = sha256_file(
            path
        )


        print(name)
        print(
            "  expected:",
            digest,
        )
        print(
            "  actual:  ",
            actual,
        )


        if actual != digest:

            raise RuntimeError(
                f"SHA mismatch: {name}"
            )


        print("  [EXACT]")


def copy_tree_exact(
    source,
    destination,
):

    if destination.exists():

        raise RuntimeError(
            "Destination already exists:\n"
            f"{destination}"
        )


    shutil.copytree(
        source,
        destination,
        copy_function=shutil.copy2,
    )


# =============================================================================
# 2. BEGIN
# =============================================================================

print("=" * 116)
print("STAGE23-6 — ZERO-FIT ATTACK-FAMILY BLOCK SEAL + PUSH")
print("=" * 116)
print()


# =============================================================================
# 3. VERIFY EXACT SEALED PARENT
# =============================================================================

if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, parent_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

_, status = git(
    "status",
    "--porcelain",
)


if branch != "main":

    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected repository parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )


if parent_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-5 parent tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-6 seal:\n"
        + status
    )


print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] worktree  : CLEAN")
print("[EXACT] Stage23 fit state: 50 / 50 SEALED")


# =============================================================================
# 4. TAG MUST NOT EXIST
# =============================================================================

_, local_tag = git(
    "tag",
    "--list",
    SEAL_TAG,
)


if local_tag:

    raise RuntimeError(
        f"Seal tag already exists locally:\n{SEAL_TAG}"
    )


_, remote_tag = git(
    "ls-remote",
    "--tags",
    "origin",
    f"refs/tags/{SEAL_TAG}",
)


if remote_tag:

    raise RuntimeError(
        f"Seal tag already exists remotely:\n{SEAL_TAG}"
    )


# =============================================================================
# 5. VERIFY STAGE23-6C
# =============================================================================

print()
print("=" * 116)
print("VERIFY STAGE23-6C")
print("=" * 116)
print()


if not SOURCE_6C.is_dir():

    raise RuntimeError(
        f"Stage23-6C source missing:\n{SOURCE_6C}"
    )


verify_named_hashes(
    SOURCE_6C,
    EXPECTED_6C,
)


state_6c_path = (
    SOURCE_6C
    / "execution_state.json"
)

if not state_6c_path.is_file():

    raise RuntimeError(
        "Stage23-6C execution state missing."
    )


state_6c = json.loads(
    state_6c_path.read_text(
        encoding="utf-8"
    )
)


if (
    state_6c.get("status")
    !=
    "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED_UNSEALED"
):

    raise RuntimeError(
        "Stage23-6C incomplete."
    )


if state_6c.get(
    "model_fits"
) != 0:

    raise RuntimeError(
        "Stage23-6C model_fits != 0."
    )


if state_6c.get(
    "model_inference"
) != 0:

    raise RuntimeError(
        "Stage23-6C model_inference != 0."
    )


if state_6c.get(
    "lightgbm_execution"
) != 0:

    raise RuntimeError(
        "Stage23-6C LightGBM execution != 0."
    )


if state_6c.get(
    "xgboost_execution"
) != 0:

    raise RuntimeError(
        "Stage23-6C XGBoost execution != 0."
    )


if state_6c.get(
    "attack_family_count"
) != 13:

    raise RuntimeError(
        "Stage23-6C family count != 13."
    )


manifest_entries_6c = verify_manifest(
    SOURCE_6C,
    SOURCE_6C
    / "checksums.sha256",
)


print()
print(
    "[EXACT] Stage23-6C checksum entries:",
    manifest_entries_6c,
)


# =============================================================================
# 6. VERIFY STAGE23-6D
# =============================================================================

print()
print("=" * 116)
print("VERIFY STAGE23-6D")
print("=" * 116)
print()


if not SOURCE_6D.is_dir():

    raise RuntimeError(
        f"Stage23-6D source missing:\n{SOURCE_6D}"
    )


verify_named_hashes(
    SOURCE_6D,
    EXPECTED_6D,
)


state_6d_path = (
    SOURCE_6D
    / "execution_state.json"
)

if not state_6d_path.is_file():

    raise RuntimeError(
        "Stage23-6D execution state missing."
    )


state_6d = json.loads(
    state_6d_path.read_text(
        encoding="utf-8"
    )
)


if (
    state_6d.get("status")
    !=
    "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Stage23-6D incomplete."
    )


if state_6d.get(
    "model_fits"
) != 0:

    raise RuntimeError(
        "Stage23-6D model_fits != 0."
    )


if state_6d.get(
    "model_inference"
) != 0:

    raise RuntimeError(
        "Stage23-6D model_inference != 0."
    )


if state_6d.get(
    "lightgbm_execution"
) != 0:

    raise RuntimeError(
        "Stage23-6D LightGBM execution != 0."
    )


if state_6d.get(
    "xgboost_execution"
) != 0:

    raise RuntimeError(
        "Stage23-6D XGBoost execution != 0."
    )


if state_6d.get(
    "metric_rows"
) != 182:

    raise RuntimeError(
        "Stage23-6D metric row count != 182."
    )


manifest_entries_6d = verify_manifest(
    SOURCE_6D,
    SOURCE_6D
    / "checksums.sha256",
)


print()
print(
    "[EXACT] Stage23-6D checksum entries:",
    manifest_entries_6d,
)


# =============================================================================
# 7. SCIENTIFIC SCHEMA / SUPPORT SANITY — NO NEW METRICS
# =============================================================================

metrics = pd.read_csv(
    SOURCE_6D
    / "stage23_6d_attack_family_metrics.csv"
)


if len(metrics) != 182:

    raise RuntimeError(
        "Stage23-6D CSV row count != 182."
    )


expected_columns = [
    "split",
    "subset",
    "attack_family",
    "interpretation_status",
    "support_attack",
    "support_benign",
    "PR_AUC",
    "ROC_AUC",
    "precision_at_0_50",
    "recall_at_0_50",
    "f1_at_0_50",
    "fpr_at_0_50",
    "fnr_at_0_50",
    "delta_f1_FULL_minus_ablated",
    "delta_recall_FULL_minus_ablated",
]


if list(
    metrics.columns
) != expected_columns:

    raise RuntimeError(
        "Stage23-6D frozen metric schema mismatch."
    )


if metrics[
    "attack_family"
].nunique() != 13:

    raise RuntimeError(
        "Stage23-6D family universe != 13."
    )


if metrics[
    "subset"
].nunique() != 7:

    raise RuntimeError(
        "Stage23-6D primary subset count != 7."
    )


if metrics[
    "split"
].nunique() != 2:

    raise RuntimeError(
        "Stage23-6D split count != 2."
    )


support = pd.read_csv(
    SOURCE_6C
    / "attack_family_support_by_split.csv"
)


random_interpretable = int(
    (
        support[
            "random_natural_validation_attack_support"
        ]
        >= 100
    ).sum()
)

chrono_interpretable = int(
    (
        support[
            "chronological_natural_validation_attack_support"
        ]
        >= 100
    ).sum()
)


if random_interpretable != 11:

    raise RuntimeError(
        "Expected 11 RANDOM interpretable families."
    )


if chrono_interpretable != 1:

    raise RuntimeError(
        "Expected 1 CHRONO interpretable family."
    )


print()
print("[EXACT] metric rows              : 182")
print("[EXACT] frozen families          : 13")
print("[EXACT] primary subsets          : 7")
print("[EXACT] splits                   : 2")
print("[EXACT] RANDOM interpretable     : 11")
print("[EXACT] CHRONO interpretable     : 1")


# =============================================================================
# 8. FILE-SIZE SAFETY BEFORE REPOSITORY MODIFICATION
# =============================================================================

print()
print("=" * 116)
print("GITHUB FILE-SIZE SAFETY")
print("=" * 116)
print()


source_files = [
    p
    for root in [
        SOURCE_6C,
        SOURCE_6D,
    ]
    for p in root.rglob("*")
    if p.is_file()
]


largest = max(
    source_files,
    key=lambda p:
        p.stat().st_size,
)


largest_mib = (
    largest.stat().st_size
    / (
        1024 ** 2
    )
)


print(
    "Largest artifact:",
    largest,
)

print(
    "Size:",
    f"{largest_mib:.2f} MiB",
)


if (
    largest.stat().st_size
    >= 95
    * 1024
    * 1024
):

    raise RuntimeError(
        "A Stage23-6 artifact is >=95 MiB.\n"
        "REFUSING GitHub commit."
    )


print(
    "[OK] all Stage23-6 artifacts below safety limit."
)


# =============================================================================
# 9. GITHUB AUTH — FRESH-SESSION SAFE
#
# Prefer environment token, then Kaggle Secrets.
# Token is NEVER printed and NEVER written into repository.
# =============================================================================

github_token = (
    os.environ.get(
        "GITHUB_TOKEN"
    )
    or os.environ.get(
        "GH_TOKEN"
    )
)


if not github_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        secrets = (
            UserSecretsClient()
        )


        for secret_name in [
            "GITHUB_TOKEN",
            "github_token",
            "GH_TOKEN",
            "github_pat",
            "GITHUB_PAT",
        ]:

            try:

                candidate = (
                    secrets.get_secret(
                        secret_name
                    )
                )

            except Exception:

                candidate = None


            if candidate:

                github_token = (
                    candidate.strip()
                )

                print(
                    "[OK] GitHub credential loaded from Kaggle Secrets:",
                    secret_name,
                )

                break

    except Exception:

        pass


if not github_token:

    raise RuntimeError(
        "GitHub token unavailable.\n"
        "No repository files have been modified."
    )


ASKPASS = Path(
    "/kaggle/working/.stage23_git_askpass.sh"
)


ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$STAGE23_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


ASKPASS.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)


auth_env = {
    "GIT_ASKPASS":
        str(
            ASKPASS
        ),

    "GIT_TERMINAL_PROMPT":
        "0",

    "STAGE23_GITHUB_TOKEN":
        github_token,
}


# =============================================================================
# 10. PUSH AUTH PREFLIGHT BEFORE REPOSITORY MODIFICATION
# =============================================================================

print()
print("=" * 116)
print("GITHUB PUSH AUTH PREFLIGHT")
print("=" * 116)
print()


rc, dryrun = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
    env=auth_env,
)


if rc != 0:

    try:
        ASKPASS.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "GitHub push authentication failed.\n"
        "No repository artifacts were modified.\n"
        + dryrun
    )


print(
    "[OK] GitHub push authentication available."
)


# =============================================================================
# 11. ONLY NOW MODIFY REPOSITORY
# =============================================================================

print()
print("=" * 116)
print("COPY STAGE23-6C + STAGE23-6D")
print("=" * 116)
print()


if DEST_ROOT.exists():

    raise RuntimeError(
        "Stage23-6 repository destination already exists:\n"
        f"{DEST_ROOT}"
    )


DEST_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


copy_tree_exact(
    SOURCE_6C,
    DEST_6C,
)

copy_tree_exact(
    SOURCE_6D,
    DEST_6D,
)


print("[OK] Stage23-6C copied.")
print("[OK] Stage23-6D copied.")


# =============================================================================
# 12. VERIFY COPIED HASHES
# =============================================================================

for name, digest in EXPECTED_6C.items():

    copied = (
        DEST_6C
        / name
    )


    if sha256_file(
        copied
    ) != digest:

        raise RuntimeError(
            f"Copied Stage23-6C hash mismatch: {name}"
        )


for name, digest in EXPECTED_6D.items():

    copied = (
        DEST_6D
        / name
    )


    if sha256_file(
        copied
    ) != digest:

        raise RuntimeError(
            f"Copied Stage23-6D hash mismatch: {name}"
        )


print(
    "[EXACT] copied scientific artifacts preserved byte-for-byte."
)


# =============================================================================
# 13. SEAL RECEIPT
# =============================================================================

write_json(
    SEAL_RECEIPT,
    {
        "stage":
            "Stage23-6 frozen attack-family analysis",

        "status":
            "SEALED",

        "sealed_utc":
            utc_now(),

        "execution_parent": {
            "commit":
                EXPECTED_PARENT,

            "tag":
                EXPECTED_PARENT_TAG,
        },

        "stage23_6c": {
            "status":
                "EXACT_ATTACK_FAMILY_ALIGNMENT_MATERIALIZED",

            "development_attack_families":
                13,

            "retained_development_attacks":
                1_972_299,

            "random_validation_rows_verified":
                2_882_481,

            "chronological_validation_rows_verified":
                593_780,

            "random_label_binary_mismatches":
                0,

            "chronological_label_binary_mismatches":
                0,

            "artifacts_sha256":
                EXPECTED_6C,
        },

        "stage23_6d": {
            "status":
                "FROZEN_ATTACK_FAMILY_PRIMARY_ABLATION_METRICS_COMPLETE",

            "family_count":
                13,

            "primary_subsets":
                7,

            "splits":
                2,

            "probability_artifacts":
                14,

            "family_metric_rows":
                182,

            "random_interpretable_families":
                11,

            "chronological_interpretable_families":
                1,

            "artifacts_sha256":
                EXPECTED_6D,
        },

        "governance": {
            "stage23_model_fit_budget":
                "50 / 50 SEALED",

            "additional_model_fits_authorized":
                0,

            "model_fits_during_stage23_6c":
                0,

            "model_fits_during_stage23_6d":
                0,

            "model_fits_during_seal":
                0,

            "model_inference_during_stage23_6":
                0,

            "lightgbm_execution_stage23_6":
                0,

            "xgboost_execution_stage23_6":
                0,

            "scientific_metrics_calculated_during_seal":
                0,

            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "retraining":
                False,
        },

        "interpretation_boundary": {
            "family_coverage":
                (
                    "The frozen chronological validation contains "
                    "Infilteration attacks only. Other frozen development "
                    "families have zero chronological attack support."
                ),

            "causality":
                (
                    "Family-specific ablation effects are descriptive "
                    "associations under frozen memberships and do not "
                    "establish causal feature dependence or leakage."
                ),
        },

        "tag":
            SEAL_TAG,
    },
)


# =============================================================================
# 14. REPOSITORY CHECKSUM MANIFEST
# =============================================================================

sealed_files = [
    p
    for root in [
        DEST_6C,
        DEST_6D,
    ]
    for p in root.rglob("*")
    if p.is_file()
]


sealed_files.append(
    SEAL_RECEIPT
)


sealed_files = sorted(
    sealed_files,
    key=lambda p:
        str(
            p.relative_to(
                REPO
            )
        )
)


REPO_CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(path)}  "
            f"{path.relative_to(REPO)}"
        )
        for path in sealed_files
    ) + "\n",
    encoding="utf-8",
)


repo_manifest_sha = (
    sha256_file(
        REPO_CHECKSUMS
    )
)


print()
print(
    "[OK] repository checksum manifest SHA256:",
    repo_manifest_sha,
)


# =============================================================================
# 15. GIT IDENTITY
# =============================================================================

_, user_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

_, user_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not user_name:

    _, prior_name = git(
        "log",
        "-1",
        "--format=%an",
    )

    git(
        "config",
        "user.name",
        prior_name,
    )


if not user_email:

    _, prior_email = git(
        "log",
        "-1",
        "--format=%ae",
    )

    git(
        "config",
        "user.email",
        prior_email,
    )


# =============================================================================
# 16. STAGE ONLY STAGE23-6
# =============================================================================

git(
    "add",
    "--",
    str(
        DEST_ROOT.relative_to(
            REPO
        )
    ),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    line
    for line in staged.splitlines()
    if line.strip()
]


if not staged_files:

    raise RuntimeError(
        "No Stage23-6 files staged."
    )


allowed_prefix = (
    str(
        DEST_ROOT.relative_to(
            REPO
        )
    )
    + "/"
)


for path in staged_files:

    if not path.startswith(
        allowed_prefix
    ):

        raise RuntimeError(
            "Unexpected staged path:\n"
            f"{path}"
        )


print()
print(
    "[OK] staged Stage23-6 files:",
    len(
        staged_files
    ),
)


# =============================================================================
# 17. COMMIT
# =============================================================================

print()
print("=" * 116)
print("COMMIT")
print("=" * 116)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, sealed_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] Stage23-6 commit:",
    sealed_commit,
)


# =============================================================================
# 18. ANNOTATED TAG
# =============================================================================

git(
    "tag",
    "-a",
    SEAL_TAG,
    sealed_commit,
    "-m",
    (
        "Stage23-6 frozen attack-family analysis complete: "
        "13 families, exact development label alignment, "
        "182 preregistered primary family metric rows, "
        "zero model fits and zero model inference"
    ),
)


# =============================================================================
# 19. ATOMIC PUSH MAIN + TAG
# =============================================================================

print()
print("=" * 116)
print("ATOMIC PUSH MAIN + TAG")
print("=" * 116)
print()


git(
    "push",
    "--atomic",
    "origin",
    "main",
    f"refs/tags/{SEAL_TAG}",
    env=auth_env,
    show=True,
)


# =============================================================================
# 20. REMOVE AUTH HELPER IMMEDIATELY
# =============================================================================

github_token = None


try:

    ASKPASS.unlink()

except Exception:

    pass


# =============================================================================
# 21. REMOTE VERIFICATION
# =============================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEAL_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Remote main could not be resolved."
    )


if not remote_tag_line:

    raise RuntimeError(
        "Remote annotated Stage23-6 tag could not be peeled."
    )


remote_main = (
    remote_main_line.split()[0]
)

remote_tag_commit = (
    remote_tag_line.split()[0]
)


if remote_main != sealed_commit:

    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_main}"
    )


if remote_tag_commit != sealed_commit:

    raise RuntimeError(
        "Remote Stage23-6 tag mismatch.\n"
        f"expected={sealed_commit}\n"
        f"actual={remote_tag_commit}"
    )


# =============================================================================
# 22. FINAL CLEAN STATE
# =============================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage23-6 seal:\n"
        + final_status
    )


# =============================================================================
# 23. FINAL
# =============================================================================

print()
print("=" * 116)
print("STAGE23-6 ATTACK-FAMILY BLOCK — SEALED AND PUSHED")
print("=" * 116)
print()

print("Commit:")
print(
    " ",
    sealed_commit,
)

print()
print("Tag:")
print(
    " ",
    SEAL_TAG,
)

print()
print("Remote main:")
print(
    " ",
    remote_main,
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    remote_tag_commit,
)

print()
print("Repository manifest SHA256:")
print(
    " ",
    repo_manifest_sha,
)

print()
print("STAGE23-6C")
print("  development families       : 13")
print("  retained attacks           : 1,972,299")
print("  RANDOM rows verified       : 2,882,481")
print("  CHRONO rows verified       : 593,780")
print("  label/binary mismatches    : 0 / 0")

print()
print("STAGE23-6D")
print("  primary subsets            : 7 / 7")
print("  splits                     : 2 / 2")
print("  probability artifacts      : 14 / 14")
print("  family metric rows         : 182")
print("  RANDOM interpretable       : 11")
print("  CHRONO interpretable       : 1")

print()
print("GOVERNANCE")
print("  Stage23 fit budget         : 50 / 50 SEALED")
print("  additional fits authorized : 0")
print("  new model fits             : 0")
print("  model inference            : 0")
print("  LightGBM execution         : 0")
print("  XGBoost execution          : 0")
print("  Raw Mar1 accessed          : NO")
print("  Raw Mar2 accessed          : NO")

print()
print("NEXT:")
print("  Stage23 consolidated scientific audit.")
print("  Then publication figures + manuscript integration.")
print("  NO ADDITIONAL MODEL FITS.")

print("=" * 116)

STAGE23-6 — ZERO-FIT ATTACK-FAMILY BLOCK SEAL + PUSH

[EXACT] HEAD      : b927499efdf7d4dd5013054cc796f1115f879f8b
[EXACT] parent tag: stage23-5-shap-proxy-absorption-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] Stage23 fit state: 50 / 50 SEALED

VERIFY STAGE23-6C

retained_development_attack_family_labels.parquet
  expected: 3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  actual:   3872d6000cbc78d8f97e47463115599c59d1d19dc2ffc8a8c72dbec8aeacbf26
  [EXACT]
attack_family_support_by_split.csv
  expected: d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  actual:   d8bffc4a0ba61b70033364db379e261b9530fde43b19b534f860785315af6377
  [EXACT]
stage23_6c_alignment_manifest.json
  expected: bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  actual:   bec2b909ac4249e351b39e466290be11f0f843e1bda8fcb247cc7122fa6272b8
  [EXACT]
checksums.sha256
  expected: a54e01b733e093602a03b9b8bff5430f7cdfa4e987e0a36577ee686e736113b4
  actual:   a54e01b733e0936

In [11]:
# ==================================================================================================
# STAGE23-7A — FINAL SCIENTIFIC SYNTHESIS + PUBLICATION TABLES + FIGURES + MANUSCRIPT INTEGRATION
# ZERO FIT / ZERO INFERENCE / ZERO NEW SHAP / ZERO BOOTSTRAP / NO PARQUET / NO MAR1-MAR2
# ==================================================================================================

from __future__ import annotations

import json
import hashlib
import math
import os
import shutil
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
ROOT = REPO / "results" / "stage23_shortcut_feature_audit"

P0 = ROOT / "stage23_0_protocol_lock"
P1 = ROOT / "stage23_1_primary_ablation"
P2 = ROOT / "stage23_2_placebo_ablation"
P3 = ROOT / "stage23_3_stump_controls"
P4 = ROOT / "stage23_4_uncertainty_analysis"
P5 = ROOT / "stage23_5b_treeshap_proxy_absorption"
P6 = ROOT / "stage23_6_attack_family_analysis" / "stage23_6d_attack_family_metrics"

OUT = Path("/kaggle/working/stage23_7_final_synthesis")
TMP = Path("/kaggle/working/stage23_7_final_synthesis_tmp")

EXPECTED_PARENT = "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
EXPECTED_PARENT_TAG = "stage23-6-attack-family-analysis-complete-v1"

SPLITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

PRIMARY_ORDER = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]

PRIMARY_ABLATIONS = PRIMARY_ORDER[1:]

PLACEBO_ORDER = [
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]

STUMP_ORDER = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]

DISPLAY = {
    "FULL": "FULL",
    "NO_DST_PORT": "No Dst Port",
    "NO_PORTS": "No Ports",
    "NO_INIT_FWD_WIN_BYTS": "No Init Fwd Win",
    "NO_FWD_SEG_SIZE_MIN": "No Fwd Seg Size Min",
    "NO_SUSPICIOUS_GROUP": "No Suspicious Group",
    "BEHAVIOR_ONLY": "Behavior Only",
    "PLACEBO_COUNTS": "Placebo Counts",
    "PLACEBO_VOLUME_DIRECTION": "Placebo Volume/Direction",
    "PLACEBO_IAT": "Placebo IAT",
    "PLACEBO_PACKET_SIZE": "Placebo Packet Size",
    "PLACEBO_ACTIVITY": "Placebo Activity",
    "RANDOM_NATURAL": "Random-natural",
    "CHRONOLOGICAL_NATURAL": "Chronological-natural",
}

SEP = "=" * 112


# ==================================================================================================
# HELPERS
# ==================================================================================================

def git(*args: str) -> str:
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    return p.stdout.strip()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.rstrip() + "\n", encoding="utf-8")


SOURCE_FILES: dict[str, str] = {}


def register_source(path: Path) -> None:
    assert path.is_file(), f"Missing source: {path}"
    rel = str(path.relative_to(REPO))
    SOURCE_FILES[rel] = sha256_file(path)


def read_json(path: Path):
    register_source(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_csv(path: Path) -> pd.DataFrame:
    register_source(path)
    return pd.read_csv(path)


def f6(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return f"{float(x):.6f}"


def f4(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return f"{float(x):.4f}"


def signed6(x) -> str:
    return f"{float(x):+.6f}"


def ci_excludes_zero(low: float, high: float) -> bool:
    return bool(low > 0.0 or high < 0.0)


def latex_escape(s: str) -> str:
    s = str(s)
    replacements = [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
        ("~", r"\textasciitilde{}"),
        ("^", r"\textasciicircum{}"),
    ]
    for a, b in replacements:
        s = s.replace(a, b)
    return s


def save_figure(fig, stem: Path) -> None:
    stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def row_for(df: pd.DataFrame, split: str, subset: str) -> pd.Series:
    x = df[(df["split"] == split) & (df["subset"] == subset)]
    assert len(x) == 1, (split, subset, len(x))
    return x.iloc[0]


# ==================================================================================================
# PREFLIGHT — REPOSITORY MUST REMAIN EXACTLY AT SEALED STAGE23-6
# ==================================================================================================

print(SEP)
print("STAGE23-7A — FINAL SYNTHESIS / TABLES / FIGURES / MANUSCRIPT INTEGRATION")
print(SEP)
print()

assert REPO.is_dir(), f"Repository missing: {REPO}"
assert ROOT.is_dir(), f"Stage23 root missing: {ROOT}"

head = git("rev-parse", "HEAD")
branch = git("rev-parse", "--abbrev-ref", "HEAD")
status = git("status", "--porcelain")
tag_commit = git("rev-list", "-n", "1", EXPECTED_PARENT_TAG)

print("[EXACT] branch    :", branch)
print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] tag commit:", tag_commit)
print("[EXACT] worktree  :", "CLEAN" if not status else "DIRTY")
print()

assert branch == "main", branch
assert head == EXPECTED_PARENT, (head, EXPECTED_PARENT)
assert tag_commit == EXPECTED_PARENT, (tag_commit, EXPECTED_PARENT)
assert status == "", f"Repository dirty before Stage23-7A:\n{status}"

for required in [P0, P1, P2, P3, P4, P5, P6]:
    assert required.exists(), required

if OUT.exists():
    raise RuntimeError(
        f"{OUT} already exists. Refusing to overwrite a completed Stage23-7A package."
    )

if TMP.exists():
    shutil.rmtree(TMP)

TMP.mkdir(parents=True)
TABLES = TMP / "tables"
FIGURES = TMP / "figures"
MANUSCRIPT = TMP / "manuscript"
AUDIT = TMP / "audit"

for p in [TABLES, FIGURES, MANUSCRIPT, AUDIT]:
    p.mkdir(parents=True, exist_ok=True)


# ==================================================================================================
# FROZEN PROTOCOL
# ==================================================================================================

print(SEP)
print("VERIFY FROZEN STAGE23 PROTOCOL")
print(SEP)
print()

stopping_rule = read_json(P0 / "stopping_rule.json")
figure_plan = read_json(P0 / "figure_plan.json")
interpretation_matrix = read_json(P0 / "interpretation_matrix.json")
metric_spec = read_json(P0 / "metric_spec.json")

assert len(stopping_rule["complete_when"]) == 7
assert stopping_rule["final_holdout"] == "Raw Mar1 and Mar2 permanently forbidden."
assert stopping_rule["rolling_forward_analysis"] == "NOT_INCLUDED_IN_FINAL_STAGE23_PROTOCOL"

assert set(figure_plan.keys()) == {
    "Figure_23_A",
    "Figure_23_B",
    "Figure_23_C",
    "Figure_23_D",
    "supplementary",
}

assert metric_spec["primary_ranking_metric"] == "PR_AUC"
assert metric_spec["headline_interaction_metrics"] == ["PR_AUC", "ROC_AUC"]
assert float(metric_spec["secondary_fixed_operating_point"]["threshold"]) == 0.5

print("[EXACT] stopping conditions :", len(stopping_rule["complete_when"]))
print("[EXACT] frozen main figures : 4")
print("[EXACT] primary metric       : PR_AUC")
print("[EXACT] fixed threshold      : 0.50")
print("[EXACT] Mar1/Mar2            : PERMANENTLY FORBIDDEN")
print()


# ==================================================================================================
# PRIMARY ABLATION RESULTS — SEALED JSON ONLY
# ==================================================================================================

print(SEP)
print("LOAD SEALED PRIMARY ABLATION RESULTS")
print(SEP)
print()

primary_result_paths = sorted(P1.glob("*/*/stage23_*_result.json"))
assert len(primary_result_paths) == 12, len(primary_result_paths)

primary_rows = []
full_refs: dict[str, list[dict]] = {s: [] for s in SPLITS}
validation_meta: dict[str, dict] = {}

for path in primary_result_paths:
    d = read_json(path)

    split = d["cell"]["split"]
    subset = d["cell"]["subset"]

    assert split in SPLITS
    assert subset in PRIMARY_ABLATIONS

    ranking = d["ranking_metrics"]
    fixed = d["fixed_threshold_0_50"]
    val = d["data"]["validation"]

    meta = {
        "rows": int(val["rows"]),
        "attack": int(val["attack"]),
        "benign": int(val["benign"]),
        "attack_prevalence": float(val["attack_prevalence"]),
    }

    if split in validation_meta:
        assert validation_meta[split] == meta
    else:
        validation_meta[split] = meta

    primary_rows.append({
        "split": split,
        "subset": subset,
        "family": "PRIMARY",
        "feature_count": int(d["cell"]["feature_count"]),
        "validation_rows": int(val["rows"]),
        "support_attack": int(val["attack"]),
        "support_benign": int(val["benign"]),
        "attack_prevalence": float(val["attack_prevalence"]),
        "pr_auc": float(ranking["pr_auc"]),
        "pr_auc_minus_attack_prevalence":
            float(ranking["pr_auc_minus_attack_prevalence"]),
        "roc_auc": float(ranking["roc_auc"]),
        "f1_at_0_50": float(fixed["f1"]),
        "recall_at_0_50": float(fixed["recall"]),
        "fpr_at_0_50": float(fixed["fpr"]),
        "fnr_at_0_50": float(fixed["fnr"]),
    })

    full_refs[split].append(d["full_reference"])

# FULL references must be identical across all six ablation results within split.
for split in SPLITS:
    assert len(full_refs[split]) == 6

    canonical = full_refs[split][0]
    for other in full_refs[split][1:]:
        for key in [
            "pr_auc",
            "roc_auc",
            "f1_at_0_50",
            "recall_at_0_50",
            "fpr_at_0_50",
        ]:
            assert float(other[key]) == float(canonical[key]), (split, key)

    prev = validation_meta[split]["attack_prevalence"]

    primary_rows.append({
        "split": split,
        "subset": "FULL",
        "family": "FULL",
        "feature_count": 70,
        "validation_rows": validation_meta[split]["rows"],
        "support_attack": validation_meta[split]["attack"],
        "support_benign": validation_meta[split]["benign"],
        "attack_prevalence": prev,
        "pr_auc": float(canonical["pr_auc"]),
        "pr_auc_minus_attack_prevalence":
            float(canonical["pr_auc"]) - prev,
        "roc_auc": float(canonical["roc_auc"]),
        "f1_at_0_50": float(canonical["f1_at_0_50"]),
        "recall_at_0_50": float(canonical["recall_at_0_50"]),
        "fpr_at_0_50": float(canonical["fpr_at_0_50"]),
        "fnr_at_0_50": 1.0 - float(canonical["recall_at_0_50"]),
    })

primary_df = pd.DataFrame(primary_rows)

assert len(primary_df) == 14
assert set(primary_df["subset"]) == set(PRIMARY_ORDER)

for split in SPLITS:
    assert set(primary_df[primary_df["split"] == split]["subset"]) == set(PRIMARY_ORDER)

order_map = {v: i for i, v in enumerate(PRIMARY_ORDER)}
split_map = {v: i for i, v in enumerate(SPLITS)}

primary_df["_split_order"] = primary_df["split"].map(split_map)
primary_df["_subset_order"] = primary_df["subset"].map(order_map)
primary_df = (
    primary_df.sort_values(["_split_order", "_subset_order"])
    .drop(columns=["_split_order", "_subset_order"])
    .reset_index(drop=True)
)

print("[EXACT] primary result JSONs :", len(primary_result_paths))
print("[EXACT] primary table rows   :", len(primary_df))
print("[EXACT] primary subsets      : 7 / 7")
print("[EXACT] splits               : 2 / 2")
print()


# ==================================================================================================
# PLACEBO RESULTS — SEALED JSON ONLY
# ==================================================================================================

print(SEP)
print("LOAD SEALED MATCHED-SIZE PLACEBO RESULTS")
print(SEP)
print()

placebo_result_paths = sorted(P2.glob("*/*/stage23_*_result.json"))
assert len(placebo_result_paths) == 10, len(placebo_result_paths)

placebo_rows = []

for path in placebo_result_paths:
    d = read_json(path)

    split = d["cell"]["split"]
    placebo = d["cell"]["placebo"]

    assert split in SPLITS
    assert placebo in PLACEBO_ORDER

    ranking = d["ranking_metrics"]
    fixed = d["fixed_threshold_0_50"]
    val = d["data"]["validation"]

    assert int(val["rows"]) == validation_meta[split]["rows"]
    assert int(val["attack"]) == validation_meta[split]["attack"]
    assert int(val["benign"]) == validation_meta[split]["benign"]

    placebo_rows.append({
        "split": split,
        "subset": placebo,
        "family": "PLACEBO",
        "feature_count": int(d["cell"]["feature_count"]),
        "validation_rows": int(val["rows"]),
        "support_attack": int(val["attack"]),
        "support_benign": int(val["benign"]),
        "attack_prevalence": float(val["attack_prevalence"]),
        "pr_auc": float(ranking["pr_auc"]),
        "pr_auc_minus_attack_prevalence":
            float(ranking["pr_auc_minus_attack_prevalence"]),
        "roc_auc": float(ranking["roc_auc"]),
        "f1_at_0_50": float(fixed["f1"]),
        "recall_at_0_50": float(fixed["recall"]),
        "fpr_at_0_50": float(fixed["fpr"]),
        "fnr_at_0_50": float(fixed["fnr"]),
    })

placebo_df = pd.DataFrame(placebo_rows)

assert len(placebo_df) == 10

for split in SPLITS:
    assert set(
        placebo_df[placebo_df["split"] == split]["subset"]
    ) == set(PLACEBO_ORDER)

placebo_order_map = {v: i for i, v in enumerate(PLACEBO_ORDER)}
placebo_df["_split_order"] = placebo_df["split"].map(split_map)
placebo_df["_subset_order"] = placebo_df["subset"].map(placebo_order_map)
placebo_df = (
    placebo_df.sort_values(["_split_order", "_subset_order"])
    .drop(columns=["_split_order", "_subset_order"])
    .reset_index(drop=True)
)

print("[EXACT] placebo result JSONs :", len(placebo_result_paths))
print("[EXACT] placebo table rows   :", len(placebo_df))
print("[EXACT] placebo subsets      : 5 / 5")
print()


# ==================================================================================================
# UNCERTAINTY — USE FROZEN SUMMARY ONLY; NO BOOTSTRAP RERUN
# ==================================================================================================

print(SEP)
print("LOAD FROZEN UNCERTAINTY SUMMARY")
print(SEP)
print()

uncertainty_json = read_json(P4 / "stage23_4_uncertainty_summary.json")

expected_comparisons = set(PRIMARY_ABLATIONS + PLACEBO_ORDER)
assert set(uncertainty_json["comparisons"].keys()) == expected_comparisons

unc_rows = []

for comparison in PRIMARY_ABLATIONS + PLACEBO_ORDER:
    c = uncertainty_json["comparisons"][comparison]
    pts = c["complete_validation_point_estimates"]
    ci = c["bootstrap_95_percentile_ci"]

    row = {
        "comparison": comparison,
        "family": c["family"],

        "random_pr_auc_removal_penalty":
            float(pts["random"]["pr_auc_removal_penalty"]),
        "random_pr_auc_ci_low":
            float(ci["random_pr_auc_removal_penalty"]["lower_2_5"]),
        "random_pr_auc_ci_high":
            float(ci["random_pr_auc_removal_penalty"]["upper_97_5"]),

        "chronological_pr_auc_removal_penalty":
            float(pts["chronological"]["pr_auc_removal_penalty"]),
        "chronological_pr_auc_ci_low":
            float(ci["chronological_pr_auc_removal_penalty"]["lower_2_5"]),
        "chronological_pr_auc_ci_high":
            float(ci["chronological_pr_auc_removal_penalty"]["upper_97_5"]),

        "pr_auc_interaction":
            float(pts["interaction"]["pr_auc"]),
        "pr_auc_interaction_ci_low":
            float(ci["pr_auc_shortcut_interaction"]["lower_2_5"]),
        "pr_auc_interaction_ci_high":
            float(ci["pr_auc_shortcut_interaction"]["upper_97_5"]),

        "random_roc_auc_removal_penalty":
            float(pts["random"]["roc_auc_removal_penalty"]),
        "random_roc_auc_ci_low":
            float(ci["random_roc_auc_removal_penalty"]["lower_2_5"]),
        "random_roc_auc_ci_high":
            float(ci["random_roc_auc_removal_penalty"]["upper_97_5"]),

        "chronological_roc_auc_removal_penalty":
            float(pts["chronological"]["roc_auc_removal_penalty"]),
        "chronological_roc_auc_ci_low":
            float(ci["chronological_roc_auc_removal_penalty"]["lower_2_5"]),
        "chronological_roc_auc_ci_high":
            float(ci["chronological_roc_auc_removal_penalty"]["upper_97_5"]),

        "roc_auc_interaction":
            float(pts["interaction"]["roc_auc"]),
        "roc_auc_interaction_ci_low":
            float(ci["roc_auc_shortcut_interaction"]["lower_2_5"]),
        "roc_auc_interaction_ci_high":
            float(ci["roc_auc_shortcut_interaction"]["upper_97_5"]),
    }

    row["pr_auc_interaction_ci_excludes_zero"] = ci_excludes_zero(
        row["pr_auc_interaction_ci_low"],
        row["pr_auc_interaction_ci_high"],
    )
    row["roc_auc_interaction_ci_excludes_zero"] = ci_excludes_zero(
        row["roc_auc_interaction_ci_low"],
        row["roc_auc_interaction_ci_high"],
    )

    unc_rows.append(row)

unc_df = pd.DataFrame(unc_rows)

assert len(unc_df) == 11
assert len(unc_df[unc_df["family"] == "PRIMARY"]) == 6
assert len(unc_df[unc_df["family"] == "PLACEBO"]) == 5

print("[EXACT] uncertainty comparisons : 11")
print("[EXACT] primary comparisons     : 6")
print("[EXACT] placebo comparisons     : 5")
print("[EXACT] new bootstrap draws     : 0")
print()


# ==================================================================================================
# STUMP CONTROLS
# ==================================================================================================

print(SEP)
print("LOAD SEALED STUMP-CONTROL SUMMARY")
print(SEP)
print()

stump_summary = read_json(P3 / "stage23_3_stump_controls_summary.json")

assert int(stump_summary["fit_accounting"]["stage23_after"]) == 50
assert int(stump_summary["fit_accounting"]["stump_fits_completed"]) == 6
assert int(stump_summary["fit_accounting"]["stump_fits_expected"]) == 6
assert set(stump_summary["paired_random_to_chronological"].keys()) == set(STUMP_ORDER)

stump_rows = []

for feature in STUMP_ORDER:
    x = stump_summary["paired_random_to_chronological"][feature]

    rm = x["random_metrics"]
    cm = x["chronological_metrics"]
    dg = x["random_to_chronological_degradation"]
    rt = x["random_tree_structure"]
    ct = x["chronological_tree_structure"]

    stump_rows.append({
        "feature": feature,

        "random_pr_auc": float(rm["PR_AUC"]),
        "chronological_pr_auc": float(cm["PR_AUC"]),
        "random_minus_chronological_pr_auc": float(dg["PR_AUC"]),

        "random_roc_auc": float(rm["ROC_AUC"]),
        "chronological_roc_auc": float(cm["ROC_AUC"]),
        "random_minus_chronological_roc_auc": float(dg["ROC_AUC"]),

        "random_f1_at_0_50": float(rm["f1"]),
        "chronological_f1_at_0_50": float(cm["f1"]),
        "random_minus_chronological_f1": float(dg["f1"]),

        "random_recall_at_0_50": float(rm["recall"]),
        "chronological_recall_at_0_50": float(cm["recall"]),

        "random_fpr_at_0_50": float(rm["fpr"]),
        "chronological_fpr_at_0_50": float(cm["fpr"]),

        "random_training_median": float(rt["missing_imputation_median"]),
        "chronological_training_median": float(ct["missing_imputation_median"]),

        "random_split_threshold": float(rt["split_threshold"]),
        "chronological_split_threshold": float(ct["split_threshold"]),

        "random_attack_branch": rt["branch_predicting_attack"],
        "chronological_attack_branch": ct["branch_predicting_attack"],
    })

stump_df = pd.DataFrame(stump_rows)

assert len(stump_df) == 3

print("[EXACT] stump controls        : 6 / 6")
print("[EXACT] stump feature pairs   : 3 / 3")
print("[EXACT] Stage23 fit budget    : 50 / 50 SEALED")
print()


# ==================================================================================================
# SHAP PROXY-ABSORPTION — SEALED DERIVED OUTPUTS ONLY
# ==================================================================================================

print(SEP)
print("LOAD SEALED SHAP PROXY-ABSORPTION RESULTS")
print(SEP)
print()

shap_summary = read_csv(P5 / "stage23_5b_proxy_absorption_summary.csv")
consensus_importance = read_csv(P5 / "stage23_5b_descriptive_consensus_importance.csv")
component_importance = read_csv(P5 / "stage23_5b_component_importance.csv")

assert len(shap_summary) == 66
assert shap_summary[["split", "subset"]].drop_duplicates().shape[0] == 22

reportings = set(shap_summary["reporting"].unique())
assert reportings == {
    "LightGBM",
    "XGBoost",
    "DESCRIPTIVE_CONSENSUS_0.5_0.5",
}

assert component_importance[
    ["split", "subset", "component"]
].drop_duplicates().shape[0] == 48

assert consensus_importance[
    ["split", "subset"]
].drop_duplicates().shape[0] == 24

shap_primary = shap_summary[
    (shap_summary["family"] == "PRIMARY")
    & (shap_summary["reporting"] == "DESCRIPTIVE_CONSENSUS_0.5_0.5")
].copy()

assert len(shap_primary) == 12
assert set(shap_primary["subset"]) == set(PRIMARY_ABLATIONS)

# Compute the largest positive normalized consensus-share change among RETAINED features.
# This is a presentation-only derivation from sealed descriptive consensus values.
gainer_records = []

for split in SPLITS:
    full_share = consensus_importance[
        (consensus_importance["split"] == split)
        & (consensus_importance["subset"] == "FULL")
    ][["feature", "consensus_normalized_importance_share"]].copy()

    full_share = full_share.rename(
        columns={
            "consensus_normalized_importance_share": "full_consensus_share"
        }
    )

    for subset in PRIMARY_ABLATIONS:
        sub_share = consensus_importance[
            (consensus_importance["split"] == split)
            & (consensus_importance["subset"] == subset)
        ][["feature", "consensus_normalized_importance_share"]].copy()

        sub_share = sub_share.rename(
            columns={
                "consensus_normalized_importance_share": "subset_consensus_share"
            }
        )

        merged = sub_share.merge(full_share, on="feature", how="inner", validate="one_to_one")
        merged["consensus_share_change"] = (
            merged["subset_consensus_share"] - merged["full_consensus_share"]
        )

        merged = merged.sort_values(
            ["consensus_share_change", "feature"],
            ascending=[False, True],
        )

        top = merged.iloc[0]

        assert float(top["consensus_share_change"]) > 0.0

        gainer_records.append({
            "split": split,
            "subset": subset,
            "top_retained_consensus_share_gainer": str(top["feature"]),
            "top_retained_consensus_share_gain":
                float(top["consensus_share_change"]),
            "top_retained_subset_consensus_share":
                float(top["subset_consensus_share"]),
            "top_retained_full_consensus_share":
                float(top["full_consensus_share"]),
        })

gainers_df = pd.DataFrame(gainer_records)

shap_primary = shap_primary.merge(
    gainers_df,
    on=["split", "subset"],
    how="left",
    validate="one_to_one",
)

assert shap_primary["top_retained_consensus_share_gainer"].notna().all()

component_top20 = component_importance[
    component_importance["rank"] <= 20
].copy()

assert component_top20[
    ["split", "subset", "component"]
].drop_duplicates().shape[0] == 48

# All Stage23 SHAP models retain at least 20 features.
assert len(component_top20) == 48 * 20

print("[EXACT] proxy split comparisons     : 22")
print("[EXACT] proxy reporting rows        : 66")
print("[EXACT] component TreeSHAP models   : 48 / 48")
print("[EXACT] descriptive consensus sets  : 24")
print("[EXACT] component top-20 rows       :", len(component_top20))
print("[EXACT] new SHAP values computed    : 0")
print()


# ==================================================================================================
# ATTACK-FAMILY RESULTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED ATTACK-FAMILY RESULTS")
print(SEP)
print()

attack_df = read_csv(P6 / "stage23_6d_attack_family_metrics.csv")

assert len(attack_df) == 182
assert attack_df["attack_family"].nunique() == 13
assert attack_df["subset"].nunique() == 7
assert set(attack_df["subset"]) == set(PRIMARY_ORDER)
assert set(attack_df["split"]) == set(SPLITS)

full_attack = attack_df[attack_df["subset"] == "FULL"].copy()
assert len(full_attack) == 26

random_full_attack = full_attack[
    full_attack["split"] == "RANDOM_NATURAL"
].copy()

chrono_full_attack = full_attack[
    full_attack["split"] == "CHRONOLOGICAL_NATURAL"
].copy()

assert len(random_full_attack) == 13
assert len(chrono_full_attack) == 13

random_interp = random_full_attack[
    random_full_attack["interpretation_status"] == "INTERPRETABLE"
]
chrono_interp = chrono_full_attack[
    chrono_full_attack["interpretation_status"] == "INTERPRETABLE"
]

assert len(random_interp) == 11
assert len(chrono_interp) == 1

# Attack-family support must exactly account for the validation attacks.
assert int(random_full_attack["support_attack"].sum()) == validation_meta["RANDOM_NATURAL"]["attack"]
assert int(chrono_full_attack["support_attack"].sum()) == validation_meta["CHRONOLOGICAL_NATURAL"]["attack"]

# Chronological attack support is entirely Infilteration under the frozen family labels.
chrono_positive = chrono_full_attack[
    chrono_full_attack["support_attack"] > 0
]

assert len(chrono_positive) == 1
assert chrono_positive.iloc[0]["attack_family"] == "Infilteration"
assert int(chrono_positive.iloc[0]["support_attack"]) == validation_meta["CHRONOLOGICAL_NATURAL"]["attack"]

attack_support_table = full_attack[
    [
        "split",
        "attack_family",
        "interpretation_status",
        "support_attack",
        "support_benign",
        "PR_AUC",
        "ROC_AUC",
        "recall_at_0_50",
        "f1_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()

print("[EXACT] family metric rows       : 182")
print("[EXACT] frozen attack families   : 13")
print("[EXACT] RANDOM interpretable     : 11")
print("[EXACT] CHRONO interpretable     : 1")
print("[EXACT] CHRONO positive families : Infilteration only")
print("[EXACT] CHRONO attack support    :", validation_meta["CHRONOLOGICAL_NATURAL"]["attack"])
print()


# ==================================================================================================
# PUBLICATION TABLES
# ==================================================================================================

print(SEP)
print("GENERATE PUBLICATION TABLES")
print(SEP)
print()

# --------------------------------------------------------------------------------------------------
# TABLE 23-1: Primary subset performance
# --------------------------------------------------------------------------------------------------

primary_wide_rows = []

for subset in PRIMARY_ORDER:
    rr = row_for(primary_df, "RANDOM_NATURAL", subset)
    cc = row_for(primary_df, "CHRONOLOGICAL_NATURAL", subset)

    primary_wide_rows.append({
        "subset": subset,
        "feature_count": int(rr["feature_count"]),

        "random_pr_auc": float(rr["pr_auc"]),
        "chronological_pr_auc": float(cc["pr_auc"]),

        "random_pr_auc_minus_prevalence":
            float(rr["pr_auc_minus_attack_prevalence"]),
        "chronological_pr_auc_minus_prevalence":
            float(cc["pr_auc_minus_attack_prevalence"]),

        "random_roc_auc": float(rr["roc_auc"]),
        "chronological_roc_auc": float(cc["roc_auc"]),

        "random_f1_at_0_50": float(rr["f1_at_0_50"]),
        "chronological_f1_at_0_50": float(cc["f1_at_0_50"]),

        "random_recall_at_0_50": float(rr["recall_at_0_50"]),
        "chronological_recall_at_0_50": float(cc["recall_at_0_50"]),

        "random_fpr_at_0_50": float(rr["fpr_at_0_50"]),
        "chronological_fpr_at_0_50": float(cc["fpr_at_0_50"]),
    })

primary_wide = pd.DataFrame(primary_wide_rows)

primary_wide.to_csv(
    TABLES / "table_23_1_primary_subset_performance.csv",
    index=False,
)

primary_df.to_csv(
    TABLES / "table_23_s0_primary_subset_performance_long.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-2: Primary removal penalties and interaction CIs
# --------------------------------------------------------------------------------------------------

unc_primary = unc_df[unc_df["family"] == "PRIMARY"].copy()
unc_primary["_order"] = unc_primary["comparison"].map(
    {x: i for i, x in enumerate(PRIMARY_ABLATIONS)}
)
unc_primary = (
    unc_primary.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

unc_primary.to_csv(
    TABLES / "table_23_2_primary_removal_penalties_and_interactions.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-3: Stump controls
# --------------------------------------------------------------------------------------------------

stump_df.to_csv(
    TABLES / "table_23_3_single_feature_stump_controls.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-4: SHAP proxy absorption
# --------------------------------------------------------------------------------------------------

shap_table_cols = [
    "split",
    "subset",
    "jaccard_at_10_vs_full",
    "jaccard_at_20_vs_full",
    "spearman_common_retained",
    "new_top10_entrants",
    "features_leaving_top10",
    "number_positive_share_gainers",
    "top_retained_consensus_share_gainer",
    "top_retained_consensus_share_gain",
    "top_retained_subset_consensus_share",
    "top_retained_full_consensus_share",
]

shap_table = shap_primary[shap_table_cols].copy()
shap_table["_split_order"] = shap_table["split"].map(split_map)
shap_table["_subset_order"] = shap_table["subset"].map(
    {x: i for i, x in enumerate(PRIMARY_ABLATIONS)}
)
shap_table = (
    shap_table.sort_values(["_split_order", "_subset_order"])
    .drop(columns=["_split_order", "_subset_order"])
    .reset_index(drop=True)
)

shap_table.to_csv(
    TABLES / "table_23_4_shap_proxy_absorption_consensus.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-5: Attack-family support/context
# --------------------------------------------------------------------------------------------------

attack_support_table.to_csv(
    TABLES / "table_23_5_attack_family_support_and_full_metrics.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# SUPPLEMENTARY TABLES
# --------------------------------------------------------------------------------------------------

unc_placebo = unc_df[unc_df["family"] == "PLACEBO"].copy()
unc_placebo["_order"] = unc_placebo["comparison"].map(placebo_order_map)
unc_placebo = (
    unc_placebo.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

unc_placebo.to_csv(
    TABLES / "table_23_s1_placebo_removal_penalties_and_interactions.csv",
    index=False,
)

placebo_df.to_csv(
    TABLES / "table_23_s2_placebo_subset_performance.csv",
    index=False,
)

attack_df.to_csv(
    TABLES / "table_23_s3_attack_family_all_frozen_metrics.csv",
    index=False,
)

component_top20.sort_values(
    ["split", "subset", "component", "rank"]
).to_csv(
    TABLES / "table_23_s4_component_specific_shap_top20.csv",
    index=False,
)

behavior_operating = primary_df[
    primary_df["subset"].isin(["FULL", "BEHAVIOR_ONLY"])
][
    [
        "split",
        "subset",
        "attack_prevalence",
        "pr_auc",
        "pr_auc_minus_attack_prevalence",
        "roc_auc",
        "f1_at_0_50",
        "recall_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()

behavior_operating.to_csv(
    TABLES / "table_23_s5_behavior_restricted_operating_metrics.csv",
    index=False,
)

print("[OK] Table 23-1  primary subset performance")
print("[OK] Table 23-2  removal penalties + interactions")
print("[OK] Table 23-3  stump controls")
print("[OK] Table 23-4  SHAP proxy absorption")
print("[OK] Table 23-5  attack-family support/context")
print("[OK] Table 23-S1 placebo interactions")
print("[OK] Table 23-S2 placebo performance")
print("[OK] Table 23-S3 all attack-family frozen metrics")
print("[OK] Table 23-S4 complete component-specific SHAP top-20")
print("[OK] Table 23-S5 behavior-restricted operating metrics")
print()


# ==================================================================================================
# PUBLICATION FIGURE STYLE
# ==================================================================================================

plt.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 9.5,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "figure.titlesize": 10,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.4,
    "lines.markersize": 4.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

cycle_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


# ==================================================================================================
# FIGURE 23-A — SUBSET × SPLIT INTERACTION
# ==================================================================================================

print(SEP)
print("GENERATE FIGURE 23-A")
print(SEP)

fig, axes = plt.subplots(1, 2, figsize=(7.25, 3.15))

x = np.arange(len(PRIMARY_ORDER))
labels = [DISPLAY[s] for s in PRIMARY_ORDER]

for idx, split in enumerate(SPLITS):
    vals = [
        float(row_for(primary_df, split, subset)["pr_auc"])
        for subset in PRIMARY_ORDER
    ]
    line = axes[0].plot(
        x,
        vals,
        marker="o",
        label=DISPLAY[split],
    )[0]

    prevalence = validation_meta[split]["attack_prevalence"]
    axes[0].axhline(
        prevalence,
        linestyle=":",
        linewidth=0.9,
        color=line.get_color(),
        alpha=0.75,
    )

axes[0].set_title("(a) PR-AUC")
axes[0].set_ylabel("PR-AUC")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=38, ha="right")
axes[0].set_ylim(0.0, 1.03)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False)

for split in SPLITS:
    vals = [
        float(row_for(primary_df, split, subset)["roc_auc"])
        for subset in PRIMARY_ORDER
    ]
    axes[1].plot(
        x,
        vals,
        marker="o",
        label=DISPLAY[split],
    )

axes[1].axhline(0.5, linestyle=":", linewidth=0.9, color="0.35")
axes[1].set_title("(b) ROC-AUC")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=38, ha="right")
axes[1].set_ylim(0.0, 1.03)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False)

fig.suptitle("Stage23 subset × split interaction")
fig.tight_layout()

save_figure(
    fig,
    FIGURES / "figure_23_a_subset_split_interaction",
)

print("[OK] Figure 23-A PNG + PDF")
print()


# ==================================================================================================
# FIGURE 23-B — REMOVAL PENALTIES + BOOTSTRAP UNCERTAINTY + INTERACTION
# ==================================================================================================

print(SEP)
print("GENERATE FIGURE 23-B")
print(SEP)

fig, axes = plt.subplots(2, 2, figsize=(7.25, 6.4))

y = np.arange(len(PRIMARY_ABLATIONS))
ylabels = [DISPLAY[s] for s in PRIMARY_ABLATIONS]

# -------------------------- PR penalties -----------------------------------------------------------

ax = axes[0, 0]
for i, row in unc_primary.reset_index(drop=True).iterrows():
    yr = i - 0.12
    yc = i + 0.12

    ax.hlines(
        yr,
        row["random_pr_auc_ci_low"],
        row["random_pr_auc_ci_high"],
        color=cycle_colors[0],
        linewidth=1.3,
    )
    ax.plot(
        row["random_pr_auc_removal_penalty"],
        yr,
        "o",
        color=cycle_colors[0],
    )

    ax.hlines(
        yc,
        row["chronological_pr_auc_ci_low"],
        row["chronological_pr_auc_ci_high"],
        color=cycle_colors[1],
        linewidth=1.3,
    )
    ax.plot(
        row["chronological_pr_auc_removal_penalty"],
        yc,
        "o",
        color=cycle_colors[1],
    )

ax.plot([], [], "o-", color=cycle_colors[0], label="Random-natural")
ax.plot([], [], "o-", color=cycle_colors[1], label="Chronological-natural")
ax.axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
ax.set_yticks(y)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()
ax.set_xlabel("FULL − ablated PR-AUC")
ax.set_title("(a) PR-AUC removal penalty")
ax.grid(axis="x", alpha=0.25)
ax.legend(frameon=False)


# -------------------------- PR interaction ---------------------------------------------------------

ax = axes[0, 1]
for i, row in unc_primary.reset_index(drop=True).iterrows():
    ax.hlines(
        i,
        row["pr_auc_interaction_ci_low"],
        row["pr_auc_interaction_ci_high"],
        color=cycle_colors[2],
        linewidth=1.4,
    )
    ax.plot(
        row["pr_auc_interaction"],
        i,
        "o",
        color=cycle_colors[2],
    )

ax.axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
ax.set_yticks(y)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()
ax.set_xlabel("Random penalty − chronological penalty")
ax.set_title("(b) PR-AUC shortcut interaction")
ax.grid(axis="x", alpha=0.25)


# -------------------------- ROC penalties ----------------------------------------------------------

ax = axes[1, 0]
for i, row in unc_primary.reset_index(drop=True).iterrows():
    yr = i - 0.12
    yc = i + 0.12

    ax.hlines(
        yr,
        row["random_roc_auc_ci_low"],
        row["random_roc_auc_ci_high"],
        color=cycle_colors[0],
        linewidth=1.3,
    )
    ax.plot(
        row["random_roc_auc_removal_penalty"],
        yr,
        "o",
        color=cycle_colors[0],
    )

    ax.hlines(
        yc,
        row["chronological_roc_auc_ci_low"],
        row["chronological_roc_auc_ci_high"],
        color=cycle_colors[1],
        linewidth=1.3,
    )
    ax.plot(
        row["chronological_roc_auc_removal_penalty"],
        yc,
        "o",
        color=cycle_colors[1],
    )

ax.plot([], [], "o-", color=cycle_colors[0], label="Random-natural")
ax.plot([], [], "o-", color=cycle_colors[1], label="Chronological-natural")
ax.axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
ax.set_yticks(y)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()
ax.set_xlabel("FULL − ablated ROC-AUC")
ax.set_title("(c) ROC-AUC removal penalty")
ax.grid(axis="x", alpha=0.25)
ax.legend(frameon=False)


# -------------------------- ROC interaction --------------------------------------------------------

ax = axes[1, 1]
for i, row in unc_primary.reset_index(drop=True).iterrows():
    ax.hlines(
        i,
        row["roc_auc_interaction_ci_low"],
        row["roc_auc_interaction_ci_high"],
        color=cycle_colors[2],
        linewidth=1.4,
    )
    ax.plot(
        row["roc_auc_interaction"],
        i,
        "o",
        color=cycle_colors[2],
    )

ax.axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
ax.set_yticks(y)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()
ax.set_xlabel("Random penalty − chronological penalty")
ax.set_title("(d) ROC-AUC shortcut interaction")
ax.grid(axis="x", alpha=0.25)

fig.suptitle(
    "Stage23 removal penalties and frozen paired-bootstrap uncertainty"
)
fig.tight_layout()

save_figure(
    fig,
    FIGURES / "figure_23_b_removal_penalties_and_interaction",
)

print("[OK] Figure 23-B PNG + PDF")
print()


# ==================================================================================================
# FIGURE 23-C — SINGLE-FEATURE STUMP DEGRADATION
# ==================================================================================================

print(SEP)
print("GENERATE FIGURE 23-C")
print(SEP)

fig, axes = plt.subplots(1, 2, figsize=(7.25, 3.15))

x = np.arange(len(STUMP_ORDER))
width = 0.34

axes[0].bar(
    x - width / 2,
    stump_df["random_pr_auc"].to_numpy(),
    width,
    label="Random-natural",
)
axes[0].bar(
    x + width / 2,
    stump_df["chronological_pr_auc"].to_numpy(),
    width,
    label="Chronological-natural",
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(STUMP_ORDER, rotation=25, ha="right")
axes[0].set_ylabel("PR-AUC")
axes[0].set_ylim(0.0, 0.34)
axes[0].set_title("(a) Single-feature PR-AUC")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False)

axes[1].bar(
    x - width / 2,
    stump_df["random_roc_auc"].to_numpy(),
    width,
    label="Random-natural",
)
axes[1].bar(
    x + width / 2,
    stump_df["chronological_roc_auc"].to_numpy(),
    width,
    label="Chronological-natural",
)
axes[1].axhline(0.5, linestyle=":", linewidth=0.9, color="0.35")
axes[1].set_xticks(x)
axes[1].set_xticklabels(STUMP_ORDER, rotation=25, ha="right")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_ylim(0.45, 0.80)
axes[1].set_title("(b) Single-feature ROC-AUC")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False)

fig.suptitle("Stage23 depth-1 stump degradation across split regimes")
fig.tight_layout()

save_figure(
    fig,
    FIGURES / "figure_23_c_single_feature_stump_degradation",
)

print("[OK] Figure 23-C PNG + PDF")
print()


# ==================================================================================================
# FIGURE 23-D — SHAP PROXY ABSORPTION
# DESCRIPTIVE CONSENSUS ONLY — NOT EXACT SHAP FOR THE PROBABILITY-AVERAGED ENSEMBLE
# ==================================================================================================

print(SEP)
print("GENERATE FIGURE 23-D")
print(SEP)

consensus_primary = shap_table.copy()

jaccard_matrix = np.zeros((len(PRIMARY_ABLATIONS), len(SPLITS)), dtype=float)
gain_matrix = np.zeros_like(jaccard_matrix)
gainer_names = np.empty(jaccard_matrix.shape, dtype=object)

for i, subset in enumerate(PRIMARY_ABLATIONS):
    for j, split in enumerate(SPLITS):
        r = consensus_primary[
            (consensus_primary["subset"] == subset)
            & (consensus_primary["split"] == split)
        ]
        assert len(r) == 1

        r = r.iloc[0]
        jaccard_matrix[i, j] = float(r["jaccard_at_10_vs_full"])
        gain_matrix[i, j] = float(r["top_retained_consensus_share_gain"])
        gainer_names[i, j] = str(r["top_retained_consensus_share_gainer"])

fig, axes = plt.subplots(1, 2, figsize=(7.25, 4.35))

im0 = axes[0].imshow(
    jaccard_matrix,
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    cmap="Blues",
)

axes[0].set_xticks(np.arange(len(SPLITS)))
axes[0].set_xticklabels([DISPLAY[s] for s in SPLITS], rotation=15)
axes[0].set_yticks(np.arange(len(PRIMARY_ABLATIONS)))
axes[0].set_yticklabels([DISPLAY[s] for s in PRIMARY_ABLATIONS])
axes[0].set_title("(a) Top-10 overlap with FULL")

for i in range(jaccard_matrix.shape[0]):
    for j in range(jaccard_matrix.shape[1]):
        axes[0].text(
            j,
            i,
            f"{jaccard_matrix[i, j]:.3f}",
            ha="center",
            va="center",
            fontsize=7,
        )

fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="Jaccard@10")

gain_max = float(np.nanmax(gain_matrix))
assert gain_max > 0.0

im1 = axes[1].imshow(
    gain_matrix,
    aspect="auto",
    vmin=0.0,
    vmax=gain_max,
    cmap="magma",
)

axes[1].set_xticks(np.arange(len(SPLITS)))
axes[1].set_xticklabels([DISPLAY[s] for s in SPLITS], rotation=15)
axes[1].set_yticks(np.arange(len(PRIMARY_ABLATIONS)))
axes[1].set_yticklabels([DISPLAY[s] for s in PRIMARY_ABLATIONS])
axes[1].set_title("(b) Largest retained-feature share gain")

for i in range(gain_matrix.shape[0]):
    for j in range(gain_matrix.shape[1]):
        name = str(gainer_names[i, j])
        wrapped = "\n".join(textwrap.wrap(name, width=16))
        axes[1].text(
            j,
            i,
            f"{wrapped}\n{gain_matrix[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.7,
        )

fig.colorbar(
    im1,
    ax=axes[1],
    fraction=0.046,
    pad=0.04,
    label="Δ normalized consensus share",
)

fig.suptitle(
    "Stage23 SHAP proxy absorption — descriptive 0.5/0.5 normalized consensus"
)
fig.tight_layout()

save_figure(
    fig,
    FIGURES / "figure_23_d_shap_proxy_absorption",
)

print("[OK] Figure 23-D PNG + PDF")
print("[NOTE] Figure 23-D consensus is descriptive only; it is NOT exact ensemble SHAP.")
print()


# ==================================================================================================
# SUPPLEMENTARY FIGURE S1 — MATCHED-SIZE PLACEBO INTERACTIONS
# ==================================================================================================

print(SEP)
print("GENERATE SUPPLEMENTARY FIGURE 23-S1 — PLACEBOS")
print(SEP)

fig, axes = plt.subplots(1, 2, figsize=(7.25, 3.35))

y = np.arange(len(PLACEBO_ORDER))
labels = [DISPLAY[x] for x in PLACEBO_ORDER]

for i, row in unc_placebo.reset_index(drop=True).iterrows():
    axes[0].hlines(
        i,
        row["pr_auc_interaction_ci_low"],
        row["pr_auc_interaction_ci_high"],
        color=cycle_colors[2],
        linewidth=1.4,
    )
    axes[0].plot(
        row["pr_auc_interaction"],
        i,
        "o",
        color=cycle_colors[2],
    )

axes[0].axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
axes[0].set_yticks(y)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Random penalty − chronological penalty")
axes[0].set_title("(a) PR-AUC interaction")
axes[0].grid(axis="x", alpha=0.25)

for i, row in unc_placebo.reset_index(drop=True).iterrows():
    axes[1].hlines(
        i,
        row["roc_auc_interaction_ci_low"],
        row["roc_auc_interaction_ci_high"],
        color=cycle_colors[2],
        linewidth=1.4,
    )
    axes[1].plot(
        row["roc_auc_interaction"],
        i,
        "o",
        color=cycle_colors[2],
    )

axes[1].axvline(0.0, linestyle=":", linewidth=0.9, color="0.35")
axes[1].set_yticks(y)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].set_xlabel("Random penalty − chronological penalty")
axes[1].set_title("(b) ROC-AUC interaction")
axes[1].grid(axis="x", alpha=0.25)

fig.suptitle("Stage23 matched-size placebo ablation interactions")
fig.tight_layout()

save_figure(
    fig,
    FIGURES / "figure_23_s1_placebo_interactions",
)

print("[OK] Supplementary Figure 23-S1 PNG + PDF")
print()


# ==================================================================================================
# SUPPLEMENTARY FIGURE S2 — ATTACK-FAMILY-CONDITIONED DELTA F1
# ==================================================================================================

print(SEP)
print("GENERATE SUPPLEMENTARY FIGURE 23-S2 — ATTACK FAMILY")
print(SEP)

random_family_order = (
    random_full_attack[
        random_full_attack["interpretation_status"] == "INTERPRETABLE"
    ]["attack_family"]
    .tolist()
)

chrono_family_order = (
    chrono_full_attack[
        chrono_full_attack["interpretation_status"] == "INTERPRETABLE"
    ]["attack_family"]
    .tolist()
)

assert len(random_family_order) == 11
assert chrono_family_order == ["Infilteration"]

rmat = (
    attack_df[
        (attack_df["split"] == "RANDOM_NATURAL")
        & (attack_df["interpretation_status"] == "INTERPRETABLE")
        & (attack_df["subset"].isin(PRIMARY_ABLATIONS))
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(index=random_family_order, columns=PRIMARY_ABLATIONS)
)

cmat = (
    attack_df[
        (attack_df["split"] == "CHRONOLOGICAL_NATURAL")
        & (attack_df["interpretation_status"] == "INTERPRETABLE")
        & (attack_df["subset"].isin(PRIMARY_ABLATIONS))
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(index=chrono_family_order, columns=PRIMARY_ABLATIONS)
)

assert not rmat.isna().any().any()
assert not cmat.isna().any().any()

allvals = np.concatenate([rmat.to_numpy().ravel(), cmat.to_numpy().ravel()])
vmin = float(np.nanmin(allvals))
vmax = float(np.nanmax(allvals))

if vmin < 0.0 < vmax:
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    image_kwargs = {"norm": norm}
else:
    image_kwargs = {"vmin": vmin, "vmax": vmax}

fig, axes = plt.subplots(
    2,
    1,
    figsize=(7.25, 6.2),
    gridspec_kw={"height_ratios": [5.0, 1.2]},
)

im = axes[0].imshow(
    rmat.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    **image_kwargs,
)

axes[0].set_xticks(np.arange(len(PRIMARY_ABLATIONS)))
axes[0].set_xticklabels(
    [DISPLAY[x] for x in PRIMARY_ABLATIONS],
    rotation=30,
    ha="right",
)
axes[0].set_yticks(np.arange(len(random_family_order)))
axes[0].set_yticklabels(random_family_order)
axes[0].set_title("(a) Random-natural — interpretable families (support ≥100)")

for i in range(rmat.shape[0]):
    for j in range(rmat.shape[1]):
        axes[0].text(
            j,
            i,
            f"{rmat.iloc[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.7,
        )

im2 = axes[1].imshow(
    cmat.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    **image_kwargs,
)

axes[1].set_xticks(np.arange(len(PRIMARY_ABLATIONS)))
axes[1].set_xticklabels(
    [DISPLAY[x] for x in PRIMARY_ABLATIONS],
    rotation=30,
    ha="right",
)
axes[1].set_yticks(np.arange(len(chrono_family_order)))
axes[1].set_yticklabels(chrono_family_order)
axes[1].set_title("(b) Chronological-natural — interpretable family")

for i in range(cmat.shape[0]):
    for j in range(cmat.shape[1]):
        axes[1].text(
            j,
            i,
            f"{cmat.iloc[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=6.2,
        )

cbar = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    fraction=0.025,
    pad=0.02,
)
cbar.set_label("ΔF1 = FULL − ablated at threshold 0.50")

fig.suptitle("Stage23 attack-family-conditioned degradation")
fig.subplots_adjust(
    left=0.22,
    right=0.90,
    top=0.91,
    bottom=0.14,
    hspace=0.75,
)

save_figure(
    fig,
    FIGURES / "figure_23_s2_attack_family_delta_f1",
)

print("[OK] Supplementary Figure 23-S2 PNG + PDF")
print()


# ==================================================================================================
# SCIENTIFIC SYNTHESIS — EXACT SEALED NUMBERS ONLY
# ==================================================================================================

print(SEP)
print("GENERATE CONSOLIDATED SCIENTIFIC SYNTHESIS")
print(SEP)
print()

full_r = row_for(primary_df, "RANDOM_NATURAL", "FULL")
full_c = row_for(primary_df, "CHRONOLOGICAL_NATURAL", "FULL")
beh_r = row_for(primary_df, "RANDOM_NATURAL", "BEHAVIOR_ONLY")
beh_c = row_for(primary_df, "CHRONOLOGICAL_NATURAL", "BEHAVIOR_ONLY")

noinit_unc = unc_primary[
    unc_primary["comparison"] == "NO_INIT_FWD_WIN_BYTS"
].iloc[0]

susp_unc = unc_primary[
    unc_primary["comparison"] == "NO_SUSPICIOUS_GROUP"
].iloc[0]

dst_unc = unc_primary[
    unc_primary["comparison"] == "NO_DST_PORT"
].iloc[0]

packet_placebo_unc = unc_placebo[
    unc_placebo["comparison"] == "PLACEBO_PACKET_SIZE"
].iloc[0]

iat_placebo_unc = unc_placebo[
    unc_placebo["comparison"] == "PLACEBO_IAT"
].iloc[0]

stump_pr_min_random = float(stump_df["random_pr_auc"].min())
stump_pr_max_random = float(stump_df["random_pr_auc"].max())
stump_pr_min_chrono = float(stump_df["chronological_pr_auc"].min())
stump_pr_max_chrono = float(stump_df["chronological_pr_auc"].max())

stump_roc_min_random = float(stump_df["random_roc_auc"].min())
stump_roc_max_random = float(stump_df["random_roc_auc"].max())
stump_roc_min_chrono = float(stump_df["chronological_roc_auc"].min())
stump_roc_max_chrono = float(stump_df["chronological_roc_auc"].max())

chrono_attack_support = validation_meta["CHRONOLOGICAL_NATURAL"]["attack"]
random_attack_support = validation_meta["RANDOM_NATURAL"]["attack"]

chrono_prev = validation_meta["CHRONOLOGICAL_NATURAL"]["attack_prevalence"]
random_prev = validation_meta["RANDOM_NATURAL"]["attack_prevalence"]

primary_pr_significant = unc_primary[
    unc_primary["pr_auc_interaction_ci_excludes_zero"]
]["comparison"].tolist()

primary_roc_significant = unc_primary[
    unc_primary["roc_auc_interaction_ci_excludes_zero"]
]["comparison"].tolist()

placebo_pr_significant = unc_placebo[
    unc_placebo["pr_auc_interaction_ci_excludes_zero"]
]["comparison"].tolist()

placebo_roc_significant = unc_placebo[
    unc_placebo["roc_auc_interaction_ci_excludes_zero"]
]["comparison"].tolist()

synthesis_md = f"""
# Stage23 Final Scientific Synthesis

## Scope

Stage23 is a frozen development-only shortcut-feature audit. It compares inherited
RANDOM_NATURAL and CHRONOLOGICAL_NATURAL validation regimes under frozen primary
ablations, matched-size placebo removals, depth-1 single-feature controls, frozen
paired-bootstrap uncertainty, component-specific TreeSHAP proxy-absorption analysis,
and frozen attack-family support rules.

No Stage23 result permits a causal leakage claim. Raw Mar1 and Mar2 remain permanently
forbidden, and Stage23 does not create a new untouched final holdout.

## 1. Dominant finding: extreme validation-regime sensitivity

Under the FULL 70-feature representation:

- RANDOM_NATURAL PR-AUC = {full_r["pr_auc"]:.12f}
- RANDOM_NATURAL ROC-AUC = {full_r["roc_auc"]:.12f}
- RANDOM_NATURAL attack prevalence = {random_prev:.12f}
- RANDOM_NATURAL PR-AUC minus prevalence = {full_r["pr_auc_minus_attack_prevalence"]:.12f}

- CHRONOLOGICAL_NATURAL PR-AUC = {full_c["pr_auc"]:.12f}
- CHRONOLOGICAL_NATURAL ROC-AUC = {full_c["roc_auc"]:.12f}
- CHRONOLOGICAL_NATURAL attack prevalence = {chrono_prev:.12f}
- CHRONOLOGICAL_NATURAL PR-AUC minus prevalence = {full_c["pr_auc_minus_attack_prevalence"]:.12f}

The model therefore shows excellent discrimination under RANDOM_NATURAL but only
near-baseline ranking under the frozen forward-temporal validation. This supports a
strong claim of validation sensitivity. It does not establish that random splitting
is universally invalid, nor that chronological splitting completely eliminates leakage.

## 2. Frozen primary ablations

The PR-AUC shortcut interaction is defined exactly as:

    (FULL_RANDOM - ABLATED_RANDOM) - (FULL_CHRONOLOGICAL - ABLATED_CHRONOLOGICAL)

Positive interaction means random validation benefits disproportionately from the
tested information relative to chronological validation. Negative interaction means
chronological validation depends more strongly on that tested information.

Primary PR-AUC interactions whose frozen 95% bootstrap CIs exclude zero:

{chr(10).join("- " + x for x in primary_pr_significant)}

Primary ROC-AUC interactions whose frozen 95% bootstrap CIs exclude zero:

{chr(10).join("- " + x for x in primary_roc_significant)}

Two important examples are:

- NO_INIT_FWD_WIN_BYTS PR interaction =
  {signed6(noinit_unc["pr_auc_interaction"])}
  [{signed6(noinit_unc["pr_auc_interaction_ci_low"])},
   {signed6(noinit_unc["pr_auc_interaction_ci_high"])}].

- NO_SUSPICIOUS_GROUP PR interaction =
  {signed6(susp_unc["pr_auc_interaction"])}
  [{signed6(susp_unc["pr_auc_interaction_ci_low"])},
   {signed6(susp_unc["pr_auc_interaction_ci_high"])}].

By contrast, NO_DST_PORT produces a negative PR interaction of
{signed6(dst_unc["pr_auc_interaction"])}
[{signed6(dst_unc["pr_auc_interaction_ci_low"])},
 {signed6(dst_unc["pr_auc_interaction_ci_high"])}],
meaning that Dst Port removal hurts chronological PR-AUC more than random PR-AUC
under these frozen validations.

The metric direction is not uniform. For NO_SUSPICIOUS_GROUP, the PR interaction is
positive while the ROC interaction is negative
({signed6(susp_unc["roc_auc_interaction"])}). This is an important reason not to
reduce Stage23 to a binary "leakage / not leakage" claim.

## 3. Matched-size placebo context

Placebo PR-AUC interactions whose frozen CIs exclude zero:

{chr(10).join("- " + x for x in placebo_pr_significant)}

Placebo ROC-AUC interactions whose frozen CIs exclude zero:

{chr(10).join("- " + x for x in placebo_roc_significant)}

Notably, PLACEBO_PACKET_SIZE has a positive PR-AUC interaction of
{signed6(packet_placebo_unc["pr_auc_interaction"])}
[{signed6(packet_placebo_unc["pr_auc_interaction_ci_low"])},
 {signed6(packet_placebo_unc["pr_auc_interaction_ci_high"])}].

PLACEBO_IAT has a negative PR-AUC interaction of
{signed6(iat_placebo_unc["pr_auc_interaction"])}
[{signed6(iat_placebo_unc["pr_auc_interaction_ci_low"])},
 {signed6(iat_placebo_unc["pr_auc_interaction_ci_high"])}].

Thus, exclusion of zero by an interaction CI is not unique to the pre-specified
shortcut-prone feature removals. Statistical separation from zero is therefore not,
by itself, evidence that a tested feature is leakage.

## 4. Single-feature stump controls

Across the three frozen depth-1 controls:

- RANDOM_NATURAL PR-AUC range =
  [{stump_pr_min_random:.6f}, {stump_pr_max_random:.6f}]
- CHRONOLOGICAL_NATURAL PR-AUC range =
  [{stump_pr_min_chrono:.6f}, {stump_pr_max_chrono:.6f}]
- RANDOM_NATURAL ROC-AUC range =
  [{stump_roc_min_random:.6f}, {stump_roc_max_random:.6f}]
- CHRONOLOGICAL_NATURAL ROC-AUC range =
  [{stump_roc_min_chrono:.6f}, {stump_roc_max_chrono:.6f}]

All three single-feature controls discriminate substantially better under random
validation and collapse toward weak or near-chance discrimination under chronological
validation. This is consistent with split-specific / shortcut-like signal that transfers
poorly. It is not proof of leakage or causality.

## 5. SHAP proxy absorption

TreeSHAP was computed separately for LightGBM and XGBoost on the frozen 5,000-row
balanced cohort for each split. Component-specific results are primary. The 0.5/0.5
consensus used in Figure 23-D is descriptive only: it averages normalized component
importance shares and is not exact SHAP for the probability-averaged ensemble.

Removal of the suspicious group and the behavior-restricted representation produce
substantial top-rank reorganization, while narrower removals such as
NO_FWD_SEG_SIZE_MIN retain considerably greater rank stability. Retained features
increase normalized importance after removals, providing evidence consistent with
proxy absorption. SHAP importance does not prove causation.

Complete component-specific top-20 tables are exported as
`table_23_s4_component_specific_shap_top20.csv`.

## 6. Behavior-restricted representation

BEHAVIOR_ONLY:

- RANDOM_NATURAL PR-AUC = {beh_r["pr_auc"]:.12f}
- RANDOM_NATURAL ROC-AUC = {beh_r["roc_auc"]:.12f}
- RANDOM_NATURAL F1@0.50 = {beh_r["f1_at_0_50"]:.12f}

- CHRONOLOGICAL_NATURAL PR-AUC = {beh_c["pr_auc"]:.12f}
- CHRONOLOGICAL_NATURAL ROC-AUC = {beh_c["roc_auc"]:.12f}
- CHRONOLOGICAL_NATURAL F1@0.50 = {beh_c["f1_at_0_50"]:.12f}

The chronological behavior-only PR-AUC is below its split-specific attack-prevalence
reference ({chrono_prev:.12f}), and ROC-AUC is below 0.5. Therefore Stage23 does not
support a claim that substantial behavior-restricted chronological capability survives.
Rather, performance under this protocol depends materially on information excluded by
the frozen behavior-only definition, while FULL chronological performance itself
remains weak.

This does not imply that BEHAVIOR_ONLY equals real-world deployment performance.

## 7. Attack-family context

RANDOM_NATURAL contains {random_attack_support:,} attack validation rows and has
11 frozen attack families meeting the >=100 interpretive-support threshold.

CHRONOLOGICAL_NATURAL contains {chrono_attack_support:,} attack validation rows.
All {chrono_attack_support:,} belong to the frozen `Infilteration` family; the other
12 frozen families have zero chronological attack support.

Consequently, the random-versus-chronological comparison combines forward-temporal
change with a major attack-family composition change. Stage23 cannot attribute the
entire split-performance gap to any single feature, mechanism, or form of leakage.

## 8. Consolidated scientific conclusion

The frozen evidence supports the following bounded conclusion:

1. The evaluated ensemble is highly sensitive to validation regime.
2. Dst Port, Init Fwd Win Byts, and Fwd Seg Size Min individually carry substantially
   more discriminative information under RANDOM_NATURAL than under CHRONOLOGICAL_NATURAL.
3. Some primary feature removals produce split-dependent interaction effects, but the
   direction depends on the metric and tested subset.
4. Matched-size placebo removals also produce non-zero and sometimes comparably sized
   interactions, so significance alone cannot identify leakage.
5. SHAP redistribution after removals is consistent with proxy absorption among retained
   features, but does not establish causal substitution.
6. The chronological split is dominated entirely by Infilteration attack support, which
   materially limits attack-family comparability with random validation.
7. Stage23 therefore supports a validation-sensitivity / poor-transfer interpretation,
   not a claim that any tested feature is proven leakage.

## 9. Claims explicitly not supported

The frozen Stage23 interpretation boundary prohibits:

{chr(10).join("- " + x for x in interpretation_matrix["prohibited_claims"])}

## 10. Governance

- Frozen Stage23 fit budget: 50 / 50 exhausted.
- New model fits in Stage23-7A: 0.
- Model inference in Stage23-7A: 0.
- New SHAP computation in Stage23-7A: 0.
- New bootstrap sampling in Stage23-7A: 0.
- Parquet reads in Stage23-7A: 0.
- Raw Mar1 access: NO.
- Raw Mar2 access: NO.
- Rolling-forward analysis: NOT INCLUDED.

The generated package satisfies the final synthesis, publication-table,
publication-figure, supplementary-output, and manuscript-integration portions of the
frozen Stage23 stopping rule. It remains UNSEALED until the separate Stage23 closure
seal/push is executed.
"""

write_text(
    TMP / "stage23_7_final_scientific_synthesis.md",
    synthesis_md,
)


# ==================================================================================================
# LATEX / IEEE MANUSCRIPT INTEGRATION
# ==================================================================================================

print(SEP)
print("GENERATE LATEX MANUSCRIPT INTEGRATION")
print(SEP)
print()

def subset_tex(x: str) -> str:
    return r"\texttt{" + latex_escape(x) + "}"


# --------------------------------------------------------------------------------------------------
# Main table snippets
# --------------------------------------------------------------------------------------------------

primary_tex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Stage23 primary subset performance under the frozen random-natural and chronological-natural development validations.}",
    r"\label{tab:stage23-primary}",
    r"\small",
    r"\begin{tabular}{lrrrr}",
    r"\hline",
    r"Subset & Random PR-AUC & Chronological PR-AUC & Random ROC-AUC & Chronological ROC-AUC \\",
    r"\hline",
]

for _, r in primary_wide.iterrows():
    primary_tex_lines.append(
        f"{subset_tex(r['subset'])} & "
        f"{r['random_pr_auc']:.6f} & "
        f"{r['chronological_pr_auc']:.6f} & "
        f"{r['random_roc_auc']:.6f} & "
        f"{r['chronological_roc_auc']:.6f} \\\\"
    )

primary_tex_lines += [
    r"\hline",
    r"\end{tabular}",
    r"\end{table*}",
    "",
]


interaction_tex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Frozen Stage23 shortcut interactions. Interactions are random removal penalty minus chronological removal penalty; intervals are frozen paired stratified bootstrap percentile 95\% CIs.}",
    r"\label{tab:stage23-interactions}",
    r"\small",
    r"\begin{tabular}{lll}",
    r"\hline",
    r"Subset & PR-AUC interaction [95\% CI] & ROC-AUC interaction [95\% CI] \\",
    r"\hline",
]

for _, r in unc_primary.iterrows():
    interaction_tex_lines.append(
        f"{subset_tex(r['comparison'])} & "
        f"{r['pr_auc_interaction']:+.6f} "
        f"[{r['pr_auc_interaction_ci_low']:+.6f}, "
        f"{r['pr_auc_interaction_ci_high']:+.6f}] & "
        f"{r['roc_auc_interaction']:+.6f} "
        f"[{r['roc_auc_interaction_ci_low']:+.6f}, "
        f"{r['roc_auc_interaction_ci_high']:+.6f}] \\\\"
    )

interaction_tex_lines += [
    r"\hline",
    r"\end{tabular}",
    r"\end{table*}",
    "",
]


stump_tex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Frozen depth-1 single-feature stump controls. Degradation is random-natural minus chronological-natural performance.}",
    r"\label{tab:stage23-stumps}",
    r"\small",
    r"\begin{tabular}{lrrrrrr}",
    r"\hline",
    r"Feature & Random PR & Chrono PR & $\Delta$PR & Random ROC & Chrono ROC & $\Delta$ROC \\",
    r"\hline",
]

for _, r in stump_df.iterrows():
    stump_tex_lines.append(
        f"{latex_escape(r['feature'])} & "
        f"{r['random_pr_auc']:.6f} & "
        f"{r['chronological_pr_auc']:.6f} & "
        f"{r['random_minus_chronological_pr_auc']:+.6f} & "
        f"{r['random_roc_auc']:.6f} & "
        f"{r['chronological_roc_auc']:.6f} & "
        f"{r['random_minus_chronological_roc_auc']:+.6f} \\\\"
    )

stump_tex_lines += [
    r"\hline",
    r"\end{tabular}",
    r"\end{table*}",
    "",
]

write_text(
    MANUSCRIPT / "stage23_main_tables.tex",
    "\n".join(
        primary_tex_lines
        + interaction_tex_lines
        + stump_tex_lines
    ),
)


# --------------------------------------------------------------------------------------------------
# Results/discussion manuscript fragment
# --------------------------------------------------------------------------------------------------

manuscript_tex = rf"""
% ================================================================================================
% STAGE23 MANUSCRIPT INTEGRATION
% Generated entirely from sealed Stage23 evidence.
% No additional model fitting, inference, SHAP computation, or bootstrap sampling.
% ================================================================================================

\subsection{{Validation-Safe Shortcut-Feature Audit}}

Stage23 evaluated whether several pre-specified feature groups exhibited validation-regime-dependent
predictive behavior under frozen random-natural and chronological-natural development splits.
The full 70-feature ensemble achieved a PR-AUC of {full_r["pr_auc"]:.6f} and ROC-AUC of
{full_r["roc_auc"]:.6f} under random-natural validation, compared with PR-AUC
{full_c["pr_auc"]:.6f} and ROC-AUC {full_c["roc_auc"]:.6f} under chronological-natural
validation. The corresponding attack prevalences were {random_prev:.6f} and
{chrono_prev:.6f}, respectively. Thus, the chronological PR-AUC was only
{full_c["pr_auc_minus_attack_prevalence"]:.6f} above its prevalence reference, while
its ROC-AUC was close to the 0.5 no-skill reference.

Table~\ref{{tab:stage23-primary}} summarizes all seven frozen primary representations.
Ablation effects were metric dependent. Removing \texttt{{Init Fwd Win Byts}} produced
a positive PR-AUC shortcut interaction of {noinit_unc["pr_auc_interaction"]:+.6f}
(95\% CI: {noinit_unc["pr_auc_interaction_ci_low"]:+.6f} to
{noinit_unc["pr_auc_interaction_ci_high"]:+.6f}), indicating that random validation
benefited disproportionately from this feature under the frozen PR-AUC definition.
Removing the complete suspicious group produced a PR-AUC interaction of
{susp_unc["pr_auc_interaction"]:+.6f}
(95\% CI: {susp_unc["pr_auc_interaction_ci_low"]:+.6f} to
{susp_unc["pr_auc_interaction_ci_high"]:+.6f}). However, its ROC-AUC interaction was
{susp_unc["roc_auc_interaction"]:+.6f}, showing that the direction of the interaction
was not invariant across ranking metrics.

Conversely, removal of \texttt{{Dst Port}} yielded a negative PR-AUC interaction of
{dst_unc["pr_auc_interaction"]:+.6f}
(95\% CI: {dst_unc["pr_auc_interaction_ci_low"]:+.6f} to
{dst_unc["pr_auc_interaction_ci_high"]:+.6f}), because the chronological removal
penalty exceeded the random removal penalty. Therefore, the frozen ablation evidence
does not support a uniform interpretation in which every pre-specified suspicious
feature disproportionately benefits random validation.

\subsubsection{{Placebo Context and Uncertainty}}

The same paired stratified bootstrap protocol was applied to five matched-size placebo
ablations. Importantly, placebo interactions also excluded zero. For example,
\texttt{{PLACEBO PACKET SIZE}} produced a PR-AUC interaction of
{packet_placebo_unc["pr_auc_interaction"]:+.6f}
(95\% CI: {packet_placebo_unc["pr_auc_interaction_ci_low"]:+.6f} to
{packet_placebo_unc["pr_auc_interaction_ci_high"]:+.6f}), while
\texttt{{PLACEBO IAT}} produced an interaction of
{iat_placebo_unc["pr_auc_interaction"]:+.6f}
(95\% CI: {iat_placebo_unc["pr_auc_interaction_ci_low"]:+.6f} to
{iat_placebo_unc["pr_auc_interaction_ci_high"]:+.6f}). Consequently, CI exclusion
from zero is not unique to the pre-specified shortcut-prone feature groups and is not,
by itself, evidence of leakage.

\subsubsection{{Single-Feature Controls}}

The frozen depth-1 stump controls provided a complementary view. Across
\texttt{{Dst Port}}, \texttt{{Init Fwd Win Byts}}, and \texttt{{Fwd Seg Size Min}},
random-natural PR-AUC ranged from {stump_pr_min_random:.6f} to
{stump_pr_max_random:.6f}, whereas chronological-natural PR-AUC ranged only from
{stump_pr_min_chrono:.6f} to {stump_pr_max_chrono:.6f}. Random-natural ROC-AUC ranged
from {stump_roc_min_random:.6f} to {stump_roc_max_random:.6f}, compared with
{stump_roc_min_chrono:.6f} to {stump_roc_max_chrono:.6f} chronologically.
These results are consistent with split-specific discriminative cues that transfer
poorly forward in time, but do not establish leakage or causality.

\subsubsection{{SHAP Proxy Absorption}}

TreeSHAP was evaluated separately for LightGBM and XGBoost using the frozen balanced
5,000-row cohort for each validation regime. Component-specific results remain the
primary explanation outputs. For visualization only, normalized component mean-absolute
SHAP shares were averaged with equal 0.5/0.5 weighting. This descriptive consensus is
not exact SHAP for the probability-averaged ensemble.

The removal experiments produced measurable reorganization of top-ranked retained
features, particularly for the suspicious-group and behavior-restricted representations.
Several retained packet-length, inter-arrival-time, header, and flow features increased
their normalized importance after removal of higher-ranked features. This behavior is
consistent with proxy absorption; it does not demonstrate causal substitution.

\subsubsection{{Behavior-Restricted Representation}}

The behavior-restricted representation retained high random-natural discrimination
(PR-AUC {beh_r["pr_auc"]:.6f}, ROC-AUC {beh_r["roc_auc"]:.6f}) but did not preserve
chronological discrimination (PR-AUC {beh_c["pr_auc"]:.6f}, ROC-AUC
{beh_c["roc_auc"]:.6f}). The chronological behavior-only PR-AUC was below the
split-specific prevalence reference ({chrono_prev:.6f}), and its ROC-AUC was below
0.5. Accordingly, the frozen protocol indicates substantial dependence on information
excluded by the behavior-only definition. This result must not be interpreted as a
direct estimate of real-world deployment performance.

\subsubsection{{Attack-Family Context}}

Attack-family conditioning revealed a major composition difference between validation
regimes. Random-natural validation contained {random_attack_support:,} attacks and
11 families meeting the frozen minimum support of 100. Chronological-natural validation
contained {chrono_attack_support:,} attacks, all of which were labeled
\texttt{{Infilteration}} under the frozen Stage23 family mapping; the other 12 frozen
families had zero chronological attack support. Therefore, the observed split-performance
gap combines temporal change with an attack-family composition change, and Stage23
cannot attribute that gap to a single feature or leakage mechanism.

\subsubsection{{Stage23 Interpretation}}

Taken together, Stage23 demonstrates strong validation sensitivity and poor temporal
transfer of several discriminative cues. Some pre-specified removals yield
split-dependent penalties and SHAP redistribution consistent with proxy absorption,
but the effects are not uniform across metrics and comparable effects are observed for
matched-size placebo removals. The evidence therefore supports a bounded
validation-sensitivity interpretation rather than a causal claim that any tested
feature is proven leakage.

% --------------------------------------------------------------------------------
% Frozen figure captions
% --------------------------------------------------------------------------------

% Figure 23-A:
% Subset x split interaction for all seven primary representations. PR-AUC no-skill
% references equal the split-specific attack prevalence; ROC-AUC no-skill reference
% equals 0.5.

% Figure 23-B:
% Full-minus-ablated PR-AUC and ROC-AUC penalties together with frozen paired
% stratified bootstrap percentile 95 percent CIs and random-minus-chronological
% shortcut interactions.

% Figure 23-C:
% Frozen depth-1 single-feature stump controls for Dst Port, Init Fwd Win Byts,
% and Fwd Seg Size Min under random-natural and chronological-natural validation.

% Figure 23-D:
% SHAP proxy-absorption summary using the descriptive equal-weight consensus of
% normalized LightGBM and XGBoost mean-absolute SHAP shares. The consensus is
% descriptive and is not exact ensemble SHAP.
"""

write_text(
    MANUSCRIPT / "stage23_results_integration.tex",
    manuscript_tex,
)

figure_captions_tex = r"""
% Stage23 frozen publication figure captions

\paragraph{Figure 23-A.}
Subset-by-split interaction across all seven frozen primary representations.
The left panel reports PR-AUC, with split-specific attack prevalence shown as the
no-skill reference; the right panel reports ROC-AUC with 0.5 as the no-skill reference.

\paragraph{Figure 23-B.}
Full-minus-ablated ranking penalties and shortcut interactions under the frozen Stage23
uncertainty protocol. Points are complete-validation estimates; intervals are frozen
paired stratified bootstrap percentile 95\% confidence intervals. Positive interaction
indicates that random validation benefits disproportionately from the tested information,
whereas negative interaction indicates greater chronological dependence.

\paragraph{Figure 23-C.}
Depth-1 single-feature stump controls for Dst Port, Init Fwd Win Byts, and
Fwd Seg Size Min. Each feature shows substantially stronger discrimination under
random-natural validation than under chronological-natural validation.

\paragraph{Figure 23-D.}
SHAP proxy-absorption summary. Top-10 Jaccard overlap quantifies rank stability relative
to FULL, while the second panel reports the largest positive normalized importance-share
change among retained features. The equal-weight LightGBM/XGBoost consensus is
descriptive only and is not exact SHAP for the probability-averaged ensemble.
"""

write_text(
    MANUSCRIPT / "stage23_figure_captions.tex",
    figure_captions_tex,
)

print("[OK] IEEE/LaTeX results integration")
print("[OK] main LaTeX table snippets")
print("[OK] frozen figure-caption fragment")
print()


# ==================================================================================================
# CONSOLIDATED SCIENTIFIC AUDIT / STOPPING-RULE EVIDENCE
# ==================================================================================================

print(SEP)
print("GENERATE CONSOLIDATED SCIENTIFIC AUDIT")
print(SEP)
print()

main_figure_bases = [
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
]

supp_figure_bases = [
    "figure_23_s1_placebo_interactions",
    "figure_23_s2_attack_family_delta_f1",
]

for base in main_figure_bases + supp_figure_bases:
    assert (FIGURES / f"{base}.png").is_file()
    assert (FIGURES / f"{base}.pdf").is_file()

table_files = sorted(TABLES.glob("*.csv"))
figure_files = sorted(FIGURES.glob("*"))
manuscript_files = sorted(MANUSCRIPT.glob("*"))

assert len(table_files) == 10
assert len(figure_files) == 12
assert len(manuscript_files) == 3

closure_conditions = [
    {
        "condition": stopping_rule["complete_when"][0],
        "satisfied": True,
        "evidence": {
            "primary_subsets": 7,
            "splits": 2,
            "primary_performance_rows": 14,
        },
    },
    {
        "condition": stopping_rule["complete_when"][1],
        "satisfied": True,
        "evidence": {
            "stump_controls": 6,
            "feature_split_pairs": 6,
        },
    },
    {
        "condition": stopping_rule["complete_when"][2],
        "satisfied": True,
        "evidence": {
            "placebo_subsets": 5,
            "splits": 2,
            "placebo_result_rows": 10,
        },
    },
    {
        "condition": stopping_rule["complete_when"][3],
        "satisfied": True,
        "evidence": {
            "component_treeshap_models": 48,
            "proxy_subset_split_comparisons": 22,
            "descriptive_consensus_sets": 24,
            "component_specific_top20_rows": int(len(component_top20)),
        },
    },
    {
        "condition": stopping_rule["complete_when"][4],
        "satisfied": True,
        "evidence": {
            "frozen_families": 13,
            "metric_rows": 182,
            "random_interpretable_families": 11,
            "chronological_interpretable_families": 1,
        },
    },
    {
        "condition": stopping_rule["complete_when"][5],
        "satisfied": True,
        "evidence": {
            "audit_artifact":
                "audit/stage23_7_consolidated_scientific_audit.json",
            "synthesis_artifact":
                "stage23_7_final_scientific_synthesis.md",
        },
    },
    {
        "condition": stopping_rule["complete_when"][6],
        "satisfied": True,
        "evidence": {
            "main_figure_bases": 4,
            "supplementary_figure_bases": 2,
            "figure_files_png_pdf": 12,
            "publication_table_csvs": 10,
            "manuscript_files": 3,
        },
    },
]

assert all(x["satisfied"] for x in closure_conditions)

audit_json = {
    "stage": "Stage23-7A",
    "status": "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "execution_parent": {
        "commit": EXPECTED_PARENT,
        "tag": EXPECTED_PARENT_TAG,
    },

    "frozen_scope": {
        "primary_subsets": PRIMARY_ORDER,
        "placebo_subsets": PLACEBO_ORDER,
        "stump_features": STUMP_ORDER,
        "splits": SPLITS,
        "headline_metrics": ["PR_AUC", "ROC_AUC"],
        "fixed_operating_threshold": 0.50,
        "rolling_forward_analysis":
            stopping_rule["rolling_forward_analysis"],
    },

    "validated_counts": {
        "primary_result_jsons": 12,
        "primary_subset_split_rows": 14,
        "placebo_result_jsons": 10,
        "placebo_subset_split_rows": 10,
        "uncertainty_comparisons": 11,
        "stump_controls": 6,
        "shap_proxy_subset_split_comparisons": 22,
        "shap_component_models": 48,
        "component_specific_shap_top20_rows":
            int(len(component_top20)),
        "attack_family_metric_rows": 182,
        "attack_families": 13,
        "random_interpretable_attack_families": 11,
        "chronological_interpretable_attack_families": 1,
    },

    "headline_full_results": {
        "random_natural": {
            "pr_auc": float(full_r["pr_auc"]),
            "roc_auc": float(full_r["roc_auc"]),
            "attack_prevalence": float(random_prev),
            "pr_auc_minus_attack_prevalence":
                float(full_r["pr_auc_minus_attack_prevalence"]),
        },
        "chronological_natural": {
            "pr_auc": float(full_c["pr_auc"]),
            "roc_auc": float(full_c["roc_auc"]),
            "attack_prevalence": float(chrono_prev),
            "pr_auc_minus_attack_prevalence":
                float(full_c["pr_auc_minus_attack_prevalence"]),
        },
    },

    "attack_family_context": {
        "random_attack_support": int(random_attack_support),
        "chronological_attack_support": int(chrono_attack_support),
        "chronological_positive_attack_families": ["Infilteration"],
        "chronological_infilteration_support":
            int(chrono_attack_support),
    },

    "uncertainty_context": {
        "primary_pr_interactions_ci_excluding_zero":
            primary_pr_significant,
        "primary_roc_interactions_ci_excluding_zero":
            primary_roc_significant,
        "placebo_pr_interactions_ci_excluding_zero":
            placebo_pr_significant,
        "placebo_roc_interactions_ci_excluding_zero":
            placebo_roc_significant,
        "interpretation_boundary":
            "CI exclusion from zero is not unique to primary shortcut-prone removals and is not leakage evidence by itself.",
    },

    "scientific_synthesis": [
        "The frozen ensemble is highly sensitive to validation regime.",
        "The three frozen stump features discriminate substantially better under RANDOM_NATURAL and transfer poorly to CHRONOLOGICAL_NATURAL.",
        "Primary ablation interactions are metric-dependent rather than uniformly shortcut-directed.",
        "Matched-size placebo removals also yield non-zero interactions, including CIs excluding zero.",
        "SHAP redistribution is consistent with proxy absorption but does not establish causation.",
        "The chronological attack support is entirely Infilteration under the frozen family mapping.",
        "The combined evidence supports validation sensitivity and poor temporal transfer, not proof that a tested feature is leakage.",
    ],

    "prohibited_claims":
        interpretation_matrix["prohibited_claims"],

    "stopping_rule_evidence":
        closure_conditions,

    "governance": {
        "stage23_fit_budget": "50/50 SEALED",
        "additional_model_fits_authorized": 0,
        "new_model_fits": 0,
        "model_inference": 0,
        "lightgbm_execution": 0,
        "xgboost_execution": 0,
        "new_shap_values_computed": 0,
        "new_bootstrap_replicates_generated": 0,
        "parquet_files_read": 0,
        "raw_mar1_accessed": False,
        "raw_mar2_accessed": False,
        "stage23_0_modified": False,
        "new_subset_created": False,
        "new_metric_added": False,
        "threshold_optimization": False,
    },

    "publication_outputs": {
        "main_figures": main_figure_bases,
        "supplementary_figures": supp_figure_bases,
        "table_files": [p.name for p in table_files],
        "manuscript_files": [p.name for p in manuscript_files],
    },

    "next_authorized_action":
        "Zero-fit Stage23 final closure seal and push only. No additional model fits or analyses.",
}

write_json(
    AUDIT / "stage23_7_consolidated_scientific_audit.json",
    audit_json,
)

print("[EXACT] stopping-rule conditions satisfied: 7 / 7")
print("[EXACT] main figure bases              : 4 / 4")
print("[EXACT] supplementary figure bases     : 2")
print("[EXACT] publication table CSVs         :", len(table_files))
print("[EXACT] manuscript integration files   :", len(manuscript_files))
print()


# ==================================================================================================
# SOURCE MANIFEST
# ==================================================================================================

source_manifest = {
    "stage": "Stage23-7A",
    "execution_parent_commit": EXPECTED_PARENT,
    "source_policy":
        "SEALED_REPOSITORY_JSON_AND_CSV_ONLY_NO_MODELS_NO_PROBABILITIES_NO_PARQUET",
    "source_files": [
        {
            "path": rel,
            "sha256": digest,
        }
        for rel, digest in sorted(SOURCE_FILES.items())
    ],
    "source_file_count": len(SOURCE_FILES),
    "raw_mar1_accessed": False,
    "raw_mar2_accessed": False,
    "parquet_files_read": 0,
    "model_files_read": 0,
    "probability_npz_files_read": 0,
}

# Fail closed: this synthesis must never have read a model, NPZ, or parquet.
for item in source_manifest["source_files"]:
    p = item["path"].lower()
    assert not p.endswith(".parquet"), p
    assert not p.endswith(".npz"), p
    assert "model.txt" not in p, p
    assert "xgboost_model" not in p, p
    assert not p.endswith(".joblib"), p

write_json(
    AUDIT / "stage23_7_source_manifest.json",
    source_manifest,
)


# ==================================================================================================
# README
# ==================================================================================================

readme = f"""
# Stage23-7 Final Synthesis Package

Status: **COMPLETE — UNSEALED**

Execution parent:

- commit: `{EXPECTED_PARENT}`
- tag: `{EXPECTED_PARENT_TAG}`

This directory contains the final zero-fit Stage23 scientific consolidation.

## Main frozen figures

1. `figure_23_a_subset_split_interaction`
2. `figure_23_b_removal_penalties_and_interaction`
3. `figure_23_c_single_feature_stump_degradation`
4. `figure_23_d_shap_proxy_absorption`

Each main figure is provided as both 300-dpi PNG and vector PDF.

## Supplementary figures

1. `figure_23_s1_placebo_interactions`
2. `figure_23_s2_attack_family_delta_f1`

## Tables

Publication and supplementary CSV tables are under `tables/`.

The complete component-specific SHAP top-20 export is:

`tables/table_23_s4_component_specific_shap_top20.csv`

The complete frozen attack-family metric export is:

`tables/table_23_s3_attack_family_all_frozen_metrics.csv`

## Manuscript integration

IEEE/LaTeX-ready integration fragments are under `manuscript/`:

- `stage23_results_integration.tex`
- `stage23_main_tables.tex`
- `stage23_figure_captions.tex`

## Scientific audit

The consolidated audit is:

`audit/stage23_7_consolidated_scientific_audit.json`

## Governance

- Stage23 model-fit budget remains 50/50 SEALED.
- New fits: 0.
- Inference: 0.
- New SHAP computation: 0.
- New bootstrap sampling: 0.
- Parquet reads: 0.
- Raw Mar1: not accessed.
- Raw Mar2: not accessed.

This package must be sealed in a separate zero-fit closure commit before Stage23
is considered repository-final.
"""

write_text(TMP / "README.md", readme)


# ==================================================================================================
# EXECUTION STATE
# ==================================================================================================

execution_state = {
    "stage": "Stage23-7A",
    "status": "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "execution_parent": {
        "commit": EXPECTED_PARENT,
        "tag": EXPECTED_PARENT_TAG,
    },

    "counts": {
        "primary_subsets": 7,
        "primary_split_results": 14,
        "placebo_subsets": 5,
        "placebo_split_results": 10,
        "stump_controls": 6,
        "uncertainty_comparisons": 11,
        "shap_component_models": 48,
        "shap_proxy_split_comparisons": 22,
        "attack_family_metric_rows": 182,
        "main_figure_bases": 4,
        "supplementary_figure_bases": 2,
        "figure_files": 12,
        "publication_table_csvs": 10,
        "manuscript_files": 3,
        "stopping_rule_conditions_satisfied": 7,
    },

    "governance": {
        "stage23_fit_budget_before": 50,
        "stage23_fit_budget_after": 50,
        "model_fits": 0,
        "model_inference": 0,
        "lightgbm_execution": 0,
        "xgboost_execution": 0,
        "shap_values_computed": 0,
        "bootstrap_replicates_generated": 0,
        "parquet_files_read": 0,
        "model_files_read": 0,
        "probability_npz_files_read": 0,
        "raw_mar1_accessed": False,
        "raw_mar2_accessed": False,
        "stage23_0_modified": False,
    },

    "next_authorized_action":
        "ZERO_FIT_FINAL_STAGE23_CLOSURE_SEAL_AND_PUSH",
}

write_json(
    TMP / "execution_state.json",
    execution_state,
)


# ==================================================================================================
# CHECKSUMS
# ==================================================================================================

print(SEP)
print("WRITE FINAL PACKAGE CHECKSUMS")
print(SEP)
print()

checksum_path = TMP / "checksums.sha256"

artifact_files = sorted(
    p for p in TMP.rglob("*")
    if p.is_file() and p != checksum_path
)

checksum_lines = []

for path in artifact_files:
    rel = path.relative_to(TMP)
    checksum_lines.append(
        f"{sha256_file(path)}  {rel.as_posix()}"
    )

write_text(
    checksum_path,
    "\n".join(checksum_lines),
)

# Verify checksum manifest immediately.
for line in checksum_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue

    digest, rel = line.split("  ", 1)
    p = TMP / rel

    assert p.is_file(), p
    actual = sha256_file(p)
    assert actual == digest, (rel, digest, actual)

package_manifest_sha = sha256_file(checksum_path)

print("[EXACT] package artifact entries :", len(checksum_lines))
print("[EXACT] checksum manifest SHA256 :", package_manifest_sha)
print()


# ==================================================================================================
# ATOMIC FINALIZATION OF UNSEALED WORKING PACKAGE
# ==================================================================================================

# Repo must still be completely untouched.
status_after = git("status", "--porcelain")
head_after = git("rev-parse", "HEAD")

assert status_after == "", status_after
assert head_after == EXPECTED_PARENT, head_after

TMP.rename(OUT)

assert OUT.is_dir()
assert not TMP.exists()

# Reverify checksum manifest after rename.
final_checksum_path = OUT / "checksums.sha256"

for line in final_checksum_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue

    digest, rel = line.split("  ", 1)
    p = OUT / rel

    assert p.is_file(), p
    assert sha256_file(p) == digest, rel

assert sha256_file(final_checksum_path) == package_manifest_sha


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print(SEP)
print("STAGE23-7A — FINAL SYNTHESIS / TABLES / FIGURES COMPLETE — UNSEALED")
print(SEP)
print()

print("Output:")
print(" ", OUT)
print()

print("HEAD:")
print(" ", head_after)
print()

print("Parent tag:")
print(" ", EXPECTED_PARENT_TAG)
print()

print("Package checksum manifest SHA256:")
print(" ", package_manifest_sha)
print()

print("SCIENTIFIC INPUT ACCOUNTING")
print("  primary subsets              : 7 / 7")
print("  primary split results        : 14")
print("  placebo subsets              : 5 / 5")
print("  placebo split results        : 10")
print("  stump controls               : 6 / 6")
print("  uncertainty comparisons      : 11 / 11")
print("  SHAP component models        : 48 / 48")
print("  SHAP proxy comparisons       : 22 / 22")
print("  attack-family metric rows    : 182")
print("  frozen families              : 13")
print()

print("PUBLICATION OUTPUTS")
print("  frozen main figure bases     : 4 / 4")
print("  supplementary figure bases   : 2")
print("  PNG/PDF figure files         : 12")
print("  publication/supp table CSVs  : 10")
print("  manuscript integration files : 3")
print("  stopping-rule conditions     : 7 / 7 SATISFIED")
print()

print("HEADLINE FULL RESULTS")
print(
    f"  RANDOM PR / ROC              : "
    f"{full_r['pr_auc']:.12f} / {full_r['roc_auc']:.12f}"
)
print(
    f"  CHRONO PR / ROC              : "
    f"{full_c['pr_auc']:.12f} / {full_c['roc_auc']:.12f}"
)
print(
    f"  CHRONO attack prevalence     : "
    f"{chrono_prev:.12f}"
)
print(
    f"  CHRONO PR - prevalence       : "
    f"{full_c['pr_auc_minus_attack_prevalence']:.12f}"
)
print()

print("ATTACK-FAMILY CONTEXT")
print("  RANDOM interpretable         : 11")
print("  CHRONO interpretable         : 1")
print("  CHRONO attack family         : Infilteration only")
print("  CHRONO attack support        :", f"{chrono_attack_support:,}")
print()

print("GOVERNANCE")
print("  Stage23 fit budget           : 50 / 50 SEALED")
print("  additional fits authorized   : 0")
print("  new model fits               : 0")
print("  model inference              : 0")
print("  LightGBM execution           : 0")
print("  XGBoost execution            : 0")
print("  new SHAP values              : 0")
print("  new bootstrap replicates     : 0")
print("  parquet reads                : 0")
print("  Raw Mar1 accessed            : NO")
print("  Raw Mar2 accessed            : NO")
print("  repository modified          : NO")
print()

print("STATUS:")
print("  FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED")
print()
print("NEXT AUTHORIZED ACTION:")
print("  ZERO-FIT FINAL STAGE23 CLOSURE SEAL + PUSH.")
print("  NO ADDITIONAL MODEL FITS.")
print(SEP)

STAGE23-7A — FINAL SYNTHESIS / TABLES / FIGURES / MANUSCRIPT INTEGRATION

[EXACT] branch    : main
[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag: stage23-6-attack-family-analysis-complete-v1
[EXACT] tag commit: 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] worktree  : CLEAN

VERIFY FROZEN STAGE23 PROTOCOL

[EXACT] stopping conditions : 7
[EXACT] frozen main figures : 4
[EXACT] primary metric       : PR_AUC
[EXACT] fixed threshold      : 0.50
[EXACT] Mar1/Mar2            : PERMANENTLY FORBIDDEN

LOAD SEALED PRIMARY ABLATION RESULTS

[EXACT] primary result JSONs : 12
[EXACT] primary table rows   : 14
[EXACT] primary subsets      : 7 / 7
[EXACT] splits               : 2 / 2

LOAD SEALED MATCHED-SIZE PLACEBO RESULTS



KeyError: 'data'

In [12]:
# ==================================================================================================
# STAGE23-7A-R1 — EXACT SYNTHESIS READER HOTFIX + CLEAN RERUN
#
# Fixes ONLY:
#   1. Stage23-2 heterogeneous placebo result schema:
#        some results have data.validation; later sealed results do not.
#   2. Publication table count:
#        actual generated CSV count = 11, not 10.
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM EXECUTION
# ZERO XGBOOST EXECUTION
# ZERO NEW SHAP
# ZERO NEW BOOTSTRAP
# ZERO PARQUET
# NO MAR1 / MAR2
# NO REPOSITORY MODIFICATION
#
# This cell recovers the exact previously executed Stage23-7A cell from
# IPython history, verifies the expected buggy source signatures, applies
# only the frozen synthesis-reader corrections above, compiles, and reruns.
# ==================================================================================================

from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")
OUT = Path("/kaggle/working/stage23_7_final_synthesis")

EXPECTED_PARENT = "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
EXPECTED_PARENT_TAG = "stage23-6-attack-family-analysis-complete-v1"

SEP = "=" * 116


def git(*args):
    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )
    return p.stdout.strip()


print(SEP)
print("STAGE23-7A-R1 — SYNTHESIS READER HOTFIX PREFLIGHT")
print(SEP)
print()


# --------------------------------------------------------------------------------------------------
# 1. SEALED PARENT MUST STILL BE EXACT
# --------------------------------------------------------------------------------------------------

assert REPO.is_dir(), REPO

head = git("rev-parse", "HEAD")
branch = git("branch", "--show-current")
status = git("status", "--porcelain")
tag_commit = git("rev-list", "-n", "1", EXPECTED_PARENT_TAG)

assert branch == "main", branch
assert head == EXPECTED_PARENT, (head, EXPECTED_PARENT)
assert tag_commit == EXPECTED_PARENT, (tag_commit, EXPECTED_PARENT)
assert status == "", f"Repository dirty:\n{status}"

print("[EXACT] branch    :", branch)
print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] worktree  : CLEAN")


# --------------------------------------------------------------------------------------------------
# 2. PREVIOUS FAILED RUN MUST NOT HAVE FINALIZED A PACKAGE
# --------------------------------------------------------------------------------------------------

if OUT.exists():
    raise RuntimeError(
        "Final Stage23-7A output already exists:\n"
        f"{OUT}\n"
        "Refusing automatic rerun."
    )

print("[EXACT] finalized Stage23-7A output does not exist.")
print("[OK] previous failure occurred before final package creation.")


# --------------------------------------------------------------------------------------------------
# 3. LOCATE EXACT FAILED STAGE23-7A SOURCE IN NOTEBOOK HISTORY
# --------------------------------------------------------------------------------------------------

ip = get_ipython()

if ip is None:
    raise RuntimeError(
        "IPython history unavailable; cannot perform exact-source hotfix."
    )

history = list(ip.history_manager.input_hist_raw)

candidates = []

for idx, source in enumerate(history):

    if not isinstance(source, str):
        continue

    signatures = [
        "STAGE23-7A — FINAL SCIENTIFIC SYNTHESIS + PUBLICATION TABLES + FIGURES",
        "LOAD SEALED MATCHED-SIZE PLACEBO RESULTS",
        'val = d["data"]["validation"]',
        "assert len(table_files) == 10",
        "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED",
        "figure_23_d_shap_proxy_absorption",
    ]

    if all(sig in source for sig in signatures):
        candidates.append((idx, source))


if not candidates:
    raise RuntimeError(
        "Could not locate the exact failed Stage23-7A source in "
        "the current notebook history.\n"
        "NO CODE EXECUTED."
    )


history_index, source = candidates[-1]

print("[EXACT] located failed Stage23-7A history entry:", history_index)


# --------------------------------------------------------------------------------------------------
# 4. VERIFY EXACT PATCH TARGETS
# --------------------------------------------------------------------------------------------------

old_schema_line = '    val = d["data"]["validation"]'

if source.count(old_schema_line) != 1:
    raise RuntimeError(
        "Unexpected placebo schema-reader occurrence count:\n"
        f"expected=1 actual={source.count(old_schema_line)}"
    )


old_table_assert = "assert len(table_files) == 10"

if source.count(old_table_assert) != 1:
    raise RuntimeError(
        "Unexpected table-count assertion occurrence count:\n"
        f"expected=1 actual={source.count(old_table_assert)}"
    )


old_table_count_json = '"publication_table_csvs": 10'

if source.count(old_table_count_json) != 2:
    raise RuntimeError(
        "Unexpected publication_table_csvs count occurrences:\n"
        f"expected=2 actual={source.count(old_table_count_json)}"
    )


old_table_final_print = (
    'print("  publication/supp table CSVs  : 10")'
)

if source.count(old_table_final_print) != 1:
    raise RuntimeError(
        "Unexpected final table-count print occurrence count:\n"
        f"expected=1 actual={source.count(old_table_final_print)}"
    )


print("[EXACT] schema patch target       : 1")
print("[EXACT] table assertion target    : 1")
print("[EXACT] table JSON count targets  : 2")
print("[EXACT] table final-print target  : 1")


# --------------------------------------------------------------------------------------------------
# 5. PATCH ONLY THE HETEROGENEOUS PLACEBO RESULT READER
#
# Earlier sealed Stage23-2 result:
#     d["data"]["validation"] exists.
#
# Later sealed Stage23-2 results:
#     top-level "data" is absent.
#     Split is frozen in d["cell"]["split"].
#     attack_prevalence is preserved in ranking_metrics.
#
# For the latter, validation population metadata is inherited from the already
# exact primary result validation_meta for the SAME frozen split.
# --------------------------------------------------------------------------------------------------

new_schema_block = '''    val = d.get("data", {}).get("validation")

    if val is None:

        # Later frozen Stage23-2 result schema intentionally omitted
        # the repeated validation-population block.
        #
        # Recover ONLY the already-verified population metadata from
        # the same frozen split's primary results.
        val = dict(validation_meta[split])

        if "attack_prevalence" not in ranking:
            raise RuntimeError(
                f"Placebo result lacks both data.validation and "
                f"ranking_metrics.attack_prevalence: {path}"
            )

        if float(ranking["attack_prevalence"]) != float(
            val["attack_prevalence"]
        ):
            raise RuntimeError(
                "Placebo fallback validation-prevalence mismatch:\\\\n"
                f"path={path}\\\\n"
                f"result={ranking['attack_prevalence']}\\\\n"
                f"expected={val['attack_prevalence']}"
            )
'''

patched = source.replace(
    old_schema_line,
    new_schema_block,
    1,
)


# --------------------------------------------------------------------------------------------------
# 6. PATCH ONLY THE VERIFIED OUTPUT TABLE COUNT
#
# Generated CSVs:
#   Table 23-1
#   Table 23-S0
#   Table 23-2
#   Table 23-3
#   Table 23-4
#   Table 23-5
#   Table 23-S1
#   Table 23-S2
#   Table 23-S3
#   Table 23-S4
#   Table 23-S5
#
# Total = 11.
# --------------------------------------------------------------------------------------------------

patched = patched.replace(
    old_table_assert,
    "assert len(table_files) == 11",
    1,
)

patched = patched.replace(
    old_table_count_json,
    '"publication_table_csvs": 11',
)

patched = patched.replace(
    old_table_final_print,
    'print("  publication/supp table CSVs  : 11")',
    1,
)


# --------------------------------------------------------------------------------------------------
# 7. POST-PATCH SOURCE INTEGRITY
# --------------------------------------------------------------------------------------------------

assert old_schema_line not in patched
assert "assert len(table_files) == 10" not in patched
assert '"publication_table_csvs": 10' not in patched
assert old_table_final_print not in patched

assert patched.count("assert len(table_files) == 11") == 1
assert patched.count('"publication_table_csvs": 11') == 2
assert (
    patched.count(
        'print("  publication/supp table CSVs  : 11")'
    )
    == 1
)

# Frozen scientific definitions must still be present.
required_frozen_signatures = [
    'EXPECTED_PARENT = "3c874e3d20b2f38d842423c1f2316fd60a5eea68"',
    '"NO_DST_PORT"',
    '"NO_PORTS"',
    '"NO_INIT_FWD_WIN_BYTS"',
    '"NO_FWD_SEG_SIZE_MIN"',
    '"NO_SUSPICIOUS_GROUP"',
    '"BEHAVIOR_ONLY"',
    '"PLACEBO_COUNTS"',
    '"PLACEBO_VOLUME_DIRECTION"',
    '"PLACEBO_IAT"',
    '"PLACEBO_PACKET_SIZE"',
    '"PLACEBO_ACTIVITY"',
    '"Dst Port"',
    '"Init Fwd Win Byts"',
    '"Fwd Seg Size Min"',
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
    "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED",
]

for sig in required_frozen_signatures:
    if sig not in patched:
        raise RuntimeError(
            f"Frozen Stage23-7A signature missing after patch: {sig}"
        )


# Compile BEFORE executing anything.
compile(
    patched,
    "<stage23_7a_r1_exact_hotfix>",
    "exec",
)

print()
print("[EXACT] corrected source compiled successfully.")
print()
print("PATCH ACCOUNTING")
print("  scientific definitions changed : 0")
print("  feature/subset definitions      : 0")
print("  metric definitions              : 0")
print("  uncertainty method              : 0")
print("  SHAP method                     : 0")
print("  attack-family rules             : 0")
print("  model settings                  : 0")
print("  model fits                      : 0")
print()
print("  schema-reader fixes             : 1")
print("  output-count corrections        : 4 occurrences")
print()
print("[RUNNING] corrected Stage23-7A from the beginning...")
print(SEP)
print()


# --------------------------------------------------------------------------------------------------
# 8. EXECUTE EXACT CORRECTED STAGE23-7A
#
# Original Stage23-7A startup logic will:
#   - verify sealed HEAD/tag/worktree
#   - delete only the failed temporary Stage23-7A working directory
#   - refuse if final OUT already exists
#   - rebuild all synthesis artifacts from sealed evidence
# --------------------------------------------------------------------------------------------------

exec(
    compile(
        patched,
        "<stage23_7a_r1_exact_hotfix>",
        "exec",
    ),
    globals(),
    globals(),
)

STAGE23-7A-R1 — SYNTHESIS READER HOTFIX PREFLIGHT

[EXACT] branch    : main
[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag: stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] finalized Stage23-7A output does not exist.
[OK] previous failure occurred before final package creation.
[EXACT] located failed Stage23-7A history entry: 12


RuntimeError: Unexpected table-count assertion occurrence count:
expected=1 actual=3

In [13]:
# ==================================================================================================
# STAGE23-7A-R2 — EXACT ORIGINAL-CELL RECOVERY + MINIMAL SYNTHESIS HOTFIX
#
# Fixes ONLY:
#   1. Stage23-2 heterogeneous placebo JSON schema.
#   2. Generated publication-table count: 11, not 10.
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM / XGBOOST EXECUTION
# ZERO NEW SHAP
# ZERO NEW BOOTSTRAP
# ZERO PARQUET
# NO MAR1 / MAR2
# NO REPOSITORY MODIFICATION
# ==================================================================================================

from pathlib import Path
import subprocess


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

OUT = Path(
    "/kaggle/working/stage23_7_final_synthesis"
)

TMP = Path(
    "/kaggle/working/stage23_7_final_synthesis_tmp"
)

EXPECTED_PARENT = (
    "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
)

EXPECTED_PARENT_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)

SEP = "=" * 116


def git(*args):

    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )

    return (
        p.stdout or ""
    ).strip()


print(SEP)
print("STAGE23-7A-R2 — EXACT ORIGINAL-CELL RECOVERY + MINIMAL HOTFIX")
print(SEP)
print()


# ==================================================================================================
# 1. VERIFY SEALED PARENT
# ==================================================================================================

assert REPO.is_dir(), REPO

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

status = git(
    "status",
    "--porcelain",
)


assert branch == "main", branch
assert head == EXPECTED_PARENT, (
    head,
    EXPECTED_PARENT,
)
assert tag_commit == EXPECTED_PARENT, (
    tag_commit,
    EXPECTED_PARENT,
)
assert status == "", (
    "Repository dirty before Stage23-7A-R2:\n"
    + status
)


print("[EXACT] branch    :", branch)
print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] worktree  : CLEAN")


# ==================================================================================================
# 2. FAILED RUN MUST NOT HAVE FINALIZED PACKAGE
# ==================================================================================================

if OUT.exists():

    raise RuntimeError(
        "Final Stage23-7A output already exists.\n"
        "REFUSING automatic rerun:\n"
        f"{OUT}"
    )


print(
    "[EXACT] finalized Stage23-7A output does not exist."
)


if TMP.exists():

    print(
        "[INFO] failed-run temporary directory exists and "
        "will be removed by the original Stage23-7A startup logic:"
    )

    print(
        " ",
        TMP,
    )

else:

    print(
        "[INFO] no failed-run temporary directory exists."
    )


# ==================================================================================================
# 3. FIND THE ACTUAL ORIGINAL STAGE23-7A CELL
#
# Key distinction:
#
# Original cell contains these as EXECUTABLE lines:
#
#     val = d["data"]["validation"]
#     assert len(table_files) == 10
#
# R1/R2 hotfix cells contain them only inside quoted strings.
# ==================================================================================================

ip = get_ipython()

if ip is None:

    raise RuntimeError(
        "IPython history unavailable."
    )


history = list(
    ip.history_manager.input_hist_raw
)


candidates = []


for idx, source in enumerate(
    history
):

    if not isinstance(
        source,
        str,
    ):

        continue


    lines = source.splitlines()

    stripped_lines = [
        line.strip()
        for line in lines
    ]


    actual_schema_lines = [
        i
        for i, line
        in enumerate(
            stripped_lines
        )
        if line
        == 'val = d["data"]["validation"]'
    ]


    actual_table_assert_lines = [
        i
        for i, line
        in enumerate(
            stripped_lines
        )
        if line
        == "assert len(table_files) == 10"
    ]


    has_stage_header = any(
        (
            "STAGE23-7A — FINAL SCIENTIFIC SYNTHESIS "
            "+ PUBLICATION TABLES + FIGURES"
        )
        in line

        for line
        in lines
    )


    has_final_status = any(
        "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED"
        in line

        for line
        in lines
    )


    # Reject hotfix wrappers explicitly.
    is_hotfix_wrapper = any(
        marker in source

        for marker in [
            "STAGE23-7A-R1",
            "STAGE23-7A-R2",
            "history_manager.input_hist_raw",
            "SYNTHESIS READER HOTFIX PREFLIGHT",
            "EXACT ORIGINAL-CELL RECOVERY",
        ]
    )


    if (
        has_stage_header
        and has_final_status
        and len(
            actual_schema_lines
        ) == 1
        and len(
            actual_table_assert_lines
        ) == 1
        and not is_hotfix_wrapper
    ):

        candidates.append(
            {
                "history_index":
                    idx,

                "source":
                    source,

                "schema_line":
                    actual_schema_lines[
                        0
                    ],

                "table_assert_line":
                    actual_table_assert_lines[
                        0
                    ],
            }
        )


if len(
    candidates
) != 1:

    print()
    print(
        "Original-cell candidates found:",
        len(
            candidates
        ),
    )

    for candidate in candidates:

        print(
            "  history index:",
            candidate[
                "history_index"
            ],
        )

    raise RuntimeError(
        "Could not uniquely identify the original failed Stage23-7A cell.\n"
        "NO CODE EXECUTED."
    )


candidate = candidates[
    0
]

history_index = candidate[
    "history_index"
]

source = candidate[
    "source"
]


print()
print(
    "[EXACT] original Stage23-7A history entry:",
    history_index,
)

print(
    "[EXACT] executable placebo-schema target : 1"
)

print(
    "[EXACT] executable table-count target     : 1"
)


# ==================================================================================================
# 4. LINE-LEVEL PATCHING
#
# We patch executable source lines, not raw substring counts.
# ==================================================================================================

lines = source.splitlines()


# --------------------------------------------------------------------------------------------------
# 4A. PLACEBO SCHEMA READER
# --------------------------------------------------------------------------------------------------

schema_target_indices = [
    i
    for i, line
    in enumerate(
        lines
    )
    if line.strip()
    == 'val = d["data"]["validation"]'
]


if len(
    schema_target_indices
) != 1:

    raise RuntimeError(
        "Executable placebo schema target is not unique."
    )


schema_idx = (
    schema_target_indices[
        0
    ]
)

indent = lines[
    schema_idx
][
    :len(
        lines[
            schema_idx
        ]
    )
    -
    len(
        lines[
            schema_idx
        ].lstrip()
    )
]


schema_replacement = [
    indent
    + 'val = d.get("data", {}).get("validation")',

    "",

    indent
    + "if val is None:",

    "",

    indent
    + "    # Later sealed Stage23-2 results omit the repeated",

    indent
    + "    # data.validation block. Reuse ONLY the validation",

    indent
    + "    # population metadata already verified for the same",

    indent
    + "    # frozen split from the primary Stage23 results.",

    indent
    + "    val = dict(validation_meta[split])",

    "",

    indent
    + '    if "attack_prevalence" not in ranking:',

    "",

    indent
    + "        raise RuntimeError(",

    indent
    + '            "Placebo result lacks both data.validation and "',

    indent
    + '            f"ranking_metrics.attack_prevalence: {path}"',

    indent
    + "        )",

    "",

    indent
    + "    if not np.isclose(",

    indent
    + '        float(ranking["attack_prevalence"]),',

    indent
    + '        float(val["attack_prevalence"]),',

    indent
    + "        rtol=0.0,",

    indent
    + "        atol=0.0,",

    indent
    + "    ):",

    "",

    indent
    + "        raise RuntimeError(",

    indent
    + '            "Placebo fallback validation-prevalence mismatch:\\n"',

    indent
    + '            f"path={path}\\n"',

    indent
    + '            f"result={ranking[\'attack_prevalence\']}\\n"',

    indent
    + '            f"expected={val[\'attack_prevalence\']}"',

    indent
    + "        )",
]


lines[
    schema_idx:
    schema_idx + 1
] = schema_replacement


# --------------------------------------------------------------------------------------------------
# 4B. TABLE COUNT ASSERTION
# --------------------------------------------------------------------------------------------------

table_assert_indices = [
    i
    for i, line
    in enumerate(
        lines
    )
    if line.strip()
    == "assert len(table_files) == 10"
]


if len(
    table_assert_indices
) != 1:

    raise RuntimeError(
        "Executable table-count assertion is not unique."
    )


idx = table_assert_indices[
    0
]

leading = lines[
    idx
][
    :len(
        lines[
            idx
        ]
    )
    -
    len(
        lines[
            idx
        ].lstrip()
    )
]


lines[
    idx
] = (
    leading
    + "assert len(table_files) == 11"
)


# ==================================================================================================
# 5. PATCH STATIC REPORTED COUNTS BY EXACT SOURCE LINES
# ==================================================================================================

json_count_changed = 0
print_count_changed = 0


for i, line in enumerate(
    lines
):

    stripped = line.strip()


    if stripped == '"publication_table_csvs": 10,':

        leading = line[
            :len(
                line
            )
            -
            len(
                line.lstrip()
            )
        ]

        lines[
            i
        ] = (
            leading
            + '"publication_table_csvs": 11,'
        )

        json_count_changed += 1


    elif (
        stripped
        ==
        'print("  publication/supp table CSVs  : 10")'
    ):

        leading = line[
            :len(
                line
            )
            -
            len(
                line.lstrip()
            )
        ]

        lines[
            i
        ] = (
            leading
            + 'print("  publication/supp table CSVs  : 11")'
        )

        print_count_changed += 1


if json_count_changed != 2:

    raise RuntimeError(
        "Unexpected publication_table_csvs static-count targets.\n"
        f"expected=2 actual={json_count_changed}"
    )


if print_count_changed != 1:

    raise RuntimeError(
        "Unexpected final table-count print target.\n"
        f"expected=1 actual={print_count_changed}"
    )


patched = "\n".join(
    lines
)


# Preserve terminal newline if original had one.
if source.endswith(
    "\n"
):

    patched += "\n"


# ==================================================================================================
# 6. POST-PATCH SEMANTIC INTEGRITY CHECK
# ==================================================================================================

patched_lines = [
    line.strip()
    for line in patched.splitlines()
]


if (
    'val = d["data"]["validation"]'
    in patched_lines
):

    raise RuntimeError(
        "Old executable placebo reader remains."
    )


if (
    "assert len(table_files) == 10"
    in patched_lines
):

    raise RuntimeError(
        "Old executable table-count assertion remains."
    )


if (
    "assert len(table_files) == 11"
    not in patched_lines
):

    raise RuntimeError(
        "Corrected executable table-count assertion absent."
    )


if patched_lines.count(
    '"publication_table_csvs": 11,'
) != 2:

    raise RuntimeError(
        "Corrected publication table JSON counts != 2."
    )


if patched_lines.count(
    'print("  publication/supp table CSVs  : 11")'
) != 1:

    raise RuntimeError(
        "Corrected final table-count print != 1."
    )


# Frozen scientific definitions must remain untouched.
required_frozen_signatures = [
    'EXPECTED_PARENT = "3c874e3d20b2f38d842423c1f2316fd60a5eea68"',
    '"NO_DST_PORT"',
    '"NO_PORTS"',
    '"NO_INIT_FWD_WIN_BYTS"',
    '"NO_FWD_SEG_SIZE_MIN"',
    '"NO_SUSPICIOUS_GROUP"',
    '"BEHAVIOR_ONLY"',
    '"PLACEBO_COUNTS"',
    '"PLACEBO_VOLUME_DIRECTION"',
    '"PLACEBO_IAT"',
    '"PLACEBO_PACKET_SIZE"',
    '"PLACEBO_ACTIVITY"',
    '"Dst Port"',
    '"Init Fwd Win Byts"',
    '"Fwd Seg Size Min"',
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
    "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED",
]


for signature in required_frozen_signatures:

    if signature not in patched:

        raise RuntimeError(
            "Frozen Stage23-7A definition missing after patch:\n"
            + signature
        )


# Must compile before any rerun.
compile(
    patched,
    "<stage23_7a_r2>",
    "exec",
)


print()
print("[EXACT] patched Stage23-7A compiles.")
print()
print("PATCH ACCOUNTING")
print("  scientific hypotheses changed      : 0")
print("  subset definitions changed         : 0")
print("  feature definitions changed        : 0")
print("  metric definitions changed         : 0")
print("  threshold changed                  : 0")
print("  uncertainty protocol changed       : 0")
print("  SHAP protocol changed              : 0")
print("  attack-family rules changed        : 0")
print("  model fits                         : 0")
print("  model inference                    : 0")
print()
print("  Stage23-2 schema-reader correction : 1")
print("  publication table count            : 10 -> 11")
print()
print(
    "[RUNNING] corrected original Stage23-7A..."
)
print(SEP)
print()


# ==================================================================================================
# 7. EXECUTE CORRECTED ORIGINAL CELL
#
# The original startup will safely remove only its failed TMP directory.
# It still refuses to overwrite a finalized OUT directory.
# ==================================================================================================

exec(
    compile(
        patched,
        "<stage23_7a_r2>",
        "exec",
    ),
    globals(),
    globals(),
)

STAGE23-7A-R2 — EXACT ORIGINAL-CELL RECOVERY + MINIMAL HOTFIX

[EXACT] branch    : main
[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag: stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] finalized Stage23-7A output does not exist.
[INFO] failed-run temporary directory exists and will be removed by the original Stage23-7A startup logic:
  /kaggle/working/stage23_7_final_synthesis_tmp

Original-cell candidates found: 0


RuntimeError: Could not uniquely identify the original failed Stage23-7A cell.
NO CODE EXECUTED.

In [14]:
# ==================================================================================================
# STAGE23-7A-R3 — STANDALONE FINAL SYNTHESIS / TABLES / FIGURES / MANUSCRIPT PACKAGE
#
# Reads ONLY already-sealed Stage23 JSON/CSV scientific outputs.
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO LIGHTGBM EXECUTION
# ZERO XGBOOST EXECUTION
# ZERO NEW SHAP
# ZERO NEW BOOTSTRAP
# ZERO MODEL FILE READS
# ZERO NPZ READS
# ZERO PARQUET READS
# NO MAR1 / MAR2
# NO REPOSITORY MODIFICATION
#
# This cell intentionally does NOT depend on IPython history.
# ==================================================================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import math
import shutil
import textwrap

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ==================================================================================================
# 0. CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
)

P0 = (
    ROOT
    / "stage23_0_protocol_lock"
)

P1 = (
    ROOT
    / "stage23_1_primary_ablation"
)

P2 = (
    ROOT
    / "stage23_2_placebo_ablation"
)

P3 = (
    ROOT
    / "stage23_3_stump_controls"
)

P4 = (
    ROOT
    / "stage23_4_uncertainty_analysis"
)

P5 = (
    ROOT
    / "stage23_5b_treeshap_proxy_absorption"
)

P6 = (
    ROOT
    / "stage23_6_attack_family_analysis"
    / "stage23_6d_attack_family_metrics"
)


EXPECTED_PARENT = (
    "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
)

EXPECTED_PARENT_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)


OUT = Path(
    "/kaggle/working/stage23_7_final_synthesis"
)

TMP = Path(
    "/kaggle/working/stage23_7_final_synthesis_tmp"
)


SPLITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]


PRIMARY_ORDER = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]


PRIMARY_ABLATIONS = (
    PRIMARY_ORDER[
        1:
    ]
)


PLACEBO_ORDER = [
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]


STUMP_ORDER = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


DISPLAY = {
    "FULL":
        "FULL",

    "NO_DST_PORT":
        "No Dst Port",

    "NO_PORTS":
        "No Ports",

    "NO_INIT_FWD_WIN_BYTS":
        "No Init Fwd Win",

    "NO_FWD_SEG_SIZE_MIN":
        "No Fwd Seg Size Min",

    "NO_SUSPICIOUS_GROUP":
        "No Suspicious Group",

    "BEHAVIOR_ONLY":
        "Behavior Only",

    "PLACEBO_COUNTS":
        "Placebo Counts",

    "PLACEBO_VOLUME_DIRECTION":
        "Placebo Volume/Direction",

    "PLACEBO_IAT":
        "Placebo IAT",

    "PLACEBO_PACKET_SIZE":
        "Placebo Packet Size",

    "PLACEBO_ACTIVITY":
        "Placebo Activity",

    "RANDOM_NATURAL":
        "Random-natural",

    "CHRONOLOGICAL_NATURAL":
        "Chronological-natural",
}


SEP = "=" * 118


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def git(*args):

    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )

    return (
        p.stdout or ""
    ).strip()


def sha256_file(
    path,
    chunk=1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


SOURCE_FILES = {}


def register_source(
    path,
):

    path = Path(
        path
    )

    if not path.is_file():

        raise RuntimeError(
            f"Missing sealed source:\n{path}"
        )


    rel = str(
        path.relative_to(
            REPO
        )
    )


    SOURCE_FILES[
        rel
    ] = sha256_file(
        path
    )


def read_json(
    path,
):

    register_source(
        path
    )

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def read_csv(
    path,
):

    register_source(
        path
    )

    return pd.read_csv(
        path
    )


def write_json(
    path,
    obj,
):

    Path(path).parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def write_text(
    path,
    text,
):

    Path(path).parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    Path(path).write_text(
        str(text).rstrip()
        + "\n",
        encoding="utf-8",
    )


def ci_excludes_zero(
    low,
    high,
):

    return bool(
        float(low) > 0.0
        or float(high) < 0.0
    )


def row_for(
    df,
    split,
    subset,
):

    g = df[
        (
            df[
                "split"
            ]
            == split
        )
        &
        (
            df[
                "subset"
            ]
            == subset
        )
    ]


    if len(
        g
    ) != 1:

        raise RuntimeError(
            f"Row not unique: {split}/{subset}; n={len(g)}"
        )


    return g.iloc[
        0
    ]


def save_figure(
    fig,
    stem,
):

    stem.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    fig.savefig(
        stem.with_suffix(
            ".png"
        ),
        dpi=300,
        bbox_inches="tight",
    )


    fig.savefig(
        stem.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )


    plt.close(
        fig
    )


def latex_escape(
    value,
):

    s = str(
        value
    )

    mapping = [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
    ]


    for a, b in mapping:

        s = s.replace(
            a,
            b,
        )


    return s


# ==================================================================================================
# 2. SEALED-PARENT PREFLIGHT
# ==================================================================================================

print(SEP)
print("STAGE23-7A-R3 — STANDALONE FINAL SYNTHESIS PACKAGE")
print(SEP)
print()


if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage23 parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-6 seal tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-7A-R3:\n"
        + status
    )


if OUT.exists():

    raise RuntimeError(
        "Final Stage23-7A package already exists.\n"
        "REFUSING overwrite:\n"
        f"{OUT}"
    )


if TMP.exists():

    shutil.rmtree(
        TMP
    )


TMP.mkdir(
    parents=True,
    exist_ok=False,
)


TABLES = (
    TMP
    / "tables"
)

FIGURES = (
    TMP
    / "figures"
)

MANUSCRIPT = (
    TMP
    / "manuscript"
)

AUDIT = (
    TMP
    / "audit"
)


for p in [
    TABLES,
    FIGURES,
    MANUSCRIPT,
    AUDIT,
]:

    p.mkdir(
        parents=True,
        exist_ok=False,
    )


print("[EXACT] branch    :", branch)
print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] worktree  : CLEAN")
print("[EXACT] Stage23 fit budget: 50 / 50 SEALED")
print()


# ==================================================================================================
# 3. FROZEN PROTOCOL
# ==================================================================================================

print(SEP)
print("VERIFY FROZEN PROTOCOL")
print(SEP)
print()


stopping_rule = read_json(
    P0
    / "stopping_rule.json"
)

figure_plan = read_json(
    P0
    / "figure_plan.json"
)

interpretation_matrix = read_json(
    P0
    / "interpretation_matrix.json"
)

metric_spec = read_json(
    P0
    / "metric_spec.json"
)


if len(
    stopping_rule[
        "complete_when"
    ]
) != 7:

    raise RuntimeError(
        "Frozen stopping-rule condition count != 7."
    )


if (
    stopping_rule[
        "final_holdout"
    ]
    !=
    "Raw Mar1 and Mar2 permanently forbidden."
):

    raise RuntimeError(
        "Frozen final-holdout rule mismatch."
    )


if (
    stopping_rule[
        "rolling_forward_analysis"
    ]
    !=
    "NOT_INCLUDED_IN_FINAL_STAGE23_PROTOCOL"
):

    raise RuntimeError(
        "Rolling-forward protocol changed."
    )


if set(
    figure_plan.keys()
) != {
    "Figure_23_A",
    "Figure_23_B",
    "Figure_23_C",
    "Figure_23_D",
    "supplementary",
}:

    raise RuntimeError(
        "Frozen figure plan mismatch."
    )


if (
    metric_spec[
        "primary_ranking_metric"
    ]
    != "PR_AUC"
):

    raise RuntimeError(
        "Primary metric mismatch."
    )


if float(
    metric_spec[
        "secondary_fixed_operating_point"
    ][
        "threshold"
    ]
) != 0.5:

    raise RuntimeError(
        "Frozen operating threshold mismatch."
    )


print("[EXACT] stopping conditions : 7")
print("[EXACT] main frozen figures : 4")
print("[EXACT] primary ranking      : PR_AUC")
print("[EXACT] fixed threshold      : 0.50")
print("[EXACT] Mar1 / Mar2          : PERMANENTLY FORBIDDEN")
print()


# ==================================================================================================
# 4. PRIMARY RESULTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED PRIMARY RESULTS")
print(SEP)
print()


primary_paths = sorted(
    P1.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    primary_paths
) != 12:

    raise RuntimeError(
        f"Expected 12 primary result JSONs; found {len(primary_paths)}"
    )


primary_rows = []

full_refs = {
    split: []
    for split
    in SPLITS
}

validation_meta = {}


for path in primary_paths:

    d = read_json(
        path
    )


    split = d[
        "cell"
    ][
        "split"
    ]

    subset = d[
        "cell"
    ][
        "subset"
    ]


    if split not in SPLITS:

        raise RuntimeError(
            f"Unexpected primary split: {split}"
        )


    if subset not in PRIMARY_ABLATIONS:

        raise RuntimeError(
            f"Unexpected primary subset: {subset}"
        )


    ranking = d[
        "ranking_metrics"
    ]

    fixed = d[
        "fixed_threshold_0_50"
    ]

    val = d[
        "data"
    ][
        "validation"
    ]


    meta = {
        "rows":
            int(
                val[
                    "rows"
                ]
            ),

        "attack":
            int(
                val[
                    "attack"
                ]
            ),

        "benign":
            int(
                val[
                    "benign"
                ]
            ),

        "attack_prevalence":
            float(
                val[
                    "attack_prevalence"
                ]
            ),
    }


    if split in validation_meta:

        if validation_meta[
            split
        ] != meta:

            raise RuntimeError(
                f"Primary validation metadata changed within {split}."
            )

    else:

        validation_meta[
            split
        ] = meta


    primary_rows.append(
        {
            "split":
                split,

            "subset":
                subset,

            "family":
                "PRIMARY",

            "feature_count":
                int(
                    d[
                        "cell"
                    ][
                        "feature_count"
                    ]
                ),

            "validation_rows":
                meta[
                    "rows"
                ],

            "support_attack":
                meta[
                    "attack"
                ],

            "support_benign":
                meta[
                    "benign"
                ],

            "attack_prevalence":
                meta[
                    "attack_prevalence"
                ],

            "pr_auc":
                float(
                    ranking[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                float(
                    ranking[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "roc_auc":
                float(
                    ranking[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    fixed[
                        "f1"
                    ]
                ),

            "recall_at_0_50":
                float(
                    fixed[
                        "recall"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    fixed[
                        "fpr"
                    ]
                ),

            "fnr_at_0_50":
                float(
                    fixed[
                        "fnr"
                    ]
                ),
        }
    )


    full_refs[
        split
    ].append(
        d[
            "full_reference"
        ]
    )


for split in SPLITS:

    if len(
        full_refs[
            split
        ]
    ) != 6:

        raise RuntimeError(
            f"{split}: expected 6 FULL references."
        )


    canonical = full_refs[
        split
    ][
        0
    ]


    for other in full_refs[
        split
    ][
        1:
    ]:

        for key in [
            "pr_auc",
            "roc_auc",
            "f1_at_0_50",
            "recall_at_0_50",
            "fpr_at_0_50",
        ]:

            if float(
                other[
                    key
                ]
            ) != float(
                canonical[
                    key
                ]
            ):

                raise RuntimeError(
                    f"{split}: inconsistent FULL reference {key}."
                )


    prev = validation_meta[
        split
    ][
        "attack_prevalence"
    ]


    primary_rows.append(
        {
            "split":
                split,

            "subset":
                "FULL",

            "family":
                "FULL",

            "feature_count":
                70,

            "validation_rows":
                validation_meta[
                    split
                ][
                    "rows"
                ],

            "support_attack":
                validation_meta[
                    split
                ][
                    "attack"
                ],

            "support_benign":
                validation_meta[
                    split
                ][
                    "benign"
                ],

            "attack_prevalence":
                prev,

            "pr_auc":
                float(
                    canonical[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                (
                    float(
                        canonical[
                            "pr_auc"
                        ]
                    )
                    -
                    prev
                ),

            "roc_auc":
                float(
                    canonical[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    canonical[
                        "f1_at_0_50"
                    ]
                ),

            "recall_at_0_50":
                float(
                    canonical[
                        "recall_at_0_50"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    canonical[
                        "fpr_at_0_50"
                    ]
                ),

            "fnr_at_0_50":
                (
                    1.0
                    -
                    float(
                        canonical[
                            "recall_at_0_50"
                        ]
                    )
                ),
        }
    )


primary_df = pd.DataFrame(
    primary_rows
)


if len(
    primary_df
) != 14:

    raise RuntimeError(
        "Primary synthesis row count != 14."
    )


for split in SPLITS:

    found = set(
        primary_df.loc[
            primary_df[
                "split"
            ]
            == split,
            "subset",
        ]
    )


    if found != set(
        PRIMARY_ORDER
    ):

        raise RuntimeError(
            f"{split}: incomplete primary subset set."
        )


split_order = {
    x: i
    for i, x
    in enumerate(
        SPLITS
    )
}

primary_order = {
    x: i
    for i, x
    in enumerate(
        PRIMARY_ORDER
    )
}


primary_df[
    "_split_order"
] = primary_df[
    "split"
].map(
    split_order
)

primary_df[
    "_subset_order"
] = primary_df[
    "subset"
].map(
    primary_order
)


primary_df = (
    primary_df
    .sort_values(
        [
            "_split_order",
            "_subset_order",
        ]
    )
    .drop(
        columns=[
            "_split_order",
            "_subset_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] primary result JSONs : 12")
print("[EXACT] primary subset rows   : 14")
print("[EXACT] primary subsets       : 7 / 7")
print("[EXACT] splits                : 2 / 2")
print()


# ==================================================================================================
# 5. PLACEBO RESULTS — ROBUST TO BOTH SEALED STAGE23-2 JSON SCHEMAS
#
# We DO NOT REQUIRE data.validation.
# Population metadata comes from the already verified frozen primary split.
# ==================================================================================================

print(SEP)
print("LOAD SEALED PLACEBO RESULTS")
print(SEP)
print()


placebo_paths = sorted(
    P2.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    placebo_paths
) != 10:

    raise RuntimeError(
        f"Expected 10 placebo result JSONs; found {len(placebo_paths)}"
    )


placebo_rows = []


for path in placebo_paths:

    d = read_json(
        path
    )


    split = d[
        "cell"
    ][
        "split"
    ]

    placebo = d[
        "cell"
    ][
        "placebo"
    ]


    if split not in SPLITS:

        raise RuntimeError(
            f"Unexpected placebo split: {split}"
        )


    if placebo not in PLACEBO_ORDER:

        raise RuntimeError(
            f"Unexpected placebo subset: {placebo}"
        )


    ranking = d[
        "ranking_metrics"
    ]

    fixed = d[
        "fixed_threshold_0_50"
    ]


    val = validation_meta[
        split
    ]


    if "attack_prevalence" not in ranking:

        raise RuntimeError(
            f"Placebo ranking result lacks attack_prevalence:\n{path}"
        )


    if not np.isclose(
        float(
            ranking[
                "attack_prevalence"
            ]
        ),
        float(
            val[
                "attack_prevalence"
            ]
        ),
        rtol=0.0,
        atol=1e-15,
    ):

        raise RuntimeError(
            "Placebo prevalence disagrees with frozen split population:\n"
            f"{path}"
        )


    # If this historical result happens to repeat data.validation,
    # verify it too, but do not require it.
    optional_val = (
        d.get(
            "data",
            {}
        ).get(
            "validation"
        )
    )


    if optional_val is not None:

        for key in [
            "rows",
            "attack",
            "benign",
        ]:

            if int(
                optional_val[
                    key
                ]
            ) != int(
                val[
                    key
                ]
            ):

                raise RuntimeError(
                    f"Optional Stage23-2 validation metadata mismatch: "
                    f"{placebo}/{split}/{key}"
                )


        if not np.isclose(
            float(
                optional_val[
                    "attack_prevalence"
                ]
            ),
            float(
                val[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"Optional Stage23-2 prevalence mismatch: "
                f"{placebo}/{split}"
            )


    placebo_rows.append(
        {
            "split":
                split,

            "subset":
                placebo,

            "family":
                "PLACEBO",

            "feature_count":
                int(
                    d[
                        "cell"
                    ][
                        "feature_count"
                    ]
                ),

            "validation_rows":
                val[
                    "rows"
                ],

            "support_attack":
                val[
                    "attack"
                ],

            "support_benign":
                val[
                    "benign"
                ],

            "attack_prevalence":
                val[
                    "attack_prevalence"
                ],

            "pr_auc":
                float(
                    ranking[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                float(
                    ranking[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "roc_auc":
                float(
                    ranking[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    fixed[
                        "f1"
                    ]
                ),

            "recall_at_0_50":
                float(
                    fixed[
                        "recall"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    fixed[
                        "fpr"
                    ]
                ),

            "fnr_at_0_50":
                float(
                    fixed[
                        "fnr"
                    ]
                ),
        }
    )


placebo_df = pd.DataFrame(
    placebo_rows
)


if len(
    placebo_df
) != 10:

    raise RuntimeError(
        "Placebo synthesis row count != 10."
    )


for split in SPLITS:

    if set(
        placebo_df.loc[
            placebo_df[
                "split"
            ]
            == split,
            "subset",
        ]
    ) != set(
        PLACEBO_ORDER
    ):

        raise RuntimeError(
            f"{split}: incomplete placebo set."
        )


placebo_order = {
    x: i
    for i, x
    in enumerate(
        PLACEBO_ORDER
    )
}


placebo_df[
    "_split_order"
] = placebo_df[
    "split"
].map(
    split_order
)

placebo_df[
    "_subset_order"
] = placebo_df[
    "subset"
].map(
    placebo_order
)


placebo_df = (
    placebo_df
    .sort_values(
        [
            "_split_order",
            "_subset_order",
        ]
    )
    .drop(
        columns=[
            "_split_order",
            "_subset_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] placebo result JSONs : 10")
print("[EXACT] placebo subset rows   : 10")
print("[EXACT] placebo subsets       : 5 / 5")
print("[EXACT] heterogeneous Stage23-2 schemas resolved without adaptation")
print()


# ==================================================================================================
# 6. UNCERTAINTY SUMMARY — NO NEW BOOTSTRAP
# ==================================================================================================

print(SEP)
print("LOAD SEALED UNCERTAINTY SUMMARY")
print(SEP)
print()


unc = read_json(
    P4
    / "stage23_4_uncertainty_summary.json"
)


expected_comparisons = set(
    PRIMARY_ABLATIONS
    + PLACEBO_ORDER
)


if set(
    unc[
        "comparisons"
    ].keys()
) != expected_comparisons:

    raise RuntimeError(
        "Frozen uncertainty comparison set mismatch."
    )


unc_rows = []


for comparison in (
    PRIMARY_ABLATIONS
    + PLACEBO_ORDER
):

    c = unc[
        "comparisons"
    ][
        comparison
    ]

    pts = c[
        "complete_validation_point_estimates"
    ]

    ci = c[
        "bootstrap_95_percentile_ci"
    ]


    row = {
        "comparison":
            comparison,

        "family":
            c[
                "family"
            ],

        "random_pr_auc_removal_penalty":
            float(
                pts[
                    "random"
                ][
                    "pr_auc_removal_penalty"
                ]
            ),

        "random_pr_auc_ci_low":
            float(
                ci[
                    "random_pr_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "random_pr_auc_ci_high":
            float(
                ci[
                    "random_pr_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "chronological_pr_auc_removal_penalty":
            float(
                pts[
                    "chronological"
                ][
                    "pr_auc_removal_penalty"
                ]
            ),

        "chronological_pr_auc_ci_low":
            float(
                ci[
                    "chronological_pr_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "chronological_pr_auc_ci_high":
            float(
                ci[
                    "chronological_pr_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "pr_auc_interaction":
            float(
                pts[
                    "interaction"
                ][
                    "pr_auc"
                ]
            ),

        "pr_auc_interaction_ci_low":
            float(
                ci[
                    "pr_auc_shortcut_interaction"
                ][
                    "lower_2_5"
                ]
            ),

        "pr_auc_interaction_ci_high":
            float(
                ci[
                    "pr_auc_shortcut_interaction"
                ][
                    "upper_97_5"
                ]
            ),

        "random_roc_auc_removal_penalty":
            float(
                pts[
                    "random"
                ][
                    "roc_auc_removal_penalty"
                ]
            ),

        "random_roc_auc_ci_low":
            float(
                ci[
                    "random_roc_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "random_roc_auc_ci_high":
            float(
                ci[
                    "random_roc_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "chronological_roc_auc_removal_penalty":
            float(
                pts[
                    "chronological"
                ][
                    "roc_auc_removal_penalty"
                ]
            ),

        "chronological_roc_auc_ci_low":
            float(
                ci[
                    "chronological_roc_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "chronological_roc_auc_ci_high":
            float(
                ci[
                    "chronological_roc_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "roc_auc_interaction":
            float(
                pts[
                    "interaction"
                ][
                    "roc_auc"
                ]
            ),

        "roc_auc_interaction_ci_low":
            float(
                ci[
                    "roc_auc_shortcut_interaction"
                ][
                    "lower_2_5"
                ]
            ),

        "roc_auc_interaction_ci_high":
            float(
                ci[
                    "roc_auc_shortcut_interaction"
                ][
                    "upper_97_5"
                ]
            ),
    }


    row[
        "pr_auc_interaction_ci_excludes_zero"
    ] = ci_excludes_zero(
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
    )


    row[
        "roc_auc_interaction_ci_excludes_zero"
    ] = ci_excludes_zero(
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
    )


    unc_rows.append(
        row
    )


unc_df = pd.DataFrame(
    unc_rows
)


if len(
    unc_df
) != 11:

    raise RuntimeError(
        "Uncertainty summary comparison count != 11."
    )


unc_primary = unc_df[
    unc_df[
        "family"
    ]
    == "PRIMARY"
].copy()


unc_placebo = unc_df[
    unc_df[
        "family"
    ]
    == "PLACEBO"
].copy()


if len(
    unc_primary
) != 6:

    raise RuntimeError(
        "Primary uncertainty count != 6."
    )


if len(
    unc_placebo
) != 5:

    raise RuntimeError(
        "Placebo uncertainty count != 5."
    )


unc_primary[
    "_order"
] = unc_primary[
    "comparison"
].map(
    {
        x: i
        for i, x
        in enumerate(
            PRIMARY_ABLATIONS
        )
    }
)


unc_primary = (
    unc_primary
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


unc_placebo[
    "_order"
] = unc_placebo[
    "comparison"
].map(
    placebo_order
)


unc_placebo = (
    unc_placebo
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] uncertainty comparisons : 11 / 11")
print("[EXACT] primary                : 6")
print("[EXACT] placebo                : 5")
print("[EXACT] new bootstrap draws    : 0")
print()


# ==================================================================================================
# 7. STUMP CONTROLS
# ==================================================================================================

print(SEP)
print("LOAD SEALED STUMP CONTROLS")
print(SEP)
print()


stump = read_json(
    P3
    / "stage23_3_stump_controls_summary.json"
)


if int(
    stump[
        "fit_accounting"
    ][
        "stage23_after"
    ]
) != 50:

    raise RuntimeError(
        "Stage23 stump summary does not end at fit 50."
    )


if int(
    stump[
        "fit_accounting"
    ][
        "stump_fits_completed"
    ]
) != 6:

    raise RuntimeError(
        "Stump completion count != 6."
    )


if set(
    stump[
        "paired_random_to_chronological"
    ].keys()
) != set(
    STUMP_ORDER
):

    raise RuntimeError(
        "Frozen stump feature set mismatch."
    )


stump_rows = []


for feature in STUMP_ORDER:

    x = stump[
        "paired_random_to_chronological"
    ][
        feature
    ]


    rm = x[
        "random_metrics"
    ]

    cm = x[
        "chronological_metrics"
    ]

    dg = x[
        "random_to_chronological_degradation"
    ]

    rt = x[
        "random_tree_structure"
    ]

    ct = x[
        "chronological_tree_structure"
    ]


    stump_rows.append(
        {
            "feature":
                feature,

            "random_pr_auc":
                float(
                    rm[
                        "PR_AUC"
                    ]
                ),

            "chronological_pr_auc":
                float(
                    cm[
                        "PR_AUC"
                    ]
                ),

            "random_minus_chronological_pr_auc":
                float(
                    dg[
                        "PR_AUC"
                    ]
                ),

            "random_roc_auc":
                float(
                    rm[
                        "ROC_AUC"
                    ]
                ),

            "chronological_roc_auc":
                float(
                    cm[
                        "ROC_AUC"
                    ]
                ),

            "random_minus_chronological_roc_auc":
                float(
                    dg[
                        "ROC_AUC"
                    ]
                ),

            "random_f1_at_0_50":
                float(
                    rm[
                        "f1"
                    ]
                ),

            "chronological_f1_at_0_50":
                float(
                    cm[
                        "f1"
                    ]
                ),

            "random_minus_chronological_f1":
                float(
                    dg[
                        "f1"
                    ]
                ),

            "random_recall_at_0_50":
                float(
                    rm[
                        "recall"
                    ]
                ),

            "chronological_recall_at_0_50":
                float(
                    cm[
                        "recall"
                    ]
                ),

            "random_fpr_at_0_50":
                float(
                    rm[
                        "fpr"
                    ]
                ),

            "chronological_fpr_at_0_50":
                float(
                    cm[
                        "fpr"
                    ]
                ),

            "random_training_median":
                float(
                    rt[
                        "missing_imputation_median"
                    ]
                ),

            "chronological_training_median":
                float(
                    ct[
                        "missing_imputation_median"
                    ]
                ),

            "random_split_threshold":
                float(
                    rt[
                        "split_threshold"
                    ]
                ),

            "chronological_split_threshold":
                float(
                    ct[
                        "split_threshold"
                    ]
                ),

            "random_attack_branch":
                rt[
                    "branch_predicting_attack"
                ],

            "chronological_attack_branch":
                ct[
                    "branch_predicting_attack"
                ],
        }
    )


stump_df = pd.DataFrame(
    stump_rows
)


print("[EXACT] stump fits complete : 6 / 6")
print("[EXACT] stump pairs         : 3 / 3")
print("[EXACT] Stage23 fit budget  : 50 / 50 SEALED")
print()


# ==================================================================================================
# 8. SHAP — SEALED DERIVED CSVs ONLY
# ==================================================================================================

print(SEP)
print("LOAD SEALED SHAP PROXY-ABSORPTION OUTPUTS")
print(SEP)
print()


shap_summary = read_csv(
    P5
    / "stage23_5b_proxy_absorption_summary.csv"
)

consensus = read_csv(
    P5
    / "stage23_5b_descriptive_consensus_importance.csv"
)

component = read_csv(
    P5
    / "stage23_5b_component_importance.csv"
)


required_shap_summary_cols = {
    "split",
    "family",
    "subset",
    "reporting",
    "jaccard_at_10_vs_full",
    "jaccard_at_20_vs_full",
    "spearman_common_retained",
    "new_top10_entrants",
    "features_leaving_top10",
    "number_positive_share_gainers",
}


if not required_shap_summary_cols.issubset(
    set(
        shap_summary.columns
    )
):

    raise RuntimeError(
        "Stage23-5 proxy summary schema mismatch."
    )


if len(
    shap_summary
) != 66:

    raise RuntimeError(
        "Stage23-5 proxy reporting rows != 66."
    )


if (
    shap_summary[
        [
            "split",
            "subset",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 22
):

    raise RuntimeError(
        "Stage23-5 proxy split/subset comparisons != 22."
    )


if (
    component[
        [
            "split",
            "subset",
            "component",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 48
):

    raise RuntimeError(
        "Stage23-5 component model count != 48."
    )


if (
    consensus[
        [
            "split",
            "subset",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 24
):

    raise RuntimeError(
        "Stage23-5 consensus set count != 24."
    )


shap_primary = shap_summary[
    (
        shap_summary[
            "family"
        ]
        == "PRIMARY"
    )
    &
    (
        shap_summary[
            "reporting"
        ]
        ==
        "DESCRIPTIVE_CONSENSUS_0.5_0.5"
    )
].copy()


if len(
    shap_primary
) != 12:

    raise RuntimeError(
        "Primary consensus proxy rows != 12."
    )


gainer_rows = []


for split in SPLITS:

    full = consensus[
        (
            consensus[
                "split"
            ]
            == split
        )
        &
        (
            consensus[
                "subset"
            ]
            == "FULL"
        )
    ][
        [
            "feature",
            "consensus_normalized_importance_share",
        ]
    ].copy()


    if len(
        full
    ) != 70:

        raise RuntimeError(
            f"{split}: FULL consensus feature count != 70."
        )


    full = full.rename(
        columns={
            "consensus_normalized_importance_share":
                "full_share"
        }
    )


    for subset in PRIMARY_ABLATIONS:

        sub = consensus[
            (
                consensus[
                    "split"
                ]
                == split
            )
            &
            (
                consensus[
                    "subset"
                ]
                == subset
            )
        ][
            [
                "feature",
                "consensus_normalized_importance_share",
            ]
        ].copy()


        merged = sub.merge(
            full,
            on="feature",
            how="inner",
            validate="one_to_one",
        )


        merged[
            "share_change"
        ] = (
            merged[
                "consensus_normalized_importance_share"
            ]
            -
            merged[
                "full_share"
            ]
        )


        merged = merged.sort_values(
            [
                "share_change",
                "feature",
            ],
            ascending=[
                False,
                True,
            ],
        )


        top = merged.iloc[
            0
        ]


        gainer_rows.append(
            {
                "split":
                    split,

                "subset":
                    subset,

                "top_retained_consensus_share_gainer":
                    str(
                        top[
                            "feature"
                        ]
                    ),

                "top_retained_consensus_share_gain":
                    float(
                        top[
                            "share_change"
                        ]
                    ),
            }
        )


gainer_df = pd.DataFrame(
    gainer_rows
)


shap_primary = shap_primary.merge(
    gainer_df,
    on=[
        "split",
        "subset",
    ],
    how="left",
    validate="one_to_one",
)


component_top20 = component[
    component[
        "rank"
    ]
    <= 20
].copy()


if len(
    component_top20
) != 960:

    raise RuntimeError(
        f"Expected 960 component top-20 rows; found {len(component_top20)}"
    )


print("[EXACT] proxy comparisons          : 22 / 22")
print("[EXACT] proxy reporting rows       : 66")
print("[EXACT] component TreeSHAP models  : 48 / 48")
print("[EXACT] consensus sets             : 24")
print("[EXACT] component top-20 rows      : 960")
print("[EXACT] new SHAP computation       : 0")
print()


# ==================================================================================================
# 9. ATTACK-FAMILY RESULTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED ATTACK-FAMILY RESULTS")
print(SEP)
print()


attack_df = read_csv(
    P6
    / "stage23_6d_attack_family_metrics.csv"
)


if len(
    attack_df
) != 182:

    raise RuntimeError(
        "Attack-family metric rows != 182."
    )


if attack_df[
    "attack_family"
].nunique() != 13:

    raise RuntimeError(
        "Frozen attack-family universe != 13."
    )


if set(
    attack_df[
        "subset"
    ]
) != set(
    PRIMARY_ORDER
):

    raise RuntimeError(
        "Attack-family subset set mismatch."
    )


full_attack = attack_df[
    attack_df[
        "subset"
    ]
    == "FULL"
].copy()


random_full_attack = full_attack[
    full_attack[
        "split"
    ]
    == "RANDOM_NATURAL"
].copy()


chrono_full_attack = full_attack[
    full_attack[
        "split"
    ]
    == "CHRONOLOGICAL_NATURAL"
].copy()


random_interp = random_full_attack[
    random_full_attack[
        "interpretation_status"
    ]
    == "INTERPRETABLE"
]


chrono_interp = chrono_full_attack[
    chrono_full_attack[
        "interpretation_status"
    ]
    == "INTERPRETABLE"
]


if len(
    random_interp
) != 11:

    raise RuntimeError(
        "RANDOM interpretable family count != 11."
    )


if len(
    chrono_interp
) != 1:

    raise RuntimeError(
        "CHRONO interpretable family count != 1."
    )


chrono_positive = chrono_full_attack[
    chrono_full_attack[
        "support_attack"
    ]
    > 0
]


if len(
    chrono_positive
) != 1:

    raise RuntimeError(
        "CHRONO has more than one positive attack family."
    )


if (
    chrono_positive.iloc[
        0
    ][
        "attack_family"
    ]
    != "Infilteration"
):

    raise RuntimeError(
        "Unexpected CHRONO attack family."
    )


print("[EXACT] metric rows              : 182")
print("[EXACT] frozen families          : 13")
print("[EXACT] RANDOM interpretable     : 11")
print("[EXACT] CHRONO interpretable     : 1")
print("[EXACT] CHRONO positive family   : Infilteration only")
print()


# ==================================================================================================
# 10. PUBLICATION TABLES
# ==================================================================================================

print(SEP)
print("GENERATE PUBLICATION TABLES")
print(SEP)
print()


# --------------------------------------------------------------------------------------------------
# TABLE 23-1
# --------------------------------------------------------------------------------------------------

wide_rows = []


for subset in PRIMARY_ORDER:

    rr = row_for(
        primary_df,
        "RANDOM_NATURAL",
        subset,
    )

    cc = row_for(
        primary_df,
        "CHRONOLOGICAL_NATURAL",
        subset,
    )


    wide_rows.append(
        {
            "subset":
                subset,

            "feature_count":
                int(
                    rr[
                        "feature_count"
                    ]
                ),

            "random_pr_auc":
                float(
                    rr[
                        "pr_auc"
                    ]
                ),

            "chronological_pr_auc":
                float(
                    cc[
                        "pr_auc"
                    ]
                ),

            "random_pr_auc_minus_prevalence":
                float(
                    rr[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "chronological_pr_auc_minus_prevalence":
                float(
                    cc[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "random_roc_auc":
                float(
                    rr[
                        "roc_auc"
                    ]
                ),

            "chronological_roc_auc":
                float(
                    cc[
                        "roc_auc"
                    ]
                ),

            "random_f1_at_0_50":
                float(
                    rr[
                        "f1_at_0_50"
                    ]
                ),

            "chronological_f1_at_0_50":
                float(
                    cc[
                        "f1_at_0_50"
                    ]
                ),

            "random_recall_at_0_50":
                float(
                    rr[
                        "recall_at_0_50"
                    ]
                ),

            "chronological_recall_at_0_50":
                float(
                    cc[
                        "recall_at_0_50"
                    ]
                ),

            "random_fpr_at_0_50":
                float(
                    rr[
                        "fpr_at_0_50"
                    ]
                ),

            "chronological_fpr_at_0_50":
                float(
                    cc[
                        "fpr_at_0_50"
                    ]
                ),
        }
    )


primary_wide = pd.DataFrame(
    wide_rows
)


primary_wide.to_csv(
    TABLES
    / "table_23_1_primary_subset_performance.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-2
# --------------------------------------------------------------------------------------------------

unc_primary.to_csv(
    TABLES
    / "table_23_2_primary_removal_penalties_and_interactions.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-3
# --------------------------------------------------------------------------------------------------

stump_df.to_csv(
    TABLES
    / "table_23_3_single_feature_stump_controls.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-4
# --------------------------------------------------------------------------------------------------

shap_table = shap_primary[
    [
        "split",
        "subset",
        "jaccard_at_10_vs_full",
        "jaccard_at_20_vs_full",
        "spearman_common_retained",
        "new_top10_entrants",
        "features_leaving_top10",
        "number_positive_share_gainers",
        "top_retained_consensus_share_gainer",
        "top_retained_consensus_share_gain",
    ]
].copy()


shap_table[
    "_split_order"
] = shap_table[
    "split"
].map(
    split_order
)

shap_table[
    "_subset_order"
] = shap_table[
    "subset"
].map(
    {
        x: i
        for i, x
        in enumerate(
            PRIMARY_ABLATIONS
        )
    }
)


shap_table = (
    shap_table
    .sort_values(
        [
            "_split_order",
            "_subset_order",
        ]
    )
    .drop(
        columns=[
            "_split_order",
            "_subset_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


shap_table.to_csv(
    TABLES
    / "table_23_4_shap_proxy_absorption_consensus.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# TABLE 23-5
# --------------------------------------------------------------------------------------------------

attack_support = full_attack[
    [
        "split",
        "attack_family",
        "interpretation_status",
        "support_attack",
        "support_benign",
        "PR_AUC",
        "ROC_AUC",
        "precision_at_0_50",
        "recall_at_0_50",
        "f1_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()


attack_support.to_csv(
    TABLES
    / "table_23_5_attack_family_support_and_full_metrics.csv",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# SUPPLEMENTARY TABLES
# --------------------------------------------------------------------------------------------------

placebo_df.to_csv(
    TABLES
    / "table_23_s1_placebo_subset_performance.csv",
    index=False,
)


unc_placebo.to_csv(
    TABLES
    / "table_23_s2_placebo_interactions.csv",
    index=False,
)


attack_df.to_csv(
    TABLES
    / "table_23_s3_attack_family_all_frozen_metrics.csv",
    index=False,
)


component_top20.sort_values(
    [
        "split",
        "subset",
        "component",
        "rank",
    ]
).to_csv(
    TABLES
    / "table_23_s4_component_specific_shap_top20.csv",
    index=False,
)


behavior_table = primary_df[
    primary_df[
        "subset"
    ].isin(
        [
            "FULL",
            "BEHAVIOR_ONLY",
        ]
    )
][
    [
        "split",
        "subset",
        "attack_prevalence",
        "pr_auc",
        "pr_auc_minus_attack_prevalence",
        "roc_auc",
        "f1_at_0_50",
        "recall_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()


behavior_table.to_csv(
    TABLES
    / "table_23_s5_behavior_restricted_operating_metrics.csv",
    index=False,
)


EXPECTED_TABLES = {
    "table_23_1_primary_subset_performance.csv",
    "table_23_2_primary_removal_penalties_and_interactions.csv",
    "table_23_3_single_feature_stump_controls.csv",
    "table_23_4_shap_proxy_absorption_consensus.csv",
    "table_23_5_attack_family_support_and_full_metrics.csv",
    "table_23_s1_placebo_subset_performance.csv",
    "table_23_s2_placebo_interactions.csv",
    "table_23_s3_attack_family_all_frozen_metrics.csv",
    "table_23_s4_component_specific_shap_top20.csv",
    "table_23_s5_behavior_restricted_operating_metrics.csv",
}


actual_tables = {
    p.name
    for p in TABLES.glob(
        "*.csv"
    )
}


if actual_tables != EXPECTED_TABLES:

    raise RuntimeError(
        "Publication table set mismatch.\n"
        f"missing={sorted(EXPECTED_TABLES - actual_tables)}\n"
        f"extra={sorted(actual_tables - EXPECTED_TABLES)}"
    )


print("[EXACT] publication/supplementary tables: 10 / 10")
print()


# ==================================================================================================
# 11. FIGURE STYLE
# ==================================================================================================

plt.rcParams.update(
    {
        "font.size":
            8.5,

        "axes.titlesize":
            9.5,

        "axes.labelsize":
            8.5,

        "xtick.labelsize":
            7.3,

        "ytick.labelsize":
            7.3,

        "legend.fontsize":
            7.3,

        "figure.titlesize":
            10,

        "axes.linewidth":
            0.8,

        "lines.linewidth":
            1.4,

        "lines.markersize":
            4.5,

        "pdf.fonttype":
            42,
    }
)


colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()[
    "color"
]


# ==================================================================================================
# 12. FIGURE 23-A — SUBSET × SPLIT
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.15,
    ),
)


x = np.arange(
    len(
        PRIMARY_ORDER
    )
)


labels = [
    DISPLAY[
        x
    ]
    for x in PRIMARY_ORDER
]


for split in SPLITS:

    values = [
        float(
            row_for(
                primary_df,
                split,
                subset,
            )[
                "pr_auc"
            ]
        )

        for subset
        in PRIMARY_ORDER
    ]


    line = axes[
        0
    ].plot(
        x,
        values,
        marker="o",
        label=DISPLAY[
            split
        ],
    )[
        0
    ]


    axes[
        0
    ].axhline(
        validation_meta[
            split
        ][
            "attack_prevalence"
        ],
        linestyle=":",
        linewidth=0.9,
        color=line.get_color(),
        alpha=0.75,
    )


axes[
    0
].set_title(
    "(a) PR-AUC"
)

axes[
    0
].set_ylabel(
    "PR-AUC"
)

axes[
    0
].set_xticks(
    x
)

axes[
    0
].set_xticklabels(
    labels,
    rotation=38,
    ha="right",
)

axes[
    0
].set_ylim(
    0,
    1.03,
)

axes[
    0
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    0
].legend(
    frameon=False
)


for split in SPLITS:

    values = [
        float(
            row_for(
                primary_df,
                split,
                subset,
            )[
                "roc_auc"
            ]
        )

        for subset
        in PRIMARY_ORDER
    ]


    axes[
        1
    ].plot(
        x,
        values,
        marker="o",
        label=DISPLAY[
            split
        ],
    )


axes[
    1
].axhline(
    0.5,
    linestyle=":",
    linewidth=0.9,
    color="0.35",
)

axes[
    1
].set_title(
    "(b) ROC-AUC"
)

axes[
    1
].set_ylabel(
    "ROC-AUC"
)

axes[
    1
].set_xticks(
    x
)

axes[
    1
].set_xticklabels(
    labels,
    rotation=38,
    ha="right",
)

axes[
    1
].set_ylim(
    0,
    1.03,
)

axes[
    1
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    1
].legend(
    frameon=False
)


fig.suptitle(
    "Stage23 subset × split interaction"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_a_subset_split_interaction",
)


# ==================================================================================================
# 13. FIGURE 23-B — REMOVAL PENALTIES + INTERACTION
# ==================================================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        7.25,
        6.2,
    ),
)


y = np.arange(
    len(
        PRIMARY_ABLATIONS
    )
)


ylabels = [
    DISPLAY[
        x
    ]
    for x in PRIMARY_ABLATIONS
]


# PR penalties
ax = axes[
    0,
    0
]


for i, row in unc_primary.iterrows():

    yr = i - 0.12
    yc = i + 0.12


    ax.hlines(
        yr,
        row[
            "random_pr_auc_ci_low"
        ],
        row[
            "random_pr_auc_ci_high"
        ],
        color=colors[
            0
        ],
    )

    ax.plot(
        row[
            "random_pr_auc_removal_penalty"
        ],
        yr,
        "o",
        color=colors[
            0
        ],
    )


    ax.hlines(
        yc,
        row[
            "chronological_pr_auc_ci_low"
        ],
        row[
            "chronological_pr_auc_ci_high"
        ],
        color=colors[
            1
        ],
    )

    ax.plot(
        row[
            "chronological_pr_auc_removal_penalty"
        ],
        yc,
        "o",
        color=colors[
            1
        ],
    )


ax.axvline(
    0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_title(
    "(a) PR-AUC removal penalty"
)

ax.set_xlabel(
    "FULL − ablated"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# PR interaction
ax = axes[
    0,
    1
]


for i, row in unc_primary.iterrows():

    ax.hlines(
        i,
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    ax.plot(
        row[
            "pr_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


ax.axvline(
    0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_title(
    "(b) PR-AUC interaction"
)

ax.set_xlabel(
    "Random penalty − chronological penalty"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# ROC penalties
ax = axes[
    1,
    0
]


for i, row in unc_primary.iterrows():

    yr = i - 0.12
    yc = i + 0.12


    ax.hlines(
        yr,
        row[
            "random_roc_auc_ci_low"
        ],
        row[
            "random_roc_auc_ci_high"
        ],
        color=colors[
            0
        ],
    )

    ax.plot(
        row[
            "random_roc_auc_removal_penalty"
        ],
        yr,
        "o",
        color=colors[
            0
        ],
    )


    ax.hlines(
        yc,
        row[
            "chronological_roc_auc_ci_low"
        ],
        row[
            "chronological_roc_auc_ci_high"
        ],
        color=colors[
            1
        ],
    )

    ax.plot(
        row[
            "chronological_roc_auc_removal_penalty"
        ],
        yc,
        "o",
        color=colors[
            1
        ],
    )


ax.axvline(
    0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_title(
    "(c) ROC-AUC removal penalty"
)

ax.set_xlabel(
    "FULL − ablated"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# ROC interaction
ax = axes[
    1,
    1
]


for i, row in unc_primary.iterrows():

    ax.hlines(
        i,
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    ax.plot(
        row[
            "roc_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


ax.axvline(
    0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_title(
    "(d) ROC-AUC interaction"
)

ax.set_xlabel(
    "Random penalty − chronological penalty"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


fig.suptitle(
    "Stage23 removal penalties and frozen uncertainty"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_b_removal_penalties_and_interaction",
)


# ==================================================================================================
# 14. FIGURE 23-C — STUMP DEGRADATION
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.15,
    ),
)


x = np.arange(
    len(
        STUMP_ORDER
    )
)

width = 0.34


axes[
    0
].bar(
    x
    -
    width
    / 2,
    stump_df[
        "random_pr_auc"
    ],
    width,
    label="Random-natural",
)

axes[
    0
].bar(
    x
    +
    width
    / 2,
    stump_df[
        "chronological_pr_auc"
    ],
    width,
    label="Chronological-natural",
)

axes[
    0
].set_xticks(
    x
)

axes[
    0
].set_xticklabels(
    STUMP_ORDER,
    rotation=25,
    ha="right",
)

axes[
    0
].set_ylabel(
    "PR-AUC"
)

axes[
    0
].set_title(
    "(a) Single-feature PR-AUC"
)

axes[
    0
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    0
].legend(
    frameon=False
)


axes[
    1
].bar(
    x
    -
    width
    / 2,
    stump_df[
        "random_roc_auc"
    ],
    width,
    label="Random-natural",
)

axes[
    1
].bar(
    x
    +
    width
    / 2,
    stump_df[
        "chronological_roc_auc"
    ],
    width,
    label="Chronological-natural",
)

axes[
    1
].axhline(
    0.5,
    linestyle=":",
    color="0.35",
)

axes[
    1
].set_xticks(
    x
)

axes[
    1
].set_xticklabels(
    STUMP_ORDER,
    rotation=25,
    ha="right",
)

axes[
    1
].set_ylabel(
    "ROC-AUC"
)

axes[
    1
].set_title(
    "(b) Single-feature ROC-AUC"
)

axes[
    1
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    1
].legend(
    frameon=False
)


fig.suptitle(
    "Stage23 depth-1 stump degradation"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_c_single_feature_stump_degradation",
)


# ==================================================================================================
# 15. FIGURE 23-D — SHAP PROXY ABSORPTION
# ==================================================================================================

jaccard_matrix = np.zeros(
    (
        len(
            PRIMARY_ABLATIONS
        ),
        len(
            SPLITS
        ),
    ),
    dtype=float,
)


gain_matrix = np.zeros_like(
    jaccard_matrix
)


gainer_matrix = np.empty(
    jaccard_matrix.shape,
    dtype=object,
)


for i, subset in enumerate(
    PRIMARY_ABLATIONS
):

    for j, split in enumerate(
        SPLITS
    ):

        r = shap_primary[
            (
                shap_primary[
                    "split"
                ]
                == split
            )
            &
            (
                shap_primary[
                    "subset"
                ]
                == subset
            )
        ]


        if len(
            r
        ) != 1:

            raise RuntimeError(
                f"SHAP consensus row not unique: {split}/{subset}"
            )


        r = r.iloc[
            0
        ]


        jaccard_matrix[
            i,
            j
        ] = float(
            r[
                "jaccard_at_10_vs_full"
            ]
        )


        gain_matrix[
            i,
            j
        ] = float(
            r[
                "top_retained_consensus_share_gain"
            ]
        )


        gainer_matrix[
            i,
            j
        ] = str(
            r[
                "top_retained_consensus_share_gainer"
            ]
        )


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        4.4,
    ),
)


im = axes[
    0
].imshow(
    jaccard_matrix,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="Blues",
)


axes[
    0
].set_xticks(
    np.arange(
        len(
            SPLITS
        )
    )
)

axes[
    0
].set_xticklabels(
    [
        DISPLAY[
            s
        ]
        for s in SPLITS
    ],
    rotation=15,
)

axes[
    0
].set_yticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    0
].set_yticklabels(
    [
        DISPLAY[
            s
        ]
        for s in PRIMARY_ABLATIONS
    ]
)

axes[
    0
].set_title(
    "(a) Top-10 overlap with FULL"
)


for i in range(
    jaccard_matrix.shape[
        0
    ]
):

    for j in range(
        jaccard_matrix.shape[
            1
        ]
    ):

        axes[
            0
        ].text(
            j,
            i,
            f"{jaccard_matrix[i, j]:.3f}",
            ha="center",
            va="center",
            fontsize=7,
        )


fig.colorbar(
    im,
    ax=axes[
        0
    ],
    fraction=0.046,
    pad=0.04,
    label="Jaccard@10",
)


gain_max = max(
    float(
        np.nanmax(
            gain_matrix
        )
    ),
    1e-9,
)


im2 = axes[
    1
].imshow(
    gain_matrix,
    aspect="auto",
    vmin=0,
    vmax=gain_max,
    cmap="magma",
)


axes[
    1
].set_xticks(
    np.arange(
        len(
            SPLITS
        )
    )
)

axes[
    1
].set_xticklabels(
    [
        DISPLAY[
            s
        ]
        for s in SPLITS
    ],
    rotation=15,
)

axes[
    1
].set_yticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    1
].set_yticklabels(
    [
        DISPLAY[
            s
        ]
        for s in PRIMARY_ABLATIONS
    ]
)

axes[
    1
].set_title(
    "(b) Largest retained-feature share gain"
)


for i in range(
    gain_matrix.shape[
        0
    ]
):

    for j in range(
        gain_matrix.shape[
            1
        ]
    ):

        name = "\n".join(
            textwrap.wrap(
                str(
                    gainer_matrix[
                        i,
                        j
                    ]
                ),
                width=15,
            )
        )


        axes[
            1
        ].text(
            j,
            i,
            f"{name}\n{gain_matrix[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.6,
        )


fig.colorbar(
    im2,
    ax=axes[
        1
    ],
    fraction=0.046,
    pad=0.04,
    label="Δ normalized consensus share",
)


fig.suptitle(
    "Stage23 SHAP proxy absorption — descriptive consensus"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_d_shap_proxy_absorption",
)


# ==================================================================================================
# 16. SUPPLEMENTARY FIGURE S1 — PLACEBO INTERACTIONS
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.4,
    ),
)


y = np.arange(
    len(
        PLACEBO_ORDER
    )
)

labels = [
    DISPLAY[
        x
    ]
    for x in PLACEBO_ORDER
]


for i, row in unc_placebo.iterrows():

    axes[
        0
    ].hlines(
        i,
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    axes[
        0
    ].plot(
        row[
            "pr_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


axes[
    0
].axvline(
    0,
    linestyle=":",
    color="0.35",
)

axes[
    0
].set_yticks(
    y
)

axes[
    0
].set_yticklabels(
    labels
)

axes[
    0
].invert_yaxis()

axes[
    0
].set_title(
    "(a) PR-AUC interaction"
)

axes[
    0
].set_xlabel(
    "Random penalty − chronological penalty"
)

axes[
    0
].grid(
    axis="x",
    alpha=0.25,
)


for i, row in unc_placebo.iterrows():

    axes[
        1
    ].hlines(
        i,
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    axes[
        1
    ].plot(
        row[
            "roc_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


axes[
    1
].axvline(
    0,
    linestyle=":",
    color="0.35",
)

axes[
    1
].set_yticks(
    y
)

axes[
    1
].set_yticklabels(
    labels
)

axes[
    1
].invert_yaxis()

axes[
    1
].set_title(
    "(b) ROC-AUC interaction"
)

axes[
    1
].set_xlabel(
    "Random penalty − chronological penalty"
)

axes[
    1
].grid(
    axis="x",
    alpha=0.25,
)


fig.suptitle(
    "Stage23 matched-size placebo interactions"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_s1_placebo_interactions",
)


# ==================================================================================================
# 17. SUPPLEMENTARY FIGURE S2 — ATTACK-FAMILY ΔF1
# ==================================================================================================

random_families = random_interp[
    "attack_family"
].tolist()


chrono_families = chrono_interp[
    "attack_family"
].tolist()


if chrono_families != [
    "Infilteration"
]:

    raise RuntimeError(
        "Unexpected chronological interpretable family set."
    )


rmat = (
    attack_df[
        (
            attack_df[
                "split"
            ]
            == "RANDOM_NATURAL"
        )
        &
        (
            attack_df[
                "interpretation_status"
            ]
            == "INTERPRETABLE"
        )
        &
        (
            attack_df[
                "subset"
            ].isin(
                PRIMARY_ABLATIONS
            )
        )
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(
        index=random_families,
        columns=PRIMARY_ABLATIONS,
    )
)


cmat = (
    attack_df[
        (
            attack_df[
                "split"
            ]
            == "CHRONOLOGICAL_NATURAL"
        )
        &
        (
            attack_df[
                "interpretation_status"
            ]
            == "INTERPRETABLE"
        )
        &
        (
            attack_df[
                "subset"
            ].isin(
                PRIMARY_ABLATIONS
            )
        )
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(
        index=chrono_families,
        columns=PRIMARY_ABLATIONS,
    )
)


if rmat.isna().any().any():

    raise RuntimeError(
        "RANDOM attack-family ΔF1 matrix contains NaN."
    )


if cmat.isna().any().any():

    raise RuntimeError(
        "CHRONO attack-family ΔF1 matrix contains NaN."
    )


all_values = np.concatenate(
    [
        rmat.to_numpy().ravel(),
        cmat.to_numpy().ravel(),
    ]
)


vlim = max(
    float(
        np.max(
            np.abs(
                all_values
            )
        )
    ),
    1e-9,
)


norm = TwoSlopeNorm(
    vmin=-vlim,
    vcenter=0.0,
    vmax=vlim,
)


fig, axes = plt.subplots(
    2,
    1,
    figsize=(
        7.25,
        6.2,
    ),
    gridspec_kw={
        "height_ratios":
            [
                5,
                1.2,
            ]
    },
)


im = axes[
    0
].imshow(
    rmat.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)


axes[
    0
].set_xticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    0
].set_xticklabels(
    [
        DISPLAY[
            x
        ]
        for x in PRIMARY_ABLATIONS
    ],
    rotation=30,
    ha="right",
)

axes[
    0
].set_yticks(
    np.arange(
        len(
            random_families
        )
    )
)

axes[
    0
].set_yticklabels(
    random_families
)

axes[
    0
].set_title(
    "(a) Random-natural — support ≥100"
)


for i in range(
    rmat.shape[
        0
    ]
):

    for j in range(
        rmat.shape[
            1
        ]
    ):

        axes[
            0
        ].text(
            j,
            i,
            f"{rmat.iloc[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.5,
        )


axes[
    1
].imshow(
    cmat.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)


axes[
    1
].set_xticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    1
].set_xticklabels(
    [
        DISPLAY[
            x
        ]
        for x in PRIMARY_ABLATIONS
    ],
    rotation=30,
    ha="right",
)

axes[
    1
].set_yticks(
    [
        0
    ]
)

axes[
    1
].set_yticklabels(
    chrono_families
)

axes[
    1
].set_title(
    "(b) Chronological-natural — support ≥100"
)


for j in range(
    cmat.shape[
        1
    ]
):

    axes[
        1
    ].text(
        j,
        0,
        f"{cmat.iloc[0, j]:+.3f}",
        ha="center",
        va="center",
        fontsize=6,
    )


cbar = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "ΔF1 = FULL − ablated at threshold 0.50"
)


fig.suptitle(
    "Stage23 attack-family-conditioned degradation"
)


fig.subplots_adjust(
    left=0.22,
    right=0.90,
    top=0.91,
    bottom=0.14,
    hspace=0.78,
)


save_figure(
    fig,
    FIGURES
    / "figure_23_s2_attack_family_delta_f1",
)


# ==================================================================================================
# 18. VERIFY FIGURE SET
# ==================================================================================================

EXPECTED_FIGURE_BASES = {
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
    "figure_23_s1_placebo_interactions",
    "figure_23_s2_attack_family_delta_f1",
}


actual_png = {
    p.stem
    for p in FIGURES.glob(
        "*.png"
    )
}

actual_pdf = {
    p.stem
    for p in FIGURES.glob(
        "*.pdf"
    )
}


if actual_png != EXPECTED_FIGURE_BASES:

    raise RuntimeError(
        "PNG figure set mismatch."
    )


if actual_pdf != EXPECTED_FIGURE_BASES:

    raise RuntimeError(
        "PDF figure set mismatch."
    )


print()
print("[EXACT] frozen main figure bases : 4 / 4")
print("[EXACT] supplementary figures    : 2")
print("[EXACT] PNG/PDF figure files     : 12")
print()


# ==================================================================================================
# 19. SCIENTIFIC SYNTHESIS
# ==================================================================================================

full_r = row_for(
    primary_df,
    "RANDOM_NATURAL",
    "FULL",
)

full_c = row_for(
    primary_df,
    "CHRONOLOGICAL_NATURAL",
    "FULL",
)

beh_r = row_for(
    primary_df,
    "RANDOM_NATURAL",
    "BEHAVIOR_ONLY",
)

beh_c = row_for(
    primary_df,
    "CHRONOLOGICAL_NATURAL",
    "BEHAVIOR_ONLY",
)


random_prev = validation_meta[
    "RANDOM_NATURAL"
][
    "attack_prevalence"
]

chrono_prev = validation_meta[
    "CHRONOLOGICAL_NATURAL"
][
    "attack_prevalence"
]


primary_pr_sig = unc_primary.loc[
    unc_primary[
        "pr_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


primary_roc_sig = unc_primary.loc[
    unc_primary[
        "roc_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


placebo_pr_sig = unc_placebo.loc[
    unc_placebo[
        "pr_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


placebo_roc_sig = unc_placebo.loc[
    unc_placebo[
        "roc_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


chrono_attack_support = int(
    chrono_full_attack[
        "support_attack"
    ].sum()
)

random_attack_support = int(
    random_full_attack[
        "support_attack"
    ].sum()
)


synthesis = f"""
# Stage23 Final Scientific Synthesis

## Frozen scope

Stage23 evaluates validation-regime sensitivity using seven primary representations,
five matched-size placebo removals, six frozen depth-1 stump controls, paired
stratified bootstrap uncertainty, component-specific TreeSHAP proxy-absorption
analysis, and frozen attack-family conditioning.

Raw Mar1 and Mar2 remain permanently forbidden. Stage23 does not create a new
untouched final holdout.

## Full-model split contrast

FULL random-natural:

- PR-AUC: {float(full_r["pr_auc"]):.12f}
- ROC-AUC: {float(full_r["roc_auc"]):.12f}
- Attack prevalence: {random_prev:.12f}
- PR-AUC minus prevalence: {float(full_r["pr_auc_minus_attack_prevalence"]):.12f}

FULL chronological-natural:

- PR-AUC: {float(full_c["pr_auc"]):.12f}
- ROC-AUC: {float(full_c["roc_auc"]):.12f}
- Attack prevalence: {chrono_prev:.12f}
- PR-AUC minus prevalence: {float(full_c["pr_auc_minus_attack_prevalence"]):.12f}

The ensemble therefore exhibits very strong validation-regime sensitivity: random
validation is highly discriminative, while chronological ranking is near the relevant
no-skill references.

## Primary ablations

The frozen shortcut interaction is:

(FULL_RANDOM - ABLATED_RANDOM)
-
(FULL_CHRONOLOGICAL - ABLATED_CHRONOLOGICAL)

Positive values indicate disproportionate random-validation benefit from the tested
information; negative values indicate stronger chronological dependence.

Primary PR-AUC interaction CIs excluding zero:

{chr(10).join("- " + x for x in primary_pr_sig)}

Primary ROC-AUC interaction CIs excluding zero:

{chr(10).join("- " + x for x in primary_roc_sig)}

The directions are not uniform across features or metrics, which prohibits reducing
the results to a binary leakage/not-leakage conclusion.

## Placebo context

Placebo PR-AUC interaction CIs excluding zero:

{chr(10).join("- " + x for x in placebo_pr_sig)}

Placebo ROC-AUC interaction CIs excluding zero:

{chr(10).join("- " + x for x in placebo_roc_sig)}

Because matched-size placebo removals also produce non-zero interactions and CI
exclusion, statistical separation from zero is not itself evidence of leakage.

## Single-feature stump controls

Each of Dst Port, Init Fwd Win Byts, and Fwd Seg Size Min discriminates much more
strongly under random-natural validation than under chronological-natural validation.
This is consistent with split-specific or shortcut-like information that transfers
poorly, but it does not prove leakage or causality.

## SHAP proxy absorption

TreeSHAP remains component-specific for LightGBM and XGBoost. The equal-weight
normalized consensus used for summary visualization is descriptive only and is not
exact SHAP for the probability-averaged ensemble.

Feature-rank and normalized-share redistribution after ablation is consistent with
proxy absorption by retained variables. SHAP importance does not prove causal
substitution.

## Behavior-restricted representation

BEHAVIOR_ONLY random-natural:

- PR-AUC: {float(beh_r["pr_auc"]):.12f}
- ROC-AUC: {float(beh_r["roc_auc"]):.12f}
- F1@0.50: {float(beh_r["f1_at_0_50"]):.12f}

BEHAVIOR_ONLY chronological-natural:

- PR-AUC: {float(beh_c["pr_auc"]):.12f}
- ROC-AUC: {float(beh_c["roc_auc"]):.12f}
- F1@0.50: {float(beh_c["f1_at_0_50"]):.12f}

The chronological behavior-restricted representation does not retain strong
discrimination under this frozen protocol. This result does not equate behavior-only
performance with real-world deployment performance.

## Attack-family context

Random-natural validation contains {random_attack_support:,} attack rows and
11 families meeting the frozen support threshold of 100.

Chronological-natural validation contains {chrono_attack_support:,} attack rows,
all belonging to Infilteration under the frozen mapping. The other 12 frozen
development families have zero chronological attack support.

Therefore, the split contrast combines temporal change with a major attack-family
composition change. Stage23 cannot attribute the complete random-to-chronological
performance gap to any single feature or leakage mechanism.

## Consolidated conclusion

The frozen evidence supports:

1. Strong validation-regime sensitivity.
2. Poor temporal transfer of several individually discriminative cues.
3. Metric-dependent effects of feature removal.
4. Non-trivial placebo interactions that caution against interpreting significance
   as leakage evidence.
5. SHAP redistribution consistent with proxy absorption.
6. Material attack-family composition differences between split regimes.
7. A bounded validation-sensitivity / poor-transfer conclusion rather than proof
   that any tested feature is leakage.

## Prohibited claims

{chr(10).join("- " + x for x in interpretation_matrix["prohibited_claims"])}

## Governance

- Stage23 fit budget: 50 / 50 SEALED.
- New fits in Stage23-7A-R3: 0.
- Model inference: 0.
- LightGBM execution: 0.
- XGBoost execution: 0.
- New SHAP computation: 0.
- New bootstrap sampling: 0.
- NPZ reads: 0.
- Parquet reads: 0.
- Raw Mar1 accessed: NO.
- Raw Mar2 accessed: NO.
"""


write_text(
    TMP
    / "stage23_7_final_scientific_synthesis.md",
    synthesis,
)


# ==================================================================================================
# 20. MANUSCRIPT INTEGRATION
# ==================================================================================================

results_tex = rf"""
% Stage23 frozen manuscript integration.
% Generated from sealed JSON/CSV evidence only.

\subsection{{Validation-Safe Shortcut-Feature Audit}}

The full 70-feature ensemble achieved a random-natural PR-AUC of
{float(full_r["pr_auc"]):.6f} and ROC-AUC of {float(full_r["roc_auc"]):.6f},
compared with chronological-natural PR-AUC {float(full_c["pr_auc"]):.6f}
and ROC-AUC {float(full_c["roc_auc"]):.6f}. The chronological attack prevalence
was {chrono_prev:.6f}, placing the chronological PR-AUC only
{float(full_c["pr_auc_minus_attack_prevalence"]):.6f} above its prevalence
reference.

Frozen ablations revealed metric-dependent split interactions. Some pre-specified
feature removals disproportionately reduced random-natural ranking, whereas others
reduced chronological ranking more strongly. Importantly, matched-size placebo
removals also generated non-zero interactions and bootstrap intervals excluding zero.
Consequently, interaction significance alone does not identify leakage.

Depth-1 controls based individually on Dst Port, Init Fwd Win Byts, and
Fwd Seg Size Min showed substantially stronger discrimination under random-natural
validation and weak transfer to chronological-natural validation. These results are
consistent with split-specific discriminative cues but do not establish causality.

Component-specific TreeSHAP analyses further demonstrated importance redistribution
after feature removal. Equal-weight normalized LightGBM/XGBoost importance shares are
used only as a descriptive consensus; they are not exact SHAP values for the
probability-averaged ensemble.

The behavior-restricted representation retained strong random-natural discrimination
but did not preserve chronological discrimination. Chronological attack-family
analysis further showed that all {chrono_attack_support:,} chronological attack rows
belonged to Infilteration, whereas random-natural validation contained 11 attack
families meeting the frozen support threshold. Hence the validation-regime contrast
contains both temporal and attack-family composition changes.

Taken together, Stage23 supports a bounded conclusion of substantial validation
sensitivity and poor forward-temporal transfer. It does not prove that any tested
feature is leakage.
"""


write_text(
    MANUSCRIPT
    / "stage23_results_integration.tex",
    results_tex,
)


table_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Stage23 primary representation performance under frozen validation regimes.}",
    r"\label{tab:stage23-primary}",
    r"\small",
    r"\begin{tabular}{lrrrr}",
    r"\hline",
    r"Subset & Random PR-AUC & Chronological PR-AUC & Random ROC-AUC & Chronological ROC-AUC \\",
    r"\hline",
]


for _, r in primary_wide.iterrows():

    table_lines.append(
        f"{latex_escape(r['subset'])} & "
        f"{r['random_pr_auc']:.6f} & "
        f"{r['chronological_pr_auc']:.6f} & "
        f"{r['random_roc_auc']:.6f} & "
        f"{r['chronological_roc_auc']:.6f} \\\\"
    )


table_lines += [
    r"\hline",
    r"\end{tabular}",
    r"\end{table*}",
]


write_text(
    MANUSCRIPT
    / "stage23_main_tables.tex",
    "\n".join(
        table_lines
    ),
)


captions = r"""
% Stage23 frozen figure captions

\paragraph{Figure 23-A.}
Subset-by-split interaction across all seven primary representations. PR-AUC is shown
against split-specific attack prevalence; ROC-AUC is shown against the 0.5 no-skill
reference.

\paragraph{Figure 23-B.}
FULL-minus-ablated PR-AUC and ROC-AUC penalties, frozen paired-bootstrap percentile
95\% confidence intervals, and random-minus-chronological shortcut interactions.

\paragraph{Figure 23-C.}
Depth-1 single-feature controls for Dst Port, Init Fwd Win Byts, and Fwd Seg Size Min
under random-natural and chronological-natural validation.

\paragraph{Figure 23-D.}
SHAP proxy-absorption summary. The equal-weight consensus represents normalized
component importance shares and is descriptive only; it is not exact ensemble SHAP.
"""


write_text(
    MANUSCRIPT
    / "stage23_figure_captions.tex",
    captions,
)


EXPECTED_MANUSCRIPT = {
    "stage23_results_integration.tex",
    "stage23_main_tables.tex",
    "stage23_figure_captions.tex",
}


actual_manuscript = {
    p.name
    for p in MANUSCRIPT.glob(
        "*.tex"
    )
}


if actual_manuscript != EXPECTED_MANUSCRIPT:

    raise RuntimeError(
        "Manuscript integration file set mismatch."
    )


print("[EXACT] manuscript integration files: 3 / 3")
print()


# ==================================================================================================
# 21. CONSOLIDATED SCIENTIFIC AUDIT / CLOSURE RULE
# ==================================================================================================

closure_conditions = [
    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                0
            ],

        "satisfied":
            True,

        "evidence":
            "7 primary subsets × 2 frozen natural splits complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                1
            ],

        "satisfied":
            True,

        "evidence":
            "6/6 frozen depth-1 stump controls complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                2
            ],

        "satisfied":
            True,

        "evidence":
            "5 placebo subsets × 2 frozen natural splits complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                3
            ],

        "satisfied":
            True,

        "evidence":
            "48 component TreeSHAP explanations; 22 subset/split comparisons",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                4
            ],

        "satisfied":
            True,

        "evidence":
            "13 frozen attack families; 182 family metric rows",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                5
            ],

        "satisfied":
            True,

        "evidence":
            "Stage23-7 consolidated scientific audit generated",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                6
            ],

        "satisfied":
            True,

        "evidence":
            "Figures 23-A through 23-D, supplementary outputs, and manuscript fragments generated",
    },
]


if not all(
    x[
        "satisfied"
    ]
    for x in closure_conditions
):

    raise RuntimeError(
        "Stage23 stopping rule not fully satisfied."
    )


audit = {
    "stage":
        "Stage23-7A-R3",

    "status":
        "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            EXPECTED_PARENT_TAG,
    },

    "counts": {
        "primary_result_jsons":
            12,

        "primary_subset_split_rows":
            14,

        "placebo_result_jsons":
            10,

        "placebo_subset_split_rows":
            10,

        "uncertainty_comparisons":
            11,

        "stump_controls":
            6,

        "shap_proxy_comparisons":
            22,

        "shap_component_models":
            48,

        "component_shap_top20_rows":
            960,

        "attack_family_metric_rows":
            182,

        "attack_families":
            13,

        "random_interpretable_families":
            11,

        "chronological_interpretable_families":
            1,

        "publication_tables":
            10,

        "main_figure_bases":
            4,

        "supplementary_figure_bases":
            2,

        "figure_files":
            12,

        "manuscript_files":
            3,

        "stopping_conditions_satisfied":
            7,
    },

    "headline_full_results": {
        "RANDOM_NATURAL": {
            "pr_auc":
                float(
                    full_r[
                        "pr_auc"
                    ]
                ),

            "roc_auc":
                float(
                    full_r[
                        "roc_auc"
                    ]
                ),

            "attack_prevalence":
                float(
                    random_prev
                ),

            "pr_auc_minus_prevalence":
                float(
                    full_r[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),
        },

        "CHRONOLOGICAL_NATURAL": {
            "pr_auc":
                float(
                    full_c[
                        "pr_auc"
                    ]
                ),

            "roc_auc":
                float(
                    full_c[
                        "roc_auc"
                    ]
                ),

            "attack_prevalence":
                float(
                    chrono_prev
                ),

            "pr_auc_minus_prevalence":
                float(
                    full_c[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),
        },
    },

    "attack_family_context": {
        "random_attack_support":
            random_attack_support,

        "chronological_attack_support":
            chrono_attack_support,

        "chronological_positive_families":
            [
                "Infilteration"
            ],
    },

    "interpretation": {
        "supported":
            (
                "Strong validation sensitivity and poor forward-temporal "
                "transfer of several discriminative cues."
            ),

        "not_supported":
            (
                "No tested feature is established as leakage or as a causal "
                "source of the split-performance gap."
            ),

        "prohibited_claims":
            interpretation_matrix[
                "prohibited_claims"
            ],
    },

    "stopping_rule_evidence":
        closure_conditions,

    "governance": {
        "stage23_fit_budget":
            "50 / 50 SEALED",

        "additional_model_fits_authorized":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_execution":
            0,

        "xgboost_execution":
            0,

        "new_shap_computation":
            0,

        "new_bootstrap_sampling":
            0,

        "model_files_read":
            0,

        "probability_npz_files_read":
            0,

        "parquet_files_read":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "repository_modified":
            False,
    },

    "next_authorized_action":
        "ZERO_FIT_FINAL_STAGE23_CLOSURE_SEAL_AND_PUSH",
}


write_json(
    AUDIT
    / "stage23_7_consolidated_scientific_audit.json",
    audit,
)


# ==================================================================================================
# 22. SOURCE MANIFEST — FAIL CLOSED AGAINST FORBIDDEN FILE TYPES
# ==================================================================================================

for rel in SOURCE_FILES:

    low = rel.lower()


    if low.endswith(
        ".parquet"
    ):

        raise RuntimeError(
            f"Forbidden parquet source read:\n{rel}"
        )


    if low.endswith(
        ".npz"
    ):

        raise RuntimeError(
            f"Forbidden NPZ source read:\n{rel}"
        )


    if "model.txt" in low:

        raise RuntimeError(
            f"Forbidden model source read:\n{rel}"
        )


    if "xgboost_model" in low:

        raise RuntimeError(
            f"Forbidden model source read:\n{rel}"
        )


    if low.endswith(
        ".joblib"
    ):

        raise RuntimeError(
            f"Forbidden model source read:\n{rel}"
        )


source_manifest = {
    "stage":
        "Stage23-7A-R3",

    "execution_parent_commit":
        EXPECTED_PARENT,

    "policy":
        "SEALED_JSON_CSV_ONLY",

    "source_file_count":
        len(
            SOURCE_FILES
        ),

    "source_files":
        [
            {
                "path":
                    rel,

                "sha256":
                    digest,
            }

            for rel, digest
            in sorted(
                SOURCE_FILES.items()
            )
        ],

    "model_files_read":
        0,

    "probability_npz_files_read":
        0,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json(
    AUDIT
    / "stage23_7_source_manifest.json",
    source_manifest,
)


# ==================================================================================================
# 23. README / EXECUTION STATE
# ==================================================================================================

readme = f"""
# Stage23 Final Synthesis Package

Status: **FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED**

Parent commit: `{EXPECTED_PARENT}`

Parent tag: `{EXPECTED_PARENT_TAG}`

## Main figures

- Figure 23-A — subset × split interaction
- Figure 23-B — removal penalties and shortcut interactions
- Figure 23-C — single-feature stump degradation
- Figure 23-D — SHAP proxy absorption

Each is provided as PNG and vector PDF.

## Supplementary outputs

- matched-size placebo interactions
- attack-family-conditioned degradation
- complete component-specific SHAP top-20 table
- behavior-restricted operating metrics

## Governance

- Stage23 fit budget: 50 / 50 SEALED
- New fits: 0
- Model inference: 0
- New SHAP: 0
- New bootstrap sampling: 0
- NPZ reads: 0
- Parquet reads: 0
- Raw Mar1: not accessed
- Raw Mar2: not accessed

Next authorized action: zero-fit final Stage23 closure seal and push.
"""


write_text(
    TMP
    / "README.md",
    readme,
)


execution_state = {
    "stage":
        "Stage23-7A-R3",

    "status":
        "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent_commit":
        EXPECTED_PARENT,

    "counts": {
        "publication_tables":
            10,

        "main_figures":
            4,

        "supplementary_figures":
            2,

        "figure_files":
            12,

        "manuscript_files":
            3,

        "stopping_conditions":
            7,
    },

    "governance": {
        "stage23_fit_budget_before":
            50,

        "stage23_fit_budget_after":
            50,

        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_execution":
            0,

        "xgboost_execution":
            0,

        "new_shap":
            0,

        "new_bootstrap":
            0,

        "npz_reads":
            0,

        "parquet_reads":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "repository_modified":
            False,
    },

    "next_authorized_action":
        "ZERO_FIT_FINAL_STAGE23_CLOSURE_SEAL_AND_PUSH",
}


write_json(
    TMP
    / "execution_state.json",
    execution_state,
)


# ==================================================================================================
# 24. CHECKSUM PACKAGE
# ==================================================================================================

checksum_path = (
    TMP
    / "checksums.sha256"
)


files_to_hash = sorted(
    p
    for p in TMP.rglob(
        "*"
    )
    if (
        p.is_file()
        and p != checksum_path
    )
)


checksum_lines = []


for path in files_to_hash:

    rel = path.relative_to(
        TMP
    )


    checksum_lines.append(
        f"{sha256_file(path)}  {rel.as_posix()}"
    )


write_text(
    checksum_path,
    "\n".join(
        checksum_lines
    ),
)


for line in checksum_path.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue


    digest, rel = line.split(
        "  ",
        1,
    )


    p = (
        TMP
        / rel
    )


    if not p.is_file():

        raise RuntimeError(
            f"Checksum target missing: {rel}"
        )


    if sha256_file(
        p
    ) != digest:

        raise RuntimeError(
            f"Checksum verification failed: {rel}"
        )


package_manifest_sha = sha256_file(
    checksum_path
)


# ==================================================================================================
# 25. REPOSITORY MUST STILL BE UNTOUCHED
# ==================================================================================================

if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Repository HEAD changed during Stage23-7A."
    )


final_repo_status = git(
    "status",
    "--porcelain",
)


if final_repo_status:

    raise RuntimeError(
        "Repository modified during Stage23-7A:\n"
        + final_repo_status
    )


# ==================================================================================================
# 26. ATOMIC FINALIZATION
# ==================================================================================================

TMP.rename(
    OUT
)


if not OUT.is_dir():

    raise RuntimeError(
        "Final Stage23-7A package rename failed."
    )


final_checksum = (
    OUT
    / "checksums.sha256"
)


if sha256_file(
    final_checksum
) != package_manifest_sha:

    raise RuntimeError(
        "Checksum manifest changed during finalization."
    )


for line in final_checksum.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue


    digest, rel = line.split(
        "  ",
        1,
    )


    if sha256_file(
        OUT
        / rel
    ) != digest:

        raise RuntimeError(
            f"Finalized artifact checksum mismatch: {rel}"
        )


# ==================================================================================================
# 27. FINAL REPORT
# ==================================================================================================

print()
print(SEP)
print("STAGE23-7A — FINAL SYNTHESIS / TABLES / FIGURES COMPLETE — UNSEALED")
print(SEP)
print()

print("Output:")
print(
    " ",
    OUT,
)

print()
print("Parent commit:")
print(
    " ",
    EXPECTED_PARENT,
)

print()
print("Package checksum manifest SHA256:")
print(
    " ",
    package_manifest_sha,
)

print()
print("SCIENTIFIC INPUT ACCOUNTING")
print("  primary subsets              : 7 / 7")
print("  primary split results        : 14")
print("  placebo subsets              : 5 / 5")
print("  placebo split results        : 10")
print("  stump controls               : 6 / 6")
print("  uncertainty comparisons      : 11 / 11")
print("  SHAP component models        : 48 / 48")
print("  SHAP proxy comparisons       : 22 / 22")
print("  attack-family metric rows    : 182")
print("  frozen attack families       : 13")

print()
print("PUBLICATION OUTPUTS")
print("  frozen main figure bases     : 4 / 4")
print("  supplementary figure bases   : 2")
print("  PNG/PDF figure files         : 12")
print("  publication/supp tables      : 10")
print("  manuscript integration files : 3")
print("  stopping-rule conditions     : 7 / 7 SATISFIED")

print()
print("HEADLINE FULL RESULTS")
print(
    "  RANDOM PR / ROC              : "
    f"{float(full_r['pr_auc']):.12f} / "
    f"{float(full_r['roc_auc']):.12f}"
)

print(
    "  CHRONO PR / ROC              : "
    f"{float(full_c['pr_auc']):.12f} / "
    f"{float(full_c['roc_auc']):.12f}"
)

print(
    "  CHRONO attack prevalence     : "
    f"{chrono_prev:.12f}"
)

print(
    "  CHRONO PR - prevalence       : "
    f"{float(full_c['pr_auc_minus_attack_prevalence']):.12f}"
)

print()
print("ATTACK-FAMILY CONTEXT")
print("  RANDOM interpretable         : 11")
print("  CHRONO interpretable         : 1")
print("  CHRONO positive family       : Infilteration only")
print(
    "  CHRONO attack support        :",
    f"{chrono_attack_support:,}",
)

print()
print("GOVERNANCE")
print("  Stage23 fit budget           : 50 / 50 SEALED")
print("  additional fits authorized   : 0")
print("  new model fits               : 0")
print("  model inference              : 0")
print("  LightGBM execution           : 0")
print("  XGBoost execution            : 0")
print("  new SHAP values              : 0")
print("  new bootstrap replicates     : 0")
print("  model files read             : 0")
print("  probability NPZ reads        : 0")
print("  parquet reads                : 0")
print("  Raw Mar1 accessed            : NO")
print("  Raw Mar2 accessed            : NO")
print("  repository modified          : NO")

print()
print("STATUS:")
print("  FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED")

print()
print("NEXT AUTHORIZED ACTION:")
print("  ZERO-FIT FINAL STAGE23 CLOSURE SEAL + PUSH.")
print("  NO ADDITIONAL MODEL FITS.")

print(SEP)

STAGE23-7A-R3 — STANDALONE FINAL SYNTHESIS PACKAGE

[EXACT] branch    : main
[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag: stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] Stage23 fit budget: 50 / 50 SEALED

VERIFY FROZEN PROTOCOL

[EXACT] stopping conditions : 7
[EXACT] main frozen figures : 4
[EXACT] primary ranking      : PR_AUC
[EXACT] fixed threshold      : 0.50
[EXACT] Mar1 / Mar2          : PERMANENTLY FORBIDDEN

LOAD SEALED PRIMARY RESULTS

[EXACT] primary result JSONs : 12
[EXACT] primary subset rows   : 14
[EXACT] primary subsets       : 7 / 7
[EXACT] splits                : 2 / 2

LOAD SEALED PLACEBO RESULTS



RuntimeError: Placebo ranking result lacks attack_prevalence:
/kaggle/working/ids2018-validation-safe-ablation/results/stage23_shortcut_feature_audit/stage23_2_placebo_ablation/random/placebo_counts/stage23_2a_placebo_counts_random_natural_result.json

In [15]:
# ==================================================================================================
# STAGE23-7A-R4 — SEALED STAGE23-2 SCHEMA CENSUS PREFLIGHT
#
# PURPOSE
#   Inspect all 10 frozen placebo result JSON schemas BEFORE another
#   final-synthesis attempt.
#
# READS
#   JSON ONLY
#
# ZERO FITS
# ZERO INFERENCE
# ZERO LIGHTGBM / XGBOOST
# ZERO SHAP
# ZERO BOOTSTRAP
# ZERO NPZ
# ZERO PARQUET
# ZERO SCIENTIFIC WRITES
# NO MAR1 / MAR2
# REPOSITORY UNCHANGED
# ==================================================================================================

from pathlib import Path
import subprocess
import json
import numpy as np


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

P1 = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_1_primary_ablation"
)

P2 = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_2_placebo_ablation"
)

OUT = Path(
    "/kaggle/working/stage23_7_final_synthesis"
)

EXPECTED_HEAD = (
    "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
)

EXPECTED_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)

EXPECTED_PLACEBOS = {
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
}

EXPECTED_SPLITS = {
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
}


def git(*args):

    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )

    return (
        p.stdout or ""
    ).strip()


print("=" * 118)
print("STAGE23-7A-R4 — SEALED STAGE23-2 SCHEMA CENSUS")
print("=" * 118)
print()


# --------------------------------------------------------------------------------------------------
# 1. SEALED STATE
# --------------------------------------------------------------------------------------------------

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "Unexpected HEAD."
    )


if tag_commit != EXPECTED_HEAD:

    raise RuntimeError(
        "Stage23-6 tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty:\n"
        + status
    )


if OUT.exists():

    raise RuntimeError(
        "Final Stage23-7A package unexpectedly exists:\n"
        f"{OUT}"
    )


print("[EXACT] HEAD      :", head)
print("[EXACT] seal tag  :", EXPECTED_TAG)
print("[EXACT] worktree  : CLEAN")
print("[EXACT] final 7A package absent")
print()


# --------------------------------------------------------------------------------------------------
# 2. DERIVE FROZEN SPLIT POPULATIONS FROM PRIMARY RESULTS
# --------------------------------------------------------------------------------------------------

primary_paths = sorted(
    P1.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    primary_paths
) != 12:

    raise RuntimeError(
        f"Expected 12 primary result JSONs; found {len(primary_paths)}."
    )


validation_meta = {}


for path in primary_paths:

    d = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    split = d[
        "cell"
    ][
        "split"
    ]


    val = d[
        "data"
    ][
        "validation"
    ]


    current = {
        "rows":
            int(
                val[
                    "rows"
                ]
            ),

        "attack":
            int(
                val[
                    "attack"
                ]
            ),

        "benign":
            int(
                val[
                    "benign"
                ]
            ),

        "attack_prevalence":
            float(
                val[
                    "attack_prevalence"
                ]
            ),
    }


    if split in validation_meta:

        if validation_meta[
            split
        ] != current:

            raise RuntimeError(
                f"Primary split metadata inconsistent for {split}."
            )

    else:

        validation_meta[
            split
        ] = current


if set(
    validation_meta
) != EXPECTED_SPLITS:

    raise RuntimeError(
        "Primary split metadata incomplete."
    )


print("FROZEN SPLIT POPULATIONS")
print("-" * 118)


for split in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:

    m = validation_meta[
        split
    ]

    print(
        f"{split:<25} "
        f"rows={m['rows']:,} "
        f"attack={m['attack']:,} "
        f"benign={m['benign']:,} "
        f"prevalence={m['attack_prevalence']:.15f}"
    )


# --------------------------------------------------------------------------------------------------
# 3. CENSUS ALL 10 SEALED PLACEBO RESULT JSONs
# --------------------------------------------------------------------------------------------------

placebo_paths = sorted(
    P2.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    placebo_paths
) != 10:

    raise RuntimeError(
        f"Expected 10 placebo result JSONs; found {len(placebo_paths)}."
    )


seen_pairs = set()

schema_counts = {
    "DATA_VALIDATION_ONLY":
        0,

    "RANKING_PREVALENCE_ONLY":
        0,

    "BOTH":
        0,

    "NEITHER":
        0,
}


required_ranking = {
    "pr_auc",
    "pr_auc_minus_attack_prevalence",
    "roc_auc",
}

required_fixed = {
    "f1",
    "recall",
    "fpr",
    "fnr",
}


print()
print("=" * 118)
print("PLACEBO RESULT SCHEMA CENSUS")
print("=" * 118)
print()


for path in placebo_paths:

    d = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    split = d[
        "cell"
    ][
        "split"
    ]

    placebo = d[
        "cell"
    ][
        "placebo"
    ]


    if split not in EXPECTED_SPLITS:

        raise RuntimeError(
            f"Unexpected split in {path}: {split}"
        )


    if placebo not in EXPECTED_PLACEBOS:

        raise RuntimeError(
            f"Unexpected placebo in {path}: {placebo}"
        )


    pair = (
        split,
        placebo,
    )


    if pair in seen_pairs:

        raise RuntimeError(
            f"Duplicate placebo/split pair: {pair}"
        )


    seen_pairs.add(
        pair
    )


    ranking = d.get(
        "ranking_metrics",
        {}
    )

    fixed = d.get(
        "fixed_threshold_0_50",
        {}
    )


    missing_ranking = (
        required_ranking
        -
        set(
            ranking
        )
    )


    if missing_ranking:

        raise RuntimeError(
            f"{path.name}: missing ranking metrics "
            f"{sorted(missing_ranking)}"
        )


    missing_fixed = (
        required_fixed
        -
        set(
            fixed
        )
    )


    if missing_fixed:

        raise RuntimeError(
            f"{path.name}: missing fixed-threshold metrics "
            f"{sorted(missing_fixed)}"
        )


    data_validation = (
        d.get(
            "data",
            {}
        ).get(
            "validation"
        )
    )


    has_data_validation = (
        data_validation
        is not None
    )


    has_ranking_prevalence = (
        "attack_prevalence"
        in ranking
    )


    if (
        has_data_validation
        and has_ranking_prevalence
    ):

        schema = "BOTH"

    elif has_data_validation:

        schema = "DATA_VALIDATION_ONLY"

    elif has_ranking_prevalence:

        schema = "RANKING_PREVALENCE_ONLY"

    else:

        schema = "NEITHER"


    schema_counts[
        schema
    ] += 1


    expected = validation_meta[
        split
    ]


    # ----------------------------------------------------------------------------------------------
    # At least ONE sealed prevalence provenance is mandatory.
    # ----------------------------------------------------------------------------------------------

    if schema == "NEITHER":

        raise RuntimeError(
            f"{path.name}: neither data.validation nor "
            "ranking_metrics.attack_prevalence is present."
        )


    # ----------------------------------------------------------------------------------------------
    # Verify data.validation whenever present.
    # ----------------------------------------------------------------------------------------------

    if has_data_validation:

        for key in [
            "rows",
            "attack",
            "benign",
        ]:

            if int(
                data_validation[
                    key
                ]
            ) != int(
                expected[
                    key
                ]
            ):

                raise RuntimeError(
                    f"{path.name}: data.validation.{key} mismatch.\n"
                    f"expected={expected[key]}\n"
                    f"actual={data_validation[key]}"
                )


        if not np.isclose(
            float(
                data_validation[
                    "attack_prevalence"
                ]
            ),
            float(
                expected[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: data.validation prevalence mismatch."
            )


    # ----------------------------------------------------------------------------------------------
    # Verify ranking prevalence whenever present.
    # ----------------------------------------------------------------------------------------------

    if has_ranking_prevalence:

        if not np.isclose(
            float(
                ranking[
                    "attack_prevalence"
                ]
            ),
            float(
                expected[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: ranking prevalence mismatch."
            )


    # ----------------------------------------------------------------------------------------------
    # If both are present, they must agree exactly within frozen numeric tolerance.
    # ----------------------------------------------------------------------------------------------

    if (
        has_data_validation
        and has_ranking_prevalence
    ):

        if not np.isclose(
            float(
                data_validation[
                    "attack_prevalence"
                ]
            ),
            float(
                ranking[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: two sealed prevalence fields disagree."
            )


    print(
        f"{split:<25} "
        f"{placebo:<27} "
        f"{schema:<25} "
        f"PR={float(ranking['pr_auc']):.12f} "
        f"ROC={float(ranking['roc_auc']):.12f}"
    )


# --------------------------------------------------------------------------------------------------
# 4. COMPLETE PAIR SET
# --------------------------------------------------------------------------------------------------

expected_pairs = {
    (
        split,
        placebo,
    )

    for split
    in EXPECTED_SPLITS

    for placebo
    in EXPECTED_PLACEBOS
}


if seen_pairs != expected_pairs:

    raise RuntimeError(
        "Frozen Stage23-2 placebo/split pair set mismatch.\n"
        f"missing={sorted(expected_pairs - seen_pairs)}\n"
        f"extra={sorted(seen_pairs - expected_pairs)}"
    )


# --------------------------------------------------------------------------------------------------
# 5. FINAL
# --------------------------------------------------------------------------------------------------

print()
print("=" * 118)
print("STAGE23-7A-R4 PLACEBO SCHEMA CENSUS — PASSED")
print("=" * 118)
print()

print("Frozen placebo result JSONs : 10 / 10")
print("Frozen placebo/split pairs  : 10 / 10")
print()

print("Schema counts:")
print(
    "  DATA_VALIDATION_ONLY      :",
    schema_counts[
        "DATA_VALIDATION_ONLY"
    ],
)

print(
    "  RANKING_PREVALENCE_ONLY   :",
    schema_counts[
        "RANKING_PREVALENCE_ONLY"
    ],
)

print(
    "  BOTH                      :",
    schema_counts[
        "BOTH"
    ],
)

print(
    "  NEITHER                   :",
    schema_counts[
        "NEITHER"
    ],
)

print()
print("Governance:")
print("  Model fits                : 0")
print("  Model inference           : 0")
print("  LightGBM execution        : 0")
print("  XGBoost execution         : 0")
print("  New SHAP                  : 0")
print("  New bootstrap             : 0")
print("  NPZ reads                 : 0")
print("  Parquet reads             : 0")
print("  Scientific writes         : 0")
print("  Raw Mar1 accessed         : NO")
print("  Raw Mar2 accessed         : NO")
print("  Repository modified       : NO")

print()
print("NEXT:")
print(
    "  Build Stage23-7A using either sealed validation provenance:"
)
print(
    "  data.validation OR ranking_metrics.attack_prevalence."
)
print("=" * 118)

STAGE23-7A-R4 — SEALED STAGE23-2 SCHEMA CENSUS

[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] seal tag  : stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] final 7A package absent

FROZEN SPLIT POPULATIONS
----------------------------------------------------------------------------------------------------------------------
RANDOM_NATURAL            rows=2,882,481 attack=394,460 benign=2,488,021 prevalence=0.136847389453738
CHRONOLOGICAL_NATURAL     rows=593,780 attack=62,256 benign=531,524 prevalence=0.104846912998080

PLACEBO RESULT SCHEMA CENSUS

CHRONOLOGICAL_NATURAL     PLACEBO_ACTIVITY            RANKING_PREVALENCE_ONLY   PR=0.106554516270 ROC=0.516519011938
CHRONOLOGICAL_NATURAL     PLACEBO_COUNTS              RANKING_PREVALENCE_ONLY   PR=0.106566505817 ROC=0.516785367433
CHRONOLOGICAL_NATURAL     PLACEBO_IAT                 RANKING_PREVALENCE_ONLY   PR=0.105297783187 ROC=0.512494147793
CHRONOLOGICAL_NATURAL     PLACEBO_PACKET_S

In [16]:
# ==================================================================================================
# STAGE23-7A-R5 — FINAL STANDALONE STAGE23 SYNTHESIS / TABLES / FIGURES / MANUSCRIPT PACKAGE
#
# IMPORTANT
#   This is a fresh standalone implementation.
#   It does NOT use notebook history.
#
# FROZEN STAGE23-2 SCHEMA CENSUS
#   DATA_VALIDATION_ONLY    = 1
#   RANKING_PREVALENCE_ONLY = 9
#   BOTH                    = 0
#   NEITHER                 = 0
#
# SCIENTIFIC EXECUTION
#   ZERO MODEL FITS
#   ZERO MODEL INFERENCE
#   ZERO LIGHTGBM EXECUTION
#   ZERO XGBOOST EXECUTION
#   ZERO NEW SHAP
#   ZERO NEW BOOTSTRAP
#   ZERO MODEL FILE READS
#   ZERO NPZ READS
#   ZERO PARQUET READS
#   NO MAR1 / MAR2
#   REPOSITORY UNMODIFIED
#
# OUTPUT
#   /kaggle/working/stage23_7_final_synthesis
#
# STATUS ON SUCCESS
#   FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED
# ==================================================================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import textwrap

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ==================================================================================================
# 0. CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

ROOT = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
)


P0 = (
    ROOT
    / "stage23_0_protocol_lock"
)

P1 = (
    ROOT
    / "stage23_1_primary_ablation"
)

P2 = (
    ROOT
    / "stage23_2_placebo_ablation"
)

P3 = (
    ROOT
    / "stage23_3_stump_controls"
)

P4 = (
    ROOT
    / "stage23_4_uncertainty_analysis"
)

P5 = (
    ROOT
    / "stage23_5b_treeshap_proxy_absorption"
)

P6 = (
    ROOT
    / "stage23_6_attack_family_analysis"
    / "stage23_6d_attack_family_metrics"
)


EXPECTED_PARENT = (
    "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
)

EXPECTED_PARENT_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)


OUT = Path(
    "/kaggle/working/stage23_7_final_synthesis"
)

TMP = Path(
    "/kaggle/working/stage23_7_final_synthesis_tmp"
)


SPLITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]


PRIMARY_ORDER = [
    "FULL",
    "NO_DST_PORT",
    "NO_PORTS",
    "NO_INIT_FWD_WIN_BYTS",
    "NO_FWD_SEG_SIZE_MIN",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]


PRIMARY_ABLATIONS = (
    PRIMARY_ORDER[
        1:
    ]
)


PLACEBO_ORDER = [
    "PLACEBO_COUNTS",
    "PLACEBO_VOLUME_DIRECTION",
    "PLACEBO_IAT",
    "PLACEBO_PACKET_SIZE",
    "PLACEBO_ACTIVITY",
]


STUMP_ORDER = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]


EXPECTED_PLACEBO_SCHEMA_COUNTS = {
    "DATA_VALIDATION_ONLY":
        1,

    "RANKING_PREVALENCE_ONLY":
        9,

    "BOTH":
        0,

    "NEITHER":
        0,
}


DISPLAY = {
    "FULL":
        "FULL",

    "NO_DST_PORT":
        "No Dst Port",

    "NO_PORTS":
        "No Ports",

    "NO_INIT_FWD_WIN_BYTS":
        "No Init Fwd Win",

    "NO_FWD_SEG_SIZE_MIN":
        "No Fwd Seg Size Min",

    "NO_SUSPICIOUS_GROUP":
        "No Suspicious Group",

    "BEHAVIOR_ONLY":
        "Behavior Only",

    "PLACEBO_COUNTS":
        "Placebo Counts",

    "PLACEBO_VOLUME_DIRECTION":
        "Placebo Volume/Direction",

    "PLACEBO_IAT":
        "Placebo IAT",

    "PLACEBO_PACKET_SIZE":
        "Placebo Packet Size",

    "PLACEBO_ACTIVITY":
        "Placebo Activity",

    "RANDOM_NATURAL":
        "Random-natural",

    "CHRONOLOGICAL_NATURAL":
        "Chronological-natural",
}


SEP = "=" * 120


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def git(*args):

    p = subprocess.run(
        ["git", *args],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )

    return (
        p.stdout
        or ""
    ).strip()


def sha256_file(
    path,
    chunk=1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


SOURCE_FILES = {}


def register_source(
    path,
):

    path = Path(
        path
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing sealed source:\n{path}"
        )


    rel = str(
        path.relative_to(
            REPO
        )
    )


    SOURCE_FILES[
        rel
    ] = sha256_file(
        path
    )


def read_json(
    path,
):

    register_source(
        path
    )

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def read_csv(
    path,
):

    register_source(
        path
    )

    return pd.read_csv(
        path
    )


def write_json(
    path,
    obj,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def write_text(
    path,
    text,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    path.write_text(
        str(
            text
        ).rstrip()
        + "\n",
        encoding="utf-8",
    )


def ci_excludes_zero(
    lo,
    hi,
):

    return bool(
        float(
            lo
        )
        > 0.0

        or

        float(
            hi
        )
        < 0.0
    )


def row_for(
    df,
    split,
    subset,
):

    g = df[
        (
            df[
                "split"
            ]
            == split
        )
        &
        (
            df[
                "subset"
            ]
            == subset
        )
    ]


    if len(
        g
    ) != 1:

        raise RuntimeError(
            f"Row not unique: "
            f"{split}/{subset}; "
            f"n={len(g)}"
        )


    return g.iloc[
        0
    ]


def save_figure(
    fig,
    stem,
):

    stem = Path(
        stem
    )

    stem.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    fig.savefig(
        stem.with_suffix(
            ".png"
        ),
        dpi=300,
        bbox_inches="tight",
    )


    fig.savefig(
        stem.with_suffix(
            ".pdf"
        ),
        bbox_inches="tight",
    )


    plt.close(
        fig
    )


def latex_escape(
    value,
):

    s = str(
        value
    )


    for src, dst in [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
    ]:

        s = s.replace(
            src,
            dst,
        )


    return s


# ==================================================================================================
# 2. SEALED-PARENT PREFLIGHT
# ==================================================================================================

print(SEP)
print("STAGE23-7A-R5 — FINAL STANDALONE SYNTHESIS PACKAGE")
print(SEP)
print()


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

status = git(
    "status",
    "--porcelain",
)


if branch != "main":

    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage23 parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-6 parent tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before Stage23-7A-R5:\n"
        + status
    )


if OUT.exists():

    raise RuntimeError(
        "Final Stage23-7A package already exists.\n"
        "REFUSING overwrite:\n"
        f"{OUT}"
    )


# Failed earlier synthesis runs can leave TMP only.
if TMP.exists():

    shutil.rmtree(
        TMP
    )


TMP.mkdir(
    parents=True,
    exist_ok=False,
)


TABLES = (
    TMP
    / "tables"
)

FIGURES = (
    TMP
    / "figures"
)

MANUSCRIPT = (
    TMP
    / "manuscript"
)

AUDIT = (
    TMP
    / "audit"
)


for path in [
    TABLES,
    FIGURES,
    MANUSCRIPT,
    AUDIT,
]:

    path.mkdir(
        parents=True,
        exist_ok=False,
    )


print("[EXACT] branch    :", branch)
print("[EXACT] HEAD      :", head)
print("[EXACT] parent tag:", EXPECTED_PARENT_TAG)
print("[EXACT] worktree  : CLEAN")
print("[EXACT] Stage23 fit budget: 50 / 50 SEALED")
print()


# ==================================================================================================
# 3. FROZEN PROTOCOL
# ==================================================================================================

print(SEP)
print("VERIFY FROZEN STAGE23 PROTOCOL")
print(SEP)
print()


stopping_rule = read_json(
    P0
    / "stopping_rule.json"
)

figure_plan = read_json(
    P0
    / "figure_plan.json"
)

interpretation_matrix = read_json(
    P0
    / "interpretation_matrix.json"
)

metric_spec = read_json(
    P0
    / "metric_spec.json"
)


if len(
    stopping_rule[
        "complete_when"
    ]
) != 7:

    raise RuntimeError(
        "Frozen stopping condition count != 7."
    )


if (
    stopping_rule[
        "final_holdout"
    ]
    !=
    "Raw Mar1 and Mar2 permanently forbidden."
):

    raise RuntimeError(
        "Final-holdout rule changed."
    )


if (
    stopping_rule[
        "rolling_forward_analysis"
    ]
    !=
    "NOT_INCLUDED_IN_FINAL_STAGE23_PROTOCOL"
):

    raise RuntimeError(
        "Rolling-forward Stage23 status changed."
    )


if set(
    figure_plan.keys()
) != {
    "Figure_23_A",
    "Figure_23_B",
    "Figure_23_C",
    "Figure_23_D",
    "supplementary",
}:

    raise RuntimeError(
        "Frozen figure plan mismatch."
    )


if (
    metric_spec[
        "primary_ranking_metric"
    ]
    !=
    "PR_AUC"
):

    raise RuntimeError(
        "Frozen primary metric mismatch."
    )


if (
    metric_spec[
        "headline_interaction_metrics"
    ]
    !=
    [
        "PR_AUC",
        "ROC_AUC",
    ]
):

    raise RuntimeError(
        "Frozen interaction metric set mismatch."
    )


if float(
    metric_spec[
        "secondary_fixed_operating_point"
    ][
        "threshold"
    ]
) != 0.50:

    raise RuntimeError(
        "Frozen threshold mismatch."
    )


print("[EXACT] stopping conditions : 7")
print("[EXACT] frozen main figures : 4")
print("[EXACT] primary metric       : PR_AUC")
print("[EXACT] interaction metrics  : PR_AUC + ROC_AUC")
print("[EXACT] fixed threshold      : 0.50")
print("[EXACT] Mar1 / Mar2          : PERMANENTLY FORBIDDEN")
print()


# ==================================================================================================
# 4. SEALED PRIMARY RESULTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED PRIMARY ABLATION RESULTS")
print(SEP)
print()


primary_paths = sorted(
    P1.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    primary_paths
) != 12:

    raise RuntimeError(
        f"Expected 12 primary result JSONs; "
        f"found {len(primary_paths)}"
    )


primary_rows = []

validation_meta = {}

full_refs = {
    split:
        []
    for split
    in SPLITS
}


for path in primary_paths:

    d = read_json(
        path
    )


    split = d[
        "cell"
    ][
        "split"
    ]

    subset = d[
        "cell"
    ][
        "subset"
    ]


    if split not in SPLITS:

        raise RuntimeError(
            f"Unknown primary split: {split}"
        )


    if subset not in PRIMARY_ABLATIONS:

        raise RuntimeError(
            f"Unknown primary subset: {subset}"
        )


    ranking = d[
        "ranking_metrics"
    ]

    fixed = d[
        "fixed_threshold_0_50"
    ]

    val = d[
        "data"
    ][
        "validation"
    ]


    meta = {
        "rows":
            int(
                val[
                    "rows"
                ]
            ),

        "attack":
            int(
                val[
                    "attack"
                ]
            ),

        "benign":
            int(
                val[
                    "benign"
                ]
            ),

        "attack_prevalence":
            float(
                val[
                    "attack_prevalence"
                ]
            ),
    }


    if split in validation_meta:

        if validation_meta[
            split
        ] != meta:

            raise RuntimeError(
                f"Primary validation metadata mismatch "
                f"within {split}."
            )

    else:

        validation_meta[
            split
        ] = meta


    primary_rows.append(
        {
            "split":
                split,

            "subset":
                subset,

            "family":
                "PRIMARY",

            "feature_count":
                int(
                    d[
                        "cell"
                    ][
                        "feature_count"
                    ]
                ),

            "validation_rows":
                meta[
                    "rows"
                ],

            "support_attack":
                meta[
                    "attack"
                ],

            "support_benign":
                meta[
                    "benign"
                ],

            "attack_prevalence":
                meta[
                    "attack_prevalence"
                ],

            "pr_auc":
                float(
                    ranking[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                float(
                    ranking[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "roc_auc":
                float(
                    ranking[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    fixed[
                        "f1"
                    ]
                ),

            "recall_at_0_50":
                float(
                    fixed[
                        "recall"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    fixed[
                        "fpr"
                    ]
                ),

            "fnr_at_0_50":
                float(
                    fixed[
                        "fnr"
                    ]
                ),
        }
    )


    full_refs[
        split
    ].append(
        d[
            "full_reference"
        ]
    )


# Recover the two already-frozen FULL points from the six repeated references per split.
for split in SPLITS:

    refs = full_refs[
        split
    ]


    if len(
        refs
    ) != 6:

        raise RuntimeError(
            f"{split}: FULL reference count != 6."
        )


    canonical = refs[
        0
    ]


    for other in refs[
        1:
    ]:

        for key in [
            "pr_auc",
            "roc_auc",
            "f1_at_0_50",
            "recall_at_0_50",
            "fpr_at_0_50",
        ]:

            if float(
                other[
                    key
                ]
            ) != float(
                canonical[
                    key
                ]
            ):

                raise RuntimeError(
                    f"{split}: FULL reference mismatch: {key}"
                )


    prev = validation_meta[
        split
    ][
        "attack_prevalence"
    ]


    primary_rows.append(
        {
            "split":
                split,

            "subset":
                "FULL",

            "family":
                "FULL",

            "feature_count":
                70,

            "validation_rows":
                validation_meta[
                    split
                ][
                    "rows"
                ],

            "support_attack":
                validation_meta[
                    split
                ][
                    "attack"
                ],

            "support_benign":
                validation_meta[
                    split
                ][
                    "benign"
                ],

            "attack_prevalence":
                prev,

            "pr_auc":
                float(
                    canonical[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                (
                    float(
                        canonical[
                            "pr_auc"
                        ]
                    )
                    -
                    prev
                ),

            "roc_auc":
                float(
                    canonical[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    canonical[
                        "f1_at_0_50"
                    ]
                ),

            "recall_at_0_50":
                float(
                    canonical[
                        "recall_at_0_50"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    canonical[
                        "fpr_at_0_50"
                    ]
                ),

            "fnr_at_0_50":
                (
                    1.0
                    -
                    float(
                        canonical[
                            "recall_at_0_50"
                        ]
                    )
                ),
        }
    )


primary_df = pd.DataFrame(
    primary_rows
)


if len(
    primary_df
) != 14:

    raise RuntimeError(
        "Primary result synthesis rows != 14."
    )


for split in SPLITS:

    found = set(
        primary_df.loc[
            primary_df[
                "split"
            ]
            == split,
            "subset",
        ]
    )


    if found != set(
        PRIMARY_ORDER
    ):

        raise RuntimeError(
            f"{split}: primary subset set incomplete."
        )


split_order = {
    value:
        index
    for index, value
    in enumerate(
        SPLITS
    )
}


primary_order = {
    value:
        index
    for index, value
    in enumerate(
        PRIMARY_ORDER
    )
}


primary_df[
    "_split"
] = primary_df[
    "split"
].map(
    split_order
)


primary_df[
    "_subset"
] = primary_df[
    "subset"
].map(
    primary_order
)


primary_df = (
    primary_df
    .sort_values(
        [
            "_split",
            "_subset",
        ]
    )
    .drop(
        columns=[
            "_split",
            "_subset",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] primary result JSONs : 12")
print("[EXACT] primary table rows   : 14")
print("[EXACT] primary subsets      : 7 / 7")
print("[EXACT] splits               : 2 / 2")
print()


# ==================================================================================================
# 5. SEALED PLACEBO RESULTS — EXACT 1/9 SCHEMA HANDLING
# ==================================================================================================

print(SEP)
print("LOAD SEALED MATCHED-SIZE PLACEBO RESULTS")
print(SEP)
print()


placebo_paths = sorted(
    P2.glob(
        "*/*/stage23_*_result.json"
    )
)


if len(
    placebo_paths
) != 10:

    raise RuntimeError(
        f"Expected 10 placebo JSONs; "
        f"found {len(placebo_paths)}"
    )


placebo_rows = []

placebo_schema_counts = {
    "DATA_VALIDATION_ONLY":
        0,

    "RANKING_PREVALENCE_ONLY":
        0,

    "BOTH":
        0,

    "NEITHER":
        0,
}


seen_placebo_pairs = set()


for path in placebo_paths:

    d = read_json(
        path
    )


    split = d[
        "cell"
    ][
        "split"
    ]

    placebo = d[
        "cell"
    ][
        "placebo"
    ]


    if split not in SPLITS:

        raise RuntimeError(
            f"Unknown placebo split: {split}"
        )


    if placebo not in PLACEBO_ORDER:

        raise RuntimeError(
            f"Unknown placebo: {placebo}"
        )


    pair = (
        split,
        placebo,
    )


    if pair in seen_placebo_pairs:

        raise RuntimeError(
            f"Duplicate placebo/split pair: {pair}"
        )


    seen_placebo_pairs.add(
        pair
    )


    ranking = d[
        "ranking_metrics"
    ]

    fixed = d[
        "fixed_threshold_0_50"
    ]


    data_validation = (
        d.get(
            "data",
            {}
        ).get(
            "validation"
        )
    )


    has_data = (
        data_validation
        is not None
    )

    has_ranking_prev = (
        "attack_prevalence"
        in ranking
    )


    if (
        has_data
        and has_ranking_prev
    ):

        schema = "BOTH"

    elif has_data:

        schema = "DATA_VALIDATION_ONLY"

    elif has_ranking_prev:

        schema = "RANKING_PREVALENCE_ONLY"

    else:

        schema = "NEITHER"


    placebo_schema_counts[
        schema
    ] += 1


    if schema == "NEITHER":

        raise RuntimeError(
            f"{path.name}: no sealed validation provenance."
        )


    expected = validation_meta[
        split
    ]


    # Verify every provenance field that exists.
    if has_data:

        for key in [
            "rows",
            "attack",
            "benign",
        ]:

            if int(
                data_validation[
                    key
                ]
            ) != int(
                expected[
                    key
                ]
            ):

                raise RuntimeError(
                    f"{path.name}: data.validation.{key} mismatch."
                )


        if not np.isclose(
            float(
                data_validation[
                    "attack_prevalence"
                ]
            ),
            float(
                expected[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: data.validation prevalence mismatch."
            )


    if has_ranking_prev:

        if not np.isclose(
            float(
                ranking[
                    "attack_prevalence"
                ]
            ),
            float(
                expected[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: ranking prevalence mismatch."
            )


    if (
        has_data
        and has_ranking_prev
    ):

        if not np.isclose(
            float(
                data_validation[
                    "attack_prevalence"
                ]
            ),
            float(
                ranking[
                    "attack_prevalence"
                ]
            ),
            rtol=0.0,
            atol=1e-15,
        ):

            raise RuntimeError(
                f"{path.name}: sealed prevalence fields disagree."
            )


    placebo_rows.append(
        {
            "split":
                split,

            "subset":
                placebo,

            "family":
                "PLACEBO",

            "source_schema":
                schema,

            "feature_count":
                int(
                    d[
                        "cell"
                    ][
                        "feature_count"
                    ]
                ),

            "validation_rows":
                int(
                    expected[
                        "rows"
                    ]
                ),

            "support_attack":
                int(
                    expected[
                        "attack"
                    ]
                ),

            "support_benign":
                int(
                    expected[
                        "benign"
                    ]
                ),

            "attack_prevalence":
                float(
                    expected[
                        "attack_prevalence"
                    ]
                ),

            "pr_auc":
                float(
                    ranking[
                        "pr_auc"
                    ]
                ),

            "pr_auc_minus_attack_prevalence":
                float(
                    ranking[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "roc_auc":
                float(
                    ranking[
                        "roc_auc"
                    ]
                ),

            "f1_at_0_50":
                float(
                    fixed[
                        "f1"
                    ]
                ),

            "recall_at_0_50":
                float(
                    fixed[
                        "recall"
                    ]
                ),

            "fpr_at_0_50":
                float(
                    fixed[
                        "fpr"
                    ]
                ),

            "fnr_at_0_50":
                float(
                    fixed[
                        "fnr"
                    ]
                ),
        }
    )


expected_pairs = {
    (
        split,
        placebo,
    )

    for split
    in SPLITS

    for placebo
    in PLACEBO_ORDER
}


if seen_placebo_pairs != expected_pairs:

    raise RuntimeError(
        "Frozen placebo/split pair set mismatch."
    )


if (
    placebo_schema_counts
    !=
    EXPECTED_PLACEBO_SCHEMA_COUNTS
):

    raise RuntimeError(
        "Stage23-2 schema distribution changed.\n"
        f"expected={EXPECTED_PLACEBO_SCHEMA_COUNTS}\n"
        f"actual={placebo_schema_counts}"
    )


placebo_df = pd.DataFrame(
    placebo_rows
)


if len(
    placebo_df
) != 10:

    raise RuntimeError(
        "Placebo synthesis rows != 10."
    )


placebo_order = {
    value:
        index
    for index, value
    in enumerate(
        PLACEBO_ORDER
    )
}


placebo_df[
    "_split"
] = placebo_df[
    "split"
].map(
    split_order
)


placebo_df[
    "_subset"
] = placebo_df[
    "subset"
].map(
    placebo_order
)


placebo_df = (
    placebo_df
    .sort_values(
        [
            "_split",
            "_subset",
        ]
    )
    .drop(
        columns=[
            "_split",
            "_subset",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] placebo result JSONs : 10")
print("[EXACT] placebo pairs        : 10 / 10")
print("[EXACT] placebo subsets      : 5 / 5")
print(
    "[EXACT] DATA_VALIDATION_ONLY:",
    placebo_schema_counts[
        "DATA_VALIDATION_ONLY"
    ],
)
print(
    "[EXACT] RANKING_PREVALENCE_ONLY:",
    placebo_schema_counts[
        "RANKING_PREVALENCE_ONLY"
    ],
)
print("[EXACT] BOTH                 : 0")
print("[EXACT] NEITHER              : 0")
print()


# ==================================================================================================
# 6. SEALED UNCERTAINTY SUMMARY
# ==================================================================================================

print(SEP)
print("LOAD SEALED UNCERTAINTY SUMMARY")
print(SEP)
print()


uncertainty = read_json(
    P4
    / "stage23_4_uncertainty_summary.json"
)


expected_uncertainty = set(
    PRIMARY_ABLATIONS
    +
    PLACEBO_ORDER
)


if set(
    uncertainty[
        "comparisons"
    ].keys()
) != expected_uncertainty:

    raise RuntimeError(
        "Frozen uncertainty comparison set mismatch."
    )


unc_rows = []


for comparison in (
    PRIMARY_ABLATIONS
    +
    PLACEBO_ORDER
):

    c = uncertainty[
        "comparisons"
    ][
        comparison
    ]


    pts = c[
        "complete_validation_point_estimates"
    ]

    ci = c[
        "bootstrap_95_percentile_ci"
    ]


    row = {
        "comparison":
            comparison,

        "family":
            c[
                "family"
            ],

        "random_pr_auc_removal_penalty":
            float(
                pts[
                    "random"
                ][
                    "pr_auc_removal_penalty"
                ]
            ),

        "random_pr_auc_ci_low":
            float(
                ci[
                    "random_pr_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "random_pr_auc_ci_high":
            float(
                ci[
                    "random_pr_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "chronological_pr_auc_removal_penalty":
            float(
                pts[
                    "chronological"
                ][
                    "pr_auc_removal_penalty"
                ]
            ),

        "chronological_pr_auc_ci_low":
            float(
                ci[
                    "chronological_pr_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "chronological_pr_auc_ci_high":
            float(
                ci[
                    "chronological_pr_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "pr_auc_interaction":
            float(
                pts[
                    "interaction"
                ][
                    "pr_auc"
                ]
            ),

        "pr_auc_interaction_ci_low":
            float(
                ci[
                    "pr_auc_shortcut_interaction"
                ][
                    "lower_2_5"
                ]
            ),

        "pr_auc_interaction_ci_high":
            float(
                ci[
                    "pr_auc_shortcut_interaction"
                ][
                    "upper_97_5"
                ]
            ),

        "random_roc_auc_removal_penalty":
            float(
                pts[
                    "random"
                ][
                    "roc_auc_removal_penalty"
                ]
            ),

        "random_roc_auc_ci_low":
            float(
                ci[
                    "random_roc_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "random_roc_auc_ci_high":
            float(
                ci[
                    "random_roc_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "chronological_roc_auc_removal_penalty":
            float(
                pts[
                    "chronological"
                ][
                    "roc_auc_removal_penalty"
                ]
            ),

        "chronological_roc_auc_ci_low":
            float(
                ci[
                    "chronological_roc_auc_removal_penalty"
                ][
                    "lower_2_5"
                ]
            ),

        "chronological_roc_auc_ci_high":
            float(
                ci[
                    "chronological_roc_auc_removal_penalty"
                ][
                    "upper_97_5"
                ]
            ),

        "roc_auc_interaction":
            float(
                pts[
                    "interaction"
                ][
                    "roc_auc"
                ]
            ),

        "roc_auc_interaction_ci_low":
            float(
                ci[
                    "roc_auc_shortcut_interaction"
                ][
                    "lower_2_5"
                ]
            ),

        "roc_auc_interaction_ci_high":
            float(
                ci[
                    "roc_auc_shortcut_interaction"
                ][
                    "upper_97_5"
                ]
            ),
    }


    row[
        "pr_auc_interaction_ci_excludes_zero"
    ] = ci_excludes_zero(
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
    )


    row[
        "roc_auc_interaction_ci_excludes_zero"
    ] = ci_excludes_zero(
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
    )


    unc_rows.append(
        row
    )


unc_df = pd.DataFrame(
    unc_rows
)


if len(
    unc_df
) != 11:

    raise RuntimeError(
        "Uncertainty comparison count != 11."
    )


unc_primary = unc_df[
    unc_df[
        "family"
    ]
    == "PRIMARY"
].copy()


unc_placebo = unc_df[
    unc_df[
        "family"
    ]
    == "PLACEBO"
].copy()


if len(
    unc_primary
) != 6:

    raise RuntimeError(
        "Primary uncertainty count != 6."
    )


if len(
    unc_placebo
) != 5:

    raise RuntimeError(
        "Placebo uncertainty count != 5."
    )


unc_primary[
    "_order"
] = unc_primary[
    "comparison"
].map(
    {
        value:
            index
        for index, value
        in enumerate(
            PRIMARY_ABLATIONS
        )
    }
)


unc_primary = (
    unc_primary
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


unc_placebo[
    "_order"
] = unc_placebo[
    "comparison"
].map(
    placebo_order
)


unc_placebo = (
    unc_placebo
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


print("[EXACT] uncertainty comparisons : 11 / 11")
print("[EXACT] primary comparisons     : 6")
print("[EXACT] placebo comparisons     : 5")
print("[EXACT] bootstrap reruns         : 0")
print()


# ==================================================================================================
# 7. SEALED STUMP CONTROLS
# ==================================================================================================

print(SEP)
print("LOAD SEALED STUMP CONTROLS")
print(SEP)
print()


stump_summary = read_json(
    P3
    / "stage23_3_stump_controls_summary.json"
)


if int(
    stump_summary[
        "fit_accounting"
    ][
        "stage23_after"
    ]
) != 50:

    raise RuntimeError(
        "Stump block does not finish at Stage23 fit 50."
    )


if int(
    stump_summary[
        "fit_accounting"
    ][
        "stump_fits_completed"
    ]
) != 6:

    raise RuntimeError(
        "Stump completion != 6."
    )


if set(
    stump_summary[
        "paired_random_to_chronological"
    ].keys()
) != set(
    STUMP_ORDER
):

    raise RuntimeError(
        "Frozen stump feature set mismatch."
    )


stump_rows = []


for feature in STUMP_ORDER:

    x = stump_summary[
        "paired_random_to_chronological"
    ][
        feature
    ]


    rm = x[
        "random_metrics"
    ]

    cm = x[
        "chronological_metrics"
    ]

    dg = x[
        "random_to_chronological_degradation"
    ]

    rt = x[
        "random_tree_structure"
    ]

    ct = x[
        "chronological_tree_structure"
    ]


    stump_rows.append(
        {
            "feature":
                feature,

            "random_pr_auc":
                float(
                    rm[
                        "PR_AUC"
                    ]
                ),

            "chronological_pr_auc":
                float(
                    cm[
                        "PR_AUC"
                    ]
                ),

            "random_minus_chronological_pr_auc":
                float(
                    dg[
                        "PR_AUC"
                    ]
                ),

            "random_roc_auc":
                float(
                    rm[
                        "ROC_AUC"
                    ]
                ),

            "chronological_roc_auc":
                float(
                    cm[
                        "ROC_AUC"
                    ]
                ),

            "random_minus_chronological_roc_auc":
                float(
                    dg[
                        "ROC_AUC"
                    ]
                ),

            "random_f1_at_0_50":
                float(
                    rm[
                        "f1"
                    ]
                ),

            "chronological_f1_at_0_50":
                float(
                    cm[
                        "f1"
                    ]
                ),

            "random_minus_chronological_f1":
                float(
                    dg[
                        "f1"
                    ]
                ),

            "random_recall_at_0_50":
                float(
                    rm[
                        "recall"
                    ]
                ),

            "chronological_recall_at_0_50":
                float(
                    cm[
                        "recall"
                    ]
                ),

            "random_fpr_at_0_50":
                float(
                    rm[
                        "fpr"
                    ]
                ),

            "chronological_fpr_at_0_50":
                float(
                    cm[
                        "fpr"
                    ]
                ),

            "random_training_median":
                float(
                    rt[
                        "missing_imputation_median"
                    ]
                ),

            "chronological_training_median":
                float(
                    ct[
                        "missing_imputation_median"
                    ]
                ),

            "random_split_threshold":
                float(
                    rt[
                        "split_threshold"
                    ]
                ),

            "chronological_split_threshold":
                float(
                    ct[
                        "split_threshold"
                    ]
                ),

            "random_attack_branch":
                str(
                    rt[
                        "branch_predicting_attack"
                    ]
                ),

            "chronological_attack_branch":
                str(
                    ct[
                        "branch_predicting_attack"
                    ]
                ),
        }
    )


stump_df = pd.DataFrame(
    stump_rows
)


if len(
    stump_df
) != 3:

    raise RuntimeError(
        "Paired stump table rows != 3."
    )


print("[EXACT] stump controls      : 6 / 6")
print("[EXACT] feature/split pairs : 6")
print("[EXACT] Stage23 fit budget  : 50 / 50 SEALED")
print()


# ==================================================================================================
# 8. SEALED SHAP OUTPUTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED SHAP PROXY-ABSORPTION OUTPUTS")
print(SEP)
print()


shap_summary = read_csv(
    P5
    / "stage23_5b_proxy_absorption_summary.csv"
)


consensus = read_csv(
    P5
    / "stage23_5b_descriptive_consensus_importance.csv"
)


component = read_csv(
    P5
    / "stage23_5b_component_importance.csv"
)


expected_component_columns = {
    "split",
    "family",
    "subset",
    "component",
    "feature",
    "mean_abs_shap",
    "normalized_importance_share",
    "rank",
}


if set(
    component.columns
) != expected_component_columns:

    raise RuntimeError(
        "Stage23-5 component importance schema changed."
    )


if len(
    shap_summary
) != 66:

    raise RuntimeError(
        "SHAP reporting rows != 66."
    )


if (
    shap_summary[
        [
            "split",
            "subset",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 22
):

    raise RuntimeError(
        "SHAP proxy comparisons != 22."
    )


if (
    component[
        [
            "split",
            "subset",
            "component",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 48
):

    raise RuntimeError(
        "Component TreeSHAP model count != 48."
    )


if (
    consensus[
        [
            "split",
            "subset",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    != 24
):

    raise RuntimeError(
        "Descriptive consensus set count != 24."
    )


shap_primary = shap_summary[
    (
        shap_summary[
            "family"
        ]
        == "PRIMARY"
    )
    &
    (
        shap_summary[
            "reporting"
        ]
        ==
        "DESCRIPTIVE_CONSENSUS_0.5_0.5"
    )
].copy()


if len(
    shap_primary
) != 12:

    raise RuntimeError(
        "Primary consensus SHAP comparison rows != 12."
    )


# Derive largest positive retained-feature normalized-share gain from sealed consensus values.
gainer_rows = []


for split in SPLITS:

    full = consensus[
        (
            consensus[
                "split"
            ]
            == split
        )
        &
        (
            consensus[
                "subset"
            ]
            == "FULL"
        )
    ][
        [
            "feature",
            "consensus_normalized_importance_share",
        ]
    ].copy()


    if len(
        full
    ) != 70:

        raise RuntimeError(
            f"{split}: FULL consensus feature count != 70."
        )


    full = full.rename(
        columns={
            "consensus_normalized_importance_share":
                "full_share"
        }
    )


    for subset in PRIMARY_ABLATIONS:

        sub = consensus[
            (
                consensus[
                    "split"
                ]
                == split
            )
            &
            (
                consensus[
                    "subset"
                ]
                == subset
            )
        ][
            [
                "feature",
                "consensus_normalized_importance_share",
            ]
        ].copy()


        merged = sub.merge(
            full,
            on="feature",
            how="inner",
            validate="one_to_one",
        )


        merged[
            "share_gain"
        ] = (
            merged[
                "consensus_normalized_importance_share"
            ]
            -
            merged[
                "full_share"
            ]
        )


        merged = merged.sort_values(
            [
                "share_gain",
                "feature",
            ],
            ascending=[
                False,
                True,
            ],
        )


        top = merged.iloc[
            0
        ]


        gainer_rows.append(
            {
                "split":
                    split,

                "subset":
                    subset,

                "top_retained_consensus_share_gainer":
                    str(
                        top[
                            "feature"
                        ]
                    ),

                "top_retained_consensus_share_gain":
                    float(
                        top[
                            "share_gain"
                        ]
                    ),
            }
        )


gainer_df = pd.DataFrame(
    gainer_rows
)


shap_primary = shap_primary.merge(
    gainer_df,
    on=[
        "split",
        "subset",
    ],
    how="left",
    validate="one_to_one",
)


component_top20 = component[
    component[
        "rank"
    ]
    <= 20
].copy()


if len(
    component_top20
) != 960:

    raise RuntimeError(
        "Expected 48×20 = 960 component SHAP top-20 rows; "
        f"found {len(component_top20)}"
    )


print("[EXACT] SHAP proxy comparisons     : 22 / 22")
print("[EXACT] SHAP reporting rows        : 66")
print("[EXACT] component TreeSHAP models  : 48 / 48")
print("[EXACT] descriptive consensus sets : 24")
print("[EXACT] component top-20 rows      : 960")
print("[EXACT] new SHAP computation       : 0")
print()


# ==================================================================================================
# 9. SEALED ATTACK-FAMILY RESULTS
# ==================================================================================================

print(SEP)
print("LOAD SEALED ATTACK-FAMILY RESULTS")
print(SEP)
print()


attack_df = read_csv(
    P6
    / "stage23_6d_attack_family_metrics.csv"
)


if len(
    attack_df
) != 182:

    raise RuntimeError(
        "Attack-family metric row count != 182."
    )


if attack_df[
    "attack_family"
].nunique() != 13:

    raise RuntimeError(
        "Attack-family universe != 13."
    )


if set(
    attack_df[
        "subset"
    ]
) != set(
    PRIMARY_ORDER
):

    raise RuntimeError(
        "Attack-family subset set mismatch."
    )


if set(
    attack_df[
        "split"
    ]
) != set(
    SPLITS
):

    raise RuntimeError(
        "Attack-family split set mismatch."
    )


full_attack = attack_df[
    attack_df[
        "subset"
    ]
    == "FULL"
].copy()


random_full_attack = full_attack[
    full_attack[
        "split"
    ]
    == "RANDOM_NATURAL"
].copy()


chrono_full_attack = full_attack[
    full_attack[
        "split"
    ]
    == "CHRONOLOGICAL_NATURAL"
].copy()


random_interp = random_full_attack[
    random_full_attack[
        "interpretation_status"
    ]
    == "INTERPRETABLE"
]


chrono_interp = chrono_full_attack[
    chrono_full_attack[
        "interpretation_status"
    ]
    == "INTERPRETABLE"
]


if len(
    random_interp
) != 11:

    raise RuntimeError(
        "RANDOM interpretable attack-family count != 11."
    )


if len(
    chrono_interp
) != 1:

    raise RuntimeError(
        "CHRONO interpretable attack-family count != 1."
    )


chrono_positive = chrono_full_attack[
    chrono_full_attack[
        "support_attack"
    ]
    > 0
]


if len(
    chrono_positive
) != 1:

    raise RuntimeError(
        "Chronological validation contains >1 positive attack family."
    )


if (
    str(
        chrono_positive.iloc[
            0
        ][
            "attack_family"
        ]
    )
    !=
    "Infilteration"
):

    raise RuntimeError(
        "Chronological positive family is not Infilteration."
    )


random_attack_support = int(
    random_full_attack[
        "support_attack"
    ].sum()
)


chrono_attack_support = int(
    chrono_full_attack[
        "support_attack"
    ].sum()
)


if (
    random_attack_support
    !=
    validation_meta[
        "RANDOM_NATURAL"
    ][
        "attack"
    ]
):

    raise RuntimeError(
        "RANDOM family support does not sum to validation attacks."
    )


if (
    chrono_attack_support
    !=
    validation_meta[
        "CHRONOLOGICAL_NATURAL"
    ][
        "attack"
    ]
):

    raise RuntimeError(
        "CHRONO family support does not sum to validation attacks."
    )


print("[EXACT] family metric rows       : 182")
print("[EXACT] frozen attack families   : 13")
print("[EXACT] RANDOM interpretable     : 11")
print("[EXACT] CHRONO interpretable     : 1")
print("[EXACT] CHRONO positive family   : Infilteration only")
print()


# ==================================================================================================
# 10. PUBLICATION TABLES
# ==================================================================================================

print(SEP)
print("GENERATE PUBLICATION TABLES")
print(SEP)
print()


# Main Table 23-1.
wide_rows = []


for subset in PRIMARY_ORDER:

    rr = row_for(
        primary_df,
        "RANDOM_NATURAL",
        subset,
    )

    cc = row_for(
        primary_df,
        "CHRONOLOGICAL_NATURAL",
        subset,
    )


    wide_rows.append(
        {
            "subset":
                subset,

            "feature_count":
                int(
                    rr[
                        "feature_count"
                    ]
                ),

            "random_pr_auc":
                float(
                    rr[
                        "pr_auc"
                    ]
                ),

            "chronological_pr_auc":
                float(
                    cc[
                        "pr_auc"
                    ]
                ),

            "random_pr_auc_minus_prevalence":
                float(
                    rr[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "chronological_pr_auc_minus_prevalence":
                float(
                    cc[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),

            "random_roc_auc":
                float(
                    rr[
                        "roc_auc"
                    ]
                ),

            "chronological_roc_auc":
                float(
                    cc[
                        "roc_auc"
                    ]
                ),

            "random_f1_at_0_50":
                float(
                    rr[
                        "f1_at_0_50"
                    ]
                ),

            "chronological_f1_at_0_50":
                float(
                    cc[
                        "f1_at_0_50"
                    ]
                ),

            "random_recall_at_0_50":
                float(
                    rr[
                        "recall_at_0_50"
                    ]
                ),

            "chronological_recall_at_0_50":
                float(
                    cc[
                        "recall_at_0_50"
                    ]
                ),

            "random_fpr_at_0_50":
                float(
                    rr[
                        "fpr_at_0_50"
                    ]
                ),

            "chronological_fpr_at_0_50":
                float(
                    cc[
                        "fpr_at_0_50"
                    ]
                ),
        }
    )


primary_wide = pd.DataFrame(
    wide_rows
)


primary_wide.to_csv(
    TABLES
    / "table_23_1_primary_subset_performance.csv",
    index=False,
)


# Main Table 23-2.
unc_primary.to_csv(
    TABLES
    / "table_23_2_primary_removal_penalties_and_interactions.csv",
    index=False,
)


# Main Table 23-3.
stump_df.to_csv(
    TABLES
    / "table_23_3_single_feature_stump_controls.csv",
    index=False,
)


# Main Table 23-4.
shap_table = shap_primary[
    [
        "split",
        "subset",
        "jaccard_at_10_vs_full",
        "jaccard_at_20_vs_full",
        "spearman_common_retained",
        "new_top10_entrants",
        "features_leaving_top10",
        "number_positive_share_gainers",
        "top_retained_consensus_share_gainer",
        "top_retained_consensus_share_gain",
    ]
].copy()


shap_table[
    "_split"
] = shap_table[
    "split"
].map(
    split_order
)


shap_table[
    "_subset"
] = shap_table[
    "subset"
].map(
    {
        value:
            index
        for index, value
        in enumerate(
            PRIMARY_ABLATIONS
        )
    }
)


shap_table = (
    shap_table
    .sort_values(
        [
            "_split",
            "_subset",
        ]
    )
    .drop(
        columns=[
            "_split",
            "_subset",
        ]
    )
    .reset_index(
        drop=True
    )
)


shap_table.to_csv(
    TABLES
    / "table_23_4_shap_proxy_absorption_consensus.csv",
    index=False,
)


# Main Table 23-5.
attack_support = full_attack[
    [
        "split",
        "attack_family",
        "interpretation_status",
        "support_attack",
        "support_benign",
        "PR_AUC",
        "ROC_AUC",
        "precision_at_0_50",
        "recall_at_0_50",
        "f1_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()


attack_support.to_csv(
    TABLES
    / "table_23_5_attack_family_support_and_full_metrics.csv",
    index=False,
)


# Supplementary tables.
placebo_df.to_csv(
    TABLES
    / "table_23_s1_placebo_subset_performance.csv",
    index=False,
)


unc_placebo.to_csv(
    TABLES
    / "table_23_s2_placebo_interactions.csv",
    index=False,
)


attack_df.to_csv(
    TABLES
    / "table_23_s3_attack_family_all_frozen_metrics.csv",
    index=False,
)


component_top20.sort_values(
    [
        "split",
        "subset",
        "component",
        "rank",
    ]
).to_csv(
    TABLES
    / "table_23_s4_component_specific_shap_top20.csv",
    index=False,
)


behavior_table = primary_df[
    primary_df[
        "subset"
    ].isin(
        [
            "FULL",
            "BEHAVIOR_ONLY",
        ]
    )
][
    [
        "split",
        "subset",
        "attack_prevalence",
        "pr_auc",
        "pr_auc_minus_attack_prevalence",
        "roc_auc",
        "f1_at_0_50",
        "recall_at_0_50",
        "fpr_at_0_50",
        "fnr_at_0_50",
    ]
].copy()


behavior_table.to_csv(
    TABLES
    / "table_23_s5_behavior_restricted_operating_metrics.csv",
    index=False,
)


EXPECTED_TABLE_FILES = {
    "table_23_1_primary_subset_performance.csv",
    "table_23_2_primary_removal_penalties_and_interactions.csv",
    "table_23_3_single_feature_stump_controls.csv",
    "table_23_4_shap_proxy_absorption_consensus.csv",
    "table_23_5_attack_family_support_and_full_metrics.csv",
    "table_23_s1_placebo_subset_performance.csv",
    "table_23_s2_placebo_interactions.csv",
    "table_23_s3_attack_family_all_frozen_metrics.csv",
    "table_23_s4_component_specific_shap_top20.csv",
    "table_23_s5_behavior_restricted_operating_metrics.csv",
}


actual_table_files = {
    path.name
    for path in TABLES.glob(
        "*.csv"
    )
}


if actual_table_files != EXPECTED_TABLE_FILES:

    raise RuntimeError(
        "Publication table set mismatch.\n"
        f"missing={sorted(EXPECTED_TABLE_FILES - actual_table_files)}\n"
        f"extra={sorted(actual_table_files - EXPECTED_TABLE_FILES)}"
    )


print("[EXACT] publication/supp tables: 10 / 10")
print()


# ==================================================================================================
# 11. FIGURE STYLE
# ==================================================================================================

plt.rcParams.update(
    {
        "font.size":
            8.5,

        "axes.titlesize":
            9.5,

        "axes.labelsize":
            8.5,

        "xtick.labelsize":
            7.3,

        "ytick.labelsize":
            7.3,

        "legend.fontsize":
            7.3,

        "figure.titlesize":
            10.0,

        "axes.linewidth":
            0.8,

        "lines.linewidth":
            1.4,

        "lines.markersize":
            4.5,

        "pdf.fonttype":
            42,
    }
)


colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()[
    "color"
]


# ==================================================================================================
# 12. FIGURE 23-A — SUBSET × SPLIT INTERACTION
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.15,
    ),
)


x = np.arange(
    len(
        PRIMARY_ORDER
    )
)


labels = [
    DISPLAY[
        subset
    ]
    for subset in PRIMARY_ORDER
]


for split in SPLITS:

    values = [
        float(
            row_for(
                primary_df,
                split,
                subset,
            )[
                "pr_auc"
            ]
        )

        for subset
        in PRIMARY_ORDER
    ]


    line = axes[
        0
    ].plot(
        x,
        values,
        marker="o",
        label=DISPLAY[
            split
        ],
    )[
        0
    ]


    axes[
        0
    ].axhline(
        validation_meta[
            split
        ][
            "attack_prevalence"
        ],
        linestyle=":",
        linewidth=0.9,
        color=line.get_color(),
        alpha=0.75,
    )


axes[
    0
].set_title(
    "(a) PR-AUC"
)

axes[
    0
].set_ylabel(
    "PR-AUC"
)

axes[
    0
].set_xticks(
    x
)

axes[
    0
].set_xticklabels(
    labels,
    rotation=38,
    ha="right",
)

axes[
    0
].set_ylim(
    0.0,
    1.03,
)

axes[
    0
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    0
].legend(
    frameon=False
)


for split in SPLITS:

    values = [
        float(
            row_for(
                primary_df,
                split,
                subset,
            )[
                "roc_auc"
            ]
        )

        for subset
        in PRIMARY_ORDER
    ]


    axes[
        1
    ].plot(
        x,
        values,
        marker="o",
        label=DISPLAY[
            split
        ],
    )


axes[
    1
].axhline(
    0.5,
    linestyle=":",
    linewidth=0.9,
    color="0.35",
)

axes[
    1
].set_title(
    "(b) ROC-AUC"
)

axes[
    1
].set_ylabel(
    "ROC-AUC"
)

axes[
    1
].set_xticks(
    x
)

axes[
    1
].set_xticklabels(
    labels,
    rotation=38,
    ha="right",
)

axes[
    1
].set_ylim(
    0.0,
    1.03,
)

axes[
    1
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    1
].legend(
    frameon=False
)


fig.suptitle(
    "Stage23 subset × split interaction"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_a_subset_split_interaction",
)


# ==================================================================================================
# 13. FIGURE 23-B — REMOVAL PENALTIES / UNCERTAINTY / INTERACTION
# ==================================================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        7.25,
        6.2,
    ),
)


y = np.arange(
    len(
        PRIMARY_ABLATIONS
    )
)


ylabels = [
    DISPLAY[
        subset
    ]
    for subset in PRIMARY_ABLATIONS
]


# PR penalties.
ax = axes[
    0,
    0
]


for i, row in unc_primary.iterrows():

    yr = i - 0.12
    yc = i + 0.12


    ax.hlines(
        yr,
        row[
            "random_pr_auc_ci_low"
        ],
        row[
            "random_pr_auc_ci_high"
        ],
        color=colors[
            0
        ],
    )

    ax.plot(
        row[
            "random_pr_auc_removal_penalty"
        ],
        yr,
        "o",
        color=colors[
            0
        ],
    )


    ax.hlines(
        yc,
        row[
            "chronological_pr_auc_ci_low"
        ],
        row[
            "chronological_pr_auc_ci_high"
        ],
        color=colors[
            1
        ],
    )

    ax.plot(
        row[
            "chronological_pr_auc_removal_penalty"
        ],
        yc,
        "o",
        color=colors[
            1
        ],
    )


ax.axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_xlabel(
    "FULL − ablated PR-AUC"
)

ax.set_title(
    "(a) PR-AUC removal penalty"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# PR interaction.
ax = axes[
    0,
    1
]


for i, row in unc_primary.iterrows():

    ax.hlines(
        i,
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    ax.plot(
        row[
            "pr_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


ax.axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_xlabel(
    "Random penalty − chronological penalty"
)

ax.set_title(
    "(b) PR-AUC shortcut interaction"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# ROC penalties.
ax = axes[
    1,
    0
]


for i, row in unc_primary.iterrows():

    yr = i - 0.12
    yc = i + 0.12


    ax.hlines(
        yr,
        row[
            "random_roc_auc_ci_low"
        ],
        row[
            "random_roc_auc_ci_high"
        ],
        color=colors[
            0
        ],
    )

    ax.plot(
        row[
            "random_roc_auc_removal_penalty"
        ],
        yr,
        "o",
        color=colors[
            0
        ],
    )


    ax.hlines(
        yc,
        row[
            "chronological_roc_auc_ci_low"
        ],
        row[
            "chronological_roc_auc_ci_high"
        ],
        color=colors[
            1
        ],
    )

    ax.plot(
        row[
            "chronological_roc_auc_removal_penalty"
        ],
        yc,
        "o",
        color=colors[
            1
        ],
    )


ax.axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_xlabel(
    "FULL − ablated ROC-AUC"
)

ax.set_title(
    "(c) ROC-AUC removal penalty"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


# ROC interaction.
ax = axes[
    1,
    1
]


for i, row in unc_primary.iterrows():

    ax.hlines(
        i,
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    ax.plot(
        row[
            "roc_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


ax.axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ylabels
)

ax.invert_yaxis()

ax.set_xlabel(
    "Random penalty − chronological penalty"
)

ax.set_title(
    "(d) ROC-AUC shortcut interaction"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


fig.suptitle(
    "Stage23 removal penalties and frozen paired-bootstrap uncertainty"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_b_removal_penalties_and_interaction",
)


# ==================================================================================================
# 14. FIGURE 23-C — SINGLE-FEATURE STUMP DEGRADATION
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.15,
    ),
)


x = np.arange(
    len(
        STUMP_ORDER
    )
)

width = 0.34


axes[
    0
].bar(
    x
    -
    width
    / 2,
    stump_df[
        "random_pr_auc"
    ],
    width,
    label="Random-natural",
)


axes[
    0
].bar(
    x
    +
    width
    / 2,
    stump_df[
        "chronological_pr_auc"
    ],
    width,
    label="Chronological-natural",
)


axes[
    0
].set_xticks(
    x
)

axes[
    0
].set_xticklabels(
    STUMP_ORDER,
    rotation=25,
    ha="right",
)

axes[
    0
].set_ylabel(
    "PR-AUC"
)

axes[
    0
].set_title(
    "(a) Single-feature PR-AUC"
)

axes[
    0
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    0
].legend(
    frameon=False
)


axes[
    1
].bar(
    x
    -
    width
    / 2,
    stump_df[
        "random_roc_auc"
    ],
    width,
    label="Random-natural",
)


axes[
    1
].bar(
    x
    +
    width
    / 2,
    stump_df[
        "chronological_roc_auc"
    ],
    width,
    label="Chronological-natural",
)


axes[
    1
].axhline(
    0.5,
    linestyle=":",
    color="0.35",
)


axes[
    1
].set_xticks(
    x
)

axes[
    1
].set_xticklabels(
    STUMP_ORDER,
    rotation=25,
    ha="right",
)

axes[
    1
].set_ylabel(
    "ROC-AUC"
)

axes[
    1
].set_title(
    "(b) Single-feature ROC-AUC"
)

axes[
    1
].grid(
    axis="y",
    alpha=0.25,
)

axes[
    1
].legend(
    frameon=False
)


fig.suptitle(
    "Stage23 depth-1 stump degradation across split regimes"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_c_single_feature_stump_degradation",
)


# ==================================================================================================
# 15. FIGURE 23-D — SHAP PROXY ABSORPTION
# ==================================================================================================

jaccard_matrix = np.zeros(
    (
        len(
            PRIMARY_ABLATIONS
        ),
        len(
            SPLITS
        ),
    ),
    dtype=float,
)


gain_matrix = np.zeros_like(
    jaccard_matrix
)


gainer_matrix = np.empty(
    jaccard_matrix.shape,
    dtype=object,
)


for i, subset in enumerate(
    PRIMARY_ABLATIONS
):

    for j, split in enumerate(
        SPLITS
    ):

        r = shap_primary[
            (
                shap_primary[
                    "split"
                ]
                == split
            )
            &
            (
                shap_primary[
                    "subset"
                ]
                == subset
            )
        ]


        if len(
            r
        ) != 1:

            raise RuntimeError(
                f"SHAP row not unique: "
                f"{split}/{subset}"
            )


        r = r.iloc[
            0
        ]


        jaccard_matrix[
            i,
            j
        ] = float(
            r[
                "jaccard_at_10_vs_full"
            ]
        )


        gain_matrix[
            i,
            j
        ] = float(
            r[
                "top_retained_consensus_share_gain"
            ]
        )


        gainer_matrix[
            i,
            j
        ] = str(
            r[
                "top_retained_consensus_share_gainer"
            ]
        )


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        4.4,
    ),
)


im0 = axes[
    0
].imshow(
    jaccard_matrix,
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
    cmap="Blues",
)


axes[
    0
].set_xticks(
    np.arange(
        len(
            SPLITS
        )
    )
)

axes[
    0
].set_xticklabels(
    [
        DISPLAY[
            split
        ]
        for split in SPLITS
    ],
    rotation=15,
)

axes[
    0
].set_yticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    0
].set_yticklabels(
    [
        DISPLAY[
            subset
        ]
        for subset in PRIMARY_ABLATIONS
    ]
)

axes[
    0
].set_title(
    "(a) Top-10 overlap with FULL"
)


for i in range(
    jaccard_matrix.shape[
        0
    ]
):

    for j in range(
        jaccard_matrix.shape[
            1
        ]
    ):

        axes[
            0
        ].text(
            j,
            i,
            f"{jaccard_matrix[i, j]:.3f}",
            ha="center",
            va="center",
            fontsize=7,
        )


fig.colorbar(
    im0,
    ax=axes[
        0
    ],
    fraction=0.046,
    pad=0.04,
    label="Jaccard@10",
)


gain_max = max(
    float(
        np.nanmax(
            gain_matrix
        )
    ),
    1e-12,
)


im1 = axes[
    1
].imshow(
    gain_matrix,
    aspect="auto",
    vmin=0.0,
    vmax=gain_max,
    cmap="magma",
)


axes[
    1
].set_xticks(
    np.arange(
        len(
            SPLITS
        )
    )
)

axes[
    1
].set_xticklabels(
    [
        DISPLAY[
            split
        ]
        for split in SPLITS
    ],
    rotation=15,
)

axes[
    1
].set_yticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    1
].set_yticklabels(
    [
        DISPLAY[
            subset
        ]
        for subset in PRIMARY_ABLATIONS
    ]
)

axes[
    1
].set_title(
    "(b) Largest retained-feature share gain"
)


for i in range(
    gain_matrix.shape[
        0
    ]
):

    for j in range(
        gain_matrix.shape[
            1
        ]
    ):

        wrapped = "\n".join(
            textwrap.wrap(
                str(
                    gainer_matrix[
                        i,
                        j
                    ]
                ),
                width=15,
            )
        )


        axes[
            1
        ].text(
            j,
            i,
            f"{wrapped}\n"
            f"{gain_matrix[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.5,
        )


fig.colorbar(
    im1,
    ax=axes[
        1
    ],
    fraction=0.046,
    pad=0.04,
    label="Δ normalized consensus share",
)


fig.suptitle(
    "Stage23 SHAP proxy absorption — descriptive 0.5/0.5 normalized consensus"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_d_shap_proxy_absorption",
)


# ==================================================================================================
# 16. SUPPLEMENTARY FIGURE 23-S1 — PLACEBO INTERACTIONS
# ==================================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        7.25,
        3.4,
    ),
)


y = np.arange(
    len(
        PLACEBO_ORDER
    )
)


labels = [
    DISPLAY[
        placebo
    ]
    for placebo in PLACEBO_ORDER
]


for i, row in unc_placebo.iterrows():

    axes[
        0
    ].hlines(
        i,
        row[
            "pr_auc_interaction_ci_low"
        ],
        row[
            "pr_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    axes[
        0
    ].plot(
        row[
            "pr_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


axes[
    0
].axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

axes[
    0
].set_yticks(
    y
)

axes[
    0
].set_yticklabels(
    labels
)

axes[
    0
].invert_yaxis()

axes[
    0
].set_xlabel(
    "Random penalty − chronological penalty"
)

axes[
    0
].set_title(
    "(a) PR-AUC interaction"
)

axes[
    0
].grid(
    axis="x",
    alpha=0.25,
)


for i, row in unc_placebo.iterrows():

    axes[
        1
    ].hlines(
        i,
        row[
            "roc_auc_interaction_ci_low"
        ],
        row[
            "roc_auc_interaction_ci_high"
        ],
        color=colors[
            2
        ],
    )

    axes[
        1
    ].plot(
        row[
            "roc_auc_interaction"
        ],
        i,
        "o",
        color=colors[
            2
        ],
    )


axes[
    1
].axvline(
    0.0,
    linestyle=":",
    color="0.35",
)

axes[
    1
].set_yticks(
    y
)

axes[
    1
].set_yticklabels(
    labels
)

axes[
    1
].invert_yaxis()

axes[
    1
].set_xlabel(
    "Random penalty − chronological penalty"
)

axes[
    1
].set_title(
    "(b) ROC-AUC interaction"
)

axes[
    1
].grid(
    axis="x",
    alpha=0.25,
)


fig.suptitle(
    "Stage23 matched-size placebo ablation interactions"
)

fig.tight_layout()


save_figure(
    fig,
    FIGURES
    / "figure_23_s1_placebo_interactions",
)


# ==================================================================================================
# 17. SUPPLEMENTARY FIGURE 23-S2 — ATTACK-FAMILY DELTA F1
# ==================================================================================================

random_families = random_interp[
    "attack_family"
].astype(
    str
).tolist()


chrono_families = chrono_interp[
    "attack_family"
].astype(
    str
).tolist()


if chrono_families != [
    "Infilteration"
]:

    raise RuntimeError(
        "Unexpected chronological interpretable family set."
    )


rmat = (
    attack_df[
        (
            attack_df[
                "split"
            ]
            == "RANDOM_NATURAL"
        )
        &
        (
            attack_df[
                "interpretation_status"
            ]
            == "INTERPRETABLE"
        )
        &
        (
            attack_df[
                "subset"
            ].isin(
                PRIMARY_ABLATIONS
            )
        )
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(
        index=random_families,
        columns=PRIMARY_ABLATIONS,
    )
)


cmat = (
    attack_df[
        (
            attack_df[
                "split"
            ]
            == "CHRONOLOGICAL_NATURAL"
        )
        &
        (
            attack_df[
                "interpretation_status"
            ]
            == "INTERPRETABLE"
        )
        &
        (
            attack_df[
                "subset"
            ].isin(
                PRIMARY_ABLATIONS
            )
        )
    ]
    .pivot(
        index="attack_family",
        columns="subset",
        values="delta_f1_FULL_minus_ablated",
    )
    .reindex(
        index=chrono_families,
        columns=PRIMARY_ABLATIONS,
    )
)


if rmat.isna().any().any():

    raise RuntimeError(
        "RANDOM attack-family heatmap contains NaN."
    )


if cmat.isna().any().any():

    raise RuntimeError(
        "CHRONO attack-family heatmap contains NaN."
    )


all_values = np.concatenate(
    [
        rmat.to_numpy(
            dtype=float
        ).ravel(),

        cmat.to_numpy(
            dtype=float
        ).ravel(),
    ]
)


vlim = max(
    float(
        np.max(
            np.abs(
                all_values
            )
        )
    ),
    1e-12,
)


norm = TwoSlopeNorm(
    vmin=-vlim,
    vcenter=0.0,
    vmax=vlim,
)


fig, axes = plt.subplots(
    2,
    1,
    figsize=(
        7.25,
        6.2,
    ),
    gridspec_kw={
        "height_ratios":
            [
                5.0,
                1.2,
            ]
    },
)


im = axes[
    0
].imshow(
    rmat.to_numpy(
        dtype=float
    ),
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)


axes[
    0
].set_xticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    0
].set_xticklabels(
    [
        DISPLAY[
            subset
        ]
        for subset in PRIMARY_ABLATIONS
    ],
    rotation=30,
    ha="right",
)

axes[
    0
].set_yticks(
    np.arange(
        len(
            random_families
        )
    )
)

axes[
    0
].set_yticklabels(
    random_families
)

axes[
    0
].set_title(
    "(a) Random-natural — interpretable families"
)


for i in range(
    rmat.shape[
        0
    ]
):

    for j in range(
        rmat.shape[
            1
        ]
    ):

        axes[
            0
        ].text(
            j,
            i,
            f"{rmat.iloc[i, j]:+.3f}",
            ha="center",
            va="center",
            fontsize=5.5,
        )


axes[
    1
].imshow(
    cmat.to_numpy(
        dtype=float
    ),
    aspect="auto",
    cmap="coolwarm",
    norm=norm,
)


axes[
    1
].set_xticks(
    np.arange(
        len(
            PRIMARY_ABLATIONS
        )
    )
)

axes[
    1
].set_xticklabels(
    [
        DISPLAY[
            subset
        ]
        for subset in PRIMARY_ABLATIONS
    ],
    rotation=30,
    ha="right",
)

axes[
    1
].set_yticks(
    [
        0
    ]
)

axes[
    1
].set_yticklabels(
    chrono_families
)

axes[
    1
].set_title(
    "(b) Chronological-natural — interpretable family"
)


for j in range(
    cmat.shape[
        1
    ]
):

    axes[
        1
    ].text(
        j,
        0,
        f"{cmat.iloc[0, j]:+.3f}",
        ha="center",
        va="center",
        fontsize=6.0,
    )


cbar = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    fraction=0.025,
    pad=0.02,
)


cbar.set_label(
    "ΔF1 = FULL − ablated at threshold 0.50"
)


fig.suptitle(
    "Stage23 attack-family-conditioned degradation"
)


fig.subplots_adjust(
    left=0.22,
    right=0.90,
    top=0.91,
    bottom=0.14,
    hspace=0.78,
)


save_figure(
    fig,
    FIGURES
    / "figure_23_s2_attack_family_delta_f1",
)


# ==================================================================================================
# 18. VERIFY EXACT FIGURE SET
# ==================================================================================================

EXPECTED_FIGURE_BASES = {
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
    "figure_23_s1_placebo_interactions",
    "figure_23_s2_attack_family_delta_f1",
}


actual_png = {
    path.stem
    for path in FIGURES.glob(
        "*.png"
    )
}


actual_pdf = {
    path.stem
    for path in FIGURES.glob(
        "*.pdf"
    )
}


if actual_png != EXPECTED_FIGURE_BASES:

    raise RuntimeError(
        "PNG figure set mismatch."
    )


if actual_pdf != EXPECTED_FIGURE_BASES:

    raise RuntimeError(
        "PDF figure set mismatch."
    )


print()
print("[EXACT] frozen main figure bases   : 4 / 4")
print("[EXACT] supplementary figure bases : 2")
print("[EXACT] PNG/PDF figure files       : 12")
print()


# ==================================================================================================
# 19. FINAL SCIENTIFIC SYNTHESIS
# ==================================================================================================

full_r = row_for(
    primary_df,
    "RANDOM_NATURAL",
    "FULL",
)

full_c = row_for(
    primary_df,
    "CHRONOLOGICAL_NATURAL",
    "FULL",
)

behavior_r = row_for(
    primary_df,
    "RANDOM_NATURAL",
    "BEHAVIOR_ONLY",
)

behavior_c = row_for(
    primary_df,
    "CHRONOLOGICAL_NATURAL",
    "BEHAVIOR_ONLY",
)


random_prev = float(
    validation_meta[
        "RANDOM_NATURAL"
    ][
        "attack_prevalence"
    ]
)


chrono_prev = float(
    validation_meta[
        "CHRONOLOGICAL_NATURAL"
    ][
        "attack_prevalence"
    ]
)


primary_pr_sig = unc_primary.loc[
    unc_primary[
        "pr_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


primary_roc_sig = unc_primary.loc[
    unc_primary[
        "roc_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


placebo_pr_sig = unc_placebo.loc[
    unc_placebo[
        "pr_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


placebo_roc_sig = unc_placebo.loc[
    unc_placebo[
        "roc_auc_interaction_ci_excludes_zero"
    ],
    "comparison",
].tolist()


synthesis_md = f"""
# Stage23 Final Scientific Synthesis

## Frozen scope

Stage23 is a development-only validation-sensitivity and shortcut-feature audit.
It contains seven frozen primary representations, five matched-size placebo removals,
six frozen depth-1 stump controls, paired stratified bootstrap uncertainty,
component-specific TreeSHAP proxy-absorption analysis, and frozen attack-family
conditioning.

Raw Mar1 and Mar2 remain permanently forbidden. Stage23 does not create a new
untouched final holdout.

## Full-model validation-regime contrast

### RANDOM_NATURAL

- PR-AUC: {float(full_r["pr_auc"]):.12f}
- ROC-AUC: {float(full_r["roc_auc"]):.12f}
- Attack prevalence: {random_prev:.12f}
- PR-AUC minus attack prevalence:
  {float(full_r["pr_auc_minus_attack_prevalence"]):.12f}

### CHRONOLOGICAL_NATURAL

- PR-AUC: {float(full_c["pr_auc"]):.12f}
- ROC-AUC: {float(full_c["roc_auc"]):.12f}
- Attack prevalence: {chrono_prev:.12f}
- PR-AUC minus attack prevalence:
  {float(full_c["pr_auc_minus_attack_prevalence"]):.12f}

The FULL ensemble therefore shows extreme validation-regime sensitivity. Random-natural
validation is highly discriminative, whereas chronological-natural ranking is close to
the relevant no-skill references.

## Primary ablations

The frozen shortcut interaction is:

(FULL_RANDOM - ABLATED_RANDOM)
-
(FULL_CHRONOLOGICAL - ABLATED_CHRONOLOGICAL)

A positive interaction means random validation benefits disproportionately from the
tested information relative to chronological validation. A negative interaction means
chronological validation depends more strongly on the tested information.

Primary PR-AUC interactions whose frozen 95% bootstrap CIs exclude zero:

{chr(10).join("- " + value for value in primary_pr_sig)}

Primary ROC-AUC interactions whose frozen 95% bootstrap CIs exclude zero:

{chr(10).join("- " + value for value in primary_roc_sig)}

The directions are not uniform across subsets or ranking metrics. Therefore Stage23
does not support a binary leakage/not-leakage interpretation.

## Matched-size placebo context

Placebo PR-AUC interactions whose frozen CIs exclude zero:

{chr(10).join("- " + value for value in placebo_pr_sig)}

Placebo ROC-AUC interactions whose frozen CIs exclude zero:

{chr(10).join("- " + value for value in placebo_roc_sig)}

Matched-size placebo removals also produce non-zero interactions and CI exclusion.
Thus CI exclusion from zero is not unique to the pre-specified suspicious groups and
cannot by itself establish leakage.

## Single-feature stump controls

Dst Port, Init Fwd Win Byts, and Fwd Seg Size Min individually discriminate much more
strongly under RANDOM_NATURAL than under CHRONOLOGICAL_NATURAL. Their forward-temporal
degradation is consistent with split-specific or shortcut-like information that
transfers poorly. It is not proof of leakage or causality.

## SHAP proxy absorption

TreeSHAP remains component-specific for LightGBM and XGBoost. Figure 23-D uses only a
descriptive equal-weight consensus of normalized component mean-absolute SHAP shares.
It is not exact SHAP for the probability-averaged ensemble.

Ablation changes retained-feature ranks and normalized importance shares. These shifts
are consistent with proxy absorption but do not establish causal substitution.

## Behavior-restricted representation

RANDOM_NATURAL BEHAVIOR_ONLY:

- PR-AUC: {float(behavior_r["pr_auc"]):.12f}
- ROC-AUC: {float(behavior_r["roc_auc"]):.12f}
- F1@0.50: {float(behavior_r["f1_at_0_50"]):.12f}

CHRONOLOGICAL_NATURAL BEHAVIOR_ONLY:

- PR-AUC: {float(behavior_c["pr_auc"]):.12f}
- ROC-AUC: {float(behavior_c["roc_auc"]):.12f}
- F1@0.50: {float(behavior_c["f1_at_0_50"]):.12f}

The behavior-restricted representation does not retain strong chronological
discrimination under the frozen Stage23 protocol. This does not mean BEHAVIOR_ONLY
equals real-world deployment performance.

## Attack-family context

RANDOM_NATURAL contains {random_attack_support:,} attack validation rows and
11 families meeting the frozen minimum support of 100.

CHRONOLOGICAL_NATURAL contains {chrono_attack_support:,} attack validation rows.
All chronological attack support belongs to Infilteration under the frozen family
mapping; the other 12 development families have zero chronological attack support.

The random-versus-chronological comparison therefore combines temporal change with
a major attack-family composition change. Stage23 cannot attribute the entire split
gap to a single feature or leakage mechanism.

## Consolidated conclusion

The frozen Stage23 evidence supports the following bounded conclusions:

1. The evaluated ensemble is highly sensitive to validation regime.
2. Several individually discriminative cues transfer poorly to chronological validation.
3. Feature-removal interactions are metric- and subset-dependent.
4. Matched-size placebo interactions caution against treating statistical significance
   as leakage evidence.
5. SHAP redistribution is consistent with proxy absorption, not causal proof.
6. Chronological attack support is entirely Infilteration, materially limiting
   attack-family comparability.
7. The overall evidence supports validation sensitivity and poor temporal transfer,
   not a claim that any tested feature is proven leakage.

## Claims explicitly prohibited by the frozen interpretation matrix

{chr(10).join("- " + value for value in interpretation_matrix["prohibited_claims"])}

## Governance

- Stage23 fit budget: 50 / 50 SEALED.
- Additional fits authorized: 0.
- Model fits in Stage23-7A-R5: 0.
- Model inference: 0.
- LightGBM execution: 0.
- XGBoost execution: 0.
- New SHAP computation: 0.
- New bootstrap sampling: 0.
- Model files read: 0.
- Probability NPZ files read: 0.
- Parquet files read: 0.
- Raw Mar1 accessed: NO.
- Raw Mar2 accessed: NO.
- Repository modified: NO.
"""


write_text(
    TMP
    / "stage23_7_final_scientific_synthesis.md",
    synthesis_md,
)


# ==================================================================================================
# 20. IEEE / LATEX MANUSCRIPT INTEGRATION
# ==================================================================================================

results_tex = rf"""
% Stage23 frozen manuscript integration.
% Generated from sealed Stage23 JSON/CSV evidence only.

\subsection{{Validation-Safe Shortcut-Feature Audit}}

The full 70-feature ensemble achieved a random-natural PR-AUC of
{float(full_r["pr_auc"]):.6f} and ROC-AUC of {float(full_r["roc_auc"]):.6f},
compared with chronological-natural PR-AUC {float(full_c["pr_auc"]):.6f}
and ROC-AUC {float(full_c["roc_auc"]):.6f}. The corresponding chronological
attack prevalence was {chrono_prev:.6f}, placing the chronological PR-AUC only
{float(full_c["pr_auc_minus_attack_prevalence"]):.6f} above its prevalence
reference.

Frozen primary ablations revealed metric-dependent split interactions. Some
pre-specified removals disproportionately reduced random-natural ranking, whereas
others affected chronological ranking more strongly. Matched-size placebo removals
also yielded non-zero interactions and bootstrap intervals excluding zero. Therefore,
interaction significance alone does not identify leakage.

The frozen depth-1 controls for Dst Port, Init Fwd Win Byts, and Fwd Seg Size Min
showed substantially stronger discrimination under random-natural validation than
under chronological-natural validation. These observations are consistent with
split-specific discriminative cues that transfer poorly, but they do not establish
causality.

Component-specific TreeSHAP analyses further demonstrated importance redistribution
after feature removal. Equal-weight normalized LightGBM/XGBoost shares are used only
as a descriptive consensus and are not exact SHAP values for the averaged-probability
ensemble.

The behavior-restricted representation retained strong random-natural performance but
did not preserve chronological discrimination. Attack-family conditioning further
showed that all {chrono_attack_support:,} chronological attack rows belonged to
Infilteration, while random-natural validation contained 11 families meeting the frozen
interpretive support threshold.

Taken together, Stage23 supports a bounded conclusion of strong validation sensitivity
and poor forward-temporal transfer. It does not prove that any tested feature is leakage.
"""


write_text(
    MANUSCRIPT
    / "stage23_results_integration.tex",
    results_tex,
)


table_tex = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Stage23 primary representation performance under the frozen random-natural and chronological-natural validation regimes.}",
    r"\label{tab:stage23-primary}",
    r"\small",
    r"\begin{tabular}{lrrrr}",
    r"\hline",
    r"Subset & Random PR-AUC & Chronological PR-AUC & Random ROC-AUC & Chronological ROC-AUC \\",
    r"\hline",
]


for _, row in primary_wide.iterrows():

    table_tex.append(
        f"{latex_escape(row['subset'])} & "
        f"{row['random_pr_auc']:.6f} & "
        f"{row['chronological_pr_auc']:.6f} & "
        f"{row['random_roc_auc']:.6f} & "
        f"{row['chronological_roc_auc']:.6f} \\\\"
    )


table_tex += [
    r"\hline",
    r"\end{tabular}",
    r"\end{table*}",
]


write_text(
    MANUSCRIPT
    / "stage23_main_tables.tex",
    "\n".join(
        table_tex
    ),
)


captions_tex = r"""
% Frozen Stage23 publication figure captions.

\paragraph{Figure 23-A.}
Subset-by-split interaction across the seven primary representations. PR-AUC is shown
with split-specific attack prevalence as the no-skill reference; ROC-AUC uses 0.5.

\paragraph{Figure 23-B.}
FULL-minus-ablated PR-AUC and ROC-AUC penalties, frozen paired stratified bootstrap
percentile 95\% confidence intervals, and random-minus-chronological shortcut
interactions.

\paragraph{Figure 23-C.}
Frozen depth-1 single-feature controls for Dst Port, Init Fwd Win Byts, and
Fwd Seg Size Min under random-natural and chronological-natural validation.

\paragraph{Figure 23-D.}
SHAP proxy-absorption summary. The displayed equal-weight normalized component
importance consensus is descriptive only and is not exact ensemble SHAP.
"""


write_text(
    MANUSCRIPT
    / "stage23_figure_captions.tex",
    captions_tex,
)


EXPECTED_MANUSCRIPT_FILES = {
    "stage23_results_integration.tex",
    "stage23_main_tables.tex",
    "stage23_figure_captions.tex",
}


actual_manuscript_files = {
    path.name
    for path in MANUSCRIPT.glob(
        "*.tex"
    )
}


if actual_manuscript_files != EXPECTED_MANUSCRIPT_FILES:

    raise RuntimeError(
        "Manuscript integration file set mismatch."
    )


print("[EXACT] manuscript integration files: 3 / 3")
print()


# ==================================================================================================
# 21. CONSOLIDATED SCIENTIFIC AUDIT / STOPPING RULE
# ==================================================================================================

closure_conditions = [
    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                0
            ],

        "satisfied":
            True,

        "evidence":
            "7 primary subsets × 2 frozen natural splits complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                1
            ],

        "satisfied":
            True,

        "evidence":
            "6/6 frozen depth-1 stump controls complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                2
            ],

        "satisfied":
            True,

        "evidence":
            "5 matched-size placebo subsets × 2 splits complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                3
            ],

        "satisfied":
            True,

        "evidence":
            "48 component TreeSHAP explanations and 22 proxy subset/split comparisons complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                4
            ],

        "satisfied":
            True,

        "evidence":
            "13 frozen attack families and 182 frozen family metric rows complete",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                5
            ],

        "satisfied":
            True,

        "evidence":
            "Stage23-7 consolidated scientific audit generated",
    },

    {
        "condition":
            stopping_rule[
                "complete_when"
            ][
                6
            ],

        "satisfied":
            True,

        "evidence":
            "Figures 23-A through 23-D, supplementary outputs, tables, and manuscript integration generated",
    },
]


if not all(
    item[
        "satisfied"
    ]
    for item in closure_conditions
):

    raise RuntimeError(
        "Stage23 stopping rule not fully satisfied."
    )


audit = {
    "stage":
        "Stage23-7A-R5",

    "status":
        "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent": {
        "commit":
            EXPECTED_PARENT,

        "tag":
            EXPECTED_PARENT_TAG,
    },

    "frozen_stage23_2_schema_distribution":
        placebo_schema_counts,

    "counts": {
        "primary_result_jsons":
            12,

        "primary_subset_split_rows":
            14,

        "placebo_result_jsons":
            10,

        "placebo_subset_split_rows":
            10,

        "uncertainty_comparisons":
            11,

        "stump_controls":
            6,

        "shap_proxy_comparisons":
            22,

        "shap_component_models":
            48,

        "component_shap_top20_rows":
            960,

        "attack_family_metric_rows":
            182,

        "attack_families":
            13,

        "random_interpretable_attack_families":
            11,

        "chronological_interpretable_attack_families":
            1,

        "publication_tables":
            10,

        "main_figure_bases":
            4,

        "supplementary_figure_bases":
            2,

        "figure_files":
            12,

        "manuscript_files":
            3,

        "stopping_conditions_satisfied":
            7,
    },

    "headline_full_results": {
        "RANDOM_NATURAL": {
            "pr_auc":
                float(
                    full_r[
                        "pr_auc"
                    ]
                ),

            "roc_auc":
                float(
                    full_r[
                        "roc_auc"
                    ]
                ),

            "attack_prevalence":
                random_prev,

            "pr_auc_minus_attack_prevalence":
                float(
                    full_r[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),
        },

        "CHRONOLOGICAL_NATURAL": {
            "pr_auc":
                float(
                    full_c[
                        "pr_auc"
                    ]
                ),

            "roc_auc":
                float(
                    full_c[
                        "roc_auc"
                    ]
                ),

            "attack_prevalence":
                chrono_prev,

            "pr_auc_minus_attack_prevalence":
                float(
                    full_c[
                        "pr_auc_minus_attack_prevalence"
                    ]
                ),
        },
    },

    "attack_family_context": {
        "random_attack_support":
            random_attack_support,

        "chronological_attack_support":
            chrono_attack_support,

        "chronological_positive_attack_families":
            [
                "Infilteration"
            ],
    },

    "scientific_conclusion":
        (
            "Strong validation sensitivity and poor forward-temporal transfer "
            "are supported; no tested feature is established as leakage or as "
            "a causal source of the split-performance gap."
        ),

    "prohibited_claims":
        interpretation_matrix[
            "prohibited_claims"
        ],

    "stopping_rule_evidence":
        closure_conditions,

    "governance": {
        "stage23_fit_budget":
            "50 / 50 SEALED",

        "additional_model_fits_authorized":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_execution":
            0,

        "xgboost_execution":
            0,

        "new_shap_computation":
            0,

        "new_bootstrap_sampling":
            0,

        "model_files_read":
            0,

        "probability_npz_files_read":
            0,

        "parquet_files_read":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "repository_modified":
            False,
    },

    "next_authorized_action":
        "ZERO_FIT_FINAL_STAGE23_CLOSURE_SEAL_AND_PUSH",
}


write_json(
    AUDIT
    / "stage23_7_consolidated_scientific_audit.json",
    audit,
)


# ==================================================================================================
# 22. SOURCE MANIFEST — JSON/CSV ONLY
# ==================================================================================================

for rel in SOURCE_FILES:

    lower = rel.lower()


    forbidden = (
        lower.endswith(
            ".parquet"
        )
        or
        lower.endswith(
            ".npz"
        )
        or
        lower.endswith(
            ".joblib"
        )
        or
        "model.txt"
        in lower
        or
        "xgboost_model"
        in lower
    )


    if forbidden:

        raise RuntimeError(
            "Forbidden Stage23-7A source type read:\n"
            f"{rel}"
        )


source_manifest = {
    "stage":
        "Stage23-7A-R5",

    "execution_parent_commit":
        EXPECTED_PARENT,

    "source_policy":
        "SEALED_REPOSITORY_JSON_AND_CSV_ONLY",

    "source_file_count":
        len(
            SOURCE_FILES
        ),

    "source_files":
        [
            {
                "path":
                    rel,

                "sha256":
                    digest,
            }

            for rel, digest
            in sorted(
                SOURCE_FILES.items()
            )
        ],

    "model_files_read":
        0,

    "probability_npz_files_read":
        0,

    "parquet_files_read":
        0,

    "raw_mar1_accessed":
        False,

    "raw_mar2_accessed":
        False,
}


write_json(
    AUDIT
    / "stage23_7_source_manifest.json",
    source_manifest,
)


# ==================================================================================================
# 23. README + EXECUTION STATE
# ==================================================================================================

readme = f"""
# Stage23 Final Synthesis Package

Status: **FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED**

Parent commit: `{EXPECTED_PARENT}`

Parent tag: `{EXPECTED_PARENT_TAG}`

## Frozen main figures

1. Figure 23-A — subset × split interaction
2. Figure 23-B — removal penalties and shortcut interaction
3. Figure 23-C — single-feature stump degradation
4. Figure 23-D — SHAP proxy absorption

All figures are exported as 300-dpi PNG and vector PDF.

## Supplementary outputs

- matched-size placebo interaction figure
- attack-family-conditioned degradation figure
- complete component-specific SHAP top-20 table
- behavior-restricted operating metrics
- complete attack-family metric table

## Stage23-2 historical schema

The frozen placebo result artifacts contain two historical layouts:

- DATA_VALIDATION_ONLY: 1
- RANKING_PREVALENCE_ONLY: 9
- BOTH: 0
- NEITHER: 0

Both layouts were verified against the exact frozen split populations before synthesis.

## Governance

- Stage23 fit budget: 50 / 50 SEALED
- New fits: 0
- Model inference: 0
- New SHAP: 0
- New bootstrap: 0
- Model files read: 0
- Probability NPZ files read: 0
- Parquet reads: 0
- Raw Mar1: not accessed
- Raw Mar2: not accessed

Next authorized action: zero-fit final Stage23 closure seal and push.
"""


write_text(
    TMP
    / "README.md",
    readme,
)


execution_state = {
    "stage":
        "Stage23-7A-R5",

    "status":
        "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent_commit":
        EXPECTED_PARENT,

    "frozen_stage23_2_schema_distribution":
        placebo_schema_counts,

    "counts": {
        "publication_tables":
            10,

        "main_figure_bases":
            4,

        "supplementary_figure_bases":
            2,

        "figure_files":
            12,

        "manuscript_files":
            3,

        "stopping_conditions":
            7,
    },

    "governance": {
        "stage23_fit_budget_before":
            50,

        "stage23_fit_budget_after":
            50,

        "model_fits":
            0,

        "model_inference":
            0,

        "lightgbm_execution":
            0,

        "xgboost_execution":
            0,

        "new_shap":
            0,

        "new_bootstrap":
            0,

        "model_files_read":
            0,

        "npz_reads":
            0,

        "parquet_reads":
            0,

        "raw_mar1_accessed":
            False,

        "raw_mar2_accessed":
            False,

        "repository_modified":
            False,
    },

    "next_authorized_action":
        "ZERO_FIT_FINAL_STAGE23_CLOSURE_SEAL_AND_PUSH",
}


write_json(
    TMP
    / "execution_state.json",
    execution_state,
)


# ==================================================================================================
# 24. PACKAGE CHECKSUMS
# ==================================================================================================

checksum_path = (
    TMP
    / "checksums.sha256"
)


files_to_hash = sorted(
    path
    for path in TMP.rglob(
        "*"
    )
    if (
        path.is_file()
        and path != checksum_path
    )
)


checksum_lines = []


for path in files_to_hash:

    rel = path.relative_to(
        TMP
    )


    checksum_lines.append(
        f"{sha256_file(path)}  {rel.as_posix()}"
    )


write_text(
    checksum_path,
    "\n".join(
        checksum_lines
    ),
)


# Immediate verification.
for line in checksum_path.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue


    digest, rel = line.split(
        "  ",
        1,
    )


    target = (
        TMP
        / rel
    )


    if not target.is_file():

        raise RuntimeError(
            f"Checksum target missing: {rel}"
        )


    actual = sha256_file(
        target
    )


    if actual != digest:

        raise RuntimeError(
            "Checksum verification failed:\n"
            f"{rel}\n"
            f"expected={digest}\n"
            f"actual={actual}"
        )


package_manifest_sha = sha256_file(
    checksum_path
)


# ==================================================================================================
# 25. REPOSITORY MUST STILL BE UNTOUCHED
# ==================================================================================================

if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_PARENT:

    raise RuntimeError(
        "Repository HEAD changed during synthesis."
    )


repo_status_after = git(
    "status",
    "--porcelain",
)


if repo_status_after:

    raise RuntimeError(
        "Repository modified during synthesis:\n"
        + repo_status_after
    )


# ==================================================================================================
# 26. ATOMIC FINALIZATION OUTSIDE REPOSITORY
# ==================================================================================================

TMP.rename(
    OUT
)


if not OUT.is_dir():

    raise RuntimeError(
        "Failed to finalize Stage23-7A package."
    )


final_checksum_path = (
    OUT
    / "checksums.sha256"
)


if sha256_file(
    final_checksum_path
) != package_manifest_sha:

    raise RuntimeError(
        "Checksum manifest changed during finalization."
    )


for line in final_checksum_path.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue


    digest, rel = line.split(
        "  ",
        1,
    )


    actual = sha256_file(
        OUT
        / rel
    )


    if actual != digest:

        raise RuntimeError(
            f"Final artifact changed during rename: {rel}"
        )


# ==================================================================================================
# 27. FINAL REPORT
# ==================================================================================================

print()
print(SEP)
print("STAGE23-7A — FINAL SYNTHESIS / TABLES / FIGURES COMPLETE — UNSEALED")
print(SEP)
print()

print("Output:")
print(
    " ",
    OUT,
)

print()
print("Parent commit:")
print(
    " ",
    EXPECTED_PARENT,
)

print()
print("Package checksum manifest SHA256:")
print(
    " ",
    package_manifest_sha,
)

print()
print("STAGE23-2 SEALED SCHEMA")
print("  DATA_VALIDATION_ONLY        : 1")
print("  RANKING_PREVALENCE_ONLY     : 9")
print("  BOTH                        : 0")
print("  NEITHER                     : 0")

print()
print("SCIENTIFIC INPUT ACCOUNTING")
print("  primary subsets              : 7 / 7")
print("  primary split results        : 14")
print("  placebo subsets              : 5 / 5")
print("  placebo split results        : 10")
print("  stump controls               : 6 / 6")
print("  uncertainty comparisons      : 11 / 11")
print("  SHAP component models        : 48 / 48")
print("  SHAP proxy comparisons       : 22 / 22")
print("  attack-family metric rows    : 182")
print("  frozen attack families       : 13")

print()
print("PUBLICATION OUTPUTS")
print("  frozen main figure bases     : 4 / 4")
print("  supplementary figure bases   : 2")
print("  PNG/PDF figure files         : 12")
print("  publication/supp tables      : 10")
print("  manuscript integration files : 3")
print("  stopping-rule conditions     : 7 / 7 SATISFIED")

print()
print("HEADLINE FULL RESULTS")
print(
    "  RANDOM PR / ROC              : "
    f"{float(full_r['pr_auc']):.12f} / "
    f"{float(full_r['roc_auc']):.12f}"
)

print(
    "  CHRONO PR / ROC              : "
    f"{float(full_c['pr_auc']):.12f} / "
    f"{float(full_c['roc_auc']):.12f}"
)

print(
    "  CHRONO attack prevalence     : "
    f"{chrono_prev:.12f}"
)

print(
    "  CHRONO PR - prevalence       : "
    f"{float(full_c['pr_auc_minus_attack_prevalence']):.12f}"
)

print()
print("ATTACK-FAMILY CONTEXT")
print("  RANDOM interpretable         : 11")
print("  CHRONO interpretable         : 1")
print("  CHRONO positive family       : Infilteration only")
print(
    "  CHRONO attack support        :",
    f"{chrono_attack_support:,}",
)

print()
print("GOVERNANCE")
print("  Stage23 fit budget           : 50 / 50 SEALED")
print("  additional fits authorized   : 0")
print("  new model fits               : 0")
print("  model inference              : 0")
print("  LightGBM execution           : 0")
print("  XGBoost execution            : 0")
print("  new SHAP values              : 0")
print("  new bootstrap replicates     : 0")
print("  model files read             : 0")
print("  probability NPZ reads        : 0")
print("  parquet reads                : 0")
print("  Raw Mar1 accessed            : NO")
print("  Raw Mar2 accessed            : NO")
print("  repository modified          : NO")

print()
print("STATUS:")
print(
    "  FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED"
)

print()
print("NEXT AUTHORIZED ACTION:")
print(
    "  ZERO-FIT FINAL STAGE23 CLOSURE SEAL + PUSH."
)
print(
    "  NO ADDITIONAL MODEL FITS."
)

print(SEP)

STAGE23-7A-R5 — FINAL STANDALONE SYNTHESIS PACKAGE

[EXACT] branch    : main
[EXACT] HEAD      : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag: stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree  : CLEAN
[EXACT] Stage23 fit budget: 50 / 50 SEALED

VERIFY FROZEN STAGE23 PROTOCOL

[EXACT] stopping conditions : 7
[EXACT] frozen main figures : 4
[EXACT] primary metric       : PR_AUC
[EXACT] interaction metrics  : PR_AUC + ROC_AUC
[EXACT] fixed threshold      : 0.50
[EXACT] Mar1 / Mar2          : PERMANENTLY FORBIDDEN

LOAD SEALED PRIMARY ABLATION RESULTS

[EXACT] primary result JSONs : 12
[EXACT] primary table rows   : 14
[EXACT] primary subsets      : 7 / 7
[EXACT] splits               : 2 / 2

LOAD SEALED MATCHED-SIZE PLACEBO RESULTS

[EXACT] placebo result JSONs : 10
[EXACT] placebo pairs        : 10 / 10
[EXACT] placebo subsets      : 5 / 5
[EXACT] DATA_VALIDATION_ONLY: 1
[EXACT] RANKING_PREVALENCE_ONLY: 9
[EXACT] BOTH                 : 0
[EXACT] NEITHER       

In [17]:
# ==================================================================================================
# STAGE23-7B — DEFINITIVE ZERO-FIT STAGE23 CLOSURE / SEAL / TAG / ATOMIC PUSH
#
# INPUT
#   /kaggle/working/stage23_7_final_synthesis
#
# EXPECTED PACKAGE MANIFEST SHA256
#   c29624f379ad32413308a4b4657d56f79109fd4fb494b3e8f754cb7e04bf5d19
#
# REPOSITORY PARENT
#   3c874e3d20b2f38d842423c1f2316fd60a5eea68
#
# FINAL TAG
#   stage23-shortcut-feature-audit-complete-v1
#
# SCIENTIFIC GOVERNANCE
#   ZERO MODEL FITS
#   ZERO MODEL INFERENCE
#   ZERO LIGHTGBM EXECUTION
#   ZERO XGBOOST EXECUTION
#   ZERO NEW SHAP
#   ZERO NEW BOOTSTRAP
#   ZERO SCIENTIFIC METRIC COMPUTATION
#   ZERO RAW DATA READS
#   NO MAR1 / MAR2
#
# This cell ONLY:
#   - verifies the completed Stage23-7A package
#   - copies it byte-for-byte into the repository
#   - creates closure/seal metadata
#   - commits
#   - creates annotated final Stage23 tag
#   - atomically pushes main + tag
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import shutil
import os
import stat


# ==================================================================================================
# 0. CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE = Path(
    "/kaggle/working/stage23_7_final_synthesis"
)

DEST = (
    REPO
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_7_final_synthesis"
)


EXPECTED_PARENT = (
    "3c874e3d20b2f38d842423c1f2316fd60a5eea68"
)

EXPECTED_PARENT_TAG = (
    "stage23-6-attack-family-analysis-complete-v1"
)

EXPECTED_PACKAGE_MANIFEST_SHA = (
    "c29624f379ad32413308a4b4657d56f79109fd4fb494b3e8f754cb7e04bf5d19"
)


FINAL_TAG = (
    "stage23-shortcut-feature-audit-complete-v1"
)

COMMIT_MESSAGE = (
    "Stage23: final synthesis and protocol closure"
)


SOURCE_CHECKSUMS = (
    SOURCE
    / "checksums.sha256"
)

SOURCE_STATE = (
    SOURCE
    / "execution_state.json"
)

SOURCE_AUDIT = (
    SOURCE
    / "audit"
    / "stage23_7_consolidated_scientific_audit.json"
)

SOURCE_MANIFEST = (
    SOURCE
    / "audit"
    / "stage23_7_source_manifest.json"
)


CLOSURE_RECEIPT = (
    DEST
    / "stage23_final_closure_receipt.json"
)

REPOSITORY_CHECKSUMS = (
    DEST
    / "stage23_final_repository_checksums.sha256"
)


EXPECTED_PACKAGE_CONTENT = {
    "tables":
        10,

    "figure_png":
        6,

    "figure_pdf":
        6,

    "manuscript_tex":
        3,

    "audit_json":
        2,

    "root_scientific_synthesis":
        1,

    "root_readme":
        1,

    "root_execution_state":
        1,

    "source_checksum_entries":
        30,

    "source_total_files_including_checksums":
        31,
}


SEP = "=" * 122


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def git(
    *args,
    check=True,
    env=None,
    show=False,
):

    merged_env = os.environ.copy()

    if env:

        merged_env.update(
            env
        )


    p = subprocess.run(
        [
            "git",
            *map(
                str,
                args,
            ),
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )


    output = (
        p.stdout
        or ""
    ).strip()


    if (
        check
        and p.returncode != 0
    ):

        raise RuntimeError(
            "git command failed:\n"
            f"git {' '.join(map(str, args))}\n"
            f"{output}"
        )


    if (
        show
        and output
    ):

        print(
            output
        )


    return (
        p.returncode,
        output,
    )


def verify_checksum_manifest(
    root,
    manifest,
):

    lines = Path(
        manifest
    ).read_text(
        encoding="utf-8"
    ).splitlines()


    entries = []


    for line in lines:

        if not line.strip():

            continue


        digest, rel = line.split(
            "  ",
            1,
        )


        target = (
            Path(root)
            / rel
        )


        if not target.is_file():

            raise RuntimeError(
                "Checksum manifest target missing:\n"
                f"{target}"
            )


        actual = sha256_file(
            target
        )


        if actual != digest:

            raise RuntimeError(
                "Checksum mismatch:\n"
                f"{rel}\n"
                f"expected={digest}\n"
                f"actual={actual}"
            )


        entries.append(
            (
                digest,
                rel,
            )
        )


    return entries


# ==================================================================================================
# 2. BEGIN
# ==================================================================================================

print(SEP)
print("STAGE23-7B — DEFINITIVE ZERO-FIT STAGE23 CLOSURE / SEAL / PUSH")
print(SEP)
print()


# ==================================================================================================
# 3. VERIFY EXACT REPOSITORY PARENT
# ==================================================================================================

if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, parent_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    EXPECTED_PARENT_TAG,
)

_, status = git(
    "status",
    "--porcelain",
)


if branch != "main":

    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "Unexpected Stage23 closure parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )


if parent_tag_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage23-6 parent tag mismatch."
    )


if status:

    raise RuntimeError(
        "Repository dirty before final Stage23 closure:\n"
        + status
    )


print("[EXACT] branch     :", branch)
print("[EXACT] HEAD       :", head)
print("[EXACT] parent tag :", EXPECTED_PARENT_TAG)
print("[EXACT] worktree   : CLEAN")
print("[EXACT] Stage23 fit budget: 50 / 50 SEALED")


# ==================================================================================================
# 4. FINAL TAG / DESTINATION MUST NOT EXIST
# ==================================================================================================

if DEST.exists():

    raise RuntimeError(
        "Final Stage23 repository destination already exists:\n"
        f"{DEST}"
    )


_, local_final_tag = git(
    "tag",
    "--list",
    FINAL_TAG,
)


if local_final_tag:

    raise RuntimeError(
        f"Final Stage23 tag already exists locally:\n{FINAL_TAG}"
    )


_, remote_final_tag = git(
    "ls-remote",
    "--tags",
    "origin",
    f"refs/tags/{FINAL_TAG}",
)


if remote_final_tag:

    raise RuntimeError(
        f"Final Stage23 tag already exists remotely:\n{FINAL_TAG}"
    )


print("[EXACT] final destination absent.")
print("[EXACT] final tag absent locally and remotely.")


# ==================================================================================================
# 5. VERIFY STAGE23-7A SOURCE PACKAGE
# ==================================================================================================

print()
print(SEP)
print("VERIFY STAGE23-7A COMPLETED PACKAGE")
print(SEP)
print()


if not SOURCE.is_dir():

    raise RuntimeError(
        f"Stage23-7A source package missing:\n{SOURCE}"
    )


for required in [
    SOURCE_CHECKSUMS,
    SOURCE_STATE,
    SOURCE_AUDIT,
    SOURCE_MANIFEST,
]:

    if not required.is_file():

        raise RuntimeError(
            f"Required Stage23-7A artifact missing:\n{required}"
        )


actual_package_manifest_sha = sha256_file(
    SOURCE_CHECKSUMS
)


print(
    "checksums.sha256 expected:",
    EXPECTED_PACKAGE_MANIFEST_SHA,
)

print(
    "checksums.sha256 actual:  ",
    actual_package_manifest_sha,
)


if (
    actual_package_manifest_sha
    !=
    EXPECTED_PACKAGE_MANIFEST_SHA
):

    raise RuntimeError(
        "Stage23-7A package checksum-manifest SHA mismatch."
    )


print("[EXACT]")


source_entries = verify_checksum_manifest(
    SOURCE,
    SOURCE_CHECKSUMS,
)


if len(
    source_entries
) != EXPECTED_PACKAGE_CONTENT[
    "source_checksum_entries"
]:

    raise RuntimeError(
        "Unexpected Stage23-7A checksum entry count.\n"
        f"expected="
        f"{EXPECTED_PACKAGE_CONTENT['source_checksum_entries']}\n"
        f"actual={len(source_entries)}"
    )


all_source_files = sorted(
    p
    for p in SOURCE.rglob(
        "*"
    )
    if p.is_file()
)


if len(
    all_source_files
) != EXPECTED_PACKAGE_CONTENT[
    "source_total_files_including_checksums"
]:

    raise RuntimeError(
        "Unexpected total Stage23-7A package file count.\n"
        f"expected="
        f"{EXPECTED_PACKAGE_CONTENT['source_total_files_including_checksums']}\n"
        f"actual={len(all_source_files)}"
    )


print(
    "[EXACT] source checksum entries:",
    len(
        source_entries
    ),
)

print(
    "[EXACT] source package files   :",
    len(
        all_source_files
    ),
)


# ==================================================================================================
# 6. VERIFY EXACT PUBLICATION STRUCTURE
# ==================================================================================================

table_files = sorted(
    (
        SOURCE
        / "tables"
    ).glob(
        "*.csv"
    )
)

png_files = sorted(
    (
        SOURCE
        / "figures"
    ).glob(
        "*.png"
    )
)

pdf_files = sorted(
    (
        SOURCE
        / "figures"
    ).glob(
        "*.pdf"
    )
)

tex_files = sorted(
    (
        SOURCE
        / "manuscript"
    ).glob(
        "*.tex"
    )
)

audit_files = sorted(
    (
        SOURCE
        / "audit"
    ).glob(
        "*.json"
    )
)


if len(
    table_files
) != 10:

    raise RuntimeError(
        "Publication/supplementary table count != 10."
    )


if len(
    png_files
) != 6:

    raise RuntimeError(
        "PNG figure count != 6."
    )


if len(
    pdf_files
) != 6:

    raise RuntimeError(
        "PDF figure count != 6."
    )


if len(
    tex_files
) != 3:

    raise RuntimeError(
        "Manuscript integration file count != 3."
    )


if len(
    audit_files
) != 2:

    raise RuntimeError(
        "Audit JSON count != 2."
    )


expected_figure_bases = {
    "figure_23_a_subset_split_interaction",
    "figure_23_b_removal_penalties_and_interaction",
    "figure_23_c_single_feature_stump_degradation",
    "figure_23_d_shap_proxy_absorption",
    "figure_23_s1_placebo_interactions",
    "figure_23_s2_attack_family_delta_f1",
}


if {
    p.stem
    for p in png_files
} != expected_figure_bases:

    raise RuntimeError(
        "PNG figure base set mismatch."
    )


if {
    p.stem
    for p in pdf_files
} != expected_figure_bases:

    raise RuntimeError(
        "PDF figure base set mismatch."
    )


print("[EXACT] publication/supp tables      : 10")
print("[EXACT] frozen main figure bases     : 4")
print("[EXACT] supplementary figure bases   : 2")
print("[EXACT] PNG/PDF figure files         : 12")
print("[EXACT] manuscript integration files : 3")


# ==================================================================================================
# 7. VERIFY EXECUTION STATE
# ==================================================================================================

state = json.loads(
    SOURCE_STATE.read_text(
        encoding="utf-8"
    )
)


if (
    state.get(
        "status"
    )
    !=
    "FINAL_SYNTHESIS_TABLES_FIGURES_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Stage23-7A execution state is not complete."
    )


if (
    state.get(
        "execution_parent_commit"
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage23-7A execution parent mismatch."
    )


state_counts = state.get(
    "counts",
    {}
)


expected_state_counts = {
    "publication_tables":
        10,

    "main_figure_bases":
        4,

    "supplementary_figure_bases":
        2,

    "figure_files":
        12,

    "manuscript_files":
        3,

    "stopping_conditions":
        7,
}


for key, expected in expected_state_counts.items():

    if int(
        state_counts.get(
            key,
            -1,
        )
    ) != expected:

        raise RuntimeError(
            f"Execution-state count mismatch: {key}"
        )


governance = state.get(
    "governance",
    {}
)


expected_zero_fields = [
    "model_fits",
    "model_inference",
    "lightgbm_execution",
    "xgboost_execution",
    "new_shap",
    "new_bootstrap",
    "model_files_read",
    "npz_reads",
    "parquet_reads",
]


for key in expected_zero_fields:

    if governance.get(
        key
    ) != 0:

        raise RuntimeError(
            f"Stage23-7A governance violation: {key}"
        )


if governance.get(
    "stage23_fit_budget_before"
) != 50:

    raise RuntimeError(
        "Stage23 fit budget before synthesis != 50."
    )


if governance.get(
    "stage23_fit_budget_after"
) != 50:

    raise RuntimeError(
        "Stage23 fit budget after synthesis != 50."
    )


if governance.get(
    "raw_mar1_accessed"
) is not False:

    raise RuntimeError(
        "Stage23-7A reports Mar1 access."
    )


if governance.get(
    "raw_mar2_accessed"
) is not False:

    raise RuntimeError(
        "Stage23-7A reports Mar2 access."
    )


if governance.get(
    "repository_modified"
) is not False:

    raise RuntimeError(
        "Stage23-7A reports repository modification."
    )


schema_distribution = state.get(
    "frozen_stage23_2_schema_distribution",
    {}
)


if schema_distribution != {
    "DATA_VALIDATION_ONLY":
        1,

    "RANKING_PREVALENCE_ONLY":
        9,

    "BOTH":
        0,

    "NEITHER":
        0,
}:

    raise RuntimeError(
        "Frozen Stage23-2 schema distribution mismatch."
    )


print()
print("[EXACT] Stage23-7A execution state complete.")
print("[EXACT] Stage23-2 schema distribution: 1 / 9 / 0 / 0")
print("[EXACT] fit budget remains: 50 / 50")


# ==================================================================================================
# 8. VERIFY CONSOLIDATED SCIENTIFIC AUDIT
# ==================================================================================================

audit = json.loads(
    SOURCE_AUDIT.read_text(
        encoding="utf-8"
    )
)


if (
    audit.get(
        "status"
    )
    !=
    "STAGE23_CLOSURE_PACKAGE_COMPLETE_UNSEALED"
):

    raise RuntimeError(
        "Consolidated scientific audit status mismatch."
    )


if (
    audit.get(
        "execution_parent",
        {}
    ).get(
        "commit"
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Audit execution parent mismatch."
    )


audit_counts = audit.get(
    "counts",
    {}
)


expected_audit_counts = {
    "primary_result_jsons":
        12,

    "primary_subset_split_rows":
        14,

    "placebo_result_jsons":
        10,

    "placebo_subset_split_rows":
        10,

    "uncertainty_comparisons":
        11,

    "stump_controls":
        6,

    "shap_proxy_comparisons":
        22,

    "shap_component_models":
        48,

    "component_shap_top20_rows":
        960,

    "attack_family_metric_rows":
        182,

    "attack_families":
        13,

    "random_interpretable_attack_families":
        11,

    "chronological_interpretable_attack_families":
        1,

    "publication_tables":
        10,

    "main_figure_bases":
        4,

    "supplementary_figure_bases":
        2,

    "figure_files":
        12,

    "manuscript_files":
        3,

    "stopping_conditions_satisfied":
        7,
}


for key, expected in expected_audit_counts.items():

    if int(
        audit_counts.get(
            key,
            -1,
        )
    ) != expected:

        raise RuntimeError(
            f"Scientific audit count mismatch: {key}"
        )


stopping_evidence = audit.get(
    "stopping_rule_evidence",
    []
)


if len(
    stopping_evidence
) != 7:

    raise RuntimeError(
        "Audit stopping-rule evidence count != 7."
    )


if not all(
    bool(
        item.get(
            "satisfied"
        )
    )
    for item in stopping_evidence
):

    raise RuntimeError(
        "At least one frozen Stage23 stopping condition is unsatisfied."
    )


audit_governance = audit.get(
    "governance",
    {}
)


for key in [
    "model_fits",
    "model_inference",
    "lightgbm_execution",
    "xgboost_execution",
    "new_shap_computation",
    "new_bootstrap_sampling",
    "model_files_read",
    "probability_npz_files_read",
    "parquet_files_read",
]:

    if audit_governance.get(
        key
    ) != 0:

        raise RuntimeError(
            f"Scientific audit governance violation: {key}"
        )


if audit_governance.get(
    "raw_mar1_accessed"
) is not False:

    raise RuntimeError(
        "Scientific audit indicates Mar1 access."
    )


if audit_governance.get(
    "raw_mar2_accessed"
) is not False:

    raise RuntimeError(
        "Scientific audit indicates Mar2 access."
    )


print()
print("[EXACT] frozen stopping rule: 7 / 7 SATISFIED")
print("[EXACT] consolidated scientific audit: COMPLETE")


# ==================================================================================================
# 9. VERIFY SOURCE-MANIFEST GOVERNANCE
# ==================================================================================================

source_manifest = json.loads(
    SOURCE_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if (
    source_manifest.get(
        "execution_parent_commit"
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage23-7 source-manifest parent mismatch."
    )


if (
    source_manifest.get(
        "source_policy"
    )
    !=
    "SEALED_REPOSITORY_JSON_AND_CSV_ONLY"
):

    raise RuntimeError(
        "Unexpected Stage23-7 source policy."
    )


for key in [
    "model_files_read",
    "probability_npz_files_read",
    "parquet_files_read",
]:

    if source_manifest.get(
        key
    ) != 0:

        raise RuntimeError(
            f"Source-manifest governance violation: {key}"
        )


if source_manifest.get(
    "raw_mar1_accessed"
) is not False:

    raise RuntimeError(
        "Source manifest reports Mar1 access."
    )


if source_manifest.get(
    "raw_mar2_accessed"
) is not False:

    raise RuntimeError(
        "Source manifest reports Mar2 access."
    )


for item in source_manifest.get(
    "source_files",
    []
):

    rel = str(
        item[
            "path"
        ]
    ).lower()


    if not (
        rel.endswith(
            ".json"
        )
        or
        rel.endswith(
            ".csv"
        )
    ):

        raise RuntimeError(
            "Stage23-7 source manifest contains "
            "non-JSON/CSV scientific source:\n"
            f"{rel}"
        )


print("[EXACT] Stage23-7 scientific source policy: JSON/CSV only")


# ==================================================================================================
# 10. GITHUB FILE-SIZE SAFETY
# ==================================================================================================

print()
print(SEP)
print("GITHUB FILE-SIZE SAFETY")
print(SEP)
print()


largest = max(
    all_source_files,
    key=lambda p:
        p.stat().st_size,
)


largest_mib = (
    largest.stat().st_size
    / (
        1024 ** 2
    )
)


print(
    "Largest Stage23-7 artifact:"
)

print(
    " ",
    largest,
)

print(
    "Size:",
    f"{largest_mib:.2f} MiB",
)


if (
    largest.stat().st_size
    >=
    95
    * 1024
    * 1024
):

    raise RuntimeError(
        "Stage23-7 contains an artifact >=95 MiB.\n"
        "REFUSING GitHub commit."
    )


print("[OK] all Stage23-7 artifacts below GitHub safety limit.")


# ==================================================================================================
# 11. GITHUB AUTH — BEFORE REPOSITORY MODIFICATION
# ==================================================================================================

github_token = (
    os.environ.get(
        "GITHUB_TOKEN"
    )
    or
    os.environ.get(
        "GH_TOKEN"
    )
)


if not github_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )


        secrets = UserSecretsClient()


        for secret_name in [
            "GITHUB_TOKEN",
            "github_token",
            "GH_TOKEN",
            "github_pat",
            "GITHUB_PAT",
        ]:

            try:

                candidate = secrets.get_secret(
                    secret_name
                )

            except Exception:

                candidate = None


            if candidate:

                github_token = candidate.strip()

                print(
                    "[OK] GitHub credential loaded from Kaggle Secrets:",
                    secret_name,
                )

                break

    except Exception:

        pass


if not github_token:

    raise RuntimeError(
        "GitHub credential unavailable.\n"
        "Repository has NOT been modified."
    )


ASKPASS = Path(
    "/kaggle/working/.stage23_final_git_askpass.sh"
)


ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$STAGE23_FINAL_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


ASKPASS.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


auth_env = {
    "GIT_ASKPASS":
        str(
            ASKPASS
        ),

    "GIT_TERMINAL_PROMPT":
        "0",

    "STAGE23_FINAL_GITHUB_TOKEN":
        github_token,
}


# ==================================================================================================
# 12. PUSH AUTH PREFLIGHT — STILL BEFORE REPOSITORY MODIFICATION
# ==================================================================================================

print()
print(SEP)
print("GITHUB PUSH AUTH PREFLIGHT")
print(SEP)
print()


rc, dryrun_output = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
    env=auth_env,
)


if rc != 0:

    github_token = None

    try:

        ASKPASS.unlink()

    except Exception:

        pass


    raise RuntimeError(
        "GitHub push-auth preflight failed.\n"
        "Repository has NOT been modified.\n"
        + dryrun_output
    )


print("[OK] GitHub push authentication available.")


# ==================================================================================================
# 13. ONLY NOW COPY STAGE23-7A INTO REPOSITORY
# ==================================================================================================

print()
print(SEP)
print("COPY FINAL STAGE23 SYNTHESIS PACKAGE")
print(SEP)
print()


DEST.parent.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copytree(
    SOURCE,
    DEST,
    copy_function=shutil.copy2,
)


print("[OK] Stage23-7A package copied into repository.")


# ==================================================================================================
# 14. VERIFY BYTE-FOR-BYTE COPY
# ==================================================================================================

copied_files = sorted(
    p
    for p in DEST.rglob(
        "*"
    )
    if p.is_file()
)


if len(
    copied_files
) != len(
    all_source_files
):

    raise RuntimeError(
        "Copied Stage23-7 package file count mismatch."
    )


for source_file in all_source_files:

    rel = source_file.relative_to(
        SOURCE
    )

    copied = (
        DEST
        / rel
    )


    if not copied.is_file():

        raise RuntimeError(
            f"Copied file missing:\n{rel}"
        )


    if sha256_file(
        copied
    ) != sha256_file(
        source_file
    ):

        raise RuntimeError(
            f"Byte-for-byte copy mismatch:\n{rel}"
        )


if sha256_file(
    DEST
    / "checksums.sha256"
) != EXPECTED_PACKAGE_MANIFEST_SHA:

    raise RuntimeError(
        "Copied package checksum-manifest SHA mismatch."
    )


print("[EXACT] scientific synthesis package preserved byte-for-byte.")
print(
    "[EXACT] copied package manifest SHA256:",
    EXPECTED_PACKAGE_MANIFEST_SHA,
)


# ==================================================================================================
# 15. CREATE FINAL CLOSURE RECEIPT — METADATA ONLY
# ==================================================================================================

write_json(
    CLOSURE_RECEIPT,
    {
        "stage":
            "Stage23 validation-safe shortcut-feature audit",

        "status":
            "FINAL_PROTOCOL_CLOSURE_SEALED",

        "sealed_utc":
            utc_now(),

        "execution_parent": {
            "commit":
                EXPECTED_PARENT,

            "tag":
                EXPECTED_PARENT_TAG,
        },

        "final_tag":
            FINAL_TAG,

        "stage23_fit_budget":
            "50 / 50 SEALED",

        "scientific_closure": {
            "frozen_stopping_conditions_satisfied":
                "7 / 7",

            "primary_subsets":
                7,

            "primary_split_results":
                14,

            "matched_size_placebo_subsets":
                5,

            "matched_size_placebo_split_results":
                10,

            "stump_controls":
                6,

            "uncertainty_comparisons":
                11,

            "component_treeshap_models":
                48,

            "shap_proxy_comparisons":
                22,

            "attack_family_metric_rows":
                182,

            "frozen_attack_families":
                13,

            "main_publication_figures":
                4,

            "supplementary_figures":
                2,

            "publication_and_supplementary_tables":
                10,

            "manuscript_integration_files":
                3,
        },

        "headline_full_results": {
            "RANDOM_NATURAL": {
                "PR_AUC":
                    0.9955900418992819,

                "ROC_AUC":
                    0.9986245647735994,
            },

            "CHRONOLOGICAL_NATURAL": {
                "PR_AUC":
                    0.10621515513397227,

                "ROC_AUC":
                    0.5149184263937692,

                "attack_prevalence":
                    0.104846912998080,

                "PR_AUC_minus_attack_prevalence":
                    0.001368242135892,
            },
        },

        "attack_family_context": {
            "random_interpretable_families":
                11,

            "chronological_interpretable_families":
                1,

            "chronological_positive_family":
                "Infilteration",

            "chronological_attack_support":
                62256,
        },

        "stage23_2_historical_schema_distribution": {
            "DATA_VALIDATION_ONLY":
                1,

            "RANKING_PREVALENCE_ONLY":
                9,

            "BOTH":
                0,

            "NEITHER":
                0,
        },

        "scientific_interpretation_boundary": {
            "supported":
                (
                    "Strong validation sensitivity and poor "
                    "forward-temporal transfer of several discriminative cues."
                ),

            "not_supported":
                (
                    "Stage23 does not establish any tested feature as leakage "
                    "or as a causal source of the validation-regime gap."
                ),

            "chronological_family_limit":
                (
                    "All frozen chronological attack support belongs to "
                    "Infilteration; the other 12 frozen development families "
                    "have zero chronological attack support."
                ),

            "shap_boundary":
                (
                    "Component-specific TreeSHAP is primary. The equal-weight "
                    "normalized consensus is descriptive only and is not exact "
                    "SHAP for the probability-averaged ensemble."
                ),
        },

        "stage23_7a_package": {
            "source_package":
                "/kaggle/working/stage23_7_final_synthesis",

            "package_checksum_manifest_sha256":
                EXPECTED_PACKAGE_MANIFEST_SHA,

            "scientific_package_copied_byte_for_byte":
                True,
        },

        "governance": {
            "additional_model_fits_authorized":
                0,

            "model_fits_during_closure":
                0,

            "model_inference_during_closure":
                0,

            "lightgbm_execution_during_closure":
                0,

            "xgboost_execution_during_closure":
                0,

            "new_shap_during_closure":
                0,

            "new_bootstrap_during_closure":
                0,

            "scientific_metrics_computed_during_closure":
                0,

            "raw_data_reads_during_closure":
                0,

            "raw_mar1_accessed":
                False,

            "raw_mar2_accessed":
                False,

            "rolling_forward_analysis":
                "NOT_INCLUDED_IN_FINAL_STAGE23_PROTOCOL",
        },

        "final_state":
            "STAGE23_COMPLETE_NO_ADDITIONAL_MODEL_FITS_AUTHORIZED",
    },
)


# ==================================================================================================
# 16. FINAL REPOSITORY CHECKSUM MANIFEST
#
# Includes:
#   - every byte-for-byte Stage23-7A package file
#   - final closure receipt
#
# Does not self-hash.
# ==================================================================================================

manifest_targets = sorted(
    [
        p
        for p in DEST.rglob(
            "*"
        )
        if (
            p.is_file()
            and
            p != REPOSITORY_CHECKSUMS
        )
    ],
    key=lambda p:
        str(
            p.relative_to(
                REPO
            )
        ),
)


REPOSITORY_CHECKSUMS.write_text(
    "\n".join(
        (
            f"{sha256_file(path)}  "
            f"{path.relative_to(REPO).as_posix()}"
        )
        for path in manifest_targets
    )
    + "\n",
    encoding="utf-8",
)


repository_manifest_sha = sha256_file(
    REPOSITORY_CHECKSUMS
)


print()
print(
    "[OK] final repository checksum entries:",
    len(
        manifest_targets
    ),
)

print(
    "[OK] final repository manifest SHA256:",
    repository_manifest_sha,
)


# ==================================================================================================
# 17. GIT IDENTITY
# ==================================================================================================

_, git_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

_, git_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)


if not git_name:

    _, prior_name = git(
        "log",
        "-1",
        "--format=%an",
    )

    git(
        "config",
        "user.name",
        prior_name,
    )


if not git_email:

    _, prior_email = git(
        "log",
        "-1",
        "--format=%ae",
    )

    git(
        "config",
        "user.email",
        prior_email,
    )


# ==================================================================================================
# 18. STAGE ONLY FINAL STAGE23 DIRECTORY
# ==================================================================================================

git(
    "add",
    "--",
    str(
        DEST.relative_to(
            REPO
        )
    ),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    line
    for line in staged.splitlines()
    if line.strip()
]


if not staged_files:

    raise RuntimeError(
        "No Stage23 final-closure files staged."
    )


allowed_prefix = (
    str(
        DEST.relative_to(
            REPO
        )
    )
    + "/"
)


for path in staged_files:

    if not path.startswith(
        allowed_prefix
    ):

        raise RuntimeError(
            "Unexpected staged path outside final Stage23 closure:\n"
            f"{path}"
        )


print()
print(
    "[EXACT] staged final Stage23 files:",
    len(
        staged_files
    ),
)


# ==================================================================================================
# 19. COMMIT FINAL STAGE23 CLOSURE
# ==================================================================================================

print()
print(SEP)
print("FINAL STAGE23 COMMIT")
print(SEP)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, final_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] final Stage23 commit:",
    final_commit,
)


# ==================================================================================================
# 20. CREATE DEFINITIVE ANNOTATED TAG
# ==================================================================================================

git(
    "tag",
    "-a",
    FINAL_TAG,
    final_commit,
    "-m",
    (
        "Stage23 validation-safe shortcut-feature audit complete: "
        "7/7 frozen stopping conditions satisfied; "
        "50/50 model-fit budget sealed; "
        "primary and placebo ablations, six stump controls, "
        "frozen uncertainty, 48 component TreeSHAP explanations, "
        "attack-family analysis, publication figures/tables, "
        "and manuscript integration complete; "
        "zero additional model fits"
    ),
)


# ==================================================================================================
# 21. ATOMIC PUSH MAIN + DEFINITIVE TAG
# ==================================================================================================

print()
print(SEP)
print("ATOMIC PUSH — MAIN + FINAL STAGE23 TAG")
print(SEP)
print()


git(
    "push",
    "--atomic",
    "origin",
    "main",
    f"refs/tags/{FINAL_TAG}",
    env=auth_env,
    show=True,
)


# ==================================================================================================
# 22. REMOVE AUTH MATERIAL IMMEDIATELY
# ==================================================================================================

github_token = None


try:

    ASKPASS.unlink()

except Exception:

    pass


# ==================================================================================================
# 23. REMOTE VERIFICATION
# ==================================================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


_, remote_tag_object_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{FINAL_TAG}",
)


_, remote_tag_peeled_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{FINAL_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Remote main could not be resolved."
    )


if not remote_tag_object_line:

    raise RuntimeError(
        "Remote final annotated tag object could not be resolved."
    )


if not remote_tag_peeled_line:

    raise RuntimeError(
        "Remote final annotated tag could not be peeled."
    )


remote_main = remote_main_line.split()[
    0
]

remote_tag_object = remote_tag_object_line.split()[
    0
]

remote_tag_commit = remote_tag_peeled_line.split()[
    0
]


if remote_main != final_commit:

    raise RuntimeError(
        "Remote main does not equal final Stage23 commit.\n"
        f"expected={final_commit}\n"
        f"actual={remote_main}"
    )


if remote_tag_commit != final_commit:

    raise RuntimeError(
        "Remote final tag does not peel to final Stage23 commit.\n"
        f"expected={final_commit}\n"
        f"actual={remote_tag_commit}"
    )


print("[EXACT] remote main matches final Stage23 commit.")
print("[EXACT] final annotated tag peels to final Stage23 commit.")


# ==================================================================================================
# 24. VERIFY COMMITTED PACKAGE DID NOT CHANGE
# ==================================================================================================

if sha256_file(
    DEST
    / "checksums.sha256"
) != EXPECTED_PACKAGE_MANIFEST_SHA:

    raise RuntimeError(
        "Scientific Stage23-7A package changed during seal."
    )


verify_checksum_manifest(
    DEST,
    DEST
    / "checksums.sha256",
)


print("[EXACT] scientific Stage23-7A package remains byte-for-byte unchanged.")


# ==================================================================================================
# 25. FINAL CLEAN WORKTREE
# ==================================================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after final Stage23 closure:\n"
        + final_status
    )


# ==================================================================================================
# 26. FINAL REPORT
# ==================================================================================================

print()
print(SEP)
print("STAGE23 — DEFINITIVELY CLOSED / SEALED / PUSHED")
print(SEP)
print()

print("Final commit:")
print(
    " ",
    final_commit,
)

print()
print("Definitive tag:")
print(
    " ",
    FINAL_TAG,
)

print()
print("Remote main:")
print(
    " ",
    remote_main,
)

print()
print("Remote annotated tag object:")
print(
    " ",
    remote_tag_object,
)

print()
print("Remote tag peeled commit:")
print(
    " ",
    remote_tag_commit,
)

print()
print("Stage23-7A scientific package manifest SHA256:")
print(
    " ",
    EXPECTED_PACKAGE_MANIFEST_SHA,
)

print()
print("Final repository manifest SHA256:")
print(
    " ",
    repository_manifest_sha,
)

print()
print("FROZEN STAGE23 STOPPING RULE")
print("  primary subsets × splits      : COMPLETE")
print("  six stump controls            : COMPLETE")
print("  five placebo subsets × splits : COMPLETE")
print("  SHAP proxy absorption         : COMPLETE")
print("  attack-family analysis        : COMPLETE")
print("  consolidated scientific audit : COMPLETE")
print("  figures + manuscript          : COMPLETE")
print("  stopping conditions           : 7 / 7 SATISFIED")

print()
print("PUBLICATION PACKAGE")
print("  main figure bases             : 4 / 4")
print("  supplementary figure bases    : 2")
print("  PNG/PDF figure files          : 12")
print("  publication/supp tables       : 10")
print("  manuscript integration files  : 3")

print()
print("SCIENTIFIC HEADLINE")
print("  FULL RANDOM PR-AUC            : 0.995590041899")
print("  FULL RANDOM ROC-AUC           : 0.998624564774")
print("  FULL CHRONO PR-AUC            : 0.106215155134")
print("  FULL CHRONO ROC-AUC           : 0.514918426394")
print("  CHRONO attack prevalence      : 0.104846912998")
print("  CHRONO attack family          : Infilteration only")
print("  CHRONO attack support         : 62,256")

print()
print("FINAL GOVERNANCE")
print("  Stage23 fit budget            : 50 / 50 SEALED")
print("  additional fits authorized    : 0")
print("  new model fits                : 0")
print("  model inference               : 0")
print("  LightGBM execution            : 0")
print("  XGBoost execution             : 0")
print("  new SHAP computation          : 0")
print("  new bootstrap sampling        : 0")
print("  scientific metrics in seal    : 0")
print("  raw-data reads in seal        : 0")
print("  Raw Mar1 accessed             : NO")
print("  Raw Mar2 accessed             : NO")
print("  worktree                      : CLEAN")

print()
print("FINAL STATUS:")
print("  STAGE23_COMPLETE")
print("  NO ADDITIONAL STAGE23 MODEL FITS AUTHORIZED")
print("  NO ROLLING-FORWARD ANALYSIS IN FINAL STAGE23 PROTOCOL")

print(SEP)

STAGE23-7B — DEFINITIVE ZERO-FIT STAGE23 CLOSURE / SEAL / PUSH

[EXACT] branch     : main
[EXACT] HEAD       : 3c874e3d20b2f38d842423c1f2316fd60a5eea68
[EXACT] parent tag : stage23-6-attack-family-analysis-complete-v1
[EXACT] worktree   : CLEAN
[EXACT] Stage23 fit budget: 50 / 50 SEALED
[EXACT] final destination absent.
[EXACT] final tag absent locally and remotely.

VERIFY STAGE23-7A COMPLETED PACKAGE

checksums.sha256 expected: c29624f379ad32413308a4b4657d56f79109fd4fb494b3e8f754cb7e04bf5d19
checksums.sha256 actual:   c29624f379ad32413308a4b4657d56f79109fd4fb494b3e8f754cb7e04bf5d19
[EXACT]
[EXACT] source checksum entries: 30
[EXACT] source package files   : 31
[EXACT] publication/supp tables      : 10
[EXACT] frozen main figure bases     : 4
[EXACT] supplementary figure bases   : 2
[EXACT] PNG/PDF figure files         : 12
[EXACT] manuscript integration files : 3

[EXACT] Stage23-7A execution state complete.
[EXACT] Stage23-2 schema distribution: 1 / 9 / 0 / 0
[EXACT] fit budget rema

In [18]:
# ==================================================================================================
# POST-STAGE23 — EXPORT KAGGLE NOTEBOOK TO PYTHON SCRIPT + PUSH TO GITHUB
#
# IMPORTANT
#   Documentation/reproducibility commit ONLY.
#   Does NOT alter any Stage23 scientific artifact.
#   Does NOT move/recreate the Stage23 completion tag.
#
# EXPECTED SEALED STAGE23 COMMIT
#   f9661acf152d6f1c5dd12de2d0f94f21e68d7170
#
# EXPECTED DEFINITIVE TAG
#   stage23-shortcut-feature-audit-complete-v1
#
# OUTPUT
#   scripts/stage23_shortcut_feature_audit_kaggle.py
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone

import subprocess
import hashlib
import json
import os
import stat


# ==================================================================================================
# 0. CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_STAGE23_COMMIT = (
    "f9661acf152d6f1c5dd12de2d0f94f21e68d7170"
)

FINAL_STAGE23_TAG = (
    "stage23-shortcut-feature-audit-complete-v1"
)

SCRIPT_REL = Path(
    "scripts/stage23_shortcut_feature_audit_kaggle.py"
)

SCRIPT_PATH = (
    REPO
    / SCRIPT_REL
)

COMMIT_MESSAGE = (
    "docs: add Stage23 Kaggle notebook script"
)

SEP = "=" * 118


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def git(
    *args,
    check=True,
    env=None,
    show=False,
):

    merged_env = os.environ.copy()

    if env:
        merged_env.update(
            env
        )

    p = subprocess.run(
        [
            "git",
            *map(str, args),
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )

    output = (
        p.stdout
        or ""
    ).strip()

    if (
        check
        and p.returncode != 0
    ):
        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n"
            f"{output}"
        )

    if (
        show
        and output
    ):
        print(output)

    return (
        p.returncode,
        output,
    )


def sha256_file(
    path,
    chunk=1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


# ==================================================================================================
# 2. VERIFY CURRENT REPOSITORY STATE
# ==================================================================================================

print(SEP)
print("POST-STAGE23 — NOTEBOOK → SCRIPT EXPORT + GITHUB PUSH")
print(SEP)
print()


if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, tag_commit = git(
    "rev-list",
    "-n",
    "1",
    FINAL_STAGE23_TAG,
)


if branch != "main":

    raise RuntimeError(
        f"Expected branch main; found {branch}"
    )


if head != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "Expected repository to still be at the "
        "definitive Stage23 closure commit before "
        "the documentation commit.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={head}"
    )


if status:

    raise RuntimeError(
        "Repository is dirty before notebook export:\n"
        + status
    )


if tag_commit != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "Definitive Stage23 tag no longer points to "
        "the sealed Stage23 commit.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={tag_commit}"
    )


if SCRIPT_PATH.exists():

    raise RuntimeError(
        "Target script already exists:\n"
        f"{SCRIPT_PATH}\n"
        "Refusing overwrite."
    )


print("[EXACT] branch            :", branch)
print("[EXACT] current HEAD      :", head)
print("[EXACT] Stage23 final tag :", FINAL_STAGE23_TAG)
print("[EXACT] tag peeled commit :", tag_commit)
print("[EXACT] worktree          : CLEAN")
print()


# ==================================================================================================
# 3. LOCATE ACTUAL KAGGLE NOTEBOOK FILE
#
# We deliberately DO NOT reconstruct from IPython history.
# ==================================================================================================

print(SEP)
print("LOCATE KAGGLE NOTEBOOK SOURCE")
print(SEP)
print()


search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/temp"),
]


candidates = []


for root in search_roots:

    if not root.exists():
        continue

    try:

        for path in root.rglob(
            "*.ipynb"
        ):

            # Never accidentally export something already inside repo
            # unless it is explicitly a notebook.
            if path.is_file():
                candidates.append(
                    path
                )

    except PermissionError:
        pass


# Remove checkpoints.
candidates = [
    p
    for p in candidates
    if ".ipynb_checkpoints"
    not in str(p)
]


# Prefer Kaggle's conventional current-notebook names.
preferred_names = {
    "__notebook__.ipynb",
    "notebook.ipynb",
}


preferred = [
    p
    for p in candidates
    if p.name
    in preferred_names
]


if len(preferred) == 1:

    NOTEBOOK = preferred[
        0
    ]


elif len(candidates) == 1:

    NOTEBOOK = candidates[
        0
    ]


else:

    print(
        "Detected .ipynb candidates:"
    )

    for idx, path in enumerate(
        candidates,
        start=1,
    ):
        print(
            f"  [{idx}] {path}"
        )

    print()

    raise RuntimeError(
        "Could not uniquely identify the actual Kaggle "
        "notebook file.\n\n"
        "If no notebook is listed, use Kaggle's "
        "'Download code' / '.ipynb' export first, upload "
        "that notebook into this session, and rerun this cell.\n"
        "No repository files have been modified."
    )


print("[EXACT] notebook source:")
print(" ", NOTEBOOK)
print()

print(
    "Notebook size:",
    f"{NOTEBOOK.stat().st_size / 1024 / 1024:.2f} MiB",
)

print(
    "Notebook SHA256:",
    sha256_file(
        NOTEBOOK
    ),
)


# ==================================================================================================
# 4. PARSE NOTEBOOK
# ==================================================================================================

print()
print(SEP)
print("CONVERT NOTEBOOK → PYTHON SCRIPT")
print(SEP)
print()


notebook = json.loads(
    NOTEBOOK.read_text(
        encoding="utf-8"
    )
)


if (
    notebook.get(
        "nbformat"
    )
    is None
):

    raise RuntimeError(
        "File does not appear to be a valid Jupyter notebook."
    )


cells = notebook.get(
    "cells",
    []
)


code_cells = [
    cell
    for cell in cells
    if cell.get(
        "cell_type"
    )
    == "code"
]


markdown_cells = [
    cell
    for cell in cells
    if cell.get(
        "cell_type"
    )
    == "markdown"
]


if not code_cells:

    raise RuntimeError(
        "Notebook contains no code cells."
    )


print(
    "[OK] notebook cells :",
    len(
        cells
    ),
)

print(
    "[OK] code cells     :",
    len(
        code_cells
    ),
)

print(
    "[OK] markdown cells :",
    len(
        markdown_cells
    ),
)


# ==================================================================================================
# 5. GENERATE SCRIPT
#
# Code cells only.
# Markdown is intentionally excluded from executable body.
# Cell boundaries and execution counts are retained as comments.
# ==================================================================================================

script_lines = [
    "#!/usr/bin/env python3",
    "# -*- coding: utf-8 -*-",
    "#",
    "# ================================================================================================",
    "# Stage23 — Validation-Safe Shortcut-Feature Audit",
    "# Kaggle notebook Python export",
    "# ================================================================================================",
    "#",
    "# This file is a reproducibility export of the Kaggle notebook.",
    "#",
    "# IMPORTANT GOVERNANCE",
    "# - The definitive Stage23 scientific state was sealed BEFORE this script-export commit.",
    "# - Definitive Stage23 tag:",
    f"#     {FINAL_STAGE23_TAG}",
    "# - Definitive Stage23 commit:",
    f"#     {EXPECTED_STAGE23_COMMIT}",
    "# - This script-export commit is documentation/reproducibility only.",
    "# - It does NOT redefine, reopen, or extend the Stage23 scientific protocol.",
    "# - Raw Mar1 and Mar2 remain permanently forbidden within Stage23.",
    "# - No additional Stage23 model fits are authorized.",
    "#",
    f"# Source notebook: {NOTEBOOK.name}",
    f"# Source notebook SHA256: {sha256_file(NOTEBOOK)}",
    f"# Exported UTC: {datetime.now(timezone.utc).isoformat()}",
    "#",
    "# ================================================================================================",
    "",
]


code_cell_number = 0


for notebook_index, cell in enumerate(
    cells,
    start=1,
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    code_cell_number += 1

    execution_count = cell.get(
        "execution_count"
    )


    source = cell.get(
        "source",
        []
    )


    if isinstance(
        source,
        list,
    ):

        source_text = "".join(
            source
        )

    elif isinstance(
        source,
        str,
    ):

        source_text = source

    else:

        raise RuntimeError(
            f"Unexpected source type in notebook cell {notebook_index}."
        )


    script_lines.extend(
        [
            "",
            "# ================================================================================================",
            (
                f"# NOTEBOOK CODE CELL {code_cell_number} "
                f"(notebook cell {notebook_index}, "
                f"execution_count={execution_count})"
            ),
            "# ================================================================================================",
            "",
        ]
    )


    # Preserve exact source text.
    script_lines.append(
        source_text.rstrip()
    )

    script_lines.append(
        ""
    )


script_text = (
    "\n".join(
        script_lines
    ).rstrip()
    + "\n"
)


# ==================================================================================================
# 6. BASIC EXPORT SAFETY CHECK
# ==================================================================================================

if (
    "STAGE23"
    not in script_text.upper()
):

    raise RuntimeError(
        "Exported notebook script does not appear "
        "to contain Stage23 content."
    )


if len(
    script_text
) < 10_000:

    raise RuntimeError(
        "Exported script is unexpectedly small; "
        "refusing to push an incomplete notebook export."
    )


SCRIPT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


SCRIPT_PATH.write_text(
    script_text,
    encoding="utf-8",
)


print("[OK] script created:")
print(" ", SCRIPT_PATH)

print()
print(
    "Script size:",
    f"{SCRIPT_PATH.stat().st_size / 1024 / 1024:.2f} MiB",
)

print(
    "Script SHA256:",
    sha256_file(
        SCRIPT_PATH
    ),
)

print(
    "Exported code cells:",
    code_cell_number,
)


# ==================================================================================================
# 7. COMPILE CHECK
#
# Some Jupyter magics/shell syntax is not valid Python.
# If present, fail closed rather than pushing a broken .py file.
# ==================================================================================================

try:

    compile(
        SCRIPT_PATH.read_text(
            encoding="utf-8"
        ),
        str(
            SCRIPT_PATH
        ),
        "exec",
    )

except SyntaxError as exc:

    # Remove the generated script before stopping.
    try:
        SCRIPT_PATH.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "Notebook contains Jupyter-only syntax that is not "
        "valid in a standalone Python script.\n"
        "Generated file was removed and nothing was staged.\n\n"
        f"{exc}"
    )


print("[EXACT] exported script compiles as Python.")


# ==================================================================================================
# 8. GITHUB FILE-SIZE SAFETY
# ==================================================================================================

if (
    SCRIPT_PATH.stat().st_size
    >=
    95
    * 1024
    * 1024
):

    SCRIPT_PATH.unlink()

    raise RuntimeError(
        "Exported script exceeds GitHub safety limit."
    )


print("[OK] GitHub file-size safety passed.")


# ==================================================================================================
# 9. GITHUB AUTH
# ==================================================================================================

github_token = (
    os.environ.get(
        "GITHUB_TOKEN"
    )
    or
    os.environ.get(
        "GH_TOKEN"
    )
)


if not github_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )


        secrets = (
            UserSecretsClient()
        )


        for secret_name in [
            "GITHUB_TOKEN",
            "github_token",
            "GH_TOKEN",
            "github_pat",
            "GITHUB_PAT",
        ]:

            try:

                candidate = (
                    secrets.get_secret(
                        secret_name
                    )
                )

            except Exception:

                candidate = None


            if candidate:

                github_token = (
                    candidate.strip()
                )

                print(
                    "[OK] GitHub credential loaded from Kaggle Secrets:",
                    secret_name,
                )

                break

    except Exception:

        pass


if not github_token:

    SCRIPT_PATH.unlink()

    raise RuntimeError(
        "GitHub credential unavailable.\n"
        "Generated script removed; repository remains unchanged."
    )


ASKPASS = Path(
    "/kaggle/working/.stage23_script_git_askpass.sh"
)


ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$STAGE23_SCRIPT_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


ASKPASS.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


auth_env = {
    "GIT_ASKPASS":
        str(
            ASKPASS
        ),

    "GIT_TERMINAL_PROMPT":
        "0",

    "STAGE23_SCRIPT_GITHUB_TOKEN":
        github_token,
}


# ==================================================================================================
# 10. AUTH PREFLIGHT
# ==================================================================================================

rc, dryrun = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
    env=auth_env,
)


if rc != 0:

    github_token = None

    try:
        ASKPASS.unlink()
    except Exception:
        pass

    try:
        SCRIPT_PATH.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "GitHub authentication preflight failed.\n"
        "Nothing has been staged or committed.\n"
        + dryrun
    )


print("[OK] GitHub push authentication available.")


# ==================================================================================================
# 11. STAGE ONLY SCRIPT
# ==================================================================================================

git(
    "add",
    "--",
    str(
        SCRIPT_REL
    ),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    line.strip()
    for line in staged.splitlines()
    if line.strip()
]


if staged_files != [
    SCRIPT_REL.as_posix()
]:

    raise RuntimeError(
        "Unexpected staged file set:\n"
        + "\n".join(
            staged_files
        )
    )


print(
    "[EXACT] staged file:",
    SCRIPT_REL,
)


# ==================================================================================================
# 12. VERIFY STAGE23 TAG STILL IMMUTABLE BEFORE COMMIT
# ==================================================================================================

_, tag_before_commit = git(
    "rev-list",
    "-n",
    "1",
    FINAL_STAGE23_TAG,
)


if (
    tag_before_commit
    !=
    EXPECTED_STAGE23_COMMIT
):

    raise RuntimeError(
        "Stage23 completion tag moved before documentation commit."
    )


# ==================================================================================================
# 13. DOCUMENTATION COMMIT
# ==================================================================================================

print()
print(SEP)
print("COMMIT NOTEBOOK SCRIPT")
print(SEP)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, script_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] script-export commit:",
    script_commit,
)


# ==================================================================================================
# 14. PUSH MAIN ONLY
#
# DO NOT PUSH / MOVE / RECREATE STAGE23 COMPLETION TAG.
# ==================================================================================================

print()
print(SEP)
print("PUSH DOCUMENTATION COMMIT — MAIN ONLY")
print(SEP)
print()


git(
    "push",
    "origin",
    "main",
    env=auth_env,
    show=True,
)


# ==================================================================================================
# 15. REMOVE AUTH MATERIAL
# ==================================================================================================

github_token = None


try:

    ASKPASS.unlink()

except Exception:

    pass


# ==================================================================================================
# 16. REMOTE VERIFICATION
# ==================================================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


_, remote_stage23_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{FINAL_STAGE23_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Could not resolve remote main."
    )


if not remote_stage23_tag_line:

    raise RuntimeError(
        "Could not resolve definitive Stage23 tag."
    )


remote_main = (
    remote_main_line.split()[
        0
    ]
)

remote_stage23_tag = (
    remote_stage23_tag_line.split()[
        0
    ]
)


if remote_main != script_commit:

    raise RuntimeError(
        "Remote main does not match script-export commit."
    )


if (
    remote_stage23_tag
    !=
    EXPECTED_STAGE23_COMMIT
):

    raise RuntimeError(
        "CRITICAL: definitive Stage23 tag moved.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={remote_stage23_tag}"
    )


# ==================================================================================================
# 17. CLEAN WORKTREE
# ==================================================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after script-export push:\n"
        + final_status
    )


# ==================================================================================================
# 18. FINAL
# ==================================================================================================

print()
print(SEP)
print("STAGE23 KAGGLE NOTEBOOK SCRIPT — PUSHED")
print(SEP)
print()

print("Script:")
print(
    " ",
    SCRIPT_REL,
)

print()
print("Script SHA256:")
print(
    " ",
    sha256_file(
        SCRIPT_PATH
    ),
)

print()
print("Documentation commit:")
print(
    " ",
    script_commit,
)

print()
print("Remote main:")
print(
    " ",
    remote_main,
)

print()
print("Definitive Stage23 scientific tag:")
print(
    " ",
    FINAL_STAGE23_TAG,
)

print()
print("Tag remains permanently at:")
print(
    " ",
    remote_stage23_tag,
)

print()
print("SCIENTIFIC STATE")
print("  Stage23 completion tag moved : NO")
print("  Stage23 scientific artifacts : UNCHANGED")
print("  additional model fits        : 0")
print("  scientific metrics computed  : 0")
print("  Raw Mar1 accessed            : NO")
print("  Raw Mar2 accessed            : NO")
print("  worktree                     : CLEAN")

print()
print("STATUS:")
print("  NOTEBOOK_SCRIPT_DOCUMENTATION_COMMIT_PUSHED")
print("  STAGE23 SCIENTIFIC CLOSURE REMAINS IMMUTABLE")

print(SEP)

POST-STAGE23 — NOTEBOOK → SCRIPT EXPORT + GITHUB PUSH

[EXACT] branch            : main
[EXACT] current HEAD      : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] Stage23 final tag : stage23-shortcut-feature-audit-complete-v1
[EXACT] tag peeled commit : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] worktree          : CLEAN

LOCATE KAGGLE NOTEBOOK SOURCE

Detected .ipynb candidates:
  [1] /kaggle/working/.virtual_documents/__notebook_source__.ipynb
  [2] /kaggle/working/ids2018-validation-safe-ablation/notebooks/original_kaggle_working_notebook.ipynb



RuntimeError: Could not uniquely identify the actual Kaggle notebook file.

If no notebook is listed, use Kaggle's 'Download code' / '.ipynb' export first, upload that notebook into this session, and rerun this cell.
No repository files have been modified.

In [19]:
# ==================================================================================================
# POST-STAGE23 — PUSH FULL KAGGLE NOTEBOOK PYTHON EXPORT
#
# Scientific Stage23 closure remains immutable.
#
# SEALED SCIENTIFIC COMMIT
#   f9661acf152d6f1c5dd12de2d0f94f21e68d7170
#
# SEALED SCIENTIFIC TAG
#   stage23-shortcut-feature-audit-complete-v1
#
# NEW REPOSITORY FILE
#   scripts/stage23_shortcut_feature_audit_kaggle.py
#
# EXPECTED SCRIPT SHA256
#   4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9
#
# THIS COMMIT:
#   documentation/reproducibility only
#   does NOT move the Stage23 tag
#   performs NO scientific computation
# ==================================================================================================

from pathlib import Path
import subprocess
import hashlib
import shutil
import os
import stat


# ==================================================================================================
# 0. CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_STAGE23_COMMIT = (
    "f9661acf152d6f1c5dd12de2d0f94f21e68d7170"
)

FINAL_STAGE23_TAG = (
    "stage23-shortcut-feature-audit-complete-v1"
)

EXPECTED_SCRIPT_SHA = (
    "4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9"
)

SCRIPT_NAME = (
    "stage23_shortcut_feature_audit_kaggle.py"
)

DEST_REL = Path(
    "scripts/stage23_shortcut_feature_audit_kaggle.py"
)

DEST = (
    REPO
    / DEST_REL
)

COMMIT_MESSAGE = (
    "docs: add full Stage23 Kaggle notebook script"
)

SEP = "=" * 120


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def git(
    *args,
    check=True,
    env=None,
    show=False,
):

    merged_env = os.environ.copy()

    if env:
        merged_env.update(
            env
        )

    p = subprocess.run(
        [
            "git",
            *map(str, args),
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )

    output = (
        p.stdout
        or ""
    ).strip()

    if (
        check
        and p.returncode != 0
    ):

        raise RuntimeError(
            f"git {' '.join(map(str, args))} failed:\n"
            f"{output}"
        )

    if (
        show
        and output
    ):
        print(output)

    return (
        p.returncode,
        output,
    )


def sha256_file(
    path,
    chunk=1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


# ==================================================================================================
# 2. VERIFY SEALED SCIENTIFIC STATE
# ==================================================================================================

print(SEP)
print("POST-STAGE23 — PUSH FULL KAGGLE NOTEBOOK SCRIPT")
print(SEP)
print()


if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, tag_commit = git(
    "rev-list",
    "-n",
    "1",
    FINAL_STAGE23_TAG,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "Unexpected HEAD before documentation commit.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={head}"
    )


if tag_commit != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "Definitive Stage23 tag does not point to "
        "the sealed scientific commit.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={tag_commit}"
    )


if status:

    raise RuntimeError(
        "Repository dirty before documentation commit:\n"
        + status
    )


if DEST.exists():

    raise RuntimeError(
        "Destination script already exists:\n"
        f"{DEST}"
    )


print("[EXACT] branch            :", branch)
print("[EXACT] scientific HEAD   :", head)
print("[EXACT] definitive tag    :", FINAL_STAGE23_TAG)
print("[EXACT] tag peeled commit :", tag_commit)
print("[EXACT] worktree          : CLEAN")
print()


# ==================================================================================================
# 3. LOCATE THE UPLOADED SCRIPT
# ==================================================================================================

print(SEP)
print("LOCATE UPLOADED SCRIPT")
print(SEP)
print()


search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]


candidates = []


for root in search_roots:

    if not root.exists():
        continue

    try:

        for path in root.rglob(
            SCRIPT_NAME
        ):

            if (
                path.is_file()
                and
                path.resolve() != DEST.resolve()
            ):

                candidates.append(
                    path
                )

    except PermissionError:
        pass


# Deduplicate exact resolved paths.
unique = {}

for path in candidates:

    unique[
        str(
            path.resolve()
        )
    ] = path


candidates = list(
    unique.values()
)


if len(candidates) != 1:

    print(
        "Detected candidates:"
    )

    for path in candidates:
        print(
            " ",
            path,
        )

    raise RuntimeError(
        "Expected exactly one uploaded Stage23 script.\n"
        "Upload the downloaded .py file to this Kaggle session "
        "and rerun this cell."
    )


SOURCE = candidates[
    0
]


print("[EXACT] source:")
print(
    " ",
    SOURCE,
)

print()


# ==================================================================================================
# 4. VERIFY SCRIPT HASH AND STRUCTURE
# ==================================================================================================

actual_sha = sha256_file(
    SOURCE
)


print(
    "Expected SHA256:",
    EXPECTED_SCRIPT_SHA,
)

print(
    "Actual SHA256:  ",
    actual_sha,
)


if actual_sha != EXPECTED_SCRIPT_SHA:

    raise RuntimeError(
        "Uploaded script SHA256 mismatch.\n"
        "Do NOT push a different export."
    )


text = SOURCE.read_text(
    encoding="utf-8"
)


required_markers = [
    "Stage23 — Validation-Safe Shortcut-Feature Audit",
    "stage23-shortcut-feature-audit-complete-v1",
    "f9661acf152d6f1c5dd12de2d0f94f21e68d7170",
    "# %% [notebook cell 1;",
    "# %% [notebook cell 75;",
]


for marker in required_markers:

    if marker not in text:

        raise RuntimeError(
            "Expected script marker missing:\n"
            + marker
        )


code_cell_markers = text.count(
    "# %% [notebook cell "
)


if code_cell_markers != 75:

    raise RuntimeError(
        "Expected 75 notebook cell markers.\n"
        f"actual={code_cell_markers}"
    )


# Syntax check only — does NOT execute script.
compile(
    text,
    str(SOURCE),
    "exec",
)


print("[EXACT] SHA256             : MATCH")
print("[EXACT] notebook cells     : 75 / 75")
print("[EXACT] Python syntax      : PASS")
print(
    "[EXACT] script size        :",
    f"{SOURCE.stat().st_size / 1024 / 1024:.3f} MiB",
)


# ==================================================================================================
# 5. GITHUB AUTH BEFORE MODIFYING REPOSITORY
# ==================================================================================================

github_token = (
    os.environ.get(
        "GITHUB_TOKEN"
    )
    or
    os.environ.get(
        "GH_TOKEN"
    )
)


if not github_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        secrets = UserSecretsClient()

        for secret_name in [
            "GITHUB_TOKEN",
            "github_token",
            "GH_TOKEN",
            "github_pat",
            "GITHUB_PAT",
        ]:

            try:

                candidate = secrets.get_secret(
                    secret_name
                )

            except Exception:

                candidate = None


            if candidate:

                github_token = candidate.strip()

                print(
                    "[OK] GitHub credential loaded from Kaggle Secrets:",
                    secret_name,
                )

                break

    except Exception:

        pass


if not github_token:

    raise RuntimeError(
        "GitHub credential unavailable.\n"
        "Repository has not been modified."
    )


ASKPASS = Path(
    "/kaggle/working/.stage23_script_push_askpass.sh"
)


ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$STAGE23_SCRIPT_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


ASKPASS.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


auth_env = {
    "GIT_ASKPASS":
        str(ASKPASS),

    "GIT_TERMINAL_PROMPT":
        "0",

    "STAGE23_SCRIPT_TOKEN":
        github_token,
}


# ==================================================================================================
# 6. PUSH AUTH PREFLIGHT
# ==================================================================================================

rc, dryrun = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
    env=auth_env,
)


if rc != 0:

    github_token = None

    try:
        ASKPASS.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "GitHub push authentication failed.\n"
        "Repository remains unchanged.\n"
        + dryrun
    )


print("[OK] GitHub push authentication available.")


# ==================================================================================================
# 7. COPY SCRIPT INTO REPOSITORY
# ==================================================================================================

DEST.parent.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    SOURCE,
    DEST,
)


if sha256_file(
    DEST
) != EXPECTED_SCRIPT_SHA:

    raise RuntimeError(
        "Copied repository script SHA mismatch."
    )


print()
print("[EXACT] script copied byte-for-byte:")
print(
    " ",
    DEST_REL,
)


# ==================================================================================================
# 8. STAGE ONLY THIS SCRIPT
# ==================================================================================================

git(
    "add",
    "--",
    DEST_REL.as_posix(),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    line.strip()
    for line in staged.splitlines()
    if line.strip()
]


if staged_files != [
    DEST_REL.as_posix()
]:

    raise RuntimeError(
        "Unexpected staged files:\n"
        + "\n".join(
            staged_files
        )
    )


print(
    "[EXACT] staged file:",
    DEST_REL.as_posix(),
)


# ==================================================================================================
# 9. VERIFY SCIENTIFIC TAG STILL IMMUTABLE
# ==================================================================================================

_, tag_before_commit = git(
    "rev-list",
    "-n",
    "1",
    FINAL_STAGE23_TAG,
)


if tag_before_commit != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "Stage23 scientific tag moved before documentation commit."
    )


# ==================================================================================================
# 10. DOCUMENTATION COMMIT
# ==================================================================================================

print()
print(SEP)
print("COMMIT FULL NOTEBOOK SCRIPT")
print(SEP)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, documentation_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] documentation commit:",
    documentation_commit,
)


# ==================================================================================================
# 11. PUSH MAIN ONLY — NEVER MOVE THE STAGE23 TAG
# ==================================================================================================

print()
print(SEP)
print("PUSH MAIN ONLY")
print(SEP)
print()


git(
    "push",
    "origin",
    "main",
    env=auth_env,
    show=True,
)


# ==================================================================================================
# 12. DESTROY AUTH HELPER
# ==================================================================================================

github_token = None


try:

    ASKPASS.unlink()

except Exception:

    pass


# ==================================================================================================
# 13. REMOTE VERIFICATION
# ==================================================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{FINAL_STAGE23_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


if not remote_tag_line:

    raise RuntimeError(
        "Unable to resolve definitive Stage23 tag."
    )


remote_main = remote_main_line.split()[
    0
]

remote_tag_commit = remote_tag_line.split()[
    0
]


if remote_main != documentation_commit:

    raise RuntimeError(
        "Remote main does not match documentation commit."
    )


if remote_tag_commit != EXPECTED_STAGE23_COMMIT:

    raise RuntimeError(
        "CRITICAL: Stage23 scientific completion tag moved.\n"
        f"expected={EXPECTED_STAGE23_COMMIT}\n"
        f"actual={remote_tag_commit}"
    )


if sha256_file(
    DEST
) != EXPECTED_SCRIPT_SHA:

    raise RuntimeError(
        "Committed script changed unexpectedly."
    )


# ==================================================================================================
# 14. FINAL CLEAN STATE
# ==================================================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after script push:\n"
        + final_status
    )


# ==================================================================================================
# 15. FINAL
# ==================================================================================================

print()
print(SEP)
print("FULL STAGE23 KAGGLE NOTEBOOK SCRIPT — PUSHED")
print(SEP)
print()

print("Repository path:")
print(
    " ",
    DEST_REL,
)

print()
print("Script SHA256:")
print(
    " ",
    EXPECTED_SCRIPT_SHA,
)

print()
print("Documentation commit:")
print(
    " ",
    documentation_commit,
)

print()
print("Remote main:")
print(
    " ",
    remote_main,
)

print()
print("Scientific Stage23 tag:")
print(
    " ",
    FINAL_STAGE23_TAG,
)

print()
print("Scientific tag remains at:")
print(
    " ",
    remote_tag_commit,
)

print()
print("VERIFICATION")
print("  notebook code cells preserved : 75 / 75")
print("  Python syntax                 : PASS")
print("  script byte hash              : EXACT")
print("  scientific Stage23 tag moved  : NO")
print("  scientific artifacts changed  : NO")
print("  new model fits                : 0")
print("  scientific metrics computed   : 0")
print("  Raw Mar1 accessed             : NO")
print("  Raw Mar2 accessed             : NO")
print("  worktree                      : CLEAN")

print()
print("STATUS:")
print("  STAGE23_NOTEBOOK_SCRIPT_PUSHED")
print("  SCIENTIFIC_STAGE23_CLOSURE_REMAINS_IMMUTABLE")

print(SEP)

POST-STAGE23 — PUSH FULL KAGGLE NOTEBOOK SCRIPT

[EXACT] branch            : main
[EXACT] scientific HEAD   : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] definitive tag    : stage23-shortcut-feature-audit-complete-v1
[EXACT] tag peeled commit : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] worktree          : CLEAN

LOCATE UPLOADED SCRIPT

Detected candidates:


RuntimeError: Expected exactly one uploaded Stage23 script.
Upload the downloaded .py file to this Kaggle session and rerun this cell.

In [20]:
# ==================================================================================================
# STAGE23 SCRIPT — BROWSER UPLOAD INTO KAGGLE
#
# Select:
#   stage23_shortcut_feature_audit_kaggle.py
#
# EXPECTED SHA256:
#   4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9
#
# This cell:
#   - creates an upload control
#   - saves the uploaded file to /kaggle/working
#   - verifies exact SHA256
#   - verifies 75 notebook cell markers
#   - performs Python syntax validation
#   - does NOT touch GitHub
# ==================================================================================================

from pathlib import Path
import hashlib
import ipywidgets as widgets
from IPython.display import display, clear_output


EXPECTED_NAME = "stage23_shortcut_feature_audit_kaggle.py"

EXPECTED_SHA256 = (
    "4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9"
)

DEST = Path(
    "/kaggle/working/stage23_shortcut_feature_audit_kaggle.py"
)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def extract_uploaded_file(value):
    """
    Supports both ipywidgets 7 and 8 FileUpload.value formats.
    """

    # ipywidgets 8:
    # value = ({'name': ..., 'content': memoryview(...)},)
    if isinstance(value, (tuple, list)):

        if len(value) != 1:
            raise RuntimeError(
                f"Expected exactly one uploaded file; got {len(value)}."
            )

        item = value[0]

        name = item.get(
            "name",
            ""
        )

        content = item.get(
            "content"
        )

        if content is None:
            raise RuntimeError(
                "Uploaded file has no content."
            )

        return (
            name,
            bytes(content),
        )

    # ipywidgets 7:
    # value = {'filename.py': {'content': ...}}
    if isinstance(value, dict):

        if len(value) != 1:
            raise RuntimeError(
                f"Expected exactly one uploaded file; got {len(value)}."
            )

        name, item = next(
            iter(value.items())
        )

        if isinstance(item, dict):

            actual_name = item.get(
                "name",
                name,
            )

            content = item.get(
                "content"
            )

        else:

            actual_name = name
            content = item

        if content is None:
            raise RuntimeError(
                "Uploaded file has no content."
            )

        return (
            actual_name,
            bytes(content),
        )

    raise RuntimeError(
        f"Unsupported FileUpload value type: {type(value)}"
    )


uploader = widgets.FileUpload(
    accept=".py",
    multiple=False,
    description="Upload Stage23 .py",
)


output = widgets.Output()


def handle_upload(change):

    with output:

        clear_output(
            wait=True
        )

        try:

            name, data = extract_uploaded_file(
                uploader.value
            )

            print("=" * 100)
            print("STAGE23 SCRIPT UPLOAD VERIFICATION")
            print("=" * 100)
            print()

            print("Uploaded filename:")
            print(" ", name)

            if name != EXPECTED_NAME:

                raise RuntimeError(
                    "Wrong filename.\n"
                    f"Expected: {EXPECTED_NAME}\n"
                    f"Received: {name}"
                )

            actual_sha = sha256_bytes(
                data
            )

            print()
            print("Expected SHA256:")
            print(" ", EXPECTED_SHA256)

            print("Actual SHA256:")
            print(" ", actual_sha)

            if actual_sha != EXPECTED_SHA256:

                raise RuntimeError(
                    "SHA256 mismatch.\n"
                    "This is NOT the exact converted Stage23 script."
                )

            # Decode before writing.
            text = data.decode(
                "utf-8"
            )

            # Verify notebook export structure.
            cell_markers = text.count(
                "# %% [notebook cell "
            )

            if cell_markers != 75:

                raise RuntimeError(
                    "Notebook-cell marker count mismatch.\n"
                    f"Expected: 75\n"
                    f"Actual:   {cell_markers}"
                )

            # Syntax validation ONLY — does not execute the script.
            compile(
                text,
                EXPECTED_NAME,
                "exec",
            )

            # Write only after every validation passes.
            DEST.write_bytes(
                data
            )

            # Verify disk copy.
            disk_sha = hashlib.sha256(
                DEST.read_bytes()
            ).hexdigest()

            if disk_sha != EXPECTED_SHA256:

                try:
                    DEST.unlink()
                except Exception:
                    pass

                raise RuntimeError(
                    "Disk SHA256 mismatch after save."
                )

            print()
            print("[EXACT] filename        : MATCH")
            print("[EXACT] SHA256          : MATCH")
            print("[EXACT] notebook cells  : 75 / 75")
            print("[EXACT] Python syntax   : PASS")
            print("[EXACT] disk copy SHA   : MATCH")

            print()
            print("Saved to:")
            print(" ", DEST)

            print()
            print(
                "Size:",
                f"{DEST.stat().st_size / 1024 / 1024:.3f} MiB"
            )

            print()
            print("=" * 100)
            print("UPLOAD COMPLETE")
            print("=" * 100)
            print()
            print("NEXT:")
            print("  Run the GitHub push cell.")

        except Exception as exc:

            # Never leave an unverified destination behind.
            if DEST.exists():

                try:

                    if hashlib.sha256(
                        DEST.read_bytes()
                    ).hexdigest() != EXPECTED_SHA256:

                        DEST.unlink()

                except Exception:
                    pass

            print()
            print("UPLOAD REJECTED")
            print()
            print(type(exc).__name__ + ":")
            print(exc)


uploader.observe(
    handle_upload,
    names="value",
)


display(
    widgets.VBox(
        [
            widgets.HTML(
                """
                <b>Stage23 notebook script upload</b><br>
                Select <code>stage23_shortcut_feature_audit_kaggle.py</code>.
                The file will be accepted only if its SHA256 exactly matches
                the verified export.
                """
            ),
            uploader,
            output,
        ]
    )
)

In [21]:
# ==================================================================================================
# POST-STAGE23 — COMMIT + PUSH VERIFIED FULL NOTEBOOK SCRIPT
#
# VERIFIED SOURCE
#   /kaggle/working/stage23_shortcut_feature_audit_kaggle.py
#
# DESTINATION
#   scripts/stage23_shortcut_feature_audit_kaggle.py
#
# SCRIPT SHA256
#   4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9
#
# SCIENTIFIC STAGE23 CLOSURE
#   commit: f9661acf152d6f1c5dd12de2d0f94f21e68d7170
#   tag:    stage23-shortcut-feature-audit-complete-v1
#
# IMPORTANT
#   Documentation/reproducibility commit only.
#   The scientific Stage23 tag MUST remain pinned to f9661acf...
# ==================================================================================================

from pathlib import Path
import subprocess
import hashlib
import shutil
import os
import stat


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SOURCE = Path(
    "/kaggle/working/stage23_shortcut_feature_audit_kaggle.py"
)

DEST_REL = Path(
    "scripts/stage23_shortcut_feature_audit_kaggle.py"
)

DEST = (
    REPO
    / DEST_REL
)

EXPECTED_SCRIPT_SHA = (
    "4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9"
)

SEALED_STAGE23_COMMIT = (
    "f9661acf152d6f1c5dd12de2d0f94f21e68d7170"
)

SEALED_STAGE23_TAG = (
    "stage23-shortcut-feature-audit-complete-v1"
)

COMMIT_MESSAGE = (
    "docs: add full Stage23 Kaggle notebook script"
)

SEP = "=" * 120


# ==================================================================================================
# HELPERS
# ==================================================================================================

def git(
    *args,
    check=True,
    env=None,
    show=False,
):

    merged_env = os.environ.copy()

    if env:
        merged_env.update(
            env
        )

    p = subprocess.run(
        [
            "git",
            *map(
                str,
                args,
            ),
        ],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )

    output = (
        p.stdout
        or ""
    ).strip()

    if (
        check
        and p.returncode != 0
    ):

        raise RuntimeError(
            "git command failed:\n"
            f"git {' '.join(map(str, args))}\n"
            f"{output}"
        )

    if (
        show
        and output
    ):
        print(
            output
        )

    return (
        p.returncode,
        output,
    )


def sha256_file(
    path,
    chunk=1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


# ==================================================================================================
# 1. PREFLIGHT
# ==================================================================================================

print(SEP)
print("POST-STAGE23 — PUSH VERIFIED FULL NOTEBOOK SCRIPT")
print(SEP)
print()


if not REPO.is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


if not SOURCE.is_file():

    raise RuntimeError(
        f"Verified uploaded script missing:\n{SOURCE}"
    )


_, branch = git(
    "branch",
    "--show-current",
)

_, head = git(
    "rev-parse",
    "HEAD",
)

_, status = git(
    "status",
    "--porcelain",
)

_, local_tag_commit = git(
    "rev-list",
    "-n",
    "1",
    SEALED_STAGE23_TAG,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != SEALED_STAGE23_COMMIT:

    raise RuntimeError(
        "Unexpected HEAD before documentation commit.\n"
        f"expected={SEALED_STAGE23_COMMIT}\n"
        f"actual={head}"
    )


if status:

    raise RuntimeError(
        "Repository dirty before documentation commit:\n"
        + status
    )


if local_tag_commit != SEALED_STAGE23_COMMIT:

    raise RuntimeError(
        "Scientific Stage23 tag moved locally.\n"
        f"expected={SEALED_STAGE23_COMMIT}\n"
        f"actual={local_tag_commit}"
    )


if DEST.exists():

    raise RuntimeError(
        "Destination already exists:\n"
        f"{DEST}\n"
        "Refusing overwrite."
    )


print("[EXACT] branch                  :", branch)
print("[EXACT] scientific HEAD         :", head)
print("[EXACT] scientific tag          :", SEALED_STAGE23_TAG)
print("[EXACT] scientific tag commit   :", local_tag_commit)
print("[EXACT] worktree                : CLEAN")
print()


# ==================================================================================================
# 2. VERIFY UPLOADED SCRIPT AGAIN
# ==================================================================================================

actual_sha = sha256_file(
    SOURCE
)


print("Expected script SHA256:")
print(
    " ",
    EXPECTED_SCRIPT_SHA
)

print("Actual script SHA256:")
print(
    " ",
    actual_sha
)


if actual_sha != EXPECTED_SCRIPT_SHA:

    raise RuntimeError(
        "Uploaded script SHA256 mismatch."
    )


text = SOURCE.read_text(
    encoding="utf-8"
)


cell_markers = text.count(
    "# %% [notebook cell "
)


if cell_markers != 75:

    raise RuntimeError(
        "Expected 75 notebook-cell markers.\n"
        f"actual={cell_markers}"
    )


# Syntax validation only — DOES NOT execute notebook code.
compile(
    text,
    str(
        SOURCE
    ),
    "exec",
)


if SOURCE.stat().st_size >= (
    95
    * 1024
    * 1024
):

    raise RuntimeError(
        "Script exceeds GitHub file-size safety limit."
    )


print()
print("[EXACT] script SHA256         : MATCH")
print("[EXACT] notebook code cells   : 75 / 75")
print("[EXACT] Python syntax         : PASS")
print(
    "[EXACT] script size           :",
    f"{SOURCE.stat().st_size / 1024 / 1024:.3f} MiB"
)


# ==================================================================================================
# 3. VERIFY REMOTE SCIENTIFIC TAG BEFORE ANY WRITE
# ==================================================================================================

_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEALED_STAGE23_TAG}^{{}}",
)


if not remote_tag_line:

    raise RuntimeError(
        "Could not resolve remote Stage23 scientific tag."
    )


remote_tag_before = remote_tag_line.split()[
    0
]


if remote_tag_before != SEALED_STAGE23_COMMIT:

    raise RuntimeError(
        "Remote Stage23 scientific tag moved.\n"
        f"expected={SEALED_STAGE23_COMMIT}\n"
        f"actual={remote_tag_before}"
    )


print(
    "[EXACT] remote scientific tag :",
    remote_tag_before,
)


# ==================================================================================================
# 4. LOAD GITHUB CREDENTIAL
# ==================================================================================================

github_token = (
    os.environ.get(
        "GITHUB_TOKEN"
    )
    or
    os.environ.get(
        "GH_TOKEN"
    )
)


if not github_token:

    try:

        from kaggle_secrets import (
            UserSecretsClient
        )

        secrets = UserSecretsClient()

        for secret_name in [
            "GITHUB_TOKEN",
            "github_token",
            "GH_TOKEN",
            "github_pat",
            "GITHUB_PAT",
        ]:

            try:

                candidate = secrets.get_secret(
                    secret_name
                )

            except Exception:

                candidate = None


            if candidate:

                github_token = candidate.strip()

                print(
                    "[OK] GitHub credential loaded from Kaggle Secrets:",
                    secret_name,
                )

                break

    except Exception:

        pass


if not github_token:

    raise RuntimeError(
        "GitHub credential unavailable.\n"
        "Repository has NOT been modified."
    )


ASKPASS = Path(
    "/kaggle/working/.stage23_notebook_script_askpass.sh"
)


ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *Password*) printf '%s\\n' "$STAGE23_NOTEBOOK_GITHUB_TOKEN" ;;
  *) printf '%s\\n' "" ;;
esac
""",
    encoding="utf-8",
)


ASKPASS.chmod(
    stat.S_IRUSR
    |
    stat.S_IWUSR
    |
    stat.S_IXUSR
)


auth_env = {
    "GIT_ASKPASS":
        str(
            ASKPASS
        ),

    "GIT_TERMINAL_PROMPT":
        "0",

    "STAGE23_NOTEBOOK_GITHUB_TOKEN":
        github_token,
}


# ==================================================================================================
# 5. AUTH PREFLIGHT — STILL NO REPOSITORY MODIFICATION
# ==================================================================================================

print()
print(SEP)
print("GITHUB PUSH AUTH PREFLIGHT")
print(SEP)
print()


rc, dryrun = git(
    "push",
    "--dry-run",
    "origin",
    "HEAD:main",
    check=False,
    env=auth_env,
)


if rc != 0:

    github_token = None

    try:
        ASKPASS.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "GitHub authentication preflight failed.\n"
        "Repository remains unchanged.\n"
        + dryrun
    )


print("[OK] GitHub push authentication available.")


# ==================================================================================================
# 6. COPY EXACT SCRIPT INTO REPOSITORY
# ==================================================================================================

print()
print(SEP)
print("COPY VERIFIED SCRIPT INTO REPOSITORY")
print(SEP)
print()


DEST.parent.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    SOURCE,
    DEST,
)


copied_sha = sha256_file(
    DEST
)


if copied_sha != EXPECTED_SCRIPT_SHA:

    try:
        DEST.unlink()
    except Exception:
        pass

    raise RuntimeError(
        "Repository copy SHA256 mismatch."
    )


print("[EXACT] copied byte-for-byte:")
print(
    " ",
    DEST_REL.as_posix()
)

print(
    "[EXACT] copied SHA256:",
    copied_sha,
)


# ==================================================================================================
# 7. STAGE ONLY SCRIPT
# ==================================================================================================

git(
    "add",
    "--",
    DEST_REL.as_posix(),
)


_, staged = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_files = [
    line.strip()
    for line in staged.splitlines()
    if line.strip()
]


if staged_files != [
    DEST_REL.as_posix()
]:

    raise RuntimeError(
        "Unexpected staged file set:\n"
        + "\n".join(
            staged_files
        )
    )


print(
    "[EXACT] staged file:",
    staged_files[
        0
    ],
)


# ==================================================================================================
# 8. COMMIT DOCUMENTATION EXPORT
# ==================================================================================================

print()
print(SEP)
print("COMMIT NOTEBOOK SCRIPT")
print(SEP)
print()


git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
    show=True,
)


_, documentation_commit = git(
    "rev-parse",
    "HEAD",
)


print()
print(
    "[OK] documentation commit:",
    documentation_commit,
)


# ==================================================================================================
# 9. VERIFY SCIENTIFIC TAG DID NOT MOVE LOCALLY
# ==================================================================================================

_, tag_after_commit = git(
    "rev-list",
    "-n",
    "1",
    SEALED_STAGE23_TAG,
)


if tag_after_commit != SEALED_STAGE23_COMMIT:

    raise RuntimeError(
        "CRITICAL: scientific Stage23 tag moved locally."
    )


# ==================================================================================================
# 10. PUSH MAIN ONLY
#
# DO NOT PUSH / MODIFY / RECREATE SCIENTIFIC TAG.
# ==================================================================================================

print()
print(SEP)
print("PUSH DOCUMENTATION COMMIT — MAIN ONLY")
print(SEP)
print()


git(
    "push",
    "origin",
    "main",
    env=auth_env,
    show=True,
)


# ==================================================================================================
# 11. REMOVE AUTH MATERIAL
# ==================================================================================================

github_token = None


try:

    ASKPASS.unlink()

except Exception:

    pass


# ==================================================================================================
# 12. REMOTE VERIFICATION
# ==================================================================================================

_, remote_main_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


_, remote_tag_line = git(
    "ls-remote",
    "origin",
    f"refs/tags/{SEALED_STAGE23_TAG}^{{}}",
)


if not remote_main_line:

    raise RuntimeError(
        "Could not resolve remote main."
    )


if not remote_tag_line:

    raise RuntimeError(
        "Could not resolve remote scientific Stage23 tag."
    )


remote_main = remote_main_line.split()[
    0
]

remote_tag_after = remote_tag_line.split()[
    0
]


if remote_main != documentation_commit:

    raise RuntimeError(
        "Remote main does not match documentation commit.\n"
        f"expected={documentation_commit}\n"
        f"actual={remote_main}"
    )


if remote_tag_after != SEALED_STAGE23_COMMIT:

    raise RuntimeError(
        "CRITICAL: scientific Stage23 tag moved remotely.\n"
        f"expected={SEALED_STAGE23_COMMIT}\n"
        f"actual={remote_tag_after}"
    )


if sha256_file(
    DEST
) != EXPECTED_SCRIPT_SHA:

    raise RuntimeError(
        "Committed repository script changed unexpectedly."
    )


# ==================================================================================================
# 13. FINAL CLEAN STATE
# ==================================================================================================

_, final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "Repository dirty after documentation push:\n"
        + final_status
    )


# ==================================================================================================
# 14. FINAL REPORT
# ==================================================================================================

print()
print(SEP)
print("FULL STAGE23 KAGGLE NOTEBOOK SCRIPT — PUSHED")
print(SEP)
print()

print("Repository file:")
print(
    " ",
    DEST_REL.as_posix(),
)

print()
print("Script SHA256:")
print(
    " ",
    EXPECTED_SCRIPT_SHA,
)

print()
print("Documentation commit:")
print(
    " ",
    documentation_commit,
)

print()
print("Remote main:")
print(
    " ",
    remote_main,
)

print()
print("Definitive scientific Stage23 tag:")
print(
    " ",
    SEALED_STAGE23_TAG,
)

print()
print("Scientific tag remains pinned to:")
print(
    " ",
    remote_tag_after,
)

print()
print("VERIFICATION")
print("  notebook code cells          : 75 / 75")
print("  Python syntax                : PASS")
print("  script SHA256                : EXACT")
print("  scientific tag moved         : NO")
print("  scientific artifacts changed : NO")
print("  additional model fits        : 0")
print("  scientific metrics computed  : 0")
print("  Raw Mar1 accessed            : NO")
print("  Raw Mar2 accessed            : NO")
print("  worktree                     : CLEAN")

print()
print("STATUS:")
print("  STAGE23_NOTEBOOK_SCRIPT_PUSHED")
print("  STAGE23_SCIENTIFIC_CLOSURE_REMAINS_IMMUTABLE")

print(SEP)

POST-STAGE23 — PUSH VERIFIED FULL NOTEBOOK SCRIPT

[EXACT] branch                  : main
[EXACT] scientific HEAD         : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] scientific tag          : stage23-shortcut-feature-audit-complete-v1
[EXACT] scientific tag commit   : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[EXACT] worktree                : CLEAN

Expected script SHA256:
  4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9
Actual script SHA256:
  4c54963f8748fbf7dd96c8e23ccd7a4ea434b0c61dd19733c229b392e15974e9

[EXACT] script SHA256         : MATCH
[EXACT] notebook code cells   : 75 / 75
[EXACT] Python syntax         : PASS
[EXACT] script size           : 2.298 MiB
[EXACT] remote scientific tag : f9661acf152d6f1c5dd12de2d0f94f21e68d7170
[OK] GitHub credential loaded from Kaggle Secrets: GITHUB_TOKEN

GITHUB PUSH AUTH PREFLIGHT

[OK] GitHub push authentication available.

COPY VERIFIED SCRIPT INTO REPOSITORY

[EXACT] copied byte-for-byte:
  scripts/stage23_shortcut